In [ ]:
%pip install numpy pandas scikit-learn scipy lightgbm jupyter

In [1]:
# ============================================================
# 0. Setup and raw data loading
# ============================================================

from pathlib import Path
import sys
import platform
import numpy as np
import pandas as pd

RANDOM_STATE = 9890
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# Change this only if your CSV files are stored in another folder.
DATA_DIR = Path(".")

def find_file(candidate_names, data_dir=DATA_DIR):
    """
    Looks for a file using a short list of possible names.
    This makes the notebook robust to files like school_covariates.csv
    versus school_covariates(2).csv.
    """
    for name in candidate_names:
        path = data_dir / name
        if path.exists():
            return path
    
    raise FileNotFoundError(
        "Could not find any of these files in "
        f"{data_dir.resolve()}:\n" + "\n".join(candidate_names)
    )

PATHS = {
    "school": find_file(["school_covariates.csv", "school_covariates(2).csv"]),
    "district": find_file(["district_covariates.csv", "district_covariates(2).csv"]),
    "train": find_file(["scores_training.csv", "scores_training(2).csv"]),
    "test": find_file(["scores_test.csv", "scores_test(2).csv"]),
}

school_covariates = pd.read_csv(PATHS["school"])
district_covariates = pd.read_csv(PATHS["district"])
scores_training = pd.read_csv(PATHS["train"])
scores_test = pd.read_csv(PATHS["test"])

# Preserve ID and categorical columns as strings.
STRING_COLS = [
    "ASSESSMENT_ID", "SCHOOL", "DISTRICT", "COUNTY",
    "SUBGROUP_NAME", "ASSESSMENT_NAME", "DISTRICT_TYPE", "REGION"
]

for df in [school_covariates, district_covariates, scores_training, scores_test]:
    for col in STRING_COLS:
        if col in df.columns:
            df[col] = df[col].astype("string")

print("Loaded files:")
for key, path in PATHS.items():
    print(f"  {key:8s}: {path}")

def raw_summary(name, df):
    return {
        "table": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "object_or_string_cols": int(
            df.select_dtypes(include=["object", "string"]).shape[1]
        ),
        "numeric_cols": int(df.select_dtypes(include=[np.number]).shape[1]),
    }

summary = pd.DataFrame([
    raw_summary("school_covariates", school_covariates),
    raw_summary("district_covariates", district_covariates),
    raw_summary("scores_training", scores_training),
    raw_summary("scores_test", scores_test),
])

print("\nRaw table summary:")
print(summary.to_string(index=False))

print("\nKey checks:")
print("school_covariates['SCHOOL'] unique:      ", school_covariates["SCHOOL"].is_unique)
print("district_covariates['DISTRICT'] unique:  ", district_covariates["DISTRICT"].is_unique)
print("scores_training['ASSESSMENT_ID'] unique: ", scores_training["ASSESSMENT_ID"].is_unique)
print("scores_test['ASSESSMENT_ID'] unique:     ", scores_test["ASSESSMENT_ID"].is_unique)

print("\nTarget checks:")
print("'PERCENT_PROFICIENT' in training:", "PERCENT_PROFICIENT" in scores_training.columns)
print("'PERCENT_PROFICIENT' in test:    ", "PERCENT_PROFICIENT" in scores_test.columns)

train_school_coverage = scores_training["SCHOOL"].isin(school_covariates["SCHOOL"]).mean()
test_school_coverage = scores_test["SCHOOL"].isin(school_covariates["SCHOOL"]).mean()

school_districts = school_covariates[["SCHOOL", "DISTRICT"]].drop_duplicates()

train_district_coverage = (
    scores_training[["SCHOOL"]]
    .drop_duplicates()
    .merge(school_districts, on="SCHOOL", how="left")["DISTRICT"]
    .isin(district_covariates["DISTRICT"])
    .mean()
)

test_district_coverage = (
    scores_test[["SCHOOL"]]
    .drop_duplicates()
    .merge(school_districts, on="SCHOOL", how="left")["DISTRICT"]
    .isin(district_covariates["DISTRICT"])
    .mean()
)

print("\nJoin coverage:")
print(f"training rows with SCHOOL in school_covariates: {train_school_coverage:.4f}")
print(f"test rows with SCHOOL in school_covariates:     {test_school_coverage:.4f}")
print(f"training unique schools with DISTRICT data:     {train_district_coverage:.4f}")
print(f"test unique schools with DISTRICT data:         {test_district_coverage:.4f}")

print("\nTarget summary:")
print(scores_training["PERCENT_PROFICIENT"].describe().to_string())

print("\nSoftware:")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Random state:", RANDOM_STATE)

Loaded files:
  school  : school_covariates.csv
  district: district_covariates.csv
  train   : scores_training.csv
  test    : scores_test.csv

Raw table summary:
              table   rows  cols  duplicate_rows  missing_cells  object_or_string_cols  numeric_cols
  school_covariates   4754    52               0          26460                      5            47
district_covariates    674     6               0              0                      1             5
    scores_training 144921     6               0              0                      4             2
        scores_test  48307     5               0              0                      4             1

Key checks:
school_covariates['SCHOOL'] unique:       True
district_covariates['DISTRICT'] unique:   True
scores_training['ASSESSMENT_ID'] unique:  True
scores_test['ASSESSMENT_ID'] unique:      True

Target checks:
'PERCENT_PROFICIENT' in training: True
'PERCENT_PROFICIENT' in test:     False

Join coverage:
training rows with 

In [2]:
# ============================================================
# 1. Merge datasets
# ============================================================

# Merge school covariates
train_full = scores_training.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one"
)

test_full = scores_test.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one"
)

# Merge district covariates
train_full = train_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one"
)

test_full = test_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one"
)

print("train_full shape:", train_full.shape)
print("test_full shape:", test_full.shape)

train_full shape: (144921, 62)
test_full shape: (48307, 61)


In [3]:
# ============================================================
# 2. Missingness overview
# ============================================================

def missing_report(df):
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    
    report = pd.DataFrame({
        "missing_count": miss,
        "missing_pct": (miss / len(df)) * 100
    })
    
    return report

train_missing = missing_report(train_full)
test_missing = missing_report(test_full)

print("Train missing columns:", train_missing.shape[0])
print("Test missing columns:", test_missing.shape[0])

print("\nTop 15 missing (train):")
print(train_missing.head(15))

print("\nTop 15 missing (test):")
print(test_missing.head(15))

Train missing columns: 52
Test missing columns: 52

Top 15 missing (train):
                                                    missing_count  missing_pct
TEACHER_TURNOVER_RATE                                      134136    92.558014
KINDERGARTEN_AVERAGE_CLASS_SIZE                            103467    71.395450
GRADE_1_AVERAGE_CLASS_SIZE                                 102899    71.003512
GRADE_2_AVERAGE_CLASS_SIZE                                 102779    70.920709
HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_...          89314    61.629439
PERCENT_DROPOUT                                             16061    11.082590
PERCENT_GED                                                 16061    11.082590
PERCENT_STILL_ENROLLED                                      16061    11.082590
PERCENT_NON_DIPLOMA                                         16061    11.082590
PERCENT_DIPLOMA                                             16061    11.082590
SCIENCE_AVERAGE_CLASS_SIZE                             

In [4]:
# ============================================================
# 3. School-level missingness check
# ============================================================

# Example column: ATTENDANCE_RATE (you can change later)
col = "ATTENDANCE_RATE"

train_missing_schools = train_full[train_full[col].isna()]["SCHOOL"].nunique()
test_missing_schools = test_full[test_full[col].isna()]["SCHOOL"].nunique()

train_missing_rows = train_full[col].isna().sum()
test_missing_rows = test_full[col].isna().sum()

print("Using column:", col)

print("\nTrain rows missing:", train_missing_rows)
print("Train unique schools missing:", train_missing_schools)

print("\nTest rows missing:", test_missing_rows)
print("Test unique schools missing:", test_missing_schools)

Using column: ATTENDANCE_RATE

Train rows missing: 3036
Train unique schools missing: 94

Test rows missing: 983
Test unique schools missing: 93


In [3]:
# ============================================================
# 4. Build X and y + preserve IDs
# ============================================================

TARGET = "PERCENT_PROFICIENT"
ID_COL = "ASSESSMENT_ID"

# Save IDs separately (needed for submission later)
train_ids = train_full[ID_COL].copy()
test_ids = test_full[ID_COL].copy()

# Target
y_train = train_full[TARGET].copy()

# Drop target from features
X_train = train_full.drop(columns=[TARGET])
X_test = test_full.copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)

X_train shape: (144921, 61)
X_test shape: (48307, 61)
y_train shape: (144921,)


In [6]:
# ============================================================
# 5. Column typing
# ============================================================

ID_COL = "ASSESSMENT_ID"

# Identify column types
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

# Remove ID from categorical if present
if ID_COL in categorical_cols:
    categorical_cols.remove(ID_COL)

# High-cardinality (simple rule: > 50 unique values)
high_cardinality_cols = [
    col for col in categorical_cols
    if X_train[col].nunique() > 50
]

low_cardinality_cols = [
    col for col in categorical_cols
    if col not in high_cardinality_cols
]

print("Numeric cols:", len(numeric_cols))
print("Categorical cols:", len(categorical_cols))
print("High-cardinality cols:", high_cardinality_cols)
print("Low-cardinality cols:", low_cardinality_cols)

Numeric cols: 53
Categorical cols: 7
High-cardinality cols: ['SCHOOL', 'DISTRICT', 'COUNTY']
Low-cardinality cols: ['SUBGROUP_NAME', 'ASSESSMENT_NAME', 'DISTRICT_TYPE', 'REGION']


In [7]:
# ============================================================
# 6A. Frequency encoding (safe, no leakage)
# ============================================================

X_train_proc = X_train.copy()
X_test_proc = X_test.copy()

freq_encoding_cols = []

for col in high_cardinality_cols:
    freq_map = X_train_proc[col].value_counts(dropna=False)
    
    new_col = col + "_freq"
    X_train_proc[new_col] = X_train_proc[col].map(freq_map).astype(float)
    X_test_proc[new_col] = X_test_proc[col].map(freq_map).fillna(0).astype(float)
    
    freq_encoding_cols.append(new_col)

print("Added frequency columns:", freq_encoding_cols)
print("X_train_proc shape:", X_train_proc.shape)
print("X_test_proc shape:", X_test_proc.shape)

Added frequency columns: ['SCHOOL_freq', 'DISTRICT_freq', 'COUNTY_freq']
X_train_proc shape: (144921, 64)
X_test_proc shape: (48307, 64)


In [8]:
# ============================================================
# 6B. Missing indicators + median imputation (more robust than mean)
# ============================================================

numeric_cols_extended = numeric_cols + freq_encoding_cols

missing_indicator_cols = []

for col in numeric_cols_extended:
    if X_train_proc[col].isna().sum() > 0:
        new_col = col + "_missing"
        
        X_train_proc[new_col] = X_train_proc[col].isna().astype(int)
        X_test_proc[new_col] = X_test_proc[col].isna().astype(int)
        
        median_val = X_train_proc[col].median()
        
        X_train_proc[col] = X_train_proc[col].fillna(median_val)
        X_test_proc[col] = X_test_proc[col].fillna(median_val)
        
        missing_indicator_cols.append(new_col)

print("Missing indicators added:", len(missing_indicator_cols))
print("New shape:", X_train_proc.shape)

Missing indicators added: 52
New shape: (144921, 116)


/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_95715/1325974498.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_proc[new_col] = X_train_proc[col].isna().astype(int)
/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_95715/1325974498.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test_proc[new_col] = X_test_proc[col].isna().astype(int)
/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_95715/1325974498.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the 

In [9]:
# ============================================================
# 6C. Drop raw high-cardinality categorical columns
# ============================================================

X_train_proc = X_train_proc.drop(columns=high_cardinality_cols)
X_test_proc = X_test_proc.drop(columns=high_cardinality_cols)

print("Shape after dropping high-cardinality cols:", X_train_proc.shape)

Shape after dropping high-cardinality cols: (144921, 113)


### Feature Encoding: High-Cardinality Variables

The raw high-cardinality categorical variables `SCHOOL`, `DISTRICT`, and `COUNTY` were removed from the modeling matrix after frequency encodings were created for them.

This prevents the model from directly using raw ID-like categorical labels while still preserving useful information about how frequently each school, district, or county appears in the training data.

After dropping these raw columns, the feature matrix has 115 columns.

In [10]:
# ============================================================
# 6D. One-hot encode low-cardinality categorical variables
# ============================================================

X_train_proc = pd.get_dummies(
    X_train_proc,
    columns=low_cardinality_cols,
    drop_first=False
)

X_test_proc = pd.get_dummies(
    X_test_proc,
    columns=low_cardinality_cols,
    drop_first=False
)

# Align columns (important!)
X_train_proc, X_test_proc = X_train_proc.align(X_test_proc, join="left", axis=1, fill_value=0)

print("Final X_train shape:", X_train_proc.shape)
print("Final X_test shape:", X_test_proc.shape)

Final X_train shape: (144921, 163)
Final X_test shape: (48307, 163)


### Final Feature Matrix

After full preprocessing:

- High-cardinality variables (`SCHOOL`, `DISTRICT`, `COUNTY`) were replaced with frequency encodings
- Missing values were handled via:
  - median imputation
  - explicit missingness indicator variables
- Low-cardinality categorical variables were one-hot encoded:
  - `SUBGROUP_NAME`, `ASSESSMENT_NAME`, `DISTRICT_TYPE`, `REGION`

Final dimensions:
- Training set: 144,921 rows × 165 features
- Test set: 48,307 rows × 165 features

The feature space is now fully numeric, aligned between train and test, and ready for modeling.

In [11]:
# ============================================================
# Create modeling copy WITHOUT ID (non-destructive)
# ============================================================

X_train_proc_model = X_train_proc.drop(columns=["ASSESSMENT_ID"])
X_test_proc_model = X_test_proc.drop(columns=["ASSESSMENT_ID"])

print("Modeling shape:", X_train_proc_model.shape)

Modeling shape: (144921, 162)


### Modeling Dataset

A separate modeling dataset was created by removing the identifier column `ASSESSMENT_ID`.

Final modeling dimensions:
- Training set: 144,921 rows × 164 features

This ensures that all features used for modeling are numeric or boolean, and that no identifier-based leakage occurs.

In [12]:
# ============================================================
# 7A. Simple Linear Regression (one feature at a time)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc_model, y_train, test_size=0.2, random_state=9890
)

simple_lr_results = []

for col in X_train_proc_model.columns:
    model = LinearRegression()
    model.fit(X_tr[[col]], y_tr)
    
    train_pred = model.predict(X_tr[[col]])
    val_pred = model.predict(X_val[[col]])
    
    simple_lr_results.append({
        "feature": col,
        "train_mse": mean_squared_error(y_tr, train_pred),
        "val_mse": mean_squared_error(y_val, val_pred)
    })

simple_lr_results = pd.DataFrame(simple_lr_results).sort_values("val_mse")

print(simple_lr_results.head(30).to_string(index=False))

                                                    feature  train_mse    val_mse
                         PERCENT_ECONOMICALLY_DISADVANTAGED 570.735244 581.066288
                                         PERCENT_FREE_LUNCH 575.082361 584.868424
                                            PERCENT_DIPLOMA 621.742550 626.029451
                                     PERCENT_STILL_ENROLLED 637.471784 641.000299
                                           PERCENT_HOMELESS 640.796399 644.070716
                                              PERCENT_BLACK 648.963234 652.648495
                                  PERCENT_WITH_DISABILITIES 650.316873 653.054139
                           PERCENT_ENGLISH_LANGUAGE_LEANERS 648.460937 655.348885
                                            ATTENDANCE_RATE 650.513825 657.793054
                                              PERCENT_WHITE 653.127789 658.601548
                                            PERCENT_DROPOUT 653.769449 661.927501
                

In [13]:
# ============================================================
# Extract best simple linear regression feature properly
# ============================================================

best_feature_row = simple_lr_results.loc[simple_lr_results["val_mse"].idxmin()]

print("Best single-feature model:")
print(best_feature_row)

Best single-feature model:
feature      PERCENT_ECONOMICALLY_DISADVANTAGED
train_mse                            570.735244
val_mse                              581.066288
Name: 43, dtype: object


### Simple Linear Regression Baseline

Each feature was tested individually in a simple linear regression model. This creates a baseline ranking of single predictors before fitting larger multiple regression models.

The goal is not to select the final model from one feature, but to identify which variables have the strongest individual linear relationship with `PERCENT_PROFICIENT`.

### Best Simple Linear Regression Feature

The best single-feature model was:

- Feature: `PERCENT_ECONOMICALLY_DISADVANTAGED`
- Train MSE: 570.74  
- Validation MSE: 581.07  

Interpretation:
- Socioeconomic disadvantage is the strongest standalone predictor of `PERCENT_PROFICIENT`.
- However, the error (~581) is substantially higher than the multiple linear regression model (~312), indicating that no single variable explains the outcome well.
- This confirms that predictive power in the dataset is distributed across multiple correlated features rather than dominated by a single factor.

Conclusion:
Simple linear regression provides insight into marginal relationships but is insufficient for accurate prediction on its own.

### Simple Linear Regression Insights

The strongest individual predictors of `PERCENT_PROFICIENT` are:

- Socioeconomic indicators:
  - `PERCENT_ECONOMICALLY_DISADVANTAGED`
  - `PERCENT_FREE_LUNCH`
- Academic outcomes:
  - `PERCENT_DIPLOMA`
  - `PERCENT_STILL_ENROLLED`
- Demographics:
  - `PERCENT_BLACK`, `PERCENT_WHITE`, `PERCENT_HISPANIC`
- Vulnerability indicators:
  - `PERCENT_HOMELESS`, `PERCENT_WITH_DISABILITIES`
- Attendance:
  - `ATTENDANCE_RATE`

Key observations:
- Socioeconomic disadvantage is the strongest single predictor.
- Many top features are highly correlated (e.g., free lunch vs economic disadvantage).
- Missingness indicators appear, confirming that missing data carries signal.
- Frequency-encoded variables are not dominant individually, suggesting entity effects are weaker in isolation.

Conclusion:
Simple linear regression highlights strong marginal relationships but does not account for interactions or multicollinearity.

In [14]:
# ============================================================
# 7B. Multiple Linear Regression (clean baseline)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Fresh split (consistent with 7A)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc_model, y_train, test_size=0.2, random_state=9890
)

model = LinearRegression()
model.fit(X_tr, y_tr)

train_pred = model.predict(X_tr)
val_pred = model.predict(X_val)

train_mse = mean_squared_error(y_tr, train_pred)
val_mse = mean_squared_error(y_val, val_pred)

print("Train MSE:", train_mse)
print("Validation MSE:", val_mse)

Train MSE: 305.12409327017724
Validation MSE: 312.6016702920729


### Multiple Linear Regression Baseline

A multiple linear regression model was fitted using all available features.

Results:
- Train MSE: 305.12  
- Validation MSE: 312.60  

Interpretation:
- The gap between training and validation error is small, indicating minimal overfitting.
- The model performs substantially better than the best simple linear regression (~581 MSE), showing that predictive power is distributed across multiple features.
- The relatively low validation error suggests that linear relationships capture a large portion of the underlying structure in the data.

Conclusion:
Multiple linear regression provides a strong and stable baseline for evaluating more complex models.

In [15]:
# Select top 10 features from simple LR
top_features = simple_lr_results.nsmallest(10, "val_mse")["feature"].tolist()

print(top_features)

['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']


In [16]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)

X_tr_poly = poly.fit_transform(X_tr[top_features])
X_val_poly = poly.transform(X_val[top_features])

print("Poly feature shape:", X_tr_poly.shape)

Poly feature shape: (115936, 65)


In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_tr_poly, y_tr)

train_pred = model.predict(X_tr_poly)
val_pred = model.predict(X_val_poly)

print("Polynomial Regression:")
print("Train MSE:", mean_squared_error(y_tr, train_pred))
print("Validation MSE:", mean_squared_error(y_val, val_pred))

Polynomial Regression:
Train MSE: 507.61481933233904
Validation MSE: 515.1911286593601


### Polynomial Regression

Polynomial regression (degree 2) was applied to the top 10 features identified from simple linear regression.

Results:
- Train MSE: 507.61  
- Validation MSE: 515.19  

Interpretation:
- Performance is significantly worse than multiple linear regression (~312 MSE).
- This indicates that restricting the model to a small subset of features removes important predictive information.
- The polynomial expansion does not compensate for the loss of breadth in the feature space.

Conclusion:
The dataset appears to benefit more from combining many features linearly rather than modeling nonlinear relationships among a small subset of variables. Polynomial regression is not effective in this setting.

In [18]:
top5_features = simple_lr_results.nsmallest(5, "val_mse")["feature"].tolist()
print(top5_features)

['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']


In [19]:
from itertools import combinations

interaction_cols = []

X_tr_int = X_tr.copy()
X_val_int = X_val.copy()

for f1, f2 in combinations(top5_features, 2):
    new_col = f"{f1}_x_{f2}"
    
    X_tr_int[new_col] = X_tr[f1] * X_tr[f2]
    X_val_int[new_col] = X_val[f1] * X_val[f2]
    
    interaction_cols.append(new_col)

print("Number of interaction features added:", len(interaction_cols))

Number of interaction features added: 10


In [20]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_tr_int, y_tr)

train_pred = model.predict(X_tr_int)
val_pred = model.predict(X_val_int)

print("Interaction Model:")
print("Train MSE:", mean_squared_error(y_tr, train_pred))
print("Validation MSE:", mean_squared_error(y_val, val_pred))

Interaction Model:
Train MSE: 304.3250676749579
Validation MSE: 311.955526746157


### Interaction Model

Pairwise interaction terms were added between the top 5 features identified from simple linear regression.

Results:
- Train MSE: 304.33  
- Validation MSE: 311.96  

Interpretation:
- The interaction model slightly improves performance compared to multiple linear regression (~312.60 → ~311.96).
- The improvement is marginal, suggesting that most of the predictive structure is already captured by additive linear effects.
- Interactions contribute some additional signal but are not a dominant factor in this dataset.

Conclusion:
While interaction terms provide a small improvement, the dataset is largely driven by additive relationships rather than strong nonlinear interactions.

In [21]:
# ============================================================
# 8A. Build controlled candidate feature spaces
# ============================================================

from itertools import combinations

# Use strongest marginal predictors as candidates for engineered terms
top10_features = simple_lr_results.nsmallest(10, "val_mse")["feature"].tolist()
top5_features = simple_lr_results.nsmallest(5, "val_mse")["feature"].tolist()

# Base space
X_tr_base = X_tr.copy()
X_val_base = X_val.copy()

# Base + polynomial squares for top10
X_tr_polyspace = X_tr.copy()
X_val_polyspace = X_val.copy()

poly_cols = []

for col in top10_features:
    new_col = col + "_squared"
    X_tr_polyspace[new_col] = X_tr[col] ** 2
    X_val_polyspace[new_col] = X_val[col] ** 2
    poly_cols.append(new_col)

# Base + pairwise interactions for top5
X_tr_intspace = X_tr.copy()
X_val_intspace = X_val.copy()

interaction_cols = []

for f1, f2 in combinations(top5_features, 2):
    new_col = f"{f1}_x_{f2}"
    X_tr_intspace[new_col] = X_tr[f1] * X_tr[f2]
    X_val_intspace[new_col] = X_val[f1] * X_val[f2]
    interaction_cols.append(new_col)

# Base + polynomial + interactions
X_tr_combined = X_tr_polyspace.copy()
X_val_combined = X_val_polyspace.copy()

for col in interaction_cols:
    X_tr_combined[col] = X_tr_intspace[col]
    X_val_combined[col] = X_val_intspace[col]

feature_spaces = {
    "base": (X_tr_base, X_val_base),
    "base_plus_poly": (X_tr_polyspace, X_val_polyspace),
    "base_plus_interactions": (X_tr_intspace, X_val_intspace),
    "base_plus_poly_interactions": (X_tr_combined, X_val_combined)
}

print("Top 10 features used for squares:")
print(top10_features)

print("\nTop 5 features used for interactions:")
print(top5_features)

print("\nPolynomial columns added:", len(poly_cols))
print("Interaction columns added:", len(interaction_cols))

print("\nFeature space shapes:")
for name, (X_train_space, X_val_space) in feature_spaces.items():
    print(name, X_train_space.shape, X_val_space.shape)

Top 10 features used for squares:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']

Top 5 features used for interactions:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']

Polynomial columns added: 10
Interaction columns added: 10

Feature space shapes:
base (115936, 162) (28985, 162)
base_plus_poly (115936, 172) (28985, 172)
base_plus_interactions (115936, 172) (28985, 172)
base_plus_poly_interactions (115936, 182) (28985, 182)


In [22]:
# ============================================================
# 8B. Forward stepwise selection across feature spaces
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd

def forward_stepwise(X_train_space, X_val_space, y_train, y_val, max_features=30):
    remaining = list(X_train_space.columns)
    selected = []
    rows = []
    best_mse = float("inf")
    
    while remaining and len(selected) < max_features:
        candidates = []
        
        for feature in remaining:
            trial_features = selected + [feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            val_mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, val_mse))
        
        best_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse < best_mse:
            selected.append(best_feature)
            remaining.remove(best_feature)
            best_mse = candidate_mse
            
            rows.append({
                "num_features": len(selected),
                "feature_added": best_feature,
                "val_mse": best_mse
            })
        else:
            break
    
    return pd.DataFrame(rows), selected

forward_summary = {}

for space_name, (X_train_space, X_val_space) in feature_spaces.items():
    results, selected = forward_stepwise(
        X_train_space, X_val_space, y_tr, y_val, max_features=30
    )
    
    forward_summary[space_name] = {
        "results": results,
        "selected_features": selected,
        "best_val_mse": results["val_mse"].min() if len(results) > 0 else None,
        "best_num_features": results.loc[results["val_mse"].idxmin(), "num_features"] if len(results) > 0 else None
    }
    
    print("\n" + "=" * 80)
    print(space_name)
    print("=" * 80)
    print(results.tail(10).to_string(index=False))
    print("Best validation MSE:", forward_summary[space_name]["best_val_mse"])

KeyboardInterrupt: 

### Forward Stepwise Design Choice

Forward stepwise selection was used to evaluate whether a smaller subset of predictors can approach the performance of the full multiple linear regression model.

Because the full feature space contains 164–184 predictors depending on the feature set, unrestricted stepwise selection would be computationally expensive and would gradually reconstruct the full model. To keep the procedure tractable and focused on model parsimony, the search was capped at 30 selected features.

This cap is not intended to imply that 30 is theoretically optimal. Instead, it provides a practical stopping limit that allows us to examine whether most predictive gains occur early in the selection path. If validation MSE is still improving near 30 features, the cap can be increased later.

### Forward Stepwise Selection

Forward stepwise selection was applied across multiple feature spaces, including:
- Base feature space
- Base + polynomial terms
- Base + interaction terms
- Base + polynomial + interaction terms

Results:
- Best validation MSE (base): 332.31  
- Best validation MSE (poly): 329.66  

Interpretation:
- All stepwise models perform significantly worse than the full multiple linear regression model (~312.60).
- This indicates that predictive performance relies on combining a large number of features rather than selecting a small subset.
- Polynomial and interaction features provide only marginal improvements within the stepwise framework.

Conclusion:
Subset selection via forward stepwise is not effective for this dataset. The data exhibits a high-dimensional additive structure where many weak predictors contribute jointly. Methods that retain all features while controlling complexity (e.g., Ridge or Lasso) are more appropriate.

In [23]:
# ============================================================
# Backward Stepwise (controlled)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd

def backward_stepwise(X_train_space, X_val_space, y_train, y_val, max_removals=30):
    
    selected = list(X_train_space.columns)
    results = []
    
    # Initial model (full)
    model = LinearRegression()
    model.fit(X_train_space[selected], y_train)
    val_pred = model.predict(X_val_space[selected])
    best_mse = mean_squared_error(y_val, val_pred)
    
    results.append({
        "num_features": len(selected),
        "removed_feature": None,
        "val_mse": best_mse
    })
    
    for _ in range(max_removals):
        candidates = []
        
        for feature in selected:
            trial_features = [f for f in selected if f != feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, mse))
        
        worst_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse <= best_mse:
            selected.remove(worst_feature)
            best_mse = candidate_mse
            
            results.append({
                "num_features": len(selected),
                "removed_feature": worst_feature,
                "val_mse": best_mse
            })
        else:
            break
    
    return pd.DataFrame(results)


backward_summary = {}

for name, (X_train_space, X_val_space) in feature_spaces.items():
    print("\n" + "="*80)
    print(name)
    print("="*80)
    
    results = backward_stepwise(X_train_space, X_val_space, y_tr, y_val, max_removals=30)
    
    backward_summary[name] = results
    
    print(results.tail(10).to_string(index=False))
    print("Best val MSE:", results["val_mse"].min())


base
 num_features                               removed_feature    val_mse
          154                               PERCENT_DIPLOMA 312.396761
          153                                      GRADE_01 312.387188
          152                                      GRADE_12 312.378367
          151                                 PERCENT_ASIAN 312.378267
          150             REGION_Capital District, New York 312.378267
          149               REGION_North Country (New York) 312.377630
          148                       PERCENT_DIPLOMA_missing 312.377630
          147 ASSESSMENT_NAME_Regents Common Core Algebra I 312.377630
          146                          ASSESSMENT_NAME_ELA4 312.377146
          145                 DISTRICT_TYPE_High-Need Rural 312.377146
Best val MSE: 312.3771461462088

base_plus_poly
 num_features                                      removed_feature    val_mse
          164                                             GRADE_07 308.801547
         

In [24]:
# ============================================================
# Hybrid Stepwise (forward + backward cleanup)
# ============================================================

def hybrid_stepwise(X_train_space, X_val_space, y_train, y_val, max_features=30):
    
    remaining = list(X_train_space.columns)
    selected = []
    results = []
    
    best_mse = float("inf")
    
    while remaining and len(selected) < max_features:
        
        # Forward step
        candidates = []
        
        for feature in remaining:
            trial_features = selected + [feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, mse))
        
        best_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse < best_mse:
            selected.append(best_feature)
            remaining.remove(best_feature)
            best_mse = candidate_mse
        else:
            break
        
        # Backward cleanup step
        improved = True
        while improved and len(selected) > 1:
            improved = False
            
            for feature in selected:
                trial_features = [f for f in selected if f != feature]
                
                model = LinearRegression()
                model.fit(X_train_space[trial_features], y_train)
                
                val_pred = model.predict(X_val_space[trial_features])
                mse = mean_squared_error(y_val, val_pred)
                
                if mse < best_mse:
                    selected.remove(feature)
                    best_mse = mse
                    improved = True
                    break
        
        results.append({
            "num_features": len(selected),
            "val_mse": best_mse
        })
    
    return pd.DataFrame(results)


hybrid_summary = {}

for name, (X_train_space, X_val_space) in feature_spaces.items():
    print("\n" + "="*80)
    print(name)
    print("="*80)
    
    results = hybrid_stepwise(X_train_space, X_val_space, y_tr, y_val, max_features=30)
    
    hybrid_summary[name] = results
    
    print(results.tail(10).to_string(index=False))
    print("Best val MSE:", results["val_mse"].min())


base
 num_features    val_mse
           21 356.639984
           22 352.619203
           23 348.915113
           24 346.330326
           25 344.123928
           26 341.579203
           27 338.762894
           28 336.466988
           29 334.174799
           30 332.314849
Best val MSE: 332.3148493948422

base_plus_poly
 num_features    val_mse
           21 352.226262
           22 348.874109
           23 346.326338
           24 343.776219
           25 341.160388
           26 338.657725
           27 336.137043
           28 333.717583
           29 331.600432
           30 329.660042
Best val MSE: 329.6600419738662

base_plus_interactions
 num_features    val_mse
           21 356.639984
           22 352.619203
           23 348.915113
           24 346.330326
           25 344.123928
           26 341.579203
           27 338.762894
           28 336.466988
           29 334.174799
           30 332.314849
Best val MSE: 332.3148493948422

base_plus_poly_interactions
 num

### Backward and Hybrid Stepwise Selection

Backward and hybrid stepwise selection were evaluated across four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Backward stepwise performed better than forward stepwise because it began with the full model and removed only features whose exclusion improved or preserved validation performance.

Best backward stepwise validation MSE values:

- Base: 312.35
- Base + polynomial: 308.77
- Base + interactions: 311.70
- Base + polynomial + interactions: 308.10

The best backward stepwise model was the base + polynomial + interaction model, with validation MSE of 308.10. This improves on the full multiple linear regression baseline of approximately 312.60.

The hybrid stepwise results matched the forward stepwise results exactly, suggesting that the backward cleanup phase did not remove any features after forward additions. Therefore, in this implementation, hybrid stepwise effectively behaved like forward stepwise.

Interpretation:
- Forward stepwise performed worse because it was limited to 30 selected features and could not capture the distributed signal across many predictors.
- Backward stepwise performed better because it retained most of the full feature space while pruning redundant or harmful variables.
- Polynomial terms provided meaningful improvement when added to the full feature space and pruned through backward selection.
- Interaction terms alone provided only modest improvement.

Conclusion:
The strongest linear-model-family result so far is backward stepwise on the combined polynomial + interaction feature space. This suggests that the dataset is mostly additive and high-dimensional, but selected nonlinear terms can improve performance when incorporated carefully.

In [23]:
# ============================================================
# Validation protocol note
# ============================================================

VALIDATION_PROTOCOL = {
    "current_stage": "development_holdout",
    "split_method": "train_test_split",
    "test_size": 0.20,
    "random_state": 9890,
    "final_stage": "cross_validation_on_shortlist",
    "note": (
        "Current MSE values are development validation scores. "
        "Cross-validation will be run later on shortlisted models."
    )
}

VALIDATION_PROTOCOL

{'current_stage': 'development_holdout',
 'split_method': 'train_test_split',
 'test_size': 0.2,
 'random_state': 9890,
 'final_stage': 'cross_validation_on_shortlist',
 'note': 'Current MSE values are development validation scores. Cross-validation will be run later on shortlisted models.'}

### Cross-Validation Strategy

At this stage, models are being evaluated using a fixed train/validation split. These results are useful for rapid model screening, but they should not be treated as final estimates of generalization performance.

Cross-validation will be applied later after the main model families have been tested. This avoids excessive computation during the exploratory phase while still allowing rigorous comparison among serious finalist models.

Current terminology:
- `val_mse`: development holdout validation MSE
- `cv_mse`: cross-validated MSE, to be computed later for shortlisted models

Planned approach:
1. Use the current validation split to screen many model classes quickly.
2. Record all results in a model ledger.
3. Shortlist the strongest models.
4. Run cross-validation on the shortlist.
5. Tune/refine the strongest cross-validated candidates.
6. Select the final model for submission.

For models involving target encoding or feature selection, special care will be needed to avoid leakage. Target encoding should be performed out-of-fold, and feature selection may need to be repeated inside folds if the selected model becomes a serious finalist.

In [24]:
# ============================================================
# 9A. Ridge Regression: two-stage alpha search + scaler comparison
# ============================================================

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Design choices
# ------------------------------------------------------------
# We compare StandardScaler and RobustScaler instead of assuming one.
# We use a two-stage alpha search:
#   1. Broad search across many orders of magnitude
#   2. Fine search around the best alpha from the broad search
#
# Cross-validation is intentionally postponed until later finalist screening.
# These are still development holdout validation results.

scaler_factories = {
    "standard": StandardScaler,
    "robust": RobustScaler
}

broad_alphas = np.logspace(-4, 8, 49)  # 0.0001 to 100,000,000

def evaluate_ridge_grid(X_train_space, X_val_space, y_train, y_val, alphas, scaler_name, scaler_factory, stage):
    rows = []
    
    for alpha in alphas:
        model = make_pipeline(
            scaler_factory(),
            Ridge(alpha=alpha)
        )
        
        model.fit(X_train_space, y_train)
        
        train_pred = model.predict(X_train_space)
        val_pred = model.predict(X_val_space)
        
        rows.append({
            "model_class": "Ridge",
            "stage": stage,
            "scaler": scaler_name,
            "alpha": alpha,
            "train_mse": mean_squared_error(y_train, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "aic": np.nan,
            "bic": np.nan
        })
    
    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Stage 1: broad search
# ------------------------------------------------------------

ridge_broad_rows = []

for feature_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for scaler_name, scaler_factory in scaler_factories.items():
        
        result = evaluate_ridge_grid(
            X_train_space=X_train_space,
            X_val_space=X_val_space,
            y_train=y_tr,
            y_val=y_val,
            alphas=broad_alphas,
            scaler_name=scaler_name,
            scaler_factory=scaler_factory,
            stage="broad"
        )
        
        result["feature_space"] = feature_space_name
        ridge_broad_rows.append(result)

ridge_broad_results = pd.concat(ridge_broad_rows, ignore_index=True)

best_broad_by_space_scaler = (
    ridge_broad_results
    .loc[ridge_broad_results.groupby(["feature_space", "scaler"])["val_mse"].idxmin()]
    .sort_values("val_mse")
)

print("Best broad Ridge result by feature space and scaler:")
print(best_broad_by_space_scaler.to_string(index=False))


# ------------------------------------------------------------
# Stage 2: fine search around each broad-search winner
# ------------------------------------------------------------

ridge_fine_rows = []

for _, row in best_broad_by_space_scaler.iterrows():
    feature_space_name = row["feature_space"]
    scaler_name = row["scaler"]
    scaler_factory = scaler_factories[scaler_name]
    best_alpha = row["alpha"]
    
    # Search within +/- 0.75 log10 units around the broad winner.
    # This is about a 5.6x range on either side.
    log_alpha = np.log10(best_alpha)
    fine_low = max(np.log10(broad_alphas.min()), log_alpha - 0.75)
    fine_high = min(np.log10(broad_alphas.max()), log_alpha + 0.75)
    
    fine_alphas = np.logspace(fine_low, fine_high, 31)
    
    X_train_space, X_val_space = feature_spaces[feature_space_name]
    
    result = evaluate_ridge_grid(
        X_train_space=X_train_space,
        X_val_space=X_val_space,
        y_train=y_tr,
        y_val=y_val,
        alphas=fine_alphas,
        scaler_name=scaler_name,
        scaler_factory=scaler_factory,
        stage="fine"
    )
    
    result["feature_space"] = feature_space_name
    ridge_fine_rows.append(result)

ridge_fine_results = pd.concat(ridge_fine_rows, ignore_index=True)


# ------------------------------------------------------------
# Combine broad + fine results and select winners
# ------------------------------------------------------------

ridge_results = pd.concat(
    [ridge_broad_results, ridge_fine_results],
    ignore_index=True
)

best_ridge_by_space = (
    ridge_results
    .loc[ridge_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_ridge_overall = ridge_results.loc[ridge_results["val_mse"].idxmin()]

print("\nBest Ridge result by feature space:")
print(best_ridge_by_space.to_string(index=False))

print("\nOverall best Ridge result:")
print(best_ridge_overall)

# ------------------------------------------------------------
# Optional warning: if best alpha is at broad grid boundary
# ------------------------------------------------------------

if best_ridge_overall["alpha"] == broad_alphas.min():
    print("\nWARNING: Best alpha is at the minimum broad-grid boundary. Consider expanding lower.")
elif best_ridge_overall["alpha"] == broad_alphas.max():
    print("\nWARNING: Best alpha is at the maximum broad-grid boundary. Consider expanding higher.")

Best broad Ridge result by feature space and scaler:
model_class stage   scaler    alpha  train_mse    val_mse  aic  bic               feature_space
      Ridge broad standard 1.778279 300.906794 308.357271  NaN  NaN base_plus_poly_interactions
      Ridge broad   robust 0.316228 300.891402 308.358944  NaN  NaN base_plus_poly_interactions
      Ridge broad standard 0.000100 301.622160 308.925244  NaN  NaN              base_plus_poly
      Ridge broad   robust 0.000100 301.622160 308.925245  NaN  NaN              base_plus_poly
      Ridge broad standard 0.000100 304.325068 311.955527  NaN  NaN      base_plus_interactions
      Ridge broad   robust 0.000100 304.325068 311.955527  NaN  NaN      base_plus_interactions
      Ridge broad standard 1.778279 305.124297 312.601534  NaN  NaN                        base
      Ridge broad   robust 0.000100 305.124093 312.601671  NaN  NaN                        base

Best Ridge result by feature space:
model_class stage   scaler    alpha  train_mse

### Ridge Regression Results

Ridge regression was evaluated across four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Both `StandardScaler` and `RobustScaler` were compared, and alpha was tuned using a two-stage holdout-validation search.

Best Ridge model:
- Feature space: base + polynomial + interactions
- Scaler: StandardScaler
- Alpha: 1.41
- Train MSE: 300.90
- Validation MSE: 308.36

Interpretation:
- Ridge performs best on the combined polynomial + interaction feature space.
- StandardScaler slightly outperforms RobustScaler.
- Ridge improves over the base multiple linear regression model but does not quite outperform the best backward stepwise model.
- The improvement appears to come mainly from the engineered feature space rather than from shrinkage alone.

Conclusion:
Ridge is a strong regularized linear model, but the current best linear-family model remains backward stepwise on the combined polynomial + interaction feature space, with validation MSE around 308.10.

In [27]:
# ============================================================
# 10A. Lasso Regression: data-driven two-stage alpha search
# ============================================================

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time
import warnings
from sklearn.exceptions import ConvergenceWarning

# ------------------------------------------------------------
# Design choices
# ------------------------------------------------------------
# - Feature spaces: all four controlled spaces
# - Scalers: StandardScaler and RobustScaler
# - Alpha search:
#     Stage 1: broad data-driven search from alpha_max downward
#     Stage 2: fine search around the best broad alpha
# - Metric: development holdout validation MSE
# - Extra output: number of nonzero coefficients
# - CV is intentionally postponed until finalist screening

scaler_factories = {  #This means every feature space will be tested twice: once with standard scaling and once with robust scaling.
    "standard": StandardScaler,
    "robust": RobustScaler
}
# The constants below control the search over Lasso’s tuning parameter,
LASSO_BROAD_GRID_SIZE = 25 # means the first search tries 25 alpha values per feature-space/scaler combination.
LASSO_FINE_GRID_SIZE = 21 # means the second search tries 21 more alpha values near the best one from the broad search.
LASSO_MIN_ALPHA_RATIO = 1e-4 # means the broad search goes from alpha_max down to alpha_max * 0.0001
LASSO_FINE_WIDTH_LOG10 = 0.5 # means the fine search checks values about 3.16 times above and below the best broad alpha, because 10^0.5 ≈ 3.16.

LASSO_MAX_ITER = 30000 # gives the solver many iterations to converge. Lasso can be harder to optimize than Ridge, especially with correlated predictors.
LASSO_TOL = 1e-4 # controls how precise the optimization has to be before it stops.

def compute_lasso_alpha_max(X_scaled, y): # This computes the largest alpha value worth trying.
    """
    Computes alpha_max for sklearn's Lasso objective:
        (1 / (2n)) * ||y - Xw||^2 + alpha * ||w||_1

    At alpha >= alpha_max, all coefficients are zero.
    """
    y_arr = np.asarray(y, dtype=float)
    y_centered = y_arr - y_arr.mean()
    n = X_scaled.shape[0] # the number of training samples (rows) in your dataset.
    alpha_max = np.max(np.abs(X_scaled.T @ y_centered)) / n  # X_scaled.T @ y_centered (@ means matrix muliplication) measures how strongly each feature is associated with the centered target. The feature with the strongest relationship to the target determines the largest penalty needed to force every coefficient to zero. This is smarter than choosing a random alpha grid. Lasso’s useful alpha range depends heavily on the scale of the features and the target. So instead of searching arbitrary values like 0.001 to 1000, this code builds a custom alpha range for each feature space and scaler. "How strongly does this feature move with the target?"
    # If a feature increases when y increases → big positive value
    # If a feature decreases when y increases → big negative value
    # If a feature is unrelated → value near 0
    return float(alpha_max)
    #the feature most strongly related to the target, That’s exactly what determines the largest alpha where Lasso wipes everything out.

def fit_lasso_path( # This function fits many Lasso models over a list of alpha values and records the results.
    X_train_space,
    X_val_space,
    y_train,
    y_val,
    alphas,
    scaler_name,
    scaler_factory,
    feature_space_name,
    stage
):
    """
    Fits Lasso models over a path of alphas using warm starts.
    Alphas should be sorted from largest to smallest.
    """
    rows = []
    # the scaler below is fit only on the training data, we learn the scaling parameters from the training split, then later apply the same transformation to the validation split.
    scaler = scaler_factory()
    X_train_scaled = scaler.fit_transform(X_train_space)
    X_val_scaled = scaler.transform(X_val_space)
    
    model = Lasso(
        alpha=alphas[0],
        fit_intercept=True, # means the model includes an intercept term. The intercept is not penalized by Lasso.
        max_iter=LASSO_MAX_ITER,
        tol=LASSO_TOL,
        warm_start=True, # warm_start=True is important. The alpha values are sorted from largest to smallest. The solution for a large alpha is a good starting point for the next slightly smaller alpha. Warm starts let the model reuse the previous fitted coefficients instead of starting from scratch each time. This can make the alpha path much faster.
        random_state=9890
    )
    
    for alpha in alphas: # the function loops through the alpha values
        model.alpha = alpha
        
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train_scaled, y_train)              # For each alpha, it fits the model, predicts on the training set and validation set, and records the MSE.
            
            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )   # A convergence warning means sklearn is saying, roughly, “I stopped before fully solving the optimization problem.” That does not always make the result useless, but it is a warning that you may need more iterations, stronger regularization, better scaling, or a different tolerance.
        
        train_pred = model.predict(X_train_scaled)
        val_pred = model.predict(X_val_scaled)
        
        nonzero_coef = int(np.sum(np.abs(model.coef_) > 1e-8))  # counts how many coefficients are nonzero:
        
        rows.append({
            "model_class": "Lasso",
            "stage": stage,
            "feature_space": feature_space_name,
            "scaler": scaler_name,
            "alpha": alpha,
            "train_mse": mean_squared_error(y_train, train_pred),  # tells you how well the model fits the training split.
            "val_mse": mean_squared_error(y_val, val_pred),  # is the main number used for model selection.
            "nonzero_coef": nonzero_coef,  #  tells you how sparse the model is.
            "n_iter": model.n_iter_,  # tells you how many iterations the solver used.
            "convergence_warning": convergence_warning,
            "aic": np.nan,
            "bic": np.nan
        })
    
    return pd.DataFrame(rows)

# ------------------------------------------------------------
# Stage 1: broad search
# ------------------------------------------------------------

start_time = time.perf_counter()

lasso_broad_rows = []
alpha_max_records = []
# This loops over every combination of feature space and scaler.There are four feature spaces and two scalers, so there are eight combinations:
for feature_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for scaler_name, scaler_factory in scaler_factories.items():
        
        scaler = scaler_factory()
        X_train_scaled = scaler.fit_transform(X_train_space)
        
        alpha_max = compute_lasso_alpha_max(X_train_scaled, y_tr)  # For each combination, it computes that setup’s alpha_max
        alpha_min = alpha_max * LASSO_MIN_ALPHA_RATIO
        
        # Descending alpha path for warm starts
        broad_alphas = np.logspace(
            np.log10(alpha_max),
            np.log10(alpha_min),
            LASSO_BROAD_GRID_SIZE
        ) # Because the first argument is larger than the second, this creates descending values. That is intentional. The model starts with the strongest penalty and moves toward weaker penalties. This works well with warm_start=True.
        
        alpha_max_records.append({ # The code records the alpha range
            "feature_space": feature_space_name,
            "scaler": scaler_name,
            "alpha_max": alpha_max,
            "alpha_min": alpha_min
        })
        
        result = fit_lasso_path( # Then it fits all 25 broad-search Lasso models for that setup:
            X_train_space=X_train_space,
            X_val_space=X_val_space,
            y_train=y_tr,
            y_val=y_val,
            alphas=broad_alphas,
            scaler_name=scaler_name,
            scaler_factory=scaler_factory,
            feature_space_name=feature_space_name,
            stage="broad"
        )
        
        lasso_broad_rows.append(result)
# After the loops finish, it combines all broad-search results:
lasso_alpha_max_table = pd.DataFrame(alpha_max_records)
lasso_broad_results = pd.concat(lasso_broad_rows, ignore_index=True) # Since there are eight setup combinations and 25 alphas each, the broad search fits 8 × 25 = 200 Lasso models

best_broad_by_space_scaler = ( # picks the best broad alpha for each feature-space/scaler pair:
    lasso_broad_results
    .loc[lasso_broad_results.groupby(["feature_space", "scaler"])["val_mse"].idxmin()]
    .sort_values("val_mse")
)   # This groups the results by feature_space and scaler, finds the row with the smallest validation MSE in each group, and sorts those eight winners from best to worst.

print("Lasso alpha_max table:")
print(lasso_alpha_max_table.to_string(index=False))

print("\nBest broad Lasso result by feature space and scaler:")
print(best_broad_by_space_scaler.to_string(index=False))


# ------------------------------------------------------------
# Stage 2: fine search around each broad-search winner
# ------------------------------------------------------------

lasso_fine_rows = []

for _, row in best_broad_by_space_scaler.iterrows():   # The code loops over the eight broad-search winners
    feature_space_name = row["feature_space"]
    scaler_name = row["scaler"]
    scaler_factory = scaler_factories[scaler_name]
    best_alpha = row["alpha"]   # For each winner, it gets the best alpha from the broad search:
    
    alpha_record = lasso_alpha_max_table[ # Then it retrieves the original alpha bounds for that feature-space/scaler combination:
        (lasso_alpha_max_table["feature_space"] == feature_space_name) &
        (lasso_alpha_max_table["scaler"] == scaler_name)
    ].iloc[0]
    
    alpha_max = alpha_record["alpha_max"]
    alpha_min = alpha_record["alpha_min"]
    # Then it builds a narrower alpha range around the broad-search winner:
    best_log = np.log10(best_alpha)
    fine_low = max(np.log10(alpha_min), best_log - LASSO_FINE_WIDTH_LOG10)
    fine_high = min(np.log10(alpha_max), best_log + LASSO_FINE_WIDTH_LOG10) # Because LASSO_FINE_WIDTH_LOG10 = 0.5, this searches approximately 3.16 times below and 3.16 times above the broad-search winner, clipped so it never goes outside the original broad-search range.
    
    # Descending alpha path for warm starts
    fine_alphas = np.logspace(fine_high, fine_low, LASSO_FINE_GRID_SIZE)  # this creates descending alpha values for warm starts. The fine search fits 21 more models for each of the eight broad winners: 8 × 21 = 168 Lasso models. so the whole cell fits 200 broad models + 168 fine models = 368 Lasso models
    
    X_train_space, X_val_space = feature_spaces[feature_space_name]
    
    result = fit_lasso_path(
        X_train_space=X_train_space,
        X_val_space=X_val_space,
        y_train=y_tr,
        y_val=y_val,
        alphas=fine_alphas,
        scaler_name=scaler_name,
        scaler_factory=scaler_factory,
        feature_space_name=feature_space_name,
        stage="fine"
    )
    
    lasso_fine_rows.append(result)

lasso_fine_results = pd.concat(lasso_fine_rows, ignore_index=True)

# ------------------------------------------------------------
# Combine broad + fine results and select winners
# ------------------------------------------------------------

lasso_results = pd.concat(
    [lasso_broad_results, lasso_fine_results],
    ignore_index=True
)

best_lasso_by_space = ( # Then it finds the best Lasso model for each feature space. This gives one winner for each of the four feature spaces, regardless of scaler, alpha, or search stage.
    lasso_results
    .loc[lasso_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_lasso_overall = lasso_results.loc[lasso_results["val_mse"].idxmin()]  # Then it finds the single best Lasso model overall

elapsed = time.perf_counter() - start_time

print("\nBest Lasso result by feature space:")
print(best_lasso_by_space.to_string(index=False))

print("\nOverall best Lasso result:")
print(best_lasso_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

print("\nConvergence warning counts:")
print(
    lasso_results
    .groupby(["feature_space", "scaler", "stage"])["convergence_warning"]
    .sum()
    .reset_index()
    .to_string(index=False)
)

Lasso alpha_max table:
              feature_space   scaler  alpha_max  alpha_min
                       base standard  11.255885   0.001126
                       base   robust   9.898994   0.000990
             base_plus_poly standard  11.255885   0.001126
             base_plus_poly   robust  23.219510   0.002322
     base_plus_interactions standard  11.255885   0.001126
     base_plus_interactions   robust   9.898994   0.000990
base_plus_poly_interactions standard  11.255885   0.001126
base_plus_poly_interactions   robust  23.219510   0.002322

Best broad Lasso result by feature space and scaler:
model_class stage               feature_space   scaler    alpha  train_mse    val_mse  nonzero_coef  n_iter  convergence_warning  aic  bic
      Lasso broad base_plus_poly_interactions standard 0.001126 301.032379 308.416448           133   30000                 True  NaN  NaN
      Lasso broad base_plus_poly_interactions   robust 0.002322 301.362320 308.808215           128    6321       

### Lasso Regression Results

Lasso regression was evaluated across the four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Both `StandardScaler` and `RobustScaler` were compared. Alpha was tuned using a data-driven two-stage search based on `alpha_max`.

Best Lasso model:
- Feature space: base + polynomial + interactions
- Scaler: StandardScaler
- Alpha: 0.001126
- Train MSE: 301.03
- Validation MSE: 308.42
- Nonzero coefficients: 134

Interpretation:
- The best Lasso model used the combined polynomial + interaction feature space.
- StandardScaler outperformed RobustScaler.
- The best alpha was at the lowest value in the search grid, indicating that weak regularization performed best.
- Lasso retained 134 nonzero coefficients, so it did not produce a highly sparse model.
- This supports the earlier finding that predictive signal is distributed across many predictors rather than concentrated in a small subset.

Conclusion:
Lasso provides useful feature selection but does not currently outperform Ridge or backward stepwise selection. The results suggest that aggressive sparsity is not ideal for this dataset.

### Process above: 

Try Lasso on every engineered feature set, try both standard and robust scaling, choose a sensible alpha range based on the data, do a broad alpha search, refine around the best alpha, track validation MSE and sparsity, then report the best Lasso configuration to compare against Ridge and earlier linear models.

In [28]:
# ============================================================
# 11A. Elastic Net Regression: two-stage search
# ============================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

# ------------------------------------------------------------
# Design choices
# ------------------------------------------------------------
# - Feature spaces: all four controlled feature spaces
# - Scalers: StandardScaler and RobustScaler
# - l1_ratio grid: from mostly-Ridge to near-Lasso
# - alpha grid: data-driven alpha_max, then two-stage search
# - Validation: current development holdout split
# - CV: intentionally postponed until finalist screening
# - Checkpointing: saves results after every path

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ELASTICNET_RESULTS_PATH = RESULTS_DIR / "elasticnet_results_holdout.csv"
ELASTICNET_BEST_BY_SPACE_PATH = RESULTS_DIR / "elasticnet_best_by_space_holdout.csv"
ELASTICNET_BEST_OVERALL_PATH = RESULTS_DIR / "elasticnet_best_overall_holdout.csv"

OVERWRITE_ELASTICNET_RESULTS = True

if OVERWRITE_ELASTICNET_RESULTS:
    for path in [
        ELASTICNET_RESULTS_PATH,
        ELASTICNET_BEST_BY_SPACE_PATH,
        ELASTICNET_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

scaler_factories = {
    "standard": StandardScaler,
    "robust": RobustScaler
}

elasticnet_l1_ratios = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

ELASTICNET_BROAD_GRID_SIZE = 25
ELASTICNET_FINE_GRID_SIZE = 21
ELASTICNET_MIN_ALPHA_RATIO = 1e-4
ELASTICNET_FINE_WIDTH_LOG10 = 0.5

ELASTICNET_MAX_ITER = 50000
ELASTICNET_TOL = 1e-4


def append_checkpoint(df, path):
    """Append results to a CSV checkpoint file."""
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)


def compute_alpha_max_lasso_base(X_scaled, y):
    """
    Computes max_j |x_j^T (y - ybar)| / n.

    For Elastic Net, alpha_max depends on l1_ratio:
        alpha_max_enet = alpha_max_lasso_base / l1_ratio

    This matches sklearn's ElasticNet objective:
        (1 / (2n)) * ||y - Xw||^2
        + alpha * l1_ratio * ||w||_1
        + 0.5 * alpha * (1 - l1_ratio) * ||w||_2^2
    """
    y_arr = np.asarray(y, dtype=float)
    y_centered = y_arr - y_arr.mean()
    n = X_scaled.shape[0]
    alpha_max_base = np.max(np.abs(X_scaled.T @ y_centered)) / n
    
    if alpha_max_base <= 0:
        raise ValueError("alpha_max_base is non-positive; check X/y inputs.")
    
    return float(alpha_max_base)


def fit_elasticnet_path_scaled(
    X_train_scaled,
    X_val_scaled,
    y_train,
    y_val,
    alphas,
    l1_ratio,
    feature_space_name,
    scaler_name,
    stage
):
    """
    Fits Elastic Net along a descending alpha path using warm starts.
    """
    rows = []
    
    alphas = np.asarray(alphas, dtype=float)
    
    model = ElasticNet(
        alpha=alphas[0],
        l1_ratio=l1_ratio,
        fit_intercept=True,
        max_iter=ELASTICNET_MAX_ITER,
        tol=ELASTICNET_TOL,
        warm_start=True,
        selection="cyclic",
        random_state=9890
    )
    
    for alpha in alphas:
        model.alpha = float(alpha)
        
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train_scaled, y_train)
            
            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )
        
        train_pred = model.predict(X_train_scaled)
        val_pred = model.predict(X_val_scaled)
        
        rows.append({
            "model_class": "ElasticNet",
            "stage": stage,
            "feature_space": feature_space_name,
            "scaler": scaler_name,
            "l1_ratio": l1_ratio,
            "alpha": float(alpha),
            "train_mse": mean_squared_error(y_train, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "nonzero_coef": int(np.sum(np.abs(model.coef_) > 1e-8)),
            "n_iter": int(model.n_iter_),
            "convergence_warning": convergence_warning,
            "aic": np.nan,
            "bic": np.nan
        })
    
    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Main search
# ------------------------------------------------------------

start_time = time.perf_counter()
elasticnet_all_parts = []

print("Elastic Net overnight search starting...")
print("Feature spaces:", list(feature_spaces.keys()))
print("l1_ratios:", elasticnet_l1_ratios)

for feature_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for scaler_name, scaler_factory in scaler_factories.items():
        
        print("\n" + "=" * 90)
        print(f"Feature space: {feature_space_name} | Scaler: {scaler_name}")
        print("=" * 90)
        
        scaler = scaler_factory()
        X_train_scaled = scaler.fit_transform(X_train_space)
        X_val_scaled = scaler.transform(X_val_space)
        
        alpha_max_base = compute_alpha_max_lasso_base(X_train_scaled, y_tr)
        
        broad_parts_this_combo = []
        
        # ----------------------------------------------------
        # Stage 1: broad search for each l1_ratio
        # ----------------------------------------------------
        for l1_ratio in elasticnet_l1_ratios:
            alpha_max = alpha_max_base / l1_ratio
            alpha_min = alpha_max * ELASTICNET_MIN_ALPHA_RATIO
            
            broad_alphas = np.logspace(
                np.log10(alpha_max),
                np.log10(alpha_min),
                ELASTICNET_BROAD_GRID_SIZE
            )
            
            broad_df = fit_elasticnet_path_scaled(
                X_train_scaled=X_train_scaled,
                X_val_scaled=X_val_scaled,
                y_train=y_tr,
                y_val=y_val,
                alphas=broad_alphas,
                l1_ratio=l1_ratio,
                feature_space_name=feature_space_name,
                scaler_name=scaler_name,
                stage="broad"
            )
            
            broad_parts_this_combo.append(broad_df)
            elasticnet_all_parts.append(broad_df)
            append_checkpoint(broad_df, ELASTICNET_RESULTS_PATH)
        
        broad_combo_df = pd.concat(broad_parts_this_combo, ignore_index=True)
        
        best_broad_by_l1 = (
            broad_combo_df
            .loc[broad_combo_df.groupby("l1_ratio")["val_mse"].idxmin()]
            .sort_values("val_mse")
        )
        
        print("\nBest broad result by l1_ratio:")
        print(
            best_broad_by_l1[
                ["l1_ratio", "alpha", "train_mse", "val_mse", "nonzero_coef", "convergence_warning"]
            ].to_string(index=False)
        )
        
        # ----------------------------------------------------
        # Stage 2: fine search around each broad winner
        # ----------------------------------------------------
        for _, row in best_broad_by_l1.iterrows():
            l1_ratio = float(row["l1_ratio"])
            best_alpha = float(row["alpha"])
            
            alpha_max = alpha_max_base / l1_ratio
            alpha_min = alpha_max * ELASTICNET_MIN_ALPHA_RATIO
            
            best_log = np.log10(best_alpha)
            fine_low = max(np.log10(alpha_min), best_log - ELASTICNET_FINE_WIDTH_LOG10)
            fine_high = min(np.log10(alpha_max), best_log + ELASTICNET_FINE_WIDTH_LOG10)
            
            fine_alphas = np.logspace(
                fine_high,
                fine_low,
                ELASTICNET_FINE_GRID_SIZE
            )
            
            fine_df = fit_elasticnet_path_scaled(
                X_train_scaled=X_train_scaled,
                X_val_scaled=X_val_scaled,
                y_train=y_tr,
                y_val=y_val,
                alphas=fine_alphas,
                l1_ratio=l1_ratio,
                feature_space_name=feature_space_name,
                scaler_name=scaler_name,
                stage="fine"
            )
            
            elasticnet_all_parts.append(fine_df)
            append_checkpoint(fine_df, ELASTICNET_RESULTS_PATH)
        
        elapsed_so_far = time.perf_counter() - start_time
        print(f"\nFinished {feature_space_name} | {scaler_name}. Elapsed seconds: {elapsed_so_far:.2f}")


# ------------------------------------------------------------
# Summarize final Elastic Net results
# ------------------------------------------------------------

elasticnet_results = pd.concat(elasticnet_all_parts, ignore_index=True)

best_elasticnet_by_space = (
    elasticnet_results
    .loc[elasticnet_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_elasticnet_overall = elasticnet_results.loc[
    elasticnet_results["val_mse"].idxmin()
]

best_elasticnet_by_space.to_csv(ELASTICNET_BEST_BY_SPACE_PATH, index=False)
best_elasticnet_overall.to_frame().T.to_csv(ELASTICNET_BEST_OVERALL_PATH, index=False)

elapsed = time.perf_counter() - start_time

print("\n" + "=" * 90)
print("Best Elastic Net result by feature space:")
print("=" * 90)
print(best_elasticnet_by_space.to_string(index=False))

print("\n" + "=" * 90)
print("Overall best Elastic Net result:")
print("=" * 90)
print(best_elasticnet_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

print("\nConvergence warning counts:")
print(
    elasticnet_results
    .groupby(["feature_space", "scaler", "stage"])["convergence_warning"]
    .sum()
    .reset_index()
    .to_string(index=False)
)

print("\nSaved files:")
print("All Elastic Net results:", ELASTICNET_RESULTS_PATH)
print("Best by feature space:", ELASTICNET_BEST_BY_SPACE_PATH)
print("Best overall:", ELASTICNET_BEST_OVERALL_PATH)

Elastic Net overnight search starting...
Feature spaces: ['base', 'base_plus_poly', 'base_plus_interactions', 'base_plus_poly_interactions']
l1_ratios: [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]

Feature space: base | Scaler: standard

Best broad result by l1_ratio:
 l1_ratio    alpha  train_mse    val_mse  nonzero_coef  convergence_warning
     0.99 0.001137 305.154413 312.619191           132                False
     0.95 0.001185 305.158777 312.623589           135                False
     0.90 0.001251 305.164429 312.629943           145                False
     0.75 0.001501 305.192217 312.655747           154                False
     0.50 0.002251 305.267042 312.737288           156                False
     0.25 0.004502 305.469335 312.955788           155                False
     0.10 0.011256 305.948373 313.426435           156                False
     0.05 0.022512 306.670214 314.083149           162                False

Finished base | standard. Elapsed seconds: 13

### Elastic Net Regression Results

Elastic Net regression was evaluated across the four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Both `StandardScaler` and `RobustScaler` were compared, and the model was tuned over multiple `l1_ratio` values and data-driven alpha values.

Best Elastic Net model:
- Feature space: base + polynomial + interactions
- Scaler: StandardScaler
- l1_ratio: 0.99
- Alpha: 0.001137
- Train MSE: 301.05
- Validation MSE: 308.43
- Nonzero coefficients: 152

Interpretation:
- Elastic Net performed best on the combined polynomial + interaction feature space.
- The best `l1_ratio` was 0.99, meaning the model behaved very similarly to Lasso.
- StandardScaler outperformed RobustScaler.
- Elastic Net retained 152 nonzero coefficients, so it did not produce a highly sparse model.
- The result is competitive but does not outperform Ridge or the best backward stepwise model.

Conclusion:
Elastic Net confirms that the strongest feature space is the combined polynomial + interaction space, but it does not improve over the current best model. The current best linear-family model remains backward stepwise on the combined feature space, with validation MSE around 308.10.

In [25]:
# ============================================================
# 12A. Step Function Models
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

# ------------------------------------------------------------
# Design:
# - Select top continuous predictors from simple LR ranking
# - Create step-function dummy variables using training-split cutpoints only
# - Add step features to each existing controlled feature space
# - Evaluate using current development holdout split
# ------------------------------------------------------------

start_time = time.perf_counter()

# Select top continuous features only.
# Exclude binary dummies / missing indicators by requiring many unique values.
step_candidate_features = []

for col in simple_lr_results["feature"].tolist():
    if col in X_tr.columns:
        unique_count = X_tr[col].nunique(dropna=False)
        if unique_count >= 20 and not col.endswith("_missing"):
            step_candidate_features.append(col)
    
    if len(step_candidate_features) >= 10:
        break

print("Step-function candidate features:")
for i, col in enumerate(step_candidate_features, start=1):
    print(f"{i:2d}. {col}")


def make_step_features(X_train_source, X_val_source, columns, n_bins, method):
    """
    Build step-function dummy features.

    Cutpoints are learned from X_train_source only.
    Then the same cutpoints are applied to X_val_source.

    method:
        'quantile'    -> bins based on training quantiles
        'equal_width' -> bins based on training min/max width
    """
    train_parts = []
    val_parts = []
    actual_step_cols = []

    for col in columns:
        x_train = X_train_source[col].astype(float)
        x_val = X_val_source[col].astype(float)

        if x_train.nunique(dropna=False) < 2:
            continue

        if method == "quantile":
            try:
                _, edges = pd.qcut(
                    x_train,
                    q=n_bins,
                    retbins=True,
                    duplicates="drop"
                )
            except ValueError:
                continue

        elif method == "equal_width":
            min_val = x_train.min()
            max_val = x_train.max()

            if not np.isfinite(min_val) or not np.isfinite(max_val) or min_val == max_val:
                continue

            edges = np.linspace(min_val, max_val, n_bins + 1)

        else:
            raise ValueError("method must be 'quantile' or 'equal_width'")

        edges = np.unique(edges)

        if len(edges) <= 2:
            continue

        # Make validation robust to values slightly outside training range.
        edges = edges.astype(float)
        edges[0] = -np.inf
        edges[-1] = np.inf

        k = len(edges) - 1

        train_codes = pd.cut(
            x_train,
            bins=edges,
            labels=False,
            include_lowest=True
        )

        val_codes = pd.cut(
            x_val,
            bins=edges,
            labels=False,
            include_lowest=True
        )

        train_cat = pd.Categorical(train_codes, categories=list(range(k)))
        val_cat = pd.Categorical(val_codes, categories=list(range(k)))

        prefix = f"{col}_step_{method}_{n_bins}"

        train_dummies = pd.get_dummies(
            train_cat,
            prefix=prefix,
            drop_first=True,
            dtype=float
        )
        val_dummies = pd.get_dummies(
            val_cat,
            prefix=prefix,
            drop_first=True,
            dtype=float
        )

        train_dummies.index = X_train_source.index
        val_dummies.index = X_val_source.index

        train_dummies, val_dummies = train_dummies.align(
            val_dummies,
            join="left",
            axis=1,
            fill_value=0
        )

        train_parts.append(train_dummies)
        val_parts.append(val_dummies)
        actual_step_cols.extend(train_dummies.columns.tolist())

    if len(train_parts) == 0:
        empty_train = pd.DataFrame(index=X_train_source.index)
        empty_val = pd.DataFrame(index=X_val_source.index)
        return empty_train, empty_val, []

    X_train_steps = pd.concat(train_parts, axis=1)
    X_val_steps = pd.concat(val_parts, axis=1)

    return X_train_steps, X_val_steps, actual_step_cols


step_methods = ["quantile", "equal_width"]
step_bins_grid = [3, 5, 10]

step_results_rows = []

for base_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for method in step_methods:
        for n_bins in step_bins_grid:

            X_train_steps, X_val_steps, step_cols = make_step_features(
                X_train_source=X_tr,
                X_val_source=X_val,
                columns=step_candidate_features,
                n_bins=n_bins,
                method=method
            )

            X_train_aug = pd.concat([X_train_space, X_train_steps], axis=1)
            X_val_aug = pd.concat([X_val_space, X_val_steps], axis=1)

            model = LinearRegression()
            model.fit(X_train_aug, y_tr)

            train_pred = model.predict(X_train_aug)
            val_pred = model.predict(X_val_aug)

            step_results_rows.append({
                "model_class": "Step Functions + Linear Regression",
                "base_space": base_space_name,
                "step_method": method,
                "n_bins": n_bins,
                "n_step_features_added": len(step_cols),
                "total_features": X_train_aug.shape[1],
                "train_mse": mean_squared_error(y_tr, train_pred),
                "val_mse": mean_squared_error(y_val, val_pred),
                "aic": np.nan,
                "bic": np.nan
            })

step_results = pd.DataFrame(step_results_rows)

best_step_by_base_space = (
    step_results
    .loc[step_results.groupby("base_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_step_overall = step_results.loc[step_results["val_mse"].idxmin()]

elapsed = time.perf_counter() - start_time

print("\nBest step-function model by base space:")
print(best_step_by_base_space.to_string(index=False))

print("\nOverall best step-function model:")
print(best_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

Step-function candidate features:
 1. PERCENT_ECONOMICALLY_DISADVANTAGED
 2. PERCENT_FREE_LUNCH
 3. PERCENT_DIPLOMA
 4. PERCENT_STILL_ENROLLED
 5. PERCENT_HOMELESS
 6. PERCENT_BLACK
 7. PERCENT_WITH_DISABILITIES
 8. PERCENT_ENGLISH_LANGUAGE_LEANERS
 9. ATTENDANCE_RATE
10. PERCENT_WHITE

Best step-function model by base space:
                       model_class                  base_space step_method  n_bins  n_step_features_added  total_features  train_mse    val_mse  aic  bic
Step Functions + Linear Regression base_plus_poly_interactions    quantile      10                     83             265 296.544041 303.570375  NaN  NaN
Step Functions + Linear Regression              base_plus_poly    quantile      10                     83             255 297.288261 304.119794  NaN  NaN
Step Functions + Linear Regression      base_plus_interactions    quantile      10                     83             255 297.540321 304.612830  NaN  NaN
Step Functions + Linear Regression                      

### Step Function Model Results

Step-function models were evaluated by binning the strongest continuous predictors from the simple linear regression ranking.

Candidate variables included:
- `PERCENT_ECONOMICALLY_DISADVANTAGED`
- `PERCENT_FREE_LUNCH`
- `PERCENT_DIPLOMA`
- `PERCENT_STILL_ENROLLED`
- `PERCENT_HOMELESS`
- `PERCENT_BLACK`
- `PERCENT_WITH_DISABILITIES`
- `PERCENT_ENGLISH_LANGUAGE_LEANERS`
- `ATTENDANCE_RATE`
- `PERCENT_WHITE`

The best step-function model used:
- Base feature space: base + polynomial + interactions
- Binning method: quantile bins
- Number of bins: 10
- Step-function features added: 83
- Total features: 265

Results:
- Train MSE: 296.54
- Validation MSE: 303.57

Interpretation:
- Step functions substantially improved performance compared with previous linear-family models.
- The improvement suggests that some important predictors have threshold-based or piecewise relationships with the target.
- Quantile binning performed best, likely because it creates balanced bins across skewed continuous predictors.
- The train-validation gap remains moderate, so the improvement does not appear to be caused by severe overfitting.

Conclusion:
Step-function feature engineering is the strongest modeling direction so far. The current best model is a linear regression model using the combined polynomial + interaction feature space plus quantile step functions.

In [26]:
# ============================================================
# 12B. Expanded Step Function Search
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

start_time = time.perf_counter()

# ------------------------------------------------------------
# Helper: get top continuous features from simple LR ranking
# ------------------------------------------------------------

def get_top_continuous_features(k, min_unique=20):
    features = []
    
    for col in simple_lr_results["feature"].tolist():
        if col not in X_tr.columns:
            continue
        
        unique_count = X_tr[col].nunique(dropna=False)
        
        if unique_count >= min_unique and not col.endswith("_missing"):
            features.append(col)
        
        if len(features) >= k:
            break
    
    return features


# ------------------------------------------------------------
# Expanded search design
# ------------------------------------------------------------

candidate_feature_counts = [10, 15, 20]
step_bins_grid = [5, 10, 15, 20]
step_method = "quantile"

expanded_step_rows = []

for k in candidate_feature_counts:
    candidate_features = get_top_continuous_features(k)
    
    print("\n" + "=" * 80)
    print(f"Top {k} continuous step-function candidates:")
    print(candidate_features)
    
    for base_space_name, (X_train_space, X_val_space) in feature_spaces.items():
        for n_bins in step_bins_grid:
            
            X_train_steps, X_val_steps, step_cols = make_step_features(
                X_train_source=X_tr,
                X_val_source=X_val,
                columns=candidate_features,
                n_bins=n_bins,
                method=step_method
            )
            
            X_train_aug = pd.concat([X_train_space, X_train_steps], axis=1)
            X_val_aug = pd.concat([X_val_space, X_val_steps], axis=1)
            
            model = LinearRegression()
            model.fit(X_train_aug, y_tr)
            
            train_pred = model.predict(X_train_aug)
            val_pred = model.predict(X_val_aug)
            
            expanded_step_rows.append({
                "model_class": "Expanded Step Functions + Linear Regression",
                "base_space": base_space_name,
                "step_method": step_method,
                "candidate_feature_count": k,
                "n_bins": n_bins,
                "n_step_features_added": len(step_cols),
                "total_features": X_train_aug.shape[1],
                "train_mse": mean_squared_error(y_tr, train_pred),
                "val_mse": mean_squared_error(y_val, val_pred),
                "aic": np.nan,
                "bic": np.nan
            })

expanded_step_results = pd.DataFrame(expanded_step_rows)

best_expanded_step_by_base_space = (
    expanded_step_results
    .loc[expanded_step_results.groupby("base_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_expanded_step_overall = expanded_step_results.loc[
    expanded_step_results["val_mse"].idxmin()
]

elapsed = time.perf_counter() - start_time

print("\nBest expanded step-function model by base space:")
print(best_expanded_step_by_base_space.to_string(index=False))

print("\nOverall best expanded step-function model:")
print(best_expanded_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")


Top 10 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']

Top 15 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11']

Top 20 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN

### Expanded Step Function Search

The expanded step-function search tested quantile-based binning over larger sets of continuous predictors.

Best model:
- Base feature space: base + polynomial + interactions
- Candidate continuous predictors: 20
- Quantile bins: 20
- Step-function features added: 252
- Total features: 434

Results:
- Train MSE: 289.14
- Validation MSE: 296.14

Interpretation:
- Step functions substantially improved validation performance compared with all previous linear-family models.
- The improvement suggests that several predictors have threshold-based or piecewise relationships with `PERCENT_PROFICIENT`.
- The best model occurred at the largest tested feature count and bin count, indicating that the useful search space may not yet be exhausted.
- The train-validation gap remains moderate, so the model does not appear to be severely overfit at this stage.

Conclusion:
Step-function feature engineering is currently the strongest modeling direction. The best model so far is a linear regression model using the combined polynomial + interaction feature space plus quantile step functions.

In [27]:
# ============================================================
# 12C. Focused expanded step-function search on best base space
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

start_time = time.perf_counter()

# Current best base space
best_base_space_name = "base_plus_poly_interactions"
X_train_best_base, X_val_best_base = feature_spaces[best_base_space_name]

# Expand beyond previous boundary.
candidate_feature_counts = [20, 25, 30]
step_bins_grid = [20, 25, 30]
step_method = "quantile"

focused_step_rows = []

for k in candidate_feature_counts:
    candidate_features = get_top_continuous_features(k)
    
    print("\n" + "=" * 80)
    print(f"Top {k} continuous step-function candidates:")
    print(candidate_features)
    
    for n_bins in step_bins_grid:
        X_train_steps, X_val_steps, step_cols = make_step_features(
            X_train_source=X_tr,
            X_val_source=X_val,
            columns=candidate_features,
            n_bins=n_bins,
            method=step_method
        )
        
        X_train_aug = pd.concat([X_train_best_base, X_train_steps], axis=1)
        X_val_aug = pd.concat([X_val_best_base, X_val_steps], axis=1)
        
        model = LinearRegression()
        model.fit(X_train_aug, y_tr)
        
        train_pred = model.predict(X_train_aug)
        val_pred = model.predict(X_val_aug)
        
        focused_step_rows.append({
            "model_class": "Focused Expanded Step Functions + Linear Regression",
            "base_space": best_base_space_name,
            "step_method": step_method,
            "candidate_feature_count": k,
            "n_bins": n_bins,
            "n_step_features_added": len(step_cols),
            "total_features": X_train_aug.shape[1],
            "train_mse": mean_squared_error(y_tr, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "aic": np.nan,
            "bic": np.nan
        })

focused_step_results = pd.DataFrame(focused_step_rows).sort_values("val_mse")

best_focused_step_overall = focused_step_results.loc[
    focused_step_results["val_mse"].idxmin()
]

elapsed = time.perf_counter() - start_time

print("\nFocused expanded step-function results:")
print(focused_step_results.to_string(index=False))

print("\nOverall best focused expanded step-function model:")
print(best_focused_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")


Top 20 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11', 'PRE_K', 'GRADE_10', 'NUMBER_OF_TEACHERS', 'GRADE_09', 'NUMBER_OF_COUNSELORS']

Top 25 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11', 'PRE_K', 'GRADE_10', 'NUMBER_OF_TEACHERS', 'GRADE_09', 'NUMBER_OF_COUNSELORS', 'N_PUPILS', 'FEDERAL_FUNDING_PER_PUPIL', 'GRADE_04', 'GRADE_03', 'SCHOOL_freq']

Top 30 continuous step-

### Focused Expanded Step Function Search

A focused step-function search was run on the strongest base feature space: `base_plus_poly_interactions`.

The search expanded the number of continuous variables used for quantile binning and increased the number of bins.

Best model:
- Base feature space: base + polynomial + interactions
- Candidate continuous predictors: 30
- Quantile bins: 30
- Step-function features added: 526
- Total features: 708

Results:
- Train MSE: 281.63
- Validation MSE: 290.11

Interpretation:
- This is the strongest model so far.
- The result substantially improves over the previous best linear-family model, which had validation MSE around 308.10.
- The improvement suggests that the target has strong piecewise or threshold-based relationships with predictors.
- The best model occurred at the largest tested feature count and bin count, suggesting that the step-function search space may not yet be exhausted.
- The train-validation gap increased but remains moderate, so there is no clear evidence of severe overfitting yet.

Conclusion:
Step-function feature engineering is currently the most successful modeling strategy. The next step should regularize this larger step-function feature space, because the model now contains many correlated bin indicators.

In [28]:
# ============================================================
# 12D. Ridge on best step-function feature space
# ============================================================

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

start_time = time.perf_counter()

# ------------------------------------------------------------
# Rebuild current best step-function feature space
# ------------------------------------------------------------

best_base_space_name = "base_plus_poly_interactions"
best_step_candidate_count = 30
best_step_bins = 30
best_step_method = "quantile"

X_train_best_base, X_val_best_base = feature_spaces[best_base_space_name]
best_step_candidates = get_top_continuous_features(best_step_candidate_count)

X_train_steps_best, X_val_steps_best, best_step_cols = make_step_features(
    X_train_source=X_tr,
    X_val_source=X_val,
    columns=best_step_candidates,
    n_bins=best_step_bins,
    method=best_step_method
)

X_tr_step_best = pd.concat([X_train_best_base, X_train_steps_best], axis=1)
X_val_step_best = pd.concat([X_val_best_base, X_val_steps_best], axis=1)

print("Best step-function design:")
print("Base space:", best_base_space_name)
print("Candidate feature count:", best_step_candidate_count)
print("Bins:", best_step_bins)
print("Step features added:", len(best_step_cols))
print("Total features:", X_tr_step_best.shape[1])

# ------------------------------------------------------------
# Sanity check: unregularized OLS on rebuilt step space
# ------------------------------------------------------------

ols_step_model = LinearRegression()
ols_step_model.fit(X_tr_step_best, y_tr)

ols_train_pred = ols_step_model.predict(X_tr_step_best)
ols_val_pred = ols_step_model.predict(X_val_step_best)

ols_step_train_mse = mean_squared_error(y_tr, ols_train_pred)
ols_step_val_mse = mean_squared_error(y_val, ols_val_pred)

print("\nOLS step-function sanity check:")
print("Train MSE:", ols_step_train_mse)
print("Validation MSE:", ols_step_val_mse)

# ------------------------------------------------------------
# Ridge tuning on this step-function feature space
# ------------------------------------------------------------

scaler_factories = {
    "standard": StandardScaler,
    "robust": RobustScaler
}

# Broad search across many orders of magnitude.
# We include very small alpha to approximate OLS and large alpha to test heavy shrinkage.
ridge_step_broad_alphas = np.logspace(-4, 8, 49)

ridge_step_rows = []

for scaler_name, scaler_factory in scaler_factories.items():
    for alpha in ridge_step_broad_alphas:
        model = make_pipeline(
            scaler_factory(),
            Ridge(alpha=alpha)
        )
        
        model.fit(X_tr_step_best, y_tr)
        
        train_pred = model.predict(X_tr_step_best)
        val_pred = model.predict(X_val_step_best)
        
        ridge_step_rows.append({
            "model_class": "Ridge + Step Functions",
            "stage": "broad",
            "scaler": scaler_name,
            "alpha": alpha,
            "train_mse": mean_squared_error(y_tr, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "aic": np.nan,
            "bic": np.nan
        })

ridge_step_broad_results = pd.DataFrame(ridge_step_rows)

best_broad_by_scaler = (
    ridge_step_broad_results
    .loc[ridge_step_broad_results.groupby("scaler")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

print("\nBest broad Ridge-step result by scaler:")
print(best_broad_by_scaler.to_string(index=False))

# ------------------------------------------------------------
# Fine search around each broad winner
# ------------------------------------------------------------

ridge_step_fine_rows = []

for _, row in best_broad_by_scaler.iterrows():
    scaler_name = row["scaler"]
    scaler_factory = scaler_factories[scaler_name]
    best_alpha = row["alpha"]
    
    log_alpha = np.log10(best_alpha)
    fine_low = max(np.log10(ridge_step_broad_alphas.min()), log_alpha - 0.75)
    fine_high = min(np.log10(ridge_step_broad_alphas.max()), log_alpha + 0.75)
    
    ridge_step_fine_alphas = np.logspace(fine_low, fine_high, 41)
    
    for alpha in ridge_step_fine_alphas:
        model = make_pipeline(
            scaler_factory(),
            Ridge(alpha=alpha)
        )
        
        model.fit(X_tr_step_best, y_tr)
        
        train_pred = model.predict(X_tr_step_best)
        val_pred = model.predict(X_val_step_best)
        
        ridge_step_fine_rows.append({
            "model_class": "Ridge + Step Functions",
            "stage": "fine",
            "scaler": scaler_name,
            "alpha": alpha,
            "train_mse": mean_squared_error(y_tr, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "aic": np.nan,
            "bic": np.nan
        })

ridge_step_fine_results = pd.DataFrame(ridge_step_fine_rows)

ridge_step_results = pd.concat(
    [ridge_step_broad_results, ridge_step_fine_results],
    ignore_index=True
)

best_ridge_step_by_scaler = (
    ridge_step_results
    .loc[ridge_step_results.groupby("scaler")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_ridge_step_overall = ridge_step_results.loc[
    ridge_step_results["val_mse"].idxmin()
]

elapsed = time.perf_counter() - start_time

print("\nBest Ridge-step result by scaler:")
print(best_ridge_step_by_scaler.to_string(index=False))

print("\nOverall best Ridge-step result:")
print(best_ridge_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

if best_ridge_step_overall["alpha"] == ridge_step_broad_alphas.min():
    print("\nWARNING: Best alpha is at the lower broad-grid boundary. Ridge may prefer nearly no shrinkage.")
elif best_ridge_step_overall["alpha"] == ridge_step_broad_alphas.max():
    print("\nWARNING: Best alpha is at the upper broad-grid boundary. Consider expanding alpha upward.")

Best step-function design:
Base space: base_plus_poly_interactions
Candidate feature count: 30
Bins: 30
Step features added: 526
Total features: 708

OLS step-function sanity check:
Train MSE: 281.63428911174634
Validation MSE: 290.1093738555459

Best broad Ridge-step result by scaler:
           model_class stage   scaler    alpha  train_mse    val_mse  aic  bic
Ridge + Step Functions broad   robust 0.562341 281.672006 290.079990  NaN  NaN
Ridge + Step Functions broad standard 1.778279 281.656987 290.084438  NaN  NaN

Best Ridge-step result by scaler:
           model_class stage   scaler    alpha  train_mse    val_mse  aic  bic
Ridge + Step Functions  fine   robust 0.473151 281.664053 290.079273  NaN  NaN
Ridge + Step Functions  fine standard 2.304093 281.665268 290.083841  NaN  NaN

Overall best Ridge-step result:
model_class    Ridge + Step Functions
stage                            fine
scaler                         robust
alpha                        0.473151
train_mse          

### Interpretation: Step Functions + Ridge Regression

We constructed a high-dimensional feature space using step functions applied to a base set of predictors (including polynomial and interaction terms). This resulted in a total of 708 features, significantly expanding the model’s flexibility through piecewise constant approximations.

The unregularized OLS model achieved:
- Train MSE: 281.63  
- Validation MSE: 290.11  

Applying Ridge regularization led to a very small improvement:
- Best Validation MSE: 290.08 (robust scaler, α ≈ 0.47)

#### Key Takeaways

- **Minimal gain from Ridge:** The negligible improvement in validation MSE suggests that the model is not suffering from substantial overfitting, despite the large feature space.
- **Train vs Validation gap is small:** This indicates low variance and good generalization stability.
- **Bias likely dominates:** Since increasing model complexity (via step functions) and adding regularization does not materially improve performance, the model may still be underfitting the underlying structure.

#### Implication

Step functions increase dimensionality but impose rigid, piecewise constant structure. While they capture some nonlinearity, they may not be flexible enough to model smooth or complex relationships in the data. Further improvements may require models that learn structure more adaptively rather than relying solely on engineered basis expansions.

In [35]:
# ============================================================
# 12E. Lasso on best step-function feature space
# ============================================================

from pathlib import Path
import gc
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

start_time = time.perf_counter()

# ------------------------------------------------------------
# Check that the previous step-function design exists
# ------------------------------------------------------------

print("Lasso on best step-function feature space")
print("Training shape:", X_tr_step_best.shape)
print("Validation shape:", X_val_step_best.shape)
print("Reference OLS step validation MSE:", ols_step_val_mse)
print("Reference Ridge-step validation MSE:", best_ridge_step_overall["val_mse"])

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

LASSO_STEP_RESULTS_PATH = RESULTS_DIR / "lasso_step_results_holdout.csv"
LASSO_STEP_BEST_BY_SCALER_PATH = RESULTS_DIR / "lasso_step_best_by_scaler_holdout.csv"
LASSO_STEP_BEST_OVERALL_PATH = RESULTS_DIR / "lasso_step_best_overall_holdout.csv"

OVERWRITE_LASSO_STEP_RESULTS = True

if OVERWRITE_LASSO_STEP_RESULTS:
    for path in [
        LASSO_STEP_RESULTS_PATH,
        LASSO_STEP_BEST_BY_SCALER_PATH,
        LASSO_STEP_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

# ------------------------------------------------------------
# Lasso helper functions
# ------------------------------------------------------------

def compute_lasso_alpha_max(X_scaled, y):
    """
    Computes alpha_max for sklearn's Lasso objective.

    At alpha >= alpha_max, all coefficients are zero.
    """
    y_arr = np.asarray(y, dtype=float)
    y_centered = y_arr - y_arr.mean()
    n = X_scaled.shape[0]
    return float(np.max(np.abs(X_scaled.T @ y_centered)) / n)

def fit_lasso_path_for_scaler(
    scaler_name,
    scaler_factory,
    alpha_ratios,
    stage,
    max_iter=50000,
    tol=1e-4
):
    """
    Fits Lasso over a sequence of alpha ratios for one scaler.
    Uses warm starts from larger alpha to smaller alpha.
    """
    print(f"\n--- {stage.upper()} search | scaler = {scaler_name} ---")
    
    scaler = scaler_factory()
    X_train_scaled = scaler.fit_transform(X_tr_step_best)
    X_val_scaled = scaler.transform(X_val_step_best)

    alpha_max = compute_lasso_alpha_max(X_train_scaled, y_tr)
    alphas = alpha_max * np.asarray(alpha_ratios)

    # Warm start works best when moving from stronger penalty to weaker penalty.
    order = np.argsort(alphas)[::-1]
    alphas = alphas[order]
    alpha_ratios_ordered = np.asarray(alpha_ratios)[order]

    rows = []

    model = Lasso(
        alpha=alphas[0],
        fit_intercept=True,
        max_iter=max_iter,
        tol=tol,
        warm_start=True,
        selection="cyclic",
        random_state=RANDOM_STATE
    )

    for alpha_ratio, alpha in zip(alpha_ratios_ordered, alphas):
        model.alpha = float(alpha)

        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train_scaled, y_tr)

            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )

        train_pred = model.predict(X_train_scaled)
        val_pred = model.predict(X_val_scaled)

        train_mse = mean_squared_error(y_tr, train_pred)
        val_mse = mean_squared_error(y_val, val_pred)
        nonzero_coef = int(np.sum(np.abs(model.coef_) > 1e-8))

        row = {
            "model_class": "Lasso + Step Functions",
            "stage": stage,
            "scaler": scaler_name,
            "alpha_max": alpha_max,
            "alpha_ratio": alpha_ratio,
            "alpha": alpha,
            "train_mse": train_mse,
            "val_mse": val_mse,
            "nonzero_coef": nonzero_coef,
            "n_iter": model.n_iter_,
            "convergence_warning": convergence_warning,
            "aic": np.nan,
            "bic": np.nan
        }

        rows.append(row)
        append_checkpoint(pd.DataFrame([row]), LASSO_STEP_RESULTS_PATH)

        print(
            f"{stage:5s} | {scaler_name:8s} | "
            f"ratio={alpha_ratio:.2e} | alpha={alpha:.8f} | "
            f"train_mse={train_mse:.6f} | val_mse={val_mse:.6f} | "
            f"nonzero={nonzero_coef:4d} | n_iter={model.n_iter_:5d} | "
            f"warning={convergence_warning}"
        )

    results = pd.DataFrame(rows)

    del X_train_scaled, X_val_scaled, model
    gc.collect()

    return results

# ------------------------------------------------------------
# Stage 1: broad search
# ------------------------------------------------------------

scaler_factories = {
    "standard": StandardScaler,
    "robust": RobustScaler
}

# Ratios are relative to alpha_max.
# This searches from fairly strong Lasso shrinkage down to very weak shrinkage.
lasso_step_broad_ratios = np.logspace(-1, -6, 26)

lasso_step_broad_results_list = []

for scaler_name, scaler_factory in scaler_factories.items():
    results = fit_lasso_path_for_scaler(
        scaler_name=scaler_name,
        scaler_factory=scaler_factory,
        alpha_ratios=lasso_step_broad_ratios,
        stage="broad",
        max_iter=50000,
        tol=1e-4
    )
    lasso_step_broad_results_list.append(results)

lasso_step_broad_results = pd.concat(lasso_step_broad_results_list, ignore_index=True)

best_lasso_step_broad_by_scaler = (
    lasso_step_broad_results
    .loc[lasso_step_broad_results.groupby("scaler")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

print("\nBest broad Lasso-step result by scaler:")
print(best_lasso_step_broad_by_scaler.to_string(index=False))

# ------------------------------------------------------------
# Stage 2: fine search around each scaler's broad winner
# ------------------------------------------------------------

lasso_step_fine_results_list = []

for _, row in best_lasso_step_broad_by_scaler.iterrows():
    scaler_name = row["scaler"]
    scaler_factory = scaler_factories[scaler_name]

    best_ratio = row["alpha_ratio"]

    fine_low = max(1e-7, best_ratio / 3.0)
    fine_high = min(1e-1, best_ratio * 3.0)

    lasso_step_fine_ratios = np.logspace(
        np.log10(fine_high),
        np.log10(fine_low),
        25
    )

    results = fit_lasso_path_for_scaler(
        scaler_name=scaler_name,
        scaler_factory=scaler_factory,
        alpha_ratios=lasso_step_fine_ratios,
        stage="fine",
        max_iter=70000,
        tol=1e-4
    )
    lasso_step_fine_results_list.append(results)

lasso_step_fine_results = pd.concat(lasso_step_fine_results_list, ignore_index=True)

# ------------------------------------------------------------
# Combine and summarize
# ------------------------------------------------------------

lasso_step_results = pd.concat(
    [lasso_step_broad_results, lasso_step_fine_results],
    ignore_index=True
)

best_lasso_step_by_scaler = (
    lasso_step_results
    .loc[lasso_step_results.groupby("scaler")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_lasso_step_overall = lasso_step_results.loc[
    lasso_step_results["val_mse"].idxmin()
]

best_lasso_step_by_scaler.to_csv(LASSO_STEP_BEST_BY_SCALER_PATH, index=False)
pd.DataFrame([best_lasso_step_overall]).to_csv(LASSO_STEP_BEST_OVERALL_PATH, index=False)

print("\nBest Lasso-step result by scaler:")
print(best_lasso_step_by_scaler.to_string(index=False))

print("\nOverall best Lasso-step result:")
print(best_lasso_step_overall)

# ------------------------------------------------------------
# Refit best Lasso-step model on the development training split
# ------------------------------------------------------------

best_lasso_step_model = make_pipeline(
    scaler_factories[best_lasso_step_overall["scaler"]](),
    Lasso(
        alpha=float(best_lasso_step_overall["alpha"]),
        fit_intercept=True,
        max_iter=100000,
        tol=1e-4,
        selection="cyclic",
        random_state=RANDOM_STATE
    )
)

best_lasso_step_model.fit(X_tr_step_best, y_tr)

best_lasso_step_train_pred = best_lasso_step_model.predict(X_tr_step_best)
best_lasso_step_val_pred = best_lasso_step_model.predict(X_val_step_best)

best_lasso_step_train_mse = mean_squared_error(y_tr, best_lasso_step_train_pred)
best_lasso_step_val_mse = mean_squared_error(y_val, best_lasso_step_val_pred)

best_lasso_step_nonzero = int(
    np.sum(np.abs(best_lasso_step_model.named_steps["lasso"].coef_) > 1e-8)
)

print("\nRefit check for best Lasso-step model:")
print("Train MSE:", best_lasso_step_train_mse)
print("Validation MSE:", best_lasso_step_val_mse)
print("Nonzero coefficients:", best_lasso_step_nonzero)

# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Lasso + Step Functions"] = {
    "model": best_lasso_step_model,
    "feature_space": "base_plus_poly_interactions + step_functions",
    "X_train_name": "X_tr_step_best",
    "X_val_name": "X_val_step_best",
    "scaler": best_lasso_step_overall["scaler"],
    "alpha": float(best_lasso_step_overall["alpha"]),
    "alpha_ratio": float(best_lasso_step_overall["alpha_ratio"]),
    "train_mse": float(best_lasso_step_train_mse),
    "val_mse": float(best_lasso_step_val_mse),
    "nonzero_coef": best_lasso_step_nonzero,
    "notes": "Two-stage holdout search on the 708-feature step-function design."
}

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")


Lasso on best step-function feature space
Training shape: (115936, 708)
Validation shape: (28985, 708)
Reference OLS step validation MSE: 290.1093738555459
Reference Ridge-step validation MSE: 290.0792732215163

--- BROAD search | scaler = standard ---
broad | standard | ratio=1.00e-01 | alpha=1.12558852 | train_mse=357.524507 | val_mse=363.873625 | nonzero=  45 | n_iter=  130 | warning=False
broad | standard | ratio=6.31e-02 | alpha=0.71019834 | train_mse=331.857870 | val_mse=338.933256 | nonzero=  50 | n_iter=   25 | warning=False
broad | standard | ratio=3.98e-02 | alpha=0.44810486 | train_mse=317.691771 | val_mse=325.009682 | nonzero=  69 | n_iter=   45 | warning=False
broad | standard | ratio=2.51e-02 | alpha=0.28273505 | train_mse=307.916339 | val_mse=315.566760 | nonzero= 122 | n_iter=   44 | warning=False
broad | standard | ratio=1.58e-02 | alpha=0.17839376 | train_mse=299.891079 | val_mse=307.729047 | nonzero= 208 | n_iter=  101 | warning=False
broad | standard | ratio=1.00e-0

/Users/saadmanchowdhury/Desktop/All Github projects/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.951e+04, tolerance: 8.086e+03
  model = cd_fast.enet_coordinate_descent(



Refit check for best Lasso-step model:
Train MSE: 281.68849705581994
Validation MSE: 290.05901785647603
Nonzero coefficients: 658

Elapsed time: 96131.03 seconds


### Interpretation: Lasso on Step-Function Feature Space

We fit Lasso regression on the same 708-feature step-function design used for the OLS and Ridge step-function models. This model used a two-stage alpha search across both standard and robust scaling.

The best Lasso-step model used standard scaling with a very small penalty:

- Best validation MSE: 290.059
- Train MSE: 281.688
- Nonzero coefficients: 658 out of 708
- Best alpha: approximately 0.000310

This slightly improves on the previous step-function models:

- OLS step-function validation MSE: 290.109
- Ridge step-function validation MSE: 290.079
- Lasso step-function validation MSE: 290.059

#### Key Takeaways

The Lasso model gives the best validation MSE so far within the step-function feature family, but the improvement is very small. Compared with Ridge, the improvement is only about 0.02 MSE.

Lasso also retained nearly all of the step-function features, keeping 658 out of 708 coefficients nonzero. Therefore, in this feature space, Lasso is not acting as a strong feature-selection method. Instead, it behaves more like a lightly regularized linear model on the full expanded design.

The very long runtime and convergence difficulty suggest that further Lasso tuning on this same step-function feature space is unlikely to be worth the computational cost. The model is useful to record as the best step-function shrinkage result, but it does not materially change the broader conclusion: engineered linear basis expansions are beginning to plateau.

#### Implication

The best current model in the step-function family is Lasso + Step Functions, but the gains over Ridge and OLS are marginal. Future improvement likely requires either a different kind of nonlinear structure, such as splines, GAMs, trees, random forests, or boosting, or a different feature representation rather than more tuning of this same step-function design.

In [ ]:
# un-run model space DO NOT RUN! yet

# ============================================================
# 12F. Elastic Net sanity check on best step-function feature space
# Deferred model: run later if time permits
# ============================================================

from pathlib import Path
import gc
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

start_time = time.perf_counter()

print("Elastic Net on best step-function feature space")
print("Training shape:", X_tr_step_best.shape)
print("Validation shape:", X_val_step_best.shape)

print("\nReference models:")
print("OLS step-function validation MSE:", ols_step_val_mse)
print("Ridge-step validation MSE:", best_ridge_step_overall["val_mse"])
print("Lasso-step validation MSE:", best_lasso_step_val_mse)

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ELASTIC_STEP_RESULTS_PATH = RESULTS_DIR / "elastic_step_results_holdout.csv"
ELASTIC_STEP_BEST_OVERALL_PATH = RESULTS_DIR / "elastic_step_best_overall_holdout.csv"

OVERWRITE_ELASTIC_STEP_RESULTS = True

if OVERWRITE_ELASTIC_STEP_RESULTS:
    for path in [ELASTIC_STEP_RESULTS_PATH, ELASTIC_STEP_BEST_OVERALL_PATH]:
        if path.exists():
            path.unlink()

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

# ------------------------------------------------------------
# Scale once using training split only
# ------------------------------------------------------------

scaler = StandardScaler()
X_tr_step_scaled = scaler.fit_transform(X_tr_step_best)
X_val_step_scaled = scaler.transform(X_val_step_best)

# ------------------------------------------------------------
# Elastic Net search grid
# ------------------------------------------------------------
# l1_ratio = 1.0 is Lasso
# l1_ratio close to 0.0 is Ridge-like, but sklearn ElasticNet should not use exactly 0.
# We focus around mixed penalties and high-L1 penalties.

l1_ratios = [0.1, 0.25, 0.5, 0.75, 0.9]

# Use the best Lasso-step alpha as the center.
# Search one order of magnitude around it.
center_alpha = float(best_lasso_step_overall["alpha"])

alpha_multipliers = np.array([
    0.25,
    0.5,
    0.75,
    1.0,
    1.5,
    2.0,
    3.0,
    5.0,
    8.0,
    12.0
])

elastic_alphas = center_alpha * alpha_multipliers

print("\nCenter alpha from best Lasso-step:", center_alpha)
print("Elastic Net alpha grid:")
print(elastic_alphas)

# ------------------------------------------------------------
# Fit Elastic Net models
# ------------------------------------------------------------

elastic_step_rows = []

for l1_ratio in l1_ratios:
    print(f"\n--- Elastic Net search | l1_ratio = {l1_ratio} ---")
    
    # Start from stronger penalty and move downward with warm starts
    ordered_alphas = np.sort(elastic_alphas)[::-1]
    
    model = ElasticNet(
        alpha=ordered_alphas[0],
        l1_ratio=l1_ratio,
        fit_intercept=True,
        max_iter=70000,
        tol=1e-4,
        warm_start=True,
        selection="cyclic",
        random_state=RANDOM_STATE
    )
    
    for alpha in ordered_alphas:
        model.alpha = float(alpha)
        model.l1_ratio = float(l1_ratio)
        
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_tr_step_scaled, y_tr)
            
            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )
        
        train_pred = model.predict(X_tr_step_scaled)
        val_pred = model.predict(X_val_step_scaled)
        
        train_mse = mean_squared_error(y_tr, train_pred)
        val_mse = mean_squared_error(y_val, val_pred)
        nonzero_coef = int(np.sum(np.abs(model.coef_) > 1e-8))
        
        row = {
            "model_class": "Elastic Net + Step Functions",
            "scaler": "standard",
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "train_mse": train_mse,
            "val_mse": val_mse,
            "nonzero_coef": nonzero_coef,
            "n_iter": model.n_iter_,
            "convergence_warning": convergence_warning,
            "aic": np.nan,
            "bic": np.nan
        }
        
        elastic_step_rows.append(row)
        append_checkpoint(pd.DataFrame([row]), ELASTIC_STEP_RESULTS_PATH)
        
        print(
            f"l1_ratio={l1_ratio:4.2f} | "
            f"alpha={alpha:.8f} | "
            f"train_mse={train_mse:.6f} | "
            f"val_mse={val_mse:.6f} | "
            f"nonzero={nonzero_coef:4d} | "
            f"n_iter={model.n_iter_:5d} | "
            f"warning={convergence_warning}"
        )

elastic_step_results = pd.DataFrame(elastic_step_rows).sort_values("val_mse")

best_elastic_step_overall = elastic_step_results.iloc[0]
pd.DataFrame([best_elastic_step_overall]).to_csv(
    ELASTIC_STEP_BEST_OVERALL_PATH,
    index=False
)

print("\nElastic Net-step results sorted by validation MSE:")
print(elastic_step_results.to_string(index=False))

print("\nOverall best Elastic Net-step result:")
print(best_elastic_step_overall)

# ------------------------------------------------------------
# Refit best Elastic Net model
# ------------------------------------------------------------

best_elastic_step_model = make_pipeline(
    StandardScaler(),
    ElasticNet(
        alpha=float(best_elastic_step_overall["alpha"]),
        l1_ratio=float(best_elastic_step_overall["l1_ratio"]),
        fit_intercept=True,
        max_iter=100000,
        tol=1e-4,
        selection="cyclic",
        random_state=RANDOM_STATE
    )
)

best_elastic_step_model.fit(X_tr_step_best, y_tr)

best_elastic_step_train_pred = best_elastic_step_model.predict(X_tr_step_best)
best_elastic_step_val_pred = best_elastic_step_model.predict(X_val_step_best)

best_elastic_step_train_mse = mean_squared_error(y_tr, best_elastic_step_train_pred)
best_elastic_step_val_mse = mean_squared_error(y_val, best_elastic_step_val_pred)

best_elastic_step_nonzero = int(
    np.sum(np.abs(best_elastic_step_model.named_steps["elasticnet"].coef_) > 1e-8)
)

print("\nRefit check for best Elastic Net-step model:")
print("Train MSE:", best_elastic_step_train_mse)
print("Validation MSE:", best_elastic_step_val_mse)
print("Nonzero coefficients:", best_elastic_step_nonzero)

# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Elastic Net + Step Functions"] = {
    "model": best_elastic_step_model,
    "feature_space": "base_plus_poly_interactions + step_functions",
    "X_train_name": "X_tr_step_best",
    "X_val_name": "X_val_step_best",
    "scaler": "standard",
    "alpha": float(best_elastic_step_overall["alpha"]),
    "l1_ratio": float(best_elastic_step_overall["l1_ratio"]),
    "train_mse": float(best_elastic_step_train_mse),
    "val_mse": float(best_elastic_step_val_mse),
    "nonzero_coef": best_elastic_step_nonzero,
    "notes": "Deferred sanity-check Elastic Net search around the best Lasso-step alpha."
}

del X_tr_step_scaled, X_val_step_scaled
gc.collect()

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

### Deferred / Un-run Model Queue

The following model classes are currently deferred rather than discarded. They may be revisited later if time allows or if tree/boosting models plateau.

| Status | Model family | Feature space | Reason deferred |
|---|---|---|---|
| Un-run | Elastic Net + Step Functions | 708-feature step-function space | Ridge and Lasso were very close; full Elastic Net may be expensive and likely marginal |
| Un-run | PCR | Base / expanded linear feature spaces | Useful dimension-reduction baseline after linear models |
| Un-run | PLS | Base / expanded linear feature spaces | Supervised dimension reduction; may improve over PCR |
| Un-run | Regression Splines | Selected continuous predictors | More flexible/smoother alternative to step functions |
| Un-run | GAMs | Selected continuous predictors | Interpretable nonlinear additive model |
| Un-run | KNN Regression | Scaled reduced feature space | Could be expensive in high dimensions; likely needs dimensionality reduction |
| Un-run | SVR | Scaled reduced feature space | Potentially expensive for this sample size |
| Un-run | Neural Network | Scaled numeric/categorical feature space | Later-stage model; needs careful tuning |
| Un-run | BART | Reduced feature space | Useful tree-based Bayesian method, but may be package/runtime dependent |
| Un-run | Elastic Net | base_plus_poly_interactions or reduced step-function space | More sensible than running Elastic Net on all 708 step-function features; avoids repeating the expensive Lasso-step search with little expected gain |

Current priority after the step-function linear family is to move to tree-based models: regression tree, bagging, random forest, and gradient boosting.

In [29]:
# ============================================================
# 13A. Regression Tree baseline (quick holdout screen)
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

start_time = time.perf_counter()

# ------------------------------------------------------------
# Design choice:
# Trees already learn threshold/step-like structure internally.
# So for the first tree baseline, we compare the controlled
# pre-step feature spaces rather than the 708-feature step space.
# ------------------------------------------------------------

tree_feature_spaces = {
    "base": feature_spaces["base"],
    "base_plus_poly": feature_spaces["base_plus_poly"],
    "base_plus_interactions": feature_spaces["base_plus_interactions"],
    "base_plus_poly_interactions": feature_spaces["base_plus_poly_interactions"]
}

print("Regression tree baseline")
print("Feature spaces:")
for name, (X_train_space, X_val_space) in tree_feature_spaces.items():
    print(f"  {name}: train {X_train_space.shape}, validation {X_val_space.shape}")

# ------------------------------------------------------------
# Reference models
# ------------------------------------------------------------

print("\nReference validation MSEs:")
if "best_ridge_overall" in globals():
    print("Best Ridge:", best_ridge_overall["val_mse"])

if "best_lasso_overall" in globals():
    print("Best Lasso:", best_lasso_overall["val_mse"])

if "best_elasticnet_overall" in globals():
    print("Best Elastic Net:", best_elasticnet_overall["val_mse"])

if "ols_step_val_mse" in globals():
    print("OLS step-functions:", ols_step_val_mse)

if "best_ridge_step_overall" in globals():
    print("Ridge + step-functions:", best_ridge_step_overall["val_mse"])

if "best_lasso_step_val_mse" in globals():
    print("Lasso + step-functions:", best_lasso_step_val_mse)

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

TREE_RESULTS_PATH = RESULTS_DIR / "regression_tree_results_holdout.csv"
TREE_BEST_BY_SPACE_PATH = RESULTS_DIR / "regression_tree_best_by_space_holdout.csv"
TREE_BEST_OVERALL_PATH = RESULTS_DIR / "regression_tree_best_overall_holdout.csv"

OVERWRITE_TREE_RESULTS = True

if OVERWRITE_TREE_RESULTS:
    for path in [
        TREE_RESULTS_PATH,
        TREE_BEST_BY_SPACE_PATH,
        TREE_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

# ------------------------------------------------------------
# Tree hyperparameter grid
# ------------------------------------------------------------
# max_depth controls tree complexity.
# min_samples_leaf controls how small terminal leaves are allowed to be.
# Larger leaves usually reduce overfitting.

max_depth_grid = [3, 5, 7, 9, 12, 15, None]
min_samples_leaf_grid = [25, 50, 100, 250, 500, 1000, 2000]

tree_rows = []

for feature_space_name, (X_train_space, X_val_space) in tree_feature_spaces.items():
    print(f"\n--- Feature space: {feature_space_name} ---")
    
    for max_depth in max_depth_grid:
        for min_samples_leaf in min_samples_leaf_grid:
            
            model = DecisionTreeRegressor(
                criterion="squared_error",
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                min_samples_split=max(2, 2 * min_samples_leaf),
                random_state=RANDOM_STATE
            )
            
            model.fit(X_train_space, y_tr)
            
            train_pred = model.predict(X_train_space)
            val_pred = model.predict(X_val_space)
            
            train_mse = mean_squared_error(y_tr, train_pred)
            val_mse = mean_squared_error(y_val, val_pred)
            
            row = {
                "model_class": "Regression Tree",
                "feature_space": feature_space_name,
                "max_depth": "None" if max_depth is None else max_depth,
                "min_samples_leaf": min_samples_leaf,
                "min_samples_split": max(2, 2 * min_samples_leaf),
                "train_mse": train_mse,
                "val_mse": val_mse,
                "n_leaves": model.get_n_leaves(),
                "tree_depth": model.get_depth(),
                "aic": np.nan,
                "bic": np.nan
            }
            
            tree_rows.append(row)
            append_checkpoint(pd.DataFrame([row]), TREE_RESULTS_PATH)
            
            print(
                f"depth={str(max_depth):>4s} | "
                f"leaf={min_samples_leaf:4d} | "
                f"train_mse={train_mse:.6f} | "
                f"val_mse={val_mse:.6f} | "
                f"leaves={model.get_n_leaves():5d} | "
                f"actual_depth={model.get_depth():3d}"
            )

tree_results = pd.DataFrame(tree_rows).sort_values("val_mse")

best_tree_by_space = (
    tree_results
    .loc[tree_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_tree_overall = tree_results.iloc[0]

best_tree_by_space.to_csv(TREE_BEST_BY_SPACE_PATH, index=False)
pd.DataFrame([best_tree_overall]).to_csv(TREE_BEST_OVERALL_PATH, index=False)

print("\nBest regression tree result by feature space:")
print(best_tree_by_space.to_string(index=False))

print("\nTop 15 regression tree results overall:")
print(tree_results.head(15).to_string(index=False))

print("\nOverall best regression tree result:")
print(best_tree_overall)

# ------------------------------------------------------------
# Refit best tree on the development training split
# ------------------------------------------------------------

best_tree_feature_space_name = best_tree_overall["feature_space"]
X_train_best_tree, X_val_best_tree = tree_feature_spaces[best_tree_feature_space_name]

best_tree_max_depth = best_tree_overall["max_depth"]
if best_tree_max_depth == "None":
    best_tree_max_depth = None
else:
    best_tree_max_depth = int(best_tree_max_depth)

best_tree_min_samples_leaf = int(best_tree_overall["min_samples_leaf"])

best_tree_model = DecisionTreeRegressor(
    criterion="squared_error",
    max_depth=best_tree_max_depth,
    min_samples_leaf=best_tree_min_samples_leaf,
    min_samples_split=max(2, 2 * best_tree_min_samples_leaf),
    random_state=RANDOM_STATE
)

best_tree_model.fit(X_train_best_tree, y_tr)

best_tree_train_pred = best_tree_model.predict(X_train_best_tree)
best_tree_val_pred = best_tree_model.predict(X_val_best_tree)

best_tree_train_mse = mean_squared_error(y_tr, best_tree_train_pred)
best_tree_val_mse = mean_squared_error(y_val, best_tree_val_pred)

print("\nRefit check for best regression tree:")
print("Feature space:", best_tree_feature_space_name)
print("Train MSE:", best_tree_train_mse)
print("Validation MSE:", best_tree_val_mse)
print("Leaves:", best_tree_model.get_n_leaves())
print("Depth:", best_tree_model.get_depth())

# ------------------------------------------------------------
# Feature importances for the best tree
# If duplicate column names exist, group them safely.
# ------------------------------------------------------------

tree_importances = (
    pd.Series(best_tree_model.feature_importances_, index=X_train_best_tree.columns)
    .groupby(level=0)
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 30 regression tree feature importances:")
print(tree_importances.head(30).to_string())

# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Regression Tree"] = {
    "model": best_tree_model,
    "feature_space": best_tree_feature_space_name,
    "X_train_name": f"feature_spaces['{best_tree_feature_space_name}'][0]",
    "X_val_name": f"feature_spaces['{best_tree_feature_space_name}'][1]",
    "max_depth": best_tree_max_depth,
    "min_samples_leaf": best_tree_min_samples_leaf,
    "min_samples_split": max(2, 2 * best_tree_min_samples_leaf),
    "train_mse": float(best_tree_train_mse),
    "val_mse": float(best_tree_val_mse),
    "n_leaves": int(best_tree_model.get_n_leaves()),
    "depth": int(best_tree_model.get_depth()),
    "notes": "Single regression tree baseline across controlled pre-step feature spaces."
}

gc.collect()

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

Regression tree baseline
Feature spaces:
  base: train (115936, 162), validation (28985, 162)
  base_plus_poly: train (115936, 172), validation (28985, 172)
  base_plus_interactions: train (115936, 172), validation (28985, 172)
  base_plus_poly_interactions: train (115936, 182), validation (28985, 182)

Reference validation MSEs:
Best Ridge: 308.35660869330025
OLS step-functions: 290.1093738555459
Ridge + step-functions: 290.0792732215163

--- Feature space: base ---
depth=   3 | leaf=  25 | train_mse=520.811279 | val_mse=527.943002 | leaves=    8 | actual_depth=  3
depth=   3 | leaf=  50 | train_mse=520.811279 | val_mse=527.943002 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 100 | train_mse=520.811279 | val_mse=527.943002 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 250 | train_mse=520.811279 | val_mse=527.943002 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 500 | train_mse=520.811279 | val_mse=527.943002 | leaves=    8 | actual_depth=  3
depth=   3 | leaf=1000 

### Interpretation: Regression Tree Holdout Baseline

The regression tree baseline produced a major improvement over the previous linear and step-function model families on the holdout validation set.

#### Best Holdout Regression Tree

- Validation MSE: approximately 240.57
- Train MSE: approximately 190.54
- Tree depth: 45
- Number of leaves: 3522
- Minimum samples per leaf: 25
- Best feature space: `base_plus_poly_interactions`

#### Comparison with Previous Best Models

| Model | Validation MSE |
|---|---:|
| OLS + Step Functions | ~290.11 |
| Ridge + Step Functions | ~290.08 |
| Lasso + Step Functions | ~290.06 |
| Regression Tree (holdout) | ~240.57 |

This represents a very large improvement relative to the previous linear-basis-expansion models.

#### Key Takeaways

The strong improvement suggests that the underlying data-generating structure contains important nonlinear interactions and threshold effects that are difficult for linear models and manually engineered step functions to capture efficiently.

Unlike the previous models, the regression tree learns interaction structure and nonlinear thresholds automatically. This appears particularly valuable for this dataset, where demographic, subgroup, grade-level, and assessment-related variables likely interact in complex ways.

The feature importances indicate that the tree heavily relies on variables related to:
- economic disadvantage,
- grade structure,
- assessment categories,
- subgroup indicators,
- attendance,
- school frequency,
- and demographic composition.

#### Important Caution

This result was obtained using the repeatedly-used holdout validation split rather than cross-validation. Because many modeling decisions have already been made using this holdout set, the result should currently be treated as an exploratory screening result rather than the final official regression-tree estimate.

Therefore, the next step is to run K-fold cross-validation for regression trees in order to determine whether this large improvement generalizes across folds or whether part of the gain reflects holdout-specific tuning.

In [43]:
# Preserve the quick holdout tree result before running CV tree selection

if "best_models_by_class" in globals() and "Regression Tree" in best_models_by_class:
    best_models_by_class["Regression Tree (holdout screen)"] = best_models_by_class["Regression Tree"]
    print("Saved holdout-screen tree as: Regression Tree (holdout screen)")
else:
    print("No holdout-screen Regression Tree found in best_models_by_class.")

Saved holdout-screen tree as: Regression Tree (holdout screen)


### Cross-Validation Status Tracker

| Model family | Current status | Notes |
|---|---|---|
| Simple Linear Regression | Holdout only | Exploratory baseline |
| Multiple Linear Regression | Holdout only | Exploratory baseline |
| Polynomial Regression | Holdout only | Used for feature-space development |
| Interaction Models | Holdout only | Used for feature-space development |
| Step Functions | Holdout only | Exploratory nonlinear basis expansion |
| Ridge Regression | Holdout only | Pre-tree regularization screen |
| Lasso | Holdout only | Pre-tree regularization screen |
| Elastic Net | Deferred / not fully run | Planned later on reduced feature space |
| Ridge + Step Functions | Holdout only | Exploratory shrinkage on expanded basis |
| Lasso + Step Functions | Holdout only | Computationally expensive exploratory result |
| Regression Tree (holdout screen) | Holdout only | Large improvement observed (~240 validation MSE) |
Regression Tree (5-fold CV) below | Complete | Official tree baseline: CV MSE ≈ 250.68; holdout MSE ≈ 241.35

### Planned Future CV Priorities

1. Regression Tree CV
2. Random Forest CV
3. Gradient Boosting / XGBoost CV
4. Compare top tree-based models against holdout-screen linear models
5. Optional: CV revisit for strongest linear-basis models if needed

### Important Validation Note

The original holdout validation split was useful for rapid exploratory screening and feature-space development. However, because many modeling decisions were made using that same split, future serious model-family comparisons should rely primarily on cross-validation, with the holdout set serving as a final secondary sanity check rather than the primary selection mechanism.

In [44]:
# ============================================================
# 13B. Regression Tree baseline with K-fold CV
# ============================================================

from pathlib import Path
from itertools import combinations
import time
import gc

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

start_time = time.perf_counter()

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Regression tree baseline with K-fold CV")
print("Base training split shape:", X_tr.shape)
print("Holdout validation shape:", X_val.shape)

# ------------------------------------------------------------
# Important validation design:
#
# We do NOT use the existing feature_spaces dictionary here,
# because those engineered spaces were built after selecting
# top features using the holdout validation split.
#
# For CV, we rebuild top polynomial/interaction candidates
# inside each CV fold using only that fold's training data.
# This avoids leaking validation-fold information into the
# engineered feature space.
# ------------------------------------------------------------

def select_top_by_abs_corr(X_train_df, y_train_array, top_n):
    """
    Select top predictors by absolute correlation with y.
    This is computed using only the training portion of a CV fold.
    """
    y_arr = np.asarray(y_train_array, dtype=float)
    y_centered = y_arr - y_arr.mean()
    y_norm = np.sqrt(np.sum(y_centered ** 2))
    
    scores = []
    
    for col in X_train_df.columns:
        x_arr = X_train_df[col].to_numpy(dtype=float)
        x_centered = x_arr - x_arr.mean()
        x_norm = np.sqrt(np.sum(x_centered ** 2))
        
        denom = x_norm * y_norm
        
        if denom == 0 or not np.isfinite(denom):
            score = 0.0
        else:
            score = abs(float(np.sum(x_centered * y_centered) / denom))
        
        scores.append((col, score))
    
    scores = sorted(scores, key=lambda z: z[1], reverse=True)
    return [col for col, score in scores[:top_n]]


def build_controlled_tree_feature_spaces(X_train_raw, X_eval_raw, y_train_array):
    """
    Build the same four controlled feature spaces, but with top
    polynomial and interaction candidates selected from X_train_raw only.
    """
    top10_features = select_top_by_abs_corr(X_train_raw, y_train_array, top_n=10)
    top5_features = select_top_by_abs_corr(X_train_raw, y_train_array, top_n=5)
    
    # Base space
    X_train_base = X_train_raw.copy()
    X_eval_base = X_eval_raw.copy()
    
    # Base + squared terms
    X_train_poly = X_train_raw.copy()
    X_eval_poly = X_eval_raw.copy()
    
    for col in top10_features:
        new_col = f"{col}__squared"
        X_train_poly[new_col] = X_train_raw[col] ** 2
        X_eval_poly[new_col] = X_eval_raw[col] ** 2
    
    # Base + pairwise interactions
    X_train_inter = X_train_raw.copy()
    X_eval_inter = X_eval_raw.copy()
    
    for f1, f2 in combinations(top5_features, 2):
        new_col = f"{f1}__x__{f2}"
        X_train_inter[new_col] = X_train_raw[f1] * X_train_raw[f2]
        X_eval_inter[new_col] = X_eval_raw[f1] * X_eval_raw[f2]
    
    # Base + squared terms + interactions
    X_train_combined = X_train_poly.copy()
    X_eval_combined = X_eval_poly.copy()
    
    for f1, f2 in combinations(top5_features, 2):
        new_col = f"{f1}__x__{f2}"
        X_train_combined[new_col] = X_train_raw[f1] * X_train_raw[f2]
        X_eval_combined[new_col] = X_eval_raw[f1] * X_eval_raw[f2]
    
    feature_spaces_fold = {
        "base": (X_train_base, X_eval_base),
        "base_plus_poly": (X_train_poly, X_eval_poly),
        "base_plus_interactions": (X_train_inter, X_eval_inter),
        "base_plus_poly_interactions": (X_train_combined, X_eval_combined)
    }
    
    return feature_spaces_fold, top10_features, top5_features


# ------------------------------------------------------------
# Reference models already completed
# ------------------------------------------------------------

print("\nReference holdout validation MSEs:")
if "ols_step_val_mse" in globals():
    print("OLS step-functions:", ols_step_val_mse)

if "best_ridge_step_overall" in globals():
    print("Ridge + step-functions:", best_ridge_step_overall["val_mse"])

if "best_lasso_step_val_mse" in globals():
    print("Lasso + step-functions:", best_lasso_step_val_mse)
elif "best_lasso_step_overall" in globals():
    print("Lasso + step-functions:", best_lasso_step_overall["val_mse"])

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

TREE_CV_FOLD_RESULTS_PATH = RESULTS_DIR / "regression_tree_cv_fold_results.csv"
TREE_CV_SUMMARY_PATH = RESULTS_DIR / "regression_tree_cv_summary.csv"
TREE_CV_BEST_OVERALL_PATH = RESULTS_DIR / "regression_tree_cv_best_overall.csv"

OVERWRITE_TREE_CV_RESULTS = True

if OVERWRITE_TREE_CV_RESULTS:
    for path in [
        TREE_CV_FOLD_RESULTS_PATH,
        TREE_CV_SUMMARY_PATH,
        TREE_CV_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

# ------------------------------------------------------------
# CV setup and tree grid
# ------------------------------------------------------------

N_SPLITS_TREE_CV = 5

kf = KFold(
    n_splits=N_SPLITS_TREE_CV,
    shuffle=True,
    random_state=RANDOM_STATE
)

max_depth_grid = [3, 5, 7, 9, 12, 15, None]
min_samples_leaf_grid = [25, 50, 100, 250, 500, 1000, 2000]

y_tr_array = np.asarray(y_tr, dtype=float)

tree_cv_rows = []

# ------------------------------------------------------------
# Run CV
# ------------------------------------------------------------

for fold_id, (cv_train_idx, cv_valid_idx) in enumerate(kf.split(X_tr), start=1):
    print(f"\n================ Fold {fold_id} / {N_SPLITS_TREE_CV} ================")
    
    X_cv_train_raw = X_tr.iloc[cv_train_idx].copy()
    X_cv_valid_raw = X_tr.iloc[cv_valid_idx].copy()
    
    y_cv_train = y_tr_array[cv_train_idx]
    y_cv_valid = y_tr_array[cv_valid_idx]
    
    fold_feature_spaces, fold_top10, fold_top5 = build_controlled_tree_feature_spaces(
        X_cv_train_raw,
        X_cv_valid_raw,
        y_cv_train
    )
    
    print("Top 10 fold features for squared terms:")
    print(fold_top10)
    print("Top 5 fold features for interactions:")
    print(fold_top5)
    
    for feature_space_name, (X_cv_train_space, X_cv_valid_space) in fold_feature_spaces.items():
        print(f"\n--- Fold {fold_id} | feature space: {feature_space_name} | shape: {X_cv_train_space.shape} ---")
        
        for max_depth in max_depth_grid:
            for min_samples_leaf in min_samples_leaf_grid:
                
                model = DecisionTreeRegressor(
                    criterion="squared_error",
                    max_depth=max_depth,
                    min_samples_leaf=min_samples_leaf,
                    min_samples_split=max(2, 2 * min_samples_leaf),
                    random_state=RANDOM_STATE
                )
                
                model.fit(X_cv_train_space, y_cv_train)
                
                train_pred = model.predict(X_cv_train_space)
                valid_pred = model.predict(X_cv_valid_space)
                
                train_mse = mean_squared_error(y_cv_train, train_pred)
                valid_mse = mean_squared_error(y_cv_valid, valid_pred)
                
                row = {
                    "model_class": "Regression Tree",
                    "fold": fold_id,
                    "feature_space": feature_space_name,
                    "max_depth": "None" if max_depth is None else max_depth,
                    "min_samples_leaf": min_samples_leaf,
                    "min_samples_split": max(2, 2 * min_samples_leaf),
                    "fold_train_mse": train_mse,
                    "fold_valid_mse": valid_mse,
                    "n_leaves": model.get_n_leaves(),
                    "tree_depth": model.get_depth()
                }
                
                tree_cv_rows.append(row)
                append_checkpoint(pd.DataFrame([row]), TREE_CV_FOLD_RESULTS_PATH)
                
                print(
                    f"depth={str(max_depth):>4s} | "
                    f"leaf={min_samples_leaf:4d} | "
                    f"fold_train_mse={train_mse:.6f} | "
                    f"fold_valid_mse={valid_mse:.6f} | "
                    f"leaves={model.get_n_leaves():5d} | "
                    f"actual_depth={model.get_depth():3d}"
                )
    
    del X_cv_train_raw, X_cv_valid_raw, fold_feature_spaces
    gc.collect()

# ------------------------------------------------------------
# Summarize CV results
# ------------------------------------------------------------

tree_cv_fold_results = pd.DataFrame(tree_cv_rows)

group_cols = [
    "model_class",
    "feature_space",
    "max_depth",
    "min_samples_leaf",
    "min_samples_split"
]

tree_cv_summary = (
    tree_cv_fold_results
    .groupby(group_cols)
    .agg(
        cv_train_mse_mean=("fold_train_mse", "mean"),
        cv_train_mse_std=("fold_train_mse", "std"),
        cv_valid_mse_mean=("fold_valid_mse", "mean"),
        cv_valid_mse_std=("fold_valid_mse", "std"),
        mean_leaves=("n_leaves", "mean"),
        mean_depth=("tree_depth", "mean")
    )
    .reset_index()
    .sort_values("cv_valid_mse_mean")
)

tree_cv_summary.to_csv(TREE_CV_SUMMARY_PATH, index=False)

best_tree_cv_by_space = (
    tree_cv_summary
    .loc[tree_cv_summary.groupby("feature_space")["cv_valid_mse_mean"].idxmin()]
    .sort_values("cv_valid_mse_mean")
)

best_tree_cv_overall = tree_cv_summary.iloc[0]
pd.DataFrame([best_tree_cv_overall]).to_csv(TREE_CV_BEST_OVERALL_PATH, index=False)

print("\nBest CV regression tree result by feature space:")
print(best_tree_cv_by_space.to_string(index=False))

print("\nTop 15 CV regression tree results overall:")
print(tree_cv_summary.head(15).to_string(index=False))

print("\nOverall best CV regression tree result:")
print(best_tree_cv_overall)

# ------------------------------------------------------------
# Refit the CV-selected tree on the full development training split
# and evaluate once on the holdout validation split
# ------------------------------------------------------------

best_tree_feature_space_name = best_tree_cv_overall["feature_space"]

final_feature_spaces, final_top10, final_top5 = build_controlled_tree_feature_spaces(
    X_tr,
    X_val,
    y_tr_array
)

X_train_best_tree, X_val_best_tree = final_feature_spaces[best_tree_feature_space_name]

best_tree_max_depth = best_tree_cv_overall["max_depth"]
if best_tree_max_depth == "None":
    best_tree_max_depth = None
else:
    best_tree_max_depth = int(best_tree_max_depth)

best_tree_min_samples_leaf = int(best_tree_cv_overall["min_samples_leaf"])

best_tree_model = DecisionTreeRegressor(
    criterion="squared_error",
    max_depth=best_tree_max_depth,
    min_samples_leaf=best_tree_min_samples_leaf,
    min_samples_split=max(2, 2 * best_tree_min_samples_leaf),
    random_state=RANDOM_STATE
)

best_tree_model.fit(X_train_best_tree, y_tr)

best_tree_train_pred = best_tree_model.predict(X_train_best_tree)
best_tree_val_pred = best_tree_model.predict(X_val_best_tree)

best_tree_train_mse = mean_squared_error(y_tr, best_tree_train_pred)
best_tree_val_mse = mean_squared_error(y_val, best_tree_val_pred)

print("\nRefit check for CV-selected regression tree:")
print("Feature space:", best_tree_feature_space_name)
print("CV mean validation MSE:", best_tree_cv_overall["cv_valid_mse_mean"])
print("CV validation MSE std:", best_tree_cv_overall["cv_valid_mse_std"])
print("Holdout train MSE:", best_tree_train_mse)
print("Holdout validation MSE:", best_tree_val_mse)
print("Leaves:", best_tree_model.get_n_leaves())
print("Depth:", best_tree_model.get_depth())

print("\nFinal top 10 features used for squared terms:")
print(final_top10)

print("\nFinal top 5 features used for interactions:")
print(final_top5)

# ------------------------------------------------------------
# Feature importances for the final CV-selected tree
# ------------------------------------------------------------

tree_importances = (
    pd.Series(best_tree_model.feature_importances_, index=X_train_best_tree.columns)
    .groupby(level=0)
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 30 regression tree feature importances:")
print(tree_importances.head(30).to_string())

# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Regression Tree"] = {
    "model": best_tree_model,
    "feature_space": best_tree_feature_space_name,
    "X_train_name": "CV-built feature space from X_tr",
    "X_val_name": "CV-built feature space from X_val",
    "max_depth": best_tree_max_depth,
    "min_samples_leaf": best_tree_min_samples_leaf,
    "min_samples_split": max(2, 2 * best_tree_min_samples_leaf),
    "cv_valid_mse_mean": float(best_tree_cv_overall["cv_valid_mse_mean"]),
    "cv_valid_mse_std": float(best_tree_cv_overall["cv_valid_mse_std"]),
    "holdout_train_mse": float(best_tree_train_mse),
    "holdout_val_mse": float(best_tree_val_mse),
    "n_leaves": int(best_tree_model.get_n_leaves()),
    "depth": int(best_tree_model.get_depth()),
    "notes": (
        "Regression tree selected by 5-fold CV on X_tr. "
        "Holdout validation used only after CV selection."
    )
}

del final_feature_spaces
gc.collect()

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

Regression tree baseline with K-fold CV
Base training split shape: (115936, 162)
Holdout validation shape: (28985, 162)

Reference holdout validation MSEs:
OLS step-functions: 290.1093738555459
Ridge + step-functions: 290.0792732215163

================ Fold 1 / 5 ================


Top 10 fold features for squared terms:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_WITH_DISABILITIES', 'ATTENDANCE_RATE', 'PERCENT_DROPOUT']
Top 5 fold features for interactions:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']

--- Fold 1 | feature space: base | shape: (92748, 162) ---
depth=   3 | leaf=  25 | fold_train_mse=520.426897 | fold_valid_mse=522.403878 | leaves=    8 | actual_depth=  3
depth=   3 | leaf=  50 | fold_train_mse=520.426897 | fold_valid_mse=522.403878 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 100 | fold_train_mse=520.426897 | fold_valid_mse=522.403878 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 250 | fold_train_mse=520.426897 | fold_valid_mse=522.403878 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 500 | fold_train_ms

### Regression Tree Cross-Validation Summary

The 5-fold cross-validation results confirm that regression trees substantially improve over the earlier linear and step-function baselines. The best CV-selected tree used the `base_plus_poly_interactions` feature space with `max_depth=None`, `min_samples_leaf=25`, and `min_samples_split=50`. This model achieved a mean CV validation MSE of approximately **250.68** with a standard deviation of approximately **4.65**.

When refit on the full training portion and evaluated on the original holdout validation set, the selected tree achieved a holdout validation MSE of approximately **241.35**. This supports the earlier holdout-screen result and suggests that the tree model captures important nonlinear and interaction structure that the linear-basis models missed.

The expanded polynomial and interaction tree feature spaces only modestly improved over the simpler base feature space. This is not surprising because regression trees already create nonlinear and interaction effects through recursive splitting. Therefore, the tree result should be treated as the first serious non-linear baseline, while random forests and gradient boosting are natural next steps.

### Interpretation: Regression Tree with 5-Fold Cross-Validation

We evaluated regression trees using 5-fold cross-validation on the development training split. Unlike the earlier holdout-only tree screen, this CV run gives a more reliable estimate of tree performance and reduces dependence on a single validation split.

The best CV-selected regression tree used:

- Feature space: `base_plus_poly_interactions`
- Maximum depth: unrestricted (`None`)
- Minimum samples per leaf: 25
- Mean CV validation MSE: 250.68
- CV validation MSE standard deviation: 4.65
- Mean number of leaves: about 2828
- Mean tree depth: about 40

After refitting this CV-selected tree on the full development training split, the holdout validation MSE was 241.35.

#### Comparison with Previous Best Models

| Model | Validation estimate |
|---|---:|
| Lasso + Step Functions | Holdout MSE ≈ 290.06 |
| Regression Tree | CV MSE ≈ 250.68 |
| Regression Tree | Holdout MSE ≈ 241.35 |

The regression tree represents a large improvement over the previous linear, regularized linear, and step-function models. The CV result confirms that the improvement is not merely an artifact of a single holdout split.

#### Key Takeaways

The tree model captures nonlinear threshold effects and interactions much more effectively than manually engineered linear basis expansions. This is consistent with the strong importance of variables related to economic disadvantage, grade level, assessment type, subgroup membership, attendance, region, school frequency, and demographic composition.

The selected tree is very large, with thousands of leaves and an unrestricted depth. This gives the model high flexibility, but it also creates a substantial gap between training error and CV validation error. Therefore, the single tree is useful as a strong baseline, but it is likely high-variance.

#### Implication

Tree-based methods are now the most promising model family. The next logical step is to use ensemble tree methods, especially bagging and random forests, to reduce variance while preserving the nonlinear and interaction-capturing strengths of regression trees.

In [30]:
# ============================================================
# 14A. Random Forest: controlled 3-fold CV screen
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.metrics import mean_squared_error

start_time = time.perf_counter()

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Random Forest controlled CV screen")
print("Using base feature space only for first RF screen.")
print("X_tr shape:", X_tr.shape)
print("X_val shape:", X_val.shape)

# ------------------------------------------------------------
# Reference: current best regression tree
# ------------------------------------------------------------

if "best_models_by_class" in globals() and "Regression Tree" in best_models_by_class:
    tree_ref = best_models_by_class["Regression Tree"]
    print("\nReference Regression Tree:")
    print("CV validation MSE mean:", tree_ref.get("cv_valid_mse_mean"))
    print("Holdout validation MSE:", tree_ref.get("holdout_val_mse"))

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

RF_CV_FOLD_RESULTS_PATH = RESULTS_DIR / "random_forest_cv_fold_results_screen.csv"
RF_CV_SUMMARY_PATH = RESULTS_DIR / "random_forest_cv_summary_screen.csv"
RF_CV_BEST_OVERALL_PATH = RESULTS_DIR / "random_forest_cv_best_overall_screen.csv"

OVERWRITE_RF_CV_RESULTS = True

if OVERWRITE_RF_CV_RESULTS:
    for path in [
        RF_CV_FOLD_RESULTS_PATH,
        RF_CV_SUMMARY_PATH,
        RF_CV_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

# ------------------------------------------------------------
# CV setup
# ------------------------------------------------------------

N_SPLITS_RF_CV = 3

kf = KFold(
    n_splits=N_SPLITS_RF_CV,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Keep this modest for the first RF pass.
# If RF is promising, we will refine later.
RF_N_ESTIMATORS = 120
RF_MAX_SAMPLES = 0.80

rf_param_grid = list(ParameterGrid({
    "max_features": ["sqrt", 0.50],
    "min_samples_leaf": [1, 5, 25],
}))

print("\nRF screen settings:")
print("CV folds:", N_SPLITS_RF_CV)
print("n_estimators:", RF_N_ESTIMATORS)
print("max_samples:", RF_MAX_SAMPLES)
print("parameter combinations:", len(rf_param_grid))

y_tr_array = np.asarray(y_tr, dtype=float)

rf_cv_rows = []

# ------------------------------------------------------------
# Run CV on base feature space
# ------------------------------------------------------------

for fold_id, (cv_train_idx, cv_valid_idx) in enumerate(kf.split(X_tr), start=1):
    print(f"\n================ RF Fold {fold_id} / {N_SPLITS_RF_CV} ================")
    
    X_cv_train = X_tr.iloc[cv_train_idx]
    X_cv_valid = X_tr.iloc[cv_valid_idx]
    
    y_cv_train = y_tr_array[cv_train_idx]
    y_cv_valid = y_tr_array[cv_valid_idx]
    
    for combo_id, params in enumerate(rf_param_grid, start=1):
        max_features = params["max_features"]
        min_samples_leaf = int(params["min_samples_leaf"])
        min_samples_split = max(2, 2 * min_samples_leaf)
        
        model = RandomForestRegressor(
            n_estimators=RF_N_ESTIMATORS,
            criterion="squared_error",
            max_depth=None,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            max_features=max_features,
            bootstrap=True,
            max_samples=RF_MAX_SAMPLES,
            oob_score=True,
            n_jobs=-1,
            random_state=RANDOM_STATE + 1000 * fold_id + combo_id
        )
        
        model.fit(X_cv_train, y_cv_train)
        
        train_pred = model.predict(X_cv_train)
        valid_pred = model.predict(X_cv_valid)
        
        train_mse = mean_squared_error(y_cv_train, train_pred)
        valid_mse = mean_squared_error(y_cv_valid, valid_pred)
        
        # OOB predictions are an internal training-set generalization check.
        oob_pred = model.oob_prediction_
        oob_mask = np.isfinite(oob_pred)
        if oob_mask.sum() > 0:
            oob_mse = mean_squared_error(y_cv_train[oob_mask], oob_pred[oob_mask])
        else:
            oob_mse = np.nan
        
        row = {
            "model_class": "Random Forest",
            "fold": fold_id,
            "feature_space": "base",
            "n_estimators": RF_N_ESTIMATORS,
            "max_samples": RF_MAX_SAMPLES,
            "max_features": str(max_features),
            "min_samples_leaf": min_samples_leaf,
            "min_samples_split": min_samples_split,
            "fold_train_mse": train_mse,
            "fold_oob_mse": oob_mse,
            "fold_valid_mse": valid_mse
        }
        
        rf_cv_rows.append(row)
        append_checkpoint(pd.DataFrame([row]), RF_CV_FOLD_RESULTS_PATH)
        
        print(
            f"combo={combo_id:2d} | "
            f"max_features={str(max_features):>4s} | "
            f"leaf={min_samples_leaf:3d} | "
            f"train_mse={train_mse:.6f} | "
            f"oob_mse={oob_mse:.6f} | "
            f"valid_mse={valid_mse:.6f}"
        )
    
    gc.collect()

# ------------------------------------------------------------
# Summarize CV results
# ------------------------------------------------------------

rf_cv_fold_results = pd.DataFrame(rf_cv_rows)

group_cols = [
    "model_class",
    "feature_space",
    "n_estimators",
    "max_samples",
    "max_features",
    "min_samples_leaf",
    "min_samples_split"
]

rf_cv_summary = (
    rf_cv_fold_results
    .groupby(group_cols)
    .agg(
        cv_train_mse_mean=("fold_train_mse", "mean"),
        cv_train_mse_std=("fold_train_mse", "std"),
        cv_oob_mse_mean=("fold_oob_mse", "mean"),
        cv_oob_mse_std=("fold_oob_mse", "std"),
        cv_valid_mse_mean=("fold_valid_mse", "mean"),
        cv_valid_mse_std=("fold_valid_mse", "std")
    )
    .reset_index()
    .sort_values("cv_valid_mse_mean")
)

rf_cv_summary.to_csv(RF_CV_SUMMARY_PATH, index=False)

best_rf_cv_overall = rf_cv_summary.iloc[0]
pd.DataFrame([best_rf_cv_overall]).to_csv(RF_CV_BEST_OVERALL_PATH, index=False)

print("\nRandom Forest CV summary:")
print(rf_cv_summary.to_string(index=False))

print("\nBest Random Forest CV result:")
print(best_rf_cv_overall)

# ------------------------------------------------------------
# Refit selected RF on full development training split
# and evaluate once on holdout validation split
# ------------------------------------------------------------

best_rf_max_features = best_rf_cv_overall["max_features"]
if best_rf_max_features != "sqrt":
    best_rf_max_features = float(best_rf_max_features)

best_rf_min_samples_leaf = int(best_rf_cv_overall["min_samples_leaf"])
best_rf_min_samples_split = max(2, 2 * best_rf_min_samples_leaf)

best_rf_model = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    criterion="squared_error",
    max_depth=None,
    min_samples_leaf=best_rf_min_samples_leaf,
    min_samples_split=best_rf_min_samples_split,
    max_features=best_rf_max_features,
    bootstrap=True,
    max_samples=RF_MAX_SAMPLES,
    oob_score=True,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

best_rf_model.fit(X_tr, y_tr)

best_rf_train_pred = best_rf_model.predict(X_tr)
best_rf_val_pred = best_rf_model.predict(X_val)

best_rf_train_mse = mean_squared_error(y_tr, best_rf_train_pred)
best_rf_val_mse = mean_squared_error(y_val, best_rf_val_pred)

best_rf_oob_pred = best_rf_model.oob_prediction_
best_rf_oob_mask = np.isfinite(best_rf_oob_pred)

if best_rf_oob_mask.sum() > 0:
    best_rf_oob_mse = mean_squared_error(
        np.asarray(y_tr, dtype=float)[best_rf_oob_mask],
        best_rf_oob_pred[best_rf_oob_mask]
    )
else:
    best_rf_oob_mse = np.nan

print("\nRefit check for CV-selected Random Forest:")
print("Feature space: base")
print("CV mean validation MSE:", best_rf_cv_overall["cv_valid_mse_mean"])
print("CV validation MSE std:", best_rf_cv_overall["cv_valid_mse_std"])
print("Holdout train MSE:", best_rf_train_mse)
print("Holdout OOB MSE:", best_rf_oob_mse)
print("Holdout validation MSE:", best_rf_val_mse)
print("max_features:", best_rf_max_features)
print("min_samples_leaf:", best_rf_min_samples_leaf)
print("min_samples_split:", best_rf_min_samples_split)

# ------------------------------------------------------------
# Feature importances
# ------------------------------------------------------------

rf_importances = (
    pd.Series(best_rf_model.feature_importances_, index=X_tr.columns)
    .groupby(level=0)
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 30 Random Forest feature importances:")
print(rf_importances.head(30).to_string())

# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Random Forest"] = {
    "model": best_rf_model,
    "feature_space": "base",
    "X_train_name": "X_tr",
    "X_val_name": "X_val",
    "n_estimators": RF_N_ESTIMATORS,
    "max_samples": RF_MAX_SAMPLES,
    "max_features": best_rf_max_features,
    "min_samples_leaf": best_rf_min_samples_leaf,
    "min_samples_split": best_rf_min_samples_split,
    "cv_valid_mse_mean": float(best_rf_cv_overall["cv_valid_mse_mean"]),
    "cv_valid_mse_std": float(best_rf_cv_overall["cv_valid_mse_std"]),
    "holdout_train_mse": float(best_rf_train_mse),
    "holdout_oob_mse": float(best_rf_oob_mse),
    "holdout_val_mse": float(best_rf_val_mse),
    "notes": (
        "First controlled Random Forest CV screen on base feature space only. "
        "Uses 3-fold CV, bootstrap sampling, and max_samples=0.80 for tractability."
    )
}

gc.collect()

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

Random Forest controlled CV screen
Using base feature space only for first RF screen.
X_tr shape: (115936, 162)
X_val shape: (28985, 162)

Reference Regression Tree:
CV validation MSE mean: None
Holdout validation MSE: None

RF screen settings:
CV folds: 3
n_estimators: 120
max_samples: 0.8
parameter combinations: 6

================ RF Fold 1 / 3 ================
combo= 1 | max_features=sqrt | leaf=  1 | train_mse=38.007839 | oob_mse=186.637481 | valid_mse=183.452533
combo= 2 | max_features=sqrt | leaf=  5 | train_mse=187.298683 | oob_mse=239.712041 | valid_mse=238.688685
combo= 3 | max_features=sqrt | leaf= 25 | train_mse=271.703122 | oob_mse=294.923165 | valid_mse=295.280890
combo= 4 | max_features= 0.5 | leaf=  1 | train_mse=31.132973 | oob_mse=152.413389 | valid_mse=149.439472
combo= 5 | max_features= 0.5 | leaf=  5 | train_mse=109.617475 | oob_mse=182.720757 | valid_mse=180.841065
combo= 6 | max_features= 0.5 | leaf= 25 | train_mse=199.974906 | oob_mse=230.424653 | valid_mse=229.

### Random Forest Preliminary Screen

A first controlled random forest screen was run using the base feature space, 3-fold cross-validation, 120 trees, and `max_samples=0.80`.

The best preliminary random forest used:

- `max_features = 0.5`
- `min_samples_leaf = 1`
- `min_samples_split = 2`

This model achieved a mean 3-fold CV validation MSE of approximately **149.55**, with a CV standard deviation of approximately **0.28**. When refit on the full development training split, it achieved a holdout validation MSE of approximately **127.48**.

This is a major improvement over the single regression tree baseline, whose CV validation MSE was approximately **250.68** and holdout validation MSE was approximately **241.35**. The result strongly suggests that variance reduction through averaging many trees is highly valuable for this problem.

Because the preliminary screen was intentionally small and used only 3-fold CV, the next step is a stronger 5-fold random forest refinement. The refinement will focus on the promising region around low leaf sizes and moderate-to-large `max_features`, while also testing whether the fold-safe polynomial/interactions feature space adds value.

In [43]:
# ============================================================
# 14B. Random Forest: stronger 5-fold CV refinement
# ============================================================

from pathlib import Path
from itertools import combinations
import time
import gc

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.metrics import mean_squared_error

start_time = time.perf_counter()

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Random Forest 5-fold CV refinement")
print("Base training split shape:", X_tr.shape)
print("Holdout validation shape:", X_val.shape)

# ------------------------------------------------------------
# Helper functions
# These are redefined here so this cell does not depend on
# whether the earlier tree-CV helper functions are still in memory.
# ------------------------------------------------------------

def select_top_by_abs_corr(X_train_df, y_train_array, top_n):
    """
    Select top predictors by absolute correlation with y.
    This is computed using only the training portion of a CV fold.
    """
    y_arr = np.asarray(y_train_array, dtype=float)
    y_centered = y_arr - y_arr.mean()
    y_norm = np.sqrt(np.sum(y_centered ** 2))
    
    scores = []
    
    for col in X_train_df.columns:
        x_arr = X_train_df[col].to_numpy(dtype=float)
        x_centered = x_arr - x_arr.mean()
        x_norm = np.sqrt(np.sum(x_centered ** 2))
        
        denom = x_norm * y_norm
        
        if denom == 0 or not np.isfinite(denom):
            score = 0.0
        else:
            score = abs(float(np.sum(x_centered * y_centered) / denom))
        
        scores.append((col, score))
    
    scores = sorted(scores, key=lambda z: z[1], reverse=True)
    return [col for col, score in scores[:top_n]]


def build_rf_feature_spaces(X_train_raw, X_eval_raw, y_train_array):
    """
    Build fold-safe RF feature spaces.

    Important: top squared/interacted features are selected using only
    the training portion of the current fold.
    """
    top10_features = select_top_by_abs_corr(X_train_raw, y_train_array, top_n=10)
    top5_features = select_top_by_abs_corr(X_train_raw, y_train_array, top_n=5)
    
    # Base
    X_train_base = X_train_raw.copy()
    X_eval_base = X_eval_raw.copy()
    
    # Base + squared terms + pairwise interactions
    X_train_combined = X_train_raw.copy()
    X_eval_combined = X_eval_raw.copy()
    
    for col in top10_features:
        new_col = f"{col}__squared"
        X_train_combined[new_col] = X_train_raw[col] ** 2
        X_eval_combined[new_col] = X_eval_raw[col] ** 2
    
    for f1, f2 in combinations(top5_features, 2):
        new_col = f"{f1}__x__{f2}"
        X_train_combined[new_col] = X_train_raw[f1] * X_train_raw[f2]
        X_eval_combined[new_col] = X_eval_raw[f1] * X_eval_raw[f2]
    
    feature_spaces_fold = {
        "base": (X_train_base, X_eval_base),
        "base_plus_poly_interactions": (X_train_combined, X_eval_combined)
    }
    
    return feature_spaces_fold, top10_features, top5_features


def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)


def parse_float_or_none(value):
    if str(value) == "None":
        return None
    return float(value)


# ------------------------------------------------------------
# Reference models
# ------------------------------------------------------------

print("\nReference models:")

if "best_models_by_class" in globals() and "Regression Tree" in best_models_by_class:
    tree_ref = best_models_by_class["Regression Tree"]
    print("Regression Tree CV MSE:", tree_ref.get("cv_valid_mse_mean"))
    print("Regression Tree holdout MSE:", tree_ref.get("holdout_val_mse"))

if "best_models_by_class" in globals() and "Random Forest" in best_models_by_class:
    rf_prelim_ref = best_models_by_class["Random Forest"]
    print("Preliminary RF CV MSE:", rf_prelim_ref.get("cv_valid_mse_mean"))
    print("Preliminary RF holdout MSE:", rf_prelim_ref.get("holdout_val_mse"))


# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

RF_REFINED_FOLD_RESULTS_PATH = RESULTS_DIR / "random_forest_cv_fold_results_refined.csv"
RF_REFINED_SUMMARY_PATH = RESULTS_DIR / "random_forest_cv_summary_refined.csv"
RF_REFINED_BEST_OVERALL_PATH = RESULTS_DIR / "random_forest_cv_best_overall_refined.csv"

OVERWRITE_RF_REFINED_RESULTS = True

if OVERWRITE_RF_REFINED_RESULTS:
    for path in [
        RF_REFINED_FOLD_RESULTS_PATH,
        RF_REFINED_SUMMARY_PATH,
        RF_REFINED_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()


# ------------------------------------------------------------
# CV setup and refined RF grid
# ------------------------------------------------------------

N_SPLITS_RF_REFINED = 5
RF_N_ESTIMATORS_REFINED = 300

kf = KFold(
    n_splits=N_SPLITS_RF_REFINED,
    shuffle=True,
    random_state=RANDOM_STATE
)

rf_refined_param_grid = list(ParameterGrid({
    "feature_space": ["base", "base_plus_poly_interactions"],
    "max_features": [0.40, 0.50, 0.70],
    "min_samples_leaf": [1, 2, 5],
    "max_samples": [0.80, None]
}))

print("\nRF refined settings:")
print("CV folds:", N_SPLITS_RF_REFINED)
print("n_estimators:", RF_N_ESTIMATORS_REFINED)
print("parameter combinations:", len(rf_refined_param_grid))
print("total fits:", N_SPLITS_RF_REFINED * len(rf_refined_param_grid))

y_tr_array = np.asarray(y_tr, dtype=float)

rf_refined_rows = []

# ------------------------------------------------------------
# Run 5-fold CV
# ------------------------------------------------------------

for fold_id, (cv_train_idx, cv_valid_idx) in enumerate(kf.split(X_tr), start=1):
    print(f"\n================ RF Refined Fold {fold_id} / {N_SPLITS_RF_REFINED} ================")
    
    X_cv_train_raw = X_tr.iloc[cv_train_idx]
    X_cv_valid_raw = X_tr.iloc[cv_valid_idx]
    
    y_cv_train = y_tr_array[cv_train_idx]
    y_cv_valid = y_tr_array[cv_valid_idx]
    
    fold_feature_spaces, fold_top10, fold_top5 = build_rf_feature_spaces(
        X_cv_train_raw,
        X_cv_valid_raw,
        y_cv_train
    )
    
    print("Top 10 fold features for squared terms:")
    print(fold_top10)
    print("Top 5 fold features for interactions:")
    print(fold_top5)
    
    for combo_id, params in enumerate(rf_refined_param_grid, start=1):
        feature_space_name = params["feature_space"]
        max_features = params["max_features"]
        min_samples_leaf = int(params["min_samples_leaf"])
        min_samples_split = max(2, 2 * min_samples_leaf)
        max_samples = params["max_samples"]
        
        X_cv_train, X_cv_valid = fold_feature_spaces[feature_space_name]
        
        model = RandomForestRegressor(
            n_estimators=RF_N_ESTIMATORS_REFINED,
            criterion="squared_error",
            max_depth=None,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            max_features=max_features,
            bootstrap=True,
            max_samples=max_samples,
            oob_score=True,
            n_jobs=-1,
            random_state=RANDOM_STATE + 1000 * fold_id + combo_id
        )
        
        model.fit(X_cv_train, y_cv_train)
        
        train_pred = model.predict(X_cv_train)
        valid_pred = model.predict(X_cv_valid)
        
        train_mse = mean_squared_error(y_cv_train, train_pred)
        valid_mse = mean_squared_error(y_cv_valid, valid_pred)
        
        oob_pred = model.oob_prediction_
        oob_mask = np.isfinite(oob_pred)
        
        if oob_mask.sum() > 0:
            oob_mse = mean_squared_error(y_cv_train[oob_mask], oob_pred[oob_mask])
        else:
            oob_mse = np.nan
        
        row = {
            "model_class": "Random Forest",
            "screen": "refined_5fold",
            "fold": fold_id,
            "feature_space": feature_space_name,
            "n_estimators": RF_N_ESTIMATORS_REFINED,
            "max_samples": "None" if max_samples is None else str(max_samples),
            "max_features": str(max_features),
            "min_samples_leaf": min_samples_leaf,
            "min_samples_split": min_samples_split,
            "fold_train_mse": train_mse,
            "fold_oob_mse": oob_mse,
            "fold_valid_mse": valid_mse
        }
        
        rf_refined_rows.append(row)
        append_checkpoint(pd.DataFrame([row]), RF_REFINED_FOLD_RESULTS_PATH)
        
        print(
            f"combo={combo_id:2d} | "
            f"space={feature_space_name:28s} | "
            f"max_samples={str(max_samples):>4s} | "
            f"max_features={str(max_features):>4s} | "
            f"leaf={min_samples_leaf:2d} | "
            f"train_mse={train_mse:.6f} | "
            f"oob_mse={oob_mse:.6f} | "
            f"valid_mse={valid_mse:.6f}"
        )
    
    del fold_feature_spaces
    gc.collect()


# ------------------------------------------------------------
# Summarize CV results
# ------------------------------------------------------------

rf_refined_fold_results = pd.DataFrame(rf_refined_rows)

group_cols = [
    "model_class",
    "screen",
    "feature_space",
    "n_estimators",
    "max_samples",
    "max_features",
    "min_samples_leaf",
    "min_samples_split"
]

rf_refined_summary = (
    rf_refined_fold_results
    .groupby(group_cols)
    .agg(
        cv_train_mse_mean=("fold_train_mse", "mean"),
        cv_train_mse_std=("fold_train_mse", "std"),
        cv_oob_mse_mean=("fold_oob_mse", "mean"),
        cv_oob_mse_std=("fold_oob_mse", "std"),
        cv_valid_mse_mean=("fold_valid_mse", "mean"),
        cv_valid_mse_std=("fold_valid_mse", "std")
    )
    .reset_index()
    .sort_values("cv_valid_mse_mean")
)

rf_refined_summary.to_csv(RF_REFINED_SUMMARY_PATH, index=False)

best_rf_refined = rf_refined_summary.iloc[0]
pd.DataFrame([best_rf_refined]).to_csv(RF_REFINED_BEST_OVERALL_PATH, index=False)

print("\nRandom Forest refined CV summary:")
print(rf_refined_summary.to_string(index=False))

print("\nBest refined Random Forest CV result:")
print(best_rf_refined)


# ------------------------------------------------------------
# Refit selected RF on full development training split
# and evaluate once on holdout validation split
# ------------------------------------------------------------

best_feature_space = best_rf_refined["feature_space"]
best_max_samples = parse_float_or_none(best_rf_refined["max_samples"])
best_max_features = float(best_rf_refined["max_features"])
best_min_samples_leaf = int(best_rf_refined["min_samples_leaf"])
best_min_samples_split = max(2, 2 * best_min_samples_leaf)

final_feature_spaces, final_top10, final_top5 = build_rf_feature_spaces(
    X_tr,
    X_val,
    y_tr
)

X_rf_train_final, X_rf_val_final = final_feature_spaces[best_feature_space]

best_rf_refined_model = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS_REFINED,
    criterion="squared_error",
    max_depth=None,
    min_samples_leaf=best_min_samples_leaf,
    min_samples_split=best_min_samples_split,
    max_features=best_max_features,
    bootstrap=True,
    max_samples=best_max_samples,
    oob_score=True,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

best_rf_refined_model.fit(X_rf_train_final, y_tr)

rf_train_pred = best_rf_refined_model.predict(X_rf_train_final)
rf_val_pred = best_rf_refined_model.predict(X_rf_val_final)

rf_train_mse = mean_squared_error(y_tr, rf_train_pred)
rf_val_mse = mean_squared_error(y_val, rf_val_pred)

rf_oob_pred = best_rf_refined_model.oob_prediction_
rf_oob_mask = np.isfinite(rf_oob_pred)

if rf_oob_mask.sum() > 0:
    rf_oob_mse = mean_squared_error(
        np.asarray(y_tr, dtype=float)[rf_oob_mask],
        rf_oob_pred[rf_oob_mask]
    )
else:
    rf_oob_mse = np.nan

print("\nRefit check for CV-selected refined Random Forest:")
print("Feature space:", best_feature_space)
print("CV mean validation MSE:", best_rf_refined["cv_valid_mse_mean"])
print("CV validation MSE std:", best_rf_refined["cv_valid_mse_std"])
print("Holdout train MSE:", rf_train_mse)
print("Holdout OOB MSE:", rf_oob_mse)
print("Holdout validation MSE:", rf_val_mse)
print("n_estimators:", RF_N_ESTIMATORS_REFINED)
print("max_samples:", best_max_samples)
print("max_features:", best_max_features)
print("min_samples_leaf:", best_min_samples_leaf)
print("min_samples_split:", best_min_samples_split)

print("\nFinal top 10 features used for squared terms:")
print(final_top10)

print("\nFinal top 5 features used for interactions:")
print(final_top5)

rf_refined_importances = (
    pd.Series(best_rf_refined_model.feature_importances_, index=X_rf_train_final.columns)
    .groupby(level=0)
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 30 refined Random Forest feature importances:")
print(rf_refined_importances.head(30).to_string())


# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

if "Random Forest" in best_models_by_class:
    best_models_by_class.setdefault(
        "Random Forest preliminary 3-fold",
        best_models_by_class["Random Forest"]
    )

best_models_by_class["Random Forest"] = {
    "model": best_rf_refined_model,
    "feature_space": best_feature_space,
    "X_train_final": X_rf_train_final,
    "X_val_final": X_rf_val_final,
    "n_estimators": RF_N_ESTIMATORS_REFINED,
    "max_samples": best_max_samples,
    "max_features": best_max_features,
    "min_samples_leaf": best_min_samples_leaf,
    "min_samples_split": best_min_samples_split,
    "cv_valid_mse_mean": float(best_rf_refined["cv_valid_mse_mean"]),
    "cv_valid_mse_std": float(best_rf_refined["cv_valid_mse_std"]),
    "holdout_train_mse": float(rf_train_mse),
    "holdout_oob_mse": float(rf_oob_mse),
    "holdout_val_mse": float(rf_val_mse),
    "notes": (
        "Refined 5-fold Random Forest screen over base and fold-safe "
        "base_plus_poly_interactions feature spaces."
    )
}

del final_feature_spaces
gc.collect()

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

Random Forest 5-fold CV refinement
Base training split shape: (115936, 162)
Holdout validation shape: (28985, 162)

Reference models:
Regression Tree CV MSE: 250.67682419719927
Regression Tree holdout MSE: 241.35313171586995
Preliminary RF CV MSE: 149.54957077027083
Preliminary RF holdout MSE: 127.47963454073466

RF refined settings:
CV folds: 5
n_estimators: 300
parameter combinations: 36
total fits: 180

================ RF Refined Fold 1 / 5 ================
Top 10 fold features for squared terms:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_WITH_DISABILITIES', 'ATTENDANCE_RATE', 'PERCENT_DROPOUT']
Top 5 fold features for interactions:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']
combo= 1 | space=base                         | max_samples= 0.8 | max_features= 0.4

### First Random Forest Submission Candidate

The refined 5-fold random forest screen selected the base feature space with 300 trees, `max_features = 0.5`, `max_samples = None`, `min_samples_leaf = 1`, and `min_samples_split = 2`.

This model achieved a mean 5-fold CV validation MSE of approximately **134.71** and a holdout validation MSE of approximately **123.21**. The holdout OOB MSE was approximately **123.64**, which is close to the holdout validation MSE and provides some reassurance that the model is not only exploiting the fixed holdout split.

Because this is the strongest model so far by a large margin, it is reasonable to train the same model specification on the full labeled training data and generate a first Kaggle submission.

In [46]:
# ============================================================
# 15A. First Kaggle submission: refined Random Forest on full training data
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Building first Random Forest submission")
print("Training feature matrix:", X_train_proc_model.shape)
print("Test feature matrix:", X_test_proc_model.shape)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert X_train_proc_model.shape[1] == X_test_proc_model.shape[1], "Train/test feature column count mismatch."
assert list(X_train_proc_model.columns) == list(X_test_proc_model.columns), "Train/test feature columns are not aligned."
assert len(test_ids) == len(X_test_proc_model), "test_ids length does not match test feature matrix."
assert test_ids.isna().sum() == 0, "Missing ASSESSMENT_ID values in test_ids."

# ------------------------------------------------------------
# Best refined RF specification from 5-fold CV
# ------------------------------------------------------------

final_rf_submission_model = RandomForestRegressor(
    n_estimators=300,
    criterion="squared_error",
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features=0.5,
    bootstrap=True,
    max_samples=None,
    oob_score=True,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

final_rf_submission_model.fit(X_train_proc_model, y_train)

# ------------------------------------------------------------
# Predict test set
# ------------------------------------------------------------

test_pred_raw = final_rf_submission_model.predict(X_test_proc_model)

print("\nRaw prediction summary:")
print(pd.Series(test_pred_raw).describe().to_string())

# Target is a percentage, so clip to the valid range.
test_pred_clipped = np.clip(test_pred_raw, 0, 100)

print("\nClipped prediction summary:")
print(pd.Series(test_pred_clipped).describe().to_string())

# ------------------------------------------------------------
# Build submission file
# ------------------------------------------------------------

submission_rf = pd.DataFrame({
    "ASSESSMENT_ID": test_ids.astype(str),
    "PERCENT_PROFICIENT": test_pred_clipped
})

assert submission_rf.shape[0] == X_test_proc_model.shape[0], "Submission row count mismatch."
assert list(submission_rf.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"], "Submission columns are wrong."
assert submission_rf["ASSESSMENT_ID"].isna().sum() == 0, "Missing ASSESSMENT_ID in submission."
assert submission_rf["PERCENT_PROFICIENT"].isna().sum() == 0, "Missing predictions in submission."

submission_path = Path("submission_rf_refined_300_base.csv")
submission_rf.to_csv(submission_path, index=False)

print("\nSubmission file written to:", submission_path.resolve())
print("Submission shape:", submission_rf.shape)
print("\nFirst 5 rows:")
print(submission_rf.head().to_string(index=False))

Building first Random Forest submission
Training feature matrix: (144921, 162)
Test feature matrix: (48307, 162)

Raw prediction summary:
count    48307.000000
mean        54.055481
std         23.207649
min          0.093333
25%         35.896667
50%         52.250000
75%         72.523333
max        100.000000

Clipped prediction summary:
count    48307.000000
mean        54.055481
std         23.207649
min          0.093333
25%         35.896667
50%         52.250000
75%         72.523333
max        100.000000

Submission file written to: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/submission_rf_refined_300_base.csv
Submission shape: (48307, 2)

First 5 rows:
ASSESSMENT_ID  PERCENT_PROFICIENT
 8af5e0382a81           59.140000
 e1591bf8db41           51.220000
 547ec44dcea6           33.786667
 0e200399fc40           67.336667
 c2c40438dac7           85.563333


### First Kaggle Submission File

A first submission file was generated using the refined random forest specification trained on the full labeled training data.

The submission file contains **48,307 rows** and the required two columns: `ASSESSMENT_ID` and `PERCENT_PROFICIENT`. Predicted values were already within the valid 0–100 range, so clipping did not alter the predictions. The prediction distribution appears plausible relative to the training target distribution, with a mean of approximately **54.06**, median of approximately **52.25**, and standard deviation of approximately **23.21**.

This file is ready for Kaggle submission as the first serious random forest benchmark.

### Random Forest Model-Class Continuation

The first full-data random forest submission achieved a public leaderboard MSE of **111.096**. This is worse than the external teammate LightGBM benchmark of **91.092**, but it is still a valid clean benchmark from the current notebook.

We will not switch directly to LightGBM yet. The project workflow is intentionally moving through model families in increasing complexity. Before leaving the random forest / bagging family, we want to extract two useful artifacts:

1. A clean out-of-fold random forest prediction vector for all labeled training rows.
2. A fold-averaged random forest test prediction file.

The out-of-fold predictions will be useful later for stacking, even if the random forest is not the final single best model.




# continue from here

In [47]:
# ============================================================
# 16A. Random Forest OOF predictions + fold-averaged test predictions
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

start_time = time.perf_counter()

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Random Forest OOF + fold-averaged test predictions")
print("Training feature matrix:", X_train_proc_model.shape)
print("Test feature matrix:", X_test_proc_model.shape)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert X_train_proc_model.shape[1] == X_test_proc_model.shape[1], "Train/test feature column count mismatch."
assert list(X_train_proc_model.columns) == list(X_test_proc_model.columns), "Train/test columns are not aligned."
assert len(y_train) == len(X_train_proc_model), "y_train length does not match training features."
assert len(train_ids) == len(X_train_proc_model), "train_ids length does not match training features."
assert len(test_ids) == len(X_test_proc_model), "test_ids length does not match test features."

# ------------------------------------------------------------
# RF settings from refined CV
# ------------------------------------------------------------

RF_OOF_N_SPLITS = 5
RF_OOF_N_ESTIMATORS = 500

rf_oof_params = {
    "n_estimators": RF_OOF_N_ESTIMATORS,
    "criterion": "squared_error",
    "max_depth": None,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "max_features": 0.5,
    "bootstrap": True,
    "max_samples": None,
    "oob_score": True,
    "n_jobs": -1
}

print("\nRF OOF settings:")
print(rf_oof_params)

kf = KFold(
    n_splits=RF_OOF_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

X_all = X_train_proc_model
X_test_all = X_test_proc_model
y_all = np.asarray(y_train, dtype=float)

oof_pred = np.zeros(len(X_all), dtype=float)
test_pred_folds = np.zeros((len(X_test_all), RF_OOF_N_SPLITS), dtype=float)

fold_rows = []
fold_models = []

# ------------------------------------------------------------
# Train fold models
# ------------------------------------------------------------

for fold_id, (tr_idx, val_idx) in enumerate(kf.split(X_all), start=1):
    print(f"\n================ RF OOF Fold {fold_id} / {RF_OOF_N_SPLITS} ================")
    
    X_fold_train = X_all.iloc[tr_idx]
    X_fold_valid = X_all.iloc[val_idx]
    y_fold_train = y_all[tr_idx]
    y_fold_valid = y_all[val_idx]
    
    model = RandomForestRegressor(
        **rf_oof_params,
        random_state=RANDOM_STATE + fold_id
    )
    
    model.fit(X_fold_train, y_fold_train)
    
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    test_pred = model.predict(X_test_all)
    
    oof_pred[val_idx] = valid_pred
    test_pred_folds[:, fold_id - 1] = test_pred
    
    train_mse = mean_squared_error(y_fold_train, train_pred)
    valid_mse = mean_squared_error(y_fold_valid, valid_pred)
    
    oob_pred = model.oob_prediction_
    oob_mask = np.isfinite(oob_pred)
    if oob_mask.sum() > 0:
        oob_mse = mean_squared_error(y_fold_train[oob_mask], oob_pred[oob_mask])
    else:
        oob_mse = np.nan
    
    row = {
        "model_class": "Random Forest",
        "artifact": "oof_fold_ensemble",
        "fold": fold_id,
        "n_estimators": RF_OOF_N_ESTIMATORS,
        "max_features": 0.5,
        "max_samples": "None",
        "min_samples_leaf": 1,
        "min_samples_split": 2,
        "fold_train_mse": train_mse,
        "fold_oob_mse": oob_mse,
        "fold_valid_mse": valid_mse
    }
    
    fold_rows.append(row)
    fold_models.append(model)
    
    print(f"fold_train_mse={train_mse:.6f}")
    print(f"fold_oob_mse={oob_mse:.6f}")
    print(f"fold_valid_mse={valid_mse:.6f}")
    
    gc.collect()

# ------------------------------------------------------------
# Summarize OOF performance
# ------------------------------------------------------------

rf_oof_fold_results = pd.DataFrame(fold_rows)
rf_oof_mse = mean_squared_error(y_all, oof_pred)

test_pred_mean_raw = test_pred_folds.mean(axis=1)
test_pred_mean = np.clip(test_pred_mean_raw, 0, 100)

print("\nRF OOF fold results:")
print(rf_oof_fold_results.to_string(index=False))

print("\nOverall RF OOF MSE:", rf_oof_mse)

print("\nRaw fold-averaged test prediction summary:")
print(pd.Series(test_pred_mean_raw).describe().to_string())

print("\nClipped fold-averaged test prediction summary:")
print(pd.Series(test_pred_mean).describe().to_string())

# ------------------------------------------------------------
# Save OOF predictions and fold-averaged submission
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

rf_oof_path = RESULTS_DIR / "oof_rf_500_base.csv"
rf_fold_test_path = RESULTS_DIR / "testpred_rf_500_base_folds.csv"
rf_fold_submission_path = Path("submission_rf_500_base_oof_foldavg.csv")

rf_oof_df = pd.DataFrame({
    "ASSESSMENT_ID": train_ids.astype(str),
    "y_true": y_all,
    "rf_500_base_oof_pred": oof_pred
})

rf_test_fold_df = pd.DataFrame({
    "ASSESSMENT_ID": test_ids.astype(str)
})

for fold_id in range(RF_OOF_N_SPLITS):
    rf_test_fold_df[f"rf_500_base_fold{fold_id + 1}_pred"] = test_pred_folds[:, fold_id]

rf_test_fold_df["rf_500_base_foldavg_pred"] = test_pred_mean

rf_submission_oof = pd.DataFrame({
    "ASSESSMENT_ID": test_ids.astype(str),
    "PERCENT_PROFICIENT": test_pred_mean
})

rf_oof_df.to_csv(rf_oof_path, index=False)
rf_test_fold_df.to_csv(rf_fold_test_path, index=False)
rf_submission_oof.to_csv(rf_fold_submission_path, index=False)

print("\nSaved RF OOF predictions to:", rf_oof_path.resolve())
print("Saved RF fold test predictions to:", rf_fold_test_path.resolve())
print("Saved RF fold-averaged submission to:", rf_fold_submission_path.resolve())

print("\nSubmission shape:", rf_submission_oof.shape)
print("\nFirst 5 submission rows:")
print(rf_submission_oof.head().to_string(index=False))

# ------------------------------------------------------------
# Store artifact in dictionary for later stacking
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Random Forest OOF 500 base"] = {
    "fold_models": fold_models,
    "oof_predictions": oof_pred,
    "test_fold_predictions": test_pred_folds,
    "test_fold_average": test_pred_mean,
    "oof_mse": float(rf_oof_mse),
    "fold_results": rf_oof_fold_results,
    "submission_path": str(rf_fold_submission_path),
    "notes": "OOF/fold-averaged RF artifact for later stacking."
}

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

Random Forest OOF + fold-averaged test predictions
Training feature matrix: (144921, 162)
Test feature matrix: (48307, 162)

RF OOF settings:
{'n_estimators': 500, 'criterion': 'squared_error', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'max_features': 0.5, 'bootstrap': True, 'max_samples': None, 'oob_score': True, 'n_jobs': -1}

================ RF OOF Fold 1 / 5 ================


fold_train_mse=16.708704
fold_oob_mse=122.960428
fold_valid_mse=122.881281

================ RF OOF Fold 2 / 5 ================
fold_train_mse=16.725163
fold_oob_mse=123.335287
fold_valid_mse=120.142245

================ RF OOF Fold 3 / 5 ================
fold_train_mse=16.707048
fold_oob_mse=122.797429
fold_valid_mse=123.546977

================ RF OOF Fold 4 / 5 ================
fold_train_mse=16.806578
fold_oob_mse=123.720044
fold_valid_mse=119.761836

================ RF OOF Fold 5 / 5 ================
fold_train_mse=16.672325
fold_oob_mse=122.655667
fold_valid_mse=125.307764

RF OOF fold results:
  model_class          artifact  fold  n_estimators  max_features max_samples  min_samples_leaf  min_samples_split  fold_train_mse  fold_oob_mse  fold_valid_mse
Random Forest oof_fold_ensemble     1           500           0.5        None                 1                  2       16.708704    122.960428      122.881281
Random Forest oof_fold_ensemble     2           500           0.5    

### Random Forest OOF and Fold-Averaged Submission

A 5-fold random forest OOF artifact was generated using the best refined RF settings with 500 trees, `max_features = 0.5`, `max_samples = None`, `min_samples_leaf = 1`, and `min_samples_split = 2`.

The overall random forest OOF MSE was approximately **122.33**. The fold validation MSEs were stable, ranging from approximately **119.76** to **125.31**. This is close to the earlier refined RF holdout MSE of approximately **123.21**, suggesting that the random forest generalization estimate is reasonably stable.

A fold-averaged test prediction file was also saved as `submission_rf_500_base_oof_foldavg.csv`. This file can be submitted as a second RF-family benchmark, but its larger value is that the saved OOF predictions can later be used as a stacking feature.

### Next Stage Plan After Random Forest OOF

After the random forest OOF artifact finishes, the next goal is to continue extracting value from the tree-ensemble / bagging family before moving to boosting.

The immediate next candidate is ExtraTrees. ExtraTrees is related to random forests but adds more randomization in split selection. This can sometimes reduce variance or produce errors that differ from random forests, which may be useful later in stacking even if it is not the best single model.

We will continue saving:
1. model-family validation metrics,
2. out-of-fold predictions,
3. fold-averaged test predictions,
4. Kaggle submission files,
5. a central model scoreboard.

The external teammate LightGBM score remains the current public benchmark, but we will not merge or copy that notebook into this workflow. When we reach the boosting stage, it can be inspected for ideas only.

In [31]:
# ============================================================
# 17A. Model scoreboard and submission tracker utilities
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# Submission tracker
# Manually update public scores after Kaggle submissions.
# ------------------------------------------------------------

submission_tracker = pd.DataFrame([
    {
        "submission_file": "submission_oof_lgbm.csv",
        "source": "external teammate reference",
        "model_family": "LightGBM",
        "public_mse": 91.092,
        "notes": "Current external team benchmark; do not commingle code."
    },
    {
        "submission_file": "submission_rf_refined_300_base.csv",
        "source": "current notebook",
        "model_family": "Random Forest",
        "public_mse": 111.096,
        "notes": "Full-data refit from refined RF CV-selected settings."
    },
    {
        "submission_file": "submission_rf_500_base_oof_foldavg.csv",
        "source": "current notebook",
        "model_family": "Random Forest",
        "public_mse": np.nan,
        "notes": "Pending RF OOF/fold-average submission candidate."
    }
])

submission_tracker_path = RESULTS_DIR / "submission_tracker.csv"
submission_tracker.to_csv(submission_tracker_path, index=False)

print("Submission tracker:")
print(submission_tracker.to_string(index=False))
print("\nSaved to:", submission_tracker_path.resolve())


# ------------------------------------------------------------
# Best-model scoreboard from best_models_by_class
# ------------------------------------------------------------

def build_model_scoreboard(best_models_by_class):
    rows = []
    
    for model_key, obj in best_models_by_class.items():
        if not isinstance(obj, dict):
            continue
        
        row = {
            "model_key": model_key,
            "feature_space": obj.get("feature_space", None),
            "cv_valid_mse_mean": obj.get("cv_valid_mse_mean", np.nan),
            "cv_valid_mse_std": obj.get("cv_valid_mse_std", np.nan),
            "holdout_val_mse": obj.get("holdout_val_mse", np.nan),
            "oof_mse": obj.get("oof_mse", np.nan),
            "holdout_oob_mse": obj.get("holdout_oob_mse", np.nan),
            "n_estimators": obj.get("n_estimators", np.nan),
            "max_features": obj.get("max_features", np.nan),
            "max_samples": obj.get("max_samples", np.nan),
            "min_samples_leaf": obj.get("min_samples_leaf", np.nan),
            "min_samples_split": obj.get("min_samples_split", np.nan),
            "submission_path": obj.get("submission_path", None),
            "notes": obj.get("notes", None)
        }
        
        rows.append(row)
    
    scoreboard = pd.DataFrame(rows)
    
    if len(scoreboard) > 0:
        sort_cols = []
        for col in ["oof_mse", "cv_valid_mse_mean", "holdout_val_mse"]:
            if col in scoreboard.columns:
                sort_cols.append(col)
        
        if sort_cols:
            scoreboard = scoreboard.sort_values(sort_cols, na_position="last")
    
    return scoreboard


if "best_models_by_class" in globals():
    model_scoreboard = build_model_scoreboard(best_models_by_class)
    model_scoreboard_path = RESULTS_DIR / "model_scoreboard.csv"
    model_scoreboard.to_csv(model_scoreboard_path, index=False)
    
    print("\nModel scoreboard:")
    print(model_scoreboard.to_string(index=False))
    print("\nSaved to:", model_scoreboard_path.resolve())
else:
    print("\nbest_models_by_class does not exist yet. Run this cell again after model cells finish.")

Submission tracker:
                       submission_file                      source  model_family  public_mse                                                   notes
               submission_oof_lgbm.csv external teammate reference      LightGBM      91.092 Current external team benchmark; do not commingle code.
    submission_rf_refined_300_base.csv            current notebook Random Forest     111.096   Full-data refit from refined RF CV-selected settings.
submission_rf_500_base_oof_foldavg.csv            current notebook Random Forest         NaN       Pending RF OOF/fold-average submission candidate.

Saved to: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/submission_tracker.csv

Model scoreboard:
      model_key  feature_space  cv_valid_mse_mean  cv_valid_mse_std  holdout_val_mse  oof_mse  holdout_oob_mse  n_estimators  max_features  max_samples  min_samples_leaf  min_samples_split submission_path        

In [32]:
# Patch metadata for RF OOF artifact in best_models_by_class

best_models_by_class["Random Forest OOF 500 base"].update({
    "feature_space": "base",
    "n_estimators": 500,
    "max_features": 0.5,
    "max_samples": None,
    "min_samples_leaf": 1,
    "min_samples_split": 2
})

KeyError: 'Random Forest OOF 500 base'

### Kernel Crash Recovery Note

The ExtraTrees OOF screen (deleted) crashed the Jupyter kernel, likely because the cell was too memory- and worker-intensive for the current environment. The crashed ExtraTrees output should not be used.

The notebook was recovered by rerunning only the preprocessing cells needed to rebuild `X_train_proc_model`, `X_test_proc_model`, `y_train`, `train_ids`, and `test_ids`. Expensive model-search cells were not rerun. Saved random forest artifacts were reloaded from disk and will continue to be used for tracking and future stacking.

# rerun these at least since kernel crash

0.. Setup and raw data loading


1.. Merge datasets


4.. Build X and y + preserve IDs


5.. Column typing


6A. Frequency encoding (safe, no leakage)


6B. Missing indicators + median imputation (more robust than mean)


6C. Drop raw high-cardinality categorical columns


6D. One-hot encode low-cardinality categorical variables


Create modeling copy WITHOUT ID (non-destructive)

7A. Simple Linear Regression (one feature at a time)


Extract best simple linear regression feature properly


8A. Build controlled candidate feature spaces


12A. Step Function Models


12B. Expanded Step Function Search

In [51]:
print(feature_spaces.keys())
print(X_tr.shape, X_val.shape)
print(make_step_features)
print(get_top_continuous_features)

dict_keys(['base', 'base_plus_poly', 'base_plus_interactions', 'base_plus_poly_interactions'])
(115936, 162) (28985, 162)
<function make_step_features at 0x334eff690>
<function get_top_continuous_features at 0x334eff3d0>


### 18B. Safe ExtraTrees screen and selected OOF artifact

The previous ExtraTrees OOF screen crashed the kernel, so this cell uses a safer two-stage design.

First, it runs a small ExtraTrees holdout screen on the base feature space only. Then it selects the best holdout configuration and, only if the validation MSE is below the safety cutoff, runs one 5-fold OOF/fold-averaged test-prediction artifact.

Design choices:
- Use only the base feature space because RF already preferred base over engineered polynomial/interaction features.
- Use fewer trees than the failed ExtraTrees run.
- Use `n_jobs=1` to reduce worker/memory pressure.
- Do not store all candidate test predictions.
- Save checkpoints after each screen configuration and after each OOF fold.
- Delete fitted model objects after each fit.

In [52]:
# ============================================================
# 18B. Safe ExtraTrees holdout screen + selected OOF artifact
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

start_time = time.perf_counter()

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Safe ExtraTrees screen + selected OOF artifact")
print("Development split:", X_tr.shape, X_val.shape)
print("Full training matrix:", X_train_proc_model.shape)
print("Test matrix:", X_test_proc_model.shape)

# ------------------------------------------------------------
# Runtime controls
# ------------------------------------------------------------
# This is intentionally conservative after the previous kernel crash.
# For a shorter run, set RUN_ET_OOF_AFTER_SCREEN = False.
# To force OOF even after a weak screen, increase ET_OOF_MSE_CUTOFF.

ET_N_JOBS = 1
ET_SCREEN_N_ESTIMATORS = 180
ET_OOF_N_ESTIMATORS = 250
ET_OOF_N_SPLITS = 5
ET_OOF_MSE_CUTOFF = 145.0
RUN_ET_OOF_AFTER_SCREEN = True

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ET_SCREEN_RESULTS_PATH = RESULTS_DIR / "extratrees_safe_holdout_screen.csv"
ET_SCREEN_BEST_PATH = RESULTS_DIR / "extratrees_safe_holdout_best.csv"
ET_OOF_FOLD_RESULTS_PATH = RESULTS_DIR / "extratrees_safe_oof_fold_results.csv"
ET_OOF_SUMMARY_PATH = RESULTS_DIR / "extratrees_safe_oof_summary.csv"
ET_OOF_PRED_PATH = RESULTS_DIR / "oof_extratrees_safe_base.csv"
ET_TESTPRED_PATH = RESULTS_DIR / "testpred_extratrees_safe_base_foldavg.csv"
ET_SUBMISSION_PATH = Path("submission_extratrees_safe_base_oof_foldavg.csv")

OVERWRITE_ET_RESULTS = True

if OVERWRITE_ET_RESULTS:
    for path in [
        ET_SCREEN_RESULTS_PATH,
        ET_SCREEN_BEST_PATH,
        ET_OOF_FOLD_RESULTS_PATH,
        ET_OOF_SUMMARY_PATH,
        ET_OOF_PRED_PATH,
        ET_TESTPRED_PATH,
        RESULTS_DIR / "extratrees_safe_oof_pred_partial.npy",
        RESULTS_DIR / "extratrees_safe_test_pred_sum_partial.npy",
    ]:
        if path.exists():
            path.unlink()
    if ET_SUBMISSION_PATH.exists():
        ET_SUBMISSION_PATH.unlink()

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert "base" in feature_spaces, "feature_spaces['base'] is missing."
assert list(feature_spaces["base"][0].columns) == list(X_tr.columns), "Base feature-space columns do not match X_tr."
assert list(feature_spaces["base"][1].columns) == list(X_val.columns), "Base feature-space columns do not match X_val."
assert X_train_proc_model.shape[1] == X_test_proc_model.shape[1], "Train/test feature counts do not match."
assert list(X_train_proc_model.columns) == list(X_test_proc_model.columns), "Train/test columns are not aligned."
assert len(y_train) == len(X_train_proc_model), "y_train length does not match X_train_proc_model."
assert len(test_ids) == len(X_test_proc_model), "test_ids length does not match X_test_proc_model."

# ------------------------------------------------------------
# Small helpers
# ------------------------------------------------------------

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)


def make_extratrees_model(config, random_state):
    params = {k: v for k, v in config.items() if k != "config_name"}
    return ExtraTreesRegressor(
        criterion="squared_error",
        max_depth=None,
        min_samples_split=2,
        n_jobs=ET_N_JOBS,
        random_state=random_state,
        **params
    )


def one_dim_id_array(ids):
    if isinstance(ids, pd.DataFrame):
        ids = ids.iloc[:, 0]
    return pd.Series(ids).astype(str).to_numpy()

# ------------------------------------------------------------
# Holdout screen on the already-restored development split
# ------------------------------------------------------------
# Keep this deliberately small. We only use the base feature space because
# random forest already preferred base, and ExtraTrees should also learn
# threshold structure without hand-built polynomial/interactions.

et_screen_configs = [
    {
        "config_name": "et_base_180_mf0.5_leaf1_no_bootstrap",
        "n_estimators": ET_SCREEN_N_ESTIMATORS,
        "max_features": 0.50,
        "min_samples_leaf": 1,
        "bootstrap": False,
    },
    {
        "config_name": "et_base_180_mf0.7_leaf1_no_bootstrap",
        "n_estimators": ET_SCREEN_N_ESTIMATORS,
        "max_features": 0.70,
        "min_samples_leaf": 1,
        "bootstrap": False,
    },
    {
        "config_name": "et_base_180_mf0.5_leaf2_no_bootstrap",
        "n_estimators": ET_SCREEN_N_ESTIMATORS,
        "max_features": 0.50,
        "min_samples_leaf": 2,
        "bootstrap": False,
    },
    {
        "config_name": "et_base_180_mf0.5_leaf1_bootstrap0.8",
        "n_estimators": ET_SCREEN_N_ESTIMATORS,
        "max_features": 0.50,
        "min_samples_leaf": 1,
        "bootstrap": True,
        "max_samples": 0.80,
    },
]

print("\nExtraTrees holdout screen settings:")
print("n_jobs:", ET_N_JOBS)
print("screen n_estimators:", ET_SCREEN_N_ESTIMATORS)
print("configs:", len(et_screen_configs))

X_screen_tr = X_tr
X_screen_val = X_val
y_screen_tr = np.asarray(y_tr, dtype=float).ravel()
y_screen_val = np.asarray(y_val, dtype=float).ravel()

et_screen_rows = []

for config_id, config in enumerate(et_screen_configs, start=1):
    config_start = time.perf_counter()
    print("\n" + "=" * 80)
    print(f"ExtraTrees holdout config {config_id} / {len(et_screen_configs)}")
    print(config)
    print("=" * 80)

    model = make_extratrees_model(
        config,
        random_state=RANDOM_STATE + 3000 + config_id
    )
    model.fit(X_screen_tr, y_screen_tr)

    train_pred = model.predict(X_screen_tr)
    val_pred = model.predict(X_screen_val)

    train_mse = mean_squared_error(y_screen_tr, train_pred)
    val_mse = mean_squared_error(y_screen_val, val_pred)
    elapsed = time.perf_counter() - config_start

    row = {
        "model_class": "ExtraTrees",
        "stage": "holdout_screen",
        "feature_space": "base",
        "config_name": config["config_name"],
        "n_estimators": config["n_estimators"],
        "max_features": config["max_features"],
        "min_samples_leaf": config["min_samples_leaf"],
        "min_samples_split": 2,
        "bootstrap": config["bootstrap"],
        "max_samples": config.get("max_samples", np.nan),
        "train_mse": train_mse,
        "holdout_val_mse": val_mse,
        "elapsed_sec": elapsed,
    }

    et_screen_rows.append(row)
    append_checkpoint(pd.DataFrame([row]), ET_SCREEN_RESULTS_PATH)

    print(f"train_mse={train_mse:.6f}")
    print(f"holdout_val_mse={val_mse:.6f}")
    print(f"elapsed_sec={elapsed:.2f}")

    del model, train_pred, val_pred
    gc.collect()

et_screen_results = pd.DataFrame(et_screen_rows).sort_values("holdout_val_mse")
et_screen_results.to_csv(ET_SCREEN_RESULTS_PATH, index=False)

best_et_screen = et_screen_results.iloc[0].to_dict()
pd.DataFrame([best_et_screen]).to_csv(ET_SCREEN_BEST_PATH, index=False)

print("\n" + "=" * 80)
print("ExtraTrees holdout screen complete")
print("=" * 80)
print(et_screen_results.to_string(index=False))
print("\nBest holdout config:")
print(pd.DataFrame([best_et_screen]).to_string(index=False))
print("\nSaved screen results to:", ET_SCREEN_RESULTS_PATH.resolve())
print("Saved best screen result to:", ET_SCREEN_BEST_PATH.resolve())

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["ExtraTrees holdout screen safe base"] = {
    "model_class": "ExtraTrees",
    "feature_space": "base",
    "holdout_val_mse": float(best_et_screen["holdout_val_mse"]),
    "train_mse": float(best_et_screen["train_mse"]),
    "n_estimators": int(best_et_screen["n_estimators"]),
    "max_features": best_et_screen["max_features"],
    "max_samples": best_et_screen["max_samples"],
    "min_samples_leaf": int(best_et_screen["min_samples_leaf"]),
    "min_samples_split": 2,
    "notes": "Safe ExtraTrees holdout screen after previous crash."
}

# ------------------------------------------------------------
# OOF artifact for the selected screen config only
# ------------------------------------------------------------

should_run_oof = (
    RUN_ET_OOF_AFTER_SCREEN
    and np.isfinite(best_et_screen["holdout_val_mse"])
    and best_et_screen["holdout_val_mse"] <= ET_OOF_MSE_CUTOFF
)

print("\nOOF gate:")
print("RUN_ET_OOF_AFTER_SCREEN:", RUN_ET_OOF_AFTER_SCREEN)
print("ET_OOF_MSE_CUTOFF:", ET_OOF_MSE_CUTOFF)
print("Best holdout MSE:", best_et_screen["holdout_val_mse"])
print("Will run OOF:", should_run_oof)

if should_run_oof:
    best_config_name = best_et_screen["config_name"]
    selected_config = [c for c in et_screen_configs if c["config_name"] == best_config_name][0].copy()
    selected_config["config_name"] = "et_safe_oof_from_" + best_config_name
    selected_config["n_estimators"] = ET_OOF_N_ESTIMATORS

    print("\nSelected OOF config:")
    print(selected_config)
    print("OOF folds:", ET_OOF_N_SPLITS)
    print("OOF n_estimators:", ET_OOF_N_ESTIMATORS)

    X_all = X_train_proc_model
    X_test_all = X_test_proc_model
    y_all = np.asarray(y_train, dtype=float).ravel()

    oof_pred = np.full(len(X_all), np.nan, dtype=float)
    test_pred_sum = np.zeros(len(X_test_all), dtype=float)

    fold_rows = []

    kf = KFold(
        n_splits=ET_OOF_N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    for fold_id, (tr_idx, val_idx) in enumerate(kf.split(X_all), start=1):
        fold_start = time.perf_counter()
        print("\n" + "=" * 80)
        print(f"ExtraTrees OOF fold {fold_id} / {ET_OOF_N_SPLITS}")
        print("=" * 80)

        model = make_extratrees_model(
            selected_config,
            random_state=RANDOM_STATE + 4000 + fold_id
        )

        model.fit(X_all.iloc[tr_idx], y_all[tr_idx])

        valid_pred = model.predict(X_all.iloc[val_idx])
        test_pred = model.predict(X_test_all)

        oof_pred[val_idx] = valid_pred
        test_pred_sum += test_pred / ET_OOF_N_SPLITS

        valid_mse = mean_squared_error(y_all[val_idx], valid_pred)
        fold_elapsed = time.perf_counter() - fold_start

        row = {
            "model_class": "ExtraTrees",
            "stage": "oof_selected_config",
            "feature_space": "base",
            "fold": fold_id,
            "config_name": selected_config["config_name"],
            "n_estimators": selected_config["n_estimators"],
            "max_features": selected_config["max_features"],
            "min_samples_leaf": selected_config["min_samples_leaf"],
            "min_samples_split": 2,
            "bootstrap": selected_config["bootstrap"],
            "max_samples": selected_config.get("max_samples", np.nan),
            "fold_valid_mse": valid_mse,
            "elapsed_sec": fold_elapsed,
        }

        fold_rows.append(row)
        append_checkpoint(pd.DataFrame([row]), ET_OOF_FOLD_RESULTS_PATH)

        # Light checkpointing after each fold. These are small arrays, not model objects.
        np.save(RESULTS_DIR / "extratrees_safe_oof_pred_partial.npy", oof_pred)
        np.save(RESULTS_DIR / "extratrees_safe_test_pred_sum_partial.npy", test_pred_sum)

        print(f"fold_valid_mse={valid_mse:.6f}")
        print(f"elapsed_sec={fold_elapsed:.2f}")
        print("Partial OOF predictions filled:", np.isfinite(oof_pred).sum(), "/", len(oof_pred))

        del model, valid_pred, test_pred
        gc.collect()

    assert np.isfinite(oof_pred).all(), "OOF predictions contain missing values."

    et_oof_fold_results = pd.DataFrame(fold_rows)
    et_oof_fold_results.to_csv(ET_OOF_FOLD_RESULTS_PATH, index=False)

    et_oof_mse = mean_squared_error(y_all, oof_pred)
    test_pred_avg = np.clip(test_pred_sum, 0, 100)
    oof_pred_clipped = np.clip(oof_pred, 0, 100)

    train_id_values = one_dim_id_array(train_ids)
    test_id_values = one_dim_id_array(test_ids)

    oof_df = pd.DataFrame({
        "ASSESSMENT_ID": train_id_values,
        "y_true": y_all,
        "oof_pred": oof_pred,
        "oof_pred_clipped": oof_pred_clipped,
    })
    oof_df.to_csv(ET_OOF_PRED_PATH, index=False)

    test_pred_df = pd.DataFrame({
        "ASSESSMENT_ID": test_id_values,
        "PERCENT_PROFICIENT": test_pred_avg,
    })
    test_pred_df.to_csv(ET_TESTPRED_PATH, index=False)

    submission_et = test_pred_df.copy()
    assert submission_et.shape[0] == X_test_proc_model.shape[0], "Submission row count mismatch."
    assert list(submission_et.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"], "Submission columns are wrong."
    assert submission_et["ASSESSMENT_ID"].isna().sum() == 0, "Missing ASSESSMENT_ID in submission."
    assert submission_et["PERCENT_PROFICIENT"].isna().sum() == 0, "Missing predictions in submission."
    submission_et.to_csv(ET_SUBMISSION_PATH, index=False)

    total_elapsed = time.perf_counter() - start_time

    et_oof_summary = pd.DataFrame([{
        "model_key": "ExtraTrees OOF safe base",
        "model_class": "ExtraTrees",
        "feature_space": "base",
        "screen_holdout_val_mse": float(best_et_screen["holdout_val_mse"]),
        "oof_mse": float(et_oof_mse),
        "fold_valid_mse_mean": float(et_oof_fold_results["fold_valid_mse"].mean()),
        "fold_valid_mse_std": float(et_oof_fold_results["fold_valid_mse"].std()),
        "n_splits": ET_OOF_N_SPLITS,
        "n_estimators": ET_OOF_N_ESTIMATORS,
        "max_features": selected_config["max_features"],
        "min_samples_leaf": selected_config["min_samples_leaf"],
        "min_samples_split": 2,
        "bootstrap": selected_config["bootstrap"],
        "max_samples": selected_config.get("max_samples", np.nan),
        "n_jobs": ET_N_JOBS,
        "submission_path": str(ET_SUBMISSION_PATH),
        "elapsed_sec": total_elapsed,
    }])
    et_oof_summary.to_csv(ET_OOF_SUMMARY_PATH, index=False)

    best_models_by_class["ExtraTrees OOF safe base"] = {
        "model_class": "ExtraTrees",
        "feature_space": "base",
        "holdout_val_mse": float(best_et_screen["holdout_val_mse"]),
        "oof_mse": float(et_oof_mse),
        "cv_valid_mse_mean": float(et_oof_fold_results["fold_valid_mse"].mean()),
        "cv_valid_mse_std": float(et_oof_fold_results["fold_valid_mse"].std()),
        "n_estimators": ET_OOF_N_ESTIMATORS,
        "max_features": selected_config["max_features"],
        "max_samples": selected_config.get("max_samples", np.nan),
        "min_samples_leaf": int(selected_config["min_samples_leaf"]),
        "min_samples_split": 2,
        "submission_path": str(ET_SUBMISSION_PATH),
        "notes": "Selected safe ExtraTrees OOF artifact; test predictions are fold-averaged."
    }

    new_submission_row = {
        "submission_file": ET_SUBMISSION_PATH.name,
        "source": "current notebook",
        "model_family": "ExtraTrees",
        "public_mse": np.nan,
        "notes": "Safe ExtraTrees OOF/fold-average submission candidate."
    }

    if "submission_tracker" in globals():
        submission_tracker = pd.concat(
            [submission_tracker, pd.DataFrame([new_submission_row])],
            ignore_index=True
        )
        submission_tracker = submission_tracker.drop_duplicates(
            subset=["submission_file"],
            keep="last"
        )
    else:
        submission_tracker = pd.DataFrame([new_submission_row])

    submission_tracker.to_csv(RESULTS_DIR / "submission_tracker.csv", index=False)

    print("\n" + "=" * 80)
    print("ExtraTrees OOF artifact complete")
    print("=" * 80)
    print("Fold results:")
    print(et_oof_fold_results.to_string(index=False))
    print("\nOOF MSE:", et_oof_mse)
    print("\nTest prediction summary:")
    print(pd.Series(test_pred_avg).describe().to_string())
    print("\nSaved files:")
    print("OOF:", ET_OOF_PRED_PATH.resolve())
    print("Fold-avg test predictions:", ET_TESTPRED_PATH.resolve())
    print("Submission:", ET_SUBMISSION_PATH.resolve())
    print("OOF summary:", ET_OOF_SUMMARY_PATH.resolve())
    print("\nSubmission shape:", submission_et.shape)
    print("First 5 submission rows:")
    print(submission_et.head().to_string(index=False))
    print("\nTotal elapsed seconds:", total_elapsed)

else:
    total_elapsed = time.perf_counter() - start_time
    print("\nOOF was skipped by the safety gate.")
    print("The holdout screen is still saved and can be reviewed before deciding whether to run OOF.")
    print("Total elapsed seconds:", total_elapsed)

Safe ExtraTrees screen + selected OOF artifact
Development split: (115936, 162) (28985, 162)
Full training matrix: (144921, 162)
Test matrix: (48307, 162)

ExtraTrees holdout screen settings:
n_jobs: 1
screen n_estimators: 180
configs: 4

ExtraTrees holdout config 1 / 4
{'config_name': 'et_base_180_mf0.5_leaf1_no_bootstrap', 'n_estimators': 180, 'max_features': 0.5, 'min_samples_leaf': 1, 'bootstrap': False}
train_mse=0.025122
holdout_val_mse=111.627310
elapsed_sec=131.23

ExtraTrees holdout config 2 / 4
{'config_name': 'et_base_180_mf0.7_leaf1_no_bootstrap', 'n_estimators': 180, 'max_features': 0.7, 'min_samples_leaf': 1, 'bootstrap': False}
train_mse=0.025122
holdout_val_mse=111.874048
elapsed_sec=163.68

ExtraTrees holdout config 3 / 4
{'config_name': 'et_base_180_mf0.5_leaf2_no_bootstrap', 'n_estimators': 180, 'max_features': 0.5, 'min_samples_leaf': 2, 'bootstrap': False}
train_mse=19.159447
holdout_val_mse=119.026918
elapsed_sec=107.84

ExtraTrees holdout config 4 / 4
{'config_na

### ExtraTrees Kaggle submission result

The safe ExtraTrees model produced the best clean non-boosting result so far.

| Submission file | Model | Feature space | OOF MSE | Public leaderboard MSE |
|---|---|---|---:|---:|
| `submission_extratrees_safe_base_oof_foldavg.csv` | ExtraTreesRegressor | `base` | 110.7493 | 102.56 |

The selected ExtraTrees configuration used:
- `n_estimators = 250` for the OOF/fold-averaged artifact
- `max_features = 0.5`
- `min_samples_leaf = 1`
- `bootstrap = False`
- feature space: `base`

This improves substantially over the earlier clean Random Forest public benchmark of 111.096. The OOF MSE and public leaderboard MSE are not directly comparable because they are evaluated on different data, but both indicate that ExtraTrees is now the strongest clean bagging-family model in the notebook.

Next, we will test whether a lightweight OOF-weighted blend of Random Forest and ExtraTrees improves over pure ExtraTrees.

### 19A. OOF blend of Random Forest and ExtraTrees

ExtraTrees is now the strongest clean non-boosting model by OOF MSE, but the earlier Random Forest may still contain complementary signal. This cell below performs a lightweight OOF blend between the saved RF 500-base artifact and the new safe ExtraTrees artifact.

No new tree models are trained here. The cell only:
- loads saved OOF predictions,
- searches blend weights using OOF MSE,
- blends the matching test predictions,
- saves a new blend submission if the OOF blend is useful.

In [ ]:
# ============================================================
# 19A. OOF blend: RF 500 base + safe ExtraTrees base
# Full corrected replacement cell
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

RESULTS_DIR = Path("model_results")
assert RESULTS_DIR.exists(), "model_results folder was not found."

# ------------------------------------------------------------
# Locate saved artifacts
# ------------------------------------------------------------

def first_existing_path(candidates, label):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {label}. Tried:\n" +
        "\n".join(str(p) for p in candidates)
    )

rf_oof_path = first_existing_path(
    [
        RESULTS_DIR / "oof_rf_500_base.csv",
    ],
    "RF OOF file"
)

rf_test_path = first_existing_path(
    [
        RESULTS_DIR / "testpred_rf_500_base_folds.csv",
        RESULTS_DIR / "testpred_rf_500_base_foldavg.csv",
        RESULTS_DIR / "testpred_rf_500_base.csv",
    ],
    "RF test-prediction file"
)

et_oof_path = first_existing_path(
    [
        RESULTS_DIR / "oof_extratrees_safe_base.csv",
    ],
    "ExtraTrees OOF file"
)

et_test_path = first_existing_path(
    [
        RESULTS_DIR / "testpred_extratrees_safe_base_foldavg.csv",
    ],
    "ExtraTrees test-prediction file"
)

print("Using files:")
print("RF OOF:       ", rf_oof_path)
print("RF testpred:  ", rf_test_path)
print("ET OOF:       ", et_oof_path)
print("ET testpred:  ", et_test_path)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def read_pred_csv(path):
    return pd.read_csv(path, dtype={"ASSESSMENT_ID": str})


def one_dim_id_array(ids):
    if isinstance(ids, pd.DataFrame):
        ids = ids.iloc[:, 0]
    return pd.Series(ids).astype(str).to_numpy()


def pick_prediction_column(df, preferred_cols, label):
    """
    Pick a prediction column robustly.
    Handles custom names such as rf_500_base_oof_pred.
    """
    for col in preferred_cols:
        if col in df.columns:
            return col

    excluded_cols = {"ASSESSMENT_ID", "y_true"}

    numeric_cols = [
        c for c in df.columns
        if c not in excluded_cols and pd.api.types.is_numeric_dtype(df[c])
    ]

    pred_like_cols = [
        c for c in numeric_cols
        if (
            "pred" in c.lower()
            or "proficient" in c.lower()
            or "oof" in c.lower()
            or "fold" in c.lower()
        )
    ]

    if len(pred_like_cols) == 1:
        return pred_like_cols[0]

    priority_terms = [
        "percent_proficient",
        "foldavg",
        "fold_avg",
        "average",
        "avg",
        "test_pred",
        "oof_pred",
        "oof",
        "pred",
    ]

    for term in priority_terms:
        matches = [c for c in pred_like_cols if term in c.lower()]
        if len(matches) == 1:
            return matches[0]

    if len(numeric_cols) == 1:
        return numeric_cols[0]

    raise ValueError(
        f"Could not identify prediction column for {label}.\n"
        f"Columns available: {list(df.columns)}\n"
        f"Numeric columns found: {numeric_cols}\n"
        f"Prediction-like columns found: {pred_like_cols}"
    )


def standardize_oof(df, model_name):
    assert "ASSESSMENT_ID" in df.columns, f"{model_name} OOF missing ASSESSMENT_ID."
    assert "y_true" in df.columns, f"{model_name} OOF missing y_true."

    pred_col = pick_prediction_column(
        df,
        preferred_cols=[
            "oof_pred",
            "rf_500_base_oof_pred",
            "et_oof_pred",
            "oof_pred_clipped",
            "pred",
            "prediction",
        ],
        label=f"{model_name} OOF"
    )

    print(f"{model_name.upper()} OOF prediction column used:", pred_col)

    out = df[["ASSESSMENT_ID", "y_true", pred_col]].copy()
    out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)
    out = out.rename(columns={pred_col: f"{model_name}_oof_pred"})
    return out


def standardize_testpred(df, model_name):
    assert "ASSESSMENT_ID" in df.columns, f"{model_name} test predictions missing ASSESSMENT_ID."

    # First try to pick a clear aggregate prediction column.
    try:
        pred_col = pick_prediction_column(
            df,
            preferred_cols=[
                "PERCENT_PROFICIENT",
                "test_pred",
                "test_pred_avg",
                "foldavg_pred",
                "fold_avg_pred",
                "rf_500_base_test_pred",
                "rf_500_base_test_pred_foldavg",
                "rf_500_base_foldavg_pred",
                "et_test_pred",
                "pred",
                "prediction",
            ],
            label=f"{model_name} test predictions"
        )

        print(f"{model_name.upper()} test prediction column used:", pred_col)

        out = df[["ASSESSMENT_ID", pred_col]].copy()
        out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)
        out = out.rename(columns={pred_col: f"{model_name}_test_pred"})
        return out

    except ValueError:
        # If the RF test file contains only fold columns and no aggregate,
        # average fold prediction columns automatically.
        excluded_cols = {"ASSESSMENT_ID", "y_true"}
        numeric_cols = [
            c for c in df.columns
            if c not in excluded_cols and pd.api.types.is_numeric_dtype(df[c])
        ]

        fold_cols = [
            c for c in numeric_cols
            if "fold" in c.lower()
        ]

        if len(fold_cols) >= 2:
            print(f"{model_name.upper()} test prediction columns averaged:", fold_cols)

            out = df[["ASSESSMENT_ID"]].copy()
            out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)
            out[f"{model_name}_test_pred"] = df[fold_cols].mean(axis=1)
            return out

        raise

# ------------------------------------------------------------
# Load and align OOF predictions
# ------------------------------------------------------------

rf_oof_raw = read_pred_csv(rf_oof_path)
et_oof_raw = read_pred_csv(et_oof_path)

print("\nRF OOF columns:")
print(list(rf_oof_raw.columns))

print("\nExtraTrees OOF columns:")
print(list(et_oof_raw.columns))

rf_oof = standardize_oof(rf_oof_raw, "rf")
et_oof = standardize_oof(et_oof_raw, "et")

assert rf_oof["ASSESSMENT_ID"].duplicated().sum() == 0, "Duplicate ASSESSMENT_ID in RF OOF."
assert et_oof["ASSESSMENT_ID"].duplicated().sum() == 0, "Duplicate ASSESSMENT_ID in ExtraTrees OOF."

oof_blend_df = rf_oof.merge(
    et_oof,
    on="ASSESSMENT_ID",
    how="inner",
    suffixes=("_rf", "_et")
)

assert len(oof_blend_df) == len(rf_oof) == len(et_oof), "OOF merge lost rows."

assert np.allclose(
    oof_blend_df["y_true_rf"].to_numpy(dtype=float),
    oof_blend_df["y_true_et"].to_numpy(dtype=float)
), "RF and ExtraTrees y_true values do not match after merge."

oof_blend_df["y_true"] = oof_blend_df["y_true_rf"].astype(float)
oof_blend_df = oof_blend_df.drop(columns=["y_true_rf", "y_true_et"])

y_oof = oof_blend_df["y_true"].to_numpy(dtype=float)
rf_oof_pred = oof_blend_df["rf_oof_pred"].to_numpy(dtype=float)
et_oof_pred = oof_blend_df["et_oof_pred"].to_numpy(dtype=float)

rf_oof_mse = mean_squared_error(y_oof, np.clip(rf_oof_pred, 0, 100))
et_oof_mse = mean_squared_error(y_oof, np.clip(et_oof_pred, 0, 100))

print("\nComponent OOF MSE:")
print(f"RF 500 base:        {rf_oof_mse:.6f}")
print(f"ExtraTrees safe:    {et_oof_mse:.6f}")

# ------------------------------------------------------------
# OOF weight search
# ------------------------------------------------------------
# w_et = 1 means pure ExtraTrees.
# w_et = 0 means pure Random Forest.

weight_rows = []

for w_et in np.linspace(0, 1, 1001):
    w_rf = 1.0 - w_et

    blend_pred_raw = w_rf * rf_oof_pred + w_et * et_oof_pred
    blend_pred_clipped = np.clip(blend_pred_raw, 0, 100)

    weight_rows.append({
        "w_rf": w_rf,
        "w_et": w_et,
        "oof_mse_raw": mean_squared_error(y_oof, blend_pred_raw),
        "oof_mse_clipped": mean_squared_error(y_oof, blend_pred_clipped),
    })

blend_weight_results = pd.DataFrame(weight_rows)
blend_weight_results = blend_weight_results.sort_values("oof_mse_clipped").reset_index(drop=True)

best_blend = blend_weight_results.iloc[0].to_dict()
best_w_rf = float(best_blend["w_rf"])
best_w_et = float(best_blend["w_et"])

print("\nBest OOF blend:")
print(pd.DataFrame([best_blend]).to_string(index=False))

print("\nTop 10 blend weights:")
print(blend_weight_results.head(10).to_string(index=False))

# Save OOF blend details
oof_blend_df["blend_pred_raw"] = best_w_rf * rf_oof_pred + best_w_et * et_oof_pred
oof_blend_df["blend_pred_clipped"] = np.clip(oof_blend_df["blend_pred_raw"], 0, 100)

weight_results_path = RESULTS_DIR / "blend_rf500_extratrees_safe_weight_screen.csv"
oof_blend_path = RESULTS_DIR / "oof_blend_rf500_extratrees_safe.csv"

blend_weight_results.to_csv(weight_results_path, index=False)
oof_blend_df.to_csv(oof_blend_path, index=False)

# ------------------------------------------------------------
# Load and align test predictions
# ------------------------------------------------------------

rf_test_raw = read_pred_csv(rf_test_path)
et_test_raw = read_pred_csv(et_test_path)

print("\nRF test prediction columns:")
print(list(rf_test_raw.columns))

print("\nExtraTrees test prediction columns:")
print(list(et_test_raw.columns))

rf_test = standardize_testpred(rf_test_raw, "rf")
et_test = standardize_testpred(et_test_raw, "et")

assert rf_test["ASSESSMENT_ID"].duplicated().sum() == 0, "Duplicate ASSESSMENT_ID in RF test predictions."
assert et_test["ASSESSMENT_ID"].duplicated().sum() == 0, "Duplicate ASSESSMENT_ID in ExtraTrees test predictions."

test_blend_df = rf_test.merge(et_test, on="ASSESSMENT_ID", how="inner")
assert len(test_blend_df) == len(rf_test) == len(et_test), "Test prediction merge lost rows."

# Put submission back in original test_ids order when available.
if "test_ids" in globals():
    test_order = pd.DataFrame({
        "ASSESSMENT_ID": one_dim_id_array(test_ids),
        "_test_order": np.arange(len(test_ids))
    })

    test_blend_df = test_order.merge(test_blend_df, on="ASSESSMENT_ID", how="left")

    assert test_blend_df["rf_test_pred"].isna().sum() == 0, "Missing RF predictions after test_ids alignment."
    assert test_blend_df["et_test_pred"].isna().sum() == 0, "Missing ET predictions after test_ids alignment."

    test_blend_df = test_blend_df.sort_values("_test_order").drop(columns=["_test_order"])

test_blend_df["PERCENT_PROFICIENT"] = np.clip(
    best_w_rf * test_blend_df["rf_test_pred"].to_numpy(dtype=float)
    + best_w_et * test_blend_df["et_test_pred"].to_numpy(dtype=float),
    0,
    100
)

test_blend_path = RESULTS_DIR / "testpred_blend_rf500_extratrees_safe.csv"
submission_blend_path = Path("submission_blend_rf500_extratrees_safe_oof_weighted.csv")

test_blend_df.to_csv(test_blend_path, index=False)

submission_blend = test_blend_df[["ASSESSMENT_ID", "PERCENT_PROFICIENT"]].copy()

expected_test_n = len(test_ids) if "test_ids" in globals() else 48307

assert submission_blend.shape[0] == expected_test_n, f"Submission row count is not {expected_test_n}."
assert list(submission_blend.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
assert submission_blend["ASSESSMENT_ID"].isna().sum() == 0
assert submission_blend["PERCENT_PROFICIENT"].isna().sum() == 0

submission_blend.to_csv(submission_blend_path, index=False)

# ------------------------------------------------------------
# Update trackers if they exist
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["RF500 + ExtraTrees safe OOF blend"] = {
    "model_class": "OOF weighted blend",
    "component_models": "RF 500 base + ExtraTrees safe base",
    "feature_space": "base",
    "oof_mse": float(best_blend["oof_mse_clipped"]),
    "oof_mse_raw": float(best_blend["oof_mse_raw"]),
    "w_rf": best_w_rf,
    "w_et": best_w_et,
    "submission_path": str(submission_blend_path),
    "notes": "Lightweight OOF-tuned blend of saved RF and ExtraTrees artifacts."
}

et_submission_row = {
    "submission_file": "submission_extratrees_safe_base_oof_foldavg.csv",
    "source": "current notebook",
    "model_family": "ExtraTrees",
    "public_mse": 102.56,
    "notes": "Safe ExtraTrees OOF/fold-average submission. Kaggle public leaderboard MSE = 102.56."
}

blend_submission_row = {
    "submission_file": submission_blend_path.name,
    "source": "current notebook",
    "model_family": "RF + ExtraTrees blend",
    "public_mse": np.nan,
    "notes": f"OOF-weighted blend; w_rf={best_w_rf:.3f}, w_et={best_w_et:.3f}, OOF MSE={best_blend['oof_mse_clipped']:.6f}."
}

tracker_new_rows = pd.DataFrame([et_submission_row, blend_submission_row])

if "submission_tracker" in globals():
    submission_tracker = pd.concat(
        [submission_tracker, tracker_new_rows],
        ignore_index=True
    )
    submission_tracker = submission_tracker.drop_duplicates(
        subset=["submission_file"],
        keep="last"
    )
else:
    submission_tracker = tracker_new_rows.copy()

submission_tracker.to_csv(RESULTS_DIR / "submission_tracker.csv", index=False)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\nSaved files:")
print("Weight screen:", weight_results_path.resolve())
print("OOF blend:", oof_blend_path.resolve())
print("Test blend predictions:", test_blend_path.resolve())
print("Blend submission:", submission_blend_path.resolve())
print("Updated submission tracker:", (RESULTS_DIR / "submission_tracker.csv").resolve())

print("\nBlend test prediction summary:")
print(submission_blend["PERCENT_PROFICIENT"].describe().to_string())

print("\nSubmission shape:", submission_blend.shape)

print("\nFirst 5 submission rows:")
print(submission_blend.head().to_string(index=False))

print("\nDecision note:")
if best_w_et >= 0.999:
    print("Best OOF blend is effectively pure ExtraTrees. Prioritize the pure ExtraTrees submission.")
elif best_blend["oof_mse_clipped"] < et_oof_mse:
    print("Blend improves OOF over ExtraTrees alone. This blend submission is worth trying after the pure ExtraTrees submission.")
else:
    print("Blend did not improve OOF over ExtraTrees alone. Prefer the pure ExtraTrees submission.")

Using files:
RF OOF:        model_results/oof_rf_500_base.csv
RF testpred:   model_results/testpred_rf_500_base_folds.csv
ET OOF:        model_results/oof_extratrees_safe_base.csv
ET testpred:   model_results/testpred_extratrees_safe_base_foldavg.csv

RF OOF columns:
['ASSESSMENT_ID', 'y_true', 'rf_500_base_oof_pred']

ExtraTrees OOF columns:
['ASSESSMENT_ID', 'y_true', 'oof_pred', 'oof_pred_clipped']
RF OOF prediction column used: rf_500_base_oof_pred
ET OOF prediction column used: oof_pred

Component OOF MSE:
RF 500 base:        122.328024
ExtraTrees safe:    110.749294

Best OOF blend:
 w_rf  w_et  oof_mse_raw  oof_mse_clipped
0.152 0.848   110.363092       110.363092

Top 10 blend weights:
 w_rf  w_et  oof_mse_raw  oof_mse_clipped
0.152 0.848   110.363092       110.363092
0.153 0.847   110.363099       110.363099
0.151 0.849   110.363118       110.363118
0.154 0.846   110.363139       110.363139
0.150 0.850   110.363178       110.363178
0.155 0.845   110.363212       110.363212
0.1

### RF + ExtraTrees OOF-weighted blend

After the safe ExtraTrees model became the strongest clean bagging-family model, we tested a lightweight OOF-weighted blend with the earlier RF 500-base OOF artifact.

No new models were trained in this step. The cell loaded saved OOF and test-prediction files, searched weights from 0 to 1, and selected the weight with the lowest OOF MSE.

| Model / blend | OOF MSE |
|---|---:|
| RF 500 base | 122.3280 |
| ExtraTrees safe base | 110.7493 |
| RF + ExtraTrees OOF blend | 110.3631 |

Best blend weights:

| Component | Weight |
|---|---:|
| RF 500 base | 0.152 |
| ExtraTrees safe base | 0.848 |

The blend improves OOF MSE by about 0.386 relative to pure ExtraTrees. This is a modest improvement, but it suggests that the RF predictions contain some complementary signal even though RF is weaker as a standalone model.

The generated blend submission file is:

`submission_blend_rf500_extratrees_safe_oof_weighted.csv`

This submission should be tested on Kaggle, but the public leaderboard result may differ from the OOF ranking because the public test split is a different evaluation set.

### Blend submission decision

The RF + ExtraTrees OOF-weighted blend improved OOF MSE only modestly:

| Candidate | OOF MSE |
|---|---:|
| ExtraTrees safe base | 110.7493 |
| RF + ExtraTrees blend | 110.3631 |

The OOF improvement is approximately 0.386 MSE points. Because Kaggle submissions are limited to 2 per day and there are 12 days remaining, this improvement is too small to justify spending a leaderboard submission slot immediately.

Decision:
- Do not submit `submission_blend_rf500_extratrees_safe_oof_weighted.csv` for now.
- Keep it saved as a fallback candidate.
- Use future Kaggle slots only for models or blends with materially stronger offline evidence, preferably from a new model family or a larger OOF improvement.

Current clean public leaderboard benchmark:
- `submission_extratrees_safe_base_oof_foldavg.csv`
- Public MSE: 102.56

In [33]:
# ============================================================
# Post-crash recovery diagnostic: no model fitting, no LightGBM
# ============================================================

from pathlib import Path
import sys
import platform
import gc

import numpy as np
import pandas as pd
import sklearn

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("scikit-learn:", sklearn.__version__)
print("Current working directory:", Path.cwd())

print("\nCore object checks:")
for name in [
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "train_ids",
    "test_ids",
    "X_tr",
    "X_val",
    "y_tr",
    "y_val",
    "feature_spaces",
    "make_step_features",
    "get_top_continuous_features",
]:
    exists = name in globals()
    obj = globals().get(name, None)
    shape = getattr(obj, "shape", None)
    print(f"{name:28s} exists={str(exists):5s} shape={shape}")

print("\nFeature spaces:")
if "feature_spaces" in globals():
    print(feature_spaces.keys())

print("\nRecovery helper functions:")
print("make_step_features:", globals().get("make_step_features", None))
print("get_top_continuous_features:", globals().get("get_top_continuous_features", None))

print("\nSaved model_results artifacts:")
results_dir = Path("model_results")
print("model_results exists:", results_dir.exists())
if results_dir.exists():
    for path in sorted(results_dir.glob("*")):
        print(" -", path.name)

gc.collect()

Python: 3.14.2
Platform: macOS-15.6-arm64-arm-64bit-Mach-O
scikit-learn: 1.8.0
Current working directory: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart

Core object checks:
X_train_proc_model           exists=True  shape=(144921, 162)
X_test_proc_model            exists=True  shape=(48307, 162)
y_train                      exists=True  shape=(144921,)
train_ids                    exists=True  shape=(144921,)
test_ids                     exists=True  shape=(48307,)
X_tr                         exists=True  shape=(115936, 162)
X_val                        exists=True  shape=(28985, 162)
y_tr                         exists=True  shape=(115936,)
y_val                        exists=True  shape=(28985,)
feature_spaces               exists=True  shape=None
make_step_features           exists=True  shape=None
get_top_continuous_features  exists=True  shape=None

Feature spaces:
dict_keys(['base', 'base_plus_poly', 'base_plus_interactio

1795

In [60]:
import lightgbm as lgb
print(lgb.__version__)

4.6.0


### Post-crash status checkpoint

The kernel recovery check passed. The core preprocessing objects, feature spaces, train/validation split, and saved model artifacts are available. LightGBM imports successfully, but fitting stability has not yet been verified after the crash. Before rerunning any LightGBM or other expensive model, we inspect saved result files and continue with small, checkpointed cells only.

In [34]:
# ============================================================
# Inspect saved result artifacts after LightGBM crash/recovery
# No model fitting in this cell.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

results_dir = Path("model_results")

files_to_check = [
    "model_scoreboard.csv",
    "submission_tracker.csv",
    "lgbm_clean_base_holdout_screen.csv",
    "extratrees_safe_oof_summary.csv",
    "extratrees_safe_oof_fold_results.csv",
    "oof_blend_rf500_extratrees_safe.csv",
    "blend_rf500_extratrees_safe_weight_screen.csv",
]

for fname in files_to_check:
    path = results_dir / fname
    print("\n" + "=" * 80)
    print(fname)
    print("exists:", path.exists())

    if not path.exists():
        continue

    df = pd.read_csv(path)
    print("shape:", df.shape)
    print("columns:", list(df.columns))

    # Try to show the most informative ordering without assuming exact column names.
    sort_candidates = [
        "public_mse",
        "oof_mse",
        "valid_mse",
        "cv_valid_mse_mean",
        "holdout_mse",
        "mse",
    ]

    sort_col = None
    for col in sort_candidates:
        if col in df.columns:
            sort_col = col
            break

    if sort_col is not None:
        print(f"sorted by: {sort_col}")
        display(df.sort_values(sort_col).head(10))
    else:
        display(df.head(10))


model_scoreboard.csv
exists: True
shape: (2, 14)
columns: ['model_key', 'feature_space', 'cv_valid_mse_mean', 'cv_valid_mse_std', 'holdout_val_mse', 'oof_mse', 'holdout_oob_mse', 'n_estimators', 'max_features', 'max_samples', 'min_samples_leaf', 'min_samples_split', 'submission_path', 'notes']
sorted by: oof_mse


,model_key,feature_space,cv_valid_mse_mean,cv_valid_mse_std,holdout_val_mse,oof_mse,holdout_oob_mse,n_estimators,max_features,max_samples,min_samples_leaf,min_samples_split,submission_path,notes
0,Random Forest,base,149.549571,0.282329,127.479635,NaN,129.17549,120.0,0.5,0.8,1,2,NaN,First controlled Random Forest CV screen on ba...
1,Regression Tree,base_plus_poly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25,50,NaN,Single regression tree baseline across control...



submission_tracker.csv
exists: True
shape: (3, 5)
columns: ['submission_file', 'source', 'model_family', 'public_mse', 'notes']
sorted by: public_mse


,submission_file,source,model_family,public_mse,notes
0,submission_oof_lgbm.csv,external teammate reference,LightGBM,91.092,Current external team benchmark; do not commin...
1,submission_rf_refined_300_base.csv,current notebook,Random Forest,111.096,Full-data refit from refined RF CV-selected se...
2,submission_rf_500_base_oof_foldavg.csv,current notebook,Random Forest,NaN,Pending RF OOF/fold-average submission candidate.



lgbm_clean_base_holdout_screen.csv
exists: True
shape: (2, 19)
columns: ['model_class', 'backend', 'stage', 'feature_space', 'config_name', 'train_mse_clipped', 'holdout_val_mse_clipped', 'n_iter_used', 'elapsed_sec', 'n_estimators', 'learning_rate', 'num_leaves', 'min_child_samples', 'subsample', 'subsample_freq', 'colsample_bytree', 'reg_alpha', 'reg_lambda', 'max_depth']


...

,model_class,stage,feature_space,fold,config_name,n_estimators,max_features,min_samples_leaf,min_samples_split,bootstrap,max_samples,fold_valid_mse,elapsed_sec
0,ExtraTrees,oof_selected_config,base,1,et_safe_oof_from_et_base_180_mf0.5_leaf1_no_bo...,250,0.5,1,2,False,NaN,111.639330,169.659820
1,ExtraTrees,oof_selected_config,base,2,et_safe_oof_from_et_base_180_mf0.5_leaf1_no_bo...,250,0.5,1,2,False,NaN,109.155334,166.818236
2,ExtraTrees,oof_selected_config,base,3,et_safe_oof_from_et_base_180_mf0.5_leaf1_no_bo...,250,0.5,1,2,False,NaN,111.870290,165.977457
3,ExtraTrees,oof_selected_config,base,4,et_safe_oof_from_et_base_180_mf0.5_leaf1_no_bo...,250,0.5,1,2,False,NaN,108.405409,166.278418
4,ExtraTrees,oof_selected_config,base,5,et_safe_oof_from_et_base_180_mf0.5_leaf1_no_bo...,250,0.5,1,2,False,NaN,112.676076,166.069512



oof_blend_rf500_extratrees_safe.csv
exists: True
shape: (144921, 6)
columns: ['ASSESSMENT_ID', 'rf_oof_pred', 'et_oof_pred', 'y_true', 'blend_pred_raw', 'blend_pred_clipped']


,ASSESSMENT_ID,rf_oof_pred,et_oof_pred,y_true,blend_pred_raw,blend_pred_clipped
0,3b6deef53665,36.480,37.668,38.0,37.487424,37.487424
1,962a3bfbfe84,60.716,61.660,65.0,61.516512,61.516512
2,ffe086287b6e,73.440,74.560,65.0,74.389760,74.389760
3,e6f80847409d,51.054,51.868,52.0,51.744272,51.744272
4,676cc6d81961,49.046,49.600,46.0,49.515792,49.515792
5,c9292c7cecae,63.482,67.844,74.0,67.180976,67.180976
6,121afff188a5,10.578,11.476,13.0,11.339504,11.339504
7,2611f38e41a1,77.030,77.668,80.0,77.571024,77.571024
8,fc3217bcfd05,86.298,87.172,91.0,87.039152,87.039152
9,0394ccf86692,60.324,53.588,56.0,54.611872,54.611872



blend_rf500_extratrees_safe_weight_screen.csv
exists: True
shape: (1001, 4)
columns: ['w_rf', 'w_et', 'oof_mse_raw', 'oof_mse_clipped']


,w_rf,w_et,oof_mse_raw,oof_mse_clipped
0,0.152,0.848,110.363092,110.363092
1,0.153,0.847,110.363099,110.363099
2,0.151,0.849,110.363118,110.363118
3,0.154,0.846,110.363139,110.363139
4,0.150,0.850,110.363178,110.363178
5,0.155,0.845,110.363212,110.363212
6,0.149,0.851,110.363272,110.363272
7,0.156,0.844,110.363318,110.363318
8,0.148,0.852,110.363398,110.363398
9,0.157,0.843,110.363458,110.363458


### LightGBM post-crash interpretation

A saved LightGBM holdout-screen artifact exists and contains two completed base-feature-space fits. The better saved configuration reached holdout validation MSE around 114.08, which is useful but not stronger than the current ExtraTrees / RF+ExtraTrees artifacts. Because the previous kernel crash occurred around the LightGBM installation/fitting stage, we will not rerun a full LightGBM screen yet. First we run a small, single-threaded smoke fit with early stopping to confirm that LightGBM fitting is stable in the current kernel.

### 20A. Clean boosting screen and selected OOF artifact

We now move from bagging-style tree ensembles to boosting.

This cell is a clean implementation from the current notebook pipeline. It does not use or copy any teammate/reference notebook code. If `lightgbm` is installed, it will run a small LightGBM screen. If `lightgbm` is not installed, it will fall back to scikit-learn's `HistGradientBoostingRegressor`.

Design:
- Use the `base` feature space because Random Forest and ExtraTrees both preferred the base feature matrix.
- Run a small holdout screen first.
- Select the best boosting configuration by holdout validation MSE.
- Run a 5-fold OOF/fold-averaged test artifact only if the screen is competitive enough.
- Save OOF predictions and a submission candidate for later use.
- Do not spend a Kaggle submission slot yet unless offline evidence is materially strong.

In [35]:
# ============================================================
# LightGBM post-crash smoke test
# Small sample, single-threaded, early stopping, no submission.
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

RNG_SEED = globals().get("RANDOM_STATE", 9890)
rng = np.random.default_rng(RNG_SEED)

def take_rows(X, idx):
    """Works for both pandas objects and numpy arrays."""
    if hasattr(X, "iloc"):
        return X.iloc[idx]
    return X[idx]

# Keep this intentionally small so we test stability without stressing the kernel.
n_train_smoke = min(20_000, X_tr.shape[0])
n_val_smoke = min(8_000, X_val.shape[0])

tr_idx = rng.choice(X_tr.shape[0], size=n_train_smoke, replace=False)
val_idx = rng.choice(X_val.shape[0], size=n_val_smoke, replace=False)

X_smoke = np.asarray(take_rows(X_tr, tr_idx), dtype=np.float32)
y_smoke = np.asarray(take_rows(y_tr, tr_idx), dtype=np.float32).ravel()

X_smoke_val = np.asarray(take_rows(X_val, val_idx), dtype=np.float32)
y_smoke_val = np.asarray(take_rows(y_val, val_idx), dtype=np.float32).ravel()

print("LightGBM version:", lgb.__version__)
print("Smoke train shape:", X_smoke.shape)
print("Smoke validation shape:", X_smoke_val.shape)

smoke_params = dict(
    objective="regression",
    metric="l2",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=80,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=5.0,
    max_depth=-1,
    random_state=RNG_SEED,
    n_jobs=1,              # important: avoid worker/thread pressure after crash
    verbosity=-1,
    force_col_wise=True,
)

start = time.time()

smoke_model = lgb.LGBMRegressor(**smoke_params)

smoke_model.fit(
    X_smoke,
    y_smoke,
    eval_set=[(X_smoke_val, y_smoke_val)],
    eval_metric="l2",
    callbacks=[
        lgb.early_stopping(stopping_rounds=40, verbose=True),
        lgb.log_evaluation(period=50),
    ],
)

elapsed = time.time() - start

best_iter = getattr(smoke_model, "best_iteration_", None)
pred_raw = smoke_model.predict(X_smoke_val, num_iteration=best_iter)
pred_clipped = np.clip(pred_raw, 0, 100)

smoke_summary = pd.DataFrame([{
    "model_class": "LightGBM",
    "stage": "post_crash_smoke_test",
    "feature_space": "base",
    "n_train_smoke": n_train_smoke,
    "n_val_smoke": n_val_smoke,
    "best_iteration": best_iter,
    "val_mse_raw": mean_squared_error(y_smoke_val, pred_raw),
    "val_mse_clipped": mean_squared_error(y_smoke_val, pred_clipped),
    "elapsed_sec": elapsed,
    **smoke_params,
}])

results_dir = Path("model_results")
results_dir.mkdir(exist_ok=True)

smoke_path = results_dir / "lgbm_post_crash_smoke_test.csv"
smoke_summary.to_csv(smoke_path, index=False)

print("\nSaved:", smoke_path)
display(smoke_summary)

# Clean up immediately.
del smoke_model, X_smoke, y_smoke, X_smoke_val, y_smoke_val, pred_raw, pred_clipped
gc.collect()

LightGBM version: 4.6.0
Smoke train shape: (20000, 162)
Smoke validation shape: (8000, 162)
Training until validation scores don't improve for 40 rounds
[50]	valid_0's l2: 302.248
[100]	valid_0's l2: 254.695
[150]	valid_0's l2: 238.767
[200]	valid_0's l2: 228.813
[250]	valid_0's l2: 222.504
[300]	valid_0's l2: 217.859
[350]	valid_0's l2: 213.881
[400]	valid_0's l2: 210.339
Did not meet early stopping. Best iteration is:
[400]	valid_0's l2: 210.339


/Users/saadmanchowdhury/Desktop/All Github projects/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Saved: model_results/lgbm_post_crash_smoke_test.csv


,model_class,stage,feature_space,n_train_smoke,n_val_smoke,best_iteration,val_mse_raw,val_mse_clipped,elapsed_sec,objective,metric,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,max_depth,random_state,n_jobs,verbosity,force_col_wise
0,LightGBM,post_crash_smoke_test,base,20000,8000,400,210.339065,209.938381,1.535752,regression,l2,400,0.05,31,80,0.8,1,0.8,0.0,5.0,-1,9890,1,-1,True


1728

In [36]:
# ============================================================
# Controlled LightGBM holdout screen
# Full X_tr / X_val split, base feature space only.
# Single-threaded, early stopping, checkpoint after each config.
# No submission and no OOF artifacts in this cell.
# ============================================================

from pathlib import Path
import time
import gc
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

RNG_SEED = globals().get("RANDOM_STATE", 9890)

results_dir = Path("model_results")
results_dir.mkdir(exist_ok=True)

screen_path = results_dir / "lgbm_controlled_base_holdout_screen.csv"

# Use NumPy arrays consistently for fit and predict.
# float32 reduces memory pressure and is fine for tree models.
X_train_lgb = np.ascontiguousarray(np.asarray(X_tr, dtype=np.float32))
X_val_lgb = np.ascontiguousarray(np.asarray(X_val, dtype=np.float32))
y_train_lgb = np.asarray(y_tr, dtype=np.float32).ravel()
y_val_lgb = np.asarray(y_val, dtype=np.float32).ravel()

print("LightGBM version:", lgb.__version__)
print("Train shape:", X_train_lgb.shape)
print("Validation shape:", X_val_lgb.shape)
print("Checkpoint file:", screen_path)

common_params = dict(
    objective="regression",
    metric="l2",
    random_state=RNG_SEED,
    n_jobs=1,                 # keep conservative after kernel crash
    verbosity=-1,
    force_col_wise=True,
)

lgbm_configs = [
    dict(
        config_name="lgbm_c01_lr03_l63_child80_l2_5_sub08_col08",
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=80,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=5.0,
        max_depth=-1,
    ),
    dict(
        config_name="lgbm_c02_lr03_l63_child120_l2_10_sub08_col08",
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=120,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=10.0,
        max_depth=-1,
    ),
    dict(
        config_name="lgbm_c03_lr02_l63_child80_l2_10_sub085_col085",
        n_estimators=3500,
        learning_rate=0.02,
        num_leaves=63,
        min_child_samples=80,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=10.0,
        max_depth=-1,
    ),
    dict(
        config_name="lgbm_c04_lr03_l127_child120_l2_20_sub08_col08",
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=127,
        min_child_samples=120,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=20.0,
        max_depth=-1,
    ),
]

if screen_path.exists():
    screen_df = pd.read_csv(screen_path)
    if "status" in screen_df.columns:
        done_configs = set(
            screen_df.loc[
                screen_df["status"].eq("completed"),
                "config_name"
            ].astype(str)
        )
    elif "config_name" in screen_df.columns:
        done_configs = set(screen_df["config_name"].astype(str))
    else:
        done_configs = set()
else:
    screen_df = pd.DataFrame()
    done_configs = set()

print("Already completed configs:", sorted(done_configs))

for cfg in lgbm_configs:
    config_name = cfg["config_name"]

    if config_name in done_configs:
        print(f"\nSkipping already completed config: {config_name}")
        continue

    print("\n" + "=" * 80)
    print("Running:", config_name)

    params = {**common_params, **cfg}
    params_for_model = params.copy()
    params_for_model.pop("config_name")

    start = time.time()

    model = lgb.LGBMRegressor(**params_for_model)

    model.fit(
        X_train_lgb,
        y_train_lgb,
        eval_set=[(X_val_lgb, y_val_lgb)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=True),
            lgb.log_evaluation(period=250),
        ],
    )

    elapsed = time.time() - start

    best_iter = getattr(model, "best_iteration_", None)

    pred_train_raw = model.predict(X_train_lgb, num_iteration=best_iter)
    pred_val_raw = model.predict(X_val_lgb, num_iteration=best_iter)

    pred_train_clip = np.clip(pred_train_raw, 0, 100)
    pred_val_clip = np.clip(pred_val_raw, 0, 100)

    row = {
        "model_class": "LightGBM",
        "backend": "lgbm",
        "stage": "controlled_holdout_screen",
        "feature_space": "base",
        "status": "completed",
        "config_name": config_name,
        "train_mse_raw": mean_squared_error(y_train_lgb, pred_train_raw),
        "train_mse_clipped": mean_squared_error(y_train_lgb, pred_train_clip),
        "holdout_val_mse_raw": mean_squared_error(y_val_lgb, pred_val_raw),
        "holdout_val_mse_clipped": mean_squared_error(y_val_lgb, pred_val_clip),
        "best_iteration": best_iter,
        "elapsed_sec": elapsed,
    }

    for key, value in cfg.items():
        if key != "config_name":
            row[key] = value

    screen_df = pd.concat(
        [screen_df, pd.DataFrame([row])],
        ignore_index=True
    )

    screen_df.to_csv(screen_path, index=False)

    print("\nCompleted:", config_name)
    print("Best iteration:", best_iter)
    print("Train MSE clipped:", row["train_mse_clipped"])
    print("Holdout MSE clipped:", row["holdout_val_mse_clipped"])
    print("Elapsed seconds:", round(elapsed, 2))
    print("Checkpoint saved:", screen_path)

    del model
    del pred_train_raw, pred_val_raw, pred_train_clip, pred_val_clip
    gc.collect()

print("\n" + "=" * 80)
print("Controlled LightGBM screen results:")

screen_df = pd.read_csv(screen_path)

display_cols = [
    "config_name",
    "train_mse_clipped",
    "holdout_val_mse_clipped",
    "best_iteration",
    "elapsed_sec",
    "n_estimators",
    "learning_rate",
    "num_leaves",
    "min_child_samples",
    "subsample",
    "colsample_bytree",
    "reg_lambda",
]

display(
    screen_df
    .sort_values("holdout_val_mse_clipped")
    [display_cols]
    .reset_index(drop=True)
)

# Clean up full LightGBM arrays.
del X_train_lgb, X_val_lgb, y_train_lgb, y_val_lgb
gc.collect()

LightGBM version: 4.6.0
Train shape: (115936, 162)
Validation shape: (28985, 162)
Checkpoint file: model_results/lgbm_controlled_base_holdout_screen.csv
Already completed configs: []

Running: lgbm_c01_lr03_l63_child80_l2_5_sub08_col08
Training until validation scores don't improve for 100 rounds
[250]	valid_0's l2: 199.162
[500]	valid_0's l2: 174.406
[750]	valid_0's l2: 163.192
[1000]	valid_0's l2: 156.177
[1250]	valid_0's l2: 151.088
[1500]	valid_0's l2: 147.206
[1750]	valid_0's l2: 144.006
[2000]	valid_0's l2: 141.114
[2250]	valid_0's l2: 138.747
[2500]	valid_0's l2: 136.724
Did not meet early stopping. Best iteration is:
[2500]	valid_0's l2: 136.724

Completed: lgbm_c01_lr03_l63_child80_l2_5_sub08_col08
Best iteration: 2500
Train MSE clipped: 92.78294057631074
Holdout MSE clipped: 136.40299287236442
Elapsed seconds: 36.1
Checkpoint saved: model_results/lgbm_controlled_base_holdout_screen.csv

Running: lgbm_c02_lr03_l63_child120_l2_10_sub08_col08
Training until validation scores don

,config_name,train_mse_clipped,holdout_val_mse_clipped,best_iteration,elapsed_sec,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda
0,lgbm_c04_lr03_l127_child120_l2_20_sub08_col08,70.014712,126.393035,2500,51.681790,2500,0.03,127,120,0.80,0.80,20.0
1,lgbm_c01_lr03_l63_child80_l2_5_sub08_col08,92.782941,136.402993,2500,36.097008,2500,0.03,63,80,0.80,0.80,5.0
2,lgbm_c03_lr02_l63_child80_l2_10_sub085_col085,95.873512,137.583865,3500,49.708509,3500,0.02,63,80,0.85,0.85,10.0
3,lgbm_c02_lr03_l63_child120_l2_10_sub08_col08,97.946833,139.495044,2500,37.048055,2500,0.03,63,120,0.80,0.80,10.0


0

In [37]:
# ============================================================
# 20B. LightGBM targeted holdout refinement
# Conservative, checkpointed, base feature space only.
# No OOF and no submission in this cell.
# ============================================================

from pathlib import Path
import time
import gc
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

RNG_SEED = globals().get("RANDOM_STATE", 9890)

results_dir = Path("model_results")
results_dir.mkdir(exist_ok=True)

refine_path = results_dir / "lgbm_targeted_base_holdout_refinement.csv"

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

required_objects = ["X_tr", "X_val", "y_tr", "y_val"]
missing_objects = [obj for obj in required_objects if obj not in globals()]
assert len(missing_objects) == 0, f"Missing objects: {missing_objects}"

print("LightGBM version:", lgb.__version__)
print("Train shape:", X_tr.shape)
print("Validation shape:", X_val.shape)
print("Checkpoint file:", refine_path)

# Use NumPy arrays consistently for fit and predict.
# float32 reduces memory pressure and avoids DataFrame/name warnings.
X_train_lgb = np.ascontiguousarray(np.asarray(X_tr, dtype=np.float32))
X_val_lgb = np.ascontiguousarray(np.asarray(X_val, dtype=np.float32))
y_train_lgb = np.asarray(y_tr, dtype=np.float32).ravel()
y_val_lgb = np.asarray(y_val, dtype=np.float32).ravel()

common_params = dict(
    objective="regression",
    metric="l2",
    random_state=RNG_SEED,
    n_jobs=1,              # keep conservative after kernel crashes
    verbosity=-1,
    force_col_wise=True,
)

# These configs refine around the strongest saved completed LightGBM region:
# leaves around 63, min_child_samples around 40, subsample/colsample around 0.9.
# We are not doing OOF yet; this is just a targeted holdout refinement.
targeted_configs = [
    dict(
        config_name="lgbm_t01_lr03_l63_child40_l2_1_sub09_col09",
        n_estimators=8000,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=40,
        subsample=0.90,
        subsample_freq=1,
        colsample_bytree=0.90,
        reg_alpha=0.0,
        reg_lambda=1.0,
        max_depth=-1,
    ),
    dict(
        config_name="lgbm_t02_lr02_l63_child40_l2_1_sub09_col09",
        n_estimators=10000,
        learning_rate=0.02,
        num_leaves=63,
        min_child_samples=40,
        subsample=0.90,
        subsample_freq=1,
        colsample_bytree=0.90,
        reg_alpha=0.0,
        reg_lambda=1.0,
        max_depth=-1,
    ),
    dict(
        config_name="lgbm_t03_lr03_l95_child60_l2_5_sub085_col09",
        n_estimators=8000,
        learning_rate=0.03,
        num_leaves=95,
        min_child_samples=60,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.90,
        reg_alpha=0.0,
        reg_lambda=5.0,
        max_depth=-1,
    ),
]

if refine_path.exists():
    refine_df = pd.read_csv(refine_path)
    done_configs = set(refine_df["config_name"].astype(str)) if "config_name" in refine_df.columns else set()
else:
    refine_df = pd.DataFrame()
    done_configs = set()

print("Already completed targeted configs:", sorted(done_configs))

for cfg in targeted_configs:
    config_name = cfg["config_name"]

    if config_name in done_configs:
        print(f"\nSkipping already completed config: {config_name}")
        continue

    print("\n" + "=" * 80)
    print("Running targeted LightGBM config:", config_name)
    print("=" * 80)

    params = {**common_params, **cfg}
    params_for_model = params.copy()
    params_for_model.pop("config_name")

    start = time.time()

    model = lgb.LGBMRegressor(**params_for_model)

    model.fit(
        X_train_lgb,
        y_train_lgb,
        eval_set=[(X_val_lgb, y_val_lgb)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=300, verbose=True),
            lgb.log_evaluation(period=500),
        ],
    )

    elapsed = time.time() - start

    best_iter = getattr(model, "best_iteration_", None)
    if best_iter is None or best_iter <= 0:
        best_iter = cfg["n_estimators"]

    pred_train_raw = model.predict(X_train_lgb, num_iteration=best_iter)
    pred_val_raw = model.predict(X_val_lgb, num_iteration=best_iter)

    pred_train_clip = np.clip(pred_train_raw, 0, 100)
    pred_val_clip = np.clip(pred_val_raw, 0, 100)

    row = {
        "model_class": "LightGBM",
        "backend": "lgbm",
        "stage": "targeted_holdout_refinement",
        "feature_space": "base",
        "status": "completed",
        "config_name": config_name,
        "train_mse_raw": mean_squared_error(y_train_lgb, pred_train_raw),
        "train_mse_clipped": mean_squared_error(y_train_lgb, pred_train_clip),
        "holdout_val_mse_raw": mean_squared_error(y_val_lgb, pred_val_raw),
        "holdout_val_mse_clipped": mean_squared_error(y_val_lgb, pred_val_clip),
        "best_iteration": best_iter,
        "elapsed_sec": elapsed,
    }

    for key, value in cfg.items():
        if key != "config_name":
            row[key] = value

    refine_df = pd.concat(
        [refine_df, pd.DataFrame([row])],
        ignore_index=True
    )
    refine_df.to_csv(refine_path, index=False)

    print("\nCompleted:", config_name)
    print("Best iteration:", best_iter)
    print("Train MSE clipped:", row["train_mse_clipped"])
    print("Holdout MSE clipped:", row["holdout_val_mse_clipped"])
    print("Elapsed seconds:", round(elapsed, 2))
    print("Checkpoint saved:", refine_path)

    del model
    del pred_train_raw, pred_val_raw, pred_train_clip, pred_val_clip
    gc.collect()

# ------------------------------------------------------------
# Consolidate all current LightGBM holdout screens
# ------------------------------------------------------------

screen_files = [
    "lgbm_clean_base_holdout_screen.csv",
    "lgbm_controlled_base_holdout_screen.csv",
    "lgbm_targeted_base_holdout_refinement.csv",
]

frames = []

for fname in screen_files:
    path = results_dir / fname
    if not path.exists():
        continue

    df = pd.read_csv(path)
    df["source_file"] = fname

    if "best_iteration" not in df.columns and "n_iter_used" in df.columns:
        df["best_iteration"] = df["n_iter_used"]

    frames.append(df)

if len(frames) > 0:
    all_lgbm_screens = pd.concat(frames, ignore_index=True)

    display_cols = [
        "source_file",
        "config_name",
        "train_mse_clipped",
        "holdout_val_mse_clipped",
        "best_iteration",
        "elapsed_sec",
        "n_estimators",
        "learning_rate",
        "num_leaves",
        "min_child_samples",
        "subsample",
        "colsample_bytree",
        "reg_lambda",
    ]
    display_cols = [c for c in display_cols if c in all_lgbm_screens.columns]

    all_lgbm_screens = all_lgbm_screens.sort_values("holdout_val_mse_clipped")

    print("\n" + "=" * 80)
    print("Combined LightGBM holdout screen leaderboard:")
    display(all_lgbm_screens[display_cols].head(15).reset_index(drop=True))

    best_lgbm = all_lgbm_screens.iloc[0]

    print("\nBest LightGBM holdout result so far:")
    print("source_file:", best_lgbm["source_file"])
    print("config_name:", best_lgbm["config_name"])
    print("holdout_val_mse_clipped:", best_lgbm["holdout_val_mse_clipped"])
    print("best_iteration:", best_lgbm.get("best_iteration", None))

    print("\nDecision guide:")
    if best_lgbm["holdout_val_mse_clipped"] <= 111.0:
        print("LightGBM is now close enough to ExtraTrees/RF+ET offline results to consider a selected OOF artifact next.")
    else:
        print("Do not run LightGBM OOF yet. Holdout is still not clearly better than the ExtraTrees/RF+ET artifacts.")

else:
    print("No LightGBM screen files found.")

# Clean up large arrays from this cell.
del X_train_lgb, X_val_lgb, y_train_lgb, y_val_lgb
gc.collect()

LightGBM version: 4.6.0
Train shape: (115936, 162)
Validation shape: (28985, 162)
Checkpoint file: model_results/lgbm_targeted_base_holdout_refinement.csv
Already completed targeted configs: []

Running targeted LightGBM config: lgbm_t01_lr03_l63_child40_l2_1_sub09_col09
Training until validation scores don't improve for 300 rounds
[500]	valid_0's l2: 173.611
[1000]	valid_0's l2: 153.444
[1500]	valid_0's l2: 142.397
[2000]	valid_0's l2: 134.757
[2500]	valid_0's l2: 129.388
[3000]	valid_0's l2: 125.208
[3500]	valid_0's l2: 121.699
[4000]	valid_0's l2: 118.833
[4500]	valid_0's l2: 116.477
[5000]	valid_0's l2: 114.247
[5500]	valid_0's l2: 112.313
[6000]	valid_0's l2: 110.83
[6500]	valid_0's l2: 109.438
[7000]	valid_0's l2: 108.097
[7500]	valid_0's l2: 107.004
[8000]	valid_0's l2: 105.968
Did not meet early stopping. Best iteration is:
[8000]	valid_0's l2: 105.968

Completed: lgbm_t01_lr03_l63_child40_l2_1_sub09_col09
Best iteration: 8000
Train MSE clipped: 35.15224971100176
Holdout MSE cl

,source_file,config_name,train_mse_clipped,holdout_val_mse_clipped,best_iteration,elapsed_sec,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda
0,lgbm_targeted_base_holdout_refinement.csv,lgbm_t03_lr03_l95_child60_l2_5_sub085_col09,27.643127,104.496354,8000,118.549880,8000,0.03,95,60,0.85,0.90,5.0
1,lgbm_targeted_base_holdout_refinement.csv,lgbm_t01_lr03_l63_child40_l2_1_sub09_col09,35.152250,105.586469,8000,103.653627,8000,0.03,63,40,0.90,0.90,1.0
2,lgbm_targeted_base_holdout_refinement.csv,lgbm_t02_lr02_l63_child40_l2_1_sub09_col09,41.509765,108.300128,10000,122.553700,10000,0.02,63,40,0.90,0.90,1.0
3,lgbm_clean_base_holdout_screen.csv,lgbm_lr03_leaves63_l2_1_sub09_col09,52.948577,114.079714,4999,62.768896,5000,0.03,63,40,0.90,0.90,1.0
4,lgbm_controlled_base_holdout_screen.csv,lgbm_c04_lr03_l127_child120_l2_20_sub08_col08,70.014712,126.393035,2500,51.681790,2500,0.03,127,120,0.80,0.80,20.0
5,lgbm_clean_base_holdout_screen.csv,lgbm_lr03_leaves31_l2_1_sub09_col09,85.639976,130.751393,5000,42.012063,5000,0.03,31,40,0.90,0.90,1.0
6,lgbm_controlled_base_holdout_screen.csv,lgbm_c01_lr03_l63_child80_l2_5_sub08_col08,92.782941,136.402993,2500,36.097008,2500,0.03,63,80,0.80,0.80,5.0
7,lgbm_controlled_base_holdout_screen.csv,lgbm_c03_lr02_l63_child80_l2_10_sub085_col085,95.873512,137.583865,3500,49.708509,3500,0.02,63,80,0.85,0.85,10.0
8,lgbm_controlled_base_holdout_screen.csv,lgbm_c02_lr03_l63_child120_l2_10_sub08_col08,97.946833,139.495044,2500,37.048055,2500,0.03,63,120,0.80,0.80,10.0



Best LightGBM holdout result so far:
source_file: lgbm_targeted_base_holdout_refinement.csv
config_name: lgbm_t03_lr03_l95_child60_l2_5_sub085_col09
holdout_val_mse_clipped: 104.49635410873547
best_iteration: 8000

Decision guide:
LightGBM is now close enough to ExtraTrees/RF+ET offline results to consider a selected OOF artifact next.


0

In [38]:
# ============================================================
# 21A. Selected LightGBM OOF artifact + fold-averaged submission
# Uses the best targeted holdout config:
# lgbm_t03_lr03_l95_child60_l2_5_sub085_col09
#
# Conservative design:
# - one config only
# - base feature space only
# - n_jobs=1
# - checkpoint after every fold
# - resume-safe
# - no old crashed ExtraTrees code involved
# ============================================================

from pathlib import Path
import time
import gc
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

results_dir = Path("model_results")
results_dir.mkdir(exist_ok=True)

artifact_name = "lgbm_t03_base_5fold_oof"

metrics_path = results_dir / f"{artifact_name}_fold_metrics.csv"
oof_npy_path = results_dir / f"{artifact_name}_oof.npy"
test_sum_npy_path = results_dir / f"{artifact_name}_test_pred_sum.npy"

oof_csv_path = results_dir / f"oof_{artifact_name}.csv"
testpred_csv_path = results_dir / f"testpred_{artifact_name}_foldavg.csv"
submission_path = Path(f"submission_{artifact_name}_foldavg.csv")

print("LightGBM version:", lgb.__version__)
print("Artifact:", artifact_name)
print("Metrics checkpoint:", metrics_path)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

required_objects = [
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]
assert len(missing_objects) == 0, f"Missing required objects: {missing_objects}"

assert X_train_proc_model.shape[0] == len(y_train), (
    X_train_proc_model.shape,
    len(y_train),
)
assert X_test_proc_model.shape[0] == len(test_ids), (
    X_test_proc_model.shape,
    len(test_ids),
)

print("Full train shape:", X_train_proc_model.shape)
print("Full test shape:", X_test_proc_model.shape)
print("Target length:", len(y_train))
print("Test ID length:", len(test_ids))

# Convert once to memory-conscious contiguous arrays.
# This avoids repeated DataFrame slicing overhead and keeps LightGBM stable.
X_full_lgb = np.ascontiguousarray(np.asarray(X_train_proc_model, dtype=np.float32))
X_test_lgb = np.ascontiguousarray(np.asarray(X_test_proc_model, dtype=np.float32))
y_full_lgb = np.asarray(y_train, dtype=np.float32).ravel()

n_train = X_full_lgb.shape[0]
n_test = X_test_lgb.shape[0]

# ------------------------------------------------------------
# Selected LightGBM config
# ------------------------------------------------------------

selected_lgbm_params = dict(
    objective="regression",
    metric="l2",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=-1,
    force_col_wise=True,

    # Selected from targeted holdout refinement
    n_estimators=12000,
    learning_rate=0.03,
    num_leaves=95,
    min_child_samples=60,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.90,
    reg_alpha=0.0,
    reg_lambda=5.0,
    max_depth=-1,
)

print("\nSelected LightGBM parameters:")
for k, v in selected_lgbm_params.items():
    print(f"  {k}: {v}")

# ------------------------------------------------------------
# Resume-safe checkpoint setup
# ------------------------------------------------------------

if metrics_path.exists():
    fold_metrics = pd.read_csv(metrics_path)
    completed_folds = set(fold_metrics["fold"].astype(int).tolist())
else:
    fold_metrics = pd.DataFrame()
    completed_folds = set()

if oof_npy_path.exists():
    oof_pred = np.load(oof_npy_path)
    assert len(oof_pred) == n_train
else:
    oof_pred = np.full(n_train, np.nan, dtype=np.float32)

if test_sum_npy_path.exists():
    test_pred_sum = np.load(test_sum_npy_path)
    assert len(test_pred_sum) == n_test
else:
    test_pred_sum = np.zeros(n_test, dtype=np.float32)

print("\nAlready completed folds:", sorted(completed_folds))

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

start_all = time.time()

# ------------------------------------------------------------
# 5-fold OOF loop
# ------------------------------------------------------------

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_full_lgb), start=1):

    if fold in completed_folds:
        print(f"\nSkipping already completed fold {fold}")
        continue

    print("\n" + "=" * 80)
    print(f"Starting LightGBM fold {fold}/5")
    print("=" * 80)

    X_fold_tr = X_full_lgb[tr_idx]
    y_fold_tr = y_full_lgb[tr_idx]
    X_fold_val = X_full_lgb[val_idx]
    y_fold_val = y_full_lgb[val_idx]

    fold_params = selected_lgbm_params.copy()
    fold_params["random_state"] = RANDOM_STATE + fold

    model = lgb.LGBMRegressor(**fold_params)

    fold_start = time.time()

    model.fit(
        X_fold_tr,
        y_fold_tr,
        eval_set=[(X_fold_val, y_fold_val)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=500, verbose=True),
            lgb.log_evaluation(period=1000),
        ],
    )

    fold_elapsed = time.time() - fold_start

    best_iter = getattr(model, "best_iteration_", None)
    if best_iter is None or best_iter <= 0:
        best_iter = selected_lgbm_params["n_estimators"]

    val_pred_raw = model.predict(X_fold_val, num_iteration=best_iter)
    val_pred_clip = np.clip(val_pred_raw, 0, 100).astype(np.float32)

    test_pred_raw = model.predict(X_test_lgb, num_iteration=best_iter)
    test_pred_clip = np.clip(test_pred_raw, 0, 100).astype(np.float32)

    oof_pred[val_idx] = val_pred_clip
    test_pred_sum += test_pred_clip / 5.0

    row = {
        "artifact_name": artifact_name,
        "model_class": "LightGBM",
        "feature_space": "base",
        "fold": fold,
        "fold_train_n": len(tr_idx),
        "fold_valid_n": len(val_idx),
        "best_iteration": best_iter,
        "fold_valid_mse_raw": mean_squared_error(y_fold_val, val_pred_raw),
        "fold_valid_mse_clipped": mean_squared_error(y_fold_val, val_pred_clip),
        "elapsed_sec": fold_elapsed,
    }

    for key, value in selected_lgbm_params.items():
        row[key] = value

    fold_metrics = pd.concat(
        [fold_metrics, pd.DataFrame([row])],
        ignore_index=True,
    )

    # Checkpoint immediately after the fold.
    fold_metrics.to_csv(metrics_path, index=False)
    np.save(oof_npy_path, oof_pred)
    np.save(test_sum_npy_path, test_pred_sum)

    print(f"\nCompleted fold {fold}")
    print("Best iteration:", best_iter)
    print("Fold valid MSE clipped:", row["fold_valid_mse_clipped"])
    print("Elapsed seconds:", round(fold_elapsed, 2))
    print("Checkpointed metrics/oof/test sum.")

    del model
    del X_fold_tr, y_fold_tr, X_fold_val, y_fold_val
    del val_pred_raw, val_pred_clip, test_pred_raw, test_pred_clip
    gc.collect()

elapsed_all = time.time() - start_all

# ------------------------------------------------------------
# Final artifact construction
# ------------------------------------------------------------

n_missing_oof = int(np.isnan(oof_pred).sum())
print("\n" + "=" * 80)
print("OOF completion check")
print("=" * 80)
print("Missing OOF predictions:", n_missing_oof)

if n_missing_oof == 0:
    overall_oof_mse = mean_squared_error(y_full_lgb, oof_pred)
    print("Overall LightGBM OOF MSE clipped:", overall_oof_mse)

    # Training IDs are helpful for stacking if available.
    if "train_ids" in globals():
        train_id_series = pd.Series(train_ids).reset_index(drop=True)
    else:
        train_id_series = pd.Series(np.arange(n_train), name="TRAIN_ROW_ID")

    oof_df = pd.DataFrame({
        "ASSESSMENT_ID": train_id_series,
        "PERCENT_PROFICIENT_TRUE": y_full_lgb,
        f"OOF_{artifact_name}": oof_pred,
    })
    oof_df.to_csv(oof_csv_path, index=False)

    test_id_series = pd.Series(test_ids).reset_index(drop=True)
    final_test_pred = np.clip(test_pred_sum, 0, 100).astype(np.float32)

    testpred_df = pd.DataFrame({
        "ASSESSMENT_ID": test_id_series,
        f"TESTPRED_{artifact_name}": final_test_pred,
    })
    testpred_df.to_csv(testpred_csv_path, index=False)

    submission_df = pd.DataFrame({
        "ASSESSMENT_ID": test_id_series,
        "PERCENT_PROFICIENT": final_test_pred,
    })
    submission_df.to_csv(submission_path, index=False)

    print("\nSaved OOF file:", oof_csv_path)
    print("Saved test prediction file:", testpred_csv_path)
    print("Saved Kaggle submission:", submission_path)
    print("\nSubmission shape:", submission_df.shape)
    print("\nSubmission prediction summary:")
    print(submission_df["PERCENT_PROFICIENT"].describe())

    print("\nFold metrics:")
    display(
        fold_metrics.sort_values("fold")[
            [
                "fold",
                "fold_valid_mse_clipped",
                "fold_valid_mse_raw",
                "best_iteration",
                "elapsed_sec",
            ]
        ].reset_index(drop=True)
    )

    print("\nTotal elapsed seconds in this run:", round(elapsed_all, 2))

else:
    print(
        "OOF artifact is not complete yet. Re-run this same cell to resume from the latest checkpoint."
    )

# Clean only local arrays from this cell.
del X_full_lgb, X_test_lgb, y_full_lgb
gc.collect()

LightGBM version: 4.6.0
Artifact: lgbm_t03_base_5fold_oof
Metrics checkpoint: model_results/lgbm_t03_base_5fold_oof_fold_metrics.csv
Full train shape: (144921, 162)
Full test shape: (48307, 162)
Target length: 144921
Test ID length: 48307

Selected LightGBM parameters:
  objective: regression
  metric: l2
  random_state: 9890
  n_jobs: 1
  verbosity: -1
  force_col_wise: True
  n_estimators: 12000
  learning_rate: 0.03
  num_leaves: 95
  min_child_samples: 60
  subsample: 0.85
  subsample_freq: 1
  colsample_bytree: 0.9
  reg_alpha: 0.0
  reg_lambda: 5.0
  max_depth: -1

Already completed folds: []

Starting LightGBM fold 1/5
Training until validation scores don't improve for 500 rounds
[1000]	valid_0's l2: 144.55
[2000]	valid_0's l2: 129.408
[3000]	valid_0's l2: 121.318
[4000]	valid_0's l2: 115.861
[5000]	valid_0's l2: 112.045
[6000]	valid_0's l2: 109.066
[7000]	valid_0's l2: 106.881
[8000]	valid_0's l2: 105.062
[9000]	valid_0's l2: 103.675
[10000]	valid_0's l2: 102.592
[11000]	valid_

,fold,fold_valid_mse_clipped,fold_valid_mse_raw,best_iteration,elapsed_sec
0,1,100.265396,100.821338,11999,187.305308
1,2,96.495171,97.028716,11998,177.751144
2,3,98.903893,99.598858,12000,181.971037
3,4,96.720520,97.372209,11998,179.059395
4,5,101.512009,102.124080,12000,364.619435



Total elapsed seconds in this run: 3695.62


0

### LightGBM OOF Submission Result

The selected LightGBM model produced the strongest clean pipeline result so far. The 5-fold OOF artifact used the base feature space and the targeted LightGBM configuration:

- `num_leaves = 95`
- `min_child_samples = 60`
- `learning_rate = 0.03`
- `subsample = 0.85`
- `colsample_bytree = 0.90`
- `reg_lambda = 5.0`
- `n_estimators = 12000`

The completed 5-fold OOF run gave an overall clipped OOF MSE of approximately **98.78**. The resulting fold-averaged Kaggle submission file was:

`submission_lgbm_t03_base_5fold_oof_foldavg.csv`

This submission achieved a new public leaderboard MSE of **90.644**, which is the best result from the clean current notebook pipeline so far.

This confirms that gradient boosting is currently the strongest model family for this problem. The result also validates the decision to move from bagging/tree ensembles into LightGBM while preserving earlier Random Forest and ExtraTrees OOF artifacts for possible blending or stacking.

One important observation is that all five LightGBM folds reached the maximum estimator limit of `12000` without early stopping. This suggests the model may still benefit from a longer selected OOF run with a higher tree cap. However, before rerunning a longer LightGBM OOF model, the next low-risk step is to run the saved OOF blend screen using Random Forest, ExtraTrees, and LightGBM. This requires no model refitting and may improve the submission by combining complementary model errors.

In [39]:
# ============================================================
# 22A. Three-model OOF blend screen:
# Random Forest + ExtraTrees + LightGBM
#
# No model fitting in this cell.
# Reads saved OOF/test prediction artifacts and creates
# a convex OOF-weighted blend submission.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

RESULTS_DIR = Path("model_results")
assert RESULTS_DIR.exists(), "model_results folder was not found."

# ------------------------------------------------------------
# Locate required saved artifacts
# ------------------------------------------------------------

artifact_paths = {
    "rf_oof": RESULTS_DIR / "oof_rf_500_base.csv",
    "rf_test": RESULTS_DIR / "testpred_rf_500_base_folds.csv",
    "et_oof": RESULTS_DIR / "oof_extratrees_safe_base.csv",
    "et_test": RESULTS_DIR / "testpred_extratrees_safe_base_foldavg.csv",
    "lgbm_oof": RESULTS_DIR / "oof_lgbm_t03_base_5fold_oof.csv",
    "lgbm_test": RESULTS_DIR / "testpred_lgbm_t03_base_5fold_oof_foldavg.csv",
}

for label, path in artifact_paths.items():
    assert path.exists(), f"Missing {label}: {path}"

print("Using saved artifacts:")
for label, path in artifact_paths.items():
    print(f"  {label}: {path}")

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def read_pred_csv(path):
    return pd.read_csv(path, dtype={"ASSESSMENT_ID": str})


def one_dim_id_array(ids):
    arr = np.asarray(ids)
    if arr.ndim > 1:
        arr = arr.ravel()
    return pd.Series(arr).astype(str).to_numpy()


def first_existing_column(df, candidates, label):
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(
        f"Could not find {label}. "
        f"Tried {candidates}. Available columns: {list(df.columns)}"
    )


def pick_prediction_column(df, preferred_cols, label, target_cols=None):
    if target_cols is None:
        target_cols = []

    for col in preferred_cols:
        if col in df.columns:
            return col

    excluded_cols = set(["ASSESSMENT_ID"] + target_cols)

    numeric_cols = [
        c for c in df.columns
        if c not in excluded_cols and pd.api.types.is_numeric_dtype(df[c])
    ]

    pred_like_cols = [
        c for c in numeric_cols
        if (
            "pred" in c.lower()
            or "proficient" in c.lower()
            or "oof" in c.lower()
            or "foldavg" in c.lower()
            or "fold_avg" in c.lower()
            or "average" in c.lower()
            or "avg" in c.lower()
        )
    ]

    priority_terms = [
        "foldavg",
        "fold_avg",
        "testpred",
        "test_pred",
        "percent_proficient",
        "oof",
        "pred",
        "avg",
    ]

    for term in priority_terms:
        matches = [c for c in pred_like_cols if term in c.lower()]
        if len(matches) == 1:
            return matches[0]

    if len(pred_like_cols) == 1:
        return pred_like_cols[0]

    if len(numeric_cols) == 1:
        return numeric_cols[0]

    raise ValueError(
        f"Could not identify prediction column for {label}.\n"
        f"Columns available: {list(df.columns)}\n"
        f"Numeric columns found: {numeric_cols}\n"
        f"Prediction-like columns found: {pred_like_cols}"
    )


def standardize_oof(path, model_name, preferred_pred_cols):
    df = read_pred_csv(path)

    assert "ASSESSMENT_ID" in df.columns, f"{model_name} OOF missing ASSESSMENT_ID."

    y_col = first_existing_column(
        df,
        [
            "y_true",
            "PERCENT_PROFICIENT_TRUE",
            "target",
            "true",
            "actual",
        ],
        f"{model_name} OOF target column"
    )

    pred_col = pick_prediction_column(
        df,
        preferred_cols=preferred_pred_cols,
        label=f"{model_name} OOF prediction",
        target_cols=[y_col],
    )

    print(f"{model_name.upper()} OOF target column: {y_col}")
    print(f"{model_name.upper()} OOF prediction column: {pred_col}")

    out = df[["ASSESSMENT_ID", y_col, pred_col]].copy()
    out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)
    out = out.rename(columns={
        y_col: f"{model_name}_y_true",
        pred_col: f"{model_name}_oof_pred",
    })

    return out


def standardize_testpred(path, model_name, preferred_pred_cols):
    df = read_pred_csv(path)

    assert "ASSESSMENT_ID" in df.columns, f"{model_name} test predictions missing ASSESSMENT_ID."

    pred_col = pick_prediction_column(
        df,
        preferred_cols=preferred_pred_cols,
        label=f"{model_name} test prediction",
        target_cols=[],
    )

    print(f"{model_name.upper()} test prediction column: {pred_col}")

    out = df[["ASSESSMENT_ID", pred_col]].copy()
    out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)
    out = out.rename(columns={pred_col: f"{model_name}_test_pred"})

    return out


# ------------------------------------------------------------
# Load and standardize OOF artifacts
# ------------------------------------------------------------

rf_oof = standardize_oof(
    artifact_paths["rf_oof"],
    "rf",
    preferred_pred_cols=[
        "rf_500_base_oof_pred",
        "oof_pred",
        "pred",
        "prediction",
    ],
)

et_oof = standardize_oof(
    artifact_paths["et_oof"],
    "et",
    preferred_pred_cols=[
        "oof_pred",          # raw ExtraTrees OOF, preferred for blending
        "oof_pred_clipped",
        "et_oof_pred",
        "pred",
        "prediction",
    ],
)

lgbm_oof = standardize_oof(
    artifact_paths["lgbm_oof"],
    "lgbm",
    preferred_pred_cols=[
        "OOF_lgbm_t03_base_5fold_oof",
        "oof_pred",
        "lgbm_oof_pred",
        "pred",
        "prediction",
    ],
)

for name, df in [("rf", rf_oof), ("et", et_oof), ("lgbm", lgbm_oof)]:
    assert df["ASSESSMENT_ID"].duplicated().sum() == 0, f"Duplicate ASSESSMENT_ID in {name} OOF."

oof_blend_df = (
    rf_oof
    .merge(et_oof, on="ASSESSMENT_ID", how="inner")
    .merge(lgbm_oof, on="ASSESSMENT_ID", how="inner")
)

assert len(oof_blend_df) == len(rf_oof) == len(et_oof) == len(lgbm_oof), "OOF merge lost rows."

# Check target alignment
assert np.allclose(
    oof_blend_df["rf_y_true"].to_numpy(dtype=float),
    oof_blend_df["et_y_true"].to_numpy(dtype=float),
), "RF and ExtraTrees targets do not match."

assert np.allclose(
    oof_blend_df["rf_y_true"].to_numpy(dtype=float),
    oof_blend_df["lgbm_y_true"].to_numpy(dtype=float),
), "RF and LightGBM targets do not match."

oof_blend_df["y_true"] = oof_blend_df["rf_y_true"].astype(float)

y_oof = oof_blend_df["y_true"].to_numpy(dtype=float)

rf_pred = oof_blend_df["rf_oof_pred"].to_numpy(dtype=float)
et_pred = oof_blend_df["et_oof_pred"].to_numpy(dtype=float)
lgbm_pred = oof_blend_df["lgbm_oof_pred"].to_numpy(dtype=float)

component_rows = []
for model_name, pred in [
    ("rf", rf_pred),
    ("et", et_pred),
    ("lgbm", lgbm_pred),
]:
    component_rows.append({
        "model": model_name,
        "oof_mse_raw": mean_squared_error(y_oof, pred),
        "oof_mse_clipped": mean_squared_error(y_oof, np.clip(pred, 0, 100)),
        "pred_mean": np.mean(np.clip(pred, 0, 100)),
        "pred_std": np.std(np.clip(pred, 0, 100)),
    })

component_results = pd.DataFrame(component_rows).sort_values("oof_mse_clipped")

print("\nComponent OOF results:")
display(component_results.reset_index(drop=True))

print("\nResidual correlation matrix, using clipped predictions:")
resid_df = pd.DataFrame({
    "rf_resid": y_oof - np.clip(rf_pred, 0, 100),
    "et_resid": y_oof - np.clip(et_pred, 0, 100),
    "lgbm_resid": y_oof - np.clip(lgbm_pred, 0, 100),
})
display(resid_df.corr())

lgbm_oof_mse = float(
    component_results.loc[
        component_results["model"] == "lgbm",
        "oof_mse_clipped"
    ].iloc[0]
)

# ------------------------------------------------------------
# Build candidate convex weights
# ------------------------------------------------------------

candidate_rows = []
seen_weights = set()

def add_weight(w_rf, w_et, w_lgbm, source):
    weights = np.array([w_rf, w_et, w_lgbm], dtype=float)

    if np.any(weights < -1e-9):
        return

    weights[np.abs(weights) < 1e-12] = 0.0

    total = weights.sum()
    if total <= 0:
        return

    weights = weights / total

    key = tuple(np.round(weights, 6))
    if key in seen_weights:
        return

    seen_weights.add(key)

    candidate_rows.append({
        "w_rf": weights[0],
        "w_et": weights[1],
        "w_lgbm": weights[2],
        "source": source,
    })


# Pure models
add_weight(1.0, 0.0, 0.0, "pure_rf")
add_weight(0.0, 1.0, 0.0, "pure_et")
add_weight(0.0, 0.0, 1.0, "pure_lgbm")

# Fine pairwise grids
pair_grid = np.linspace(0, 1, 1001)

for w_lgbm in pair_grid:
    add_weight(0.0, 1.0 - w_lgbm, w_lgbm, "pair_et_lgbm")

for w_lgbm in pair_grid:
    add_weight(1.0 - w_lgbm, 0.0, w_lgbm, "pair_rf_lgbm")

for w_et in pair_grid:
    add_weight(1.0 - w_et, w_et, 0.0, "pair_rf_et")

# Coarse three-way grid focused around LightGBM dominance.
# Since LightGBM is much stronger OOF than RF/ET, we do not waste
# many candidates with tiny LightGBM weights.
step = 0.01

for w_lgbm in np.arange(0.60, 1.0 + step / 2, step):
    remainder = 1.0 - w_lgbm
    n_steps = int(round(remainder / step))

    for k in range(n_steps + 1):
        w_rf = k * step
        w_et = remainder - w_rf
        add_weight(w_rf, w_et, w_lgbm, "three_way_lgbm_focused_coarse")

candidate_df = pd.DataFrame(candidate_rows)

print("\nInitial candidate weight count:", len(candidate_df))

# ------------------------------------------------------------
# Evaluate candidate weights
# ------------------------------------------------------------

pred_matrix = np.vstack([
    rf_pred,
    et_pred,
    lgbm_pred,
]).astype(np.float32)

def evaluate_weight_df(weight_df):
    rows = []

    for row in weight_df.itertuples(index=False):
        weights = np.array([row.w_rf, row.w_et, row.w_lgbm], dtype=np.float32)

        blend_raw = (
            weights[0] * pred_matrix[0]
            + weights[1] * pred_matrix[1]
            + weights[2] * pred_matrix[2]
        )

        blend_clipped = np.clip(blend_raw, 0, 100)

        rows.append({
            "w_rf": float(weights[0]),
            "w_et": float(weights[1]),
            "w_lgbm": float(weights[2]),
            "source": row.source,
            "oof_mse_raw": mean_squared_error(y_oof, blend_raw),
            "oof_mse_clipped": mean_squared_error(y_oof, blend_clipped),
        })

    return (
        pd.DataFrame(rows)
        .sort_values("oof_mse_clipped")
        .reset_index(drop=True)
    )

blend_weight_results_initial = evaluate_weight_df(candidate_df)
best_initial = blend_weight_results_initial.iloc[0]

print("\nBest initial blend:")
display(pd.DataFrame([best_initial]))

# Local fine search around the best initial weights.
local_step = 0.002
window = 0.04

center_rf = float(best_initial["w_rf"])
center_et = float(best_initial["w_et"])
center_lgbm = float(best_initial["w_lgbm"])

for w_rf in np.arange(
    max(0.0, center_rf - window),
    min(1.0, center_rf + window) + local_step / 2,
    local_step,
):
    for w_et in np.arange(
        max(0.0, center_et - window),
        min(1.0, center_et + window) + local_step / 2,
        local_step,
    ):
        w_lgbm = 1.0 - w_rf - w_et
        add_weight(w_rf, w_et, w_lgbm, "local_fine_around_best")

candidate_df = pd.DataFrame(candidate_rows)

print("Final candidate weight count after local fine search:", len(candidate_df))

blend_weight_results = evaluate_weight_df(candidate_df)
best_blend = blend_weight_results.iloc[0].to_dict()

best_w_rf = float(best_blend["w_rf"])
best_w_et = float(best_blend["w_et"])
best_w_lgbm = float(best_blend["w_lgbm"])
best_oof_mse = float(best_blend["oof_mse_clipped"])

print("\nFinal best OOF blend:")
display(pd.DataFrame([best_blend]))

print("\nTop 15 blend weights:")
display(blend_weight_results.head(15))

blend_gain_vs_lgbm = lgbm_oof_mse - best_oof_mse

print("\nBlend gain versus pure LightGBM OOF:")
print("Pure LightGBM OOF MSE:", lgbm_oof_mse)
print("Best blend OOF MSE:   ", best_oof_mse)
print("OOF MSE gain:         ", blend_gain_vs_lgbm)

# ------------------------------------------------------------
# Save OOF blend artifacts
# ------------------------------------------------------------

oof_blend_df["blend_pred_raw"] = (
    best_w_rf * rf_pred
    + best_w_et * et_pred
    + best_w_lgbm * lgbm_pred
)

oof_blend_df["blend_pred_clipped"] = np.clip(
    oof_blend_df["blend_pred_raw"],
    0,
    100,
)

weight_results_path = RESULTS_DIR / "blend_rf_et_lgbm_weight_screen.csv"
oof_blend_path = RESULTS_DIR / "oof_blend_rf_et_lgbm_weighted.csv"

blend_weight_results.to_csv(weight_results_path, index=False)
oof_blend_df.to_csv(oof_blend_path, index=False)

# ------------------------------------------------------------
# Load, standardize, and align test predictions
# ------------------------------------------------------------

rf_test = standardize_testpred(
    artifact_paths["rf_test"],
    "rf",
    preferred_pred_cols=[
        "rf_500_base_foldavg_pred",
        "rf_500_base_test_pred",
        "PERCENT_PROFICIENT",
        "test_pred",
        "pred",
        "prediction",
    ],
)

et_test = standardize_testpred(
    artifact_paths["et_test"],
    "et",
    preferred_pred_cols=[
        "PERCENT_PROFICIENT",
        "et_test_pred",
        "test_pred_avg",
        "test_pred",
        "pred",
        "prediction",
    ],
)

lgbm_test = standardize_testpred(
    artifact_paths["lgbm_test"],
    "lgbm",
    preferred_pred_cols=[
        "TESTPRED_lgbm_t03_base_5fold_oof",
        "PERCENT_PROFICIENT",
        "lgbm_test_pred",
        "test_pred",
        "pred",
        "prediction",
    ],
)

for name, df in [("rf", rf_test), ("et", et_test), ("lgbm", lgbm_test)]:
    assert df["ASSESSMENT_ID"].duplicated().sum() == 0, f"Duplicate ASSESSMENT_ID in {name} test predictions."

test_blend_df = (
    rf_test
    .merge(et_test, on="ASSESSMENT_ID", how="inner")
    .merge(lgbm_test, on="ASSESSMENT_ID", how="inner")
)

assert len(test_blend_df) == len(rf_test) == len(et_test) == len(lgbm_test), "Test prediction merge lost rows."

# Restore original test_ids order when available.
if "test_ids" in globals():
    test_order = pd.DataFrame({
        "ASSESSMENT_ID": one_dim_id_array(test_ids),
        "_test_order": np.arange(len(test_ids)),
    })

    test_blend_df = test_order.merge(test_blend_df, on="ASSESSMENT_ID", how="left")

    assert test_blend_df["rf_test_pred"].isna().sum() == 0, "Missing RF predictions after test_ids alignment."
    assert test_blend_df["et_test_pred"].isna().sum() == 0, "Missing ExtraTrees predictions after test_ids alignment."
    assert test_blend_df["lgbm_test_pred"].isna().sum() == 0, "Missing LightGBM predictions after test_ids alignment."

    test_blend_df = (
        test_blend_df
        .sort_values("_test_order")
        .drop(columns=["_test_order"])
        .reset_index(drop=True)
    )

test_blend_df["PERCENT_PROFICIENT"] = np.clip(
    best_w_rf * test_blend_df["rf_test_pred"].to_numpy(dtype=float)
    + best_w_et * test_blend_df["et_test_pred"].to_numpy(dtype=float)
    + best_w_lgbm * test_blend_df["lgbm_test_pred"].to_numpy(dtype=float),
    0,
    100,
)

test_blend_path = RESULTS_DIR / "testpred_blend_rf_et_lgbm_weighted.csv"
submission_blend_path = Path("submission_blend_rf_et_lgbm_oof_weighted.csv")

test_blend_df.to_csv(test_blend_path, index=False)

submission_blend = test_blend_df[["ASSESSMENT_ID", "PERCENT_PROFICIENT"]].copy()

expected_test_n = len(test_ids) if "test_ids" in globals() else 48307

assert submission_blend.shape[0] == expected_test_n, f"Submission row count is not {expected_test_n}."
assert list(submission_blend.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
assert submission_blend["ASSESSMENT_ID"].isna().sum() == 0
assert submission_blend["PERCENT_PROFICIENT"].isna().sum() == 0

submission_blend.to_csv(submission_blend_path, index=False)

# ------------------------------------------------------------
# Update notebook trackers
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["LightGBM T03 base OOF"] = {
    "model_class": "LightGBM",
    "feature_space": "base",
    "oof_mse": lgbm_oof_mse,
    "submission_path": "submission_lgbm_t03_base_5fold_oof_foldavg.csv",
    "notes": "Selected LightGBM OOF artifact from targeted holdout refinement."
}

best_models_by_class["RF + ExtraTrees + LightGBM OOF blend"] = {
    "model_class": "OOF weighted blend",
    "component_models": "RF 500 base + ExtraTrees safe base + LightGBM T03 base",
    "feature_space": "base",
    "oof_mse": best_oof_mse,
    "oof_mse_raw": float(best_blend["oof_mse_raw"]),
    "w_rf": best_w_rf,
    "w_et": best_w_et,
    "w_lgbm": best_w_lgbm,
    "submission_path": str(submission_blend_path),
    "notes": "Convex OOF-weighted blend of saved RF, ExtraTrees, and LightGBM artifacts."
}

tracker_path = RESULTS_DIR / "submission_tracker.csv"

if "submission_tracker" in globals():
    tracker_base = submission_tracker.copy()
elif tracker_path.exists():
    tracker_base = pd.read_csv(tracker_path)
else:
    tracker_base = pd.DataFrame()

tracker_new_rows = pd.DataFrame([
    {
        "submission_file": "submission_lgbm_t03_base_5fold_oof_foldavg.csv",
        "source": "current notebook",
        "model_family": "LightGBM",
        "public_mse": np.nan,
        "notes": f"Selected LightGBM OOF artifact; OOF MSE={lgbm_oof_mse:.6f}.",
    },
    {
        "submission_file": submission_blend_path.name,
        "source": "current notebook",
        "model_family": "RF + ExtraTrees + LightGBM blend",
        "public_mse": np.nan,
        "notes": (
            f"OOF-weighted blend; "
            f"w_rf={best_w_rf:.3f}, "
            f"w_et={best_w_et:.3f}, "
            f"w_lgbm={best_w_lgbm:.3f}, "
            f"OOF MSE={best_oof_mse:.6f}."
        ),
    },
])

if len(tracker_base) > 0:
    submission_tracker = pd.concat(
        [tracker_base, tracker_new_rows],
        ignore_index=True,
    )
else:
    submission_tracker = tracker_new_rows.copy()

submission_tracker = submission_tracker.drop_duplicates(
    subset=["submission_file"],
    keep="last",
)

submission_tracker.to_csv(tracker_path, index=False)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\nSaved files:")
print("Weight screen:", weight_results_path.resolve())
print("OOF blend:", oof_blend_path.resolve())
print("Test blend predictions:", test_blend_path.resolve())
print("Blend submission:", submission_blend_path.resolve())
print("Updated submission tracker:", tracker_path.resolve())

print("\nBlend submission prediction summary:")
print(submission_blend["PERCENT_PROFICIENT"].describe().to_string())

print("\nSubmission shape:", submission_blend.shape)

print("\nFirst 5 submission rows:")
print(submission_blend.head().to_string(index=False))

print("\nDecision note:")
if best_w_lgbm >= 0.995:
    print("Best blend is effectively pure LightGBM. Submit/keep the pure LightGBM file first.")
elif blend_gain_vs_lgbm >= 0.25:
    print("Blend improves OOF over pure LightGBM by at least 0.25 MSE. This blend is worth a Kaggle submission after pure LightGBM.")
elif blend_gain_vs_lgbm > 0:
    print("Blend improves OOF only modestly. Save it, but prioritize pure LightGBM unless you have extra submission slots.")
else:
    print("Blend does not improve OOF over pure LightGBM. Prefer the pure LightGBM submission.")

Using saved artifacts:
  rf_oof: model_results/oof_rf_500_base.csv
  rf_test: model_results/testpred_rf_500_base_folds.csv
  et_oof: model_results/oof_extratrees_safe_base.csv
  et_test: model_results/testpred_extratrees_safe_base_foldavg.csv
  lgbm_oof: model_results/oof_lgbm_t03_base_5fold_oof.csv
  lgbm_test: model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv
RF OOF target column: y_true
RF OOF prediction column: rf_500_base_oof_pred
ET OOF target column: y_true
ET OOF prediction column: oof_pred
LGBM OOF target column: PERCENT_PROFICIENT_TRUE
LGBM OOF prediction column: OOF_lgbm_t03_base_5fold_oof

Component OOF results:


,model,oof_mse_raw,oof_mse_clipped,pred_mean,pred_std
0,lgbm,98.779406,98.779406,54.152808,24.283770
1,et,110.749294,110.749294,54.127367,23.857745
2,rf,122.328024,122.328024,54.134143,22.839348



Residual correlation matrix, using clipped predictions:


,rf_resid,et_resid,lgbm_resid
rf_resid,1.000000,0.929709,0.842448
et_resid,0.929709,1.000000,0.802689
lgbm_resid,0.842448,0.802689,1.000000



Initial candidate weight count: 3780

Best initial blend:


...

Final candidate weight count after local fine search: 4574

Final best OOF blend:


,w_rf,w_et,w_lgbm,source,oof_mse_raw,oof_mse_clipped
0,0.0,0.356,0.644,pair_et_lgbm,93.499458,93.499458



Top 15 blend weights:


,w_rf,w_et,w_lgbm,source,oof_mse_raw,oof_mse_clipped
0,0.0,0.356,0.644,pair_et_lgbm,93.499458,93.499458
1,0.0,0.357,0.643,pair_et_lgbm,93.499484,93.499484
2,0.0,0.355,0.645,pair_et_lgbm,93.499516,93.499516
3,0.0,0.358,0.642,pair_et_lgbm,93.499592,93.499592
4,0.0,0.354,0.646,pair_et_lgbm,93.499655,93.499655
5,0.0,0.359,0.641,pair_et_lgbm,93.499786,93.499786
6,0.0,0.353,0.647,pair_et_lgbm,93.499879,93.499879
7,0.0,0.360,0.640,pair_et_lgbm,93.500061,93.500061
8,0.0,0.352,0.648,pair_et_lgbm,93.500187,93.500187
9,0.0,0.361,0.639,pair_et_lgbm,93.500420,93.500420



Blend gain versus pure LightGBM OOF:
Pure LightGBM OOF MSE: 98.77940606294962
Best blend OOF MSE:    93.49945771419596
OOF MSE gain:          5.279948348753663
RF test prediction column: rf_500_base_foldavg_pred
ET test prediction column: PERCENT_PROFICIENT
LGBM test prediction column: TESTPRED_lgbm_t03_base_5fold_oof

Saved files:
Weight screen: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/blend_rf_et_lgbm_weight_screen.csv
OOF blend: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/oof_blend_rf_et_lgbm_weighted.csv
Test blend predictions: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/testpred_blend_rf_et_lgbm_weighted.csv
Blend submission: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/submission_blend_rf_et_lgbm

### RF + ExtraTrees + LightGBM OOF Blend Result

The saved OOF blend screen compared the current Random Forest, ExtraTrees, and LightGBM artifacts without refitting any models.

Component OOF results:

- Random Forest OOF MSE: **122.33**
- ExtraTrees OOF MSE: **110.75**
- LightGBM OOF MSE: **98.78**

The best convex blend excluded Random Forest and used:

- `w_rf = 0.000`
- `w_et = 0.356`
- `w_lgbm = 0.644`

This produced a blended OOF MSE of approximately **93.50**, improving over pure LightGBM by about **5.28 MSE**.

The resulting blend submission file is:

`submission_blend_rf_et_lgbm_oof_weighted.csv`

Interpretation: although ExtraTrees is weaker than LightGBM as a standalone model, its residuals are not identical to LightGBM's residuals, so it contributes useful complementary signal. Random Forest does not add value in this convex blend once ExtraTrees and LightGBM are included.

The pure LightGBM OOF submission already achieved a public leaderboard MSE of **90.644**, the best clean current-notebook result so far. Because the blend improves OOF substantially, the blend submission is worth a Kaggle submission before starting the next longer LightGBM run.

Next modeling step: run a longer selected LightGBM OOF artifact using the same successful configuration but a higher tree cap, since all five folds reached the previous `12000` estimator limit without early stopping.

### RF + ExtraTrees + LightGBM Blend Submission Result

The saved OOF blend screen combined the existing Random Forest, ExtraTrees, and LightGBM OOF artifacts without refitting any models.

Standalone OOF results:

- Random Forest OOF MSE: **122.33**
- ExtraTrees OOF MSE: **110.75**
- LightGBM OOF MSE: **98.78**

The best convex blend assigned zero weight to Random Forest and used:

- `w_rf = 0.000`
- `w_et = 0.356`
- `w_lgbm = 0.644`

This produced a blended OOF MSE of approximately **93.50**, improving over pure LightGBM by about **5.28 OOF MSE**.

The blend submission file was:

`submission_blend_rf_et_lgbm_oof_weighted.csv`

Kaggle public leaderboard result:

- Pure selected LightGBM OOF submission: **90.644**
- RF + ExtraTrees + LightGBM OOF blend submission: **87.054**

This is the strongest clean current-notebook result so far. 

Interpretation: ExtraTrees is weaker than LightGBM as a standalone model, but it adds complementary signal when blended with LightGBM. Random Forest does not add marginal value in this blend after ExtraTrees and LightGBM are included.

Next modeling decision: because the 12k LightGBM OOF run reached the estimator cap on every fold and still produced the best single-model public score so far, the next step is a longer selected LightGBM OOF run. After that finishes, we should rebuild the blend screen using ExtraTrees, the original 12k LightGBM, and the new longer LightGBM artifacts.

In [40]:
# ============================================================
# 23A. Overnight selected LightGBM OOF suite + automatic re-blend
#
# Why this replaces the earlier 25k cell:
# - Pure 12k LGBM scored 90.644 public MSE.
# - ET + LGBM blend scored 87.054 public MSE.
# - All 12k LGBM folds reached the estimator cap.
# - So overnight time is better used by running longer selected
#   LightGBM artifacts and then re-blending automatically.
#
# Runs:
#   1. T03 long80k, lr=0.03
#   2. T03 long100k, lr=0.02
#
# Conservative safeguards:
# - n_jobs=1
# - one fold checkpoint at a time
# - per-fold test prediction files
# - resume-safe
# - no parameter grid explosion
# ============================================================

from pathlib import Path
import time
import gc
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

print("LightGBM version:", lgb.__version__)
print("Results folder:", RESULTS_DIR.resolve())

# ------------------------------------------------------------
# Update tracker with known public scores
# ------------------------------------------------------------

tracker_path = RESULTS_DIR / "submission_tracker.csv"

if tracker_path.exists():
    submission_tracker = pd.read_csv(tracker_path)
elif "submission_tracker" in globals():
    submission_tracker = submission_tracker.copy()
else:
    submission_tracker = pd.DataFrame(
        columns=[
            "submission_file",
            "source",
            "model_family",
            "public_mse",
            "notes",
        ]
    )

tracker_updates = pd.DataFrame([
    {
        "submission_file": "submission_lgbm_t03_base_5fold_oof_foldavg.csv",
        "source": "current notebook",
        "model_family": "LightGBM",
        "public_mse": 90.644,
        "notes": "Selected 12k LightGBM OOF/fold-average submission.",
    },
    {
        "submission_file": "submission_blend_rf_et_lgbm_oof_weighted.csv",
        "source": "current notebook",
        "model_family": "ExtraTrees + LightGBM blend",
        "public_mse": 87.054,
        "notes": "OOF blend: w_rf=0.000, w_et=0.356, w_lgbm=0.644; OOF MSE=93.499458.",
    },
])

submission_tracker = pd.concat(
    [submission_tracker, tracker_updates],
    ignore_index=True
)

submission_tracker = submission_tracker.drop_duplicates(
    subset=["submission_file"],
    keep="last"
)

submission_tracker.to_csv(tracker_path, index=False)

print("\nUpdated submission tracker with latest public scores.")
print("Tracker path:", tracker_path)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

required_objects = [
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]
assert len(missing_objects) == 0, f"Missing required objects: {missing_objects}"

assert X_train_proc_model.shape[0] == len(y_train), (
    X_train_proc_model.shape,
    len(y_train),
)

assert X_test_proc_model.shape[0] == len(test_ids), (
    X_test_proc_model.shape,
    len(test_ids),
)

print("\nFull train shape:", X_train_proc_model.shape)
print("Full test shape:", X_test_proc_model.shape)
print("Target length:", len(y_train))
print("Test ID length:", len(test_ids))

def as_1d_str_series(x):
    return pd.Series(np.asarray(x).ravel()).astype(str)

# Convert once to memory-conscious contiguous arrays.
X_full_lgb = np.ascontiguousarray(np.asarray(X_train_proc_model, dtype=np.float32))
X_test_lgb = np.ascontiguousarray(np.asarray(X_test_proc_model, dtype=np.float32))
y_full_lgb = np.asarray(y_train, dtype=np.float32).ravel()

n_train = X_full_lgb.shape[0]
n_test = X_test_lgb.shape[0]

# ------------------------------------------------------------
# Overnight LightGBM jobs
# ------------------------------------------------------------

base_lgbm_params = dict(
    objective="regression",
    metric="l2",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=-1,
    force_col_wise=True,

    # Selected T03 structure
    num_leaves=95,
    min_child_samples=60,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.90,
    reg_alpha=0.0,
    reg_lambda=5.0,
    max_depth=-1,
)

overnight_jobs = [
    {
        "artifact_name": "lgbm_t03_base_5fold_oof_long80k_lr03",
        "learning_rate": 0.03,
        "n_estimators": 80000,
        "stopping_rounds": 2500,
        "log_period": 2000,
    },
    {
        "artifact_name": "lgbm_t03_base_5fold_oof_long100k_lr02",
        "learning_rate": 0.02,
        "n_estimators": 100000,
        "stopping_rounds": 3000,
        "log_period": 2500,
    },
]

print("\nPlanned overnight jobs:")
for job in overnight_jobs:
    print(
        f"  {job['artifact_name']}: "
        f"lr={job['learning_rate']}, "
        f"n_estimators={job['n_estimators']}, "
        f"early_stopping={job['stopping_rounds']}"
    )

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

completed_lgbm_artifacts = []

# ------------------------------------------------------------
# Function: run one selected LightGBM OOF artifact
# ------------------------------------------------------------

def run_lgbm_oof_artifact(job):
    artifact_name = job["artifact_name"]

    metrics_path = RESULTS_DIR / f"{artifact_name}_fold_metrics.csv"
    oof_npy_path = RESULTS_DIR / f"{artifact_name}_oof.npy"

    oof_csv_path = RESULTS_DIR / f"oof_{artifact_name}.csv"
    testpred_csv_path = RESULTS_DIR / f"testpred_{artifact_name}_foldavg.csv"
    submission_path = Path(f"submission_{artifact_name}_foldavg.csv")

    print("\n" + "#" * 90)
    print("Starting artifact:", artifact_name)
    print("#" * 90)

    params = base_lgbm_params.copy()
    params.update(
        dict(
            n_estimators=job["n_estimators"],
            learning_rate=job["learning_rate"],
        )
    )

    print("\nLightGBM parameters:")
    for k, v in params.items():
        print(f"  {k}: {v}")

    if metrics_path.exists():
        fold_metrics = pd.read_csv(metrics_path)
    else:
        fold_metrics = pd.DataFrame()

    if oof_npy_path.exists():
        oof_pred = np.load(oof_npy_path)
        assert len(oof_pred) == n_train
    else:
        oof_pred = np.full(n_train, np.nan, dtype=np.float32)

    artifact_start = time.time()

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_full_lgb), start=1):

        fold_test_pred_path = RESULTS_DIR / f"{artifact_name}_fold{fold}_test_pred.npy"

        completed_metric_folds = (
            set(fold_metrics["fold"].astype(int).tolist())
            if len(fold_metrics) > 0 and "fold" in fold_metrics.columns
            else set()
        )

        fold_done_in_metrics = fold in completed_metric_folds
        fold_oof_done = np.isfinite(oof_pred[val_idx]).all()
        fold_test_done = fold_test_pred_path.exists()

        if fold_done_in_metrics and fold_oof_done and fold_test_done:
            print(f"\nSkipping already completed fold {fold} for {artifact_name}")
            continue

        if fold_done_in_metrics or fold_oof_done or fold_test_done:
            print(f"\nFold {fold} has incomplete checkpoint state. Rerunning fold cleanly.")

            if len(fold_metrics) > 0 and "fold" in fold_metrics.columns:
                fold_metrics = fold_metrics[fold_metrics["fold"].astype(int) != fold].copy()
                fold_metrics.to_csv(metrics_path, index=False)

            if fold_test_pred_path.exists():
                fold_test_pred_path.unlink()

            oof_pred[val_idx] = np.nan
            np.save(oof_npy_path, oof_pred)

        print("\n" + "=" * 80)
        print(f"Starting {artifact_name}, fold {fold}/5")
        print("=" * 80)

        X_fold_tr = X_full_lgb[tr_idx]
        y_fold_tr = y_full_lgb[tr_idx]
        X_fold_val = X_full_lgb[val_idx]
        y_fold_val = y_full_lgb[val_idx]

        fold_params = params.copy()
        fold_params["random_state"] = RANDOM_STATE + fold

        model = lgb.LGBMRegressor(**fold_params)

        fold_start = time.time()

        model.fit(
            X_fold_tr,
            y_fold_tr,
            eval_set=[(X_fold_val, y_fold_val)],
            eval_metric="l2",
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=job["stopping_rounds"],
                    verbose=True,
                ),
                lgb.log_evaluation(period=job["log_period"]),
            ],
        )

        fold_elapsed = time.time() - fold_start

        best_iter = getattr(model, "best_iteration_", None)
        if best_iter is None or best_iter <= 0:
            best_iter = params["n_estimators"]

        val_pred_raw = model.predict(X_fold_val, num_iteration=best_iter)
        val_pred_clip = np.clip(val_pred_raw, 0, 100).astype(np.float32)

        test_pred_raw = model.predict(X_test_lgb, num_iteration=best_iter)
        test_pred_clip = np.clip(test_pred_raw, 0, 100).astype(np.float32)

        oof_pred[val_idx] = val_pred_clip
        np.save(oof_npy_path, oof_pred)
        np.save(fold_test_pred_path, test_pred_clip)

        row = {
            "artifact_name": artifact_name,
            "model_class": "LightGBM",
            "feature_space": "base",
            "fold": fold,
            "fold_train_n": len(tr_idx),
            "fold_valid_n": len(val_idx),
            "best_iteration": best_iter,
            "fold_valid_mse_raw": mean_squared_error(y_fold_val, val_pred_raw),
            "fold_valid_mse_clipped": mean_squared_error(y_fold_val, val_pred_clip),
            "elapsed_sec": fold_elapsed,
        }

        for key, value in params.items():
            row[key] = value

        fold_metrics = pd.concat(
            [fold_metrics, pd.DataFrame([row])],
            ignore_index=True,
        )

        fold_metrics = fold_metrics.sort_values("fold").reset_index(drop=True)
        fold_metrics.to_csv(metrics_path, index=False)

        print(f"\nCompleted {artifact_name}, fold {fold}")
        print("Best iteration:", best_iter)
        print("Fold valid MSE clipped:", row["fold_valid_mse_clipped"])
        print("Elapsed seconds:", round(fold_elapsed, 2))
        print("Saved metrics checkpoint:", metrics_path)
        print("Saved OOF checkpoint:", oof_npy_path)
        print("Saved fold test prediction:", fold_test_pred_path)

        del model
        del X_fold_tr, y_fold_tr, X_fold_val, y_fold_val
        del val_pred_raw, val_pred_clip, test_pred_raw, test_pred_clip
        gc.collect()

    # --------------------------------------------------------
    # Finalize artifact
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("Completion check for:", artifact_name)
    print("=" * 80)

    n_missing_oof = int(np.isnan(oof_pred).sum())
    print("Missing OOF predictions:", n_missing_oof)

    fold_test_preds = []
    missing_fold_test_files = []

    for fold in range(1, 6):
        fold_test_pred_path = RESULTS_DIR / f"{artifact_name}_fold{fold}_test_pred.npy"

        if not fold_test_pred_path.exists():
            missing_fold_test_files.append(str(fold_test_pred_path))
            continue

        arr = np.load(fold_test_pred_path)
        assert len(arr) == n_test, f"Bad test prediction length for fold {fold}: {len(arr)}"
        fold_test_preds.append(arr.astype(np.float32))

    print("Missing fold test prediction files:", len(missing_fold_test_files))

    if n_missing_oof != 0 or len(missing_fold_test_files) != 0:
        print("\nArtifact is incomplete. Re-run this same cell to resume.")
        for p in missing_fold_test_files:
            print("Missing:", p)

        return {
            "artifact_name": artifact_name,
            "completed": False,
            "oof_mse": np.nan,
            "oof_csv_path": oof_csv_path,
            "testpred_csv_path": testpred_csv_path,
            "submission_path": submission_path,
            "metrics_path": metrics_path,
        }

    overall_oof_mse = mean_squared_error(y_full_lgb, oof_pred)

    final_test_pred = np.clip(
        np.mean(np.vstack(fold_test_preds), axis=0),
        0,
        100,
    ).astype(np.float32)

    if "train_ids" in globals():
        train_id_series = as_1d_str_series(train_ids)
    else:
        train_id_series = pd.Series(np.arange(n_train)).astype(str)

    test_id_series = as_1d_str_series(test_ids)

    oof_df = pd.DataFrame({
        "ASSESSMENT_ID": train_id_series,
        "PERCENT_PROFICIENT_TRUE": y_full_lgb,
        f"OOF_{artifact_name}": oof_pred,
    })
    oof_df.to_csv(oof_csv_path, index=False)

    testpred_df = pd.DataFrame({
        "ASSESSMENT_ID": test_id_series,
        f"TESTPRED_{artifact_name}": final_test_pred,
    })
    testpred_df.to_csv(testpred_csv_path, index=False)

    submission_df = pd.DataFrame({
        "ASSESSMENT_ID": test_id_series,
        "PERCENT_PROFICIENT": final_test_pred,
    })

    assert submission_df.shape[0] == n_test
    assert list(submission_df.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
    assert submission_df["ASSESSMENT_ID"].isna().sum() == 0
    assert submission_df["PERCENT_PROFICIENT"].isna().sum() == 0

    submission_df.to_csv(submission_path, index=False)

    artifact_elapsed = time.time() - artifact_start

    print("\nOverall OOF MSE clipped:", overall_oof_mse)
    print("Saved OOF file:", oof_csv_path)
    print("Saved test prediction file:", testpred_csv_path)
    print("Saved Kaggle submission:", submission_path)

    print("\nSubmission shape:", submission_df.shape)

    print("\nSubmission prediction summary:")
    print(submission_df["PERCENT_PROFICIENT"].describe().to_string())

    print("\nFold metrics:")
    display(
        fold_metrics.sort_values("fold")[
            [
                "fold",
                "fold_valid_mse_clipped",
                "fold_valid_mse_raw",
                "best_iteration",
                "elapsed_sec",
            ]
        ].reset_index(drop=True)
    )

    print("\nArtifact elapsed seconds:", round(artifact_elapsed, 2))

    # Update trackers
    if "best_models_by_class" not in globals():
        globals()["best_models_by_class"] = {}

    best_models_by_class[artifact_name] = {
        "model_class": "LightGBM",
        "feature_space": "base",
        "oof_mse": float(overall_oof_mse),
        "submission_path": str(submission_path),
        "notes": "Overnight selected LightGBM OOF artifact.",
    }

    tracker_row = pd.DataFrame([
        {
            "submission_file": submission_path.name,
            "source": "current notebook",
            "model_family": "LightGBM",
            "public_mse": np.nan,
            "notes": f"Overnight LightGBM artifact {artifact_name}; OOF MSE={overall_oof_mse:.6f}.",
        }
    ])

    global submission_tracker
    submission_tracker = pd.concat(
        [submission_tracker, tracker_row],
        ignore_index=True,
    )

    submission_tracker = submission_tracker.drop_duplicates(
        subset=["submission_file"],
        keep="last",
    )

    submission_tracker.to_csv(tracker_path, index=False)

    return {
        "artifact_name": artifact_name,
        "completed": True,
        "oof_mse": float(overall_oof_mse),
        "oof_csv_path": oof_csv_path,
        "testpred_csv_path": testpred_csv_path,
        "submission_path": submission_path,
        "metrics_path": metrics_path,
    }


# ------------------------------------------------------------
# Run overnight LightGBM jobs
# ------------------------------------------------------------

suite_start = time.time()

for job in overnight_jobs:
    result = run_lgbm_oof_artifact(job)
    completed_lgbm_artifacts.append(result)
    gc.collect()

suite_elapsed = time.time() - suite_start

print("\n" + "#" * 90)
print("Overnight LightGBM suite summary")
print("#" * 90)

suite_summary = pd.DataFrame(completed_lgbm_artifacts)
display(suite_summary)

print("Suite elapsed seconds:", round(suite_elapsed, 2))

# ------------------------------------------------------------
# Automatic re-blend screen
# Includes:
# - RF 500
# - ExtraTrees safe
# - original 12k LightGBM
# - completed overnight LightGBM artifacts
# ------------------------------------------------------------

print("\n" + "#" * 90)
print("Automatic re-blend screen")
print("#" * 90)

blend_components = [
    {
        "name": "rf500",
        "oof_path": RESULTS_DIR / "oof_rf_500_base.csv",
        "test_path": RESULTS_DIR / "testpred_rf_500_base_folds.csv",
        "oof_pred_preferred": ["rf_500_base_oof_pred", "oof_pred", "pred"],
        "test_pred_preferred": ["rf_500_base_foldavg_pred", "PERCENT_PROFICIENT", "test_pred", "pred"],
    },
    {
        "name": "et_safe",
        "oof_path": RESULTS_DIR / "oof_extratrees_safe_base.csv",
        "test_path": RESULTS_DIR / "testpred_extratrees_safe_base_foldavg.csv",
        "oof_pred_preferred": ["oof_pred", "oof_pred_clipped", "pred"],
        "test_pred_preferred": ["PERCENT_PROFICIENT", "test_pred_avg", "test_pred", "pred"],
    },
    {
        "name": "lgbm12k",
        "oof_path": RESULTS_DIR / "oof_lgbm_t03_base_5fold_oof.csv",
        "test_path": RESULTS_DIR / "testpred_lgbm_t03_base_5fold_oof_foldavg.csv",
        "oof_pred_preferred": ["OOF_lgbm_t03_base_5fold_oof", "oof_pred", "pred"],
        "test_pred_preferred": ["TESTPRED_lgbm_t03_base_5fold_oof", "PERCENT_PROFICIENT", "test_pred", "pred"],
    },
]

for result in completed_lgbm_artifacts:
    if result["completed"]:
        artifact_name = result["artifact_name"]
        blend_components.append({
            "name": artifact_name.replace("lgbm_t03_base_5fold_oof_", "lgbm_"),
            "oof_path": Path(result["oof_csv_path"]),
            "test_path": Path(result["testpred_csv_path"]),
            "oof_pred_preferred": [f"OOF_{artifact_name}", "oof_pred", "pred"],
            "test_pred_preferred": [f"TESTPRED_{artifact_name}", "PERCENT_PROFICIENT", "test_pred", "pred"],
        })

available_components = []

for comp in blend_components:
    if comp["oof_path"].exists() and comp["test_path"].exists():
        available_components.append(comp)
    else:
        print("Skipping missing component:", comp["name"])

print("\nAvailable blend components:")
for comp in available_components:
    print(" ", comp["name"])

assert len(available_components) >= 2, "Need at least two components for blending."

def read_pred_csv(path):
    return pd.read_csv(path, dtype={"ASSESSMENT_ID": str})

def first_existing_column(df, candidates, label):
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(
        f"Could not find {label}. Tried {candidates}. Available columns: {list(df.columns)}"
    )

def pick_prediction_column(df, preferred_cols, label, target_cols=None):
    if target_cols is None:
        target_cols = []

    for col in preferred_cols:
        if col in df.columns:
            return col

    excluded_cols = set(["ASSESSMENT_ID"] + target_cols)

    numeric_cols = [
        c for c in df.columns
        if c not in excluded_cols and pd.api.types.is_numeric_dtype(df[c])
    ]

    pred_like_cols = [
        c for c in numeric_cols
        if (
            "pred" in c.lower()
            or "proficient" in c.lower()
            or "oof" in c.lower()
            or "foldavg" in c.lower()
            or "fold_avg" in c.lower()
            or "avg" in c.lower()
        )
    ]

    if len(pred_like_cols) == 1:
        return pred_like_cols[0]

    if len(numeric_cols) == 1:
        return numeric_cols[0]

    raise ValueError(
        f"Could not identify prediction column for {label}.\n"
        f"Columns: {list(df.columns)}\n"
        f"Numeric columns: {numeric_cols}\n"
        f"Prediction-like columns: {pred_like_cols}"
    )

def load_oof_component(comp):
    df = read_pred_csv(comp["oof_path"])

    y_col = first_existing_column(
        df,
        ["y_true", "PERCENT_PROFICIENT_TRUE", "target", "true", "actual"],
        f"{comp['name']} target column",
    )

    pred_col = pick_prediction_column(
        df,
        comp["oof_pred_preferred"],
        f"{comp['name']} OOF prediction",
        target_cols=[y_col],
    )

    out = df[["ASSESSMENT_ID", y_col, pred_col]].copy()
    out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)

    out = out.rename(columns={
        y_col: "y_true",
        pred_col: comp["name"],
    })

    print(f"{comp['name']} OOF prediction column:", pred_col)

    return out

def load_test_component(comp):
    df = read_pred_csv(comp["test_path"])

    pred_col = pick_prediction_column(
        df,
        comp["test_pred_preferred"],
        f"{comp['name']} test prediction",
        target_cols=[],
    )

    out = df[["ASSESSMENT_ID", pred_col]].copy()
    out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)

    out = out.rename(columns={pred_col: comp["name"]})

    print(f"{comp['name']} test prediction column:", pred_col)

    return out

# Load and merge OOF components
merged_oof = None
component_names = []

for comp in available_components:
    this_oof = load_oof_component(comp)
    component_names.append(comp["name"])

    if merged_oof is None:
        merged_oof = this_oof.copy()
    else:
        merged_oof = merged_oof.merge(
            this_oof,
            on="ASSESSMENT_ID",
            how="inner",
            suffixes=("", "_new"),
        )

        assert np.allclose(
            merged_oof["y_true"].to_numpy(dtype=float),
            merged_oof["y_true_new"].to_numpy(dtype=float),
        ), f"Target mismatch after merging {comp['name']}"

        merged_oof = merged_oof.drop(columns=["y_true_new"])

assert len(merged_oof) == n_train, f"OOF merge row count {len(merged_oof)} != {n_train}"

y_blend = merged_oof["y_true"].to_numpy(dtype=float)

component_oof_rows = []

for name in component_names:
    pred = merged_oof[name].to_numpy(dtype=float)

    component_oof_rows.append({
        "component": name,
        "oof_mse_clipped": mean_squared_error(y_blend, np.clip(pred, 0, 100)),
        "pred_mean": np.mean(np.clip(pred, 0, 100)),
        "pred_std": np.std(np.clip(pred, 0, 100)),
    })

component_oof_results = (
    pd.DataFrame(component_oof_rows)
    .sort_values("oof_mse_clipped")
    .reset_index(drop=True)
)

print("\nComponent OOF results:")
display(component_oof_results)

pred_matrix = np.vstack([
    merged_oof[name].to_numpy(dtype=float)
    for name in component_names
]).astype(np.float32)

# ------------------------------------------------------------
# Weight screen
# For 3+ components, use strong candidates:
# - pure models
# - pairwise fine grids
# - lgbm-focused random Dirichlet candidates
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_STATE)

weights = []
sources = []

def add_weights(w, source):
    w = np.asarray(w, dtype=float)

    if np.any(w < -1e-12):
        return

    total = w.sum()
    if total <= 0:
        return

    w = w / total
    w[np.abs(w) < 1e-12] = 0.0

    key = tuple(np.round(w, 6))

    if key in seen:
        return

    seen.add(key)
    weights.append(w)
    sources.append(source)

seen = set()
n_comp = len(component_names)

# Pure models
for j in range(n_comp):
    w = np.zeros(n_comp)
    w[j] = 1.0
    add_weights(w, "pure")

# Pairwise fine grids
grid = np.linspace(0, 1, 1001)

for i in range(n_comp):
    for j in range(i + 1, n_comp):
        for a in grid:
            w = np.zeros(n_comp)
            w[i] = a
            w[j] = 1.0 - a
            add_weights(w, "pairwise_grid")

# Random lgbm-focused convex search.
# This lets the search consider 3+ way blends without a huge grid.
# Stronger components receive larger Dirichlet concentration.
oof_mse_by_name = {
    row["component"]: row["oof_mse_clipped"]
    for _, row in component_oof_results.iterrows()
}

best_single_mse = min(oof_mse_by_name.values())

alpha = []

for name in component_names:
    mse = oof_mse_by_name[name]

    if mse <= best_single_mse + 2:
        alpha.append(8.0)
    elif "lgbm" in name.lower():
        alpha.append(5.0)
    elif "et" in name.lower():
        alpha.append(3.0)
    else:
        alpha.append(1.0)

alpha = np.asarray(alpha, dtype=float)

for _ in range(30000):
    add_weights(rng.dirichlet(alpha), "dirichlet_focused")

weight_matrix = np.vstack(weights).astype(np.float32)

print("\nNumber of blend candidates:", len(weight_matrix))

blend_rows = []

for idx, w in enumerate(weight_matrix):
    pred = np.dot(w, pred_matrix)
    pred_clip = np.clip(pred, 0, 100)

    row = {
        "source": sources[idx],
        "oof_mse_clipped": mean_squared_error(y_blend, pred_clip),
        "oof_mse_raw": mean_squared_error(y_blend, pred),
    }

    for name, val in zip(component_names, w):
        row[f"w_{name}"] = float(val)

    blend_rows.append(row)

blend_results = (
    pd.DataFrame(blend_rows)
    .sort_values("oof_mse_clipped")
    .reset_index(drop=True)
)

best_blend = blend_results.iloc[0].to_dict()

print("\nBest automatic re-blend:")
display(pd.DataFrame([best_blend]))

print("\nTop 20 automatic re-blend candidates:")
display(blend_results.head(20))

# ------------------------------------------------------------
# Create automatic re-blend submission
# ------------------------------------------------------------

merged_test = None

for comp in available_components:
    this_test = load_test_component(comp)

    if merged_test is None:
        merged_test = this_test.copy()
    else:
        merged_test = merged_test.merge(this_test, on="ASSESSMENT_ID", how="inner")

assert len(merged_test) == n_test, f"Test merge row count {len(merged_test)} != {n_test}"

# Restore original test_ids order
test_order = pd.DataFrame({
    "ASSESSMENT_ID": as_1d_str_series(test_ids),
    "_test_order": np.arange(n_test),
})

merged_test = test_order.merge(merged_test, on="ASSESSMENT_ID", how="left")

for name in component_names:
    assert merged_test[name].isna().sum() == 0, f"Missing test predictions for {name}"

merged_test = (
    merged_test
    .sort_values("_test_order")
    .drop(columns=["_test_order"])
    .reset_index(drop=True)
)

best_weights = np.array(
    [best_blend[f"w_{name}"] for name in component_names],
    dtype=np.float32,
)

test_matrix = np.vstack([
    merged_test[name].to_numpy(dtype=float)
    for name in component_names
]).astype(np.float32)

final_blend_pred = np.clip(
    np.dot(best_weights, test_matrix),
    0,
    100,
).astype(np.float32)

auto_blend_name = "blend_auto_et_lgbm_long_oof_weighted"

weight_screen_path = RESULTS_DIR / f"{auto_blend_name}_weight_screen.csv"
testpred_blend_path = RESULTS_DIR / f"testpred_{auto_blend_name}.csv"
submission_blend_path = Path(f"submission_{auto_blend_name}.csv")

blend_results.to_csv(weight_screen_path, index=False)

testpred_blend_df = merged_test[["ASSESSMENT_ID"] + component_names].copy()
testpred_blend_df["PERCENT_PROFICIENT"] = final_blend_pred
testpred_blend_df.to_csv(testpred_blend_path, index=False)

submission_blend = pd.DataFrame({
    "ASSESSMENT_ID": as_1d_str_series(test_ids),
    "PERCENT_PROFICIENT": final_blend_pred,
})

assert submission_blend.shape == (n_test, 2)
assert list(submission_blend.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
assert submission_blend["ASSESSMENT_ID"].isna().sum() == 0
assert submission_blend["PERCENT_PROFICIENT"].isna().sum() == 0

submission_blend.to_csv(submission_blend_path, index=False)

print("\nSaved automatic re-blend files:")
print("Weight screen:", weight_screen_path.resolve())
print("Test blend predictions:", testpred_blend_path.resolve())
print("Blend submission:", submission_blend_path.resolve())

print("\nAutomatic re-blend submission summary:")
print(submission_blend["PERCENT_PROFICIENT"].describe().to_string())

print("\nAutomatic re-blend submission shape:", submission_blend.shape)

print("\nFirst 5 rows:")
print(submission_blend.head().to_string(index=False))

# Update tracker for automatic blend
best_auto_blend_oof = float(best_blend["oof_mse_clipped"])

weight_note_parts = []
for name in component_names:
    weight_note_parts.append(f"{name}={best_blend[f'w_{name}']:.4f}")

tracker_auto_blend_row = pd.DataFrame([
    {
        "submission_file": submission_blend_path.name,
        "source": "current notebook",
        "model_family": "automatic OOF blend",
        "public_mse": np.nan,
        "notes": (
            f"Automatic post-long-run OOF blend; "
            f"OOF MSE={best_auto_blend_oof:.6f}; "
            + ", ".join(weight_note_parts)
        ),
    }
])

submission_tracker = pd.concat(
    [submission_tracker, tracker_auto_blend_row],
    ignore_index=True,
)

submission_tracker = submission_tracker.drop_duplicates(
    subset=["submission_file"],
    keep="last",
)

submission_tracker.to_csv(tracker_path, index=False)

print("\nUpdated submission tracker:", tracker_path.resolve())

print("\nDecision guide:")
print("Known public scores:")
print("  Pure 12k LightGBM: 90.644")
print("  ET + 12k LightGBM blend: 87.054")
print("\nCompare the new long-run OOF results and automatic blend OOF against:")
print("  Old pure LightGBM OOF: 98.7794")
print("  Old ET + LightGBM blend OOF: 93.4995")
print("\nIf the automatic long blend improves OOF versus 93.4995, it is the next natural Kaggle submission.")

# Clean local arrays
del X_full_lgb, X_test_lgb, y_full_lgb
gc.collect()

LightGBM version: 4.6.0
Results folder: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results

Updated submission tracker with latest public scores.
Tracker path: model_results/submission_tracker.csv

Full train shape: (144921, 162)
Full test shape: (48307, 162)
Target length: 144921
Test ID length: 48307

Planned overnight jobs:
  lgbm_t03_base_5fold_oof_long80k_lr03: lr=0.03, n_estimators=80000, early_stopping=2500
  lgbm_t03_base_5fold_oof_long100k_lr02: lr=0.02, n_estimators=100000, early_stopping=3000

##########################################################################################
Starting artifact: lgbm_t03_base_5fold_oof_long80k_lr03
##########################################################################################

LightGBM parameters:
  objective: regression
  metric: l2
  random_state: 9890
  n_jobs: 1
  verbosity: -1
  force_col_wise: True
  num_leaves: 95
  min_child_samples: 60
  subsample:

,fold,fold_valid_mse_clipped,fold_valid_mse_raw,best_iteration,elapsed_sec
0,1,95.764534,96.739767,39024,2553.609186
1,2,91.844963,92.593493,39385,1485.937457
2,3,94.334686,95.260975,34065,591.087250
3,4,92.318062,93.309677,36884,628.429420
4,5,96.555138,97.543099,40463,688.799824



Artifact elapsed seconds: 13004.18

##########################################################################################
Starting artifact: lgbm_t03_base_5fold_oof_long100k_lr02
##########################################################################################

LightGBM parameters:
  objective: regression
  metric: l2
  random_state: 9890
  n_jobs: 1
  verbosity: -1
  force_col_wise: True
  num_leaves: 95
  min_child_samples: 60
  subsample: 0.85
  subsample_freq: 1
  colsample_bytree: 0.9
  reg_alpha: 0.0
  reg_lambda: 5.0
  max_depth: -1
  n_estimators: 100000
  learning_rate: 0.02

Starting lgbm_t03_base_5fold_oof_long100k_lr02, fold 1/5
Training until validation scores don't improve for 3000 rounds
[2500]	valid_0's l2: 132.896
[5000]	valid_0's l2: 119.103
[7500]	valid_0's l2: 111.575
[10000]	valid_0's l2: 107.136
[12500]	valid_0's l2: 104.191
[15000]	valid_0's l2: 102.111
[17500]	valid_0's l2: 100.632
[20000]	valid_0's l2: 99.4575
[22500]	valid_0's l2: 98.6736
[25000

,fold,fold_valid_mse_clipped,fold_valid_mse_raw,best_iteration,elapsed_sec
0,1,95.241920,96.194398,54700,935.653679
1,2,91.291603,92.035591,52940,884.257007
2,3,93.777443,94.710169,59506,1025.118408
3,4,92.410759,93.340667,53392,894.905689
4,5,95.909294,96.860579,64138,1080.716788



Artifact elapsed seconds: 11441.8

##########################################################################################
Overnight LightGBM suite summary
##########################################################################################


...

,source,oof_mse_clipped,oof_mse_raw,w_rf500,w_et_safe,w_lgbm12k,w_lgbm_long80k_lr03,w_lgbm_long100k_lr02
0,pairwise_grid,88.966368,88.966368,0.0,0.319,0.0,0.0,0.681



Top 20 automatic re-blend candidates:


,source,oof_mse_clipped,oof_mse_raw,w_rf500,w_et_safe,w_lgbm12k,w_lgbm_long80k_lr03,w_lgbm_long100k_lr02
0,pairwise_grid,88.966368,88.966368,0.0,0.319,0.0,0.0,0.681
1,pairwise_grid,88.966373,88.966373,0.0,0.318,0.0,0.0,0.682
2,pairwise_grid,88.966458,88.966458,0.0,0.320,0.0,0.0,0.680
3,pairwise_grid,88.966470,88.966470,0.0,0.317,0.0,0.0,0.683
4,pairwise_grid,88.966640,88.966640,0.0,0.321,0.0,0.0,0.679
5,pairwise_grid,88.966662,88.966662,0.0,0.316,0.0,0.0,0.684
6,pairwise_grid,88.966919,88.966919,0.0,0.322,0.0,0.0,0.678
7,pairwise_grid,88.966949,88.966949,0.0,0.315,0.0,0.0,0.685
8,pairwise_grid,88.967288,88.967288,0.0,0.323,0.0,0.0,0.677
9,pairwise_grid,88.967329,88.967329,0.0,0.314,0.0,0.0,0.686


rf500 test prediction column: rf_500_base_foldavg_pred
et_safe test prediction column: PERCENT_PROFICIENT
lgbm12k test prediction column: TESTPRED_lgbm_t03_base_5fold_oof
lgbm_long80k_lr03 test prediction column: TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03
lgbm_long100k_lr02 test prediction column: TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02

Saved automatic re-blend files:
Weight screen: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/blend_auto_et_lgbm_long_oof_weighted_weight_screen.csv
Test blend predictions: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/testpred_blend_auto_et_lgbm_long_oof_weighted.csv
Blend submission: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/submission_blend_auto_et_lgbm_long_oof_weighted.csv

Automatic re-blend submission summary:
count    48307.000000

0

### Overnight selected LightGBM long-run suite and automatic re-blend

The earlier 12k selected LightGBM artifact reached the estimator cap on all folds, so two longer versions of the same T03 configuration were trained with fold-level checkpointing:

- `long80k_lr03`: learning rate 0.03, 80,000 maximum estimators, early stopping 2,500
- `long100k_lr02`: learning rate 0.02, 100,000 maximum estimators, early stopping 3,000

Both completed successfully. The 12k LightGBM OOF MSE was 98.7794. The long 80k run improved OOF MSE to 94.1635, and the long 100k run improved OOF MSE to 93.7262. This confirms that the original 12k LightGBM was under-trained.

An automatic convex re-blend was then run using the saved OOF artifacts from Random Forest, ExtraTrees, the original 12k LightGBM, and the completed long LightGBM runs. The best blend used:

- ExtraTrees safe base: 0.319
- LightGBM long100k_lr02: 0.681

All other components received zero weight. The new blend OOF MSE was 88.9664, improving substantially over the previous ET + 12k LightGBM blend OOF MSE of 93.4995. The corresponding submission file is:

`submission_blend_auto_et_lgbm_long_oof_weighted.csv`

This is the next primary Kaggle submission candidate.

### Kaggle result: automatic long LightGBM + ExtraTrees blend

The automatic long-run blend was submitted to Kaggle as:

`submission_blend_auto_et_lgbm_long_oof_weighted.csv`

This submission achieved a public leaderboard MSE of **80.662**, making it the best clean current-notebook result so far.

This is a large improvement over the previous best clean blend:

- Previous best clean blend: `submission_blend_rf_et_lgbm_oof_weighted.csv`
  - Public MSE: **87.054**
- New best clean blend: `submission_blend_auto_et_lgbm_long_oof_weighted.csv`
  - Public MSE: **80.662**
- Public leaderboard improvement: **6.392 MSE points**

The local OOF evidence was directionally consistent with the public leaderboard result. The previous ExtraTrees + 12k LightGBM blend had OOF MSE **93.4995**, while the new automatic blend had OOF MSE **88.9664**. The best automatic blend used:

- ExtraTrees safe base: **0.319**
- LightGBM `long100k_lr02`: **0.681**

Random Forest, the original 12k LightGBM, and the `long80k_lr03` LightGBM variant received zero weight in the best automatic blend.

This result confirms that the original 12k LightGBM was under-trained and that the longer `long100k_lr02` LightGBM artifact added substantial predictive strength. It also confirms that ExtraTrees, although weaker as a standalone model, provides useful complementary signal when blended with LightGBM.

Current best clean current-notebook submission:

`submission_blend_auto_et_lgbm_long_oof_weighted.csv`  
Public MSE: **80.662**

In [41]:
# 22A. Post-80.662: update tracker + refine/stack saved OOF artifacts
# This cell is mostly fast. It:
# 1. records the new 80.662 public score,
# 2. reloads saved RF / ExtraTrees / LightGBM OOF artifacts,
# 3. refines the ET + long100k blend continuously,
# 4. tries conservative calibration and simple OOF meta-stacking,
# 5. saves candidate submissions for review.

import os
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize, minimize_scalar
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV, LinearRegression

warnings.filterwarnings("ignore")

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

TARGET_NAME = "PERCENT_PROFICIENT"
ID_NAME = "ASSESSMENT_ID"

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clipped(y_true, pred):
    y_true = np.asarray(y_true, dtype=float)
    pred = clip100(pred)
    return float(np.mean((y_true - pred) ** 2))

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError("None of these files exist:\n" + "\n".join(map(str, paths)))

def ids_from_global(primary_name, fallback_name, expected_len):
    if primary_name in globals():
        obj = globals()[primary_name]
        if isinstance(obj, pd.DataFrame):
            if ID_NAME in obj.columns:
                s = obj[ID_NAME]
            else:
                s = obj.iloc[:, 0]
        else:
            s = pd.Series(obj)
    elif fallback_name in globals():
        obj = globals()[fallback_name]
        if isinstance(obj, pd.DataFrame) and ID_NAME in obj.columns:
            s = obj[ID_NAME]
        else:
            return None
    else:
        return None

    s = pd.Series(s).astype(str).reset_index(drop=True)
    if len(s) != expected_len:
        raise ValueError(f"{primary_name}/{fallback_name} length mismatch: {len(s)} vs {expected_len}")
    return pd.Series(s.values, name=ID_NAME)

def pick_pred_col(df, kind="oof", preferred=None):
    if preferred is not None and preferred in df.columns:
        return preferred

    exclude = {ID_NAME, "fold", "FOLD", "index", "Unnamed: 0"}
    if kind == "oof":
        exclude |= {TARGET_NAME, "target", "TARGET", "y", "Y", "actual", "truth"}

    numeric_like = []
    for c in df.columns:
        if c in exclude:
            continue
        converted = pd.to_numeric(df[c], errors="coerce")
        if converted.notna().mean() > 0.95:
            numeric_like.append(c)

    if not numeric_like:
        raise ValueError(f"No numeric-like prediction columns found. Columns were: {list(df.columns)}")

    priority_keywords = ["oof", "testpred", "foldavg", "pred", "percent_proficient"]
    lower_map = {c: c.lower() for c in numeric_like}

    for kw in priority_keywords:
        for c in numeric_like:
            if kw in lower_map[c]:
                return c

    if len(numeric_like) == 1:
        return numeric_like[0]

    print("Multiple numeric-like columns found; using the last one:")
    print(numeric_like)
    return numeric_like[-1]

def load_prediction(path, expected_len, expected_ids=None, preferred_col=None, kind="oof"):
    path = Path(path)
    df = pd.read_csv(path)
    pred_col = pick_pred_col(df, kind=kind, preferred=preferred_col)

    if ID_NAME in df.columns and expected_ids is not None:
        left = pd.DataFrame({ID_NAME: expected_ids.astype(str).values})
        right = df[[ID_NAME, pred_col]].copy()
        right[ID_NAME] = right[ID_NAME].astype(str)

        merged = left.merge(right, on=ID_NAME, how="left")
        missing = merged[pred_col].isna().sum()
        if missing > 0:
            raise ValueError(f"{path} has {missing} missing predictions after ID merge.")

        return pd.to_numeric(merged[pred_col], errors="raise").to_numpy(dtype=float), pred_col

    if len(df) != expected_len:
        raise ValueError(f"{path} length mismatch: {len(df)} vs expected {expected_len}")

    return pd.to_numeric(df[pred_col], errors="raise").to_numpy(dtype=float), pred_col

def save_candidate(candidate_name, test_pred, oof_pred=None, extra_info=None):
    test_pred = clip100(test_pred)

    testpred_path = RESULTS_DIR / f"testpred_{candidate_name}.csv"
    sub_path = Path(f"submission_{candidate_name}.csv")

    pd.DataFrame({
        ID_NAME: test_id_series.astype(str).values,
        f"TESTPRED_{candidate_name}": test_pred
    }).to_csv(testpred_path, index=False)

    pd.DataFrame({
        ID_NAME: test_id_series.astype(str).values,
        TARGET_NAME: test_pred
    }).to_csv(sub_path, index=False)

    if oof_pred is not None:
        oof_pred = clip100(oof_pred)
        if train_id_series is not None:
            oof_df = pd.DataFrame({
                ID_NAME: train_id_series.astype(str).values,
                TARGET_NAME: y,
                f"OOF_{candidate_name}": oof_pred
            })
        else:
            oof_df = pd.DataFrame({
                TARGET_NAME: y,
                f"OOF_{candidate_name}": oof_pred
            })
        oof_df.to_csv(RESULTS_DIR / f"oof_{candidate_name}.csv", index=False)

    saved_submissions.append({
        "candidate": candidate_name,
        "submission_path": str(sub_path),
        "testpred_path": str(testpred_path),
        **(extra_info or {})
    })

    return sub_path

# --------------------------------------------------------------------------------------
# Update submission tracker with the 80.662 public score
# --------------------------------------------------------------------------------------

tracker_path = RESULTS_DIR / "submission_tracker.csv"

new_public_row = {
    "file": "submission_blend_auto_et_lgbm_long_oof_weighted.csv",
    "source": "current notebook",
    "model_family": "OOF blend",
    "public_mse": 80.662,
    "notes": "Automatic long-run blend: 0.319 ExtraTrees safe + 0.681 LightGBM long100k_lr02"
}

if tracker_path.exists():
    tracker = pd.read_csv(tracker_path)
else:
    tracker = pd.DataFrame()

for col in new_public_row:
    if col not in tracker.columns:
        tracker[col] = np.nan

row_to_add = {col: new_public_row.get(col, np.nan) for col in tracker.columns}
tracker = pd.concat([tracker, pd.DataFrame([row_to_add])], ignore_index=True)

file_col = "file" if "file" in tracker.columns else tracker.columns[0]
tracker = tracker.drop_duplicates(subset=[file_col], keep="last")
tracker.to_csv(tracker_path, index=False)

print(f"Updated tracker: {tracker_path}")

# --------------------------------------------------------------------------------------
# Load target and IDs
# --------------------------------------------------------------------------------------

y = np.asarray(y_train, dtype=float).ravel()
n_train = len(y)

train_id_series = ids_from_global("train_ids", "scores_training", n_train)
test_id_series = ids_from_global("test_ids", "scores_test", len(X_test_proc_model))

if test_id_series is None:
    raise ValueError("Could not find test IDs. Expected test_ids or scores_test with ASSESSMENT_ID.")

# --------------------------------------------------------------------------------------
# Load saved OOF/test prediction components
# --------------------------------------------------------------------------------------

component_specs = {
    "rf500": {
        "oof_paths": ["model_results/oof_rf_500_base.csv"],
        "test_paths": ["model_results/testpred_rf_500_base_folds.csv"],
        "oof_col": "rf_500_base_oof_pred",
        "test_col": "rf_500_base_foldavg_pred",
    },
    "et_safe": {
        "oof_paths": ["model_results/oof_extratrees_safe_base.csv"],
        "test_paths": [
            "model_results/testpred_extratrees_safe_base_foldavg.csv",
            "model_results/testpred_extratrees_safe_base.csv",
        ],
        "oof_col": "oof_pred",
        "test_col": TARGET_NAME,
    },
    "lgbm12k": {
        "oof_paths": ["model_results/oof_lgbm_t03_base_5fold_oof.csv"],
        "test_paths": ["model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv"],
        "oof_col": "OOF_lgbm_t03_base_5fold_oof",
        "test_col": "TESTPRED_lgbm_t03_base_5fold_oof",
    },
    "lgbm_long80k_lr03": {
        "oof_paths": ["model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv"],
        "test_paths": ["model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv"],
        "oof_col": "OOF_lgbm_t03_base_5fold_oof_long80k_lr03",
        "test_col": "TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03",
    },
    "lgbm_long100k_lr02": {
        "oof_paths": ["model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv"],
        "test_paths": ["model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv"],
        "oof_col": "OOF_lgbm_t03_base_5fold_oof_long100k_lr02",
        "test_col": "TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02",
    },
}

oof_components = {}
test_components = {}
loaded_rows = []

for name, spec in component_specs.items():
    try:
        oof_path = first_existing(spec["oof_paths"])
        test_path = first_existing(spec["test_paths"])

        oof_pred, oof_col = load_prediction(
            oof_path,
            expected_len=n_train,
            expected_ids=train_id_series,
            preferred_col=spec.get("oof_col"),
            kind="oof"
        )

        test_pred, test_col = load_prediction(
            test_path,
            expected_len=len(test_id_series),
            expected_ids=test_id_series,
            preferred_col=spec.get("test_col"),
            kind="test"
        )

        oof_components[name] = clip100(oof_pred)
        test_components[name] = clip100(test_pred)

        loaded_rows.append({
            "component": name,
            "oof_path": str(oof_path),
            "oof_col": oof_col,
            "test_path": str(test_path),
            "test_col": test_col,
            "oof_mse_clipped": mse_clipped(y, oof_pred),
            "oof_pred_mean": float(np.mean(clip100(oof_pred))),
            "oof_pred_std": float(np.std(clip100(oof_pred))),
            "test_pred_mean": float(np.mean(clip100(test_pred))),
            "test_pred_std": float(np.std(clip100(test_pred))),
        })

    except Exception as e:
        print(f"Skipped {name}: {e}")

loaded_df = pd.DataFrame(loaded_rows).sort_values("oof_mse_clipped")
print("\nLoaded components:")
try:
    display(loaded_df)
except NameError:
    print(loaded_df.to_string(index=False))

required = {"et_safe", "lgbm_long100k_lr02"}
missing_required = required - set(oof_components)
if missing_required:
    raise ValueError(f"Missing required components for the main refinement: {missing_required}")

# --------------------------------------------------------------------------------------
# Candidate screen
# --------------------------------------------------------------------------------------

summary_rows = []
saved_submissions = []

def add_summary(name, oof_pred, test_pred=None, source="candidate", details=None):
    row = {
        "name": name,
        "source": source,
        "oof_mse_clipped": mse_clipped(y, oof_pred),
        "pred_mean_oof": float(np.mean(clip100(oof_pred))),
        "pred_std_oof": float(np.std(clip100(oof_pred))),
    }

    if test_pred is not None:
        row.update({
            "pred_mean_test": float(np.mean(clip100(test_pred))),
            "pred_std_test": float(np.std(clip100(test_pred))),
            "pred_min_test": float(np.min(clip100(test_pred))),
            "pred_max_test": float(np.max(clip100(test_pred))),
        })

    if details:
        row.update(details)

    summary_rows.append(row)

# Standalone components
for name in oof_components:
    add_summary(
        name=name,
        oof_pred=oof_components[name],
        test_pred=test_components[name],
        source="component"
    )

# Recreate submitted 80.662 blend
submitted_oof = (
    0.319 * oof_components["et_safe"]
    + 0.681 * oof_components["lgbm_long100k_lr02"]
)
submitted_test = (
    0.319 * test_components["et_safe"]
    + 0.681 * test_components["lgbm_long100k_lr02"]
)
add_summary(
    name="submitted_public80p662_recreated",
    oof_pred=submitted_oof,
    test_pred=submitted_test,
    source="known_submission",
    details={"w_et_safe": 0.319, "w_lgbm_long100k_lr02": 0.681}
)

# 1D continuous refinement of ET + long100k weight
def scalar_blend_loss(w_et):
    p = (
        w_et * oof_components["et_safe"]
        + (1.0 - w_et) * oof_components["lgbm_long100k_lr02"]
    )
    return mse_clipped(y, p)

res_scalar = minimize_scalar(
    scalar_blend_loss,
    bounds=(0.0, 1.0),
    method="bounded",
    options={"xatol": 1e-10}
)

w_et_refined = float(res_scalar.x)
w_lgbm_refined = 1.0 - w_et_refined

scalar_oof = (
    w_et_refined * oof_components["et_safe"]
    + w_lgbm_refined * oof_components["lgbm_long100k_lr02"]
)
scalar_test = (
    w_et_refined * test_components["et_safe"]
    + w_lgbm_refined * test_components["lgbm_long100k_lr02"]
)

add_summary(
    name="blend_et_lgbm_long100_scalar_refined",
    oof_pred=scalar_oof,
    test_pred=scalar_test,
    source="scalar_minimize",
    details={"w_et_safe": w_et_refined, "w_lgbm_long100k_lr02": w_lgbm_refined}
)

save_candidate(
    "blend_et_lgbm_long100_scalar_refined",
    scalar_test,
    scalar_oof,
    extra_info={"w_et_safe": w_et_refined, "w_lgbm_long100k_lr02": w_lgbm_refined}
)

# Convex all-component optimization
component_order = [
    name for name in [
        "rf500",
        "et_safe",
        "lgbm12k",
        "lgbm_long80k_lr03",
        "lgbm_long100k_lr02",
    ]
    if name in oof_components
]

X_meta = np.column_stack([oof_components[name] for name in component_order])
X_test_meta = np.column_stack([test_components[name] for name in component_order])

def convex_loss(w):
    return mse_clipped(y, X_meta @ np.asarray(w))

x0 = np.ones(len(component_order)) / len(component_order)
if "et_safe" in component_order and "lgbm_long100k_lr02" in component_order:
    x0 = np.zeros(len(component_order))
    x0[component_order.index("et_safe")] = w_et_refined
    x0[component_order.index("lgbm_long100k_lr02")] = w_lgbm_refined

res_convex = minimize(
    convex_loss,
    x0=x0,
    method="SLSQP",
    bounds=[(0.0, 1.0)] * len(component_order),
    constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
    options={"maxiter": 1000, "ftol": 1e-12}
)

w_convex = np.asarray(res_convex.x, dtype=float)
convex_oof = X_meta @ w_convex
convex_test = X_test_meta @ w_convex

details_convex = {
    "convex_success": bool(res_convex.success),
    "convex_message": str(res_convex.message),
}
for name, w in zip(component_order, w_convex):
    details_convex[f"w_{name}"] = float(w)

add_summary(
    name="blend_all_components_convex_refined",
    oof_pred=convex_oof,
    test_pred=convex_test,
    source="slsqp_convex",
    details=details_convex
)

save_candidate(
    "blend_all_components_convex_refined",
    convex_test,
    convex_oof,
    extra_info=details_convex
)

# Conservative affine calibration around the refined ET + long100 blend
# Bounds are intentionally conservative to avoid making a wild leaderboard-overfit correction.
def calibration_loss(ab):
    a, b = ab
    return mse_clipped(y, a * scalar_oof + b)

res_cal = minimize(
    calibration_loss,
    x0=np.array([1.0, 0.0]),
    method="L-BFGS-B",
    bounds=[(0.90, 1.10), (-3.0, 3.0)],
    options={"maxiter": 1000, "ftol": 1e-12}
)

a_cal, b_cal = map(float, res_cal.x)
cal_oof = a_cal * scalar_oof + b_cal
cal_test = a_cal * scalar_test + b_cal

add_summary(
    name="blend_et_lgbm_long100_scalar_refined_calibrated",
    oof_pred=cal_oof,
    test_pred=cal_test,
    source="conservative_affine_calibration",
    details={
        "a": a_cal,
        "b": b_cal,
        "calibration_success": bool(res_cal.success),
        "base_w_et_safe": w_et_refined,
        "base_w_lgbm_long100k_lr02": w_lgbm_refined,
    }
)

save_candidate(
    "blend_et_lgbm_long100_scalar_refined_calibrated",
    cal_test,
    cal_oof,
    extra_info={
        "a": a_cal,
        "b": b_cal,
        "base_w_et_safe": w_et_refined,
        "base_w_lgbm_long100k_lr02": w_lgbm_refined,
    }
)

# Ridge meta-stack with 5-fold CV over OOF-prediction features
def run_meta_cv(model_factory, model_name):
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    meta_oof = np.zeros(n_train, dtype=float)

    fold_rows = []
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_meta), start=1):
        model = model_factory()
        model.fit(X_meta[tr_idx], y[tr_idx])
        meta_oof[va_idx] = model.predict(X_meta[va_idx])

        fold_rows.append({
            "fold": fold,
            "fold_mse_clipped": mse_clipped(y[va_idx], meta_oof[va_idx])
        })

    final_model = model_factory()
    final_model.fit(X_meta, y)
    meta_test = final_model.predict(X_test_meta)

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(RESULTS_DIR / f"{model_name}_meta_cv_fold_metrics.csv", index=False)

    details = {
        "meta_cv_mse_clipped": mse_clipped(y, meta_oof),
        "meta_fold_mse_mean": float(fold_df["fold_mse_clipped"].mean()),
        "meta_fold_mse_std": float(fold_df["fold_mse_clipped"].std(ddof=1)),
    }

    if hasattr(final_model, "intercept_"):
        details["intercept"] = float(np.ravel(final_model.intercept_)[0])

    if hasattr(final_model, "coef_"):
        coefs = np.ravel(final_model.coef_)
        for name, coef in zip(component_order, coefs):
            details[f"coef_{name}"] = float(coef)

    if hasattr(final_model, "alpha_"):
        details["alpha"] = float(final_model.alpha_)

    add_summary(
        name=model_name,
        oof_pred=meta_oof,
        test_pred=meta_test,
        source="meta_model_cv",
        details=details
    )

    save_candidate(model_name, meta_test, meta_oof, extra_info=details)

    return final_model, fold_df

alphas = np.logspace(-6, 3, 60)

ridge_model, ridge_folds = run_meta_cv(
    model_factory=lambda: RidgeCV(alphas=alphas, fit_intercept=True),
    model_name="stack_ridge_oof_components"
)

positive_lr_model, positive_lr_folds = run_meta_cv(
    model_factory=lambda: LinearRegression(positive=True),
    model_name="stack_positive_linear_oof_components"
)

# --------------------------------------------------------------------------------------
# Save and display screen
# --------------------------------------------------------------------------------------

screen_df = pd.DataFrame(summary_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
screen_path = RESULTS_DIR / "post80p662_blend_refinement_screen.csv"
screen_df.to_csv(screen_path, index=False)

saved_df = pd.DataFrame(saved_submissions)
saved_path = RESULTS_DIR / "post80p662_saved_candidate_submissions.csv"
saved_df.to_csv(saved_path, index=False)

print("\nPost-80.662 blend / stack refinement screen:")
try:
    display(screen_df)
except NameError:
    print(screen_df.to_string(index=False))

print("\nSaved candidate submissions:")
try:
    display(saved_df)
except NameError:
    print(saved_df.to_string(index=False))

print("\nKey files:")
print("Screen:", screen_path)
print("Saved submissions list:", saved_path)
print("Tracker:", tracker_path)

print("\nDecision guide:")
print("- Do not submit these automatically before review.")
print("- If scalar_refined or convex_refined only improves by ~0.001 OOF, it is probably not worth a Kaggle slot.")
print("- If ridge/positive stack improves OOF materially and has sane coefficients/test summary, it may be a candidate.")
print("- Paste this output when you return.")

Updated tracker: model_results/submission_tracker.csv

Loaded components:


,component,oof_path,oof_col,test_path,test_col,oof_mse_clipped,oof_pred_mean,oof_pred_std,test_pred_mean,test_pred_std
4,lgbm_long100k_lr02,model_results/oof_lgbm_t03_base_5fold_oof_long...,OOF_lgbm_t03_base_5fold_oof_long100k_lr02,model_results/testpred_lgbm_t03_base_5fold_oof...,TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02,93.726214,54.153144,24.734667,54.074208,24.605725
3,lgbm_long80k_lr03,model_results/oof_lgbm_t03_base_5fold_oof_long...,OOF_lgbm_t03_base_5fold_oof_long80k_lr03,model_results/testpred_lgbm_t03_base_5fold_oof...,TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03,94.163485,54.147019,24.743266,54.067601,24.607933
2,lgbm12k,model_results/oof_lgbm_t03_base_5fold_oof.csv,OOF_lgbm_t03_base_5fold_oof,model_results/testpred_lgbm_t03_base_5fold_oof...,TESTPRED_lgbm_t03_base_5fold_oof,98.779406,54.152808,24.283770,54.086671,24.207260
1,et_safe,model_results/oof_extratrees_safe_base.csv,oof_pred,model_results/testpred_extratrees_safe_base_fo...,PERCENT_PROFICIENT,110.749294,54.127367,23.857745,54.040173,23.782901
0,rf500,model_results/oof_rf_500_base.csv,rf_500_base_oof_pred,model_results/testpred_rf_500_base_folds.csv,rf_500_base_foldavg_pred,122.328024,54.134143,22.839348,54.056767,22.817463



Post-80.662 blend / stack refinement screen:


,name,source,oof_mse_clipped,pred_mean_oof,pred_std_oof,pred_mean_test,pred_std_test,pred_min_test,pred_max_test,w_et_safe,w_lgbm_long100k_lr02,convex_success,convex_message,w_rf500,w_lgbm12k,w_lgbm_long80k_lr03,a,b,calibration_success,base_w_et_safe,base_w_lgbm_long100k_lr02,meta_cv_mse_clipped,meta_fold_mse_mean,meta_fold_mse_std,intercept,coef_rf500,coef_et_safe,coef_lgbm12k,coef_lgbm_long80k_lr03,coef_lgbm_long100k_lr02,alpha
0,stack_ridge_oof_components,meta_model_cv,87.976479,54.179374,24.697050,54.095697,24.611060,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.976479,87.976470,1.775049,-0.333024,-0.171527,0.480278,-0.159616,0.402591,0.455197,1000.0
1,stack_positive_linear_oof_components,meta_model_cv,88.658727,54.175720,24.678509,54.094807,24.614145,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88.658727,88.658718,1.776098,-0.983962,0.000000,0.333300,0.000000,0.248013,0.437605,NaN
2,blend_et_lgbm_long100_scalar_refined_calibrated,conservative_affine_calibration,88.755227,54.188640,24.709775,54.107978,24.643046,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.019759,-1.017553,True,0.318548,0.681452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,blend_all_components_convex_refined,slsqp_convex,88.865535,54.143501,24.251630,54.061842,24.181663,0.076160,100.0000,0.315230,0.437056,True,Optimization terminated successfully,1.406089e-14,2.778263e-15,0.247714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,blend_et_lgbm_long100_scalar_refined,scalar_minimize,88.966359,54.144933,24.249691,54.063366,24.178358,0.076961,100.0000,0.318548,0.681452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,submitted_public80p662_recreated,known_submission,88.966368,54.144921,24.249135,54.063351,24.177857,0.077070,100.0000,0.319000,0.681000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,lgbm_long100k_lr02,component,93.726214,54.153144,24.734667,54.074208,24.605725,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,lgbm_long80k_lr03,component,94.163485,54.147019,24.743266,54.067601,24.607933,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,lgbm12k,component,98.779406,54.152808,24.283770,54.086671,24.207260,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,et_safe,component,110.749294,54.127367,23.857745,54.040173,23.782901,0.100800,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Saved candidate submissions:


,candidate,submission_path,testpred_path,w_et_safe,w_lgbm_long100k_lr02,convex_success,convex_message,w_rf500,w_lgbm12k,w_lgbm_long80k_lr03,a,b,base_w_et_safe,base_w_lgbm_long100k_lr02,meta_cv_mse_clipped,meta_fold_mse_mean,meta_fold_mse_std,intercept,coef_rf500,coef_et_safe,coef_lgbm12k,coef_lgbm_long80k_lr03,coef_lgbm_long100k_lr02,alpha
0,blend_et_lgbm_long100_scalar_refined,submission_blend_et_lgbm_long100_scalar_refine...,model_results/testpred_blend_et_lgbm_long100_s...,0.318548,0.681452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,blend_all_components_convex_refined,submission_blend_all_components_convex_refined...,model_results/testpred_blend_all_components_co...,0.315230,0.437056,True,Optimization terminated successfully,1.406089e-14,2.778263e-15,0.247714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,blend_et_lgbm_long100_scalar_refined_calibrated,submission_blend_et_lgbm_long100_scalar_refine...,model_results/testpred_blend_et_lgbm_long100_s...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.019759,-1.017553,0.318548,0.681452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,stack_ridge_oof_components,submission_stack_ridge_oof_components.csv,model_results/testpred_stack_ridge_oof_compone...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.976479,87.976470,1.775049,-0.333024,-0.171527,0.480278,-0.159616,0.402591,0.455197,1000.0
4,stack_positive_linear_oof_components,submission_stack_positive_linear_oof_component...,model_results/testpred_stack_positive_linear_o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88.658727,88.658718,1.776098,-0.983962,0.000000,0.333300,0.000000,0.248013,0.437605,NaN



Key files:
Screen: model_results/post80p662_blend_refinement_screen.csv
Saved submissions list: model_results/post80p662_saved_candidate_submissions.csv
Tracker: model_results/submission_tracker.csv

Decision guide:
- Do not submit these automatically before review.
- If scalar_refined or convex_refined only improves by ~0.001 OOF, it is probably not worth a Kaggle slot.
- If ridge/positive stack improves OOF materially and has sane coefficients/test summary, it may be a candidate.
- Paste this output when you return.


In [43]:
# 22B fixed: LightGBM diversity holdout screen with safe feature names
# Fix: convert X_tr / X_val to NumPy arrays and pass simple feature names to LightGBM.

import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from scipy import sparse

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse(y_true, pred):
    pred = clip100(pred)
    return float(np.mean((np.asarray(y_true, dtype=float) - pred) ** 2))

def to_lgbm_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32)
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

# Use existing development split if available; otherwise recreate it.
if all(name in globals() for name in ["X_tr", "X_val", "y_tr", "y_val"]):
    X_tr_use, X_val_use = X_tr, X_val
    y_tr_use = np.asarray(y_tr, dtype=float).ravel()
    y_val_use = np.asarray(y_val, dtype=float).ravel()
    print("Using existing X_tr / X_val split.")
else:
    X_tr_use, X_val_use, y_tr_use, y_val_use = train_test_split(
        X_train_proc_model,
        np.asarray(y_train, dtype=float).ravel(),
        test_size=0.20,
        random_state=RANDOM_STATE
    )
    print("Recreated 80/20 split.")

X_tr_lgbm = to_lgbm_matrix(X_tr_use)
X_val_lgbm = to_lgbm_matrix(X_val_use)

safe_feature_names = [f"f{i}" for i in range(X_tr_lgbm.shape[1])]

print("Train shape:", X_tr_lgbm.shape)
print("Validation shape:", X_val_lgbm.shape)

base_params = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,
}

jobs = [
    {
        "name": "lgbm_div01_l63_child80_l2_10_lr02",
        "params": {
            **base_params,
            "learning_rate": 0.02,
            "n_estimators": 70000,
            "num_leaves": 63,
            "min_child_samples": 80,
            "subsample": 0.85,
            "subsample_freq": 1,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.05,
            "reg_lambda": 10.0,
            "max_depth": -1,
        },
        "stopping_rounds": 2500,
        "log_period": 2500,
    },
    {
        "name": "lgbm_div02_l127_child50_l2_15_lr02",
        "params": {
            **base_params,
            "learning_rate": 0.02,
            "n_estimators": 80000,
            "num_leaves": 127,
            "min_child_samples": 50,
            "subsample": 0.80,
            "subsample_freq": 1,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.0,
            "reg_lambda": 15.0,
            "max_depth": -1,
        },
        "stopping_rounds": 3000,
        "log_period": 2500,
    },
    {
        "name": "lgbm_div03_extra_trees_l95_child60_l2_8_lr025",
        "params": {
            **base_params,
            "learning_rate": 0.025,
            "n_estimators": 70000,
            "num_leaves": 95,
            "min_child_samples": 60,
            "subsample": 0.90,
            "subsample_freq": 1,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.0,
            "reg_lambda": 8.0,
            "max_depth": -1,
            "extra_trees": True,
        },
        "stopping_rounds": 2500,
        "log_period": 2500,
    },
]

screen_path = RESULTS_DIR / "lgbm_diversity_holdout_screen_post80p662.csv"

if screen_path.exists():
    screen_df = pd.read_csv(screen_path)
    rows = screen_df.to_dict("records")
    completed = set(screen_df["name"].astype(str)) if "name" in screen_df.columns else set()
    print(f"Loaded existing screen with {len(completed)} completed jobs.")
else:
    rows = []
    completed = set()

reference_t03_holdout_mse = 104.4964
overall_start = time.time()

for job in jobs:
    name = job["name"]

    if name in completed:
        print(f"\nSkipping completed job: {name}")
        continue

    print("\n" + "#" * 90)
    print("Starting:", name)
    print("#" * 90)

    start = time.time()

    model = lgb.LGBMRegressor(**job["params"])
    model.fit(
        X_tr_lgbm,
        y_tr_use,
        eval_set=[(X_val_lgbm, y_val_use)],
        eval_metric="l2",
        feature_name=safe_feature_names,
        callbacks=[
            lgb.early_stopping(job["stopping_rounds"], verbose=True),
            lgb.log_evaluation(period=job["log_period"]),
        ],
    )

    best_iter = model.best_iteration_ or job["params"]["n_estimators"]

    train_pred = model.predict(X_tr_lgbm, num_iteration=best_iter)
    val_pred = model.predict(X_val_lgbm, num_iteration=best_iter)

    row = {
        "name": name,
        "best_iteration": int(best_iter),
        "train_mse_clipped": mse(y_tr_use, train_pred),
        "valid_mse_clipped": mse(y_val_use, val_pred),
        "valid_pred_mean": float(np.mean(clip100(val_pred))),
        "valid_pred_std": float(np.std(clip100(val_pred))),
        "elapsed_sec": float(time.time() - start),
        "delta_vs_old_t03_holdout": mse(y_val_use, val_pred) - reference_t03_holdout_mse,
    }

    for k, v in job["params"].items():
        row[f"param_{k}"] = v

    rows.append(row)

    screen_df = pd.DataFrame(rows).sort_values("valid_mse_clipped").reset_index(drop=True)
    screen_df.to_csv(screen_path, index=False)

    print("\nCompleted:", name)
    print("Best iteration:", row["best_iteration"])
    print("Validation MSE clipped:", row["valid_mse_clipped"])
    print("Delta vs old T03 holdout:", row["delta_vs_old_t03_holdout"])
    print("Saved checkpoint:", screen_path)

    del model, train_pred, val_pred
    gc.collect()

print("\nFinal LightGBM diversity holdout screen:")
screen_df = pd.DataFrame(rows).sort_values("valid_mse_clipped").reset_index(drop=True)
screen_df.to_csv(screen_path, index=False)

try:
    display(screen_df)
except NameError:
    print(screen_df.to_string(index=False))

print("\nScreen path:", screen_path)
print("Total elapsed minutes:", round((time.time() - overall_start) / 60, 2))

Using existing X_tr / X_val split.
Train shape: (115936, 162)
Validation shape: (28985, 162)

##########################################################################################
Starting: lgbm_div01_l63_child80_l2_10_lr02
##########################################################################################
Training until validation scores don't improve for 2500 rounds
[2500]	valid_0's l2: 145.07
[5000]	valid_0's l2: 130.559
[7500]	valid_0's l2: 122.539
[10000]	valid_0's l2: 116.925
[12500]	valid_0's l2: 113.127
[15000]	valid_0's l2: 110.18
[17500]	valid_0's l2: 107.911
[20000]	valid_0's l2: 106.053
[22500]	valid_0's l2: 104.635
[25000]	valid_0's l2: 103.55
[27500]	valid_0's l2: 102.57
[30000]	valid_0's l2: 101.815
[32500]	valid_0's l2: 101.095
[35000]	valid_0's l2: 100.581
[37500]	valid_0's l2: 100.137
[40000]	valid_0's l2: 99.7268
[42500]	valid_0's l2: 99.404
[45000]	valid_0's l2: 99.0987
[47500]	valid_0's l2: 98.8982
[50000]	valid_0's l2: 98.7031
[52500]	valid_0's l2: 98.

,name,best_iteration,train_mse_clipped,valid_mse_clipped,valid_pred_mean,valid_pred_std,elapsed_sec,delta_vs_old_t03_holdout,param_objective,param_metric,param_random_state,param_n_jobs,param_verbosity,param_force_col_wise,param_learning_rate,param_n_estimators,param_num_leaves,param_min_child_samples,param_subsample,param_subsample_freq,param_colsample_bytree,param_reg_alpha,param_reg_lambda,param_max_depth,param_extra_trees
0,lgbm_div02_l127_child50_l2_15_lr02,47124,3.485951,95.127949,54.119015,24.777036,3551.669870,-9.368451,regression,l2,9890,1,-1,True,0.020,80000,127,50,0.80,1,0.85,0.00,15.0,-1,NaN
1,lgbm_div01_l63_child80_l2_10_lr02,69974,7.093457,96.932557,54.121766,24.721065,3513.916109,-7.563843,regression,l2,9890,1,-1,True,0.020,70000,63,80,0.85,1,0.85,0.05,10.0,-1,NaN
2,lgbm_div03_extra_trees_l95_child60_l2_8_lr025,69799,10.281223,97.325755,54.107728,24.648221,4286.634597,-7.170645,regression,l2,9890,1,-1,True,0.025,70000,95,60,0.90,1,0.90,0.00,8.0,-1,True



Screen path: model_results/lgbm_diversity_holdout_screen_post80p662.csv
Total elapsed minutes: 189.22


In [44]:
# 23A. Promote best diversity LightGBM to 3-fold OOF artifact + automatic blend
# Purpose:
# - Train the best holdout diversity candidate with 3-fold OOF.
# - Save fold checkpoints so the cell can resume.
# - If all 3 folds finish, create a fold-averaged submission.
# - Then automatically re-blend RF / ET / old LGBMs / new div02.

import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy import sparse
from scipy.optimize import minimize
from sklearn.model_selection import KFold

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"

artifact = "lgbm_div02_l127_child50_l2_15_lr02_3fold_oof"

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y, p):
    return float(np.mean((np.asarray(y, dtype=float) - clip100(p)) ** 2))

def to_lgbm_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32)
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def get_ids(name, fallback_df_name, expected_len):
    if name in globals():
        s = pd.Series(globals()[name]).astype(str).reset_index(drop=True)
    elif fallback_df_name in globals() and ID_COL in globals()[fallback_df_name].columns:
        s = globals()[fallback_df_name][ID_COL].astype(str).reset_index(drop=True)
    else:
        return None
    if len(s) != expected_len:
        raise ValueError(f"{name} length mismatch: {len(s)} vs {expected_len}")
    return s

X_all = to_lgbm_matrix(X_train_proc_model)
X_test_all = to_lgbm_matrix(X_test_proc_model)
y_all = np.asarray(y_train, dtype=float).ravel()

n_train = X_all.shape[0]
n_test = X_test_all.shape[0]

train_id_series = get_ids("train_ids", "scores_training", n_train)
test_id_series = get_ids("test_ids", "scores_test", n_test)

if test_id_series is None:
    raise ValueError("Could not find test IDs from test_ids or scores_test.")

safe_feature_names = [f"f{i}" for i in range(X_all.shape[1])]

params = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,
    "learning_rate": 0.02,
    "n_estimators": 80000,
    "num_leaves": 127,
    "min_child_samples": 50,
    "subsample": 0.80,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.0,
    "reg_lambda": 15.0,
    "max_depth": -1,
}

metrics_path = RESULTS_DIR / f"{artifact}_fold_metrics.csv"
oof_npy_path = RESULTS_DIR / f"{artifact}_oof.npy"

if oof_npy_path.exists():
    oof_pred = np.load(oof_npy_path)
else:
    oof_pred = np.full(n_train, np.nan, dtype=np.float32)

if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
else:
    metrics_df = pd.DataFrame()

kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

print("Artifact:", artifact)
print("Train matrix:", X_all.shape)
print("Test matrix:", X_test_all.shape)
print("Existing completed folds:", [] if metrics_df.empty else list(metrics_df["fold"]))

overall_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all), start=1):
    fold_test_path = RESULTS_DIR / f"{artifact}_fold{fold}_test_pred.npy"

    already_done = (
        (not metrics_df.empty)
        and (fold in set(metrics_df["fold"]))
        and fold_test_path.exists()
        and (not np.isnan(oof_pred[va_idx]).any())
    )

    if already_done:
        print(f"\nSkipping completed fold {fold}/3")
        continue

    print("\n" + "=" * 80)
    print(f"Starting {artifact}, fold {fold}/3")
    print("=" * 80)

    start = time.time()

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_all[tr_idx],
        y_all[tr_idx],
        eval_set=[(X_all[va_idx], y_all[va_idx])],
        eval_metric="l2",
        feature_name=safe_feature_names,
        callbacks=[
            lgb.early_stopping(stopping_rounds=3000, verbose=True),
            lgb.log_evaluation(period=2500),
        ],
    )

    best_iter = model.best_iteration_ or params["n_estimators"]

    va_pred = model.predict(X_all[va_idx], num_iteration=best_iter)
    test_pred = model.predict(X_test_all, num_iteration=best_iter)

    oof_pred[va_idx] = va_pred.astype(np.float32)
    np.save(oof_npy_path, oof_pred)
    np.save(fold_test_path, test_pred.astype(np.float32))

    row = {
        "fold": fold,
        "fold_valid_mse_clipped": mse_clip(y_all[va_idx], va_pred),
        "fold_valid_mse_raw": float(np.mean((y_all[va_idx] - va_pred) ** 2)),
        "best_iteration": int(best_iter),
        "elapsed_sec": float(time.time() - start),
    }

    metrics_df = metrics_df[metrics_df["fold"] != fold] if not metrics_df.empty else metrics_df
    metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
    metrics_df = metrics_df.sort_values("fold").reset_index(drop=True)
    metrics_df.to_csv(metrics_path, index=False)

    print("\nCompleted fold:", fold)
    print("Fold valid MSE clipped:", row["fold_valid_mse_clipped"])
    print("Best iteration:", row["best_iteration"])
    print("Saved checkpoint:", metrics_path)

    del model, va_pred, test_pred
    gc.collect()

completed = (not np.isnan(oof_pred).any()) and all(
    (RESULTS_DIR / f"{artifact}_fold{fold}_test_pred.npy").exists()
    for fold in range(1, 4)
)

print("\n" + "#" * 90)
print("3-fold artifact completion check")
print("#" * 90)
print("Missing OOF predictions:", int(np.isnan(oof_pred).sum()))
print("Completed:", completed)

if not completed:
    print("\nNot all folds are complete yet. Rerun this same cell later to resume.")
else:
    oof_mse = mse_clip(y_all, oof_pred)
    print("Overall 3-fold OOF MSE clipped:", oof_mse)

    fold_test_preds = [
        np.load(RESULTS_DIR / f"{artifact}_fold{fold}_test_pred.npy")
        for fold in range(1, 4)
    ]
    test_foldavg = np.mean(fold_test_preds, axis=0)

    oof_csv_path = RESULTS_DIR / f"oof_{artifact}.csv"
    testpred_csv_path = RESULTS_DIR / f"testpred_{artifact}_foldavg.csv"
    submission_path = Path(f"submission_{artifact}_foldavg.csv")

    oof_df = pd.DataFrame({
        TARGET_COL: y_all,
        f"OOF_{artifact}": oof_pred
    })
    if train_id_series is not None:
        oof_df.insert(0, ID_COL, train_id_series.values)

    oof_df.to_csv(oof_csv_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        f"TESTPRED_{artifact}": clip100(test_foldavg)
    }).to_csv(testpred_csv_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        TARGET_COL: clip100(test_foldavg)
    }).to_csv(submission_path, index=False)

    print("Saved OOF:", oof_csv_path)
    print("Saved test predictions:", testpred_csv_path)
    print("Saved standalone submission:", submission_path)

    print("\nStandalone div02 3-fold submission summary:")
    print(pd.Series(clip100(test_foldavg)).describe())

    # ------------------------------------------------------------------
    # Automatic convex blend including new div02 artifact
    # ------------------------------------------------------------------

    def read_pred(path, preferred_col):
        df = pd.read_csv(path)
        if preferred_col in df.columns:
            return pd.to_numeric(df[preferred_col], errors="raise").to_numpy(dtype=float)

        numeric_cols = []
        for c in df.columns:
            if c in [ID_COL, TARGET_COL, "fold", "Unnamed: 0"]:
                continue
            vals = pd.to_numeric(df[c], errors="coerce")
            if vals.notna().mean() > 0.95:
                numeric_cols.append(c)

        if len(numeric_cols) == 0:
            raise ValueError(f"No usable prediction column found in {path}")

        print(f"Using fallback column {numeric_cols[-1]} from {path}")
        return pd.to_numeric(df[numeric_cols[-1]], errors="raise").to_numpy(dtype=float)

    components = {
        "rf500": {
            "oof": ("model_results/oof_rf_500_base.csv", "rf_500_base_oof_pred"),
            "test": ("model_results/testpred_rf_500_base_folds.csv", "rf_500_base_foldavg_pred"),
        },
        "et_safe": {
            "oof": ("model_results/oof_extratrees_safe_base.csv", "oof_pred"),
            "test": ("model_results/testpred_extratrees_safe_base_foldavg.csv", TARGET_COL),
        },
        "lgbm12k": {
            "oof": ("model_results/oof_lgbm_t03_base_5fold_oof.csv", "OOF_lgbm_t03_base_5fold_oof"),
            "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof"),
        },
        "lgbm_long80k_lr03": {
            "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv", "OOF_lgbm_t03_base_5fold_oof_long80k_lr03"),
            "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03"),
        },
        "lgbm_long100k_lr02": {
            "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv", "OOF_lgbm_t03_base_5fold_oof_long100k_lr02"),
            "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02"),
        },
        "lgbm_div02_3fold": {
            "oof": (str(oof_csv_path), f"OOF_{artifact}"),
            "test": (str(testpred_csv_path), f"TESTPRED_{artifact}"),
        },
    }

    names = []
    oof_list = []
    test_list = []
    comp_rows = []

    for name, spec in components.items():
        try:
            oof_p = clip100(read_pred(*spec["oof"]))
            test_p = clip100(read_pred(*spec["test"]))

            if len(oof_p) != n_train or len(test_p) != n_test:
                print(f"Skipping {name}: length mismatch")
                continue

            names.append(name)
            oof_list.append(oof_p)
            test_list.append(test_p)

            comp_rows.append({
                "component": name,
                "oof_mse_clipped": mse_clip(y_all, oof_p),
                "oof_mean": float(np.mean(oof_p)),
                "oof_std": float(np.std(oof_p)),
                "test_mean": float(np.mean(test_p)),
                "test_std": float(np.std(test_p)),
            })

        except Exception as e:
            print(f"Skipping {name}: {e}")

    Xo = np.column_stack(oof_list)
    Xt = np.column_stack(test_list)

    def blend_loss(w):
        return mse_clip(y_all, Xo @ w)

    starts = []

    starts.append(np.ones(len(names)) / len(names))

    known = np.zeros(len(names))
    if "et_safe" in names and "lgbm_long100k_lr02" in names:
        known[names.index("et_safe")] = 0.319
        known[names.index("lgbm_long100k_lr02")] = 0.681
        starts.append(known)

    for i in range(len(names)):
        pure = np.zeros(len(names))
        pure[i] = 1.0
        starts.append(pure)

    best_res = None
    for x0 in starts:
        res = minimize(
            blend_loss,
            x0=x0,
            method="SLSQP",
            bounds=[(0.0, 1.0)] * len(names),
            constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
            options={"maxiter": 1000, "ftol": 1e-12},
        )
        if best_res is None or res.fun < best_res.fun:
            best_res = res

    weights = np.asarray(best_res.x, dtype=float)
    blend_oof = Xo @ weights
    blend_test = Xt @ weights

    blend_name = "blend_auto_with_lgbm_div02_3fold_oof_weighted"

    weight_df = pd.DataFrame({
        "component": names,
        "weight": weights,
    }).sort_values("weight", ascending=False)

    comp_df = pd.DataFrame(comp_rows).sort_values("oof_mse_clipped")

    weight_path = RESULTS_DIR / f"{blend_name}_weights.csv"
    testblend_path = RESULTS_DIR / f"testpred_{blend_name}.csv"
    blend_submission_path = Path(f"submission_{blend_name}.csv")

    weight_df.to_csv(weight_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        f"TESTPRED_{blend_name}": clip100(blend_test)
    }).to_csv(testblend_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        TARGET_COL: clip100(blend_test)
    }).to_csv(blend_submission_path, index=False)

    print("\nBlend component OOF results:")
    try:
        display(comp_df)
    except NameError:
        print(comp_df.to_string(index=False))

    print("\nBest blend weights:")
    try:
        display(weight_df)
    except NameError:
        print(weight_df.to_string(index=False))

    print("\nBest blend OOF MSE clipped:", mse_clip(y_all, blend_oof))
    print("Previous submitted blend OOF MSE was about: 88.9664")
    print("Saved weights:", weight_path)
    print("Saved blend test predictions:", testblend_path)
    print("Saved blend submission:", blend_submission_path)

    print("\nBlend submission summary:")
    print(pd.Series(clip100(blend_test)).describe())

print("\nTotal elapsed minutes:", round((time.time() - overall_start) / 60, 2))

Artifact: lgbm_div02_l127_child50_l2_15_lr02_3fold_oof
Train matrix: (144921, 162)
Test matrix: (48307, 162)
Existing completed folds: []

Starting lgbm_div02_l127_child50_l2_15_lr02_3fold_oof, fold 1/3
Training until validation scores don't improve for 3000 rounds
[2500]	valid_0's l2: 131.203
[5000]	valid_0's l2: 119.548
[7500]	valid_0's l2: 113.556
[10000]	valid_0's l2: 110.141
[12500]	valid_0's l2: 108.057
[15000]	valid_0's l2: 106.683
[17500]	valid_0's l2: 105.74
[20000]	valid_0's l2: 105.083
[22500]	valid_0's l2: 104.662
[25000]	valid_0's l2: 104.348
[27500]	valid_0's l2: 104.128
[30000]	valid_0's l2: 103.975
[32500]	valid_0's l2: 103.875
[35000]	valid_0's l2: 103.791
[37500]	valid_0's l2: 103.718
[40000]	valid_0's l2: 103.715
[42500]	valid_0's l2: 103.706
Early stopping, best iteration is:
[41059]	valid_0's l2: 103.693

Completed fold: 1
Fold valid MSE clipped: 102.8387506655704
Best iteration: 41059
Saved checkpoint: model_results/lgbm_div02_l127_child50_l2_15_lr02_3fold_oof_fol

,component,oof_mse_clipped,oof_mean,oof_std,test_mean,test_std
4,lgbm_long100k_lr02,93.726214,54.153144,24.734667,54.074208,24.605725
3,lgbm_long80k_lr03,94.163485,54.147019,24.743266,54.067601,24.607933
2,lgbm12k,98.779406,54.152808,24.283770,54.086671,24.207260
5,lgbm_div02_3fold,104.083036,54.131471,24.596497,54.084634,24.384543
1,et_safe,110.749294,54.127367,23.857745,54.040173,23.782901
0,rf500,122.328024,54.134143,22.839348,54.056767,22.817463



Best blend weights:


,component,weight
4,lgbm_long100k_lr02,0.391029
1,et_safe,0.308273
3,lgbm_long80k_lr03,0.220189
5,lgbm_div02_3fold,0.080509
0,rf500,0.000000
2,lgbm12k,0.000000



Best blend OOF MSE clipped: 88.76684691394617
Previous submitted blend OOF MSE was about: 88.9664
Saved weights: model_results/blend_auto_with_lgbm_div02_3fold_oof_weighted_weights.csv
Saved blend test predictions: model_results/testpred_blend_auto_with_lgbm_div02_3fold_oof_weighted.csv
Saved blend submission: submission_blend_auto_with_lgbm_div02_3fold_oof_weighted.csv

Blend submission summary:
count    48307.000000
mean        54.063100
std         24.170248
min          0.074479
25%         35.017689
50%         52.562427
75%         73.413524
max        100.000000
dtype: float64

Total elapsed minutes: 117.43


In [45]:
# 24A. Conservative residual stack on top of the current best 80.662 blend
# Goal:
# - Use the current best submitted blend as the base prediction.
# - Train a small LightGBM residual model with proper OOF validation.
# - Choose a conservative shrinkage factor for the residual correction.
# - Save candidate submissions, but do NOT submit automatically.

import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy import sparse
from sklearn.model_selection import KFold

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"

artifact = "residstack_lgbm_on_best80p662_blend_5fold"

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y, p):
    return float(np.mean((np.asarray(y, dtype=float) - clip100(p)) ** 2))

def read_pred(path, preferred_col):
    df = pd.read_csv(path)

    if preferred_col in df.columns:
        return pd.to_numeric(df[preferred_col], errors="raise").to_numpy(dtype=float)

    numeric_cols = []
    for c in df.columns:
        if c in [ID_COL, TARGET_COL, "fold", "Unnamed: 0"]:
            continue
        vals = pd.to_numeric(df[c], errors="coerce")
        if vals.notna().mean() > 0.95:
            numeric_cols.append(c)

    if not numeric_cols:
        raise ValueError(f"No prediction column found in {path}")

    print(f"Using fallback prediction column {numeric_cols[-1]} from {path}")
    return pd.to_numeric(df[numeric_cols[-1]], errors="raise").to_numpy(dtype=float)

def get_test_ids():
    if "test_ids" in globals():
        obj = globals()["test_ids"]
        if isinstance(obj, pd.DataFrame):
            if ID_COL in obj.columns:
                return obj[ID_COL].astype(str).reset_index(drop=True)
            return obj.iloc[:, 0].astype(str).reset_index(drop=True)
        return pd.Series(obj).astype(str).reset_index(drop=True)

    if "scores_test" in globals() and ID_COL in scores_test.columns:
        return scores_test[ID_COL].astype(str).reset_index(drop=True)

    raise ValueError("Could not find test IDs.")

def to_float_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32).tocsr()
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def add_meta_features(X, M):
    M = np.asarray(M, dtype=np.float32)
    if sparse.issparse(X):
        return sparse.hstack([X, sparse.csr_matrix(M)], format="csr")
    return np.hstack([X, M])

# ---------------------------------------------------------------------
# Load base matrices and saved OOF/test prediction artifacts
# ---------------------------------------------------------------------

X_base = to_float_matrix(X_train_proc_model)
X_test_base = to_float_matrix(X_test_proc_model)
y = np.asarray(y_train, dtype=float).ravel()
test_id_series = get_test_ids()

components = {
    "rf500": {
        "oof": ("model_results/oof_rf_500_base.csv", "rf_500_base_oof_pred"),
        "test": ("model_results/testpred_rf_500_base_folds.csv", "rf_500_base_foldavg_pred"),
    },
    "et_safe": {
        "oof": ("model_results/oof_extratrees_safe_base.csv", "oof_pred"),
        "test": ("model_results/testpred_extratrees_safe_base_foldavg.csv", TARGET_COL),
    },
    "lgbm12k": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof.csv", "OOF_lgbm_t03_base_5fold_oof"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof"),
    },
    "lgbm_long80k_lr03": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv", "OOF_lgbm_t03_base_5fold_oof_long80k_lr03"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03"),
    },
    "lgbm_long100k_lr02": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv", "OOF_lgbm_t03_base_5fold_oof_long100k_lr02"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02"),
    },
    "lgbm_div02_3fold": {
        "oof": ("model_results/oof_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof.csv", "OOF_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof"),
        "test": ("model_results/testpred_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof_foldavg.csv", "TESTPRED_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof"),
    },
}

oof_preds = {}
test_preds = {}

for name, spec in components.items():
    try:
        oof_p = clip100(read_pred(*spec["oof"]))
        test_p = clip100(read_pred(*spec["test"]))

        if len(oof_p) != len(y) or len(test_p) != len(test_id_series):
            print(f"Skipping {name}: length mismatch")
            continue

        oof_preds[name] = oof_p
        test_preds[name] = test_p
        print(f"Loaded {name}: OOF MSE = {mse_clip(y, oof_p):.6f}")

    except Exception as e:
        print(f"Skipping {name}: {e}")

if "et_safe" not in oof_preds or "lgbm_long100k_lr02" not in oof_preds:
    raise ValueError("Need et_safe and lgbm_long100k_lr02 for the current best blend.")

# Current best submitted 80.662 blend
base_oof = 0.319 * oof_preds["et_safe"] + 0.681 * oof_preds["lgbm_long100k_lr02"]
base_test = 0.319 * test_preds["et_safe"] + 0.681 * test_preds["lgbm_long100k_lr02"]

base_mse = mse_clip(y, base_oof)
print("\nCurrent best submitted blend OOF MSE:", base_mse)

# ---------------------------------------------------------------------
# Build meta features
# ---------------------------------------------------------------------

component_order = list(oof_preds.keys())

M_train_main = np.column_stack([oof_preds[name] for name in component_order])
M_test_main = np.column_stack([test_preds[name] for name in component_order])

lgbm_names = [n for n in component_order if n.startswith("lgbm")]
lgbm_train = np.column_stack([oof_preds[n] for n in lgbm_names])
lgbm_test = np.column_stack([test_preds[n] for n in lgbm_names])

M_train_extra = np.column_stack([
    base_oof,
    M_train_main.mean(axis=1),
    M_train_main.std(axis=1),
    M_train_main.max(axis=1) - M_train_main.min(axis=1),
    lgbm_train.mean(axis=1),
    lgbm_train.std(axis=1),
])

M_test_extra = np.column_stack([
    base_test,
    M_test_main.mean(axis=1),
    M_test_main.std(axis=1),
    M_test_main.max(axis=1) - M_test_main.min(axis=1),
    lgbm_test.mean(axis=1),
    lgbm_test.std(axis=1),
])

M_train = np.hstack([M_train_main, M_train_extra]).astype(np.float32)
M_test = np.hstack([M_test_main, M_test_extra]).astype(np.float32)

Z_train = add_meta_features(X_base, M_train)
Z_test = add_meta_features(X_test_base, M_test)

residual_y = y - base_oof

print("Residual target summary:")
print(pd.Series(residual_y).describe())

print("\nMeta training matrix:", Z_train.shape)
print("Meta test matrix:", Z_test.shape)

# ---------------------------------------------------------------------
# 5-fold residual model with checkpointing
# ---------------------------------------------------------------------

params = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,
    "learning_rate": 0.03,
    "n_estimators": 20000,
    "num_leaves": 31,
    "max_depth": 6,
    "min_child_samples": 250,
    "subsample": 0.80,
    "subsample_freq": 1,
    "colsample_bytree": 0.80,
    "reg_alpha": 1.0,
    "reg_lambda": 50.0,
}

resid_oof_path = RESULTS_DIR / f"{artifact}_resid_oof.npy"
metrics_path = RESULTS_DIR / f"{artifact}_fold_metrics.csv"

if resid_oof_path.exists():
    resid_oof = np.load(resid_oof_path)
else:
    resid_oof = np.full(len(y), np.nan, dtype=np.float32)

if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
else:
    metrics_df = pd.DataFrame()

safe_feature_names = [f"f{i}" for i in range(Z_train.shape[1])]
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

overall_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(Z_train), start=1):
    fold_test_path = RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy"

    already_done = (
        (not metrics_df.empty)
        and (fold in set(metrics_df["fold"]))
        and fold_test_path.exists()
        and (not np.isnan(resid_oof[va_idx]).any())
    )

    if already_done:
        print(f"\nSkipping completed fold {fold}/5")
        continue

    print("\n" + "=" * 80)
    print(f"Starting residual stack fold {fold}/5")
    print("=" * 80)

    start = time.time()

    model = lgb.LGBMRegressor(**params)
    model.fit(
        Z_train[tr_idx],
        residual_y[tr_idx],
        eval_set=[(Z_train[va_idx], residual_y[va_idx])],
        eval_metric="l2",
        feature_name=safe_feature_names,
        callbacks=[
            lgb.early_stopping(stopping_rounds=1000, verbose=True),
            lgb.log_evaluation(period=1000),
        ],
    )

    best_iter = model.best_iteration_ or params["n_estimators"]

    va_resid_pred = model.predict(Z_train[va_idx], num_iteration=best_iter)
    test_resid_pred = model.predict(Z_test, num_iteration=best_iter)

    resid_oof[va_idx] = va_resid_pred.astype(np.float32)
    np.save(resid_oof_path, resid_oof)
    np.save(fold_test_path, test_resid_pred.astype(np.float32))

    row = {
        "fold": fold,
        "best_iteration": int(best_iter),
        "residual_valid_mse": float(np.mean((residual_y[va_idx] - va_resid_pred) ** 2)),
        "corrected_mse_lambda_1": mse_clip(y[va_idx], base_oof[va_idx] + va_resid_pred),
        "base_mse_on_fold": mse_clip(y[va_idx], base_oof[va_idx]),
        "elapsed_sec": float(time.time() - start),
    }

    metrics_df = metrics_df[metrics_df["fold"] != fold] if not metrics_df.empty else metrics_df
    metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
    metrics_df = metrics_df.sort_values("fold").reset_index(drop=True)
    metrics_df.to_csv(metrics_path, index=False)

    print("Completed fold:", fold)
    print("Base fold MSE:", row["base_mse_on_fold"])
    print("Corrected fold MSE with lambda=1:", row["corrected_mse_lambda_1"])
    print("Best iteration:", row["best_iteration"])
    print("Saved checkpoint:", metrics_path)

    del model, va_resid_pred, test_resid_pred
    gc.collect()

completed = (not np.isnan(resid_oof).any()) and all(
    (RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy").exists()
    for fold in range(1, 6)
)

print("\nCompleted residual stack:", completed)
print("Missing residual OOF predictions:", int(np.isnan(resid_oof).sum()))

if completed:
    test_resids = [
        np.load(RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy")
        for fold in range(1, 6)
    ]
    resid_test = np.mean(test_resids, axis=0)

    lambda_grid = np.linspace(0.0, 0.60, 121)
    lambda_rows = []

    for lam in lambda_grid:
        pred = base_oof + lam * resid_oof
        lambda_rows.append({
            "lambda": float(lam),
            "oof_mse_clipped": mse_clip(y, pred),
            "gain_vs_base_oof": base_mse - mse_clip(y, pred),
            "pred_mean": float(np.mean(clip100(pred))),
            "pred_std": float(np.std(clip100(pred))),
        })

    lambda_df = pd.DataFrame(lambda_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
    lambda_path = RESULTS_DIR / f"{artifact}_lambda_screen.csv"
    lambda_df.to_csv(lambda_path, index=False)

    print("\nResidual shrinkage screen:")
    try:
        display(lambda_df.head(15))
    except NameError:
        print(lambda_df.head(15).to_string(index=False))

    best_lambda = float(lambda_df.loc[0, "lambda"])
    conservative_lambda = round(best_lambda * 0.5, 4)

    save_lambdas = sorted(set([
        best_lambda,
        conservative_lambda,
        0.10,
        0.20,
        0.30,
    ]))

    saved_rows = []

    for lam in save_lambdas:
        final_oof = clip100(base_oof + lam * resid_oof)
        final_test = clip100(base_test + lam * resid_test)

        name = f"{artifact}_lambda_{str(lam).replace('.', 'p')}"
        oof_path = RESULTS_DIR / f"oof_{name}.csv"
        testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
        sub_path = Path(f"submission_{name}.csv")

        pd.DataFrame({
            TARGET_COL: y,
            f"OOF_{name}": final_oof
        }).to_csv(oof_path, index=False)

        pd.DataFrame({
            ID_COL: test_id_series.values,
            f"TESTPRED_{name}": final_test
        }).to_csv(testpred_path, index=False)

        pd.DataFrame({
            ID_COL: test_id_series.values,
            TARGET_COL: final_test
        }).to_csv(sub_path, index=False)

        saved_rows.append({
            "lambda": lam,
            "oof_mse_clipped": mse_clip(y, final_oof),
            "gain_vs_base_oof": base_mse - mse_clip(y, final_oof),
            "submission_path": str(sub_path),
            "test_mean": float(np.mean(final_test)),
            "test_std": float(np.std(final_test)),
            "test_min": float(np.min(final_test)),
            "test_max": float(np.max(final_test)),
        })

    saved_df = pd.DataFrame(saved_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
    saved_path = RESULTS_DIR / f"{artifact}_saved_candidates.csv"
    saved_df.to_csv(saved_path, index=False)

    print("\nSaved residual-stack candidate submissions:")
    try:
        display(saved_df)
    except NameError:
        print(saved_df.to_string(index=False))

    corr = np.corrcoef(residual_y, resid_oof)[0, 1]
    print("\nResidual OOF prediction correlation with true residual:", corr)
    print("Base OOF MSE:", base_mse)
    print("Best residual-stack OOF MSE:", float(saved_df.loc[0, "oof_mse_clipped"]))
    print("Best OOF gain:", float(saved_df.loc[0, "gain_vs_base_oof"]))
    print("Lambda screen:", lambda_path)
    print("Saved candidates:", saved_path)

print("\nTotal elapsed minutes:", round((time.time() - overall_start) / 60, 2))

Loaded rf500: OOF MSE = 122.328024
Loaded et_safe: OOF MSE = 110.749294
Loaded lgbm12k: OOF MSE = 98.779406
Loaded lgbm_long80k_lr03: OOF MSE = 94.163485
Loaded lgbm_long100k_lr02: OOF MSE = 93.726214
Loaded lgbm_div02_3fold: OOF MSE = 104.083036

Current best submitted blend OOF MSE: 88.96636822734054
Residual target summary:
count    144921.000000
mean          0.038600
std           9.432152
min         -74.711172
25%          -4.382987
50%           0.127390
75%           4.493513
max          84.650332
dtype: float64

Meta training matrix: (144921, 174)
Meta test matrix: (48307, 174)

Starting residual stack fold 1/5
Training until validation scores don't improve for 1000 rounds
[1000]	valid_0's l2: 89.8431
Early stopping, best iteration is:
[248]	valid_0's l2: 88.7583
Completed fold: 1
Base fold MSE: 90.31439437895932
Corrected fold MSE with lambda=1: 88.75596944176321
Best iteration: 248
Saved checkpoint: model_results/residstack_lgbm_on_best80p662_blend_5fold_fold_metrics.csv



,lambda,oof_mse_clipped,gain_vs_base_oof,pred_mean,pred_std
0,0.600,87.543031,1.423337,54.167175,24.526060
1,0.595,87.550667,1.415702,54.166992,24.523684
2,0.590,87.558373,1.407995,54.166809,24.521309
3,0.585,87.566151,1.400217,54.166626,24.518935
4,0.580,87.573999,1.392369,54.166443,24.516562
5,0.575,87.581919,1.384449,54.166260,24.514191
6,0.570,87.589909,1.376459,54.166077,24.511821
7,0.565,87.597971,1.368397,54.165894,24.509451
8,0.560,87.606104,1.360265,54.165711,24.507083
9,0.555,87.614307,1.352061,54.165527,24.504716



Saved residual-stack candidate submissions:


,lambda,oof_mse_clipped,gain_vs_base_oof,submission_path,test_mean,test_std,test_min,test_max
0,0.6,87.543031,1.423337,submission_residstack_lgbm_on_best80p662_blend...,54.081737,24.436239,0.148440,100.0
1,0.3,88.126863,0.839505,submission_residstack_lgbm_on_best80p662_blend...,54.072594,24.305647,0.125898,100.0
2,0.2,88.378293,0.588075,submission_residstack_lgbm_on_best80p662_blend...,54.069531,24.262751,0.118384,100.0
3,0.1,88.658121,0.308247,submission_residstack_lgbm_on_best80p662_blend...,54.066449,24.220152,0.110870,100.0



Residual OOF prediction correlation with true residual: 0.14331489148998913
Base OOF MSE: 88.96636822734054
Best residual-stack OOF MSE: 87.54303119204958
Best OOF gain: 1.423337035290956
Lambda screen: model_results/residstack_lgbm_on_best80p662_blend_5fold_lambda_screen.csv
Saved candidates: model_results/residstack_lgbm_on_best80p662_blend_5fold_saved_candidates.csv

Total elapsed minutes: 1.42


In [46]:
# 24B. Expanded residual-stack lambda screen
# No model training here. This reuses the saved residual OOF/test predictions from 24A.

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"

artifact = "residstack_lgbm_on_best80p662_blend_5fold"

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y, p):
    return float(np.mean((np.asarray(y, dtype=float) - clip100(p)) ** 2))

def read_pred(path, preferred_col):
    df = pd.read_csv(path)

    if preferred_col in df.columns:
        return pd.to_numeric(df[preferred_col], errors="raise").to_numpy(dtype=float)

    numeric_cols = []
    for c in df.columns:
        if c in [ID_COL, TARGET_COL, "fold", "Unnamed: 0"]:
            continue
        vals = pd.to_numeric(df[c], errors="coerce")
        if vals.notna().mean() > 0.95:
            numeric_cols.append(c)

    if not numeric_cols:
        raise ValueError(f"No prediction column found in {path}")

    print(f"Using fallback prediction column {numeric_cols[-1]} from {path}")
    return pd.to_numeric(df[numeric_cols[-1]], errors="raise").to_numpy(dtype=float)

def get_test_ids():
    if "test_ids" in globals():
        obj = globals()["test_ids"]
        if isinstance(obj, pd.DataFrame):
            if ID_COL in obj.columns:
                return obj[ID_COL].astype(str).reset_index(drop=True)
            return obj.iloc[:, 0].astype(str).reset_index(drop=True)
        return pd.Series(obj).astype(str).reset_index(drop=True)

    if "scores_test" in globals() and ID_COL in scores_test.columns:
        return scores_test[ID_COL].astype(str).reset_index(drop=True)

    raise ValueError("Could not find test IDs.")

y = np.asarray(y_train, dtype=float).ravel()
test_id_series = get_test_ids()

# Rebuild the current best submitted 80.662 blend.
et_oof = clip100(read_pred("model_results/oof_extratrees_safe_base.csv", "oof_pred"))
et_test = clip100(read_pred("model_results/testpred_extratrees_safe_base_foldavg.csv", TARGET_COL))

lgb_oof = clip100(read_pred(
    "model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv",
    "OOF_lgbm_t03_base_5fold_oof_long100k_lr02"
))
lgb_test = clip100(read_pred(
    "model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv",
    "TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02"
))

base_oof = 0.319 * et_oof + 0.681 * lgb_oof
base_test = 0.319 * et_test + 0.681 * lgb_test

base_mse = mse_clip(y, base_oof)

# Load residual OOF and averaged test residuals from 24A.
resid_oof = np.load(RESULTS_DIR / f"{artifact}_resid_oof.npy")

resid_test_parts = []
for fold in range(1, 6):
    p = RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy"
    if not p.exists():
        raise FileNotFoundError(f"Missing residual test prediction file: {p}")
    resid_test_parts.append(np.load(p))

resid_test = np.mean(resid_test_parts, axis=0)

print("Base OOF MSE:", base_mse)
print("Residual OOF shape:", resid_oof.shape)
print("Residual test shape:", resid_test.shape)

# Expanded lambda screen.
lambda_grid = np.round(np.arange(0.00, 1.505, 0.005), 3)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(kf.split(base_oof))

rows = []

for lam in lambda_grid:
    oof_pred = base_oof + lam * resid_oof
    overall_mse = mse_clip(y, oof_pred)

    fold_gains = []
    fold_mses = []

    for fold, (_, va_idx) in enumerate(fold_indices, start=1):
        fold_base_mse = mse_clip(y[va_idx], base_oof[va_idx])
        fold_new_mse = mse_clip(y[va_idx], oof_pred[va_idx])
        fold_gains.append(fold_base_mse - fold_new_mse)
        fold_mses.append(fold_new_mse)

    test_pred = clip100(base_test + lam * resid_test)

    rows.append({
        "lambda": float(lam),
        "oof_mse_clipped": overall_mse,
        "gain_vs_base_oof": base_mse - overall_mse,
        "min_fold_gain": float(np.min(fold_gains)),
        "mean_fold_gain": float(np.mean(fold_gains)),
        "max_fold_gain": float(np.max(fold_gains)),
        "fold_mse_std": float(np.std(fold_mses, ddof=1)),
        "test_mean": float(np.mean(test_pred)),
        "test_std": float(np.std(test_pred)),
        "test_min": float(np.min(test_pred)),
        "test_max": float(np.max(test_pred)),
    })

screen_df = pd.DataFrame(rows).sort_values("oof_mse_clipped").reset_index(drop=True)
screen_path = RESULTS_DIR / f"{artifact}_expanded_lambda_screen.csv"
screen_df.to_csv(screen_path, index=False)

print("\nTop expanded lambda candidates:")
try:
    display(screen_df.head(25))
except NameError:
    print(screen_df.head(25).to_string(index=False))

best_lambda = float(screen_df.loc[0, "lambda"])
print("\nBest lambda:", best_lambda)
print("Best OOF MSE:", float(screen_df.loc[0, "oof_mse_clipped"]))
print("Best OOF gain vs base:", float(screen_df.loc[0, "gain_vs_base_oof"]))
print("Minimum fold gain at best lambda:", float(screen_df.loc[0, "min_fold_gain"]))

# Save a few sensible candidate submissions.
candidate_lambdas = sorted(set([
    0.60,
    0.80,
    1.00,
    round(best_lambda, 3),
    round(0.75 * best_lambda, 3),
]))

saved_rows = []

for lam in candidate_lambdas:
    final_oof = clip100(base_oof + lam * resid_oof)
    final_test = clip100(base_test + lam * resid_test)

    name = f"{artifact}_expanded_lambda_{str(lam).replace('.', 'p')}"

    oof_path = RESULTS_DIR / f"oof_{name}.csv"
    testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
    sub_path = Path(f"submission_{name}.csv")

    pd.DataFrame({
        TARGET_COL: y,
        f"OOF_{name}": final_oof
    }).to_csv(oof_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        f"TESTPRED_{name}": final_test
    }).to_csv(testpred_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        TARGET_COL: final_test
    }).to_csv(sub_path, index=False)

    saved_rows.append({
        "lambda": lam,
        "oof_mse_clipped": mse_clip(y, final_oof),
        "gain_vs_base_oof": base_mse - mse_clip(y, final_oof),
        "submission_path": str(sub_path),
        "test_mean": float(np.mean(final_test)),
        "test_std": float(np.std(final_test)),
        "test_min": float(np.min(final_test)),
        "test_max": float(np.max(final_test)),
    })

saved_df = pd.DataFrame(saved_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
saved_path = RESULTS_DIR / f"{artifact}_expanded_saved_candidates.csv"
saved_df.to_csv(saved_path, index=False)

print("\nSaved expanded residual-stack candidates:")
try:
    display(saved_df)
except NameError:
    print(saved_df.to_string(index=False))

print("\nScreen saved to:", screen_path)
print("Saved candidates list:", saved_path)

Base OOF MSE: 88.96636822734054
Residual OOF shape: (144921,)
Residual test shape: (48307,)

Top expanded lambda candidates:


,lambda,oof_mse_clipped,gain_vs_base_oof,min_fold_gain,mean_fold_gain,max_fold_gain,fold_mse_std,test_mean,test_std,test_min,test_max
0,1.135,87.136049,1.830319,1.552019,1.830321,2.114976,1.919453,54.097842,24.676033,0.188640,100.0
1,1.140,87.136071,1.830297,1.550795,1.830299,2.115073,1.919480,54.097991,24.678314,0.189016,100.0
2,1.130,87.136097,1.830271,1.553172,1.830273,2.114798,1.919418,54.097693,24.673752,0.188265,100.0
3,1.145,87.136164,1.830204,1.549500,1.830206,2.115089,1.919500,54.098139,24.680596,0.189392,100.0
4,1.125,87.136217,1.830152,1.554254,1.830153,2.114540,1.919376,54.097543,24.671472,0.187889,100.0
5,1.150,87.136329,1.830040,1.548135,1.830042,2.115024,1.919513,54.098288,24.682879,0.189768,100.0
6,1.120,87.136407,1.829961,1.555266,1.829963,2.114201,1.919326,54.097394,24.669192,0.187513,100.0
7,1.155,87.136564,1.829805,1.546699,1.829806,2.114879,1.919518,54.098437,24.685163,0.190143,100.0
8,1.115,87.136668,1.829700,1.556207,1.829702,2.113781,1.919269,54.097245,24.666913,0.187138,100.0
9,1.160,87.136870,1.829499,1.545192,1.829500,2.114653,1.919516,54.098586,24.687448,0.190519,100.0



Best lambda: 1.135
Best OOF MSE: 87.1360487514482
Best OOF gain vs base: 1.8303194758923382
Minimum fold gain at best lambda: 1.552018866620017

Saved expanded residual-stack candidates:


,lambda,oof_mse_clipped,gain_vs_base_oof,submission_path,test_mean,test_std,test_min,test_max
0,1.135,87.136049,1.830319,submission_residstack_lgbm_on_best80p662_blend...,54.097842,24.676033,0.188640,100.0
1,1.000,87.162181,1.804188,submission_residstack_lgbm_on_best80p662_blend...,54.093804,24.614705,0.178496,100.0
2,0.851,87.250959,1.715410,submission_residstack_lgbm_on_best80p662_blend...,54.089329,24.547659,0.167301,100.0
3,0.800,87.295825,1.670544,submission_residstack_lgbm_on_best80p662_blend...,54.087791,24.524863,0.163468,100.0
4,0.600,87.543031,1.423337,submission_residstack_lgbm_on_best80p662_blend...,54.081737,24.436239,0.148440,100.0



Screen saved to: model_results/residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_screen.csv
Saved candidates list: model_results/residstack_lgbm_on_best80p662_blend_5fold_expanded_saved_candidates.csv


### Expanded Residual Stack on Best 80.662 Blend

This experiment fit a residual model on top of the current best submitted blend, whose OOF MSE was 88.9664. The expanded residual-stack screen improved the best clipped OOF MSE to 87.1360 at lambda = 1.135, corresponding to an OOF gain of approximately 1.83 MSE points. Importantly, the improvement was positive on every fold, with the minimum fold-level gain around 1.55, suggesting that the residual correction is not driven by a single lucky validation fold.

The resulting test prediction distribution appears reasonable, with mean near 54.10, standard deviation near 24.68, and predictions clipped to the valid [0, 100] target range. However, because this model is a residual correction on top of the already-best ET/LGBM blend, it may still be exploiting similar structure rather than adding a clearly independent source of predictive signal. Therefore, this candidate is saved as a strong backlog submission candidate, but it is not submitted immediately.

The next priority is to seek genuinely different model diversity through the planned div02 / diversity-style OOF artifact and blend experiment, rather than continuing to tune small residual-stack or lambda refinements.

### Kaggle result: expanded residual stack on best 80.662 blend

The expanded residual-stack candidate was submitted as:

`submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv`

This submission improved the public leaderboard MSE from the prior best `80.662` to `77.277`.

This is a meaningful improvement, so the residual-stack model is now the new leaderboard anchor. The result also confirms that the previous ET + long LightGBM blend had systematic residual structure that could be learned from the existing training features and model-prediction meta-features.

The best internal lambda-screen candidate used `lambda = 1.135`, with OOF MSE approximately `87.1360`, OOF gain approximately `1.8303` versus the old base blend, and positive fold-level gains across all folds. Because the public leaderboard also improved substantially, future work should focus on adding diversity to the residual-correction layer rather than returning to small base-blend or lambda-only tuning.

In [48]:
# ============================================================
# 25A. Residual-stack diversity model after 77.277 public score
# ============================================================
#
# Current confirmed public anchor:
# submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv
# Public MSE: 77.277
#
# Goal:
# - Do NOT tune only lambda again.
# - Do NOT build a stage-2 residual on top of the 77.277 model yet.
# - Instead, train a different first-level residual correction on the original
#   80.662 base blend residuals.
# - Then blend the original successful residual correction with this new diverse
#   residual correction using OOF weights.
#
# Submit only if this cell finds a real OOF gain versus the 1.135 residual-stack
# anchor, preferably with positive fold-gain diagnostics.

import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy import sparse
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"

PUBLIC_ANCHOR_FILE = "submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv"
PUBLIC_ANCHOR_SCORE = 77.277

old_resid_artifact = "residstack_lgbm_on_best80p662_blend_5fold"
new_artifact = "residstack_huber_extra_on_best80p662_blend_5fold"

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y_true, pred):
    return float(np.mean((np.asarray(y_true, dtype=float) - clip100(pred)) ** 2))

def read_pred(path, preferred_col):
    df = pd.read_csv(path)

    if preferred_col in df.columns:
        return pd.to_numeric(df[preferred_col], errors="raise").to_numpy(dtype=float)

    numeric_cols = []
    for c in df.columns:
        if c in [ID_COL, TARGET_COL, "fold", "Unnamed: 0"]:
            continue
        vals = pd.to_numeric(df[c], errors="coerce")
        if vals.notna().mean() > 0.95:
            numeric_cols.append(c)

    if not numeric_cols:
        raise ValueError(f"No prediction column found in {path}")

    print(f"Using fallback prediction column {numeric_cols[-1]} from {path}")
    return pd.to_numeric(df[numeric_cols[-1]], errors="raise").to_numpy(dtype=float)

def get_test_ids():
    if "test_ids" in globals():
        obj = globals()["test_ids"]
        if isinstance(obj, pd.DataFrame):
            if ID_COL in obj.columns:
                return obj[ID_COL].astype(str).reset_index(drop=True)
            return obj.iloc[:, 0].astype(str).reset_index(drop=True)
        return pd.Series(obj).astype(str).reset_index(drop=True)

    if "scores_test" in globals() and ID_COL in scores_test.columns:
        return scores_test[ID_COL].astype(str).reset_index(drop=True)

    raise ValueError("Could not find test IDs.")

def to_float_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32).tocsr()
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def add_meta_features(X, M):
    M = np.asarray(M, dtype=np.float32)
    if sparse.issparse(X):
        return sparse.hstack([X, sparse.csr_matrix(M)], format="csr")
    return np.hstack([X, M])

def mean_test_residual_parts(artifact, n_folds=5):
    parts = []
    for fold in range(1, n_folds + 1):
        p = RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy"
        if not p.exists():
            raise FileNotFoundError(f"Missing residual test prediction file: {p}")
        parts.append(np.load(p))
    return np.mean(parts, axis=0)

# ------------------------------------------------------------
# Log public checkpoint in tracker
# ------------------------------------------------------------

tracker_path = RESULTS_DIR / "submission_tracker.csv"
checkpoint_row = {
    "file": PUBLIC_ANCHOR_FILE,
    "public_mse": PUBLIC_ANCHOR_SCORE,
    "private_mse": np.nan,
    "notes": (
        "New public anchor. Expanded residual stack on prior 80.662 ET+longLGBM blend; "
        "lambda=1.135; public MSE improved to 77.277."
    ),
}

if tracker_path.exists():
    tracker_df = pd.read_csv(tracker_path)
else:
    tracker_df = pd.DataFrame()

if len(tracker_df) == 0 or PUBLIC_ANCHOR_FILE not in set(tracker_df.get("file", pd.Series(dtype=str)).astype(str)):
    tracker_df = pd.concat([tracker_df, pd.DataFrame([checkpoint_row])], ignore_index=True)
    tracker_df.to_csv(tracker_path, index=False)
    print("Added public checkpoint to tracker:", tracker_path)
else:
    print("Public checkpoint already present in tracker:", tracker_path)

# ------------------------------------------------------------
# Load base matrices and model prediction artifacts
# ------------------------------------------------------------

X_base = to_float_matrix(X_train_proc_model)
X_test_base = to_float_matrix(X_test_proc_model)
y = np.asarray(y_train, dtype=float).ravel()
test_id_series = get_test_ids()

components = {
    "rf500": {
        "oof": ("model_results/oof_rf_500_base.csv", "rf_500_base_oof_pred"),
        "test": ("model_results/testpred_rf_500_base_folds.csv", "rf_500_base_foldavg_pred"),
    },
    "et_safe": {
        "oof": ("model_results/oof_extratrees_safe_base.csv", "oof_pred"),
        "test": ("model_results/testpred_extratrees_safe_base_foldavg.csv", TARGET_COL),
    },
    "lgbm12k": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof.csv", "OOF_lgbm_t03_base_5fold_oof"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof"),
    },
    "lgbm_long80k_lr03": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv", "OOF_lgbm_t03_base_5fold_oof_long80k_lr03"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03"),
    },
    "lgbm_long100k_lr02": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv", "OOF_lgbm_t03_base_5fold_oof_long100k_lr02"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02"),
    },
    "lgbm_div02_3fold": {
        "oof": ("model_results/oof_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof.csv", "OOF_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof"),
        "test": ("model_results/testpred_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof_foldavg.csv", "TESTPRED_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof"),
    },
}

oof_preds = {}
test_preds = {}

for name, spec in components.items():
    try:
        oof_p = clip100(read_pred(*spec["oof"]))
        test_p = clip100(read_pred(*spec["test"]))

        if len(oof_p) != len(y) or len(test_p) != len(test_id_series):
            print(f"Skipping {name}: length mismatch")
            continue

        oof_preds[name] = oof_p
        test_preds[name] = test_p
        print(f"Loaded {name}: OOF MSE = {mse_clip(y, oof_p):.6f}")

    except Exception as e:
        print(f"Skipping {name}: {e}")

if "et_safe" not in oof_preds or "lgbm_long100k_lr02" not in oof_preds:
    raise ValueError("Need et_safe and lgbm_long100k_lr02 to rebuild the old 80.662 base blend.")

# Old submitted 80.662 base blend.
base_oof = 0.319 * oof_preds["et_safe"] + 0.681 * oof_preds["lgbm_long100k_lr02"]
base_test = 0.319 * test_preds["et_safe"] + 0.681 * test_preds["lgbm_long100k_lr02"]
base_mse = mse_clip(y, base_oof)

# Existing successful residual correction from 24A/24B.
old_resid_oof = np.load(RESULTS_DIR / f"{old_resid_artifact}_resid_oof.npy")
old_resid_test = mean_test_residual_parts(old_resid_artifact, n_folds=5)

old_lambda = 1.135
current_anchor_oof = clip100(base_oof + old_lambda * old_resid_oof)
current_anchor_test = clip100(base_test + old_lambda * old_resid_test)
current_anchor_mse = mse_clip(y, current_anchor_oof)

print("\nOld 80.662 base OOF MSE:", base_mse)
print("Current residual-stack anchor lambda:", old_lambda)
print("Current residual-stack anchor OOF MSE:", current_anchor_mse)
print("Current public anchor MSE:", PUBLIC_ANCHOR_SCORE)

# ------------------------------------------------------------
# Build feature matrix for a diverse first-level residual model
# ------------------------------------------------------------

component_order = list(oof_preds.keys())

M_train_main = np.column_stack([oof_preds[name] for name in component_order])
M_test_main = np.column_stack([test_preds[name] for name in component_order])

lgbm_names = [n for n in component_order if n.startswith("lgbm")]
lgbm_train = np.column_stack([oof_preds[n] for n in lgbm_names])
lgbm_test = np.column_stack([test_preds[n] for n in lgbm_names])

extra_train_cols = [
    base_oof,
    M_train_main.mean(axis=1),
    M_train_main.std(axis=1),
    M_train_main.min(axis=1),
    M_train_main.max(axis=1),
    M_train_main.max(axis=1) - M_train_main.min(axis=1),
    lgbm_train.mean(axis=1),
    lgbm_train.std(axis=1),
]

extra_test_cols = [
    base_test,
    M_test_main.mean(axis=1),
    M_test_main.std(axis=1),
    M_test_main.min(axis=1),
    M_test_main.max(axis=1),
    M_test_main.max(axis=1) - M_test_main.min(axis=1),
    lgbm_test.mean(axis=1),
    lgbm_test.std(axis=1),
]

# Explicit disagreement features. The model could learn these from the raw
# component predictions, but adding them helps a constrained residual model.
if "et_safe" in oof_preds and "lgbm_long100k_lr02" in oof_preds:
    d_tr = oof_preds["et_safe"] - oof_preds["lgbm_long100k_lr02"]
    d_te = test_preds["et_safe"] - test_preds["lgbm_long100k_lr02"]
    extra_train_cols.extend([d_tr, np.abs(d_tr)])
    extra_test_cols.extend([d_te, np.abs(d_te)])

if "lgbm_long80k_lr03" in oof_preds and "lgbm_long100k_lr02" in oof_preds:
    d_tr = oof_preds["lgbm_long80k_lr03"] - oof_preds["lgbm_long100k_lr02"]
    d_te = test_preds["lgbm_long80k_lr03"] - test_preds["lgbm_long100k_lr02"]
    extra_train_cols.extend([d_tr, np.abs(d_tr)])
    extra_test_cols.extend([d_te, np.abs(d_te)])

if "lgbm_div02_3fold" in oof_preds and "lgbm_long100k_lr02" in oof_preds:
    d_tr = oof_preds["lgbm_div02_3fold"] - oof_preds["lgbm_long100k_lr02"]
    d_te = test_preds["lgbm_div02_3fold"] - test_preds["lgbm_long100k_lr02"]
    extra_train_cols.extend([d_tr, np.abs(d_tr)])
    extra_test_cols.extend([d_te, np.abs(d_te)])

M_train_extra = np.column_stack(extra_train_cols).astype(np.float32)
M_test_extra = np.column_stack(extra_test_cols).astype(np.float32)

M_train = np.hstack([M_train_main.astype(np.float32), M_train_extra])
M_test = np.hstack([M_test_main.astype(np.float32), M_test_extra])

Z_train = add_meta_features(X_base, M_train)
Z_test = add_meta_features(X_test_base, M_test)

# Important: target the original base residual, not the 77.277 anchor residual.
# This keeps this as another first-level residual correction.
residual_y = y - base_oof

print("\nResidual target summary:")
print(pd.Series(residual_y).describe())

print("\nNew residual model train matrix:", Z_train.shape)
print("New residual model test matrix:", Z_test.shape)

# ------------------------------------------------------------
# Diverse residual model: Huber + Extra-Trees-style LightGBM
# ------------------------------------------------------------

params = {
    "objective": "huber",
    "metric": "l2",
    "alpha": 0.85,
    "random_state": RANDOM_STATE + 2026,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,
    "learning_rate": 0.025,
    "n_estimators": 30000,
    "num_leaves": 63,
    "max_depth": 8,
    "min_child_samples": 180,
    "subsample": 0.75,
    "subsample_freq": 1,
    "colsample_bytree": 0.75,
    "reg_alpha": 2.0,
    "reg_lambda": 80.0,
    "extra_trees": True,
}

resid_oof_path = RESULTS_DIR / f"{new_artifact}_resid_oof.npy"
metrics_path = RESULTS_DIR / f"{new_artifact}_fold_metrics.csv"

if resid_oof_path.exists():
    new_resid_oof = np.load(resid_oof_path)
else:
    new_resid_oof = np.full(len(y), np.nan, dtype=np.float32)

if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
else:
    metrics_df = pd.DataFrame()

safe_feature_names = [f"f{i}" for i in range(Z_train.shape[1])]

# Different seed/split for model diversity.
kf_model = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 2026)

overall_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf_model.split(Z_train), start=1):
    fold_test_path = RESULTS_DIR / f"{new_artifact}_fold{fold}_test_resid.npy"

    already_done = (
        (not metrics_df.empty)
        and (fold in set(metrics_df["fold"]))
        and fold_test_path.exists()
        and (not np.isnan(new_resid_oof[va_idx]).any())
    )

    if already_done:
        print(f"\nSkipping completed fold {fold}/5")
        continue

    print("\n" + "=" * 80)
    print(f"Starting diverse residual model fold {fold}/5")
    print("=" * 80)

    start = time.time()

    model = lgb.LGBMRegressor(**params)
    model.fit(
        Z_train[tr_idx],
        residual_y[tr_idx],
        eval_set=[(Z_train[va_idx], residual_y[va_idx])],
        eval_metric="l2",
        feature_name=safe_feature_names,
        callbacks=[
            lgb.early_stopping(stopping_rounds=1500, verbose=True),
            lgb.log_evaluation(period=1500),
        ],
    )

    best_iter = model.best_iteration_ or params["n_estimators"]

    va_resid_pred = model.predict(Z_train[va_idx], num_iteration=best_iter)
    test_resid_pred = model.predict(Z_test, num_iteration=best_iter)

    new_resid_oof[va_idx] = va_resid_pred.astype(np.float32)
    np.save(resid_oof_path, new_resid_oof)
    np.save(fold_test_path, test_resid_pred.astype(np.float32))

    row = {
        "fold": fold,
        "best_iteration": int(best_iter),
        "residual_valid_mse": float(np.mean((residual_y[va_idx] - va_resid_pred) ** 2)),
        "corrected_mse_lambda_1": mse_clip(y[va_idx], base_oof[va_idx] + va_resid_pred),
        "base_mse_on_fold": mse_clip(y[va_idx], base_oof[va_idx]),
        "elapsed_sec": float(time.time() - start),
    }

    metrics_df = metrics_df[metrics_df["fold"] != fold] if not metrics_df.empty else metrics_df
    metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
    metrics_df = metrics_df.sort_values("fold").reset_index(drop=True)
    metrics_df.to_csv(metrics_path, index=False)

    print("\nCompleted fold:", fold)
    print("Fold corrected MSE at lambda=1:", row["corrected_mse_lambda_1"])
    print("Fold base MSE:", row["base_mse_on_fold"])
    print("Best iteration:", row["best_iteration"])

    del model, va_resid_pred, test_resid_pred
    gc.collect()

completed = (not np.isnan(new_resid_oof).any()) and all(
    (RESULTS_DIR / f"{new_artifact}_fold{fold}_test_resid.npy").exists()
    for fold in range(1, 6)
)

print("\n" + "#" * 90)
print("Diverse residual artifact completion check")
print("#" * 90)
print("Missing OOF residual predictions:", int(np.isnan(new_resid_oof).sum()))
print("Completed:", completed)

if not completed:
    print("\nNot all folds are complete yet. Re-execute this same cell to resume from checkpoints.")

else:
    new_resid_test = mean_test_residual_parts(new_artifact, n_folds=5)

    standalone_new_mse = mse_clip(y, base_oof + new_resid_oof)
    corr_old_new = float(np.corrcoef(old_resid_oof, new_resid_oof)[0, 1])
    corr_new_true = float(np.corrcoef(residual_y, new_resid_oof)[0, 1])

    print("\nStandalone diverse residual correction OOF MSE at lambda=1:", standalone_new_mse)
    print("Correlation old residual correction vs new residual correction:", corr_old_new)
    print("Correlation new residual correction vs true base residual:", corr_new_true)

    # --------------------------------------------------------
    # Blend old successful residual with new diverse residual
    # --------------------------------------------------------

    a_grid = sorted(set(np.round(np.arange(0.85, 1.306, 0.025), 3).tolist() + [old_lambda]))
    b_grid = sorted(set(np.round(np.arange(0.00, 0.801, 0.025), 3).tolist()))

    # Fold diagnostics against the current 1.135 residual-stack anchor.
    # These folds are diagnostic only; the OOF predictions themselves are already cross-fit.
    kf_diag = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    diag_folds = list(kf_diag.split(y))

    rows = []

    for a in a_grid:
        for b in b_grid:
            pred_oof = clip100(base_oof + a * old_resid_oof + b * new_resid_oof)
            overall_mse = mse_clip(y, pred_oof)

            fold_gains = []
            fold_mses = []

            for fold, (_, va_idx) in enumerate(diag_folds, start=1):
                anchor_fold_mse = mse_clip(y[va_idx], current_anchor_oof[va_idx])
                new_fold_mse = mse_clip(y[va_idx], pred_oof[va_idx])
                fold_gains.append(anchor_fold_mse - new_fold_mse)
                fold_mses.append(new_fold_mse)

            pred_test = clip100(base_test + a * old_resid_test + b * new_resid_test)

            rows.append({
                "old_resid_weight": float(a),
                "new_resid_weight": float(b),
                "oof_mse_clipped": overall_mse,
                "gain_vs_current_anchor_oof": current_anchor_mse - overall_mse,
                "min_fold_gain_vs_current": float(np.min(fold_gains)),
                "mean_fold_gain_vs_current": float(np.mean(fold_gains)),
                "max_fold_gain_vs_current": float(np.max(fold_gains)),
                "fold_mse_std": float(np.std(fold_mses, ddof=1)),
                "test_mean": float(np.mean(pred_test)),
                "test_std": float(np.std(pred_test)),
                "test_min": float(np.min(pred_test)),
                "test_max": float(np.max(pred_test)),
            })

    screen_df = pd.DataFrame(rows).sort_values("oof_mse_clipped").reset_index(drop=True)

    screen_path = RESULTS_DIR / f"{new_artifact}_blend_with_old_resid_screen.csv"
    screen_df.to_csv(screen_path, index=False)

    print("\nTop residual-diversity blend candidates:")
    try:
        display(screen_df.head(25))
    except NameError:
        print(screen_df.head(25).to_string(index=False))

    print("\nCurrent anchor OOF MSE:", current_anchor_mse)
    print("Best new blend OOF MSE:", float(screen_df.loc[0, "oof_mse_clipped"]))
    print("Best new blend OOF gain vs current anchor:", float(screen_df.loc[0, "gain_vs_current_anchor_oof"]))
    print("Best new blend min fold gain vs current:", float(screen_df.loc[0, "min_fold_gain_vs_current"]))
    print("Screen saved to:", screen_path)

    # --------------------------------------------------------
    # Save only candidates that beat the current OOF anchor
    # --------------------------------------------------------

    improved = screen_df[
        (screen_df["gain_vs_current_anchor_oof"] > 0.0)
    ].copy()

    saved_rows = []

    if improved.empty:
        print("\nNo OOF-improving residual-diversity blend was found. Do not submit from this cell.")
    else:
        # Prefer candidates with positive fold-gain evidence, but keep the best OOF candidate too.
        positive_fold = improved[improved["min_fold_gain_vs_current"] > 0].copy()

        candidate_pool = positive_fold if len(positive_fold) > 0 else improved
        candidate_pool = candidate_pool.sort_values("oof_mse_clipped").head(5).reset_index(drop=True)

        for _, row in candidate_pool.iterrows():
            a = float(row["old_resid_weight"])
            b = float(row["new_resid_weight"])

            final_oof = clip100(base_oof + a * old_resid_oof + b * new_resid_oof)
            final_test = clip100(base_test + a * old_resid_test + b * new_resid_test)

            name = (
                f"{new_artifact}_oldw_{str(round(a, 3)).replace('.', 'p')}"
                f"_neww_{str(round(b, 3)).replace('.', 'p')}"
            )

            oof_path = RESULTS_DIR / f"oof_{name}.csv"
            testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
            sub_path = Path(f"submission_{name}.csv")

            pd.DataFrame({
                TARGET_COL: y,
                f"OOF_{name}": final_oof
            }).to_csv(oof_path, index=False)

            pd.DataFrame({
                ID_COL: test_id_series.values,
                f"TESTPRED_{name}": final_test
            }).to_csv(testpred_path, index=False)

            pd.DataFrame({
                ID_COL: test_id_series.values,
                TARGET_COL: final_test
            }).to_csv(sub_path, index=False)

            saved_rows.append({
                "candidate": name,
                "old_resid_weight": a,
                "new_resid_weight": b,
                "oof_mse_clipped": mse_clip(y, final_oof),
                "gain_vs_current_anchor_oof": current_anchor_mse - mse_clip(y, final_oof),
                "submission_path": str(sub_path),
                "testpred_path": str(testpred_path),
                "test_mean": float(np.mean(final_test)),
                "test_std": float(np.std(final_test)),
                "test_min": float(np.min(final_test)),
                "test_max": float(np.max(final_test)),
            })

        saved_df = pd.DataFrame(saved_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
        saved_path = RESULTS_DIR / f"{new_artifact}_saved_candidates.csv"
        saved_df.to_csv(saved_path, index=False)

        print("\nSaved OOF-improving residual-diversity candidates:")
        try:
            display(saved_df)
        except NameError:
            print(saved_df.to_string(index=False))

        print("Saved candidates list:", saved_path)

        best_gain = float(saved_df.loc[0, "gain_vs_current_anchor_oof"])
        best_sub = saved_df.loc[0, "submission_path"]

        if best_gain >= 0.15:
            print("\nSubmission candidate worth considering:", best_sub)
            print("Reason: OOF gain versus current anchor is at least 0.15.")
        else:
            print("\nGain is small. Hold this candidate unless there are no better modeling directions.")

    print("\nTotal elapsed minutes:", round((time.time() - overall_start) / 60, 2))

Public checkpoint already present in tracker: model_results/submission_tracker.csv
Loaded rf500: OOF MSE = 122.328024
Loaded et_safe: OOF MSE = 110.749294
Loaded lgbm12k: OOF MSE = 98.779406
Loaded lgbm_long80k_lr03: OOF MSE = 94.163485
Loaded lgbm_long100k_lr02: OOF MSE = 93.726214
Loaded lgbm_div02_3fold: OOF MSE = 104.083036

Old 80.662 base OOF MSE: 88.96636822734054
Current residual-stack anchor lambda: 1.135
Current residual-stack anchor OOF MSE: 87.13604874858268
Current public anchor MSE: 77.277

Residual target summary:
count    144921.000000
mean          0.038600
std           9.432152
min         -74.711172
25%          -4.382987
50%           0.127390
75%           4.493513
max          84.650332
dtype: float64

New residual model train matrix: (144921, 182)
New residual model test matrix: (48307, 182)

Skipping completed fold 1/5

Skipping completed fold 2/5

Skipping completed fold 3/5

Skipping completed fold 4/5

Skipping completed fold 5/5

###########################

,old_resid_weight,new_resid_weight,oof_mse_clipped,gain_vs_current_anchor_oof,min_fold_gain_vs_current,mean_fold_gain_vs_current,max_fold_gain_vs_current,fold_mse_std,test_mean,test_std,test_min,test_max
0,0.850,0.525,86.101600,1.034449,0.832125,1.034449,1.260125,2.024438,54.135944,24.990570,0.000000,100.0
1,0.850,0.500,86.103925,1.032124,0.838290,1.032124,1.246989,2.019299,54.133858,24.969032,0.000000,100.0
2,0.850,0.550,86.104379,1.031670,0.820974,1.031670,1.268000,2.029641,54.137978,25.012076,0.000000,100.0
3,0.850,0.475,86.111472,1.024576,0.839445,1.024576,1.228558,2.014096,54.131729,24.947484,0.000000,100.0
4,0.850,0.575,86.112234,1.023815,0.804855,1.023815,1.270733,2.034937,54.139975,25.033576,0.000000,100.0
5,0.850,0.450,86.124204,1.011845,0.835587,1.011844,1.204820,2.008816,54.129571,24.925946,0.000000,100.0
6,0.850,0.600,86.125240,1.010808,0.783856,1.010808,1.268266,2.040195,54.141928,25.055058,0.000000,100.0
7,0.875,0.525,86.129413,1.006636,0.804662,1.006635,1.232918,2.023738,54.136588,25.002140,0.000000,100.0
8,0.875,0.500,86.129549,1.006500,0.812929,1.006500,1.222086,2.018712,54.134528,24.980623,0.000000,100.0
9,0.875,0.550,86.134341,1.001708,0.791425,1.001707,1.238502,2.028886,54.138599,25.023630,0.000000,100.0



Current anchor OOF MSE: 87.13604874858268
Best new blend OOF MSE: 86.10159951777065
Best new blend OOF gain vs current anchor: 1.0344492308120294
Best new blend min fold gain vs current: 0.8321252420166019
Screen saved to: model_results/residstack_huber_extra_on_best80p662_blend_5fold_blend_with_old_resid_screen.csv

Saved OOF-improving residual-diversity candidates:


,candidate,old_resid_weight,new_resid_weight,oof_mse_clipped,gain_vs_current_anchor_oof,submission_path,testpred_path,test_mean,test_std,test_min,test_max
0,residstack_huber_extra_on_best80p662_blend_5fo...,0.85,0.525,86.101600,1.034449,submission_residstack_huber_extra_on_best80p66...,model_results/testpred_residstack_huber_extra_...,54.135944,24.990570,0.0,100.0
1,residstack_huber_extra_on_best80p662_blend_5fo...,0.85,0.500,86.103925,1.032124,submission_residstack_huber_extra_on_best80p66...,model_results/testpred_residstack_huber_extra_...,54.133858,24.969032,0.0,100.0
2,residstack_huber_extra_on_best80p662_blend_5fo...,0.85,0.550,86.104379,1.031670,submission_residstack_huber_extra_on_best80p66...,model_results/testpred_residstack_huber_extra_...,54.137978,25.012076,0.0,100.0
3,residstack_huber_extra_on_best80p662_blend_5fo...,0.85,0.475,86.111472,1.024576,submission_residstack_huber_extra_on_best80p66...,model_results/testpred_residstack_huber_extra_...,54.131729,24.947484,0.0,100.0
4,residstack_huber_extra_on_best80p662_blend_5fo...,0.85,0.575,86.112234,1.023815,submission_residstack_huber_extra_on_best80p66...,model_results/testpred_residstack_huber_extra_...,54.139975,25.033576,0.0,100.0


Saved candidates list: model_results/residstack_huber_extra_on_best80p662_blend_5fold_saved_candidates.csv

Submission candidate worth considering: submission_residstack_huber_extra_on_best80p662_blend_5fold_oldw_0p85_neww_0p525.csv
Reason: OOF gain versus current anchor is at least 0.15.

Total elapsed minutes: 0.03


### 25A Huber/extra-trees-style residual model checkpoint

The diverse residual model completed successfully and all five fold artifacts were available from checkpointed predictions.

This model was trained as a new first-level residual correction on the original best 80.662 base blend residuals, rather than as a second-stage correction on top of the 77.277 model. The standalone new residual correction achieved OOF MSE `85.6994`, compared with `87.1360` for the current 77.277 public anchor, giving an OOF gain of approximately `1.4367`.

The new residual correction is correlated with the previous residual correction at about `0.7401`, so it is related but not identical. Its correlation with the original base residual target is about `0.1931`, which is enough to produce a large OOF improvement.

The initial old+new residual blend screen found a best constrained blend at old residual weight `0.850` and new residual weight `0.525`, with OOF MSE `86.1016` and positive fold gains versus the current anchor. However, this constrained screen did not allow the old residual weight to approach zero, while the standalone new residual model already has better OOF MSE than the saved blend candidates.

Therefore, the next step is an expanded coefficient screen over both residual corrections, including the standalone new residual model and low/zero old-residual weights. No additional model training is needed for this step.

In [49]:
# ============================================================
# 25B. Expanded coefficient screen for old vs new residuals
# ============================================================
#
# 25A found:
# - current anchor OOF MSE: 87.1360
# - standalone new residual OOF MSE: 85.6994
# - constrained old+new blend best OOF MSE: 86.1016
#
# The constrained blend screen did not allow old_resid_weight near zero.
# This cell does no training. It only screens:
#
#     base + old_weight * old_residual + new_weight * new_residual
#
# over a wider grid, then saves the best OOF-backed candidates.

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y_true, pred):
    return float(np.mean((np.asarray(y_true, dtype=float) - clip100(pred)) ** 2))

def safe_token(x):
    return str(round(float(x), 4)).replace("-", "m").replace(".", "p")

def mean_test_residual_parts(artifact, n_folds=5):
    return np.mean(
        [np.load(RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy") for fold in range(1, n_folds + 1)],
        axis=0,
    )

# These should already exist because 25A just ran.
# The fallbacks only reload residual artifacts if needed.
old_artifact = "residstack_lgbm_on_best80p662_blend_5fold"
new_artifact = "residstack_huber_extra_on_best80p662_blend_5fold"

if "old_resid_oof" not in globals():
    old_resid_oof = np.load(RESULTS_DIR / f"{old_artifact}_resid_oof.npy")

if "old_resid_test" not in globals():
    old_resid_test = mean_test_residual_parts(old_artifact, n_folds=5)

if "new_resid_oof" not in globals():
    new_resid_oof = np.load(RESULTS_DIR / f"{new_artifact}_resid_oof.npy")

if "new_resid_test" not in globals():
    new_resid_test = mean_test_residual_parts(new_artifact, n_folds=5)

if "current_anchor_oof" not in globals():
    current_anchor_oof = clip100(base_oof + 1.135 * old_resid_oof)

current_anchor_mse = mse_clip(y, current_anchor_oof)

standalone_new_oof = clip100(base_oof + 1.0 * new_resid_oof)
standalone_new_mse = mse_clip(y, standalone_new_oof)

print("Current anchor OOF MSE:", current_anchor_mse)
print("Standalone new residual OOF MSE:", standalone_new_mse)
print("Standalone new gain vs current anchor:", current_anchor_mse - standalone_new_mse)

# Wider coefficient screen.
# old weight is allowed to fall to zero or slightly negative because the new residual
# may already contain much of the old residual signal.
old_grid = np.round(np.arange(-0.300, 1.251, 0.025), 3)
new_grid = np.round(np.arange(0.000, 1.601, 0.025), 3)

# Add exact known points.
old_grid = np.array(sorted(set(old_grid.tolist() + [0.0, 0.85, 1.135])))
new_grid = np.array(sorted(set(new_grid.tolist() + [0.525, 1.0])))

diag_seed = globals().get("RANDOM_STATE", 9890)
diag_folds = list(KFold(n_splits=5, shuffle=True, random_state=diag_seed).split(y))

rows = []

for ow in old_grid:
    old_part_oof = base_oof + ow * old_resid_oof
    old_part_test = base_test + ow * old_resid_test

    for nw in new_grid:
        pred_oof = clip100(old_part_oof + nw * new_resid_oof)
        oof_mse = mse_clip(y, pred_oof)

        fold_gains = []
        fold_mses = []

        for fold, (_, va_idx) in enumerate(diag_folds, start=1):
            anchor_fold_mse = mse_clip(y[va_idx], current_anchor_oof[va_idx])
            cand_fold_mse = mse_clip(y[va_idx], pred_oof[va_idx])
            fold_gains.append(anchor_fold_mse - cand_fold_mse)
            fold_mses.append(cand_fold_mse)

        pred_test = clip100(old_part_test + nw * new_resid_test)

        rows.append({
            "old_resid_weight": float(ow),
            "new_resid_weight": float(nw),
            "oof_mse_clipped": float(oof_mse),
            "gain_vs_current_anchor_oof": float(current_anchor_mse - oof_mse),
            "gain_vs_standalone_new_oof": float(standalone_new_mse - oof_mse),
            "min_fold_gain_vs_current": float(np.min(fold_gains)),
            "mean_fold_gain_vs_current": float(np.mean(fold_gains)),
            "max_fold_gain_vs_current": float(np.max(fold_gains)),
            "fold_mse_std": float(np.std(fold_mses, ddof=1)),
            "test_mean": float(np.mean(pred_test)),
            "test_std": float(np.std(pred_test)),
            "test_min": float(np.min(pred_test)),
            "test_max": float(np.max(pred_test)),
        })

screen_df = pd.DataFrame(rows).sort_values("oof_mse_clipped").reset_index(drop=True)

screen_path = RESULTS_DIR / "residstack_oldnew_expanded_weight_screen.csv"
screen_df.to_csv(screen_path, index=False)

print("\nTop expanded old/new residual coefficient candidates:")
try:
    display(screen_df.head(30))
except NameError:
    print(screen_df.head(30).to_string(index=False))

print("\nScreen saved to:", screen_path)
print("Best expanded OOF MSE:", float(screen_df.loc[0, "oof_mse_clipped"]))
print("Best expanded gain vs current anchor:", float(screen_df.loc[0, "gain_vs_current_anchor_oof"]))
print("Best expanded gain vs standalone new:", float(screen_df.loc[0, "gain_vs_standalone_new_oof"]))
print("Best expanded min fold gain vs current:", float(screen_df.loc[0, "min_fold_gain_vs_current"]))

# Save top candidates that beat the current anchor and have positive fold-gain diagnostics.
eligible = screen_df[
    (screen_df["gain_vs_current_anchor_oof"] > 0)
    & (screen_df["min_fold_gain_vs_current"] > 0)
].copy()

if eligible.empty:
    print("\nNo eligible expanded candidate found. This would be unexpected given standalone new OOF.")
else:
    saved_rows = []

    # Save top 8, but avoid saving many nearly identical rows if desired later.
    for _, row in eligible.head(8).iterrows():
        ow = float(row["old_resid_weight"])
        nw = float(row["new_resid_weight"])

        final_oof = clip100(base_oof + ow * old_resid_oof + nw * new_resid_oof)
        final_test = clip100(base_test + ow * old_resid_test + nw * new_resid_test)

        name = f"residstack_oldnew_expanded_oldw_{safe_token(ow)}_neww_{safe_token(nw)}"

        oof_path = RESULTS_DIR / f"oof_{name}.csv"
        testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
        sub_path = Path(f"submission_{name}.csv")

        pd.DataFrame({
            TARGET_COL: y,
            f"OOF_{name}": final_oof,
        }).to_csv(oof_path, index=False)

        pd.DataFrame({
            ID_COL: test_id_series.values,
            f"TESTPRED_{name}": final_test,
        }).to_csv(testpred_path, index=False)

        pd.DataFrame({
            ID_COL: test_id_series.values,
            TARGET_COL: final_test,
        }).to_csv(sub_path, index=False)

        saved_rows.append({
            "candidate": name,
            "old_resid_weight": ow,
            "new_resid_weight": nw,
            "oof_mse_clipped": mse_clip(y, final_oof),
            "gain_vs_current_anchor_oof": current_anchor_mse - mse_clip(y, final_oof),
            "gain_vs_standalone_new_oof": standalone_new_mse - mse_clip(y, final_oof),
            "submission_path": str(sub_path),
            "testpred_path": str(testpred_path),
            "test_mean": float(np.mean(final_test)),
            "test_std": float(np.std(final_test)),
            "test_min": float(np.min(final_test)),
            "test_max": float(np.max(final_test)),
        })

    saved_df = pd.DataFrame(saved_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
    saved_path = RESULTS_DIR / "residstack_oldnew_expanded_saved_candidates.csv"
    saved_df.to_csv(saved_path, index=False)

    print("\nSaved expanded old/new residual candidates:")
    try:
        display(saved_df)
    except NameError:
        print(saved_df.to_string(index=False))

    print("\nSaved candidates list:", saved_path)
    print("Best submission candidate:", saved_df.loc[0, "submission_path"])

Current anchor OOF MSE: 87.13604874858268
Standalone new residual OOF MSE: 85.69939570622593
Standalone new gain vs current anchor: 1.4366530423567525

Top expanded old/new residual coefficient candidates:


,old_resid_weight,new_resid_weight,oof_mse_clipped,gain_vs_current_anchor_oof,gain_vs_standalone_new_oof,min_fold_gain_vs_current,mean_fold_gain_vs_current,max_fold_gain_vs_current,fold_mse_std,test_mean,test_std,test_min,test_max
0,0.025,0.875,85.647385,1.488664,0.052011,1.181558,1.488663,1.776519,2.075271,54.142717,24.913286,0.0,100.0
1,-0.025,0.900,85.647596,1.488453,0.051800,1.175174,1.488451,1.777796,2.076937,54.143403,24.912080,0.0,100.0
2,0.000,0.875,85.647722,1.488326,0.051673,1.182816,1.488325,1.770613,2.071575,54.141995,24.901855,0.0,100.0
3,0.000,0.900,85.647759,1.488290,0.051637,1.173405,1.488289,1.783309,2.080667,54.144118,24.923507,0.0,100.0
4,0.050,0.875,85.648805,1.487244,0.050591,1.178659,1.487243,1.780495,2.078763,54.143432,24.924726,0.0,100.0
5,-0.050,0.900,85.649191,1.486858,0.050205,1.175296,1.486856,1.770344,2.073001,54.142681,24.900660,0.0,100.0
6,0.050,0.850,85.649498,1.486551,0.049898,1.187350,1.486550,1.767150,2.069890,54.141305,24.903074,0.0,100.0
7,0.025,0.900,85.649672,1.486377,0.049724,1.169992,1.486376,1.786900,2.084213,54.144829,24.934944,0.0,100.0
8,-0.025,0.875,85.649823,1.486225,0.049572,1.182432,1.486224,1.762759,2.067673,54.141270,24.890437,0.0,100.0
9,-0.050,0.925,85.649944,1.486105,0.049451,1.165134,1.486103,1.782429,2.082340,54.144801,24.922316,0.0,100.0



Screen saved to: model_results/residstack_oldnew_expanded_weight_screen.csv
Best expanded OOF MSE: 85.64738488864343
Best expanded gain vs current anchor: 1.4886638599392512
Best expanded gain vs standalone new: 0.0520108175824987
Best expanded min fold gain vs current: 1.181558077853552

Saved expanded old/new residual candidates:


,candidate,old_resid_weight,new_resid_weight,oof_mse_clipped,gain_vs_current_anchor_oof,gain_vs_standalone_new_oof,submission_path,testpred_path,test_mean,test_std,test_min,test_max
0,residstack_oldnew_expanded_oldw_0p025_neww_0p875,0.025,0.875,85.647385,1.488664,0.052011,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.142717,24.913286,0.0,100.0
1,residstack_oldnew_expanded_oldw_m0p025_neww_0p9,-0.025,0.900,85.647596,1.488453,0.051800,submission_residstack_oldnew_expanded_oldw_m0p...,model_results/testpred_residstack_oldnew_expan...,54.143403,24.912080,0.0,100.0
2,residstack_oldnew_expanded_oldw_0p0_neww_0p875,0.000,0.875,85.647722,1.488326,0.051673,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.141995,24.901855,0.0,100.0
3,residstack_oldnew_expanded_oldw_0p0_neww_0p9,0.000,0.900,85.647759,1.488290,0.051637,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.144118,24.923507,0.0,100.0
4,residstack_oldnew_expanded_oldw_0p05_neww_0p875,0.050,0.875,85.648805,1.487244,0.050591,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.143432,24.924726,0.0,100.0
5,residstack_oldnew_expanded_oldw_m0p05_neww_0p9,-0.050,0.900,85.649191,1.486858,0.050205,submission_residstack_oldnew_expanded_oldw_m0p...,model_results/testpred_residstack_oldnew_expan...,54.142681,24.900660,0.0,100.0
6,residstack_oldnew_expanded_oldw_0p05_neww_0p85,0.050,0.850,85.649498,1.486551,0.049898,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.141305,24.903074,0.0,100.0
7,residstack_oldnew_expanded_oldw_0p025_neww_0p9,0.025,0.900,85.649672,1.486377,0.049724,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.144829,24.934944,0.0,100.0



Saved candidates list: model_results/residstack_oldnew_expanded_saved_candidates.csv
Best submission candidate: submission_residstack_oldnew_expanded_oldw_0p025_neww_0p875.csv


### 25B expanded old/new residual coefficient screen

The expanded old/new residual screen tested combinations of the original LightGBM residual correction and the newer Huber/extra-trees-style residual correction.

The best internal candidate was:

`residstack_oldnew_expanded_oldw_0p025_neww_0p875`

with OOF MSE `85.6474`, an OOF gain of approximately `1.4887` versus the current submitted 77.277 public anchor, and a positive minimum fold gain of approximately `1.1816`.

However, the gain versus the standalone new residual correction was only about `0.0520` OOF MSE. The top region of the screen was also very flat, with old residual weights near zero and new residual weights around `0.875–0.900`. This suggests that the newer Huber/extra residual model mostly supersedes the older residual correction.

Because this coefficient screen is mainly a small refinement of an already-discovered direction, it is not submitted immediately. Instead, this candidate is kept as the new internal OOF anchor for further modeling.

After the strong residual models, are there still systematic residual biases by
school, district, assessment, subgroup, region, or combinations of these?

In [50]:
# ============================================================
# 26A. Cross-fitted grouped residual calibration
# ============================================================
#
# Purpose:
# - Do NOT submit the 25B coefficient tweak yet.
# - Use the best 25B candidate as an internal OOF anchor.
# - Test whether remaining residuals contain systematic group-level structure:
#   school, district, county, assessment, subgroup, region, and combinations.
#
# This is a leakage-safe residual target encoding / empirical Bayes correction:
# - OOF corrections are built fold-by-fold.
# - A validation row's group residual mean is computed only from other folds.
# - Test corrections use full training residual statistics.
#
# This cell is much faster than training another full model.

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y_true, pred):
    return float(np.mean((np.asarray(y_true, dtype=float) - clip100(pred)) ** 2))

def safe_token(s):
    return str(s).replace(" ", "_").replace("/", "_").replace("-", "m").replace(".", "p")

def get_test_ids():
    if "test_id_series" in globals():
        return pd.Series(test_id_series).astype(str).reset_index(drop=True)
    if "test_ids" in globals():
        return pd.Series(test_ids).astype(str).reset_index(drop=True)
    if "scores_test" in globals() and ID_COL in scores_test.columns:
        return scores_test[ID_COL].astype(str).reset_index(drop=True)
    raise ValueError("Could not find test IDs.")

def make_key(df, cols):
    tmp = df.loc[:, list(cols)].copy()
    for c in cols:
        tmp[c] = tmp[c].astype("string").fillna("__NA__")
    return tmp.astype(str).agg("||".join, axis=1).to_numpy()

def smooth_group_map(keys, values, alpha):
    d = pd.DataFrame({"key": keys, "value": values})
    stats = d.groupby("key")["value"].agg(["count", "mean"])
    global_mean = float(np.mean(values))
    stats["smooth"] = (
        stats["count"] * stats["mean"] + alpha * global_mean
    ) / (stats["count"] + alpha)
    return stats["smooth"], global_mean

def map_with_default(keys, smooth_map, default):
    return pd.Series(keys).map(smooth_map).fillna(default).to_numpy(dtype=float)

# ------------------------------------------------------------
# Internal anchor from 25B
# ------------------------------------------------------------

# Best 25B candidate:
# old_resid_weight = 0.025
# new_resid_weight = 0.875

internal_old_w = 0.025
internal_new_w = 0.875

internal_anchor_name = (
    f"oldnew_internal_oldw_{str(internal_old_w).replace('.', 'p')}"
    f"_neww_{str(internal_new_w).replace('.', 'p')}"
)

internal_anchor_oof = clip100(
    base_oof + internal_old_w * old_resid_oof + internal_new_w * new_resid_oof
)

internal_anchor_test = clip100(
    base_test + internal_old_w * old_resid_test + internal_new_w * new_resid_test
)

internal_anchor_mse = mse_clip(y, internal_anchor_oof)

print("Internal 25B anchor:", internal_anchor_name)
print("Internal 25B anchor OOF MSE:", internal_anchor_mse)

# Residual left after current best internal candidate.
resid_left = np.asarray(y, dtype=float) - internal_anchor_oof

print("\nRemaining residual summary:")
print(pd.Series(resid_left).describe())

# ------------------------------------------------------------
# Metadata table for group corrections
# ------------------------------------------------------------

train_meta = train_full.copy()
test_meta = test_full.copy()

# Add N_STUDENTS bins, because assessment size may change residual behavior.
if "N_STUDENTS" in train_meta.columns and "N_STUDENTS" in test_meta.columns:
    train_bins, bin_edges = pd.qcut(
        train_meta["N_STUDENTS"],
        q=10,
        duplicates="drop",
        retbins=True,
    )
    train_meta["N_STUDENTS_BIN"] = train_bins.astype(str)
    test_meta["N_STUDENTS_BIN"] = pd.cut(
        test_meta["N_STUDENTS"],
        bins=bin_edges,
        include_lowest=True,
    ).astype(str)

# Candidate group structures.
raw_group_specs = [
    ("ASSESSMENT_NAME",),
    ("SUBGROUP_NAME",),
    ("SCHOOL",),
    ("DISTRICT",),
    ("COUNTY",),
    ("REGION",),
    ("DISTRICT_TYPE",),

    ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("ASSESSMENT_NAME", "N_STUDENTS_BIN"),
    ("SUBGROUP_NAME", "N_STUDENTS_BIN"),

    ("SCHOOL", "ASSESSMENT_NAME"),
    ("SCHOOL", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME"),
    ("DISTRICT", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME"),
    ("COUNTY", "SUBGROUP_NAME"),
    ("REGION", "ASSESSMENT_NAME"),
    ("DISTRICT_TYPE", "ASSESSMENT_NAME"),

    ("SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
]

group_specs = []
for spec in raw_group_specs:
    if all(c in train_meta.columns and c in test_meta.columns for c in spec):
        group_specs.append(spec)

print("\nGroup specs used:")
for spec in group_specs:
    print("  ", spec)

alpha_grid = [5, 20, 100, 300]

# ------------------------------------------------------------
# Build cross-fitted group residual correction features
# ------------------------------------------------------------

n = len(y)
nt = len(test_meta)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

group_feature_names = []
group_oof_cols = []
group_test_cols = []
individual_scores = []

for spec in group_specs:
    train_keys = make_key(train_meta, spec)
    test_keys = make_key(test_meta, spec)

    for alpha in alpha_grid:
        feature_name = "grp_" + "__".join(spec) + f"__a{alpha}"
        print("\nBuilding:", feature_name)

        corr_oof = np.zeros(n, dtype=float)

        for fold, (tr_idx, va_idx) in enumerate(kf.split(train_meta), start=1):
            smap, default = smooth_group_map(
                train_keys[tr_idx],
                resid_left[tr_idx],
                alpha=alpha,
            )
            corr_oof[va_idx] = map_with_default(
                train_keys[va_idx],
                smap,
                default,
            )

        # Full-train mapping for test.
        smap_full, default_full = smooth_group_map(
            train_keys,
            resid_left,
            alpha=alpha,
        )
        corr_test = map_with_default(
            test_keys,
            smap_full,
            default_full,
        )

        # Individual correction screen: best scalar lambda on this correction.
        best_lam = None
        best_mse = np.inf
        for lam in np.round(np.arange(-0.50, 1.501, 0.025), 3):
            m = mse_clip(y, internal_anchor_oof + lam * corr_oof)
            if m < best_mse:
                best_mse = m
                best_lam = float(lam)

        individual_scores.append({
            "feature": feature_name,
            "group_spec": "|".join(spec),
            "alpha": alpha,
            "best_lambda": best_lam,
            "oof_mse": best_mse,
            "gain_vs_internal_anchor": internal_anchor_mse - best_mse,
            "corr_oof_std": float(np.std(corr_oof)),
            "corr_test_std": float(np.std(corr_test)),
        })

        group_feature_names.append(feature_name)
        group_oof_cols.append(corr_oof)
        group_test_cols.append(corr_test)

group_oof = np.column_stack(group_oof_cols)
group_test = np.column_stack(group_test_cols)

individual_df = pd.DataFrame(individual_scores).sort_values(
    "oof_mse"
).reset_index(drop=True)

individual_path = RESULTS_DIR / "group_residual_individual_screen.csv"
individual_df.to_csv(individual_path, index=False)

print("\nTop individual group residual corrections:")
try:
    display(individual_df.head(25))
except NameError:
    print(individual_df.head(25).to_string(index=False))

print("Individual screen saved to:", individual_path)

# ------------------------------------------------------------
# Cross-fitted ridge meta-combination of group corrections
# ------------------------------------------------------------

meta_oof = np.zeros(n, dtype=float)
meta_test_folds = []

ridge_alphas = np.logspace(-3, 4, 20)
meta_folds = list(KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 2601).split(group_oof))

meta_fold_rows = []

for fold, (tr_idx, va_idx) in enumerate(meta_folds, start=1):
    model = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=ridge_alphas)
    )

    model.fit(group_oof[tr_idx], resid_left[tr_idx])

    pred_va = model.predict(group_oof[va_idx])
    pred_test = model.predict(group_test)

    meta_oof[va_idx] = pred_va
    meta_test_folds.append(pred_test)

    fold_mse_before = mse_clip(y[va_idx], internal_anchor_oof[va_idx])
    fold_mse_after_raw = mse_clip(y[va_idx], internal_anchor_oof[va_idx] + pred_va)

    meta_fold_rows.append({
        "fold": fold,
        "mse_before": fold_mse_before,
        "mse_after_raw_meta": fold_mse_after_raw,
        "gain_raw_meta": fold_mse_before - fold_mse_after_raw,
    })

meta_test = np.mean(meta_test_folds, axis=0)

meta_fold_df = pd.DataFrame(meta_fold_rows)
print("\nMeta fold diagnostics before lambda shrink:")
try:
    display(meta_fold_df)
except NameError:
    print(meta_fold_df.to_string(index=False))

# ------------------------------------------------------------
# Lambda screen for the meta correction
# ------------------------------------------------------------

rows = []

for lam in np.round(np.arange(-0.50, 1.501, 0.005), 3):
    pred_oof = clip100(internal_anchor_oof + lam * meta_oof)

    fold_gains = []
    fold_mses = []

    for fold, (_, va_idx) in enumerate(meta_folds, start=1):
        before = mse_clip(y[va_idx], internal_anchor_oof[va_idx])
        after = mse_clip(y[va_idx], pred_oof[va_idx])
        fold_gains.append(before - after)
        fold_mses.append(after)

    pred_test = clip100(internal_anchor_test + lam * meta_test)

    rows.append({
        "lambda": float(lam),
        "oof_mse_clipped": mse_clip(y, pred_oof),
        "gain_vs_internal_anchor": internal_anchor_mse - mse_clip(y, pred_oof),
        "min_fold_gain": float(np.min(fold_gains)),
        "mean_fold_gain": float(np.mean(fold_gains)),
        "max_fold_gain": float(np.max(fold_gains)),
        "fold_mse_std": float(np.std(fold_mses, ddof=1)),
        "test_mean": float(np.mean(pred_test)),
        "test_std": float(np.std(pred_test)),
        "test_min": float(np.min(pred_test)),
        "test_max": float(np.max(pred_test)),
    })

screen_df = pd.DataFrame(rows).sort_values("oof_mse_clipped").reset_index(drop=True)

screen_path = RESULTS_DIR / "group_residual_meta_lambda_screen.csv"
screen_df.to_csv(screen_path, index=False)

print("\nTop grouped residual meta-correction lambda candidates:")
try:
    display(screen_df.head(30))
except NameError:
    print(screen_df.head(30).to_string(index=False))

print("\nGrouped residual screen saved to:", screen_path)
print("Internal anchor OOF MSE:", internal_anchor_mse)
print("Best grouped residual OOF MSE:", float(screen_df.loc[0, "oof_mse_clipped"]))
print("Best grouped residual gain:", float(screen_df.loc[0, "gain_vs_internal_anchor"]))
print("Best grouped residual min fold gain:", float(screen_df.loc[0, "min_fold_gain"]))

# ------------------------------------------------------------
# Save candidate only if it improves the internal anchor
# ------------------------------------------------------------

eligible = screen_df[
    (screen_df["gain_vs_internal_anchor"] > 0)
    & (screen_df["min_fold_gain"] > 0)
].copy()

if eligible.empty:
    print("\nNo positive-fold grouped residual candidate found. Do not submit from this cell.")
else:
    best = eligible.iloc[0]
    lam = float(best["lambda"])

    final_oof = clip100(internal_anchor_oof + lam * meta_oof)
    final_test = clip100(internal_anchor_test + lam * meta_test)

    name = f"group_resid_meta_on_{internal_anchor_name}_lambda_{safe_token(lam)}"

    oof_path = RESULTS_DIR / f"oof_{name}.csv"
    testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
    sub_path = Path(f"submission_{name}.csv")

    pd.DataFrame({
        TARGET_COL: y,
        f"OOF_{name}": final_oof,
    }).to_csv(oof_path, index=False)

    pd.DataFrame({
        ID_COL: get_test_ids().values,
        f"TESTPRED_{name}": final_test,
    }).to_csv(testpred_path, index=False)

    pd.DataFrame({
        ID_COL: get_test_ids().values,
        TARGET_COL: final_test,
    }).to_csv(sub_path, index=False)

    saved = pd.DataFrame([{
        "candidate": name,
        "lambda": lam,
        "oof_mse_clipped": mse_clip(y, final_oof),
        "gain_vs_internal_anchor": internal_anchor_mse - mse_clip(y, final_oof),
        "submission_path": str(sub_path),
        "testpred_path": str(testpred_path),
        "test_mean": float(np.mean(final_test)),
        "test_std": float(np.std(final_test)),
        "test_min": float(np.min(final_test)),
        "test_max": float(np.max(final_test)),
    }])

    saved_path = RESULTS_DIR / "group_residual_meta_saved_candidate.csv"
    saved.to_csv(saved_path, index=False)

    print("\nSaved grouped residual candidate:")
    try:
        display(saved)
    except NameError:
        print(saved.to_string(index=False))

    print("Saved candidate list:", saved_path)
    print("Candidate file:", sub_path)

    # Coefficient inspection from full model, for insight only.
    final_model = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=ridge_alphas)
    )
    final_model.fit(group_oof, resid_left)

    ridge = final_model.named_steps["ridgecv"]
    coef_df = pd.DataFrame({
        "feature": group_feature_names,
        "coef_scaled_space": ridge.coef_,
    }).assign(abs_coef=lambda d: d["coef_scaled_space"].abs())

    coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)
    coef_path = RESULTS_DIR / "group_residual_meta_top_coefficients.csv"
    coef_df.to_csv(coef_path, index=False)

    print("\nTop grouped residual meta features by coefficient magnitude:")
    try:
        display(coef_df.head(25))
    except NameError:
        print(coef_df.head(25).to_string(index=False))

    print("Coefficient table saved to:", coef_path)

Internal 25B anchor: oldnew_internal_oldw_0p025_neww_0p875
Internal 25B anchor OOF MSE: 85.64738488718324

Remaining residual summary:
count    144921.000000
mean         -0.057888
std           9.254438
min         -79.243600
25%          -4.158437
50%           0.027997
75%           4.053417
max          84.574910
dtype: float64

Group specs used:
   ('ASSESSMENT_NAME',)
   ('SUBGROUP_NAME',)
   ('SCHOOL',)
   ('DISTRICT',)
   ('COUNTY',)
   ('REGION',)
   ('DISTRICT_TYPE',)
   ('ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('ASSESSMENT_NAME', 'N_STUDENTS_BIN')
   ('SUBGROUP_NAME', 'N_STUDENTS_BIN')
   ('SCHOOL', 'ASSESSMENT_NAME')
   ('SCHOOL', 'SUBGROUP_NAME')
   ('DISTRICT', 'ASSESSMENT_NAME')
   ('DISTRICT', 'SUBGROUP_NAME')
   ('COUNTY', 'ASSESSMENT_NAME')
   ('COUNTY', 'SUBGROUP_NAME')
   ('REGION', 'ASSESSMENT_NAME')
   ('DISTRICT_TYPE', 'ASSESSMENT_NAME')
   ('SCHOOL', 'ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('DISTRICT', 'ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('COUNTY', 'ASSESSMENT_

,feature,group_spec,alpha,best_lambda,oof_mse,gain_vs_internal_anchor,corr_oof_std,corr_test_std
0,grp_SCHOOL__SUBGROUP_NAME__a5,SCHOOL|SUBGROUP_NAME,5,-0.500,81.334289,4.313096,1.719158,1.465856
1,grp_SCHOOL__ASSESSMENT_NAME__a5,SCHOOL|ASSESSMENT_NAME,5,-0.500,82.666780,2.980605,1.617578,1.946658
2,grp_SCHOOL__SUBGROUP_NAME__a20,SCHOOL|SUBGROUP_NAME,20,-0.500,83.776380,1.871004,0.722063,0.653960
3,grp_DISTRICT__ASSESSMENT_NAME__a5,DISTRICT|ASSESSMENT_NAME,5,-0.500,83.835018,1.812367,1.379330,1.437408
4,grp_SCHOOL__a5,SCHOOL,5,-0.500,84.060439,1.586946,1.341112,1.079777
5,grp_DISTRICT__SUBGROUP_NAME__a5,DISTRICT|SUBGROUP_NAME,5,-0.500,84.329519,1.317866,1.020284,0.779819
6,grp_SCHOOL__ASSESSMENT_NAME__a20,SCHOOL|ASSESSMENT_NAME,20,-0.500,84.529056,1.118329,0.534157,0.656322
7,grp_SCHOOL__a20,SCHOOL,20,-0.500,84.551904,1.095481,0.894762,0.764166
8,grp_DISTRICT__ASSESSMENT_NAME__a20,DISTRICT|ASSESSMENT_NAME,20,-0.500,84.773057,0.874328,0.696646,0.684216
9,grp_DISTRICT__SUBGROUP_NAME__a20,DISTRICT|SUBGROUP_NAME,20,-0.500,84.818798,0.828587,0.640020,0.520420


Individual screen saved to: model_results/group_residual_individual_screen.csv

Meta fold diagnostics before lambda shrink:


,fold,mse_before,mse_after_raw_meta,gain_raw_meta
0,1,86.161319,71.362821,14.798498
1,2,83.772147,68.903692,14.868455
2,3,85.657469,71.209126,14.448343
3,4,86.072973,71.273155,14.799818
4,5,86.572999,71.271056,15.301943



Top grouped residual meta-correction lambda candidates:


...


Grouped residual screen saved to: model_results/group_residual_meta_lambda_screen.csv
Internal anchor OOF MSE: 85.64738488718324
Best grouped residual OOF MSE: 70.8011606501738
Best grouped residual gain: 14.846224237009437
Best grouped residual min fold gain: 14.442496884824408

Saved grouped residual candidate:


,candidate,lambda,oof_mse_clipped,gain_vs_internal_anchor,submission_path,testpred_path,test_mean,test_std,test_min,test_max
0,group_resid_meta_on_oldnew_internal_oldw_0p025...,1.015,70.801161,14.846224,submission_group_resid_meta_on_oldnew_internal...,model_results/testpred_group_resid_meta_on_old...,54.115372,24.496373,0.0,100.0


Saved candidate list: model_results/group_residual_meta_saved_candidate.csv
Candidate file: submission_group_resid_meta_on_oldnew_internal_oldw_0p025_neww_0p875_lambda_1p015.csv

Top grouped residual meta features by coefficient magnitude:


,feature,coef_scaled_space,abs_coef
0,grp_SCHOOL__a100,64.191121,64.191121
1,grp_SCHOOL__a300,-41.944601,41.944601
2,grp_SCHOOL__SUBGROUP_NAME__a100,30.898091,30.898091
3,grp_SCHOOL__a20,-28.024627,28.024627
4,grp_SUBGROUP_NAME__a300,-24.430791,24.430791
5,grp_SCHOOL__SUBGROUP_NAME__a300,-23.708511,23.708511
6,grp_SCHOOL__ASSESSMENT_NAME__a20,15.344819,15.344819
7,grp_SUBGROUP_NAME__a5,12.852220,12.852220
8,grp_SUBGROUP_NAME__a20,10.935674,10.935674
9,grp_SUBGROUP_NAME__N_STUDENTS_BIN__a100,10.711082,10.711082


Coefficient table saved to: model_results/group_residual_meta_top_coefficients.csv


### 26A grouped residual calibration checkpoint

The grouped residual calibration experiment found a very large apparent OOF improvement. Starting from the internal 25B anchor with OOF MSE `85.6474`, the grouped residual meta-correction reached OOF MSE `70.8012` at lambda `1.015`, an apparent gain of approximately `14.8462`. Fold-level gains were also large and positive, with minimum fold gain around `14.4425`.

The strongest individual group residual signals were mostly school- and district-based combinations, especially `SCHOOL × SUBGROUP_NAME`, `SCHOOL × ASSESSMENT_NAME`, `DISTRICT × ASSESSMENT_NAME`, `SCHOOL`, and `DISTRICT × SUBGROUP_NAME`. The ridge meta-combination was also dominated by school-level smoothed residual features.

However, this result is too large to submit immediately. The current grouped residual feature construction used global cross-fitted encodings before a second meta-model cross-validation step. This can allow validation rows to indirectly influence each other through shared group residual statistics, especially for high-cardinality groups such as school and school-by-subgroup. The individual group screens also frequently selected negative lambdas at the lower search boundary, which suggests that the grouped residual structure requires further audit.

Therefore, the 26A result is treated as a major modeling insight rather than a valid submission candidate. The next step is a strictly nested, leakage-safe grouped residual calibration audit in which each outer validation fold is completely excluded from both the group residual maps and the meta-model fit.

In [51]:
# ============================================================
# 26B. Strict nested leakage-safe grouped residual audit
# ============================================================
#
# 26A found a huge grouped-residual gain, but the meta-CV may have allowed
# validation rows to influence each other through precomputed global group encodings.
#
# This cell performs a stricter nested audit:
#
# Outer fold:
#   - outer validation rows are fully held out.
#   - all group residual maps for outer validation are built only from outer-train rows.
#   - the ridge meta-model is fit only on outer-train rows.
#
# Inner fold inside each outer-train split:
#   - group features for outer-train rows are themselves built out-of-fold.
#   - this prevents the meta-model from learning from target-encoded features that
#     directly include the row's own residual.
#
# No submission should be made until this nested audit confirms a meaningful gain.

import gc
import time
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y_true, pred):
    return float(np.mean((np.asarray(y_true, dtype=float) - clip100(pred)) ** 2))

def safe_token(x):
    return str(round(float(x), 4)).replace("-", "m").replace(".", "p")

def get_test_ids_safe():
    if "test_id_series" in globals():
        return pd.Series(test_id_series).astype(str).reset_index(drop=True)
    if "test_ids" in globals():
        return pd.Series(test_ids).astype(str).reset_index(drop=True)
    if "scores_test" in globals() and ID_COL in scores_test.columns:
        return scores_test[ID_COL].astype(str).reset_index(drop=True)
    raise ValueError("Could not find test IDs.")

def make_key(df, cols):
    tmp = df.loc[:, list(cols)].copy()
    for c in cols:
        tmp[c] = tmp[c].astype("string").fillna("__NA__")
    return tmp.astype(str).agg("||".join, axis=1).to_numpy()

def smooth_group_map(keys, values, alpha):
    d = pd.DataFrame({"key": keys, "value": values})
    stats = d.groupby("key")["value"].agg(["count", "mean"])
    global_mean = float(np.mean(values))
    stats["smooth"] = (
        stats["count"] * stats["mean"] + alpha * global_mean
    ) / (stats["count"] + alpha)
    return stats["smooth"], global_mean

def map_with_default(keys, smooth_map, default):
    return pd.Series(keys).map(smooth_map).fillna(default).to_numpy(dtype=float)

def build_nested_train_and_valid_features(
    train_meta,
    train_keys_by_feature,
    resid,
    outer_tr_idx,
    outer_va_idx,
    feature_specs,
    inner_splits=5,
    inner_seed=12345,
):
    """
    Returns:
      X_outer_train: inner-OOF group residual features for outer-train rows
      X_outer_valid: group residual features for outer-valid rows, mapped from all outer-train rows

    This is the leakage-safe part.
    """
    n_outer_tr = len(outer_tr_idx)
    n_outer_va = len(outer_va_idx)
    p = len(feature_specs)

    X_tr = np.zeros((n_outer_tr, p), dtype=np.float32)
    X_va = np.zeros((n_outer_va, p), dtype=np.float32)

    inner_kf = KFold(n_splits=inner_splits, shuffle=True, random_state=inner_seed)

    for j, (feature_name, spec, alpha) in enumerate(feature_specs):
        keys_all = train_keys_by_feature[feature_name]

        # Build inner-OOF features for outer-train rows.
        for inner_tr_rel, inner_va_rel in inner_kf.split(outer_tr_idx):
            inner_tr_abs = outer_tr_idx[inner_tr_rel]
            inner_va_abs = outer_tr_idx[inner_va_rel]

            smap, default = smooth_group_map(
                keys_all[inner_tr_abs],
                resid[inner_tr_abs],
                alpha=alpha,
            )

            X_tr[inner_va_rel, j] = map_with_default(
                keys_all[inner_va_abs],
                smap,
                default,
            )

        # Build outer-valid features from all outer-train rows only.
        smap_outer, default_outer = smooth_group_map(
            keys_all[outer_tr_idx],
            resid[outer_tr_idx],
            alpha=alpha,
        )

        X_va[:, j] = map_with_default(
            keys_all[outer_va_idx],
            smap_outer,
            default_outer,
        )

    return X_tr, X_va

def build_full_oof_and_test_features(
    train_meta,
    test_meta,
    train_keys_by_feature,
    test_keys_by_feature,
    resid,
    feature_specs,
    inner_splits=5,
    seed=12345,
):
    """
    Final training features are full-data OOF encodings.
    Final test features are full-train maps applied to test.
    """
    n = len(train_meta)
    nt = len(test_meta)
    p = len(feature_specs)

    X_oof = np.zeros((n, p), dtype=np.float32)
    X_test = np.zeros((nt, p), dtype=np.float32)

    kf = KFold(n_splits=inner_splits, shuffle=True, random_state=seed)

    for j, (feature_name, spec, alpha) in enumerate(feature_specs):
        keys_train = train_keys_by_feature[feature_name]
        keys_test = test_keys_by_feature[feature_name]

        for tr_idx, va_idx in kf.split(train_meta):
            smap, default = smooth_group_map(
                keys_train[tr_idx],
                resid[tr_idx],
                alpha=alpha,
            )

            X_oof[va_idx, j] = map_with_default(
                keys_train[va_idx],
                smap,
                default,
            )

        smap_full, default_full = smooth_group_map(
            keys_train,
            resid,
            alpha=alpha,
        )

        X_test[:, j] = map_with_default(
            keys_test,
            smap_full,
            default_full,
        )

    return X_oof, X_test

# ------------------------------------------------------------
# Recreate current internal 25B anchor
# ------------------------------------------------------------

internal_old_w = 0.025
internal_new_w = 0.875

internal_anchor_name = (
    f"oldnew_internal_oldw_{str(internal_old_w).replace('.', 'p')}"
    f"_neww_{str(internal_new_w).replace('.', 'p')}"
)

internal_anchor_oof = clip100(
    base_oof + internal_old_w * old_resid_oof + internal_new_w * new_resid_oof
)

internal_anchor_test = clip100(
    base_test + internal_old_w * old_resid_test + internal_new_w * new_resid_test
)

internal_anchor_mse = mse_clip(y, internal_anchor_oof)
resid_left = np.asarray(y, dtype=float) - internal_anchor_oof

print("Internal 25B anchor:", internal_anchor_name)
print("Internal 25B anchor OOF MSE:", internal_anchor_mse)
print("Residual-left std:", float(np.std(resid_left)))

# ------------------------------------------------------------
# Metadata and feature specs
# ------------------------------------------------------------

train_meta = train_full.copy().reset_index(drop=True)
test_meta = test_full.copy().reset_index(drop=True)

if "N_STUDENTS" in train_meta.columns and "N_STUDENTS" in test_meta.columns:
    train_bins, bin_edges = pd.qcut(
        train_meta["N_STUDENTS"],
        q=10,
        duplicates="drop",
        retbins=True,
    )
    train_meta["N_STUDENTS_BIN"] = train_bins.astype(str)
    test_meta["N_STUDENTS_BIN"] = pd.cut(
        test_meta["N_STUDENTS"],
        bins=bin_edges,
        include_lowest=True,
    ).astype(str)

# Compact but targeted list based on the 26A individual screen and coefficient table.
# This keeps the nested audit feasible while focusing on the strongest discovered structures.
raw_specs = [
    ("SCHOOL",),
    ("SCHOOL", "SUBGROUP_NAME"),
    ("SCHOOL", "ASSESSMENT_NAME"),
    ("SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"),

    ("DISTRICT",),
    ("DISTRICT", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME"),

    ("COUNTY",),
    ("COUNTY", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME"),

    ("SUBGROUP_NAME",),
    ("ASSESSMENT_NAME",),
    ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("SUBGROUP_NAME", "N_STUDENTS_BIN"),
    ("ASSESSMENT_NAME", "N_STUDENTS_BIN"),
]

group_specs = []
for spec in raw_specs:
    if all(c in train_meta.columns and c in test_meta.columns for c in spec):
        group_specs.append(spec)

alpha_grid = [5, 20, 100, 300]

feature_specs = []
for spec in group_specs:
    for alpha in alpha_grid:
        feature_name = "grp_" + "__".join(spec) + f"__a{alpha}"
        feature_specs.append((feature_name, spec, alpha))

print("\nNested grouped audit feature count:", len(feature_specs))
print("Group specs:")
for spec in group_specs:
    print("  ", spec)

# Precompute keys once.
train_keys_by_feature = {}
test_keys_by_feature = {}

for feature_name, spec, alpha in feature_specs:
    # Same key for different alphas under same spec, but this simple cache name is fine.
    train_keys_by_feature[feature_name] = make_key(train_meta, spec)
    test_keys_by_feature[feature_name] = make_key(test_meta, spec)

# ------------------------------------------------------------
# Strict nested outer-CV meta residual prediction
# ------------------------------------------------------------

nested_oof_path = RESULTS_DIR / "group_residual_nested_meta_oof.npy"
nested_fold_path = RESULTS_DIR / "group_residual_nested_meta_fold_metrics.csv"

if nested_oof_path.exists():
    nested_meta_oof = np.load(nested_oof_path)
else:
    nested_meta_oof = np.full(len(y), np.nan, dtype=np.float32)

if nested_fold_path.exists():
    nested_fold_df = pd.read_csv(nested_fold_path)
else:
    nested_fold_df = pd.DataFrame()

outer_kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 2602)
outer_folds = list(outer_kf.split(train_meta))

ridge_alphas = np.logspace(-3, 5, 25)

overall_start = time.time()

for outer_fold, (outer_tr_idx, outer_va_idx) in enumerate(outer_folds, start=1):
    already_done = (
        (not nested_fold_df.empty)
        and (outer_fold in set(nested_fold_df["fold"]))
        and (not np.isnan(nested_meta_oof[outer_va_idx]).any())
    )

    if already_done:
        print(f"\nSkipping completed nested outer fold {outer_fold}/5")
        continue

    print("\n" + "=" * 90)
    print(f"Nested grouped residual audit: outer fold {outer_fold}/5")
    print("=" * 90)

    start = time.time()

    X_outer_tr, X_outer_va = build_nested_train_and_valid_features(
        train_meta=train_meta,
        train_keys_by_feature=train_keys_by_feature,
        resid=resid_left,
        outer_tr_idx=outer_tr_idx,
        outer_va_idx=outer_va_idx,
        feature_specs=feature_specs,
        inner_splits=5,
        inner_seed=RANDOM_STATE + 2700 + outer_fold,
    )

    model = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=ridge_alphas)
    )

    model.fit(X_outer_tr, resid_left[outer_tr_idx])
    pred_outer_va = model.predict(X_outer_va)

    nested_meta_oof[outer_va_idx] = pred_outer_va.astype(np.float32)
    np.save(nested_oof_path, nested_meta_oof)

    before_mse = mse_clip(y[outer_va_idx], internal_anchor_oof[outer_va_idx])
    after_raw_mse = mse_clip(y[outer_va_idx], internal_anchor_oof[outer_va_idx] + pred_outer_va)

    row = {
        "fold": outer_fold,
        "mse_before": before_mse,
        "mse_after_raw_meta": after_raw_mse,
        "gain_raw_meta": before_mse - after_raw_mse,
        "elapsed_sec": float(time.time() - start),
    }

    nested_fold_df = nested_fold_df[nested_fold_df["fold"] != outer_fold] if not nested_fold_df.empty else nested_fold_df
    nested_fold_df = pd.concat([nested_fold_df, pd.DataFrame([row])], ignore_index=True)
    nested_fold_df = nested_fold_df.sort_values("fold").reset_index(drop=True)
    nested_fold_df.to_csv(nested_fold_path, index=False)

    print("Fold before MSE:", before_mse)
    print("Fold after raw nested meta MSE:", after_raw_mse)
    print("Fold raw gain:", before_mse - after_raw_mse)
    print("Elapsed minutes:", round((time.time() - start) / 60, 2))

    del X_outer_tr, X_outer_va, model, pred_outer_va
    gc.collect()

missing = int(np.isnan(nested_meta_oof).sum())
print("\nNested OOF missing rows:", missing)

if missing > 0:
    print("Not all nested folds are complete. Re-run this same cell to resume.")
else:
    print("\nNested outer-fold diagnostics:")
    try:
        display(nested_fold_df)
    except NameError:
        print(nested_fold_df.to_string(index=False))

    # --------------------------------------------------------
    # Lambda screen using strictly nested OOF meta correction
    # --------------------------------------------------------

    rows = []

    for lam in np.round(np.arange(-1.000, 1.501, 0.005), 3):
        pred_oof = clip100(internal_anchor_oof + lam * nested_meta_oof)

        fold_gains = []
        fold_mses = []

        for fold, (_, va_idx) in enumerate(outer_folds, start=1):
            before = mse_clip(y[va_idx], internal_anchor_oof[va_idx])
            after = mse_clip(y[va_idx], pred_oof[va_idx])
            fold_gains.append(before - after)
            fold_mses.append(after)

        rows.append({
            "lambda": float(lam),
            "oof_mse_clipped": mse_clip(y, pred_oof),
            "gain_vs_internal_anchor": internal_anchor_mse - mse_clip(y, pred_oof),
            "min_fold_gain": float(np.min(fold_gains)),
            "mean_fold_gain": float(np.mean(fold_gains)),
            "max_fold_gain": float(np.max(fold_gains)),
            "fold_mse_std": float(np.std(fold_mses, ddof=1)),
        })

    screen_df = pd.DataFrame(rows).sort_values("oof_mse_clipped").reset_index(drop=True)

    screen_path = RESULTS_DIR / "group_residual_nested_meta_lambda_screen.csv"
    screen_df.to_csv(screen_path, index=False)

    print("\nTop strict nested grouped residual lambda candidates:")
    try:
        display(screen_df.head(30))
    except NameError:
        print(screen_df.head(30).to_string(index=False))

    print("\nStrict nested grouped screen saved to:", screen_path)
    print("Internal anchor OOF MSE:", internal_anchor_mse)
    print("Best strict nested grouped OOF MSE:", float(screen_df.loc[0, "oof_mse_clipped"]))
    print("Best strict nested grouped gain:", float(screen_df.loc[0, "gain_vs_internal_anchor"]))
    print("Best strict nested grouped min fold gain:", float(screen_df.loc[0, "min_fold_gain"]))

    # --------------------------------------------------------
    # Build final full-train OOF features and test features
    # only if nested audit confirms real signal.
    # --------------------------------------------------------

    best = screen_df.iloc[0]
    best_gain = float(best["gain_vs_internal_anchor"])
    best_min_fold_gain = float(best["min_fold_gain"])
    best_lam = float(best["lambda"])

    if best_gain <= 0 or best_min_fold_gain <= 0:
        print("\nNested audit did not confirm a positive stable gain. Do not use grouped residual submission.")
    else:
        print("\nNested audit confirms positive grouped residual signal.")
        print("Now building full OOF/test grouped features for saved candidate.")

        X_full_oof, X_test_group = build_full_oof_and_test_features(
            train_meta=train_meta,
            test_meta=test_meta,
            train_keys_by_feature=train_keys_by_feature,
            test_keys_by_feature=test_keys_by_feature,
            resid=resid_left,
            feature_specs=feature_specs,
            inner_splits=5,
            seed=RANDOM_STATE + 2800,
        )

        final_model = make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=ridge_alphas)
        )

        final_model.fit(X_full_oof, resid_left)
        final_meta_test = final_model.predict(X_test_group)

        final_oof = clip100(internal_anchor_oof + best_lam * nested_meta_oof)
        final_test = clip100(internal_anchor_test + best_lam * final_meta_test)

        name = f"group_resid_nested_on_{internal_anchor_name}_lambda_{safe_token(best_lam)}"

        oof_path = RESULTS_DIR / f"oof_{name}.csv"
        testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
        sub_path = Path(f"submission_{name}.csv")

        pd.DataFrame({
            TARGET_COL: y,
            f"OOF_{name}": final_oof,
        }).to_csv(oof_path, index=False)

        pd.DataFrame({
            ID_COL: get_test_ids_safe().values,
            f"TESTPRED_{name}": final_test,
        }).to_csv(testpred_path, index=False)

        pd.DataFrame({
            ID_COL: get_test_ids_safe().values,
            TARGET_COL: final_test,
        }).to_csv(sub_path, index=False)

        saved = pd.DataFrame([{
            "candidate": name,
            "lambda": best_lam,
            "oof_mse_clipped": mse_clip(y, final_oof),
            "gain_vs_internal_anchor": internal_anchor_mse - mse_clip(y, final_oof),
            "min_fold_gain": best_min_fold_gain,
            "submission_path": str(sub_path),
            "testpred_path": str(testpred_path),
            "test_mean": float(np.mean(final_test)),
            "test_std": float(np.std(final_test)),
            "test_min": float(np.min(final_test)),
            "test_max": float(np.max(final_test)),
        }])

        saved_path = RESULTS_DIR / "group_residual_nested_meta_saved_candidate.csv"
        saved.to_csv(saved_path, index=False)

        print("\nSaved strict nested grouped residual candidate:")
        try:
            display(saved)
        except NameError:
            print(saved.to_string(index=False))

        print("Saved candidate list:", saved_path)
        print("Candidate file:", sub_path)

        # Coefficients for interpretation only.
        ridge = final_model.named_steps["ridgecv"]
        coef_df = pd.DataFrame({
            "feature": [f[0] for f in feature_specs],
            "coef_scaled_space": ridge.coef_,
        }).assign(abs_coef=lambda d: d["coef_scaled_space"].abs())

        coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)
        coef_path = RESULTS_DIR / "group_residual_nested_meta_top_coefficients.csv"
        coef_df.to_csv(coef_path, index=False)

        print("\nTop strict nested grouped residual meta features:")
        try:
            display(coef_df.head(30))
        except NameError:
            print(coef_df.head(30).to_string(index=False))

        print("Coefficient table saved to:", coef_path)

    print("\nTotal elapsed minutes:", round((time.time() - overall_start) / 60, 2))

Internal 25B anchor: oldnew_internal_oldw_0p025_neww_0p875
Internal 25B anchor OOF MSE: 85.64738488718324
Residual-left std: 9.25440618958025

Nested grouped audit feature count: 64
Group specs:
   ('SCHOOL',)
   ('SCHOOL', 'SUBGROUP_NAME')
   ('SCHOOL', 'ASSESSMENT_NAME')
   ('SCHOOL', 'ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('DISTRICT',)
   ('DISTRICT', 'SUBGROUP_NAME')
   ('DISTRICT', 'ASSESSMENT_NAME')
   ('DISTRICT', 'ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('COUNTY',)
   ('COUNTY', 'SUBGROUP_NAME')
   ('COUNTY', 'ASSESSMENT_NAME')
   ('SUBGROUP_NAME',)
   ('ASSESSMENT_NAME',)
   ('ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('SUBGROUP_NAME', 'N_STUDENTS_BIN')
   ('ASSESSMENT_NAME', 'N_STUDENTS_BIN')

Nested grouped residual audit: outer fold 1/5
Fold before MSE: 86.81741151120198
Fold after raw nested meta MSE: 81.98623487551272
Fold raw gain: 4.8311766356892605
Elapsed minutes: 0.14

Nested grouped residual audit: outer fold 2/5
Fold before MSE: 84.71751538100786
Fold after raw nested me

,fold,mse_before,mse_after_raw_meta,gain_raw_meta,elapsed_sec
0,1,86.817412,81.986235,4.831177,8.564025
1,2,84.717515,79.826294,4.891221,8.369281
2,3,85.375399,80.594340,4.781059,7.687655
3,4,85.305619,80.843871,4.461749,7.783906
4,5,86.020939,81.733959,4.286980,7.551215



Top strict nested grouped residual lambda candidates:


,lambda,oof_mse_clipped,gain_vs_internal_anchor,min_fold_gain,mean_fold_gain,max_fold_gain,fold_mse_std
0,1.175,80.892639,4.754746,4.289962,4.754744,5.056076,0.913218
1,1.180,80.892721,4.754664,4.286859,4.754662,5.057837,0.914379
2,1.170,80.892726,4.754659,4.292897,4.754657,5.054151,0.912063
3,1.185,80.892974,4.754411,4.283575,4.754409,5.059435,0.915549
4,1.165,80.892981,4.754404,4.295673,4.754403,5.052061,0.910911
5,1.190,80.893396,4.753989,4.280110,4.753987,5.060871,0.916729
6,1.160,80.893402,4.753983,4.298294,4.753982,5.049805,0.909760
7,1.195,80.893989,4.753396,4.276464,4.753395,5.062147,0.917919
8,1.155,80.893993,4.753392,4.300735,4.753390,5.047383,0.908616
9,1.200,80.894752,4.752633,4.272641,4.752631,5.063257,0.919116



Strict nested grouped screen saved to: model_results/group_residual_nested_meta_lambda_screen.csv
Internal anchor OOF MSE: 85.64738488718324
Best strict nested grouped OOF MSE: 80.89263904946873
Best strict nested grouped gain: 4.754745837714509
Best strict nested grouped min fold gain: 4.289962241156971

Nested audit confirms positive grouped residual signal.
Now building full OOF/test grouped features for saved candidate.

Saved strict nested grouped residual candidate:


,candidate,lambda,oof_mse_clipped,gain_vs_internal_anchor,min_fold_gain,submission_path,testpred_path,test_mean,test_std,test_min,test_max
0,group_resid_nested_on_oldnew_internal_oldw_0p0...,1.175,80.892639,4.754746,4.289962,submission_group_resid_nested_on_oldnew_intern...,model_results/testpred_group_resid_nested_on_o...,54.10565,24.709315,0.0,100.0


Saved candidate list: model_results/group_residual_nested_meta_saved_candidate.csv
Candidate file: submission_group_resid_nested_on_oldnew_internal_oldw_0p025_neww_0p875_lambda_1p175.csv

Top strict nested grouped residual meta features:


,feature,coef_scaled_space,abs_coef
0,grp_SCHOOL__SUBGROUP_NAME__a100,58.158787,58.158787
1,grp_SCHOOL__a100,57.613281,57.613281
2,grp_SCHOOL__SUBGROUP_NAME__a300,-42.132343,42.132343
3,grp_SCHOOL__a300,-37.478886,37.478886
4,grp_SUBGROUP_NAME__N_STUDENTS_BIN__a100,34.236969,34.236969
5,grp_SCHOOL__ASSESSMENT_NAME__a20,-26.756861,26.756861
6,grp_SCHOOL__a20,-25.956554,25.956554
7,grp_SCHOOL__SUBGROUP_NAME__a20,-18.420261,18.420261
8,grp_SUBGROUP_NAME__N_STUDENTS_BIN__a5,-17.465593,17.465593
9,grp_SCHOOL__ASSESSMENT_NAME__a300,11.704391,11.704391


Coefficient table saved to: model_results/group_residual_nested_meta_top_coefficients.csv

Total elapsed minutes: 0.85


### 26B strict nested grouped residual submission — public check failed

Submitted file:

`submission_group_resid_nested_on_oldnew_internal_oldw_0p025_neww_0p875_lambda_1p175.csv`

Public leaderboard MSE:

**83.131**

Local strict nested OOF diagnostics had looked strong:

- internal 25B anchor OOF MSE: **85.647385**
- strict nested grouped residual OOF MSE: **80.892639**
- local OOF gain: **4.754746**
- all five outer folds had positive gains
- minimum outer-fold gain: about **4.29**

However, the public leaderboard result is much worse than the current submitted anchor:

`submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv`

with public MSE **77.277**.

Interpretation:

The 26B nested audit reduced leakage risk relative to 26A, but the public result shows that the direct high-cardinality grouped residual correction is not robust to the public test distribution. The grouped residual signal may still describe real training-set structure, but it is too public/private-sensitive in this form. Do not submit 26A, do not submit further small 26B lambda variants, and do not continue optimizing direct school/group residual maps for Kaggle slots.

Decision:

Demote the grouped residual meta-correction branch to diagnostics/backlog. Continue from the 77.277 residual-stack public anchor and pivot to safer residual diversity models that avoid direct high-cardinality grouped residual maps.

In [52]:
# 27A. Log failed 26B public result and protect current public anchor

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

tracker_path = RESULTS_DIR / "submission_tracker.csv"

new_row = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "file": "submission_group_resid_nested_on_oldnew_internal_oldw_0p025_neww_0p875_lambda_1p175.csv",
    "source": "current notebook",
    "model_family": "strict nested grouped residual meta-correction",
    "local_oof_mse": 80.89263904946873,
    "public_mse": 83.131,
    "current_public_anchor_file": "submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv",
    "current_public_anchor_mse": 77.277,
    "decision": "failed public check; do not continue direct high-cardinality grouped residual submissions",
    "notes": (
        "Local nested grouped OOF gain was large and fold-stable, but public MSE was much worse "
        "than the 77.277 residual-stack anchor. Treat direct group residual maps as public-fragile."
    ),
}

if tracker_path.exists():
    tracker = pd.read_csv(tracker_path)
else:
    tracker = pd.DataFrame()

# Remove older duplicate row for the same file, then append the corrected public result.
if len(tracker) > 0 and "file" in tracker.columns:
    tracker = tracker[tracker["file"] != new_row["file"]].copy()

tracker = pd.concat([tracker, pd.DataFrame([new_row])], ignore_index=True)
tracker.to_csv(tracker_path, index=False)

print("Saved tracker:", tracker_path)
display(tracker.tail(10))

Saved tracker: model_results/submission_tracker.csv


,submission_file,source,model_family,public_mse,notes,file,private_mse,timestamp,local_oof_mse,current_public_anchor_file,current_public_anchor_mse,decision
0,submission_blend_auto_et_lgbm_long_oof_weighte...,current notebook,automatic OOF blend,NaN,Automatic post-long-run OOF blend; OOF MSE=88....,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,current notebook,OOF blend,80.662,Automatic long-run blend: 0.319 ExtraTrees saf...,submission_blend_auto_et_lgbm_long_oof_weighte...,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,77.277,New public anchor. Expanded residual stack on ...,submission_residstack_lgbm_on_best80p662_blend...,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,current notebook,strict nested grouped residual meta-correction,83.131,Local nested grouped OOF gain was large and fo...,submission_group_resid_nested_on_oldnew_intern...,NaN,2026-05-09 03:34:53,80.892639,submission_residstack_lgbm_on_best80p662_blend...,77.277,failed public check; do not continue direct hi...


### 27B fresh restart after invalid zero-residual run

The previous 27 overnight no-group residual suite is discarded.

Reason:
The run finished suspiciously fast and produced impossible diagnostics, including zero OOF MSE candidates. This indicates that the 27B anchor loader likely selected the true training target instead of the real OOF prediction for the submitted 77.277 residual-stack anchor.

Decision:
Restart the 27 branch from a corrected 27B. The corrected 27B must explicitly verify that the anchor OOF prediction has MSE near the known residual-stack OOF value, about **87.136**, and must reject any target-like column with MSE near zero.

Only the bad 27 branch artifacts are archived. Older RF, ExtraTrees, LightGBM, blend, and residual-stack artifacts are kept.

In [56]:
# 27B fresh restart: archive bad resid27 artifacts and rebuild feature matrix from verified 77.277 anchor

from pathlib import Path
import glob
import shutil
import gc

import numpy as np
import pandas as pd
from scipy import sparse

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
PRED_COL = "PERCENT_PROFICIENT"
EXPECTED_ANCHOR_OOF_MSE = 87.136049

# ---------------------------------------------------------------------
# 1. Archive bad resid27 files so the next run cannot resume from them
# ---------------------------------------------------------------------

BAD_DIR = RESULTS_DIR / "bad_resid27_zero_residual_run"
BAD_DIR.mkdir(exist_ok=True)

bad_patterns = [
    RESULTS_DIR / "resid27_nogroup_*",
    RESULTS_DIR / "oofcorr_resid27_nogroup_*.csv",
    RESULTS_DIR / "testcorr_resid27_nogroup_*.csv",
    RESULTS_DIR / "oofpred_resid27_nogroup_*.csv",
    RESULTS_DIR / "testpred_resid27_nogroup_*.csv",
    RESULTS_DIR / "resid27_overnight_*.csv",
    Path("submission_resid27_nogroup_*.csv"),
]

moved = []

for pattern in bad_patterns:
    for path_str in glob.glob(str(pattern)):
        path = Path(path_str)
        if not path.exists() or not path.is_file():
            continue

        dest = BAD_DIR / path.name
        k = 1
        while dest.exists():
            dest = BAD_DIR / f"{path.stem}_dup{k}{path.suffix}"
            k += 1

        shutil.move(str(path), str(dest))
        moved.append((str(path), str(dest)))

print(f"Archived {len(moved)} old/bad resid27 files.")
if moved:
    display(pd.DataFrame(moved, columns=["from", "to"]).head(40))

# ---------------------------------------------------------------------
# 2. Basic checks and helpers
# ---------------------------------------------------------------------

required_objects = [
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "train_ids",
    "test_ids",
]

missing = [x for x in required_objects if x not in globals()]
if missing:
    raise RuntimeError(
        "Missing required objects. Rerun preprocessing/setup cells first: "
        + ", ".join(missing)
    )

y_arr = np.asarray(y_train, dtype=np.float64).reshape(-1)
n_train = len(y_arr)
n_test = X_test_proc_model.shape[0]

def mse_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    return float(np.mean((y_true - y_pred) ** 2))

def get_id_series(id_obj, expected_len, label):
    if isinstance(id_obj, pd.DataFrame):
        if ID_COL in id_obj.columns:
            vals = id_obj[ID_COL].to_numpy()
        else:
            vals = id_obj.iloc[:, 0].to_numpy()
    elif isinstance(id_obj, pd.Series):
        vals = id_obj.to_numpy()
    else:
        vals = np.asarray(id_obj).reshape(-1)

    if len(vals) != expected_len:
        raise ValueError(f"{label} length mismatch: got {len(vals)}, expected {expected_len}")

    return pd.Series(vals, name=ID_COL).astype(str).reset_index(drop=True)

train_id_values = get_id_series(train_ids, n_train, "train_ids")
test_id_values = get_id_series(test_ids, n_test, "test_ids")

def read_aligned_numeric_column(path, expected_ids, col):
    df = pd.read_csv(path)

    if col not in df.columns:
        raise ValueError(f"Column {col} not found in {path}")

    vals = pd.to_numeric(df[col], errors="coerce")

    if ID_COL in df.columns:
        tmp = pd.DataFrame({
            ID_COL: df[ID_COL].astype(str),
            "_pred": vals,
        })

        if tmp[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in {path}")

        aligned = tmp.set_index(ID_COL).reindex(expected_ids)["_pred"]

        if aligned.isna().any():
            raise ValueError(
                f"{path} column {col} failed ID alignment; missing {int(aligned.isna().sum())} rows"
            )

        return aligned.to_numpy(dtype=float)

    if len(vals) != len(expected_ids):
        raise ValueError(f"{path} has no ID column and length does not match expected IDs")

    if vals.isna().any():
        raise ValueError(f"{path} column {col} contains missing/non-numeric values")

    return vals.to_numpy(dtype=float)

def scan_numeric_prediction_columns(path, expected_ids, y_true=None):
    df = pd.read_csv(path)
    rows = []

    for col in df.columns:
        if col == ID_COL:
            continue

        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().mean() < 0.95:
            continue

        try:
            pred = read_aligned_numeric_column(path, expected_ids, col)

            row = {
                "path": str(path),
                "column": col,
                "n": len(pred),
                "mean": float(np.mean(pred)),
                "std": float(np.std(pred)),
                "min": float(np.min(pred)),
                "max": float(np.max(pred)),
                "missing": int(np.isnan(pred).sum()),
                "outside_0_100": int(((pred < 0) | (pred > 100)).sum()),
            }

            if y_true is not None:
                mse = mse_np(y_true, pred)
                row["mse_vs_y"] = mse
                row["target_like"] = bool(mse < 1e-8)
                row["abs_from_expected_anchor"] = abs(mse - EXPECTED_ANCHOR_OOF_MSE)

            rows.append(row)

        except Exception as e:
            rows.append({
                "path": str(path),
                "column": col,
                "error": repr(e),
            })

    return rows

# ---------------------------------------------------------------------
# 3. Find and verify the real submitted 77.277 residual-stack OOF anchor
# ---------------------------------------------------------------------

anchor_oof_patterns = [
    RESULTS_DIR / "oofpred_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv",
    RESULTS_DIR / "oof_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv",
    Path("oofpred_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv"),
    Path("oof_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv"),
    RESULTS_DIR / "*oof*residstack*lgbm*best80p662*1p135*.csv",
    RESULTS_DIR / "*oof*residstack*lgbm*lambda_1p135*.csv",
    Path("*oof*residstack*lgbm*best80p662*1p135*.csv"),
]

anchor_oof_files = []

for pattern in anchor_oof_patterns:
    for p in glob.glob(str(pattern)):
        p = Path(p)
        if p.exists() and p.is_file() and p not in anchor_oof_files:
            anchor_oof_files.append(p)

if not anchor_oof_files:
    raise FileNotFoundError(
        "Could not find the OOF prediction file for the submitted 77.277 residual-stack anchor."
    )

scan_rows = []
for p in anchor_oof_files:
    scan_rows.extend(scan_numeric_prediction_columns(p, train_id_values, y_true=y_arr))

scan_df = pd.DataFrame(scan_rows)

print("\nCandidate OOF columns for the 77.277 anchor:")
display(
    scan_df
    .sort_values(["target_like", "abs_from_expected_anchor"], ascending=[True, True])
    .head(50)
)

valid_anchor_rows = scan_df[
    (scan_df["missing"] == 0)
    & (scan_df["target_like"] == False)
    & (scan_df["mse_vs_y"].between(85.0, 89.5))
].copy()

if valid_anchor_rows.empty:
    raise RuntimeError(
        "No valid 77.277-anchor OOF column found. "
        "The correct OOF MSE should be around 87.136, not 0. Paste the scan table."
    )

best_anchor_row = valid_anchor_rows.sort_values("abs_from_expected_anchor").iloc[0]

anchor27_oof_path = Path(best_anchor_row["path"])
anchor27_oof_col = best_anchor_row["column"]

anchor27_oof = read_aligned_numeric_column(anchor27_oof_path, train_id_values, anchor27_oof_col)
anchor27_oof = np.clip(anchor27_oof, 0, 100)
anchor27_oof_mse = mse_np(y_arr, anchor27_oof)

print("\nSelected verified 77.277-anchor OOF:")
print("path:", anchor27_oof_path)
print("column:", anchor27_oof_col)
print("OOF MSE:", anchor27_oof_mse)

if not (86.5 <= anchor27_oof_mse <= 87.8):
    raise RuntimeError(
        f"Selected anchor OOF MSE is {anchor27_oof_mse}, not close enough to expected 87.136. Stop."
    )

# ---------------------------------------------------------------------
# 4. Load matching submitted test predictions
# ---------------------------------------------------------------------

anchor_test_patterns = [
    Path("submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv"),
    RESULTS_DIR / "testpred_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv",
    RESULTS_DIR / "*testpred*residstack*lgbm*best80p662*1p135*.csv",
    RESULTS_DIR / "*testpred*residstack*lgbm*lambda_1p135*.csv",
    Path("submission*residstack*lgbm*best80p662*1p135*.csv"),
]

anchor_test_files = []

for pattern in anchor_test_patterns:
    for p in glob.glob(str(pattern)):
        p = Path(p)
        if p.exists() and p.is_file() and p not in anchor_test_files:
            anchor_test_files.append(p)

if not anchor_test_files:
    raise FileNotFoundError(
        "Could not find the matching test/submission file for the submitted 77.277 anchor."
    )

test_scan_rows = []
for p in anchor_test_files:
    test_scan_rows.extend(scan_numeric_prediction_columns(p, test_id_values, y_true=None))

test_scan_df = pd.DataFrame(test_scan_rows)

print("\nCandidate test columns for the 77.277 anchor:")
display(test_scan_df)

valid_test_rows = test_scan_df[
    (test_scan_df["n"] == n_test)
    & (test_scan_df["missing"] == 0)
    & (test_scan_df["outside_0_100"] == 0)
].copy()

if valid_test_rows.empty:
    raise RuntimeError("No valid matching test prediction found. Paste the test scan table.")

valid_test_rows["is_exact_submission"] = valid_test_rows["path"].str.contains(
    "submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv",
    regex=False
)

best_test_row = valid_test_rows.sort_values("is_exact_submission", ascending=False).iloc[0]

anchor27_test_path = Path(best_test_row["path"])
anchor27_test_col = best_test_row["column"]

anchor27_test = read_aligned_numeric_column(anchor27_test_path, test_id_values, anchor27_test_col)
anchor27_test = np.clip(anchor27_test, 0, 100)

print("\nSelected verified 77.277-anchor test/submission prediction:")
print("path:", anchor27_test_path)
print("column:", anchor27_test_col)
print(pd.Series(anchor27_test).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

# ---------------------------------------------------------------------
# 5. Load saved component predictions as residual-model meta-features
# ---------------------------------------------------------------------

component_paths = {
    "rf500": (
        RESULTS_DIR / "oof_rf_500_base.csv",
        RESULTS_DIR / "testpred_rf_500_base_folds.csv",
    ),
    "et_safe": (
        RESULTS_DIR / "oof_extratrees_safe_base.csv",
        RESULTS_DIR / "testpred_extratrees_safe_base_foldavg.csv",
    ),
    "lgbm_12k": (
        RESULTS_DIR / "oof_lgbm_t03_base_5fold_oof.csv",
        RESULTS_DIR / "testpred_lgbm_t03_base_5fold_oof_foldavg.csv",
    ),
    "lgbm_long80k": (
        RESULTS_DIR / "oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv",
        RESULTS_DIR / "testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv",
    ),
    "lgbm_long100k": (
        RESULTS_DIR / "oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv",
        RESULTS_DIR / "testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv",
    ),
    "lgbm_div02_3fold": (
        RESULTS_DIR / "oof_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof.csv",
        RESULTS_DIR / "testpred_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof_foldavg.csv",
    ),
}

def auto_load_oof_component(path, expected_ids, y_true, label):
    rows = scan_numeric_prediction_columns(path, expected_ids, y_true=y_true)
    df = pd.DataFrame(rows)

    valid = df[
        (df["missing"] == 0)
        & (df["target_like"] == False)
        & (df["mse_vs_y"] > 1.0)
    ].copy()

    if valid.empty:
        raise RuntimeError(f"No valid prediction column found for {label}: {path}")

    # Usually there is one prediction column. If more than one, choose the best non-target-like MSE.
    row = valid.sort_values("mse_vs_y").iloc[0]
    pred = read_aligned_numeric_column(path, expected_ids, row["column"])

    print(f"Loaded {label}: {path} | column={row['column']} | OOF MSE={row['mse_vs_y']:.6f}")
    return pred

def auto_load_test_component(path, expected_ids, label):
    rows = scan_numeric_prediction_columns(path, expected_ids, y_true=None)
    df = pd.DataFrame(rows)

    valid = df[
        (df["n"] == len(expected_ids))
        & (df["missing"] == 0)
    ].copy()

    if valid.empty:
        raise RuntimeError(f"No valid test prediction column found for {label}: {path}")

    # Prefer standard submission column if present.
    valid["preferred"] = (valid["column"] == PRED_COL).astype(int)
    row = valid.sort_values("preferred", ascending=False).iloc[0]
    pred = read_aligned_numeric_column(path, expected_ids, row["column"])

    print(f"Loaded {label}: {path} | column={row['column']}")
    return pred

component_oof = {}
component_test = {}

print("\nLoading component prediction artifacts:")

for name, (oof_path, test_path) in component_paths.items():
    if Path(oof_path).exists() and Path(test_path).exists():
        component_oof[name] = auto_load_oof_component(oof_path, train_id_values, y_arr, f"{name} OOF")
        component_test[name] = auto_load_test_component(test_path, test_id_values, f"{name} test")
    else:
        print(f"Skipping missing component: {name}")

# ---------------------------------------------------------------------
# 6. Build corrected augmented residual-model matrix
# ---------------------------------------------------------------------

def to_csr_float32(X):
    if sparse.issparse(X):
        return X.tocsr().astype(np.float32)
    if isinstance(X, pd.DataFrame):
        arr = X.to_numpy()
    else:
        arr = np.asarray(X)
    return sparse.csr_matrix(arr.astype(np.float32))

X_base_train_27B = to_csr_float32(X_train_proc_model)
X_base_test_27B = to_csr_float32(X_test_proc_model)

extra_train = []
extra_test = []
extra_names_27B = []

def add_extra_feature(name, train_values, test_values):
    train_values = np.asarray(train_values, dtype=np.float32).reshape(-1)
    test_values = np.asarray(test_values, dtype=np.float32).reshape(-1)

    if len(train_values) != n_train:
        raise ValueError(f"{name} train length mismatch")
    if len(test_values) != n_test:
        raise ValueError(f"{name} test length mismatch")

    extra_names_27B.append(name)
    extra_train.append(train_values)
    extra_test.append(test_values)

add_extra_feature("anchor77_residstack_pred", anchor27_oof, anchor27_test)

for name in sorted(component_oof):
    tr = np.clip(component_oof[name], 0, 100)
    te = np.clip(component_test[name], 0, 100)

    add_extra_feature(f"pred_{name}", tr, te)
    add_extra_feature(f"diff_{name}_minus_anchor77", tr - anchor27_oof, te - anchor27_test)

if len(component_oof) >= 2:
    comp_names_sorted = sorted(component_oof)

    comp_train_mat = np.column_stack([
        np.clip(component_oof[n], 0, 100) for n in comp_names_sorted
    ])

    comp_test_mat = np.column_stack([
        np.clip(component_test[n], 0, 100) for n in comp_names_sorted
    ])

    add_extra_feature("component_mean", comp_train_mat.mean(axis=1), comp_test_mat.mean(axis=1))
    add_extra_feature("component_std", comp_train_mat.std(axis=1), comp_test_mat.std(axis=1))
    add_extra_feature("component_min", comp_train_mat.min(axis=1), comp_test_mat.min(axis=1))
    add_extra_feature("component_max", comp_train_mat.max(axis=1), comp_test_mat.max(axis=1))
    add_extra_feature(
        "component_range",
        comp_train_mat.max(axis=1) - comp_train_mat.min(axis=1),
        comp_test_mat.max(axis=1) - comp_test_mat.min(axis=1),
    )

extra_train_mat_27B = np.column_stack(extra_train).astype(np.float32)
extra_test_mat_27B = np.column_stack(extra_test).astype(np.float32)

X_aug_train_27B = sparse.hstack(
    [X_base_train_27B, sparse.csr_matrix(extra_train_mat_27B)],
    format="csr"
).astype(np.float32)

X_aug_test_27B = sparse.hstack(
    [X_base_test_27B, sparse.csr_matrix(extra_test_mat_27B)],
    format="csr"
).astype(np.float32)

resid_target_27B = (y_arr - anchor27_oof).astype(np.float32)

print("\nCorrected 27B complete.")
print("Verified anchor OOF MSE:", anchor27_oof_mse)
print("Residual target summary:")
print(pd.Series(resid_target_27B).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print()
print("Extra feature count:", len(extra_names_27B))
print("Extra feature names:", extra_names_27B)
print("Augmented train shape:", X_aug_train_27B.shape)
print("Augmented test shape:", X_aug_test_27B.shape)

# ---------------------------------------------------------------------
# 7. Compatibility aliases for the old 27C residual-training cell
# ---------------------------------------------------------------------
# These aliases let the old 27C cell use the corrected 27B objects.

anchor_oof = anchor27_oof
anchor_test = anchor27_test
anchor_oof_mse = anchor27_oof_mse

X_aug_train = X_aug_train_27B
X_aug_test = X_aug_test_27B
resid_target_27 = resid_target_27B
extra_names = extra_names_27B

print("\nCompatibility aliases set:")
print("anchor_oof, anchor_test, anchor_oof_mse")
print("X_aug_train, X_aug_test, resid_target_27, extra_names")

gc.collect()

Archived 63 old/bad resid27 files.


,from,to
0,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...
1,model_results/resid27_nogroup_huber_l95_child1...,model_results/bad_resid27_zero_residual_run/re...
2,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...
3,model_results/resid27_nogroup_huber_l95_child1...,model_results/bad_resid27_zero_residual_run/re...
4,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...
5,model_results/resid27_nogroup_huber_l95_child1...,model_results/bad_resid27_zero_residual_run/re...
6,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...
7,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...
8,model_results/resid27_nogroup_huber_l95_child1...,model_results/bad_resid27_zero_residual_run/re...
9,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...



Candidate OOF columns for the 77.277 anchor:


,path,column,n,mean,std,min,max,missing,outside_0_100,mse_vs_y,target_like,abs_from_expected_anchor
1,model_results/oof_residstack_lgbm_on_best80p66...,OOF_residstack_lgbm_on_best80p662_blend_5fold_...,144921,54.186484,24.786756,0.0,100.0,0,0,87.136049,False,2.514173e-07
0,model_results/oof_residstack_lgbm_on_best80p66...,PERCENT_PROFICIENT,144921,54.183521,26.429834,0.0,100.0,0,0,0.000000,True,8.713605e+01



Selected verified 77.277-anchor OOF:
path: model_results/oof_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv
column: OOF_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135
OOF MSE: 87.13604874858268

Candidate test columns for the 77.277 anchor:


,path,column,n,mean,std,min,max,missing,outside_0_100
0,submission_residstack_lgbm_on_best80p662_blend...,PERCENT_PROFICIENT,48307,54.097842,24.676033,0.18864,100.0,0,0
1,model_results/testpred_residstack_lgbm_on_best...,TESTPRED_residstack_lgbm_on_best80p662_blend_5...,48307,54.097842,24.676033,0.18864,100.0,0,0



Selected verified 77.277-anchor test/submission prediction:
path: submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv
column: PERCENT_PROFICIENT
count    48307.000000
mean        54.097842
std         24.676288
min          0.188640
1%           7.488321
5%          15.889755
50%         52.544583
95%         95.256640
99%         99.462163
max        100.000000
dtype: float64

Loading component prediction artifacts:
Loaded rf500 OOF: model_results/oof_rf_500_base.csv | column=rf_500_base_oof_pred | OOF MSE=122.328024
Loaded rf500 test: model_results/testpred_rf_500_base_folds.csv | column=rf_500_base_fold1_pred
Loaded et_safe OOF: model_results/oof_extratrees_safe_base.csv | column=oof_pred | OOF MSE=110.749294
Loaded et_safe test: model_results/testpred_extratrees_safe_base_foldavg.csv | column=PERCENT_PROFICIENT
Loaded lgbm_12k OOF: model_results/oof_lgbm_t03_base_5fold_oof.csv | column=OOF_lgbm_t03_base_5fold_oof | OOF MSE=98.779406
Loaded lgbm_12k test:

98

In [57]:
# 27B.1 patch: replace RF test feature with fold-average, then rebuild augmented matrices

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import gc

rf_test_path = Path("model_results/testpred_rf_500_base_folds.csv")

if not rf_test_path.exists():
    raise FileNotFoundError(rf_test_path)

rf_test_df = pd.read_csv(rf_test_path)

rf_num_cols = [
    c for c in rf_test_df.columns
    if c != ID_COL and pd.api.types.is_numeric_dtype(rf_test_df[c])
]

print("RF test numeric columns:")
print(rf_num_cols)

avg_like_cols = [
    c for c in rf_num_cols
    if any(tok in c.lower() for tok in ["avg", "mean", "foldavg", "fold_avg"])
]

fold_cols = [
    c for c in rf_num_cols
    if "fold" in c.lower()
]

if avg_like_cols:
    rf_used_cols = [avg_like_cols[0]]
    rf_test_fixed_method = f"existing average column: {avg_like_cols[0]}"
elif len(fold_cols) >= 2:
    rf_used_cols = fold_cols
    rf_test_fixed_method = f"mean of {len(fold_cols)} fold columns"
else:
    rf_used_cols = rf_num_cols
    rf_test_fixed_method = f"mean of all {len(rf_num_cols)} numeric prediction columns"

if len(rf_used_cols) == 0:
    raise RuntimeError("Could not identify RF test prediction columns.")

tmp = rf_test_df[[ID_COL] + rf_used_cols].copy()
tmp[ID_COL] = tmp[ID_COL].astype(str)

if tmp[ID_COL].duplicated().any():
    raise RuntimeError("Duplicate ASSESSMENT_ID in RF test file.")

aligned_rf = tmp.set_index(ID_COL).reindex(test_id_values)

if aligned_rf[rf_used_cols].isna().any().any():
    raise RuntimeError("RF test predictions failed ID alignment.")

rf500_test_fixed = aligned_rf[rf_used_cols].mean(axis=1).to_numpy(dtype=np.float64)

component_test["rf500"] = rf500_test_fixed

print("\nRF test fixed using:", rf_test_fixed_method)
print(pd.Series(rf500_test_fixed).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

# Rebuild augmented features using corrected component_test["rf500"]

extra_train = []
extra_test = []
extra_names_27B = []

def add_extra_feature_27B(name, train_values, test_values):
    train_values = np.asarray(train_values, dtype=np.float32).reshape(-1)
    test_values = np.asarray(test_values, dtype=np.float32).reshape(-1)

    if len(train_values) != n_train:
        raise ValueError(f"{name} train length mismatch")
    if len(test_values) != n_test:
        raise ValueError(f"{name} test length mismatch")

    extra_names_27B.append(name)
    extra_train.append(train_values)
    extra_test.append(test_values)

add_extra_feature_27B("anchor77_residstack_pred", anchor27_oof, anchor27_test)

for name in sorted(component_oof):
    tr = np.clip(component_oof[name], 0, 100)
    te = np.clip(component_test[name], 0, 100)

    add_extra_feature_27B(f"pred_{name}", tr, te)
    add_extra_feature_27B(f"diff_{name}_minus_anchor77", tr - anchor27_oof, te - anchor27_test)

if len(component_oof) >= 2:
    comp_names_sorted = sorted(component_oof)

    comp_train_mat = np.column_stack([
        np.clip(component_oof[n], 0, 100) for n in comp_names_sorted
    ])

    comp_test_mat = np.column_stack([
        np.clip(component_test[n], 0, 100) for n in comp_names_sorted
    ])

    add_extra_feature_27B("component_mean", comp_train_mat.mean(axis=1), comp_test_mat.mean(axis=1))
    add_extra_feature_27B("component_std", comp_train_mat.std(axis=1), comp_test_mat.std(axis=1))
    add_extra_feature_27B("component_min", comp_train_mat.min(axis=1), comp_test_mat.min(axis=1))
    add_extra_feature_27B("component_max", comp_train_mat.max(axis=1), comp_test_mat.max(axis=1))
    add_extra_feature_27B(
        "component_range",
        comp_train_mat.max(axis=1) - comp_train_mat.min(axis=1),
        comp_test_mat.max(axis=1) - comp_test_mat.min(axis=1),
    )

extra_train_mat_27B = np.column_stack(extra_train).astype(np.float32)
extra_test_mat_27B = np.column_stack(extra_test).astype(np.float32)

X_aug_train_27B = sparse.hstack(
    [X_base_train_27B, sparse.csr_matrix(extra_train_mat_27B)],
    format="csr"
).astype(np.float32)

X_aug_test_27B = sparse.hstack(
    [X_base_test_27B, sparse.csr_matrix(extra_test_mat_27B)],
    format="csr"
).astype(np.float32)

# Reset compatibility aliases for 27C
X_aug_train = X_aug_train_27B
X_aug_test = X_aug_test_27B
resid_target_27 = resid_target_27B
extra_names = extra_names_27B
anchor_oof = anchor27_oof
anchor_test = anchor27_test
anchor_oof_mse = anchor27_oof_mse

print("\n27B.1 patch complete.")
print("Verified anchor OOF MSE:", anchor_oof_mse)
print("Residual target std:", float(np.std(resid_target_27)))
print("Extra feature count:", len(extra_names))
print("Augmented train shape:", X_aug_train.shape)
print("Augmented test shape:", X_aug_test.shape)

gc.collect()

RF test numeric columns:
['rf_500_base_fold1_pred', 'rf_500_base_fold2_pred', 'rf_500_base_fold3_pred', 'rf_500_base_fold4_pred', 'rf_500_base_fold5_pred', 'rf_500_base_foldavg_pred']

RF test fixed using: existing average column: rf_500_base_foldavg_pred
count    48307.000000
mean        54.056767
std         22.817699
min          0.322400
1%           9.781608
5%          19.065240
50%         52.255200
95%         92.641520
99%         98.827408
max         99.995200
dtype: float64

27B.1 patch complete.
Verified anchor OOF MSE: 87.13604874858268
Residual target std: 9.33466911315918
Extra feature count: 18
Augmented train shape: (144921, 180)
Augmented test shape: (48307, 180)


0

# This above modelling direction has been stopped

In [59]:
# ============================================================
# 28A. New branch diagnostic:
# bounded/proportion target + hierarchical key feasibility
# ============================================================
#
# Purpose:
# - Do not train a model yet.
# - Check whether PERCENT_PROFICIENT behaves like a bounded proportion.
# - Inspect N_STUDENTS as a reliability/noise proxy.
# - Confirm which raw grouping columns are available.
# - Measure train/test coverage and repetition for candidate hierarchical keys.
#
# This sets up the next branch:
# cross-fitted statistical/target-encoding features + bounded target modeling.
# ============================================================

import numpy as np
import pandas as pd

TARGET_COL = "PERCENT_PROFICIENT"
ID_COL = "ASSESSMENT_ID"

# ------------------------------------------------------------
# 1. Locate raw merged train/test frames
# ------------------------------------------------------------

if "train_full" in globals() and "test_full" in globals():
    diag_train = train_full.copy()
    diag_test = test_full.copy()
    source_used = "existing train_full/test_full"

elif all(name in globals() for name in [
    "scores_training", "scores_test", "school_covariates", "district_covariates"
]):
    diag_train = scores_training.merge(
        school_covariates, on="SCHOOL", how="left", validate="many_to_one"
    )
    diag_test = scores_test.merge(
        school_covariates, on="SCHOOL", how="left", validate="many_to_one"
    )

    diag_train = diag_train.merge(
        district_covariates, on="DISTRICT", how="left", validate="many_to_one"
    )
    diag_test = diag_test.merge(
        district_covariates, on="DISTRICT", how="left", validate="many_to_one"
    )

    source_used = "rebuilt from scores + school/district covariates"

else:
    raise ValueError(
        "Could not find train_full/test_full or the raw score/covariate tables. "
        "Please rerun the raw loading and merge cells first."
    )

print("Source used:", source_used)
print("diag_train shape:", diag_train.shape)
print("diag_test shape: ", diag_test.shape)

# ------------------------------------------------------------
# 2. Basic alignment checks
# ------------------------------------------------------------

print("\n--- Alignment checks ---")

if "y_train" in globals():
    print("len(y_train):", len(y_train))
    print("diag_train rows match y_train:", len(diag_train) == len(y_train))
else:
    print("y_train not found in globals; using target column from diag_train.")

if "X_train_proc_model" in globals():
    print("X_train_proc_model rows:", X_train_proc_model.shape[0])
    print("diag_train rows match X_train_proc_model:", len(diag_train) == X_train_proc_model.shape[0])

if "X_test_proc_model" in globals():
    print("X_test_proc_model rows:", X_test_proc_model.shape[0])
    print("diag_test rows match X_test_proc_model:", len(diag_test) == X_test_proc_model.shape[0])

if "test_ids" in globals():
    print("len(test_ids):", len(test_ids))
    print("diag_test rows match test_ids:", len(diag_test) == len(test_ids))

# ------------------------------------------------------------
# 3. Raw column availability
# ------------------------------------------------------------

important_cols = [
    ID_COL,
    "SCHOOL",
    "DISTRICT",
    "COUNTY",
    "REGION",
    "DISTRICT_TYPE",
    "SUBGROUP_NAME",
    "ASSESSMENT_NAME",
    "N_STUDENTS",
    TARGET_COL,
]

availability = []
for col in important_cols:
    availability.append({
        "column": col,
        "in_train": col in diag_train.columns,
        "in_test": col in diag_test.columns,
        "train_missing_pct": (
            float(diag_train[col].isna().mean() * 100) if col in diag_train.columns else np.nan
        ),
        "test_missing_pct": (
            float(diag_test[col].isna().mean() * 100) if col in diag_test.columns else np.nan
        ),
        "train_unique": (
            int(diag_train[col].nunique(dropna=False)) if col in diag_train.columns else np.nan
        ),
        "test_unique": (
            int(diag_test[col].nunique(dropna=False)) if col in diag_test.columns else np.nan
        ),
    })

availability_df = pd.DataFrame(availability)

print("\n--- Important raw column availability ---")
display(availability_df)

# ------------------------------------------------------------
# 4. Target bounded/proportion diagnostics
# ------------------------------------------------------------

print("\n--- Target diagnostics ---")

if "y_train" in globals():
    y_diag = pd.Series(y_train).astype(float).reset_index(drop=True)
else:
    y_diag = pd.Series(diag_train[TARGET_COL]).astype(float).reset_index(drop=True)

print(y_diag.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

print("\nTarget boundary / discreteness checks:")
print("share target == 0:   ", float((y_diag == 0).mean()))
print("share target == 100: ", float((y_diag == 100).mean()))
print("share target < 0:    ", float((y_diag < 0).mean()))
print("share target > 100:  ", float((y_diag > 100).mean()))
print("share integer-like:  ", float(np.isclose(y_diag, np.round(y_diag), atol=1e-9).mean()))
print("number of unique target values:", int(y_diag.nunique()))

# ------------------------------------------------------------
# 5. N_STUDENTS reliability / rounded-count plausibility
# ------------------------------------------------------------

print("\n--- N_STUDENTS diagnostics ---")

if "N_STUDENTS" in diag_train.columns:
    n_train = pd.Series(diag_train["N_STUDENTS"]).astype(float).reset_index(drop=True)

    print(n_train.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

    valid_n = n_train.notna() & (n_train > 0) & y_diag.notna()
    n_valid = n_train[valid_n].to_numpy(dtype=float)
    y_valid = y_diag[valid_n].to_numpy(dtype=float)

    # Check whether the reported percent is compatible with an integer number proficient,
    # allowing for whole-percent reporting.
    k_hat = np.rint((y_valid / 100.0) * n_valid)
    pct_from_count = 100.0 * k_hat / n_valid
    pct_from_count_rounded = np.rint(pct_from_count)

    exact_count_diff = np.abs(y_valid - pct_from_count)
    rounded_count_diff = np.abs(y_valid - pct_from_count_rounded)

    print("\nRounded-count plausibility:")
    print("valid rows checked:", int(valid_n.sum()))
    print("median abs diff vs nearest exact count percent:", float(np.median(exact_count_diff)))
    print("share compatible with rounded whole-percent count:", float((rounded_count_diff <= 1e-9).mean()))
    print("share within <= 0.5 percentage point of nearest count percent:", float((exact_count_diff <= 0.5).mean()))
    print("share within <= 1.0 percentage point of nearest count percent:", float((exact_count_diff <= 1.0).mean()))

    if "N_STUDENTS" in diag_test.columns:
        n_test = pd.Series(diag_test["N_STUDENTS"]).astype(float)
        print("\nTest N_STUDENTS summary:")
        print(n_test.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

else:
    print("N_STUDENTS not found. Bounded target modeling is still possible, but reliability-aware diagnostics are limited.")

# ------------------------------------------------------------
# 6. Candidate hierarchical key coverage
# ------------------------------------------------------------

print("\n--- Candidate hierarchical key coverage ---")

def make_key(df, cols):
    cols = tuple(cols)
    if len(cols) == 1:
        return df[cols[0]].astype("string").fillna("<NA>").astype(str)
    tmp = df.loc[:, list(cols)].astype("string").fillna("<NA>").astype(str)
    return tmp.agg(" || ".join, axis=1)

candidate_keys = [
    ("SCHOOL",),
    ("DISTRICT",),
    ("COUNTY",),
    ("REGION",),
    ("DISTRICT_TYPE",),
    ("ASSESSMENT_NAME",),
    ("SUBGROUP_NAME",),
    ("SCHOOL", "ASSESSMENT_NAME"),
    ("SCHOOL", "SUBGROUP_NAME"),
    ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME"),
    ("DISTRICT", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME"),
    ("COUNTY", "SUBGROUP_NAME"),
    ("REGION", "ASSESSMENT_NAME"),
    ("DISTRICT_TYPE", "ASSESSMENT_NAME"),
    ("SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
]

if "N_STUDENTS" in diag_train.columns and "N_STUDENTS" in diag_test.columns:
    # Small, stable bins for interaction diagnostics only.
    train_bins = pd.cut(
        pd.Series(diag_train["N_STUDENTS"]).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"]
    ).astype("string").fillna("<NA>")

    test_bins = pd.cut(
        pd.Series(diag_test["N_STUDENTS"]).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"]
    ).astype("string").fillna("<NA>")

    diag_train["_N_STUDENTS_BIN_DIAG"] = train_bins
    diag_test["_N_STUDENTS_BIN_DIAG"] = test_bins

    candidate_keys += [
        ("ASSESSMENT_NAME", "_N_STUDENTS_BIN_DIAG"),
        ("SUBGROUP_NAME", "_N_STUDENTS_BIN_DIAG"),
        ("ASSESSMENT_NAME", "SUBGROUP_NAME", "_N_STUDENTS_BIN_DIAG"),
    ]

coverage_rows = []

for cols in candidate_keys:
    cols = tuple(cols)

    if not all(c in diag_train.columns for c in cols):
        continue
    if not all(c in diag_test.columns for c in cols):
        continue

    tr_key = make_key(diag_train, cols)
    te_key = make_key(diag_test, cols)

    vc = tr_key.value_counts(dropna=False)
    seen_set = set(vc.index)

    test_seen = te_key.isin(seen_set)

    coverage_rows.append({
        "key": " x ".join(cols).replace("_N_STUDENTS_BIN_DIAG", "N_STUDENTS_BIN"),
        "train_groups": int(vc.shape[0]),
        "test_groups": int(te_key.nunique(dropna=False)),
        "test_row_coverage": float(test_seen.mean()),
        "median_train_count": float(vc.median()),
        "mean_train_count": float(vc.mean()),
        "p90_train_count": float(vc.quantile(0.90)),
        "p99_train_count": float(vc.quantile(0.99)),
        "singleton_group_share": float((vc == 1).mean()),
        "max_train_count": int(vc.max()),
    })

coverage_df = pd.DataFrame(coverage_rows)

if len(coverage_df) > 0:
    coverage_df = coverage_df.sort_values(
        ["test_row_coverage", "median_train_count", "train_groups"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    display(coverage_df)

    recommended_start = coverage_df[
        (coverage_df["test_row_coverage"] >= 0.80) &
        (coverage_df["median_train_count"] >= 2)
    ].copy()

    print("\nKeys that look initially usable for cross-fitted statistical features:")
    if len(recommended_start) == 0:
        print("No keys passed the simple initial screen. We may need lower thresholds or fallback-only encodings.")
    else:
        display(recommended_start[[
            "key",
            "test_row_coverage",
            "train_groups",
            "median_train_count",
            "singleton_group_share",
            "p90_train_count",
            "p99_train_count"
        ]])

else:
    print("No candidate keys could be evaluated.")

# Save diagnostics for later cells.
encoding_key_diagnostics = coverage_df
raw_column_availability_diagnostics = availability_df

print("\nDone. Next step will depend on these diagnostics.")

Source used: existing train_full/test_full
diag_train shape: (144921, 62)
diag_test shape:  (48307, 61)

--- Alignment checks ---
len(y_train): 144921
diag_train rows match y_train: True
X_train_proc_model rows: 144921
diag_train rows match X_train_proc_model: True
X_test_proc_model rows: 48307
diag_test rows match X_test_proc_model: True
len(test_ids): 48307
diag_test rows match test_ids: True

--- Important raw column availability ---


,column,in_train,in_test,train_missing_pct,test_missing_pct,train_unique,test_unique
0,ASSESSMENT_ID,True,True,0.0,0.0,144921,48307.0
1,SCHOOL,True,True,0.0,0.0,4469,4448.0
2,DISTRICT,True,True,0.0,0.0,710,707.0
3,COUNTY,True,True,0.0,0.0,62,62.0
4,REGION,True,True,0.0,0.0,10,10.0
5,DISTRICT_TYPE,True,True,0.0,0.0,7,7.0
6,SUBGROUP_NAME,True,True,0.0,0.0,5,5.0
7,ASSESSMENT_NAME,True,True,0.0,0.0,32,32.0
8,N_STUDENTS,True,True,0.0,0.0,736,594.0
9,PERCENT_PROFICIENT,True,False,0.0,NaN,101,NaN



--- Target diagnostics ---
count    144921.000000
mean         54.183521
std          26.429925
min           0.000000
1%            0.000000
5%           13.000000
25%          33.000000
50%          53.000000
75%          76.000000
95%          98.000000
99%         100.000000
max         100.000000
Name: PERCENT_PROFICIENT, dtype: float64

Target boundary / discreteness checks:
share target == 0:    0.01107499948247666
share target == 100:  0.04582496670599844
share target < 0:     0.0
share target > 100:   0.0
share integer-like:   1.0
number of unique target values: 101

--- N_STUDENTS diagnostics ---
count    144921.000000
mean         57.664748
std          67.211682
min           5.000000
1%            5.000000
5%            8.000000
25%          21.000000
50%          38.000000
75%          69.000000
95%         173.000000
99%         336.000000
max        1683.000000
Name: N_STUDENTS, dtype: float64

Rounded-count plausibility:
valid rows checked: 144921
median abs diff vs n

,key,train_groups,test_groups,test_row_coverage,median_train_count,mean_train_count,p90_train_count,p99_train_count,singleton_group_share,max_train_count
0,SUBGROUP_NAME,5,5,1.000000,29110.0,28984.200000,33771.8,36417.08,0.000000,36711
1,DISTRICT_TYPE,7,7,1.000000,13986.0,20703.000000,42308.8,43986.58,0.000000,44173
2,REGION,10,10,1.000000,9424.0,14492.100000,21815.9,51955.19,0.000000,55304
3,ASSESSMENT_NAME,32,32,1.000000,4532.0,4528.781250,8296.4,8389.21,0.000000,8392
4,SUBGROUP_NAME x N_STUDENTS_BIN,30,30,1.000000,3874.5,4830.700000,10516.9,13959.36,0.000000,13964
5,ASSESSMENT_NAME x SUBGROUP_NAME,132,132,1.000000,1004.5,1097.886364,1771.3,1837.76,0.000000,1883
6,COUNTY,62,62,1.000000,958.0,2337.435484,7086.5,14432.88,0.000000,17112
7,ASSESSMENT_NAME x N_STUDENTS_BIN,191,188,1.000000,470.0,758.748691,1786.0,3972.70,0.010471,3985
8,DISTRICT_TYPE x ASSESSMENT_NAME,222,221,1.000000,438.5,652.797297,1523.2,2507.06,0.009009,2545
9,REGION x ASSESSMENT_NAME,315,314,1.000000,295.0,460.066667,960.8,3118.04,0.000000,3140



Keys that look initially usable for cross-fitted statistical features:


,key,test_row_coverage,train_groups,median_train_count,singleton_group_share,p90_train_count,p99_train_count
0,SUBGROUP_NAME,1.000000,5,29110.0,0.000000,33771.8,36417.08
1,DISTRICT_TYPE,1.000000,7,13986.0,0.000000,42308.8,43986.58
2,REGION,1.000000,10,9424.0,0.000000,21815.9,51955.19
3,ASSESSMENT_NAME,1.000000,32,4532.0,0.000000,8296.4,8389.21
4,SUBGROUP_NAME x N_STUDENTS_BIN,1.000000,30,3874.5,0.000000,10516.9,13959.36
5,ASSESSMENT_NAME x SUBGROUP_NAME,1.000000,132,1004.5,0.000000,1771.3,1837.76
6,COUNTY,1.000000,62,958.0,0.000000,7086.5,14432.88
7,ASSESSMENT_NAME x N_STUDENTS_BIN,1.000000,191,470.0,0.010471,1786.0,3972.70
8,DISTRICT_TYPE x ASSESSMENT_NAME,1.000000,222,438.5,0.009009,1523.2,2507.06
9,REGION x ASSESSMENT_NAME,1.000000,315,295.0,0.000000,960.8,3118.04



Done. Next step will depend on these diagnostics.


In [60]:
# ============================================================
# 28B. Leakage-safe target/statistical encoding branch:
# holdout construction diagnostic
# ============================================================
#
# What this does:
# - Creates a fresh train/validation screen split by row index.
# - Builds target/statistical encoding features using ONLY the screen-training rows.
# - For screen-training rows, encodings are internally OOF/cross-fitted.
# - For screen-validation rows, encodings are mapped from screen-training only.
# - Adds both row-wise group mean and N_STUDENTS-weighted rounded-count group mean.
# - Creates augmented matrices for the next LightGBM holdout screen.
#
# This cell does NOT train a model.
# ============================================================

import os
import gc
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from scipy import sparse

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = "PERCENT_PROFICIENT"

required_objects = [
    "train_full",
    "test_full",
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
]

missing_required = [name for name in required_objects if name not in globals()]
if missing_required:
    raise ValueError(f"Missing required objects: {missing_required}")

raw_train_te = train_full.reset_index(drop=True).copy()
raw_test_te = test_full.reset_index(drop=True).copy()
y_arr_te = np.asarray(y_train, dtype=np.float32).reshape(-1)

if len(raw_train_te) != len(y_arr_te):
    raise ValueError("train_full and y_train are not aligned.")
if X_train_proc_model.shape[0] != len(y_arr_te):
    raise ValueError("X_train_proc_model and y_train are not aligned.")
if X_test_proc_model.shape[0] != len(raw_test_te):
    raise ValueError("X_test_proc_model and test_full are not aligned.")

def add_n_students_bin(df):
    out = df.copy()
    out["_N_STUDENTS_BIN_TE"] = pd.cut(
        pd.Series(out["N_STUDENTS"]).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"]
    ).astype("string").fillna("<NA>")
    return out

raw_train_te = add_n_students_bin(raw_train_te)
raw_test_te = add_n_students_bin(raw_test_te)

candidate_key_specs = [
    # cols, alpha for row-wise mean, alpha_n for student-count-weighted mean
    (("ASSESSMENT_NAME",), 20.0, 200.0),
    (("SUBGROUP_NAME",), 20.0, 200.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME"), 20.0, 200.0),
    (("ASSESSMENT_NAME", "N_STUDENTS_BIN_TE"), 30.0, 250.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME", "_N_STUDENTS_BIN_TE"), 50.0, 300.0),

    (("COUNTY",), 30.0, 250.0),
    (("COUNTY", "SUBGROUP_NAME"), 40.0, 300.0),
    (("COUNTY", "ASSESSMENT_NAME"), 60.0, 400.0),
    (("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"), 100.0, 600.0),

    (("REGION", "ASSESSMENT_NAME"), 40.0, 300.0),
    (("DISTRICT_TYPE", "ASSESSMENT_NAME"), 40.0, 300.0),

    (("DISTRICT",), 60.0, 400.0),
    (("DISTRICT", "SUBGROUP_NAME"), 90.0, 500.0),
    (("DISTRICT", "ASSESSMENT_NAME"), 140.0, 700.0),

    (("SCHOOL",), 100.0, 600.0),
    (("SCHOOL", "SUBGROUP_NAME"), 140.0, 800.0),
    (("SCHOOL", "ASSESSMENT_NAME"), 220.0, 1000.0),
]

# Fix accidental name mismatch in one candidate.
candidate_key_specs = [
    (tuple("_N_STUDENTS_BIN_TE" if c == "N_STUDENTS_BIN_TE" else c for c in cols), alpha, alpha_n)
    for cols, alpha, alpha_n in candidate_key_specs
]

te_key_specs = []
for cols, alpha, alpha_n in candidate_key_specs:
    if all(c in raw_train_te.columns for c in cols) and all(c in raw_test_te.columns for c in cols):
        te_key_specs.append((cols, alpha, alpha_n))

if len(te_key_specs) == 0:
    raise ValueError("No usable target-encoding key specs found.")

def safe_key_name(cols):
    name = "__".join(cols)
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def make_group_key(df, cols):
    cols = tuple(cols)
    if len(cols) == 1:
        return df[cols[0]].astype("string").fillna("<NA>").astype(str).reset_index(drop=True)
    return (
        df.loc[:, list(cols)]
        .astype("string")
        .fillna("<NA>")
        .astype(str)
        .agg(" || ".join, axis=1)
        .reset_index(drop=True)
    )

def fit_group_stats(keys, y, n_students, alpha, alpha_n):
    yy = np.asarray(y, dtype=np.float64)
    nn = np.asarray(n_students, dtype=np.float64)

    good_n = np.isfinite(nn) & (nn > 0)
    if not good_n.all():
        fill_n = np.nanmedian(nn[good_n]) if good_n.any() else 1.0
        nn = np.where(good_n, nn, fill_n)

    yy_clip = np.clip(yy, 0.0, 100.0)
    proficient_counts = np.rint((yy_clip / 100.0) * nn)
    proficient_counts = np.clip(proficient_counts, 0.0, nn)

    global_mean = float(np.mean(yy))
    global_std = float(np.std(yy, ddof=0))
    global_wmean = float(100.0 * proficient_counts.sum() / max(nn.sum(), 1.0))

    tmp = pd.DataFrame({
        "key": pd.Series(keys).astype(str).to_numpy(),
        "y": yy,
        "n": nn,
        "k": proficient_counts,
    })

    stats = tmp.groupby("key", sort=False).agg(
        cnt=("y", "size"),
        sum_y=("y", "sum"),
        std_y=("y", "std"),
        sum_n=("n", "sum"),
        sum_k=("k", "sum"),
    )

    stats["mean_s"] = (stats["sum_y"] + alpha * global_mean) / (stats["cnt"] + alpha)
    stats["wmean_s"] = 100.0 * (
        stats["sum_k"] + alpha_n * (global_wmean / 100.0)
    ) / (stats["sum_n"] + alpha_n)

    stats["std_y"] = stats["std_y"].fillna(global_std)
    stats["log_count"] = np.log1p(stats["cnt"].astype(float))
    stats["log_students"] = np.log1p(stats["sum_n"].astype(float))

    defaults = {
        "global_mean": global_mean,
        "global_wmean": global_wmean,
        "global_std": global_std,
    }

    return stats, defaults

def apply_group_stats(keys_apply, stats, defaults, prefix):
    kk = pd.Series(keys_apply).astype(str).reset_index(drop=True)

    out = pd.DataFrame(index=np.arange(len(kk)))
    out[f"{prefix}_mean"] = kk.map(stats["mean_s"]).fillna(defaults["global_mean"]).astype(np.float32)
    out[f"{prefix}_wmean"] = kk.map(stats["wmean_s"]).fillna(defaults["global_wmean"]).astype(np.float32)
    out[f"{prefix}_log_count"] = kk.map(stats["log_count"]).fillna(0.0).astype(np.float32)
    out[f"{prefix}_std"] = kk.map(stats["std_y"]).fillna(defaults["global_std"]).astype(np.float32)

    return out

def build_te_oof_and_apply(raw_fit, y_fit, raw_apply, key_specs, n_splits=5, random_state=9890):
    raw_fit = raw_fit.reset_index(drop=True)
    raw_apply = raw_apply.reset_index(drop=True)
    y_fit = np.asarray(y_fit, dtype=np.float32).reshape(-1)

    n_fit = pd.Series(raw_fit["N_STUDENTS"]).astype(float).to_numpy()
    n_rows_fit = len(raw_fit)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_parts = []
    apply_parts = []
    summary_rows = []

    for cols, alpha, alpha_n in key_specs:
        cols = tuple(cols)
        prefix = f"te_{safe_key_name(cols)}_a{int(alpha)}_n{int(alpha_n)}"

        key_fit_all = make_group_key(raw_fit, cols)
        key_apply_all = make_group_key(raw_apply, cols)

        feature_cols = [
            f"{prefix}_mean",
            f"{prefix}_wmean",
            f"{prefix}_log_count",
            f"{prefix}_std",
        ]

        oof_arr = np.zeros((n_rows_fit, len(feature_cols)), dtype=np.float32)

        for inner_tr_idx, inner_va_idx in kf.split(np.arange(n_rows_fit)):
            stats_fold, defaults_fold = fit_group_stats(
                key_fit_all.iloc[inner_tr_idx],
                y_fit[inner_tr_idx],
                n_fit[inner_tr_idx],
                alpha=alpha,
                alpha_n=alpha_n,
            )

            enc_va = apply_group_stats(
                key_fit_all.iloc[inner_va_idx],
                stats_fold,
                defaults_fold,
                prefix,
            )

            oof_arr[inner_va_idx, :] = enc_va[feature_cols].to_numpy(dtype=np.float32)

        stats_full, defaults_full = fit_group_stats(
            key_fit_all,
            y_fit,
            n_fit,
            alpha=alpha,
            alpha_n=alpha_n,
        )

        apply_df = apply_group_stats(
            key_apply_all,
            stats_full,
            defaults_full,
            prefix,
        )

        oof_parts.append(pd.DataFrame(oof_arr, columns=feature_cols))
        apply_parts.append(apply_df[feature_cols])

        fit_counts = key_fit_all.value_counts(dropna=False)
        apply_seen = key_apply_all.isin(set(fit_counts.index))

        summary_rows.append({
            "key": " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
            "alpha": alpha,
            "alpha_n": alpha_n,
            "fit_groups": int(fit_counts.shape[0]),
            "apply_groups": int(key_apply_all.nunique(dropna=False)),
            "apply_row_coverage": float(apply_seen.mean()),
            "median_fit_count": float(fit_counts.median()),
            "singleton_share": float((fit_counts == 1).mean()),
            "features_added": len(feature_cols),
        })

    oof_features = pd.concat(oof_parts, axis=1)
    apply_features = pd.concat(apply_parts, axis=1)
    summary = pd.DataFrame(summary_rows)

    return oof_features, apply_features, summary

def take_rows(X, idx):
    if sparse.issparse(X):
        return X[idx]
    if isinstance(X, pd.DataFrame):
        return X.iloc[idx]
    return X[idx]

def to_float32_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32).tocsr()
    if isinstance(X, pd.DataFrame):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def append_features(X_base, add_df):
    add = add_df.to_numpy(dtype=np.float32)
    if sparse.issparse(X_base):
        return sparse.hstack([X_base, sparse.csr_matrix(add)], format="csr")
    return np.hstack([X_base, add])

# Fresh screen split.
all_idx_te = np.arange(len(y_arr_te))
tr_idx_te_screen, val_idx_te_screen = train_test_split(
    all_idx_te,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True,
)

raw_fit_screen = raw_train_te.iloc[tr_idx_te_screen].reset_index(drop=True)
raw_val_screen = raw_train_te.iloc[val_idx_te_screen].reset_index(drop=True)
y_fit_screen = y_arr_te[tr_idx_te_screen]
y_val_screen = y_arr_te[val_idx_te_screen]

te_tr_screen, te_val_screen, te_screen_key_summary = build_te_oof_and_apply(
    raw_fit=raw_fit_screen,
    y_fit=y_fit_screen,
    raw_apply=raw_val_screen,
    key_specs=te_key_specs,
    n_splits=5,
    random_state=RANDOM_STATE,
)

X_tr_base_screen = to_float32_matrix(take_rows(X_train_proc_model, tr_idx_te_screen))
X_val_base_screen = to_float32_matrix(take_rows(X_train_proc_model, val_idx_te_screen))

X_tr_te_screen = append_features(X_tr_base_screen, te_tr_screen)
X_val_te_screen = append_features(X_val_base_screen, te_val_screen)

finite_ok = (
    np.isfinite(te_tr_screen.to_numpy(dtype=np.float32)).all()
    and np.isfinite(te_val_screen.to_numpy(dtype=np.float32)).all()
)

shape_ok = (
    te_tr_screen.shape[0] == len(tr_idx_te_screen)
    and te_val_screen.shape[0] == len(val_idx_te_screen)
    and X_tr_te_screen.shape[0] == len(tr_idx_te_screen)
    and X_val_te_screen.shape[0] == len(val_idx_te_screen)
)

feature_count_ok = te_tr_screen.shape[1] > 0
column_ok = te_tr_screen.columns.is_unique and te_val_screen.columns.is_unique

TE_DIAGNOSTIC_PASS = bool(finite_ok and shape_ok and feature_count_ok and column_ok)

print("TE_DIAGNOSTIC_PASS =", TE_DIAGNOSTIC_PASS)
print("Screen train rows:", len(tr_idx_te_screen))
print("Screen valid rows:", len(val_idx_te_screen))
print("Base screen shapes:", X_tr_base_screen.shape, X_val_base_screen.shape)
print("Augmented TE screen shapes:", X_tr_te_screen.shape, X_val_te_screen.shape)
print("TE feature count:", te_tr_screen.shape[1])
print("Finite features:", finite_ok)
print("Unique TE columns:", column_ok)

print("\nTarget-encoding key summary:")
display(te_screen_key_summary.sort_values(
    ["apply_row_coverage", "median_fit_count"],
    ascending=[False, False]
).reset_index(drop=True))

print("\nTE train feature summary, first 10 columns:")
display(te_tr_screen.iloc[:, :10].describe().T)

gc.collect()

TE_DIAGNOSTIC_PASS = True
Screen train rows: 115936
Screen valid rows: 28985
Base screen shapes: (115936, 162) (28985, 162)
Augmented TE screen shapes: (115936, 230) (28985, 230)
TE feature count: 68
Finite features: True
Unique TE columns: True

Target-encoding key summary:


,key,alpha,alpha_n,fit_groups,apply_groups,apply_row_coverage,median_fit_count,singleton_share,features_added
0,SUBGROUP_NAME,20.0,200.0,5,5,1.000000,23233.0,0.000000,4
1,ASSESSMENT_NAME,20.0,200.0,32,32,1.000000,3601.0,0.000000,4
2,ASSESSMENT_NAME x SUBGROUP_NAME,20.0,200.0,132,132,1.000000,801.0,0.000000,4
3,COUNTY,30.0,250.0,62,62,1.000000,756.5,0.000000,4
4,DISTRICT_TYPE x ASSESSMENT_NAME,40.0,300.0,222,220,1.000000,354.5,0.009009,4
5,REGION x ASSESSMENT_NAME,40.0,300.0,315,314,1.000000,235.0,0.003175,4
6,COUNTY x SUBGROUP_NAME,40.0,300.0,310,309,1.000000,155.5,0.000000,4
7,DISTRICT,60.0,400.0,710,709,1.000000,78.0,0.000000,4
8,ASSESSMENT_NAME x N_STUDENTS_BIN,30.0,250.0,190,190,0.999965,372.5,0.005263,4
9,ASSESSMENT_NAME x SUBGROUP_NAME x N_STUDENTS_BIN,50.0,300.0,788,761,0.999965,109.0,0.013959,4



TE train feature summary, first 10 columns:


,count,mean,std,min,25%,50%,75%,max
te_ASSESSMENT_NAME_a20_n200_mean,115936.0,54.177391,12.351819,34.510574,43.745338,52.818825,65.268974,86.532150
te_ASSESSMENT_NAME_a20_n200_wmean,115936.0,54.782864,12.578279,31.266104,44.283440,53.665527,62.845219,84.903481
te_ASSESSMENT_NAME_a20_n200_log_count,115936.0,8.170941,0.494378,2.833213,7.962764,8.215007,8.561975,8.594524
te_ASSESSMENT_NAME_a20_n200_std,115936.0,23.093555,2.983797,2.500000,20.966339,22.582537,24.001675,30.780113
te_SUBGROUP_NAME_a20_n200_mean,115936.0,54.198841,4.719690,47.752342,52.528522,53.656490,54.151058,63.541710
te_SUBGROUP_NAME_a20_n200_wmean,115936.0,58.028111,7.377084,47.665009,55.763378,57.394348,57.541130,72.618271
te_SUBGROUP_NAME_a20_n200_log_count,115936.0,9.839026,0.148617,9.657715,9.686512,9.831508,10.059722,10.066923
te_SUBGROUP_NAME_a20_n200_std,115936.0,25.977516,0.553061,24.855640,25.627665,26.247187,26.384045,26.495983
te_ASSESSMENT_NAME_SUBGROUP_NAME_a20_n200_mean,115936.0,54.151497,13.043442,28.668512,43.414429,53.396889,64.705765,86.532150
te_ASSESSMENT_NAME_SUBGROUP_NAME_a20_n200_wmean,115936.0,55.334736,14.385706,26.434465,44.359715,53.848438,64.508759,90.540169


0

In [61]:
# ============================================================
# 28C. Holdout screen:
# base LightGBM vs base + leakage-safe target/stat encodings
# ============================================================
#
# What this does:
# - Trains one base LightGBM on the screen split.
# - Trains one augmented LightGBM on the same split using the TE features from 28B.
# - Compares validation MSE directly.
#
# This cell does NOT create a Kaggle submission.
# ============================================================

import os
import time
import gc
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

if not globals().get("TE_DIAGNOSTIC_PASS", False):
    raise ValueError("28B did not pass. Do not run 28C until TE_DIAGNOSTIC_PASS is True.")

os.makedirs("model_results", exist_ok=True)

def fit_lgbm_holdout_screen(X_tr_mat, y_tr_vec, X_val_mat, y_val_vec, label):
    params = {
        "objective": "regression",
        "metric": "l2",
        "random_state": RANDOM_STATE,
        "n_jobs": 1,
        "verbosity": -1,
        "force_col_wise": True,

        # Same general family as the successful long LightGBM,
        # but shorter for a screen.
        "n_estimators": 20000,
        "learning_rate": 0.03,
        "num_leaves": 95,
        "min_child_samples": 60,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "max_depth": -1,
    }

    model = lgb.LGBMRegressor(**params)

    t0 = time.time()
    model.fit(
        X_tr_mat,
        y_tr_vec,
        eval_set=[(X_val_mat, y_val_vec)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=1000, verbose=False),
            lgb.log_evaluation(period=1000),
        ],
    )
    elapsed = time.time() - t0

    best_iter = int(model.best_iteration_ or params["n_estimators"])

    pred_tr = np.clip(
        model.predict(X_tr_mat, num_iteration=best_iter),
        0,
        100,
    )
    pred_val = np.clip(
        model.predict(X_val_mat, num_iteration=best_iter),
        0,
        100,
    )

    result = {
        "label": label,
        "train_mse_clipped": float(mean_squared_error(y_tr_vec, pred_tr)),
        "valid_mse_clipped": float(mean_squared_error(y_val_vec, pred_val)),
        "best_iteration": best_iter,
        "elapsed_seconds": float(elapsed),
        "n_features": int(X_tr_mat.shape[1]),
        "pred_valid_mean": float(np.mean(pred_val)),
        "pred_valid_std": float(np.std(pred_val)),
        "pred_valid_min": float(np.min(pred_val)),
        "pred_valid_max": float(np.max(pred_val)),
    }

    return model, result, pred_val

screen_results = []

print("Training base LightGBM screen model...")
base_model_te_screen, base_result_te_screen, pred_val_base_te_screen = fit_lgbm_holdout_screen(
    X_tr_base_screen,
    y_fit_screen,
    X_val_base_screen,
    y_val_screen,
    label="base_lgbm_screen",
)
screen_results.append(base_result_te_screen)

gc.collect()

print("\nTraining base + target/stat encoding LightGBM screen model...")
aug_model_te_screen, aug_result_te_screen, pred_val_aug_te_screen = fit_lgbm_holdout_screen(
    X_tr_te_screen,
    y_fit_screen,
    X_val_te_screen,
    y_val_screen,
    label="base_plus_te_lgbm_screen",
)
screen_results.append(aug_result_te_screen)

lgbm_te_holdout_results = pd.DataFrame(screen_results)

base_mse = float(lgbm_te_holdout_results.loc[
    lgbm_te_holdout_results["label"] == "base_lgbm_screen",
    "valid_mse_clipped"
].iloc[0])

aug_mse = float(lgbm_te_holdout_results.loc[
    lgbm_te_holdout_results["label"] == "base_plus_te_lgbm_screen",
    "valid_mse_clipped"
].iloc[0])

te_holdout_gain = base_mse - aug_mse
TE_HOLDOUT_PASS = bool(te_holdout_gain >= 1.0)

lgbm_te_holdout_results["valid_gain_vs_base"] = base_mse - lgbm_te_holdout_results["valid_mse_clipped"]

print("\nHoldout screen results:")
display(lgbm_te_holdout_results)

print("\nBase valid MSE:       ", base_mse)
print("Base + TE valid MSE:  ", aug_mse)
print("TE holdout gain:      ", te_holdout_gain)
print("TE_HOLDOUT_PASS =     ", TE_HOLDOUT_PASS)

lgbm_te_holdout_results.to_csv(
    "model_results/lgbm_te_holdout_screen_results.csv",
    index=False,
)

# Keep predictions/results, but release heavy model objects unless you want feature importance later.
del base_model_te_screen, aug_model_te_screen
gc.collect()

Training base LightGBM screen model...
[1000]	valid_0's l2: 144.669
[2000]	valid_0's l2: 129.649
[3000]	valid_0's l2: 121.526
[4000]	valid_0's l2: 115.833
[5000]	valid_0's l2: 112.05
[6000]	valid_0's l2: 109.168
[7000]	valid_0's l2: 106.872
[8000]	valid_0's l2: 105.021
[9000]	valid_0's l2: 103.557
[10000]	valid_0's l2: 102.335
[11000]	valid_0's l2: 101.414
[12000]	valid_0's l2: 100.64
[13000]	valid_0's l2: 99.9944
[14000]	valid_0's l2: 99.4335
[15000]	valid_0's l2: 99.0222
[16000]	valid_0's l2: 98.6226
[17000]	valid_0's l2: 98.3013
[18000]	valid_0's l2: 98.0536
[19000]	valid_0's l2: 97.8065
[20000]	valid_0's l2: 97.6178

Training base + target/stat encoding LightGBM screen model...
[1000]	valid_0's l2: 90.631
[2000]	valid_0's l2: 89.6559
[3000]	valid_0's l2: 89.2743
[4000]	valid_0's l2: 89.1099
[5000]	valid_0's l2: 89.1189

Holdout screen results:


,label,train_mse_clipped,valid_mse_clipped,best_iteration,elapsed_seconds,n_features,pred_valid_mean,pred_valid_std,pred_valid_min,pred_valid_max,valid_gain_vs_base
0,base_lgbm_screen,9.283496,96.849902,19992,354.543891,162,54.109896,24.564745,0.0,100.0,0.00000
1,base_plus_te_lgbm_screen,12.467271,88.808052,4868,157.827635,230,55.502333,25.017372,0.0,100.0,8.04185



Base valid MSE:        96.84990207455618
Base + TE valid MSE:   88.80805200212511
TE holdout gain:       8.041850072431075
TE_HOLDOUT_PASS =      True


11834

In [62]:
# ============================================================
# 28D. Preflight for full OOF target/stat-encoding LightGBM
# ============================================================
#
# Purpose:
# - Confirm the 28B/28C objects are still available.
# - Confirm row alignment.
# - Confirm the full OOF run will add the expected TE features.
# - Estimate the model artifact names before launching the heavy cell.
#
# Run 28E only if TE_OOF_PREFLIGHT_PASS = True.
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy import sparse

os.makedirs("model_results", exist_ok=True)

required_28d = [
    "TE_HOLDOUT_PASS",
    "raw_train_te",
    "raw_test_te",
    "te_key_specs",
    "build_te_oof_and_apply",
    "fit_group_stats",
    "apply_group_stats",
    "make_group_key",
    "safe_key_name",
    "append_features",
    "to_float32_matrix",
    "take_rows",
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
]

missing_28d = [name for name in required_28d if name not in globals()]

alignment_ok_28d = True
alignment_notes_28d = []

if len(np.asarray(y_train).reshape(-1)) != raw_train_te.shape[0]:
    alignment_ok_28d = False
    alignment_notes_28d.append("y_train length does not match raw_train_te rows.")

if X_train_proc_model.shape[0] != raw_train_te.shape[0]:
    alignment_ok_28d = False
    alignment_notes_28d.append("X_train_proc_model rows do not match raw_train_te rows.")

if X_test_proc_model.shape[0] != raw_test_te.shape[0]:
    alignment_ok_28d = False
    alignment_notes_28d.append("X_test_proc_model rows do not match raw_test_te rows.")

if len(test_ids) != raw_test_te.shape[0]:
    alignment_ok_28d = False
    alignment_notes_28d.append("test_ids length does not match raw_test_te rows.")

holdout_ok_28d = bool(globals().get("TE_HOLDOUT_PASS", False))
key_ok_28d = ("te_key_specs" in globals()) and (len(te_key_specs) > 0)

expected_te_feature_count_28d = int(len(te_key_specs) * 4) if key_ok_28d else 0
expected_aug_feature_count_28d = int(X_train_proc_model.shape[1] + expected_te_feature_count_28d)

TE_OOF_ARTIFACT_NAME = "lgbm_te_base_5fold_oof_v1"

TE_OOF_PREFLIGHT_PASS = bool(
    len(missing_28d) == 0
    and holdout_ok_28d
    and alignment_ok_28d
    and key_ok_28d
    and expected_te_feature_count_28d > 0
)

print("TE_OOF_PREFLIGHT_PASS =", TE_OOF_PREFLIGHT_PASS)
print("Missing required objects:", missing_28d)
print("TE_HOLDOUT_PASS:", holdout_ok_28d)
print("Alignment OK:", alignment_ok_28d)
print("Alignment notes:", alignment_notes_28d)
print("Number of TE key specs:", len(te_key_specs) if key_ok_28d else 0)
print("Expected TE feature count:", expected_te_feature_count_28d)
print("Base feature count:", X_train_proc_model.shape[1])
print("Expected augmented feature count:", expected_aug_feature_count_28d)
print("Artifact name:", TE_OOF_ARTIFACT_NAME)

print("\nTE keys to be used:")
for cols, alpha, alpha_n in te_key_specs:
    print(" -", " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
          "| alpha =", alpha, "| alpha_n =", alpha_n)

TE_OOF_PREFLIGHT_PASS = True
Missing required objects: []
TE_HOLDOUT_PASS: True
Alignment OK: True
Alignment notes: []
Number of TE key specs: 17
Expected TE feature count: 68
Base feature count: 162
Expected augmented feature count: 230
Artifact name: lgbm_te_base_5fold_oof_v1

TE keys to be used:
 - ASSESSMENT_NAME | alpha = 20.0 | alpha_n = 200.0
 - SUBGROUP_NAME | alpha = 20.0 | alpha_n = 200.0
 - ASSESSMENT_NAME x SUBGROUP_NAME | alpha = 20.0 | alpha_n = 200.0
 - ASSESSMENT_NAME x N_STUDENTS_BIN | alpha = 30.0 | alpha_n = 250.0
 - ASSESSMENT_NAME x SUBGROUP_NAME x N_STUDENTS_BIN | alpha = 50.0 | alpha_n = 300.0
 - COUNTY | alpha = 30.0 | alpha_n = 250.0
 - COUNTY x SUBGROUP_NAME | alpha = 40.0 | alpha_n = 300.0
 - COUNTY x ASSESSMENT_NAME | alpha = 60.0 | alpha_n = 400.0
 - COUNTY x ASSESSMENT_NAME x SUBGROUP_NAME | alpha = 100.0 | alpha_n = 600.0
 - REGION x ASSESSMENT_NAME | alpha = 40.0 | alpha_n = 300.0
 - DISTRICT_TYPE x ASSESSMENT_NAME | alpha = 40.0 | alpha_n = 300.0
 - DIS

In [63]:
# ============================================================
# 28E. Full 5-fold OOF artifact:
# LightGBM on base + leakage-safe target/stat encodings
# ============================================================
#
# What this does:
# - For each outer fold:
#     1. Uses only the outer-training rows to build TE mappings.
#     2. Builds inner-OOF TE features for the outer-training rows.
#     3. Builds validation TE features from outer-training mappings only.
#     4. Builds test TE features from outer-training mappings only.
#     5. Trains LightGBM and saves fold predictions.
# - Saves:
#     model_results/oof_lgbm_te_base_5fold_oof_v1.csv
#     model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv
#     submission_lgbm_te_base_5fold_oof_v1_foldavg.csv
#
# This is a serious OOF artifact, not just a holdout screen.
# ============================================================

import os
import time
import gc
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

if not globals().get("TE_OOF_PREFLIGHT_PASS", False):
    raise ValueError("28D did not pass. Do not run 28E until TE_OOF_PREFLIGHT_PASS is True.")

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
ARTIFACT = globals().get("TE_OOF_ARTIFACT_NAME", "lgbm_te_base_5fold_oof_v1")

os.makedirs("model_results", exist_ok=True)

n_train_28e = X_train_proc_model.shape[0]
n_test_28e = X_test_proc_model.shape[0]
y_arr_28e = np.asarray(y_train, dtype=np.float32).reshape(-1)

metrics_path_28e = f"model_results/{ARTIFACT}_fold_metrics.csv"
oof_raw_path_28e = f"model_results/{ARTIFACT}_oof_raw.npy"
oof_clip_path_28e = f"model_results/{ARTIFACT}_oof_clipped.npy"

oof_csv_path_28e = f"model_results/oof_{ARTIFACT}.csv"
test_csv_path_28e = f"model_results/testpred_{ARTIFACT}_foldavg.csv"
submission_path_28e = f"submission_{ARTIFACT}_foldavg.csv"

def build_te_oof_and_apply_many(raw_fit, y_fit, raw_apply_dict, key_specs, n_splits=5, random_state=9890):
    """
    Build leakage-safe inner-OOF TE features for raw_fit,
    plus mapping-applied TE features for each raw_apply frame.

    raw_fit: rows used to train the outer-fold model
    raw_apply_dict: {"valid": raw_valid, "test": raw_test}
    """
    raw_fit = raw_fit.reset_index(drop=True)
    raw_apply_dict = {
        name: df.reset_index(drop=True)
        for name, df in raw_apply_dict.items()
    }

    y_fit = np.asarray(y_fit, dtype=np.float32).reshape(-1)
    n_fit = pd.Series(raw_fit["N_STUDENTS"]).astype(float).to_numpy()
    n_rows_fit = len(raw_fit)

    kf_inner = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_parts = []
    apply_parts = {name: [] for name in raw_apply_dict.keys()}
    summary_rows = []

    for cols, alpha, alpha_n in key_specs:
        cols = tuple(cols)
        prefix = f"te_{safe_key_name(cols)}_a{int(alpha)}_n{int(alpha_n)}"

        key_fit_all = make_group_key(raw_fit, cols)
        key_apply_all = {
            name: make_group_key(df, cols)
            for name, df in raw_apply_dict.items()
        }

        feature_cols = [
            f"{prefix}_mean",
            f"{prefix}_wmean",
            f"{prefix}_log_count",
            f"{prefix}_std",
        ]

        oof_arr = np.zeros((n_rows_fit, len(feature_cols)), dtype=np.float32)

        for inner_tr_idx, inner_va_idx in kf_inner.split(np.arange(n_rows_fit)):
            stats_fold, defaults_fold = fit_group_stats(
                key_fit_all.iloc[inner_tr_idx],
                y_fit[inner_tr_idx],
                n_fit[inner_tr_idx],
                alpha=alpha,
                alpha_n=alpha_n,
            )

            enc_inner_va = apply_group_stats(
                key_fit_all.iloc[inner_va_idx],
                stats_fold,
                defaults_fold,
                prefix,
            )

            oof_arr[inner_va_idx, :] = enc_inner_va[feature_cols].to_numpy(dtype=np.float32)

        stats_full, defaults_full = fit_group_stats(
            key_fit_all,
            y_fit,
            n_fit,
            alpha=alpha,
            alpha_n=alpha_n,
        )

        oof_parts.append(pd.DataFrame(oof_arr, columns=feature_cols))

        for name, key_apply in key_apply_all.items():
            enc_apply = apply_group_stats(
                key_apply,
                stats_full,
                defaults_full,
                prefix,
            )
            apply_parts[name].append(enc_apply[feature_cols])

        fit_counts = key_fit_all.value_counts(dropna=False)

        row = {
            "key": " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
            "alpha": alpha,
            "alpha_n": alpha_n,
            "fit_groups": int(fit_counts.shape[0]),
            "median_fit_count": float(fit_counts.median()),
            "singleton_share": float((fit_counts == 1).mean()),
            "features_added": len(feature_cols),
        }

        for name, key_apply in key_apply_all.items():
            row[f"{name}_groups"] = int(key_apply.nunique(dropna=False))
            row[f"{name}_row_coverage"] = float(key_apply.isin(set(fit_counts.index)).mean())

        summary_rows.append(row)

    oof_features = pd.concat(oof_parts, axis=1)
    apply_features = {
        name: pd.concat(parts, axis=1)
        for name, parts in apply_parts.items()
    }

    summary = pd.DataFrame(summary_rows)

    return oof_features, apply_features, summary

def fold_test_pred_path_28e(fold_num):
    return f"model_results/{ARTIFACT}_fold{fold_num}_test_pred.npy"

# Set up fold indices once so resume checks are stable.
outer_kf_28e = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_indices_28e = list(outer_kf_28e.split(np.arange(n_train_28e)))

if os.path.exists(oof_raw_path_28e):
    oof_pred_raw_28e = np.load(oof_raw_path_28e)
else:
    oof_pred_raw_28e = np.full(n_train_28e, np.nan, dtype=np.float32)

if os.path.exists(oof_clip_path_28e):
    oof_pred_clip_28e = np.load(oof_clip_path_28e)
else:
    oof_pred_clip_28e = np.full(n_train_28e, np.nan, dtype=np.float32)

if os.path.exists(metrics_path_28e):
    fold_metrics_28e = pd.read_csv(metrics_path_28e)
else:
    fold_metrics_28e = pd.DataFrame()

done_folds_28e = set()
if len(fold_metrics_28e) > 0 and "fold" in fold_metrics_28e.columns:
    done_folds_28e = set(fold_metrics_28e["fold"].astype(int).tolist())

params_28e = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,

    "n_estimators": 20000,
    "learning_rate": 0.03,
    "num_leaves": 95,
    "min_child_samples": 60,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "max_depth": -1,
}

print("Artifact:", ARTIFACT)
print("Rows:", n_train_28e, "train |", n_test_28e, "test")
print("Base features:", X_train_proc_model.shape[1])
print("TE features:", len(te_key_specs) * 4)
print("Augmented features expected:", X_train_proc_model.shape[1] + len(te_key_specs) * 4)
print("Already completed folds:", sorted(done_folds_28e))

for fold_num, (tr_idx, va_idx) in enumerate(fold_indices_28e, start=1):
    fold_test_path = fold_test_pred_path_28e(fold_num)

    if (
        fold_num in done_folds_28e
        and os.path.exists(fold_test_path)
        and np.isfinite(oof_pred_clip_28e[va_idx]).all()
    ):
        print(f"\nFold {fold_num} already complete. Skipping.")
        continue

    print(f"\n========== Fold {fold_num}/5 ==========")
    t0_fold = time.time()

    raw_fit_fold = raw_train_te.iloc[tr_idx].reset_index(drop=True)
    raw_val_fold = raw_train_te.iloc[va_idx].reset_index(drop=True)

    y_fit_fold = y_arr_28e[tr_idx]
    y_val_fold = y_arr_28e[va_idx]

    print("Building TE features for fold...")
    te_fit_fold, apply_dict_fold, te_key_summary_fold = build_te_oof_and_apply_many(
        raw_fit=raw_fit_fold,
        y_fit=y_fit_fold,
        raw_apply_dict={
            "valid": raw_val_fold,
            "test": raw_test_te,
        },
        key_specs=te_key_specs,
        n_splits=5,
        random_state=RANDOM_STATE + 100 * fold_num,
    )

    finite_te_ok = (
        np.isfinite(te_fit_fold.to_numpy(dtype=np.float32)).all()
        and np.isfinite(apply_dict_fold["valid"].to_numpy(dtype=np.float32)).all()
        and np.isfinite(apply_dict_fold["test"].to_numpy(dtype=np.float32)).all()
    )

    if not finite_te_ok:
        raise ValueError(f"Non-finite TE features detected in fold {fold_num}.")

    X_fit_base_fold = to_float32_matrix(take_rows(X_train_proc_model, tr_idx))
    X_val_base_fold = to_float32_matrix(take_rows(X_train_proc_model, va_idx))
    X_test_base_28e = to_float32_matrix(X_test_proc_model)

    X_fit_fold = append_features(X_fit_base_fold, te_fit_fold)
    X_val_fold = append_features(X_val_base_fold, apply_dict_fold["valid"])
    X_test_fold = append_features(X_test_base_28e, apply_dict_fold["test"])

    print("Fold train shape:", X_fit_fold.shape)
    print("Fold valid shape:", X_val_fold.shape)
    print("Fold test shape: ", X_test_fold.shape)

    model_fold = lgb.LGBMRegressor(**params_28e)

    print("Training LightGBM...")
    model_fold.fit(
        X_fit_fold,
        y_fit_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=1000, verbose=False),
            lgb.log_evaluation(period=1000),
        ],
    )

    best_iter_fold = int(model_fold.best_iteration_ or params_28e["n_estimators"])

    pred_fit_raw = model_fold.predict(X_fit_fold, num_iteration=best_iter_fold)
    pred_val_raw = model_fold.predict(X_val_fold, num_iteration=best_iter_fold)
    pred_test_raw = model_fold.predict(X_test_fold, num_iteration=best_iter_fold)

    pred_fit_clip = np.clip(pred_fit_raw, 0, 100).astype(np.float32)
    pred_val_clip = np.clip(pred_val_raw, 0, 100).astype(np.float32)
    pred_test_clip = np.clip(pred_test_raw, 0, 100).astype(np.float32)

    oof_pred_raw_28e[va_idx] = pred_val_raw.astype(np.float32)
    oof_pred_clip_28e[va_idx] = pred_val_clip

    np.save(oof_raw_path_28e, oof_pred_raw_28e)
    np.save(oof_clip_path_28e, oof_pred_clip_28e)
    np.save(fold_test_path, pred_test_clip)

    fold_elapsed = time.time() - t0_fold

    fold_row = {
        "artifact": ARTIFACT,
        "fold": fold_num,
        "train_rows": int(len(tr_idx)),
        "valid_rows": int(len(va_idx)),
        "n_features": int(X_fit_fold.shape[1]),
        "best_iteration": best_iter_fold,
        "train_mse_raw": float(mean_squared_error(y_fit_fold, pred_fit_raw)),
        "train_mse_clipped": float(mean_squared_error(y_fit_fold, pred_fit_clip)),
        "valid_mse_raw": float(mean_squared_error(y_val_fold, pred_val_raw)),
        "valid_mse_clipped": float(mean_squared_error(y_val_fold, pred_val_clip)),
        "pred_valid_mean": float(np.mean(pred_val_clip)),
        "pred_valid_std": float(np.std(pred_val_clip)),
        "pred_valid_min": float(np.min(pred_val_clip)),
        "pred_valid_max": float(np.max(pred_val_clip)),
        "elapsed_seconds": float(fold_elapsed),
    }

    fold_metrics_28e = pd.concat(
        [fold_metrics_28e[fold_metrics_28e.get("fold", pd.Series(dtype=int)) != fold_num], pd.DataFrame([fold_row])],
        ignore_index=True,
    ).sort_values("fold").reset_index(drop=True)

    fold_metrics_28e.to_csv(metrics_path_28e, index=False)

    print("Fold result:")
    print(pd.DataFrame([fold_row]).T)

    # Free memory aggressively.
    del model_fold
    del raw_fit_fold, raw_val_fold
    del te_fit_fold, apply_dict_fold, te_key_summary_fold
    del X_fit_base_fold, X_val_base_fold, X_test_base_28e
    del X_fit_fold, X_val_fold, X_test_fold
    del pred_fit_raw, pred_val_raw, pred_test_raw
    del pred_fit_clip, pred_val_clip, pred_test_clip
    gc.collect()

# Finalize only if all folds are complete.
TE_OOF_RUN_PASS = bool(np.isfinite(oof_pred_clip_28e).all())

print("\nTE_OOF_RUN_PASS =", TE_OOF_RUN_PASS)

if TE_OOF_RUN_PASS:
    overall_oof_mse_raw_28e = float(mean_squared_error(y_arr_28e, oof_pred_raw_28e))
    overall_oof_mse_clip_28e = float(mean_squared_error(y_arr_28e, oof_pred_clip_28e))

    print("Overall OOF MSE raw:    ", overall_oof_mse_raw_28e)
    print("Overall OOF MSE clipped:", overall_oof_mse_clip_28e)

    print("\nFold metrics:")
    display(fold_metrics_28e)

    fold_test_preds = []
    for fold_num in range(1, 6):
        fold_path = fold_test_pred_path_28e(fold_num)
        if not os.path.exists(fold_path):
            raise FileNotFoundError(f"Missing fold test prediction file: {fold_path}")
        fold_test_preds.append(np.load(fold_path).astype(np.float32))

    test_pred_28e = np.mean(np.vstack(fold_test_preds), axis=0)
    test_pred_28e = np.clip(test_pred_28e, 0, 100).astype(np.float32)

    oof_df_28e = pd.DataFrame({
        "row_index": np.arange(n_train_28e),
        "PERCENT_PROFICIENT": y_arr_28e,
        "pred_raw": oof_pred_raw_28e,
        "pred_clipped": oof_pred_clip_28e,
    })
    oof_df_28e.to_csv(oof_csv_path_28e, index=False)

    testpred_df_28e = pd.DataFrame({
        "ASSESSMENT_ID": np.asarray(test_ids),
        "PERCENT_PROFICIENT": test_pred_28e,
    })
    testpred_df_28e.to_csv(test_csv_path_28e, index=False)

    submission_28e = testpred_df_28e[["ASSESSMENT_ID", "PERCENT_PROFICIENT"]].copy()
    submission_28e.to_csv(submission_path_28e, index=False)

    print("\nSaved files:")
    print(" -", metrics_path_28e)
    print(" -", oof_raw_path_28e)
    print(" -", oof_clip_path_28e)
    print(" -", oof_csv_path_28e)
    print(" -", test_csv_path_28e)
    print(" -", submission_path_28e)

    print("\nSubmission prediction summary:")
    print(submission_28e["PERCENT_PROFICIENT"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

    print("\nAnchor comparisons:")
    print("Best pure long100k LGBM OOF:          93.726214")
    print("Best pre-residual blend OOF:          88.966368")
    print("Current 77.277 residual-stack OOF:    87.136049")
    print("This TE OOF clipped MSE:             ", overall_oof_mse_clip_28e)
    print("Gain vs pure long100k LGBM:          ", 93.726214 - overall_oof_mse_clip_28e)
    print("Gain vs pre-residual blend:          ", 88.966368 - overall_oof_mse_clip_28e)
    print("Gain vs current residual-stack OOF:  ", 87.136049 - overall_oof_mse_clip_28e)

else:
    print("OOF predictions are incomplete. Rerun this same cell to continue/checkpoint.")

Artifact: lgbm_te_base_5fold_oof_v1
Rows: 144921 train | 48307 test
Base features: 162
TE features: 68
Augmented features expected: 230
Already completed folds: []

========== Fold 1/5 ==========
Building TE features for fold...
Fold train shape: (115936, 230)
Fold valid shape: (28985, 230)
Fold test shape:  (48307, 230)
Training LightGBM...
[1000]	valid_0's l2: 85.3052
[2000]	valid_0's l2: 84.3444
[3000]	valid_0's l2: 84.0699
[4000]	valid_0's l2: 84.0502
[5000]	valid_0's l2: 83.9653
[6000]	valid_0's l2: 84.0682
Fold result:
                                           0
artifact           lgbm_te_base_5fold_oof_v1
fold                                       1
train_rows                            115936
valid_rows                             28985
n_features                               230
best_iteration                          5033
train_mse_raw                      11.666033
train_mse_clipped                  11.650144
valid_mse_raw                      83.961103
valid_mse_clipped  

,artifact,fold,train_rows,valid_rows,n_features,best_iteration,train_mse_raw,train_mse_clipped,valid_mse_raw,valid_mse_clipped,pred_valid_mean,pred_valid_std,pred_valid_min,pred_valid_max,elapsed_seconds
0,lgbm_te_base_5fold_oof_v1,1,115936,28985,230,5033,11.666033,11.650144,83.961103,83.846176,54.934315,24.891148,0.0,100.0,281.356708
1,lgbm_te_base_5fold_oof_v1,2,115937,28984,230,5379,10.535665,10.520914,79.906313,79.800461,53.856457,25.086536,0.0,100.0,298.724768
2,lgbm_te_base_5fold_oof_v1,3,115937,28984,230,5629,9.836355,9.822016,84.624473,84.527496,54.604282,25.013144,0.0,100.0,277.812274
3,lgbm_te_base_5fold_oof_v1,4,115937,28984,230,4379,14.653784,14.633527,83.625740,83.476036,54.854794,24.772558,0.0,100.0,224.089123
4,lgbm_te_base_5fold_oof_v1,5,115937,28984,230,4240,15.136295,15.119184,82.946000,82.887871,54.425667,24.821636,0.0,100.0,219.875332



Saved files:
 - model_results/lgbm_te_base_5fold_oof_v1_fold_metrics.csv
 - model_results/lgbm_te_base_5fold_oof_v1_oof_raw.npy
 - model_results/lgbm_te_base_5fold_oof_v1_oof_clipped.npy
 - model_results/oof_lgbm_te_base_5fold_oof_v1.csv
 - model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv
 - submission_lgbm_te_base_5fold_oof_v1_foldavg.csv

Submission prediction summary:
count    48307.000000
mean        54.500694
std         24.779079
min          0.000000
1%           8.264243
5%          16.623557
25%         34.708241
50%         52.084106
75%         75.007858
95%         95.584126
99%         99.703726
max        100.000000
Name: PERCENT_PROFICIENT, dtype: float64

Anchor comparisons:
Best pure long100k LGBM OOF:          93.726214
Best pre-residual blend OOF:          88.966368
Current 77.277 residual-stack OOF:    87.136049
This TE OOF clipped MSE:              82.90760803222656
Gain vs pure long100k LGBM:           10.818605967773436
Gain vs pre-residual blend:   

### 28E checkpoint: target/statistical encoding LightGBM OOF breakthrough

The leakage-safe cross-fitted target/statistical encoding branch completed successfully.

Artifact: `lgbm_te_base_5fold_oof_v1`

Base features: 162  
Target/statistical encoding features: 68  
Total features: 230  

OOF clipped MSE: **82.907608**

This beats:
- best pure long100k LightGBM OOF, 93.726214, by about 10.82 MSE points
- pre-residual ET + long100k blend OOF, 88.966368, by about 6.06 MSE points
- current 77.277 public residual-stack anchor OOF, 87.136049, by about 4.23 MSE points

Interpretation: cross-fitted hierarchical target/statistical encodings add major new signal. This branch should now be treated as a primary modeling direction, not a side experiment.

In [64]:
# ============================================================
# 28F. Preflight for blending the new TE OOF artifact
# ============================================================
#
# Purpose:
# - Load the new target/statistical-encoding LightGBM OOF artifact.
# - Load existing saved OOF/test artifacts if available.
# - Infer prediction columns safely.
# - Confirm shapes, finiteness, and component OOF MSEs.
#
# Run 28G only if BLEND28F_PASS = True.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

if "y_train" not in globals():
    raise ValueError("y_train is missing.")
if "test_ids" not in globals():
    raise ValueError("test_ids is missing.")

y_blend_28f = np.asarray(y_train, dtype=np.float64).reshape(-1)
test_ids_28f = np.asarray(test_ids)

n_train_28f = len(y_blend_28f)
n_test_28f = len(test_ids_28f)

def _existing(paths):
    return [p for p in paths if p is not None and os.path.exists(p)]

def _unique_keep_order(paths):
    out = []
    seen = set()
    for p in paths:
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out

def infer_oof_prediction(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row count mismatch for {path}: {len(df)} vs {len(y_ref)}")

    numeric_cols = []
    for col in df.columns:
        if col.lower() in ["row_index", "assessment_id", "fold"]:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_cols.append(col)

    candidates = []
    for col in numeric_cols:
        arr = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
        if len(arr) != len(y_ref):
            continue
        if not np.isfinite(arr).all():
            continue

        arr_clip = np.clip(arr, 0, 100)
        mse = float(mean_squared_error(y_ref, arr_clip))

        # Exclude the true target column if present.
        if mse < 1e-8:
            continue

        # Exclude obvious index-like columns.
        if np.nanstd(arr_clip) < 1e-8:
            continue

        candidates.append((mse, col, arr_clip))

    if len(candidates) == 0:
        raise ValueError(f"Could not infer OOF prediction column for {path}. Columns: {list(df.columns)}")

    candidates = sorted(candidates, key=lambda x: x[0])
    best_mse, best_col, best_arr = candidates[0]
    return best_arr.astype(np.float32), best_col, best_mse, list(df.columns)

def infer_test_prediction(path, test_ids_ref):
    df = pd.read_csv(path)

    if "ASSESSMENT_ID" in df.columns:
        # Align to current test_ids if needed.
        if len(df) == len(test_ids_ref) and np.array_equal(df["ASSESSMENT_ID"].to_numpy(), test_ids_ref):
            aligned = df.copy()
        else:
            key_df = pd.DataFrame({"ASSESSMENT_ID": test_ids_ref})
            aligned = key_df.merge(df, on="ASSESSMENT_ID", how="left")
            if len(aligned) != len(test_ids_ref):
                raise ValueError(f"Test merge row mismatch for {path}")
    else:
        aligned = df.copy()

    if len(aligned) != len(test_ids_ref):
        raise ValueError(f"Test row count mismatch for {path}: {len(aligned)} vs {len(test_ids_ref)}")

    priority_cols = [
        "PERCENT_PROFICIENT",
        "pred_clipped",
        "prediction",
        "pred",
        "test_pred",
        "foldavg",
        "fold_avg",
        "mean_pred",
    ]

    for col in priority_cols:
        if col in aligned.columns and pd.api.types.is_numeric_dtype(aligned[col]):
            arr = pd.to_numeric(aligned[col], errors="coerce").to_numpy(dtype=np.float64)
            if np.isfinite(arr).all():
                return np.clip(arr, 0, 100).astype(np.float32), col, list(aligned.columns)

    numeric_cols = []
    for col in aligned.columns:
        if col.lower() in ["assessment_id", "row_index", "fold"]:
            continue
        if pd.api.types.is_numeric_dtype(aligned[col]):
            numeric_cols.append(col)

    pred_like_cols = [
        c for c in numeric_cols
        if ("pred" in c.lower()) or ("fold" in c.lower())
    ]

    if len(pred_like_cols) >= 2:
        arr = aligned[pred_like_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
        if np.isfinite(arr).all():
            return np.clip(arr.mean(axis=1), 0, 100).astype(np.float32), "mean_of_pred_like_cols", list(aligned.columns)

    if len(numeric_cols) == 1:
        col = numeric_cols[0]
        arr = pd.to_numeric(aligned[col], errors="coerce").to_numpy(dtype=np.float64)
        if np.isfinite(arr).all():
            return np.clip(arr, 0, 100).astype(np.float32), col, list(aligned.columns)

    raise ValueError(f"Could not infer test prediction column for {path}. Columns: {list(aligned.columns)}")

def load_component(name, oof_candidates, test_candidates, y_ref, test_ids_ref):
    oof_candidates = _unique_keep_order(_existing(oof_candidates))
    test_candidates = _unique_keep_order(_existing(test_candidates))

    if len(oof_candidates) == 0:
        return None, f"{name}: no OOF file found"
    if len(test_candidates) == 0:
        return None, f"{name}: no test/submission file found"

    best_oof = None
    oof_errors = []

    for oof_path in oof_candidates:
        try:
            pred_oof, oof_col, oof_mse, oof_cols = infer_oof_prediction(oof_path, y_ref)
            if best_oof is None or oof_mse < best_oof["oof_mse"]:
                best_oof = {
                    "path": oof_path,
                    "pred": pred_oof,
                    "col": oof_col,
                    "oof_mse": oof_mse,
                    "columns": oof_cols,
                }
        except Exception as e:
            oof_errors.append((oof_path, str(e)))

    if best_oof is None:
        return None, f"{name}: OOF files found but none loaded. Errors: {oof_errors[:3]}"

    best_test = None
    test_errors = []

    for test_path in test_candidates:
        try:
            pred_test, test_col, test_cols = infer_test_prediction(test_path, test_ids_ref)
            best_test = {
                "path": test_path,
                "pred": pred_test,
                "col": test_col,
                "columns": test_cols,
            }
            break
        except Exception as e:
            test_errors.append((test_path, str(e)))

    if best_test is None:
        return None, f"{name}: test files found but none loaded. Errors: {test_errors[:3]}"

    comp = {
        "name": name,
        "oof_path": best_oof["path"],
        "test_path": best_test["path"],
        "oof_col": best_oof["col"],
        "test_col": best_test["col"],
        "oof_pred": best_oof["pred"],
        "test_pred": best_test["pred"],
        "oof_mse": best_oof["oof_mse"],
    }

    return comp, f"{name}: loaded"

# Known core artifacts.
artifact_specs_28f = [
    {
        "name": "rf500",
        "oof": ["model_results/oof_rf_500_base.csv"],
        "test": [
            "submission_rf_500_base_oof_foldavg.csv",
            "model_results/testpred_rf_500_base_folds.csv",
        ],
    },
    {
        "name": "extratrees_safe",
        "oof": ["model_results/oof_extratrees_safe_base.csv"],
        "test": ["model_results/testpred_extratrees_safe_base_foldavg.csv"],
    },
    {
        "name": "lgbm_12k",
        "oof": ["model_results/oof_lgbm_t03_base_5fold_oof.csv"],
        "test": ["model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv"],
    },
    {
        "name": "lgbm_long80k",
        "oof": ["model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv"],
        "test": ["model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv"],
    },
    {
        "name": "lgbm_long100k",
        "oof": ["model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv"],
        "test": ["model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv"],
    },
    {
        "name": "current_best_pre_resid_blend_80p662",
        "oof": ["model_results/oof_blend_auto_et_lgbm_long_oof_weighted.csv"],
        "test": [
            "model_results/testpred_blend_auto_et_lgbm_long_oof_weighted.csv",
            "submission_blend_auto_et_lgbm_long_oof_weighted.csv",
        ],
    },
    {
        "name": "lgbm_te_v1",
        "oof": ["model_results/oof_lgbm_te_base_5fold_oof_v1.csv"],
        "test": ["model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv"],
    },
]

# Try to include the current 77.277 residual-stack anchor if an OOF file exists.
resid_oof_candidates_28f = _unique_keep_order(
    glob.glob("model_results/*resid*stack*oof*.csv") +
    glob.glob("model_results/oof*resid*stack*.csv") +
    glob.glob("model_results/*residstack*.csv")
)

resid_test_candidates_28f = _unique_keep_order(
    glob.glob("submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv") +
    glob.glob("model_results/testpred*resid*stack*1p135*.csv") +
    glob.glob("model_results/*resid*stack*test*.csv")
)

artifact_specs_28f.append({
    "name": "residstack_best77_if_oof_available",
    "oof": resid_oof_candidates_28f,
    "test": resid_test_candidates_28f,
})

components_28f = []
load_messages_28f = []

for spec in artifact_specs_28f:
    comp, msg = load_component(
        name=spec["name"],
        oof_candidates=spec["oof"],
        test_candidates=spec["test"],
        y_ref=y_blend_28f,
        test_ids_ref=test_ids_28f,
    )
    load_messages_28f.append(msg)
    if comp is not None:
        components_28f.append(comp)

print("Load messages:")
for msg in load_messages_28f:
    print(" -", msg)

have_te_28f = any(c["name"] == "lgbm_te_v1" for c in components_28f)

shape_ok_28f = all(
    len(c["oof_pred"]) == n_train_28f and len(c["test_pred"]) == n_test_28f
    for c in components_28f
)

finite_ok_28f = all(
    np.isfinite(c["oof_pred"]).all() and np.isfinite(c["test_pred"]).all()
    for c in components_28f
)

BLEND28F_PASS = bool(have_te_28f and len(components_28f) >= 1 and shape_ok_28f and finite_ok_28f)

summary_28f = pd.DataFrame([
    {
        "name": c["name"],
        "oof_mse": c["oof_mse"],
        "oof_col": c["oof_col"],
        "test_col": c["test_col"],
        "oof_path": c["oof_path"],
        "test_path": c["test_path"],
        "oof_mean": float(np.mean(c["oof_pred"])),
        "oof_std": float(np.std(c["oof_pred"])),
        "test_mean": float(np.mean(c["test_pred"])),
        "test_std": float(np.std(c["test_pred"])),
    }
    for c in components_28f
]).sort_values("oof_mse").reset_index(drop=True)

print("\nBLEND28F_PASS =", BLEND28F_PASS)
print("Loaded components:", len(components_28f))
print("Have TE component:", have_te_28f)
print("Shape OK:", shape_ok_28f)
print("Finite OK:", finite_ok_28f)

display(summary_28f)

blend_component_names_28f = [c["name"] for c in components_28f]
blend_oof_mat_28f = np.column_stack([c["oof_pred"] for c in components_28f]).astype(np.float32)
blend_test_mat_28f = np.column_stack([c["test_pred"] for c in components_28f]).astype(np.float32)
blend_y_28f = y_blend_28f.astype(np.float32)
blend_component_summary_28f = summary_28f.copy()

if len(components_28f) >= 2:
    resid_mat_28f = blend_y_28f.reshape(-1, 1) - blend_oof_mat_28f
    resid_corr_28f = pd.DataFrame(
        resid_mat_28f,
        columns=blend_component_names_28f
    ).corr()
    print("\nResidual correlation matrix:")
    display(resid_corr_28f)

Load messages:
 - rf500: loaded
 - extratrees_safe: loaded
 - lgbm_12k: loaded
 - lgbm_long80k: loaded
 - lgbm_long100k: loaded
 - current_best_pre_resid_blend_80p662: no OOF file found
 - lgbm_te_v1: loaded
 - residstack_best77_if_oof_available: loaded

BLEND28F_PASS = True
Loaded components: 7
Have TE component: True
Shape OK: True
Finite OK: True


,name,oof_mse,oof_col,test_col,oof_path,test_path,oof_mean,oof_std,test_mean,test_std
0,lgbm_te_v1,82.907614,pred_raw,PERCENT_PROFICIENT,model_results/oof_lgbm_te_base_5fold_oof_v1.csv,model_results/testpred_lgbm_te_base_5fold_oof_...,54.535107,24.920244,54.500694,24.778822
1,residstack_best77_if_oof_available,85.647385,OOF_residstack_oldnew_expanded_oldw_0p025_neww...,PERCENT_PROFICIENT,model_results/oof_residstack_oldnew_expanded_o...,submission_residstack_lgbm_on_best80p662_blend...,54.241405,25.043358,54.097843,24.676031
2,lgbm_long100k,93.726214,OOF_lgbm_t03_base_5fold_oof_long100k_lr02,TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02,model_results/oof_lgbm_t03_base_5fold_oof_long...,model_results/testpred_lgbm_t03_base_5fold_oof...,54.153145,24.734667,54.074207,24.605724
3,lgbm_long80k,94.163485,OOF_lgbm_t03_base_5fold_oof_long80k_lr03,TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03,model_results/oof_lgbm_t03_base_5fold_oof_long...,model_results/testpred_lgbm_t03_base_5fold_oof...,54.147018,24.743267,54.067600,24.607933
4,lgbm_12k,98.779406,OOF_lgbm_t03_base_5fold_oof,TESTPRED_lgbm_t03_base_5fold_oof,model_results/oof_lgbm_t03_base_5fold_oof.csv,model_results/testpred_lgbm_t03_base_5fold_oof...,54.152809,24.283770,54.086674,24.207260
5,extratrees_safe,110.749294,oof_pred,PERCENT_PROFICIENT,model_results/oof_extratrees_safe_base.csv,model_results/testpred_extratrees_safe_base_fo...,54.127365,23.857744,54.040169,23.782902
6,rf500,122.328024,rf_500_base_oof_pred,PERCENT_PROFICIENT,model_results/oof_rf_500_base.csv,submission_rf_500_base_oof_foldavg.csv,54.134144,22.839348,54.056767,22.817463



Residual correlation matrix:


,rf500,extratrees_safe,lgbm_12k,lgbm_long80k,lgbm_long100k,lgbm_te_v1,residstack_best77_if_oof_available
rf500,1.000000,0.929709,0.842448,0.791132,0.792623,0.759557,0.834595
extratrees_safe,0.929709,1.000000,0.802689,0.771684,0.773277,0.789169,0.873778
lgbm_12k,0.842448,0.802689,1.000000,0.972530,0.965221,0.797072,0.935210
lgbm_long80k,0.791132,0.771684,0.972530,1.000000,0.991213,0.787838,0.955306
lgbm_long100k,0.792623,0.773277,0.965221,0.991213,1.000000,0.789823,0.958859
lgbm_te_v1,0.759557,0.789169,0.797072,0.787838,0.789823,1.000000,0.837032
residstack_best77_if_oof_available,0.834595,0.873778,0.935210,0.955306,0.958859,0.837032,1.000000


### public leaderboard checkpoint: target/statistical encoding breakthrough

Submitted file:

`submission_lgbm_te_base_5fold_oof_v1_foldavg.csv`

Public leaderboard MSE:

**70.997**

Artifact:

`lgbm_te_base_5fold_oof_v1`

Model:

LightGBM on the 162-feature base matrix plus 68 leakage-safe cross-fitted target/statistical encoding features, for a total of 230 features.

OOF clipped MSE:

**82.907608**

Fold validation MSEs:

- Fold 1: 83.846176
- Fold 2: 79.800461
- Fold 3: 84.527496
- Fold 4: 83.476036
- Fold 5: 82.887871

Interpretation:

This is the strongest public result so far and a major validation of the new hierarchical target/statistical encoding direction. The previous best public score was 77.277 from the residual-stack branch. The new public score, 70.997, beats it by about 6.28 MSE points.

This branch should now be treated as a primary modeling direction. Future work should include repaired blending with verified artifacts, residual stacking on the TE model, and model-family expansion using the same target/statistical encoding representation.

In [66]:
# ============================================================
# 28G REPAIRED. Safe convex blend with TE artifact
# Excludes residual-stack artifacts unless explicitly verified later.
# ============================================================
#
# Why this replaces the previous 28G:
# - Previous 28G may have mixed a residual-stack OOF file with a different
#   residual-stack test/submission file.
# - This repaired cell uses only verified non-residual components:
#     rf500
#     extratrees_safe
#     lgbm_12k
#     lgbm_long80k
#     lgbm_long100k
#     lgbm_te_v1
#
# Output submission:
#   submission_blend_te_verified_noresid_weighted.csv
#
# Paste the output before submitting.
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

if "components_28f" not in globals():
    raise ValueError("components_28f is missing. Re-run 28F first, then run this repaired 28G.")

if "test_ids_28f" not in globals():
    raise ValueError("test_ids_28f is missing. Re-run 28F first, then run this repaired 28G.")

if "y_train" not in globals():
    raise ValueError("y_train is missing.")

os.makedirs("model_results", exist_ok=True)

allowed_safe_components_28g = {
    "rf500",
    "extratrees_safe",
    "lgbm_12k",
    "lgbm_long80k",
    "lgbm_long100k",
    "lgbm_te_v1",
}

safe_components_28g = [
    c for c in components_28f
    if c["name"] in allowed_safe_components_28g
]

safe_names_28g = [c["name"] for c in safe_components_28g]

missing_required_28g = [
    name for name in ["lgbm_te_v1"]
    if name not in safe_names_28g
]

SAFE_BLEND28G_PREFLIGHT_PASS = bool(
    len(safe_components_28g) >= 2
    and len(missing_required_28g) == 0
)

print("SAFE_BLEND28G_PREFLIGHT_PASS =", SAFE_BLEND28G_PREFLIGHT_PASS)
print("Allowed safe components loaded:", safe_names_28g)
print("Missing required components:", missing_required_28g)

if not SAFE_BLEND28G_PREFLIGHT_PASS:
    raise ValueError("Safe blend preflight failed. Do not continue.")

y_safe_28g = np.asarray(y_train, dtype=np.float64).reshape(-1)
test_ids_safe_28g = np.asarray(test_ids_28f)

P_safe_28g = np.column_stack([
    np.asarray(c["oof_pred"], dtype=np.float64)
    for c in safe_components_28g
])

T_safe_28g = np.column_stack([
    np.asarray(c["test_pred"], dtype=np.float64)
    for c in safe_components_28g
])

# Since all component predictions are clipped to [0, 100],
# convex combinations remain inside [0, 100].
n_safe_28g, k_safe_28g = P_safe_28g.shape

shape_ok_safe_28g = (
    P_safe_28g.shape[0] == len(y_safe_28g)
    and T_safe_28g.shape[0] == len(test_ids_safe_28g)
)

finite_ok_safe_28g = (
    np.isfinite(P_safe_28g).all()
    and np.isfinite(T_safe_28g).all()
)

if not shape_ok_safe_28g or not finite_ok_safe_28g:
    raise ValueError("Shape or finite check failed.")

component_mses_safe_28g = np.array([
    float(mean_squared_error(y_safe_28g, P_safe_28g[:, i]))
    for i in range(k_safe_28g)
])

component_summary_safe_28g = pd.DataFrame({
    "component": safe_names_28g,
    "component_oof_mse": component_mses_safe_28g,
    "oof_mean": P_safe_28g.mean(axis=0),
    "oof_std": P_safe_28g.std(axis=0),
    "test_mean": T_safe_28g.mean(axis=0),
    "test_std": T_safe_28g.std(axis=0),
}).sort_values("component_oof_mse").reset_index(drop=True)

print("\nSafe component summary:")
display(component_summary_safe_28g)

A_safe_28g = (P_safe_28g.T @ P_safe_28g) / n_safe_28g
b_safe_28g = (P_safe_28g.T @ y_safe_28g) / n_safe_28g
c_safe_28g = float((y_safe_28g @ y_safe_28g) / n_safe_28g)

def mse_for_weights_safe_28g(w):
    w = np.asarray(w, dtype=np.float64)
    return float(w @ A_safe_28g @ w - 2.0 * (w @ b_safe_28g) + c_safe_28g)

def mse_for_weight_matrix_safe_28g(W):
    W = np.asarray(W, dtype=np.float64)
    return (
        np.einsum("ij,jk,ik->i", W, A_safe_28g, W)
        - 2.0 * (W @ b_safe_28g)
        + c_safe_28g
    )

candidate_rows_safe_28g = []

def add_candidate_safe_28g(label, w):
    w = np.asarray(w, dtype=np.float64)
    w = np.maximum(w, 0)
    if w.sum() <= 0:
        return
    w = w / w.sum()
    row = {
        "label": label,
        "mse": mse_for_weights_safe_28g(w),
        "weights": w,
    }
    candidate_rows_safe_28g.append(row)

# Pure models.
for i, name in enumerate(safe_names_28g):
    w = np.zeros(k_safe_28g)
    w[i] = 1.0
    add_candidate_safe_28g(f"pure_{name}", w)

# Pairwise fine grid.
grid_safe_28g = np.linspace(0.0, 1.0, 1001)

for i in range(k_safe_28g):
    for j in range(i + 1, k_safe_28g):
        W = np.zeros((len(grid_safe_28g), k_safe_28g), dtype=np.float64)
        W[:, i] = grid_safe_28g
        W[:, j] = 1.0 - grid_safe_28g

        mses = mse_for_weight_matrix_safe_28g(W)
        best_idx = int(np.argmin(mses))

        add_candidate_safe_28g(
            f"pair_{safe_names_28g[i]}__{safe_names_28g[j]}",
            W[best_idx],
        )

# Random simplex searches.
rng_safe_28g = np.random.default_rng(globals().get("RANDOM_STATE", 9890) + 2809)

# Uniform search.
W_uniform_safe = rng_safe_28g.dirichlet(np.ones(k_safe_28g), size=50000)
mses_uniform_safe = mse_for_weight_matrix_safe_28g(W_uniform_safe)
add_candidate_safe_28g(
    "random_dirichlet_uniform_50k",
    W_uniform_safe[int(np.argmin(mses_uniform_safe))]
)

# TE-focused search.
te_idx_safe_28g = safe_names_28g.index("lgbm_te_v1")

alpha_te_safe = np.ones(k_safe_28g) * 0.25
alpha_te_safe[te_idx_safe_28g] = 10.0

# Also allow mass on the best older LGBM artifacts.
order_safe_28g = np.argsort(component_mses_safe_28g)
for idx in order_safe_28g[:min(3, k_safe_28g)]:
    alpha_te_safe[idx] = max(alpha_te_safe[idx], 2.5)

W_te_safe = rng_safe_28g.dirichlet(alpha_te_safe, size=75000)
mses_te_safe = mse_for_weight_matrix_safe_28g(W_te_safe)
add_candidate_safe_28g(
    "random_dirichlet_te_focused_75k",
    W_te_safe[int(np.argmin(mses_te_safe))]
)

# Top-components focused search.
alpha_top_safe = np.ones(k_safe_28g) * 0.20
for rank, idx in enumerate(order_safe_28g[:min(4, k_safe_28g)]):
    alpha_top_safe[idx] = 6.0 / (rank + 1)

W_top_safe = rng_safe_28g.dirichlet(alpha_top_safe, size=75000)
mses_top_safe = mse_for_weight_matrix_safe_28g(W_top_safe)
add_candidate_safe_28g(
    "random_dirichlet_top_focused_75k",
    W_top_safe[int(np.argmin(mses_top_safe))]
)

best_candidate_safe_28g = min(candidate_rows_safe_28g, key=lambda d: d["mse"])
best_w_safe_28g = best_candidate_safe_28g["weights"]

blend_oof_safe_28g = np.clip(P_safe_28g @ best_w_safe_28g, 0, 100).astype(np.float32)
blend_test_safe_28g = np.clip(T_safe_28g @ best_w_safe_28g, 0, 100).astype(np.float32)

best_mse_safe_28g = float(mean_squared_error(y_safe_28g, blend_oof_safe_28g))
pure_te_mse_safe_28g = float(mean_squared_error(y_safe_28g, P_safe_28g[:, te_idx_safe_28g]))
gain_vs_te_safe_28g = pure_te_mse_safe_28g - best_mse_safe_28g

weight_table_safe_28g = pd.DataFrame({
    "component": safe_names_28g,
    "weight": best_w_safe_28g,
    "component_oof_mse": component_mses_safe_28g,
}).sort_values("weight", ascending=False).reset_index(drop=True)

screen_rows_safe_28g = []
for cand in candidate_rows_safe_28g:
    row = {
        "label": cand["label"],
        "mse": cand["mse"],
    }
    for name, w_val in zip(safe_names_28g, cand["weights"]):
        row[f"w_{name}"] = w_val
    screen_rows_safe_28g.append(row)

blend_screen_safe_28g = (
    pd.DataFrame(screen_rows_safe_28g)
    .sort_values("mse")
    .reset_index(drop=True)
)

blend_oof_path_safe_28g = "model_results/oof_blend_te_verified_noresid_weighted.csv"
blend_test_path_safe_28g = "model_results/testpred_blend_te_verified_noresid_weighted.csv"
blend_submission_path_safe_28g = "submission_blend_te_verified_noresid_weighted.csv"
blend_screen_path_safe_28g = "model_results/blend_te_verified_noresid_weight_screen.csv"
blend_weight_path_safe_28g = "model_results/blend_te_verified_noresid_best_weights.csv"

pd.DataFrame({
    "row_index": np.arange(len(y_safe_28g)),
    "PERCENT_PROFICIENT": y_safe_28g,
    "pred_clipped": blend_oof_safe_28g,
}).to_csv(blend_oof_path_safe_28g, index=False)

pd.DataFrame({
    "ASSESSMENT_ID": test_ids_safe_28g,
    "PERCENT_PROFICIENT": blend_test_safe_28g,
}).to_csv(blend_test_path_safe_28g, index=False)

pd.DataFrame({
    "ASSESSMENT_ID": test_ids_safe_28g,
    "PERCENT_PROFICIENT": blend_test_safe_28g,
}).to_csv(blend_submission_path_safe_28g, index=False)

blend_screen_safe_28g.to_csv(blend_screen_path_safe_28g, index=False)
weight_table_safe_28g.to_csv(blend_weight_path_safe_28g, index=False)

SAFE_BLEND28G_PASS = True
SAFE_BLEND28G_SUBMISSION_WORTHY = bool(gain_vs_te_safe_28g >= 1.0)

print("\nSAFE_BLEND28G_PASS =", SAFE_BLEND28G_PASS)
print("Best candidate label:", best_candidate_safe_28g["label"])
print("Best safe blend OOF MSE:", best_mse_safe_28g)
print("Pure TE OOF MSE:", pure_te_mse_safe_28g)
print("Gain vs pure TE OOF:", gain_vs_te_safe_28g)
print("Current pure TE public MSE: 70.997")
print("SAFE_BLEND28G_SUBMISSION_WORTHY =", SAFE_BLEND28G_SUBMISSION_WORTHY)

print("\nBest safe weights:")
display(weight_table_safe_28g)

print("\nTop 20 safe blend candidates:")
display(blend_screen_safe_28g.head(20))

print("\nSaved files:")
print(" -", blend_oof_path_safe_28g)
print(" -", blend_test_path_safe_28g)
print(" -", blend_submission_path_safe_28g)
print(" -", blend_screen_path_safe_28g)
print(" -", blend_weight_path_safe_28g)

print("\nSubmission prediction summary:")
print(pd.Series(blend_test_safe_28g).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

if SAFE_BLEND28G_SUBMISSION_WORTHY:
    print("\nCandidate to consider after review:")
    print(blend_submission_path_safe_28g)
else:
    print("\nSafe blend did not improve enough over pure TE. Keep it in backlog.")

SAFE_BLEND28G_PREFLIGHT_PASS = True
Allowed safe components loaded: ['rf500', 'extratrees_safe', 'lgbm_12k', 'lgbm_long80k', 'lgbm_long100k', 'lgbm_te_v1']
Missing required components: []

Safe component summary:


,component,component_oof_mse,oof_mean,oof_std,test_mean,test_std
0,lgbm_te_v1,82.907614,54.535106,24.920245,54.500693,24.778823
1,lgbm_long100k,93.726214,54.153144,24.734667,54.074208,24.605725
2,lgbm_long80k,94.163485,54.147019,24.743266,54.067601,24.607933
3,lgbm_12k,98.779406,54.152808,24.283770,54.086671,24.207260
4,extratrees_safe,110.749294,54.127367,23.857745,54.040173,23.782901
5,rf500,122.328024,54.134143,22.839348,54.056767,22.817463



SAFE_BLEND28G_PASS = True
Best candidate label: random_dirichlet_te_focused_75k
Best safe blend OOF MSE: 78.07151033226157
Pure TE OOF MSE: 82.90761366243048
Gain vs pure TE OOF: 4.8361033301689105
Current pure TE public MSE: 70.997
SAFE_BLEND28G_SUBMISSION_WORTHY = True

Best safe weights:


,component,weight,component_oof_mse
0,lgbm_te_v1,0.624592,82.907614
1,lgbm_long100k,0.207687,93.726214
2,lgbm_long80k,0.127595,94.163485
3,extratrees_safe,0.040027,110.749294
4,lgbm_12k,0.000082,98.779406
5,rf500,0.000017,122.328024



Top 20 safe blend candidates:


,label,mse,w_rf500,w_extratrees_safe,w_lgbm_12k,w_lgbm_long80k,w_lgbm_long100k,w_lgbm_te_v1
0,random_dirichlet_te_focused_75k,78.071510,0.000017,0.040027,0.000082,0.127595,0.207687,0.624592
1,random_dirichlet_top_focused_75k,78.085494,0.000026,0.034714,0.002192,0.130861,0.196293,0.635913
2,random_dirichlet_uniform_50k,78.129569,0.001443,0.029954,0.011069,0.100940,0.249669,0.606925
3,pair_lgbm_long100k__lgbm_te_v1,78.158831,0.000000,0.000000,0.000000,0.000000,0.356000,0.644000
4,pair_lgbm_long80k__lgbm_te_v1,78.206523,0.000000,0.000000,0.000000,0.352000,0.000000,0.648000
5,pair_lgbm_12k__lgbm_te_v1,79.778222,0.000000,0.000000,0.289000,0.000000,0.000000,0.711000
6,pair_extratrees_safe__lgbm_te_v1,81.633548,0.000000,0.173000,0.000000,0.000000,0.000000,0.827000
7,pair_rf500__lgbm_te_v1,82.103802,0.124000,0.000000,0.000000,0.000000,0.000000,0.876000
8,pure_lgbm_te_v1,82.907614,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
9,pair_extratrees_safe__lgbm_long100k,88.966368,0.000000,0.319000,0.000000,0.000000,0.681000,0.000000



Saved files:
 - model_results/oof_blend_te_verified_noresid_weighted.csv
 - model_results/testpred_blend_te_verified_noresid_weighted.csv
 - submission_blend_te_verified_noresid_weighted.csv
 - model_results/blend_te_verified_noresid_weight_screen.csv
 - model_results/blend_te_verified_noresid_best_weights.csv

Submission prediction summary:
count    48307.000000
mean        54.338383
std         24.551657
min          0.201202
1%           7.931219
5%          16.555257
25%         34.791862
50%         52.371319
75%         74.537514
95%         95.197914
99%         99.362563
max         99.999191
dtype: float64

Candidate to consider after review:
submission_blend_te_verified_noresid_weighted.csv


## Checkpoint: New Public Best from Repaired Safe TE Blend

The repaired no-residual target/statistical-encoding blend was submitted to Kaggle and became the new best public result.

**Submitted file:**  
`submission_blend_te_verified_noresid_weighted.csv`

**Public leaderboard MSE:**  
`69.689`

**Local OOF MSE:**  
`78.071510`

This improves over the previous pure target/statistical-encoding LightGBM submission:

**Previous best TE file:**  
`submission_lgbm_te_base_5fold_oof_v1_foldavg.csv`

**Previous public MSE:**  
`70.997`

The repaired blend used only verified non-residual OOF/test prediction pairs and excluded the unsafe earlier blend artifact. Its main weight was on the target/statistical-encoding LightGBM, with additional complementary signal from the long LightGBM models and a small ExtraTrees contribution.

Approximate blend structure:

- `lgbm_te_v1`: dominant component
- `lgbm_long100k`: secondary component
- `lgbm_long80k`: secondary component
- `extratrees_safe`: small complementary component
- `lgbm_12k` and `rf500`: essentially zero weight

Interpretation:

This is now the clean public anchor to beat. The result confirms that the target/statistical-encoding branch is the strongest modeling direction so far, and that verified blending with older tree-ensemble artifacts still adds useful complementary signal. We should not spend the next Kaggle slot on tiny refinements around this same blend unless slots remain. The next priority should be a genuinely new model-family or feature branch, such as XGBoost on the 230-feature TE matrix, CatBoost if installation is safe, bounded/proportion-aware modeling, or a carefully designed residual model on the new TE-blend anchor.

In [67]:
# 29A. Package install/import check for next model-family branch

import sys
import subprocess
import importlib.util

def install_and_check(pkg_name, import_name=None):
    if import_name is None:
        import_name = pkg_name

    print("=" * 80)
    print(f"Checking {pkg_name}...")

    before = importlib.util.find_spec(import_name)
    if before is None:
        print(f"{pkg_name} not found. Installing with current notebook Python:")
        print(sys.executable)
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            pkg_name
        ])
    else:
        print(f"{pkg_name} already appears installed.")

    module = __import__(import_name)
    version = getattr(module, "__version__", "version not found")
    location = getattr(module, "__file__", "location not found")

    print(f"{pkg_name} import OK")
    print("version:", version)
    print("location:", location)

# Safer first priority
install_and_check("xgboost", "xgboost")

# Secondary; if this fails, stop and paste the error.
install_and_check("catboost", "catboost")

Checking xgboost...
xgboost not found. Installing with current notebook Python:
/Users/saadmanchowdhury/Desktop/All Github projects/.venv/bin/python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 33.5 MB/s  0:00:00



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


xgboost import OK
version: 3.2.0
location: /Users/saadmanchowdhury/Desktop/All Github projects/.venv/lib/python3.14/site-packages/xgboost/__init__.py
Checking catboost...
catboost not found. Installing with current notebook Python:
/Users/saadmanchowdhury/Desktop/All Github projects/.venv/bin/python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 44.1 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 52.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [catboost]3/4 [catboost]



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


catboost import OK
version: 1.2.10
location: /Users/saadmanchowdhury/Desktop/All Github projects/.venv/lib/python3.14/site-packages/catboost/__init__.py


In [70]:
# 29B replacement. Build or alias TE-augmented holdout-screen matrices for XGBoost/CatBoost
# This does NOT train a model.
#
# Expected output if successful:
#   X_tr_xgb_te_screen   shape = (screen train rows, 230)
#   X_val_xgb_te_screen  shape = (screen valid rows, 230)
#   y_tr_xgb_te_screen
#   y_val_xgb_te_screen
#
# Optional:
#   X_test_xgb_te_screen exists only if the cell has to rebuild the screen matrices.

import os
import re
import gc
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.model_selection import train_test_split, KFold

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = "PERCENT_PROFICIENT"

FORCE_REBUILD_TE_SCREEN = False

print("29B replacement: TE holdout-screen matrix setup")
print("------------------------------------------------")

required_base_29b = [
    "train_full",
    "test_full",
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
]

missing_base_29b = [name for name in required_base_29b if name not in globals()]
if missing_base_29b:
    raise ValueError(f"Missing required base objects: {missing_base_29b}")

y_arr_29b = np.asarray(y_train, dtype=np.float32).reshape(-1)

def _shape2(obj):
    try:
        s = tuple(getattr(obj, "shape"))
        if len(s) == 2:
            return s
    except Exception:
        pass
    return None

def take_rows_29b(X, idx):
    if sparse.issparse(X):
        return X[idx]
    if isinstance(X, pd.DataFrame):
        return X.iloc[idx]
    return X[idx]

def to_float32_matrix_29b(X):
    if sparse.issparse(X):
        return X.astype(np.float32).tocsr()
    if isinstance(X, pd.DataFrame):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def append_features_29b(X_base, add_df):
    add = add_df.to_numpy(dtype=np.float32)
    if sparse.issparse(X_base):
        return sparse.hstack([X_base, sparse.csr_matrix(add)], format="csr")
    return np.hstack([X_base, add])

def add_n_students_bin_29b(df):
    out = df.copy()
    out["_N_STUDENTS_BIN_TE"] = pd.cut(
        pd.Series(out["N_STUDENTS"]).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"],
    ).astype("string").fillna("<NA>")
    return out

def safe_key_name_29b(cols):
    name = "__".join(cols)
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def make_group_key_29b(df, cols):
    cols = tuple(cols)
    if len(cols) == 1:
        return df[cols[0]].astype("string").fillna("<NA>").astype(str).reset_index(drop=True)

    return (
        df.loc[:, list(cols)]
        .astype("string")
        .fillna("<NA>")
        .astype(str)
        .agg(" || ".join, axis=1)
        .reset_index(drop=True)
    )

def fit_group_stats_29b(keys, y, n_students, alpha, alpha_n):
    yy = np.asarray(y, dtype=np.float64)
    nn = np.asarray(n_students, dtype=np.float64)

    good_n = np.isfinite(nn) & (nn > 0)
    if not good_n.all():
        fill_n = np.nanmedian(nn[good_n]) if good_n.any() else 1.0
        nn = np.where(good_n, nn, fill_n)

    yy_clip = np.clip(yy, 0.0, 100.0)
    proficient_counts = np.rint((yy_clip / 100.0) * nn)
    proficient_counts = np.clip(proficient_counts, 0.0, nn)

    global_mean = float(np.mean(yy))
    global_std = float(np.std(yy, ddof=0))
    global_wmean = float(100.0 * proficient_counts.sum() / max(nn.sum(), 1.0))

    tmp = pd.DataFrame({
        "key": pd.Series(keys).astype(str).to_numpy(),
        "y": yy,
        "n": nn,
        "k": proficient_counts,
    })

    stats = tmp.groupby("key", sort=False).agg(
        cnt=("y", "size"),
        sum_y=("y", "sum"),
        std_y=("y", "std"),
        sum_n=("n", "sum"),
        sum_k=("k", "sum"),
    )

    stats["mean_s"] = (stats["sum_y"] + alpha * global_mean) / (stats["cnt"] + alpha)
    stats["wmean_s"] = 100.0 * (
        stats["sum_k"] + alpha_n * (global_wmean / 100.0)
    ) / (stats["sum_n"] + alpha_n)

    stats["std_y"] = stats["std_y"].fillna(global_std)
    stats["log_count"] = np.log1p(stats["cnt"].astype(float))

    defaults = {
        "global_mean": global_mean,
        "global_wmean": global_wmean,
        "global_std": global_std,
    }

    return stats, defaults

def apply_group_stats_29b(keys_apply, stats, defaults, prefix):
    kk = pd.Series(keys_apply).astype(str).reset_index(drop=True)

    out = pd.DataFrame(index=np.arange(len(kk)))
    out[f"{prefix}_mean"] = kk.map(stats["mean_s"]).fillna(defaults["global_mean"]).astype(np.float32)
    out[f"{prefix}_wmean"] = kk.map(stats["wmean_s"]).fillna(defaults["global_wmean"]).astype(np.float32)
    out[f"{prefix}_log_count"] = kk.map(stats["log_count"]).fillna(0.0).astype(np.float32)
    out[f"{prefix}_std"] = kk.map(stats["std_y"]).fillna(defaults["global_std"]).astype(np.float32)

    return out

def build_te_oof_and_apply_many_29b(raw_fit, y_fit, raw_apply_dict, key_specs, n_splits=5, random_state=9890):
    raw_fit = raw_fit.reset_index(drop=True)
    raw_apply_dict = {
        name: df.reset_index(drop=True)
        for name, df in raw_apply_dict.items()
    }

    y_fit = np.asarray(y_fit, dtype=np.float32).reshape(-1)
    n_fit = pd.Series(raw_fit["N_STUDENTS"]).astype(float).to_numpy()
    n_rows_fit = len(raw_fit)

    kf_inner = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_parts = []
    apply_parts = {name: [] for name in raw_apply_dict.keys()}
    summary_rows = []

    for cols, alpha, alpha_n in key_specs:
        cols = tuple(cols)
        prefix = f"te_{safe_key_name_29b(cols)}_a{int(alpha)}_n{int(alpha_n)}"

        key_fit_all = make_group_key_29b(raw_fit, cols)
        key_apply_all = {
            name: make_group_key_29b(df, cols)
            for name, df in raw_apply_dict.items()
        }

        feature_cols = [
            f"{prefix}_mean",
            f"{prefix}_wmean",
            f"{prefix}_log_count",
            f"{prefix}_std",
        ]

        oof_arr = np.zeros((n_rows_fit, len(feature_cols)), dtype=np.float32)

        for inner_tr_idx, inner_va_idx in kf_inner.split(np.arange(n_rows_fit)):
            stats_fold, defaults_fold = fit_group_stats_29b(
                key_fit_all.iloc[inner_tr_idx],
                y_fit[inner_tr_idx],
                n_fit[inner_tr_idx],
                alpha=alpha,
                alpha_n=alpha_n,
            )

            enc_inner_va = apply_group_stats_29b(
                key_fit_all.iloc[inner_va_idx],
                stats_fold,
                defaults_fold,
                prefix,
            )

            oof_arr[inner_va_idx, :] = enc_inner_va[feature_cols].to_numpy(dtype=np.float32)

        stats_full, defaults_full = fit_group_stats_29b(
            key_fit_all,
            y_fit,
            n_fit,
            alpha=alpha,
            alpha_n=alpha_n,
        )

        oof_parts.append(pd.DataFrame(oof_arr, columns=feature_cols))

        for name, key_apply in key_apply_all.items():
            enc_apply = apply_group_stats_29b(
                key_apply,
                stats_full,
                defaults_full,
                prefix,
            )
            apply_parts[name].append(enc_apply[feature_cols])

        fit_counts = key_fit_all.value_counts(dropna=False)

        summary_rows.append({
            "key": " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
            "alpha": alpha,
            "alpha_n": alpha_n,
            "fit_groups": int(fit_counts.shape[0]),
            "median_fit_count": float(fit_counts.median()),
            "singleton_share": float((fit_counts == 1).mean()),
            "features_added": len(feature_cols),
        })

    oof_features = pd.concat(oof_parts, axis=1)
    apply_features = {
        name: pd.concat(parts, axis=1)
        for name, parts in apply_parts.items()
    }
    summary = pd.DataFrame(summary_rows)

    return oof_features, apply_features, summary

# If the 28B holdout-screen matrices are already warm, use them directly.
existing_screen_ok_29b = (
    (not FORCE_REBUILD_TE_SCREEN)
    and "X_tr_te_screen" in globals()
    and "X_val_te_screen" in globals()
    and _shape2(X_tr_te_screen) is not None
    and _shape2(X_val_te_screen) is not None
    and _shape2(X_tr_te_screen)[1] == 230
    and _shape2(X_val_te_screen)[1] == 230
)

# Recover y_fit_screen/y_val_screen if only the indices exist.
if "y_fit_screen" not in globals() and "tr_idx_te_screen" in globals():
    y_fit_screen = y_arr_29b[np.asarray(tr_idx_te_screen)]
if "y_val_screen" not in globals() and "val_idx_te_screen" in globals():
    y_val_screen = y_arr_29b[np.asarray(val_idx_te_screen)]

if existing_screen_ok_29b and "y_fit_screen" in globals() and "y_val_screen" in globals():
    source_29b = "reused existing X_tr_te_screen / X_val_te_screen"

    X_tr_xgb_te_screen = X_tr_te_screen
    X_val_xgb_te_screen = X_val_te_screen
    y_tr_xgb_te_screen = np.asarray(y_fit_screen, dtype=np.float32).reshape(-1)
    y_val_xgb_te_screen = np.asarray(y_val_screen, dtype=np.float32).reshape(-1)

    if "X_test_te_screen" in globals() and _shape2(X_test_te_screen) is not None:
        X_test_xgb_te_screen = X_test_te_screen

else:
    source_29b = "rebuilt TE holdout-screen matrices"

    raw_train_te_29b = train_full.reset_index(drop=True).copy()
    raw_test_te_29b = test_full.reset_index(drop=True).copy()

    raw_train_te_29b = add_n_students_bin_29b(raw_train_te_29b)
    raw_test_te_29b = add_n_students_bin_29b(raw_test_te_29b)

    candidate_key_specs_29b = [
        (("ASSESSMENT_NAME",), 20.0, 200.0),
        (("SUBGROUP_NAME",), 20.0, 200.0),
        (("ASSESSMENT_NAME", "SUBGROUP_NAME"), 20.0, 200.0),
        (("ASSESSMENT_NAME", "_N_STUDENTS_BIN_TE"), 30.0, 250.0),
        (("ASSESSMENT_NAME", "SUBGROUP_NAME", "_N_STUDENTS_BIN_TE"), 50.0, 300.0),

        (("COUNTY",), 30.0, 250.0),
        (("COUNTY", "SUBGROUP_NAME"), 40.0, 300.0),
        (("COUNTY", "ASSESSMENT_NAME"), 60.0, 400.0),
        (("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"), 100.0, 600.0),

        (("REGION", "ASSESSMENT_NAME"), 40.0, 300.0),
        (("DISTRICT_TYPE", "ASSESSMENT_NAME"), 40.0, 300.0),

        (("DISTRICT",), 60.0, 400.0),
        (("DISTRICT", "SUBGROUP_NAME"), 90.0, 500.0),
        (("DISTRICT", "ASSESSMENT_NAME"), 140.0, 700.0),

        (("SCHOOL",), 100.0, 600.0),
        (("SCHOOL", "SUBGROUP_NAME"), 140.0, 800.0),
        (("SCHOOL", "ASSESSMENT_NAME"), 220.0, 1000.0),
    ]

    te_key_specs_29b = []
    for cols, alpha, alpha_n in candidate_key_specs_29b:
        if all(c in raw_train_te_29b.columns for c in cols) and all(c in raw_test_te_29b.columns for c in cols):
            te_key_specs_29b.append((cols, alpha, alpha_n))

    if len(te_key_specs_29b) == 0:
        raise ValueError("No usable TE key specs found.")

    all_idx_29b = np.arange(len(y_arr_29b))
    tr_idx_te_screen, val_idx_te_screen = train_test_split(
        all_idx_29b,
        test_size=0.20,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    raw_fit_screen_29b = raw_train_te_29b.iloc[tr_idx_te_screen].reset_index(drop=True)
    raw_val_screen_29b = raw_train_te_29b.iloc[val_idx_te_screen].reset_index(drop=True)

    y_fit_screen = y_arr_29b[tr_idx_te_screen]
    y_val_screen = y_arr_29b[val_idx_te_screen]

    print("Rebuilding TE features. This is lighter than model training but may take a bit.")

    te_tr_screen_29b, apply_dict_29b, te_screen_key_summary_29b = build_te_oof_and_apply_many_29b(
        raw_fit=raw_fit_screen_29b,
        y_fit=y_fit_screen,
        raw_apply_dict={
            "valid": raw_val_screen_29b,
            "test": raw_test_te_29b,
        },
        key_specs=te_key_specs_29b,
        n_splits=5,
        random_state=RANDOM_STATE,
    )

    te_val_screen_29b = apply_dict_29b["valid"]
    te_test_screen_29b = apply_dict_29b["test"]

    X_tr_base_screen_29b = to_float32_matrix_29b(take_rows_29b(X_train_proc_model, tr_idx_te_screen))
    X_val_base_screen_29b = to_float32_matrix_29b(take_rows_29b(X_train_proc_model, val_idx_te_screen))
    X_test_base_screen_29b = to_float32_matrix_29b(X_test_proc_model)

    X_tr_te_screen = append_features_29b(X_tr_base_screen_29b, te_tr_screen_29b)
    X_val_te_screen = append_features_29b(X_val_base_screen_29b, te_val_screen_29b)
    X_test_te_screen = append_features_29b(X_test_base_screen_29b, te_test_screen_29b)

    X_tr_xgb_te_screen = X_tr_te_screen
    X_val_xgb_te_screen = X_val_te_screen
    X_test_xgb_te_screen = X_test_te_screen

    y_tr_xgb_te_screen = np.asarray(y_fit_screen, dtype=np.float32).reshape(-1)
    y_val_xgb_te_screen = np.asarray(y_val_screen, dtype=np.float32).reshape(-1)

    te_key_specs = te_key_specs_29b
    te_screen_key_summary_29b.to_csv("model_results/te_screen_29b_key_summary.csv", index=False)

# Final checks.
train_shape_29b = _shape2(X_tr_xgb_te_screen)
valid_shape_29b = _shape2(X_val_xgb_te_screen)

TE_SCREEN_29B_PASS = bool(
    train_shape_29b is not None
    and valid_shape_29b is not None
    and train_shape_29b[1] == 230
    and valid_shape_29b[1] == 230
    and len(y_tr_xgb_te_screen) == train_shape_29b[0]
    and len(y_val_xgb_te_screen) == valid_shape_29b[0]
)

print("\nSource:", source_29b)
print("TE_SCREEN_29B_PASS =", TE_SCREEN_29B_PASS)
print("X_tr_xgb_te_screen shape: ", train_shape_29b)
print("X_val_xgb_te_screen shape:", valid_shape_29b)
print("y_tr_xgb_te_screen shape: ", np.asarray(y_tr_xgb_te_screen).shape)
print("y_val_xgb_te_screen shape:", np.asarray(y_val_xgb_te_screen).shape)

if "X_test_xgb_te_screen" in globals():
    print("X_test_xgb_te_screen shape:", _shape2(X_test_xgb_te_screen))
else:
    print("X_test_xgb_te_screen: not built; fine for holdout screening.")

print("\nBase feature count:", X_train_proc_model.shape[1])
print("Expected TE feature count:", 230 - X_train_proc_model.shape[1])
print("Current public anchor to beat: submission_blend_te_verified_noresid_weighted.csv, public MSE 69.689")

gc.collect()

29B replacement: TE holdout-screen matrix setup
------------------------------------------------

Source: reused existing X_tr_te_screen / X_val_te_screen
TE_SCREEN_29B_PASS = True
X_tr_xgb_te_screen shape:  (115936, 230)
X_val_xgb_te_screen shape: (28985, 230)
y_tr_xgb_te_screen shape:  (115936,)
y_val_xgb_te_screen shape: (28985,)
X_test_xgb_te_screen: not built; fine for holdout screening.

Base feature count: 162
Expected TE feature count: 68
Current public anchor to beat: submission_blend_te_verified_noresid_weighted.csv, public MSE 69.689


3711

# XGBOOST

coming down next

In [71]:
# 29C. XGBoost holdout screen on TE-augmented 230-feature matrix
# This is a model-family screen only, not a full OOF/submission run.

import os
import time
import gc
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.metrics import mean_squared_error

import xgboost as xgb
from xgboost import XGBRegressor

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

print("29C. XGBoost TE holdout screen")
print("------------------------------")
print("xgboost version:", xgb.__version__)

required_29C = [
    "X_tr_xgb_te_screen",
    "X_val_xgb_te_screen",
    "y_tr_xgb_te_screen",
    "y_val_xgb_te_screen",
]

missing_29C = [name for name in required_29C if name not in globals()]
if missing_29C:
    raise ValueError(f"Missing required 29C objects: {missing_29C}. Run 29B replacement first.")

def as_xgb_matrix_29C(X):
    if sparse.issparse(X):
        return X.astype(np.float32).tocsr()
    if isinstance(X, pd.DataFrame):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def mse_clip_29C(y_true, pred):
    pred_clip = np.clip(np.asarray(pred, dtype=np.float64), 0.0, 100.0)
    return float(mean_squared_error(np.asarray(y_true, dtype=np.float64), pred_clip))

def pred_summary_29C(pred):
    s = pd.Series(np.clip(np.asarray(pred, dtype=np.float64), 0.0, 100.0))
    return s.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

X_tr_29C = as_xgb_matrix_29C(X_tr_xgb_te_screen)
X_val_29C = as_xgb_matrix_29C(X_val_xgb_te_screen)
y_tr_29C = np.asarray(y_tr_xgb_te_screen, dtype=np.float32).reshape(-1)
y_val_29C = np.asarray(y_val_xgb_te_screen, dtype=np.float32).reshape(-1)

print("Train matrix:", X_tr_29C.shape)
print("Valid matrix:", X_val_29C.shape)
print("Train target:", y_tr_29C.shape)
print("Valid target:", y_val_29C.shape)

if X_tr_29C.shape[1] != 230 or X_val_29C.shape[1] != 230:
    raise ValueError("Expected 230 TE-augmented features. Check 29B output.")

# Optional same-split benchmarks from saved full OOF files, if val_idx_te_screen exists.
benchmark_rows_29C = []

def load_oof_prediction_col_29C(path):
    df = pd.read_csv(path)
    bad_names = {"ASSESSMENT_ID", "PERCENT_PROFICIENT", "target", "y", "y_train"}
    numeric_cols = [
        c for c in df.columns
        if c not in bad_names and pd.api.types.is_numeric_dtype(df[c])
    ]
    if len(numeric_cols) == 0:
        raise ValueError(f"No usable numeric prediction column found in {path}")
    # Prefer OOF/pred-looking columns, otherwise use the last numeric non-ID column.
    preferred = [
        c for c in numeric_cols
        if any(tok in c.lower() for tok in ["oof", "pred", "prediction", "blend"])
    ]
    col = preferred[-1] if preferred else numeric_cols[-1]
    return df[col].to_numpy(dtype=np.float64), col

if "val_idx_te_screen" in globals():
    val_idx_29C = np.asarray(val_idx_te_screen)
    possible_benchmarks_29C = {
        "pure_te_lgbm_oof": "model_results/oof_lgbm_te_base_5fold_oof_v1.csv",
        "safe_te_blend_oof": "model_results/oof_blend_te_verified_noresid_weighted.csv",
    }

    print("\nSame-validation-split saved OOF benchmarks")
    print("-------------------------------------------")

    for bench_name, bench_path in possible_benchmarks_29C.items():
        if os.path.exists(bench_path):
            try:
                pred_full, pred_col = load_oof_prediction_col_29C(bench_path)
                if len(pred_full) == len(np.asarray(y_train)):
                    pred_val = pred_full[val_idx_29C]
                    bench_mse = mse_clip_29C(y_val_29C, pred_val)
                    benchmark_rows_29C.append({
                        "name": bench_name,
                        "path": bench_path,
                        "column": pred_col,
                        "val_mse_clip": bench_mse,
                    })
                    print(f"{bench_name}: val clipped MSE = {bench_mse:.6f} using column {pred_col}")
                else:
                    print(f"{bench_name}: skipped; length {len(pred_full)} does not match y_train.")
            except Exception as e:
                print(f"{bench_name}: skipped due to error: {e}")
        else:
            print(f"{bench_name}: file not found:", bench_path)
else:
    print("\nval_idx_te_screen not found, so same-split OOF benchmarks are skipped.")

# Conservative first XGBoost screen.
# These are deliberately not huge. If one is promising, we promote to OOF later.
xgb_configs_29C = [
    {
        "name": "xgb_te_h01_depth6_lr03_lam10",
        "n_estimators": 9000,
        "learning_rate": 0.03,
        "max_depth": 6,
        "min_child_weight": 30.0,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
        "early_stopping_rounds": 500,
    },
    {
        "name": "xgb_te_h02_depth7_lr02_lam20",
        "n_estimators": 12000,
        "learning_rate": 0.02,
        "max_depth": 7,
        "min_child_weight": 50.0,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_lambda": 20.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
        "early_stopping_rounds": 700,
    },
]

screen_rows_29C = []
best_val_mse_29C = np.inf
best_xgb_te_screen_model_29C = None
best_xgb_te_screen_name_29C = None
best_xgb_te_val_pred_29C = None

for cfg in xgb_configs_29C:
    cfg = cfg.copy()
    name = cfg.pop("name")
    early_rounds = cfg.pop("early_stopping_rounds")

    print("\n" + "=" * 80)
    print("Training:", name)
    print("Config:", cfg)
    print("early_stopping_rounds:", early_rounds)

    start = time.time()

    early_stop = xgb.callback.EarlyStopping(
        rounds=early_rounds,
        save_best=True,
        maximize=False,
    )

    model = XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        device="cpu",
        random_state=RANDOM_STATE,
        n_jobs=2,
        verbosity=1,
        callbacks=[early_stop],
        **cfg,
    )

    model.fit(
        X_tr_29C,
        y_tr_29C,
        eval_set=[(X_val_29C, y_val_29C)],
        verbose=500,
    )

    elapsed = time.time() - start

    pred_tr = model.predict(X_tr_29C)
    pred_val = model.predict(X_val_29C)

    train_mse_raw = float(mean_squared_error(y_tr_29C, pred_tr))
    train_mse_clip = mse_clip_29C(y_tr_29C, pred_tr)
    val_mse_raw = float(mean_squared_error(y_val_29C, pred_val))
    val_mse_clip = mse_clip_29C(y_val_29C, pred_val)

    try:
        best_iteration = int(model.best_iteration)
    except Exception:
        best_iteration = None

    try:
        boosted_rounds = int(model.get_booster().num_boosted_rounds())
    except Exception:
        boosted_rounds = None

    row = {
        "name": name,
        "train_mse_raw": train_mse_raw,
        "train_mse_clip": train_mse_clip,
        "val_mse_raw": val_mse_raw,
        "val_mse_clip": val_mse_clip,
        "best_iteration": best_iteration,
        "boosted_rounds": boosted_rounds,
        "elapsed_seconds": elapsed,
        **cfg,
        "early_stopping_rounds": early_rounds,
    }

    screen_rows_29C.append(row)

    print("\nResult:", name)
    print("train MSE raw:     ", train_mse_raw)
    print("train MSE clipped: ", train_mse_clip)
    print("valid MSE raw:     ", val_mse_raw)
    print("valid MSE clipped: ", val_mse_clip)
    print("best_iteration:    ", best_iteration)
    print("boosted_rounds:    ", boosted_rounds)
    print("elapsed seconds:   ", elapsed)

    print("\nValidation prediction summary, clipped")
    print(pred_summary_29C(pred_val))

    if val_mse_clip < best_val_mse_29C:
        if best_xgb_te_screen_model_29C is not None:
            del best_xgb_te_screen_model_29C
            gc.collect()

        best_val_mse_29C = val_mse_clip
        best_xgb_te_screen_model_29C = model
        best_xgb_te_screen_name_29C = name
        best_xgb_te_val_pred_29C = np.clip(np.asarray(pred_val, dtype=np.float64), 0.0, 100.0)

        model_path = f"model_results/{name}_holdout_model.json"
        best_xgb_te_screen_model_29C.save_model(model_path)
        print("New best XGBoost holdout model saved:", model_path)
    else:
        del model
        gc.collect()

screen_df_29C = pd.DataFrame(screen_rows_29C).sort_values("val_mse_clip").reset_index(drop=True)
screen_path_29C = "model_results/xgb_te_holdout_screen_29C.csv"
screen_df_29C.to_csv(screen_path_29C, index=False)

print("\n" + "=" * 80)
print("29C XGBoost holdout screen summary")
print("----------------------------------")
print(screen_df_29C.to_string(index=False))
print("\nSaved screen table:", screen_path_29C)

if benchmark_rows_29C:
    bench_df_29C = pd.DataFrame(benchmark_rows_29C).sort_values("val_mse_clip").reset_index(drop=True)
    bench_path_29C = "model_results/xgb_te_holdout_benchmarks_29C.csv"
    bench_df_29C.to_csv(bench_path_29C, index=False)

    print("\nSame-split OOF benchmarks")
    print("-------------------------")
    print(bench_df_29C.to_string(index=False))
    print("Saved benchmark table:", bench_path_29C)

print("\nBest XGBoost holdout model")
print("--------------------------")
print("best_xgb_te_screen_name_29C:", best_xgb_te_screen_name_29C)
print("best_val_mse_29C:", best_val_mse_29C)

if best_xgb_te_val_pred_29C is not None:
    val_pred_path_29C = "model_results/xgb_te_holdout_best_valpred_29C.csv"

    if "val_idx_te_screen" in globals():
        row_index = np.asarray(val_idx_te_screen)
    else:
        row_index = np.arange(len(y_val_29C))

    pd.DataFrame({
        "row_index": row_index,
        "y_true": y_val_29C,
        "xgb_te_holdout_pred": best_xgb_te_val_pred_29C,
    }).to_csv(val_pred_path_29C, index=False)

    print("Saved best validation predictions:", val_pred_path_29C)

gc.collect()

29C. XGBoost TE holdout screen
------------------------------
xgboost version: 3.2.0
Train matrix: (115936, 230)
Valid matrix: (28985, 230)
Train target: (115936,)
Valid target: (28985,)

Same-validation-split saved OOF benchmarks
-------------------------------------------
pure_te_lgbm_oof: val clipped MSE = 83.846176 using column pred_clipped
safe_te_blend_oof: val clipped MSE = 79.297003 using column pred_clipped

Training: xgb_te_h01_depth6_lr03_lam10
Config: {'n_estimators': 9000, 'learning_rate': 0.03, 'max_depth': 6, 'min_child_weight': 30.0, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'gamma': 0.0}
early_stopping_rounds: 500
[0]	validation_0-rmse:25.89449
[500]	validation_0-rmse:9.75548
[1000]	validation_0-rmse:9.60005
[1500]	validation_0-rmse:9.53053
[2000]	validation_0-rmse:9.50221
[2500]	validation_0-rmse:9.47756
[3000]	validation_0-rmse:9.47066
[3328]	validation_0-rmse:9.46927

Result: xgb_te_h01_depth6_lr03_lam10
train MSE raw:      5

0

## Checkpoint: XGBoost Screen on Target-Encoded Feature Matrix

We ran a controlled XGBoost holdout screen on the same 230-feature target/statistical-encoding matrix used for the TE LightGBM branch.

**Feature matrix used:**

- Training screen matrix: `X_tr_xgb_te_screen`, shape `(115936, 230)`
- Validation screen matrix: `X_val_xgb_te_screen`, shape `(28985, 230)`
- Base features: `162`
- Added target/statistical encoding features: `68`

This confirmed that we were testing XGBoost on the correct TE-augmented feature space rather than accidentally reverting to the older 162-feature base matrix.

### Same-split benchmark results

Saved OOF benchmark predictions on the same validation split gave:

- Pure TE LightGBM validation MSE: `83.846176`
- Repaired safe TE blend validation MSE: `79.297003`

### XGBoost holdout results

Two XGBoost configurations were screened:

1. `xgb_te_h01_depth6_lr03_lam10`
   - Validation clipped MSE: `89.280096`
   - Best iteration: `2829`

2. `xgb_te_h02_depth7_lr02_lam20`
   - Validation clipped MSE: `86.896025`
   - Best iteration: `5018`

The better XGBoost configuration was:

`xgb_te_h02_depth7_lr02_lam20`

with validation clipped MSE:

`86.896025`

### Interpretation

XGBoost trained successfully and gave a valid model-family screen, but it was weaker than both TE LightGBM and the repaired safe TE blend on the same validation split.

Compared with same-split benchmarks:

- XGBoost was about `3.05` MSE worse than pure TE LightGBM.
- XGBoost was about `7.60` MSE worse than the repaired safe TE blend.

Therefore, XGBoost is **not promoted to a full 5-fold OOF artifact right now**. This model family has been fairly tested on the current TE numeric feature space, but the first controlled screen does not justify a long OOF run or Kaggle submission.

Next branch: run a comparable numeric-only CatBoost holdout screen on the same 230-feature TE matrix. If numeric CatBoost is also clearly weaker, then future CatBoost work should only be considered if we build a true raw-categorical CatBoost branch rather than using only the already-numeric TE matrix.

# CATBOOST

coming down next

In [72]:
# 29D. CatBoost holdout screen on TE-augmented 230-feature matrix
# Numeric-only CatBoost screen using the same TE holdout split as 29C.
# This is not a full OOF/submission run.

import os
import time
import gc
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.metrics import mean_squared_error

import catboost
from catboost import CatBoostRegressor

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

print("29D. CatBoost TE holdout screen")
print("-------------------------------")
print("catboost version:", catboost.__version__)

required_29D = [
    "X_tr_xgb_te_screen",
    "X_val_xgb_te_screen",
    "y_tr_xgb_te_screen",
    "y_val_xgb_te_screen",
]

missing_29D = [name for name in required_29D if name not in globals()]
if missing_29D:
    raise ValueError(f"Missing required 29D objects: {missing_29D}. Run 29B replacement first.")

def as_cat_matrix_29D(X):
    # CatBoost can handle some sparse inputs, but dense float32 is simpler and safer here.
    # 115936 x 230 is manageable as float32.
    if sparse.issparse(X):
        return X.astype(np.float32).toarray()
    if isinstance(X, pd.DataFrame):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def mse_clip_29D(y_true, pred):
    pred_clip = np.clip(np.asarray(pred, dtype=np.float64), 0.0, 100.0)
    return float(mean_squared_error(np.asarray(y_true, dtype=np.float64), pred_clip))

def pred_summary_29D(pred):
    s = pd.Series(np.clip(np.asarray(pred, dtype=np.float64), 0.0, 100.0))
    return s.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

X_tr_29D = as_cat_matrix_29D(X_tr_xgb_te_screen)
X_val_29D = as_cat_matrix_29D(X_val_xgb_te_screen)
y_tr_29D = np.asarray(y_tr_xgb_te_screen, dtype=np.float32).reshape(-1)
y_val_29D = np.asarray(y_val_xgb_te_screen, dtype=np.float32).reshape(-1)

print("Train matrix:", X_tr_29D.shape, X_tr_29D.dtype)
print("Valid matrix:", X_val_29D.shape, X_val_29D.dtype)
print("Train target:", y_tr_29D.shape)
print("Valid target:", y_val_29D.shape)

if X_tr_29D.shape[1] != 230 or X_val_29D.shape[1] != 230:
    raise ValueError("Expected 230 TE-augmented features. Check 29B output.")

# Same-split saved OOF benchmarks.
benchmark_rows_29D = []

def load_oof_prediction_col_29D(path):
    df = pd.read_csv(path)
    bad_names = {"ASSESSMENT_ID", "PERCENT_PROFICIENT", "target", "y", "y_train"}
    numeric_cols = [
        c for c in df.columns
        if c not in bad_names and pd.api.types.is_numeric_dtype(df[c])
    ]
    if len(numeric_cols) == 0:
        raise ValueError(f"No usable numeric prediction column found in {path}")

    preferred = [
        c for c in numeric_cols
        if any(tok in c.lower() for tok in ["oof", "pred", "prediction", "blend"])
    ]
    col = preferred[-1] if preferred else numeric_cols[-1]
    return df[col].to_numpy(dtype=np.float64), col

if "val_idx_te_screen" in globals():
    val_idx_29D = np.asarray(val_idx_te_screen)
    possible_benchmarks_29D = {
        "pure_te_lgbm_oof": "model_results/oof_lgbm_te_base_5fold_oof_v1.csv",
        "safe_te_blend_oof": "model_results/oof_blend_te_verified_noresid_weighted.csv",
    }

    print("\nSame-validation-split saved OOF benchmarks")
    print("-------------------------------------------")

    for bench_name, bench_path in possible_benchmarks_29D.items():
        if os.path.exists(bench_path):
            try:
                pred_full, pred_col = load_oof_prediction_col_29D(bench_path)
                if len(pred_full) == len(np.asarray(y_train)):
                    pred_val = pred_full[val_idx_29D]
                    bench_mse = mse_clip_29D(y_val_29D, pred_val)
                    benchmark_rows_29D.append({
                        "name": bench_name,
                        "path": bench_path,
                        "column": pred_col,
                        "val_mse_clip": bench_mse,
                    })
                    print(f"{bench_name}: val clipped MSE = {bench_mse:.6f} using column {pred_col}")
                else:
                    print(f"{bench_name}: skipped; length {len(pred_full)} does not match y_train.")
            except Exception as e:
                print(f"{bench_name}: skipped due to error: {e}")
        else:
            print(f"{bench_name}: file not found:", bench_path)
else:
    print("\nval_idx_te_screen not found, so same-split OOF benchmarks are skipped.")

cat_configs_29D = [
    {
        "name": "cat_te_h01_depth6_lr03_l2_20",
        "iterations": 8000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 20.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.5,
        "od_wait": 500,
    },
    {
        "name": "cat_te_h02_depth7_lr02_l2_40",
        "iterations": 10000,
        "learning_rate": 0.02,
        "depth": 7,
        "l2_leaf_reg": 40.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.25,
        "od_wait": 700,
    },
]

screen_rows_29D = []
best_val_mse_29D = np.inf
best_cat_te_screen_model_29D = None
best_cat_te_screen_name_29D = None
best_cat_te_val_pred_29D = None

for cfg in cat_configs_29D:
    cfg = cfg.copy()
    name = cfg.pop("name")
    od_wait = cfg.pop("od_wait")

    print("\n" + "=" * 80)
    print("Training:", name)
    print("Config:", cfg)
    print("od_wait:", od_wait)

    start = time.time()

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        bootstrap_type="Bayesian",
        random_seed=RANDOM_STATE,
        thread_count=2,
        task_type="CPU",
        od_type="Iter",
        od_wait=od_wait,
        use_best_model=True,
        allow_writing_files=False,
        verbose=500,
        **cfg,
    )

    model.fit(
        X_tr_29D,
        y_tr_29D,
        eval_set=(X_val_29D, y_val_29D),
    )

    elapsed = time.time() - start

    pred_tr = model.predict(X_tr_29D)
    pred_val = model.predict(X_val_29D)

    train_mse_raw = float(mean_squared_error(y_tr_29D, pred_tr))
    train_mse_clip = mse_clip_29D(y_tr_29D, pred_tr)
    val_mse_raw = float(mean_squared_error(y_val_29D, pred_val))
    val_mse_clip = mse_clip_29D(y_val_29D, pred_val)

    try:
        best_iteration = int(model.get_best_iteration())
    except Exception:
        best_iteration = None

    row = {
        "name": name,
        "train_mse_raw": train_mse_raw,
        "train_mse_clip": train_mse_clip,
        "val_mse_raw": val_mse_raw,
        "val_mse_clip": val_mse_clip,
        "best_iteration": best_iteration,
        "elapsed_seconds": elapsed,
        **cfg,
        "od_wait": od_wait,
    }

    screen_rows_29D.append(row)

    print("\nResult:", name)
    print("train MSE raw:     ", train_mse_raw)
    print("train MSE clipped: ", train_mse_clip)
    print("valid MSE raw:     ", val_mse_raw)
    print("valid MSE clipped: ", val_mse_clip)
    print("best_iteration:    ", best_iteration)
    print("elapsed seconds:   ", elapsed)

    print("\nValidation prediction summary, clipped")
    print(pred_summary_29D(pred_val))

    if val_mse_clip < best_val_mse_29D:
        if best_cat_te_screen_model_29D is not None:
            del best_cat_te_screen_model_29D
            gc.collect()

        best_val_mse_29D = val_mse_clip
        best_cat_te_screen_model_29D = model
        best_cat_te_screen_name_29D = name
        best_cat_te_val_pred_29D = np.clip(np.asarray(pred_val, dtype=np.float64), 0.0, 100.0)

        model_path = f"model_results/{name}_holdout_model.cbm"
        best_cat_te_screen_model_29D.save_model(model_path)
        print("New best CatBoost holdout model saved:", model_path)
    else:
        del model
        gc.collect()

screen_df_29D = pd.DataFrame(screen_rows_29D).sort_values("val_mse_clip").reset_index(drop=True)
screen_path_29D = "model_results/cat_te_holdout_screen_29D.csv"
screen_df_29D.to_csv(screen_path_29D, index=False)

print("\n" + "=" * 80)
print("29D CatBoost holdout screen summary")
print("-----------------------------------")
print(screen_df_29D.to_string(index=False))
print("\nSaved screen table:", screen_path_29D)

if benchmark_rows_29D:
    bench_df_29D = pd.DataFrame(benchmark_rows_29D).sort_values("val_mse_clip").reset_index(drop=True)
    bench_path_29D = "model_results/cat_te_holdout_benchmarks_29D.csv"
    bench_df_29D.to_csv(bench_path_29D, index=False)

    print("\nSame-split OOF benchmarks")
    print("-------------------------")
    print(bench_df_29D.to_string(index=False))
    print("Saved benchmark table:", bench_path_29D)

print("\nBest CatBoost holdout model")
print("---------------------------")
print("best_cat_te_screen_name_29D:", best_cat_te_screen_name_29D)
print("best_val_mse_29D:", best_val_mse_29D)

if best_cat_te_val_pred_29D is not None:
    val_pred_path_29D = "model_results/cat_te_holdout_best_valpred_29D.csv"

    if "val_idx_te_screen" in globals():
        row_index = np.asarray(val_idx_te_screen)
    else:
        row_index = np.arange(len(y_val_29D))

    pd.DataFrame({
        "row_index": row_index,
        "y_true": y_val_29D,
        "cat_te_holdout_pred": best_cat_te_val_pred_29D,
    }).to_csv(val_pred_path_29D, index=False)

    print("Saved best validation predictions:", val_pred_path_29D)

gc.collect()

29D. CatBoost TE holdout screen
-------------------------------
catboost version: 1.2.10
Train matrix: (115936, 230) float32
Valid matrix: (28985, 230) float32
Train target: (115936,)
Valid target: (28985,)

Same-validation-split saved OOF benchmarks
-------------------------------------------
pure_te_lgbm_oof: val clipped MSE = 83.846176 using column pred_clipped
safe_te_blend_oof: val clipped MSE = 79.297003 using column pred_clipped

Training: cat_te_h01_depth6_lr03_l2_20
Config: {'iterations': 8000, 'learning_rate': 0.03, 'depth': 6, 'l2_leaf_reg': 20.0, 'random_strength': 1.0, 'bagging_temperature': 0.5}
od_wait: 500
0:	learn: 25.8881588	test: 25.9614949	best: 25.9614949 (0)	total: 96.5ms	remaining: 12m 51s
500:	learn: 10.5694822	test: 10.3054297	best: 10.3054297 (500)	total: 13.2s	remaining: 3m 17s
1000:	learn: 10.0108638	test: 10.0703606	best: 10.0670181 (981)	total: 26.2s	remaining: 3m 3s
1500:	learn: 9.7374500	test: 9.9895960	best: 9.9895960 (1500)	total: 39.1s	remaining: 2m 4

0

## Checkpoint: Numeric CatBoost Screen and Next CatBoost Direction

We ran a numeric-only CatBoost holdout screen on the same 230-feature target/statistical-encoding matrix used for the XGBoost screen.

### Same-split benchmarks

- Pure TE LightGBM validation MSE: `83.846176`
- Repaired safe TE blend validation MSE: `79.297003`

### Numeric CatBoost results

Two CatBoost configurations were screened:

1. `cat_te_h01_depth6_lr03_l2_20`
   - Validation clipped MSE: `94.287162`
   - Best iteration: `7999`
   - Hit the iteration cap.

2. `cat_te_h02_depth7_lr02_l2_40`
   - Validation clipped MSE: `92.964362`
   - Best iteration: `9999`
   - Hit the iteration cap.

The better numeric-only CatBoost model was:

`cat_te_h02_depth7_lr02_l2_40`

with validation clipped MSE:

`92.964362`

### Interpretation

Numeric-only CatBoost trained successfully, and both configurations were still improving at their iteration caps. Therefore, CatBoost should not be discarded as a model family.

However, this numeric-only screen was much weaker than the TE LightGBM and the repaired safe TE blend on the same validation split. The gap is too large to justify an immediate full OOF run using only the already-numeric TE matrix.

The more meaningful CatBoost branch is a **raw-categorical CatBoost branch**, where CatBoost can use high-cardinality categorical variables directly, especially:

- `SCHOOL`
- `DISTRICT`
- `COUNTY`
- `ASSESSMENT_NAME`
- `SUBGROUP_NAME`
- `REGION`
- `DISTRICT_TYPE`

Current decision:

- Do not promote numeric-only CatBoost to full OOF yet.
- Do not submit any numeric CatBoost candidate.
- Keep CatBoost active as a future model family.
- Next step: run a raw-categorical CatBoost feasibility diagnostic before training.

In [73]:
# 29E. Raw-categorical CatBoost feasibility diagnostic
# This does NOT train a model.
# Goal: prepare and inspect a CatBoost-ready raw feature matrix using categorical columns directly.

import os
import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from catboost import Pool

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = "PERCENT_PROFICIENT"

print("29E. Raw-categorical CatBoost feasibility diagnostic")
print("----------------------------------------------------")

required_29E = [
    "train_full",
    "test_full",
    "y_train",
    "test_ids",
]

missing_29E = [name for name in required_29E if name not in globals()]
if missing_29E:
    raise ValueError(f"Missing required objects: {missing_29E}")

raw_train_29E = train_full.reset_index(drop=True).copy()
raw_test_29E = test_full.reset_index(drop=True).copy()
y_29E = np.asarray(y_train, dtype=np.float32).reshape(-1)

print("raw_train_29E shape:", raw_train_29E.shape)
print("raw_test_29E shape: ", raw_test_29E.shape)
print("y_29E shape:        ", y_29E.shape)

if len(raw_train_29E) != len(y_29E):
    raise ValueError("train_full length does not match y_train length.")

# Reuse the same holdout split as the TE/XGBoost/CatBoost screens if available.
if "tr_idx_te_screen" in globals() and "val_idx_te_screen" in globals():
    tr_idx_cat_29E = np.asarray(tr_idx_te_screen)
    val_idx_cat_29E = np.asarray(val_idx_te_screen)
    split_source_29E = "reused tr_idx_te_screen / val_idx_te_screen"
else:
    all_idx_29E = np.arange(len(y_29E))
    tr_idx_cat_29E, val_idx_cat_29E = train_test_split(
        all_idx_29E,
        test_size=0.20,
        random_state=RANDOM_STATE,
        shuffle=True,
    )
    split_source_29E = "rebuilt 80/20 split with RANDOM_STATE"

print("\nSplit source:", split_source_29E)
print("train screen rows:", len(tr_idx_cat_29E))
print("valid screen rows:", len(val_idx_cat_29E))

# Use columns present in both train and test.
common_cols_29E = [c for c in raw_train_29E.columns if c in raw_test_29E.columns]

# Drop row identifiers and the target.
# ASSESSMENT_ID is the submission ID, not a modeling signal.
drop_exact_29E = {
    TARGET_COL,
    "ASSESSMENT_ID",
}

feature_cols_29E = [
    c for c in common_cols_29E
    if c not in drop_exact_29E
]

if len(feature_cols_29E) == 0:
    raise ValueError("No usable common feature columns found.")

# Preferred categorical columns if present.
preferred_cat_cols_29E = [
    "SCHOOL",
    "DISTRICT",
    "COUNTY",
    "ASSESSMENT_NAME",
    "SUBGROUP_NAME",
    "REGION",
    "DISTRICT_TYPE",
]

# Also include any object/string/category/bool columns as categoricals.
dtype_cat_cols_29E = []
for c in feature_cols_29E:
    dt = raw_train_29E[c].dtype
    if (
        pd.api.types.is_object_dtype(dt)
        or pd.api.types.is_string_dtype(dt)
        or pd.api.types.is_categorical_dtype(dt)
        or pd.api.types.is_bool_dtype(dt)
    ):
        dtype_cat_cols_29E.append(c)

cat_cols_29E = []
for c in preferred_cat_cols_29E + dtype_cat_cols_29E:
    if c in feature_cols_29E and c not in cat_cols_29E:
        cat_cols_29E.append(c)

num_cols_29E = [c for c in feature_cols_29E if c not in cat_cols_29E]

print("\nFeature column counts")
print("---------------------")
print("Total common usable features:", len(feature_cols_29E))
print("Categorical features:", len(cat_cols_29E))
print("Numeric/raw continuous features:", len(num_cols_29E))

print("\nCategorical columns")
print("-------------------")
print(cat_cols_29E)

print("\nFirst 30 numeric columns")
print("------------------------")
print(num_cols_29E[:30])

def build_catboost_frame_29E(df, feature_cols, cat_cols, num_cols, num_medians=None):
    out = pd.DataFrame(index=np.arange(len(df)))

    # Categorical features: CatBoost requires consistent string-like values.
    for c in cat_cols:
        out[c] = (
            df[c]
            .astype("string")
            .fillna("<NA>")
            .astype(str)
        )

    # Numeric features: coerce to numeric and median-impute using training-screen medians.
    if num_medians is None:
        num_medians = {}

    for c in num_cols:
        vals = pd.to_numeric(df[c], errors="coerce")
        if c in num_medians:
            fill_value = num_medians[c]
        else:
            fill_value = vals.median()
            if not np.isfinite(fill_value):
                fill_value = 0.0
            num_medians[c] = float(fill_value)

        out[c] = vals.fillna(fill_value).astype(np.float32)

    # Keep stable feature ordering.
    out = out[cat_cols + num_cols]

    return out, num_medians

raw_fit_29E = raw_train_29E.iloc[tr_idx_cat_29E].reset_index(drop=True)
raw_val_29E = raw_train_29E.iloc[val_idx_cat_29E].reset_index(drop=True)
raw_test_for_cat_29E = raw_test_29E.reset_index(drop=True)

X_cat_tr_29E, num_medians_29E = build_catboost_frame_29E(
    raw_fit_29E,
    feature_cols_29E,
    cat_cols_29E,
    num_cols_29E,
    num_medians=None,
)

X_cat_val_29E, _ = build_catboost_frame_29E(
    raw_val_29E,
    feature_cols_29E,
    cat_cols_29E,
    num_cols_29E,
    num_medians=num_medians_29E,
)

X_cat_test_29E, _ = build_catboost_frame_29E(
    raw_test_for_cat_29E,
    feature_cols_29E,
    cat_cols_29E,
    num_cols_29E,
    num_medians=num_medians_29E,
)

y_cat_tr_29E = y_29E[tr_idx_cat_29E]
y_cat_val_29E = y_29E[val_idx_cat_29E]

cat_feature_indices_29E = [X_cat_tr_29E.columns.get_loc(c) for c in cat_cols_29E]

print("\nBuilt CatBoost raw frames")
print("-------------------------")
print("X_cat_tr_29E:  ", X_cat_tr_29E.shape)
print("X_cat_val_29E: ", X_cat_val_29E.shape)
print("X_cat_test_29E:", X_cat_test_29E.shape)
print("y_cat_tr_29E:  ", y_cat_tr_29E.shape)
print("y_cat_val_29E: ", y_cat_val_29E.shape)
print("cat_feature_indices_29E length:", len(cat_feature_indices_29E))

# Cardinality and coverage diagnostics.
coverage_rows_29E = []

for c in cat_cols_29E:
    fit_vals = set(X_cat_tr_29E[c].astype(str).unique())
    val_vals = X_cat_val_29E[c].astype(str)
    test_vals = X_cat_test_29E[c].astype(str)

    coverage_rows_29E.append({
        "column": c,
        "train_unique": int(len(fit_vals)),
        "valid_unique": int(val_vals.nunique()),
        "test_unique": int(test_vals.nunique()),
        "valid_seen_share": float(val_vals.isin(fit_vals).mean()),
        "test_seen_share": float(test_vals.isin(fit_vals).mean()),
        "top_train_values": ", ".join(
            X_cat_tr_29E[c].value_counts(dropna=False).head(5).index.astype(str).tolist()
        ),
    })

coverage_df_29E = pd.DataFrame(coverage_rows_29E)

if not coverage_df_29E.empty:
    coverage_df_29E = coverage_df_29E.sort_values(
        ["train_unique", "column"],
        ascending=[False, True],
    ).reset_index(drop=True)

    print("\nCategorical cardinality and coverage")
    print("------------------------------------")
    print(coverage_df_29E.to_string(index=False, max_rows=50))

    coverage_path_29E = "model_results/catboost_rawcat_29E_coverage.csv"
    coverage_df_29E.to_csv(coverage_path_29E, index=False)
    print("\nSaved coverage diagnostics:", coverage_path_29E)

# Numeric diagnostic.
num_diag_rows_29E = []
for c in num_cols_29E:
    vals = X_cat_tr_29E[c]
    num_diag_rows_29E.append({
        "column": c,
        "train_mean": float(vals.mean()),
        "train_std": float(vals.std()),
        "train_min": float(vals.min()),
        "train_max": float(vals.max()),
        "median_impute": float(num_medians_29E[c]),
    })

num_diag_df_29E = pd.DataFrame(num_diag_rows_29E)
num_diag_path_29E = "model_results/catboost_rawcat_29E_numeric_diag.csv"
num_diag_df_29E.to_csv(num_diag_path_29E, index=False)
print("Saved numeric diagnostics:", num_diag_path_29E)

# Smoke-test Pool construction on a small sample.
smoke_n_29E = min(2000, len(X_cat_tr_29E))
try:
    smoke_pool_29E = Pool(
        X_cat_tr_29E.iloc[:smoke_n_29E],
        y_cat_tr_29E[:smoke_n_29E],
        cat_features=cat_feature_indices_29E,
    )
    CATBOOST_RAWCAT_29E_POOL_PASS = True
except Exception as e:
    CATBOOST_RAWCAT_29E_POOL_PASS = False
    print("\nPool construction error:")
    print(repr(e))

print("\nPool smoke test")
print("---------------")
print("CATBOOST_RAWCAT_29E_POOL_PASS =", CATBOOST_RAWCAT_29E_POOL_PASS)
print("smoke_n:", smoke_n_29E)

# Save compact metadata for later.
meta_29E = {
    "split_source": split_source_29E,
    "n_train_screen": int(len(tr_idx_cat_29E)),
    "n_valid_screen": int(len(val_idx_cat_29E)),
    "n_test": int(len(X_cat_test_29E)),
    "n_features": int(len(feature_cols_29E)),
    "n_cat_features": int(len(cat_cols_29E)),
    "n_num_features": int(len(num_cols_29E)),
    "pool_pass": bool(CATBOOST_RAWCAT_29E_POOL_PASS),
}

pd.DataFrame([meta_29E]).to_csv("model_results/catboost_rawcat_29E_meta.csv", index=False)

print("\nMetadata")
print("--------")
print(pd.DataFrame([meta_29E]).to_string(index=False))

gc.collect()

29E. Raw-categorical CatBoost feasibility diagnostic
----------------------------------------------------
raw_train_29E shape: (144921, 62)
raw_test_29E shape:  (48307, 61)
y_29E shape:         (144921,)

Split source: reused tr_idx_te_screen / val_idx_te_screen
train screen rows: 115936
valid screen rows: 28985

Feature column counts
---------------------
Total common usable features: 60
Categorical features: 7
Numeric/raw continuous features: 53

Categorical columns
-------------------
['SCHOOL', 'DISTRICT', 'COUNTY', 'ASSESSMENT_NAME', 'SUBGROUP_NAME', 'REGION', 'DISTRICT_TYPE']

First 30 numeric columns
------------------------
['N_STUDENTS', 'ATTENDANCE_RATE', 'LANGUAGE_ARTS_AVERAGE_CLASS_SIZE', 'MATHEMATICS_AVERAGE_CLASS_SIZE', 'SCIENCE_AVERAGE_CLASS_SIZE', 'HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE', 'GRADE_1_AVERAGE_CLASS_SIZE', 'GRADE_2_AVERAGE_CLASS_SIZE', 'KINDERGARTEN_AVERAGE_CLASS_SIZE', 'PERCENT_FREE_LUNCH', 'PERCENT_REDUCED_LUNCH', 'NUMBER_OF_TEACHERS', 'NUMBER

0

In [74]:
# ============================================================
# 29F. Raw-categorical CatBoost holdout screen
# ============================================================
# Purpose:
#   - Train CatBoost directly on raw categorical columns from 29E.
#   - Compare against same-validation-split TE LightGBM and safe TE blend.
#   - Decide whether raw-categorical CatBoost deserves full OOF promotion.
#
# This cell does NOT create a Kaggle submission.
# ============================================================

import os
import time
import gc
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from IPython.display import display

import catboost
from catboost import CatBoostRegressor, Pool

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

print("29F. Raw-categorical CatBoost holdout screen")
print("--------------------------------------------")
print("catboost version:", catboost.__version__)

required_29F = [
    "X_cat_tr_29E",
    "X_cat_val_29E",
    "y_cat_tr_29E",
    "y_cat_val_29E",
    "cat_feature_indices_29E",
]

missing_29F = [name for name in required_29F if name not in globals()]
if missing_29F:
    raise ValueError(f"Missing required 29F objects: {missing_29F}. Run 29E first.")

print("\nInput shapes")
print("------------")
print("X_cat_tr_29E: ", X_cat_tr_29E.shape)
print("X_cat_val_29E:", X_cat_val_29E.shape)
print("y_cat_tr_29E: ", np.asarray(y_cat_tr_29E).shape)
print("y_cat_val_29E:", np.asarray(y_cat_val_29E).shape)
print("cat features: ", len(cat_feature_indices_29E), cat_feature_indices_29E)

y_tr_29F = np.asarray(y_cat_tr_29E, dtype=np.float32).reshape(-1)
y_val_29F = np.asarray(y_cat_val_29E, dtype=np.float32).reshape(-1)

def mse_clip_29F(y_true, pred):
    pred_clip = np.clip(np.asarray(pred, dtype=np.float64), 0.0, 100.0)
    return float(mean_squared_error(np.asarray(y_true, dtype=np.float64), pred_clip))

def pred_summary_29F(pred):
    s = pd.Series(np.clip(np.asarray(pred, dtype=np.float64), 0.0, 100.0))
    return s.describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99])

train_pool_29F = Pool(
    data=X_cat_tr_29E,
    label=y_tr_29F,
    cat_features=cat_feature_indices_29E,
)

valid_pool_29F = Pool(
    data=X_cat_val_29E,
    label=y_val_29F,
    cat_features=cat_feature_indices_29E,
)

# ------------------------------------------------------------
# Same-validation-split benchmark check
# ------------------------------------------------------------

benchmark_specs_29F = [
    {
        "name": "pure_te_lgbm_oof",
        "path": "model_results/oof_lgbm_te_base_5fold_oof_v1.csv",
    },
    {
        "name": "safe_te_blend_oof",
        "path": "model_results/oof_blend_te_verified_noresid_weighted.csv",
    },
]

bench_rows_29F = []

print("\nSame-validation-split saved OOF benchmarks")
print("------------------------------------------")

if "val_idx_cat_29E" in globals():
    val_idx_for_bench_29F = np.asarray(val_idx_cat_29E)

    for spec in benchmark_specs_29F:
        path = spec["path"]
        name = spec["name"]

        if not os.path.exists(path):
            print(f"{name}: file not found -> {path}")
            continue

        df_bench = pd.read_csv(path)

        if "pred_clipped" in df_bench.columns:
            pred_col = "pred_clipped"
        elif "prediction" in df_bench.columns:
            pred_col = "prediction"
        elif "PERCENT_PROFICIENT" in df_bench.columns:
            pred_col = "PERCENT_PROFICIENT"
        else:
            numeric_cols = df_bench.select_dtypes(include=[np.number]).columns.tolist()
            numeric_cols = [c for c in numeric_cols if c not in ["ASSESSMENT_ID"]]
            if not numeric_cols:
                print(f"{name}: no usable prediction column found.")
                continue
            pred_col = numeric_cols[-1]

        pred_full = np.asarray(df_bench[pred_col], dtype=np.float64)

        if len(pred_full) != len(globals().get("y_train", pred_full)):
            print(f"{name}: skipped; length {len(pred_full)} does not match full y_train length.")
            continue

        pred_val = pred_full[val_idx_for_bench_29F]
        val_mse = mse_clip_29F(y_val_29F, pred_val)

        bench_rows_29F.append({
            "name": name,
            "path": path,
            "column": pred_col,
            "val_mse_clip": val_mse,
        })

        print(f"{name}: val clipped MSE = {val_mse:.6f} using column {pred_col}")

else:
    print("val_idx_cat_29E not found, so benchmark alignment is skipped.")

bench_29F = pd.DataFrame(bench_rows_29F)
if len(bench_29F):
    bench_29F = bench_29F.sort_values("val_mse_clip").reset_index(drop=True)
    bench_29F.to_csv("model_results/cat_raw_holdout_benchmarks_29F.csv", index=False)

# ------------------------------------------------------------
# Raw-categorical CatBoost configurations
# ------------------------------------------------------------

cat_raw_configs_29F = [
    {
        "name": "cat_raw_h01_depth6_lr03_l2_30_ctr1",
        "iterations": 9000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 30.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.50,
        "one_hot_max_size": 10,
        "max_ctr_complexity": 1,
        "od_wait": 700,
    },
    {
        "name": "cat_raw_h02_depth7_lr02_l2_50_ctr2",
        "iterations": 12000,
        "learning_rate": 0.02,
        "depth": 7,
        "l2_leaf_reg": 50.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.25,
        "one_hot_max_size": 10,
        "max_ctr_complexity": 2,
        "od_wait": 900,
    },
]

screen_rows_29F = []
best_val_mse_29F = np.inf
best_cat_raw_screen_model_29F = None
best_cat_raw_screen_name_29F = None
best_cat_raw_val_pred_29F = None

for cfg in cat_raw_configs_29F:
    cfg = cfg.copy()
    name = cfg.pop("name")
    od_wait = cfg.pop("od_wait")

    print("\n" + "=" * 80)
    print("Training:", name)
    print("Config:", cfg)
    print("od_wait:", od_wait)

    start = time.time()

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        bootstrap_type="Bayesian",
        random_seed=RANDOM_STATE,
        thread_count=2,
        task_type="CPU",
        od_type="Iter",
        od_wait=od_wait,
        use_best_model=True,
        allow_writing_files=False,
        verbose=500,
        **cfg,
    )

    model.fit(
        train_pool_29F,
        eval_set=valid_pool_29F,
    )

    elapsed = time.time() - start

    pred_tr = model.predict(train_pool_29F)
    pred_val = model.predict(valid_pool_29F)

    train_mse_raw = float(mean_squared_error(y_tr_29F, pred_tr))
    train_mse_clip = mse_clip_29F(y_tr_29F, pred_tr)

    val_mse_raw = float(mean_squared_error(y_val_29F, pred_val))
    val_mse_clip = mse_clip_29F(y_val_29F, pred_val)

    best_iter = model.get_best_iteration()
    if best_iter is None:
        best_iter = getattr(model, "tree_count_", np.nan)

    row = {
        "name": name,
        "train_mse_raw": train_mse_raw,
        "train_mse_clip": train_mse_clip,
        "val_mse_raw": val_mse_raw,
        "val_mse_clip": val_mse_clip,
        "best_iteration": best_iter,
        "tree_count": getattr(model, "tree_count_", np.nan),
        "elapsed_seconds": elapsed,
        **cfg,
        "od_wait": od_wait,
    }

    screen_rows_29F.append(row)

    print("\nResult:", name)
    print("train MSE raw:     ", train_mse_raw)
    print("train MSE clipped: ", train_mse_clip)
    print("valid MSE raw:     ", val_mse_raw)
    print("valid MSE clipped: ", val_mse_clip)
    print("best_iteration:    ", best_iter)
    print("tree_count:        ", getattr(model, "tree_count_", None))
    print("elapsed seconds:   ", elapsed)

    print("\nValidation prediction summary, clipped")
    print(pred_summary_29F(pred_val))

    if val_mse_clip < best_val_mse_29F:
        best_val_mse_29F = val_mse_clip
        best_cat_raw_screen_name_29F = name
        best_cat_raw_val_pred_29F = np.asarray(pred_val, dtype=np.float64)

        # Replace old best object to avoid keeping unnecessary models.
        if best_cat_raw_screen_model_29F is not None:
            try:
                del best_cat_raw_screen_model_29F
                gc.collect()
            except Exception:
                pass

        best_cat_raw_screen_model_29F = model

        model_path = f"model_results/{name}_rawcat_holdout_model.cbm"
        best_cat_raw_screen_model_29F.save_model(model_path)
        print("New best raw-categorical CatBoost model saved:", model_path)

    else:
        del model

    del pred_tr, pred_val
    gc.collect()

screen_29F = pd.DataFrame(screen_rows_29F).sort_values("val_mse_clip").reset_index(drop=True)
screen_path_29F = "model_results/cat_raw_holdout_screen_29F.csv"
screen_29F.to_csv(screen_path_29F, index=False)

print("\n" + "=" * 80)
print("29F raw-categorical CatBoost holdout screen summary")
print("---------------------------------------------------")
display(screen_29F)
print("Saved screen table:", screen_path_29F)

if len(bench_29F):
    print("\nSame-split OOF benchmark table")
    print("------------------------------")
    display(bench_29F)
    print("Saved benchmark table: model_results/cat_raw_holdout_benchmarks_29F.csv")

print("\nBest raw-categorical CatBoost holdout model")
print("-------------------------------------------")
print("best_cat_raw_screen_name_29F:", best_cat_raw_screen_name_29F)
print("best_val_mse_29F:", best_val_mse_29F)

if best_cat_raw_val_pred_29F is not None:
    val_indices_out_29F = (
        np.asarray(val_idx_cat_29E)
        if "val_idx_cat_29E" in globals()
        else np.arange(len(y_val_29F))
    )

    best_valpred_29F = pd.DataFrame({
        "row_index": val_indices_out_29F,
        "y_true": y_val_29F,
        "pred_raw": best_cat_raw_val_pred_29F,
        "pred_clipped": np.clip(best_cat_raw_val_pred_29F, 0.0, 100.0),
    })

    valpred_path_29F = "model_results/cat_raw_holdout_best_valpred_29F.csv"
    best_valpred_29F.to_csv(valpred_path_29F, index=False)
    print("Saved best validation predictions:", valpred_path_29F)

# Lightweight decision hint. Final decision comes after reviewing output.
if len(bench_29F):
    best_benchmark_29F = float(bench_29F["val_mse_clip"].min())
    pure_te_rows = bench_29F.loc[bench_29F["name"] == "pure_te_lgbm_oof", "val_mse_clip"]

    print("\nDecision hint")
    print("-------------")
    print("Best raw-cat CatBoost val MSE:", best_val_mse_29F)
    print("Best benchmark val MSE:       ", best_benchmark_29F)

    if len(pure_te_rows):
        pure_te_mse_29F = float(pure_te_rows.iloc[0])
        print("Pure TE LightGBM val MSE:     ", pure_te_mse_29F)

        if best_val_mse_29F < pure_te_mse_29F:
            print("Raw-cat CatBoost beat pure TE LightGBM on this split -> potential OOF candidate.")
        elif best_val_mse_29F < pure_te_mse_29F + 3.0:
            print("Raw-cat CatBoost is close enough to keep as possible blend diversity.")
        else:
            print("Raw-cat CatBoost is clearly weaker on this first screen; likely no OOF promotion yet.")

29F. Raw-categorical CatBoost holdout screen
--------------------------------------------
catboost version: 1.2.10

Input shapes
------------
X_cat_tr_29E:  (115936, 60)
X_cat_val_29E: (28985, 60)
y_cat_tr_29E:  (115936,)
y_cat_val_29E: (28985,)
cat features:  7 [0, 1, 2, 3, 4, 5, 6]

Same-validation-split saved OOF benchmarks
------------------------------------------
pure_te_lgbm_oof: val clipped MSE = 83.846176 using column pred_clipped
safe_te_blend_oof: val clipped MSE = 79.297003 using column pred_clipped

Training: cat_raw_h01_depth6_lr03_l2_30_ctr1
Config: {'iterations': 9000, 'learning_rate': 0.03, 'depth': 6, 'l2_leaf_reg': 30.0, 'random_strength': 1.0, 'bagging_temperature': 0.5, 'one_hot_max_size': 10, 'max_ctr_complexity': 1}
od_wait: 700
0:	learn: 26.0623913	test: 26.1668066	best: 26.1668066 (0)	total: 76.3ms	remaining: 11m 26s
500:	learn: 14.2259378	test: 13.8386917	best: 13.8386917 (500)	total: 10.9s	remaining: 3m 5s
1000:	learn: 13.6938560	test: 13.4276557	best: 13.427

,name,train_mse_raw,train_mse_clip,val_mse_raw,val_mse_clip,best_iteration,tree_count,elapsed_seconds,iterations,learning_rate,depth,l2_leaf_reg,random_strength,bagging_temperature,one_hot_max_size,max_ctr_complexity,od_wait
0,cat_raw_h02_depth7_lr02_l2_50_ctr2,77.266669,77.124415,116.220999,116.085417,11999,12000,545.766839,12000,0.02,7,50.0,1.0,0.25,10,2,900
1,cat_raw_h01_depth6_lr03_l2_30_ctr1,119.468128,119.292806,145.846843,145.656654,8999,9000,206.696925,9000,0.03,6,30.0,1.0,0.50,10,1,700


Saved screen table: model_results/cat_raw_holdout_screen_29F.csv

Same-split OOF benchmark table
------------------------------


,name,path,column,val_mse_clip
0,safe_te_blend_oof,model_results/oof_blend_te_verified_noresid_we...,pred_clipped,79.297003
1,pure_te_lgbm_oof,model_results/oof_lgbm_te_base_5fold_oof_v1.csv,pred_clipped,83.846176


Saved benchmark table: model_results/cat_raw_holdout_benchmarks_29F.csv

Best raw-categorical CatBoost holdout model
-------------------------------------------
best_cat_raw_screen_name_29F: cat_raw_h02_depth7_lr02_l2_50_ctr2
best_val_mse_29F: 116.08541653604
Saved best validation predictions: model_results/cat_raw_holdout_best_valpred_29F.csv

Decision hint
-------------
Best raw-cat CatBoost val MSE: 116.08541653604
Best benchmark val MSE:        79.29700321638711
Pure TE LightGBM val MSE:      83.84617589529451
Raw-cat CatBoost is clearly weaker on this first screen; likely no OOF promotion yet.


In [75]:
# ============================================================
# 29G. Raw-categorical CatBoost blend-diversity diagnostic
# ============================================================
# Purpose:
#   - Do NOT train anything new.
#   - Check whether the weak raw-cat CatBoost validation prediction
#     adds any useful diversity to the TE LightGBM / safe TE blend
#     on the exact same validation split.
#   - Decide whether raw-cat CatBoost deserves OOF promotion.
#
# This cell does NOT create a submission.
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from IPython.display import display

os.makedirs("model_results", exist_ok=True)

print("29G. Raw-categorical CatBoost blend-diversity diagnostic")
print("--------------------------------------------------------")

required_29G = ["y_cat_val_29E", "val_idx_cat_29E"]
missing_29G = [name for name in required_29G if name not in globals()]
if missing_29G:
    raise ValueError(f"Missing required objects: {missing_29G}. Run 29E/29F context first.")

y_val_29G = np.asarray(y_cat_val_29E, dtype=np.float64).reshape(-1)
val_idx_29G = np.asarray(val_idx_cat_29E)

print("Validation rows:", len(y_val_29G))
print("Validation index rows:", len(val_idx_29G))

def mse_clip_29G(y_true, pred):
    pred_clip = np.clip(np.asarray(pred, dtype=np.float64), 0.0, 100.0)
    return float(mean_squared_error(np.asarray(y_true, dtype=np.float64), pred_clip))

def find_pred_col_29G(df):
    preferred = ["pred_raw", "pred_clipped", "prediction", "PERCENT_PROFICIENT"]
    for c in preferred:
        if c in df.columns:
            return c

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["ASSESSMENT_ID", "row_index", "y_true"]]
    if not numeric_cols:
        raise ValueError("No usable numeric prediction column found.")
    return numeric_cols[-1]

def load_oof_on_val_29G(path, val_idx, expected_full_len=None):
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    pred_col = find_pred_col_29G(df)
    pred_full = np.asarray(df[pred_col], dtype=np.float64)

    if expected_full_len is not None and len(pred_full) != expected_full_len:
        raise ValueError(
            f"{path} length {len(pred_full)} != expected_full_len {expected_full_len}"
        )

    return pred_full[val_idx], pred_col

# ------------------------------------------------------------
# Load raw-cat CatBoost validation prediction from memory or file
# ------------------------------------------------------------

if "best_cat_raw_val_pred_29F" in globals() and best_cat_raw_val_pred_29F is not None:
    pred_rawcat_29G = np.asarray(best_cat_raw_val_pred_29F, dtype=np.float64).reshape(-1)
    rawcat_source_29G = "memory: best_cat_raw_val_pred_29F"
else:
    rawcat_path_29G = "model_results/cat_raw_holdout_best_valpred_29F.csv"
    if not os.path.exists(rawcat_path_29G):
        raise FileNotFoundError(
            "Could not find best_cat_raw_val_pred_29F in memory or "
            f"{rawcat_path_29G} on disk."
        )

    df_rawcat_29G = pd.read_csv(rawcat_path_29G)
    raw_pred_col_29G = "pred_raw" if "pred_raw" in df_rawcat_29G.columns else find_pred_col_29G(df_rawcat_29G)

    if "row_index" not in df_rawcat_29G.columns:
        raise ValueError("Raw-cat validation file needs row_index for alignment.")

    raw_lookup_29G = df_rawcat_29G.set_index("row_index")[raw_pred_col_29G]
    pred_rawcat_29G = raw_lookup_29G.loc[val_idx_29G].to_numpy(dtype=np.float64)
    rawcat_source_29G = f"disk: {rawcat_path_29G}, column={raw_pred_col_29G}"

if len(pred_rawcat_29G) != len(y_val_29G):
    raise ValueError(
        f"Raw-cat prediction length {len(pred_rawcat_29G)} != validation length {len(y_val_29G)}"
    )

print("\nRaw-cat source:", rawcat_source_29G)
print("Raw-cat validation clipped MSE:", mse_clip_29G(y_val_29G, pred_rawcat_29G))

# ------------------------------------------------------------
# Load same-split benchmark predictions
# ------------------------------------------------------------

expected_full_len_29G = len(y_train) if "y_train" in globals() else None

benchmark_paths_29G = {
    "pure_te_lgbm": "model_results/oof_lgbm_te_base_5fold_oof_v1.csv",
    "safe_te_blend": "model_results/oof_blend_te_verified_noresid_weighted.csv",
}

preds_29G = {
    "rawcat_cb": pred_rawcat_29G,
}

load_rows_29G = []

for name, path in benchmark_paths_29G.items():
    if not os.path.exists(path):
        print(f"Missing benchmark file, skipped: {name} -> {path}")
        continue

    pred_val, pred_col = load_oof_on_val_29G(
        path=path,
        val_idx=val_idx_29G,
        expected_full_len=expected_full_len_29G,
    )

    preds_29G[name] = pred_val

    load_rows_29G.append({
        "name": name,
        "path": path,
        "column": pred_col,
        "val_mse_clip": mse_clip_29G(y_val_29G, pred_val),
    })

bench_loaded_29G = pd.DataFrame(load_rows_29G).sort_values("val_mse_clip").reset_index(drop=True)

print("\nLoaded same-split benchmarks")
print("----------------------------")
display(bench_loaded_29G)

# ------------------------------------------------------------
# Component MSE + residual correlations
# ------------------------------------------------------------

component_rows_29G = []
resid_dict_29G = {}

for name, pred in preds_29G.items():
    pred_clip = np.clip(np.asarray(pred, dtype=np.float64), 0.0, 100.0)
    component_rows_29G.append({
        "component": name,
        "val_mse_clip": mse_clip_29G(y_val_29G, pred),
        "pred_mean_clip": float(np.mean(pred_clip)),
        "pred_std_clip": float(np.std(pred_clip)),
        "pred_min_clip": float(np.min(pred_clip)),
        "pred_max_clip": float(np.max(pred_clip)),
    })
    resid_dict_29G[name] = y_val_29G - pred_clip

component_29G = pd.DataFrame(component_rows_29G).sort_values("val_mse_clip").reset_index(drop=True)

print("\nComponent summary")
print("-----------------")
display(component_29G)

resid_corr_29G = pd.DataFrame(resid_dict_29G).corr()

print("\nResidual correlation matrix")
print("---------------------------")
display(resid_corr_29G)

# ------------------------------------------------------------
# Convex blend screen: benchmark + raw-cat CatBoost
# ------------------------------------------------------------

blend_rows_29G = []

for base_name in ["safe_te_blend", "pure_te_lgbm"]:
    if base_name not in preds_29G:
        continue

    base_pred = np.asarray(preds_29G[base_name], dtype=np.float64)
    raw_pred = np.asarray(preds_29G["rawcat_cb"], dtype=np.float64)

    for w_rawcat in np.round(np.linspace(0.0, 0.50, 501), 4):
        pred_blend = (1.0 - w_rawcat) * base_pred + w_rawcat * raw_pred
        blend_rows_29G.append({
            "base": base_name,
            "w_rawcat": float(w_rawcat),
            "w_base": float(1.0 - w_rawcat),
            "val_mse_clip": mse_clip_29G(y_val_29G, pred_blend),
        })

blend_screen_29G = pd.DataFrame(blend_rows_29G)

if len(blend_screen_29G):
    blend_screen_29G = blend_screen_29G.sort_values("val_mse_clip").reset_index(drop=True)
    blend_path_29G = "model_results/cat_raw_blend_diversity_29G.csv"
    blend_screen_29G.to_csv(blend_path_29G, index=False)

    print("\nBest raw-cat blend rows")
    print("-----------------------")
    display(blend_screen_29G.head(20))
    print("Saved:", blend_path_29G)

    # Decision variables for later notebook cells
    best_row_29G = blend_screen_29G.iloc[0].to_dict()
    CAT_RAW_BEST_BLEND_29G = best_row_29G

    if "safe_te_blend" in preds_29G:
        safe_base_mse_29G = mse_clip_29G(y_val_29G, preds_29G["safe_te_blend"])
        best_safe_rows_29G = blend_screen_29G.loc[
            blend_screen_29G["base"] == "safe_te_blend"
        ].sort_values("val_mse_clip")

        best_safe_29G = best_safe_rows_29G.iloc[0].to_dict()
        safe_gain_29G = safe_base_mse_29G - float(best_safe_29G["val_mse_clip"])

        CAT_RAW_SAFE_BLEND_GAIN_29G = safe_gain_29G
        CAT_RAW_SAFE_BLEND_WEIGHT_29G = float(best_safe_29G["w_rawcat"])

        print("\nDecision check against safe TE blend")
        print("------------------------------------")
        print("safe_te_blend base MSE:", safe_base_mse_29G)
        print("best safe+rawcat MSE: ", float(best_safe_29G["val_mse_clip"]))
        print("best rawcat weight:   ", CAT_RAW_SAFE_BLEND_WEIGHT_29G)
        print("gain vs safe blend:   ", CAT_RAW_SAFE_BLEND_GAIN_29G)

        CAT_RAW_OOF_PROMOTION_RECOMMENDED_29G = (
            (CAT_RAW_SAFE_BLEND_WEIGHT_29G >= 0.05) and
            (CAT_RAW_SAFE_BLEND_GAIN_29G >= 0.30)
        )

        print("\nCAT_RAW_OOF_PROMOTION_RECOMMENDED_29G:", CAT_RAW_OOF_PROMOTION_RECOMMENDED_29G)

        if CAT_RAW_OOF_PROMOTION_RECOMMENDED_29G:
            print("Raw-cat CatBoost shows enough same-split blend value to discuss OOF promotion.")
        else:
            print("Raw-cat CatBoost does not show enough same-split blend value; demote this branch.")

else:
    print("No benchmark blend rows were produced.")

29G. Raw-categorical CatBoost blend-diversity diagnostic
--------------------------------------------------------
Validation rows: 28985
Validation index rows: 28985

Raw-cat source: memory: best_cat_raw_val_pred_29F
Raw-cat validation clipped MSE: 116.08541653604

Loaded same-split benchmarks
----------------------------


,name,path,column,val_mse_clip
0,safe_te_blend,model_results/oof_blend_te_verified_noresid_we...,pred_clipped,79.297003
1,pure_te_lgbm,model_results/oof_lgbm_te_base_5fold_oof_v1.csv,pred_raw,83.846176



Component summary
-----------------


,component,val_mse_clip,pred_mean_clip,pred_std_clip,pred_min_clip,pred_max_clip
0,safe_te_blend,79.297003,54.628159,24.621564,0.012534,100.0
1,pure_te_lgbm,83.846176,54.934316,24.891149,0.000000,100.0
2,rawcat_cb,116.085417,54.109661,24.016293,0.000000,100.0



Residual correlation matrix
---------------------------


,rawcat_cb,pure_te_lgbm,safe_te_blend
rawcat_cb,1.000000,0.811789,0.841301
pure_te_lgbm,0.811789,1.000000,0.972172
safe_te_blend,0.841301,0.972172,1.000000



Best raw-cat blend rows
-----------------------


,base,w_rawcat,w_base,val_mse_clip
0,safe_te_blend,0.000,1.000,79.297003
1,safe_te_blend,0.001,0.999,79.299575
2,safe_te_blend,0.002,0.998,79.302215
3,safe_te_blend,0.003,0.997,79.304924
4,safe_te_blend,0.004,0.996,79.307702
5,safe_te_blend,0.005,0.995,79.310548
6,safe_te_blend,0.006,0.994,79.313464
7,safe_te_blend,0.007,0.993,79.316448
8,safe_te_blend,0.008,0.992,79.319500
9,safe_te_blend,0.009,0.991,79.322622


Saved: model_results/cat_raw_blend_diversity_29G.csv

Decision check against safe TE blend
------------------------------------
safe_te_blend base MSE: 79.29700321638711
best safe+rawcat MSE:  79.29700321638711
best rawcat weight:    0.0
gain vs safe blend:    0.0

CAT_RAW_OOF_PROMOTION_RECOMMENDED_29G: False
Raw-cat CatBoost does not show enough same-split blend value; demote this branch.


## Checkpoint: Raw-Categorical CatBoost Demoted

We tested whether CatBoost could recover useful signal by using raw categorical variables directly, including `SCHOOL`, `DISTRICT`, `COUNTY`, `ASSESSMENT_NAME`, `SUBGROUP_NAME`, `REGION`, and `DISTRICT_TYPE`.

The best raw-categorical CatBoost holdout model was:

`cat_raw_h02_depth7_lr02_l2_50_ctr2`

with validation clipped MSE:

`116.085417`

On the same validation slice, the benchmarks were:

- Pure TE LightGBM: `83.846176`
- Repaired safe TE blend: `79.297003`

A follow-up blend-diversity diagnostic showed that raw-categorical CatBoost received optimal weight `0.0` when blended with the repaired safe TE blend. The gain versus the safe TE blend was exactly `0.0`.

Decision:

- Do not promote raw-categorical CatBoost to full OOF.
- Do not submit any CatBoost candidate.
- Demote CatBoost for now.
- Pivot back to target/statistical-encoding and grouped historical signal, which has been the strongest modeling direction so far.

In [76]:
# ============================================================
# 30A-lite. Leakage-safe grouped historical mean diagnostic
# ============================================================
# Purpose:
#   - Build simple OOF target-mean predictors for important groups.
#   - Avoid leakage by computing each validation fold's group means
#     using only the other folds.
#   - Rank grouped historical signals by OOF clipped MSE.
#
# This cell does NOT create test predictions or a submission.
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from IPython.display import display

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

print("30A-lite. Leakage-safe grouped historical mean diagnostic")
print("--------------------------------------------------------")

required = ["train_full", "y_train"]
missing = [x for x in required if x not in globals()]
if missing:
    raise ValueError(f"Missing required objects: {missing}")

raw_train = train_full.reset_index(drop=True).copy()
y = np.asarray(y_train, dtype=float).reshape(-1)

if len(raw_train) != len(y):
    raise ValueError(f"train_full rows {len(raw_train)} != y_train length {len(y)}")

print("raw_train shape:", raw_train.shape)
print("y shape:", y.shape)
print("global target mean:", y.mean())


def mse_clip(y_true, pred):
    pred = np.clip(np.asarray(pred, dtype=float), 0, 100)
    return float(mean_squared_error(y_true, pred))


def make_key(df, cols):
    """Create one string key from one or more categorical columns."""
    key = df[cols[0]].astype("string").fillna("<NA>")
    for c in cols[1:]:
        key = key + "||" + df[c].astype("string").fillna("<NA>")
    return key


def oof_group_mean(df, y, cols, alpha=50, n_splits=5):
    """
    Cross-fitted smoothed target mean.

    For each fold:
      - fit group means on training folds only
      - apply them to held-out fold
      - unseen groups fall back to that fold's training mean
    """
    oof = np.zeros(len(y), dtype=float)
    seen = np.zeros(len(y), dtype=bool)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    for fit_idx, val_idx in kf.split(df):
        df_fit = df.iloc[fit_idx]
        df_val = df.iloc[val_idx]

        y_fit = y[fit_idx]
        fold_mean = float(y_fit.mean())

        fit_key = make_key(df_fit, cols)
        val_key = make_key(df_val, cols)

        tmp = pd.DataFrame({
            "key": fit_key,
            "y": y_fit,
        })

        stats = tmp.groupby("key")["y"].agg(["count", "sum"])
        smooth_mean = (stats["sum"] + alpha * fold_mean) / (stats["count"] + alpha)

        mapped = val_key.map(smooth_mean)

        seen[val_idx] = mapped.notna().to_numpy()
        oof[val_idx] = mapped.fillna(fold_mean).to_numpy(dtype=float)

    return np.clip(oof, 0, 100), seen


group_specs = [
    (["ASSESSMENT_NAME"], 20),
    (["SUBGROUP_NAME"], 20),
    (["ASSESSMENT_NAME", "SUBGROUP_NAME"], 40),

    (["COUNTY"], 50),
    (["COUNTY", "ASSESSMENT_NAME"], 80),
    (["COUNTY", "SUBGROUP_NAME"], 80),
    (["COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"], 120),

    (["DISTRICT"], 80),
    (["DISTRICT", "ASSESSMENT_NAME"], 150),
    (["DISTRICT", "SUBGROUP_NAME"], 150),
    (["DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME"], 250),

    (["SCHOOL"], 120),
    (["SCHOOL", "ASSESSMENT_NAME"], 250),
    (["SCHOOL", "SUBGROUP_NAME"], 250),
    (["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"], 400),
]

results = []
oof_preds = pd.DataFrame({"row_index": np.arange(len(y))})

for cols, alpha in group_specs:
    missing_cols = [c for c in cols if c not in raw_train.columns]
    if missing_cols:
        print("Skipping missing columns:", cols)
        continue

    name = "__".join(cols)
    print("Building:", name, "| alpha:", alpha)

    pred, seen = oof_group_mean(
        df=raw_train,
        y=y,
        cols=cols,
        alpha=alpha,
        n_splits=5,
    )

    oof_preds[name] = pred

    results.append({
        "group": name,
        "alpha": alpha,
        "oof_mse_clip": mse_clip(y, pred),
        "seen_rate": float(seen.mean()),
        "pred_mean": float(pred.mean()),
        "pred_std": float(pred.std()),
    })

summary_30A = (
    pd.DataFrame(results)
    .sort_values("oof_mse_clip")
    .reset_index(drop=True)
)

print("\nGrouped historical mean summary")
print("-------------------------------")
display(summary_30A)

oof_path = "model_results/oof_hist_group_means_30A_lite.csv"
summary_path = "model_results/hist_group_means_summary_30A_lite.csv"

oof_preds.to_csv(oof_path, index=False)
summary_30A.to_csv(summary_path, index=False)

print("\nSaved:")
print(oof_path)
print(summary_path)

print("\nBest grouped predictor:")
print(summary_30A.iloc[0])

30A-lite. Leakage-safe grouped historical mean diagnostic
--------------------------------------------------------
raw_train shape: (144921, 62)
y shape: (144921,)
global target mean: 54.18352067678252
Building: ASSESSMENT_NAME | alpha: 20
Building: SUBGROUP_NAME | alpha: 20
Building: ASSESSMENT_NAME__SUBGROUP_NAME | alpha: 40
Building: COUNTY | alpha: 50
Building: COUNTY__ASSESSMENT_NAME | alpha: 80
Building: COUNTY__SUBGROUP_NAME | alpha: 80
Building: COUNTY__ASSESSMENT_NAME__SUBGROUP_NAME | alpha: 120
Building: DISTRICT | alpha: 80
Building: DISTRICT__ASSESSMENT_NAME | alpha: 150
Building: DISTRICT__SUBGROUP_NAME | alpha: 150
Building: DISTRICT__ASSESSMENT_NAME__SUBGROUP_NAME | alpha: 250
Building: SCHOOL | alpha: 120
Building: SCHOOL__ASSESSMENT_NAME | alpha: 250
Building: SCHOOL__SUBGROUP_NAME | alpha: 250
Building: SCHOOL__ASSESSMENT_NAME__SUBGROUP_NAME | alpha: 400

Grouped historical mean summary
-------------------------------


,group,alpha,oof_mse_clip,seen_rate,pred_mean,pred_std
0,ASSESSMENT_NAME__SUBGROUP_NAME,40,518.259587,1.000000,54.116538,12.843963
1,COUNTY__ASSESSMENT_NAME,80,527.892294,0.999738,53.708100,8.973764
2,ASSESSMENT_NAME,20,542.528852,1.000000,54.165847,12.412135
3,DISTRICT,80,569.774684,1.000000,53.332531,9.091868
4,SCHOOL,120,583.080487,0.999959,54.422763,3.728641
5,COUNTY__ASSESSMENT_NAME__SUBGROUP_NAME,120,612.441156,0.996184,53.885140,4.112980
6,DISTRICT__SUBGROUP_NAME,150,614.334283,0.999821,53.162099,5.442560
7,DISTRICT__ASSESSMENT_NAME,150,627.384140,0.968811,53.784061,3.723250
8,COUNTY__SUBGROUP_NAME,80,647.228883,1.000000,54.089416,6.494915
9,COUNTY,50,670.251812,1.000000,54.165604,5.276547



Saved:
model_results/oof_hist_group_means_30A_lite.csv
model_results/hist_group_means_summary_30A_lite.csv

Best grouped predictor:
group           ASSESSMENT_NAME__SUBGROUP_NAME
alpha                                       40
oof_mse_clip                        518.259587
seen_rate                                  1.0
pred_mean                            54.116538
pred_std                             12.843963
Name: 0, dtype: object


##  30A-lite Grouped Historical Mean Diagnostic

We tested simple leakage-safe grouped historical mean predictors using 5-fold out-of-fold construction. Each validation row was encoded using group statistics computed only from the other four folds, so the diagnostic avoids direct target leakage.

The best standalone grouped predictor was:

`ASSESSMENT_NAME__SUBGROUP_NAME`

with:

- OOF clipped MSE: `518.259587`
- Seen rate: `1.000000`
- Prediction mean: `54.116538`
- Prediction standard deviation: `12.843963`

This is much weaker than the current target/statistical-encoding LightGBM and blend artifacts, so these simple grouped means are not useful as standalone models.

However, this result does not fully rule out grouped historical signal. Several high-cardinality groupings were probably over-smoothed. For example, `SCHOOL__ASSESSMENT_NAME` had a prediction standard deviation of only about `0.23`, meaning it was effectively shrunk back to the global mean. Before closing this branch, we will run a small alpha sensitivity diagnostic to see whether lower smoothing values recover useful signal.

In [77]:
# ============================================================
# 30B. Alpha sensitivity for grouped historical means
# ============================================================
# Purpose:
#   - Re-test the grouped historical mean idea with several smoothing values.
#   - Keep the diagnostic leakage-safe with 5-fold OOF construction.
#   - Decide whether poor 30A-lite results were caused by over-smoothing.
#
# This cell does NOT create test predictions or a submission.
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from IPython.display import display

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

print("30B. Alpha sensitivity for grouped historical means")
print("--------------------------------------------------")

# ------------------------------------------------------------
# Basic inputs
# ------------------------------------------------------------

raw_train_30B = train_full.reset_index(drop=True).copy()
y_30B = np.asarray(y_train, dtype=float).reshape(-1)

print("raw_train_30B shape:", raw_train_30B.shape)
print("y_30B shape:", y_30B.shape)
print("global target mean:", y_30B.mean())


# ------------------------------------------------------------
# Small helper functions
# ------------------------------------------------------------

def mse_clip_30B(y_true, pred):
    pred = np.clip(np.asarray(pred, dtype=float), 0, 100)
    return float(mean_squared_error(y_true, pred))


def make_key_30B(df, cols):
    key = df[cols[0]].astype("string").fillna("<NA>")
    for col in cols[1:]:
        key = key + "||" + df[col].astype("string").fillna("<NA>")
    return key


def score_group_alpha_grid_30B(df, y, cols, alpha_grid, n_splits=5):
    """
    For one group definition, compute OOF smoothed target means
    for several alpha values.

    alpha = 0 means no smoothing.
    Larger alpha means stronger shrinkage toward the fold training mean.
    """

    n = len(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    oof_by_alpha = {
        alpha: np.zeros(n, dtype=float)
        for alpha in alpha_grid
    }
    seen = np.zeros(n, dtype=bool)

    for fit_idx, val_idx in kf.split(df):
        df_fit = df.iloc[fit_idx]
        df_val = df.iloc[val_idx]

        y_fit = y[fit_idx]
        fold_mean = float(y_fit.mean())

        fit_key = make_key_30B(df_fit, cols)
        val_key = make_key_30B(df_val, cols)

        stats = (
            pd.DataFrame({"key": fit_key, "y": y_fit})
            .groupby("key")["y"]
            .agg(["count", "sum"])
        )

        seen[val_idx] = val_key.isin(stats.index).to_numpy()

        for alpha in alpha_grid:
            smooth_mean = (
                stats["sum"] + alpha * fold_mean
            ) / (
                stats["count"] + alpha
            )

            mapped = val_key.map(smooth_mean).fillna(fold_mean)
            oof_by_alpha[alpha][val_idx] = mapped.to_numpy(dtype=float)

    rows = []

    group_name = "__".join(cols)
    seen_rate = float(seen.mean())

    for alpha, pred in oof_by_alpha.items():
        pred = np.clip(pred, 0, 100)

        rows.append({
            "group": group_name,
            "alpha": alpha,
            "oof_mse_clip": mse_clip_30B(y, pred),
            "seen_rate": seen_rate,
            "pred_mean": float(pred.mean()),
            "pred_std": float(pred.std()),
        })

    return rows


# ------------------------------------------------------------
# Groups and smoothing values to test
# ------------------------------------------------------------

group_candidates_30B = [
    ["ASSESSMENT_NAME"],
    ["SUBGROUP_NAME"],
    ["ASSESSMENT_NAME", "SUBGROUP_NAME"],

    ["COUNTY", "ASSESSMENT_NAME"],
    ["COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"],

    ["DISTRICT"],
    ["DISTRICT", "ASSESSMENT_NAME"],
    ["DISTRICT", "SUBGROUP_NAME"],
    ["DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME"],

    ["SCHOOL"],
    ["SCHOOL", "ASSESSMENT_NAME"],
    ["SCHOOL", "SUBGROUP_NAME"],
    ["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"],
]

alpha_grid_30B = [0, 1, 2, 5, 10, 20, 40, 80, 150, 250, 400]


# ------------------------------------------------------------
# Run alpha screen
# ------------------------------------------------------------

all_rows_30B = []

for cols in group_candidates_30B:
    missing_cols = [col for col in cols if col not in raw_train_30B.columns]

    if missing_cols:
        print("Skipping missing group:", cols)
        continue

    print("Scoring group:", "__".join(cols))

    rows = score_group_alpha_grid_30B(
        df=raw_train_30B,
        y=y_30B,
        cols=cols,
        alpha_grid=alpha_grid_30B,
        n_splits=5,
    )

    all_rows_30B.extend(rows)


summary_30B = (
    pd.DataFrame(all_rows_30B)
    .sort_values("oof_mse_clip")
    .reset_index(drop=True)
)

summary_path_30B = "model_results/hist_group_alpha_sensitivity_30B.csv"
summary_30B.to_csv(summary_path_30B, index=False)

print("\nTop alpha sensitivity results")
print("-----------------------------")
display(summary_30B.head(30))

print("Saved:", summary_path_30B)

print("\nBest result")
print("-----------")
print(summary_30B.iloc[0])

30B. Alpha sensitivity for grouped historical means
--------------------------------------------------
raw_train_30B shape: (144921, 62)
y_30B shape: (144921,)
global target mean: 54.18352067678252
Scoring group: ASSESSMENT_NAME
Scoring group: SUBGROUP_NAME
Scoring group: ASSESSMENT_NAME__SUBGROUP_NAME
Scoring group: COUNTY__ASSESSMENT_NAME
Scoring group: COUNTY__ASSESSMENT_NAME__SUBGROUP_NAME
Scoring group: DISTRICT
Scoring group: DISTRICT__ASSESSMENT_NAME
Scoring group: DISTRICT__SUBGROUP_NAME
Scoring group: DISTRICT__ASSESSMENT_NAME__SUBGROUP_NAME
Scoring group: SCHOOL
Scoring group: SCHOOL__ASSESSMENT_NAME
Scoring group: SCHOOL__SUBGROUP_NAME
Scoring group: SCHOOL__ASSESSMENT_NAME__SUBGROUP_NAME

Top alpha sensitivity results
-----------------------------


,group,alpha,oof_mse_clip,seen_rate,pred_mean,pred_std
0,SCHOOL__ASSESSMENT_NAME,0,194.126402,0.912573,54.085244,24.254689
1,SCHOOL__ASSESSMENT_NAME,1,235.726770,0.912573,54.227788,16.290114
2,SCHOOL__ASSESSMENT_NAME,2,300.786406,0.912573,54.252232,12.547522
3,DISTRICT__ASSESSMENT_NAME,0,334.751505,0.968811,53.873197,20.338740
4,DISTRICT__ASSESSMENT_NAME,1,345.839129,0.968811,53.331887,16.812348
5,SCHOOL,1,364.781269,0.999959,54.253394,18.172359
6,SCHOOL,0,365.115454,0.999959,54.190408,18.946202
7,SCHOOL,2,365.726029,0.999959,54.307305,17.483437
8,DISTRICT__ASSESSMENT_NAME,2,367.897753,0.968811,53.088623,15.057862
9,SCHOOL,5,372.712880,0.999959,54.423358,15.761600


Saved: model_results/hist_group_alpha_sensitivity_30B.csv

Best result
-----------
group           SCHOOL__ASSESSMENT_NAME
alpha                                 0
oof_mse_clip                 194.126402
seen_rate                      0.912573
pred_mean                     54.085244
pred_std                      24.254689
Name: 0, dtype: object


## 30B Grouped Historical Mean Alpha Sensitivity

The initial grouped historical mean diagnostic in 30A-lite was too aggressively smoothed for high-cardinality groups. In 30B, we re-tested selected group definitions across a grid of smoothing strengths.

The best result was:

`SCHOOL__ASSESSMENT_NAME`, `alpha = 0`

with:

- OOF clipped MSE: `194.126402`
- Seen rate: `0.912573`
- Prediction mean: `54.085244`
- Prediction standard deviation: `24.254689`

This is a large improvement over the best 30A-lite grouped mean result of about `518.26`, confirming that the earlier grouped-mean test was over-smoothed.

However, even the best grouped historical mean is still much weaker than the current TE LightGBM / TE blend artifacts. Therefore, grouped historical means are not submission-worthy as standalone models. The next question is narrower: whether the best grouped historical predictors add any diversity when blended with the current TE anchor.



### Alpha Sensitivity for Grouped Historical Means

This diagnostic tested simple grouped historical means with different smoothing strengths.

Here, `alpha` is a smoothing parameter for grouped target averages:

`smoothed_mean = (group_sum + alpha * global_mean) / (group_count + alpha)`

Intuitively, `alpha` behaves like adding `alpha` fake observations equal to the global mean. A larger alpha pulls each group average toward the global mean more strongly. An alpha of `0` means no smoothing.

The best result was:

- Group: `SCHOOL__ASSESSMENT_NAME`
- Alpha: `0`
- OOF clipped MSE: `194.126402`
- Seen rate: `0.912573`
- Prediction mean: `54.085244`
- Prediction standard deviation: `24.254689`

This showed that the earlier 30A-lite grouped means were over-smoothed. The grouped historical signal exists, but even the best grouped mean is still much weaker than the current target-encoding LightGBM and TE blend artifacts.

In [78]:
# ============================================================
# 30C. Blend check for selected grouped historical predictors
# ============================================================
# Purpose:
#   - Rebuild a small set of the best grouped historical OOF predictors.
#   - Compare them to the current TE OOF anchors.
#   - Check whether any grouped predictor adds blend value.
#
# This cell does NOT create test predictions or a submission.
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from IPython.display import display

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

print("30C. Blend check for selected grouped historical predictors")
print("----------------------------------------------------------")

# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------

raw_train_30C = train_full.reset_index(drop=True).copy()
y_30C = np.asarray(y_train, dtype=float).reshape(-1)

print("raw_train_30C shape:", raw_train_30C.shape)
print("y_30C shape:", y_30C.shape)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def mse_clip_30C(y_true, pred):
    pred = np.clip(np.asarray(pred, dtype=float), 0, 100)
    return float(mean_squared_error(y_true, pred))


def make_group_key_30C(df, cols):
    key = df[cols[0]].astype("string").fillna("<NA>")
    for col in cols[1:]:
        key = key + "||" + df[col].astype("string").fillna("<NA>")
    return key


def build_oof_group_mean_30C(df, y, cols, alpha, n_splits=5):
    """
    Leakage-safe OOF grouped mean.

    Each validation fold receives group means computed only from
    the other folds. Unseen groups fall back to that fold's train mean.
    """

    oof = np.zeros(len(y), dtype=float)
    seen = np.zeros(len(y), dtype=bool)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    for fit_idx, val_idx in kf.split(df):
        df_fit = df.iloc[fit_idx]
        df_val = df.iloc[val_idx]

        y_fit = y[fit_idx]
        fold_mean = float(y_fit.mean())

        fit_key = make_group_key_30C(df_fit, cols)
        val_key = make_group_key_30C(df_val, cols)

        stats = (
            pd.DataFrame({"key": fit_key, "y": y_fit})
            .groupby("key")["y"]
            .agg(["count", "sum"])
        )

        smoothed_mean = (
            stats["sum"] + alpha * fold_mean
        ) / (
            stats["count"] + alpha
        )

        mapped = val_key.map(smoothed_mean)

        seen[val_idx] = mapped.notna().to_numpy()
        oof[val_idx] = mapped.fillna(fold_mean).to_numpy(dtype=float)

    return np.clip(oof, 0, 100), seen


def load_oof_prediction_30C(path):
    """
    Load one saved OOF prediction file and choose a prediction column.
    """

    df = pd.read_csv(path)

    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_30C):
        raise ValueError(f"{path} has {len(df)} rows, expected {len(y_30C)}")

    preferred_cols = [
        "pred_clipped",
        "pred_raw",
        "prediction",
        "oof_pred",
        "blend_pred",
    ]

    for col in preferred_cols:
        if col in df.columns:
            return np.clip(df[col].to_numpy(dtype=float), 0, 100), col

    numeric_cols = [
        col for col in df.select_dtypes(include=[np.number]).columns
        if col not in ["row_index", "ASSESSMENT_ID", "fold", "y_true"]
    ]

    if not numeric_cols:
        raise ValueError(f"No usable prediction column found in {path}")

    col = numeric_cols[-1]
    return np.clip(df[col].to_numpy(dtype=float), 0, 100), col


# ------------------------------------------------------------
# Rebuild selected grouped predictors from 30B
# ------------------------------------------------------------

selected_groups_30C = [
    ("hist_school_assessment_a0", ["SCHOOL", "ASSESSMENT_NAME"], 0),
    ("hist_district_assessment_a0", ["DISTRICT", "ASSESSMENT_NAME"], 0),
    ("hist_school_a0", ["SCHOOL"], 0),
    ("hist_school_a1", ["SCHOOL"], 1),
    ("hist_school_subgroup_a1", ["SCHOOL", "SUBGROUP_NAME"], 1),
    ("hist_county_assessment_a0", ["COUNTY", "ASSESSMENT_NAME"], 0),
]

hist_preds_30C = {}
hist_rows_30C = []

for name, cols, alpha in selected_groups_30C:
    missing_cols = [col for col in cols if col not in raw_train_30C.columns]

    if missing_cols:
        print(f"Skipping {name}; missing columns: {missing_cols}")
        continue

    print(f"Building {name}: columns={cols}, alpha={alpha}")

    pred, seen = build_oof_group_mean_30C(
        df=raw_train_30C,
        y=y_30C,
        cols=cols,
        alpha=alpha,
        n_splits=5,
    )

    hist_preds_30C[name] = pred

    hist_rows_30C.append({
        "component": name,
        "columns": "__".join(cols),
        "alpha": alpha,
        "oof_mse_clip": mse_clip_30C(y_30C, pred),
        "seen_rate": float(seen.mean()),
        "pred_mean": float(pred.mean()),
        "pred_std": float(pred.std()),
    })

hist_summary_30C = (
    pd.DataFrame(hist_rows_30C)
    .sort_values("oof_mse_clip")
    .reset_index(drop=True)
)

print("\nSelected grouped predictor summary")
print("----------------------------------")
display(hist_summary_30C)

hist_oof_30C = pd.DataFrame({"row_index": np.arange(len(y_30C))})
for name, pred in hist_preds_30C.items():
    hist_oof_30C[name] = pred

hist_oof_path_30C = "model_results/oof_hist_group_selected_30C.csv"
hist_oof_30C.to_csv(hist_oof_path_30C, index=False)

print("Saved selected grouped OOF predictions:", hist_oof_path_30C)


# ------------------------------------------------------------
# Load current TE anchors
# ------------------------------------------------------------

anchor_files_30C = {
    "safe_te_blend": "model_results/oof_blend_te_verified_noresid_weighted.csv",
    "pure_te_lgbm": "model_results/oof_lgbm_te_base_5fold_oof_v1.csv",
}

anchor_preds_30C = {}
anchor_rows_30C = []

for anchor_name, path in anchor_files_30C.items():
    if not os.path.exists(path):
        print(f"Missing anchor file, skipped: {path}")
        continue

    pred, pred_col = load_oof_prediction_30C(path)

    anchor_preds_30C[anchor_name] = pred

    anchor_rows_30C.append({
        "anchor": anchor_name,
        "path": path,
        "column": pred_col,
        "oof_mse_clip": mse_clip_30C(y_30C, pred),
    })

anchor_summary_30C = (
    pd.DataFrame(anchor_rows_30C)
    .sort_values("oof_mse_clip")
    .reset_index(drop=True)
)

print("\nLoaded TE anchor summary")
print("------------------------")
display(anchor_summary_30C)

if len(anchor_summary_30C) == 0:
    raise ValueError("No TE anchor OOF files were found.")


# ------------------------------------------------------------
# Blend each selected grouped predictor with the best available anchor
# ------------------------------------------------------------

anchor_name_30C = anchor_summary_30C.loc[0, "anchor"]
anchor_pred_30C = anchor_preds_30C[anchor_name_30C]
anchor_mse_30C = mse_clip_30C(y_30C, anchor_pred_30C)

print("Using anchor:", anchor_name_30C)
print("Anchor OOF MSE:", anchor_mse_30C)

blend_rows_30C = []

for hist_name, hist_pred in hist_preds_30C.items():
    for w_hist in np.round(np.linspace(0.0, 0.30, 301), 4):
        pred_blend = (1.0 - w_hist) * anchor_pred_30C + w_hist * hist_pred

        blend_rows_30C.append({
            "anchor": anchor_name_30C,
            "hist_component": hist_name,
            "w_hist": float(w_hist),
            "w_anchor": float(1.0 - w_hist),
            "oof_mse_clip": mse_clip_30C(y_30C, pred_blend),
        })

blend_summary_30C = (
    pd.DataFrame(blend_rows_30C)
    .sort_values("oof_mse_clip")
    .reset_index(drop=True)
)

blend_path_30C = "model_results/hist_group_selected_blend_screen_30C.csv"
blend_summary_30C.to_csv(blend_path_30C, index=False)

print("\nBest anchor + grouped historical blend rows")
print("-------------------------------------------")
display(blend_summary_30C.head(30))

print("Saved blend screen:", blend_path_30C)


# ------------------------------------------------------------
# Decision summary
# ------------------------------------------------------------

best_blend_30C = blend_summary_30C.iloc[0].to_dict()
gain_30C = anchor_mse_30C - float(best_blend_30C["oof_mse_clip"])

print("\nDecision diagnostic")
print("-------------------")
print("Anchor:", anchor_name_30C)
print("Anchor OOF MSE:", anchor_mse_30C)
print("Best blend OOF MSE:", float(best_blend_30C["oof_mse_clip"]))
print("Best grouped component:", best_blend_30C["hist_component"])
print("Best grouped weight:", float(best_blend_30C["w_hist"]))
print("Gain vs anchor:", gain_30C)

HIST_GROUP_SELECTED_BEST_BLEND_30C = best_blend_30C
HIST_GROUP_SELECTED_GAIN_30C = gain_30C

print("\n30C complete.")

30C. Blend check for selected grouped historical predictors
----------------------------------------------------------
raw_train_30C shape: (144921, 62)
y_30C shape: (144921,)
Building hist_school_assessment_a0: columns=['SCHOOL', 'ASSESSMENT_NAME'], alpha=0
Building hist_district_assessment_a0: columns=['DISTRICT', 'ASSESSMENT_NAME'], alpha=0
Building hist_school_a0: columns=['SCHOOL'], alpha=0
Building hist_school_a1: columns=['SCHOOL'], alpha=1
Building hist_school_subgroup_a1: columns=['SCHOOL', 'SUBGROUP_NAME'], alpha=1
Building hist_county_assessment_a0: columns=['COUNTY', 'ASSESSMENT_NAME'], alpha=0

Selected grouped predictor summary
----------------------------------


,component,columns,alpha,oof_mse_clip,seen_rate,pred_mean,pred_std
0,hist_school_assessment_a0,SCHOOL__ASSESSMENT_NAME,0,194.126402,0.912573,54.085244,24.254689
1,hist_district_assessment_a0,DISTRICT__ASSESSMENT_NAME,0,334.751505,0.968811,53.873197,20.338740
2,hist_school_a1,SCHOOL,1,364.781269,0.999959,54.253394,18.172359
3,hist_school_a0,SCHOOL,0,365.115454,0.999959,54.190408,18.946202
4,hist_school_subgroup_a1,SCHOOL__SUBGROUP_NAME,1,401.599449,0.990774,54.419232,16.842598
5,hist_county_assessment_a0,COUNTY__ASSESSMENT_NAME,0,455.100995,0.999738,54.170198,15.968256


Saved selected grouped OOF predictions: model_results/oof_hist_group_selected_30C.csv

Loaded TE anchor summary
------------------------


,anchor,path,column,oof_mse_clip
0,safe_te_blend,model_results/oof_blend_te_verified_noresid_we...,pred_clipped,78.071510
1,pure_te_lgbm,model_results/oof_lgbm_te_base_5fold_oof_v1.csv,pred_clipped,82.907614


Using anchor: safe_te_blend
Anchor OOF MSE: 78.07151037231891

Best anchor + grouped historical blend rows
-------------------------------------------


,anchor,hist_component,w_hist,w_anchor,oof_mse_clip
0,safe_te_blend,hist_school_assessment_a0,0.014,0.986,78.048814
1,safe_te_blend,hist_school_assessment_a0,0.013,0.987,78.048883
2,safe_te_blend,hist_school_assessment_a0,0.015,0.985,78.048983
3,safe_te_blend,hist_school_assessment_a0,0.012,0.988,78.049192
4,safe_te_blend,hist_school_assessment_a0,0.016,0.984,78.049391
5,safe_te_blend,hist_school_assessment_a0,0.011,0.989,78.049739
6,safe_te_blend,hist_school_assessment_a0,0.017,0.983,78.050037
7,safe_te_blend,hist_school_assessment_a0,0.010,0.990,78.050525
8,safe_te_blend,hist_school_assessment_a0,0.018,0.982,78.050922
9,safe_te_blend,hist_school_assessment_a0,0.009,0.991,78.051549


Saved blend screen: model_results/hist_group_selected_blend_screen_30C.csv

Decision diagnostic
-------------------
Anchor: safe_te_blend
Anchor OOF MSE: 78.07151037231891
Best blend OOF MSE: 78.0488138131083
Best grouped component: hist_school_assessment_a0
Best grouped weight: 0.014
Gain vs anchor: 0.02269655921061542

30C complete.


## 30C Grouped Historical Mean Blend Check

We checked whether the best simple grouped historical predictors add useful diversity to the current safe TE blend.

The best standalone grouped predictor was:

- Component: `hist_school_assessment_a0`
- Meaning: OOF mean target for `SCHOOL × ASSESSMENT_NAME`, with no smoothing
- OOF clipped MSE: `194.126402`
- Seen rate: `0.912573`

The current TE anchor was:

- Anchor: `safe_te_blend`
- OOF clipped MSE: `78.071510`

The best blend was:

- `0.986 * safe_te_blend`
- `0.014 * hist_school_assessment_a0`
- OOF clipped MSE: `78.048814`
- Gain versus anchor: `0.022697`

Decision:

The grouped historical mean predictor adds only a tiny amount of blend diversity. This is not enough to justify a submission or a larger modeling branch. We will close this branch and move on to a genuinely different source of signal.

In [79]:
# ============================================================
# 31A. Overnight run:
# Linear/ElasticNet/PLS + bounded-target LightGBM on TE features
# ============================================================
#
# Runs in sequence:
#   1. Ridge / ElasticNet / PLS on base + TE features
#   2. Bounded/proportion-aware LightGBM using transformed targets
#   3. Final blend diagnostic against current safe TE blend anchor
#
# This cell creates OOF/test artifacts but does NOT recommend auto-submission.
# ============================================================

import os
import gc
import time
import warnings
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.cross_decomposition import PLSRegression

import lightgbm as lgb

warnings.filterwarnings("ignore")
os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5

print("=" * 90)
print("31A overnight: Linear/PLS + bounded LightGBM on base + TE features")
print("=" * 90)

# ------------------------------------------------------------
# 0. Required-object checks and helper aliases
# ------------------------------------------------------------

required_base = [
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
    "raw_train_te",
    "raw_test_te",
    "te_key_specs",
]

missing_base = [x for x in required_base if x not in globals()]
if missing_base:
    raise ValueError(
        f"Missing required objects: {missing_base}. "
        "Rerun the TE branch setup cells around 28B/28D before this overnight cell."
    )

# Use the existing TE helper if available. If not, try the 29B suffixed version.
if "build_te_oof_and_apply_many" not in globals():
    if "build_te_oof_and_apply_many_29b" in globals():
        build_te_oof_and_apply_many = build_te_oof_and_apply_many_29b
    else:
        raise ValueError("Missing build_te_oof_and_apply_many helper. Rerun 28E or 29B setup.")

if "take_rows" not in globals():
    if "take_rows_29b" in globals():
        take_rows = take_rows_29b
    else:
        raise ValueError("Missing take_rows helper.")

if "to_float32_matrix" not in globals():
    if "to_float32_matrix_29b" in globals():
        to_float32_matrix = to_float32_matrix_29b
    else:
        raise ValueError("Missing to_float32_matrix helper.")

if "append_features" not in globals():
    if "append_features_29b" in globals():
        append_features = append_features_29b
    else:
        raise ValueError("Missing append_features helper.")

y_arr = np.asarray(y_train, dtype=np.float32).reshape(-1)
test_ids_arr = np.asarray(test_ids)

n_train = len(y_arr)
n_test = len(test_ids_arr)

if X_train_proc_model.shape[0] != n_train:
    raise ValueError("X_train_proc_model row count does not match y_train.")
if X_test_proc_model.shape[0] != n_test:
    raise ValueError("X_test_proc_model row count does not match test_ids.")
if raw_train_te.shape[0] != n_train:
    raise ValueError("raw_train_te row count does not match y_train.")
if raw_test_te.shape[0] != n_test:
    raise ValueError("raw_test_te row count does not match test_ids.")

print("Train rows:", n_train)
print("Test rows: ", n_test)
print("Base features:", X_train_proc_model.shape[1])
print("TE key specs:", len(te_key_specs))
print("Expected TE features:", len(te_key_specs) * 4)
print("Expected augmented features:", X_train_proc_model.shape[1] + len(te_key_specs) * 4)

# ------------------------------------------------------------
# 1. Target transforms for bounded LightGBM
# ------------------------------------------------------------

def transform_target(y, kind):
    y = np.asarray(y, dtype=np.float64)
    p = np.clip(y / 100.0, 0.005, 0.995)

    if kind == "logit":
        return np.log(p / (1.0 - p)).astype(np.float32)

    if kind == "arcsine":
        return np.arcsin(np.sqrt(p)).astype(np.float32)

    raise ValueError(f"Unknown transform kind: {kind}")

def inverse_transform_target(z, kind):
    z = np.asarray(z, dtype=np.float64)

    if kind == "logit":
        p = 1.0 / (1.0 + np.exp(-z))
        return np.clip(100.0 * p, 0.0, 100.0).astype(np.float32)

    if kind == "arcsine":
        z_clip = np.clip(z, 0.0, np.pi / 2.0)
        p = np.sin(z_clip) ** 2
        return np.clip(100.0 * p, 0.0, 100.0).astype(np.float32)

    raise ValueError(f"Unknown transform kind: {kind}")

# ------------------------------------------------------------
# 2. Model specs
# ------------------------------------------------------------

linear_specs = [
    {
        "name": "linear31_ridge_alpha100",
        "kind": "ridge",
        "model": lambda: make_pipeline(
            StandardScaler(with_mean=False),
            Ridge(alpha=100.0, solver="lsqr")
        ),
    },
    {
        "name": "linear31_ridge_alpha1000",
        "kind": "ridge",
        "model": lambda: make_pipeline(
            StandardScaler(with_mean=False),
            Ridge(alpha=1000.0, solver="lsqr")
        ),
    },
    {
        "name": "linear31_elasticnet_a0005_l1_005",
        "kind": "elasticnet",
        "model": lambda: make_pipeline(
            StandardScaler(with_mean=False),
            ElasticNet(
                alpha=0.0005,
                l1_ratio=0.05,
                max_iter=5000,
                tol=1e-4,
                selection="random",
                random_state=RANDOM_STATE,
            )
        ),
    },
    {
        "name": "linear31_pls_30",
        "kind": "pls",
        "n_components": 30,
        "model": lambda: PLSRegression(n_components=30, scale=True, max_iter=500),
    },
]

bounded_lgbm_specs = [
    {
        "name": "bounded31_lgbm_logit_l95_child60_lr02",
        "transform": "logit",
        "stopping_rounds": 2500,
        "params": {
            "objective": "regression",
            "metric": "l2",
            "random_state": RANDOM_STATE,
            "n_jobs": 1,
            "verbosity": -1,
            "force_col_wise": True,
            "n_estimators": 60000,
            "learning_rate": 0.02,
            "num_leaves": 95,
            "min_child_samples": 60,
            "subsample": 0.85,
            "subsample_freq": 1,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.0,
            "reg_lambda": 5.0,
            "max_depth": -1,
        },
    },
    {
        "name": "bounded31_lgbm_arcsine_l95_child60_lr02",
        "transform": "arcsine",
        "stopping_rounds": 2500,
        "params": {
            "objective": "regression",
            "metric": "l2",
            "random_state": RANDOM_STATE + 31,
            "n_jobs": 1,
            "verbosity": -1,
            "force_col_wise": True,
            "n_estimators": 60000,
            "learning_rate": 0.02,
            "num_leaves": 95,
            "min_child_samples": 60,
            "subsample": 0.85,
            "subsample_freq": 1,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.0,
            "reg_lambda": 5.0,
            "max_depth": -1,
        },
    },
]

all_artifact_names = [s["name"] for s in linear_specs] + [s["name"] for s in bounded_lgbm_specs]

# ------------------------------------------------------------
# 3. Resume-safe storage helpers
# ------------------------------------------------------------

def oof_npy_path(name):
    return f"model_results/{name}_oof_clipped.npy"

def fold_test_path(name, fold_num):
    return f"model_results/{name}_fold{fold_num}_test_pred.npy"

def fold_metrics_path(name):
    return f"model_results/{name}_fold_metrics.csv"

def oof_csv_path(name):
    return f"model_results/oof_{name}.csv"

def test_csv_path(name):
    return f"model_results/testpred_{name}_foldavg.csv"

def submission_path(name):
    return f"submission_{name}_foldavg.csv"

def load_oof_array(name):
    path = oof_npy_path(name)
    if os.path.exists(path):
        arr = np.load(path)
        if len(arr) == n_train:
            return arr.astype(np.float32)
    return np.full(n_train, np.nan, dtype=np.float32)

def load_metrics_df(name):
    path = fold_metrics_path(name)
    if os.path.exists(path):
        return pd.read_csv(path)
    return pd.DataFrame()

def completed_folds_from_metrics(name):
    df = load_metrics_df(name)
    if len(df) == 0 or "fold" not in df.columns:
        return set()
    return set(df["fold"].astype(int).tolist())

def save_fold_metric(name, row):
    path = fold_metrics_path(name)
    old = load_metrics_df(name)
    if len(old) > 0 and "fold" in old.columns:
        old = old[old["fold"].astype(int) != int(row["fold"])]
    new = pd.concat([old, pd.DataFrame([row])], ignore_index=True)
    new = new.sort_values("fold").reset_index(drop=True)
    new.to_csv(path, index=False)
    return new

def dense_float32(X):
    if sparse.issparse(X):
        return X.toarray().astype(np.float32, copy=False)
    if isinstance(X, pd.DataFrame):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

# Initialize OOF arrays.
oof_arrays = {name: load_oof_array(name) for name in all_artifact_names}

# ------------------------------------------------------------
# 4. Main 5-fold loop
# ------------------------------------------------------------

outer_kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(outer_kf.split(np.arange(n_train)))

overall_t0 = time.time()

for fold_num, (tr_idx, va_idx) in enumerate(fold_indices, start=1):
    print("\n" + "=" * 90)
    print(f"OUTER FOLD {fold_num}/{N_SPLITS}")
    print("=" * 90)

    need_anything_this_fold = False

    for name in all_artifact_names:
        fold_done = (
            os.path.exists(fold_test_path(name, fold_num))
            and np.isfinite(oof_arrays[name][va_idx]).all()
            and fold_num in completed_folds_from_metrics(name)
        )
        if not fold_done:
            need_anything_this_fold = True
            break

    if not need_anything_this_fold:
        print(f"Fold {fold_num} already complete for all artifacts. Skipping TE rebuild.")
        continue

    fold_t0 = time.time()

    raw_fit_fold = raw_train_te.iloc[tr_idx].reset_index(drop=True)
    raw_val_fold = raw_train_te.iloc[va_idx].reset_index(drop=True)

    y_fit = y_arr[tr_idx]
    y_val = y_arr[va_idx]

    print("Building leakage-safe TE features for this fold...")
    te_fit_fold, apply_dict_fold, te_key_summary_fold = build_te_oof_and_apply_many(
        raw_fit=raw_fit_fold,
        y_fit=y_fit,
        raw_apply_dict={
            "valid": raw_val_fold,
            "test": raw_test_te,
        },
        key_specs=te_key_specs,
        n_splits=5,
        random_state=RANDOM_STATE + 1000 + fold_num,
    )

    X_fit_base = to_float32_matrix(take_rows(X_train_proc_model, tr_idx))
    X_val_base = to_float32_matrix(take_rows(X_train_proc_model, va_idx))
    X_test_base = to_float32_matrix(X_test_proc_model)

    X_fit = append_features(X_fit_base, te_fit_fold)
    X_val = append_features(X_val_base, apply_dict_fold["valid"])
    X_test = append_features(X_test_base, apply_dict_fold["test"])

    print("Fold train shape:", X_fit.shape)
    print("Fold valid shape:", X_val.shape)
    print("Fold test shape: ", X_test.shape)

    # --------------------------------------------------------
    # 4A. Linear / ElasticNet / PLS branch
    # --------------------------------------------------------

    print("\n--- Linear / ElasticNet / PLS branch ---")

    for spec in linear_specs:
        name = spec["name"]

        fold_done = (
            os.path.exists(fold_test_path(name, fold_num))
            and np.isfinite(oof_arrays[name][va_idx]).all()
            and fold_num in completed_folds_from_metrics(name)
        )

        if fold_done:
            print(f"{name}: fold {fold_num} already complete. Skipping.")
            continue

        print(f"\nTraining {name}...")
        model_t0 = time.time()

        try:
            model = spec["model"]()

            if spec["kind"] == "pls":
                X_fit_model = dense_float32(X_fit)
                X_val_model = dense_float32(X_val)
                X_test_model = dense_float32(X_test)
            else:
                X_fit_model = X_fit
                X_val_model = X_val
                X_test_model = X_test

            model.fit(X_fit_model, y_fit)

            pred_val = np.asarray(model.predict(X_val_model)).reshape(-1)
            pred_test = np.asarray(model.predict(X_test_model)).reshape(-1)

            pred_val_clip = np.clip(pred_val, 0, 100).astype(np.float32)
            pred_test_clip = np.clip(pred_test, 0, 100).astype(np.float32)

            oof_arrays[name][va_idx] = pred_val_clip
            np.save(oof_npy_path(name), oof_arrays[name])
            np.save(fold_test_path(name, fold_num), pred_test_clip)

            fold_mse = float(mean_squared_error(y_val, pred_val_clip))
            model_elapsed = time.time() - model_t0

            row = {
                "artifact": name,
                "branch": "linear_pls",
                "fold": fold_num,
                "valid_mse_clipped": fold_mse,
                "valid_pred_mean": float(np.mean(pred_val_clip)),
                "valid_pred_std": float(np.std(pred_val_clip)),
                "valid_pred_min": float(np.min(pred_val_clip)),
                "valid_pred_max": float(np.max(pred_val_clip)),
                "elapsed_seconds": float(model_elapsed),
            }

            save_fold_metric(name, row)

            print(f"{name}: fold {fold_num} clipped MSE = {fold_mse:.6f} | elapsed {model_elapsed:.1f}s")

            del model, pred_val, pred_test, pred_val_clip, pred_test_clip
            if spec["kind"] == "pls":
                del X_fit_model, X_val_model, X_test_model

        except Exception as e:
            print(f"ERROR in {name}, fold {fold_num}: {repr(e)}")
            row = {
                "artifact": name,
                "branch": "linear_pls",
                "fold": fold_num,
                "valid_mse_clipped": np.nan,
                "error": repr(e),
                "elapsed_seconds": float(time.time() - model_t0),
            }
            save_fold_metric(name, row)

        gc.collect()

    # --------------------------------------------------------
    # 4B. Bounded transformed-target LightGBM branch
    # --------------------------------------------------------

    print("\n--- Bounded transformed-target LightGBM branch ---")

    for spec in bounded_lgbm_specs:
        name = spec["name"]
        transform_kind = spec["transform"]

        fold_done = (
            os.path.exists(fold_test_path(name, fold_num))
            and np.isfinite(oof_arrays[name][va_idx]).all()
            and fold_num in completed_folds_from_metrics(name)
        )

        if fold_done:
            print(f"{name}: fold {fold_num} already complete. Skipping.")
            continue

        print(f"\nTraining {name}...")
        print("Target transform:", transform_kind)

        model_t0 = time.time()

        try:
            y_fit_t = transform_target(y_fit, transform_kind)
            y_val_t = transform_target(y_val, transform_kind)

            model = lgb.LGBMRegressor(**spec["params"])

            model.fit(
                X_fit,
                y_fit_t,
                eval_set=[(X_val, y_val_t)],
                eval_metric="l2",
                callbacks=[
                    lgb.early_stopping(stopping_rounds=spec["stopping_rounds"], verbose=False),
                    lgb.log_evaluation(period=2500),
                ],
            )

            best_iter = int(model.best_iteration_ or spec["params"]["n_estimators"])

            pred_val_t = model.predict(X_val, num_iteration=best_iter)
            pred_test_t = model.predict(X_test, num_iteration=best_iter)

            pred_val_clip = inverse_transform_target(pred_val_t, transform_kind)
            pred_test_clip = inverse_transform_target(pred_test_t, transform_kind)

            oof_arrays[name][va_idx] = pred_val_clip
            np.save(oof_npy_path(name), oof_arrays[name])
            np.save(fold_test_path(name, fold_num), pred_test_clip)

            fold_mse = float(mean_squared_error(y_val, pred_val_clip))
            transform_mse = float(mean_squared_error(y_val_t, pred_val_t))
            model_elapsed = time.time() - model_t0

            row = {
                "artifact": name,
                "branch": "bounded_lgbm",
                "transform": transform_kind,
                "fold": fold_num,
                "best_iteration": best_iter,
                "valid_mse_clipped_original_scale": fold_mse,
                "valid_mse_transform_scale": transform_mse,
                "valid_pred_mean": float(np.mean(pred_val_clip)),
                "valid_pred_std": float(np.std(pred_val_clip)),
                "valid_pred_min": float(np.min(pred_val_clip)),
                "valid_pred_max": float(np.max(pred_val_clip)),
                "elapsed_seconds": float(model_elapsed),
            }

            save_fold_metric(name, row)

            print(f"{name}: fold {fold_num} clipped MSE = {fold_mse:.6f}")
            print(f"{name}: best_iteration = {best_iter} | elapsed {model_elapsed:.1f}s")

            del model, y_fit_t, y_val_t
            del pred_val_t, pred_test_t, pred_val_clip, pred_test_clip

        except Exception as e:
            print(f"ERROR in {name}, fold {fold_num}: {repr(e)}")
            row = {
                "artifact": name,
                "branch": "bounded_lgbm",
                "transform": transform_kind,
                "fold": fold_num,
                "valid_mse_clipped_original_scale": np.nan,
                "error": repr(e),
                "elapsed_seconds": float(time.time() - model_t0),
            }
            save_fold_metric(name, row)

        gc.collect()

    # Fold cleanup.
    del raw_fit_fold, raw_val_fold
    del te_fit_fold, apply_dict_fold, te_key_summary_fold
    del X_fit_base, X_val_base, X_test_base
    del X_fit, X_val, X_test
    gc.collect()

    print(f"\nFold {fold_num} total elapsed: {time.time() - fold_t0:.1f}s")

# ------------------------------------------------------------
# 5. Finalize completed artifacts
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("Finalizing completed artifacts")
print("=" * 90)

artifact_summary_rows = []
completed_artifacts = []

for name in all_artifact_names:
    oof_arr = load_oof_array(name)
    complete_oof = bool(np.isfinite(oof_arr).all())

    fold_test_files = [fold_test_path(name, f) for f in range(1, N_SPLITS + 1)]
    complete_test = all(os.path.exists(p) for p in fold_test_files)

    if not complete_oof or not complete_test:
        print(f"{name}: incomplete. OOF complete={complete_oof}, test folds complete={complete_test}")
        continue

    fold_test_preds = [np.load(p).astype(np.float32) for p in fold_test_files]
    test_pred = np.clip(np.mean(np.vstack(fold_test_preds), axis=0), 0, 100).astype(np.float32)

    oof_mse = float(mean_squared_error(y_arr, oof_arr))

    pd.DataFrame({
        "row_index": np.arange(n_train),
        "PERCENT_PROFICIENT": y_arr,
        "pred_clipped": oof_arr,
    }).to_csv(oof_csv_path(name), index=False)

    pd.DataFrame({
        "ASSESSMENT_ID": test_ids_arr,
        "PERCENT_PROFICIENT": test_pred,
    }).to_csv(test_csv_path(name), index=False)

    pd.DataFrame({
        "ASSESSMENT_ID": test_ids_arr,
        "PERCENT_PROFICIENT": test_pred,
    }).to_csv(submission_path(name), index=False)

    metrics_df = load_metrics_df(name)

    row = {
        "artifact": name,
        "oof_mse_clipped": oof_mse,
        "test_mean": float(np.mean(test_pred)),
        "test_std": float(np.std(test_pred)),
        "test_min": float(np.min(test_pred)),
        "test_max": float(np.max(test_pred)),
        "oof_file": oof_csv_path(name),
        "test_file": test_csv_path(name),
        "submission_file": submission_path(name),
    }

    artifact_summary_rows.append(row)
    completed_artifacts.append(name)

    print(f"\n{name}")
    print("OOF MSE:", oof_mse)
    print("Saved:", oof_csv_path(name))
    print("Saved:", test_csv_path(name))
    print("Saved:", submission_path(name))

artifact_summary = pd.DataFrame(artifact_summary_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
artifact_summary_path = "model_results/overnight31_linear_bounded_artifact_summary.csv"
artifact_summary.to_csv(artifact_summary_path, index=False)

print("\nCompleted artifact summary:")
display(artifact_summary)
print("Saved summary:", artifact_summary_path)

# ------------------------------------------------------------
# 6. Blend diagnostic against current safe TE blend anchor
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("Blend diagnostic against current safe TE blend")
print("=" * 90)

def load_oof_prediction_simple(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}")

    candidate_cols = []
    for col in df.columns:
        if col.lower() in ["row_index", "assessment_id", "fold"]:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue

        arr = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
        if not np.isfinite(arr).all():
            continue

        arr_clip = np.clip(arr, 0, 100)
        mse = float(mean_squared_error(y_ref, arr_clip))

        # Avoid selecting true target.
        if mse < 1e-8:
            continue

        candidate_cols.append((mse, col, arr_clip.astype(np.float32)))

    if len(candidate_cols) == 0:
        raise ValueError(f"No prediction column found for {path}")

    candidate_cols = sorted(candidate_cols, key=lambda x: x[0])
    return candidate_cols[0][2], candidate_cols[0][1], candidate_cols[0][0]

def load_test_prediction_simple(path, test_ids_ref):
    df = pd.read_csv(path)

    if "ASSESSMENT_ID" in df.columns:
        if len(df) == len(test_ids_ref) and np.array_equal(df["ASSESSMENT_ID"].to_numpy(), test_ids_ref):
            aligned = df.copy()
        else:
            aligned = pd.DataFrame({"ASSESSMENT_ID": test_ids_ref}).merge(df, on="ASSESSMENT_ID", how="left")
    else:
        aligned = df.copy()

    if len(aligned) != len(test_ids_ref):
        raise ValueError(f"Test row mismatch for {path}")

    preferred_cols = ["PERCENT_PROFICIENT", "pred_clipped", "prediction", "pred", "test_pred"]
    for col in preferred_cols:
        if col in aligned.columns and pd.api.types.is_numeric_dtype(aligned[col]):
            arr = pd.to_numeric(aligned[col], errors="coerce").to_numpy(dtype=np.float64)
            if np.isfinite(arr).all():
                return np.clip(arr, 0, 100).astype(np.float32), col

    numeric_cols = [
        c for c in aligned.columns
        if c.lower() not in ["assessment_id", "row_index", "fold"]
        and pd.api.types.is_numeric_dtype(aligned[c])
    ]

    if len(numeric_cols) == 1:
        col = numeric_cols[0]
        arr = pd.to_numeric(aligned[col], errors="coerce").to_numpy(dtype=np.float64)
        if np.isfinite(arr).all():
            return np.clip(arr, 0, 100).astype(np.float32), col

    raise ValueError(f"No test prediction column found for {path}")

blend_components = []

anchor_oof_path = "model_results/oof_blend_te_verified_noresid_weighted.csv"
anchor_test_path = "model_results/testpred_blend_te_verified_noresid_weighted.csv"

if os.path.exists(anchor_oof_path) and os.path.exists(anchor_test_path):
    anchor_oof, anchor_oof_col, anchor_mse = load_oof_prediction_simple(anchor_oof_path, y_arr)
    anchor_test, anchor_test_col = load_test_prediction_simple(anchor_test_path, test_ids_arr)

    blend_components.append({
        "name": "safe_te_blend_anchor_69p689",
        "oof": anchor_oof,
        "test": anchor_test,
        "oof_mse": anchor_mse,
        "oof_path": anchor_oof_path,
        "test_path": anchor_test_path,
    })

    print("Loaded safe TE blend anchor.")
    print("Anchor OOF MSE:", anchor_mse)
else:
    print("Safe TE blend anchor files not found. Blend diagnostic will use only new artifacts.")

for name in completed_artifacts:
    oof_path = oof_csv_path(name)
    tst_path = test_csv_path(name)

    if not os.path.exists(oof_path) or not os.path.exists(tst_path):
        continue

    pred_oof, pred_col, pred_mse = load_oof_prediction_simple(oof_path, y_arr)
    pred_test, test_col = load_test_prediction_simple(tst_path, test_ids_arr)

    blend_components.append({
        "name": name,
        "oof": pred_oof,
        "test": pred_test,
        "oof_mse": pred_mse,
        "oof_path": oof_path,
        "test_path": tst_path,
    })

if len(blend_components) < 2:
    print("Not enough completed components for blend diagnostic.")
else:
    names = [c["name"] for c in blend_components]
    P = np.column_stack([c["oof"] for c in blend_components]).astype(np.float64)
    T = np.column_stack([c["test"] for c in blend_components]).astype(np.float64)

    n, k = P.shape

    component_summary = pd.DataFrame({
        "component": names,
        "oof_mse": [c["oof_mse"] for c in blend_components],
        "oof_mean": P.mean(axis=0),
        "oof_std": P.std(axis=0),
        "test_mean": T.mean(axis=0),
        "test_std": T.std(axis=0),
        "oof_path": [c["oof_path"] for c in blend_components],
        "test_path": [c["test_path"] for c in blend_components],
    }).sort_values("oof_mse").reset_index(drop=True)

    print("\nBlend component summary:")
    display(component_summary)

    A = (P.T @ P) / n
    b = (P.T @ y_arr.astype(np.float64)) / n
    c0 = float((y_arr.astype(np.float64) @ y_arr.astype(np.float64)) / n)

    def mse_w(w):
        w = np.asarray(w, dtype=np.float64)
        return float(w @ A @ w - 2.0 * (w @ b) + c0)

    def mse_W(W):
        W = np.asarray(W, dtype=np.float64)
        return (
            np.einsum("ij,jk,ik->i", W, A, W)
            - 2.0 * (W @ b)
            + c0
        )

    candidate_rows = []

    def add_candidate(label, w):
        w = np.asarray(w, dtype=np.float64)
        w = np.maximum(w, 0)
        if w.sum() <= 0:
            return
        w = w / w.sum()
        row = {"label": label, "mse": mse_w(w)}
        for nm, ww in zip(names, w):
            row[f"w_{nm}"] = ww
        candidate_rows.append(row)

    # Pure models.
    for i, nm in enumerate(names):
        w = np.zeros(k)
        w[i] = 1.0
        add_candidate(f"pure_{nm}", w)

    # Pairwise fine grids.
    grid = np.linspace(0, 1, 1001)
    for i in range(k):
        for j in range(i + 1, k):
            W = np.zeros((len(grid), k), dtype=np.float64)
            W[:, i] = grid
            W[:, j] = 1.0 - grid
            mses = mse_W(W)
            add_candidate(f"pair_{names[i]}__{names[j]}", W[int(np.argmin(mses))])

    # Focused random convex search.
    rng = np.random.default_rng(RANDOM_STATE + 3101)

    rank = np.argsort([c["oof_mse"] for c in blend_components])
    alpha = np.ones(k) * 0.20
    for r, idx in enumerate(rank[:min(5, k)]):
        alpha[idx] = 8.0 / (r + 1)

    W = rng.dirichlet(alpha, size=50000)
    mses = mse_W(W)
    add_candidate("random_dirichlet_top_focused_50k", W[int(np.argmin(mses))])

    blend_screen = pd.DataFrame(candidate_rows).sort_values("mse").reset_index(drop=True)
    blend_screen_path = "model_results/overnight31_linear_bounded_blend_screen.csv"
    blend_screen.to_csv(blend_screen_path, index=False)

    best = blend_screen.iloc[0]
    weight_cols = [c for c in blend_screen.columns if c.startswith("w_")]
    best_w = best[weight_cols].to_numpy(dtype=np.float64)
    best_w = best_w / best_w.sum()

    blend_oof = np.clip(P @ best_w, 0, 100).astype(np.float32)
    blend_test = np.clip(T @ best_w, 0, 100).astype(np.float32)

    best_mse = float(mean_squared_error(y_arr, blend_oof))

    blend_oof_path = "model_results/oof_blend_overnight31_linear_bounded_te_weighted.csv"
    blend_test_path = "model_results/testpred_blend_overnight31_linear_bounded_te_weighted.csv"
    blend_submission_path = "submission_blend_overnight31_linear_bounded_te_weighted.csv"
    blend_weights_path = "model_results/blend_overnight31_linear_bounded_te_best_weights.csv"

    pd.DataFrame({
        "row_index": np.arange(n_train),
        "PERCENT_PROFICIENT": y_arr,
        "pred_clipped": blend_oof,
    }).to_csv(blend_oof_path, index=False)

    pd.DataFrame({
        "ASSESSMENT_ID": test_ids_arr,
        "PERCENT_PROFICIENT": blend_test,
    }).to_csv(blend_test_path, index=False)

    pd.DataFrame({
        "ASSESSMENT_ID": test_ids_arr,
        "PERCENT_PROFICIENT": blend_test,
    }).to_csv(blend_submission_path, index=False)

    weight_table = pd.DataFrame({
        "component": names,
        "weight": best_w,
        "component_oof_mse": [c["oof_mse"] for c in blend_components],
    }).sort_values("weight", ascending=False).reset_index(drop=True)

    weight_table.to_csv(blend_weights_path, index=False)

    anchor_mse_for_gain = None
    for c in blend_components:
        if c["name"] == "safe_te_blend_anchor_69p689":
            anchor_mse_for_gain = c["oof_mse"]

    print("\nTop blend candidates:")
    display(blend_screen.head(20))

    print("\nBest blend weights:")
    display(weight_table)

    print("\nBest overnight31 blend OOF MSE:", best_mse)
    if anchor_mse_for_gain is not None:
        print("Safe TE blend anchor OOF MSE:", anchor_mse_for_gain)
        print("Gain vs safe TE blend anchor:", anchor_mse_for_gain - best_mse)

    print("\nSaved blend files:")
    print(" -", blend_screen_path)
    print(" -", blend_weights_path)
    print(" -", blend_oof_path)
    print(" -", blend_test_path)
    print(" -", blend_submission_path)

    print("\nBlend test prediction summary:")
    print(pd.Series(blend_test).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

print("\n" + "=" * 90)
print("31A overnight run finished or checkpointed.")
print("Total elapsed seconds:", round(time.time() - overall_t0, 1))
print("=" * 90)

31A overnight: Linear/PLS + bounded LightGBM on base + TE features
Train rows: 144921
Test rows:  48307
Base features: 162
TE key specs: 17
Expected TE features: 68
Expected augmented features: 230

OUTER FOLD 1/5
Building leakage-safe TE features for this fold...
Fold train shape: (115936, 230)
Fold valid shape: (28985, 230)
Fold test shape:  (48307, 230)

--- Linear / ElasticNet / PLS branch ---

Training linear31_ridge_alpha100...
linear31_ridge_alpha100: fold 1 clipped MSE = 148.617325 | elapsed 0.7s

Training linear31_ridge_alpha1000...
linear31_ridge_alpha1000: fold 1 clipped MSE = 149.602829 | elapsed 0.4s

Training linear31_elasticnet_a0005_l1_005...
linear31_elasticnet_a0005_l1_005: fold 1 clipped MSE = 149.494873 | elapsed 13.1s

Training linear31_pls_30...
linear31_pls_30: fold 1 clipped MSE = 148.584198 | elapsed 2.9s

--- Bounded transformed-target LightGBM branch ---

Training bounded31_lgbm_logit_l95_child60_lr02...
Target transform: logit
[2500]	valid_0's l2: 0.516938
[

,artifact,oof_mse_clipped,test_mean,test_std,test_min,test_max,oof_file,test_file,submission_file
0,bounded31_lgbm_arcsine_l95_child60_lr02,83.952621,54.116909,25.367729,0.016324,100.000000,model_results/oof_bounded31_lgbm_arcsine_l95_c...,model_results/testpred_bounded31_lgbm_arcsine_...,submission_bounded31_lgbm_arcsine_l95_child60_...
1,bounded31_lgbm_logit_l95_child60_lr02,91.060333,54.145927,26.180801,0.269396,99.794357,model_results/oof_bounded31_lgbm_logit_l95_chi...,model_results/testpred_bounded31_lgbm_logit_l9...,submission_bounded31_lgbm_logit_l95_child60_lr...
2,linear31_pls_30,145.309433,54.082096,24.836098,0.000000,100.000000,model_results/oof_linear31_pls_30.csv,model_results/testpred_linear31_pls_30_foldavg...,submission_linear31_pls_30_foldavg.csv
3,linear31_elasticnet_a0005_l1_005,145.613022,54.365833,24.824835,0.000000,100.000000,model_results/oof_linear31_elasticnet_a0005_l1...,model_results/testpred_linear31_elasticnet_a00...,submission_linear31_elasticnet_a0005_l1_005_fo...
4,linear31_ridge_alpha100,145.735809,53.845928,24.854681,0.000000,100.000000,model_results/oof_linear31_ridge_alpha100.csv,model_results/testpred_linear31_ridge_alpha100...,submission_linear31_ridge_alpha100_foldavg.csv
5,linear31_ridge_alpha1000,146.095566,54.102894,24.780504,0.000000,100.000000,model_results/oof_linear31_ridge_alpha1000.csv,model_results/testpred_linear31_ridge_alpha100...,submission_linear31_ridge_alpha1000_foldavg.csv


Saved summary: model_results/overnight31_linear_bounded_artifact_summary.csv

Blend diagnostic against current safe TE blend
Loaded safe TE blend anchor.
Anchor OOF MSE: 78.07151037231891

Blend component summary:


,component,oof_mse,oof_mean,oof_std,test_mean,test_std,oof_path,test_path
0,safe_te_blend_anchor_69p689,78.071510,54.389900,24.623243,54.338382,24.551402,model_results/oof_blend_te_verified_noresid_we...,model_results/testpred_blend_te_verified_nores...
1,bounded31_lgbm_arcsine_l95_child60_lr02,83.952616,54.151213,25.545005,54.116908,25.367731,model_results/oof_bounded31_lgbm_arcsine_l95_c...,model_results/testpred_bounded31_lgbm_arcsine_...
2,bounded31_lgbm_logit_l95_child60_lr02,91.060335,54.186669,26.400574,54.145924,26.180803,model_results/oof_bounded31_lgbm_logit_l95_chi...,model_results/testpred_bounded31_lgbm_logit_l9...
3,linear31_pls_30,145.309431,54.122328,24.962932,54.082095,24.836099,model_results/oof_linear31_pls_30.csv,model_results/testpred_linear31_pls_30_foldavg...
4,linear31_elasticnet_a0005_l1_005,145.613008,54.404463,24.960608,54.365831,24.824835,model_results/oof_linear31_elasticnet_a0005_l1...,model_results/testpred_linear31_elasticnet_a00...
5,linear31_ridge_alpha100,145.735810,53.883836,24.988039,53.845928,24.854680,model_results/oof_linear31_ridge_alpha100.csv,model_results/testpred_linear31_ridge_alpha100...
6,linear31_ridge_alpha1000,146.095573,54.134137,24.899515,54.102894,24.780505,model_results/oof_linear31_ridge_alpha1000.csv,model_results/testpred_linear31_ridge_alpha100...



Top blend candidates:


,label,mse,w_safe_te_blend_anchor_69p689,w_linear31_ridge_alpha100,w_linear31_ridge_alpha1000,w_linear31_elasticnet_a0005_l1_005,w_linear31_pls_30,w_bounded31_lgbm_logit_l95_child60_lr02,w_bounded31_lgbm_arcsine_l95_child60_lr02
0,pair_safe_te_blend_anchor_69p689__bounded31_lg...,77.148170,0.731000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.269000
1,random_dirichlet_top_focused_50k,77.242932,0.696348,9.003037e-09,0.002565,0.006409,0.005318,0.030535,0.258825
2,pair_safe_te_blend_anchor_69p689__bounded31_lg...,77.421525,0.821000,0.000000e+00,0.000000,0.000000,0.000000,0.179000,0.000000
3,pair_safe_te_blend_anchor_69p689__linear31_pls_30,78.071510,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
4,pair_safe_te_blend_anchor_69p689__linear31_ela...,78.071510,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
5,pair_safe_te_blend_anchor_69p689__linear31_rid...,78.071510,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
6,pair_safe_te_blend_anchor_69p689__linear31_rid...,78.071510,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
7,pure_safe_te_blend_anchor_69p689,78.071510,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
8,pair_linear31_elasticnet_a0005_l1_005__bounded...,83.463816,0.000000,0.000000e+00,0.000000,0.081000,0.000000,0.000000,0.919000
9,pair_linear31_pls_30__bounded31_lgbm_arcsine_l...,83.498726,0.000000,0.000000e+00,0.000000,0.000000,0.079000,0.000000,0.921000



Best blend weights:


,component,weight,component_oof_mse
0,safe_te_blend_anchor_69p689,0.731,78.071510
1,bounded31_lgbm_arcsine_l95_child60_lr02,0.269,83.952616
2,linear31_ridge_alpha100,0.000,145.735810
3,linear31_ridge_alpha1000,0.000,146.095573
4,linear31_elasticnet_a0005_l1_005,0.000,145.613008
5,linear31_pls_30,0.000,145.309431
6,bounded31_lgbm_logit_l95_child60_lr02,0.000,91.060335



Best overnight31 blend OOF MSE: 77.1481704711914
Safe TE blend anchor OOF MSE: 78.07151037231891
Gain vs safe TE blend anchor: 0.9233399011275054

Saved blend files:
 - model_results/overnight31_linear_bounded_blend_screen.csv
 - model_results/blend_overnight31_linear_bounded_te_best_weights.csv
 - model_results/oof_blend_overnight31_linear_bounded_te_weighted.csv
 - model_results/testpred_blend_overnight31_linear_bounded_te_weighted.csv
 - submission_blend_overnight31_linear_bounded_te_weighted.csv

Blend test prediction summary:
count    48307.000000
mean        54.278812
std         24.752636
min          0.203759
1%           7.770110
5%          16.207087
25%         34.487080
50%         52.269962
75%         74.758732
95%         95.346964
99%         99.302391
max         99.993156
dtype: float64

31A overnight run finished or checkpointed.
Total elapsed seconds: 3104.5


### 31A. Linear / PLS and bounded-target TE modeling

This section tested two model-family directions on the 230-feature base + target/statistical encoding matrix.

The regularized linear and PLS models were weak as standalone predictors, with OOF MSE around 145–146, and received zero weight in the final blend. This branch is closed for now as a direct predictor branch.

The bounded-target LightGBM branch was more useful. The logit-transformed target model had OOF MSE 91.060 and did not receive final blend weight. The arcsine-square-root transformed target model had standalone OOF MSE 83.953, weaker than the current safe TE blend, but it added meaningful diversity. Blending the current safe TE blend with the arcsine model improved OOF MSE from 78.071510 to 77.148170, a gain of 0.923340.

The best saved candidate from this section is:

`submission_blend_overnight31_linear_bounded_te_weighted.csv`

This is a legitimate backlog / possible-submission candidate, but it should be audited by fold and target bucket before using a Kaggle slot.

In [80]:
# ============================================================
# 31B. Diagnostic audit for bounded-arcsine TE blend candidate
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

y_arr = np.asarray(y_train, dtype=np.float64).reshape(-1)

anchor_oof_path = "model_results/oof_blend_te_verified_noresid_weighted.csv"
arcsine_oof_path = "model_results/oof_bounded31_lgbm_arcsine_l95_child60_lr02.csv"
blend31_oof_path = "model_results/oof_blend_overnight31_linear_bounded_te_weighted.csv"

anchor_test_path = "model_results/testpred_blend_te_verified_noresid_weighted.csv"
arcsine_test_path = "model_results/testpred_bounded31_lgbm_arcsine_l95_child60_lr02_foldavg.csv"
blend31_test_path = "model_results/testpred_blend_overnight31_linear_bounded_te_weighted.csv"

required_paths = [
    anchor_oof_path, arcsine_oof_path, blend31_oof_path,
    anchor_test_path, arcsine_test_path, blend31_test_path
]

missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

def load_oof_pred_strict(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"Row mismatch for {path}: {len(df)} vs {len(y_ref)}")

    preferred = ["pred_clipped", "prediction", "pred", "OOF", "oof_pred"]

    candidate_cols = []
    for col in preferred + list(df.columns):
        if col not in df.columns:
            continue
        if col in ["row_index", "ASSESSMENT_ID", "fold"]:
            continue
        if col == "PERCENT_PROFICIENT":
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue

        arr = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
        if not np.isfinite(arr).all():
            continue

        arr = np.clip(arr, 0, 100)
        mse = mean_squared_error(y_ref, arr)

        # Safety check: reject target-like columns.
        if mse < 1e-8:
            continue

        candidate_cols.append((mse, col, arr))

    if not candidate_cols:
        raise ValueError(f"No safe prediction column found in {path}")

    candidate_cols = sorted(candidate_cols, key=lambda x: x[0])
    mse, col, arr = candidate_cols[0]
    print(f"Loaded OOF: {path}")
    print(f"  selected column: {col}")
    print(f"  OOF MSE: {mse:.6f}")

    return arr

def load_test_pred_strict(path):
    df = pd.read_csv(path)

    if "PERCENT_PROFICIENT" in df.columns:
        col = "PERCENT_PROFICIENT"
    elif "pred_clipped" in df.columns:
        col = "pred_clipped"
    elif "prediction" in df.columns:
        col = "prediction"
    elif "pred" in df.columns:
        col = "pred"
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["ASSESSMENT_ID", "row_index", "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) != 1:
            raise ValueError(f"Cannot identify test prediction column in {path}")
        col = numeric_cols[0]

    arr = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(arr).all():
        raise ValueError(f"Non-finite test predictions in {path}")

    arr = np.clip(arr, 0, 100)
    print(f"Loaded test: {path}")
    print(f"  selected column: {col}")
    return arr

anchor_oof = load_oof_pred_strict(anchor_oof_path, y_arr)
arcsine_oof = load_oof_pred_strict(arcsine_oof_path, y_arr)
blend31_oof = load_oof_pred_strict(blend31_oof_path, y_arr)

anchor_test = load_test_pred_strict(anchor_test_path)
arcsine_test = load_test_pred_strict(arcsine_test_path)
blend31_test = load_test_pred_strict(blend31_test_path)

# ------------------------------------------------------------
# Overall metrics and residual correlations
# ------------------------------------------------------------

overall_rows = []
for name, pred in [
    ("safe_te_anchor", anchor_oof),
    ("bounded_arcsine", arcsine_oof),
    ("blend31_anchor_arcsine", blend31_oof),
]:
    overall_rows.append({
        "model": name,
        "oof_mse": mean_squared_error(y_arr, pred),
        "pred_mean": np.mean(pred),
        "pred_std": np.std(pred),
        "pred_min": np.min(pred),
        "pred_max": np.max(pred),
    })

overall_df = pd.DataFrame(overall_rows)
overall_df["gain_vs_anchor"] = overall_df.loc[overall_df["model"].eq("safe_te_anchor"), "oof_mse"].iloc[0] - overall_df["oof_mse"]

print("\nOverall OOF summary:")
display(overall_df)

resid_df = pd.DataFrame({
    "anchor_resid": y_arr - anchor_oof,
    "arcsine_resid": y_arr - arcsine_oof,
    "blend31_resid": y_arr - blend31_oof,
})

print("\nResidual correlation matrix:")
display(resid_df.corr())

# ------------------------------------------------------------
# Fold stability using the same 5-fold split convention
# ------------------------------------------------------------

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fold_rows = []
for fold_num, (_, va_idx) in enumerate(kf.split(np.arange(len(y_arr))), start=1):
    y_f = y_arr[va_idx]
    anchor_mse = mean_squared_error(y_f, anchor_oof[va_idx])
    arcsine_mse = mean_squared_error(y_f, arcsine_oof[va_idx])
    blend_mse = mean_squared_error(y_f, blend31_oof[va_idx])

    fold_rows.append({
        "fold": fold_num,
        "anchor_mse": anchor_mse,
        "arcsine_mse": arcsine_mse,
        "blend31_mse": blend_mse,
        "blend_gain_vs_anchor": anchor_mse - blend_mse,
        "arcsine_gain_vs_anchor": anchor_mse - arcsine_mse,
    })

fold_df = pd.DataFrame(fold_rows)

print("\nFold-level diagnostic:")
display(fold_df)

print("\nFold gain summary:")
display(fold_df[["blend_gain_vs_anchor", "arcsine_gain_vs_anchor"]].describe())

# ------------------------------------------------------------
# Error by true-target bucket
# ------------------------------------------------------------

bins = [-0.001, 5, 10, 20, 35, 50, 65, 80, 90, 95, 100.001]
labels = ["0-5", "5-10", "10-20", "20-35", "35-50", "50-65", "65-80", "80-90", "90-95", "95-100"]

bucket = pd.cut(y_arr, bins=bins, labels=labels, include_lowest=True)

bucket_rows = []
for b in labels:
    idx = np.asarray(bucket == b)
    if idx.sum() == 0:
        continue

    anchor_mse = mean_squared_error(y_arr[idx], anchor_oof[idx])
    arcsine_mse = mean_squared_error(y_arr[idx], arcsine_oof[idx])
    blend_mse = mean_squared_error(y_arr[idx], blend31_oof[idx])

    bucket_rows.append({
        "target_bucket": b,
        "n": int(idx.sum()),
        "anchor_mse": anchor_mse,
        "arcsine_mse": arcsine_mse,
        "blend31_mse": blend_mse,
        "blend_gain_vs_anchor": anchor_mse - blend_mse,
        "arcsine_gain_vs_anchor": anchor_mse - arcsine_mse,
        "anchor_pred_mean": np.mean(anchor_oof[idx]),
        "arcsine_pred_mean": np.mean(arcsine_oof[idx]),
        "blend31_pred_mean": np.mean(blend31_oof[idx]),
        "true_mean": np.mean(y_arr[idx]),
    })

bucket_df = pd.DataFrame(bucket_rows)

print("\nTarget-bucket diagnostic:")
display(bucket_df)

# ------------------------------------------------------------
# Test prediction shift vs current safe TE anchor
# ------------------------------------------------------------

delta_test = blend31_test - anchor_test

test_shift = pd.Series(delta_test).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
test_summary = pd.DataFrame({
    "safe_te_anchor": pd.Series(anchor_test).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]),
    "bounded_arcsine": pd.Series(arcsine_test).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]),
    "blend31_anchor_arcsine": pd.Series(blend31_test).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]),
})

print("\nTest prediction summaries:")
display(test_summary)

print("\nBlend31 minus safe-anchor test shift:")
display(test_shift)

print("\nLargest absolute test shifts:")
largest_shift = pd.DataFrame({
    "ASSESSMENT_ID": np.asarray(test_ids),
    "anchor_pred": anchor_test,
    "arcsine_pred": arcsine_test,
    "blend31_pred": blend31_test,
    "blend31_minus_anchor": delta_test,
    "abs_shift": np.abs(delta_test),
}).sort_values("abs_shift", ascending=False).head(20)

display(largest_shift)

# Save diagnostics.
os.makedirs("model_results", exist_ok=True)
overall_df.to_csv("model_results/diagnostic31b_overall_summary.csv", index=False)
fold_df.to_csv("model_results/diagnostic31b_fold_summary.csv", index=False)
bucket_df.to_csv("model_results/diagnostic31b_target_bucket_summary.csv", index=False)
largest_shift.to_csv("model_results/diagnostic31b_largest_test_shifts.csv", index=False)

print("\nSaved:")
print(" - model_results/diagnostic31b_overall_summary.csv")
print(" - model_results/diagnostic31b_fold_summary.csv")
print(" - model_results/diagnostic31b_target_bucket_summary.csv")
print(" - model_results/diagnostic31b_largest_test_shifts.csv")

Loaded OOF: model_results/oof_blend_te_verified_noresid_weighted.csv
  selected column: pred_clipped
  OOF MSE: 78.071510
Loaded OOF: model_results/oof_bounded31_lgbm_arcsine_l95_child60_lr02.csv
  selected column: pred_clipped
  OOF MSE: 83.952616
Loaded OOF: model_results/oof_blend_overnight31_linear_bounded_te_weighted.csv
  selected column: pred_clipped
  OOF MSE: 77.148170
Loaded test: model_results/testpred_blend_te_verified_noresid_weighted.csv
  selected column: PERCENT_PROFICIENT
Loaded test: model_results/testpred_bounded31_lgbm_arcsine_l95_child60_lr02_foldavg.csv
  selected column: PERCENT_PROFICIENT
Loaded test: model_results/testpred_blend_overnight31_linear_bounded_te_weighted.csv
  selected column: PERCENT_PROFICIENT

Overall OOF summary:


,model,oof_mse,pred_mean,pred_std,pred_min,pred_max,gain_vs_anchor
0,safe_te_anchor,78.071510,54.389900,24.623243,0.000010,100.0,0.000000
1,bounded_arcsine,83.952616,54.151213,25.545005,0.000000,100.0,-5.881106
2,blend31_anchor_arcsine,77.148170,54.325693,24.824370,0.000472,100.0,0.923340



Residual correlation matrix:


,anchor_resid,arcsine_resid,blend31_resid
anchor_resid,1.000000,0.922311,0.994101
arcsine_resid,0.922311,1.000000,0.958783
blend31_resid,0.994101,0.958783,1.000000



Fold-level diagnostic:


...


Blend31 minus safe-anchor test shift:


count    48307.000000
mean        -0.059576
std          0.622474
min         -5.561236
1%          -1.757797
5%          -1.049059
25%         -0.389225
50%         -0.049075
75%          0.278310
95%          0.892381
99%          1.605285
max          7.018882
dtype: float64


Largest absolute test shifts:


,ASSESSMENT_ID,anchor_pred,arcsine_pred,blend31_pred,blend31_minus_anchor,abs_shift
37324,134f4d188ca1,47.901780,73.994280,54.920662,7.018882,7.018882
39326,f34edb0b05a9,53.541985,76.690750,59.769000,6.227015,6.227015
26139,3f4917dcf272,25.784653,5.110918,20.223417,-5.561236,5.561236
45475,b24a2d91aa05,31.971199,11.603750,26.492355,-5.478844,5.478844
21685,c461d73c67ca,26.207335,7.858086,21.271387,-4.935948,4.935948
45479,df0d0fc670f0,77.319090,94.377840,81.907900,4.588810,4.588810
34494,3034e4e5731b,76.954414,93.554344,81.419790,4.465376,4.465376
9217,37b0d0836a98,67.641380,83.806440,71.989784,4.348404,4.348404
42135,0adcae119cdd,31.371962,15.539269,27.112967,-4.258995,4.258995
47648,c14bca56d7fb,57.170580,41.698410,53.008568,-4.162012,4.162012



Saved:
 - model_results/diagnostic31b_overall_summary.csv
 - model_results/diagnostic31b_fold_summary.csv
 - model_results/diagnostic31b_target_bucket_summary.csv
 - model_results/diagnostic31b_largest_test_shifts.csv


### 31B. Audit of bounded-arcsine TE blend

The bounded-arcsine LightGBM model was weaker than the safe TE blend as a standalone model, with OOF MSE 83.953 versus 78.072 for the safe TE anchor. However, it added useful diversity when blended with the safe TE anchor.

The best blend used approximately:

- 0.731 safe TE blend
- 0.269 bounded-arcsine LightGBM

This improved OOF MSE from 78.071510 to 77.148170, a gain of 0.923340.

The gain was positive on all five folds, with fold gains between about 0.56 and 1.37 MSE. Test prediction shifts were moderate overall, with a median shift near zero and only a small number of rows moving by more than 4–5 points.

The target-bucket diagnostic showed that the blend mainly helps near the low and high ends of the target range, while slightly hurting some middle target buckets. Overall, the fold-stable OOF gain and sane test distribution make this a legitimate Kaggle candidate.

Candidate file:

`submission_blend_overnight31_linear_bounded_te_weighted.csv`

### Kaggle checkpoint: bounded-arcsine TE blend

Submitted:

`submission_blend_overnight31_linear_bounded_te_weighted.csv`

Public MSE:

**68.986**

This improved over the previous public best:

`submission_blend_te_verified_noresid_weighted.csv`

Public MSE:

**69.689**

The improvement was consistent with the OOF audit. Locally, the blend improved OOF MSE from 78.071510 to 77.148170, a gain of 0.923340, with positive gain on all five folds.

Interpretation:

The bounded-arcsine LightGBM model was weaker than the safe TE blend as a standalone model, but it added useful diversity. Its main value was correcting boundary behavior near 0 and 100. The target-bucket diagnostic showed gains in the low and high target ranges, with some tradeoff in the middle buckets. The public improvement confirms that bounded-target modeling is a useful branch for this competition.

Current best public submission:

`submission_blend_overnight31_linear_bounded_te_weighted.csv`

Current best public MSE:

**68.986**

### 32A. Why did the bounded-arcsine blend improve publicly?

The bounded-arcsine TE blend improved the public score from 69.689 to 68.986.

This diagnostic investigates where the improvement came from. Since test labels are hidden, the analysis uses OOF rows to estimate which segments benefited locally, then compares those same segments against the test prediction distribution.

The main questions are:

1. Did the arcsine blend help mostly near the 0 and 100 boundaries?
2. Did it help specific assessments or subgroups?
3. Are the test rows concentrated in segments where OOF suggested the arcsine blend helps?
4. Are the prediction shifts moderate and directionally sensible?

In [82]:
# ============================================================
# 32A. Diagnostic: why did bounded-arcsine TE blend improve?
# ============================================================

import os
import re
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

y_arr = np.asarray(y_train, dtype=np.float64).reshape(-1)
test_ids_arr = np.asarray(test_ids)

anchor_oof_path = "model_results/oof_blend_te_verified_noresid_weighted.csv"
arcsine_oof_path = "model_results/oof_bounded31_lgbm_arcsine_l95_child60_lr02.csv"
blend_oof_path = "model_results/oof_blend_overnight31_linear_bounded_te_weighted.csv"

anchor_test_path = "model_results/testpred_blend_te_verified_noresid_weighted.csv"
arcsine_test_path = "model_results/testpred_bounded31_lgbm_arcsine_l95_child60_lr02_foldavg.csv"
blend_test_path = "model_results/testpred_blend_overnight31_linear_bounded_te_weighted.csv"

required_paths = [
    anchor_oof_path, arcsine_oof_path, blend_oof_path,
    anchor_test_path, arcsine_test_path, blend_test_path,
]

missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

def load_oof_pred(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: {len(df)} vs {len(y_ref)}")

    candidate_cols = []
    preferred = ["pred_clipped", "prediction", "pred", "oof_pred", "OOF"]

    for col in preferred + list(df.columns):
        if col not in df.columns:
            continue
        if col in ["row_index", "ASSESSMENT_ID", "fold", "PERCENT_PROFICIENT"]:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue

        arr = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
        if not np.isfinite(arr).all():
            continue

        arr = np.clip(arr, 0, 100)
        mse = mean_squared_error(y_ref, arr)

        # Safety: reject target-like columns.
        if mse < 1e-8:
            continue

        candidate_cols.append((mse, col, arr))

    if not candidate_cols:
        raise ValueError(f"No safe OOF prediction column found in {path}")

    candidate_cols = sorted(candidate_cols, key=lambda x: x[0])
    mse, col, arr = candidate_cols[0]

    print(f"Loaded OOF: {path}")
    print(f"  selected column: {col}")
    print(f"  OOF MSE: {mse:.6f}")

    return arr

def load_test_pred(path, test_ids_ref):
    df = pd.read_csv(path)

    if "ASSESSMENT_ID" in df.columns:
        if len(df) == len(test_ids_ref) and np.array_equal(df["ASSESSMENT_ID"].to_numpy(), test_ids_ref):
            aligned = df.copy()
        else:
            aligned = pd.DataFrame({"ASSESSMENT_ID": test_ids_ref}).merge(df, on="ASSESSMENT_ID", how="left")
    else:
        aligned = df.copy()

    if len(aligned) != len(test_ids_ref):
        raise ValueError(f"Test row mismatch for {path}: {len(aligned)} vs {len(test_ids_ref)}")

    preferred = ["PERCENT_PROFICIENT", "pred_clipped", "prediction", "pred", "test_pred"]

    for col in preferred:
        if col in aligned.columns and pd.api.types.is_numeric_dtype(aligned[col]):
            arr = pd.to_numeric(aligned[col], errors="coerce").to_numpy(dtype=np.float64)
            if np.isfinite(arr).all():
                print(f"Loaded test: {path}")
                print(f"  selected column: {col}")
                return np.clip(arr, 0, 100)

    numeric_cols = [
        c for c in aligned.columns
        if c not in ["ASSESSMENT_ID", "row_index", "fold"]
        and pd.api.types.is_numeric_dtype(aligned[c])
    ]

    if len(numeric_cols) != 1:
        raise ValueError(f"Could not identify test prediction column in {path}")

    col = numeric_cols[0]
    arr = pd.to_numeric(aligned[col], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(arr).all():
        raise ValueError(f"Non-finite test predictions in {path}")

    print(f"Loaded test: {path}")
    print(f"  selected column: {col}")
    return np.clip(arr, 0, 100)

anchor_oof = load_oof_pred(anchor_oof_path, y_arr)
arcsine_oof = load_oof_pred(arcsine_oof_path, y_arr)
blend_oof = load_oof_pred(blend_oof_path, y_arr)

anchor_test = load_test_pred(anchor_test_path, test_ids_arr)
arcsine_test = load_test_pred(arcsine_test_path, test_ids_arr)
blend_test = load_test_pred(blend_test_path, test_ids_arr)

# ------------------------------------------------------------
# 1. Row-level OOF diagnostics
# ------------------------------------------------------------

diag = pd.DataFrame({
    "y": y_arr,
    "anchor": anchor_oof,
    "arcsine": arcsine_oof,
    "blend": blend_oof,
})

diag["anchor_se"] = (diag["y"] - diag["anchor"]) ** 2
diag["arcsine_se"] = (diag["y"] - diag["arcsine"]) ** 2
diag["blend_se"] = (diag["y"] - diag["blend"]) ** 2

diag["anchor_resid"] = diag["y"] - diag["anchor"]
diag["arcsine_minus_anchor"] = diag["arcsine"] - diag["anchor"]
diag["blend_minus_anchor"] = diag["blend"] - diag["anchor"]

diag["gain_blend_vs_anchor"] = diag["anchor_se"] - diag["blend_se"]
diag["gain_arcsine_vs_anchor"] = diag["anchor_se"] - diag["arcsine_se"]

diag["abs_anchor_error"] = np.abs(diag["anchor_resid"])
diag["abs_blend_shift"] = np.abs(diag["blend_minus_anchor"])

# Directional check:
# If anchor underpredicts, positive shift helps.
# If anchor overpredicts, negative shift helps.
diag["shift_direction_helped_anchor"] = np.sign(diag["blend_minus_anchor"]) == np.sign(diag["anchor_resid"])
# Directional check:
# If anchor underpredicts, positive shift helps.
# If anchor overpredicts, negative shift helps.
# Use float so we can store NaN safely.
diag["shift_direction_helped_anchor"] = np.where(
    diag["blend_minus_anchor"].abs() < 1e-12,
    np.nan,
    (
        np.sign(diag["blend_minus_anchor"].to_numpy())
        == np.sign(diag["anchor_resid"].to_numpy())
    ).astype(float)
)

test_diag = pd.DataFrame({
    "ASSESSMENT_ID": test_ids_arr,
    "anchor": anchor_test,
    "arcsine": arcsine_test,
    "blend": blend_test,
})

test_diag["arcsine_minus_anchor"] = test_diag["arcsine"] - test_diag["anchor"]
test_diag["blend_minus_anchor"] = test_diag["blend"] - test_diag["anchor"]
test_diag["abs_blend_shift"] = np.abs(test_diag["blend_minus_anchor"])

print("\nOverall OOF metrics:")
overall = pd.DataFrame([
    {
        "model": "safe_te_anchor",
        "oof_mse": mean_squared_error(y_arr, anchor_oof),
        "pred_mean": anchor_oof.mean(),
        "pred_std": anchor_oof.std(),
    },
    {
        "model": "bounded_arcsine",
        "oof_mse": mean_squared_error(y_arr, arcsine_oof),
        "pred_mean": arcsine_oof.mean(),
        "pred_std": arcsine_oof.std(),
    },
    {
        "model": "anchor_arcsine_blend",
        "oof_mse": mean_squared_error(y_arr, blend_oof),
        "pred_mean": blend_oof.mean(),
        "pred_std": blend_oof.std(),
    },
])
overall["gain_vs_anchor"] = overall.loc[overall["model"].eq("safe_te_anchor"), "oof_mse"].iloc[0] - overall["oof_mse"]
display(overall)

print("\nRow-level alignment diagnostics:")
alignment = pd.DataFrame({
    "metric": [
        "corr(anchor_residual, blend_shift)",
        "corr(anchor_residual, arcsine_minus_anchor)",
        "mean_abs_blend_shift",
        "median_abs_blend_shift",
        "share_shift_direction_helped_anchor",
        "OOF_gain_blend_vs_anchor",
    ],
    "value": [
        diag["anchor_resid"].corr(diag["blend_minus_anchor"]),
        diag["anchor_resid"].corr(diag["arcsine_minus_anchor"]),
        diag["abs_blend_shift"].mean(),
        diag["abs_blend_shift"].median(),
        diag["shift_direction_helped_anchor"].mean(),
        diag["gain_blend_vs_anchor"].mean(),
    ],
})
display(alignment)

# ------------------------------------------------------------
# 2. Buckets available on both OOF and test
# ------------------------------------------------------------

bucket_edges = [-0.001, 5, 10, 20, 35, 50, 65, 80, 90, 95, 100.001]
bucket_labels = ["0-5", "5-10", "10-20", "20-35", "35-50", "50-65", "65-80", "80-90", "90-95", "95-100"]

diag["true_target_bucket"] = pd.cut(diag["y"], bins=bucket_edges, labels=bucket_labels, include_lowest=True)
diag["anchor_pred_bucket"] = pd.cut(diag["anchor"], bins=bucket_edges, labels=bucket_labels, include_lowest=True)
test_diag["anchor_pred_bucket"] = pd.cut(test_diag["anchor"], bins=bucket_edges, labels=bucket_labels, include_lowest=True)

# Error-size bucket is useful because bounded corrections may help rows where the anchor was uncertain/wrong.
diag["anchor_abs_error_bucket"] = pd.qcut(
    diag["abs_anchor_error"],
    q=[0, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00],
    labels=["0-50%", "50-75%", "75-90%", "90-95%", "95-99%", "99-100%"],
    duplicates="drop",
)

def summarize_train_only(col):
    out = (
        diag
        .groupby(col, dropna=False)
        .agg(
            n_train=("y", "size"),
            y_mean=("y", "mean"),
            anchor_mse=("anchor_se", "mean"),
            arcsine_mse=("arcsine_se", "mean"),
            blend_mse=("blend_se", "mean"),
            gain_blend_vs_anchor=("gain_blend_vs_anchor", "mean"),
            gain_arcsine_vs_anchor=("gain_arcsine_vs_anchor", "mean"),
            anchor_pred_mean=("anchor", "mean"),
            arcsine_pred_mean=("arcsine", "mean"),
            blend_pred_mean=("blend", "mean"),
            mean_blend_shift=("blend_minus_anchor", "mean"),
            mean_abs_blend_shift=("abs_blend_shift", "mean"),
            shift_help_rate=("shift_direction_helped_anchor", "mean"),
        )
        .reset_index()
    )

    out["train_share"] = out["n_train"] / len(diag)
    out["oof_gain_contribution"] = out["gain_blend_vs_anchor"] * out["train_share"]
    return out.sort_values("oof_gain_contribution", ascending=False).reset_index(drop=True)

print("\nTrue-target bucket contribution summary:")
true_bucket_summary = summarize_train_only("true_target_bucket")
display(true_bucket_summary)

print("\nAnchor absolute-error bucket summary:")
error_bucket_summary = summarize_train_only("anchor_abs_error_bucket")
display(error_bucket_summary)

# ------------------------------------------------------------
# 3. Attach raw columns if available
# ------------------------------------------------------------

raw_train = globals().get("raw_train_te", None)
raw_test = globals().get("raw_test_te", None)

segment_cols = ["anchor_pred_bucket"]

if raw_train is not None and raw_test is not None and len(raw_train) == len(diag) and len(raw_test) == len(test_diag):
    raw_train_reset = raw_train.reset_index(drop=True)
    raw_test_reset = raw_test.reset_index(drop=True)

    base_raw_cols = [
        "ASSESSMENT_NAME",
        "SUBGROUP_NAME",
        "REGION",
        "DISTRICT_TYPE",
        "SCHOOL",
        "DISTRICT",
        "COUNTY",
        "N_STUDENTS_BIN",
    ]

    for col in base_raw_cols:
        if col in raw_train_reset.columns and col in raw_test_reset.columns:
            diag[col] = raw_train_reset[col].astype(str).fillna("MISSING")
            test_diag[col] = raw_test_reset[col].astype(str).fillna("MISSING")
            segment_cols.append(col)

    # If N_STUDENTS_BIN does not exist but N_STUDENTS exists, create a simple train-quantile bin.
    if "N_STUDENTS_BIN" not in diag.columns and "N_STUDENTS" in raw_train_reset.columns and "N_STUDENTS" in raw_test_reset.columns:
        train_n = pd.to_numeric(raw_train_reset["N_STUDENTS"], errors="coerce")
        test_n = pd.to_numeric(raw_test_reset["N_STUDENTS"], errors="coerce")

        qs = np.nanquantile(train_n, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
        qs = np.unique(qs)

        if len(qs) >= 3:
            diag["N_STUDENTS_BIN"] = pd.cut(train_n, bins=qs, include_lowest=True).astype(str).fillna("MISSING")
            test_diag["N_STUDENTS_BIN"] = pd.cut(test_n, bins=qs, include_lowest=True).astype(str).fillna("MISSING")
            segment_cols.append("N_STUDENTS_BIN")

    # Compact compound keys.
    compound_specs = [
        ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
        ("ASSESSMENT_NAME", "N_STUDENTS_BIN"),
        ("SUBGROUP_NAME", "N_STUDENTS_BIN"),
    ]

    for a, b in compound_specs:
        if a in diag.columns and b in diag.columns and a in test_diag.columns and b in test_diag.columns:
            new_col = f"{a}__{b}"
            diag[new_col] = diag[a].astype(str) + " | " + diag[b].astype(str)
            test_diag[new_col] = test_diag[a].astype(str) + " | " + test_diag[b].astype(str)
            segment_cols.append(new_col)

else:
    print("\nraw_train_te/raw_test_te not available or not aligned. Will only summarize prediction buckets.")

# Keep the displayed segment list focused.
preferred_segment_cols = [
    "anchor_pred_bucket",
    "ASSESSMENT_NAME",
    "SUBGROUP_NAME",
    "ASSESSMENT_NAME__SUBGROUP_NAME",
    "N_STUDENTS_BIN",
    "ASSESSMENT_NAME__N_STUDENTS_BIN",
    "SUBGROUP_NAME__N_STUDENTS_BIN",
    "REGION",
    "DISTRICT_TYPE",
]

segment_cols_to_show = [c for c in preferred_segment_cols if c in segment_cols]

print("\nSegment columns available for train/test comparison:")
print(segment_cols_to_show)

# ------------------------------------------------------------
# 4. Segment summaries: OOF gain + test representation
# ------------------------------------------------------------

def safe_filename_piece(x):
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(x)).strip("_").lower()

def summarize_train_test_segment(col, min_train=150, min_test=50):
    train_group = (
        diag
        .groupby(col, dropna=False)
        .agg(
            n_train=("y", "size"),
            y_mean=("y", "mean"),
            anchor_mse=("anchor_se", "mean"),
            arcsine_mse=("arcsine_se", "mean"),
            blend_mse=("blend_se", "mean"),
            gain_blend_vs_anchor=("gain_blend_vs_anchor", "mean"),
            gain_arcsine_vs_anchor=("gain_arcsine_vs_anchor", "mean"),
            anchor_pred_mean=("anchor", "mean"),
            arcsine_pred_mean=("arcsine", "mean"),
            blend_pred_mean=("blend", "mean"),
            mean_blend_shift=("blend_minus_anchor", "mean"),
            mean_abs_blend_shift=("abs_blend_shift", "mean"),
            shift_help_rate=("shift_direction_helped_anchor", "mean"),
        )
        .reset_index()
    )

    train_group["train_share"] = train_group["n_train"] / len(diag)
    train_group["oof_gain_contribution"] = train_group["gain_blend_vs_anchor"] * train_group["train_share"]

    test_group = (
        test_diag
        .groupby(col, dropna=False)
        .agg(
            n_test=("anchor", "size"),
            test_anchor_mean=("anchor", "mean"),
            test_arcsine_mean=("arcsine", "mean"),
            test_blend_mean=("blend", "mean"),
            test_mean_blend_shift=("blend_minus_anchor", "mean"),
            test_mean_abs_blend_shift=("abs_blend_shift", "mean"),
            test_max_abs_blend_shift=("abs_blend_shift", "max"),
        )
        .reset_index()
    )

    test_group["test_share"] = test_group["n_test"] / len(test_diag)

    merged = train_group.merge(test_group, on=col, how="outer")

    # A rough proxy: if a segment gained in OOF and is common in test, it may explain public improvement.
    merged["test_weighted_gain_proxy"] = merged["gain_blend_vs_anchor"] * merged["test_share"]

    merged = merged[
        (merged["n_train"].fillna(0) >= min_train) &
        (merged["n_test"].fillna(0) >= min_test)
    ].copy()

    merged = merged.sort_values("test_weighted_gain_proxy", ascending=False).reset_index(drop=True)

    return merged

combined_segment_summaries = []

for col in segment_cols_to_show:
    print("\n" + "=" * 90)
    print(f"Segment diagnostic: {col}")
    print("=" * 90)

    summary = summarize_train_test_segment(col)

    if len(summary) == 0:
        print(f"No segments passed minimum-count filters for {col}.")
        continue

    summary.insert(0, "segment_type", col)
    combined_segment_summaries.append(summary)

    save_path = f"model_results/diagnostic32a_segment_{safe_filename_piece(col)}.csv"
    summary.to_csv(save_path, index=False)

    print(f"Saved: {save_path}")

    print("\nTop OOF-positive / test-relevant segments:")
    display(summary.head(12))

    print("\nMost OOF-negative / test-relevant segments:")
    display(summary.sort_values("test_weighted_gain_proxy", ascending=True).head(8))

if combined_segment_summaries:
    combined = pd.concat(combined_segment_summaries, ignore_index=True)
    combined_path = "model_results/diagnostic32a_all_segment_summaries.csv"
    combined.to_csv(combined_path, index=False)

    print("\n" + "=" * 90)
    print("Best segment explanations across all segment types")
    print("=" * 90)

    print("\nTop positive test-weighted gain proxies:")
    display(combined.sort_values("test_weighted_gain_proxy", ascending=False).head(25))

    print("\nTop negative test-weighted gain proxies:")
    display(combined.sort_values("test_weighted_gain_proxy", ascending=True).head(15))

    print("Saved combined segment summary:", combined_path)

# ------------------------------------------------------------
# 5. Test shift distribution
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("Test shift summary")
print("=" * 90)

test_shift_summary = pd.Series(test_diag["blend_minus_anchor"]).describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

display(test_shift_summary)

largest_test_shifts = test_diag.assign(
    abs_shift=test_diag["abs_blend_shift"]
).sort_values("abs_shift", ascending=False).head(30)

largest_shift_path = "model_results/diagnostic32a_largest_test_shifts.csv"
largest_test_shifts.to_csv(largest_shift_path, index=False)

print("Saved largest test shifts:", largest_shift_path)
display(largest_test_shifts)

print("\n32A diagnostic complete.")

Loaded OOF: model_results/oof_blend_te_verified_noresid_weighted.csv
  selected column: pred_clipped
  OOF MSE: 78.071510
Loaded OOF: model_results/oof_bounded31_lgbm_arcsine_l95_child60_lr02.csv
  selected column: pred_clipped
  OOF MSE: 83.952616
Loaded OOF: model_results/oof_blend_overnight31_linear_bounded_te_weighted.csv
  selected column: pred_clipped
  OOF MSE: 77.148170
Loaded test: model_results/testpred_blend_te_verified_noresid_weighted.csv
  selected column: PERCENT_PROFICIENT
Loaded test: model_results/testpred_bounded31_lgbm_arcsine_l95_child60_lr02_foldavg.csv
  selected column: PERCENT_PROFICIENT
Loaded test: model_results/testpred_blend_overnight31_linear_bounded_te_weighted.csv
  selected column: PERCENT_PROFICIENT

Overall OOF metrics:


,model,oof_mse,pred_mean,pred_std,gain_vs_anchor
0,safe_te_anchor,78.071510,54.389900,24.623243,0.000000
1,bounded_arcsine,83.952616,54.151213,25.545005,-5.881106
2,anchor_arcsine_blend,77.148170,54.325693,24.824370,0.923340



Row-level alignment diagnostics:


,metric,value
0,"corr(anchor_residual, blend_shift)",0.107459
1,"corr(anchor_residual, arcsine_minus_anchor)",0.107459
2,mean_abs_blend_shift,0.711517
3,median_abs_blend_shift,0.541110
4,share_shift_direction_helped_anchor,0.541440
5,OOF_gain_blend_vs_anchor,0.923340



True-target bucket contribution summary:


...

Saved combined segment summary: model_results/diagnostic32a_all_segment_summaries.csv

Test shift summary


count    48307.000000
mean        -0.059576
std          0.622474
min         -5.561236
1%          -1.757797
5%          -1.049059
25%         -0.389225
50%         -0.049075
75%          0.278310
95%          0.892381
99%          1.605285
max          7.018882
Name: blend_minus_anchor, dtype: float64

Saved largest test shifts: model_results/diagnostic32a_largest_test_shifts.csv


,ASSESSMENT_ID,anchor,arcsine,blend,arcsine_minus_anchor,blend_minus_anchor,abs_blend_shift,anchor_pred_bucket,ASSESSMENT_NAME,SUBGROUP_NAME,REGION,DISTRICT_TYPE,SCHOOL,DISTRICT,COUNTY,N_STUDENTS_BIN,ASSESSMENT_NAME__SUBGROUP_NAME,ASSESSMENT_NAME__N_STUDENTS_BIN,SUBGROUP_NAME__N_STUDENTS_BIN,abs_shift
37324,134f4d188ca1,47.901780,73.994280,54.920662,26.092500,7.018882,7.018882,35-50,RegentsScience8,All Students,Southern Tier,Average Need,5c1f9619,549eb20a,49223df1,"(4.999, 18.0]",RegentsScience8 | All Students,"RegentsScience8 | (4.999, 18.0]","All Students | (4.999, 18.0]",7.018882
39326,f34edb0b05a9,53.541985,76.690750,59.769000,23.148765,6.227015,6.227015,50-65,RegentsMath8,All Students,Southern Tier,Average Need,5c1f9619,549eb20a,49223df1,"(4.999, 18.0]",RegentsMath8 | All Students,"RegentsMath8 | (4.999, 18.0]","All Students | (4.999, 18.0]",6.227015
26139,3f4917dcf272,25.784653,5.110918,20.223417,-20.673735,-5.561236,5.561236,20-35,Regents Phy Set/Chemistry,Not Economically Disadvantaged,New York City,Charter School,06c1fc22,03759bb1,062c154c,"(4.999, 18.0]",Regents Phy Set/Chemistry | Not Economically D...,"Regents Phy Set/Chemistry | (4.999, 18.0]","Not Economically Disadvantaged | (4.999, 18.0]",5.561236
45475,b24a2d91aa05,31.971199,11.603750,26.492355,-20.367449,-5.478844,5.478844,20-35,ELA6,Female,Central New York,Average Need,011a9e5d,8c25b58e,fd1f6ae1,"(4.999, 18.0]",ELA6 | Female,"ELA6 | (4.999, 18.0]","Female | (4.999, 18.0]",5.478844
21685,c461d73c67ca,26.207335,7.858086,21.271387,-18.349249,-4.935948,4.935948,20-35,Regents Phy Set/Physics,Not Economically Disadvantaged,New York City,NYC,00f3540f,5ccd3c25,8af58576,"(4.999, 18.0]",Regents Phy Set/Physics | Not Economically Dis...,"Regents Phy Set/Physics | (4.999, 18.0]","Not Economically Disadvantaged | (4.999, 18.0]",4.935948
45479,df0d0fc670f0,77.319090,94.377840,81.907900,17.058750,4.588810,4.588810,65-80,Regents Phy Set/Chemistry,Economically Disadvantaged,Long Island,Low Need,1a74f953,ef70b3e0,9d7af256,"(4.999, 18.0]",Regents Phy Set/Chemistry | Economically Disad...,"Regents Phy Set/Chemistry | (4.999, 18.0]","Economically Disadvantaged | (4.999, 18.0]",4.588810
34494,3034e4e5731b,76.954414,93.554344,81.419790,16.599930,4.465376,4.465376,65-80,ELA5,Not Economically Disadvantaged,New York City,Charter School,ce560b5e,03759bb1,613b61ae,"(4.999, 18.0]",ELA5 | Not Economically Disadvantaged,"ELA5 | (4.999, 18.0]","Not Economically Disadvantaged | (4.999, 18.0]",4.465376
9217,37b0d0836a98,67.641380,83.806440,71.989784,16.165060,4.348404,4.348404,65-80,MATH5,Female,New York City,Charter School,68ad687e,03759bb1,613b61ae,"(18.0, 30.0]",MATH5 | Female,"MATH5 | (18.0, 30.0]","Female | (18.0, 30.0]",4.348404
42135,0adcae119cdd,31.371962,15.539269,27.112967,-15.832693,-4.258995,4.258995,20-35,Regents Common Core Algebra I,All Students,New York City,Charter School,e5e57648,03759bb1,613b61ae,"(4.999, 18.0]",Regents Common Core Algebra I | All Students,"Regents Common Core Algebra I | (4.999, 18.0]","All Students | (4.999, 18.0]",4.258995
47648,c14bca56d7fb,57.170580,41.698410,53.008568,-15.472170,-4.162012,4.162012,50-65,ELA4,All Students,New York City,Charter School,dd34faa6,03759bb1,613b61ae,"(47.0, 80.0]",ELA4 | All Students,"ELA4 | (47.0, 80.0]","All Students | (47.0, 80.0]",4.162012



32A diagnostic complete.


The arcsine target transform helped because percentages from small student counts are noisy and bounded. The transform gives a different error geometry that blends well with the TE anchor, especially for small cohorts and boundary-sensitive rows.

In [83]:
# ============================================================
# 33A. Segment-adaptive arcsine blend candidate
# ============================================================
# Builds a new candidate from:
#   safe_TE_anchor + segment_weight * (bounded_arcsine - safe_TE_anchor)
#
# This uses the 32A objects directly. No artifact reloads, no inventory checks.
# The segment weights are cross-fitted, then refit on all train rows for test.
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = 9890
N_SPLITS = 5

y = np.asarray(y_arr, dtype=float)
base_oof = np.asarray(anchor_oof, dtype=float)
arc_oof = np.asarray(arcsine_oof, dtype=float)
current_oof = np.asarray(blend_oof, dtype=float)

base_test = np.asarray(anchor_test, dtype=float)
arc_test = np.asarray(arcsine_test, dtype=float)

delta_oof = arc_oof - base_oof
delta_test = arc_test - base_test

current_mse = mean_squared_error(y, current_oof)
base_mse = mean_squared_error(y, base_oof)
arc_mse = mean_squared_error(y, arc_oof)

print("33A. Segment-adaptive arcsine blend")
print("----------------------------------")
print(f"Safe TE anchor OOF MSE:      {base_mse:.6f}")
print(f"Bounded arcsine OOF MSE:     {arc_mse:.6f}")
print(f"Current 68.986 OOF MSE:      {current_mse:.6f}")

def mse(pred):
    return mean_squared_error(y, np.clip(pred, 0, 100))

def best_weight(rows, low=0.0, high=0.70):
    d = delta_oof[rows]
    denom = np.dot(d, d)
    if denom <= 1e-12:
        return 0.0
    w = np.dot(d, y[rows] - base_oof[rows]) / denom
    return float(np.clip(w, low, high))

def make_segment_values(cols, frame):
    vals = frame[cols[0]].astype(str).copy()
    for c in cols[1:]:
        vals = vals + " | " + frame[c].astype(str)
    return vals.to_numpy(dtype=object)

def fit_segment_weights(seg, fit_rows, min_count, shrink):
    global_w = best_weight(fit_rows)

    tmp = pd.DataFrame({
        "seg": seg[fit_rows],
        "row": fit_rows,
    })

    weights = {}
    counts = tmp["seg"].value_counts()

    for s, n in counts.items():
        if n < min_count:
            continue

        rows = tmp.loc[tmp["seg"].eq(s), "row"].to_numpy()
        raw_w = best_weight(rows)
        shrink_factor = n / (n + shrink)

        weights[s] = global_w + shrink_factor * (raw_w - global_w)

    return weights, global_w

def predict_with_weights(seg, rows, weights, global_w):
    w = pd.Series(seg[rows]).map(weights).fillna(global_w).to_numpy(dtype=float)
    pred = base_oof[rows] + w * delta_oof[rows]
    return np.clip(pred, 0, 100), w

segment_specs = {
    "pred_bucket": ["anchor_pred_bucket"],
    "n_students": ["N_STUDENTS_BIN"],
    "assessment": ["ASSESSMENT_NAME"],
    "subgroup": ["SUBGROUP_NAME"],
    "assessment_subgroup": ["ASSESSMENT_NAME", "SUBGROUP_NAME"],
    "assessment_n": ["ASSESSMENT_NAME", "N_STUDENTS_BIN"],
    "subgroup_n": ["SUBGROUP_NAME", "N_STUDENTS_BIN"],
    "pred_bucket_n": ["anchor_pred_bucket", "N_STUDENTS_BIN"],
    "pred_bucket_assessment": ["anchor_pred_bucket", "ASSESSMENT_NAME"],
    "pred_bucket_subgroup": ["anchor_pred_bucket", "SUBGROUP_NAME"],
}

segment_specs = {
    name: cols for name, cols in segment_specs.items()
    if all(c in diag.columns and c in test_diag.columns for c in cols)
}

settings = [
    (300, 1000.0),
    (600, 2000.0),
    (1000, 3000.0),
]

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(len(y))))

screen_rows = []
oof_by_label = {}

for spec_name, cols in segment_specs.items():
    seg = make_segment_values(cols, diag)

    for min_count, shrink in settings:
        oof_pred = np.zeros(len(y), dtype=float)
        oof_w = np.zeros(len(y), dtype=float)
        fold_gains = []
        segment_counts = []

        for tr_rows, va_rows in folds:
            weights, global_w = fit_segment_weights(seg, tr_rows, min_count, shrink)
            pred_va, w_va = predict_with_weights(seg, va_rows, weights, global_w)

            oof_pred[va_rows] = pred_va
            oof_w[va_rows] = w_va

            fold_gain = mean_squared_error(y[va_rows], current_oof[va_rows]) - mean_squared_error(y[va_rows], pred_va)
            fold_gains.append(fold_gain)
            segment_counts.append(len(weights))

        label = f"seg33a_{spec_name}_min{min_count}_shrink{int(shrink)}"
        cand_mse = mean_squared_error(y, oof_pred)

        screen_rows.append({
            "label": label,
            "segment_spec": spec_name,
            "cols": " + ".join(cols),
            "min_count": min_count,
            "shrink": shrink,
            "oof_mse": cand_mse,
            "gain_vs_current": current_mse - cand_mse,
            "min_fold_gain": float(np.min(fold_gains)),
            "mean_fold_gain": float(np.mean(fold_gains)),
            "mean_weight": float(np.mean(oof_w)),
            "std_weight": float(np.std(oof_w)),
            "avg_segment_weights": float(np.mean(segment_counts)),
        })

        oof_by_label[label] = oof_pred.astype(np.float32)

screen = pd.DataFrame(screen_rows).sort_values(
    ["oof_mse", "min_fold_gain"],
    ascending=[True, False]
).reset_index(drop=True)

screen_path = "model_results/seg33a_arcsine_weight_screen.csv"
screen.to_csv(screen_path, index=False)

stable = screen[(screen["gain_vs_current"] > 0) & (screen["min_fold_gain"] > 0)].copy()

if len(stable) > 0:
    best = stable.iloc[0]
    selection_note = "best fold-stable improvement"
else:
    best = screen.iloc[0]
    selection_note = "best OOF candidate, but not fold-stable"

best_label = best["label"]
best_oof = oof_by_label[best_label]

best_cols = segment_specs[best["segment_spec"]]
best_seg_train = make_segment_values(best_cols, diag)
best_seg_test = make_segment_values(best_cols, test_diag)

all_rows = np.arange(len(y))
final_weights, final_global_w = fit_segment_weights(
    best_seg_train,
    all_rows,
    int(best["min_count"]),
    float(best["shrink"]),
)

test_w = pd.Series(best_seg_test).map(final_weights).fillna(final_global_w).to_numpy(dtype=float)
best_test = np.clip(base_test + test_w * delta_test, 0, 100)

oof_path = "model_results/oof_seg33a_arcsine_adaptive.csv"
test_path = "model_results/testpred_seg33a_arcsine_adaptive.csv"
submission_path = "submission_seg33a_arcsine_adaptive.csv"
weight_path = "model_results/seg33a_arcsine_adaptive_segment_weights.csv"

pd.DataFrame({
    "row_index": np.arange(len(y)),
    "PERCENT_PROFICIENT": y,
    "pred_clipped": best_oof,
}).to_csv(oof_path, index=False)

pd.DataFrame({
    "ASSESSMENT_ID": test_ids_arr,
    "PERCENT_PROFICIENT": best_test,
}).to_csv(test_path, index=False)

pd.DataFrame({
    "ASSESSMENT_ID": test_ids_arr,
    "PERCENT_PROFICIENT": best_test,
}).to_csv(submission_path, index=False)

pd.DataFrame({
    "segment": list(final_weights.keys()),
    "weight": list(final_weights.values()),
}).sort_values("weight", ascending=False).to_csv(weight_path, index=False)

print("\nTop 10 screened candidates")
print("--------------------------")
print(
    screen.head(10)[[
        "label",
        "oof_mse",
        "gain_vs_current",
        "min_fold_gain",
        "mean_weight",
        "avg_segment_weights",
    ]].to_string(index=False)
)

selected_mse = mean_squared_error(y, best_oof)

print("\nSelected candidate")
print("------------------")
print("selection:", selection_note)
print("label:", best_label)
print("segment columns:", best["cols"])
print("candidate OOF MSE:", selected_mse)
print("current OOF MSE:", current_mse)
print("OOF gain vs current:", current_mse - selected_mse)
print("min fold gain:", best["min_fold_gain"])
print("final global fallback weight:", final_global_w)
print("number of final segment weights:", len(final_weights))

print("\nSaved files")
print("-----------")
print(screen_path)
print(weight_path)
print(oof_path)
print(test_path)
print(submission_path)

print("\nCandidate test prediction summary")
print("---------------------------------")
print(
    pd.Series(best_test).describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).to_string()
)

33A. Segment-adaptive arcsine blend
----------------------------------
Safe TE anchor OOF MSE:      78.071510
Bounded arcsine OOF MSE:     83.952616
Current 68.986 OOF MSE:      77.148170

Top 10 screened candidates
--------------------------
                                          label   oof_mse  gain_vs_current  min_fold_gain  mean_weight  avg_segment_weights
         seg33a_pred_bucket_n_min300_shrink1000 77.049862         0.098308       0.046554     0.253545                 44.0
         seg33a_pred_bucket_n_min600_shrink2000 77.066430         0.081740       0.026026     0.252616                 40.0
  seg33a_pred_bucket_subgroup_min300_shrink1000 77.071549         0.076621       0.055422     0.269164                 44.0
        seg33a_pred_bucket_n_min1000_shrink3000 77.081122         0.067048       0.009728     0.251204                 35.0
           seg33a_pred_bucket_min300_shrink1000 77.088785         0.059385       0.031127     0.288482                 10.0
  seg33a_pred

### 33A Submission Result — Segment-Adaptive Arcsine Blend

Submitted candidate:

`submission_seg33a_arcsine_adaptive.csv`

This model extended the previous bounded-arcsine + TE blend by allowing the arcsine correction weight to vary across prediction regimes and student-count regimes. The strongest configuration used:

- segment columns:
  - `anchor_pred_bucket`
  - `N_STUDENTS_BIN`
- minimum segment count:
  - `300`
- shrink parameter:
  - `1000`

Key local validation results:

- previous current-anchor OOF MSE:
  - `77.148170`
- 33A candidate OOF MSE:
  - `77.049862`
- local OOF gain:
  - `0.098308`
- minimum fold gain:
  - `0.046554`

Public leaderboard result:

- previous public MSE:
  - `68.986`
- new public MSE:
  - `68.873`
- public gain:
  - `0.113`

Interpretation:

Although the local OOF gain was small, the candidate generalized in the correct direction on the public leaderboard. This suggests that prediction-regime-aware weighting and student-count-aware weighting contain real signal, even if the overall improvement is modest.

The result is treated as a successful incremental improvement and becomes the new current public anchor until surpassed by a stronger candidate.

### 33B. Huber + ExtraTrees-Style LightGBM on TE Features

The previous 33A segment-adaptive arcsine blend produced a small but fold-stable local OOF gain of about 0.098 MSE versus the current 68.986 public-anchor blend. Because this is only a small same-branch blend refinement, it is saved as a backlog candidate rather than treated as an immediate Kaggle submission.

This next cell builds a genuinely new first-level model artifact using the same leakage-safe TE feature construction, but changes the model behavior through a Huber objective and `extra_trees=True` LightGBM splitting. The purpose is to test whether a more robust and more randomized boosting model contributes useful diversity beyond the current anchor.

The cell trains 5-fold OOF predictions, saves fold-averaged test predictions, and then screens a simple convex blend between the current 68.986 anchor and the new Huber/ExtraTrees-style LightGBM artifact. The key things to inspect are the standalone OOF MSE, the blend weight assigned to the new model, the blend OOF gain, and whether fold gains are stable.

In [84]:
# ============================================================
# 33B. Huber + ExtraTrees-style LightGBM on TE features
# ============================================================
# New first-level model:
#   - same leakage-safe base + TE feature construction as 28E/31A
#   - raw percentage target
#   - Huber objective
#   - LightGBM extra_trees=True for diversity
#
# Then blends the new OOF artifact with the current 68.986 anchor.
# ============================================================

import os
import gc
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = 9890
N_SPLITS = 5

ARTIFACT = "lgbm33b_te_huber_extra_l127_child80_l2_10_lr03"

print("=" * 90)
print("33B. Huber + ExtraTrees-style LightGBM on TE features")
print("=" * 90)

y_33b = np.asarray(y_train, dtype=np.float32).reshape(-1)
test_ids_33b = np.asarray(test_ids)

n_train_33b = len(y_33b)
n_test_33b = len(test_ids_33b)

current_oof_33b = np.asarray(blend_oof, dtype=np.float32)
current_test_33b = np.asarray(blend_test, dtype=np.float32)

folds_33b = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_33b))
)

metrics_path_33b = f"model_results/{ARTIFACT}_fold_metrics.csv"
oof_raw_path_33b = f"model_results/{ARTIFACT}_oof_raw.npy"
oof_clip_path_33b = f"model_results/{ARTIFACT}_oof_clipped.npy"

def fold_test_path_33b(fold_num):
    return f"model_results/{ARTIFACT}_fold{fold_num}_test_pred.npy"

oof_csv_path_33b = f"model_results/oof_{ARTIFACT}.csv"
testpred_path_33b = f"model_results/testpred_{ARTIFACT}_foldavg.csv"
submission_pure_path_33b = f"submission_{ARTIFACT}_foldavg.csv"

if os.path.exists(oof_raw_path_33b):
    oof_raw_33b = np.load(oof_raw_path_33b)
else:
    oof_raw_33b = np.full(n_train_33b, np.nan, dtype=np.float32)

if os.path.exists(oof_clip_path_33b):
    oof_clip_33b = np.load(oof_clip_path_33b)
else:
    oof_clip_33b = np.full(n_train_33b, np.nan, dtype=np.float32)

if os.path.exists(metrics_path_33b):
    fold_metrics_33b = pd.read_csv(metrics_path_33b)
else:
    fold_metrics_33b = pd.DataFrame()

done_folds_33b = set()
if len(fold_metrics_33b) > 0 and "fold" in fold_metrics_33b.columns:
    done_folds_33b = set(fold_metrics_33b["fold"].astype(int).tolist())

params_33b = {
    "objective": "huber",
    "metric": "l2",
    "alpha": 0.90,
    "random_state": RANDOM_STATE + 33,
    "extra_seed": RANDOM_STATE + 330,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,

    "n_estimators": 30000,
    "learning_rate": 0.03,
    "num_leaves": 127,
    "min_child_samples": 80,
    "subsample": 0.80,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.0,
    "reg_lambda": 10.0,
    "max_depth": -1,
    "extra_trees": True,
}

print("Artifact:", ARTIFACT)
print("Rows:", n_train_33b, "train |", n_test_33b, "test")
print("Already completed folds:", sorted(done_folds_33b))
print("Current 68.986 anchor OOF MSE:", mean_squared_error(y_33b, current_oof_33b))

for fold_num, (tr_idx, va_idx) in enumerate(folds_33b, start=1):
    fold_test_path = fold_test_path_33b(fold_num)

    if (
        fold_num in done_folds_33b
        and os.path.exists(fold_test_path)
        and np.isfinite(oof_clip_33b[va_idx]).all()
    ):
        print(f"\nFold {fold_num} already complete. Skipping.")
        continue

    print("\n" + "=" * 70)
    print(f"Fold {fold_num}/{N_SPLITS}")
    print("=" * 70)

    t0 = time.time()

    raw_fit = raw_train_te.iloc[tr_idx].reset_index(drop=True)
    raw_val = raw_train_te.iloc[va_idx].reset_index(drop=True)

    y_fit = y_33b[tr_idx]
    y_val = y_33b[va_idx]

    print("Building leakage-safe TE features...")
    te_fit, apply_dict, _ = build_te_oof_and_apply_many(
        raw_fit=raw_fit,
        y_fit=y_fit,
        raw_apply_dict={
            "valid": raw_val,
            "test": raw_test_te,
        },
        key_specs=te_key_specs,
        n_splits=5,
        random_state=RANDOM_STATE + 100 * fold_num,
    )

    X_fit_base = to_float32_matrix(take_rows(X_train_proc_model, tr_idx))
    X_val_base = to_float32_matrix(take_rows(X_train_proc_model, va_idx))
    X_test_base = to_float32_matrix(X_test_proc_model)

    X_fit = append_features(X_fit_base, te_fit)
    X_val = append_features(X_val_base, apply_dict["valid"])
    X_test = append_features(X_test_base, apply_dict["test"])

    print("Fold train shape:", X_fit.shape)
    print("Fold valid shape:", X_val.shape)
    print("Fold test shape: ", X_test.shape)

    model = lgb.LGBMRegressor(**params_33b)

    model.fit(
        X_fit,
        y_fit,
        eval_set=[(X_val, y_val)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=1500, verbose=False),
            lgb.log_evaluation(period=1000),
        ],
    )

    best_iter = int(model.best_iteration_ or params_33b["n_estimators"])

    pred_fit_raw = model.predict(X_fit, num_iteration=best_iter)
    pred_val_raw = model.predict(X_val, num_iteration=best_iter)
    pred_test_raw = model.predict(X_test, num_iteration=best_iter)

    pred_fit_clip = np.clip(pred_fit_raw, 0, 100).astype(np.float32)
    pred_val_clip = np.clip(pred_val_raw, 0, 100).astype(np.float32)
    pred_test_clip = np.clip(pred_test_raw, 0, 100).astype(np.float32)

    oof_raw_33b[va_idx] = pred_val_raw.astype(np.float32)
    oof_clip_33b[va_idx] = pred_val_clip

    np.save(oof_raw_path_33b, oof_raw_33b)
    np.save(oof_clip_path_33b, oof_clip_33b)
    np.save(fold_test_path, pred_test_clip)

    elapsed = time.time() - t0

    row = {
        "artifact": ARTIFACT,
        "fold": fold_num,
        "train_rows": int(len(tr_idx)),
        "valid_rows": int(len(va_idx)),
        "n_features": int(X_fit.shape[1]),
        "best_iteration": best_iter,
        "train_mse_clipped": float(mean_squared_error(y_fit, pred_fit_clip)),
        "valid_mse_clipped": float(mean_squared_error(y_val, pred_val_clip)),
        "current_anchor_fold_mse": float(mean_squared_error(y_val, current_oof_33b[va_idx])),
        "elapsed_seconds": float(elapsed),
    }

    fold_metrics_33b = pd.concat(
        [
            fold_metrics_33b[fold_metrics_33b.get("fold", pd.Series(dtype=int)) != fold_num],
            pd.DataFrame([row]),
        ],
        ignore_index=True,
    ).sort_values("fold").reset_index(drop=True)

    fold_metrics_33b.to_csv(metrics_path_33b, index=False)

    print("Fold result:")
    print(pd.DataFrame([row]).T.to_string())

    del model
    del raw_fit, raw_val
    del te_fit, apply_dict
    del X_fit_base, X_val_base, X_test_base
    del X_fit, X_val, X_test
    del pred_fit_raw, pred_val_raw, pred_test_raw
    del pred_fit_clip, pred_val_clip, pred_test_clip
    gc.collect()

print("\n" + "=" * 90)
print("Finalizing 33B")
print("=" * 90)

if not np.isfinite(oof_clip_33b).all():
    print("OOF predictions are incomplete. Rerun this same cell to resume.")
else:
    huber_oof_mse = float(mean_squared_error(y_33b, oof_clip_33b))
    current_mse_33b = float(mean_squared_error(y_33b, current_oof_33b))

    fold_test_preds = []
    for fold_num in range(1, N_SPLITS + 1):
        fold_test_preds.append(np.load(fold_test_path_33b(fold_num)).astype(np.float32))

    huber_test_33b = np.clip(np.mean(np.vstack(fold_test_preds), axis=0), 0, 100).astype(np.float32)

    pd.DataFrame({
        "row_index": np.arange(n_train_33b),
        "PERCENT_PROFICIENT": y_33b,
        "pred_raw": oof_raw_33b,
        "pred_clipped": oof_clip_33b,
    }).to_csv(oof_csv_path_33b, index=False)

    pd.DataFrame({
        "ASSESSMENT_ID": test_ids_33b,
        "PERCENT_PROFICIENT": huber_test_33b,
    }).to_csv(testpred_path_33b, index=False)

    pd.DataFrame({
        "ASSESSMENT_ID": test_ids_33b,
        "PERCENT_PROFICIENT": huber_test_33b,
    }).to_csv(submission_pure_path_33b, index=False)

    # --------------------------------------------------------
    # Blend the new first-level model with the current 68.986 anchor
    # --------------------------------------------------------

    blend_rows = []

    for w_huber in np.linspace(0.0, 0.60, 601):
        pred = np.clip((1.0 - w_huber) * current_oof_33b + w_huber * oof_clip_33b, 0, 100)
        blend_rows.append({
            "w_current_anchor": 1.0 - w_huber,
            "w_huber33b": w_huber,
            "oof_mse": float(mean_squared_error(y_33b, pred)),
        })

    blend_screen_33b = pd.DataFrame(blend_rows).sort_values("oof_mse").reset_index(drop=True)
    best_blend_33b = blend_screen_33b.iloc[0]

    w_huber_best = float(best_blend_33b["w_huber33b"])
    w_current_best = float(best_blend_33b["w_current_anchor"])

    blend_oof_33b = np.clip(
        w_current_best * current_oof_33b + w_huber_best * oof_clip_33b,
        0,
        100,
    ).astype(np.float32)

    blend_test_33b = np.clip(
        w_current_best * current_test_33b + w_huber_best * huber_test_33b,
        0,
        100,
    ).astype(np.float32)

    blend_mse_33b = float(mean_squared_error(y_33b, blend_oof_33b))

    fold_gain_rows = []
    for fold_num, (_, va_idx) in enumerate(folds_33b, start=1):
        current_fold_mse = mean_squared_error(y_33b[va_idx], current_oof_33b[va_idx])
        blend_fold_mse = mean_squared_error(y_33b[va_idx], blend_oof_33b[va_idx])
        huber_fold_mse = mean_squared_error(y_33b[va_idx], oof_clip_33b[va_idx])

        fold_gain_rows.append({
            "fold": fold_num,
            "current_anchor_mse": current_fold_mse,
            "huber33b_mse": huber_fold_mse,
            "blend33b_mse": blend_fold_mse,
            "blend_gain_vs_current": current_fold_mse - blend_fold_mse,
        })

    fold_gain_33b = pd.DataFrame(fold_gain_rows)

    blend_screen_path_33b = "model_results/blend33b_current_huber_te_screen.csv"
    blend_oof_path_33b = "model_results/oof_blend33b_current_huber_te.csv"
    blend_test_path_33b = "model_results/testpred_blend33b_current_huber_te.csv"
    blend_submission_path_33b = "submission_blend33b_current_huber_te.csv"
    blend_fold_path_33b = "model_results/blend33b_current_huber_te_fold_gains.csv"

    blend_screen_33b.to_csv(blend_screen_path_33b, index=False)
    fold_gain_33b.to_csv(blend_fold_path_33b, index=False)

    pd.DataFrame({
        "row_index": np.arange(n_train_33b),
        "PERCENT_PROFICIENT": y_33b,
        "pred_clipped": blend_oof_33b,
    }).to_csv(blend_oof_path_33b, index=False)

    pd.DataFrame({
        "ASSESSMENT_ID": test_ids_33b,
        "PERCENT_PROFICIENT": blend_test_33b,
    }).to_csv(blend_test_path_33b, index=False)

    pd.DataFrame({
        "ASSESSMENT_ID": test_ids_33b,
        "PERCENT_PROFICIENT": blend_test_33b,
    }).to_csv(blend_submission_path_33b, index=False)

    print("\n33B standalone model")
    print("--------------------")
    print("Huber/extra TE OOF MSE:", huber_oof_mse)
    print("Current 68.986 anchor OOF MSE:", current_mse_33b)
    print("Standalone gain vs current:", current_mse_33b - huber_oof_mse)

    print("\n33B blend result")
    print("----------------")
    print("Best w_current_anchor:", w_current_best)
    print("Best w_huber33b:", w_huber_best)
    print("Blend OOF MSE:", blend_mse_33b)
    print("Blend gain vs current:", current_mse_33b - blend_mse_33b)

    print("\nFold gains")
    print("----------")
    print(fold_gain_33b.to_string(index=False))
    print("Min fold gain:", float(fold_gain_33b["blend_gain_vs_current"].min()))

    print("\nSaved files")
    print("-----------")
    print(metrics_path_33b)
    print(oof_csv_path_33b)
    print(testpred_path_33b)
    print(submission_pure_path_33b)
    print(blend_screen_path_33b)
    print(blend_fold_path_33b)
    print(blend_oof_path_33b)
    print(blend_test_path_33b)
    print(blend_submission_path_33b)

    print("\nBlend test prediction summary")
    print("-----------------------------")
    print(
        pd.Series(blend_test_33b)
        .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
        .to_string()
    )

33B. Huber + ExtraTrees-style LightGBM on TE features
Artifact: lgbm33b_te_huber_extra_l127_child80_l2_10_lr03
Rows: 144921 train | 48307 test
Already completed folds: []
Current 68.986 anchor OOF MSE: 77.1481704711914

Fold 1/5
Building leakage-safe TE features...
Fold train shape: (115936, 230)
Fold valid shape: (28985, 230)
Fold test shape:  (48307, 230)
[1000]	valid_0's l2: 193.051
[2000]	valid_0's l2: 112.094
[3000]	valid_0's l2: 99.8307
[4000]	valid_0's l2: 95.0257
[5000]	valid_0's l2: 92.4366
[6000]	valid_0's l2: 90.8781
[7000]	valid_0's l2: 89.764
[8000]	valid_0's l2: 88.9792
[9000]	valid_0's l2: 88.3503
[10000]	valid_0's l2: 87.8434
[11000]	valid_0's l2: 87.4332
[12000]	valid_0's l2: 87.084
[13000]	valid_0's l2: 86.7815
[14000]	valid_0's l2: 86.52
[15000]	valid_0's l2: 86.2807
[16000]	valid_0's l2: 86.0672
[17000]	valid_0's l2: 85.8874
[18000]	valid_0's l2: 85.7298
[19000]	valid_0's l2: 85.5706
[20000]	valid_0's l2: 85.4255
[21000]	valid_0's l2: 85.2991
[22000]	valid_0's l2: 8

### 33B Result and 33C Comparison Setup

33B trained a Huber + ExtraTrees-style LightGBM model on the 230-feature base + TE matrix. As a standalone model, it was weaker than the current anchor, with OOF MSE around `82.795`. However, it received nonzero blend weight against the older bounded-arcsine anchor and improved OOF from `77.148` to `76.952`, with positive gains on all five folds.

Because the current public anchor is now the 33A segment-adaptive arcsine blend with public MSE `68.873`, the next step is not to submit 33B blindly. Instead, 33C directly compares and blends the 33A adaptive anchor with the pure 33B Huber/ExtraTrees-style model. If this produces a meaningful, fold-stable OOF gain over 33A, then the resulting 33C candidate can be considered for submission.

In [85]:
# ============================================================
# 33C. Direct comparison/blend: current 33A anchor + pure 33B
# ============================================================
# This is a light comparison cell, not a new model fit.
# It answers:
#   Does the 33B Huber/ExtraTrees TE model improve the new 33A anchor?
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = 9890
N_SPLITS = 5

print("33C. Blend current 33A anchor with pure 33B Huber/ExtraTrees TE")
print("----------------------------------------------------------------")

# Current public anchor from 33A
oof33a_path = "model_results/oof_seg33a_arcsine_adaptive.csv"
test33a_path = "model_results/testpred_seg33a_arcsine_adaptive.csv"

# Pure 33B model artifact
oof33b_path = "model_results/oof_lgbm33b_te_huber_extra_l127_child80_l2_10_lr03.csv"
test33b_path = "model_results/testpred_lgbm33b_te_huber_extra_l127_child80_l2_10_lr03_foldavg.csv"

# Existing old-anchor + 33B blend, only for reference
oof33b_oldblend_path = "model_results/oof_blend33b_current_huber_te.csv"

oof33a = pd.read_csv(oof33a_path).sort_values("row_index").reset_index(drop=True)
oof33b = pd.read_csv(oof33b_path).sort_values("row_index").reset_index(drop=True)
oof33b_oldblend = pd.read_csv(oof33b_oldblend_path).sort_values("row_index").reset_index(drop=True)

test33a = pd.read_csv(test33a_path)
test33b = pd.read_csv(test33b_path)

y = oof33a["PERCENT_PROFICIENT"].to_numpy(dtype=np.float32)

pred33a = oof33a["pred_clipped"].to_numpy(dtype=np.float32)
pred33b = oof33b["pred_clipped"].to_numpy(dtype=np.float32)
pred33b_oldblend = oof33b_oldblend["pred_clipped"].to_numpy(dtype=np.float32)

test_pred33a = test33a["PERCENT_PROFICIENT"].to_numpy(dtype=np.float32)
test_pred33b = test33b["PERCENT_PROFICIENT"].to_numpy(dtype=np.float32)

mse33a = mean_squared_error(y, pred33a)
mse33b = mean_squared_error(y, pred33b)
mse33b_oldblend = mean_squared_error(y, pred33b_oldblend)

rows = []

for w33b in np.linspace(0.0, 0.35, 351):
    pred = np.clip((1.0 - w33b) * pred33a + w33b * pred33b, 0, 100)
    rows.append({
        "w_33a_anchor": 1.0 - w33b,
        "w_33b_huber": w33b,
        "oof_mse": mean_squared_error(y, pred),
        "gain_vs_33a": mse33a - mean_squared_error(y, pred),
    })

screen33c = pd.DataFrame(rows).sort_values("oof_mse").reset_index(drop=True)
best33c = screen33c.iloc[0]

w33b_best = float(best33c["w_33b_huber"])
w33a_best = float(best33c["w_33a_anchor"])

blend_oof33c = np.clip(w33a_best * pred33a + w33b_best * pred33b, 0, 100).astype(np.float32)
blend_test33c = np.clip(w33a_best * test_pred33a + w33b_best * test_pred33b, 0, 100).astype(np.float32)

fold_rows = []
folds = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(len(y)))
)

for fold_num, (_, va_idx) in enumerate(folds, start=1):
    fold_mse_33a = mean_squared_error(y[va_idx], pred33a[va_idx])
    fold_mse_33c = mean_squared_error(y[va_idx], blend_oof33c[va_idx])

    fold_rows.append({
        "fold": fold_num,
        "mse_33a_anchor": fold_mse_33a,
        "mse_33c_blend": fold_mse_33c,
        "gain_vs_33a": fold_mse_33a - fold_mse_33c,
    })

fold33c = pd.DataFrame(fold_rows)

screen_path = "model_results/blend33c_33a_huber33b_screen.csv"
fold_path = "model_results/blend33c_33a_huber33b_fold_gains.csv"
oof_path = "model_results/oof_blend33c_33a_huber33b.csv"
test_path = "model_results/testpred_blend33c_33a_huber33b.csv"
submission_path = "submission_blend33c_33a_huber33b.csv"

screen33c.to_csv(screen_path, index=False)
fold33c.to_csv(fold_path, index=False)

pd.DataFrame({
    "row_index": np.arange(len(y)),
    "PERCENT_PROFICIENT": y,
    "pred_clipped": blend_oof33c,
}).to_csv(oof_path, index=False)

pd.DataFrame({
    "ASSESSMENT_ID": test33a["ASSESSMENT_ID"],
    "PERCENT_PROFICIENT": blend_test33c,
}).to_csv(test_path, index=False)

pd.DataFrame({
    "ASSESSMENT_ID": test33a["ASSESSMENT_ID"],
    "PERCENT_PROFICIENT": blend_test33c,
}).to_csv(submission_path, index=False)

print("\nOOF comparison")
print("--------------")
print(f"33A current public anchor OOF MSE:       {mse33a:.6f}")
print(f"Pure 33B Huber/ExtraTrees OOF MSE:       {mse33b:.6f}")
print(f"Old-anchor 33B blend OOF MSE reference: {mse33b_oldblend:.6f}")

print("\nBest 33C blend")
print("--------------")
print(f"w_33a_anchor: {w33a_best:.3f}")
print(f"w_33b_huber:  {w33b_best:.3f}")
print(f"33C OOF MSE:  {mean_squared_error(y, blend_oof33c):.6f}")
print(f"Gain vs 33A:  {mse33a - mean_squared_error(y, blend_oof33c):.6f}")
print(f"Min fold gain vs 33A: {fold33c['gain_vs_33a'].min():.6f}")

print("\nFold gains vs 33A")
print("-----------------")
print(fold33c.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path)
print(fold_path)
print(oof_path)
print(test_path)
print(submission_path)

print("\n33C test prediction summary")
print("---------------------------")
print(
    pd.Series(blend_test33c)
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .to_string()
)

33C. Blend current 33A anchor with pure 33B Huber/ExtraTrees TE
----------------------------------------------------------------

OOF comparison
--------------
33A current public anchor OOF MSE:       77.049866
Pure 33B Huber/ExtraTrees OOF MSE:       82.794830
Old-anchor 33B blend OOF MSE reference: 76.951591

Best 33C blend
--------------
w_33a_anchor: 0.848
w_33b_huber:  0.152
33C OOF MSE:  76.859200
Gain vs 33A:  0.190666
Min fold gain vs 33A: 0.067055

Fold gains vs 33A
-----------------
 fold  mse_33a_anchor  mse_33c_blend  gain_vs_33a
    1       78.427116      78.354721     0.072395
    2       74.554970      74.339523     0.215446
    3       78.135284      77.834740     0.300545
    4       76.396492      76.329437     0.067055
    5       77.735397      77.437508     0.297890

Saved files
-----------
model_results/blend33c_33a_huber33b_screen.csv
model_results/blend33c_33a_huber33b_fold_gains.csv
model_results/oof_blend33c_33a_huber33b.csv
model_results/testpred_blend33c_33a

In [86]:
# ============================================================
# 34A. Regression splines / GAM-style additive residual screen
# ============================================================
#
# Textbook family:
#   12. Regression splines / cubic splines
#   13. Smoothing-spline-like shrinkage via Ridge penalty
#   15. GAM-style additive model
#
# What this does:
# - Uses the same leakage-safe TE construction pattern as 28E/31A/33B.
# - Builds cubic B-spline basis expansions on selected continuous + TE features.
# - Fits Ridge-regularized additive spline models.
# - Tests two spline/GAM variants:
#     1. Direct target model
#     2. Residual correction to the current 33A public anchor
#
# This is a model-family screen, not another LightGBM tweak.
# ============================================================

import os
import gc
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")
os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = "PERCENT_PROFICIENT"
ID_COL = "ASSESSMENT_ID"

# ------------------------------------------------------------
# 0. Current public anchor
# ------------------------------------------------------------
# Default: use 33A because 33C was publicly worse.
# Change these two lines only if the current best public anchor changes.

ANCHOR_OOF_PATH_34A = "model_results/oof_seg33a_arcsine_adaptive.csv"
ANCHOR_TEST_PATH_34A = "model_results/testpred_seg33a_arcsine_adaptive.csv"

ARTIFACT_PREFIX_34A = "spline34a_gam_te_additive"

print("=" * 90)
print("34A. Regression splines / GAM-style additive model")
print("=" * 90)
print("Anchor OOF path: ", ANCHOR_OOF_PATH_34A)
print("Anchor test path:", ANCHOR_TEST_PATH_34A)

# ------------------------------------------------------------
# 1. Required-object checks
# ------------------------------------------------------------

required_34a = [
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "raw_train_te",
    "raw_test_te",
    "te_key_specs",
]

missing_34a = [x for x in required_34a if x not in globals()]
if missing_34a:
    raise ValueError(
        f"Missing required objects: {missing_34a}. "
        "Rerun the TE setup cells around 28B/28D/31A before running 34A."
    )

if not isinstance(X_train_proc_model, pd.DataFrame) or not isinstance(X_test_proc_model, pd.DataFrame):
    raise ValueError(
        "34A expects X_train_proc_model and X_test_proc_model to be pandas DataFrames "
        "with column names. The earlier preprocessing cells created them this way."
    )

# Helper aliases copied from the established TE branch pattern.
if "build_te_oof_and_apply_many" not in globals():
    if "build_te_oof_and_apply_many_29b" in globals():
        build_te_oof_and_apply_many = build_te_oof_and_apply_many_29b
    else:
        raise ValueError("Missing build_te_oof_and_apply_many helper. Rerun 28E or 29B setup.")

y_arr_34a = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_34a = len(y_arr_34a)
n_test_34a = X_test_proc_model.shape[0]

if X_train_proc_model.shape[0] != n_train_34a:
    raise ValueError("X_train_proc_model row count does not match y_train.")
if raw_train_te.shape[0] != n_train_34a:
    raise ValueError("raw_train_te row count does not match y_train.")
if X_test_proc_model.shape[0] != raw_test_te.shape[0]:
    raise ValueError("X_test_proc_model row count does not match raw_test_te.")

# ------------------------------------------------------------
# 2. Anchor loaders
# ------------------------------------------------------------

def load_oof_prediction_34a(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    y_from_file = None
    if TARGET_COL in df.columns:
        y_from_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)

    pred_priority = [
        "pred_clipped",
        "prediction",
        "pred",
        "oof_pred",
        "PREDICTED_PERCENT_PROFICIENT",
    ]

    pred_col = None
    for c in pred_priority:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find a prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}, column {pred_col}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col, y_from_file


def load_test_prediction_34a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}, column {pred_col}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no global test_ids found.")

    if len(pred) != n_test_34a:
        raise ValueError(f"Test row mismatch for {path}: got {len(pred)}, expected {n_test_34a}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col


anchor_oof_34a, anchor_oof_col_34a, anchor_y_file_34a = load_oof_prediction_34a(
    ANCHOR_OOF_PATH_34A,
    y_arr_34a,
)
anchor_test_34a, test_ids_34a, anchor_test_col_34a = load_test_prediction_34a(
    ANCHOR_TEST_PATH_34A
)

if anchor_y_file_34a is not None:
    max_y_diff_34a = float(np.nanmax(np.abs(anchor_y_file_34a - y_arr_34a)))
    if max_y_diff_34a > 1e-5:
        raise ValueError(
            f"Anchor OOF target values do not align with y_train. Max difference: {max_y_diff_34a}"
        )

anchor_mse_34a = float(mean_squared_error(y_arr_34a, anchor_oof_34a))

print("\nAnchor check")
print("------------")
print("Anchor OOF prediction column:", anchor_oof_col_34a)
print("Anchor test prediction column:", anchor_test_col_34a)
print(f"Anchor OOF MSE: {anchor_mse_34a:.6f}")

# ------------------------------------------------------------
# 3. Continuous base-feature candidates for spline basis
# ------------------------------------------------------------

common_base_cols_34a = [
    c for c in X_train_proc_model.columns
    if c in X_test_proc_model.columns
]

base_cont_cols_34a = []

for c in common_base_cols_34a:
    if not pd.api.types.is_numeric_dtype(X_train_proc_model[c]):
        continue

    if str(c).endswith("_missing"):
        continue

    s = X_train_proc_model[c]
    nunique = int(s.nunique(dropna=True))

    # Exclude one-hot / near-binary / very low-cardinality dummies.
    if nunique <= 8:
        continue

    base_cont_cols_34a.append(c)

print("\nSpline feature pool")
print("-------------------")
print("Continuous base candidates:", len(base_cont_cols_34a))
print("TE key specs:", len(te_key_specs))
print("Expected TE columns per fold:", len(te_key_specs) * 4)

# ------------------------------------------------------------
# 4. Spline/GAM configs
# ------------------------------------------------------------

spline_configs_34a = [
    {"top_k": 8,  "n_knots": 4, "degree": 3, "alpha": 100.0},
    {"top_k": 8,  "n_knots": 6, "degree": 3, "alpha": 100.0},
    {"top_k": 12, "n_knots": 4, "degree": 3, "alpha": 100.0},
    {"top_k": 12, "n_knots": 6, "degree": 3, "alpha": 100.0},
    {"top_k": 20, "n_knots": 4, "degree": 3, "alpha": 1000.0},
    {"top_k": 20, "n_knots": 6, "degree": 3, "alpha": 1000.0},
]

for cfg in spline_configs_34a:
    cfg["name"] = (
        f"{ARTIFACT_PREFIX_34A}"
        f"_top{cfg['top_k']}"
        f"_knots{cfg['n_knots']}"
        f"_deg{cfg['degree']}"
        f"_alpha{str(cfg['alpha']).replace('.', 'p')}"
    )

print("\nConfigs")
print("-------")
for cfg in spline_configs_34a:
    print(cfg["name"])

# ------------------------------------------------------------
# 5. Helpers
# ------------------------------------------------------------

def make_spline_ridge_34a(n_knots, degree, alpha):
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        SplineTransformer(
            n_knots=int(n_knots),
            degree=int(degree),
            include_bias=False,
            extrapolation="constant",
        ),
        StandardScaler(),
        Ridge(alpha=float(alpha), solver="lsqr"),
    )


def rank_columns_by_abs_corr_34a(X_df, target):
    y = np.asarray(target, dtype=np.float64).reshape(-1)
    y = y - np.nanmean(y)
    y_den = np.sqrt(np.dot(y, y))

    rows = []

    for c in X_df.columns:
        x = pd.to_numeric(X_df[c], errors="coerce").to_numpy(dtype=np.float64)

        if not np.isfinite(x).all():
            finite = np.isfinite(x)
            fill_value = np.nanmedian(x[finite]) if finite.any() else 0.0
            x = np.where(finite, x, fill_value)

        x = x - np.mean(x)
        x_den = np.sqrt(np.dot(x, x))

        if x_den <= 1e-12 or y_den <= 1e-12:
            corr = 0.0
        else:
            corr = abs(float(np.dot(x, y) / (x_den * y_den)))

        rows.append((c, corr))

    ranked = pd.DataFrame(rows, columns=["feature", "abs_corr"])
    ranked = ranked.sort_values("abs_corr", ascending=False).reset_index(drop=True)
    return ranked


def fold_mse_table_34a(pred, label):
    rows = []
    for fold_num, (_, va_idx) in enumerate(folds_34a, start=1):
        anchor_fold_mse = float(mean_squared_error(y_arr_34a[va_idx], anchor_oof_34a[va_idx]))
        cand_fold_mse = float(mean_squared_error(y_arr_34a[va_idx], pred[va_idx]))
        rows.append({
            "candidate": label,
            "fold": fold_num,
            "anchor_mse": anchor_fold_mse,
            "candidate_mse": cand_fold_mse,
            "gain_vs_anchor": anchor_fold_mse - cand_fold_mse,
        })
    return pd.DataFrame(rows)


def save_candidate_34a(label, pred_oof, pred_test):
    oof_path = f"model_results/oof_{label}.csv"
    test_path = f"model_results/testpred_{label}.csv"
    submission_path = f"submission_{label}.csv"

    pd.DataFrame({
        "row_index": np.arange(n_train_34a),
        TARGET_COL: y_arr_34a,
        "pred_clipped": np.clip(pred_oof, 0, 100).astype(np.float32),
    }).to_csv(oof_path, index=False)

    pd.DataFrame({
        ID_COL: test_ids_34a,
        TARGET_COL: np.clip(pred_test, 0, 100).astype(np.float32),
    }).to_csv(test_path, index=False)

    pd.DataFrame({
        ID_COL: test_ids_34a,
        TARGET_COL: np.clip(pred_test, 0, 100).astype(np.float32),
    }).to_csv(submission_path, index=False)

    return oof_path, test_path, submission_path


# ------------------------------------------------------------
# 6. Main OOF loop
# ------------------------------------------------------------

folds_34a = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_34a))
)

direct_oof_34a = {
    cfg["name"]: np.full(n_train_34a, np.nan, dtype=np.float32)
    for cfg in spline_configs_34a
}
resid_oof_34a = {
    cfg["name"]: np.full(n_train_34a, np.nan, dtype=np.float32)
    for cfg in spline_configs_34a
}

direct_test_sum_34a = {
    cfg["name"]: np.zeros(n_test_34a, dtype=np.float64)
    for cfg in spline_configs_34a
}
resid_test_sum_34a = {
    cfg["name"]: np.zeros(n_test_34a, dtype=np.float64)
    for cfg in spline_configs_34a
}

fold_metric_rows_34a = []
selected_feature_rows_34a = []

overall_t0_34a = time.time()

for fold_num, (tr_idx, va_idx) in enumerate(folds_34a, start=1):
    print("\n" + "=" * 90)
    print(f"34A OUTER FOLD {fold_num}/{N_SPLITS}")
    print("=" * 90)

    fold_t0 = time.time()

    raw_fit_fold = raw_train_te.iloc[tr_idx].reset_index(drop=True)
    raw_val_fold = raw_train_te.iloc[va_idx].reset_index(drop=True)

    y_fit = y_arr_34a[tr_idx]
    y_val = y_arr_34a[va_idx]

    anchor_fit = anchor_oof_34a[tr_idx]
    anchor_val = anchor_oof_34a[va_idx]

    resid_fit = y_fit - anchor_fit

    print("Building leakage-safe TE features for spline/GAM fold...")

    te_fit_fold, apply_dict_fold, te_key_summary_fold = build_te_oof_and_apply_many(
        raw_fit=raw_fit_fold,
        y_fit=y_fit,
        raw_apply_dict={
            "valid": raw_val_fold,
            "test": raw_test_te,
        },
        key_specs=te_key_specs,
        n_splits=5,
        random_state=RANDOM_STATE + 3400 + fold_num,
    )

    # Prefix TE columns so they do not collide with base columns.
    te_fit_fold = te_fit_fold.reset_index(drop=True).add_prefix("TE__")
    te_val_fold = apply_dict_fold["valid"].reset_index(drop=True).add_prefix("TE__")
    te_test_fold = apply_dict_fold["test"].reset_index(drop=True).add_prefix("TE__")

    X_fit_base_fold = X_train_proc_model.iloc[tr_idx][base_cont_cols_34a].reset_index(drop=True)
    X_val_base_fold = X_train_proc_model.iloc[va_idx][base_cont_cols_34a].reset_index(drop=True)
    X_test_base_fold = X_test_proc_model[base_cont_cols_34a].reset_index(drop=True)

    X_fit_all = pd.concat([X_fit_base_fold, te_fit_fold], axis=1)
    X_val_all = pd.concat([X_val_base_fold, te_val_fold], axis=1)
    X_test_all = pd.concat([X_test_base_fold, te_test_fold], axis=1)

    # Remove duplicated columns defensively.
    X_fit_all = X_fit_all.loc[:, ~X_fit_all.columns.duplicated()]
    X_val_all = X_val_all[X_fit_all.columns]
    X_test_all = X_test_all[X_fit_all.columns]

    print("Fold spline candidate matrix:", X_fit_all.shape)

    direct_rank = rank_columns_by_abs_corr_34a(X_fit_all, y_fit)
    resid_rank = rank_columns_by_abs_corr_34a(X_fit_all, resid_fit)

    print("Top direct-correlation features:")
    print(direct_rank.head(8).to_string(index=False))

    print("\nTop residual-correlation features:")
    print(resid_rank.head(8).to_string(index=False))

    for cfg in spline_configs_34a:
        name = cfg["name"]
        top_k = int(cfg["top_k"])

        direct_cols = direct_rank["feature"].head(min(top_k, len(direct_rank))).tolist()
        resid_cols = resid_rank["feature"].head(min(top_k, len(resid_rank))).tolist()

        for rank_pos, col in enumerate(direct_cols, start=1):
            selected_feature_rows_34a.append({
                "fold": fold_num,
                "config": name,
                "mode": "direct",
                "rank": rank_pos,
                "feature": col,
            })

        for rank_pos, col in enumerate(resid_cols, start=1):
            selected_feature_rows_34a.append({
                "fold": fold_num,
                "config": name,
                "mode": "residual",
                "rank": rank_pos,
                "feature": col,
            })

        # -----------------------------
        # Direct target spline/GAM model
        # -----------------------------
        try:
            t0 = time.time()

            direct_model = make_spline_ridge_34a(
                n_knots=cfg["n_knots"],
                degree=cfg["degree"],
                alpha=cfg["alpha"],
            )

            direct_model.fit(X_fit_all[direct_cols], y_fit)

            pred_val_direct = np.asarray(
                direct_model.predict(X_val_all[direct_cols])
            ).reshape(-1)
            pred_test_direct = np.asarray(
                direct_model.predict(X_test_all[direct_cols])
            ).reshape(-1)

            pred_val_direct_clip = np.clip(pred_val_direct, 0, 100).astype(np.float32)
            pred_test_direct_clip = np.clip(pred_test_direct, 0, 100).astype(np.float32)

            direct_oof_34a[name][va_idx] = pred_val_direct_clip
            direct_test_sum_34a[name] += pred_test_direct_clip.astype(np.float64)

            fold_mse_direct = float(mean_squared_error(y_val, pred_val_direct_clip))
            fold_metric_rows_34a.append({
                "fold": fold_num,
                "config": name,
                "mode": "direct_target",
                "top_k": cfg["top_k"],
                "n_knots": cfg["n_knots"],
                "degree": cfg["degree"],
                "alpha": cfg["alpha"],
                "fold_mse": fold_mse_direct,
                "fold_gain_vs_anchor": float(mean_squared_error(y_val, anchor_val) - fold_mse_direct),
                "elapsed_seconds": float(time.time() - t0),
                "status": "ok",
            })

            print(
                f"{name} | direct target: fold MSE {fold_mse_direct:.6f} "
                f"| gain vs anchor {mean_squared_error(y_val, anchor_val) - fold_mse_direct:.6f}"
            )

            del direct_model, pred_val_direct, pred_test_direct

        except Exception as e:
            print(f"ERROR direct target | {name} | fold {fold_num}: {repr(e)}")
            fold_metric_rows_34a.append({
                "fold": fold_num,
                "config": name,
                "mode": "direct_target",
                "top_k": cfg["top_k"],
                "n_knots": cfg["n_knots"],
                "degree": cfg["degree"],
                "alpha": cfg["alpha"],
                "fold_mse": np.nan,
                "fold_gain_vs_anchor": np.nan,
                "elapsed_seconds": np.nan,
                "status": repr(e),
            })

        # --------------------------------
        # Residual correction spline/GAM
        # --------------------------------
        try:
            t0 = time.time()

            resid_model = make_spline_ridge_34a(
                n_knots=cfg["n_knots"],
                degree=cfg["degree"],
                alpha=cfg["alpha"],
            )

            resid_model.fit(X_fit_all[resid_cols], resid_fit)

            pred_val_resid = np.asarray(
                resid_model.predict(X_val_all[resid_cols])
            ).reshape(-1)
            pred_test_resid = np.asarray(
                resid_model.predict(X_test_all[resid_cols])
            ).reshape(-1)

            resid_oof_34a[name][va_idx] = pred_val_resid.astype(np.float32)
            resid_test_sum_34a[name] += pred_test_resid.astype(np.float64)

            corrected_val_lambda1 = np.clip(anchor_val + pred_val_resid, 0, 100)
            fold_mse_resid_lam1 = float(mean_squared_error(y_val, corrected_val_lambda1))

            fold_metric_rows_34a.append({
                "fold": fold_num,
                "config": name,
                "mode": "residual_lambda1",
                "top_k": cfg["top_k"],
                "n_knots": cfg["n_knots"],
                "degree": cfg["degree"],
                "alpha": cfg["alpha"],
                "fold_mse": fold_mse_resid_lam1,
                "fold_gain_vs_anchor": float(mean_squared_error(y_val, anchor_val) - fold_mse_resid_lam1),
                "elapsed_seconds": float(time.time() - t0),
                "status": "ok",
            })

            print(
                f"{name} | residual lambda=1: fold MSE {fold_mse_resid_lam1:.6f} "
                f"| gain vs anchor {mean_squared_error(y_val, anchor_val) - fold_mse_resid_lam1:.6f}"
            )

            del resid_model, pred_val_resid, pred_test_resid

        except Exception as e:
            print(f"ERROR residual | {name} | fold {fold_num}: {repr(e)}")
            fold_metric_rows_34a.append({
                "fold": fold_num,
                "config": name,
                "mode": "residual_lambda1",
                "top_k": cfg["top_k"],
                "n_knots": cfg["n_knots"],
                "degree": cfg["degree"],
                "alpha": cfg["alpha"],
                "fold_mse": np.nan,
                "fold_gain_vs_anchor": np.nan,
                "elapsed_seconds": np.nan,
                "status": repr(e),
            })

    print(f"\nFold {fold_num} elapsed: {time.time() - fold_t0:.1f}s")

    del te_fit_fold, te_val_fold, te_test_fold
    del X_fit_base_fold, X_val_base_fold, X_test_base_fold
    del X_fit_all, X_val_all, X_test_all
    gc.collect()

# ------------------------------------------------------------
# 7. Final screen: direct, direct-blend, residual correction
# ------------------------------------------------------------

direct_test_34a = {
    name: np.clip(direct_test_sum_34a[name] / N_SPLITS, 0, 100).astype(np.float32)
    for name in direct_test_sum_34a
}
resid_test_34a = {
    name: (resid_test_sum_34a[name] / N_SPLITS).astype(np.float32)
    for name in resid_test_sum_34a
}

screen_rows_34a = []

for cfg in spline_configs_34a:
    name = cfg["name"]

    # Direct model by itself.
    direct_oof = direct_oof_34a[name]
    if np.isfinite(direct_oof).all():
        direct_mse = float(mean_squared_error(y_arr_34a, np.clip(direct_oof, 0, 100)))

        screen_rows_34a.append({
            "candidate_type": "direct_raw",
            "config": name,
            "top_k": cfg["top_k"],
            "n_knots": cfg["n_knots"],
            "degree": cfg["degree"],
            "alpha": cfg["alpha"],
            "oof_mse": direct_mse,
            "gain_vs_anchor": anchor_mse_34a - direct_mse,
            "w_anchor": np.nan,
            "residual_lambda": np.nan,
        })

        # Direct model blended with the anchor.
        best_direct_blend = None
        for w_anchor in np.linspace(0.0, 1.0, 501):
            pred = np.clip(w_anchor * anchor_oof_34a + (1.0 - w_anchor) * direct_oof, 0, 100)
            mse = float(mean_squared_error(y_arr_34a, pred))

            row = {
                "candidate_type": "direct_anchor_blend",
                "config": name,
                "top_k": cfg["top_k"],
                "n_knots": cfg["n_knots"],
                "degree": cfg["degree"],
                "alpha": cfg["alpha"],
                "oof_mse": mse,
                "gain_vs_anchor": anchor_mse_34a - mse,
                "w_anchor": float(w_anchor),
                "residual_lambda": np.nan,
            }

            if best_direct_blend is None or row["oof_mse"] < best_direct_blend["oof_mse"]:
                best_direct_blend = row

        screen_rows_34a.append(best_direct_blend)

    # Residual model as anchor + lambda * residual.
    resid_oof = resid_oof_34a[name]
    if np.isfinite(resid_oof).all():
        best_resid = None
        for lam in np.linspace(-0.50, 1.50, 501):
            pred = np.clip(anchor_oof_34a + lam * resid_oof, 0, 100)
            mse = float(mean_squared_error(y_arr_34a, pred))

            row = {
                "candidate_type": "residual_correction",
                "config": name,
                "top_k": cfg["top_k"],
                "n_knots": cfg["n_knots"],
                "degree": cfg["degree"],
                "alpha": cfg["alpha"],
                "oof_mse": mse,
                "gain_vs_anchor": anchor_mse_34a - mse,
                "w_anchor": np.nan,
                "residual_lambda": float(lam),
            }

            if best_resid is None or row["oof_mse"] < best_resid["oof_mse"]:
                best_resid = row

        screen_rows_34a.append(best_resid)

screen34a = pd.DataFrame(screen_rows_34a)

if len(screen34a) == 0:
    raise ValueError("No complete 34A spline/GAM candidates were produced.")

screen34a = screen34a.sort_values("oof_mse").reset_index(drop=True)

def prediction_from_screen_row_34a(row):
    name = row["config"]
    ctype = row["candidate_type"]

    if ctype == "direct_raw":
        pred_oof = np.clip(direct_oof_34a[name], 0, 100).astype(np.float32)
        pred_test = np.clip(direct_test_34a[name], 0, 100).astype(np.float32)
        return pred_oof, pred_test

    if ctype == "direct_anchor_blend":
        w_anchor = float(row["w_anchor"])
        pred_oof = np.clip(
            w_anchor * anchor_oof_34a + (1.0 - w_anchor) * direct_oof_34a[name],
            0,
            100,
        ).astype(np.float32)
        pred_test = np.clip(
            w_anchor * anchor_test_34a + (1.0 - w_anchor) * direct_test_34a[name],
            0,
            100,
        ).astype(np.float32)
        return pred_oof, pred_test

    if ctype == "residual_correction":
        lam = float(row["residual_lambda"])
        pred_oof = np.clip(anchor_oof_34a + lam * resid_oof_34a[name], 0, 100).astype(np.float32)
        pred_test = np.clip(anchor_test_34a + lam * resid_test_34a[name], 0, 100).astype(np.float32)
        return pred_oof, pred_test

    raise ValueError(f"Unknown candidate_type: {ctype}")

best34a = screen34a.iloc[0]
best_oof34a, best_test34a = prediction_from_screen_row_34a(best34a)

best_label34a = "spline34a_gam_best"
best_oof_path34a, best_test_path34a, best_submission_path34a = save_candidate_34a(
    best_label34a,
    best_oof34a,
    best_test34a,
)

fold_gains34a = fold_mse_table_34a(best_oof34a, best_label34a)

# Also save best residual candidate separately if it exists.
best_resid_paths34a = None
resid_only34a = screen34a[screen34a["candidate_type"] == "residual_correction"].copy()
if len(resid_only34a) > 0:
    best_resid34a = resid_only34a.iloc[0]
    best_resid_oof34a, best_resid_test34a = prediction_from_screen_row_34a(best_resid34a)
    best_resid_paths34a = save_candidate_34a(
        "spline34a_gam_best_residual",
        best_resid_oof34a,
        best_resid_test34a,
    )

# Save diagnostics.
screen_path34a = "model_results/spline34a_gam_screen.csv"
fold_metrics_path34a = "model_results/spline34a_gam_fold_metrics.csv"
fold_gains_path34a = "model_results/spline34a_gam_best_fold_gains.csv"
selected_features_path34a = "model_results/spline34a_gam_selected_features.csv"
feature_counts_path34a = "model_results/spline34a_gam_selected_feature_counts.csv"

fold_metrics34a = pd.DataFrame(fold_metric_rows_34a)
selected_features34a = pd.DataFrame(selected_feature_rows_34a)

screen34a.to_csv(screen_path34a, index=False)
fold_metrics34a.to_csv(fold_metrics_path34a, index=False)
fold_gains34a.to_csv(fold_gains_path34a, index=False)
selected_features34a.to_csv(selected_features_path34a, index=False)

if len(selected_features34a) > 0:
    feature_counts34a = (
        selected_features34a
        .groupby(["mode", "feature"])
        .size()
        .reset_index(name="times_selected")
        .sort_values(["times_selected", "mode", "feature"], ascending=[False, True, True])
        .reset_index(drop=True)
    )
else:
    feature_counts34a = pd.DataFrame(columns=["mode", "feature", "times_selected"])

feature_counts34a.to_csv(feature_counts_path34a, index=False)

# ------------------------------------------------------------
# 8. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("34A spline/GAM screen complete")
print("=" * 90)

print("\nAnchor")
print("------")
print(f"Anchor OOF MSE: {anchor_mse_34a:.6f}")

print("\nTop 15 spline/GAM candidates")
print("----------------------------")
display(screen34a.head(15))

print("\nBest 34A candidate")
print("------------------")
print(best34a.to_string())
print(f"\nBest 34A OOF MSE: {mean_squared_error(y_arr_34a, best_oof34a):.6f}")
print(f"Gain vs anchor:   {anchor_mse_34a - mean_squared_error(y_arr_34a, best_oof34a):.6f}")
print(f"Min fold gain:    {fold_gains34a['gain_vs_anchor'].min():.6f}")

print("\nBest candidate fold gains")
print("-------------------------")
print(fold_gains34a.to_string(index=False))

print("\nMost frequently selected spline/GAM features")
print("--------------------------------------------")
display(feature_counts34a.head(30))

print("\nSaved files")
print("-----------")
print(screen_path34a)
print(fold_metrics_path34a)
print(fold_gains_path34a)
print(selected_features_path34a)
print(feature_counts_path34a)
print(best_oof_path34a)
print(best_test_path34a)
print(best_submission_path34a)

if best_resid_paths34a is not None:
    print("\nBest residual-only saved files")
    print("------------------------------")
    for p in best_resid_paths34a:
        print(p)

print("\nRuntime seconds:", round(time.time() - overall_t0_34a, 1))

print("\nDecision rule")
print("-------------")
if anchor_mse_34a - mean_squared_error(y_arr_34a, best_oof34a) >= 5.0:
    print("This spline/GAM family has meaningful signal. Continue within spline/GAM before moving to KNN/local methods.")
elif anchor_mse_34a - mean_squared_error(y_arr_34a, best_oof34a) >= 1.0:
    print("This spline/GAM family has some signal, but not enough yet. Try weighted or segmented spline/GAM next.")
else:
    print("This spline/GAM screen did not show material improvement. Do not submit unless public validation says otherwise.")

34A. Regression splines / GAM-style additive model
Anchor OOF path:  model_results/oof_seg33a_arcsine_adaptive.csv
Anchor test path: model_results/testpred_seg33a_arcsine_adaptive.csv

Anchor check
------------
Anchor OOF prediction column: pred_clipped
Anchor test prediction column: PERCENT_PROFICIENT
Anchor OOF MSE: 77.049866

Spline feature pool
-------------------
Continuous base candidates: 54
TE key specs: 17
Expected TE columns per fold: 68

Configs
-------
spline34a_gam_te_additive_top8_knots4_deg3_alpha100p0
spline34a_gam_te_additive_top8_knots6_deg3_alpha100p0
spline34a_gam_te_additive_top12_knots4_deg3_alpha100p0
spline34a_gam_te_additive_top12_knots6_deg3_alpha100p0
spline34a_gam_te_additive_top20_knots4_deg3_alpha1000p0
spline34a_gam_te_additive_top20_knots6_deg3_alpha1000p0

34A OUTER FOLD 1/5
Building leakage-safe TE features for spline/GAM fold...
Fold spline candidate matrix: (115936, 122)
Top direct-correlation features:
                                            fea

,candidate_type,config,top_k,n_knots,degree,alpha,oof_mse,gain_vs_anchor,w_anchor,residual_lambda
0,residual_correction,spline34a_gam_te_additive_top20_knots4_deg3_al...,20,4,3,1000.0,76.929332,0.120533,NaN,0.416
1,residual_correction,spline34a_gam_te_additive_top20_knots6_deg3_al...,20,6,3,1000.0,76.956146,0.093720,NaN,0.352
2,residual_correction,spline34a_gam_te_additive_top12_knots4_deg3_al...,12,4,3,100.0,76.961279,0.088587,NaN,0.372
3,residual_correction,spline34a_gam_te_additive_top12_knots6_deg3_al...,12,6,3,100.0,76.994931,0.054935,NaN,0.248
4,residual_correction,spline34a_gam_te_additive_top8_knots4_deg3_alp...,8,4,3,100.0,77.008787,0.041079,NaN,0.292
5,residual_correction,spline34a_gam_te_additive_top8_knots6_deg3_alp...,8,6,3,100.0,77.025153,0.024713,NaN,0.196
6,direct_anchor_blend,spline34a_gam_te_additive_top8_knots6_deg3_alp...,8,6,3,100.0,77.049862,0.000003,1.0,NaN
7,direct_anchor_blend,spline34a_gam_te_additive_top12_knots4_deg3_al...,12,4,3,100.0,77.049862,0.000003,1.0,NaN
8,direct_anchor_blend,spline34a_gam_te_additive_top20_knots6_deg3_al...,20,6,3,1000.0,77.049862,0.000003,1.0,NaN
9,direct_anchor_blend,spline34a_gam_te_additive_top12_knots6_deg3_al...,12,6,3,100.0,77.049862,0.000003,1.0,NaN



Best 34A candidate
------------------
candidate_type                                   residual_correction
config             spline34a_gam_te_additive_top20_knots4_deg3_al...
top_k                                                             20
n_knots                                                            4
degree                                                             3
alpha                                                         1000.0
oof_mse                                                    76.929332
gain_vs_anchor                                              0.120533
w_anchor                                                         NaN
residual_lambda                                                0.416

Best 34A OOF MSE: 76.929337
Gain vs anchor:   0.120529
Min fold gain:    0.043991

Best candidate fold gains
-------------------------
         candidate  fold  anchor_mse  candidate_mse  gain_vs_anchor
spline34a_gam_best     1   78.427116      78.287201        0.139915

,mode,feature,times_selected
0,direct,TE__te_DISTRICT_TYPE_ASSESSMENT_NAME_a40_n300_...,30
1,direct,TE__te_DISTRICT_TYPE_ASSESSMENT_NAME_a40_n300_...,30
2,direct,TE__te_SCHOOL_ASSESSMENT_NAME_a220_n1000_mean,30
3,direct,TE__te_SCHOOL_ASSESSMENT_NAME_a220_n1000_wmean,30
4,direct,TE__te_SCHOOL_SUBGROUP_NAME_a140_n800_mean,30
5,direct,TE__te_SCHOOL_a100_n600_mean,30
6,direct,TE__te_SCHOOL_a100_n600_wmean,30
7,residual,TE__te_SCHOOL_ASSESSMENT_NAME_a220_n1000_mean,30
8,residual,TE__te_SCHOOL_ASSESSMENT_NAME_a220_n1000_wmean,30
9,residual,TE__te_SUBGROUP_NAME_a20_n200_mean,30



Saved files
-----------
model_results/spline34a_gam_screen.csv
model_results/spline34a_gam_fold_metrics.csv
model_results/spline34a_gam_best_fold_gains.csv
model_results/spline34a_gam_selected_features.csv
model_results/spline34a_gam_selected_feature_counts.csv
model_results/oof_spline34a_gam_best.csv
model_results/testpred_spline34a_gam_best.csv
submission_spline34a_gam_best.csv

Best residual-only saved files
------------------------------
model_results/oof_spline34a_gam_best_residual.csv
model_results/testpred_spline34a_gam_best_residual.csv
submission_spline34a_gam_best_residual.csv

Runtime seconds: 157.6

Decision rule
-------------
This spline/GAM screen did not show material improvement. Do not submit unless public validation says otherwise.


### 34A. Regression Splines / GAM-Style Additive Residual Screen

This section tested the regression spline / GAM-style additive model family after the target-encoding and boosting branch. The goal was to determine whether the current 33A anchor had smooth residual structure that could be captured by cubic B-spline basis functions with Ridge regularization.

The 33A anchor OOF MSE was `77.049866`. The best 34A spline/GAM candidate was a residual-correction model using the top 20 selected features, 4 cubic spline knots, degree 3 splines, Ridge alpha `1000.0`, and a residual shrinkage value of `0.416`.

The best 34A OOF MSE was `76.929337`, for a gain of only `0.120529` versus the 33A anchor. The minimum fold gain was positive at `0.043991`, so the correction was fold-stable, but the improvement was far too small to be considered material.

The direct spline/GAM target models were not competitive. The best direct raw spline model still had OOF MSE around `157.010483`, much worse than the anchor. The direct-anchor blend selected weight `1.0` on the anchor, meaning the direct spline predictions added no useful standalone signal.

Conclusion: generic additive splines on the TE feature space are not a promising standalone direction. The only useful result was a very small residual correction, so this model should not be submitted. Before closing the spline/GAM family entirely, the next step is a more targeted spline/GAM calibration model: smooth the residual as a function of the anchor prediction itself, optionally with categorical factor effects for assessment/subgroup segments.

In [87]:
# ============================================================
# 34B. Spline/GAM anchor calibration + additive factor effects
# ============================================================
#
# Textbook family:
#   12. Regression splines / cubic splines
#   13. Smoothing-spline-like shrinkage via Ridge
#   15. GAM-style additive model
#
# Purpose:
# - 34A showed generic TE splines gave only a tiny residual correction.
# - 34B tests a more targeted GAM idea:
#       residual = y - anchor
#       residual ~ smooth(anchor prediction) + smooth(selected numeric covariates)
#                  + categorical factor effects
#
# This is still a spline/GAM-family model, not a LightGBM tweak.
# ============================================================

import os
import gc
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer, OneHotEncoder
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")
os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

ANCHOR_OOF_PATH_34B = "model_results/oof_seg33a_arcsine_adaptive.csv"
ANCHOR_TEST_PATH_34B = "model_results/testpred_seg33a_arcsine_adaptive.csv"

ARTIFACT_PREFIX_34B = "spline34b_calib_gam"

print("=" * 90)
print("34B. Spline/GAM anchor calibration + additive factor effects")
print("=" * 90)

# ------------------------------------------------------------
# 1. Basic checks and anchor loading
# ------------------------------------------------------------

if "y_train" not in globals():
    raise ValueError("Missing y_train.")

y_arr_34b = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_34b = len(y_arr_34b)

if "X_train_proc_model" not in globals() or "X_test_proc_model" not in globals():
    raise ValueError("Missing X_train_proc_model or X_test_proc_model.")

if not isinstance(X_train_proc_model, pd.DataFrame) or not isinstance(X_test_proc_model, pd.DataFrame):
    raise ValueError("34B expects X_train_proc_model and X_test_proc_model to be pandas DataFrames.")

n_test_34b = X_test_proc_model.shape[0]

def load_oof_prediction_34b(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_priority = [
        "pred_clipped",
        "prediction",
        "pred",
        "oof_pred",
        "PREDICTED_PERCENT_PROFICIENT",
    ]

    pred_col = None
    for c in pred_priority:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite predictions in {path}, column {pred_col}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_y_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_y_diff > 1e-5:
            raise ValueError(f"Anchor OOF target mismatch. Max difference: {max_y_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col


def load_test_prediction_34b(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}, column {pred_col}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids_34a" in globals():
        ids = np.asarray(test_ids_34a)
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no global test ids found.")

    if len(pred) != n_test_34b:
        raise ValueError(f"Test row mismatch for {path}: got {len(pred)}, expected {n_test_34b}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col


if "anchor_oof_34a" in globals() and len(anchor_oof_34a) == n_train_34b:
    anchor_oof_34b = np.asarray(anchor_oof_34a, dtype=np.float32)
    anchor_oof_col_34b = "anchor_oof_34a"
else:
    anchor_oof_34b, anchor_oof_col_34b = load_oof_prediction_34b(
        ANCHOR_OOF_PATH_34B,
        y_arr_34b,
    )

if "anchor_test_34a" in globals() and len(anchor_test_34a) == n_test_34b:
    anchor_test_34b = np.asarray(anchor_test_34a, dtype=np.float32)
    test_ids_34b = np.asarray(test_ids_34a) if "test_ids_34a" in globals() else np.asarray(test_ids)
    anchor_test_col_34b = "anchor_test_34a"
else:
    anchor_test_34b, test_ids_34b, anchor_test_col_34b = load_test_prediction_34b(
        ANCHOR_TEST_PATH_34B
    )

anchor_mse_34b = float(mean_squared_error(y_arr_34b, anchor_oof_34b))

print("\nAnchor check")
print("------------")
print("Anchor OOF source:", anchor_oof_col_34b)
print("Anchor test source:", anchor_test_col_34b)
print(f"Anchor OOF MSE: {anchor_mse_34b:.6f}")

# ------------------------------------------------------------
# 2. Build calibration feature frame
# ------------------------------------------------------------

cal_train_34b = pd.DataFrame({
    "anchor_pred": anchor_oof_34b.astype(np.float32)
})

cal_test_34b = pd.DataFrame({
    "anchor_pred": anchor_test_34b.astype(np.float32)
})

numeric_covariate_candidates_34b = [
    "N_STUDENTS",
    "TOTAL_STUDENTS",
    "ENROLLMENT",
    "PERCENT_FREE_LUNCH",
    "PERCENT_REDUCED_LUNCH",
    "PERCENT_ECONOMICALLY_DISADVANTAGED",
    "PERCENT_ENGLISH_LANGUAGE_LEANERS",
    "PERCENT_STUDENTS_WITH_DISABILITIES",
    "PERCENT_MIGRANT",
    "PERCENT_HOMELESS",
    "PERCENT_FEMALE",
    "PERCENT_MALE",
]

numeric_covariates_34b = []

for c in numeric_covariate_candidates_34b:
    if c in X_train_proc_model.columns and c in X_test_proc_model.columns:
        if pd.api.types.is_numeric_dtype(X_train_proc_model[c]):
            nunique = int(X_train_proc_model[c].nunique(dropna=True))
            if nunique > 8:
                train_vals = (
                    pd.to_numeric(X_train_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )
                test_vals = (
                    pd.to_numeric(X_test_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )

                cal_train_34b[c] = train_vals
                cal_test_34b[c] = test_vals
                numeric_covariates_34b.append(c)

def clean_cat_34b(s):
    s = pd.Series(s).astype("object")
    s = s.where(pd.notna(s), "__NA__")
    return s.astype(str)

categorical_factor_cols_34b = []

if "raw_train_te" in globals() and "raw_test_te" in globals():
    if len(raw_train_te) != n_train_34b or len(raw_test_te) != n_test_34b:
        raise ValueError("raw_train_te/raw_test_te row counts do not match train/test sizes.")

    single_cat_candidates_34b = [
        "SUBGROUP_NAME",
        "ASSESSMENT_NAME",
        "GRADE",
        "TESTED_GRADE",
        "SUBJECT",
        "SCHOOL_TYPE",
        "DISTRICT_TYPE",
        "COUNTY",
        "REGION",
        "CHARTER",
        "TITLE_I_STATUS",
    ]

    for c in single_cat_candidates_34b:
        if c in raw_train_te.columns and c in raw_test_te.columns:
            train_s = clean_cat_34b(raw_train_te[c])
            test_s = clean_cat_34b(raw_test_te[c])
            card = int(train_s.nunique(dropna=True))

            if 2 <= card <= 500:
                new_c = f"CAT__{c}"
                cal_train_34b[new_c] = train_s.to_numpy()
                cal_test_34b[new_c] = test_s.to_numpy()
                categorical_factor_cols_34b.append(new_c)

    combo_candidates_34b = [
        ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
        ("GRADE", "SUBGROUP_NAME"),
        ("TESTED_GRADE", "SUBGROUP_NAME"),
        ("DISTRICT_TYPE", "ASSESSMENT_NAME"),
        ("REGION", "ASSESSMENT_NAME"),
        ("COUNTY", "SUBGROUP_NAME"),
    ]

    for combo in combo_candidates_34b:
        if all(c in raw_train_te.columns for c in combo) and all(c in raw_test_te.columns for c in combo):
            combo_name = "CAT__" + "__X__".join(combo)

            train_parts = [clean_cat_34b(raw_train_te[c]) for c in combo]
            test_parts = [clean_cat_34b(raw_test_te[c]) for c in combo]

            train_combo = train_parts[0].copy()
            test_combo = test_parts[0].copy()

            for p in train_parts[1:]:
                train_combo = train_combo + "__" + p
            for p in test_parts[1:]:
                test_combo = test_combo + "__" + p

            card = int(train_combo.nunique(dropna=True))

            if 2 <= card <= 700:
                cal_train_34b[combo_name] = train_combo.to_numpy()
                cal_test_34b[combo_name] = test_combo.to_numpy()
                categorical_factor_cols_34b.append(combo_name)

else:
    print("raw_train_te/raw_test_te not found; 34B will run global smooth calibration only.")

# Keep the config search controlled.
categorical_factor_cols_34b = categorical_factor_cols_34b[:10]

num_sets_34b = {
    "anchor": ["anchor_pred"]
}

if len(numeric_covariates_34b) > 0:
    num_sets_34b["anchor_covars"] = ["anchor_pred"] + numeric_covariates_34b

print("\nCalibration feature pool")
print("------------------------")
print("Numeric covariates:", numeric_covariates_34b)
print("Categorical factor columns:", categorical_factor_cols_34b)
print("Calibration train shape:", cal_train_34b.shape)
print("Calibration test shape: ", cal_test_34b.shape)

# ------------------------------------------------------------
# 3. Configs
# ------------------------------------------------------------

def safe_name_34b(x):
    x = str(x)
    x = x.replace("CAT__", "")
    x = x.replace("__X__", "x")
    x = x.replace("__", "_")
    x = x.replace(" ", "")
    x = x.replace("/", "")
    return x[:80]

configs_34b = []

# Global smooth calibration.
for num_key in num_sets_34b:
    for n_knots in [4, 6, 8]:
        for alpha in [10.0, 100.0, 1000.0]:
            configs_34b.append({
                "name": (
                    f"{ARTIFACT_PREFIX_34B}_{num_key}"
                    f"_knots{n_knots}_alpha{str(alpha).replace('.', 'p')}"
                ),
                "num_key": num_key,
                "cat_cols": [],
                "n_knots": n_knots,
                "degree": 3,
                "alpha": alpha,
            })

# Additive factor-effect GAM calibration.
for cat_col in categorical_factor_cols_34b:
    for alpha in [100.0, 1000.0, 10000.0]:
        configs_34b.append({
            "name": (
                f"{ARTIFACT_PREFIX_34B}_anchor_factor_{safe_name_34b(cat_col)}"
                f"_knots4_alpha{str(alpha).replace('.', 'p')}"
            ),
            "num_key": "anchor",
            "cat_cols": [cat_col],
            "n_knots": 4,
            "degree": 3,
            "alpha": alpha,
        })

        if "anchor_covars" in num_sets_34b:
            configs_34b.append({
                "name": (
                    f"{ARTIFACT_PREFIX_34B}_anchorcov_factor_{safe_name_34b(cat_col)}"
                    f"_knots4_alpha{str(alpha).replace('.', 'p')}"
                ),
                "num_key": "anchor_covars",
                "cat_cols": [cat_col],
                "n_knots": 4,
                "degree": 3,
                "alpha": alpha,
            })

print("\n34B configs")
print("-----------")
print("Number of configs:", len(configs_34b))
for cfg in configs_34b[:25]:
    print(cfg["name"])
if len(configs_34b) > 25:
    print(f"... {len(configs_34b) - 25} more configs")

# ------------------------------------------------------------
# 4. Model helper
# ------------------------------------------------------------

def make_one_hot_34b():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def make_calib_gam_model_34b(num_cols, cat_cols, n_knots, degree, alpha):
    transformers = []

    if len(num_cols) > 0:
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler1", StandardScaler()),
            ("spline", SplineTransformer(
                n_knots=int(n_knots),
                degree=int(degree),
                include_bias=False,
                extrapolation="constant",
            )),
            ("scaler2", StandardScaler(with_mean=False)),
        ])
        transformers.append(("num_smooths", num_pipe, num_cols))

    if len(cat_cols) > 0:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="__NA__")),
            ("onehot", make_one_hot_34b()),
        ])
        transformers.append(("cat_factors", cat_pipe, cat_cols))

    pre = ColumnTransformer(
        transformers=transformers,
        sparse_threshold=0.75,
    )

    model = Pipeline([
        ("pre", pre),
        ("ridge", Ridge(alpha=float(alpha), solver="lsqr")),
    ])

    return model

# ------------------------------------------------------------
# 5. OOF calibration loop
# ------------------------------------------------------------

folds_34b = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_34b))
)

resid_oof_34b = {
    cfg["name"]: np.full(n_train_34b, np.nan, dtype=np.float32)
    for cfg in configs_34b
}

resid_test_sum_34b = {
    cfg["name"]: np.zeros(n_test_34b, dtype=np.float64)
    for cfg in configs_34b
}

fold_metric_rows_34b = []

overall_t0_34b = time.time()

for fold_num, (tr_idx, va_idx) in enumerate(folds_34b, start=1):
    print("\n" + "=" * 90)
    print(f"34B OUTER FOLD {fold_num}/{N_SPLITS}")
    print("=" * 90)

    fold_t0 = time.time()

    y_fit = y_arr_34b[tr_idx]
    y_val = y_arr_34b[va_idx]

    anchor_fit = anchor_oof_34b[tr_idx]
    anchor_val = anchor_oof_34b[va_idx]

    resid_fit = y_fit - anchor_fit

    fit_df = cal_train_34b.iloc[tr_idx].reset_index(drop=True)
    val_df = cal_train_34b.iloc[va_idx].reset_index(drop=True)
    test_df = cal_test_34b.reset_index(drop=True)

    fold_anchor_mse = float(mean_squared_error(y_val, anchor_val))
    print(f"Fold anchor MSE: {fold_anchor_mse:.6f}")
    print(f"Fitting {len(configs_34b)} calibration/GAM configs...")

    for cfg_i, cfg in enumerate(configs_34b, start=1):
        name = cfg["name"]
        num_cols = num_sets_34b[cfg["num_key"]]
        cat_cols = cfg["cat_cols"]
        used_cols = num_cols + cat_cols

        try:
            t0 = time.time()

            model = make_calib_gam_model_34b(
                num_cols=num_cols,
                cat_cols=cat_cols,
                n_knots=cfg["n_knots"],
                degree=cfg["degree"],
                alpha=cfg["alpha"],
            )

            model.fit(fit_df[used_cols], resid_fit)

            pred_val_resid = np.asarray(
                model.predict(val_df[used_cols])
            ).reshape(-1)

            pred_test_resid = np.asarray(
                model.predict(test_df[used_cols])
            ).reshape(-1)

            resid_oof_34b[name][va_idx] = pred_val_resid.astype(np.float32)
            resid_test_sum_34b[name] += pred_test_resid.astype(np.float64)

            pred_val_lam1 = np.clip(anchor_val + pred_val_resid, 0, 100)
            mse_lam1 = float(mean_squared_error(y_val, pred_val_lam1))

            fold_metric_rows_34b.append({
                "fold": fold_num,
                "config": name,
                "num_key": cfg["num_key"],
                "cat_cols": "|".join(cat_cols),
                "n_knots": cfg["n_knots"],
                "degree": cfg["degree"],
                "alpha": cfg["alpha"],
                "fold_anchor_mse": fold_anchor_mse,
                "fold_mse_lambda1": mse_lam1,
                "fold_gain_lambda1": fold_anchor_mse - mse_lam1,
                "elapsed_seconds": float(time.time() - t0),
                "status": "ok",
            })

            del model, pred_val_resid, pred_test_resid

        except Exception as e:
            print(f"ERROR | fold {fold_num} | {name}: {repr(e)}")

            fold_metric_rows_34b.append({
                "fold": fold_num,
                "config": name,
                "num_key": cfg["num_key"],
                "cat_cols": "|".join(cat_cols),
                "n_knots": cfg["n_knots"],
                "degree": cfg["degree"],
                "alpha": cfg["alpha"],
                "fold_anchor_mse": fold_anchor_mse,
                "fold_mse_lambda1": np.nan,
                "fold_gain_lambda1": np.nan,
                "elapsed_seconds": np.nan,
                "status": repr(e),
            })

    print(f"Fold {fold_num} elapsed: {time.time() - fold_t0:.1f}s")

    del fit_df, val_df, test_df
    gc.collect()

# ------------------------------------------------------------
# 6. Screen shrinkage values
# ------------------------------------------------------------

lambda_grid_34b = np.unique(
    np.concatenate([
        np.linspace(-1.0, 1.5, 626),
        np.array([0.0, 0.25, 0.5, 0.75, 1.0]),
    ])
)

screen_rows_34b = []

for cfg in configs_34b:
    name = cfg["name"]
    resid_oof = resid_oof_34b[name]

    if not np.isfinite(resid_oof).all():
        continue

    best_row = None

    for lam in lambda_grid_34b:
        pred = np.clip(anchor_oof_34b + float(lam) * resid_oof, 0, 100)
        mse = float(mean_squared_error(y_arr_34b, pred))

        row = {
            "config": name,
            "num_key": cfg["num_key"],
            "cat_cols": "|".join(cfg["cat_cols"]),
            "n_knots": cfg["n_knots"],
            "degree": cfg["degree"],
            "alpha": cfg["alpha"],
            "residual_lambda": float(lam),
            "oof_mse": mse,
            "gain_vs_anchor": anchor_mse_34b - mse,
        }

        if best_row is None or row["oof_mse"] < best_row["oof_mse"]:
            best_row = row

    screen_rows_34b.append(best_row)

screen34b = pd.DataFrame(screen_rows_34b)

if len(screen34b) == 0:
    raise ValueError("No complete 34B candidates were produced.")

screen34b = screen34b.sort_values("oof_mse").reset_index(drop=True)

best34b = screen34b.iloc[0]
best_name34b = best34b["config"]
best_lambda34b = float(best34b["residual_lambda"])

best_resid_oof34b = resid_oof_34b[best_name34b]
best_resid_test34b = resid_test_sum_34b[best_name34b] / N_SPLITS

best_oof34b = np.clip(anchor_oof_34b + best_lambda34b * best_resid_oof34b, 0, 100).astype(np.float32)
best_test34b = np.clip(anchor_test_34b + best_lambda34b * best_resid_test34b, 0, 100).astype(np.float32)

best_mse34b = float(mean_squared_error(y_arr_34b, best_oof34b))
best_gain34b = anchor_mse_34b - best_mse34b

# ------------------------------------------------------------
# 7. Fold gains + saves
# ------------------------------------------------------------

fold_gains_rows_34b = []

for fold_num, (_, va_idx) in enumerate(folds_34b, start=1):
    anchor_fold_mse = float(mean_squared_error(y_arr_34b[va_idx], anchor_oof_34b[va_idx]))
    cand_fold_mse = float(mean_squared_error(y_arr_34b[va_idx], best_oof34b[va_idx]))

    fold_gains_rows_34b.append({
        "fold": fold_num,
        "anchor_mse": anchor_fold_mse,
        "candidate_mse": cand_fold_mse,
        "gain_vs_anchor": anchor_fold_mse - cand_fold_mse,
    })

fold_gains34b = pd.DataFrame(fold_gains_rows_34b)

screen_path34b = "model_results/spline34b_calib_gam_screen.csv"
fold_metrics_path34b = "model_results/spline34b_calib_gam_fold_metrics.csv"
fold_gains_path34b = "model_results/spline34b_calib_gam_best_fold_gains.csv"
oof_path34b = "model_results/oof_spline34b_calib_gam_best.csv"
testpred_path34b = "model_results/testpred_spline34b_calib_gam_best.csv"
submission_path34b = "submission_spline34b_calib_gam_best.csv"

screen34b.to_csv(screen_path34b, index=False)
pd.DataFrame(fold_metric_rows_34b).to_csv(fold_metrics_path34b, index=False)
fold_gains34b.to_csv(fold_gains_path34b, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_34b),
    TARGET_COL: y_arr_34b,
    "pred_clipped": best_oof34b,
}).to_csv(oof_path34b, index=False)

pd.DataFrame({
    ID_COL: test_ids_34b,
    TARGET_COL: best_test34b,
}).to_csv(testpred_path34b, index=False)

pd.DataFrame({
    ID_COL: test_ids_34b,
    TARGET_COL: best_test34b,
}).to_csv(submission_path34b, index=False)

# ------------------------------------------------------------
# 8. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("34B spline/GAM calibration screen complete")
print("=" * 90)

print("\nAnchor")
print("------")
print(f"Anchor OOF MSE: {anchor_mse_34b:.6f}")

print("\nTop 20 calibration/GAM candidates")
print("---------------------------------")
display(screen34b.head(20))

print("\nBest 34B candidate")
print("------------------")
print(best34b.to_string())
print(f"\nBest 34B OOF MSE: {best_mse34b:.6f}")
print(f"Gain vs anchor:   {best_gain34b:.6f}")
print(f"Min fold gain:    {fold_gains34b['gain_vs_anchor'].min():.6f}")

print("\nBest candidate fold gains")
print("-------------------------")
print(fold_gains34b.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path34b)
print(fold_metrics_path34b)
print(fold_gains_path34b)
print(oof_path34b)
print(testpred_path34b)
print(submission_path34b)

print("\nRuntime seconds:", round(time.time() - overall_t0_34b, 1))

print("\nDecision rule")
print("-------------")
if best_gain34b >= 5.0:
    print("Material spline/GAM calibration signal found. Continue within spline/GAM with a larger segmented/factor-smooth screen.")
elif best_gain34b >= 1.0:
    print("Some spline/GAM calibration signal found. Try one more targeted spline/GAM expansion before closing this family.")
else:
    print("No material spline/GAM calibration signal. If fold gains are also unstable or tiny, close the spline/GAM family after this output.")

34B. Spline/GAM anchor calibration + additive factor effects

Anchor check
------------
Anchor OOF source: anchor_oof_34a
Anchor test source: anchor_test_34a
Anchor OOF MSE: 77.049866

Calibration feature pool
------------------------
Numeric covariates: ['N_STUDENTS', 'PERCENT_FREE_LUNCH', 'PERCENT_REDUCED_LUNCH', 'PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_MIGRANT', 'PERCENT_HOMELESS', 'PERCENT_FEMALE', 'PERCENT_MALE']
Categorical factor columns: ['CAT__SUBGROUP_NAME', 'CAT__ASSESSMENT_NAME', 'CAT__DISTRICT_TYPE', 'CAT__COUNTY', 'CAT__REGION', 'CAT__ASSESSMENT_NAME__X__SUBGROUP_NAME', 'CAT__DISTRICT_TYPE__X__ASSESSMENT_NAME', 'CAT__REGION__X__ASSESSMENT_NAME', 'CAT__COUNTY__X__SUBGROUP_NAME']
Calibration train shape: (144921, 19)
Calibration test shape:  (48307, 19)

34B configs
-----------
Number of configs: 72
spline34b_calib_gam_anchor_knots4_alpha10p0
spline34b_calib_gam_anchor_knots4_alpha100p0
spline34b_calib_gam_anchor_knots4_alpha1000p0


,config,num_key,cat_cols,n_knots,degree,alpha,residual_lambda,oof_mse,gain_vs_anchor
0,spline34b_calib_gam_anchorcov_factor_ASSESSMEN...,anchor_covars,CAT__ASSESSMENT_NAME__X__SUBGROUP_NAME,4,3,100.0,0.492,76.963020,0.086845
1,spline34b_calib_gam_anchorcov_factor_ASSESSMEN...,anchor_covars,CAT__ASSESSMENT_NAME__X__SUBGROUP_NAME,4,3,1000.0,0.660,76.973801,0.076065
2,spline34b_calib_gam_anchorcov_factor_ASSESSMEN...,anchor_covars,CAT__ASSESSMENT_NAME,4,3,100.0,0.556,76.984009,0.065857
3,spline34b_calib_gam_anchor_factor_ASSESSMENT_N...,anchor,CAT__ASSESSMENT_NAME__X__SUBGROUP_NAME,4,3,100.0,0.492,76.984604,0.065262
4,spline34b_calib_gam_anchor_factor_ASSESSMENT_N...,anchor,CAT__ASSESSMENT_NAME__X__SUBGROUP_NAME,4,3,1000.0,0.780,76.985718,0.064148
5,spline34b_calib_gam_anchorcov_factor_ASSESSMEN...,anchor_covars,CAT__ASSESSMENT_NAME,4,3,1000.0,0.624,76.987244,0.062622
6,spline34b_calib_gam_anchorcov_factor_ASSESSMEN...,anchor_covars,CAT__ASSESSMENT_NAME__X__SUBGROUP_NAME,4,3,10000.0,0.684,77.005997,0.043869
7,spline34b_calib_gam_anchorcov_factor_DISTRICT_...,anchor_covars,CAT__DISTRICT_TYPE,4,3,100.0,0.544,77.006844,0.043022
8,spline34b_calib_gam_anchorcov_factor_ASSESSMEN...,anchor_covars,CAT__ASSESSMENT_NAME,4,3,10000.0,0.664,77.007088,0.042778
9,spline34b_calib_gam_anchor_factor_ASSESSMENT_N...,anchor,CAT__ASSESSMENT_NAME,4,3,100.0,0.612,77.007568,0.042297



Best 34B candidate
------------------
config             spline34b_calib_gam_anchorcov_factor_ASSESSMEN...
num_key                                                anchor_covars
cat_cols                      CAT__ASSESSMENT_NAME__X__SUBGROUP_NAME
n_knots                                                            4
degree                                                             3
alpha                                                          100.0
residual_lambda                                                0.492
oof_mse                                                     76.96302
gain_vs_anchor                                              0.086845

Best 34B OOF MSE: 76.963020
Gain vs anchor:   0.086845
Min fold gain:    0.017570

Best candidate fold gains
-------------------------
 fold  anchor_mse  candidate_mse  gain_vs_anchor
    1   78.427116      78.232887        0.194229
    2   74.554970      74.521584        0.033386
    3   78.135284      78.033188        0.102097
    4   

### 34B. Spline/GAM Anchor Calibration with Additive Factor Effects

This section continued the spline/GAM model family by testing a more targeted calibration model. Instead of applying splines directly to the target-encoded features, this model treated the current 33A prediction as an anchor and modeled the residual:

`residual = observed target - 33A anchor prediction`

The calibration model used cubic spline smooths of the anchor prediction and selected numeric covariates, with optional additive categorical factor effects for variables such as `ASSESSMENT_NAME`, `SUBGROUP_NAME`, `DISTRICT_TYPE`, and their combinations.

The 33A anchor OOF MSE was `77.049866`. The best 34B model used anchor prediction plus numeric covariates and an additive factor effect for `ASSESSMENT_NAME × SUBGROUP_NAME`. Its best residual shrinkage value was `0.492`.

The best 34B OOF MSE was `76.963020`, for a gain of only `0.086845` versus the 33A anchor. The fold gains were all positive, but very small:

- Fold 1 gain: `0.194229`
- Fold 2 gain: `0.033386`
- Fold 3 gain: `0.102097`
- Fold 4 gain: `0.086914`
- Fold 5 gain: `0.017570`

Conclusion: this additive spline/GAM calibration model is fold-stable but not materially useful. The best gain is smaller than the 34A residual spline gain and far below the threshold needed for a meaningful modeling improvement. Before closing the spline/GAM family, one final more flexible GAM-style variant should be tested: factor-specific smooths, where the calibration curve is allowed to vary by assessment/subgroup segment rather than only adding a categorical intercept.

In [88]:
# ============================================================
# 34C. Factor-specific smooth GAM calibration
# ============================================================
#
# Textbook family:
#   12. Regression splines / cubic splines
#   13. Smoothing-spline-like shrinkage via Ridge
#   15. Generalized Additive Models
#
# Why this follows 34A/34B:
# - 34A: generic spline/GAM on TE features gave tiny residual signal.
# - 34B: global anchor calibration + additive factor effects gave tiny signal.
# - 34C: final stronger GAM attempt:
#       residual ~ global smooth(anchor)
#                  + group intercepts
#                  + group-specific smooth(anchor)
#
# This tests whether calibration curves differ by assessment/subgroup segment.
# ============================================================

import os
import gc
import time
import warnings
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")
os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

ANCHOR_OOF_PATH_34C = "model_results/oof_seg33a_arcsine_adaptive.csv"
ANCHOR_TEST_PATH_34C = "model_results/testpred_seg33a_arcsine_adaptive.csv"

ARTIFACT_PREFIX_34C = "spline34c_factor_smooth_gam"
OTHER_LEVEL_34C = "__OTHER_LEVEL_34C__"

print("=" * 90)
print("34C. Factor-specific smooth GAM calibration")
print("=" * 90)

# ------------------------------------------------------------
# 1. Checks and anchor loading
# ------------------------------------------------------------

if "y_train" not in globals():
    raise ValueError("Missing y_train.")

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("34C requires raw_train_te and raw_test_te for segment definitions.")

y_arr_34c = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_34c = len(y_arr_34c)
n_test_34c = raw_test_te.shape[0]

def load_oof_prediction_34c(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_priority = [
        "pred_clipped",
        "prediction",
        "pred",
        "oof_pred",
        "PREDICTED_PERCENT_PROFICIENT",
    ]

    pred_col = None
    for c in pred_priority:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}, column {pred_col}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_y_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_y_diff > 1e-5:
            raise ValueError(f"Anchor OOF target mismatch. Max difference: {max_y_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col


def load_test_prediction_34c(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}, column {pred_col}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids_34b" in globals():
        ids = np.asarray(test_ids_34b)
    elif "test_ids_34a" in globals():
        ids = np.asarray(test_ids_34a)
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no global test ids found.")

    if len(pred) != n_test_34c:
        raise ValueError(f"Test row mismatch for {path}: got {len(pred)}, expected {n_test_34c}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col


if "anchor_oof_34b" in globals() and len(anchor_oof_34b) == n_train_34c:
    anchor_oof_34c = np.asarray(anchor_oof_34b, dtype=np.float32)
    anchor_oof_col_34c = "anchor_oof_34b"
elif "anchor_oof_34a" in globals() and len(anchor_oof_34a) == n_train_34c:
    anchor_oof_34c = np.asarray(anchor_oof_34a, dtype=np.float32)
    anchor_oof_col_34c = "anchor_oof_34a"
else:
    anchor_oof_34c, anchor_oof_col_34c = load_oof_prediction_34c(
        ANCHOR_OOF_PATH_34C,
        y_arr_34c,
    )

if "anchor_test_34b" in globals() and len(anchor_test_34b) == n_test_34c:
    anchor_test_34c = np.asarray(anchor_test_34b, dtype=np.float32)
    test_ids_34c = np.asarray(test_ids_34b)
    anchor_test_col_34c = "anchor_test_34b"
elif "anchor_test_34a" in globals() and len(anchor_test_34a) == n_test_34c:
    anchor_test_34c = np.asarray(anchor_test_34a, dtype=np.float32)
    test_ids_34c = np.asarray(test_ids_34a)
    anchor_test_col_34c = "anchor_test_34a"
else:
    anchor_test_34c, test_ids_34c, anchor_test_col_34c = load_test_prediction_34c(
        ANCHOR_TEST_PATH_34C
    )

anchor_mse_34c = float(mean_squared_error(y_arr_34c, anchor_oof_34c))

print("\nAnchor check")
print("------------")
print("Anchor OOF source:", anchor_oof_col_34c)
print("Anchor test source:", anchor_test_col_34c)
print(f"Anchor OOF MSE: {anchor_mse_34c:.6f}")

# ------------------------------------------------------------
# 2. Segment helpers
# ------------------------------------------------------------

def clean_cat_34c(s):
    s = pd.Series(s).astype("object")
    s = s.where(pd.notna(s), "__NA__")
    return s.astype(str)

def make_group_values_34c(raw_df, group_spec):
    parts = [clean_cat_34c(raw_df[c]) for c in group_spec]
    out = parts[0].copy()

    for p in parts[1:]:
        out = out + "__" + p

    return out.astype(str).to_numpy()

def group_spec_name_34c(group_spec):
    return "x".join(group_spec)

def select_levels_34c(group_values_fit, max_levels=75, min_count=150):
    vc = pd.Series(group_values_fit).value_counts(dropna=False)
    keep = vc[vc >= int(min_count)].head(int(max_levels)).index.astype(str).tolist()

    keep = [x for x in keep if x != OTHER_LEVEL_34C]
    keep.append(OTHER_LEVEL_34C)

    return keep

def map_group_codes_34c(group_values, selected_levels):
    level_to_code = {v: i for i, v in enumerate(selected_levels)}
    other_code = level_to_code[OTHER_LEVEL_34C]

    codes = np.empty(len(group_values), dtype=np.int32)
    for i, v in enumerate(group_values):
        codes[i] = level_to_code.get(str(v), other_code)

    return codes

def build_factor_smooth_design_34c(anchor_pred, group_values, spline, selected_levels):
    anchor_pred = np.asarray(anchor_pred, dtype=np.float32).reshape(-1, 1)
    n = anchor_pred.shape[0]

    basis = spline.transform(anchor_pred).astype(np.float32)
    nbasis = basis.shape[1]

    codes = map_group_codes_34c(group_values, selected_levels)
    nlevels = len(selected_levels)

    # Global smooth(anchor)
    X_global = sparse.csr_matrix(basis)

    # Group intercepts
    X_group_intercept = sparse.csr_matrix(
        (
            np.ones(n, dtype=np.float32),
            (np.arange(n, dtype=np.int32), codes),
        ),
        shape=(n, nlevels),
    )

    # Group-specific smooth(anchor): selected group indicator multiplied by spline basis.
    row_idx = np.repeat(np.arange(n, dtype=np.int32), nbasis)
    basis_idx = np.tile(np.arange(nbasis, dtype=np.int32), n)
    col_idx = np.repeat(codes * nbasis, nbasis) + basis_idx
    data = basis.reshape(-1).astype(np.float32)

    X_factor_smooth = sparse.csr_matrix(
        (data, (row_idx, col_idx)),
        shape=(n, nlevels * nbasis),
    )

    X = sparse.hstack(
        [X_global, X_group_intercept, X_factor_smooth],
        format="csr",
    )

    return X

# ------------------------------------------------------------
# 3. Build valid group specs and configs
# ------------------------------------------------------------

candidate_group_specs_34c = [
    ("ASSESSMENT_NAME",),
    ("SUBGROUP_NAME",),
    ("DISTRICT_TYPE",),
    ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("DISTRICT_TYPE", "ASSESSMENT_NAME"),
    ("REGION", "ASSESSMENT_NAME"),
    ("COUNTY", "SUBGROUP_NAME"),
]

valid_group_specs_34c = []
for spec in candidate_group_specs_34c:
    if all(c in raw_train_te.columns for c in spec) and all(c in raw_test_te.columns for c in spec):
        valid_group_specs_34c.append(spec)

if len(valid_group_specs_34c) == 0:
    raise ValueError("No valid group specs found in raw_train_te/raw_test_te.")

configs_34c = []

for spec in valid_group_specs_34c:
    spec_name = group_spec_name_34c(spec)

    for n_knots in [4, 6]:
        for alpha in [100.0, 1000.0, 10000.0]:
            configs_34c.append({
                "name": (
                    f"{ARTIFACT_PREFIX_34C}_{spec_name}"
                    f"_levels75_knots{n_knots}_alpha{str(alpha).replace('.', 'p')}"
                ),
                "group_spec": spec,
                "group_name": spec_name,
                "max_levels": 75,
                "min_count": 150,
                "n_knots": n_knots,
                "degree": 3,
                "alpha": alpha,
            })

print("\n34C group specs")
print("---------------")
for spec in valid_group_specs_34c:
    full_group = make_group_values_34c(raw_train_te, spec)
    print(
        group_spec_name_34c(spec),
        "| unique train levels:",
        pd.Series(full_group).nunique()
    )

print("\n34C configs")
print("-----------")
print("Number of configs:", len(configs_34c))
for cfg in configs_34c[:30]:
    print(cfg["name"])
if len(configs_34c) > 30:
    print(f"... {len(configs_34c) - 30} more configs")

# ------------------------------------------------------------
# 4. OOF loop
# ------------------------------------------------------------

folds_34c = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_34c))
)

resid_oof_34c = {
    cfg["name"]: np.full(n_train_34c, np.nan, dtype=np.float32)
    for cfg in configs_34c
}

resid_test_sum_34c = {
    cfg["name"]: np.zeros(n_test_34c, dtype=np.float64)
    for cfg in configs_34c
}

fold_metric_rows_34c = []
level_summary_rows_34c = []

overall_t0_34c = time.time()

for fold_num, (tr_idx, va_idx) in enumerate(folds_34c, start=1):
    print("\n" + "=" * 90)
    print(f"34C OUTER FOLD {fold_num}/{N_SPLITS}")
    print("=" * 90)

    fold_t0 = time.time()

    y_fit = y_arr_34c[tr_idx]
    y_val = y_arr_34c[va_idx]

    anchor_fit = anchor_oof_34c[tr_idx]
    anchor_val = anchor_oof_34c[va_idx]

    resid_fit = y_fit - anchor_fit

    raw_fit_fold = raw_train_te.iloc[tr_idx]
    raw_val_fold = raw_train_te.iloc[va_idx]
    raw_test_fold = raw_test_te

    fold_anchor_mse = float(mean_squared_error(y_val, anchor_val))
    print(f"Fold anchor MSE: {fold_anchor_mse:.6f}")
    print(f"Fitting {len(configs_34c)} factor-specific smooth configs...")

    for cfg in configs_34c:
        name = cfg["name"]
        spec = cfg["group_spec"]

        try:
            t0 = time.time()

            group_fit = make_group_values_34c(raw_fit_fold, spec)
            group_val = make_group_values_34c(raw_val_fold, spec)
            group_test = make_group_values_34c(raw_test_fold, spec)

            selected_levels = select_levels_34c(
                group_fit,
                max_levels=cfg["max_levels"],
                min_count=cfg["min_count"],
            )

            spline = SplineTransformer(
                n_knots=int(cfg["n_knots"]),
                degree=int(cfg["degree"]),
                include_bias=False,
                extrapolation="constant",
            )
            spline.fit(anchor_fit.reshape(-1, 1))

            X_fit = build_factor_smooth_design_34c(
                anchor_fit,
                group_fit,
                spline,
                selected_levels,
            )
            X_val = build_factor_smooth_design_34c(
                anchor_val,
                group_val,
                spline,
                selected_levels,
            )
            X_test = build_factor_smooth_design_34c(
                anchor_test_34c,
                group_test,
                spline,
                selected_levels,
            )

            model = Ridge(
                alpha=float(cfg["alpha"]),
                solver="lsqr",
                fit_intercept=True,
            )

            model.fit(X_fit, resid_fit)

            pred_val_resid = np.asarray(model.predict(X_val)).reshape(-1)
            pred_test_resid = np.asarray(model.predict(X_test)).reshape(-1)

            resid_oof_34c[name][va_idx] = pred_val_resid.astype(np.float32)
            resid_test_sum_34c[name] += pred_test_resid.astype(np.float64)

            pred_val_lam1 = np.clip(anchor_val + pred_val_resid, 0, 100)
            mse_lam1 = float(mean_squared_error(y_val, pred_val_lam1))

            fold_metric_rows_34c.append({
                "fold": fold_num,
                "config": name,
                "group_name": cfg["group_name"],
                "group_spec": "|".join(spec),
                "n_selected_levels": len(selected_levels),
                "max_levels": cfg["max_levels"],
                "min_count": cfg["min_count"],
                "n_knots": cfg["n_knots"],
                "degree": cfg["degree"],
                "alpha": cfg["alpha"],
                "fold_anchor_mse": fold_anchor_mse,
                "fold_mse_lambda1": mse_lam1,
                "fold_gain_lambda1": fold_anchor_mse - mse_lam1,
                "elapsed_seconds": float(time.time() - t0),
                "status": "ok",
            })

            level_summary_rows_34c.append({
                "fold": fold_num,
                "config": name,
                "group_name": cfg["group_name"],
                "n_selected_levels": len(selected_levels),
                "first_15_levels": " | ".join(selected_levels[:15]),
            })

            del X_fit, X_val, X_test, model
            del pred_val_resid, pred_test_resid

        except Exception as e:
            print(f"ERROR | fold {fold_num} | {name}: {repr(e)}")

            fold_metric_rows_34c.append({
                "fold": fold_num,
                "config": name,
                "group_name": cfg["group_name"],
                "group_spec": "|".join(spec),
                "n_selected_levels": np.nan,
                "max_levels": cfg["max_levels"],
                "min_count": cfg["min_count"],
                "n_knots": cfg["n_knots"],
                "degree": cfg["degree"],
                "alpha": cfg["alpha"],
                "fold_anchor_mse": fold_anchor_mse,
                "fold_mse_lambda1": np.nan,
                "fold_gain_lambda1": np.nan,
                "elapsed_seconds": np.nan,
                "status": repr(e),
            })

    print(f"Fold {fold_num} elapsed: {time.time() - fold_t0:.1f}s")
    gc.collect()

# ------------------------------------------------------------
# 5. Global residual shrinkage scan
# ------------------------------------------------------------

lambda_grid_34c = np.unique(
    np.concatenate([
        np.linspace(-1.0, 1.5, 626),
        np.array([0.0, 0.25, 0.5, 0.75, 1.0]),
    ])
)

screen_rows_34c = []

for cfg in configs_34c:
    name = cfg["name"]
    resid_oof = resid_oof_34c[name]

    if not np.isfinite(resid_oof).all():
        continue

    best_row = None

    for lam in lambda_grid_34c:
        pred = np.clip(anchor_oof_34c + float(lam) * resid_oof, 0, 100)
        mse = float(mean_squared_error(y_arr_34c, pred))

        row = {
            "config": name,
            "group_name": cfg["group_name"],
            "group_spec": "|".join(cfg["group_spec"]),
            "max_levels": cfg["max_levels"],
            "min_count": cfg["min_count"],
            "n_knots": cfg["n_knots"],
            "degree": cfg["degree"],
            "alpha": cfg["alpha"],
            "residual_lambda": float(lam),
            "oof_mse": mse,
            "gain_vs_anchor": anchor_mse_34c - mse,
        }

        if best_row is None or row["oof_mse"] < best_row["oof_mse"]:
            best_row = row

    screen_rows_34c.append(best_row)

screen34c = pd.DataFrame(screen_rows_34c)

if len(screen34c) == 0:
    raise ValueError("No complete 34C candidates were produced.")

screen34c = screen34c.sort_values("oof_mse").reset_index(drop=True)

best34c = screen34c.iloc[0]
best_name34c = best34c["config"]
best_lambda34c = float(best34c["residual_lambda"])

best_resid_oof34c = resid_oof_34c[best_name34c]
best_resid_test34c = resid_test_sum_34c[best_name34c] / N_SPLITS

best_oof34c = np.clip(anchor_oof_34c + best_lambda34c * best_resid_oof34c, 0, 100).astype(np.float32)
best_test34c = np.clip(anchor_test_34c + best_lambda34c * best_resid_test34c, 0, 100).astype(np.float32)

best_mse34c = float(mean_squared_error(y_arr_34c, best_oof34c))
best_gain34c = anchor_mse_34c - best_mse34c

# ------------------------------------------------------------
# 6. Fold gains and saves
# ------------------------------------------------------------

fold_gains_rows_34c = []

for fold_num, (_, va_idx) in enumerate(folds_34c, start=1):
    anchor_fold_mse = float(mean_squared_error(y_arr_34c[va_idx], anchor_oof_34c[va_idx]))
    cand_fold_mse = float(mean_squared_error(y_arr_34c[va_idx], best_oof34c[va_idx]))

    fold_gains_rows_34c.append({
        "fold": fold_num,
        "anchor_mse": anchor_fold_mse,
        "candidate_mse": cand_fold_mse,
        "gain_vs_anchor": anchor_fold_mse - cand_fold_mse,
    })

fold_gains34c = pd.DataFrame(fold_gains_rows_34c)

screen_path34c = "model_results/spline34c_factor_smooth_gam_screen.csv"
fold_metrics_path34c = "model_results/spline34c_factor_smooth_gam_fold_metrics.csv"
fold_gains_path34c = "model_results/spline34c_factor_smooth_gam_best_fold_gains.csv"
level_summary_path34c = "model_results/spline34c_factor_smooth_gam_level_summary.csv"
oof_path34c = "model_results/oof_spline34c_factor_smooth_gam_best.csv"
testpred_path34c = "model_results/testpred_spline34c_factor_smooth_gam_best.csv"
submission_path34c = "submission_spline34c_factor_smooth_gam_best.csv"

screen34c.to_csv(screen_path34c, index=False)
pd.DataFrame(fold_metric_rows_34c).to_csv(fold_metrics_path34c, index=False)
fold_gains34c.to_csv(fold_gains_path34c, index=False)
pd.DataFrame(level_summary_rows_34c).to_csv(level_summary_path34c, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_34c),
    TARGET_COL: y_arr_34c,
    "pred_clipped": best_oof34c,
}).to_csv(oof_path34c, index=False)

pd.DataFrame({
    ID_COL: test_ids_34c,
    TARGET_COL: best_test34c,
}).to_csv(testpred_path34c, index=False)

pd.DataFrame({
    ID_COL: test_ids_34c,
    TARGET_COL: best_test34c,
}).to_csv(submission_path34c, index=False)

# ------------------------------------------------------------
# 7. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("34C factor-specific smooth GAM screen complete")
print("=" * 90)

print("\nAnchor")
print("------")
print(f"Anchor OOF MSE: {anchor_mse_34c:.6f}")

print("\nTop 20 factor-specific smooth GAM candidates")
print("--------------------------------------------")
display(screen34c.head(20))

print("\nBest 34C candidate")
print("------------------")
print(best34c.to_string())
print(f"\nBest 34C OOF MSE: {best_mse34c:.6f}")
print(f"Gain vs anchor:   {best_gain34c:.6f}")
print(f"Min fold gain:    {fold_gains34c['gain_vs_anchor'].min():.6f}")

print("\nBest candidate fold gains")
print("-------------------------")
print(fold_gains34c.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path34c)
print(fold_metrics_path34c)
print(fold_gains_path34c)
print(level_summary_path34c)
print(oof_path34c)
print(testpred_path34c)
print(submission_path34c)

print("\nRuntime seconds:", round(time.time() - overall_t0_34c, 1))

print("\nDecision rule")
print("-------------")
if best_gain34c >= 5.0:
    print("Material factor-specific spline/GAM signal found. Continue spline/GAM with larger level counts and segment-specific configs.")
elif best_gain34c >= 1.0:
    print("Some factor-specific spline/GAM signal found. One more expanded spline/GAM run may be justified.")
else:
    print("No material spline/GAM signal. Close the spline/GAM family and move to the next textbook family.")

34C. Factor-specific smooth GAM calibration

Anchor check
------------
Anchor OOF source: anchor_oof_34b
Anchor test source: anchor_test_34b
Anchor OOF MSE: 77.049866

34C group specs
---------------
ASSESSMENT_NAME | unique train levels: 32
SUBGROUP_NAME | unique train levels: 5
DISTRICT_TYPE | unique train levels: 7
ASSESSMENT_NAMExSUBGROUP_NAME | unique train levels: 132
DISTRICT_TYPExASSESSMENT_NAME | unique train levels: 222
REGIONxASSESSMENT_NAME | unique train levels: 315
COUNTYxSUBGROUP_NAME | unique train levels: 310

34C configs
-----------
Number of configs: 42
spline34c_factor_smooth_gam_ASSESSMENT_NAME_levels75_knots4_alpha100p0
spline34c_factor_smooth_gam_ASSESSMENT_NAME_levels75_knots4_alpha1000p0
spline34c_factor_smooth_gam_ASSESSMENT_NAME_levels75_knots4_alpha10000p0
spline34c_factor_smooth_gam_ASSESSMENT_NAME_levels75_knots6_alpha100p0
spline34c_factor_smooth_gam_ASSESSMENT_NAME_levels75_knots6_alpha1000p0
spline34c_factor_smooth_gam_ASSESSMENT_NAME_levels75_knots6_al

,config,group_name,group_spec,max_levels,min_count,n_knots,degree,alpha,residual_lambda,oof_mse,gain_vs_anchor
0,spline34c_factor_smooth_gam_ASSESSMENT_NAMExSU...,ASSESSMENT_NAMExSUBGROUP_NAME,ASSESSMENT_NAME|SUBGROUP_NAME,75,150,6,3,100.0,0.532,76.969872,0.079994
1,spline34c_factor_smooth_gam_ASSESSMENT_NAMExSU...,ASSESSMENT_NAMExSUBGROUP_NAME,ASSESSMENT_NAME|SUBGROUP_NAME,75,150,6,3,1000.0,0.792,76.972656,0.077209
2,spline34c_factor_smooth_gam_ASSESSMENT_NAMExSU...,ASSESSMENT_NAMExSUBGROUP_NAME,ASSESSMENT_NAME|SUBGROUP_NAME,75,150,4,3,100.0,0.540,76.974762,0.075104
3,spline34c_factor_smooth_gam_SUBGROUP_NAME_leve...,SUBGROUP_NAME,SUBGROUP_NAME,75,150,6,3,100.0,0.644,76.976921,0.072945
4,spline34c_factor_smooth_gam_SUBGROUP_NAME_leve...,SUBGROUP_NAME,SUBGROUP_NAME,75,150,6,3,1000.0,0.704,76.986061,0.063805
5,spline34c_factor_smooth_gam_ASSESSMENT_NAMExSU...,ASSESSMENT_NAMExSUBGROUP_NAME,ASSESSMENT_NAME|SUBGROUP_NAME,75,150,4,3,1000.0,0.748,76.987831,0.062035
6,spline34c_factor_smooth_gam_SUBGROUP_NAME_leve...,SUBGROUP_NAME,SUBGROUP_NAME,75,150,4,3,100.0,0.628,76.992737,0.057129
7,spline34c_factor_smooth_gam_ASSESSMENT_NAME_le...,ASSESSMENT_NAME,ASSESSMENT_NAME,75,150,6,3,1000.0,0.748,76.997948,0.051918
8,spline34c_factor_smooth_gam_ASSESSMENT_NAME_le...,ASSESSMENT_NAME,ASSESSMENT_NAME,75,150,4,3,100.0,0.560,77.003700,0.046165
9,spline34c_factor_smooth_gam_ASSESSMENT_NAME_le...,ASSESSMENT_NAME,ASSESSMENT_NAME,75,150,6,3,100.0,0.500,77.004654,0.045212



Best 34C candidate
------------------
config             spline34c_factor_smooth_gam_ASSESSMENT_NAMExSU...
group_name                             ASSESSMENT_NAMExSUBGROUP_NAME
group_spec                             ASSESSMENT_NAME|SUBGROUP_NAME
max_levels                                                        75
min_count                                                        150
n_knots                                                            6
degree                                                             3
alpha                                                          100.0
residual_lambda                                                0.532
oof_mse                                                    76.969872
gain_vs_anchor                                              0.079994

Best 34C OOF MSE: 76.969872
Gain vs anchor:   0.079994
Min fold gain:    0.032654

Best candidate fold gains
-------------------------
 fold  anchor_mse  candidate_mse  gain_vs_anchor
    1   78.427116

### 34C. Factor-Specific Smooth GAM Calibration

This section tested the final spline/GAM variant: factor-specific smooth calibration. The model again used the 33A anchor prediction and modeled residuals, but allowed the smooth calibration curve to vary by categorical segment. This is closer to a GAM with segment-specific smooth effects.

The 33A anchor OOF MSE was `77.049866`. The best 34C candidate used factor-specific smooths by `ASSESSMENT_NAME × SUBGROUP_NAME`, with 75 retained levels, minimum group count 150, 6 knots, cubic splines, Ridge alpha `100.0`, and residual shrinkage `0.532`.

The best 34C OOF MSE was `76.969872`, for a gain of only `0.079994` versus the 33A anchor. Fold gains were all positive but very small:

- Fold 1 gain: `0.153564`
- Fold 2 gain: `0.052811`
- Fold 3 gain: `0.061279`
- Fold 4 gain: `0.099625`
- Fold 5 gain: `0.032654`

Conclusion: the spline/GAM family produced only tiny, stable residual corrections. The 34A, 34B, and 34C results show that there is no material spline/GAM signal left to exploit relative to the current target-encoded boosting anchor. This family should be closed and not submitted. The next step is to move to the next textbook family: local regression / KNN-style regression.

In [90]:
# ============================================================
# 35A. Local Regression / KNN-style residual model
# ============================================================
#
# Textbook family:
#   14. Local Regression / LOESS-style local smoothing
#   16. k-Nearest Neighbors Regression
#
# Practical adaptation:
# - Full LOESS is not practical on ~145k rows.
# - This cell uses KNN residual smoothing as the scalable local-regression analogue.
# - It predicts residuals from the current 33A anchor:
#
#       residual = y - anchor_prediction
#
#   Then:
#
#       final_prediction = anchor_prediction + lambda * local_residual_prediction
#
# - KNN is run in low-dimensional standardized spaces and optionally within
#   assessment/subgroup segments.
# ============================================================

import os
import gc
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

ANCHOR_OOF_PATH_35A = "model_results/oof_seg33a_arcsine_adaptive.csv"
ANCHOR_TEST_PATH_35A = "model_results/testpred_seg33a_arcsine_adaptive.csv"

ARTIFACT_PREFIX_35A = "knn35a_local_residual"

print("=" * 90)
print("35A. Local Regression / KNN-style residual model")
print("=" * 90)

# ------------------------------------------------------------
# 1. Checks and anchor loading
# ------------------------------------------------------------

if "y_train" not in globals():
    raise ValueError("Missing y_train.")

if "X_train_proc_model" not in globals() or "X_test_proc_model" not in globals():
    raise ValueError("Missing X_train_proc_model or X_test_proc_model.")

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("35A requires raw_train_te and raw_test_te for segmented local neighborhoods.")

if not isinstance(X_train_proc_model, pd.DataFrame) or not isinstance(X_test_proc_model, pd.DataFrame):
    raise ValueError("35A expects X_train_proc_model and X_test_proc_model to be pandas DataFrames.")

y_arr_35a = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_35a = len(y_arr_35a)
n_test_35a = X_test_proc_model.shape[0]

def load_oof_prediction_35a(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_priority = [
        "pred_clipped",
        "prediction",
        "pred",
        "oof_pred",
        "PREDICTED_PERCENT_PROFICIENT",
    ]

    pred_col = None
    for c in pred_priority:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}, column {pred_col}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_y_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_y_diff > 1e-5:
            raise ValueError(f"Anchor OOF target mismatch. Max difference: {max_y_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col


def load_test_prediction_35a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}, column {pred_col}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids_34c" in globals():
        ids = np.asarray(test_ids_34c)
    elif "test_ids_34b" in globals():
        ids = np.asarray(test_ids_34b)
    elif "test_ids_34a" in globals():
        ids = np.asarray(test_ids_34a)
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no global test ids found.")

    if len(pred) != n_test_35a:
        raise ValueError(f"Test row mismatch for {path}: got {len(pred)}, expected {n_test_35a}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col


if "anchor_oof_34c" in globals() and len(anchor_oof_34c) == n_train_35a:
    anchor_oof_35a = np.asarray(anchor_oof_34c, dtype=np.float32)
    anchor_oof_source_35a = "anchor_oof_34c"
elif "anchor_oof_34a" in globals() and len(anchor_oof_34a) == n_train_35a:
    anchor_oof_35a = np.asarray(anchor_oof_34a, dtype=np.float32)
    anchor_oof_source_35a = "anchor_oof_34a"
else:
    anchor_oof_35a, anchor_oof_source_35a = load_oof_prediction_35a(
        ANCHOR_OOF_PATH_35A,
        y_arr_35a,
    )

if "anchor_test_34c" in globals() and len(anchor_test_34c) == n_test_35a:
    anchor_test_35a = np.asarray(anchor_test_34c, dtype=np.float32)
    test_ids_35a = np.asarray(test_ids_34c)
    anchor_test_source_35a = "anchor_test_34c"
elif "anchor_test_34a" in globals() and len(anchor_test_34a) == n_test_35a:
    anchor_test_35a = np.asarray(anchor_test_34a, dtype=np.float32)
    test_ids_35a = np.asarray(test_ids_34a)
    anchor_test_source_35a = "anchor_test_34a"
else:
    anchor_test_35a, test_ids_35a, anchor_test_source_35a = load_test_prediction_35a(
        ANCHOR_TEST_PATH_35A
    )

anchor_mse_35a = float(mean_squared_error(y_arr_35a, anchor_oof_35a))

print("\nAnchor check")
print("------------")
print("Anchor OOF source:", anchor_oof_source_35a)
print("Anchor test source:", anchor_test_source_35a)
print(f"Anchor OOF MSE: {anchor_mse_35a:.6f}")

# ------------------------------------------------------------
# 2. Feature frame for local neighborhoods
# ------------------------------------------------------------

def add_numeric_col_35a(train_df, test_df, source_train, source_test, col):
    train_vals = (
        pd.to_numeric(source_train[col], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=np.float32)
    )
    test_vals = (
        pd.to_numeric(source_test[col], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=np.float32)
    )

    train_df[col] = train_vals
    test_df[col] = test_vals

local_train_35a = pd.DataFrame({
    "anchor_pred": anchor_oof_35a.astype(np.float32)
})

local_test_35a = pd.DataFrame({
    "anchor_pred": anchor_test_35a.astype(np.float32)
})

candidate_numeric_cols_35a = [
    "N_STUDENTS",
    "PERCENT_FREE_LUNCH",
    "PERCENT_REDUCED_LUNCH",
    "PERCENT_ECONOMICALLY_DISADVANTAGED",
    "PERCENT_ENGLISH_LANGUAGE_LEARNERS",
    "PERCENT_ENGLISH_LANGUAGE_LEANERS",
    "PERCENT_WITH_DISABILITIES",
    "PERCENT_STUDENTS_WITH_DISABILITIES",
    "PERCENT_HOMELESS",
    "PERCENT_MIGRANT",
    "ATTENDANCE_RATE",
    "PERCENT_FEMALE",
    "PERCENT_MALE",
]

available_numeric_cols_35a = []

for c in candidate_numeric_cols_35a:
    if c in X_train_proc_model.columns and c in X_test_proc_model.columns:
        if pd.api.types.is_numeric_dtype(X_train_proc_model[c]):
            nunique = int(X_train_proc_model[c].nunique(dropna=True))
            if nunique > 8 and c not in available_numeric_cols_35a:
                add_numeric_col_35a(
                    local_train_35a,
                    local_test_35a,
                    X_train_proc_model,
                    X_test_proc_model,
                    c,
                )
                available_numeric_cols_35a.append(c)

feature_sets_35a = {
    "anchor": ["anchor_pred"],
}

if "N_STUDENTS" in available_numeric_cols_35a:
    feature_sets_35a["anchor_size"] = ["anchor_pred", "N_STUDENTS"]

demog_cols_35a = [
    c for c in [
        "N_STUDENTS",
        "PERCENT_FREE_LUNCH",
        "PERCENT_ECONOMICALLY_DISADVANTAGED",
        "PERCENT_ENGLISH_LANGUAGE_LEANERS",
        "PERCENT_ENGLISH_LANGUAGE_LEARNERS",
        "PERCENT_WITH_DISABILITIES",
        "PERCENT_STUDENTS_WITH_DISABILITIES",
        "PERCENT_HOMELESS",
    ]
    if c in available_numeric_cols_35a
]

if len(demog_cols_35a) >= 2:
    feature_sets_35a["anchor_demog"] = ["anchor_pred"] + demog_cols_35a

print("\nLocal feature sets")
print("------------------")
for k, v in feature_sets_35a.items():
    print(k, ":", v)

# ------------------------------------------------------------
# 3. Segment helpers and configs
# ------------------------------------------------------------

def clean_cat_35a(s):
    s = pd.Series(s).astype("object")
    s = s.where(pd.notna(s), "__NA__")
    return s.astype(str)

def make_group_values_35a(raw_df, group_spec):
    if group_spec is None:
        return np.array(["__GLOBAL__"] * len(raw_df), dtype=object)

    parts = [clean_cat_35a(raw_df[c]) for c in group_spec]
    out = parts[0].copy()

    for p in parts[1:]:
        out = out + "__" + p

    return out.astype(str).to_numpy()

def group_name_35a(group_spec):
    if group_spec is None:
        return "global"
    return "x".join(group_spec)

candidate_group_specs_35a = [
    None,
    ("ASSESSMENT_NAME",),
    ("SUBGROUP_NAME",),
    ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
]

valid_group_specs_35a = []
for spec in candidate_group_specs_35a:
    if spec is None:
        valid_group_specs_35a.append(spec)
    elif all(c in raw_train_te.columns for c in spec) and all(c in raw_test_te.columns for c in spec):
        valid_group_specs_35a.append(spec)

configs_35a = []

for feature_key in feature_sets_35a:
    for group_spec in valid_group_specs_35a:
        gname = group_name_35a(group_spec)

        # Smaller neighborhoods inside tighter segments; larger neighborhoods globally.
        if group_spec is None:
            k_values = [100, 300, 700]
            min_group_train = 1
        elif group_spec == ("ASSESSMENT_NAME", "SUBGROUP_NAME"):
            k_values = [25, 75, 150]
            min_group_train = 80
        else:
            k_values = [50, 150, 300]
            min_group_train = 120

        for k_neighbors in k_values:
            configs_35a.append({
                "name": (
                    f"{ARTIFACT_PREFIX_35A}_{feature_key}_{gname}"
                    f"_k{k_neighbors}_dist"
                ),
                "feature_key": feature_key,
                "group_spec": group_spec,
                "group_name": gname,
                "k_neighbors": int(k_neighbors),
                "weights": "distance",
                "min_group_train": int(min_group_train),
            })

print("\n35A configs")
print("-----------")
print("Number of configs:", len(configs_35a))
for cfg in configs_35a:
    print(cfg["name"])

# ------------------------------------------------------------
# 4. KNN prediction helper
# ------------------------------------------------------------

def knn_weighted_mean_35a(distances, neighbor_indices, residuals, weights):
    neigh_resid = residuals[neighbor_indices]

    if weights == "uniform":
        return neigh_resid.mean(axis=1)

    w = 1.0 / (distances + 1e-6)
    return (w * neigh_resid).sum(axis=1) / w.sum(axis=1)

def segmented_knn_predict_resid_35a(
    X_fit,
    resid_fit,
    group_fit,
    X_query,
    group_query,
    k_neighbors,
    weights,
    min_group_train,
):
    X_fit = np.asarray(X_fit, dtype=np.float32)
    X_query = np.asarray(X_query, dtype=np.float32)
    resid_fit = np.asarray(resid_fit, dtype=np.float32).reshape(-1)

    n_fit = X_fit.shape[0]
    n_query = X_query.shape[0]

    pred = np.zeros(n_query, dtype=np.float32)

    global_k = int(min(k_neighbors, n_fit))
    global_nn = NearestNeighbors(
        n_neighbors=global_k,
        algorithm="auto",
        metric="euclidean",
        n_jobs=-1,
    )
    global_nn.fit(X_fit)

    group_fit_series = pd.Series(group_fit).astype(str).reset_index(drop=True)
    group_query_series = pd.Series(group_query).astype(str).reset_index(drop=True)

    fit_group_indices = {
        g: idx.to_numpy(dtype=np.int64)
        for g, idx in group_fit_series.groupby(group_fit_series).groups.items()
    }

    query_group_indices = {
        g: idx.to_numpy(dtype=np.int64)
        for g, idx in group_query_series.groupby(group_query_series).groups.items()
    }

    for g, q_idx in query_group_indices.items():
        f_idx = fit_group_indices.get(g, None)

        if f_idx is None or len(f_idx) < min_group_train:
            nn = global_nn
            resid_source = resid_fit
            k_eff = global_k
            X_source = X_fit
        else:
            X_source = X_fit[f_idx]
            resid_source = resid_fit[f_idx]
            k_eff = int(min(k_neighbors, len(f_idx)))

            nn = NearestNeighbors(
                n_neighbors=k_eff,
                algorithm="auto",
                metric="euclidean",
                n_jobs=-1,
            )
            nn.fit(X_source)

        distances, neighbor_indices = nn.kneighbors(X_query[q_idx], n_neighbors=k_eff)

        pred[q_idx] = knn_weighted_mean_35a(
            distances=distances,
            neighbor_indices=neighbor_indices,
            residuals=resid_source,
            weights=weights,
        ).astype(np.float32)

    return pred

# ------------------------------------------------------------
# 5. OOF loop
# ------------------------------------------------------------

folds_35a = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_35a))
)

resid_oof_35a = {
    cfg["name"]: np.full(n_train_35a, np.nan, dtype=np.float32)
    for cfg in configs_35a
}

resid_test_sum_35a = {
    cfg["name"]: np.zeros(n_test_35a, dtype=np.float64)
    for cfg in configs_35a
}

fold_metric_rows_35a = []

overall_t0_35a = time.time()

for fold_num, (tr_idx, va_idx) in enumerate(folds_35a, start=1):
    print("\n" + "=" * 90)
    print(f"35A OUTER FOLD {fold_num}/{N_SPLITS}")
    print("=" * 90)

    fold_t0 = time.time()

    y_fit = y_arr_35a[tr_idx]
    y_val = y_arr_35a[va_idx]

    anchor_fit = anchor_oof_35a[tr_idx]
    anchor_val = anchor_oof_35a[va_idx]

    resid_fit = y_fit - anchor_fit

    fold_anchor_mse = float(mean_squared_error(y_val, anchor_val))
    print(f"Fold anchor MSE: {fold_anchor_mse:.6f}")

    fold_rows_start = len(fold_metric_rows_35a)

    for cfg in configs_35a:
        name = cfg["name"]
        feature_cols = feature_sets_35a[cfg["feature_key"]]

        try:
            t0 = time.time()

            X_fit_raw = local_train_35a.iloc[tr_idx][feature_cols].reset_index(drop=True)
            X_val_raw = local_train_35a.iloc[va_idx][feature_cols].reset_index(drop=True)
            X_test_raw = local_test_35a[feature_cols].reset_index(drop=True)

            imputer = SimpleImputer(strategy="median")
            scaler = StandardScaler()

            X_fit_imp = imputer.fit_transform(X_fit_raw)
            X_val_imp = imputer.transform(X_val_raw)
            X_test_imp = imputer.transform(X_test_raw)

            X_fit_scaled = scaler.fit_transform(X_fit_imp).astype(np.float32)
            X_val_scaled = scaler.transform(X_val_imp).astype(np.float32)
            X_test_scaled = scaler.transform(X_test_imp).astype(np.float32)

            group_fit = make_group_values_35a(
                raw_train_te.iloc[tr_idx].reset_index(drop=True),
                cfg["group_spec"],
            )
            group_val = make_group_values_35a(
                raw_train_te.iloc[va_idx].reset_index(drop=True),
                cfg["group_spec"],
            )
            group_test = make_group_values_35a(
                raw_test_te.reset_index(drop=True),
                cfg["group_spec"],
            )

            X_query = np.vstack([X_val_scaled, X_test_scaled])
            group_query = np.concatenate([group_val, group_test])

            pred_query_resid = segmented_knn_predict_resid_35a(
                X_fit=X_fit_scaled,
                resid_fit=resid_fit,
                group_fit=group_fit,
                X_query=X_query,
                group_query=group_query,
                k_neighbors=cfg["k_neighbors"],
                weights=cfg["weights"],
                min_group_train=cfg["min_group_train"],
            )

            pred_val_resid = pred_query_resid[:len(va_idx)]
            pred_test_resid = pred_query_resid[len(va_idx):]

            resid_oof_35a[name][va_idx] = pred_val_resid.astype(np.float32)
            resid_test_sum_35a[name] += pred_test_resid.astype(np.float64)

            pred_val_lam1 = np.clip(anchor_val + pred_val_resid, 0, 100)
            mse_lam1 = float(mean_squared_error(y_val, pred_val_lam1))

            fold_metric_rows_35a.append({
                "fold": fold_num,
                "config": name,
                "feature_key": cfg["feature_key"],
                "features": "|".join(feature_cols),
                "group_name": cfg["group_name"],
                "k_neighbors": cfg["k_neighbors"],
                "weights": cfg["weights"],
                "min_group_train": cfg["min_group_train"],
                "fold_anchor_mse": fold_anchor_mse,
                "fold_mse_lambda1": mse_lam1,
                "fold_gain_lambda1": fold_anchor_mse - mse_lam1,
                "elapsed_seconds": float(time.time() - t0),
                "status": "ok",
            })

            del X_fit_raw, X_val_raw, X_test_raw
            del X_fit_imp, X_val_imp, X_test_imp
            del X_fit_scaled, X_val_scaled, X_test_scaled
            del X_query, pred_query_resid, pred_val_resid, pred_test_resid

        except Exception as e:
            print(f"ERROR | fold {fold_num} | {name}: {repr(e)}")

            fold_metric_rows_35a.append({
                "fold": fold_num,
                "config": name,
                "feature_key": cfg["feature_key"],
                "features": "|".join(feature_sets_35a[cfg["feature_key"]]),
                "group_name": cfg["group_name"],
                "k_neighbors": cfg["k_neighbors"],
                "weights": cfg["weights"],
                "min_group_train": cfg["min_group_train"],
                "fold_anchor_mse": fold_anchor_mse,
                "fold_mse_lambda1": np.nan,
                "fold_gain_lambda1": np.nan,
                "elapsed_seconds": np.nan,
                "status": repr(e),
            })

    fold_summary = pd.DataFrame(fold_metric_rows_35a[fold_rows_start:])
    fold_summary = fold_summary.sort_values("fold_gain_lambda1", ascending=False)

    print("\nTop fold candidates by lambda=1 gain:")
    print(
        fold_summary[
            ["config", "fold_mse_lambda1", "fold_gain_lambda1", "elapsed_seconds"]
        ].head(8).to_string(index=False)
    )

    print(f"\nFold {fold_num} elapsed: {time.time() - fold_t0:.1f}s")
    gc.collect()

# ------------------------------------------------------------
# 6. Global shrinkage scan
# ------------------------------------------------------------

lambda_grid_35a = np.unique(
    np.concatenate([
        np.linspace(-1.0, 1.5, 626),
        np.array([0.0, 0.25, 0.5, 0.75, 1.0]),
    ])
)

screen_rows_35a = []

for cfg in configs_35a:
    name = cfg["name"]
    resid_oof = resid_oof_35a[name]

    if not np.isfinite(resid_oof).all():
        continue

    best_row = None

    for lam in lambda_grid_35a:
        pred = np.clip(anchor_oof_35a + float(lam) * resid_oof, 0, 100)
        mse = float(mean_squared_error(y_arr_35a, pred))

        row = {
            "config": name,
            "feature_key": cfg["feature_key"],
            "features": "|".join(feature_sets_35a[cfg["feature_key"]]),
            "group_name": cfg["group_name"],
            "k_neighbors": cfg["k_neighbors"],
            "weights": cfg["weights"],
            "min_group_train": cfg["min_group_train"],
            "residual_lambda": float(lam),
            "oof_mse": mse,
            "gain_vs_anchor": anchor_mse_35a - mse,
        }

        if best_row is None or row["oof_mse"] < best_row["oof_mse"]:
            best_row = row

    screen_rows_35a.append(best_row)

screen35a = pd.DataFrame(screen_rows_35a)

if len(screen35a) == 0:
    raise ValueError("No complete 35A candidates were produced.")

screen35a = screen35a.sort_values("oof_mse").reset_index(drop=True)

best35a = screen35a.iloc[0]
best_name35a = best35a["config"]
best_lambda35a = float(best35a["residual_lambda"])

best_resid_oof35a = resid_oof_35a[best_name35a]
best_resid_test35a = resid_test_sum_35a[best_name35a] / N_SPLITS

best_oof35a = np.clip(anchor_oof_35a + best_lambda35a * best_resid_oof35a, 0, 100).astype(np.float32)
best_test35a = np.clip(anchor_test_35a + best_lambda35a * best_resid_test35a, 0, 100).astype(np.float32)

best_mse35a = float(mean_squared_error(y_arr_35a, best_oof35a))
best_gain35a = anchor_mse_35a - best_mse35a

# ------------------------------------------------------------
# 7. Fold gains and saves
# ------------------------------------------------------------

fold_gains_rows_35a = []

for fold_num, (_, va_idx) in enumerate(folds_35a, start=1):
    anchor_fold_mse = float(mean_squared_error(y_arr_35a[va_idx], anchor_oof_35a[va_idx]))
    cand_fold_mse = float(mean_squared_error(y_arr_35a[va_idx], best_oof35a[va_idx]))

    fold_gains_rows_35a.append({
        "fold": fold_num,
        "anchor_mse": anchor_fold_mse,
        "candidate_mse": cand_fold_mse,
        "gain_vs_anchor": anchor_fold_mse - cand_fold_mse,
    })

fold_gains35a = pd.DataFrame(fold_gains_rows_35a)

screen_path35a = "model_results/knn35a_local_residual_screen.csv"
fold_metrics_path35a = "model_results/knn35a_local_residual_fold_metrics.csv"
fold_gains_path35a = "model_results/knn35a_local_residual_best_fold_gains.csv"
oof_path35a = "model_results/oof_knn35a_local_residual_best.csv"
testpred_path35a = "model_results/testpred_knn35a_local_residual_best.csv"
submission_path35a = "submission_knn35a_local_residual_best.csv"

screen35a.to_csv(screen_path35a, index=False)
pd.DataFrame(fold_metric_rows_35a).to_csv(fold_metrics_path35a, index=False)
fold_gains35a.to_csv(fold_gains_path35a, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_35a),
    TARGET_COL: y_arr_35a,
    "pred_clipped": best_oof35a,
}).to_csv(oof_path35a, index=False)

pd.DataFrame({
    ID_COL: test_ids_35a,
    TARGET_COL: best_test35a,
}).to_csv(testpred_path35a, index=False)

pd.DataFrame({
    ID_COL: test_ids_35a,
    TARGET_COL: best_test35a,
}).to_csv(submission_path35a, index=False)

# ------------------------------------------------------------
# 8. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("35A local regression / KNN residual screen complete")
print("=" * 90)

print("\nAnchor")
print("------")
print(f"Anchor OOF MSE: {anchor_mse_35a:.6f}")

print("\nTop 20 KNN/local residual candidates")
print("------------------------------------")
display(screen35a.head(20))

print("\nBest 35A candidate")
print("------------------")
print(best35a.to_string())
print(f"\nBest 35A OOF MSE: {best_mse35a:.6f}")
print(f"Gain vs anchor:   {best_gain35a:.6f}")
print(f"Min fold gain:    {fold_gains35a['gain_vs_anchor'].min():.6f}")

print("\nBest candidate fold gains")
print("-------------------------")
print(fold_gains35a.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path35a)
print(fold_metrics_path35a)
print(fold_gains_path35a)
print(oof_path35a)
print(testpred_path35a)
print(submission_path35a)

print("\nRuntime seconds:", round(time.time() - overall_t0_35a, 1))

print("\nDecision rule")
print("-------------")
if best_gain35a >= 5.0:
    print("Material local/KNN signal found. Continue this family with a broader local-regression screen.")
elif best_gain35a >= 1.0:
    print("Some local/KNN signal found. Consider one targeted expansion before moving on.")
else:
    print("No material local/KNN signal. Close this family and move to the next textbook family.")

35A. Local Regression / KNN-style residual model

Anchor check
------------
Anchor OOF source: anchor_oof_34c
Anchor test source: anchor_test_34c
Anchor OOF MSE: 77.049866

Local feature sets
------------------
anchor : ['anchor_pred']
anchor_size : ['anchor_pred', 'N_STUDENTS']
anchor_demog : ['anchor_pred', 'N_STUDENTS', 'PERCENT_FREE_LUNCH', 'PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_WITH_DISABILITIES', 'PERCENT_HOMELESS']

35A configs
-----------
Number of configs: 36
knn35a_local_residual_anchor_global_k100_dist
knn35a_local_residual_anchor_global_k300_dist
knn35a_local_residual_anchor_global_k700_dist
knn35a_local_residual_anchor_ASSESSMENT_NAME_k50_dist
knn35a_local_residual_anchor_ASSESSMENT_NAME_k150_dist
knn35a_local_residual_anchor_ASSESSMENT_NAME_k300_dist
knn35a_local_residual_anchor_SUBGROUP_NAME_k50_dist
knn35a_local_residual_anchor_SUBGROUP_NAME_k150_dist
knn35a_local_residual_anchor_SUBGROUP_NAME_k300_dist
knn35a_local_residual_a

,config,feature_key,features,group_name,k_neighbors,weights,min_group_train,residual_lambda,oof_mse,gain_vs_anchor
0,knn35a_local_residual_anchor_demog_ASSESSMENT_...,anchor_demog,anchor_pred|N_STUDENTS|PERCENT_FREE_LUNCH|PERC...,ASSESSMENT_NAME,50,distance,120,-0.552,76.453072,0.596794
1,knn35a_local_residual_anchor_demog_ASSESSMENT_...,anchor_demog,anchor_pred|N_STUDENTS|PERCENT_FREE_LUNCH|PERC...,ASSESSMENT_NAME,150,distance,120,-0.596,76.782829,0.267036
2,knn35a_local_residual_anchor_demog_global_k100...,anchor_demog,anchor_pred|N_STUDENTS|PERCENT_FREE_LUNCH|PERC...,global,100,distance,1,-0.416,76.798904,0.250961
3,knn35a_local_residual_anchor_demog_ASSESSMENT_...,anchor_demog,anchor_pred|N_STUDENTS|PERCENT_FREE_LUNCH|PERC...,ASSESSMENT_NAME,300,distance,120,-0.552,76.919090,0.130775
4,knn35a_local_residual_anchor_demog_global_k300...,anchor_demog,anchor_pred|N_STUDENTS|PERCENT_FREE_LUNCH|PERC...,global,300,distance,1,-0.432,76.942200,0.107666
5,knn35a_local_residual_anchor_demog_SUBGROUP_NA...,anchor_demog,anchor_pred|N_STUDENTS|PERCENT_FREE_LUNCH|PERC...,SUBGROUP_NAME,50,distance,120,-0.160,76.981377,0.068489
6,knn35a_local_residual_anchor_demog_global_k700...,anchor_demog,anchor_pred|N_STUDENTS|PERCENT_FREE_LUNCH|PERC...,global,700,distance,1,-0.336,77.014946,0.034920
7,knn35a_local_residual_anchor_demog_ASSESSMENT_...,anchor_demog,anchor_pred|N_STUDENTS|PERCENT_FREE_LUNCH|PERC...,ASSESSMENT_NAMExSUBGROUP_NAME,150,distance,80,0.140,77.031883,0.017982
8,knn35a_local_residual_anchor_size_ASSESSMENT_N...,anchor_size,anchor_pred|N_STUDENTS,ASSESSMENT_NAMExSUBGROUP_NAME,150,distance,80,0.076,77.035637,0.014229
9,knn35a_local_residual_anchor_demog_SUBGROUP_NA...,anchor_demog,anchor_pred|N_STUDENTS|PERCENT_FREE_LUNCH|PERC...,SUBGROUP_NAME,150,distance,120,-0.108,77.037285,0.012581



Best 35A candidate
------------------
config             knn35a_local_residual_anchor_demog_ASSESSMENT_...
feature_key                                             anchor_demog
features           anchor_pred|N_STUDENTS|PERCENT_FREE_LUNCH|PERC...
group_name                                           ASSESSMENT_NAME
k_neighbors                                                       50
weights                                                     distance
min_group_train                                                  120
residual_lambda                                               -0.552
oof_mse                                                    76.453072
gain_vs_anchor                                              0.596794

Best 35A OOF MSE: 76.453072
Gain vs anchor:   0.596794
Min fold gain:    0.492233

Best candidate fold gains
-------------------------
 fold  anchor_mse  candidate_mse  gain_vs_anchor
    1   78.427116      77.934883        0.492233
    2   74.554970      73.880463     

### 35A. Local Regression / KNN-Style Residual Model

This section tested the local regression / KNN model family. Since full LOESS is not computationally practical on this dataset size, the implementation used KNN residual smoothing as a scalable local-regression analogue.

The model used the current 33A anchor and modeled residuals:

`residual = observed target - 33A anchor prediction`

The best KNN/local residual model used the `anchor_demog` feature set, grouped neighborhoods by `ASSESSMENT_NAME`, used `k = 50` nearest neighbors, distance weighting, and minimum group size 120. The selected residual shrinkage value was negative, `-0.552`, meaning the raw KNN residual estimates were more useful as an inverse correction than as a direct residual estimate.

The 33A anchor OOF MSE was `77.049866`. The best 35A OOF MSE was `76.453072`, for a gain of `0.596794` versus the anchor. Fold gains were stable and positive:

- Fold 1 gain: `0.492233`
- Fold 2 gain: `0.674507`
- Fold 3 gain: `0.578018`
- Fold 4 gain: `0.688377`
- Fold 5 gain: `0.550758`

Although this is the largest post-33A residual improvement so far, it is still far below the threshold for a material modeling breakthrough. The negative shrinkage also suggests that the KNN residual smoother is not directly learning residuals in a clean way; instead, it is providing a small anti-local calibration effect.

Conclusion: the local regression / KNN family gives a stable but small improvement. It should be saved as an artifact, but not treated as a serious submission direction. The next step is to revisit the step-function family, since the earlier notebook showed meaningful threshold-based signal before the target-encoding breakthrough.

In [91]:
# ============================================================
# 36A. Step-function residual correction to current anchor
# ============================================================
#
# Textbook family:
#   5. Step Functions
#
# Why revisit this family:
# - Earlier notebook results showed meaningful signal from step functions
#   relative to the early linear-model pipeline.
# - 34A-34C showed that smooth spline/GAM residual corrections are tiny.
# - 35A showed local/KNN residual correction is stable but still not material.
#
# This cell tests whether piecewise-constant threshold effects remain in the
# residuals of the current 33A anchor.
#
# Model:
#   residual = y - anchor_prediction
#   residual ~ one-hot quantile bins of selected continuous / TE features
#   final_prediction = anchor_prediction + lambda * step_residual_prediction
# ============================================================

import os
import gc
import time
import warnings
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")
os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

ANCHOR_OOF_PATH_36A = "model_results/oof_seg33a_arcsine_adaptive.csv"
ANCHOR_TEST_PATH_36A = "model_results/testpred_seg33a_arcsine_adaptive.csv"

ARTIFACT_PREFIX_36A = "step36a_residual_bins"

print("=" * 90)
print("36A. Step-function residual correction")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required checks and anchor loading
# ------------------------------------------------------------

required_36a = ["y_train", "X_train_proc_model", "X_test_proc_model"]
missing_36a = [x for x in required_36a if x not in globals()]
if missing_36a:
    raise ValueError(f"Missing required objects: {missing_36a}")

if not isinstance(X_train_proc_model, pd.DataFrame) or not isinstance(X_test_proc_model, pd.DataFrame):
    raise ValueError("36A expects X_train_proc_model and X_test_proc_model to be pandas DataFrames.")

y_arr_36a = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_36a = len(y_arr_36a)
n_test_36a = X_test_proc_model.shape[0]

def load_oof_prediction_36a(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_priority = [
        "pred_clipped",
        "prediction",
        "pred",
        "oof_pred",
        "PREDICTED_PERCENT_PROFICIENT",
    ]

    pred_col = None
    for c in pred_priority:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}, column {pred_col}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_y_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_y_diff > 1e-5:
            raise ValueError(f"Anchor OOF target mismatch. Max difference: {max_y_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col


def load_test_prediction_36a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}, column {pred_col}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids_35a" in globals():
        ids = np.asarray(test_ids_35a)
    elif "test_ids_34c" in globals():
        ids = np.asarray(test_ids_34c)
    elif "test_ids_34a" in globals():
        ids = np.asarray(test_ids_34a)
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no global test ids found.")

    if len(pred) != n_test_36a:
        raise ValueError(f"Test row mismatch for {path}: got {len(pred)}, expected {n_test_36a}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col


# Important: use the 33A-style anchor, not the small-improvement 35A candidate.
if "anchor_oof_35a" in globals() and len(anchor_oof_35a) == n_train_36a:
    anchor_oof_36a = np.asarray(anchor_oof_35a, dtype=np.float32)
    anchor_oof_source_36a = "anchor_oof_35a"
elif "anchor_oof_34a" in globals() and len(anchor_oof_34a) == n_train_36a:
    anchor_oof_36a = np.asarray(anchor_oof_34a, dtype=np.float32)
    anchor_oof_source_36a = "anchor_oof_34a"
else:
    anchor_oof_36a, anchor_oof_source_36a = load_oof_prediction_36a(
        ANCHOR_OOF_PATH_36A,
        y_arr_36a,
    )

if "anchor_test_35a" in globals() and len(anchor_test_35a) == n_test_36a:
    anchor_test_36a = np.asarray(anchor_test_35a, dtype=np.float32)
    test_ids_36a = np.asarray(test_ids_35a)
    anchor_test_source_36a = "anchor_test_35a"
elif "anchor_test_34a" in globals() and len(anchor_test_34a) == n_test_36a:
    anchor_test_36a = np.asarray(anchor_test_34a, dtype=np.float32)
    test_ids_36a = np.asarray(test_ids_34a)
    anchor_test_source_36a = "anchor_test_34a"
else:
    anchor_test_36a, test_ids_36a, anchor_test_source_36a = load_test_prediction_36a(
        ANCHOR_TEST_PATH_36A
    )

anchor_mse_36a = float(mean_squared_error(y_arr_36a, anchor_oof_36a))

print("\nAnchor check")
print("------------")
print("Anchor OOF source:", anchor_oof_source_36a)
print("Anchor test source:", anchor_test_source_36a)
print(f"Anchor OOF MSE: {anchor_mse_36a:.6f}")

# ------------------------------------------------------------
# 2. Feature pool
# ------------------------------------------------------------

base_step_train_36a = pd.DataFrame({
    "anchor_pred": anchor_oof_36a.astype(np.float32)
})

base_step_test_36a = pd.DataFrame({
    "anchor_pred": anchor_test_36a.astype(np.float32)
})

candidate_numeric_cols_36a = [
    "N_STUDENTS",
    "PERCENT_FREE_LUNCH",
    "PERCENT_REDUCED_LUNCH",
    "PERCENT_ECONOMICALLY_DISADVANTAGED",
    "PERCENT_ENGLISH_LANGUAGE_LEARNERS",
    "PERCENT_ENGLISH_LANGUAGE_LEANERS",
    "PERCENT_WITH_DISABILITIES",
    "PERCENT_STUDENTS_WITH_DISABILITIES",
    "PERCENT_HOMELESS",
    "PERCENT_MIGRANT",
    "PERCENT_FEMALE",
    "PERCENT_MALE",
    "ATTENDANCE_RATE",
]

available_numeric_cols_36a = []

for c in candidate_numeric_cols_36a:
    if c in X_train_proc_model.columns and c in X_test_proc_model.columns:
        if pd.api.types.is_numeric_dtype(X_train_proc_model[c]):
            nunique = int(X_train_proc_model[c].nunique(dropna=True))
            if nunique > 8 and c not in available_numeric_cols_36a:
                base_step_train_36a[c] = (
                    pd.to_numeric(X_train_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )
                base_step_test_36a[c] = (
                    pd.to_numeric(X_test_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )
                available_numeric_cols_36a.append(c)

print("\nBase step-function feature pool")
print("--------------------------------")
print("Base numeric columns:", list(base_step_train_36a.columns))

# TE features are optional but preferred, using the same leakage-safe helper from the TE branch.
use_te_36a = (
    "raw_train_te" in globals()
    and "raw_test_te" in globals()
    and "te_key_specs" in globals()
    and (
        "build_te_oof_and_apply_many" in globals()
        or "build_te_oof_and_apply_many_29b" in globals()
    )
)

if use_te_36a and "build_te_oof_and_apply_many" not in globals():
    build_te_oof_and_apply_many = build_te_oof_and_apply_many_29b

print("Use leakage-safe TE features inside folds:", use_te_36a)

# ------------------------------------------------------------
# 3. Helpers
# ------------------------------------------------------------

def clean_numeric_array_36a(x, fill_value=None):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    finite = np.isfinite(x)

    if fill_value is None:
        fill_value = float(np.nanmedian(x[finite])) if finite.any() else 0.0

    x = np.where(finite, x, fill_value)
    return x.astype(np.float32), float(fill_value)


def rank_columns_by_abs_corr_36a(X_df, target):
    y = np.asarray(target, dtype=np.float64).reshape(-1)
    y = y - np.nanmean(y)
    y_den = np.sqrt(np.dot(y, y))

    rows = []

    for c in X_df.columns:
        x_raw = pd.to_numeric(X_df[c], errors="coerce").to_numpy(dtype=np.float64)
        x, _ = clean_numeric_array_36a(x_raw)
        x = x.astype(np.float64)
        x = x - np.mean(x)
        x_den = np.sqrt(np.dot(x, x))

        if x_den <= 1e-12 or y_den <= 1e-12:
            corr = 0.0
        else:
            corr = abs(float(np.dot(x, y) / (x_den * y_den)))

        rows.append((c, corr))

    ranked = pd.DataFrame(rows, columns=["feature", "abs_corr"])
    ranked = ranked.sort_values("abs_corr", ascending=False).reset_index(drop=True)
    return ranked


def select_step_features_36a(selection_mode, top_k, resid_rank, y_rank, forced_cols):
    forced_cols = list(dict.fromkeys([c for c in forced_cols if c is not None]))

    if selection_mode == "residual_rank":
        candidates = resid_rank["feature"].tolist()

    elif selection_mode == "target_rank":
        candidates = y_rank["feature"].tolist()

    elif selection_mode == "hybrid":
        half = max(1, int(np.ceil(top_k / 2)))
        candidates = (
            resid_rank["feature"].head(half).tolist()
            + y_rank["feature"].head(half).tolist()
            + resid_rank["feature"].tolist()
            + y_rank["feature"].tolist()
        )

    else:
        raise ValueError(f"Unknown selection_mode: {selection_mode}")

    selected = []
    for c in forced_cols + candidates:
        if c not in selected:
            selected.append(c)
        if len(selected) >= top_k:
            break

    return selected


def fit_step_bin_info_36a(X_fit_df, selected_cols, n_bins):
    bin_info = []

    for c in selected_cols:
        x_raw = pd.to_numeric(X_fit_df[c], errors="coerce").to_numpy(dtype=np.float64)
        x, fill_value = clean_numeric_array_36a(x_raw)

        unique_vals = np.unique(x)
        if len(unique_vals) <= 1:
            continue

        q = np.linspace(0, 1, int(n_bins) + 1)
        edges = np.quantile(x, q)
        thresholds = np.unique(edges[1:-1])

        # If quantiles collapse too much, fall back to unique midpoints.
        if len(thresholds) == 0:
            mids = (unique_vals[:-1] + unique_vals[1:]) / 2.0
            thresholds = mids[: min(len(mids), int(n_bins) - 1)]

        thresholds = np.asarray(thresholds, dtype=np.float32)
        n_bins_eff = int(len(thresholds) + 1)

        if n_bins_eff <= 1:
            continue

        bin_info.append({
            "feature": c,
            "fill_value": float(fill_value),
            "thresholds": thresholds,
            "n_bins_eff": n_bins_eff,
        })

    if len(bin_info) == 0:
        raise ValueError("No usable step-function bin features were created.")

    return bin_info


def transform_step_design_36a(X_df, bin_info):
    n = len(X_df)

    row_parts = []
    col_parts = []
    data_parts = []

    col_offset = 0

    for info in bin_info:
        c = info["feature"]
        thresholds = info["thresholds"]
        n_bins_eff = info["n_bins_eff"]
        fill_value = info["fill_value"]

        x_raw = pd.to_numeric(X_df[c], errors="coerce").to_numpy(dtype=np.float64)
        x, _ = clean_numeric_array_36a(x_raw, fill_value=fill_value)

        codes = np.digitize(x, thresholds, right=False).astype(np.int32)
        codes = np.clip(codes, 0, n_bins_eff - 1)

        rows = np.arange(n, dtype=np.int32)
        cols = col_offset + codes

        row_parts.append(rows)
        col_parts.append(cols)
        data_parts.append(np.ones(n, dtype=np.float32))

        col_offset += n_bins_eff

    row_idx = np.concatenate(row_parts)
    col_idx = np.concatenate(col_parts)
    data = np.concatenate(data_parts)

    X_step = sparse.csr_matrix(
        (data, (row_idx, col_idx)),
        shape=(n, col_offset),
    )

    return X_step


def fold_mse_table_36a(pred, label):
    rows = []

    for fold_num, (_, va_idx) in enumerate(folds_36a, start=1):
        anchor_fold_mse = float(mean_squared_error(y_arr_36a[va_idx], anchor_oof_36a[va_idx]))
        cand_fold_mse = float(mean_squared_error(y_arr_36a[va_idx], pred[va_idx]))

        rows.append({
            "candidate": label,
            "fold": fold_num,
            "anchor_mse": anchor_fold_mse,
            "candidate_mse": cand_fold_mse,
            "gain_vs_anchor": anchor_fold_mse - cand_fold_mse,
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 4. Configs
# ------------------------------------------------------------

configs_36a = []

for selection_mode in ["residual_rank", "target_rank", "hybrid"]:
    for top_k in [8, 16, 32]:
        for n_bins in [5, 10, 20]:
            for alpha in [100.0, 1000.0]:
                configs_36a.append({
                    "name": (
                        f"{ARTIFACT_PREFIX_36A}_{selection_mode}"
                        f"_top{top_k}_bins{n_bins}_alpha{str(alpha).replace('.', 'p')}"
                    ),
                    "selection_mode": selection_mode,
                    "top_k": int(top_k),
                    "n_bins": int(n_bins),
                    "alpha": float(alpha),
                })

print("\n36A configs")
print("-----------")
print("Number of configs:", len(configs_36a))
for cfg in configs_36a[:20]:
    print(cfg["name"])
if len(configs_36a) > 20:
    print(f"... {len(configs_36a) - 20} more configs")

# ------------------------------------------------------------
# 5. Main OOF loop
# ------------------------------------------------------------

folds_36a = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_36a))
)

resid_oof_36a = {
    cfg["name"]: np.full(n_train_36a, np.nan, dtype=np.float32)
    for cfg in configs_36a
}

resid_test_sum_36a = {
    cfg["name"]: np.zeros(n_test_36a, dtype=np.float64)
    for cfg in configs_36a
}

fold_metric_rows_36a = []
selected_feature_rows_36a = []

overall_t0_36a = time.time()

for fold_num, (tr_idx, va_idx) in enumerate(folds_36a, start=1):
    print("\n" + "=" * 90)
    print(f"36A OUTER FOLD {fold_num}/{N_SPLITS}")
    print("=" * 90)

    fold_t0 = time.time()

    y_fit = y_arr_36a[tr_idx]
    y_val = y_arr_36a[va_idx]

    anchor_fit = anchor_oof_36a[tr_idx]
    anchor_val = anchor_oof_36a[va_idx]

    resid_fit = y_fit - anchor_fit

    fold_anchor_mse = float(mean_squared_error(y_val, anchor_val))
    print(f"Fold anchor MSE: {fold_anchor_mse:.6f}")

    X_fit_base = base_step_train_36a.iloc[tr_idx].reset_index(drop=True)
    X_val_base = base_step_train_36a.iloc[va_idx].reset_index(drop=True)
    X_test_base = base_step_test_36a.reset_index(drop=True)

    if use_te_36a:
        print("Building leakage-safe TE features for step-function fold...")

        raw_fit_fold = raw_train_te.iloc[tr_idx].reset_index(drop=True)
        raw_val_fold = raw_train_te.iloc[va_idx].reset_index(drop=True)

        te_fit_fold, apply_dict_fold, _ = build_te_oof_and_apply_many(
            raw_fit=raw_fit_fold,
            y_fit=y_fit,
            raw_apply_dict={
                "valid": raw_val_fold,
                "test": raw_test_te.reset_index(drop=True),
            },
            key_specs=te_key_specs,
            n_splits=5,
            random_state=RANDOM_STATE + 3600 + fold_num,
        )

        te_fit_fold = te_fit_fold.reset_index(drop=True).add_prefix("TE__")
        te_val_fold = apply_dict_fold["valid"].reset_index(drop=True).add_prefix("TE__")
        te_test_fold = apply_dict_fold["test"].reset_index(drop=True).add_prefix("TE__")

        X_fit_all = pd.concat([X_fit_base, te_fit_fold], axis=1)
        X_val_all = pd.concat([X_val_base, te_val_fold], axis=1)
        X_test_all = pd.concat([X_test_base, te_test_fold], axis=1)

        del te_fit_fold, te_val_fold, te_test_fold

    else:
        X_fit_all = X_fit_base.copy()
        X_val_all = X_val_base.copy()
        X_test_all = X_test_base.copy()

    X_fit_all = X_fit_all.loc[:, ~X_fit_all.columns.duplicated()]
    X_val_all = X_val_all[X_fit_all.columns]
    X_test_all = X_test_all[X_fit_all.columns]

    print("Step-function candidate matrix:", X_fit_all.shape)

    resid_rank = rank_columns_by_abs_corr_36a(X_fit_all, resid_fit)
    y_rank = rank_columns_by_abs_corr_36a(X_fit_all, y_fit)

    print("\nTop residual-ranked features:")
    print(resid_rank.head(10).to_string(index=False))

    print("\nTop target-ranked features:")
    print(y_rank.head(10).to_string(index=False))

    fold_rows_start = len(fold_metric_rows_36a)

    for cfg in configs_36a:
        name = cfg["name"]

        try:
            t0 = time.time()

            selected_cols = select_step_features_36a(
                selection_mode=cfg["selection_mode"],
                top_k=cfg["top_k"],
                resid_rank=resid_rank,
                y_rank=y_rank,
                forced_cols=["anchor_pred"],
            )

            bin_info = fit_step_bin_info_36a(
                X_fit_df=X_fit_all,
                selected_cols=selected_cols,
                n_bins=cfg["n_bins"],
            )

            X_fit_step = transform_step_design_36a(X_fit_all, bin_info)
            X_val_step = transform_step_design_36a(X_val_all, bin_info)
            X_test_step = transform_step_design_36a(X_test_all, bin_info)

            model = Ridge(
                alpha=cfg["alpha"],
                solver="lsqr",
                fit_intercept=True,
            )

            model.fit(X_fit_step, resid_fit)

            pred_val_resid = np.asarray(model.predict(X_val_step)).reshape(-1)
            pred_test_resid = np.asarray(model.predict(X_test_step)).reshape(-1)

            resid_oof_36a[name][va_idx] = pred_val_resid.astype(np.float32)
            resid_test_sum_36a[name] += pred_test_resid.astype(np.float64)

            pred_val_lam1 = np.clip(anchor_val + pred_val_resid, 0, 100)
            mse_lam1 = float(mean_squared_error(y_val, pred_val_lam1))

            fold_metric_rows_36a.append({
                "fold": fold_num,
                "config": name,
                "selection_mode": cfg["selection_mode"],
                "top_k": cfg["top_k"],
                "n_bins": cfg["n_bins"],
                "alpha": cfg["alpha"],
                "n_selected_features": len(selected_cols),
                "n_step_columns": X_fit_step.shape[1],
                "fold_anchor_mse": fold_anchor_mse,
                "fold_mse_lambda1": mse_lam1,
                "fold_gain_lambda1": fold_anchor_mse - mse_lam1,
                "elapsed_seconds": float(time.time() - t0),
                "status": "ok",
            })

            for rank_pos, col in enumerate(selected_cols, start=1):
                selected_feature_rows_36a.append({
                    "fold": fold_num,
                    "config": name,
                    "selection_mode": cfg["selection_mode"],
                    "top_k": cfg["top_k"],
                    "n_bins": cfg["n_bins"],
                    "alpha": cfg["alpha"],
                    "rank": rank_pos,
                    "feature": col,
                })

            del X_fit_step, X_val_step, X_test_step, model
            del pred_val_resid, pred_test_resid

        except Exception as e:
            print(f"ERROR | fold {fold_num} | {name}: {repr(e)}")

            fold_metric_rows_36a.append({
                "fold": fold_num,
                "config": name,
                "selection_mode": cfg["selection_mode"],
                "top_k": cfg["top_k"],
                "n_bins": cfg["n_bins"],
                "alpha": cfg["alpha"],
                "n_selected_features": np.nan,
                "n_step_columns": np.nan,
                "fold_anchor_mse": fold_anchor_mse,
                "fold_mse_lambda1": np.nan,
                "fold_gain_lambda1": np.nan,
                "elapsed_seconds": np.nan,
                "status": repr(e),
            })

    fold_summary = pd.DataFrame(fold_metric_rows_36a[fold_rows_start:])
    fold_summary = fold_summary.sort_values("fold_gain_lambda1", ascending=False)

    print("\nTop fold candidates by lambda=1 gain:")
    print(
        fold_summary[
            [
                "config",
                "fold_mse_lambda1",
                "fold_gain_lambda1",
                "n_selected_features",
                "n_step_columns",
                "elapsed_seconds",
            ]
        ].head(10).to_string(index=False)
    )

    print(f"\nFold {fold_num} elapsed: {time.time() - fold_t0:.1f}s")

    del X_fit_base, X_val_base, X_test_base
    del X_fit_all, X_val_all, X_test_all
    gc.collect()

# ------------------------------------------------------------
# 6. Global residual shrinkage scan
# ------------------------------------------------------------

lambda_grid_36a = np.unique(
    np.concatenate([
        np.linspace(-1.0, 1.5, 626),
        np.array([0.0, 0.25, 0.5, 0.75, 1.0]),
    ])
)

screen_rows_36a = []

for cfg in configs_36a:
    name = cfg["name"]
    resid_oof = resid_oof_36a[name]

    if not np.isfinite(resid_oof).all():
        continue

    best_row = None

    for lam in lambda_grid_36a:
        pred = np.clip(anchor_oof_36a + float(lam) * resid_oof, 0, 100)
        mse = float(mean_squared_error(y_arr_36a, pred))

        row = {
            "config": name,
            "selection_mode": cfg["selection_mode"],
            "top_k": cfg["top_k"],
            "n_bins": cfg["n_bins"],
            "alpha": cfg["alpha"],
            "residual_lambda": float(lam),
            "oof_mse": mse,
            "gain_vs_anchor": anchor_mse_36a - mse,
        }

        if best_row is None or row["oof_mse"] < best_row["oof_mse"]:
            best_row = row

    screen_rows_36a.append(best_row)

screen36a = pd.DataFrame(screen_rows_36a)

if len(screen36a) == 0:
    raise ValueError("No complete 36A candidates were produced.")

screen36a = screen36a.sort_values("oof_mse").reset_index(drop=True)

best36a = screen36a.iloc[0]
best_name36a = best36a["config"]
best_lambda36a = float(best36a["residual_lambda"])

best_resid_oof36a = resid_oof_36a[best_name36a]
best_resid_test36a = resid_test_sum_36a[best_name36a] / N_SPLITS

best_oof36a = np.clip(anchor_oof_36a + best_lambda36a * best_resid_oof36a, 0, 100).astype(np.float32)
best_test36a = np.clip(anchor_test_36a + best_lambda36a * best_resid_test36a, 0, 100).astype(np.float32)

best_mse36a = float(mean_squared_error(y_arr_36a, best_oof36a))
best_gain36a = anchor_mse_36a - best_mse36a

fold_gains36a = fold_mse_table_36a(best_oof36a, "step36a_residual_bins_best")

# ------------------------------------------------------------
# 7. Save artifacts
# ------------------------------------------------------------

screen_path36a = "model_results/step36a_residual_bins_screen.csv"
fold_metrics_path36a = "model_results/step36a_residual_bins_fold_metrics.csv"
fold_gains_path36a = "model_results/step36a_residual_bins_best_fold_gains.csv"
selected_features_path36a = "model_results/step36a_residual_bins_selected_features.csv"
feature_counts_path36a = "model_results/step36a_residual_bins_selected_feature_counts.csv"
oof_path36a = "model_results/oof_step36a_residual_bins_best.csv"
testpred_path36a = "model_results/testpred_step36a_residual_bins_best.csv"
submission_path36a = "submission_step36a_residual_bins_best.csv"

screen36a.to_csv(screen_path36a, index=False)
pd.DataFrame(fold_metric_rows_36a).to_csv(fold_metrics_path36a, index=False)
fold_gains36a.to_csv(fold_gains_path36a, index=False)

selected_features36a = pd.DataFrame(selected_feature_rows_36a)
selected_features36a.to_csv(selected_features_path36a, index=False)

if len(selected_features36a) > 0:
    feature_counts36a = (
        selected_features36a
        .groupby(["selection_mode", "feature"])
        .size()
        .reset_index(name="times_selected")
        .sort_values(["times_selected", "selection_mode", "feature"], ascending=[False, True, True])
        .reset_index(drop=True)
    )
else:
    feature_counts36a = pd.DataFrame(columns=["selection_mode", "feature", "times_selected"])

feature_counts36a.to_csv(feature_counts_path36a, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_36a),
    TARGET_COL: y_arr_36a,
    "pred_clipped": best_oof36a,
}).to_csv(oof_path36a, index=False)

pd.DataFrame({
    ID_COL: test_ids_36a,
    TARGET_COL: best_test36a,
}).to_csv(testpred_path36a, index=False)

pd.DataFrame({
    ID_COL: test_ids_36a,
    TARGET_COL: best_test36a,
}).to_csv(submission_path36a, index=False)

# ------------------------------------------------------------
# 8. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("36A step-function residual screen complete")
print("=" * 90)

print("\nAnchor")
print("------")
print(f"Anchor OOF MSE: {anchor_mse_36a:.6f}")

print("\nTop 20 step-function residual candidates")
print("----------------------------------------")
display(screen36a.head(20))

print("\nBest 36A candidate")
print("------------------")
print(best36a.to_string())
print(f"\nBest 36A OOF MSE: {best_mse36a:.6f}")
print(f"Gain vs anchor:   {best_gain36a:.6f}")
print(f"Min fold gain:    {fold_gains36a['gain_vs_anchor'].min():.6f}")

print("\nBest candidate fold gains")
print("-------------------------")
print(fold_gains36a.to_string(index=False))

print("\nMost frequently selected step-function features")
print("-----------------------------------------------")
display(feature_counts36a.head(30))

print("\nSaved files")
print("-----------")
print(screen_path36a)
print(fold_metrics_path36a)
print(fold_gains_path36a)
print(selected_features_path36a)
print(feature_counts_path36a)
print(oof_path36a)
print(testpred_path36a)
print(submission_path36a)

print("\nRuntime seconds:", round(time.time() - overall_t0_36a, 1))

print("\nDecision rule")
print("-------------")
if best_gain36a >= 5.0:
    print("Material step-function signal found. Continue this family with segmented step functions and interactions.")
elif best_gain36a >= 1.0:
    print("Some step-function signal found. Try one targeted expansion before moving on.")
else:
    print("No material step-function signal. Close this family and move to the next textbook family.")

36A. Step-function residual correction

Anchor check
------------
Anchor OOF source: anchor_oof_35a
Anchor test source: anchor_test_35a
Anchor OOF MSE: 77.049866

Base step-function feature pool
--------------------------------
Base numeric columns: ['anchor_pred', 'N_STUDENTS', 'PERCENT_FREE_LUNCH', 'PERCENT_REDUCED_LUNCH', 'PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_WITH_DISABILITIES', 'PERCENT_HOMELESS', 'PERCENT_MIGRANT', 'PERCENT_FEMALE', 'PERCENT_MALE', 'ATTENDANCE_RATE']
Use leakage-safe TE features inside folds: True

36A configs
-----------
Number of configs: 54
step36a_residual_bins_residual_rank_top8_bins5_alpha100p0
step36a_residual_bins_residual_rank_top8_bins5_alpha1000p0
step36a_residual_bins_residual_rank_top8_bins10_alpha100p0
step36a_residual_bins_residual_rank_top8_bins10_alpha1000p0
step36a_residual_bins_residual_rank_top8_bins20_alpha100p0
step36a_residual_bins_residual_rank_top8_bins20_alpha1000p0
step36a_residual_bins_residu

,config,selection_mode,top_k,n_bins,alpha,residual_lambda,oof_mse,gain_vs_anchor
0,step36a_residual_bins_residual_rank_top16_bins...,residual_rank,16,5,1000.0,0.420,76.954796,0.095070
1,step36a_residual_bins_target_rank_top32_bins5_...,target_rank,32,5,1000.0,0.400,76.955544,0.094322
2,step36a_residual_bins_hybrid_top32_bins5_alpha...,hybrid,32,5,1000.0,0.388,76.956329,0.093536
3,step36a_residual_bins_hybrid_top16_bins5_alpha...,hybrid,16,5,1000.0,0.408,76.959015,0.090851
4,step36a_residual_bins_residual_rank_top32_bins...,residual_rank,32,5,1000.0,0.380,76.961441,0.088425
5,step36a_residual_bins_residual_rank_top16_bins...,residual_rank,16,5,100.0,0.384,76.962112,0.087753
6,step36a_residual_bins_target_rank_top32_bins5_...,target_rank,32,5,100.0,0.356,76.963516,0.086349
7,step36a_residual_bins_hybrid_top32_bins5_alpha...,hybrid,32,5,100.0,0.340,76.967010,0.082855
8,step36a_residual_bins_hybrid_top16_bins5_alpha...,hybrid,16,5,100.0,0.364,76.968826,0.081039
9,step36a_residual_bins_residual_rank_top32_bins...,residual_rank,32,10,1000.0,0.340,76.969810,0.080055



Best 36A candidate
------------------
config             step36a_residual_bins_residual_rank_top16_bins...
selection_mode                                         residual_rank
top_k                                                             16
n_bins                                                             5
alpha                                                         1000.0
residual_lambda                                                 0.42
oof_mse                                                    76.954796
gain_vs_anchor                                               0.09507

Best 36A OOF MSE: 76.954796
Gain vs anchor:   0.095070
Min fold gain:    0.029579

Best candidate fold gains
-------------------------
                 candidate  fold  anchor_mse  candidate_mse  gain_vs_anchor
step36a_residual_bins_best     1   78.427116      78.315094        0.112022
step36a_residual_bins_best     2   74.554970      74.525391        0.029579
step36a_residual_bins_best     3   78.135284 

,selection_mode,feature,times_selected
0,hybrid,TE__te_SCHOOL_ASSESSMENT_NAME_a220_n1000_mean,90
1,hybrid,TE__te_SCHOOL_ASSESSMENT_NAME_a220_n1000_wmean,90
2,hybrid,TE__te_SCHOOL_a100_n600_mean,90
3,hybrid,TE__te_SCHOOL_a100_n600_wmean,90
4,hybrid,TE__te_SUBGROUP_NAME_a20_n200_mean,90
5,hybrid,TE__te_SUBGROUP_NAME_a20_n200_wmean,90
6,hybrid,anchor_pred,90
7,residual_rank,TE__te_SCHOOL_ASSESSMENT_NAME_a220_n1000_mean,90
8,residual_rank,TE__te_SCHOOL_ASSESSMENT_NAME_a220_n1000_wmean,90
9,residual_rank,TE__te_SUBGROUP_NAME_a20_n200_mean,90



Saved files
-----------
model_results/step36a_residual_bins_screen.csv
model_results/step36a_residual_bins_fold_metrics.csv
model_results/step36a_residual_bins_best_fold_gains.csv
model_results/step36a_residual_bins_selected_features.csv
model_results/step36a_residual_bins_selected_feature_counts.csv
model_results/oof_step36a_residual_bins_best.csv
model_results/testpred_step36a_residual_bins_best.csv
submission_step36a_residual_bins_best.csv

Runtime seconds: 240.3

Decision rule
-------------
No material step-function signal. Close this family and move to the next textbook family.


### 36A. Step-Function Residual Correction

This section revisited the step-function model family using the current 33A anchor rather than the older linear-model pipeline. The goal was to test whether piecewise-constant threshold effects remained in the residuals after the target-encoded boosting model.

The model used:

`residual = observed target - 33A anchor prediction`

and fit Ridge-regularized step-function features based on quantile bins of selected continuous and target-encoded predictors. Candidate features were selected using residual correlation, target correlation, and hybrid rankings.

The 33A anchor OOF MSE was `77.049866`. The best 36A candidate used residual-ranked features, top 16 predictors, 5 bins, Ridge alpha `1000.0`, and residual shrinkage `0.420`.

The best 36A OOF MSE was `76.954796`, for a gain of only `0.095070` versus the anchor. Fold gains were positive but very small:

- Fold 1 gain: `0.112022`
- Fold 2 gain: `0.029579`
- Fold 3 gain: `0.137787`
- Fold 4 gain: `0.083908`
- Fold 5 gain: `0.112022`

Conclusion: although earlier step-function models showed meaningful improvement in the weaker linear-model stage, step functions do not provide material residual signal after the current target-encoded boosting anchor. The family should be closed and saved only as a project artifact. The next model family is neural networks.

In [93]:
# ============================================================
# 37A/38A prerequisite: install + verify PyTorch
# ============================================================

import sys
import subprocess
import importlib.util
import platform

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

if importlib.util.find_spec("torch") is None:
    print("\nPyTorch not found. Installing torch, torchvision, torchaudio...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "torch", "torchvision", "torchaudio"
    ])
else:
    print("\nPyTorch already installed.")

import torch

print("\nPyTorch verification")
print("--------------------")
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if hasattr(torch.backends, "mps"):
    print("MPS available:", torch.backends.mps.is_available())
else:
    print("MPS available: False")

device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else torch.device("cpu")
)

print("Selected device:", device)
print("Test tensor:", torch.rand(2, 3, device=device))

Python executable: /Users/saadmanchowdhury/Desktop/All Github projects/.venv/bin/python
Python version: 3.14.2 (main, Dec  5 2025, 16:49:16) [Clang 17.0.0 (clang-1700.4.4.1)]
Platform: macOS-15.6-arm64-arm-64bit-Mach-O

PyTorch not found. Installing torch, torchvision, torchaudio...
  Using cached torchaudio-2.11.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (6.9 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 67.9 MB/s  0:00:01 eta 0:00:01
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 44.6 MB/s  0:00:00
Using cached torchaudio-2.11.0-cp314-cp314-macosx_12_0_arm64.whl (679 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip



PyTorch verification
--------------------
torch version: 2.11.0
CUDA available: False
MPS available: True
Selected device: mps
Test tensor: tensor([[0.4002, 0.7695, 0.0103],
        [0.9067, 0.4123, 0.9119]], device='mps:0')


### 37A/38A. Neural Network Attempt with MPS

The first neural-network attempt used Apple MPS acceleration. The kernel crashed during the first outer fold after building leakage-safe target-encoding features and before completing any neural-network candidate. Because this was a kernel-level failure rather than a normal Python exception, the MPS neural-network run is not treated as a valid model-family result.

To continue the neural-network family safely, the next run forces PyTorch to CPU, removes MPS-sensitive training components, uses smaller batches, and writes checkpointed predictions after each fold/config. This preserves the textbook neural-network progression while avoiding another MPS kernel crash.

In [22]:
# ============================================================
# SAFE RECOVERY CELL after kernel crash: restore TE setup/helpers only
# ============================================================
# Run this after cells 0, 1, 4, 5, 6, 7, 8, 10, 12.
# This does NOT train LightGBM, XGBoost, CatBoost, splines, KNN, or NN.
# It only restores:
#   raw_train_te
#   raw_test_te
#   te_key_specs
#   build_te_oof_and_apply_many
#   build_te_oof_and_apply
#   take_rows
#   to_float32_matrix
#   append_features
# ============================================================

import os
import re
import gc
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.model_selection import KFold

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

required_recovery = [
    "train_full",
    "test_full",
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
]

missing_recovery = [x for x in required_recovery if x not in globals()]
if missing_recovery:
    raise ValueError(
        f"Missing required setup objects: {missing_recovery}. "
        "Rerun only cells 0, 1, 4, 5, 6, 7, 8, 10, 12 first."
    )

raw_train_te = train_full.reset_index(drop=True).copy()
raw_test_te = test_full.reset_index(drop=True).copy()
y_arr_te = np.asarray(y_train, dtype=np.float32).reshape(-1)

if len(raw_train_te) != len(y_arr_te):
    raise ValueError("train_full and y_train are not aligned.")
if X_train_proc_model.shape[0] != len(y_arr_te):
    raise ValueError("X_train_proc_model and y_train are not aligned.")
if X_test_proc_model.shape[0] != len(raw_test_te):
    raise ValueError("X_test_proc_model and test_full are not aligned.")

def add_n_students_bin_recovery(df):
    out = df.copy()
    out["_N_STUDENTS_BIN_TE"] = pd.cut(
        pd.Series(out["N_STUDENTS"]).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"],
    ).astype("string").fillna("<NA>")
    return out

raw_train_te = add_n_students_bin_recovery(raw_train_te)
raw_test_te = add_n_students_bin_recovery(raw_test_te)

candidate_key_specs = [
    (("ASSESSMENT_NAME",), 20.0, 200.0),
    (("SUBGROUP_NAME",), 20.0, 200.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME"), 20.0, 200.0),
    (("ASSESSMENT_NAME", "_N_STUDENTS_BIN_TE"), 30.0, 250.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME", "_N_STUDENTS_BIN_TE"), 50.0, 300.0),

    (("COUNTY",), 30.0, 250.0),
    (("COUNTY", "SUBGROUP_NAME"), 40.0, 300.0),
    (("COUNTY", "ASSESSMENT_NAME"), 60.0, 400.0),
    (("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"), 100.0, 600.0),

    (("REGION", "ASSESSMENT_NAME"), 40.0, 300.0),
    (("DISTRICT_TYPE", "ASSESSMENT_NAME"), 40.0, 300.0),

    (("DISTRICT",), 60.0, 400.0),
    (("DISTRICT", "SUBGROUP_NAME"), 90.0, 500.0),
    (("DISTRICT", "ASSESSMENT_NAME"), 140.0, 700.0),

    (("SCHOOL",), 100.0, 600.0),
    (("SCHOOL", "SUBGROUP_NAME"), 140.0, 800.0),
    (("SCHOOL", "ASSESSMENT_NAME"), 220.0, 1000.0),
]

te_key_specs = []
for cols, alpha, alpha_n in candidate_key_specs:
    if all(c in raw_train_te.columns for c in cols) and all(c in raw_test_te.columns for c in cols):
        te_key_specs.append((cols, alpha, alpha_n))

if len(te_key_specs) == 0:
    raise ValueError("No usable target-encoding key specs found.")

def safe_key_name(cols):
    name = "__".join(cols)
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def make_group_key(df, cols):
    cols = tuple(cols)

    if len(cols) == 1:
        return (
            df[cols[0]]
            .astype("string")
            .fillna("<NA>")
            .astype(str)
            .reset_index(drop=True)
        )

    return (
        df.loc[:, list(cols)]
        .astype("string")
        .fillna("<NA>")
        .astype(str)
        .agg(" || ".join, axis=1)
        .reset_index(drop=True)
    )

def fit_group_stats(keys, y, n_students, alpha, alpha_n):
    yy = np.asarray(y, dtype=np.float64)
    nn = np.asarray(n_students, dtype=np.float64)

    good_n = np.isfinite(nn) & (nn > 0)
    if not good_n.all():
        fill_n = np.nanmedian(nn[good_n]) if good_n.any() else 1.0
        nn = np.where(good_n, nn, fill_n)

    yy_clip = np.clip(yy, 0.0, 100.0)
    proficient_counts = np.rint((yy_clip / 100.0) * nn)
    proficient_counts = np.clip(proficient_counts, 0.0, nn)

    global_mean = float(np.mean(yy))
    global_std = float(np.std(yy, ddof=0))
    global_wmean = float(100.0 * proficient_counts.sum() / max(nn.sum(), 1.0))

    tmp = pd.DataFrame({
        "key": pd.Series(keys).astype(str).to_numpy(),
        "y": yy,
        "n": nn,
        "k": proficient_counts,
    })

    stats = tmp.groupby("key", sort=False).agg(
        cnt=("y", "size"),
        sum_y=("y", "sum"),
        std_y=("y", "std"),
        sum_n=("n", "sum"),
        sum_k=("k", "sum"),
    )

    stats["mean_s"] = (stats["sum_y"] + alpha * global_mean) / (stats["cnt"] + alpha)

    stats["wmean_s"] = 100.0 * (
        stats["sum_k"] + alpha_n * (global_wmean / 100.0)
    ) / (stats["sum_n"] + alpha_n)

    stats["std_y"] = stats["std_y"].fillna(global_std)
    stats["log_count"] = np.log1p(stats["cnt"].astype(float))
    stats["log_students"] = np.log1p(stats["sum_n"].astype(float))

    defaults = {
        "global_mean": global_mean,
        "global_wmean": global_wmean,
        "global_std": global_std,
    }

    return stats, defaults

def apply_group_stats(keys_apply, stats, defaults, prefix):
    kk = pd.Series(keys_apply).astype(str).reset_index(drop=True)

    out = pd.DataFrame(index=np.arange(len(kk)))
    out[f"{prefix}_mean"] = kk.map(stats["mean_s"]).fillna(defaults["global_mean"]).astype(np.float32)
    out[f"{prefix}_wmean"] = kk.map(stats["wmean_s"]).fillna(defaults["global_wmean"]).astype(np.float32)
    out[f"{prefix}_log_count"] = kk.map(stats["log_count"]).fillna(0.0).astype(np.float32)
    out[f"{prefix}_std"] = kk.map(stats["std_y"]).fillna(defaults["global_std"]).astype(np.float32)

    return out

def build_te_oof_and_apply_many(
    raw_fit,
    y_fit,
    raw_apply_dict,
    key_specs,
    n_splits=5,
    random_state=9890,
):
    raw_fit = raw_fit.reset_index(drop=True)
    raw_apply_dict = {
        name: df.reset_index(drop=True)
        for name, df in raw_apply_dict.items()
    }

    y_fit = np.asarray(y_fit, dtype=np.float32).reshape(-1)
    n_fit = pd.Series(raw_fit["N_STUDENTS"]).astype(float).to_numpy()
    n_rows_fit = len(raw_fit)

    kf_inner = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_parts = []
    apply_parts = {name: [] for name in raw_apply_dict.keys()}
    summary_rows = []

    for cols, alpha, alpha_n in key_specs:
        cols = tuple(cols)
        prefix = f"te_{safe_key_name(cols)}_a{int(alpha)}_n{int(alpha_n)}"

        key_fit_all = make_group_key(raw_fit, cols)
        key_apply_all = {
            name: make_group_key(df, cols)
            for name, df in raw_apply_dict.items()
        }

        feature_cols = [
            f"{prefix}_mean",
            f"{prefix}_wmean",
            f"{prefix}_log_count",
            f"{prefix}_std",
        ]

        oof_arr = np.zeros((n_rows_fit, len(feature_cols)), dtype=np.float32)

        for inner_tr_idx, inner_va_idx in kf_inner.split(np.arange(n_rows_fit)):
            stats_fold, defaults_fold = fit_group_stats(
                key_fit_all.iloc[inner_tr_idx],
                y_fit[inner_tr_idx],
                n_fit[inner_tr_idx],
                alpha=alpha,
                alpha_n=alpha_n,
            )

            enc_va = apply_group_stats(
                key_fit_all.iloc[inner_va_idx],
                stats_fold,
                defaults_fold,
                prefix,
            )

            oof_arr[inner_va_idx, :] = enc_va[feature_cols].to_numpy(dtype=np.float32)

        stats_full, defaults_full = fit_group_stats(
            key_fit_all,
            y_fit,
            n_fit,
            alpha=alpha,
            alpha_n=alpha_n,
        )

        oof_parts.append(pd.DataFrame(oof_arr, columns=feature_cols))

        for name, keys_apply in key_apply_all.items():
            apply_df = apply_group_stats(
                keys_apply,
                stats_full,
                defaults_full,
                prefix,
            )
            apply_parts[name].append(apply_df[feature_cols])

        fit_counts = key_fit_all.value_counts(dropna=False)

        summary_row = {
            "key": " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
            "alpha": alpha,
            "alpha_n": alpha_n,
            "fit_groups": int(fit_counts.shape[0]),
            "median_fit_count": float(fit_counts.median()),
            "singleton_share": float((fit_counts == 1).mean()),
            "features_added": len(feature_cols),
        }

        for name, keys_apply in key_apply_all.items():
            apply_seen = keys_apply.isin(set(fit_counts.index))
            summary_row[f"{name}_apply_groups"] = int(keys_apply.nunique(dropna=False))
            summary_row[f"{name}_apply_row_coverage"] = float(apply_seen.mean())

        summary_rows.append(summary_row)

    oof_features = pd.concat(oof_parts, axis=1)
    apply_features = {
        name: pd.concat(parts, axis=1)
        for name, parts in apply_parts.items()
    }
    summary = pd.DataFrame(summary_rows)

    return oof_features, apply_features, summary

def build_te_oof_and_apply(raw_fit, y_fit, raw_apply, key_specs, n_splits=5, random_state=9890):
    oof_features, apply_dict, summary = build_te_oof_and_apply_many(
        raw_fit=raw_fit,
        y_fit=y_fit,
        raw_apply_dict={"apply": raw_apply},
        key_specs=key_specs,
        n_splits=n_splits,
        random_state=random_state,
    )
    return oof_features, apply_dict["apply"], summary

def take_rows(X, idx):
    if sparse.issparse(X):
        return X[idx]
    if isinstance(X, pd.DataFrame):
        return X.iloc[idx]
    return X[idx]

def to_float32_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32).tocsr()
    if isinstance(X, pd.DataFrame):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def append_features(X_base, add_df):
    add = add_df.to_numpy(dtype=np.float32)
    if sparse.issparse(X_base):
        return sparse.hstack([X_base, sparse.csr_matrix(add)], format="csr")
    return np.hstack([X_base, add])

print("SAFE RECOVERY PASS")
print("------------------")
print("raw_train_te:", raw_train_te.shape)
print("raw_test_te: ", raw_test_te.shape)
print("X_train_proc_model:", X_train_proc_model.shape)
print("X_test_proc_model: ", X_test_proc_model.shape)
print("te_key_specs:", len(te_key_specs))
print("Expected TE feature count:", len(te_key_specs) * 4)
print("Helper available: build_te_oof_and_apply_many =", callable(build_te_oof_and_apply_many))

gc.collect()

SAFE RECOVERY PASS
------------------
raw_train_te: (144921, 63)
raw_test_te:  (48307, 62)
X_train_proc_model: (144921, 162)
X_test_proc_model:  (48307, 162)
te_key_specs: 17
Expected TE feature count: 68
Helper available: build_te_oof_and_apply_many = True


8

In [24]:
# ============================================================
# 37B / 38B. CPU-safe neural network family
# ============================================================
#
# Textbook family:
#   23. Neural Networks, single-layer / shallow MLP
#   24. Deep Neural Networks
#
# This replaces the failed MPS run.
#
# Key safety changes:
# - Forces PyTorch to CPU.
# - No MPS.
# - No BatchNorm.
# - Smaller batches.
# - Checkpoints prediction outputs after each fold/config.
# - If rerun, completed fold/config predictions are loaded from checkpoints.
#
# Main target:
#   residual = y - current_anchor_prediction
#
# Final prediction:
#   anchor_prediction + lambda * NN_residual_prediction
#
# Direct-target neural nets are also screened by blending with the anchor.
# ============================================================

import os
import gc
import time
import json
import math
import random
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

os.makedirs("model_results", exist_ok=True)
os.makedirs("model_results/nn37b_cpu_checkpoints", exist_ok=True)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# 0. Runtime / reproducibility / CPU safety
# ------------------------------------------------------------

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

MAX_RUNTIME_SECONDS_37B = 24 * 60 * 60
SAFETY_SECONDS_37B = 10 * 60

BATCH_SIZE_37B = 2048
INNER_VALID_FRAC_37B = 0.12
MIN_INNER_VALID_37B = 4096

USE_LEAKAGE_SAFE_TE_37B = True

ANCHOR_OOF_PATH_37B = "model_results/oof_seg33a_arcsine_adaptive.csv"
ANCHOR_TEST_PATH_37B = "model_results/testpred_seg33a_arcsine_adaptive.csv"

ARTIFACT_PREFIX_37B = "nn37b38b_cpu_safe"

# Force CPU. Do not use MPS.
device_37b = torch.device("cpu")

try:
    torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))
except Exception:
    pass

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

deadline_37b = time.time() + MAX_RUNTIME_SECONDS_37B

print("=" * 90)
print("37B / 38B. CPU-safe neural network family")
print("=" * 90)
print("torch version:", torch.__version__)
print("Forced device:", device_37b)
print("torch num threads:", torch.get_num_threads())
print("Max runtime hours:", MAX_RUNTIME_SECONDS_37B / 3600)

def time_left_37b():
    return deadline_37b - time.time()

def should_stop_for_time_37b():
    return time_left_37b() <= SAFETY_SECONDS_37B

# ------------------------------------------------------------
# 1. Required checks and anchor loading
# ------------------------------------------------------------

required_37b = ["y_train", "X_train_proc_model", "X_test_proc_model"]
missing_37b = [x for x in required_37b if x not in globals()]

if missing_37b:
    raise ValueError(
        f"Missing required objects: {missing_37b}. "
        "After the kernel restart, rerun the data/setup cells before running 37B."
    )

if not isinstance(X_train_proc_model, pd.DataFrame) or not isinstance(X_test_proc_model, pd.DataFrame):
    raise ValueError("37B expects X_train_proc_model and X_test_proc_model to be pandas DataFrames.")

y_arr_37b = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_37b = len(y_arr_37b)
n_test_37b = X_test_proc_model.shape[0]

def load_oof_prediction_37b(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_priority = [
        "pred_clipped",
        "prediction",
        "pred",
        "oof_pred",
        "PREDICTED_PERCENT_PROFICIENT",
    ]

    pred_col = None
    for c in pred_priority:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}, column {pred_col}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_y_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_y_diff > 1e-5:
            raise ValueError(f"Anchor OOF target mismatch. Max difference: {max_y_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col

def load_test_prediction_37b(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}, column {pred_col}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids_34a" in globals():
        ids = np.asarray(test_ids_34a)
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no global test ids found.")

    if len(pred) != n_test_37b:
        raise ValueError(f"Test row mismatch for {path}: got {len(pred)}, expected {n_test_37b}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col

# Use the true 33A anchor, not small residual corrections from later families.
anchor_oof_37b, anchor_oof_source_37b = load_oof_prediction_37b(
    ANCHOR_OOF_PATH_37B,
    y_arr_37b,
)
anchor_test_37b, test_ids_37b, anchor_test_source_37b = load_test_prediction_37b(
    ANCHOR_TEST_PATH_37B
)

anchor_mse_37b = float(mean_squared_error(y_arr_37b, anchor_oof_37b))

print("\nAnchor check")
print("------------")
print("Anchor OOF source:", anchor_oof_source_37b)
print("Anchor test source:", anchor_test_source_37b)
print(f"Anchor OOF MSE: {anchor_mse_37b:.6f}")

# ------------------------------------------------------------
# 2. Base numeric features
# ------------------------------------------------------------

base_numeric_train_37b = pd.DataFrame({
    "anchor_pred": anchor_oof_37b.astype(np.float32)
})

base_numeric_test_37b = pd.DataFrame({
    "anchor_pred": anchor_test_37b.astype(np.float32)
})

candidate_numeric_cols_37b = [
    "N_STUDENTS",
    "PERCENT_FREE_LUNCH",
    "PERCENT_REDUCED_LUNCH",
    "PERCENT_ECONOMICALLY_DISADVANTAGED",
    "PERCENT_ENGLISH_LANGUAGE_LEARNERS",
    "PERCENT_ENGLISH_LANGUAGE_LEANERS",
    "PERCENT_WITH_DISABILITIES",
    "PERCENT_STUDENTS_WITH_DISABILITIES",
    "PERCENT_HOMELESS",
    "PERCENT_MIGRANT",
    "PERCENT_FEMALE",
    "PERCENT_MALE",
    "ATTENDANCE_RATE",
]

available_numeric_cols_37b = []

for c in candidate_numeric_cols_37b:
    if c in X_train_proc_model.columns and c in X_test_proc_model.columns:
        if pd.api.types.is_numeric_dtype(X_train_proc_model[c]):
            nunique = int(X_train_proc_model[c].nunique(dropna=True))
            if nunique > 8 and c not in available_numeric_cols_37b:
                base_numeric_train_37b[c] = (
                    pd.to_numeric(X_train_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )
                base_numeric_test_37b[c] = (
                    pd.to_numeric(X_test_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )
                available_numeric_cols_37b.append(c)

base_numeric_cols_37b = list(base_numeric_train_37b.columns)

print("\nBase numeric features")
print("---------------------")
print(base_numeric_cols_37b)

# ------------------------------------------------------------
# 3. Optional leakage-safe TE features
# ------------------------------------------------------------

use_te_37b = (
    USE_LEAKAGE_SAFE_TE_37B
    and "raw_train_te" in globals()
    and "raw_test_te" in globals()
    and "te_key_specs" in globals()
    and (
        "build_te_oof_and_apply_many" in globals()
        or "build_te_oof_and_apply_many_29b" in globals()
    )
)

if use_te_37b and "build_te_oof_and_apply_many" not in globals():
    build_te_oof_and_apply_many = build_te_oof_and_apply_many_29b

print("\nUse leakage-safe TE features inside NN folds:", use_te_37b)

# ------------------------------------------------------------
# 4. Numeric ranking helpers
# ------------------------------------------------------------

def clean_numeric_array_37b(x, fill_value=None):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    finite = np.isfinite(x)

    if fill_value is None:
        fill_value = float(np.nanmedian(x[finite])) if finite.any() else 0.0

    x = np.where(finite, x, fill_value)
    return x.astype(np.float32), float(fill_value)

def rank_columns_by_abs_corr_37b(X_df, target):
    y = np.asarray(target, dtype=np.float64).reshape(-1)
    y = y - np.nanmean(y)
    y_den = np.sqrt(np.dot(y, y))

    rows = []

    for c in X_df.columns:
        x_raw = pd.to_numeric(X_df[c], errors="coerce").to_numpy(dtype=np.float64)
        x, _ = clean_numeric_array_37b(x_raw)
        x = x.astype(np.float64)
        x = x - np.mean(x)
        x_den = np.sqrt(np.dot(x, x))

        if x_den <= 1e-12 or y_den <= 1e-12:
            corr = 0.0
        else:
            corr = abs(float(np.dot(x, y) / (x_den * y_den)))

        rows.append((c, corr))

    return (
        pd.DataFrame(rows, columns=["feature", "abs_corr"])
        .sort_values("abs_corr", ascending=False)
        .reset_index(drop=True)
    )

def select_numeric_cols_37b(cfg, X_fit_all, resid_rank, y_rank):
    selected = list(base_numeric_cols_37b)

    top_k_extra = int(cfg["top_k_extra"])
    if top_k_extra <= 0:
        return [c for c in selected if c in X_fit_all.columns]

    rank_df = resid_rank if cfg["target_mode"] == "residual" else y_rank

    for c in rank_df["feature"].tolist():
        if c not in selected:
            selected.append(c)
        if len(selected) >= len(base_numeric_cols_37b) + top_k_extra:
            break

    return [c for c in selected if c in X_fit_all.columns]

def make_numeric_arrays_37b(X_fit, X_val, X_test, selected_cols):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_fit_num = imputer.fit_transform(X_fit[selected_cols])
    X_val_num = imputer.transform(X_val[selected_cols])
    X_test_num = imputer.transform(X_test[selected_cols])

    X_fit_num = scaler.fit_transform(X_fit_num).astype(np.float32)
    X_val_num = scaler.transform(X_val_num).astype(np.float32)
    X_test_num = scaler.transform(X_test_num).astype(np.float32)

    return X_fit_num, X_val_num, X_test_num

# ------------------------------------------------------------
# 5. Categorical encoding helpers for embedding models
# ------------------------------------------------------------

def clean_cat_37b(s):
    s = pd.Series(s).astype("object")
    s = s.where(pd.notna(s), "__NA__")
    return s.astype(str)

def make_cat_values_37b(raw_df, col_tuple):
    parts = [clean_cat_37b(raw_df[c]) for c in col_tuple]
    out = parts[0].copy()
    for p in parts[1:]:
        out = out + "__" + p
    return out.astype(str).to_numpy()

def cat_def_name_37b(col_tuple):
    return "x".join(col_tuple)

def existing_cat_def_37b(col_tuple):
    if "raw_train_te" not in globals() or "raw_test_te" not in globals():
        return False
    return all(c in raw_train_te.columns for c in col_tuple) and all(c in raw_test_te.columns for c in col_tuple)

def cat_defs_for_mode_37b(mode):
    if mode == "none":
        return []

    assessment_defs = [
        (("ASSESSMENT_NAME",), 500, 1),
        (("SUBGROUP_NAME",), 100, 1),
        (("DISTRICT_TYPE",), 100, 1),
        (("ASSESSMENT_NAME", "SUBGROUP_NAME"), 800, 1),
    ]

    core_defs = assessment_defs + [
        (("COUNTY",), 250, 1),
        (("REGION",), 100, 1),
        (("DISTRICT",), 3000, 2),
        (("SCHOOL",), 12000, 2),
    ]

    combo_defs = core_defs + [
        (("DISTRICT", "ASSESSMENT_NAME"), 8000, 3),
        (("SCHOOL", "ASSESSMENT_NAME"), 20000, 3),
        (("SCHOOL", "SUBGROUP_NAME"), 20000, 3),
        (("COUNTY", "SUBGROUP_NAME"), 2500, 2),
    ]

    if mode == "assessment":
        candidate_defs = assessment_defs
    elif mode == "core":
        candidate_defs = core_defs
    elif mode == "combo":
        candidate_defs = combo_defs
    else:
        raise ValueError(f"Unknown cat_mode: {mode}")

    out = []
    seen = set()

    for col_tuple, max_levels, min_count in candidate_defs:
        if col_tuple in seen:
            continue
        if existing_cat_def_37b(col_tuple):
            out.append({
                "cols": col_tuple,
                "name": cat_def_name_37b(col_tuple),
                "max_levels": int(max_levels),
                "min_count": int(min_count),
            })
            seen.add(col_tuple)

    return out

def fit_transform_cats_37b(raw_fit, raw_val, raw_test, cat_defs):
    if len(cat_defs) == 0:
        return (
            np.zeros((len(raw_fit), 0), dtype=np.int64),
            np.zeros((len(raw_val), 0), dtype=np.int64),
            np.zeros((len(raw_test), 0), dtype=np.int64),
            [],
            [],
        )

    fit_codes = []
    val_codes = []
    test_codes = []
    cat_cards = []
    cat_summaries = []

    for d in cat_defs:
        fit_values = make_cat_values_37b(raw_fit, d["cols"])
        val_values = make_cat_values_37b(raw_val, d["cols"])
        test_values = make_cat_values_37b(raw_test, d["cols"])

        vc = pd.Series(fit_values).value_counts(dropna=False)
        keep = vc[vc >= d["min_count"]].head(d["max_levels"] - 1).index.astype(str).tolist()

        mapping = {v: i + 1 for i, v in enumerate(keep)}
        card = len(keep) + 1

        def map_values(values):
            return np.array([mapping.get(str(v), 0) for v in values], dtype=np.int64)

        fit_codes.append(map_values(fit_values))
        val_codes.append(map_values(val_values))
        test_codes.append(map_values(test_values))
        cat_cards.append(card)

        cat_summaries.append({
            "name": d["name"],
            "cols": "|".join(d["cols"]),
            "cardinality_used": card,
            "max_levels": d["max_levels"],
            "min_count": d["min_count"],
        })

    return (
        np.column_stack(fit_codes).astype(np.int64),
        np.column_stack(val_codes).astype(np.int64),
        np.column_stack(test_codes).astype(np.int64),
        cat_cards,
        cat_summaries,
    )

# ------------------------------------------------------------
# 6. PyTorch model helpers
# ------------------------------------------------------------

class TabularDataset37B(Dataset):
    def __init__(self, X_num, X_cat, y=None):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return self.X_num.shape[0]

    def __getitem__(self, idx):
        if self.y is None:
            return self.X_num[idx], self.X_cat[idx]
        return self.X_num[idx], self.X_cat[idx], self.y[idx]

def emb_dim_37b(card):
    card = int(card)
    if card <= 2:
        return 1
    if card <= 10:
        return min(4, card)
    if card <= 50:
        return 8
    if card <= 500:
        return 12
    if card <= 5000:
        return 16
    return 24

class TabularMLP37B(nn.Module):
    def __init__(self, n_num, cat_cards, hidden_layers, dropout):
        super().__init__()

        self.embeddings = nn.ModuleList()
        emb_total = 0

        for card in cat_cards:
            dim = emb_dim_37b(card)
            self.embeddings.append(nn.Embedding(int(card), int(dim)))
            emb_total += int(dim)

        input_dim = int(n_num) + int(emb_total)

        layers = []
        prev = input_dim

        for h in hidden_layers:
            h = int(h)
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(float(dropout)))
            prev = h

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        pieces = [x_num]

        if len(self.embeddings) > 0:
            emb_pieces = []
            for j, emb in enumerate(self.embeddings):
                emb_pieces.append(emb(x_cat[:, j]))
            pieces.append(torch.cat(emb_pieces, dim=1))

        x = torch.cat(pieces, dim=1)
        return self.net(x)

def make_loss_37b(loss_name):
    if loss_name == "mse":
        return nn.MSELoss()
    if loss_name == "huber":
        return nn.SmoothL1Loss()
    raise ValueError(f"Unknown loss_name: {loss_name}")

def train_predict_nn_37b(
    X_fit_num,
    X_val_num,
    X_test_num,
    X_fit_cat,
    X_val_cat,
    X_test_cat,
    y_fit_raw,
    cfg,
    cat_cards,
    seed,
):
    rng = np.random.default_rng(seed)
    n = X_fit_num.shape[0]

    perm = rng.permutation(n)
    n_valid = max(MIN_INNER_VALID_37B, int(round(INNER_VALID_FRAC_37B * n)))
    n_valid = min(n_valid, max(1, n // 4))

    inner_valid_idx = perm[:n_valid]
    inner_train_idx = perm[n_valid:]

    y_train_part = np.asarray(y_fit_raw[inner_train_idx], dtype=np.float32)
    y_mean = float(np.mean(y_train_part))
    y_std = float(np.std(y_train_part))

    if not np.isfinite(y_std) or y_std < 1e-6:
        y_std = 1.0

    y_scaled = ((np.asarray(y_fit_raw, dtype=np.float32) - y_mean) / y_std).astype(np.float32)

    train_ds = TabularDataset37B(
        X_fit_num[inner_train_idx],
        X_fit_cat[inner_train_idx],
        y_scaled[inner_train_idx],
    )
    valid_ds = TabularDataset37B(
        X_fit_num[inner_valid_idx],
        X_fit_cat[inner_valid_idx],
        y_scaled[inner_valid_idx],
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=int(cfg["batch_size"]),
        shuffle=True,
        num_workers=0,
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=int(cfg["batch_size"]) * 2,
        shuffle=False,
        num_workers=0,
    )

    torch.manual_seed(seed)

    model = TabularMLP37B(
        n_num=X_fit_num.shape[1],
        cat_cards=cat_cards,
        hidden_layers=cfg["hidden_layers"],
        dropout=cfg["dropout"],
    ).to(device_37b)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"]),
    )

    loss_fn = make_loss_37b(cfg["loss"])

    best_loss = np.inf
    best_epoch = 0
    best_state = None
    bad_epochs = 0

    for epoch in range(1, int(cfg["max_epochs"]) + 1):
        if should_stop_for_time_37b():
            break

        model.train()

        for xb_num, xb_cat, yb in train_loader:
            xb_num = xb_num.to(device_37b)
            xb_cat = xb_cat.to(device_37b)
            yb = yb.to(device_37b)

            optimizer.zero_grad(set_to_none=True)
            pred = model(xb_num, xb_cat)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb_num, xb_cat, yb in valid_loader:
                xb_num = xb_num.to(device_37b)
                xb_cat = xb_cat.to(device_37b)
                yb = yb.to(device_37b)

                pred = model(xb_num, xb_cat)
                loss = loss_fn(pred, yb)
                val_losses.append(float(loss.detach().cpu().item()))

        valid_loss = float(np.mean(val_losses)) if len(val_losses) > 0 else np.inf

        if valid_loss < best_loss - 1e-6:
            best_loss = valid_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= int(cfg["patience"]):
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    def predict_array(X_num, X_cat):
        ds = TabularDataset37B(X_num, X_cat, None)
        loader = DataLoader(
            ds,
            batch_size=int(cfg["batch_size"]) * 2,
            shuffle=False,
            num_workers=0,
        )

        preds = []

        model.eval()
        with torch.no_grad():
            for xb_num, xb_cat in loader:
                xb_num = xb_num.to(device_37b)
                xb_cat = xb_cat.to(device_37b)

                pred_scaled = model(xb_num, xb_cat).detach().cpu().numpy().reshape(-1)
                pred_raw = pred_scaled * y_std + y_mean
                preds.append(pred_raw.astype(np.float32))

        return np.concatenate(preds).astype(np.float32)

    pred_val = predict_array(X_val_num, X_val_cat)
    pred_test = predict_array(X_test_num, X_test_cat)

    del model, optimizer, train_loader, valid_loader, train_ds, valid_ds
    gc.collect()

    return pred_val, pred_test, int(best_epoch), float(best_loss), float(y_mean), float(y_std)

# ------------------------------------------------------------
# 7. Configs
# ------------------------------------------------------------

configs_37b = [
    {
        "name": "nn37b_shallow_numeric_base_resid",
        "target_mode": "residual",
        "cat_mode": "none",
        "top_k_extra": 0,
        "hidden_layers": [64],
        "dropout": 0.05,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss": "huber",
        "max_epochs": 60,
        "patience": 6,
        "batch_size": BATCH_SIZE_37B,
    },
    {
        "name": "nn37b_shallow_numeric_te24_resid",
        "target_mode": "residual",
        "cat_mode": "none",
        "top_k_extra": 24,
        "hidden_layers": [128],
        "dropout": 0.08,
        "lr": 8e-4,
        "weight_decay": 2e-4,
        "loss": "huber",
        "max_epochs": 70,
        "patience": 7,
        "batch_size": BATCH_SIZE_37B,
    },
    {
        "name": "nn37b_shallow_numeric_te64_resid",
        "target_mode": "residual",
        "cat_mode": "none",
        "top_k_extra": 64,
        "hidden_layers": [192],
        "dropout": 0.10,
        "lr": 7e-4,
        "weight_decay": 2e-4,
        "loss": "huber",
        "max_epochs": 80,
        "patience": 8,
        "batch_size": BATCH_SIZE_37B,
    },
    {
        "name": "nn38b_deep_numeric_te64_resid",
        "target_mode": "residual",
        "cat_mode": "none",
        "top_k_extra": 64,
        "hidden_layers": [256, 128, 64],
        "dropout": 0.12,
        "lr": 6e-4,
        "weight_decay": 3e-4,
        "loss": "huber",
        "max_epochs": 90,
        "patience": 8,
        "batch_size": BATCH_SIZE_37B,
    },
    {
        "name": "nn37b_embed_assessment_base_resid",
        "target_mode": "residual",
        "cat_mode": "assessment",
        "top_k_extra": 0,
        "hidden_layers": [128, 64],
        "dropout": 0.10,
        "lr": 7e-4,
        "weight_decay": 2e-4,
        "loss": "huber",
        "max_epochs": 80,
        "patience": 8,
        "batch_size": BATCH_SIZE_37B,
    },
    {
        "name": "nn37b_embed_assessment_te24_resid",
        "target_mode": "residual",
        "cat_mode": "assessment",
        "top_k_extra": 24,
        "hidden_layers": [192, 96],
        "dropout": 0.12,
        "lr": 6e-4,
        "weight_decay": 3e-4,
        "loss": "huber",
        "max_epochs": 90,
        "patience": 8,
        "batch_size": BATCH_SIZE_37B,
    },
    {
        "name": "nn38b_embed_core_te48_resid",
        "target_mode": "residual",
        "cat_mode": "core",
        "top_k_extra": 48,
        "hidden_layers": [256, 128, 64],
        "dropout": 0.15,
        "lr": 5e-4,
        "weight_decay": 4e-4,
        "loss": "huber",
        "max_epochs": 100,
        "patience": 9,
        "batch_size": BATCH_SIZE_37B,
    },
    {
        "name": "nn37b_direct_numeric_te48",
        "target_mode": "direct",
        "cat_mode": "none",
        "top_k_extra": 48,
        "hidden_layers": [192, 96],
        "dropout": 0.10,
        "lr": 7e-4,
        "weight_decay": 3e-4,
        "loss": "huber",
        "max_epochs": 80,
        "patience": 8,
        "batch_size": BATCH_SIZE_37B,
    },
    {
        "name": "nn38b_direct_embed_core_te48",
        "target_mode": "direct",
        "cat_mode": "core",
        "top_k_extra": 48,
        "hidden_layers": [256, 128, 64],
        "dropout": 0.15,
        "lr": 5e-4,
        "weight_decay": 4e-4,
        "loss": "huber",
        "max_epochs": 90,
        "patience": 9,
        "batch_size": BATCH_SIZE_37B,
    },
]

print("\nNeural-network configs")
print("----------------------")
print("Number of configs:", len(configs_37b))
for cfg in configs_37b:
    print(
        cfg["name"],
        "| target:", cfg["target_mode"],
        "| cats:", cfg["cat_mode"],
        "| extra numeric:", cfg["top_k_extra"],
        "| hidden:", cfg["hidden_layers"],
    )

# ------------------------------------------------------------
# 8. Smoke test
# ------------------------------------------------------------

print("\nCPU smoke test")
print("--------------")
_smoke_model = TabularMLP37B(n_num=3, cat_cards=[], hidden_layers=[8], dropout=0.0).to(device_37b)
_smoke_x = torch.randn(16, 3, device=device_37b)
_smoke_cat = torch.zeros((16, 0), dtype=torch.long, device=device_37b)
_smoke_y = _smoke_model(_smoke_x, _smoke_cat)
print("Smoke output shape:", tuple(_smoke_y.shape))
del _smoke_model, _smoke_x, _smoke_cat, _smoke_y
gc.collect()

# ------------------------------------------------------------
# 9. OOF training loop
# ------------------------------------------------------------

folds_37b = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_37b))
)

raw_oof_37b = {
    cfg["name"]: np.full(n_train_37b, np.nan, dtype=np.float32)
    for cfg in configs_37b
}

raw_test_sum_37b = {
    cfg["name"]: np.zeros(n_test_37b, dtype=np.float64)
    for cfg in configs_37b
}

fold_metric_rows_37b = []
selected_feature_rows_37b = []
cat_summary_rows_37b = []

overall_t0_37b = time.time()
stop_due_to_time_37b = False

for fold_num, (tr_idx, va_idx) in enumerate(folds_37b, start=1):
    if should_stop_for_time_37b():
        stop_due_to_time_37b = True
        print("\nStopping before next fold because the runtime budget is nearly exhausted.")
        break

    print("\n" + "=" * 90)
    print(f"37B/38B OUTER FOLD {fold_num}/{N_SPLITS}")
    print("=" * 90)

    fold_t0 = time.time()

    y_fit = y_arr_37b[tr_idx]
    y_val = y_arr_37b[va_idx]

    anchor_fit = anchor_oof_37b[tr_idx]
    anchor_val = anchor_oof_37b[va_idx]

    resid_fit = y_fit - anchor_fit

    fold_anchor_mse = float(mean_squared_error(y_val, anchor_val))
    print(f"Fold anchor MSE: {fold_anchor_mse:.6f}")

    X_fit_base = base_numeric_train_37b.iloc[tr_idx].reset_index(drop=True)
    X_val_base = base_numeric_train_37b.iloc[va_idx].reset_index(drop=True)
    X_test_base = base_numeric_test_37b.reset_index(drop=True)

    raw_fit_fold = None
    raw_val_fold = None
    raw_test_fold = None

    if use_te_37b:
        print("Building leakage-safe TE features for NN fold...")

        raw_fit_fold = raw_train_te.iloc[tr_idx].reset_index(drop=True)
        raw_val_fold = raw_train_te.iloc[va_idx].reset_index(drop=True)
        raw_test_fold = raw_test_te.reset_index(drop=True)

        te_fit_fold, apply_dict_fold, _ = build_te_oof_and_apply_many(
            raw_fit=raw_fit_fold,
            y_fit=y_fit,
            raw_apply_dict={
                "valid": raw_val_fold,
                "test": raw_test_fold,
            },
            key_specs=te_key_specs,
            n_splits=5,
            random_state=RANDOM_STATE + 37000 + fold_num,
        )

        te_fit_fold = te_fit_fold.reset_index(drop=True).add_prefix("TE__")
        te_val_fold = apply_dict_fold["valid"].reset_index(drop=True).add_prefix("TE__")
        te_test_fold = apply_dict_fold["test"].reset_index(drop=True).add_prefix("TE__")

        X_fit_all = pd.concat([X_fit_base, te_fit_fold], axis=1)
        X_val_all = pd.concat([X_val_base, te_val_fold], axis=1)
        X_test_all = pd.concat([X_test_base, te_test_fold], axis=1)

        del te_fit_fold, te_val_fold, te_test_fold

    else:
        if "raw_train_te" in globals() and "raw_test_te" in globals():
            raw_fit_fold = raw_train_te.iloc[tr_idx].reset_index(drop=True)
            raw_val_fold = raw_train_te.iloc[va_idx].reset_index(drop=True)
            raw_test_fold = raw_test_te.reset_index(drop=True)

        X_fit_all = X_fit_base.copy()
        X_val_all = X_val_base.copy()
        X_test_all = X_test_base.copy()

    X_fit_all = X_fit_all.loc[:, ~X_fit_all.columns.duplicated()]
    X_val_all = X_val_all[X_fit_all.columns]
    X_test_all = X_test_all[X_fit_all.columns]

    print("NN numeric candidate matrix:", X_fit_all.shape)

    resid_rank = rank_columns_by_abs_corr_37b(X_fit_all, resid_fit)
    y_rank = rank_columns_by_abs_corr_37b(X_fit_all, y_fit)

    print("\nTop residual-ranked numeric features:")
    print(resid_rank.head(8).to_string(index=False))

    print("\nTop target-ranked numeric features:")
    print(y_rank.head(8).to_string(index=False))

    fold_rows_start = len(fold_metric_rows_37b)

    for cfg_i, cfg in enumerate(configs_37b, start=1):
        if should_stop_for_time_37b:
            pass

        if should_stop_for_time_37b():
            stop_due_to_time_37b = True
            print("\nStopping config loop because runtime budget is nearly exhausted.")
            break

        name = cfg["name"]
        ckpt_path = f"model_results/nn37b_cpu_checkpoints/{name}_fold{fold_num}.npz"

        try:
            t0 = time.time()

            if os.path.exists(ckpt_path):
                ckpt = np.load(ckpt_path, allow_pickle=True)
                pred_val_output = ckpt["pred_val_output"].astype(np.float32)
                pred_test_output = ckpt["pred_test_output"].astype(np.float32)
                best_epoch = int(ckpt["best_epoch"])
                best_inner_loss = float(ckpt["best_inner_loss"])
                selected_cols = list(ckpt["selected_cols"])
                n_cat_features = int(ckpt["n_cat_features"])
                status = "loaded_checkpoint"

            else:
                selected_cols = select_numeric_cols_37b(
                    cfg=cfg,
                    X_fit_all=X_fit_all,
                    resid_rank=resid_rank,
                    y_rank=y_rank,
                )

                X_fit_num, X_val_num, X_test_num = make_numeric_arrays_37b(
                    X_fit=X_fit_all,
                    X_val=X_val_all,
                    X_test=X_test_all,
                    selected_cols=selected_cols,
                )

                cat_defs = cat_defs_for_mode_37b(cfg["cat_mode"])

                if cfg["cat_mode"] != "none" and (raw_fit_fold is None or raw_val_fold is None or raw_test_fold is None):
                    raise ValueError("Embedding config requested raw categorical data, but raw_train_te/raw_test_te are unavailable.")

                if len(cat_defs) > 0:
                    X_fit_cat, X_val_cat, X_test_cat, cat_cards, cat_summaries = fit_transform_cats_37b(
                        raw_fit=raw_fit_fold,
                        raw_val=raw_val_fold,
                        raw_test=raw_test_fold,
                        cat_defs=cat_defs,
                    )
                else:
                    X_fit_cat = np.zeros((len(X_fit_num), 0), dtype=np.int64)
                    X_val_cat = np.zeros((len(X_val_num), 0), dtype=np.int64)
                    X_test_cat = np.zeros((len(X_test_num), 0), dtype=np.int64)
                    cat_cards = []
                    cat_summaries = []

                if cfg["target_mode"] == "residual":
                    train_target = resid_fit.astype(np.float32)
                elif cfg["target_mode"] == "direct":
                    train_target = y_fit.astype(np.float32)
                else:
                    raise ValueError(f"Unknown target_mode: {cfg['target_mode']}")

                pred_val_output, pred_test_output, best_epoch, best_inner_loss, y_mean_model, y_std_model = train_predict_nn_37b(
                    X_fit_num=X_fit_num,
                    X_val_num=X_val_num,
                    X_test_num=X_test_num,
                    X_fit_cat=X_fit_cat,
                    X_val_cat=X_val_cat,
                    X_test_cat=X_test_cat,
                    y_fit_raw=train_target,
                    cfg=cfg,
                    cat_cards=cat_cards,
                    seed=RANDOM_STATE + fold_num * 10000 + cfg_i,
                )

                np.savez_compressed(
                    ckpt_path,
                    pred_val_output=pred_val_output.astype(np.float32),
                    pred_test_output=pred_test_output.astype(np.float32),
                    best_epoch=np.array(best_epoch),
                    best_inner_loss=np.array(best_inner_loss),
                    selected_cols=np.array(selected_cols, dtype=object),
                    n_cat_features=np.array(len(cat_cards)),
                )

                for rank_pos, col in enumerate(selected_cols, start=1):
                    selected_feature_rows_37b.append({
                        "fold": fold_num,
                        "config": name,
                        "target_mode": cfg["target_mode"],
                        "cat_mode": cfg["cat_mode"],
                        "rank": rank_pos,
                        "feature": col,
                    })

                for cs in cat_summaries:
                    row = dict(cs)
                    row.update({
                        "fold": fold_num,
                        "config": name,
                        "target_mode": cfg["target_mode"],
                        "cat_mode": cfg["cat_mode"],
                    })
                    cat_summary_rows_37b.append(row)

                n_cat_features = len(cat_cards)

                del X_fit_num, X_val_num, X_test_num
                del X_fit_cat, X_val_cat, X_test_cat
                gc.collect()

                status = "ok"

            raw_oof_37b[name][va_idx] = pred_val_output.astype(np.float32)
            raw_test_sum_37b[name] += pred_test_output.astype(np.float64)

            if cfg["target_mode"] == "residual":
                pred_val_lam1 = np.clip(anchor_val + pred_val_output, 0, 100)
                fold_mse_raw = float(mean_squared_error(y_val, pred_val_lam1))
                fold_gain_raw = fold_anchor_mse - fold_mse_raw
                model_output_mse = float(mean_squared_error(y_val - anchor_val, pred_val_output))
            else:
                pred_val_direct = np.clip(pred_val_output, 0, 100)
                fold_mse_raw = float(mean_squared_error(y_val, pred_val_direct))
                fold_gain_raw = fold_anchor_mse - fold_mse_raw
                model_output_mse = fold_mse_raw

            fold_metric_rows_37b.append({
                "fold": fold_num,
                "config": name,
                "target_mode": cfg["target_mode"],
                "cat_mode": cfg["cat_mode"],
                "top_k_extra": cfg["top_k_extra"],
                "hidden_layers": str(cfg["hidden_layers"]),
                "dropout": cfg["dropout"],
                "lr": cfg["lr"],
                "weight_decay": cfg["weight_decay"],
                "loss": cfg["loss"],
                "best_epoch": best_epoch,
                "best_inner_loss": best_inner_loss,
                "n_numeric_features": len(selected_cols),
                "n_cat_features": n_cat_features,
                "fold_anchor_mse": fold_anchor_mse,
                "fold_mse_raw_or_lambda1": fold_mse_raw,
                "fold_gain_raw_or_lambda1": fold_gain_raw,
                "model_output_mse": model_output_mse,
                "elapsed_seconds": float(time.time() - t0),
                "status": status,
            })

            print(
                f"{name} | fold {fold_num} | mode={cfg['target_mode']} | cats={cfg['cat_mode']} | "
                f"raw/lambda1 MSE {fold_mse_raw:.6f} | gain {fold_gain_raw:.6f} | "
                f"epoch {best_epoch} | status {status}"
            )

        except Exception as e:
            print(f"ERROR | fold {fold_num} | {name}: {repr(e)}")

            fold_metric_rows_37b.append({
                "fold": fold_num,
                "config": name,
                "target_mode": cfg["target_mode"],
                "cat_mode": cfg["cat_mode"],
                "top_k_extra": cfg["top_k_extra"],
                "hidden_layers": str(cfg["hidden_layers"]),
                "dropout": cfg["dropout"],
                "lr": cfg["lr"],
                "weight_decay": cfg["weight_decay"],
                "loss": cfg["loss"],
                "best_epoch": np.nan,
                "best_inner_loss": np.nan,
                "n_numeric_features": np.nan,
                "n_cat_features": np.nan,
                "fold_anchor_mse": fold_anchor_mse,
                "fold_mse_raw_or_lambda1": np.nan,
                "fold_gain_raw_or_lambda1": np.nan,
                "model_output_mse": np.nan,
                "elapsed_seconds": np.nan,
                "status": repr(e),
            })

    fold_summary = pd.DataFrame(fold_metric_rows_37b[fold_rows_start:])
    if len(fold_summary) > 0:
        fold_summary = fold_summary.sort_values("fold_gain_raw_or_lambda1", ascending=False)
        print("\nTop fold candidates by raw/lambda=1 gain:")
        print(
            fold_summary[
                [
                    "config",
                    "target_mode",
                    "cat_mode",
                    "fold_mse_raw_or_lambda1",
                    "fold_gain_raw_or_lambda1",
                    "best_epoch",
                    "elapsed_seconds",
                    "status",
                ]
            ].head(10).to_string(index=False)
        )

    print(f"\nFold {fold_num} elapsed: {time.time() - fold_t0:.1f}s")

    del X_fit_base, X_val_base, X_test_base
    del X_fit_all, X_val_all, X_test_all
    gc.collect()

    if stop_due_to_time_37b:
        break

# ------------------------------------------------------------
# 10. Final OOF shrinkage/blend screen
# ------------------------------------------------------------

lambda_grid_37b = np.unique(
    np.concatenate([
        np.linspace(-1.0, 1.5, 626),
        np.array([0.0, 0.25, 0.5, 0.75, 1.0]),
    ])
)

blend_grid_37b = np.linspace(0.0, 1.0, 501)

screen_rows_37b = []
cfg_by_name_37b = {cfg["name"]: cfg for cfg in configs_37b}

for cfg in configs_37b:
    name = cfg["name"]
    raw_oof = raw_oof_37b[name]

    if not np.isfinite(raw_oof).all():
        print(f"Skipping incomplete config in final screen: {name}")
        continue

    if cfg["target_mode"] == "residual":
        best_row = None

        for lam in lambda_grid_37b:
            pred = np.clip(anchor_oof_37b + float(lam) * raw_oof, 0, 100)
            mse = float(mean_squared_error(y_arr_37b, pred))

            row = {
                "candidate_type": "residual_correction",
                "config": name,
                "target_mode": cfg["target_mode"],
                "cat_mode": cfg["cat_mode"],
                "top_k_extra": cfg["top_k_extra"],
                "hidden_layers": str(cfg["hidden_layers"]),
                "dropout": cfg["dropout"],
                "lr": cfg["lr"],
                "weight_decay": cfg["weight_decay"],
                "loss": cfg["loss"],
                "residual_lambda": float(lam),
                "w_anchor": np.nan,
                "oof_mse": mse,
                "gain_vs_anchor": anchor_mse_37b - mse,
            }

            if best_row is None or row["oof_mse"] < best_row["oof_mse"]:
                best_row = row

        screen_rows_37b.append(best_row)

    elif cfg["target_mode"] == "direct":
        raw_direct = np.clip(raw_oof, 0, 100)
        direct_mse = float(mean_squared_error(y_arr_37b, raw_direct))

        screen_rows_37b.append({
            "candidate_type": "direct_raw",
            "config": name,
            "target_mode": cfg["target_mode"],
            "cat_mode": cfg["cat_mode"],
            "top_k_extra": cfg["top_k_extra"],
            "hidden_layers": str(cfg["hidden_layers"]),
            "dropout": cfg["dropout"],
            "lr": cfg["lr"],
            "weight_decay": cfg["weight_decay"],
            "loss": cfg["loss"],
            "residual_lambda": np.nan,
            "w_anchor": np.nan,
            "oof_mse": direct_mse,
            "gain_vs_anchor": anchor_mse_37b - direct_mse,
        })

        best_blend = None

        for w_anchor in blend_grid_37b:
            pred = np.clip(
                float(w_anchor) * anchor_oof_37b + (1.0 - float(w_anchor)) * raw_direct,
                0,
                100,
            )
            mse = float(mean_squared_error(y_arr_37b, pred))

            row = {
                "candidate_type": "direct_anchor_blend",
                "config": name,
                "target_mode": cfg["target_mode"],
                "cat_mode": cfg["cat_mode"],
                "top_k_extra": cfg["top_k_extra"],
                "hidden_layers": str(cfg["hidden_layers"]),
                "dropout": cfg["dropout"],
                "lr": cfg["lr"],
                "weight_decay": cfg["weight_decay"],
                "loss": cfg["loss"],
                "residual_lambda": np.nan,
                "w_anchor": float(w_anchor),
                "oof_mse": mse,
                "gain_vs_anchor": anchor_mse_37b - mse,
            }

            if best_blend is None or row["oof_mse"] < best_blend["oof_mse"]:
                best_blend = row

        screen_rows_37b.append(best_blend)

screen37b = pd.DataFrame(screen_rows_37b)

if len(screen37b) == 0:
    raise ValueError("No complete neural-network candidates were available for final screening.")

screen37b = screen37b.sort_values("oof_mse").reset_index(drop=True)

def prediction_from_screen_row_37b(row):
    name = row["config"]
    raw_oof = raw_oof_37b[name]
    raw_test = raw_test_sum_37b[name] / N_SPLITS

    if row["candidate_type"] == "residual_correction":
        lam = float(row["residual_lambda"])
        pred_oof = np.clip(anchor_oof_37b + lam * raw_oof, 0, 100).astype(np.float32)
        pred_test = np.clip(anchor_test_37b + lam * raw_test, 0, 100).astype(np.float32)
        return pred_oof, pred_test

    if row["candidate_type"] == "direct_raw":
        pred_oof = np.clip(raw_oof, 0, 100).astype(np.float32)
        pred_test = np.clip(raw_test, 0, 100).astype(np.float32)
        return pred_oof, pred_test

    if row["candidate_type"] == "direct_anchor_blend":
        w_anchor = float(row["w_anchor"])
        pred_oof = np.clip(w_anchor * anchor_oof_37b + (1.0 - w_anchor) * raw_oof, 0, 100).astype(np.float32)
        pred_test = np.clip(w_anchor * anchor_test_37b + (1.0 - w_anchor) * raw_test, 0, 100).astype(np.float32)
        return pred_oof, pred_test

    raise ValueError(f"Unknown candidate_type: {row['candidate_type']}")

best37b = screen37b.iloc[0]
best_oof37b, best_test37b = prediction_from_screen_row_37b(best37b)

best_mse37b = float(mean_squared_error(y_arr_37b, best_oof37b))
best_gain37b = anchor_mse_37b - best_mse37b

fold_gains_rows_37b = []

for fold_num, (_, va_idx) in enumerate(folds_37b, start=1):
    anchor_fold_mse = float(mean_squared_error(y_arr_37b[va_idx], anchor_oof_37b[va_idx]))
    cand_fold_mse = float(mean_squared_error(y_arr_37b[va_idx], best_oof37b[va_idx]))

    fold_gains_rows_37b.append({
        "fold": fold_num,
        "anchor_mse": anchor_fold_mse,
        "candidate_mse": cand_fold_mse,
        "gain_vs_anchor": anchor_fold_mse - cand_fold_mse,
    })

fold_gains37b = pd.DataFrame(fold_gains_rows_37b)

# ------------------------------------------------------------
# 11. Save artifacts
# ------------------------------------------------------------

screen_path37b = "model_results/nn37b38b_cpu_safe_screen.csv"
fold_metrics_path37b = "model_results/nn37b38b_cpu_safe_fold_metrics.csv"
fold_gains_path37b = "model_results/nn37b38b_cpu_safe_best_fold_gains.csv"
selected_features_path37b = "model_results/nn37b38b_cpu_safe_selected_features.csv"
cat_summary_path37b = "model_results/nn37b38b_cpu_safe_cat_summary.csv"
saved_candidates_path37b = "model_results/nn37b38b_cpu_safe_saved_candidates.csv"

screen37b.to_csv(screen_path37b, index=False)
pd.DataFrame(fold_metric_rows_37b).to_csv(fold_metrics_path37b, index=False)
fold_gains37b.to_csv(fold_gains_path37b, index=False)
pd.DataFrame(selected_feature_rows_37b).to_csv(selected_features_path37b, index=False)
pd.DataFrame(cat_summary_rows_37b).to_csv(cat_summary_path37b, index=False)

saved_candidate_paths_37b = []

top_save_n_37b = min(3, len(screen37b))

for rank_i in range(top_save_n_37b):
    row = screen37b.iloc[rank_i]
    pred_oof_i, pred_test_i = prediction_from_screen_row_37b(row)

    label = f"nn37b38b_cpu_safe_rank{rank_i + 1}"
    oof_path_i = f"model_results/oof_{label}.csv"
    test_path_i = f"model_results/testpred_{label}.csv"
    submission_path_i = f"submission_{label}.csv"

    pd.DataFrame({
        "row_index": np.arange(n_train_37b),
        TARGET_COL: y_arr_37b,
        "pred_clipped": pred_oof_i.astype(np.float32),
    }).to_csv(oof_path_i, index=False)

    pd.DataFrame({
        ID_COL: test_ids_37b,
        TARGET_COL: pred_test_i.astype(np.float32),
    }).to_csv(test_path_i, index=False)

    pd.DataFrame({
        ID_COL: test_ids_37b,
        TARGET_COL: pred_test_i.astype(np.float32),
    }).to_csv(submission_path_i, index=False)

    saved_candidate_paths_37b.append({
        "rank": rank_i + 1,
        "config": row["config"],
        "candidate_type": row["candidate_type"],
        "oof_path": oof_path_i,
        "testpred_path": test_path_i,
        "submission_path": submission_path_i,
        "oof_mse": float(mean_squared_error(y_arr_37b, pred_oof_i)),
        "gain_vs_anchor": float(anchor_mse_37b - mean_squared_error(y_arr_37b, pred_oof_i)),
    })

pd.DataFrame(saved_candidate_paths_37b).to_csv(saved_candidates_path37b, index=False)

# ------------------------------------------------------------
# 12. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("37B/38B CPU-safe neural-network family complete")
print("=" * 90)

print("\nRuntime")
print("-------")
print("Runtime seconds:", round(time.time() - overall_t0_37b, 1))
print("Stopped due to time budget:", stop_due_to_time_37b)

print("\nAnchor")
print("------")
print(f"Anchor OOF MSE: {anchor_mse_37b:.6f}")

print("\nTop 20 neural-network candidates")
print("--------------------------------")
display(screen37b.head(20))

print("\nBest neural-network candidate")
print("-----------------------------")
print(best37b.to_string())
print(f"\nBest neural-network OOF MSE: {best_mse37b:.6f}")
print(f"Gain vs anchor:              {best_gain37b:.6f}")
print(f"Min fold gain:               {fold_gains37b['gain_vs_anchor'].min():.6f}")

print("\nBest candidate fold gains")
print("-------------------------")
print(fold_gains37b.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path37b)
print(fold_metrics_path37b)
print(fold_gains_path37b)
print(selected_features_path37b)
print(cat_summary_path37b)
print(saved_candidates_path37b)

print("\nSaved top candidate submissions")
print("-------------------------------")
for row in saved_candidate_paths_37b:
    print(row)

print("\nDecision rule")
print("-------------")
if best_gain37b >= 5.0:
    print("Material neural-network signal found. Continue neural networks with a focused expansion around the winning architecture.")
elif best_gain37b >= 1.0:
    print("Some neural-network signal found. Consider one focused expansion before moving to SVR/kernel methods.")
else:
    print("No material neural-network signal. Close the neural-network family and move to SVR/kernel methods.")

37B / 38B. CPU-safe neural network family
torch version: 2.11.0
Forced device: cpu
torch num threads: 8
Max runtime hours: 24.0

Anchor check
------------
Anchor OOF source: pred_clipped
Anchor test source: PERCENT_PROFICIENT
Anchor OOF MSE: 77.049866

Base numeric features
---------------------
['anchor_pred', 'N_STUDENTS', 'PERCENT_FREE_LUNCH', 'PERCENT_REDUCED_LUNCH', 'PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_WITH_DISABILITIES', 'PERCENT_HOMELESS', 'PERCENT_MIGRANT', 'PERCENT_FEMALE', 'PERCENT_MALE', 'ATTENDANCE_RATE']

Use leakage-safe TE features inside NN folds: True

Neural-network configs
----------------------
Number of configs: 9
nn37b_shallow_numeric_base_resid | target: residual | cats: none | extra numeric: 0 | hidden: [64]
nn37b_shallow_numeric_te24_resid | target: residual | cats: none | extra numeric: 24 | hidden: [128]
nn37b_shallow_numeric_te64_resid | target: residual | cats: none | extra numeric: 64 | hidden: [192]
nn38b_deep

,candidate_type,config,target_mode,cat_mode,top_k_extra,hidden_layers,dropout,lr,weight_decay,loss,residual_lambda,w_anchor,oof_mse,gain_vs_anchor
0,residual_correction,nn37b_shallow_numeric_te64_resid,residual,none,64,[192],0.10,0.0007,0.0002,huber,0.236,NaN,76.849442,0.200424
1,residual_correction,nn38b_deep_numeric_te64_resid,residual,none,64,"[256, 128, 64]",0.12,0.0006,0.0003,huber,0.180,NaN,76.883743,0.166122
2,residual_correction,nn37b_embed_assessment_te24_resid,residual,assessment,24,"[192, 96]",0.12,0.0006,0.0003,huber,0.320,NaN,76.913986,0.135880
3,residual_correction,nn37b_shallow_numeric_te24_resid,residual,none,24,[128],0.08,0.0008,0.0002,huber,0.300,NaN,76.928535,0.121330
4,residual_correction,nn37b_embed_assessment_base_resid,residual,assessment,0,"[128, 64]",0.10,0.0007,0.0002,huber,0.368,NaN,76.973351,0.076515
5,direct_anchor_blend,nn37b_direct_numeric_te48,direct,none,48,"[192, 96]",0.10,0.0007,0.0003,huber,NaN,0.876,77.013786,0.036079
6,residual_correction,nn38b_embed_core_te48_resid,residual,core,48,"[256, 128, 64]",0.15,0.0005,0.0004,huber,0.184,NaN,77.015884,0.033981
7,residual_correction,nn37b_shallow_numeric_base_resid,residual,none,0,[64],0.05,0.0010,0.0001,huber,0.250,NaN,77.036110,0.013756
8,direct_anchor_blend,nn38b_direct_embed_core_te48,direct,core,48,"[256, 128, 64]",0.15,0.0005,0.0004,huber,NaN,1.000,77.049866,0.000000
9,direct_raw,nn37b_direct_numeric_te48,direct,none,48,"[192, 96]",0.10,0.0007,0.0003,huber,NaN,NaN,78.816711,-1.766846



Best neural-network candidate
-----------------------------
candidate_type                  residual_correction
config             nn37b_shallow_numeric_te64_resid
target_mode                                residual
cat_mode                                       none
top_k_extra                                      64
hidden_layers                                 [192]
dropout                                         0.1
lr                                           0.0007
weight_decay                                 0.0002
loss                                          huber
residual_lambda                               0.236
w_anchor                                        NaN
oof_mse                                   76.849442
gain_vs_anchor                             0.200424

Best neural-network OOF MSE: 76.849442
Gain vs anchor:              0.200424
Min fold gain:               0.071922

Best candidate fold gains
-------------------------
 fold  anchor_mse  candidate_mse  gain_vs_

### 37B/38B. CPU-Safe Neural Network Family

This section tested the neural-network model family after the earlier MPS-based run caused a kernel crash. The replacement implementation forced PyTorch to CPU, removed MPS-sensitive components, and checkpointed candidate predictions.

The models included shallow residual MLPs, deeper residual MLPs, assessment-level embedding models, core entity embedding models, and direct-target neural networks blended with the current 33A anchor.

The 33A anchor OOF MSE was `77.049866`. The best neural-network candidate was `nn37b_shallow_numeric_te64_resid`, a shallow residual neural network using 64 selected numeric/target-encoding features. Its optimal residual shrinkage was `0.236`.

The best neural-network OOF MSE was `76.849442`, for a gain of `0.200424` versus the anchor. Fold gains were positive but small:

- Fold 1 gain: `0.176636`
- Fold 2 gain: `0.173096`
- Fold 3 gain: `0.266533`
- Fold 4 gain: `0.071922`
- Fold 5 gain: `0.313934`

Conclusion: the neural-network family produced a stable but small residual correction. This does not explain the large public-leaderboard gap, so generic neural-network tuning should be closed as a low-priority branch.

In [25]:
# ============================================================
# 39A. Subgroup accounting / sibling-row reconstruction diagnostic
# ============================================================
#
# Structural idea:
#   For a fixed SCHOOL x ASSESSMENT_NAME, subgroup rows may be related.
#
# Example:
#   All Students count ≈ Male count + Female count
#   Proficient_All ≈ Proficient_Male + Proficient_Female
#
# Since target is a percentage and N_STUDENTS is available:
#   proficient_count ≈ round(PERCENT_PROFICIENT / 100 * N_STUDENTS)
#
# This cell:
#   1. Discovers likely complementary subgroup pairs from N_STUDENTS structure.
#   2. Performs OOF-safe sibling reconstruction using only fold-training sibling rows.
#   3. Blends/replaces the current 33A anchor only where reconstruction is available.
#   4. Applies the same reconstruction to the test set using full training labels.
#
# This is not a generic model family. It tests structural signal in the data.
# ============================================================

import os
import gc
import itertools
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

ANCHOR_OOF_PATH_39A = "model_results/oof_seg33a_arcsine_adaptive.csv"
ANCHOR_TEST_PATH_39A = "model_results/testpred_seg33a_arcsine_adaptive.csv"

print("=" * 90)
print("39A. Subgroup accounting / sibling-row reconstruction diagnostic")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required checks
# ------------------------------------------------------------

required_39a = ["raw_train_te", "raw_test_te", "y_train"]

missing_39a = [x for x in required_39a if x not in globals()]
if missing_39a:
    raise ValueError(f"Missing required objects: {missing_39a}")

needed_cols_39a = ["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]
for c in needed_cols_39a:
    if c not in raw_train_te.columns or c not in raw_test_te.columns:
        raise ValueError(f"Missing required column in raw_train_te/raw_test_te: {c}")

y_arr_39a = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_39a = len(y_arr_39a)
n_test_39a = len(raw_test_te)

if len(raw_train_te) != n_train_39a:
    raise ValueError("raw_train_te and y_train length mismatch.")

# ------------------------------------------------------------
# 2. Anchor loading
# ------------------------------------------------------------

def load_oof_prediction_39a(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_col = None
    for c in ["pred_clipped", "prediction", "pred", "oof_pred", TARGET_COL]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}, column {pred_col}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_y_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_y_diff > 1e-5:
            raise ValueError(f"Anchor OOF target mismatch. Max difference: {max_y_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col

def load_test_prediction_39a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}, column {pred_col}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no global test_ids found.")

    if len(pred) != n_test_39a:
        raise ValueError(f"Test row mismatch for {path}: got {len(pred)}, expected {n_test_39a}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col

anchor_oof_39a, anchor_oof_col_39a = load_oof_prediction_39a(
    ANCHOR_OOF_PATH_39A,
    y_arr_39a,
)
anchor_test_39a, test_ids_39a, anchor_test_col_39a = load_test_prediction_39a(
    ANCHOR_TEST_PATH_39A
)

anchor_mse_39a = float(mean_squared_error(y_arr_39a, anchor_oof_39a))

print("\nAnchor check")
print("------------")
print("Anchor OOF column:", anchor_oof_col_39a)
print("Anchor test column:", anchor_test_col_39a)
print(f"Anchor OOF MSE: {anchor_mse_39a:.6f}")

# ------------------------------------------------------------
# 3. Build compact row frames
# ------------------------------------------------------------

def clean_str_39a(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def safe_n_39a(x):
    x = pd.to_numeric(x, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return x.to_numpy(dtype=np.float64)

train39 = pd.DataFrame({
    "row_index": np.arange(n_train_39a),
    "SCHOOL": clean_str_39a(raw_train_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_str_39a(raw_train_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str_39a(raw_train_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_n_39a(raw_train_te["N_STUDENTS"]),
    TARGET_COL: y_arr_39a,
})

test39 = pd.DataFrame({
    "row_index": np.arange(n_test_39a),
    "SCHOOL": clean_str_39a(raw_test_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_str_39a(raw_test_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str_39a(raw_test_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_n_39a(raw_test_te["N_STUDENTS"]),
})

train39["group_key"] = train39["SCHOOL"] + "||" + train39["ASSESSMENT_NAME"]
test39["group_key"] = test39["SCHOOL"] + "||" + test39["ASSESSMENT_NAME"]

# Approximate proficient counts.
train39["N_STUDENTS"] = np.where(
    np.isfinite(train39["N_STUDENTS"]) & (train39["N_STUDENTS"] > 0),
    train39["N_STUDENTS"],
    np.nan,
)

test39["N_STUDENTS"] = np.where(
    np.isfinite(test39["N_STUDENTS"]) & (test39["N_STUDENTS"] > 0),
    test39["N_STUDENTS"],
    np.nan,
)

train39["prof_count"] = np.rint(
    np.clip(train39[TARGET_COL].to_numpy(dtype=np.float64), 0, 100)
    / 100.0
    * train39["N_STUDENTS"].to_numpy(dtype=np.float64)
)

train39["prof_count"] = np.clip(
    train39["prof_count"],
    0,
    train39["N_STUDENTS"],
)

print("\nSubgroup values")
print("---------------")
subgroup_counts39 = (
    train39["SUBGROUP_NAME"]
    .value_counts(dropna=False)
    .rename_axis("SUBGROUP_NAME")
    .reset_index(name="train_rows")
)

subgroup_test_counts39 = (
    test39["SUBGROUP_NAME"]
    .value_counts(dropna=False)
    .rename_axis("SUBGROUP_NAME")
    .reset_index(name="test_rows")
)

subgroup_summary39 = subgroup_counts39.merge(
    subgroup_test_counts39,
    on="SUBGROUP_NAME",
    how="outer",
).fillna(0)

subgroup_summary39["train_rows"] = subgroup_summary39["train_rows"].astype(int)
subgroup_summary39["test_rows"] = subgroup_summary39["test_rows"].astype(int)

print(subgroup_summary39.to_string(index=False))

# ------------------------------------------------------------
# 4. Discover likely complementary subgroup pairs
# ------------------------------------------------------------

subgroups39 = sorted(train39["SUBGROUP_NAME"].unique().tolist())

# Wide count/proficient matrices by SCHOOL x ASSESSMENT_NAME.
n_wide39 = train39.pivot_table(
    index="group_key",
    columns="SUBGROUP_NAME",
    values="N_STUDENTS",
    aggfunc="first",
)

k_wide39 = train39.pivot_table(
    index="group_key",
    columns="SUBGROUP_NAME",
    values="prof_count",
    aggfunc="first",
)

pair_rows39 = []

for all_s in subgroups39:
    for a_s, b_s in itertools.combinations([s for s in subgroups39 if s != all_s], 2):
        if all_s not in n_wide39.columns or a_s not in n_wide39.columns or b_s not in n_wide39.columns:
            continue

        n_all = n_wide39[all_s]
        n_a = n_wide39[a_s]
        n_b = n_wide39[b_s]

        complete = n_all.notna() & n_a.notna() & n_b.notna()

        if int(complete.sum()) == 0:
            continue

        n_all_v = n_all[complete].astype(float)
        n_sum_v = n_a[complete].astype(float) + n_b[complete].astype(float)

        n_tol = np.maximum(1.0, 0.02 * n_all_v.to_numpy())
        n_abs_err = np.abs(n_all_v.to_numpy() - n_sum_v.to_numpy())
        n_match = n_abs_err <= n_tol

        k_match_rate = np.nan
        k_mae = np.nan

        if all_s in k_wide39.columns and a_s in k_wide39.columns and b_s in k_wide39.columns:
            k_all = k_wide39[all_s]
            k_a = k_wide39[a_s]
            k_b = k_wide39[b_s]

            k_complete = complete & k_all.notna() & k_a.notna() & k_b.notna()

            if int(k_complete.sum()) > 0:
                k_all_v = k_all[k_complete].astype(float)
                k_sum_v = k_a[k_complete].astype(float) + k_b[k_complete].astype(float)
                k_tol = np.maximum(1.0, 0.02 * n_wide39.loc[k_complete, all_s].astype(float).to_numpy())
                k_abs_err = np.abs(k_all_v.to_numpy() - k_sum_v.to_numpy())
                k_match_rate = float(np.mean(k_abs_err <= k_tol))
                k_mae = float(np.mean(k_abs_err))

        pair_rows39.append({
            "all_subgroup": all_s,
            "part_a": a_s,
            "part_b": b_s,
            "complete_groups": int(complete.sum()),
            "n_match_rate": float(np.mean(n_match)),
            "n_mae": float(np.mean(n_abs_err)),
            "k_match_rate": k_match_rate,
            "k_mae": k_mae,
            "name_has_all": int("all" in all_s.lower()),
        })

pair_candidates39 = pd.DataFrame(pair_rows39)

if len(pair_candidates39) == 0:
    raise ValueError("No subgroup pair candidates could be formed.")

pair_candidates39["selection_score"] = (
    pair_candidates39["n_match_rate"] * np.log1p(pair_candidates39["complete_groups"])
    + 0.25 * pair_candidates39["name_has_all"]
)

pair_candidates39 = pair_candidates39.sort_values(
    ["selection_score", "n_match_rate", "complete_groups"],
    ascending=False,
).reset_index(drop=True)

print("\nTop candidate subgroup accounting relationships")
print("-----------------------------------------------")
print(
    pair_candidates39[
        [
            "all_subgroup",
            "part_a",
            "part_b",
            "complete_groups",
            "n_match_rate",
            "n_mae",
            "k_match_rate",
            "k_mae",
            "name_has_all",
        ]
    ].head(20).to_string(index=False)
)

pair_path39 = "model_results/account39a_pair_candidates.csv"
pair_candidates39.to_csv(pair_path39, index=False)

selected_pairs39 = pair_candidates39[
    (pair_candidates39["complete_groups"] >= 50)
    & (pair_candidates39["n_match_rate"] >= 0.70)
].copy()

if len(selected_pairs39) == 0:
    selected_pairs39 = pair_candidates39.head(3).copy()
else:
    selected_pairs39 = selected_pairs39.head(10).copy()

selected_pair_tuples39 = [
    (r["all_subgroup"], r["part_a"], r["part_b"])
    for _, r in selected_pairs39.iterrows()
]

print("\nSelected accounting relationships")
print("---------------------------------")
for all_s, a_s, b_s in selected_pair_tuples39:
    print(f"{all_s} ≈ {a_s} + {b_s}")

# ------------------------------------------------------------
# 5. Group lookup + reconstruction functions
# ------------------------------------------------------------

def build_lookup_39a(df, row_indices, has_target=True):
    out = {}

    sub_df = df.iloc[row_indices]

    for r in sub_df.itertuples(index=False):
        n = float(r.N_STUDENTS) if np.isfinite(r.N_STUDENTS) else np.nan

        if not np.isfinite(n) or n <= 0:
            continue

        key = r.group_key
        subgroup = r.SUBGROUP_NAME

        if key not in out:
            out[key] = {}

        if has_target:
            k = float(r.prof_count) if np.isfinite(r.prof_count) else np.nan
            if not np.isfinite(k):
                continue
            out[key][subgroup] = {
                "n": n,
                "k": k,
                "row_index": int(r.row_index),
            }
        else:
            out[key][subgroup] = {
                "n": n,
                "k": np.nan,
                "row_index": int(r.row_index),
            }

    return out

def n_tolerance_39a(n):
    if not np.isfinite(n):
        return 1.0
    return max(1.0, 0.02 * float(n))

def reconstruct_one_39a(row, lookup, selected_pairs):
    key = row["group_key"]
    target_s = row["SUBGROUP_NAME"]
    n_target = float(row["N_STUDENTS"])

    if not np.isfinite(n_target) or n_target <= 0:
        return np.nan, np.nan, ""

    group = lookup.get(key, None)

    if group is None:
        return np.nan, np.nan, ""

    preds = []
    methods = []

    for all_s, a_s, b_s in selected_pairs:
        # Predict All from A + B
        if target_s == all_s and a_s in group and b_s in group:
            n_pred = group[a_s]["n"] + group[b_s]["n"]

            if abs(n_target - n_pred) <= n_tolerance_39a(n_target):
                k_pred = group[a_s]["k"] + group[b_s]["k"]
                k_pred = np.clip(k_pred, 0, n_target)
                preds.append(100.0 * k_pred / n_target)
                methods.append(f"{all_s}<={a_s}+{b_s}")

        # Predict A from All - B
        if target_s == a_s and all_s in group and b_s in group:
            n_pred = group[all_s]["n"] - group[b_s]["n"]

            if abs(n_target - n_pred) <= n_tolerance_39a(n_target):
                k_pred = group[all_s]["k"] - group[b_s]["k"]
                k_pred = np.clip(k_pred, 0, n_target)
                preds.append(100.0 * k_pred / n_target)
                methods.append(f"{a_s}<={all_s}-{b_s}")

        # Predict B from All - A
        if target_s == b_s and all_s in group and a_s in group:
            n_pred = group[all_s]["n"] - group[a_s]["n"]

            if abs(n_target - n_pred) <= n_tolerance_39a(n_target):
                k_pred = group[all_s]["k"] - group[a_s]["k"]
                k_pred = np.clip(k_pred, 0, n_target)
                preds.append(100.0 * k_pred / n_target)
                methods.append(f"{b_s}<={all_s}-{a_s}")

    if len(preds) == 0:
        return np.nan, np.nan, ""

    pred = float(np.mean(preds))
    pred = float(np.clip(pred, 0, 100))

    return pred, float(len(preds)), " | ".join(methods)

def reconstruct_many_39a(query_df, lookup, selected_pairs):
    pred = np.full(len(query_df), np.nan, dtype=np.float32)
    n_methods = np.zeros(len(query_df), dtype=np.float32)
    method = np.array([""] * len(query_df), dtype=object)

    for j, (_, row) in enumerate(query_df.iterrows()):
        p, m, s = reconstruct_one_39a(row, lookup, selected_pairs)
        if np.isfinite(p):
            pred[j] = p
            n_methods[j] = m
            method[j] = s

    return pred, n_methods, method

# ------------------------------------------------------------
# 6. OOF-safe reconstruction
# ------------------------------------------------------------

folds39 = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_39a))
)

acct_oof39 = np.full(n_train_39a, np.nan, dtype=np.float32)
acct_oof_methods39 = np.array([""] * n_train_39a, dtype=object)
acct_oof_n_methods39 = np.zeros(n_train_39a, dtype=np.float32)

fold_rows39 = []

for fold_num, (tr_idx, va_idx) in enumerate(folds39, start=1):
    lookup_fold = build_lookup_39a(train39, tr_idx, has_target=True)

    pred_fold, n_methods_fold, method_fold = reconstruct_many_39a(
        train39.iloc[va_idx].reset_index(drop=True),
        lookup_fold,
        selected_pair_tuples39,
    )

    acct_oof39[va_idx] = pred_fold
    acct_oof_n_methods39[va_idx] = n_methods_fold
    acct_oof_methods39[va_idx] = method_fold

    covered = np.isfinite(pred_fold)
    n_covered = int(covered.sum())

    if n_covered > 0:
        y_val = y_arr_39a[va_idx][covered]
        anchor_val = anchor_oof_39a[va_idx][covered]
        acct_val = pred_fold[covered]

        acct_mse = float(mean_squared_error(y_val, acct_val))
        anchor_subset_mse = float(mean_squared_error(y_val, anchor_val))
        gain_subset = anchor_subset_mse - acct_mse
    else:
        acct_mse = np.nan
        anchor_subset_mse = np.nan
        gain_subset = np.nan

    fold_rows39.append({
        "fold": fold_num,
        "n_valid_rows": len(va_idx),
        "n_accounting_covered": n_covered,
        "coverage_rate": n_covered / len(va_idx),
        "anchor_mse_on_covered": anchor_subset_mse,
        "accounting_mse_on_covered": acct_mse,
        "gain_on_covered": gain_subset,
    })

fold_diag39 = pd.DataFrame(fold_rows39)

covered_oof39 = np.isfinite(acct_oof39)

print("\nOOF accounting coverage")
print("-----------------------")
print(fold_diag39.to_string(index=False))
print("\nTotal covered OOF rows:", int(covered_oof39.sum()), "of", n_train_39a)
print("Total OOF coverage:", float(covered_oof39.mean()))

# ------------------------------------------------------------
# 7. Blend/replace screen against anchor
# ------------------------------------------------------------

screen_rows39 = []

if int(covered_oof39.sum()) > 0:
    acct_raw_full39 = anchor_oof_39a.copy()
    acct_raw_full39[covered_oof39] = acct_oof39[covered_oof39]

    raw_replacement_mse39 = float(mean_squared_error(y_arr_39a, acct_raw_full39))

    screen_rows39.append({
        "candidate": "accounting_replace_covered",
        "covered_rows": int(covered_oof39.sum()),
        "coverage_rate": float(covered_oof39.mean()),
        "lambda_accounting": 1.0,
        "oof_mse": raw_replacement_mse39,
        "gain_vs_anchor": anchor_mse_39a - raw_replacement_mse39,
    })

    best_row = None

    # Only alter covered rows; uncovered rows remain anchor.
    for lam in np.linspace(-0.50, 1.50, 501):
        pred = anchor_oof_39a.copy()
        pred[covered_oof39] = (
            anchor_oof_39a[covered_oof39]
            + float(lam) * (acct_oof39[covered_oof39] - anchor_oof_39a[covered_oof39])
        )
        pred = np.clip(pred, 0, 100)

        mse = float(mean_squared_error(y_arr_39a, pred))

        row = {
            "candidate": "accounting_blend_covered",
            "covered_rows": int(covered_oof39.sum()),
            "coverage_rate": float(covered_oof39.mean()),
            "lambda_accounting": float(lam),
            "oof_mse": mse,
            "gain_vs_anchor": anchor_mse_39a - mse,
        }

        if best_row is None or row["oof_mse"] < best_row["oof_mse"]:
            best_row = row

    screen_rows39.append(best_row)

else:
    print("No OOF rows covered by accounting reconstruction.")

screen39 = pd.DataFrame(screen_rows39)

if len(screen39) == 0:
    screen39 = pd.DataFrame(columns=[
        "candidate",
        "covered_rows",
        "coverage_rate",
        "lambda_accounting",
        "oof_mse",
        "gain_vs_anchor",
    ])

screen39 = screen39.sort_values("oof_mse").reset_index(drop=True)

# ------------------------------------------------------------
# 8. Test reconstruction using full training labels
# ------------------------------------------------------------

lookup_full39 = build_lookup_39a(train39, np.arange(n_train_39a), has_target=True)

acct_test39, acct_test_n_methods39, acct_test_methods39 = reconstruct_many_39a(
    test39.reset_index(drop=True),
    lookup_full39,
    selected_pair_tuples39,
)

covered_test39 = np.isfinite(acct_test39)

print("\nTest accounting coverage")
print("------------------------")
print("Covered test rows:", int(covered_test39.sum()), "of", n_test_39a)
print("Test coverage rate:", float(covered_test39.mean()))

# ------------------------------------------------------------
# 9. Save best artifact
# ------------------------------------------------------------

if len(screen39) > 0:
    best39 = screen39.iloc[0]
    best_lambda39 = float(best39["lambda_accounting"])

    best_oof39 = anchor_oof_39a.copy()
    best_oof39[covered_oof39] = (
        anchor_oof_39a[covered_oof39]
        + best_lambda39 * (acct_oof39[covered_oof39] - anchor_oof_39a[covered_oof39])
    )
    best_oof39 = np.clip(best_oof39, 0, 100).astype(np.float32)

    best_test39 = anchor_test_39a.copy()
    best_test39[covered_test39] = (
        anchor_test_39a[covered_test39]
        + best_lambda39 * (acct_test39[covered_test39] - anchor_test_39a[covered_test39])
    )
    best_test39 = np.clip(best_test39, 0, 100).astype(np.float32)

    best_mse39 = float(mean_squared_error(y_arr_39a, best_oof39))
    best_gain39 = anchor_mse_39a - best_mse39

    fold_gain_rows39 = []
    for fold_num, (_, va_idx) in enumerate(folds39, start=1):
        anchor_fold_mse = float(mean_squared_error(y_arr_39a[va_idx], anchor_oof_39a[va_idx]))
        cand_fold_mse = float(mean_squared_error(y_arr_39a[va_idx], best_oof39[va_idx]))
        fold_gain_rows39.append({
            "fold": fold_num,
            "anchor_mse": anchor_fold_mse,
            "candidate_mse": cand_fold_mse,
            "gain_vs_anchor": anchor_fold_mse - cand_fold_mse,
            "covered_rows": int(np.isfinite(acct_oof39[va_idx]).sum()),
        })

    fold_gains39 = pd.DataFrame(fold_gain_rows39)

else:
    best39 = None
    best_lambda39 = 0.0
    best_oof39 = anchor_oof_39a.copy()
    best_test39 = anchor_test_39a.copy()
    best_mse39 = anchor_mse_39a
    best_gain39 = 0.0
    fold_gains39 = pd.DataFrame()

screen_path39 = "model_results/account39a_screen.csv"
fold_diag_path39 = "model_results/account39a_fold_coverage_diag.csv"
fold_gains_path39 = "model_results/account39a_best_fold_gains.csv"
oof_path39 = "model_results/oof_account39a_best.csv"
testpred_path39 = "model_results/testpred_account39a_best.csv"
submission_path39 = "submission_account39a_best.csv"
raw_oof_path39 = "model_results/account39a_raw_oof_reconstruction.csv"
raw_test_path39 = "model_results/account39a_raw_test_reconstruction.csv"

screen39.to_csv(screen_path39, index=False)
fold_diag39.to_csv(fold_diag_path39, index=False)
fold_gains39.to_csv(fold_gains_path39, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_39a),
    TARGET_COL: y_arr_39a,
    "anchor_pred": anchor_oof_39a,
    "accounting_pred": acct_oof39,
    "accounting_covered": covered_oof39.astype(int),
    "accounting_n_methods": acct_oof_n_methods39,
    "accounting_method": acct_oof_methods39,
    "pred_clipped": best_oof39,
}).to_csv(oof_path39, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_39a),
    TARGET_COL: y_arr_39a,
    "anchor_pred": anchor_oof_39a,
    "accounting_pred": acct_oof39,
    "accounting_covered": covered_oof39.astype(int),
    "accounting_n_methods": acct_oof_n_methods39,
    "accounting_method": acct_oof_methods39,
}).to_csv(raw_oof_path39, index=False)

pd.DataFrame({
    ID_COL: test_ids_39a,
    "anchor_pred": anchor_test_39a,
    "accounting_pred": acct_test39,
    "accounting_covered": covered_test39.astype(int),
    "accounting_n_methods": acct_test_n_methods39,
    "accounting_method": acct_test_methods39,
    TARGET_COL: best_test39,
}).to_csv(testpred_path39, index=False)

pd.DataFrame({
    ID_COL: test_ids_39a,
    "anchor_pred": anchor_test_39a,
    "accounting_pred": acct_test39,
    "accounting_covered": covered_test39.astype(int),
    "accounting_n_methods": acct_test_n_methods39,
    "accounting_method": acct_test_methods39,
}).to_csv(raw_test_path39, index=False)

pd.DataFrame({
    ID_COL: test_ids_39a,
    TARGET_COL: best_test39,
}).to_csv(submission_path39, index=False)

# ------------------------------------------------------------
# 10. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("39A subgroup accounting diagnostic complete")
print("=" * 90)

print("\nSelected accounting relationships")
print("---------------------------------")
for all_s, a_s, b_s in selected_pair_tuples39:
    print(f"{all_s} ≈ {a_s} + {b_s}")

print("\nOOF screen")
print("----------")
display(screen39)

print("\nBest 39A candidate")
print("------------------")
if best39 is not None:
    print(best39.to_string())
print(f"\nBest 39A OOF MSE: {best_mse39:.6f}")
print(f"Gain vs anchor:   {best_gain39:.6f}")
if len(fold_gains39) > 0:
    print(f"Min fold gain:    {fold_gains39['gain_vs_anchor'].min():.6f}")

print("\nBest candidate fold gains")
print("-------------------------")
print(fold_gains39.to_string(index=False))

print("\nCoverage summary")
print("----------------")
print("OOF covered rows:", int(covered_oof39.sum()), "of", n_train_39a, "| rate:", round(float(covered_oof39.mean()), 6))
print("Test covered rows:", int(covered_test39.sum()), "of", n_test_39a, "| rate:", round(float(covered_test39.mean()), 6))

if int(covered_oof39.sum()) > 0:
    print("\nAccounting-only performance on covered OOF rows")
    print("-----------------------------------------------")
    print("Anchor MSE on covered rows:", float(mean_squared_error(y_arr_39a[covered_oof39], anchor_oof_39a[covered_oof39])))
    print("Accounting MSE on covered rows:", float(mean_squared_error(y_arr_39a[covered_oof39], acct_oof39[covered_oof39])))
    print("Gain on covered rows:", float(mean_squared_error(y_arr_39a[covered_oof39], anchor_oof_39a[covered_oof39]) - mean_squared_error(y_arr_39a[covered_oof39], acct_oof39[covered_oof39])))

print("\nSaved files")
print("-----------")
print(pair_path39)
print(screen_path39)
print(fold_diag_path39)
print(fold_gains_path39)
print(raw_oof_path39)
print(raw_test_path39)
print(oof_path39)
print(testpred_path39)
print(submission_path39)

print("\nDecision rule")
print("-------------")
if best_gain39 >= 5.0:
    print("Material accounting signal found. Make this the main branch and expand to more accounting identities.")
elif best_gain39 >= 1.0:
    print("Some accounting signal found. Expand accounting identities and segment-specific use before moving on.")
elif int(covered_test39.sum()) > int(covered_oof39.sum()) * 0.5 and int(covered_test39.sum()) > 1000:
    print("OOF gain is small, but test coverage is substantial. Inspect raw accounting predictions before dismissing.")
else:
    print("No material accounting signal in this first sibling-reconstruction pass.")

39A. Subgroup accounting / sibling-row reconstruction diagnostic

Anchor check
------------
Anchor OOF column: pred_clipped
Anchor test column: PERCENT_PROFICIENT
Anchor OOF MSE: 77.049866

Subgroup values
---------------
                 SUBGROUP_NAME  train_rows  test_rows
                  All Students       36711      12428
    Economically Disadvantaged       25137       8487
                        Female       29110       9777
                          Male       29363       9589
Not Economically Disadvantaged       24600       8026

Top candidate subgroup accounting relationships
-----------------------------------------------
                  all_subgroup                     part_a                         part_b  complete_groups  n_match_rate      n_mae  k_match_rate     k_mae  name_has_all
                  All Students                     Female                           Male            16014      0.999750   0.004559      0.999750  0.111902             1
                  A

,candidate,covered_rows,coverage_rate,lambda_accounting,oof_mse,gain_vs_anchor
0,accounting_blend_covered,53786,0.37114,0.992,55.642780,21.407085
1,accounting_replace_covered,53786,0.37114,1.000,55.644047,21.405819



Best 39A candidate
------------------
candidate            accounting_blend_covered
covered_rows                            53786
coverage_rate                         0.37114
lambda_accounting                       0.992
oof_mse                              55.64278
gain_vs_anchor                      21.407085

Best 39A OOF MSE: 55.642780
Gain vs anchor:   21.407085
Min fold gain:    20.800014

Best candidate fold gains
-------------------------
 fold  anchor_mse  candidate_mse  gain_vs_anchor  covered_rows
    1   78.427116      55.819286       22.607830         10837
    2   74.554970      53.740479       20.814491         10622
    3   78.135284      56.340160       21.795124         10838
    4   76.396492      55.596478       20.800014         10796
    5   77.735397      56.717499       21.017899         10693

Coverage summary
----------------
OOF covered rows: 53786 of 144921 | rate: 0.37114
Test covered rows: 27298 of 48307 | rate: 0.565094

Accounting-only performance on c

### 39A. Subgroup Accounting / Sibling-Row Reconstruction

This section tested a structural reconstruction approach rather than another generic regression model. The idea was to use known algebraic relationships among subgroup rows within the same `SCHOOL × ASSESSMENT_NAME` group.

Because the target is a percentage and each row includes `N_STUDENTS`, each labeled row can be converted into an approximate proficient count:

`proficient_count ≈ round(PERCENT_PROFICIENT / 100 × N_STUDENTS)`

The diagnostic discovered two nearly exact subgroup accounting identities:

`All Students ≈ Female + Male`

`All Students ≈ Economically Disadvantaged + Not Economically Disadvantaged`

These identities were extremely reliable in the training data. For `All Students ≈ Female + Male`, the student-count match rate was `0.999750`. For `All Students ≈ Economically Disadvantaged + Not Economically Disadvantaged`, the student-count match rate was `1.000000`.

Using only fold-training sibling rows, the accounting reconstruction covered `53,786` OOF rows, or `37.1%` of the training set. On those covered rows, the current anchor had MSE `58.274662`, while the accounting reconstruction had MSE only `0.598845`.

The full OOF result was:

- 33A anchor OOF MSE: `77.049866`
- 39A accounting OOF MSE: `55.642780`
- Gain versus anchor: `21.407085`
- Minimum fold gain: `20.800014`

The test set coverage was even higher: `27,298` of `48,307` rows, or `56.5%`.

Conclusion: this is the first material breakthrough after the target-encoding branch. The large gain shows that many rows are not merely predictable statistically; they are nearly reconstructable from sibling subgroup rows. This likely explains how much lower public leaderboard scores are possible. The accounting branch should become the main modeling direction.

In [26]:
# ============================================================
# Validate 39A accounting submission
# ============================================================

import pandas as pd
import numpy as np

submission_path = "submission_account39a_best.csv"

sub = pd.read_csv(submission_path)

print("Submission path:", submission_path)
print("Shape:", sub.shape)
print("Columns:", list(sub.columns))

assert sub.shape[0] == 48307, f"Expected 48307 rows, got {sub.shape[0]}"
assert list(sub.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"], list(sub.columns)
assert sub["ASSESSMENT_ID"].notna().all(), "Missing ASSESSMENT_ID"
assert sub["PERCENT_PROFICIENT"].notna().all(), "Missing predictions"
assert np.isfinite(sub["PERCENT_PROFICIENT"]).all(), "Non-finite predictions"
assert sub["PERCENT_PROFICIENT"].between(0, 100).all(), "Predictions outside [0, 100]"

print("\nPrediction summary:")
print(sub["PERCENT_PROFICIENT"].describe())

print("\nValidation passed. Submit this file:")
print(submission_path)

Submission path: submission_account39a_best.csv
Shape: (48307, 2)
Columns: ['ASSESSMENT_ID', 'PERCENT_PROFICIENT']

Prediction summary:
count    48307.000000
mean        54.184972
std         25.610511
min          0.001630
25%         33.641388
50%         52.689290
75%         75.103700
max         99.999970
Name: PERCENT_PROFICIENT, dtype: float64

Validation passed. Submit this file:
submission_account39a_best.csv


# Algorithmically, 39A does this for each OOF fold:

For each fold:
    Use only the fold-training rows as known labeled rows.
    Build lookup table by SCHOOL × ASSESSMENT_NAME × SUBGROUP_NAME.
    
    For each validation row:
        If target subgroup is All Students:
            If Female and Male are known:
                All = Female + Male
            If Economically Disadvantaged and Not Economically Disadvantaged are known:
                All = ED + Not ED
        
        If target subgroup is Female:
            If All Students and Male are known:
                Female = All - Male
        
        If target subgroup is Male:
            If All Students and Female are known:
                Male = All - Female
        
        If target subgroup is Economically Disadvantaged:
            If All Students and Not Economically Disadvantaged are known:
                ED = All - Not ED
        
        If target subgroup is Not Economically Disadvantaged:
            If All Students and Economically Disadvantaged are known:
                Not ED = All - ED
        
        Convert reconstructed proficient count back to percent.

In [27]:
# ============================================================
# 39B. Constrained subgroup accounting solver + prior/fallback blending
# ============================================================
#
# Builds on 39A:
#   39A directly reconstructs a row when its sibling rows are known.
#
# 39B generalizes this:
#   For each SCHOOL x ASSESSMENT_NAME group, solve unknown subgroup proficient
#   counts subject to accounting equations:
#
#       All Students = Female + Male
#       All Students = Economically Disadvantaged + Not Economically Disadvantaged
#
# Known training-fold rows are fixed.
# Unknown validation/test rows are variables with prior predictions from a base model.
#
# This can cover cases where multiple sibling rows are unknown simultaneously.
#
# It also tests multiple prior/fallback models:
#   - 33A anchor
#   - KNN 35A if present
#   - NN 37B if present
#   - step 36A if present
#   - prior blend artifacts if present
#
# Final prediction:
#   base_prediction + lambda * (solver_prediction - base_prediction)
# only on rows solved by the accounting system.
# ============================================================

import os
import gc
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

try:
    from scipy.optimize import lsq_linear
    HAVE_LSQ_LINEAR_39B = True
except Exception:
    HAVE_LSQ_LINEAR_39B = False

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("39B. Constrained subgroup accounting solver + prior/fallback blending")
print("=" * 90)
print("Have scipy.optimize.lsq_linear:", HAVE_LSQ_LINEAR_39B)

# ------------------------------------------------------------
# 1. Required checks
# ------------------------------------------------------------

required_39b = ["raw_train_te", "raw_test_te", "y_train"]

missing_39b = [x for x in required_39b if x not in globals()]
if missing_39b:
    raise ValueError(f"Missing required objects: {missing_39b}")

needed_cols_39b = ["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]
for c in needed_cols_39b:
    if c not in raw_train_te.columns or c not in raw_test_te.columns:
        raise ValueError(f"Missing required column in raw_train_te/raw_test_te: {c}")

y_arr_39b = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_39b = len(y_arr_39b)
n_test_39b = len(raw_test_te)

if len(raw_train_te) != n_train_39b:
    raise ValueError("raw_train_te and y_train length mismatch.")

# ------------------------------------------------------------
# 2. Prediction loading helpers
# ------------------------------------------------------------

def load_oof_pred_39b(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_col = None
    for c in ["pred_clipped", "prediction", "pred", "oof_pred", TARGET_COL]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            # Avoid accidentally using target if pred_clipped exists.
            if c == TARGET_COL and "pred_clipped" in df.columns:
                continue
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find OOF prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}, column {pred_col}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_y_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_y_diff > 1e-5:
            raise ValueError(f"OOF target mismatch for {path}. Max difference: {max_y_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col


def load_test_pred_39b(path):
    df = pd.read_csv(path)

    if len(df) != n_test_39b:
        raise ValueError(f"Test row mismatch for {path}: got {len(df)}, expected {n_test_39b}")

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"Could not find test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}, column {pred_col}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no global test_ids found.")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col


candidate_paths_39b = [
    {
        "name": "anchor33a",
        "oof_path": "model_results/oof_seg33a_arcsine_adaptive.csv",
        "test_path": "model_results/testpred_seg33a_arcsine_adaptive.csv",
    },
    {
        "name": "knn35a",
        "oof_path": "model_results/oof_knn35a_local_residual_best.csv",
        "test_path": "model_results/testpred_knn35a_local_residual_best.csv",
    },
    {
        "name": "nn37b_rank1",
        "oof_path": "model_results/oof_nn37b38b_cpu_safe_rank1.csv",
        "test_path": "model_results/testpred_nn37b38b_cpu_safe_rank1.csv",
    },
    {
        "name": "step36a",
        "oof_path": "model_results/oof_step36a_residual_bins_best.csv",
        "test_path": "model_results/testpred_step36a_residual_bins_best.csv",
    },
    {
        "name": "blend33c",
        "oof_path": "model_results/oof_blend33c_33a_huber33b.csv",
        "test_path": "model_results/testpred_blend33c_33a_huber33b.csv",
    },
    {
        "name": "blend33b_oldanchor",
        "oof_path": "model_results/oof_blend33b_current_huber_te.csv",
        "test_path": "model_results/testpred_blend33b_current_huber_te.csv",
    },
]

base_preds_39b = {}

test_ids_39b = None

for cand in candidate_paths_39b:
    name = cand["name"]
    oof_path = cand["oof_path"]
    test_path = cand["test_path"]

    if not Path(oof_path).exists() or not Path(test_path).exists():
        print(f"Skipping missing base candidate: {name}")
        continue

    try:
        oof_pred, oof_col = load_oof_pred_39b(oof_path, y_arr_39b)
        test_pred, ids, test_col = load_test_pred_39b(test_path)

        if test_ids_39b is None:
            test_ids_39b = ids
        else:
            if len(ids) != len(test_ids_39b):
                raise ValueError("test id length mismatch")

        base_preds_39b[name] = {
            "oof": oof_pred,
            "test": test_pred,
            "oof_path": oof_path,
            "test_path": test_path,
            "oof_col": oof_col,
            "test_col": test_col,
            "oof_mse": float(mean_squared_error(y_arr_39b, oof_pred)),
        }

        print(f"Loaded base candidate {name:24s} | OOF MSE {base_preds_39b[name]['oof_mse']:.6f}")

    except Exception as e:
        print(f"Skipping base candidate {name} due to error: {repr(e)}")

if "anchor33a" not in base_preds_39b:
    raise ValueError("anchor33a base candidate is required but was not loaded.")

anchor_oof_39b = base_preds_39b["anchor33a"]["oof"]
anchor_test_39b = base_preds_39b["anchor33a"]["test"]
anchor_mse_39b = float(mean_squared_error(y_arr_39b, anchor_oof_39b))

print("\nBase candidates")
print("---------------")
for name, d in base_preds_39b.items():
    print(f"{name:24s} OOF MSE = {d['oof_mse']:.6f} | gain vs 33A = {anchor_mse_39b - d['oof_mse']:.6f}")

# ------------------------------------------------------------
# 3. Build compact row frames
# ------------------------------------------------------------

def clean_str_39b(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def safe_n_39b(x):
    return (
        pd.to_numeric(x, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=np.float64)
    )

train39b = pd.DataFrame({
    "row_index": np.arange(n_train_39b),
    "SCHOOL": clean_str_39b(raw_train_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_str_39b(raw_train_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str_39b(raw_train_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_n_39b(raw_train_te["N_STUDENTS"]),
    TARGET_COL: y_arr_39b,
})

test39b = pd.DataFrame({
    "row_index": np.arange(n_test_39b),
    "SCHOOL": clean_str_39b(raw_test_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_str_39b(raw_test_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str_39b(raw_test_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_n_39b(raw_test_te["N_STUDENTS"]),
})

train39b["group_key"] = train39b["SCHOOL"] + "||" + train39b["ASSESSMENT_NAME"]
test39b["group_key"] = test39b["SCHOOL"] + "||" + test39b["ASSESSMENT_NAME"]

train39b["N_STUDENTS"] = np.where(
    np.isfinite(train39b["N_STUDENTS"]) & (train39b["N_STUDENTS"] > 0),
    train39b["N_STUDENTS"],
    np.nan,
)

test39b["N_STUDENTS"] = np.where(
    np.isfinite(test39b["N_STUDENTS"]) & (test39b["N_STUDENTS"] > 0),
    test39b["N_STUDENTS"],
    np.nan,
)

train39b["prof_count"] = np.rint(
    np.clip(train39b[TARGET_COL].to_numpy(dtype=np.float64), 0, 100)
    / 100.0
    * train39b["N_STUDENTS"].to_numpy(dtype=np.float64)
)

train39b["prof_count"] = np.clip(
    train39b["prof_count"],
    0,
    train39b["N_STUDENTS"],
)

print("\nSubgroup values")
print("---------------")
print(
    pd.concat([
        train39b["SUBGROUP_NAME"].value_counts().rename("train_rows"),
        test39b["SUBGROUP_NAME"].value_counts().rename("test_rows"),
    ], axis=1).fillna(0).astype(int).to_string()
)

# ------------------------------------------------------------
# 4. Accounting identities
# ------------------------------------------------------------

subgroups_available_39b = set(train39b["SUBGROUP_NAME"].unique()).union(set(test39b["SUBGROUP_NAME"].unique()))

candidate_identities_39b = [
    ("All Students", "Female", "Male"),
    ("All Students", "Economically Disadvantaged", "Not Economically Disadvantaged"),
]

accounting_identities_39b = [
    tup for tup in candidate_identities_39b
    if all(s in subgroups_available_39b for s in tup)
]

if len(accounting_identities_39b) == 0:
    raise ValueError("No accounting identities found in available subgroup names.")

print("\nAccounting identities used")
print("--------------------------")
for all_s, a_s, b_s in accounting_identities_39b:
    print(f"{all_s} = {a_s} + {b_s}")

def n_tolerance_39b(n):
    if not np.isfinite(n):
        return 1.0
    return max(1.0, 0.02 * float(n))

# ------------------------------------------------------------
# 5. Known lookup and solver
# ------------------------------------------------------------

def build_known_lookup_39b(df, row_indices):
    sub = df.iloc[row_indices]

    lookup = {}

    for group_key, g in sub.groupby("group_key", sort=False):
        group_dict = {}

        for subgroup, sg in g.groupby("SUBGROUP_NAME", sort=False):
            n_vals = sg["N_STUDENTS"].to_numpy(dtype=np.float64)
            k_vals = sg["prof_count"].to_numpy(dtype=np.float64)

            good = np.isfinite(n_vals) & (n_vals > 0) & np.isfinite(k_vals)

            if not good.any():
                continue

            # There should normally be one row per subgroup in a group.
            # If duplicates occur, average the counts defensively.
            n = float(np.mean(n_vals[good]))
            k = float(np.mean(k_vals[good]))

            group_dict[str(subgroup)] = {
                "n": n,
                "k": float(np.clip(k, 0, n)),
            }

        if len(group_dict) > 0:
            lookup[str(group_key)] = group_dict

    return lookup

def solve_one_group_39b(query_group_df, known_group, base_prior_pct, identities, equation_weight, prior_weight):
    # query_group_df is reset-indexed within one group and contains local_pos.
    n_query = len(query_group_df)

    pred_pct = np.full(n_query, np.nan, dtype=np.float32)
    touched = np.zeros(n_query, dtype=bool)
    eq_count_used = 0

    # Create at most one variable per subgroup. Duplicates are rare; extras remain unsolved.
    subgroup_to_var = {}
    var_to_local = []
    var_n = []
    var_prior_k = []

    for j, row in query_group_df.iterrows():
        subgroup = str(row["SUBGROUP_NAME"])
        n = float(row["N_STUDENTS"])
        prior_pct = float(base_prior_pct[j])

        if not np.isfinite(n) or n <= 0 or not np.isfinite(prior_pct):
            continue

        if subgroup in subgroup_to_var:
            continue

        v = len(var_to_local)
        subgroup_to_var[subgroup] = v
        var_to_local.append(j)
        var_n.append(n)
        var_prior_k.append(float(np.clip(prior_pct, 0, 100) / 100.0 * n))

    n_vars = len(var_to_local)

    if n_vars == 0:
        return pred_pct, touched, eq_count_used

    def subgroup_present(s):
        return (s in subgroup_to_var) or (known_group is not None and s in known_group)

    def subgroup_n(s):
        if s in subgroup_to_var:
            return var_n[subgroup_to_var[s]]
        return known_group[s]["n"]

    def subgroup_known_k(s):
        return known_group[s]["k"]

    A_rows = []
    b_vals = []

    sqrt_prior = float(np.sqrt(prior_weight))
    sqrt_eq = float(np.sqrt(equation_weight))

    # Prior rows.
    for v in range(n_vars):
        row = np.zeros(n_vars, dtype=np.float64)
        row[v] = sqrt_prior
        A_rows.append(row)
        b_vals.append(sqrt_prior * var_prior_k[v])

    touched_vars = set()

    # Equation rows.
    for all_s, a_s, b_s in identities:
        if not (subgroup_present(all_s) and subgroup_present(a_s) and subgroup_present(b_s)):
            continue

        n_all = subgroup_n(all_s)
        n_a = subgroup_n(a_s)
        n_b = subgroup_n(b_s)

        if not (np.isfinite(n_all) and np.isfinite(n_a) and np.isfinite(n_b)):
            continue

        if abs(n_all - (n_a + n_b)) > n_tolerance_39b(n_all):
            continue

        # all - a - b = 0
        signs = {
            all_s: 1.0,
            a_s: -1.0,
            b_s: -1.0,
        }

        row = np.zeros(n_vars, dtype=np.float64)
        known_sum = 0.0
        vars_in_eq = []

        for s, sign in signs.items():
            if s in subgroup_to_var:
                v = subgroup_to_var[s]
                row[v] += sign
                vars_in_eq.append(v)
            else:
                known_sum += sign * subgroup_known_k(s)

        if len(vars_in_eq) == 0:
            continue

        A_rows.append(sqrt_eq * row)
        b_vals.append(sqrt_eq * (-known_sum))
        eq_count_used += 1

        for v in vars_in_eq:
            touched_vars.add(v)

    if eq_count_used == 0 or len(touched_vars) == 0:
        return pred_pct, touched, eq_count_used

    A = np.vstack(A_rows)
    b = np.asarray(b_vals, dtype=np.float64)

    lower = np.zeros(n_vars, dtype=np.float64)
    upper = np.asarray(var_n, dtype=np.float64)

    try:
        if HAVE_LSQ_LINEAR_39B:
            sol = lsq_linear(
                A,
                b,
                bounds=(lower, upper),
                method="trf",
                lsmr_tol="auto",
                max_iter=100,
            )
            x = sol.x
        else:
            x, *_ = np.linalg.lstsq(A, b, rcond=None)
            x = np.clip(x, lower, upper)

    except Exception:
        x, *_ = np.linalg.lstsq(A, b, rcond=None)
        x = np.clip(x, lower, upper)

    for v in touched_vars:
        local_j = var_to_local[v]
        n = var_n[v]

        if np.isfinite(n) and n > 0:
            p = 100.0 * float(x[v]) / n
            pred_pct[local_j] = np.float32(np.clip(p, 0, 100))
            touched[local_j] = True

    return pred_pct, touched, eq_count_used

def solve_many_39b(query_df, known_lookup, base_prior_pct, identities, equation_weight, prior_weight):
    query_df = query_df.reset_index(drop=True).copy()
    query_df["local_pos"] = np.arange(len(query_df))

    base_prior_pct = np.asarray(base_prior_pct, dtype=np.float32).reshape(-1)

    pred = np.full(len(query_df), np.nan, dtype=np.float32)
    touched = np.zeros(len(query_df), dtype=bool)
    eq_counts = np.zeros(len(query_df), dtype=np.float32)

    for group_key, g in query_df.groupby("group_key", sort=False):
        local_positions = g["local_pos"].to_numpy(dtype=np.int64)
        known_group = known_lookup.get(str(group_key), {})

        pred_g, touched_g, eq_count_g = solve_one_group_39b(
            query_group_df=g.reset_index(drop=True),
            known_group=known_group,
            base_prior_pct=base_prior_pct[local_positions],
            identities=identities,
            equation_weight=equation_weight,
            prior_weight=prior_weight,
        )

        pred[local_positions] = pred_g
        touched[local_positions] = touched_g
        eq_counts[local_positions] = eq_count_g

    return pred, touched, eq_counts

# ------------------------------------------------------------
# 6. Configs
# ------------------------------------------------------------

configs_39b = []

for base_name in base_preds_39b.keys():
    for eq_weight in [100.0, 1000.0, 10000.0, 100000.0]:
        configs_39b.append({
            "name": f"account39b_solver_{base_name}_eq{int(eq_weight)}",
            "base_name": base_name,
            "equation_weight": float(eq_weight),
            "prior_weight": 1.0,
        })

print("\n39B configs")
print("-----------")
print("Number of configs:", len(configs_39b))
for cfg in configs_39b:
    print(cfg["name"])

# ------------------------------------------------------------
# 7. OOF solver loop
# ------------------------------------------------------------

folds39b = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_39b))
)

solver_oof_39b = {
    cfg["name"]: np.full(n_train_39b, np.nan, dtype=np.float32)
    for cfg in configs_39b
}

solver_touched_39b = {
    cfg["name"]: np.zeros(n_train_39b, dtype=bool)
    for cfg in configs_39b
}

fold_rows_39b = []

for cfg in configs_39b:
    name = cfg["name"]
    base_name = cfg["base_name"]
    base_oof = base_preds_39b[base_name]["oof"]

    print("\n" + "=" * 90)
    print(f"Running OOF solver config: {name}")
    print("=" * 90)

    for fold_num, (tr_idx, va_idx) in enumerate(folds39b, start=1):
        known_lookup_fold = build_known_lookup_39b(train39b, tr_idx)

        query_fold = train39b.iloc[va_idx].reset_index(drop=True)
        base_prior_fold = base_oof[va_idx]

        pred_fold, touched_fold, eq_counts_fold = solve_many_39b(
            query_df=query_fold,
            known_lookup=known_lookup_fold,
            base_prior_pct=base_prior_fold,
            identities=accounting_identities_39b,
            equation_weight=cfg["equation_weight"],
            prior_weight=cfg["prior_weight"],
        )

        solver_oof_39b[name][va_idx] = pred_fold
        solver_touched_39b[name][va_idx] = touched_fold

        covered = np.isfinite(pred_fold) & touched_fold
        n_cov = int(covered.sum())

        fold_anchor_mse = float(mean_squared_error(y_arr_39b[va_idx], anchor_oof_39b[va_idx]))
        fold_base_mse = float(mean_squared_error(y_arr_39b[va_idx], base_prior_fold))

        if n_cov > 0:
            y_cov = y_arr_39b[va_idx][covered]
            anchor_cov = anchor_oof_39b[va_idx][covered]
            base_cov = base_prior_fold[covered]
            solver_cov = pred_fold[covered]

            anchor_mse_cov = float(mean_squared_error(y_cov, anchor_cov))
            base_mse_cov = float(mean_squared_error(y_cov, base_cov))
            solver_mse_cov = float(mean_squared_error(y_cov, solver_cov))
        else:
            anchor_mse_cov = np.nan
            base_mse_cov = np.nan
            solver_mse_cov = np.nan

        fold_rows_39b.append({
            "config": name,
            "base_name": base_name,
            "fold": fold_num,
            "n_valid_rows": len(va_idx),
            "covered_rows": n_cov,
            "coverage_rate": n_cov / len(va_idx),
            "fold_anchor_mse": fold_anchor_mse,
            "fold_base_mse": fold_base_mse,
            "anchor_mse_on_covered": anchor_mse_cov,
            "base_mse_on_covered": base_mse_cov,
            "solver_mse_on_covered": solver_mse_cov,
            "equation_weight": cfg["equation_weight"],
            "prior_weight": cfg["prior_weight"],
        })

    cov_total = int(np.isfinite(solver_oof_39b[name]).sum())
    print(f"Completed {name} | OOF solved rows: {cov_total}")

fold_diag39b = pd.DataFrame(fold_rows_39b)

# ------------------------------------------------------------
# 8. OOF lambda scan
# ------------------------------------------------------------

lambda_grid_39b = np.unique(
    np.concatenate([
        np.linspace(-0.50, 1.50, 501),
        np.array([0.0, 0.25, 0.50, 0.75, 0.992, 1.0]),
    ])
)

screen_rows_39b = []

for cfg in configs_39b:
    name = cfg["name"]
    base_name = cfg["base_name"]
    base_oof = base_preds_39b[base_name]["oof"]
    base_mse = base_preds_39b[base_name]["oof_mse"]

    solver_oof = solver_oof_39b[name]
    covered = np.isfinite(solver_oof) & solver_touched_39b[name]

    if int(covered.sum()) == 0:
        continue

    best_row = None

    for lam in lambda_grid_39b:
        pred = base_oof.copy()
        pred[covered] = base_oof[covered] + float(lam) * (solver_oof[covered] - base_oof[covered])
        pred = np.clip(pred, 0, 100).astype(np.float32)

        mse = float(mean_squared_error(y_arr_39b, pred))

        row = {
            "config": name,
            "base_name": base_name,
            "equation_weight": cfg["equation_weight"],
            "prior_weight": cfg["prior_weight"],
            "covered_rows": int(covered.sum()),
            "coverage_rate": float(covered.mean()),
            "lambda_solver": float(lam),
            "base_oof_mse": base_mse,
            "oof_mse": mse,
            "gain_vs_33a_anchor": anchor_mse_39b - mse,
            "gain_vs_base": base_mse - mse,
        }

        if best_row is None or row["oof_mse"] < best_row["oof_mse"]:
            best_row = row

    screen_rows_39b.append(best_row)

screen39b = pd.DataFrame(screen_rows_39b)

if len(screen39b) == 0:
    raise ValueError("No 39B solver candidates covered any rows.")

screen39b = screen39b.sort_values("oof_mse").reset_index(drop=True)

best39b = screen39b.iloc[0]
best_cfg_name39b = best39b["config"]
best_base_name39b = best39b["base_name"]
best_lambda39b = float(best39b["lambda_solver"])

best_base_oof39b = base_preds_39b[best_base_name39b]["oof"]
best_base_test39b = base_preds_39b[best_base_name39b]["test"]

best_solver_oof39b = solver_oof_39b[best_cfg_name39b]
best_covered_oof39b = np.isfinite(best_solver_oof39b) & solver_touched_39b[best_cfg_name39b]

best_oof39b = best_base_oof39b.copy()
best_oof39b[best_covered_oof39b] = (
    best_base_oof39b[best_covered_oof39b]
    + best_lambda39b * (best_solver_oof39b[best_covered_oof39b] - best_base_oof39b[best_covered_oof39b])
)
best_oof39b = np.clip(best_oof39b, 0, 100).astype(np.float32)

best_mse39b = float(mean_squared_error(y_arr_39b, best_oof39b))
best_gain39b = anchor_mse_39b - best_mse39b

# ------------------------------------------------------------
# 9. Test solve for best config
# ------------------------------------------------------------

best_cfg39b = None
for cfg in configs_39b:
    if cfg["name"] == best_cfg_name39b:
        best_cfg39b = cfg
        break

if best_cfg39b is None:
    raise ValueError("Could not recover best config.")

known_lookup_full39b = build_known_lookup_39b(train39b, np.arange(n_train_39b))

solver_test39b, touched_test39b, eq_counts_test39b = solve_many_39b(
    query_df=test39b.reset_index(drop=True),
    known_lookup=known_lookup_full39b,
    base_prior_pct=best_base_test39b,
    identities=accounting_identities_39b,
    equation_weight=best_cfg39b["equation_weight"],
    prior_weight=best_cfg39b["prior_weight"],
)

covered_test39b = np.isfinite(solver_test39b) & touched_test39b

best_test39b = best_base_test39b.copy()
best_test39b[covered_test39b] = (
    best_base_test39b[covered_test39b]
    + best_lambda39b * (solver_test39b[covered_test39b] - best_base_test39b[covered_test39b])
)
best_test39b = np.clip(best_test39b, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 10. Fold gains + comparison to 39A if available
# ------------------------------------------------------------

fold_gain_rows39b = []

for fold_num, (_, va_idx) in enumerate(folds39b, start=1):
    anchor_fold_mse = float(mean_squared_error(y_arr_39b[va_idx], anchor_oof_39b[va_idx]))
    base_fold_mse = float(mean_squared_error(y_arr_39b[va_idx], best_base_oof39b[va_idx]))
    cand_fold_mse = float(mean_squared_error(y_arr_39b[va_idx], best_oof39b[va_idx]))

    fold_gain_rows39b.append({
        "fold": fold_num,
        "anchor_mse": anchor_fold_mse,
        "base_mse": base_fold_mse,
        "candidate_mse": cand_fold_mse,
        "gain_vs_33a_anchor": anchor_fold_mse - cand_fold_mse,
        "gain_vs_base": base_fold_mse - cand_fold_mse,
        "covered_rows": int(best_covered_oof39b[va_idx].sum()),
    })

fold_gains39b = pd.DataFrame(fold_gain_rows39b)

account39a_mse = np.nan
account39a_gain = np.nan

if Path("model_results/oof_account39a_best.csv").exists():
    try:
        acct39a_oof, _ = load_oof_pred_39b("model_results/oof_account39a_best.csv", y_arr_39b)
        account39a_mse = float(mean_squared_error(y_arr_39b, acct39a_oof))
        account39a_gain = anchor_mse_39b - account39a_mse
    except Exception as e:
        print("Could not compare to 39A:", repr(e))

# ------------------------------------------------------------
# 11. Save artifacts
# ------------------------------------------------------------

screen_path39b = "model_results/account39b_solver_screen.csv"
fold_diag_path39b = "model_results/account39b_solver_fold_diag.csv"
fold_gains_path39b = "model_results/account39b_solver_best_fold_gains.csv"
oof_path39b = "model_results/oof_account39b_solver_best.csv"
testpred_path39b = "model_results/testpred_account39b_solver_best.csv"
submission_path39b = "submission_account39b_solver_best.csv"
raw_oof_path39b = "model_results/account39b_solver_raw_oof.csv"
raw_test_path39b = "model_results/account39b_solver_raw_test.csv"

screen39b.to_csv(screen_path39b, index=False)
fold_diag39b.to_csv(fold_diag_path39b, index=False)
fold_gains39b.to_csv(fold_gains_path39b, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_39b),
    TARGET_COL: y_arr_39b,
    "anchor33a_pred": anchor_oof_39b,
    "base_name": best_base_name39b,
    "base_pred": best_base_oof39b,
    "solver_pred": best_solver_oof39b,
    "solver_covered": best_covered_oof39b.astype(int),
    "lambda_solver": best_lambda39b,
    "pred_clipped": best_oof39b,
}).to_csv(oof_path39b, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_39b),
    TARGET_COL: y_arr_39b,
    "anchor33a_pred": anchor_oof_39b,
    "base_name": best_base_name39b,
    "base_pred": best_base_oof39b,
    "solver_pred": best_solver_oof39b,
    "solver_covered": best_covered_oof39b.astype(int),
}).to_csv(raw_oof_path39b, index=False)

pd.DataFrame({
    ID_COL: test_ids_39b,
    "anchor33a_pred": anchor_test_39b,
    "base_name": best_base_name39b,
    "base_pred": best_base_test39b,
    "solver_pred": solver_test39b,
    "solver_covered": covered_test39b.astype(int),
    "lambda_solver": best_lambda39b,
    TARGET_COL: best_test39b,
}).to_csv(testpred_path39b, index=False)

pd.DataFrame({
    ID_COL: test_ids_39b,
    "anchor33a_pred": anchor_test_39b,
    "base_name": best_base_name39b,
    "base_pred": best_base_test39b,
    "solver_pred": solver_test39b,
    "solver_covered": covered_test39b.astype(int),
}).to_csv(raw_test_path39b, index=False)

pd.DataFrame({
    ID_COL: test_ids_39b,
    TARGET_COL: best_test39b,
}).to_csv(submission_path39b, index=False)

# ------------------------------------------------------------
# 12. Validate submission
# ------------------------------------------------------------

sub39b = pd.read_csv(submission_path39b)

assert sub39b.shape[0] == n_test_39b, f"Expected {n_test_39b} rows, got {sub39b.shape[0]}"
assert list(sub39b.columns) == [ID_COL, TARGET_COL], list(sub39b.columns)
assert sub39b[ID_COL].notna().all(), "Missing IDs"
assert sub39b[TARGET_COL].notna().all(), "Missing predictions"
assert np.isfinite(sub39b[TARGET_COL]).all(), "Non-finite predictions"
assert sub39b[TARGET_COL].between(0, 100).all(), "Predictions outside [0,100]"

# ------------------------------------------------------------
# 13. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("39B constrained accounting solver complete")
print("=" * 90)

print("\nBest 39B candidate")
print("------------------")
print(best39b.to_string())

print("\nOOF comparison")
print("--------------")
print(f"33A anchor OOF MSE:       {anchor_mse_39b:.6f}")
if np.isfinite(account39a_mse):
    print(f"39A accounting OOF MSE:   {account39a_mse:.6f} | gain vs 33A {account39a_gain:.6f}")
print(f"39B best OOF MSE:         {best_mse39b:.6f}")
print(f"39B gain vs 33A:          {best_gain39b:.6f}")
print(f"39B gain vs best base:    {base_preds_39b[best_base_name39b]['oof_mse'] - best_mse39b:.6f}")
if np.isfinite(account39a_mse):
    print(f"39B gain vs 39A:          {account39a_mse - best_mse39b:.6f}")

print("\nCoverage")
print("--------")
print("Best OOF covered rows:", int(best_covered_oof39b.sum()), "of", n_train_39b, "| rate:", round(float(best_covered_oof39b.mean()), 6))
print("Best test covered rows:", int(covered_test39b.sum()), "of", n_test_39b, "| rate:", round(float(covered_test39b.mean()), 6))

if int(best_covered_oof39b.sum()) > 0:
    print("\nCovered-row performance")
    print("-----------------------")
    print("Anchor MSE on covered OOF:", float(mean_squared_error(y_arr_39b[best_covered_oof39b], anchor_oof_39b[best_covered_oof39b])))
    print("Base MSE on covered OOF:", float(mean_squared_error(y_arr_39b[best_covered_oof39b], best_base_oof39b[best_covered_oof39b])))
    print("Raw solver MSE on covered OOF:", float(mean_squared_error(y_arr_39b[best_covered_oof39b], best_solver_oof39b[best_covered_oof39b])))

print("\nBest candidate fold gains")
print("-------------------------")
print(fold_gains39b.to_string(index=False))
print("Min fold gain vs 33A:", float(fold_gains39b["gain_vs_33a_anchor"].min()))

print("\nTop 20 39B candidates")
print("---------------------")
display(screen39b.head(20))

print("\nSaved files")
print("-----------")
print(screen_path39b)
print(fold_diag_path39b)
print(fold_gains_path39b)
print(raw_oof_path39b)
print(raw_test_path39b)
print(oof_path39b)
print(testpred_path39b)
print(submission_path39b)

print("\nSubmission validation")
print("---------------------")
print("Shape:", sub39b.shape)
print("Prediction summary:")
print(sub39b[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
if np.isfinite(account39a_mse) and best_mse39b < account39a_mse - 1.0:
    print("39B materially improves over 39A. Prefer submitting 39B.")
elif np.isfinite(account39a_mse) and best_mse39b <= account39a_mse + 0.5 and int(covered_test39b.sum()) > 27298:
    print("39B is close to 39A and has higher test coverage. Consider submitting 39B before 39A.")
elif np.isfinite(account39a_mse):
    print("39B does not clearly beat 39A. Prefer submitting 39A first.")
else:
    print("No 39A comparison available. Use 39B if fold gains and coverage look strong.")

39B. Constrained subgroup accounting solver + prior/fallback blending
Have scipy.optimize.lsq_linear: True
Loaded base candidate anchor33a                | OOF MSE 77.049866
Loaded base candidate knn35a                   | OOF MSE 76.453072
Loaded base candidate nn37b_rank1              | OOF MSE 76.849442
Loaded base candidate step36a                  | OOF MSE 76.954796
Loaded base candidate blend33c                 | OOF MSE 76.859200
Loaded base candidate blend33b_oldanchor       | OOF MSE 76.951591

Base candidates
---------------
anchor33a                OOF MSE = 77.049866 | gain vs 33A = 0.000000
knn35a                   OOF MSE = 76.453072 | gain vs 33A = 0.596794
nn37b_rank1              OOF MSE = 76.849442 | gain vs 33A = 0.200424
step36a                  OOF MSE = 76.954796 | gain vs 33A = 0.095070
blend33c                 OOF MSE = 76.859200 | gain vs 33A = 0.190666
blend33b_oldanchor       OOF MSE = 76.951591 | gain vs 33A = 0.098274

Subgroup values
---------------
     

KeyboardInterrupt: 

### 39B. Constrained Subgroup Accounting Solver with Prior/Fallback Blending

This section extended the 39A subgroup accounting approach. In 39A, rows were reconstructed only when enough labeled sibling rows were directly available. In 39B, each `SCHOOL × ASSESSMENT_NAME` group was treated as a small constrained accounting system.

The solver used the subgroup identities:

`All Students = Female + Male`

`All Students = Economically Disadvantaged + Not Economically Disadvantaged`

Known training-fold rows were treated as fixed proficient counts. Unknown validation or test rows were solved using a prior prediction while enforcing the subgroup accounting equations. Several prior/fallback models were tested, including the 33A anchor, the 35A KNN residual model, the 37B neural-network model, the 36A step-function model, and previous blend artifacts.

The best 39B model used the `knn35a` prior, equation weight `100000.0`, prior weight `1.0`, and solver shrinkage `0.924`.

Results:

- 33A anchor OOF MSE: `77.049866`
- 39A accounting OOF MSE: `55.642780`
- 39B best OOF MSE: `53.720547`
- 39B gain versus 33A: `23.329319`
- 39B gain versus 39A: `1.922234`
- 39B gain versus its `knn35a` base: `22.732525`

The best 39B model covered `81,593` OOF rows, or `56.3%` of the training set, and `45,105` test rows, or `93.4%` of the test set. Fold gains versus the 33A anchor were large and stable:

- Fold 1 gain: `24.372295`
- Fold 2 gain: `23.028576`
- Fold 3 gain: `22.917091`
- Fold 4 gain: `22.853306`
- Fold 5 gain: `23.475281`

Conclusion: 39B is the strongest model so far and materially improves over both the original target-encoding anchor and the direct 39A accounting reconstruction. The large, stable OOF gain and very high test coverage indicate that subgroup accounting is the main signal source that previous generic regression models failed to exploit. The 39B submission file should be treated as the current primary Kaggle candidate.

### Public Leaderboard Check: 39B Accounting Solver

The 39B constrained accounting solver submission produced a public leaderboard MSE of `32.113`. This is a major improvement over the previous best submissions near the high 60s and confirms that the dominant missing signal was not another generic regression model, but the subgroup accounting structure.

The result supports the interpretation from the OOF analysis: many rows are reconstructable or partially reconstructable from sibling subgroup rows within the same `SCHOOL × ASSESSMENT_NAME` group. The identities

`All Students = Female + Male`

and

`All Students = Economically Disadvantaged + Not Economically Disadvantaged`

are the main source of the improvement.

Although 39B is the strongest OOF model so far, the public score suggests that a more conservative trust-tiered accounting hybrid may improve further. The next step is to combine 39A’s nearly exact direct reconstructions with 39B’s broader constrained solver coverage.

In [ ]:
# ============================================================
# 39C. Trust-tiered accounting hybrid + conservative blend screen
# ============================================================
#
# Purpose:
#   39A direct accounting is almost exact where it applies.
#   39B constrained solver covers many more rows, but is less exact per covered row.
#
# 39C tests:
#   Tier 1: rows covered by 39A direct reconstruction
#   Tier 2: rows covered by 39B solver but not 39A
#   Tier 3: rows covered by both, with overlap source screened
#   Tier 4: rows covered by neither, use best fallback/base prediction
#
# Final prediction form:
#   pred = base
#   pred[tier] = base[tier] + lambda_tier * (accounting_source[tier] - base[tier])
#
# Then it does a conservative final pairwise blend screen against 39A, 39B,
# and the main prior models to check whether stacking/blending improves OOF.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("39C. Trust-tiered accounting hybrid + conservative blend screen")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load core artifacts
# ------------------------------------------------------------

required_paths_39c = [
    "model_results/account39a_raw_oof_reconstruction.csv",
    "model_results/account39a_raw_test_reconstruction.csv",
    "model_results/oof_account39a_best.csv",
    "model_results/testpred_account39a_best.csv",
    "model_results/account39b_solver_raw_oof.csv",
    "model_results/account39b_solver_raw_test.csv",
    "model_results/oof_account39b_solver_best.csv",
    "model_results/testpred_account39b_solver_best.csv",
]

missing_paths_39c = [p for p in required_paths_39c if not Path(p).exists()]
if missing_paths_39c:
    raise FileNotFoundError(f"Missing required 39A/39B artifacts: {missing_paths_39c}")

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

n_train_39c = len(raw39a_oof)
n_test_39c = len(raw39a_test)

if len(raw39b_oof) != n_train_39c:
    raise ValueError("39A and 39B OOF row counts differ.")
if len(raw39b_test) != n_test_39c:
    raise ValueError("39A and 39B test row counts differ.")

if TARGET_COL in raw39a_oof.columns:
    y_arr_39c = raw39a_oof[TARGET_COL].to_numpy(dtype=np.float32)
elif "y_train" in globals():
    y_arr_39c = np.asarray(y_train, dtype=np.float32).reshape(-1)
else:
    raise ValueError("Could not recover y values from 39A raw OOF or y_train.")

if len(y_arr_39c) != n_train_39c:
    raise ValueError("Target length mismatch.")

# IDs for final submission.
if ID_COL in raw39b_test.columns:
    test_ids_39c = raw39b_test[ID_COL].to_numpy()
elif ID_COL in raw39a_test.columns:
    test_ids_39c = raw39a_test[ID_COL].to_numpy()
elif "test_ids" in globals():
    test_ids_39c = np.asarray(test_ids)
else:
    raise ValueError("Could not recover test IDs.")

# Core accounting predictions.
direct_oof_39c = raw39a_oof["accounting_pred"].to_numpy(dtype=np.float32)
direct_test_39c = raw39a_test["accounting_pred"].to_numpy(dtype=np.float32)

direct_cov_oof_39c = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test_39c = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)

solver_oof_39c = raw39b_oof["solver_pred"].to_numpy(dtype=np.float32)
solver_test_39c = raw39b_test["solver_pred"].to_numpy(dtype=np.float32)

solver_cov_oof_39c = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test_39c = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

# Defensive finite masks.
direct_cov_oof_39c = direct_cov_oof_39c & np.isfinite(direct_oof_39c)
direct_cov_test_39c = direct_cov_test_39c & np.isfinite(direct_test_39c)

solver_cov_oof_39c = solver_cov_oof_39c & np.isfinite(solver_oof_39c)
solver_cov_test_39c = solver_cov_test_39c & np.isfinite(solver_test_39c)

overlap_oof_39c = direct_cov_oof_39c & solver_cov_oof_39c
direct_only_oof_39c = direct_cov_oof_39c & ~solver_cov_oof_39c
solver_only_oof_39c = solver_cov_oof_39c & ~direct_cov_oof_39c
none_oof_39c = ~(direct_cov_oof_39c | solver_cov_oof_39c)

overlap_test_39c = direct_cov_test_39c & solver_cov_test_39c
direct_only_test_39c = direct_cov_test_39c & ~solver_cov_test_39c
solver_only_test_39c = solver_cov_test_39c & ~direct_cov_test_39c
none_test_39c = ~(direct_cov_test_39c | solver_cov_test_39c)

print("\nTier coverage")
print("-------------")
print("OOF overlap direct & solver:", int(overlap_oof_39c.sum()))
print("OOF direct only:            ", int(direct_only_oof_39c.sum()))
print("OOF solver only:            ", int(solver_only_oof_39c.sum()))
print("OOF neither:                ", int(none_oof_39c.sum()))
print("OOF total accounting union: ", int((direct_cov_oof_39c | solver_cov_oof_39c).sum()), "/", n_train_39c)

print("\nTest overlap direct & solver:", int(overlap_test_39c.sum()))
print("Test direct only:            ", int(direct_only_test_39c.sum()))
print("Test solver only:            ", int(solver_only_test_39c.sum()))
print("Test neither:                ", int(none_test_39c.sum()))
print("Test total accounting union: ", int((direct_cov_test_39c | solver_cov_test_39c).sum()), "/", n_test_39c)

# ------------------------------------------------------------
# 2. Load base/fallback candidates
# ------------------------------------------------------------

def load_oof_pred_39c(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_col = None
    for c in ["pred_clipped", "prediction", "pred", "oof_pred", TARGET_COL]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            if c == TARGET_COL and "pred_clipped" in df.columns:
                continue
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column found in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF prediction in {path}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_diff > 1e-5:
            raise ValueError(f"Target mismatch in {path}: max diff {max_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col

def load_test_pred_39c(path):
    df = pd.read_csv(path)

    if len(df) != n_test_39c:
        raise ValueError(f"Test row mismatch for {path}: got {len(df)}, expected {n_test_39c}")

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No test prediction column found in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test prediction in {path}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    else:
        ids = test_ids_39c

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col

base_candidate_paths_39c = [
    {
        "name": "anchor33a",
        "oof_path": "model_results/oof_seg33a_arcsine_adaptive.csv",
        "test_path": "model_results/testpred_seg33a_arcsine_adaptive.csv",
    },
    {
        "name": "knn35a",
        "oof_path": "model_results/oof_knn35a_local_residual_best.csv",
        "test_path": "model_results/testpred_knn35a_local_residual_best.csv",
    },
    {
        "name": "nn37b_rank1",
        "oof_path": "model_results/oof_nn37b38b_cpu_safe_rank1.csv",
        "test_path": "model_results/testpred_nn37b38b_cpu_safe_rank1.csv",
    },
    {
        "name": "step36a",
        "oof_path": "model_results/oof_step36a_residual_bins_best.csv",
        "test_path": "model_results/testpred_step36a_residual_bins_best.csv",
    },
    {
        "name": "blend33c",
        "oof_path": "model_results/oof_blend33c_33a_huber33b.csv",
        "test_path": "model_results/testpred_blend33c_33a_huber33b.csv",
    },
    {
        "name": "account39a_best",
        "oof_path": "model_results/oof_account39a_best.csv",
        "test_path": "model_results/testpred_account39a_best.csv",
    },
    {
        "name": "account39b_best",
        "oof_path": "model_results/oof_account39b_solver_best.csv",
        "test_path": "model_results/testpred_account39b_solver_best.csv",
    },
]

base_preds_39c = {}

for cand in base_candidate_paths_39c:
    name = cand["name"]
    oof_path = cand["oof_path"]
    test_path = cand["test_path"]

    if not Path(oof_path).exists() or not Path(test_path).exists():
        print(f"Skipping missing base candidate: {name}")
        continue

    try:
        oof_pred, oof_col = load_oof_pred_39c(oof_path, y_arr_39c)
        test_pred, ids, test_col = load_test_pred_39c(test_path)

        base_preds_39c[name] = {
            "oof": oof_pred,
            "test": test_pred,
            "oof_path": oof_path,
            "test_path": test_path,
            "oof_col": oof_col,
            "test_col": test_col,
            "oof_mse": float(mean_squared_error(y_arr_39c, oof_pred)),
        }

        print(f"Loaded {name:20s} | OOF MSE {base_preds_39c[name]['oof_mse']:.6f}")

    except Exception as e:
        print(f"Skipping {name} due to error: {repr(e)}")

if "anchor33a" not in base_preds_39c:
    raise ValueError("anchor33a candidate is required.")

if "account39a_best" not in base_preds_39c:
    raise ValueError("account39a_best candidate is required.")

if "account39b_best" not in base_preds_39c:
    raise ValueError("account39b_best candidate is required.")

anchor_mse_39c = base_preds_39c["anchor33a"]["oof_mse"]
mse39a_39c = base_preds_39c["account39a_best"]["oof_mse"]
mse39b_39c = base_preds_39c["account39b_best"]["oof_mse"]

# ------------------------------------------------------------
# 3. Fast tier-wise lambda optimizer
# ------------------------------------------------------------

lambda_grid_39c = np.unique(
    np.concatenate([
        np.linspace(-0.25, 1.50, 351),
        np.array([0.0, 0.25, 0.50, 0.75, 0.924, 0.992, 1.0]),
    ])
)

def best_lambda_for_tier_39c(y, base, source, mask, lambda_grid):
    mask = np.asarray(mask, dtype=bool)

    if int(mask.sum()) == 0:
        return {
            "lambda": 0.0,
            "sse": 0.0,
            "mse": np.nan,
            "n": 0,
            "base_mse": np.nan,
            "source_mse": np.nan,
        }

    y_m = y[mask].astype(np.float64)
    base_m = base[mask].astype(np.float64)
    src_m = source[mask].astype(np.float64)

    best = None

    for lam in lambda_grid:
        pred = base_m + float(lam) * (src_m - base_m)
        pred = np.clip(pred, 0, 100)
        err = y_m - pred
        sse = float(np.dot(err, err))

        if best is None or sse < best["sse"]:
            best = {
                "lambda": float(lam),
                "sse": sse,
            }

    base_err = y_m - base_m
    src_err = y_m - np.clip(src_m, 0, 100)

    best["mse"] = best["sse"] / len(y_m)
    best["n"] = int(len(y_m))
    best["base_mse"] = float(np.dot(base_err, base_err) / len(y_m))
    best["source_mse"] = float(np.dot(src_err, src_err) / len(y_m))

    return best

def compute_tiered_prediction_39c(
    base,
    overlap_source,
    lambda_overlap,
    lambda_direct_only,
    lambda_solver_only,
    clip=True,
):
    pred = base.copy().astype(np.float64)

    if int(overlap_oof_39c.sum()) > 0:
        pred[overlap_oof_39c] = (
            base[overlap_oof_39c]
            + lambda_overlap * (overlap_source[overlap_oof_39c] - base[overlap_oof_39c])
        )

    if int(direct_only_oof_39c.sum()) > 0:
        pred[direct_only_oof_39c] = (
            base[direct_only_oof_39c]
            + lambda_direct_only * (direct_oof_39c[direct_only_oof_39c] - base[direct_only_oof_39c])
        )

    if int(solver_only_oof_39c.sum()) > 0:
        pred[solver_only_oof_39c] = (
            base[solver_only_oof_39c]
            + lambda_solver_only * (solver_oof_39c[solver_only_oof_39c] - base[solver_only_oof_39c])
        )

    if clip:
        pred = np.clip(pred, 0, 100)

    return pred.astype(np.float32)

def compute_tiered_test_prediction_39c(
    base_test,
    overlap_source_test,
    lambda_overlap,
    lambda_direct_only,
    lambda_solver_only,
    clip=True,
):
    pred = base_test.copy().astype(np.float64)

    if int(overlap_test_39c.sum()) > 0:
        pred[overlap_test_39c] = (
            base_test[overlap_test_39c]
            + lambda_overlap * (overlap_source_test[overlap_test_39c] - base_test[overlap_test_39c])
        )

    if int(direct_only_test_39c.sum()) > 0:
        pred[direct_only_test_39c] = (
            base_test[direct_only_test_39c]
            + lambda_direct_only * (direct_test_39c[direct_only_test_39c] - base_test[direct_only_test_39c])
        )

    if int(solver_only_test_39c.sum()) > 0:
        pred[solver_only_test_39c] = (
            base_test[solver_only_test_39c]
            + lambda_solver_only * (solver_test_39c[solver_only_test_39c] - base_test[solver_only_test_39c])
        )

    if clip:
        pred = np.clip(pred, 0, 100)

    return pred.astype(np.float32)

# ------------------------------------------------------------
# 4. Tiered hybrid screen
# ------------------------------------------------------------

overlap_source_defs_39c = {
    "direct39a": {
        "oof": direct_oof_39c,
        "test": direct_test_39c,
    },
    "solver39b": {
        "oof": solver_oof_39c,
        "test": solver_test_39c,
    },
    "avg_direct_solver": {
        "oof": np.nanmean(np.vstack([direct_oof_39c, solver_oof_39c]), axis=0).astype(np.float32),
        "test": np.nanmean(np.vstack([direct_test_39c, solver_test_39c]), axis=0).astype(np.float32),
    },
}

tier_screen_rows_39c = []
tier_detail_rows_39c = []

# Use pure non-accounting models as fallback bases. Accounting candidates are saved for final blend screen,
# but using account39b as a base here would double-apply solver corrections.
fallback_base_names_39c = [
    name for name in base_preds_39c.keys()
    if name not in ["account39a_best", "account39b_best"]
]

for base_name in fallback_base_names_39c:
    base_oof = base_preds_39c[base_name]["oof"]

    none_sse = float(np.sum((y_arr_39c[none_oof_39c] - base_oof[none_oof_39c]) ** 2))
    none_n = int(none_oof_39c.sum())
    none_mse = none_sse / none_n if none_n > 0 else np.nan

    direct_only_best = best_lambda_for_tier_39c(
        y=y_arr_39c,
        base=base_oof,
        source=direct_oof_39c,
        mask=direct_only_oof_39c,
        lambda_grid=lambda_grid_39c,
    )

    solver_only_best = best_lambda_for_tier_39c(
        y=y_arr_39c,
        base=base_oof,
        source=solver_oof_39c,
        mask=solver_only_oof_39c,
        lambda_grid=lambda_grid_39c,
    )

    for overlap_source_name, overlap_source_obj in overlap_source_defs_39c.items():
        overlap_source_oof = overlap_source_obj["oof"]

        overlap_best = best_lambda_for_tier_39c(
            y=y_arr_39c,
            base=base_oof,
            source=overlap_source_oof,
            mask=overlap_oof_39c,
            lambda_grid=lambda_grid_39c,
        )

        total_sse = (
            none_sse
            + direct_only_best["sse"]
            + solver_only_best["sse"]
            + overlap_best["sse"]
        )

        total_mse = total_sse / n_train_39c

        row = {
            "candidate_type": "tiered_accounting_hybrid",
            "base_name": base_name,
            "overlap_source": overlap_source_name,
            "lambda_overlap": overlap_best["lambda"],
            "lambda_direct_only": direct_only_best["lambda"],
            "lambda_solver_only": solver_only_best["lambda"],
            "oof_mse": float(total_mse),
            "gain_vs_33a": float(anchor_mse_39c - total_mse),
            "gain_vs_39a": float(mse39a_39c - total_mse),
            "gain_vs_39b": float(mse39b_39c - total_mse),
            "n_overlap": int(overlap_oof_39c.sum()),
            "n_direct_only": int(direct_only_oof_39c.sum()),
            "n_solver_only": int(solver_only_oof_39c.sum()),
            "n_none": int(none_oof_39c.sum()),
            "none_mse_base": none_mse,
            "overlap_mse": overlap_best["mse"],
            "direct_only_mse": direct_only_best["mse"],
            "solver_only_mse": solver_only_best["mse"],
        }

        tier_screen_rows_39c.append(row)

        tier_detail_rows_39c.extend([
            {
                "base_name": base_name,
                "overlap_source": overlap_source_name,
                "tier": "overlap",
                **overlap_best,
            },
            {
                "base_name": base_name,
                "overlap_source": overlap_source_name,
                "tier": "direct_only",
                **direct_only_best,
            },
            {
                "base_name": base_name,
                "overlap_source": overlap_source_name,
                "tier": "solver_only",
                **solver_only_best,
            },
            {
                "base_name": base_name,
                "overlap_source": overlap_source_name,
                "tier": "none",
                "lambda": 0.0,
                "sse": none_sse,
                "mse": none_mse,
                "n": none_n,
                "base_mse": none_mse,
                "source_mse": np.nan,
            },
        ])

tier_screen39c = pd.DataFrame(tier_screen_rows_39c).sort_values("oof_mse").reset_index(drop=True)
tier_detail39c = pd.DataFrame(tier_detail_rows_39c)

if len(tier_screen39c) == 0:
    raise ValueError("No 39C tiered candidates were created.")

best_tier39c = tier_screen39c.iloc[0]

best_base_name39c = best_tier39c["base_name"]
best_overlap_source_name39c = best_tier39c["overlap_source"]

best_base_oof39c = base_preds_39c[best_base_name39c]["oof"]
best_base_test39c = base_preds_39c[best_base_name39c]["test"]

best_overlap_source_oof39c = overlap_source_defs_39c[best_overlap_source_name39c]["oof"]
best_overlap_source_test39c = overlap_source_defs_39c[best_overlap_source_name39c]["test"]

tiered_oof39c = compute_tiered_prediction_39c(
    base=best_base_oof39c,
    overlap_source=best_overlap_source_oof39c,
    lambda_overlap=float(best_tier39c["lambda_overlap"]),
    lambda_direct_only=float(best_tier39c["lambda_direct_only"]),
    lambda_solver_only=float(best_tier39c["lambda_solver_only"]),
)

tiered_test39c = compute_tiered_test_prediction_39c(
    base_test=best_base_test39c,
    overlap_source_test=best_overlap_source_test39c,
    lambda_overlap=float(best_tier39c["lambda_overlap"]),
    lambda_direct_only=float(best_tier39c["lambda_direct_only"]),
    lambda_solver_only=float(best_tier39c["lambda_solver_only"]),
)

tiered_mse39c = float(mean_squared_error(y_arr_39c, tiered_oof39c))

# ------------------------------------------------------------
# 5. Conservative final blend screen
# ------------------------------------------------------------

blend_candidates_39c = {
    "tiered_best": {
        "oof": tiered_oof39c,
        "test": tiered_test39c,
        "oof_mse": tiered_mse39c,
    }
}

for name, d in base_preds_39c.items():
    blend_candidates_39c[name] = {
        "oof": d["oof"],
        "test": d["test"],
        "oof_mse": d["oof_mse"],
    }

blend_rows_39c = []

# Identity candidate.
blend_rows_39c.append({
    "candidate_type": "identity",
    "left": "tiered_best",
    "right": "",
    "w_left": 1.0,
    "oof_mse": tiered_mse39c,
    "gain_vs_33a": anchor_mse_39c - tiered_mse39c,
    "gain_vs_39a": mse39a_39c - tiered_mse39c,
    "gain_vs_39b": mse39b_39c - tiered_mse39c,
})

blend_grid_39c = np.linspace(0.0, 1.0, 501)

for other_name, other_obj in blend_candidates_39c.items():
    if other_name == "tiered_best":
        continue

    left = tiered_oof39c.astype(np.float64)
    right = other_obj["oof"].astype(np.float64)

    best_blend = None

    for w_left in blend_grid_39c:
        pred = np.clip(float(w_left) * left + (1.0 - float(w_left)) * right, 0, 100)
        mse = float(mean_squared_error(y_arr_39c, pred))

        row = {
            "candidate_type": "pairwise_blend_with_tiered",
            "left": "tiered_best",
            "right": other_name,
            "w_left": float(w_left),
            "oof_mse": mse,
            "gain_vs_33a": anchor_mse_39c - mse,
            "gain_vs_39a": mse39a_39c - mse,
            "gain_vs_39b": mse39b_39c - mse,
        }

        if best_blend is None or row["oof_mse"] < best_blend["oof_mse"]:
            best_blend = row

    blend_rows_39c.append(best_blend)

blend_screen39c = pd.DataFrame(blend_rows_39c).sort_values("oof_mse").reset_index(drop=True)

best_blend39c = blend_screen39c.iloc[0]

if best_blend39c["candidate_type"] == "identity":
    final_oof39c = tiered_oof39c.copy()
    final_test39c = tiered_test39c.copy()
else:
    w_left = float(best_blend39c["w_left"])
    right_name = best_blend39c["right"]

    final_oof39c = np.clip(
        w_left * tiered_oof39c + (1.0 - w_left) * blend_candidates_39c[right_name]["oof"],
        0,
        100,
    ).astype(np.float32)

    final_test39c = np.clip(
        w_left * tiered_test39c + (1.0 - w_left) * blend_candidates_39c[right_name]["test"],
        0,
        100,
    ).astype(np.float32)

final_mse39c = float(mean_squared_error(y_arr_39c, final_oof39c))

# ------------------------------------------------------------
# 6. Fold gains
# ------------------------------------------------------------

folds39c = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_39c))
)

fold_gain_rows39c = []

for fold_num, (_, va_idx) in enumerate(folds39c, start=1):
    anchor_fold_mse = float(mean_squared_error(y_arr_39c[va_idx], base_preds_39c["anchor33a"]["oof"][va_idx]))
    acc39a_fold_mse = float(mean_squared_error(y_arr_39c[va_idx], base_preds_39c["account39a_best"]["oof"][va_idx]))
    acc39b_fold_mse = float(mean_squared_error(y_arr_39c[va_idx], base_preds_39c["account39b_best"]["oof"][va_idx]))
    tiered_fold_mse = float(mean_squared_error(y_arr_39c[va_idx], tiered_oof39c[va_idx]))
    final_fold_mse = float(mean_squared_error(y_arr_39c[va_idx], final_oof39c[va_idx]))

    fold_gain_rows39c.append({
        "fold": fold_num,
        "anchor33a_mse": anchor_fold_mse,
        "account39a_mse": acc39a_fold_mse,
        "account39b_mse": acc39b_fold_mse,
        "tiered39c_mse": tiered_fold_mse,
        "final39c_mse": final_fold_mse,
        "final_gain_vs_33a": anchor_fold_mse - final_fold_mse,
        "final_gain_vs_39a": acc39a_fold_mse - final_fold_mse,
        "final_gain_vs_39b": acc39b_fold_mse - final_fold_mse,
        "n_overlap": int(overlap_oof_39c[va_idx].sum()),
        "n_direct_only": int(direct_only_oof_39c[va_idx].sum()),
        "n_solver_only": int(solver_only_oof_39c[va_idx].sum()),
        "n_none": int(none_oof_39c[va_idx].sum()),
    })

fold_gains39c = pd.DataFrame(fold_gain_rows39c)

# ------------------------------------------------------------
# 7. Diagnostics by tier for final tiered candidate
# ------------------------------------------------------------

tier_perf_rows39c = []

tier_defs_oof39c = {
    "overlap": overlap_oof_39c,
    "direct_only": direct_only_oof_39c,
    "solver_only": solver_only_oof_39c,
    "none": none_oof_39c,
    "direct_any": direct_cov_oof_39c,
    "solver_any": solver_cov_oof_39c,
}

for tier_name, mask in tier_defs_oof39c.items():
    n = int(mask.sum())
    if n == 0:
        continue

    tier_perf_rows39c.append({
        "tier": tier_name,
        "n_rows": n,
        "anchor33a_mse": float(mean_squared_error(y_arr_39c[mask], base_preds_39c["anchor33a"]["oof"][mask])),
        "base_mse": float(mean_squared_error(y_arr_39c[mask], best_base_oof39c[mask])),
        "account39a_best_mse": float(mean_squared_error(y_arr_39c[mask], base_preds_39c["account39a_best"]["oof"][mask])),
        "account39b_best_mse": float(mean_squared_error(y_arr_39c[mask], base_preds_39c["account39b_best"]["oof"][mask])),
        "tiered39c_mse": float(mean_squared_error(y_arr_39c[mask], tiered_oof39c[mask])),
        "final39c_mse": float(mean_squared_error(y_arr_39c[mask], final_oof39c[mask])),
    })

tier_perf39c = pd.DataFrame(tier_perf_rows39c)

# ------------------------------------------------------------
# 8. Save artifacts
# ------------------------------------------------------------

tier_screen_path39c = "model_results/account39c_tiered_screen.csv"
tier_detail_path39c = "model_results/account39c_tier_detail.csv"
blend_screen_path39c = "model_results/account39c_final_blend_screen.csv"
fold_gains_path39c = "model_results/account39c_best_fold_gains.csv"
tier_perf_path39c = "model_results/account39c_tier_performance.csv"

oof_tiered_path39c = "model_results/oof_account39c_tiered_best.csv"
test_tiered_path39c = "model_results/testpred_account39c_tiered_best.csv"
submission_tiered_path39c = "submission_account39c_tiered_best.csv"

oof_final_path39c = "model_results/oof_account39c_final_best.csv"
test_final_path39c = "model_results/testpred_account39c_final_best.csv"
submission_final_path39c = "submission_account39c_final_best.csv"

tier_screen39c.to_csv(tier_screen_path39c, index=False)
tier_detail39c.to_csv(tier_detail_path39c, index=False)
blend_screen39c.to_csv(blend_screen_path39c, index=False)
fold_gains39c.to_csv(fold_gains_path39c, index=False)
tier_perf39c.to_csv(tier_perf_path39c, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_39c),
    TARGET_COL: y_arr_39c,
    "anchor33a_pred": base_preds_39c["anchor33a"]["oof"],
    "account39a_pred": base_preds_39c["account39a_best"]["oof"],
    "account39b_pred": base_preds_39c["account39b_best"]["oof"],
    "base_name": best_base_name39c,
    "base_pred": best_base_oof39c,
    "direct39a_raw": direct_oof_39c,
    "direct39a_covered": direct_cov_oof_39c.astype(int),
    "solver39b_raw": solver_oof_39c,
    "solver39b_covered": solver_cov_oof_39c.astype(int),
    "overlap": overlap_oof_39c.astype(int),
    "direct_only": direct_only_oof_39c.astype(int),
    "solver_only": solver_only_oof_39c.astype(int),
    "pred_clipped": tiered_oof39c,
}).to_csv(oof_tiered_path39c, index=False)

pd.DataFrame({
    ID_COL: test_ids_39c,
    "anchor33a_pred": base_preds_39c["anchor33a"]["test"],
    "account39a_pred": base_preds_39c["account39a_best"]["test"],
    "account39b_pred": base_preds_39c["account39b_best"]["test"],
    "base_name": best_base_name39c,
    "base_pred": best_base_test39c,
    "direct39a_raw": direct_test_39c,
    "direct39a_covered": direct_cov_test_39c.astype(int),
    "solver39b_raw": solver_test_39c,
    "solver39b_covered": solver_cov_test_39c.astype(int),
    "overlap": overlap_test_39c.astype(int),
    "direct_only": direct_only_test_39c.astype(int),
    "solver_only": solver_only_test_39c.astype(int),
    TARGET_COL: tiered_test39c,
}).to_csv(test_tiered_path39c, index=False)

pd.DataFrame({
    ID_COL: test_ids_39c,
    TARGET_COL: tiered_test39c,
}).to_csv(submission_tiered_path39c, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_39c),
    TARGET_COL: y_arr_39c,
    "tiered_pred": tiered_oof39c,
    "pred_clipped": final_oof39c,
}).to_csv(oof_final_path39c, index=False)

pd.DataFrame({
    ID_COL: test_ids_39c,
    "tiered_pred": tiered_test39c,
    TARGET_COL: final_test39c,
}).to_csv(test_final_path39c, index=False)

pd.DataFrame({
    ID_COL: test_ids_39c,
    TARGET_COL: final_test39c,
}).to_csv(submission_final_path39c, index=False)

# ------------------------------------------------------------
# 9. Validate final submission
# ------------------------------------------------------------

sub39c = pd.read_csv(submission_final_path39c)

assert sub39c.shape[0] == n_test_39c, f"Expected {n_test_39c}, got {sub39c.shape[0]}"
assert list(sub39c.columns) == [ID_COL, TARGET_COL], list(sub39c.columns)
assert sub39c[ID_COL].notna().all(), "Missing IDs"
assert sub39c[TARGET_COL].notna().all(), "Missing predictions"
assert np.isfinite(sub39c[TARGET_COL]).all(), "Non-finite predictions"
assert sub39c[TARGET_COL].between(0, 100).all(), "Predictions outside [0,100]"

# ------------------------------------------------------------
# 10. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("39C trust-tiered accounting hybrid complete")
print("=" * 90)

print("\nBaseline OOF comparison")
print("-----------------------")
print(f"33A anchor OOF MSE: {anchor_mse_39c:.6f}")
print(f"39A best OOF MSE:   {mse39a_39c:.6f}")
print(f"39B best OOF MSE:   {mse39b_39c:.6f}")

print("\nBest tiered 39C candidate")
print("-------------------------")
print(best_tier39c.to_string())
print(f"\nTiered 39C OOF MSE: {tiered_mse39c:.6f}")
print(f"Tiered gain vs 39B: {mse39b_39c - tiered_mse39c:.6f}")

print("\nFinal conservative blend choice")
print("-------------------------------")
print(best_blend39c.to_string())
print(f"\nFinal 39C OOF MSE: {final_mse39c:.6f}")
print(f"Final gain vs 33A: {anchor_mse_39c - final_mse39c:.6f}")
print(f"Final gain vs 39A: {mse39a_39c - final_mse39c:.6f}")
print(f"Final gain vs 39B: {mse39b_39c - final_mse39c:.6f}")

print("\nBest 39C fold gains")
print("-------------------")
print(fold_gains39c.to_string(index=False))
print("Min final gain vs 39B:", float(fold_gains39c["final_gain_vs_39b"].min()))

print("\nTier performance")
print("----------------")
print(tier_perf39c.to_string(index=False))

print("\nTop 20 tiered candidates")
print("------------------------")
display(tier_screen39c.head(20))

print("\nTop 20 final blend candidates")
print("-----------------------------")
display(blend_screen39c.head(20))

print("\nSaved files")
print("-----------")
print(tier_screen_path39c)
print(tier_detail_path39c)
print(blend_screen_path39c)
print(fold_gains_path39c)
print(tier_perf_path39c)
print(oof_tiered_path39c)
print(test_tiered_path39c)
print(submission_tiered_path39c)
print(oof_final_path39c)
print(test_final_path39c)
print(submission_final_path39c)

print("\nSubmission validation")
print("---------------------")
print("Final submission:", submission_final_path39c)
print("Shape:", sub39c.shape)
print("Prediction summary:")
print(sub39c[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
if final_mse39c < mse39b_39c - 1.0:
    print("39C materially improves over 39B. Prefer submitting 39C next.")
elif final_mse39c < mse39b_39c - 0.1:
    print("39C improves over 39B modestly. Consider submitting 39C if public slots remain.")
elif final_mse39c <= mse39b_39c + 0.1:
    print("39C is essentially tied with 39B OOF. Use public score/coverage diagnostics to decide.")
else:
    print("39C does not beat 39B OOF. Keep 39B as primary.")

39C. Trust-tiered accounting hybrid + conservative blend screen

Tier coverage
-------------
OOF overlap direct & solver: 53786
OOF direct only:             0
OOF solver only:             27807
OOF neither:                 63328
OOF total accounting union:  81593 / 144921

Test overlap direct & solver: 27298
Test direct only:             0
Test solver only:             17807
Test neither:                 3202
Test total accounting union:  45105 / 48307
Loaded anchor33a            | OOF MSE 77.049866
Loaded knn35a               | OOF MSE 76.453072
Loaded nn37b_rank1          | OOF MSE 76.849442
Loaded step36a              | OOF MSE 76.954796
Loaded blend33c             | OOF MSE 76.859200
Loaded account39a_best      | OOF MSE 55.642780
Loaded account39b_best      | OOF MSE 53.720547

39C trust-tiered accounting hybrid complete

Baseline OOF comparison
-----------------------
33A anchor OOF MSE: 77.049866
39A best OOF MSE:   55.642780
39B best OOF MSE:   53.720547

Best tiered 39C candid

,candidate_type,base_name,overlap_source,lambda_overlap,lambda_direct_only,lambda_solver_only,oof_mse,gain_vs_33a,gain_vs_39a,gain_vs_39b,n_overlap,n_direct_only,n_solver_only,n_none,none_mse_base,overlap_mse,direct_only_mse,solver_only_mse
0,tiered_accounting_hybrid,knn35a,direct39a,0.992,0.0,0.650,53.228146,23.821720,2.414634,0.492401,53786,0,27807,63328,92.374242,0.595324,NaN,65.881974
1,tiered_accounting_hybrid,knn35a,avg_direct_solver,0.992,0.0,0.650,53.228146,23.821719,2.414634,0.492400,53786,0,27807,63328,92.374242,0.595325,NaN,65.881974
2,tiered_accounting_hybrid,knn35a,solver39b,0.992,0.0,0.650,53.228146,23.821719,2.414634,0.492400,53786,0,27807,63328,92.374242,0.595325,NaN,65.881974
3,tiered_accounting_hybrid,nn37b_rank1,direct39a,0.992,0.0,0.655,53.283203,23.766662,2.359577,0.437343,53786,0,27807,63328,92.469618,0.595445,NaN,65.951469
4,tiered_accounting_hybrid,nn37b_rank1,avg_direct_solver,0.992,0.0,0.655,53.283203,23.766662,2.359577,0.437343,53786,0,27807,63328,92.469618,0.595445,NaN,65.951469
5,tiered_accounting_hybrid,nn37b_rank1,solver39b,0.992,0.0,0.655,53.283203,23.766662,2.359577,0.437343,53786,0,27807,63328,92.469618,0.595445,NaN,65.951469
6,tiered_accounting_hybrid,step36a,direct39a,0.992,0.0,0.655,53.310174,23.739692,2.332606,0.410373,53786,0,27807,63328,92.533255,0.595409,NaN,65.947173
7,tiered_accounting_hybrid,step36a,avg_direct_solver,0.992,0.0,0.655,53.310174,23.739692,2.332606,0.410373,53786,0,27807,63328,92.533255,0.595410,NaN,65.947173
8,tiered_accounting_hybrid,step36a,solver39b,0.992,0.0,0.655,53.310174,23.739692,2.332606,0.410373,53786,0,27807,63328,92.533255,0.595410,NaN,65.947173
9,tiered_accounting_hybrid,anchor33a,direct39a,0.992,0.0,0.655,53.380856,23.669010,2.261924,0.339691,53786,0,27807,63328,92.678799,0.595420,NaN,65.984061



Top 20 final blend candidates
-----------------------------


,candidate_type,left,right,w_left,oof_mse,gain_vs_33a,gain_vs_39a,gain_vs_39b
0,pairwise_blend_with_tiered,tiered_best,account39a_best,0.976,53.226633,23.823233,2.416148,0.493914
1,pairwise_blend_with_tiered,tiered_best,nn37b_rank1,0.992,53.226708,23.823158,2.416073,0.493839
2,pairwise_blend_with_tiered,tiered_best,step36a,0.994,53.227224,23.822641,2.415556,0.493322
3,pairwise_blend_with_tiered,tiered_best,blend33c,0.996,53.227801,23.822065,2.414979,0.492746
4,pairwise_blend_with_tiered,tiered_best,anchor33a,0.998,53.228016,23.821850,2.414764,0.492531
5,pairwise_blend_with_tiered,tiered_best,knn35a,1.000,53.228148,23.821718,2.414632,0.492399
6,pairwise_blend_with_tiered,tiered_best,account39b_best,1.000,53.228148,23.821718,2.414632,0.492399
7,identity,tiered_best,,1.000,53.228149,23.821716,2.414631,0.492397



Saved files
-----------
model_results/account39c_tiered_screen.csv
model_results/account39c_tier_detail.csv
model_results/account39c_final_blend_screen.csv
model_results/account39c_best_fold_gains.csv
model_results/account39c_tier_performance.csv
model_results/oof_account39c_tiered_best.csv
model_results/testpred_account39c_tiered_best.csv
submission_account39c_tiered_best.csv
model_results/oof_account39c_final_best.csv
model_results/testpred_account39c_final_best.csv
submission_account39c_final_best.csv

Submission validation
---------------------
Final submission: submission_account39c_final_best.csv
Shape: (48307, 2)
Prediction summary:
count    48307.000000
mean        54.186695
std         25.771232
min          0.008765
25%         33.435155
50%         52.786070
75%         75.249203
max        100.000000
Name: PERCENT_PROFICIENT, dtype: float64

Decision rule
-------------
39C improves over 39B modestly. Consider submitting 39C if public slots remain.


### 39C. Trust-Tiered Accounting Hybrid and Conservative Blend

This section refined the accounting branch by separating rows into trust tiers. The 39A direct reconstruction was nearly exact where available, while the 39B constrained solver covered many more rows but was less exact. The goal of 39C was to use each accounting source where it was most reliable.

Rows were divided into:

- overlap rows covered by both 39A and 39B,
- direct-only rows covered only by 39A,
- solver-only rows covered only by 39B,
- and rows covered by neither accounting method.

In this dataset, there were no direct-only rows. The OOF tier counts were:

- 39A/39B overlap: `53,786`
- 39A direct only: `0`
- 39B solver only: `27,807`
- neither: `63,328`

The test tier counts were:

- 39A/39B overlap: `27,298`
- 39A direct only: `0`
- 39B solver only: `17,807`
- neither: `3,202`

The best tiered model used `knn35a` as the base/fallback model, used the 39A direct reconstruction on overlap rows, and used a reduced solver shrinkage on solver-only rows. The fitted tier weights were:

- overlap source: `direct39a`
- overlap lambda: `0.992`
- solver-only lambda: `0.650`
- base/fallback model: `knn35a`

The tiered model had OOF MSE `53.228149`, improving over the 39B OOF MSE of `53.720547` by about `0.492397`.

A final conservative pairwise blend with the 39A model gave the best final result:

- final 39C OOF MSE: `53.226631`
- gain versus 33A anchor: `23.823235`
- gain versus 39A: `2.416149`
- gain versus 39B: `0.493916`

The fold gains versus 39B were all positive:

- Fold 1: `0.495617`
- Fold 2: `0.465733`
- Fold 3: `0.834633`
- Fold 4: `0.392185`
- Fold 5: `0.281387`

Conclusion: 39C is a modest but stable improvement over 39B. It is structurally preferable because it trusts exact direct accounting on rows where direct reconstruction is available, while shrinking the broader constrained solver more conservatively on rows that are only solver-covered. The file `submission_account39c_final_best.csv` should be treated as the next Kaggle candidate if a submission slot remains.

### Checkpoint: Accounting Branch Breakthrough and Submission Results

The project’s main breakthrough came from exploiting subgroup accounting structure rather than from generic regression-model tuning. Earlier model-family screens — splines/GAMs, KNN/local regression, step functions, and neural networks — produced only small residual improvements relative to the 33A target-encoded boosting anchor.

The key discovered subgroup identities were:

`All Students = Female + Male`

`All Students = Economically Disadvantaged + Not Economically Disadvantaged`

These relationships allow many target rows to be reconstructed or constrained using sibling rows within the same `SCHOOL × ASSESSMENT_NAME` group.

Important accounting models:

- **39A direct accounting reconstruction**
  - OOF MSE: `55.642780`
  - Public MSE: `37.424`
  - Strength: very accurate on directly reconstructable rows.
  - Weakness: lower test coverage.

- **39B constrained accounting solver**
  - OOF MSE: `53.720547`
  - Public MSE: `32.113`
  - Strength: much higher test coverage by solving subgroup systems with model priors.
  - Weakness: solver-only rows are less exact than direct 39A reconstructions.

- **39C trust-tiered accounting hybrid**
  - OOF MSE: `53.226631`
  - Public MSE: `31.431`
  - Strength: uses direct 39A reconstruction on high-trust overlap rows and a more conservative solver adjustment on solver-only rows.
  - Current best public submission.

39C improved public MSE over 39B by only `0.682`, so future submissions should be reserved for structurally meaningful improvements. Going forward, a new candidate should generally need either an OOF improvement of roughly `3+` MSE over 39C or a clear structural reason to expect a large public gain before using a Kaggle submission slot.

In [ ]:
# ============================================================
# 39D. Subgroup-specific solver shrinkage + none-tier fallback selection
# ============================================================
#
# Context:
#   39C is current best public: 31.431.
#   39C only improved public over 39B by 0.682, so future submissions
#   should require larger expected gains.
#
# Goal:
#   Improve the weak tiers from 39C:
#     - solver_only rows: covered by 39B solver but not direct 39A
#     - none rows: not covered by accounting
#
# 39D tests:
#   1. Solver-only shrinkage tuned by subgroup.
#   2. Solver-only shrinkage tuned by subgroup x N_STUDENTS bin.
#   3. None-tier fallback model selected by subgroup.
#
# Submission rule:
#   Do NOT submit 39D unless it beats 39C by around 3+ OOF MSE
#   or reveals a clear structural public-score opportunity.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("39D. Subgroup-specific solver shrinkage + none-tier fallback selection")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required artifact checks
# ------------------------------------------------------------

required_paths_39d = [
    "model_results/account39a_raw_oof_reconstruction.csv",
    "model_results/account39a_raw_test_reconstruction.csv",
    "model_results/account39b_solver_raw_oof.csv",
    "model_results/account39b_solver_raw_test.csv",
    "model_results/oof_account39a_best.csv",
    "model_results/testpred_account39a_best.csv",
    "model_results/oof_account39b_solver_best.csv",
    "model_results/testpred_account39b_solver_best.csv",
    "model_results/oof_account39c_final_best.csv",
    "model_results/testpred_account39c_final_best.csv",
]

missing_paths_39d = [p for p in required_paths_39d if not Path(p).exists()]
if missing_paths_39d:
    raise FileNotFoundError(f"Missing required artifacts: {missing_paths_39d}")

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("39D requires raw_train_te/raw_test_te. Rerun the safe recovery cell if needed.")

for c in ["SUBGROUP_NAME", "N_STUDENTS"]:
    if c not in raw_train_te.columns or c not in raw_test_te.columns:
        raise ValueError(f"Missing required column in raw_train_te/raw_test_te: {c}")

# ------------------------------------------------------------
# 2. Load accounting raw artifacts
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

n_train_39d = len(raw39a_oof)
n_test_39d = len(raw39a_test)

if len(raw39b_oof) != n_train_39d:
    raise ValueError("39A/39B OOF row count mismatch.")
if len(raw39b_test) != n_test_39d:
    raise ValueError("39A/39B test row count mismatch.")

if TARGET_COL in raw39a_oof.columns:
    y_arr_39d = raw39a_oof[TARGET_COL].to_numpy(dtype=np.float32)
elif "y_train" in globals():
    y_arr_39d = np.asarray(y_train, dtype=np.float32).reshape(-1)
else:
    raise ValueError("Could not recover y_train.")

if len(y_arr_39d) != n_train_39d:
    raise ValueError("Target length mismatch.")

if ID_COL in raw39b_test.columns:
    test_ids_39d = raw39b_test[ID_COL].to_numpy()
elif ID_COL in raw39a_test.columns:
    test_ids_39d = raw39a_test[ID_COL].to_numpy()
elif "test_ids" in globals():
    test_ids_39d = np.asarray(test_ids)
else:
    raise ValueError("Could not recover test IDs.")

direct_oof_39d = raw39a_oof["accounting_pred"].to_numpy(dtype=np.float32)
direct_test_39d = raw39a_test["accounting_pred"].to_numpy(dtype=np.float32)

direct_cov_oof_39d = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test_39d = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)

solver_oof_39d = raw39b_oof["solver_pred"].to_numpy(dtype=np.float32)
solver_test_39d = raw39b_test["solver_pred"].to_numpy(dtype=np.float32)

solver_cov_oof_39d = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test_39d = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

direct_cov_oof_39d = direct_cov_oof_39d & np.isfinite(direct_oof_39d)
direct_cov_test_39d = direct_cov_test_39d & np.isfinite(direct_test_39d)

solver_cov_oof_39d = solver_cov_oof_39d & np.isfinite(solver_oof_39d)
solver_cov_test_39d = solver_cov_test_39d & np.isfinite(solver_test_39d)

overlap_oof_39d = direct_cov_oof_39d & solver_cov_oof_39d
direct_only_oof_39d = direct_cov_oof_39d & ~solver_cov_oof_39d
solver_only_oof_39d = solver_cov_oof_39d & ~direct_cov_oof_39d
none_oof_39d = ~(direct_cov_oof_39d | solver_cov_oof_39d)

overlap_test_39d = direct_cov_test_39d & solver_cov_test_39d
direct_only_test_39d = direct_cov_test_39d & ~solver_cov_test_39d
solver_only_test_39d = solver_cov_test_39d & ~direct_cov_test_39d
none_test_39d = ~(direct_cov_test_39d | solver_cov_test_39d)

# Raw subgroup / size labels.
def clean_str_39d(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str).to_numpy()

subgroup_train_39d = clean_str_39d(raw_train_te["SUBGROUP_NAME"])
subgroup_test_39d = clean_str_39d(raw_test_te["SUBGROUP_NAME"])

n_train_students_39d = pd.to_numeric(raw_train_te["N_STUDENTS"], errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)
n_test_students_39d = pd.to_numeric(raw_test_te["N_STUDENTS"], errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)

def make_n_bin_39d(x):
    return pd.cut(
        pd.Series(x).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"],
    ).astype("string").fillna("<NA>").astype(str).to_numpy()

n_bin_train_39d = make_n_bin_39d(n_train_students_39d)
n_bin_test_39d = make_n_bin_39d(n_test_students_39d)

segment_train_defs_39d = {
    "global": np.array(["GLOBAL"] * n_train_39d, dtype=object),
    "subgroup": subgroup_train_39d.astype(object),
    "subgroup_nbin": np.array([f"{a}||{b}" for a, b in zip(subgroup_train_39d, n_bin_train_39d)], dtype=object),
}

segment_test_defs_39d = {
    "global": np.array(["GLOBAL"] * n_test_39d, dtype=object),
    "subgroup": subgroup_test_39d.astype(object),
    "subgroup_nbin": np.array([f"{a}||{b}" for a, b in zip(subgroup_test_39d, n_bin_test_39d)], dtype=object),
}

print("\nTier coverage")
print("-------------")
print("OOF overlap:      ", int(overlap_oof_39d.sum()))
print("OOF direct only:  ", int(direct_only_oof_39d.sum()))
print("OOF solver only:  ", int(solver_only_oof_39d.sum()))
print("OOF neither:      ", int(none_oof_39d.sum()))
print("Test overlap:     ", int(overlap_test_39d.sum()))
print("Test direct only: ", int(direct_only_test_39d.sum()))
print("Test solver only: ", int(solver_only_test_39d.sum()))
print("Test neither:     ", int(none_test_39d.sum()))

# ------------------------------------------------------------
# 3. Load prediction artifacts
# ------------------------------------------------------------

def load_oof_pred_39d(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_col = None
    for c in ["pred_clipped", "prediction", "pred", "oof_pred", TARGET_COL]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            if c == TARGET_COL and "pred_clipped" in df.columns:
                continue
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column found in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_diff > 1e-5:
            raise ValueError(f"Target mismatch in {path}: max diff {max_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col

def load_test_pred_39d(path):
    df = pd.read_csv(path)

    if len(df) != n_test_39d:
        raise ValueError(f"Test row mismatch for {path}: got {len(df)}, expected {n_test_39d}")

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No test prediction column found in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    else:
        ids = test_ids_39d

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col

candidate_paths_39d = [
    {
        "name": "anchor33a",
        "oof_path": "model_results/oof_seg33a_arcsine_adaptive.csv",
        "test_path": "model_results/testpred_seg33a_arcsine_adaptive.csv",
        "kind": "non_accounting",
    },
    {
        "name": "knn35a",
        "oof_path": "model_results/oof_knn35a_local_residual_best.csv",
        "test_path": "model_results/testpred_knn35a_local_residual_best.csv",
        "kind": "non_accounting",
    },
    {
        "name": "nn37b_rank1",
        "oof_path": "model_results/oof_nn37b38b_cpu_safe_rank1.csv",
        "test_path": "model_results/testpred_nn37b38b_cpu_safe_rank1.csv",
        "kind": "non_accounting",
    },
    {
        "name": "step36a",
        "oof_path": "model_results/oof_step36a_residual_bins_best.csv",
        "test_path": "model_results/testpred_step36a_residual_bins_best.csv",
        "kind": "non_accounting",
    },
    {
        "name": "blend33c",
        "oof_path": "model_results/oof_blend33c_33a_huber33b.csv",
        "test_path": "model_results/testpred_blend33c_33a_huber33b.csv",
        "kind": "non_accounting",
    },
    {
        "name": "account39a_best",
        "oof_path": "model_results/oof_account39a_best.csv",
        "test_path": "model_results/testpred_account39a_best.csv",
        "kind": "accounting",
    },
    {
        "name": "account39b_best",
        "oof_path": "model_results/oof_account39b_solver_best.csv",
        "test_path": "model_results/testpred_account39b_solver_best.csv",
        "kind": "accounting",
    },
    {
        "name": "account39c_final",
        "oof_path": "model_results/oof_account39c_final_best.csv",
        "test_path": "model_results/testpred_account39c_final_best.csv",
        "kind": "accounting",
    },
]

preds_39d = {}

for cand in candidate_paths_39d:
    name = cand["name"]

    if not Path(cand["oof_path"]).exists() or not Path(cand["test_path"]).exists():
        print(f"Skipping missing candidate: {name}")
        continue

    try:
        oof_pred, oof_col = load_oof_pred_39d(cand["oof_path"], y_arr_39d)
        test_pred, ids, test_col = load_test_pred_39d(cand["test_path"])

        preds_39d[name] = {
            "oof": oof_pred,
            "test": test_pred,
            "kind": cand["kind"],
            "oof_mse": float(mean_squared_error(y_arr_39d, oof_pred)),
            "oof_col": oof_col,
            "test_col": test_col,
        }

        print(f"Loaded {name:20s} | kind={cand['kind']:14s} | OOF MSE {preds_39d[name]['oof_mse']:.6f}")

    except Exception as e:
        print(f"Skipping {name} because of error: {repr(e)}")

required_loaded_39d = ["anchor33a", "knn35a", "account39a_best", "account39b_best", "account39c_final"]
missing_loaded_39d = [x for x in required_loaded_39d if x not in preds_39d]
if missing_loaded_39d:
    raise ValueError(f"Missing required loaded predictions: {missing_loaded_39d}")

mse33a_39d = preds_39d["anchor33a"]["oof_mse"]
mse39a_39d = preds_39d["account39a_best"]["oof_mse"]
mse39b_39d = preds_39d["account39b_best"]["oof_mse"]
mse39c_39d = preds_39d["account39c_final"]["oof_mse"]

non_accounting_names_39d = [
    name for name, d in preds_39d.items()
    if d["kind"] == "non_accounting"
]

print("\nBaseline OOF")
print("------------")
print(f"33A: {mse33a_39d:.6f}")
print(f"39A: {mse39a_39d:.6f}")
print(f"39B: {mse39b_39d:.6f}")
print(f"39C: {mse39c_39d:.6f}")

# ------------------------------------------------------------
# 4. Optimization helpers
# ------------------------------------------------------------

lambda_grid_39d = np.unique(
    np.concatenate([
        np.linspace(-0.25, 1.50, 351),
        np.array([0.0, 0.25, 0.50, 0.65, 0.75, 0.924, 0.992, 1.0]),
    ])
)

def opt_lambda_39d(y, base, source, mask, lambda_grid):
    mask = np.asarray(mask, dtype=bool)

    if int(mask.sum()) == 0:
        return 0.0, np.nan, 0

    y_m = y[mask].astype(np.float64)
    b_m = base[mask].astype(np.float64)
    s_m = source[mask].astype(np.float64)

    best_lam = 0.0
    best_sse = np.inf

    for lam in lambda_grid:
        pred = b_m + float(lam) * (s_m - b_m)
        pred = np.clip(pred, 0, 100)
        err = y_m - pred
        sse = float(np.dot(err, err))

        if sse < best_sse:
            best_sse = sse
            best_lam = float(lam)

    return best_lam, best_sse / len(y_m), int(len(y_m))

def fit_segment_lambdas_39d(y, base, source, mask, segment, lambda_grid, min_rows):
    global_lam, global_mse, global_n = opt_lambda_39d(
        y=y,
        base=base,
        source=source,
        mask=mask,
        lambda_grid=lambda_grid,
    )

    segment = np.asarray(segment, dtype=object)
    lambdas = {}
    rows = []

    unique_segments = sorted(pd.Series(segment[mask]).dropna().astype(str).unique().tolist())

    for seg in unique_segments:
        m = mask & (segment.astype(str) == str(seg))
        n = int(m.sum())

        if n < min_rows:
            continue

        lam, mse, _ = opt_lambda_39d(
            y=y,
            base=base,
            source=source,
            mask=m,
            lambda_grid=lambda_grid,
        )

        lambdas[str(seg)] = lam

        base_mse = float(mean_squared_error(y[m], base[m]))
        source_mse = float(mean_squared_error(y[m], np.clip(source[m], 0, 100)))
        pred = np.clip(base[m] + lam * (source[m] - base[m]), 0, 100)
        pred_mse = float(mean_squared_error(y[m], pred))

        rows.append({
            "segment": str(seg),
            "n_rows": n,
            "lambda": lam,
            "base_mse": base_mse,
            "source_mse": source_mse,
            "segment_tuned_mse": pred_mse,
        })

    return {
        "global_lambda": global_lam,
        "global_mse": global_mse,
        "global_n": global_n,
        "segment_lambdas": lambdas,
        "rows": rows,
    }

def apply_segment_lambdas_39d(base, source, mask, segment, global_lambda, segment_lambdas):
    pred = base.copy().astype(np.float64)
    segment = np.asarray(segment, dtype=object)

    idx = np.where(mask)[0]

    for i in idx:
        seg = str(segment[i])
        lam = segment_lambdas.get(seg, global_lambda)
        pred[i] = base[i] + lam * (source[i] - base[i])

    return np.clip(pred, 0, 100).astype(np.float32)

def choose_none_fallback_by_segment_39d(y, pred_dict, mask, segment, min_rows):
    segment = np.asarray(segment, dtype=object)
    names = list(pred_dict.keys())

    # Global best fallback on none rows.
    global_scores = []
    for name in names:
        pred = pred_dict[name]
        mse = float(mean_squared_error(y[mask], pred[mask])) if int(mask.sum()) > 0 else np.nan
        global_scores.append((name, mse))

    global_scores = sorted(global_scores, key=lambda x: x[1])
    global_best = global_scores[0][0]

    chosen = {}
    rows = []

    unique_segments = sorted(pd.Series(segment[mask]).dropna().astype(str).unique().tolist())

    for seg in unique_segments:
        m = mask & (segment.astype(str) == str(seg))
        n = int(m.sum())

        if n < min_rows:
            continue

        scores = []
        for name in names:
            pred = pred_dict[name]
            mse = float(mean_squared_error(y[m], pred[m]))
            scores.append((name, mse))

        scores = sorted(scores, key=lambda x: x[1])
        chosen[str(seg)] = scores[0][0]

        row = {
            "segment": str(seg),
            "n_rows": n,
            "chosen_base": scores[0][0],
            "chosen_mse": scores[0][1],
            "global_base": global_best,
            "global_base_mse_on_segment": float(mean_squared_error(y[m], pred_dict[global_best][m])),
        }

        for name, mse in scores:
            row[f"mse_{name}"] = mse

        rows.append(row)

    return {
        "global_best": global_best,
        "chosen_by_segment": chosen,
        "rows": rows,
        "global_scores": global_scores,
    }

def apply_none_fallback_39d(default_base, pred_dict, mask, segment, global_best, chosen_by_segment):
    pred = default_base.copy().astype(np.float64)
    segment = np.asarray(segment, dtype=object)

    idx = np.where(mask)[0]

    for i in idx:
        seg = str(segment[i])
        name = chosen_by_segment.get(seg, global_best)
        pred[i] = pred_dict[name][i]

    return np.clip(pred, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 5. Candidate construction
# ------------------------------------------------------------

screen_rows_39d = []
segment_lambda_rows_39d = []
none_selection_rows_39d = []

# Non-accounting prediction dictionaries for fallback selection.
nonacct_oof_dict_39d = {
    name: preds_39d[name]["oof"]
    for name in non_accounting_names_39d
}
nonacct_test_dict_39d = {
    name: preds_39d[name]["test"]
    for name in non_accounting_names_39d
}

configs_39d = []

for base_name in non_accounting_names_39d:
    for solver_segment_mode in ["global", "subgroup", "subgroup_nbin"]:
        for none_policy in ["same_base", "segment_best_base"]:
            configs_39d.append({
                "base_name": base_name,
                "solver_segment_mode": solver_segment_mode,
                "none_policy": none_policy,
                "min_rows_solver": 300 if solver_segment_mode == "subgroup" else 500,
                "min_rows_none": 500,
            })

print("\n39D configs:", len(configs_39d))
for cfg in configs_39d:
    print(cfg)

candidate_store_39d = {}

for cfg_i, cfg in enumerate(configs_39d, start=1):
    base_name = cfg["base_name"]
    base_oof = preds_39d[base_name]["oof"]
    base_test = preds_39d[base_name]["test"]

    solver_segment_mode = cfg["solver_segment_mode"]

    seg_train_solver = segment_train_defs_39d[solver_segment_mode]
    seg_test_solver = segment_test_defs_39d[solver_segment_mode]

    # 39C already showed direct accounting should be trusted at lambda near 0.992.
    # Still tune overlap lambda for this base defensively.
    lambda_overlap, overlap_mse, overlap_n = opt_lambda_39d(
        y=y_arr_39d,
        base=base_oof,
        source=direct_oof_39d,
        mask=overlap_oof_39d,
        lambda_grid=lambda_grid_39d,
    )

    lambda_direct_only, direct_only_mse, direct_only_n = opt_lambda_39d(
        y=y_arr_39d,
        base=base_oof,
        source=direct_oof_39d,
        mask=direct_only_oof_39d,
        lambda_grid=lambda_grid_39d,
    )

    solver_fit = fit_segment_lambdas_39d(
        y=y_arr_39d,
        base=base_oof,
        source=solver_oof_39d,
        mask=solver_only_oof_39d,
        segment=seg_train_solver,
        lambda_grid=lambda_grid_39d,
        min_rows=cfg["min_rows_solver"],
    )

    for row in solver_fit["rows"]:
        row2 = dict(row)
        row2.update({
            "candidate_id": cfg_i,
            "base_name": base_name,
            "solver_segment_mode": solver_segment_mode,
            "none_policy": cfg["none_policy"],
        })
        segment_lambda_rows_39d.append(row2)

    # Start with base.
    pred_oof = base_oof.copy().astype(np.float64)
    pred_test = base_test.copy().astype(np.float64)

    # Overlap: use direct 39A.
    pred_oof[overlap_oof_39d] = (
        base_oof[overlap_oof_39d]
        + lambda_overlap * (direct_oof_39d[overlap_oof_39d] - base_oof[overlap_oof_39d])
    )
    pred_test[overlap_test_39d] = (
        base_test[overlap_test_39d]
        + lambda_overlap * (direct_test_39d[overlap_test_39d] - base_test[overlap_test_39d])
    )

    # Direct-only: probably none, but keep generic.
    if int(direct_only_oof_39d.sum()) > 0:
        pred_oof[direct_only_oof_39d] = (
            base_oof[direct_only_oof_39d]
            + lambda_direct_only * (direct_oof_39d[direct_only_oof_39d] - base_oof[direct_only_oof_39d])
        )
    if int(direct_only_test_39d.sum()) > 0:
        pred_test[direct_only_test_39d] = (
            base_test[direct_only_test_39d]
            + lambda_direct_only * (direct_test_39d[direct_only_test_39d] - base_test[direct_only_test_39d])
        )

    # Solver-only: segmented lambdas.
    solver_adjusted_oof = apply_segment_lambdas_39d(
        base=base_oof,
        source=solver_oof_39d,
        mask=solver_only_oof_39d,
        segment=seg_train_solver,
        global_lambda=solver_fit["global_lambda"],
        segment_lambdas=solver_fit["segment_lambdas"],
    )

    solver_adjusted_test = apply_segment_lambdas_39d(
        base=base_test,
        source=solver_test_39d,
        mask=solver_only_test_39d,
        segment=seg_test_solver,
        global_lambda=solver_fit["global_lambda"],
        segment_lambdas=solver_fit["segment_lambdas"],
    )

    pred_oof[solver_only_oof_39d] = solver_adjusted_oof[solver_only_oof_39d]
    pred_test[solver_only_test_39d] = solver_adjusted_test[solver_only_test_39d]

    # None tier: either same base or best base by subgroup.
    none_policy = cfg["none_policy"]
    none_global_best = base_name
    none_chosen_by_segment = {}

    if none_policy == "segment_best_base":
        none_selection = choose_none_fallback_by_segment_39d(
            y=y_arr_39d,
            pred_dict=nonacct_oof_dict_39d,
            mask=none_oof_39d,
            segment=subgroup_train_39d,
            min_rows=cfg["min_rows_none"],
        )

        none_global_best = none_selection["global_best"]
        none_chosen_by_segment = none_selection["chosen_by_segment"]

        for row in none_selection["rows"]:
            row2 = dict(row)
            row2.update({
                "candidate_id": cfg_i,
                "base_name": base_name,
                "solver_segment_mode": solver_segment_mode,
                "none_policy": none_policy,
            })
            none_selection_rows_39d.append(row2)

        none_oof_adjusted = apply_none_fallback_39d(
            default_base=base_oof,
            pred_dict=nonacct_oof_dict_39d,
            mask=none_oof_39d,
            segment=subgroup_train_39d,
            global_best=none_global_best,
            chosen_by_segment=none_chosen_by_segment,
        )

        none_test_adjusted = apply_none_fallback_39d(
            default_base=base_test,
            pred_dict=nonacct_test_dict_39d,
            mask=none_test_39d,
            segment=subgroup_test_39d,
            global_best=none_global_best,
            chosen_by_segment=none_chosen_by_segment,
        )

        pred_oof[none_oof_39d] = none_oof_adjusted[none_oof_39d]
        pred_test[none_test_39d] = none_test_adjusted[none_test_39d]

    pred_oof = np.clip(pred_oof, 0, 100).astype(np.float32)
    pred_test = np.clip(pred_test, 0, 100).astype(np.float32)

    mse = float(mean_squared_error(y_arr_39d, pred_oof))

    screen_row = {
        "candidate_id": cfg_i,
        "base_name": base_name,
        "solver_segment_mode": solver_segment_mode,
        "none_policy": none_policy,
        "lambda_overlap": lambda_overlap,
        "lambda_direct_only": lambda_direct_only,
        "lambda_solver_global": solver_fit["global_lambda"],
        "n_solver_segment_lambdas": len(solver_fit["segment_lambdas"]),
        "none_global_best": none_global_best,
        "oof_mse": mse,
        "gain_vs_33a": mse33a_39d - mse,
        "gain_vs_39a": mse39a_39d - mse,
        "gain_vs_39b": mse39b_39d - mse,
        "gain_vs_39c": mse39c_39d - mse,
        "overlap_mse": float(mean_squared_error(y_arr_39d[overlap_oof_39d], pred_oof[overlap_oof_39d])) if int(overlap_oof_39d.sum()) > 0 else np.nan,
        "solver_only_mse": float(mean_squared_error(y_arr_39d[solver_only_oof_39d], pred_oof[solver_only_oof_39d])) if int(solver_only_oof_39d.sum()) > 0 else np.nan,
        "none_mse": float(mean_squared_error(y_arr_39d[none_oof_39d], pred_oof[none_oof_39d])) if int(none_oof_39d.sum()) > 0 else np.nan,
    }

    screen_rows_39d.append(screen_row)

    candidate_store_39d[cfg_i] = {
        "oof": pred_oof,
        "test": pred_test,
        "cfg": cfg,
        "screen_row": screen_row,
        "solver_segment_lambdas": solver_fit["segment_lambdas"],
        "solver_global_lambda": solver_fit["global_lambda"],
        "none_global_best": none_global_best,
        "none_chosen_by_segment": none_chosen_by_segment,
    }

screen39d = pd.DataFrame(screen_rows_39d).sort_values("oof_mse").reset_index(drop=True)

if len(screen39d) == 0:
    raise ValueError("No 39D candidates were created.")

best39d = screen39d.iloc[0]
best_id39d = int(best39d["candidate_id"])
best_oof39d = candidate_store_39d[best_id39d]["oof"]
best_test39d = candidate_store_39d[best_id39d]["test"]
best_mse39d = float(mean_squared_error(y_arr_39d, best_oof39d))

# ------------------------------------------------------------
# 6. Conservative final blend screen against existing best accounting models
# ------------------------------------------------------------

blend_candidates_39d = {
    "account39d_best": {
        "oof": best_oof39d,
        "test": best_test39d,
        "mse": best_mse39d,
    }
}

for name in ["account39c_final", "account39b_best", "account39a_best", "knn35a", "nn37b_rank1", "step36a", "anchor33a"]:
    if name in preds_39d:
        blend_candidates_39d[name] = {
            "oof": preds_39d[name]["oof"],
            "test": preds_39d[name]["test"],
            "mse": preds_39d[name]["oof_mse"],
        }

blend_grid_39d = np.linspace(0.0, 1.0, 501)
blend_rows_39d = []

blend_rows_39d.append({
    "candidate_type": "identity",
    "left": "account39d_best",
    "right": "",
    "w_left": 1.0,
    "oof_mse": best_mse39d,
    "gain_vs_39c": mse39c_39d - best_mse39d,
})

for other_name, other_obj in blend_candidates_39d.items():
    if other_name == "account39d_best":
        continue

    left = best_oof39d.astype(np.float64)
    right = other_obj["oof"].astype(np.float64)

    best_blend = None

    for w_left in blend_grid_39d:
        pred = np.clip(float(w_left) * left + (1.0 - float(w_left)) * right, 0, 100)
        mse = float(mean_squared_error(y_arr_39d, pred))

        row = {
            "candidate_type": "pairwise_blend_with_39d",
            "left": "account39d_best",
            "right": other_name,
            "w_left": float(w_left),
            "oof_mse": mse,
            "gain_vs_39c": mse39c_39d - mse,
        }

        if best_blend is None or row["oof_mse"] < best_blend["oof_mse"]:
            best_blend = row

    blend_rows_39d.append(best_blend)

blend_screen39d = pd.DataFrame(blend_rows_39d).sort_values("oof_mse").reset_index(drop=True)

best_blend39d = blend_screen39d.iloc[0]

if best_blend39d["candidate_type"] == "identity":
    final_oof39d = best_oof39d.copy()
    final_test39d = best_test39d.copy()
else:
    w_left = float(best_blend39d["w_left"])
    right_name = best_blend39d["right"]

    final_oof39d = np.clip(
        w_left * best_oof39d + (1.0 - w_left) * blend_candidates_39d[right_name]["oof"],
        0,
        100,
    ).astype(np.float32)

    final_test39d = np.clip(
        w_left * best_test39d + (1.0 - w_left) * blend_candidates_39d[right_name]["test"],
        0,
        100,
    ).astype(np.float32)

final_mse39d = float(mean_squared_error(y_arr_39d, final_oof39d))

# ------------------------------------------------------------
# 7. Fold gains
# ------------------------------------------------------------

folds39d = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_39d))
)

fold_rows_39d = []

for fold_num, (_, va_idx) in enumerate(folds39d, start=1):
    fold_rows_39d.append({
        "fold": fold_num,
        "mse_33a": float(mean_squared_error(y_arr_39d[va_idx], preds_39d["anchor33a"]["oof"][va_idx])),
        "mse_39a": float(mean_squared_error(y_arr_39d[va_idx], preds_39d["account39a_best"]["oof"][va_idx])),
        "mse_39b": float(mean_squared_error(y_arr_39d[va_idx], preds_39d["account39b_best"]["oof"][va_idx])),
        "mse_39c": float(mean_squared_error(y_arr_39d[va_idx], preds_39d["account39c_final"]["oof"][va_idx])),
        "mse_39d": float(mean_squared_error(y_arr_39d[va_idx], final_oof39d[va_idx])),
        "gain_39d_vs_39c": float(mean_squared_error(y_arr_39d[va_idx], preds_39d["account39c_final"]["oof"][va_idx]) - mean_squared_error(y_arr_39d[va_idx], final_oof39d[va_idx])),
        "n_overlap": int(overlap_oof_39d[va_idx].sum()),
        "n_solver_only": int(solver_only_oof_39d[va_idx].sum()),
        "n_none": int(none_oof_39d[va_idx].sum()),
    })

fold_gains39d = pd.DataFrame(fold_rows_39d)

# ------------------------------------------------------------
# 8. Tier performance for final 39D
# ------------------------------------------------------------

tier_perf_rows39d = []

tier_masks_39d = {
    "overlap": overlap_oof_39d,
    "solver_only": solver_only_oof_39d,
    "none": none_oof_39d,
    "solver_any": solver_cov_oof_39d,
    "accounting_union": direct_cov_oof_39d | solver_cov_oof_39d,
}

for tier_name, mask in tier_masks_39d.items():
    n = int(mask.sum())
    if n == 0:
        continue

    tier_perf_rows39d.append({
        "tier": tier_name,
        "n_rows": n,
        "mse_33a": float(mean_squared_error(y_arr_39d[mask], preds_39d["anchor33a"]["oof"][mask])),
        "mse_39a": float(mean_squared_error(y_arr_39d[mask], preds_39d["account39a_best"]["oof"][mask])),
        "mse_39b": float(mean_squared_error(y_arr_39d[mask], preds_39d["account39b_best"]["oof"][mask])),
        "mse_39c": float(mean_squared_error(y_arr_39d[mask], preds_39d["account39c_final"]["oof"][mask])),
        "mse_39d": float(mean_squared_error(y_arr_39d[mask], final_oof39d[mask])),
        "gain_39d_vs_39c": float(mean_squared_error(y_arr_39d[mask], preds_39d["account39c_final"]["oof"][mask]) - mean_squared_error(y_arr_39d[mask], final_oof39d[mask])),
    })

tier_perf39d = pd.DataFrame(tier_perf_rows39d)

# ------------------------------------------------------------
# 9. Save artifacts
# ------------------------------------------------------------

screen_path39d = "model_results/account39d_subgroup_solver_screen.csv"
segment_lambda_path39d = "model_results/account39d_segment_lambdas.csv"
none_selection_path39d = "model_results/account39d_none_base_selection.csv"
blend_screen_path39d = "model_results/account39d_blend_screen.csv"
fold_gains_path39d = "model_results/account39d_best_fold_gains.csv"
tier_perf_path39d = "model_results/account39d_tier_performance.csv"

oof_path39d = "model_results/oof_account39d_best.csv"
testpred_path39d = "model_results/testpred_account39d_best.csv"
submission_path39d = "submission_account39d_best.csv"

screen39d.to_csv(screen_path39d, index=False)
pd.DataFrame(segment_lambda_rows_39d).to_csv(segment_lambda_path39d, index=False)
pd.DataFrame(none_selection_rows_39d).to_csv(none_selection_path39d, index=False)
blend_screen39d.to_csv(blend_screen_path39d, index=False)
fold_gains39d.to_csv(fold_gains_path39d, index=False)
tier_perf39d.to_csv(tier_perf_path39d, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_39d),
    TARGET_COL: y_arr_39d,
    "pred_39c": preds_39d["account39c_final"]["oof"],
    "pred_39d_unblended": best_oof39d,
    "pred_clipped": final_oof39d,
    "subgroup": subgroup_train_39d,
    "n_bin": n_bin_train_39d,
    "overlap": overlap_oof_39d.astype(int),
    "solver_only": solver_only_oof_39d.astype(int),
    "none": none_oof_39d.astype(int),
}).to_csv(oof_path39d, index=False)

pd.DataFrame({
    ID_COL: test_ids_39d,
    "pred_39c": preds_39d["account39c_final"]["test"],
    "pred_39d_unblended": best_test39d,
    TARGET_COL: final_test39d,
    "subgroup": subgroup_test_39d,
    "n_bin": n_bin_test_39d,
    "overlap": overlap_test_39d.astype(int),
    "solver_only": solver_only_test_39d.astype(int),
    "none": none_test_39d.astype(int),
}).to_csv(testpred_path39d, index=False)

pd.DataFrame({
    ID_COL: test_ids_39d,
    TARGET_COL: final_test39d,
}).to_csv(submission_path39d, index=False)

# Validate.
sub39d = pd.read_csv(submission_path39d)
assert sub39d.shape[0] == n_test_39d
assert list(sub39d.columns) == [ID_COL, TARGET_COL]
assert sub39d[ID_COL].notna().all()
assert sub39d[TARGET_COL].notna().all()
assert np.isfinite(sub39d[TARGET_COL]).all()
assert sub39d[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 10. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("39D subgroup-specific solver shrinkage complete")
print("=" * 90)

print("\nBaseline comparison")
print("-------------------")
print(f"33A OOF MSE: {mse33a_39d:.6f}")
print(f"39A OOF MSE: {mse39a_39d:.6f}")
print(f"39B OOF MSE: {mse39b_39d:.6f}")
print(f"39C OOF MSE: {mse39c_39d:.6f}")

print("\nBest unblended 39D candidate")
print("----------------------------")
print(best39d.to_string())
print(f"\nBest unblended 39D OOF MSE: {best_mse39d:.6f}")
print(f"Gain vs 39C: {mse39c_39d - best_mse39d:.6f}")

print("\nBest final 39D blend")
print("--------------------")
print(best_blend39d.to_string())
print(f"\nFinal 39D OOF MSE: {final_mse39d:.6f}")
print(f"Final gain vs 39C: {mse39c_39d - final_mse39d:.6f}")

print("\nFold gains vs 39C")
print("-----------------")
print(fold_gains39d.to_string(index=False))
print("Min fold gain vs 39C:", float(fold_gains39d["gain_39d_vs_39c"].min()))

print("\nTier performance")
print("----------------")
print(tier_perf39d.to_string(index=False))

print("\nTop 20 39D candidates")
print("---------------------")
display(screen39d.head(20))

print("\nTop 20 39D final blends")
print("-----------------------")
display(blend_screen39d.head(20))

print("\nSaved files")
print("-----------")
print(screen_path39d)
print(segment_lambda_path39d)
print(none_selection_path39d)
print(blend_screen_path39d)
print(fold_gains_path39d)
print(tier_perf_path39d)
print(oof_path39d)
print(testpred_path39d)
print(submission_path39d)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path39d)
print("Shape:", sub39d.shape)
print(sub39d[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
if mse39c_39d - final_mse39d >= 3.0:
    print("39D clears the 3+ MSE threshold versus 39C. This is submission-worthy.")
elif mse39c_39d - final_mse39d >= 1.0:
    print("39D improves OOF but does not clear the 3+ threshold. Save artifact; submit only with strong structural justification.")
elif final_mse39d < mse39c_39d:
    print("39D gives only a marginal OOF gain. Do not submit under the current threshold.")
else:
    print("39D does not improve over 39C. Keep 39C as current best.")

39D. Subgroup-specific solver shrinkage + none-tier fallback selection

Tier coverage
-------------
OOF overlap:       53786
OOF direct only:   0
OOF solver only:   27807
OOF neither:       63328
Test overlap:      27298
Test direct only:  0
Test solver only:  17807
Test neither:      3202
Loaded anchor33a            | kind=non_accounting | OOF MSE 77.049866
Loaded knn35a               | kind=non_accounting | OOF MSE 76.453072
Loaded nn37b_rank1          | kind=non_accounting | OOF MSE 76.849442
Loaded step36a              | kind=non_accounting | OOF MSE 76.954796
Loaded blend33c             | kind=non_accounting | OOF MSE 76.859200
Loaded account39a_best      | kind=accounting     | OOF MSE 55.642780
Loaded account39b_best      | kind=accounting     | OOF MSE 53.720547
Loaded account39c_final     | kind=accounting     | OOF MSE 53.226631

Baseline OOF
------------
33A: 77.049866
39A: 55.642780
39B: 53.720547
39C: 53.226631

39D configs: 30
{'base_name': 'anchor33a', 'solver_segment_mo

,candidate_id,base_name,solver_segment_mode,none_policy,lambda_overlap,lambda_direct_only,lambda_solver_global,n_solver_segment_lambdas,none_global_best,oof_mse,gain_vs_33a,gain_vs_39a,gain_vs_39b,gain_vs_39c,overlap_mse,solver_only_mse,none_mse
0,10,knn35a,subgroup,segment_best_base,0.992,0.0,0.650,5,knn35a,52.826656,24.223209,2.816124,0.893890,0.399975,0.595324,64.277260,92.160088
1,16,nn37b_rank1,subgroup,segment_best_base,0.992,0.0,0.655,5,knn35a,52.830059,24.219807,2.812721,0.890488,0.396572,0.595445,64.294746,92.160088
2,28,blend33c,subgroup,segment_best_base,0.992,0.0,0.655,5,knn35a,52.831638,24.218227,2.811142,0.888908,0.394993,0.595360,64.303177,92.160088
3,22,step36a,subgroup,segment_best_base,0.992,0.0,0.655,5,knn35a,52.831985,24.217880,2.810795,0.888561,0.394646,0.595409,64.304848,92.160088
4,4,anchor33a,subgroup,segment_best_base,0.992,0.0,0.655,5,knn35a,52.834843,24.215023,2.807938,0.885704,0.391788,0.595420,64.319740,92.160088
5,12,knn35a,subgroup_nbin,segment_best_base,0.992,0.0,0.650,17,knn35a,52.862972,24.186893,2.779808,0.857574,0.363659,0.595324,64.466530,92.160088
6,18,nn37b_rank1,subgroup_nbin,segment_best_base,0.992,0.0,0.655,17,knn35a,52.866714,24.183151,2.776066,0.853832,0.359917,0.595445,64.485794,92.160088
7,30,blend33c,subgroup_nbin,segment_best_base,0.992,0.0,0.655,17,knn35a,52.868507,24.181358,2.774273,0.852039,0.358124,0.595360,64.495316,92.160088
8,24,step36a,subgroup_nbin,segment_best_base,0.992,0.0,0.655,17,knn35a,52.870205,24.179661,2.772575,0.850342,0.356426,0.595409,64.504059,92.160088
9,6,anchor33a,subgroup_nbin,segment_best_base,0.992,0.0,0.655,17,knn35a,52.871544,24.178322,2.771236,0.849003,0.355087,0.595420,64.511024,92.160088



Top 20 39D final blends
-----------------------


,candidate_type,left,right,w_left,oof_mse,gain_vs_39c
0,pairwise_blend_with_39d,account39d_best,account39c_final,0.894,52.820926,0.405705
1,pairwise_blend_with_39d,account39d_best,account39b_best,0.944,52.823427,0.403204
2,pairwise_blend_with_39d,account39d_best,knn35a,0.998,52.826544,0.400087
3,pairwise_blend_with_39d,account39d_best,account39a_best,1.000,52.826656,0.399976
4,pairwise_blend_with_39d,account39d_best,nn37b_rank1,1.000,52.826656,0.399976
5,pairwise_blend_with_39d,account39d_best,step36a,1.000,52.826656,0.399976
6,pairwise_blend_with_39d,account39d_best,anchor33a,1.000,52.826656,0.399976
7,identity,account39d_best,,1.000,52.826656,0.399975



Saved files
-----------
model_results/account39d_subgroup_solver_screen.csv
model_results/account39d_segment_lambdas.csv
model_results/account39d_none_base_selection.csv
model_results/account39d_blend_screen.csv
model_results/account39d_best_fold_gains.csv
model_results/account39d_tier_performance.csv
model_results/oof_account39d_best.csv
model_results/testpred_account39d_best.csv
submission_account39d_best.csv

Submission validation
---------------------
File: submission_account39d_best.csv
Shape: (48307, 2)
count    48307.000000
mean        54.168934
std         25.804494
min          0.008840
25%         33.426792
50%         52.788116
75%         75.247940
max        100.000000
Name: PERCENT_PROFICIENT, dtype: float64

Decision rule
-------------
39D gives only a marginal OOF gain. Do not submit under the current threshold.


### 39D. Subgroup-Specific Solver Shrinkage

This section tested whether the accounting solver could be improved by tuning shrinkage separately by subgroup and by selecting fallback models for rows not covered by accounting.

The best 39D model used `knn35a` as the base model, subgroup-specific solver shrinkage, and segment-specific fallback selection. It improved OOF MSE from the 39C value of `53.226631` to `52.820927`, a gain of only `0.405704`.

Although the fold gains were positive, this improvement is below the current threshold for using another Kaggle submission. Since 39C already reached public MSE `31.431`, further accounting-only shrinkage risks overfitting without enough expected leaderboard gain.

Conclusion: the accounting branch remains the main breakthrough, but marginal accounting refinements should be stopped unless they produce a much larger OOF improvement. The next work should move to other structural model families.

### Public Leaderboard Check: 39D Becomes Protected Anchor

The 39D subgroup-specific solver shrinkage model produced a public leaderboard MSE of `30.752`, improving over the previous best 39C public MSE of `31.431`.

This result changes the protected anchor for the remainder of the project:

`protected anchor = 39D`

Although 39D only improved 39C by about `0.406` OOF MSE, it transferred well to the public leaderboard. This suggests that small accounting-covered improvements can matter more than large ordinary OOF gains concentrated in the `none` tier.

Going forward, new candidates should be compared against 39D. In particular, models that mainly improve the training `none` tier should be treated with skepticism, while models that improve the public-heavy accounting-covered or solver-only tiers may be more valuable.

In [ ]:
# another kernel crash happened in next cell for neural nets, it was deleted

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

paths = [
    "submission_account39c_final_best.csv",
    "model_results/oof_account39c_final_best.csv",
    "model_results/testpred_account39c_final_best.csv",
    "submission_account39b_solver_best.csv",
    "model_results/oof_account39b_solver_best.csv",
    "model_results/testpred_account39b_solver_best.csv",
    "submission_account39a_best.csv",
    "model_results/oof_account39a_best.csv",
    "model_results/testpred_account39a_best.csv",
    "model_results/oof_account39d_best.csv",
    "model_results/testpred_account39d_best.csv",
]

print("Artifact recovery check")
print("-----------------------")
for p in paths:
    print(f"{p:65s}", Path(p).exists())

sub = pd.read_csv("submission_account39c_final_best.csv")
print("\n39C submission check")
print("--------------------")
print(sub.shape)
print(sub.columns.tolist())
print(sub["PERCENT_PROFICIENT"].describe())

assert sub.shape == (48307, 2)
assert sub.columns.tolist() == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
assert sub["PERCENT_PROFICIENT"].notna().all()
assert np.isfinite(sub["PERCENT_PROFICIENT"]).all()
assert sub["PERCENT_PROFICIENT"].between(0, 100).all()

print("\nRecovery OK. Current best remains submission_account39c_final_best.csv")

Artifact recovery check
-----------------------
submission_account39c_final_best.csv                              True
model_results/oof_account39c_final_best.csv                       True
model_results/testpred_account39c_final_best.csv                  True
submission_account39b_solver_best.csv                             True
model_results/oof_account39b_solver_best.csv                      True
model_results/testpred_account39b_solver_best.csv                 True
submission_account39a_best.csv                                    True
model_results/oof_account39a_best.csv                             True
model_results/testpred_account39a_best.csv                        True
model_results/oof_account39d_best.csv                             True
model_results/testpred_account39d_best.csv                        True

39C submission check
--------------------
(48307, 2)
['ASSESSMENT_ID', 'PERCENT_PROFICIENT']
count    48307.000000
mean        54.186695
std         25.771232
min        

In [30]:
# ============================================================
# 40A-1. Hierarchical encoding setup for 39C residuals
# ============================================================
#
# Safe diagnostic/setup cell only.
#
# This cell:
#   1. Loads current best 39C OOF/test predictions.
#   2. Computes residual_39C = y - pred_39C.
#   3. Recovers accounting tiers: overlap / solver_only / none.
#   4. Audits candidate hierarchy keys for target/residual encoding.
#   5. Saves a small setup summary.
#
# It does NOT:
#   - build a huge dense matrix,
#   - fit LightGBM,
#   - fit neural nets,
#   - construct all features at once.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("40A-1. Hierarchical encoding setup for 39C residuals")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required checks
# ------------------------------------------------------------

required = ["raw_train_te", "raw_test_te", "y_train"]
missing = [x for x in required if x not in globals()]
if missing:
    raise ValueError(f"Missing required objects: {missing}. Rerun safe recovery/setup first.")

y_arr_40a = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_40a = len(y_arr_40a)
n_test_40a = len(raw_test_te)

if len(raw_train_te) != n_train_40a:
    raise ValueError("raw_train_te and y_train row count mismatch.")

# ------------------------------------------------------------
# 2. Load 39C predictions
# ------------------------------------------------------------

oof39c_path = "model_results/oof_account39c_final_best.csv"
test39c_path = "model_results/testpred_account39c_final_best.csv"

if not Path(oof39c_path).exists() or not Path(test39c_path).exists():
    raise FileNotFoundError("Missing 39C OOF/test artifacts.")

oof39c = pd.read_csv(oof39c_path)
test39c = pd.read_csv(test39c_path)

if "row_index" in oof39c.columns:
    oof39c = oof39c.sort_values("row_index").reset_index(drop=True)

if len(oof39c) != n_train_40a:
    raise ValueError("39C OOF row count mismatch.")
if len(test39c) != n_test_40a:
    raise ValueError("39C test row count mismatch.")

if "pred_clipped" in oof39c.columns:
    pred39c_oof = oof39c["pred_clipped"].to_numpy(dtype=np.float32)
elif TARGET_COL in oof39c.columns:
    pred39c_oof = oof39c[TARGET_COL].to_numpy(dtype=np.float32)
else:
    raise ValueError("Could not find 39C OOF prediction column.")

if TARGET_COL in test39c.columns:
    pred39c_test = test39c[TARGET_COL].to_numpy(dtype=np.float32)
else:
    numeric_cols = [
        c for c in test39c.columns
        if c != ID_COL and pd.api.types.is_numeric_dtype(test39c[c])
    ]
    if len(numeric_cols) == 0:
        raise ValueError("Could not find 39C test prediction column.")
    pred39c_test = test39c[numeric_cols[0]].to_numpy(dtype=np.float32)

if ID_COL in test39c.columns:
    test_ids_40a = test39c[ID_COL].to_numpy()
elif "test_ids" in globals():
    test_ids_40a = np.asarray(test_ids)
else:
    raise ValueError("Could not recover test IDs.")

pred39c_oof = np.clip(pred39c_oof, 0, 100).astype(np.float32)
pred39c_test = np.clip(pred39c_test, 0, 100).astype(np.float32)

resid39c = (y_arr_40a - pred39c_oof).astype(np.float32)
mse39c = float(mean_squared_error(y_arr_40a, pred39c_oof))

print("\n39C baseline")
print("------------")
print(f"39C OOF MSE: {mse39c:.6f}")
print(f"Residual mean: {float(np.mean(resid39c)):.6f}")
print(f"Residual std:  {float(np.std(resid39c)):.6f}")

# ------------------------------------------------------------
# 3. Recover accounting tiers from 39A/39B raw artifacts
# ------------------------------------------------------------

raw39a_oof_path = "model_results/account39a_raw_oof_reconstruction.csv"
raw39a_test_path = "model_results/account39a_raw_test_reconstruction.csv"
raw39b_oof_path = "model_results/account39b_solver_raw_oof.csv"
raw39b_test_path = "model_results/account39b_solver_raw_test.csv"

for p in [raw39a_oof_path, raw39a_test_path, raw39b_oof_path, raw39b_test_path]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing tier artifact: {p}")

raw39a_oof = pd.read_csv(raw39a_oof_path)
raw39a_test = pd.read_csv(raw39a_test_path)
raw39b_oof = pd.read_csv(raw39b_oof_path)
raw39b_test = pd.read_csv(raw39b_test_path)

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_cov_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)

solver_cov_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_cov_oof & solver_cov_oof
direct_only_oof = direct_cov_oof & ~solver_cov_oof
solver_only_oof = solver_cov_oof & ~direct_cov_oof
none_oof = ~(direct_cov_oof | solver_cov_oof)

overlap_test = direct_cov_test & solver_cov_test
direct_only_test = direct_cov_test & ~solver_cov_test
solver_only_test = solver_cov_test & ~direct_cov_test
none_test = ~(direct_cov_test | solver_cov_test)

tier_masks_oof_40a = {
    "overlap": overlap_oof,
    "direct_only": direct_only_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test_40a = {
    "overlap": overlap_test,
    "direct_only": direct_only_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

tier_rows = []

print("\n39C residual/tier diagnostic")
print("----------------------------")

for tier, mask in tier_masks_oof_40a.items():
    n = int(mask.sum())
    n_test_tier = int(tier_masks_test_40a[tier].sum())

    if n > 0:
        mse_tier = float(mean_squared_error(y_arr_40a[mask], pred39c_oof[mask]))
        resid_mean = float(np.mean(resid39c[mask]))
        resid_std = float(np.std(resid39c[mask]))
    else:
        mse_tier = np.nan
        resid_mean = np.nan
        resid_std = np.nan

    tier_rows.append({
        "tier": tier,
        "train_rows": n,
        "test_rows": n_test_tier,
        "train_rate": n / n_train_40a,
        "test_rate": n_test_tier / n_test_40a,
        "mse_39c": mse_tier,
        "residual_mean": resid_mean,
        "residual_std": resid_std,
    })

tier_diag40a = pd.DataFrame(tier_rows)

print(tier_diag40a.to_string(index=False))

# ------------------------------------------------------------
# 4. Ensure N_STUDENTS bin exists
# ------------------------------------------------------------

def make_n_bin_40a(x):
    return pd.cut(
        pd.Series(x).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"],
    ).astype("string").fillna("<NA>").astype(str)

raw_train_te = raw_train_te.copy()
raw_test_te = raw_test_te.copy()

if "_N_STUDENTS_BIN_TE" not in raw_train_te.columns:
    raw_train_te["_N_STUDENTS_BIN_TE"] = make_n_bin_40a(raw_train_te["N_STUDENTS"]).to_numpy()

if "_N_STUDENTS_BIN_TE" not in raw_test_te.columns:
    raw_test_te["_N_STUDENTS_BIN_TE"] = make_n_bin_40a(raw_test_te["N_STUDENTS"]).to_numpy()

# ------------------------------------------------------------
# 5. Audit candidate hierarchy keys
# ------------------------------------------------------------

candidate_keys_40a = [
    ("ASSESSMENT_NAME",),
    ("SUBGROUP_NAME",),
    ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("ASSESSMENT_NAME", "_N_STUDENTS_BIN_TE"),
    ("ASSESSMENT_NAME", "SUBGROUP_NAME", "_N_STUDENTS_BIN_TE"),

    ("SCHOOL",),
    ("SCHOOL", "ASSESSMENT_NAME"),
    ("SCHOOL", "SUBGROUP_NAME"),

    ("DISTRICT",),
    ("DISTRICT", "ASSESSMENT_NAME"),
    ("DISTRICT", "SUBGROUP_NAME"),

    ("COUNTY",),
    ("COUNTY", "ASSESSMENT_NAME"),
    ("COUNTY", "SUBGROUP_NAME"),

    ("REGION",),
    ("REGION", "ASSESSMENT_NAME"),

    ("DISTRICT_TYPE",),
    ("DISTRICT_TYPE", "ASSESSMENT_NAME"),
]

usable_keys_40a = []

def make_group_key_40a(df, cols):
    cols = tuple(cols)
    if len(cols) == 1:
        return pd.Series(df[cols[0]]).astype("string").fillna("<NA>").astype(str)
    return (
        df.loc[:, list(cols)]
        .astype("string")
        .fillna("<NA>")
        .astype(str)
        .agg(" || ".join, axis=1)
    )

key_rows = []

for cols in candidate_keys_40a:
    cols = tuple(cols)

    if not all(c in raw_train_te.columns for c in cols):
        continue
    if not all(c in raw_test_te.columns for c in cols):
        continue

    train_key = make_group_key_40a(raw_train_te, cols)
    test_key = make_group_key_40a(raw_test_te, cols)

    train_vc = train_key.value_counts(dropna=False)
    test_vc = test_key.value_counts(dropna=False)

    seen_train = set(train_vc.index.astype(str))
    test_covered = test_key.astype(str).isin(seen_train)

    usable_keys_40a.append(cols)

    key_rows.append({
        "key": " × ".join(cols),
        "n_train_groups": int(train_vc.shape[0]),
        "n_test_groups": int(test_vc.shape[0]),
        "median_train_count": float(train_vc.median()),
        "mean_train_count": float(train_vc.mean()),
        "singleton_share": float((train_vc == 1).mean()),
        "test_row_coverage": float(test_covered.mean()),
        "test_uncovered_rows": int((~test_covered).sum()),
    })

key_audit40a = pd.DataFrame(key_rows).sort_values(
    ["test_row_coverage", "median_train_count"],
    ascending=[False, False],
).reset_index(drop=True)

print("\nCandidate hierarchy key audit")
print("-----------------------------")
print(key_audit40a.to_string(index=False))

# ------------------------------------------------------------
# 6. Save setup artifacts
# ------------------------------------------------------------

tier_diag_path40a = "model_results/hte40a_tier_setup_diag.csv"
key_audit_path40a = "model_results/hte40a_key_audit.csv"
setup_summary_path40a = "model_results/hte40a_setup_summary.csv"

tier_diag40a.to_csv(tier_diag_path40a, index=False)
key_audit40a.to_csv(key_audit_path40a, index=False)

pd.DataFrame({
    "item": [
        "mse39c",
        "n_train",
        "n_test",
        "n_usable_keys",
        "overlap_train_rows",
        "solver_only_train_rows",
        "none_train_rows",
        "overlap_test_rows",
        "solver_only_test_rows",
        "none_test_rows",
    ],
    "value": [
        mse39c,
        n_train_40a,
        n_test_40a,
        len(usable_keys_40a),
        int(overlap_oof.sum()),
        int(solver_only_oof.sum()),
        int(none_oof.sum()),
        int(overlap_test.sum()),
        int(solver_only_test.sum()),
        int(none_test.sum()),
    ],
}).to_csv(setup_summary_path40a, index=False)

print("\nSaved setup files")
print("-----------------")
print(tier_diag_path40a)
print(key_audit_path40a)
print(setup_summary_path40a)

print("\n40A-1 complete.")
print("Next step should be 40A-2: build ONE or TWO hierarchy feature blocks only, then check memory and residual signal.")

40A-1. Hierarchical encoding setup for 39C residuals

39C baseline
------------
39C OOF MSE: 53.226631
Residual mean: -0.147212
Residual std:  7.294173

39C residual/tier diagnostic
----------------------------
       tier  train_rows  test_rows  train_rate  test_rate   mse_39c  residual_mean  residual_std
    overlap       53786      27298    0.371140   0.565094  0.595326       0.003323      0.771567
direct_only           0          0    0.000000   0.000000       NaN            NaN           NaN
solver_only       27807      17807    0.191877   0.368622 65.890625      -0.245121      8.113603
       none       63328       3202    0.436983   0.066284 92.366982      -0.232073      9.607971

Candidate hierarchy key audit
-----------------------------
                                                 key  n_train_groups  n_test_groups  median_train_count  mean_train_count  singleton_share  test_row_coverage  test_uncovered_rows
                                       SUBGROUP_NAME            

In [31]:
# ============================================================
# 40A-2. Build compact OOF-safe hierarchical residual encodings
# ============================================================
#
# Requires 40A-1 variables:
#   y_arr_40a
#   pred39c_oof
#   pred39c_test
#   resid39c
#   raw_train_te
#   raw_test_te
#   tier_masks_oof_40a
#   tier_masks_test_40a
#
# This cell:
#   - Builds a small OOF-safe hierarchy-encoding feature matrix.
#   - Uses the same 5 folds for feature construction that 40A-3 will use.
#   - Saves features to disk as compressed .npz.
#   - Prints signal diagnostics by residual correlation, including solver_only / none tiers.
#
# It does NOT fit LightGBM or any large model.
# ============================================================

import os
import re
import gc
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")

print("=" * 90)
print("40A-2. Compact OOF-safe hierarchical residual encodings")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required object checks
# ------------------------------------------------------------

required_40a2 = [
    "y_arr_40a",
    "pred39c_oof",
    "pred39c_test",
    "resid39c",
    "raw_train_te",
    "raw_test_te",
    "tier_masks_oof_40a",
    "tier_masks_test_40a",
]

missing_40a2 = [x for x in required_40a2 if x not in globals()]
if missing_40a2:
    raise ValueError(f"Missing required 40A-1 objects: {missing_40a2}. Run 40A-1 first.")

y_arr_40a = np.asarray(y_arr_40a, dtype=np.float32).reshape(-1)
pred39c_oof = np.asarray(pred39c_oof, dtype=np.float32).reshape(-1)
pred39c_test = np.asarray(pred39c_test, dtype=np.float32).reshape(-1)
resid39c = np.asarray(resid39c, dtype=np.float32).reshape(-1)

n_train_40a2 = len(y_arr_40a)
n_test_40a2 = len(raw_test_te)

if len(raw_train_te) != n_train_40a2:
    raise ValueError("raw_train_te length mismatch.")
if len(pred39c_oof) != n_train_40a2:
    raise ValueError("pred39c_oof length mismatch.")
if len(pred39c_test) != n_test_40a2:
    raise ValueError("pred39c_test length mismatch.")

mse39c_40a2 = float(mean_squared_error(y_arr_40a, pred39c_oof))

print("\nBaseline")
print("--------")
print(f"39C OOF MSE: {mse39c_40a2:.6f}")

# ------------------------------------------------------------
# 2. Utility helpers
# ------------------------------------------------------------

def safe_name_40a2(cols):
    name = "__".join(cols)
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def clean_str_40a2(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def make_group_key_40a2(df, cols):
    cols = tuple(cols)

    if len(cols) == 1:
        return clean_str_40a2(df[cols[0]]).reset_index(drop=True)

    return (
        df.loc[:, list(cols)]
        .astype("string")
        .fillna("<NA>")
        .astype(str)
        .agg(" || ".join, axis=1)
        .reset_index(drop=True)
    )

def fit_group_stats_40a2(keys, residual, target, alpha):
    keys = pd.Series(keys).astype(str).reset_index(drop=True)
    residual = np.asarray(residual, dtype=np.float64).reshape(-1)
    target = np.asarray(target, dtype=np.float64).reshape(-1)

    global_res_mean = float(np.nanmean(residual))
    global_abs_res_mean = float(np.nanmean(np.abs(residual)))
    global_target_mean = float(np.nanmean(target))

    tmp = pd.DataFrame({
        "key": keys,
        "residual": residual,
        "abs_residual": np.abs(residual),
        "target": target,
    })

    stats = tmp.groupby("key", sort=False).agg(
        cnt=("residual", "size"),
        sum_residual=("residual", "sum"),
        sum_abs_residual=("abs_residual", "sum"),
        sum_target=("target", "sum"),
    )

    stats["res_mean_s"] = (
        stats["sum_residual"] + float(alpha) * global_res_mean
    ) / (stats["cnt"] + float(alpha))

    stats["abs_res_mean_s"] = (
        stats["sum_abs_residual"] + float(alpha) * global_abs_res_mean
    ) / (stats["cnt"] + float(alpha))

    stats["target_mean_s"] = (
        stats["sum_target"] + float(alpha) * global_target_mean
    ) / (stats["cnt"] + float(alpha))

    stats["log_count"] = np.log1p(stats["cnt"].astype(float))

    defaults = {
        "res_mean_s": global_res_mean,
        "abs_res_mean_s": global_abs_res_mean,
        "target_mean_s": global_target_mean,
        "log_count": 0.0,
    }

    return stats, defaults

def apply_group_stats_40a2(keys, stats, defaults, prefix):
    keys = pd.Series(keys).astype(str).reset_index(drop=True)

    out = pd.DataFrame(index=np.arange(len(keys)))

    out[f"{prefix}_res_mean"] = (
        keys.map(stats["res_mean_s"])
        .fillna(defaults["res_mean_s"])
        .astype(np.float32)
    )

    out[f"{prefix}_abs_res_mean"] = (
        keys.map(stats["abs_res_mean_s"])
        .fillna(defaults["abs_res_mean_s"])
        .astype(np.float32)
    )

    out[f"{prefix}_target_mean"] = (
        keys.map(stats["target_mean_s"])
        .fillna(defaults["target_mean_s"])
        .astype(np.float32)
    )

    out[f"{prefix}_log_count"] = (
        keys.map(stats["log_count"])
        .fillna(defaults["log_count"])
        .astype(np.float32)
    )

    return out

def abs_corr_40a2(x, y):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)

    finite = np.isfinite(x) & np.isfinite(y)
    if finite.sum() < 3:
        return np.nan

    x = x[finite]
    y = y[finite]

    x = x - x.mean()
    y = y - y.mean()

    dx = np.sqrt(np.dot(x, x))
    dy = np.sqrt(np.dot(y, y))

    if dx <= 1e-12 or dy <= 1e-12:
        return 0.0

    return abs(float(np.dot(x, y) / (dx * dy)))

# ------------------------------------------------------------
# 3. Compact hierarchy keys
# ------------------------------------------------------------
# We are deliberately not using every possible key.
# This block is small enough to inspect and reuse safely.

encoding_specs_40a2 = [
    (("ASSESSMENT_NAME",), 20.0),
    (("SUBGROUP_NAME",), 20.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME"), 40.0),

    (("SCHOOL",), 100.0),
    (("SCHOOL", "ASSESSMENT_NAME"), 220.0),
    (("SCHOOL", "SUBGROUP_NAME"), 140.0),

    (("DISTRICT",), 60.0),
    (("DISTRICT", "ASSESSMENT_NAME"), 140.0),

    (("COUNTY",), 40.0),
    (("COUNTY", "ASSESSMENT_NAME"), 80.0),
]

usable_specs_40a2 = []

for cols, alpha in encoding_specs_40a2:
    cols = tuple(cols)
    if all(c in raw_train_te.columns for c in cols) and all(c in raw_test_te.columns for c in cols):
        usable_specs_40a2.append((cols, float(alpha)))

print("\nEncoding specs used")
print("-------------------")
for cols, alpha in usable_specs_40a2:
    print(f"{cols} | alpha={alpha}")

if len(usable_specs_40a2) == 0:
    raise ValueError("No usable hierarchy specs found.")

prefix_by_key_40a2 = {
    cols: f"hte40a2_{safe_name_40a2(cols)}"
    for cols, _ in usable_specs_40a2
}

# ------------------------------------------------------------
# 4. Build OOF-safe features
# ------------------------------------------------------------

folds40a2 = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_40a2))
)

fold_id_40a2 = np.full(n_train_40a2, -1, dtype=np.int16)

for fold_num, (_, va_idx) in enumerate(folds40a2, start=1):
    fold_id_40a2[va_idx] = fold_num

if (fold_id_40a2 < 0).any():
    raise ValueError("Some rows did not receive a fold id.")

feature_parts_oof_40a2 = []
feature_parts_test_40a2 = []

for spec_i, (cols, alpha) in enumerate(usable_specs_40a2, start=1):
    print(f"\nBuilding OOF encoding {spec_i}/{len(usable_specs_40a2)}: {cols}")

    prefix = prefix_by_key_40a2[cols]

    key_train_all = make_group_key_40a2(raw_train_te, cols)
    key_test_all = make_group_key_40a2(raw_test_te, cols)

    feature_cols = [
        f"{prefix}_res_mean",
        f"{prefix}_abs_res_mean",
        f"{prefix}_target_mean",
        f"{prefix}_log_count",
    ]

    oof_arr = np.zeros((n_train_40a2, len(feature_cols)), dtype=np.float32)

    for fold_num, (tr_idx, va_idx) in enumerate(folds40a2, start=1):
        stats_fold, defaults_fold = fit_group_stats_40a2(
            keys=key_train_all.iloc[tr_idx],
            residual=resid39c[tr_idx],
            target=y_arr_40a[tr_idx],
            alpha=alpha,
        )

        enc_val = apply_group_stats_40a2(
            keys=key_train_all.iloc[va_idx],
            stats=stats_fold,
            defaults=defaults_fold,
            prefix=prefix,
        )

        oof_arr[va_idx, :] = enc_val[feature_cols].to_numpy(dtype=np.float32)

    # Full fit for test features.
    stats_full, defaults_full = fit_group_stats_40a2(
        keys=key_train_all,
        residual=resid39c,
        target=y_arr_40a,
        alpha=alpha,
    )

    enc_test = apply_group_stats_40a2(
        keys=key_test_all,
        stats=stats_full,
        defaults=defaults_full,
        prefix=prefix,
    )

    feature_parts_oof_40a2.append(pd.DataFrame(oof_arr, columns=feature_cols))
    feature_parts_test_40a2.append(enc_test[feature_cols].reset_index(drop=True))

    print("  added columns:", feature_cols)

X_hte_oof_40a2 = pd.concat(feature_parts_oof_40a2, axis=1)
X_hte_test_40a2 = pd.concat(feature_parts_test_40a2, axis=1)

# ------------------------------------------------------------
# 5. Add explicit hierarchy deltas
# ------------------------------------------------------------

def add_delta_40a2(df, hi_key, lo_key, suffix, name):
    hi_pref = prefix_by_key_40a2.get(tuple(hi_key))
    lo_pref = prefix_by_key_40a2.get(tuple(lo_key))

    if hi_pref is None or lo_pref is None:
        return df

    hi_col = f"{hi_pref}_{suffix}"
    lo_col = f"{lo_pref}_{suffix}"

    if hi_col not in df.columns or lo_col not in df.columns:
        return df

    df[f"delta_{name}_{suffix}"] = (df[hi_col] - df[lo_col]).astype(np.float32)
    return df

delta_specs_40a2 = [
    (("SCHOOL", "ASSESSMENT_NAME"), ("DISTRICT", "ASSESSMENT_NAME"), "res_mean", "school_assess_minus_district_assess"),
    (("DISTRICT", "ASSESSMENT_NAME"), ("COUNTY", "ASSESSMENT_NAME"), "res_mean", "district_assess_minus_county_assess"),
    (("COUNTY", "ASSESSMENT_NAME"), ("ASSESSMENT_NAME",), "res_mean", "county_assess_minus_assess"),
    (("SCHOOL", "SUBGROUP_NAME"), ("SCHOOL",), "res_mean", "school_subgroup_minus_school"),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME"), ("ASSESSMENT_NAME",), "res_mean", "assess_subgroup_minus_assess"),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME"), ("SUBGROUP_NAME",), "res_mean", "assess_subgroup_minus_subgroup"),

    (("SCHOOL", "ASSESSMENT_NAME"), ("DISTRICT", "ASSESSMENT_NAME"), "target_mean", "school_assess_minus_district_assess"),
    (("DISTRICT", "ASSESSMENT_NAME"), ("COUNTY", "ASSESSMENT_NAME"), "target_mean", "district_assess_minus_county_assess"),
    (("COUNTY", "ASSESSMENT_NAME"), ("ASSESSMENT_NAME",), "target_mean", "county_assess_minus_assess"),
    (("SCHOOL", "SUBGROUP_NAME"), ("SCHOOL",), "target_mean", "school_subgroup_minus_school"),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME"), ("ASSESSMENT_NAME",), "target_mean", "assess_subgroup_minus_assess"),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME"), ("SUBGROUP_NAME",), "target_mean", "assess_subgroup_minus_subgroup"),
]

for hi_key, lo_key, suffix, name in delta_specs_40a2:
    X_hte_oof_40a2 = add_delta_40a2(X_hte_oof_40a2, hi_key, lo_key, suffix, name)
    X_hte_test_40a2 = add_delta_40a2(X_hte_test_40a2, hi_key, lo_key, suffix, name)

# Sanity align.
X_hte_test_40a2 = X_hte_test_40a2[X_hte_oof_40a2.columns]

print("\nFinal compact HTE feature matrix")
print("--------------------------------")
print("OOF shape: ", X_hte_oof_40a2.shape)
print("Test shape:", X_hte_test_40a2.shape)
print("Memory OOF MB:", round(X_hte_oof_40a2.memory_usage(deep=True).sum() / 1024**2, 2))
print("Memory test MB:", round(X_hte_test_40a2.memory_usage(deep=True).sum() / 1024**2, 2))

# ------------------------------------------------------------
# 6. Residual-signal diagnostics
# ------------------------------------------------------------

signal_rows_40a2 = []

for col in X_hte_oof_40a2.columns:
    x = X_hte_oof_40a2[col].to_numpy(dtype=np.float32)

    row = {
        "feature": col,
        "abs_corr_all": abs_corr_40a2(x, resid39c),
    }

    for tier, mask in tier_masks_oof_40a.items():
        if int(mask.sum()) >= 100:
            row[f"abs_corr_{tier}"] = abs_corr_40a2(x[mask], resid39c[mask])
        else:
            row[f"abs_corr_{tier}"] = np.nan

    signal_rows_40a2.append(row)

signal40a2 = pd.DataFrame(signal_rows_40a2)

sort_cols = ["abs_corr_none", "abs_corr_solver_only", "abs_corr_all"]
sort_cols = [c for c in sort_cols if c in signal40a2.columns]

signal40a2 = signal40a2.sort_values(
    sort_cols,
    ascending=False,
).reset_index(drop=True)

print("\nTop residual-signal features")
print("----------------------------")
display(signal40a2.head(30))

# ------------------------------------------------------------
# 7. Save compact feature block
# ------------------------------------------------------------

features_npz_path40a2 = "model_results/hte40a2_compact_features.npz"
feature_cols_path40a2 = "model_results/hte40a2_feature_columns.csv"
signal_path40a2 = "model_results/hte40a2_feature_signal.csv"
fold_path40a2 = "model_results/hte40a2_fold_ids.csv"

np.savez_compressed(
    features_npz_path40a2,
    X_oof=X_hte_oof_40a2.to_numpy(dtype=np.float32),
    X_test=X_hte_test_40a2.to_numpy(dtype=np.float32),
    fold_id=fold_id_40a2,
    columns=np.array(X_hte_oof_40a2.columns, dtype=object),
)

pd.DataFrame({"feature": X_hte_oof_40a2.columns}).to_csv(feature_cols_path40a2, index=False)
signal40a2.to_csv(signal_path40a2, index=False)
pd.DataFrame({"row_index": np.arange(n_train_40a2), "fold_id": fold_id_40a2}).to_csv(fold_path40a2, index=False)

print("\nSaved files")
print("-----------")
print(features_npz_path40a2)
print(feature_cols_path40a2)
print(signal_path40a2)
print(fold_path40a2)

print("\n40A-2 complete.")
print("Next step: 40A-3 will train a small residual model using only this compact feature block.")
gc.collect()

40A-2. Compact OOF-safe hierarchical residual encodings

Baseline
--------
39C OOF MSE: 53.226631

Encoding specs used
-------------------
('ASSESSMENT_NAME',) | alpha=20.0
('SUBGROUP_NAME',) | alpha=20.0
('ASSESSMENT_NAME', 'SUBGROUP_NAME') | alpha=40.0
('SCHOOL',) | alpha=100.0
('SCHOOL', 'ASSESSMENT_NAME') | alpha=220.0
('SCHOOL', 'SUBGROUP_NAME') | alpha=140.0
('DISTRICT',) | alpha=60.0
('DISTRICT', 'ASSESSMENT_NAME') | alpha=140.0
('COUNTY',) | alpha=40.0
('COUNTY', 'ASSESSMENT_NAME') | alpha=80.0

Building OOF encoding 1/10: ('ASSESSMENT_NAME',)
  added columns: ['hte40a2_ASSESSMENT_NAME_res_mean', 'hte40a2_ASSESSMENT_NAME_abs_res_mean', 'hte40a2_ASSESSMENT_NAME_target_mean', 'hte40a2_ASSESSMENT_NAME_log_count']

Building OOF encoding 2/10: ('SUBGROUP_NAME',)
  added columns: ['hte40a2_SUBGROUP_NAME_res_mean', 'hte40a2_SUBGROUP_NAME_abs_res_mean', 'hte40a2_SUBGROUP_NAME_target_mean', 'hte40a2_SUBGROUP_NAME_log_count']

Building OOF encoding 3/10: ('ASSESSMENT_NAME', 'SUBGROUP_NAM

,feature,abs_corr_all,abs_corr_overlap,abs_corr_direct_only,abs_corr_solver_only,abs_corr_none
0,hte40a2_SCHOOL_ASSESSMENT_NAME_res_mean,0.139017,0.003270,NaN,0.068506,0.202979
1,hte40a2_SCHOOL_res_mean,0.090660,0.003100,NaN,0.061622,0.132501
2,hte40a2_SCHOOL_SUBGROUP_NAME_res_mean,0.089473,0.006983,NaN,0.083307,0.122392
3,delta_school_subgroup_minus_school_res_mean,0.061721,0.000294,NaN,0.031258,0.094286
4,hte40a2_DISTRICT_ASSESSMENT_NAME_res_mean,0.045152,0.003427,NaN,0.016615,0.070548
5,hte40a2_DISTRICT_res_mean,0.035171,0.001323,NaN,0.013920,0.055289
6,delta_county_assess_minus_assess_res_mean,0.023766,0.000314,NaN,0.007064,0.037314
7,hte40a2_SCHOOL_ASSESSMENT_NAME_target_mean,0.016357,0.004146,NaN,0.009310,0.033347
8,hte40a2_COUNTY_res_mean,0.020521,0.001929,NaN,0.017135,0.028902
9,hte40a2_COUNTY_ASSESSMENT_NAME_res_mean,0.015251,0.000856,NaN,0.003985,0.024356



Saved files
-----------
model_results/hte40a2_compact_features.npz
model_results/hte40a2_feature_columns.csv
model_results/hte40a2_feature_signal.csv
model_results/hte40a2_fold_ids.csv

40A-2 complete.
Next step: 40A-3 will train a small residual model using only this compact feature block.


0

In [32]:
# ============================================================
# 40A-3. Small residual model using compact hierarchical encodings
# ============================================================
#
# Uses the saved 40A-2 feature block only:
#   model_results/hte40a2_compact_features.npz
#
# Target:
#   residual_39C = y - pred_39C
#
# Prediction:
#   final = pred_39C + lambda * predicted_residual
#
# This cell is intentionally small:
#   - 52 compact HTE features + a few tier flags
#   - Ridge and small HistGradientBoosting only
#   - 5-fold OOF using the existing fold_id from 40A-2
#   - global and tier-specific lambda scans
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

try:
    from sklearn.ensemble import HistGradientBoostingRegressor
    HAVE_HGB_40A3 = True
except Exception as e:
    HAVE_HGB_40A3 = False
    print("HistGradientBoostingRegressor unavailable:", repr(e))

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("40A-3. Small residual model using compact hierarchical encodings")
print("=" * 90)
print("Have HistGradientBoostingRegressor:", HAVE_HGB_40A3)

# ------------------------------------------------------------
# 1. Load compact HTE features
# ------------------------------------------------------------

feature_path_40a3 = "model_results/hte40a2_compact_features.npz"

if not Path(feature_path_40a3).exists():
    raise FileNotFoundError("Missing 40A-2 feature block. Run 40A-2 first.")

feat_npz = np.load(feature_path_40a3, allow_pickle=True)

X_hte_oof_40a3 = feat_npz["X_oof"].astype(np.float32)
X_hte_test_40a3 = feat_npz["X_test"].astype(np.float32)
fold_id_40a3 = feat_npz["fold_id"].astype(int)
feature_cols_40a3 = [str(x) for x in feat_npz["columns"]]

n_train_40a3 = X_hte_oof_40a3.shape[0]
n_test_40a3 = X_hte_test_40a3.shape[0]

print("\nLoaded compact features")
print("-----------------------")
print("X_oof shape: ", X_hte_oof_40a3.shape)
print("X_test shape:", X_hte_test_40a3.shape)
print("Fold IDs:", sorted(np.unique(fold_id_40a3).tolist()))

# ------------------------------------------------------------
# 2. Load 39C baseline and tiers
# ------------------------------------------------------------

def load_oof_40a3(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(f"No prediction column in {path}")

    if TARGET_COL in df.columns:
        y = df[TARGET_COL].to_numpy(dtype=np.float32)
    elif "y_train" in globals():
        y = np.asarray(y_train, dtype=np.float32).reshape(-1)
    else:
        raise ValueError("Could not recover y_train.")

    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_40a3(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No test prediction column in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("Could not recover test IDs.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

oof39c_df_40a3, y_arr_40a3, pred39c_oof_40a3 = load_oof_40a3(
    "model_results/oof_account39c_final_best.csv"
)
test39c_df_40a3, test_ids_40a3, pred39c_test_40a3 = load_test_40a3(
    "model_results/testpred_account39c_final_best.csv"
)

if len(y_arr_40a3) != n_train_40a3:
    raise ValueError("y length does not match feature block.")
if len(pred39c_test_40a3) != n_test_40a3:
    raise ValueError("test prediction length does not match feature block.")

resid39c_40a3 = (y_arr_40a3 - pred39c_oof_40a3).astype(np.float32)
mse39c_40a3 = float(mean_squared_error(y_arr_40a3, pred39c_oof_40a3))

# Tiers from raw 39A/39B artifacts.
raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_cov_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_cov_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_cov_oof & solver_cov_oof
solver_only_oof = solver_cov_oof & ~direct_cov_oof
none_oof = ~(direct_cov_oof | solver_cov_oof)

overlap_test = direct_cov_test & solver_cov_test
solver_only_test = solver_cov_test & ~direct_cov_test
none_test = ~(direct_cov_test | solver_cov_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

print("\n39C baseline and tiers")
print("----------------------")
print(f"39C OOF MSE: {mse39c_40a3:.6f}")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train n={int(mask.sum()):6d} "
        f"| test n={int(tier_masks_test[tier].sum()):6d} "
        f"| 39C MSE={mean_squared_error(y_arr_40a3[mask], pred39c_oof_40a3[mask]) if mask.sum() else np.nan:.6f}"
    )

# ------------------------------------------------------------
# 3. Add tiny non-HTE feature block: pred39c + tier flags
# ------------------------------------------------------------

extra_oof_40a3 = np.column_stack([
    pred39c_oof_40a3.astype(np.float32),
    overlap_oof.astype(np.float32),
    solver_only_oof.astype(np.float32),
    none_oof.astype(np.float32),
])

extra_test_40a3 = np.column_stack([
    pred39c_test_40a3.astype(np.float32),
    overlap_test.astype(np.float32),
    solver_only_test.astype(np.float32),
    none_test.astype(np.float32),
])

extra_cols_40a3 = [
    "pred39c",
    "tier_overlap",
    "tier_solver_only",
    "tier_none",
]

X_oof_40a3 = np.hstack([X_hte_oof_40a3, extra_oof_40a3]).astype(np.float32)
X_test_40a3 = np.hstack([X_hte_test_40a3, extra_test_40a3]).astype(np.float32)
all_feature_cols_40a3 = feature_cols_40a3 + extra_cols_40a3

print("\nTraining feature matrix")
print("-----------------------")
print("OOF train matrix:", X_oof_40a3.shape)
print("Test matrix:     ", X_test_40a3.shape)
print("Approx OOF MB:   ", round(X_oof_40a3.nbytes / 1024**2, 2))

# ------------------------------------------------------------
# 4. Model configs
# ------------------------------------------------------------

configs_40a3 = [
    {
        "name": "hte40a3_ridge_alpha100_all",
        "model_type": "ridge",
        "alpha": 100.0,
        "weight_scheme": "all",
    },
    {
        "name": "hte40a3_ridge_alpha1000_focus",
        "model_type": "ridge",
        "alpha": 1000.0,
        "weight_scheme": "focus",
    },
    {
        "name": "hte40a3_ridge_alpha10000_focus",
        "model_type": "ridge",
        "alpha": 10000.0,
        "weight_scheme": "focus",
    },
]

if HAVE_HGB_40A3:
    configs_40a3.extend([
        {
            "name": "hte40a3_hgb_small_focus",
            "model_type": "hgb",
            "weight_scheme": "focus",
            "params": {
                "max_iter": 160,
                "learning_rate": 0.04,
                "max_leaf_nodes": 15,
                "max_depth": 4,
                "min_samples_leaf": 80,
                "l2_regularization": 3.0,
                "random_state": RANDOM_STATE,
            },
        },
        {
            "name": "hte40a3_hgb_tiny_noneheavy",
            "model_type": "hgb",
            "weight_scheme": "none_heavy",
            "params": {
                "max_iter": 140,
                "learning_rate": 0.04,
                "max_leaf_nodes": 11,
                "max_depth": 3,
                "min_samples_leaf": 120,
                "l2_regularization": 5.0,
                "random_state": RANDOM_STATE + 11,
            },
        },
    ])

print("\nConfigs")
print("-------")
for cfg in configs_40a3:
    print(cfg)

def make_weights_40a3(idx, scheme):
    w = np.ones(len(idx), dtype=np.float32)

    ov = overlap_oof[idx]
    so = solver_only_oof[idx]
    no = none_oof[idx]

    if scheme == "all":
        return w

    if scheme == "focus":
        w[ov] = 0.05
        w[so] = 1.75
        w[no] = 1.75
        return w

    if scheme == "none_heavy":
        w[ov] = 0.03
        w[so] = 1.00
        w[no] = 2.75
        return w

    raise ValueError(f"Unknown weight scheme: {scheme}")

# ------------------------------------------------------------
# 5. OOF residual training
# ------------------------------------------------------------

resid_oof_pred_40a3 = {
    cfg["name"]: np.full(n_train_40a3, np.nan, dtype=np.float32)
    for cfg in configs_40a3
}

resid_test_sum_40a3 = {
    cfg["name"]: np.zeros(n_test_40a3, dtype=np.float64)
    for cfg in configs_40a3
}

fold_metric_rows_40a3 = []

fold_values_40a3 = sorted(np.unique(fold_id_40a3).tolist())

for cfg in configs_40a3:
    name = cfg["name"]

    print("\n" + "=" * 90)
    print("Training config:", name)
    print("=" * 90)

    for fold in fold_values_40a3:
        va_idx = np.where(fold_id_40a3 == fold)[0]
        tr_idx = np.where(fold_id_40a3 != fold)[0]

        X_tr = X_oof_40a3[tr_idx]
        X_va = X_oof_40a3[va_idx]
        y_tr_resid = resid39c_40a3[tr_idx]

        sample_weight = make_weights_40a3(tr_idx, cfg["weight_scheme"])

        if cfg["model_type"] == "ridge":
            imputer = SimpleImputer(strategy="median")
            scaler = StandardScaler()

            X_tr_imp = imputer.fit_transform(X_tr)
            X_va_imp = imputer.transform(X_va)
            X_te_imp = imputer.transform(X_test_40a3)

            X_tr_s = scaler.fit_transform(X_tr_imp)
            X_va_s = scaler.transform(X_va_imp)
            X_te_s = scaler.transform(X_te_imp)

            model = Ridge(alpha=float(cfg["alpha"]))
            model.fit(X_tr_s, y_tr_resid, sample_weight=sample_weight)

            pred_va_resid = model.predict(X_va_s).astype(np.float32)
            pred_te_resid = model.predict(X_te_s).astype(np.float32)

            del X_tr_imp, X_va_imp, X_te_imp, X_tr_s, X_va_s, X_te_s

        elif cfg["model_type"] == "hgb":
            imputer = SimpleImputer(strategy="median")
            X_tr_imp = imputer.fit_transform(X_tr)
            X_va_imp = imputer.transform(X_va)
            X_te_imp = imputer.transform(X_test_40a3)

            model = HistGradientBoostingRegressor(**cfg["params"])
            model.fit(X_tr_imp, y_tr_resid, sample_weight=sample_weight)

            pred_va_resid = model.predict(X_va_imp).astype(np.float32)
            pred_te_resid = model.predict(X_te_imp).astype(np.float32)

            del X_tr_imp, X_va_imp, X_te_imp

        else:
            raise ValueError(f"Unknown model_type: {cfg['model_type']}")

        resid_oof_pred_40a3[name][va_idx] = pred_va_resid
        resid_test_sum_40a3[name] += pred_te_resid.astype(np.float64)

        pred_va_lam1 = np.clip(pred39c_oof_40a3[va_idx] + pred_va_resid, 0, 100)

        row = {
            "config": name,
            "fold": int(fold),
            "model_type": cfg["model_type"],
            "weight_scheme": cfg["weight_scheme"],
            "mse_39c": float(mean_squared_error(y_arr_40a3[va_idx], pred39c_oof_40a3[va_idx])),
            "mse_lam1": float(mean_squared_error(y_arr_40a3[va_idx], pred_va_lam1)),
            "resid_pred_mse": float(mean_squared_error(resid39c_40a3[va_idx], pred_va_resid)),
        }
        row["gain_lam1_vs_39c"] = row["mse_39c"] - row["mse_lam1"]

        for tier, mask_full in tier_masks_oof.items():
            mask_fold = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask_fold.sum())
            if int(mask_fold.sum()) > 0:
                row[f"mse39c_{tier}"] = float(mean_squared_error(
                    y_arr_40a3[va_idx][mask_fold],
                    pred39c_oof_40a3[va_idx][mask_fold],
                ))
                row[f"mse_lam1_{tier}"] = float(mean_squared_error(
                    y_arr_40a3[va_idx][mask_fold],
                    pred_va_lam1[mask_fold],
                ))
                row[f"gain_lam1_{tier}"] = row[f"mse39c_{tier}"] - row[f"mse_lam1_{tier}"]

        fold_metric_rows_40a3.append(row)

        print(
            f"fold {fold} | lam1 MSE {row['mse_lam1']:.6f} "
            f"| gain {row['gain_lam1_vs_39c']:.6f} "
            f"| resid_pred_mse {row['resid_pred_mse']:.6f}"
        )

        del model
        gc.collect()

# ------------------------------------------------------------
# 6. Lambda scans
# ------------------------------------------------------------

lambda_grid_40a3 = np.unique(
    np.concatenate([
        np.linspace(-1.0, 1.5, 626),
        np.array([0.0, 0.05, 0.10, 0.25, 0.50, 0.75, 1.0]),
    ])
)

def best_lambda_mask_40a3(resid_pred, mask):
    mask = np.asarray(mask, dtype=bool)

    if int(mask.sum()) == 0:
        return {"lambda": 0.0, "mse": np.nan, "sse": 0.0, "n": 0}

    y_m = y_arr_40a3[mask].astype(np.float64)
    base_m = pred39c_oof_40a3[mask].astype(np.float64)
    r_m = resid_pred[mask].astype(np.float64)

    best = None

    for lam in lambda_grid_40a3:
        pred = np.clip(base_m + float(lam) * r_m, 0, 100)
        err = y_m - pred
        sse = float(np.dot(err, err))

        row = {
            "lambda": float(lam),
            "sse": sse,
            "mse": sse / len(y_m),
            "n": int(len(y_m)),
        }

        if best is None or row["sse"] < best["sse"]:
            best = row

    return best

def make_pred_global_40a3(resid_pred, lam, is_test=False):
    base = pred39c_test_40a3 if is_test else pred39c_oof_40a3
    return np.clip(base + float(lam) * resid_pred, 0, 100).astype(np.float32)

def make_pred_tier_40a3(resid_pred, lams, is_test=False):
    base = pred39c_test_40a3.copy() if is_test else pred39c_oof_40a3.copy()
    masks = tier_masks_test if is_test else tier_masks_oof

    pred = base.astype(np.float64)

    for tier, mask in masks.items():
        if int(mask.sum()) == 0:
            continue
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * resid_pred[mask]

    return np.clip(pred, 0, 100).astype(np.float32)

screen_rows_40a3 = []

for cfg in configs_40a3:
    name = cfg["name"]
    resid_oof = resid_oof_pred_40a3[name]

    if not np.isfinite(resid_oof).all():
        print("Skipping incomplete:", name)
        continue

    # Global lambda.
    best_global = best_lambda_mask_40a3(
        resid_pred=resid_oof,
        mask=np.ones(n_train_40a3, dtype=bool),
    )

    pred_global = make_pred_global_40a3(resid_oof, best_global["lambda"], is_test=False)
    mse_global = float(mean_squared_error(y_arr_40a3, pred_global))

    screen_rows_40a3.append({
        "candidate_type": "global_lambda",
        "config": name,
        "model_type": cfg["model_type"],
        "weight_scheme": cfg["weight_scheme"],
        "lambda_global": best_global["lambda"],
        "lambda_overlap": np.nan,
        "lambda_solver_only": np.nan,
        "lambda_none": np.nan,
        "oof_mse": mse_global,
        "gain_vs_39c": mse39c_40a3 - mse_global,
    })

    # Tier-specific lambda.
    lams = {}
    tier_mses = {}

    for tier, mask in tier_masks_oof.items():
        opt = best_lambda_mask_40a3(resid_pred=resid_oof, mask=mask)
        lams[tier] = opt["lambda"]
        tier_mses[tier] = opt["mse"]

    pred_tier = make_pred_tier_40a3(resid_oof, lams, is_test=False)
    mse_tier = float(mean_squared_error(y_arr_40a3, pred_tier))

    screen_rows_40a3.append({
        "candidate_type": "tier_specific_lambda",
        "config": name,
        "model_type": cfg["model_type"],
        "weight_scheme": cfg["weight_scheme"],
        "lambda_global": np.nan,
        "lambda_overlap": lams.get("overlap", np.nan),
        "lambda_solver_only": lams.get("solver_only", np.nan),
        "lambda_none": lams.get("none", np.nan),
        "oof_mse": mse_tier,
        "gain_vs_39c": mse39c_40a3 - mse_tier,
        "overlap_mse": tier_mses.get("overlap", np.nan),
        "solver_only_mse": tier_mses.get("solver_only", np.nan),
        "none_mse": tier_mses.get("none", np.nan),
    })

screen40a3 = pd.DataFrame(screen_rows_40a3).sort_values("oof_mse").reset_index(drop=True)

best40a3 = screen40a3.iloc[0]
best_name_40a3 = best40a3["config"]
best_resid_oof_40a3 = resid_oof_pred_40a3[best_name_40a3]
best_resid_test_40a3 = (resid_test_sum_40a3[best_name_40a3] / len(fold_values_40a3)).astype(np.float32)

if best40a3["candidate_type"] == "global_lambda":
    final_oof_40a3 = make_pred_global_40a3(
        best_resid_oof_40a3,
        float(best40a3["lambda_global"]),
        is_test=False,
    )
    final_test_40a3 = make_pred_global_40a3(
        best_resid_test_40a3,
        float(best40a3["lambda_global"]),
        is_test=True,
    )
else:
    best_lams_40a3 = {
        "overlap": float(best40a3["lambda_overlap"]) if np.isfinite(best40a3["lambda_overlap"]) else 0.0,
        "solver_only": float(best40a3["lambda_solver_only"]) if np.isfinite(best40a3["lambda_solver_only"]) else 0.0,
        "none": float(best40a3["lambda_none"]) if np.isfinite(best40a3["lambda_none"]) else 0.0,
    }

    final_oof_40a3 = make_pred_tier_40a3(
        best_resid_oof_40a3,
        best_lams_40a3,
        is_test=False,
    )
    final_test_40a3 = make_pred_tier_40a3(
        best_resid_test_40a3,
        best_lams_40a3,
        is_test=True,
    )

best_mse_40a3 = float(mean_squared_error(y_arr_40a3, final_oof_40a3))
best_gain_40a3 = mse39c_40a3 - best_mse_40a3

# ------------------------------------------------------------
# 7. Fold and tier diagnostics
# ------------------------------------------------------------

fold_gain_rows_40a3 = []

for fold in fold_values_40a3:
    idx = np.where(fold_id_40a3 == fold)[0]

    row = {
        "fold": int(fold),
        "mse_39c": float(mean_squared_error(y_arr_40a3[idx], pred39c_oof_40a3[idx])),
        "mse_40a3": float(mean_squared_error(y_arr_40a3[idx], final_oof_40a3[idx])),
    }
    row["gain_40a3_vs_39c"] = row["mse_39c"] - row["mse_40a3"]

    for tier, mask_full in tier_masks_oof.items():
        mask_fold = mask_full[idx]
        row[f"n_{tier}"] = int(mask_fold.sum())

        if int(mask_fold.sum()) > 0:
            row[f"mse39c_{tier}"] = float(mean_squared_error(
                y_arr_40a3[idx][mask_fold],
                pred39c_oof_40a3[idx][mask_fold],
            ))
            row[f"mse40a3_{tier}"] = float(mean_squared_error(
                y_arr_40a3[idx][mask_fold],
                final_oof_40a3[idx][mask_fold],
            ))
            row[f"gain_{tier}"] = row[f"mse39c_{tier}"] - row[f"mse40a3_{tier}"]

    fold_gain_rows_40a3.append(row)

fold_gains40a3 = pd.DataFrame(fold_gain_rows_40a3)

tier_perf_rows_40a3 = []

for tier, mask in tier_masks_oof.items():
    if int(mask.sum()) == 0:
        continue

    tier_perf_rows_40a3.append({
        "tier": tier,
        "n_rows": int(mask.sum()),
        "mse_39c": float(mean_squared_error(y_arr_40a3[mask], pred39c_oof_40a3[mask])),
        "mse_40a3": float(mean_squared_error(y_arr_40a3[mask], final_oof_40a3[mask])),
        "gain_vs_39c": float(mean_squared_error(y_arr_40a3[mask], pred39c_oof_40a3[mask]) - mean_squared_error(y_arr_40a3[mask], final_oof_40a3[mask])),
    })

tier_perf40a3 = pd.DataFrame(tier_perf_rows_40a3)

# ------------------------------------------------------------
# 8. Save artifacts
# ------------------------------------------------------------

screen_path_40a3 = "model_results/hte40a3_residual_model_screen.csv"
fold_metrics_path_40a3 = "model_results/hte40a3_fold_metrics.csv"
fold_gains_path_40a3 = "model_results/hte40a3_best_fold_gains.csv"
tier_perf_path_40a3 = "model_results/hte40a3_tier_performance.csv"
oof_path_40a3 = "model_results/oof_hte40a3_residual_best.csv"
testpred_path_40a3 = "model_results/testpred_hte40a3_residual_best.csv"
submission_path_40a3 = "submission_hte40a3_residual_best.csv"

screen40a3.to_csv(screen_path_40a3, index=False)
pd.DataFrame(fold_metric_rows_40a3).to_csv(fold_metrics_path_40a3, index=False)
fold_gains40a3.to_csv(fold_gains_path_40a3, index=False)
tier_perf40a3.to_csv(tier_perf_path_40a3, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_40a3),
    TARGET_COL: y_arr_40a3,
    "pred_39c": pred39c_oof_40a3,
    "residual_39c": resid39c_40a3,
    "residual_pred_40a3": best_resid_oof_40a3,
    "pred_clipped": final_oof_40a3,
    "overlap": overlap_oof.astype(int),
    "solver_only": solver_only_oof.astype(int),
    "none": none_oof.astype(int),
}).to_csv(oof_path_40a3, index=False)

pd.DataFrame({
    ID_COL: test_ids_40a3,
    "pred_39c": pred39c_test_40a3,
    "residual_pred_40a3": best_resid_test_40a3,
    TARGET_COL: final_test_40a3,
    "overlap": overlap_test.astype(int),
    "solver_only": solver_only_test.astype(int),
    "none": none_test.astype(int),
}).to_csv(testpred_path_40a3, index=False)

pd.DataFrame({
    ID_COL: test_ids_40a3,
    TARGET_COL: final_test_40a3,
}).to_csv(submission_path_40a3, index=False)

sub40a3 = pd.read_csv(submission_path_40a3)
assert sub40a3.shape == (n_test_40a3, 2)
assert list(sub40a3.columns) == [ID_COL, TARGET_COL]
assert sub40a3[TARGET_COL].notna().all()
assert np.isfinite(sub40a3[TARGET_COL]).all()
assert sub40a3[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 9. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("40A-3 residual model complete")
print("=" * 90)

print("\nBaseline")
print("--------")
print(f"39C OOF MSE: {mse39c_40a3:.6f}")

print("\nTop 20 40A-3 candidates")
print("-----------------------")
display(screen40a3.head(20))

print("\nBest 40A-3 candidate")
print("--------------------")
print(best40a3.to_string())
print(f"\nBest 40A-3 OOF MSE: {best_mse_40a3:.6f}")
print(f"Gain vs 39C:         {best_gain_40a3:.6f}")

print("\nFold gains")
print("----------")
print(fold_gains40a3.to_string(index=False))
print("Min fold gain vs 39C:", float(fold_gains40a3["gain_40a3_vs_39c"].min()))

print("\nTier performance")
print("----------------")
print(tier_perf40a3.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path_40a3)
print(fold_metrics_path_40a3)
print(fold_gains_path_40a3)
print(tier_perf_path_40a3)
print(oof_path_40a3)
print(testpred_path_40a3)
print(submission_path_40a3)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path_40a3)
print("Shape:", sub40a3.shape)
print(sub40a3[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
if best_gain_40a3 >= 3.0:
    print("40A-3 clears the 3+ OOF MSE threshold versus 39C. This is submission-worthy.")
elif best_gain_40a3 >= 1.0:
    print("40A-3 improves OOF but does not clear the 3+ threshold. Save artifact; submit only with strong structural justification.")
elif best_gain_40a3 > 0:
    print("40A-3 gives only a marginal OOF gain. Do not submit under the current threshold.")
else:
    print("40A-3 does not improve over 39C. Keep 39C as current best.")

40A-3. Small residual model using compact hierarchical encodings
Have HistGradientBoostingRegressor: True

Loaded compact features
-----------------------
X_oof shape:  (144921, 52)
X_test shape: (48307, 52)
Fold IDs: [1, 2, 3, 4, 5]

39C baseline and tiers
----------------------
39C OOF MSE: 53.226631
overlap      train n= 53786 | test n= 27298 | 39C MSE=0.595326
solver_only  train n= 27807 | test n= 17807 | 39C MSE=65.890625
none         train n= 63328 | test n=  3202 | 39C MSE=92.366982

Training feature matrix
-----------------------
OOF train matrix: (144921, 56)
Test matrix:      (48307, 56)
Approx OOF MB:    30.96

Configs
-------
{'name': 'hte40a3_ridge_alpha100_all', 'model_type': 'ridge', 'alpha': 100.0, 'weight_scheme': 'all'}
{'name': 'hte40a3_ridge_alpha1000_focus', 'model_type': 'ridge', 'alpha': 1000.0, 'weight_scheme': 'focus'}
{'name': 'hte40a3_ridge_alpha10000_focus', 'model_type': 'ridge', 'alpha': 10000.0, 'weight_scheme': 'focus'}
{'name': 'hte40a3_hgb_small_focus'

,candidate_type,config,model_type,weight_scheme,lambda_global,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_39c,overlap_mse,solver_only_mse,none_mse
0,tier_specific_lambda,hte40a3_hgb_small_focus,hgb,focus,NaN,0.000,0.496,0.992,49.689606,3.537025,0.595326,64.945831,84.687634
1,tier_specific_lambda,hte40a3_hgb_tiny_noneheavy,hgb,none_heavy,NaN,0.000,0.448,1.016,50.116100,3.110531,0.595326,65.134523,85.580780
2,tier_specific_lambda,hte40a3_ridge_alpha1000_focus,ridge,focus,NaN,0.000,0.344,0.876,50.739738,2.486893,0.595326,65.227957,86.966898
3,tier_specific_lambda,hte40a3_ridge_alpha10000_focus,ridge,focus,NaN,0.000,0.384,0.964,50.741196,2.485435,0.595326,65.224353,86.971812
4,global_lambda,hte40a3_hgb_small_focus,hgb,focus,0.652,NaN,NaN,NaN,50.776306,2.450325,NaN,NaN,NaN
5,tier_specific_lambda,hte40a3_ridge_alpha100_all,ridge,all,NaN,0.004,0.600,1.396,50.790077,2.436554,0.595323,65.173023,87.106229
6,global_lambda,hte40a3_hgb_tiny_noneheavy,hgb,none_heavy,0.612,NaN,NaN,NaN,51.237061,1.989571,NaN,NaN,NaN
7,global_lambda,hte40a3_ridge_alpha100_all,ridge,all,0.744,NaN,NaN,NaN,51.831150,1.395481,NaN,NaN,NaN
8,global_lambda,hte40a3_ridge_alpha10000_focus,ridge,focus,0.496,NaN,NaN,NaN,51.837612,1.389019,NaN,NaN,NaN
9,global_lambda,hte40a3_ridge_alpha1000_focus,ridge,focus,0.444,NaN,NaN,NaN,51.859642,1.366989,NaN,NaN,NaN



Best 40A-3 candidate
--------------------
candidate_type           tier_specific_lambda
config                hte40a3_hgb_small_focus
model_type                                hgb
weight_scheme                           focus
lambda_global                             NaN
lambda_overlap                            0.0
lambda_solver_only                      0.496
lambda_none                             0.992
oof_mse                             49.689606
gain_vs_39c                          3.537025
overlap_mse                          0.595326
solver_only_mse                     64.945831
none_mse                            84.687634

Best 40A-3 OOF MSE: 49.689606
Gain vs 39C:         3.537025

Fold gains
----------
 fold   mse_39c  mse_40a3  gain_40a3_vs_39c  n_overlap  mse39c_overlap  mse40a3_overlap  gain_overlap  n_solver_only  mse39c_solver_only  mse40a3_solver_only  gain_solver_only  n_none  mse39c_none  mse40a3_none  gain_none
    1 53.559204 49.163811          4.395393      1083

### 40A. Hierarchical Target-Encoding Residual Model After 39C

This section returned to feature engineering after the accounting branch. The current best public model, 39C, solved the direct-accounting overlap rows very well, but still had high error on rows not directly handled by subgroup accounting.

The model targeted the residual from 39C:

`residual_39C = observed target - 39C prediction`

A compact, OOF-safe hierarchy-encoding block was constructed using only selected hierarchy keys:

`ASSESSMENT_NAME`, `SUBGROUP_NAME`, `ASSESSMENT_NAME × SUBGROUP_NAME`, `SCHOOL`, `SCHOOL × ASSESSMENT_NAME`, `SCHOOL × SUBGROUP_NAME`, `DISTRICT`, `DISTRICT × ASSESSMENT_NAME`, `COUNTY`, and `COUNTY × ASSESSMENT_NAME`.

The feature matrix was intentionally kept small, with only `52` hierarchical encoding features from 40A-2, plus a few tier flags and the 39C prediction. This avoided the memory problems from the earlier failed all-in-one hierarchical model attempt.

The best 40A-3 residual model was `hte40a3_hgb_small_focus`, a small histogram gradient boosting model trained on the compact hierarchy features. It used tier-specific residual shrinkage:

- overlap lambda: `0.000`
- solver-only lambda: `0.496`
- none lambda: `0.992`

This means the model left the already-solved direct-accounting overlap rows untouched, made a modest correction to solver-only rows, and applied a much stronger correction to rows not covered by accounting.

Results:

- 39C OOF MSE: `53.226631`
- 40A-3 OOF MSE: `49.689606`
- Gain versus 39C: `3.537025`

Tier performance:

- overlap MSE stayed unchanged at `0.595326`
- solver-only MSE improved from `65.890625` to `64.945831`
- none-tier MSE improved from `92.366982` to `84.687637`

Fold gains were all positive:

- Fold 1 gain: `4.395393`
- Fold 2 gain: `1.045525`
- Fold 3 gain: `3.947090`
- Fold 4 gain: `4.300766`
- Fold 5 gain: `3.996334`

Conclusion: 40A-3 is a meaningful structural improvement over 39C. Unlike the marginal 39D accounting refinement, this model attacks the unsolved `none` tier using hierarchy-based residual signal. Because it clears the project’s current `3+` OOF MSE threshold, `submission_hte40a3_residual_best.csv` is submission-worthy.

### Public Leaderboard Check: 40A-3 Hierarchical Residual Model

The 40A-3 hierarchical target-encoding residual model improved OOF MSE from `53.226631` to `49.689606`, a gain of `3.537025`. However, its public leaderboard MSE was `33.919`, worse than the current best 39C public MSE of `31.431`.

This indicates that the OOF improvement did not transfer to the public test set. The likely reason is tier distribution mismatch. In training OOF, the `none` tier was large, with `63,328` rows, while in the test set only `3,202` rows were in the `none` tier. The 40A-3 model improved the `none` tier substantially in OOF, but the public test set is much more dominated by accounting-covered rows.

Current public ranking:

- 39C: public MSE `31.431`
- 39B: public MSE `32.113`
- 40A-3: public MSE `33.919`
- 39A: public MSE `37.424`

Conclusion: 40A-3 should not be used as a final model. Future model selection should include a test-tier-weighted OOF metric, not just ordinary OOF MSE.

In [33]:
# ============================================================
# 40B. Lightweight constrained stacking over saved artifacts
# ============================================================
#
# Purpose:
#   Test whether previous model-family artifacts add useful signal
#   after the accounting branch, without rebuilding features.
#
# This cell:
#   - Loads saved OOF/test predictions.
#   - Builds global and tier-specific convex stacks.
#   - Optionally includes 40A-3 as a candidate, but lets the stack downweight it.
#   - Reports ordinary OOF and test-tier-weighted OOF.
#
# Safe:
#   - No LightGBM
#   - No PyTorch
#   - No target encoding construction
#   - No large feature matrices
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

try:
    from scipy.optimize import minimize
    HAVE_SCIPY_MINIMIZE_40B = True
except Exception:
    HAVE_SCIPY_MINIMIZE_40B = False

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("40B. Lightweight constrained stacking over saved artifacts")
print("=" * 90)
print("Have scipy.optimize.minimize:", HAVE_SCIPY_MINIMIZE_40B)

# ------------------------------------------------------------
# 1. Load saved prediction artifacts
# ------------------------------------------------------------

artifact_specs_40b = [
    ("anchor33a", "model_results/oof_seg33a_arcsine_adaptive.csv", "model_results/testpred_seg33a_arcsine_adaptive.csv"),
    ("knn35a", "model_results/oof_knn35a_local_residual_best.csv", "model_results/testpred_knn35a_local_residual_best.csv"),
    ("nn37b_rank1", "model_results/oof_nn37b38b_cpu_safe_rank1.csv", "model_results/testpred_nn37b38b_cpu_safe_rank1.csv"),
    ("step36a", "model_results/oof_step36a_residual_bins_best.csv", "model_results/testpred_step36a_residual_bins_best.csv"),
    ("blend33c", "model_results/oof_blend33c_33a_huber33b.csv", "model_results/testpred_blend33c_33a_huber33b.csv"),
    ("account39a", "model_results/oof_account39a_best.csv", "model_results/testpred_account39a_best.csv"),
    ("account39b", "model_results/oof_account39b_solver_best.csv", "model_results/testpred_account39b_solver_best.csv"),
    ("account39c", "model_results/oof_account39c_final_best.csv", "model_results/testpred_account39c_final_best.csv"),
    ("account39d", "model_results/oof_account39d_best.csv", "model_results/testpred_account39d_best.csv"),
    ("hte40a3", "model_results/oof_hte40a3_residual_best.csv", "model_results/testpred_hte40a3_residual_best.csv"),
]

def load_oof_pred_40b(path):
    df = pd.read_csv(path)

    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    pred_col = None
    for c in ["pred_clipped", "prediction", "pred", "oof_pred"]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No OOF prediction column found in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if TARGET_COL in df.columns:
        y = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
    else:
        y = None

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}")

    return np.clip(pred, 0, 100).astype(np.float32), y, pred_col

def load_test_pred_40b(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No test prediction column found in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no global test_ids found.")

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col

oof_preds_40b = {}
test_preds_40b = {}
artifact_rows_40b = []

y_40b = None
test_ids_40b = None

for name, oof_path, test_path in artifact_specs_40b:
    if not Path(oof_path).exists() or not Path(test_path).exists():
        print(f"Skipping missing artifact: {name}")
        continue

    try:
        oof_pred, y_file, oof_col = load_oof_pred_40b(oof_path)
        test_pred, ids, test_col = load_test_pred_40b(test_path)

        if y_40b is None:
            if y_file is not None:
                y_40b = y_file.astype(np.float32)
            elif "y_train" in globals():
                y_40b = np.asarray(y_train, dtype=np.float32).reshape(-1)
            else:
                raise ValueError("Could not recover y_train.")
        else:
            if y_file is not None:
                max_diff = float(np.nanmax(np.abs(y_file - y_40b)))
                if max_diff > 1e-5:
                    raise ValueError(f"Target mismatch for {name}: max diff {max_diff}")

        if test_ids_40b is None:
            test_ids_40b = ids

        if len(oof_pred) != len(y_40b):
            raise ValueError(f"OOF length mismatch for {name}")
        if len(test_pred) != len(test_ids_40b):
            raise ValueError(f"Test length mismatch for {name}")

        oof_preds_40b[name] = oof_pred
        test_preds_40b[name] = test_pred

        mse = float(mean_squared_error(y_40b, oof_pred))

        artifact_rows_40b.append({
            "name": name,
            "oof_mse": mse,
            "oof_path": oof_path,
            "test_path": test_path,
            "oof_col": oof_col,
            "test_col": test_col,
        })

        print(f"Loaded {name:14s} | OOF MSE {mse:.6f}")

    except Exception as e:
        print(f"Skipping {name} due to error: {repr(e)}")

artifact_summary40b = (
    pd.DataFrame(artifact_rows_40b)
    .sort_values("oof_mse")
    .reset_index(drop=True)
)

if "account39c" not in oof_preds_40b:
    raise ValueError("account39c must be available because it is the current public best.")

n_train_40b = len(y_40b)
n_test_40b = len(test_ids_40b)

mse39c_40b = float(mean_squared_error(y_40b, oof_preds_40b["account39c"]))

print("\nArtifact summary")
print("----------------")
display(artifact_summary40b)

# ------------------------------------------------------------
# 2. Recover tier flags
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_cov_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_cov_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_cov_oof & solver_cov_oof
solver_only_oof = solver_cov_oof & ~direct_cov_oof
none_oof = ~(direct_cov_oof | solver_cov_oof)

overlap_test = direct_cov_test & solver_cov_test
solver_only_test = solver_cov_test & ~direct_cov_test
none_test = ~(direct_cov_test | solver_cov_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

print("\nTier coverage")
print("-------------")
for tier in ["overlap", "solver_only", "none"]:
    print(
        f"{tier:12s} train n={int(tier_masks_oof[tier].sum()):6d} "
        f"| test n={int(tier_masks_test[tier].sum()):6d}"
    )

# ------------------------------------------------------------
# 3. Helper metrics
# ------------------------------------------------------------

test_tier_rates_40b = {
    tier: float(mask.mean())
    for tier, mask in tier_masks_test.items()
}

def ordinary_mse_40b(pred):
    return float(mean_squared_error(y_40b, np.clip(pred, 0, 100)))

def tier_weighted_mse_40b(pred):
    pred = np.clip(pred, 0, 100)
    total = 0.0

    for tier, mask in tier_masks_oof.items():
        if int(mask.sum()) == 0:
            continue
        tier_mse = float(mean_squared_error(y_40b[mask], pred[mask]))
        total += test_tier_rates_40b[tier] * tier_mse

    return float(total)

def tier_mse_dict_40b(pred):
    pred = np.clip(pred, 0, 100)
    out = {}

    for tier, mask in tier_masks_oof.items():
        if int(mask.sum()) > 0:
            out[f"mse_{tier}"] = float(mean_squared_error(y_40b[mask], pred[mask]))
        else:
            out[f"mse_{tier}"] = np.nan

    return out

baseline_metric_rows_40b = []

for name, pred in oof_preds_40b.items():
    row = {
        "name": name,
        "ordinary_oof_mse": ordinary_mse_40b(pred),
        "test_tier_weighted_mse": tier_weighted_mse_40b(pred),
        "gain_vs_39c_oof": mse39c_40b - ordinary_mse_40b(pred),
        "gain_vs_39c_weighted": tier_weighted_mse_40b(oof_preds_40b["account39c"]) - tier_weighted_mse_40b(pred),
    }
    row.update(tier_mse_dict_40b(pred))
    baseline_metric_rows_40b.append(row)

baseline_metrics40b = (
    pd.DataFrame(baseline_metric_rows_40b)
    .sort_values("test_tier_weighted_mse")
    .reset_index(drop=True)
)

print("\nBaseline artifact metrics")
print("-------------------------")
display(baseline_metrics40b)

# ------------------------------------------------------------
# 4. Convex blend optimizer
# ------------------------------------------------------------

def fit_convex_weights_40b(P, y, sample_weight=None, ridge=1e-8):
    P = np.asarray(P, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)

    n, k = P.shape

    if sample_weight is None:
        w = np.ones(n, dtype=np.float64)
    else:
        w = np.asarray(sample_weight, dtype=np.float64).reshape(-1)
        w = np.where(np.isfinite(w) & (w > 0), w, 1.0)

    if k == 1:
        return np.ones(1, dtype=np.float64)

    def obj(beta):
        pred = P @ beta
        err = y - pred
        return float(np.sum(w * err * err) / np.sum(w) + ridge * np.sum(beta * beta))

    cons = [{"type": "eq", "fun": lambda beta: np.sum(beta) - 1.0}]
    bounds = [(0.0, 1.0)] * k
    x0 = np.ones(k, dtype=np.float64) / k

    if HAVE_SCIPY_MINIMIZE_40B:
        res = minimize(
            obj,
            x0,
            method="SLSQP",
            bounds=bounds,
            constraints=cons,
            options={"maxiter": 1000, "ftol": 1e-12, "disp": False},
        )

        if res.success and np.isfinite(res.x).all():
            beta = np.clip(res.x, 0, 1)
            beta = beta / beta.sum()
            return beta

    # Fallback: use inverse-MSE weights.
    mses = []
    for j in range(k):
        mse_j = mean_squared_error(y, P[:, j], sample_weight=w)
        mses.append(max(mse_j, 1e-9))

    beta = 1.0 / np.asarray(mses)
    beta = beta / beta.sum()
    return beta

def make_weighted_tier_sample_weights_40b(idx):
    # Makes train-fold weighted loss approximate test tier proportions.
    sw = np.ones(len(idx), dtype=np.float64)

    for tier, mask_full in tier_masks_oof.items():
        mask_local = mask_full[idx]
        n_local = int(mask_local.sum())
        if n_local == 0:
            continue
        train_rate = n_local / len(idx)
        test_rate = test_tier_rates_40b[tier]
        sw[mask_local] = test_rate / max(train_rate, 1e-12)

    return sw

# ------------------------------------------------------------
# 5. Stack candidate definitions
# ------------------------------------------------------------

# Candidate sets. Include hte40a3 only in some stacks because it failed public,
# but it may have useful none-tier signal if constrained correctly.
candidate_sets_40b = {
    "accounting_only": [
        n for n in ["account39a", "account39b", "account39c", "account39d"]
        if n in oof_preds_40b
    ],
    "accounting_plus_ml": [
        n for n in ["account39a", "account39b", "account39c", "account39d", "knn35a", "nn37b_rank1", "step36a", "blend33c", "anchor33a"]
        if n in oof_preds_40b
    ],
    "accounting_plus_hte": [
        n for n in ["account39a", "account39b", "account39c", "account39d", "hte40a3", "knn35a"]
        if n in oof_preds_40b
    ],
}

print("\nCandidate sets")
print("--------------")
for set_name, names in candidate_sets_40b.items():
    print(set_name, ":", names)

folds40b = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_40b))
)

stack_rows_40b = []
stack_store_40b = {}

# ------------------------------------------------------------
# 6. Global convex stacks
# ------------------------------------------------------------

for set_name, names in candidate_sets_40b.items():
    if len(names) == 0:
        continue

    P_oof = np.column_stack([oof_preds_40b[n] for n in names]).astype(np.float64)
    P_test = np.column_stack([test_preds_40b[n] for n in names]).astype(np.float64)

    for weight_mode in ["ordinary", "test_tier_weighted"]:
        pred_oof = np.full(n_train_40b, np.nan, dtype=np.float32)
        test_pred_sum = np.zeros(n_test_40b, dtype=np.float64)
        fold_weight_rows = []

        for fold_num, (tr_idx, va_idx) in enumerate(folds40b, start=1):
            if weight_mode == "ordinary":
                sw = None
            else:
                sw = make_weighted_tier_sample_weights_40b(tr_idx)

            beta = fit_convex_weights_40b(
                P=P_oof[tr_idx],
                y=y_40b[tr_idx],
                sample_weight=sw,
            )

            pred_oof[va_idx] = np.clip(P_oof[va_idx] @ beta, 0, 100).astype(np.float32)
            test_pred_sum += np.clip(P_test @ beta, 0, 100)

            for name, b in zip(names, beta):
                fold_weight_rows.append({
                    "stack_name": f"global_{set_name}_{weight_mode}",
                    "fold": fold_num,
                    "model": name,
                    "weight": float(b),
                })

        pred_test = (test_pred_sum / N_SPLITS).astype(np.float32)

        stack_name = f"global_{set_name}_{weight_mode}"

        mse_oof = ordinary_mse_40b(pred_oof)
        mse_weighted = tier_weighted_mse_40b(pred_oof)

        row = {
            "stack_name": stack_name,
            "stack_type": "global_convex",
            "candidate_set": set_name,
            "weight_mode": weight_mode,
            "ordinary_oof_mse": mse_oof,
            "test_tier_weighted_mse": mse_weighted,
            "gain_vs_39c_oof": mse39c_40b - mse_oof,
            "gain_vs_39c_weighted": tier_weighted_mse_40b(oof_preds_40b["account39c"]) - mse_weighted,
        }
        row.update(tier_mse_dict_40b(pred_oof))

        stack_rows_40b.append(row)
        stack_store_40b[stack_name] = {
            "oof": pred_oof,
            "test": pred_test,
            "weights": pd.DataFrame(fold_weight_rows),
        }

# ------------------------------------------------------------
# 7. Tier-specific convex stacks
# ------------------------------------------------------------

for set_name, names in candidate_sets_40b.items():
    if len(names) == 0:
        continue

    P_oof = np.column_stack([oof_preds_40b[n] for n in names]).astype(np.float64)
    P_test = np.column_stack([test_preds_40b[n] for n in names]).astype(np.float64)

    pred_oof = np.full(n_train_40b, np.nan, dtype=np.float32)
    test_pred_sum = np.zeros(n_test_40b, dtype=np.float64)
    fold_weight_rows = []

    for fold_num, (tr_idx, va_idx) in enumerate(folds40b, start=1):
        pred_fold = np.zeros(len(va_idx), dtype=np.float64)
        pred_test_fold = np.zeros(n_test_40b, dtype=np.float64)

        for tier in ["overlap", "solver_only", "none"]:
            tr_mask = tier_masks_oof[tier][tr_idx]
            va_mask = tier_masks_oof[tier][va_idx]
            test_mask = tier_masks_test[tier]

            if int(va_mask.sum()) == 0 and int(test_mask.sum()) == 0:
                continue

            # If tier has too few training rows, fallback to all training rows.
            if int(tr_mask.sum()) >= 500:
                tr_rows = tr_idx[tr_mask]
            else:
                tr_rows = tr_idx

            beta = fit_convex_weights_40b(
                P=P_oof[tr_rows],
                y=y_40b[tr_rows],
                sample_weight=None,
            )

            if int(va_mask.sum()) > 0:
                pred_fold[va_mask] = np.clip(P_oof[va_idx[va_mask]] @ beta, 0, 100)

            if int(test_mask.sum()) > 0:
                pred_test_fold[test_mask] = np.clip(P_test[test_mask] @ beta, 0, 100)

            for name, b in zip(names, beta):
                fold_weight_rows.append({
                    "stack_name": f"tiered_{set_name}",
                    "fold": fold_num,
                    "tier": tier,
                    "model": name,
                    "weight": float(b),
                    "n_train_tier": int(tr_mask.sum()),
                })

        pred_oof[va_idx] = pred_fold.astype(np.float32)
        test_pred_sum += pred_test_fold

    pred_test = (test_pred_sum / N_SPLITS).astype(np.float32)

    stack_name = f"tiered_{set_name}"

    mse_oof = ordinary_mse_40b(pred_oof)
    mse_weighted = tier_weighted_mse_40b(pred_oof)

    row = {
        "stack_name": stack_name,
        "stack_type": "tier_specific_convex",
        "candidate_set": set_name,
        "weight_mode": "tier_specific",
        "ordinary_oof_mse": mse_oof,
        "test_tier_weighted_mse": mse_weighted,
        "gain_vs_39c_oof": mse39c_40b - mse_oof,
        "gain_vs_39c_weighted": tier_weighted_mse_40b(oof_preds_40b["account39c"]) - mse_weighted,
    }
    row.update(tier_mse_dict_40b(pred_oof))

    stack_rows_40b.append(row)
    stack_store_40b[stack_name] = {
        "oof": pred_oof,
        "test": pred_test,
        "weights": pd.DataFrame(fold_weight_rows),
    }

# ------------------------------------------------------------
# 8. Screen and save
# ------------------------------------------------------------

stack_screen40b = pd.DataFrame(stack_rows_40b).sort_values(
    ["test_tier_weighted_mse", "ordinary_oof_mse"]
).reset_index(drop=True)

best_weighted_name_40b = stack_screen40b.iloc[0]["stack_name"]
best_weighted_oof_40b = stack_store_40b[best_weighted_name_40b]["oof"]
best_weighted_test_40b = stack_store_40b[best_weighted_name_40b]["test"]

stack_screen_by_oof40b = stack_screen40b.sort_values(
    ["ordinary_oof_mse", "test_tier_weighted_mse"]
).reset_index(drop=True)

best_oof_name_40b = stack_screen_by_oof40b.iloc[0]["stack_name"]
best_oof_oof_40b = stack_store_40b[best_oof_name_40b]["oof"]
best_oof_test_40b = stack_store_40b[best_oof_name_40b]["test"]

# Save both best-by-weighted and best-by-OOF, even if same.
screen_path40b = "model_results/stack40b_screen.csv"
baseline_path40b = "model_results/stack40b_baseline_artifact_metrics.csv"
weights_path40b = "model_results/stack40b_all_weights.csv"

oof_weighted_path40b = "model_results/oof_stack40b_best_weighted.csv"
test_weighted_path40b = "model_results/testpred_stack40b_best_weighted.csv"
submission_weighted_path40b = "submission_stack40b_best_weighted.csv"

oof_oof_path40b = "model_results/oof_stack40b_best_oof.csv"
test_oof_path40b = "model_results/testpred_stack40b_best_oof.csv"
submission_oof_path40b = "submission_stack40b_best_oof.csv"

stack_screen40b.to_csv(screen_path40b, index=False)
baseline_metrics40b.to_csv(baseline_path40b, index=False)

all_weights40b = []
for stack_name, obj in stack_store_40b.items():
    wdf = obj["weights"].copy()
    if len(wdf) > 0:
        all_weights40b.append(wdf)

if len(all_weights40b) > 0:
    pd.concat(all_weights40b, axis=0).to_csv(weights_path40b, index=False)
else:
    pd.DataFrame().to_csv(weights_path40b, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_40b),
    TARGET_COL: y_40b,
    "pred_39c": oof_preds_40b["account39c"],
    "pred_clipped": best_weighted_oof_40b,
}).to_csv(oof_weighted_path40b, index=False)

pd.DataFrame({
    ID_COL: test_ids_40b,
    TARGET_COL: best_weighted_test_40b,
}).to_csv(submission_weighted_path40b, index=False)

pd.DataFrame({
    ID_COL: test_ids_40b,
    TARGET_COL: best_weighted_test_40b,
}).to_csv(test_weighted_path40b, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_40b),
    TARGET_COL: y_40b,
    "pred_39c": oof_preds_40b["account39c"],
    "pred_clipped": best_oof_oof_40b,
}).to_csv(oof_oof_path40b, index=False)

pd.DataFrame({
    ID_COL: test_ids_40b,
    TARGET_COL: best_oof_test_40b,
}).to_csv(submission_oof_path40b, index=False)

pd.DataFrame({
    ID_COL: test_ids_40b,
    TARGET_COL: best_oof_test_40b,
}).to_csv(test_oof_path40b, index=False)

# Validate submissions.
for p in [submission_weighted_path40b, submission_oof_path40b]:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test_40b, 2)
    assert list(sub.columns) == [ID_COL, TARGET_COL]
    assert sub[TARGET_COL].notna().all()
    assert np.isfinite(sub[TARGET_COL]).all()
    assert sub[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 9. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("40B stacking complete")
print("=" * 90)

print("\nReference")
print("---------")
print(f"39C ordinary OOF MSE:       {ordinary_mse_40b(oof_preds_40b['account39c']):.6f}")
print(f"39C test-tier weighted MSE: {tier_weighted_mse_40b(oof_preds_40b['account39c']):.6f}")

print("\nTop stack candidates by test-tier weighted MSE")
print("----------------------------------------------")
display(stack_screen40b.head(20))

print("\nTop stack candidates by ordinary OOF MSE")
print("----------------------------------------")
display(stack_screen_by_oof40b.head(20))

print("\nBest weighted stack")
print("-------------------")
print(stack_screen40b.iloc[0].to_string())

print("\nBest ordinary-OOF stack")
print("-----------------------")
print(stack_screen_by_oof40b.iloc[0].to_string())

print("\nSaved files")
print("-----------")
print(screen_path40b)
print(baseline_path40b)
print(weights_path40b)
print(oof_weighted_path40b)
print(test_weighted_path40b)
print(submission_weighted_path40b)
print(oof_oof_path40b)
print(test_oof_path40b)
print(submission_oof_path40b)

print("\nDecision rule")
print("-------------")
best_weighted_gain = float(stack_screen40b.iloc[0]["gain_vs_39c_weighted"])
best_oof_gain = float(stack_screen_by_oof40b.iloc[0]["gain_vs_39c_oof"])

if best_weighted_gain >= 3.0 and best_oof_gain >= 0.0:
    print("Stack clears the weighted 3+ threshold and does not hurt ordinary OOF. This may be submission-worthy.")
elif best_oof_gain >= 3.0 and best_weighted_gain >= 0.0:
    print("Stack clears ordinary OOF but not weighted strongly. Treat cautiously because 40A-3 failed public despite OOF gain.")
elif best_weighted_gain > 0.5 and best_oof_gain >= 0.0:
    print("Stack gives modest weighted improvement. Save artifact; do not submit unless there is strong public/structural justification.")
else:
    print("Stack does not provide enough evidence for submission. Move to 41A entity-embedding / matrix-factorization residual model.")

40B. Lightweight constrained stacking over saved artifacts
Have scipy.optimize.minimize: True
Loaded anchor33a      | OOF MSE 77.049866
Loaded knn35a         | OOF MSE 76.453072
Loaded nn37b_rank1    | OOF MSE 76.849442
Loaded step36a        | OOF MSE 76.954796
Loaded blend33c       | OOF MSE 76.859200
Loaded account39a     | OOF MSE 55.642780
Loaded account39b     | OOF MSE 53.720547
Loaded account39c     | OOF MSE 53.226631
Loaded account39d     | OOF MSE 52.820927
Loaded hte40a3        | OOF MSE 49.689606

Artifact summary
----------------


,name,oof_mse,oof_path,test_path,oof_col,test_col
0,hte40a3,49.689606,model_results/oof_hte40a3_residual_best.csv,model_results/testpred_hte40a3_residual_best.csv,pred_clipped,PERCENT_PROFICIENT
1,account39d,52.820927,model_results/oof_account39d_best.csv,model_results/testpred_account39d_best.csv,pred_clipped,PERCENT_PROFICIENT
2,account39c,53.226631,model_results/oof_account39c_final_best.csv,model_results/testpred_account39c_final_best.csv,pred_clipped,PERCENT_PROFICIENT
3,account39b,53.720547,model_results/oof_account39b_solver_best.csv,model_results/testpred_account39b_solver_best.csv,pred_clipped,PERCENT_PROFICIENT
4,account39a,55.642780,model_results/oof_account39a_best.csv,model_results/testpred_account39a_best.csv,pred_clipped,PERCENT_PROFICIENT
5,knn35a,76.453072,model_results/oof_knn35a_local_residual_best.csv,model_results/testpred_knn35a_local_residual_b...,pred_clipped,PERCENT_PROFICIENT
6,nn37b_rank1,76.849442,model_results/oof_nn37b38b_cpu_safe_rank1.csv,model_results/testpred_nn37b38b_cpu_safe_rank1...,pred_clipped,PERCENT_PROFICIENT
7,blend33c,76.859200,model_results/oof_blend33c_33a_huber33b.csv,model_results/testpred_blend33c_33a_huber33b.csv,pred_clipped,PERCENT_PROFICIENT
8,step36a,76.954796,model_results/oof_step36a_residual_bins_best.csv,model_results/testpred_step36a_residual_bins_b...,pred_clipped,PERCENT_PROFICIENT
9,anchor33a,77.049866,model_results/oof_seg33a_arcsine_adaptive.csv,model_results/testpred_seg33a_arcsine_adaptive...,pred_clipped,PERCENT_PROFICIENT



Tier coverage
-------------
overlap      train n= 53786 | test n= 27298
solver_only  train n= 27807 | test n= 17807
none         train n= 63328 | test n=  3202

Baseline artifact metrics
-------------------------


,name,ordinary_oof_mse,test_tier_weighted_mse,gain_vs_39c_oof,gain_vs_39c_weighted,mse_overlap,mse_solver_only,mse_none
0,hte40a3,49.689606,29.890315,3.537025,0.857292,0.595326,64.945831,84.687637
1,account39d,52.820927,30.147487,0.405704,0.600120,0.595325,64.304314,92.135094
2,account39c,53.226631,30.747607,0.000000,0.000000,0.595326,65.890625,92.366982
3,account39b,53.720547,31.651285,-0.493916,-0.903678,0.862888,67.930656,92.374245
4,account39a,55.642780,35.148235,-2.416149,-4.400628,0.595420,77.772476,92.678802
5,knn35a,76.453072,66.983315,-23.226440,-36.235708,57.247768,77.342117,92.374245
6,blend33c,76.859200,67.434008,-23.632568,-36.686402,57.788399,77.677948,92.696976
7,nn37b_rank1,76.849442,67.546247,-23.622810,-36.798640,58.091324,77.558929,92.469620
8,step36a,76.954796,67.675987,-23.728165,-36.928381,58.249962,77.656258,92.533257
9,anchor33a,77.049866,67.742433,-23.823235,-36.994827,58.274662,77.772476,92.678802



Candidate sets
--------------
accounting_only : ['account39a', 'account39b', 'account39c', 'account39d']
accounting_plus_ml : ['account39a', 'account39b', 'account39c', 'account39d', 'knn35a', 'nn37b_rank1', 'step36a', 'blend33c', 'anchor33a']
accounting_plus_hte : ['account39a', 'account39b', 'account39c', 'account39d', 'hte40a3', 'knn35a']

40B stacking complete

Reference
---------
39C ordinary OOF MSE:       53.226631
39C test-tier weighted MSE: 30.747607

Top stack candidates by test-tier weighted MSE
----------------------------------------------


,stack_name,stack_type,candidate_set,weight_mode,ordinary_oof_mse,test_tier_weighted_mse,gain_vs_39c_oof,gain_vs_39c_weighted,mse_overlap,mse_solver_only,mse_none
0,tiered_accounting_plus_hte,tier_specific_convex,accounting_plus_hte,tier_specific,49.621124,29.582633,3.605507,1.164974,0.595349,64.070160,84.915390
1,global_accounting_plus_hte_test_tier_weighted,global_convex,accounting_plus_hte,test_tier_weighted,50.049191,29.700316,3.177441,1.047291,0.595325,64.225563,85.826790
2,global_accounting_plus_hte_ordinary,global_convex,accounting_plus_hte,ordinary,49.813713,29.832814,3.412918,0.914793,0.595326,64.721024,85.070358
3,tiered_accounting_plus_ml,tier_specific_convex,accounting_plus_ml,tier_specific,52.817196,30.148261,0.409435,0.599346,0.595351,64.308220,92.124825
4,tiered_accounting_only,tier_specific_convex,accounting_only,tier_specific,52.817398,30.148290,0.409233,0.599317,0.595349,64.308220,92.125275
5,global_accounting_only_test_tier_weighted,global_convex,accounting_only,test_tier_weighted,52.821938,30.149470,0.404694,0.598136,0.595339,64.309685,92.135025
6,global_accounting_plus_ml_test_tier_weighted,global_convex,accounting_plus_ml,test_tier_weighted,52.821938,30.149470,0.404694,0.598136,0.595339,64.309685,92.135025
7,global_accounting_only_ordinary,global_convex,accounting_only,ordinary,52.827183,30.160243,0.399448,0.587364,0.595596,64.338684,92.134079
8,global_accounting_plus_ml_ordinary,global_convex,accounting_plus_ml,ordinary,52.827187,30.160261,0.399445,0.587346,0.595589,64.338745,92.134071



Top stack candidates by ordinary OOF MSE
----------------------------------------


,stack_name,stack_type,candidate_set,weight_mode,ordinary_oof_mse,test_tier_weighted_mse,gain_vs_39c_oof,gain_vs_39c_weighted,mse_overlap,mse_solver_only,mse_none
0,tiered_accounting_plus_hte,tier_specific_convex,accounting_plus_hte,tier_specific,49.621124,29.582633,3.605507,1.164974,0.595349,64.070160,84.915390
1,global_accounting_plus_hte_ordinary,global_convex,accounting_plus_hte,ordinary,49.813713,29.832814,3.412918,0.914793,0.595326,64.721024,85.070358
2,global_accounting_plus_hte_test_tier_weighted,global_convex,accounting_plus_hte,test_tier_weighted,50.049191,29.700316,3.177441,1.047291,0.595325,64.225563,85.826790
3,tiered_accounting_plus_ml,tier_specific_convex,accounting_plus_ml,tier_specific,52.817196,30.148261,0.409435,0.599346,0.595351,64.308220,92.124825
4,tiered_accounting_only,tier_specific_convex,accounting_only,tier_specific,52.817398,30.148290,0.409233,0.599317,0.595349,64.308220,92.125275
5,global_accounting_only_test_tier_weighted,global_convex,accounting_only,test_tier_weighted,52.821938,30.149470,0.404694,0.598136,0.595339,64.309685,92.135025
6,global_accounting_plus_ml_test_tier_weighted,global_convex,accounting_plus_ml,test_tier_weighted,52.821938,30.149470,0.404694,0.598136,0.595339,64.309685,92.135025
7,global_accounting_only_ordinary,global_convex,accounting_only,ordinary,52.827183,30.160243,0.399448,0.587364,0.595596,64.338684,92.134079
8,global_accounting_plus_ml_ordinary,global_convex,accounting_plus_ml,ordinary,52.827187,30.160261,0.399445,0.587346,0.595589,64.338745,92.134071



Best weighted stack
-------------------
stack_name                tiered_accounting_plus_hte
stack_type                      tier_specific_convex
candidate_set                    accounting_plus_hte
weight_mode                            tier_specific
ordinary_oof_mse                           49.621124
test_tier_weighted_mse                     29.582633
gain_vs_39c_oof                             3.605507
gain_vs_39c_weighted                        1.164974
mse_overlap                                 0.595349
mse_solver_only                             64.07016
mse_none                                    84.91539

Best ordinary-OOF stack
-----------------------
stack_name                tiered_accounting_plus_hte
stack_type                      tier_specific_convex
candidate_set                    accounting_plus_hte
weight_mode                            tier_specific
ordinary_oof_mse                           49.621124
test_tier_weighted_mse                     29.582633
gain_vs_3

### 40B. Lightweight Stacking of Saved Model Artifacts

This section tested whether previously trained model-family artifacts could improve the accounting-based models through constrained stacking. The stack included the original target-encoded boosting anchor, KNN residual model, neural-network residual model, step-function residual model, previous blends, and the accounting models 39A–39D.

The results show that the accounting models dominate the stack. Older generic ML artifacts such as KNN, neural networks, step functions, and boosting blends had OOF MSE around `76–77`, while the accounting models were much stronger, with 39C at `53.226631` and 39D at `52.820927`. As a result, adding the older regular ML artifacts produced almost no improvement beyond the accounting-only stack.

The best accounting-only/tiered stack achieved OOF MSE around `52.817`, a modest improvement over 39C but not large enough to justify another submission. The best stack overall included the hierarchical residual model 40A-3 and achieved OOF MSE `49.621124`, but this was considered high-risk because the standalone 40A-3 submission performed worse on the public leaderboard.

Conclusion: stacking confirmed that the dominant signal source is the subgroup accounting structure. Generic ML artifacts provide little additional value after accounting. Hierarchical residual features show OOF signal, but their public validation behavior is unreliable, so they should not be trusted for submission without stronger evidence.

In [34]:
# ============================================================
# 41A-1. Entity-embedding / matrix-factorization setup
# ============================================================
#
# New-signal branch:
#   Latent entity effects for school / district / assessment / subgroup.
#
# Target for later model:
#   residual_39C = y - pred_39C
#
# This cell:
#   - loads 39C predictions,
#   - rebuilds tier flags,
#   - audits categorical entity coverage,
#   - builds a small numeric feature frame,
#   - saves setup diagnostics.
#
# It does NOT train a model yet.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("41A-1. Entity-embedding / matrix-factorization setup")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required checks
# ------------------------------------------------------------

required_41a1 = ["raw_train_te", "raw_test_te", "X_train_proc_model", "X_test_proc_model", "y_train"]
missing_41a1 = [x for x in required_41a1 if x not in globals()]

if missing_41a1:
    raise ValueError(f"Missing required setup objects: {missing_41a1}. Rerun safe recovery/setup first.")

y_41a = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_41a = len(y_41a)
n_test_41a = len(raw_test_te)

if len(raw_train_te) != n_train_41a:
    raise ValueError("raw_train_te and y_train row count mismatch.")
if len(raw_test_te) != X_test_proc_model.shape[0]:
    raise ValueError("raw_test_te and X_test_proc_model row count mismatch.")

# ------------------------------------------------------------
# 2. Load 39C public-best artifact
# ------------------------------------------------------------

oof39c_path_41a = "model_results/oof_account39c_final_best.csv"
test39c_path_41a = "model_results/testpred_account39c_final_best.csv"

if not Path(oof39c_path_41a).exists() or not Path(test39c_path_41a).exists():
    raise FileNotFoundError("Missing 39C OOF/test artifacts.")

oof39c_41a = pd.read_csv(oof39c_path_41a)
test39c_41a = pd.read_csv(test39c_path_41a)

if "row_index" in oof39c_41a.columns:
    oof39c_41a = oof39c_41a.sort_values("row_index").reset_index(drop=True)

if "pred_clipped" in oof39c_41a.columns:
    pred39c_oof_41a = oof39c_41a["pred_clipped"].to_numpy(dtype=np.float32)
elif TARGET_COL in oof39c_41a.columns:
    pred39c_oof_41a = oof39c_41a[TARGET_COL].to_numpy(dtype=np.float32)
else:
    raise ValueError("Could not find 39C OOF prediction column.")

if TARGET_COL in test39c_41a.columns:
    pred39c_test_41a = test39c_41a[TARGET_COL].to_numpy(dtype=np.float32)
else:
    numeric_cols = [
        c for c in test39c_41a.columns
        if c != ID_COL and pd.api.types.is_numeric_dtype(test39c_41a[c])
    ]
    if len(numeric_cols) == 0:
        raise ValueError("Could not find 39C test prediction column.")
    pred39c_test_41a = test39c_41a[numeric_cols[0]].to_numpy(dtype=np.float32)

if ID_COL in test39c_41a.columns:
    test_ids_41a = test39c_41a[ID_COL].to_numpy()
elif "test_ids" in globals():
    test_ids_41a = np.asarray(test_ids)
else:
    raise ValueError("Could not recover test IDs.")

pred39c_oof_41a = np.clip(pred39c_oof_41a, 0, 100).astype(np.float32)
pred39c_test_41a = np.clip(pred39c_test_41a, 0, 100).astype(np.float32)

resid39c_41a = (y_41a - pred39c_oof_41a).astype(np.float32)
mse39c_41a = float(mean_squared_error(y_41a, pred39c_oof_41a))

print("\n39C baseline")
print("------------")
print(f"39C OOF MSE: {mse39c_41a:.6f}")
print("Residual mean:", float(np.mean(resid39c_41a)))
print("Residual std: ", float(np.std(resid39c_41a)))

# ------------------------------------------------------------
# 3. Recover accounting tiers
# ------------------------------------------------------------

tier_files_41a = [
    "model_results/account39a_raw_oof_reconstruction.csv",
    "model_results/account39a_raw_test_reconstruction.csv",
    "model_results/account39b_solver_raw_oof.csv",
    "model_results/account39b_solver_raw_test.csv",
]

missing_tier_files_41a = [p for p in tier_files_41a if not Path(p).exists()]
if missing_tier_files_41a:
    raise FileNotFoundError(f"Missing accounting tier files: {missing_tier_files_41a}")

raw39a_oof_41a = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test_41a = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof_41a = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test_41a = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof_41a.columns:
    raw39a_oof_41a = raw39a_oof_41a.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof_41a.columns:
    raw39b_oof_41a = raw39b_oof_41a.sort_values("row_index").reset_index(drop=True)

direct_cov_oof_41a = raw39a_oof_41a["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test_41a = raw39a_test_41a["accounting_covered"].astype(int).to_numpy().astype(bool)

solver_cov_oof_41a = raw39b_oof_41a["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test_41a = raw39b_test_41a["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof_41a = direct_cov_oof_41a & solver_cov_oof_41a
solver_only_oof_41a = solver_cov_oof_41a & ~direct_cov_oof_41a
none_oof_41a = ~(direct_cov_oof_41a | solver_cov_oof_41a)

overlap_test_41a = direct_cov_test_41a & solver_cov_test_41a
solver_only_test_41a = solver_cov_test_41a & ~direct_cov_test_41a
none_test_41a = ~(direct_cov_test_41a | solver_cov_test_41a)

tier_masks_oof_41a = {
    "overlap": overlap_oof_41a,
    "solver_only": solver_only_oof_41a,
    "none": none_oof_41a,
}

tier_masks_test_41a = {
    "overlap": overlap_test_41a,
    "solver_only": solver_only_test_41a,
    "none": none_test_41a,
}

tier_rows_41a = []

for tier, mask in tier_masks_oof_41a.items():
    tier_rows_41a.append({
        "tier": tier,
        "train_rows": int(mask.sum()),
        "test_rows": int(tier_masks_test_41a[tier].sum()),
        "train_rate": float(mask.mean()),
        "test_rate": float(tier_masks_test_41a[tier].mean()),
        "mse_39c": float(mean_squared_error(y_41a[mask], pred39c_oof_41a[mask])) if int(mask.sum()) else np.nan,
        "residual_mean": float(np.mean(resid39c_41a[mask])) if int(mask.sum()) else np.nan,
        "residual_std": float(np.std(resid39c_41a[mask])) if int(mask.sum()) else np.nan,
    })

tier_diag41a = pd.DataFrame(tier_rows_41a)

print("\nTier diagnostic")
print("---------------")
print(tier_diag41a.to_string(index=False))

# ------------------------------------------------------------
# 4. Entity definitions and coverage audit
# ------------------------------------------------------------

def clean_cat_41a(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def make_entity_values_41a(df, cols):
    cols = tuple(cols)
    if len(cols) == 1:
        return clean_cat_41a(df[cols[0]]).to_numpy()

    parts = [clean_cat_41a(df[c]) for c in cols]
    out = parts[0].copy()

    for p in parts[1:]:
        out = out + "||" + p

    return out.to_numpy()

candidate_entities_41a = [
    {"name": "school", "cols": ("SCHOOL",), "max_levels": 6000, "min_count": 1},
    {"name": "district", "cols": ("DISTRICT",), "max_levels": 1000, "min_count": 1},
    {"name": "county", "cols": ("COUNTY",), "max_levels": 100, "min_count": 1},
    {"name": "region", "cols": ("REGION",), "max_levels": 50, "min_count": 1},
    {"name": "assessment", "cols": ("ASSESSMENT_NAME",), "max_levels": 100, "min_count": 1},
    {"name": "subgroup", "cols": ("SUBGROUP_NAME",), "max_levels": 20, "min_count": 1},
    {"name": "district_type", "cols": ("DISTRICT_TYPE",), "max_levels": 20, "min_count": 1},

    {"name": "assessment_subgroup", "cols": ("ASSESSMENT_NAME", "SUBGROUP_NAME"), "max_levels": 300, "min_count": 1},
    {"name": "school_assessment", "cols": ("SCHOOL", "ASSESSMENT_NAME"), "max_levels": 50000, "min_count": 1},
    {"name": "school_subgroup", "cols": ("SCHOOL", "SUBGROUP_NAME"), "max_levels": 30000, "min_count": 1},
    {"name": "district_assessment", "cols": ("DISTRICT", "ASSESSMENT_NAME"), "max_levels": 25000, "min_count": 1},
]

entity_defs_41a = []
entity_audit_rows_41a = []

for d in candidate_entities_41a:
    cols = tuple(d["cols"])

    if not all(c in raw_train_te.columns for c in cols):
        continue
    if not all(c in raw_test_te.columns for c in cols):
        continue

    train_vals = make_entity_values_41a(raw_train_te, cols)
    test_vals = make_entity_values_41a(raw_test_te, cols)

    vc = pd.Series(train_vals).value_counts(dropna=False)
    keep = vc[vc >= int(d["min_count"])].head(int(d["max_levels"]) - 1).index.astype(str).tolist()
    keep_set = set(keep)

    test_seen = pd.Series(test_vals).astype(str).isin(keep_set)

    entity_defs_41a.append(d)

    entity_audit_rows_41a.append({
        "entity": d["name"],
        "cols": " × ".join(cols),
        "train_unique": int(pd.Series(train_vals).nunique()),
        "test_unique": int(pd.Series(test_vals).nunique()),
        "levels_kept_plus_unknown": len(keep) + 1,
        "min_count": d["min_count"],
        "max_levels": d["max_levels"],
        "test_row_coverage_after_cap": float(test_seen.mean()),
        "test_uncovered_rows": int((~test_seen).sum()),
    })

entity_audit41a = pd.DataFrame(entity_audit_rows_41a)

print("\nEntity coverage audit")
print("---------------------")
print(entity_audit41a.to_string(index=False))

if len(entity_defs_41a) == 0:
    raise ValueError("No usable entity definitions found.")

# ------------------------------------------------------------
# 5. Numeric feature frame for entity model
# ------------------------------------------------------------

num_train_41a = pd.DataFrame(index=np.arange(n_train_41a))
num_test_41a = pd.DataFrame(index=np.arange(n_test_41a))

num_train_41a["pred39c"] = pred39c_oof_41a.astype(np.float32)
num_test_41a["pred39c"] = pred39c_test_41a.astype(np.float32)

# Add available prediction differences from saved artifacts.
def load_pred_pair_41a(name, oof_path, test_path):
    if not Path(oof_path).exists() or not Path(test_path).exists():
        return None, None

    oof = pd.read_csv(oof_path)
    test = pd.read_csv(test_path)

    if "row_index" in oof.columns:
        oof = oof.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in oof.columns:
        p_oof = oof["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in oof.columns:
        p_oof = oof[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        return None, None

    if TARGET_COL in test.columns:
        p_test = test[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in test.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(test[c])
        ]
        if len(numeric_cols) == 0:
            return None, None
        p_test = test[numeric_cols[0]].to_numpy(dtype=np.float32)

    return np.clip(p_oof, 0, 100), np.clip(p_test, 0, 100)

prediction_diff_specs_41a = [
    ("account39b", "model_results/oof_account39b_solver_best.csv", "model_results/testpred_account39b_solver_best.csv"),
    ("account39d", "model_results/oof_account39d_best.csv", "model_results/testpred_account39d_best.csv"),
    ("knn35a", "model_results/oof_knn35a_local_residual_best.csv", "model_results/testpred_knn35a_local_residual_best.csv"),
    ("nn37b", "model_results/oof_nn37b38b_cpu_safe_rank1.csv", "model_results/testpred_nn37b38b_cpu_safe_rank1.csv"),
]

for name, oof_path, test_path in prediction_diff_specs_41a:
    p_oof, p_test = load_pred_pair_41a(name, oof_path, test_path)

    if p_oof is not None and len(p_oof) == n_train_41a and len(p_test) == n_test_41a:
        num_train_41a[f"diff_{name}_minus_39c"] = (p_oof - pred39c_oof_41a).astype(np.float32)
        num_test_41a[f"diff_{name}_minus_39c"] = (p_test - pred39c_test_41a).astype(np.float32)

# Tier flags.
num_train_41a["tier_overlap"] = overlap_oof_41a.astype(np.float32)
num_train_41a["tier_solver_only"] = solver_only_oof_41a.astype(np.float32)
num_train_41a["tier_none"] = none_oof_41a.astype(np.float32)

num_test_41a["tier_overlap"] = overlap_test_41a.astype(np.float32)
num_test_41a["tier_solver_only"] = solver_only_test_41a.astype(np.float32)
num_test_41a["tier_none"] = none_test_41a.astype(np.float32)

# Numeric covariates.
numeric_covariates_41a = [
    "N_STUDENTS",
    "PERCENT_FREE_LUNCH",
    "PERCENT_REDUCED_LUNCH",
    "PERCENT_ECONOMICALLY_DISADVANTAGED",
    "PERCENT_ENGLISH_LANGUAGE_LEARNERS",
    "PERCENT_ENGLISH_LANGUAGE_LEANERS",
    "PERCENT_WITH_DISABILITIES",
    "PERCENT_STUDENTS_WITH_DISABILITIES",
    "PERCENT_HOMELESS",
    "PERCENT_MIGRANT",
    "PERCENT_FEMALE",
    "PERCENT_MALE",
    "ATTENDANCE_RATE",
]

added_num_cols_41a = []

for c in numeric_covariates_41a:
    if c in X_train_proc_model.columns and c in X_test_proc_model.columns:
        if pd.api.types.is_numeric_dtype(X_train_proc_model[c]):
            if c not in added_num_cols_41a:
                train_vals = (
                    pd.to_numeric(X_train_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )
                test_vals = (
                    pd.to_numeric(X_test_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )

                num_train_41a[f"num_{c}"] = train_vals
                num_test_41a[f"num_{c}"] = test_vals
                added_num_cols_41a.append(c)

if "num_N_STUDENTS" in num_train_41a.columns:
    num_train_41a["num_log1p_N_STUDENTS"] = np.log1p(
        np.maximum(num_train_41a["num_N_STUDENTS"].to_numpy(dtype=np.float32), 0)
    )
    num_test_41a["num_log1p_N_STUDENTS"] = np.log1p(
        np.maximum(num_test_41a["num_N_STUDENTS"].to_numpy(dtype=np.float32), 0)
    )

num_train_41a = num_train_41a.replace([np.inf, -np.inf], np.nan)
num_test_41a = num_test_41a.replace([np.inf, -np.inf], np.nan)

print("\nNumeric entity-model features")
print("-----------------------------")
print("Train numeric shape:", num_train_41a.shape)
print("Test numeric shape: ", num_test_41a.shape)
print("Numeric columns:")
print(list(num_train_41a.columns))

# ------------------------------------------------------------
# 6. Save setup diagnostics
# ------------------------------------------------------------

entity_audit_path41a = "model_results/entity41a_entity_audit.csv"
tier_diag_path41a = "model_results/entity41a_tier_diag.csv"
num_cols_path41a = "model_results/entity41a_numeric_columns.csv"

entity_audit41a.to_csv(entity_audit_path41a, index=False)
tier_diag41a.to_csv(tier_diag_path41a, index=False)
pd.DataFrame({"numeric_feature": list(num_train_41a.columns)}).to_csv(num_cols_path41a, index=False)

print("\nSaved setup files")
print("-----------------")
print(entity_audit_path41a)
print(tier_diag_path41a)
print(num_cols_path41a)

print("\n41A-1 complete.")
print("Next: run 41A-2 to train one small CPU-only entity-embedding residual model.")

41A-1. Entity-embedding / matrix-factorization setup

39C baseline
------------
39C OOF MSE: 53.226631
Residual mean: -0.14721155166625977
Residual std:  7.294173240661621

Tier diagnostic
---------------
       tier  train_rows  test_rows  train_rate  test_rate   mse_39c  residual_mean  residual_std
    overlap       53786      27298    0.371140   0.565094  0.595326       0.003323      0.771567
solver_only       27807      17807    0.191877   0.368622 65.890625      -0.245121      8.113603
       none       63328       3202    0.436983   0.066284 92.366982      -0.232073      9.607971

Entity coverage audit
---------------------
             entity                            cols  train_unique  test_unique  levels_kept_plus_unknown  min_count  max_levels  test_row_coverage_after_cap  test_uncovered_rows
             school                          SCHOOL          4469         4448                      4470          1        6000                     1.000000                    0
      

In [35]:
# ============================================================
# 41A-2. CPU-safe entity-embedding residual model
# ============================================================
#
# Requires 41A-1 variables.
#
# Model:
#   entity embeddings + numeric features -> residual_39C
#
# Prediction:
#   final = pred39C + lambda * predicted_residual
#
# This is a new-signal branch:
#   It learns latent entity effects rather than group means or accounting rules.
#
# Submission rule:
#   Do not submit unless OOF gain is very strong and tier diagnostics support it.
# ============================================================

import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
os.makedirs("model_results", exist_ok=True)
os.makedirs("model_results/entity41a_checkpoints", exist_ok=True)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

# Force CPU. Do not use MPS.
device_41a = torch.device("cpu")

try:
    torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))
except Exception:
    pass

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("=" * 90)
print("41A-2. CPU-safe entity-embedding residual model")
print("=" * 90)
print("Device:", device_41a)
print("Torch threads:", torch.get_num_threads())

# ------------------------------------------------------------
# 1. Required object checks
# ------------------------------------------------------------

required_41a2 = [
    "raw_train_te",
    "raw_test_te",
    "y_41a",
    "pred39c_oof_41a",
    "pred39c_test_41a",
    "resid39c_41a",
    "test_ids_41a",
    "entity_defs_41a",
    "num_train_41a",
    "num_test_41a",
    "tier_masks_oof_41a",
    "tier_masks_test_41a",
]

missing_41a2 = [x for x in required_41a2 if x not in globals()]
if missing_41a2:
    raise ValueError(f"Missing 41A-1 objects: {missing_41a2}. Run 41A-1 first.")

y_41a = np.asarray(y_41a, dtype=np.float32).reshape(-1)
pred39c_oof_41a = np.asarray(pred39c_oof_41a, dtype=np.float32).reshape(-1)
pred39c_test_41a = np.asarray(pred39c_test_41a, dtype=np.float32).reshape(-1)
resid39c_41a = np.asarray(resid39c_41a, dtype=np.float32).reshape(-1)

n_train_41a = len(y_41a)
n_test_41a = len(pred39c_test_41a)

mse39c_41a = float(mean_squared_error(y_41a, pred39c_oof_41a))

print("\nBaseline")
print("--------")
print(f"39C OOF MSE: {mse39c_41a:.6f}")

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def clean_cat_41a2(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def make_entity_values_41a2(df, cols):
    cols = tuple(cols)

    if len(cols) == 1:
        return clean_cat_41a2(df[cols[0]]).to_numpy()

    parts = [clean_cat_41a2(df[c]) for c in cols]
    out = parts[0].copy()

    for p in parts[1:]:
        out = out + "||" + p

    return out.to_numpy()

def fit_transform_entities_41a2(raw_fit, raw_val, raw_test, entity_defs):
    fit_codes = []
    val_codes = []
    test_codes = []
    cards = []
    summary_rows = []

    for d in entity_defs:
        name = d["name"]
        cols = tuple(d["cols"])

        fit_vals = make_entity_values_41a2(raw_fit, cols)
        val_vals = make_entity_values_41a2(raw_val, cols)
        test_vals = make_entity_values_41a2(raw_test, cols)

        vc = pd.Series(fit_vals).value_counts(dropna=False)
        keep = vc[vc >= int(d["min_count"])].head(int(d["max_levels"]) - 1).index.astype(str).tolist()

        mapping = {v: i + 1 for i, v in enumerate(keep)}
        card = len(keep) + 1

        def map_vals(vals):
            return np.array([mapping.get(str(v), 0) for v in vals], dtype=np.int64)

        fit_c = map_vals(fit_vals)
        val_c = map_vals(val_vals)
        test_c = map_vals(test_vals)

        fit_codes.append(fit_c)
        val_codes.append(val_c)
        test_codes.append(test_c)
        cards.append(card)

        summary_rows.append({
            "entity": name,
            "cols": " × ".join(cols),
            "cardinality": int(card),
            "fit_unique": int(pd.Series(fit_vals).nunique()),
            "val_unknown_rate": float(np.mean(val_c == 0)),
            "test_unknown_rate": float(np.mean(test_c == 0)),
        })

    return (
        np.column_stack(fit_codes).astype(np.int64),
        np.column_stack(val_codes).astype(np.int64),
        np.column_stack(test_codes).astype(np.int64),
        cards,
        pd.DataFrame(summary_rows),
    )

def emb_dim_41a(card):
    card = int(card)
    if card <= 2:
        return 1
    if card <= 10:
        return min(4, card)
    if card <= 50:
        return 8
    if card <= 500:
        return 12
    if card <= 5000:
        return 16
    return 24

class EntityDataset41A(Dataset):
    def __init__(self, X_num, X_cat, y=None, w=None):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32).view(-1, 1)
        self.w = None if w is None else torch.tensor(w, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return self.X_num.shape[0]

    def __getitem__(self, idx):
        if self.y is None:
            return self.X_num[idx], self.X_cat[idx]
        return self.X_num[idx], self.X_cat[idx], self.y[idx], self.w[idx]

class EntityMLP41A(nn.Module):
    def __init__(self, n_num, cat_cards, hidden_layers=(128, 64), dropout=0.10):
        super().__init__()

        self.embeddings = nn.ModuleList()
        emb_total = 0

        for card in cat_cards:
            dim = emb_dim_41a(card)
            self.embeddings.append(nn.Embedding(int(card), int(dim)))
            emb_total += int(dim)

        input_dim = int(n_num) + int(emb_total)

        layers = []
        prev = input_dim

        for h in hidden_layers:
            layers.append(nn.Linear(prev, int(h)))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(float(dropout)))
            prev = int(h)

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        pieces = [x_num]

        if len(self.embeddings) > 0:
            emb_parts = []
            for j, emb in enumerate(self.embeddings):
                emb_parts.append(emb(x_cat[:, j]))
            pieces.append(torch.cat(emb_parts, dim=1))

        x = torch.cat(pieces, dim=1)
        return self.net(x)

def make_weights_41a2(idx, scheme):
    w = np.ones(len(idx), dtype=np.float32)

    ov = tier_masks_oof_41a["overlap"][idx]
    so = tier_masks_oof_41a["solver_only"][idx]
    no = tier_masks_oof_41a["none"][idx]

    if scheme == "test_tier":
        # approximate test composition, but do not overweight overlap errors because it is already solved
        train_rates = {
            "overlap": max(float(ov.mean()), 1e-12),
            "solver_only": max(float(so.mean()), 1e-12),
            "none": max(float(no.mean()), 1e-12),
        }
        test_rates = {
            k: float(tier_masks_test_41a[k].mean())
            for k in ["overlap", "solver_only", "none"]
        }

        w[ov] = 0.10 * test_rates["overlap"] / train_rates["overlap"]
        w[so] = 1.00 * test_rates["solver_only"] / train_rates["solver_only"]
        w[no] = 0.50 * test_rates["none"] / train_rates["none"]
        return w

    if scheme == "solver_focus":
        w[ov] = 0.05
        w[so] = 2.00
        w[no] = 0.50
        return w

    if scheme == "all":
        return w

    raise ValueError(f"Unknown weight scheme: {scheme}")

def train_entity_model_41a2(
    X_fit_num,
    X_val_num,
    X_test_num,
    X_fit_cat,
    X_val_cat,
    X_test_cat,
    y_fit_resid,
    sample_weight,
    cat_cards,
    seed,
    epochs=32,
    batch_size=4096,
    lr=7e-4,
    weight_decay=3e-4,
    hidden_layers=(128, 64),
    dropout=0.10,
):
    torch.manual_seed(seed)

    y_fit_resid = np.asarray(y_fit_resid, dtype=np.float32).reshape(-1)
    sample_weight = np.asarray(sample_weight, dtype=np.float32).reshape(-1)

    y_mean = float(np.average(y_fit_resid, weights=sample_weight))
    y_var = float(np.average((y_fit_resid - y_mean) ** 2, weights=sample_weight))
    y_std = float(np.sqrt(max(y_var, 1e-6)))

    y_scaled = ((y_fit_resid - y_mean) / y_std).astype(np.float32)

    ds = EntityDataset41A(X_fit_num, X_fit_cat, y_scaled, sample_weight)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0)

    model = EntityMLP41A(
        n_num=X_fit_num.shape[1],
        cat_cards=cat_cards,
        hidden_layers=hidden_layers,
        dropout=dropout,
    ).to(device_41a)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []

        for xb_num, xb_cat, yb, wb in loader:
            xb_num = xb_num.to(device_41a)
            xb_cat = xb_cat.to(device_41a)
            yb = yb.to(device_41a)
            wb = wb.to(device_41a)

            opt.zero_grad(set_to_none=True)
            pred = model(xb_num, xb_cat)
            loss_raw = (pred - yb) ** 2
            loss = (loss_raw * wb).sum() / (wb.sum() + 1e-8)
            loss.backward()
            opt.step()

            losses.append(float(loss.detach().cpu().item()))

        if epoch in [1, 5, 10, 20, epochs]:
            print(f"    epoch {epoch:03d} | weighted train loss {np.mean(losses):.6f}")

    def predict(X_num, X_cat):
        pred_ds = EntityDataset41A(X_num, X_cat, None, None)
        pred_loader = DataLoader(pred_ds, batch_size=batch_size * 2, shuffle=False, num_workers=0)

        out = []
        model.eval()

        with torch.no_grad():
            for xb_num, xb_cat in pred_loader:
                xb_num = xb_num.to(device_41a)
                xb_cat = xb_cat.to(device_41a)

                pred_scaled = model(xb_num, xb_cat).detach().cpu().numpy().reshape(-1)
                pred_raw = pred_scaled * y_std + y_mean
                out.append(pred_raw.astype(np.float32))

        return np.concatenate(out).astype(np.float32)

    pred_val = predict(X_val_num, X_val_cat)
    pred_test = predict(X_test_num, X_test_cat)

    del model, opt, loader, ds
    gc.collect()

    return pred_val, pred_test

# ------------------------------------------------------------
# 3. Configs
# ------------------------------------------------------------

configs_41a2 = [
    {
        "name": "entity41a_solver_focus_core",
        "weight_scheme": "solver_focus",
        "epochs": 32,
        "hidden_layers": (128, 64),
        "dropout": 0.10,
        "lr": 7e-4,
        "weight_decay": 3e-4,
    },
    {
        "name": "entity41a_testtier_core",
        "weight_scheme": "test_tier",
        "epochs": 32,
        "hidden_layers": (128, 64),
        "dropout": 0.10,
        "lr": 7e-4,
        "weight_decay": 3e-4,
    },
]

print("\nConfigs")
print("-------")
for cfg in configs_41a2:
    print(cfg)

# ------------------------------------------------------------
# 4. OOF training loop
# ------------------------------------------------------------

folds41a = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_41a))
)

resid_oof_pred_41a = {
    cfg["name"]: np.full(n_train_41a, np.nan, dtype=np.float32)
    for cfg in configs_41a2
}

resid_test_sum_41a = {
    cfg["name"]: np.zeros(n_test_41a, dtype=np.float64)
    for cfg in configs_41a2
}

fold_metric_rows_41a = []
entity_summary_rows_41a = []

for fold_num, (tr_idx, va_idx) in enumerate(folds41a, start=1):
    print("\n" + "=" * 90)
    print(f"41A OUTER FOLD {fold_num}/{N_SPLITS}")
    print("=" * 90)

    raw_fit = raw_train_te.iloc[tr_idx].reset_index(drop=True)
    raw_val = raw_train_te.iloc[va_idx].reset_index(drop=True)
    raw_test = raw_test_te.reset_index(drop=True)

    X_fit_num_raw = num_train_41a.iloc[tr_idx].reset_index(drop=True)
    X_val_num_raw = num_train_41a.iloc[va_idx].reset_index(drop=True)
    X_test_num_raw = num_test_41a.reset_index(drop=True)

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_fit_num = imputer.fit_transform(X_fit_num_raw).astype(np.float32)
    X_val_num = imputer.transform(X_val_num_raw).astype(np.float32)
    X_test_num = imputer.transform(X_test_num_raw).astype(np.float32)

    X_fit_num = scaler.fit_transform(X_fit_num).astype(np.float32)
    X_val_num = scaler.transform(X_val_num).astype(np.float32)
    X_test_num = scaler.transform(X_test_num).astype(np.float32)

    X_fit_cat, X_val_cat, X_test_cat, cat_cards, entity_summary = fit_transform_entities_41a2(
        raw_fit=raw_fit,
        raw_val=raw_val,
        raw_test=raw_test,
        entity_defs=entity_defs_41a,
    )

    entity_summary["fold"] = fold_num
    entity_summary_rows_41a.extend(entity_summary.to_dict("records"))

    print("Numeric shape:", X_fit_num.shape)
    print("Cat shape:", X_fit_cat.shape)
    print("Total embedding cards:", sum(cat_cards))
    print("Top entity unknown rates:")
    print(entity_summary.sort_values("test_unknown_rate", ascending=False).head(8).to_string(index=False))

    for cfg in configs_41a2:
        name = cfg["name"]

        ckpt_path = f"model_results/entity41a_checkpoints/{name}_fold{fold_num}.npz"

        if Path(ckpt_path).exists():
            ckpt = np.load(ckpt_path)
            pred_val_resid = ckpt["pred_val_resid"].astype(np.float32)
            pred_test_resid = ckpt["pred_test_resid"].astype(np.float32)
            status = "loaded_checkpoint"
        else:
            print(f"\nTraining {name}")

            sample_weight = make_weights_41a2(tr_idx, cfg["weight_scheme"])

            pred_val_resid, pred_test_resid = train_entity_model_41a2(
                X_fit_num=X_fit_num,
                X_val_num=X_val_num,
                X_test_num=X_test_num,
                X_fit_cat=X_fit_cat,
                X_val_cat=X_val_cat,
                X_test_cat=X_test_cat,
                y_fit_resid=resid39c_41a[tr_idx],
                sample_weight=sample_weight,
                cat_cards=cat_cards,
                seed=RANDOM_STATE + fold_num * 1000 + len(name),
                epochs=cfg["epochs"],
                batch_size=4096,
                lr=cfg["lr"],
                weight_decay=cfg["weight_decay"],
                hidden_layers=cfg["hidden_layers"],
                dropout=cfg["dropout"],
            )

            np.savez_compressed(
                ckpt_path,
                pred_val_resid=pred_val_resid.astype(np.float32),
                pred_test_resid=pred_test_resid.astype(np.float32),
            )

            status = "ok"

        resid_oof_pred_41a[name][va_idx] = pred_val_resid
        resid_test_sum_41a[name] += pred_test_resid.astype(np.float64)

        pred_val_lam1 = np.clip(pred39c_oof_41a[va_idx] + pred_val_resid, 0, 100)

        row = {
            "fold": fold_num,
            "config": name,
            "weight_scheme": cfg["weight_scheme"],
            "mse_39c": float(mean_squared_error(y_41a[va_idx], pred39c_oof_41a[va_idx])),
            "mse_lam1": float(mean_squared_error(y_41a[va_idx], pred_val_lam1)),
            "resid_pred_mse": float(mean_squared_error(resid39c_41a[va_idx], pred_val_resid)),
            "status": status,
        }
        row["gain_lam1_vs_39c"] = row["mse_39c"] - row["mse_lam1"]

        for tier, mask_full in tier_masks_oof_41a.items():
            mask_fold = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask_fold.sum())

            if int(mask_fold.sum()) > 0:
                row[f"mse39c_{tier}"] = float(mean_squared_error(
                    y_41a[va_idx][mask_fold],
                    pred39c_oof_41a[va_idx][mask_fold],
                ))
                row[f"mse_lam1_{tier}"] = float(mean_squared_error(
                    y_41a[va_idx][mask_fold],
                    pred_val_lam1[mask_fold],
                ))
                row[f"gain_lam1_{tier}"] = row[f"mse39c_{tier}"] - row[f"mse_lam1_{tier}"]

        fold_metric_rows_41a.append(row)

        print(
            f"{name} | fold {fold_num} | lam1 MSE {row['mse_lam1']:.6f} "
            f"| gain {row['gain_lam1_vs_39c']:.6f} | {status}"
        )

    del X_fit_num, X_val_num, X_test_num
    del X_fit_cat, X_val_cat, X_test_cat
    gc.collect()

# ------------------------------------------------------------
# 5. Lambda scans
# ------------------------------------------------------------

lambda_grid_41a = np.unique(
    np.concatenate([
        np.linspace(-1.0, 1.5, 626),
        np.array([0.0, 0.05, 0.10, 0.25, 0.50, 0.75, 1.0]),
    ])
)

def best_lambda_mask_41a(resid_pred, mask):
    mask = np.asarray(mask, dtype=bool)

    if int(mask.sum()) == 0:
        return {"lambda": 0.0, "mse": np.nan, "sse": 0.0, "n": 0}

    y_m = y_41a[mask].astype(np.float64)
    base_m = pred39c_oof_41a[mask].astype(np.float64)
    r_m = resid_pred[mask].astype(np.float64)

    best = None

    for lam in lambda_grid_41a:
        pred = np.clip(base_m + float(lam) * r_m, 0, 100)
        err = y_m - pred
        sse = float(np.dot(err, err))

        row = {
            "lambda": float(lam),
            "sse": sse,
            "mse": sse / len(y_m),
            "n": int(len(y_m)),
        }

        if best is None or row["sse"] < best["sse"]:
            best = row

    return best

def make_pred_global_41a(resid_pred, lam, is_test=False):
    base = pred39c_test_41a if is_test else pred39c_oof_41a
    return np.clip(base + float(lam) * resid_pred, 0, 100).astype(np.float32)

def make_pred_tier_41a(resid_pred, lams, is_test=False):
    base = pred39c_test_41a.copy() if is_test else pred39c_oof_41a.copy()
    masks = tier_masks_test_41a if is_test else tier_masks_oof_41a

    pred = base.astype(np.float64)

    for tier, mask in masks.items():
        if int(mask.sum()) == 0:
            continue
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * resid_pred[mask]

    return np.clip(pred, 0, 100).astype(np.float32)

screen_rows_41a = []

for cfg in configs_41a2:
    name = cfg["name"]
    resid_oof = resid_oof_pred_41a[name]

    if not np.isfinite(resid_oof).all():
        print("Skipping incomplete config:", name)
        continue

    best_global = best_lambda_mask_41a(
        resid_pred=resid_oof,
        mask=np.ones(n_train_41a, dtype=bool),
    )

    pred_global = make_pred_global_41a(resid_oof, best_global["lambda"], is_test=False)
    mse_global = float(mean_squared_error(y_41a, pred_global))

    screen_rows_41a.append({
        "candidate_type": "global_lambda",
        "config": name,
        "lambda_global": best_global["lambda"],
        "lambda_overlap": np.nan,
        "lambda_solver_only": np.nan,
        "lambda_none": np.nan,
        "oof_mse": mse_global,
        "gain_vs_39c": mse39c_41a - mse_global,
    })

    lams = {}
    tier_mses = {}

    for tier, mask in tier_masks_oof_41a.items():
        opt = best_lambda_mask_41a(resid_pred=resid_oof, mask=mask)
        lams[tier] = opt["lambda"]
        tier_mses[tier] = opt["mse"]

    pred_tier = make_pred_tier_41a(resid_oof, lams, is_test=False)
    mse_tier = float(mean_squared_error(y_41a, pred_tier))

    screen_rows_41a.append({
        "candidate_type": "tier_specific_lambda",
        "config": name,
        "lambda_global": np.nan,
        "lambda_overlap": lams.get("overlap", np.nan),
        "lambda_solver_only": lams.get("solver_only", np.nan),
        "lambda_none": lams.get("none", np.nan),
        "oof_mse": mse_tier,
        "gain_vs_39c": mse39c_41a - mse_tier,
        "overlap_mse": tier_mses.get("overlap", np.nan),
        "solver_only_mse": tier_mses.get("solver_only", np.nan),
        "none_mse": tier_mses.get("none", np.nan),
    })

screen41a = pd.DataFrame(screen_rows_41a).sort_values("oof_mse").reset_index(drop=True)

best41a = screen41a.iloc[0]
best_name_41a = best41a["config"]
best_resid_oof_41a = resid_oof_pred_41a[best_name_41a]
best_resid_test_41a = (resid_test_sum_41a[best_name_41a] / N_SPLITS).astype(np.float32)

if best41a["candidate_type"] == "global_lambda":
    final_oof_41a = make_pred_global_41a(
        best_resid_oof_41a,
        float(best41a["lambda_global"]),
        is_test=False,
    )
    final_test_41a = make_pred_global_41a(
        best_resid_test_41a,
        float(best41a["lambda_global"]),
        is_test=True,
    )
else:
    best_lams_41a = {
        "overlap": float(best41a["lambda_overlap"]) if np.isfinite(best41a["lambda_overlap"]) else 0.0,
        "solver_only": float(best41a["lambda_solver_only"]) if np.isfinite(best41a["lambda_solver_only"]) else 0.0,
        "none": float(best41a["lambda_none"]) if np.isfinite(best41a["lambda_none"]) else 0.0,
    }

    final_oof_41a = make_pred_tier_41a(best_resid_oof_41a, best_lams_41a, is_test=False)
    final_test_41a = make_pred_tier_41a(best_resid_test_41a, best_lams_41a, is_test=True)

best_mse_41a = float(mean_squared_error(y_41a, final_oof_41a))
best_gain_41a = mse39c_41a - best_mse_41a

# ------------------------------------------------------------
# 6. Diagnostics and saves
# ------------------------------------------------------------

fold_rows_41a = []

for fold_num, (_, va_idx) in enumerate(folds41a, start=1):
    row = {
        "fold": fold_num,
        "mse_39c": float(mean_squared_error(y_41a[va_idx], pred39c_oof_41a[va_idx])),
        "mse_41a": float(mean_squared_error(y_41a[va_idx], final_oof_41a[va_idx])),
    }
    row["gain_41a_vs_39c"] = row["mse_39c"] - row["mse_41a"]

    for tier, mask_full in tier_masks_oof_41a.items():
        mask_fold = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask_fold.sum())

        if int(mask_fold.sum()) > 0:
            row[f"mse39c_{tier}"] = float(mean_squared_error(
                y_41a[va_idx][mask_fold],
                pred39c_oof_41a[va_idx][mask_fold],
            ))
            row[f"mse41a_{tier}"] = float(mean_squared_error(
                y_41a[va_idx][mask_fold],
                final_oof_41a[va_idx][mask_fold],
            ))
            row[f"gain_{tier}"] = row[f"mse39c_{tier}"] - row[f"mse41a_{tier}"]

    fold_rows_41a.append(row)

fold_gains41a = pd.DataFrame(fold_rows_41a)

tier_perf_rows_41a = []

for tier, mask in tier_masks_oof_41a.items():
    if int(mask.sum()) == 0:
        continue

    tier_perf_rows_41a.append({
        "tier": tier,
        "n_rows": int(mask.sum()),
        "mse_39c": float(mean_squared_error(y_41a[mask], pred39c_oof_41a[mask])),
        "mse_41a": float(mean_squared_error(y_41a[mask], final_oof_41a[mask])),
        "gain_vs_39c": float(mean_squared_error(y_41a[mask], pred39c_oof_41a[mask]) - mean_squared_error(y_41a[mask], final_oof_41a[mask])),
    })

tier_perf41a = pd.DataFrame(tier_perf_rows_41a)

screen_path41a = "model_results/entity41a_screen.csv"
fold_metrics_path41a = "model_results/entity41a_fold_metrics.csv"
fold_gains_path41a = "model_results/entity41a_best_fold_gains.csv"
tier_perf_path41a = "model_results/entity41a_tier_performance.csv"
entity_summary_path41a = "model_results/entity41a_fold_entity_summary.csv"

oof_path41a = "model_results/oof_entity41a_best.csv"
testpred_path41a = "model_results/testpred_entity41a_best.csv"
submission_path41a = "submission_entity41a_best.csv"

screen41a.to_csv(screen_path41a, index=False)
pd.DataFrame(fold_metric_rows_41a).to_csv(fold_metrics_path41a, index=False)
fold_gains41a.to_csv(fold_gains_path41a, index=False)
tier_perf41a.to_csv(tier_perf_path41a, index=False)
pd.DataFrame(entity_summary_rows_41a).to_csv(entity_summary_path41a, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_41a),
    TARGET_COL: y_41a,
    "pred_39c": pred39c_oof_41a,
    "residual_39c": resid39c_41a,
    "residual_pred_41a": best_resid_oof_41a,
    "pred_clipped": final_oof_41a,
    "overlap": overlap_oof_41a.astype(int),
    "solver_only": solver_only_oof_41a.astype(int),
    "none": none_oof_41a.astype(int),
}).to_csv(oof_path41a, index=False)

pd.DataFrame({
    ID_COL: test_ids_41a,
    "pred_39c": pred39c_test_41a,
    "residual_pred_41a": best_resid_test_41a,
    TARGET_COL: final_test_41a,
    "overlap": overlap_test_41a.astype(int),
    "solver_only": solver_only_test_41a.astype(int),
    "none": none_test_41a.astype(int),
}).to_csv(testpred_path41a, index=False)

pd.DataFrame({
    ID_COL: test_ids_41a,
    TARGET_COL: final_test_41a,
}).to_csv(submission_path41a, index=False)

sub41a = pd.read_csv(submission_path41a)
assert sub41a.shape == (n_test_41a, 2)
assert list(sub41a.columns) == [ID_COL, TARGET_COL]
assert sub41a[TARGET_COL].notna().all()
assert np.isfinite(sub41a[TARGET_COL]).all()
assert sub41a[TARGET_COL].between(0, 100).all()

print("\n" + "=" * 90)
print("41A entity-embedding residual model complete")
print("=" * 90)

print("\nBaseline")
print("--------")
print(f"39C OOF MSE: {mse39c_41a:.6f}")

print("\nTop 41A candidates")
print("------------------")
display(screen41a)

print("\nBest 41A candidate")
print("------------------")
print(best41a.to_string())
print(f"\nBest 41A OOF MSE: {best_mse_41a:.6f}")
print(f"Gain vs 39C:       {best_gain_41a:.6f}")

print("\nFold gains")
print("----------")
print(fold_gains41a.to_string(index=False))
print("Min fold gain vs 39C:", float(fold_gains41a["gain_41a_vs_39c"].min()))

print("\nTier performance")
print("----------------")
print(tier_perf41a.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path41a)
print(fold_metrics_path41a)
print(fold_gains_path41a)
print(tier_perf_path41a)
print(entity_summary_path41a)
print(oof_path41a)
print(testpred_path41a)
print(submission_path41a)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path41a)
print("Shape:", sub41a.shape)
print(sub41a[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
if best_gain_41a >= 3.0:
    print("41A clears the 3+ OOF threshold versus 39C. Still compare tier behavior carefully before submitting.")
elif best_gain_41a >= 1.0:
    print("41A has some OOF signal but does not clear the 3+ threshold. Save artifact; do not submit yet.")
elif best_gain_41a > 0:
    print("41A gives only a marginal OOF gain. Do not submit under the current threshold.")
else:
    print("41A does not improve over 39C. Keep 39C as current best.")

41A-2. CPU-safe entity-embedding residual model
Device: cpu
Torch threads: 8

Baseline
--------
39C OOF MSE: 53.226631

Configs
-------
{'name': 'entity41a_solver_focus_core', 'weight_scheme': 'solver_focus', 'epochs': 32, 'hidden_layers': (128, 64), 'dropout': 0.1, 'lr': 0.0007, 'weight_decay': 0.0003}
{'name': 'entity41a_testtier_core', 'weight_scheme': 'test_tier', 'epochs': 32, 'hidden_layers': (128, 64), 'dropout': 0.1, 'lr': 0.0007, 'weight_decay': 0.0003}

41A OUTER FOLD 1/5
Numeric shape: (115936, 20)
Cat shape: (115936, 11)
Total embedding cards: 89959
Top entity unknown rates:
             entity                       cols  cardinality  fit_unique  val_unknown_rate  test_unknown_rate  fold
  school_assessment   SCHOOL × ASSESSMENT_NAME        44637       44636          0.088943           0.089407     1
district_assessment DISTRICT × ASSESSMENT_NAME        18515       18514          0.030499           0.030099     1
    school_subgroup     SCHOOL × SUBGROUP_NAME        21373  

,candidate_type,config,lambda_global,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_39c,overlap_mse,solver_only_mse,none_mse
0,tier_specific_lambda,entity41a_solver_focus_core,NaN,-0.004,0.040,-0.252,52.760006,0.466625,0.595309,65.855698,91.314493
1,global_lambda,entity41a_solver_focus_core,-0.128,NaN,NaN,NaN,53.014221,0.212410,NaN,NaN,NaN
2,tier_specific_lambda,entity41a_testtier_core,NaN,0.000,0.068,-0.208,53.097397,0.129234,0.595326,65.783254,92.118383
3,global_lambda,entity41a_testtier_core,-0.032,NaN,NaN,NaN,53.219456,0.007175,NaN,NaN,NaN



Best 41A candidate
------------------
candidate_type               tier_specific_lambda
config                entity41a_solver_focus_core
lambda_global                                 NaN
lambda_overlap                             -0.004
lambda_solver_only                           0.04
lambda_none                                -0.252
oof_mse                                 52.760006
gain_vs_39c                              0.466625
overlap_mse                              0.595309
solver_only_mse                         65.855698
none_mse                                91.314493

Best 41A OOF MSE: 52.760006
Gain vs 39C:       0.466625

Fold gains
----------
 fold   mse_39c   mse_41a  gain_41a_vs_39c  n_overlap  mse39c_overlap  mse41a_overlap  gain_overlap  n_solver_only  mse39c_solver_only  mse41a_solver_only  gain_solver_only  n_none  mse39c_none  mse41a_none  gain_none
    1 53.559204 52.935390         0.623814      10837        0.488838        0.489095     -0.000257           549

### 41A. Entity-Embedding / Matrix-Factorization Residual Model

This section tested a latent entity representation as a new signal branch after the accounting models. The model used embeddings for school, district, county, region, assessment, subgroup, district type, and selected entity interactions such as `SCHOOL × ASSESSMENT_NAME`, `SCHOOL × SUBGROUP_NAME`, and `DISTRICT × ASSESSMENT_NAME`.

The target was the residual from the current best accounting model:

`residual_39C = observed target - 39C prediction`

The best 41A candidate was `entity41a_solver_focus_core` with tier-specific shrinkage. Its OOF MSE was `52.760006`, compared with the 39C OOF MSE of `53.226631`, for a gain of `0.466625`.

The gain was fold-stable but small. The minimum fold gain versus 39C was `0.258396`. Tier performance showed that the model barely affected the already-solved overlap rows and gave only a tiny improvement on solver-only rows:

- overlap MSE: `0.595326 → 0.595309`
- solver-only MSE: `65.890625 → 65.855698`
- none-tier MSE: `92.366982 → 91.314491`

Conclusion: the entity-embedding branch produced only marginal residual signal, mostly on the `none` tier, which is a small part of the test set. Since the gain is far below the current submission threshold, `submission_entity41a_best.csv` should not be submitted.

In [36]:
# ============================================================
# 42A-1. Integer / rounding accounting audit
# ============================================================
#
# New branch:
#   Latent integer proficient-count reconstruction.
#
# Motivation:
#   A row reports:
#       PERCENT_PROFICIENT
#       N_STUDENTS
#
#   So the hidden object is approximately:
#       k = number of proficient students
#       PERCENT_PROFICIENT ~= 100 * k / N_STUDENTS
#
# 39A/39B/39C used rounded point counts. 42A checks whether
# treating known rows as integer count intervals can improve the
# accounting solver.
#
# This cell:
#   1. Audits row-level count integrality.
#   2. Builds integer intervals for several percent tolerances.
#   3. Checks feasibility of:
#        All Students = Female + Male
#        All Students = Economically Disadvantaged + Not Economically Disadvantaged
#   4. Saves diagnostics.
#
# It does NOT solve test predictions yet.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("42A-1. Integer / rounding accounting audit")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required checks
# ------------------------------------------------------------

required_42a1 = ["raw_train_te", "raw_test_te", "y_train"]
missing_42a1 = [x for x in required_42a1 if x not in globals()]

if missing_42a1:
    raise ValueError(f"Missing required objects: {missing_42a1}. Rerun safe recovery/setup first.")

needed_cols_42a1 = ["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]

for c in needed_cols_42a1:
    if c not in raw_train_te.columns:
        raise ValueError(f"Missing {c} in raw_train_te.")
    if c not in raw_test_te.columns:
        raise ValueError(f"Missing {c} in raw_test_te.")

y_42a1 = np.asarray(y_train, dtype=np.float64).reshape(-1)
n_train_42a1 = len(y_42a1)
n_test_42a1 = len(raw_test_te)

if len(raw_train_te) != n_train_42a1:
    raise ValueError("raw_train_te and y_train row count mismatch.")

# ------------------------------------------------------------
# 2. Compact train/test frames
# ------------------------------------------------------------

def clean_str_42a1(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def safe_num_42a1(s):
    return (
        pd.to_numeric(s, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=np.float64)
    )

train42a1 = pd.DataFrame({
    "row_index": np.arange(n_train_42a1),
    "SCHOOL": clean_str_42a1(raw_train_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_str_42a1(raw_train_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str_42a1(raw_train_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_num_42a1(raw_train_te["N_STUDENTS"]),
    TARGET_COL: y_42a1,
})

test42a1 = pd.DataFrame({
    "row_index": np.arange(n_test_42a1),
    "SCHOOL": clean_str_42a1(raw_test_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_str_42a1(raw_test_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str_42a1(raw_test_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_num_42a1(raw_test_te["N_STUDENTS"]),
})

train42a1["group_key"] = train42a1["SCHOOL"] + "||" + train42a1["ASSESSMENT_NAME"]
test42a1["group_key"] = test42a1["SCHOOL"] + "||" + test42a1["ASSESSMENT_NAME"]

train42a1["N_STUDENTS"] = np.where(
    np.isfinite(train42a1["N_STUDENTS"]) & (train42a1["N_STUDENTS"] > 0),
    train42a1["N_STUDENTS"],
    np.nan,
)

test42a1["N_STUDENTS"] = np.where(
    np.isfinite(test42a1["N_STUDENTS"]) & (test42a1["N_STUDENTS"] > 0),
    test42a1["N_STUDENTS"],
    np.nan,
)

# Rounded point count used by earlier accounting models.
train42a1["k_float"] = (
    np.clip(train42a1[TARGET_COL].to_numpy(dtype=np.float64), 0, 100)
    / 100.0
    * train42a1["N_STUDENTS"].to_numpy(dtype=np.float64)
)

train42a1["k_round"] = np.rint(train42a1["k_float"])
train42a1["k_round"] = np.clip(
    train42a1["k_round"],
    0,
    train42a1["N_STUDENTS"],
)

train42a1["k_round_error_abs"] = np.abs(train42a1["k_float"] - train42a1["k_round"])

print("\nSubgroups")
print("---------")
subgroup_summary42a1 = pd.concat([
    train42a1["SUBGROUP_NAME"].value_counts().rename("train_rows"),
    test42a1["SUBGROUP_NAME"].value_counts().rename("test_rows"),
], axis=1).fillna(0).astype(int)

print(subgroup_summary42a1.to_string())

# ------------------------------------------------------------
# 3. Row-level integer compatibility audit
# ------------------------------------------------------------

integer_tol_grid_42a1 = [
    1e-12,
    1e-9,
    1e-6,
    1e-4,
    1e-3,
    1e-2,
    5e-2,
    1e-1,
    5e-1,
]

integrality_rows_42a1 = []

finite_k = np.isfinite(train42a1["k_float"].to_numpy()) & np.isfinite(train42a1["N_STUDENTS"].to_numpy())

for tol in integer_tol_grid_42a1:
    integrality_rows_42a1.append({
        "count_abs_tolerance": tol,
        "row_share_within_tolerance": float((train42a1.loc[finite_k, "k_round_error_abs"] <= tol).mean()),
        "n_rows_within_tolerance": int((train42a1.loc[finite_k, "k_round_error_abs"] <= tol).sum()),
    })

integrality42a1 = pd.DataFrame(integrality_rows_42a1)

print("\nRow-level integer compatibility")
print("-------------------------------")
print(integrality42a1.to_string(index=False))

print("\nRounded count error summary")
print("---------------------------")
print(train42a1.loc[finite_k, "k_round_error_abs"].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]).to_string())

# ------------------------------------------------------------
# 4. Build interval bounds for a given percent tolerance
# ------------------------------------------------------------

def interval_bounds_from_percent_42a1(percent, n_students, pct_tol):
    """
    Returns integer lower/upper bounds for k satisfying:
        |100*k/n - percent| <= pct_tol

    This generalizes the other thread's ±0.5 percentage interval.
    If pct_tol = 0.5, this corresponds to nearest-integer percent rounding.
    Smaller pct_tol values test whether reported percentages are more precise.
    """
    p = np.asarray(percent, dtype=np.float64)
    n = np.asarray(n_students, dtype=np.float64)

    lower_real = n * (p - float(pct_tol)) / 100.0
    upper_real = n * (p + float(pct_tol)) / 100.0

    lower = np.ceil(lower_real - 1e-12)
    upper = np.floor(upper_real + 1e-12)

    lower = np.where(np.isfinite(lower), lower, np.nan)
    upper = np.where(np.isfinite(upper), upper, np.nan)

    lower = np.maximum(lower, 0)
    upper = np.minimum(upper, n)

    return lower, upper

def interval_width_summary_42a1(df, pct_tol):
    lower, upper = interval_bounds_from_percent_42a1(
        percent=df[TARGET_COL].to_numpy(dtype=np.float64),
        n_students=df["N_STUDENTS"].to_numpy(dtype=np.float64),
        pct_tol=pct_tol,
    )

    valid = np.isfinite(lower) & np.isfinite(upper) & (lower <= upper)
    width = upper - lower + 1

    return {
        "pct_tol": pct_tol,
        "valid_interval_share": float(valid.mean()),
        "median_width": float(np.nanmedian(width[valid])) if valid.any() else np.nan,
        "mean_width": float(np.nanmean(width[valid])) if valid.any() else np.nan,
        "max_width": float(np.nanmax(width[valid])) if valid.any() else np.nan,
        "share_width_1": float((width[valid] == 1).mean()) if valid.any() else np.nan,
        "share_width_le_3": float((width[valid] <= 3).mean()) if valid.any() else np.nan,
    }

pct_tol_grid_42a1 = [
    1e-6,
    1e-4,
    1e-3,
    1e-2,
    5e-2,
    1e-1,
    5e-1,
]

interval_summary42a1 = pd.DataFrame([
    interval_width_summary_42a1(train42a1, pct_tol)
    for pct_tol in pct_tol_grid_42a1
])

print("\nInteger interval width summary")
print("------------------------------")
print(interval_summary42a1.to_string(index=False))

# ------------------------------------------------------------
# 5. Identity feasibility audit under intervals
# ------------------------------------------------------------

candidate_identities_42a1 = [
    ("All Students", "Female", "Male"),
    ("All Students", "Economically Disadvantaged", "Not Economically Disadvantaged"),
]

available_subgroups_42a1 = set(train42a1["SUBGROUP_NAME"].unique()).union(set(test42a1["SUBGROUP_NAME"].unique()))

identities_42a1 = [
    t for t in candidate_identities_42a1
    if all(s in available_subgroups_42a1 for s in t)
]

if len(identities_42a1) == 0:
    raise ValueError("No expected accounting identities found in subgroup names.")

print("\nAccounting identities audited")
print("-----------------------------")
for all_s, a_s, b_s in identities_42a1:
    print(f"{all_s} = {a_s} + {b_s}")

# Pivot helpers.
def build_wide_for_tolerance_42a1(df, pct_tol):
    df = df.copy()

    lower, upper = interval_bounds_from_percent_42a1(
        percent=df[TARGET_COL].to_numpy(dtype=np.float64),
        n_students=df["N_STUDENTS"].to_numpy(dtype=np.float64),
        pct_tol=pct_tol,
    )

    df["k_lower"] = lower
    df["k_upper"] = upper

    wide_n = df.pivot_table(
        index="group_key",
        columns="SUBGROUP_NAME",
        values="N_STUDENTS",
        aggfunc="first",
    )

    wide_kr = df.pivot_table(
        index="group_key",
        columns="SUBGROUP_NAME",
        values="k_round",
        aggfunc="first",
    )

    wide_l = df.pivot_table(
        index="group_key",
        columns="SUBGROUP_NAME",
        values="k_lower",
        aggfunc="first",
    )

    wide_u = df.pivot_table(
        index="group_key",
        columns="SUBGROUP_NAME",
        values="k_upper",
        aggfunc="first",
    )

    return wide_n, wide_kr, wide_l, wide_u

def identity_feasibility_42a1(wide_n, wide_kr, wide_l, wide_u, identity):
    all_s, a_s, b_s = identity

    needed = [all_s, a_s, b_s]

    for s in needed:
        if s not in wide_n.columns:
            return None

    complete = (
        wide_n[all_s].notna()
        & wide_n[a_s].notna()
        & wide_n[b_s].notna()
        & wide_l[all_s].notna()
        & wide_l[a_s].notna()
        & wide_l[b_s].notna()
    )

    if int(complete.sum()) == 0:
        return {
            "complete_groups": 0,
            "n_match_rate": np.nan,
            "rounded_count_identity_rate": np.nan,
            "interval_feasible_rate": np.nan,
            "interval_rescues_rounded_mismatch_rate": np.nan,
            "interval_width_all_median": np.nan,
            "interval_width_a_median": np.nan,
            "interval_width_b_median": np.nan,
        }

    n_all = wide_n.loc[complete, all_s].to_numpy(dtype=np.float64)
    n_a = wide_n.loc[complete, a_s].to_numpy(dtype=np.float64)
    n_b = wide_n.loc[complete, b_s].to_numpy(dtype=np.float64)

    # Student-count consistency. Keep same tolerance style as accounting code.
    n_tol = np.maximum(1.0, 0.02 * n_all)
    n_match = np.abs(n_all - (n_a + n_b)) <= n_tol

    kr_all = wide_kr.loc[complete, all_s].to_numpy(dtype=np.float64)
    kr_a = wide_kr.loc[complete, a_s].to_numpy(dtype=np.float64)
    kr_b = wide_kr.loc[complete, b_s].to_numpy(dtype=np.float64)

    rounded_identity = kr_all == (kr_a + kr_b)

    l_all = wide_l.loc[complete, all_s].to_numpy(dtype=np.float64)
    u_all = wide_u.loc[complete, all_s].to_numpy(dtype=np.float64)

    l_a = wide_l.loc[complete, a_s].to_numpy(dtype=np.float64)
    u_a = wide_u.loc[complete, a_s].to_numpy(dtype=np.float64)

    l_b = wide_l.loc[complete, b_s].to_numpy(dtype=np.float64)
    u_b = wide_u.loc[complete, b_s].to_numpy(dtype=np.float64)

    # There exists k_all with:
    #   k_all in [l_all, u_all]
    #   k_all in [l_a + l_b, u_a + u_b]
    inter_low = np.maximum(l_all, l_a + l_b)
    inter_high = np.minimum(u_all, u_a + u_b)

    interval_feasible = inter_low <= inter_high

    rounded_mismatch = ~rounded_identity
    rescued = interval_feasible & rounded_mismatch

    return {
        "complete_groups": int(complete.sum()),
        "n_match_rate": float(np.mean(n_match)),
        "rounded_count_identity_rate": float(np.mean(rounded_identity)),
        "interval_feasible_rate": float(np.mean(interval_feasible)),
        "interval_rescues_rounded_mismatch_rate": float(rescued.sum() / max(rounded_mismatch.sum(), 1)),
        "n_rounded_mismatches": int(rounded_mismatch.sum()),
        "n_interval_rescues": int(rescued.sum()),
        "interval_width_all_median": float(np.nanmedian(u_all - l_all + 1)),
        "interval_width_a_median": float(np.nanmedian(u_a - l_a + 1)),
        "interval_width_b_median": float(np.nanmedian(u_b - l_b + 1)),
    }

identity_rows_42a1 = []

for pct_tol in pct_tol_grid_42a1:
    wide_n, wide_kr, wide_l, wide_u = build_wide_for_tolerance_42a1(train42a1, pct_tol)

    for identity in identities_42a1:
        stats = identity_feasibility_42a1(wide_n, wide_kr, wide_l, wide_u, identity)

        if stats is None:
            continue

        all_s, a_s, b_s = identity

        row = {
            "pct_tol": pct_tol,
            "identity": f"{all_s} = {a_s} + {b_s}",
            "all_subgroup": all_s,
            "part_a": a_s,
            "part_b": b_s,
        }
        row.update(stats)
        identity_rows_42a1.append(row)

identity_audit42a1 = pd.DataFrame(identity_rows_42a1)

print("\nIdentity interval feasibility audit")
print("-----------------------------------")
print(identity_audit42a1.to_string(index=False))

# ------------------------------------------------------------
# 6. Test group structural coverage
# ------------------------------------------------------------

def test_identity_coverage_42a1(test_df, identity):
    all_s, a_s, b_s = identity

    wide_n_test = test_df.pivot_table(
        index="group_key",
        columns="SUBGROUP_NAME",
        values="N_STUDENTS",
        aggfunc="first",
    )

    for s in [all_s, a_s, b_s]:
        if s not in wide_n_test.columns:
            return None

    complete = (
        wide_n_test[all_s].notna()
        & wide_n_test[a_s].notna()
        & wide_n_test[b_s].notna()
    )

    if int(complete.sum()) == 0:
        return {
            "test_complete_groups": 0,
            "test_n_match_rate": np.nan,
            "test_complete_rows_in_identity_subgroups": 0,
        }

    n_all = wide_n_test.loc[complete, all_s].to_numpy(dtype=np.float64)
    n_a = wide_n_test.loc[complete, a_s].to_numpy(dtype=np.float64)
    n_b = wide_n_test.loc[complete, b_s].to_numpy(dtype=np.float64)

    n_tol = np.maximum(1.0, 0.02 * n_all)
    n_match = np.abs(n_all - (n_a + n_b)) <= n_tol

    complete_group_keys = set(wide_n_test.loc[complete].index.astype(str))

    rows_in_identity = test_df[
        test_df["group_key"].astype(str).isin(complete_group_keys)
        & test_df["SUBGROUP_NAME"].isin([all_s, a_s, b_s])
    ]

    return {
        "test_complete_groups": int(complete.sum()),
        "test_n_match_rate": float(np.mean(n_match)),
        "test_complete_rows_in_identity_subgroups": int(len(rows_in_identity)),
    }

test_rows_42a1 = []

for identity in identities_42a1:
    stats = test_identity_coverage_42a1(test42a1, identity)

    if stats is None:
        continue

    all_s, a_s, b_s = identity

    row = {
        "identity": f"{all_s} = {a_s} + {b_s}",
        "all_subgroup": all_s,
        "part_a": a_s,
        "part_b": b_s,
    }
    row.update(stats)
    test_rows_42a1.append(row)

test_identity_audit42a1 = pd.DataFrame(test_rows_42a1)

print("\nTest identity structural coverage")
print("---------------------------------")
print(test_identity_audit42a1.to_string(index=False))

# ------------------------------------------------------------
# 7. Choose default tolerance for 42A-2
# ------------------------------------------------------------
# Pick the tightest tolerance with high interval validity and high identity feasibility.
# This is only a recommendation, not hard-coded into future cells.

recommendation_rows_42a1 = []

for pct_tol in pct_tol_grid_42a1:
    id_sub = identity_audit42a1[identity_audit42a1["pct_tol"] == pct_tol]

    if len(id_sub) == 0:
        continue

    interval_sub = interval_summary42a1[interval_summary42a1["pct_tol"] == pct_tol].iloc[0]

    recommendation_rows_42a1.append({
        "pct_tol": pct_tol,
        "valid_interval_share": float(interval_sub["valid_interval_share"]),
        "median_width": float(interval_sub["median_width"]),
        "share_width_1": float(interval_sub["share_width_1"]),
        "min_interval_feasible_rate": float(id_sub["interval_feasible_rate"].min()),
        "mean_interval_feasible_rate": float(id_sub["interval_feasible_rate"].mean()),
        "total_interval_rescues": int(id_sub["n_interval_rescues"].sum()),
        "total_rounded_mismatches": int(id_sub["n_rounded_mismatches"].sum()),
    })

tolerance_recommendation42a1 = pd.DataFrame(recommendation_rows_42a1)

print("\nTolerance recommendation table")
print("------------------------------")
print(tolerance_recommendation42a1.to_string(index=False))

# Simple suggested tolerance.
eligible = tolerance_recommendation42a1[
    (tolerance_recommendation42a1["valid_interval_share"] >= 0.99)
    & (tolerance_recommendation42a1["min_interval_feasible_rate"] >= 0.99)
].copy()

if len(eligible) > 0:
    suggested_pct_tol_42a1 = float(eligible.sort_values(["median_width", "pct_tol"]).iloc[0]["pct_tol"])
else:
    suggested_pct_tol_42a1 = 0.5

print("\nSuggested pct_tol for 42A-2:", suggested_pct_tol_42a1)

# ------------------------------------------------------------
# 8. Save artifacts
# ------------------------------------------------------------

integrality_path42a1 = "model_results/int42a1_row_integrality_audit.csv"
interval_summary_path42a1 = "model_results/int42a1_interval_width_summary.csv"
identity_audit_path42a1 = "model_results/int42a1_identity_feasibility.csv"
test_identity_audit_path42a1 = "model_results/int42a1_test_identity_coverage.csv"
tolerance_recommendation_path42a1 = "model_results/int42a1_tolerance_recommendation.csv"
row_counts_path42a1 = "model_results/int42a1_train_row_counts_basic.csv"

integrality42a1.to_csv(integrality_path42a1, index=False)
interval_summary42a1.to_csv(interval_summary_path42a1, index=False)
identity_audit42a1.to_csv(identity_audit_path42a1, index=False)
test_identity_audit42a1.to_csv(test_identity_audit_path42a1, index=False)
tolerance_recommendation42a1.to_csv(tolerance_recommendation_path42a1, index=False)

train42a1[
    [
        "row_index",
        "group_key",
        "SUBGROUP_NAME",
        "N_STUDENTS",
        TARGET_COL,
        "k_float",
        "k_round",
        "k_round_error_abs",
    ]
].to_csv(row_counts_path42a1, index=False)

print("\nSaved files")
print("-----------")
print(integrality_path42a1)
print(interval_summary_path42a1)
print(identity_audit_path42a1)
print(test_identity_audit_path42a1)
print(tolerance_recommendation_path42a1)
print(row_counts_path42a1)

print("\n42A-1 complete.")
print("Next: 42A-2 will implement the exact integer interval accounting solver if this audit shows feasible interval signal.")

42A-1. Integer / rounding accounting audit

Subgroups
---------
                                train_rows  test_rows
SUBGROUP_NAME                                        
All Students                         36711      12428
Male                                 29363       9589
Female                               29110       9777
Economically Disadvantaged           25137       8487
Not Economically Disadvantaged       24600       8026

Row-level integer compatibility
-------------------------------
 count_abs_tolerance  row_share_within_tolerance  n_rows_within_tolerance
        1.000000e-12                    0.179870                    26067
        1.000000e-09                    0.179870                    26067
        1.000000e-06                    0.179870                    26067
        1.000000e-04                    0.179870                    26067
        1.000000e-03                    0.179870                    26067
        1.000000e-02                    0.197176 

In [37]:
# ============================================================
# 42A-2. Integer interval accounting solver
# ============================================================
#
# Uses the 42A-1 audit result:
#   pct_tol = 0.5
#
# Interpretation:
#   A known training row with percentage y and N students gives an
#   integer proficient-count interval:
#
#       ceil(N * (y - 0.5) / 100) <= k <= floor(N * (y + 0.5) / 100)
#
#   Instead of fixing known rows at round(y*N/100), this solver allows
#   known rows to vary within that interval while enforcing:
#
#       All Students = Female + Male
#       All Students = Econ Disadv + Not Econ Disadv
#
# Unknown validation/test rows are guided by a prior prediction.
#
# The solver is tiny per SCHOOL x ASSESSMENT_NAME group:
#   up to five subgroup counts.
#
# It uses integer candidate search, not a large mixed-integer optimizer.
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

PCT_TOL_42A2 = 0.5

print("=" * 90)
print("42A-2. Integer interval accounting solver")
print("=" * 90)
print("pct_tol:", PCT_TOL_42A2)

# ------------------------------------------------------------
# 1. Required checks
# ------------------------------------------------------------

required_42a2 = ["raw_train_te", "raw_test_te", "y_train"]
missing_42a2 = [x for x in required_42a2 if x not in globals()]

if missing_42a2:
    raise ValueError(f"Missing required objects: {missing_42a2}. Rerun safe recovery/setup first.")

needed_cols_42a2 = ["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]
for c in needed_cols_42a2:
    if c not in raw_train_te.columns or c not in raw_test_te.columns:
        raise ValueError(f"Missing required column: {c}")

y_42a2 = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_42a2 = len(y_42a2)
n_test_42a2 = len(raw_test_te)

# ------------------------------------------------------------
# 2. Prediction artifact loaders
# ------------------------------------------------------------

def load_oof_pred_42a2(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_col = None
    for c in ["pred_clipped", "prediction", "pred", "oof_pred"]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF prediction in {path}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_diff > 1e-5:
            raise ValueError(f"Target mismatch in {path}: max diff {max_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col

def load_test_pred_42a2(path):
    df = pd.read_csv(path)

    if len(df) != n_test_42a2:
        raise ValueError(f"Test row mismatch for {path}: got {len(df)}, expected {n_test_42a2}")

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test prediction in {path}")

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} in {path} and no global test_ids.")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col

prior_specs_42a2 = [
    ("account39c", "model_results/oof_account39c_final_best.csv", "model_results/testpred_account39c_final_best.csv"),
    ("account39d", "model_results/oof_account39d_best.csv", "model_results/testpred_account39d_best.csv"),
    ("knn35a", "model_results/oof_knn35a_local_residual_best.csv", "model_results/testpred_knn35a_local_residual_best.csv"),
]

prior_preds_42a2 = {}
test_ids_42a2 = None

for name, oof_path, test_path in prior_specs_42a2:
    if not Path(oof_path).exists() or not Path(test_path).exists():
        print(f"Skipping missing prior: {name}")
        continue

    oof_pred, oof_col = load_oof_pred_42a2(oof_path, y_42a2)
    test_pred, ids, test_col = load_test_pred_42a2(test_path)

    if test_ids_42a2 is None:
        test_ids_42a2 = ids

    prior_preds_42a2[name] = {
        "oof": oof_pred,
        "test": test_pred,
        "oof_mse": float(mean_squared_error(y_42a2, oof_pred)),
        "oof_col": oof_col,
        "test_col": test_col,
    }

    print(f"Loaded prior {name:10s} | OOF MSE {prior_preds_42a2[name]['oof_mse']:.6f}")

if "account39c" not in prior_preds_42a2:
    raise ValueError("account39c prior is required.")

base39c_oof_42a2 = prior_preds_42a2["account39c"]["oof"]
base39c_test_42a2 = prior_preds_42a2["account39c"]["test"]
mse39c_42a2 = prior_preds_42a2["account39c"]["oof_mse"]

# ------------------------------------------------------------
# 3. Compact train/test frames
# ------------------------------------------------------------

def clean_str_42a2(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def safe_num_42a2(s):
    return (
        pd.to_numeric(s, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=np.float64)
    )

train42 = pd.DataFrame({
    "row_index": np.arange(n_train_42a2),
    "SCHOOL": clean_str_42a2(raw_train_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_str_42a2(raw_train_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str_42a2(raw_train_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_num_42a2(raw_train_te["N_STUDENTS"]),
    TARGET_COL: y_42a2,
})

test42 = pd.DataFrame({
    "row_index": np.arange(n_test_42a2),
    "SCHOOL": clean_str_42a2(raw_test_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_str_42a2(raw_test_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str_42a2(raw_test_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_num_42a2(raw_test_te["N_STUDENTS"]),
})

train42["group_key"] = train42["SCHOOL"] + "||" + train42["ASSESSMENT_NAME"]
test42["group_key"] = test42["SCHOOL"] + "||" + test42["ASSESSMENT_NAME"]

train42["N_STUDENTS"] = np.where(
    np.isfinite(train42["N_STUDENTS"]) & (train42["N_STUDENTS"] > 0),
    train42["N_STUDENTS"],
    np.nan,
)

test42["N_STUDENTS"] = np.where(
    np.isfinite(test42["N_STUDENTS"]) & (test42["N_STUDENTS"] > 0),
    test42["N_STUDENTS"],
    np.nan,
)

# ------------------------------------------------------------
# 4. Integer interval helpers
# ------------------------------------------------------------

IDENTITIES_42A2 = [
    ("All Students", "Female", "Male"),
    ("All Students", "Economically Disadvantaged", "Not Economically Disadvantaged"),
]

def n_tolerance_42a2(n):
    if not np.isfinite(n):
        return 1.0
    return max(1.0, 0.02 * float(n))

def interval_bounds_42a2(percent, n_students, pct_tol=PCT_TOL_42A2):
    p = float(percent)
    n = float(n_students)

    if not np.isfinite(p) or not np.isfinite(n) or n <= 0:
        return None

    n_int = int(round(n))

    lo = int(np.ceil(n_int * (p - pct_tol) / 100.0 - 1e-12))
    hi = int(np.floor(n_int * (p + pct_tol) / 100.0 + 1e-12))

    lo = max(0, lo)
    hi = min(n_int, hi)

    if lo > hi:
        return None

    return lo, hi, n_int

def rounded_count_42a2(percent, n_students):
    if not np.isfinite(percent) or not np.isfinite(n_students) or n_students <= 0:
        return np.nan
    n_int = int(round(float(n_students)))
    k = int(round(np.clip(float(percent), 0, 100) / 100.0 * n_int))
    return int(np.clip(k, 0, n_int))

def build_known_lookup_42a2(df, row_indices):
    out = {}

    sub = df.iloc[row_indices]

    for r in sub.itertuples(index=False):
        bounds = interval_bounds_42a2(
            percent=getattr(r, TARGET_COL),
            n_students=r.N_STUDENTS,
            pct_tol=PCT_TOL_42A2,
        )

        if bounds is None:
            continue

        lo, hi, n_int = bounds
        k_round = rounded_count_42a2(getattr(r, TARGET_COL), r.N_STUDENTS)

        group_key = str(r.group_key)
        subgroup = str(r.SUBGROUP_NAME)

        if group_key not in out:
            out[group_key] = {}

        # If duplicates ever occur, keep the tighter intersection.
        if subgroup in out[group_key]:
            old = out[group_key][subgroup]
            lo = max(lo, old["lo"])
            hi = min(hi, old["hi"])
            if lo > hi:
                continue

        out[group_key][subgroup] = {
            "n": n_int,
            "lo": int(lo),
            "hi": int(hi),
            "prior": float(k_round),
            "query": False,
        }

    return out

def best_split_for_sum_42a2(k_sum, a_info, b_info):
    la, ua = a_info["lo"], a_info["hi"]
    lb, ub = b_info["lo"], b_info["hi"]

    low = max(la, k_sum - ub)
    high = min(ua, k_sum - lb)

    if low > high:
        return None

    wa = float(a_info.get("weight", 1.0))
    wb = float(b_info.get("weight", 1.0))
    pa = float(a_info.get("prior", 0.0))
    pb = float(b_info.get("prior", 0.0))

    denom = wa + wb
    if denom <= 0:
        opt = 0.5 * (low + high)
    else:
        opt = (wa * pa + wb * (k_sum - pb)) / denom

    candidates = set()
    for v in [np.floor(opt), np.ceil(opt), round(opt), low, high]:
        vv = int(v)
        vv = max(low, min(high, vv))
        candidates.add(vv)

    best = None

    for ka in candidates:
        kb = int(k_sum - ka)

        if kb < lb or kb > ub:
            continue

        obj = wa * (ka - pa) ** 2 + wb * (kb - pb) ** 2

        if best is None or obj < best["obj"]:
            best = {
                "ka": int(ka),
                "kb": int(kb),
                "obj": float(obj),
            }

    return best

def candidate_values_for_all_42a2(all_info, active_pairs, max_full_enum=4000):
    lo, hi = int(all_info["lo"]), int(all_info["hi"])

    if hi < lo:
        return []

    # Intersect with pair feasible sum ranges.
    for a_info, b_info in active_pairs:
        lo = max(lo, int(a_info["lo"]) + int(b_info["lo"]))
        hi = min(hi, int(a_info["hi"]) + int(b_info["hi"]))

    if hi < lo:
        return []

    width = hi - lo + 1

    if width <= max_full_enum:
        return list(range(lo, hi + 1))

    # Safe fallback if N is unexpectedly huge:
    # prioritize values near prior and near boundaries.
    prior = float(all_info.get("prior", 0.5 * (lo + hi)))
    center = int(round(prior))

    values = set([lo, hi, center])

    for delta in range(-250, 251):
        v = center + delta
        if lo <= v <= hi:
            values.add(v)

    # Add a coarse grid.
    for v in np.linspace(lo, hi, 301):
        values.add(int(round(v)))

    return sorted(values)

def solve_group_integer_42a2(query_group_df, known_group, prior_pct_query):
    """
    Returns:
        pred_pct_local: array length len(query_group_df), NaN unless solved/touched
        touched_local: bool array
        active_equations: number of equations used
    """
    query_group_df = query_group_df.reset_index(drop=True)
    prior_pct_query = np.asarray(prior_pct_query, dtype=np.float64).reshape(-1)

    n_query = len(query_group_df)

    pred_pct = np.full(n_query, np.nan, dtype=np.float32)
    touched = np.zeros(n_query, dtype=bool)

    entries = {}

    # Add known entries.
    if known_group is not None:
        for subgroup, info in known_group.items():
            entries[str(subgroup)] = dict(info)

    # Add query entries. Query overrides known if duplicated.
    local_for_subgroup = {}

    for j, r in query_group_df.iterrows():
        subgroup = str(r["SUBGROUP_NAME"])
        n = float(r["N_STUDENTS"])

        if not np.isfinite(n) or n <= 0:
            continue

        n_int = int(round(n))
        prior_pct = float(prior_pct_query[j])
        prior_k = np.clip(prior_pct, 0, 100) / 100.0 * n_int

        entries[subgroup] = {
            "n": n_int,
            "lo": 0,
            "hi": n_int,
            "prior": float(prior_k),
            "query": True,
            "local_index": int(j),
        }

        local_for_subgroup[subgroup] = int(j)

    active_identities = []

    for all_s, a_s, b_s in IDENTITIES_42A2:
        if all_s not in entries or a_s not in entries or b_s not in entries:
            continue

        n_all = entries[all_s]["n"]
        n_a = entries[a_s]["n"]
        n_b = entries[b_s]["n"]

        if abs(n_all - (n_a + n_b)) > n_tolerance_42a2(n_all):
            continue

        active_identities.append((all_s, a_s, b_s))

    if len(active_identities) == 0:
        return pred_pct, touched, 0

    # There is one shared All Students variable in the observed identities.
    if "All Students" not in entries:
        return pred_pct, touched, 0

    all_info = entries["All Students"]

    active_pairs = [(entries[a_s], entries[b_s]) for _, a_s, b_s in active_identities]

    k_all_candidates = candidate_values_for_all_42a2(all_info, active_pairs)

    if len(k_all_candidates) == 0:
        return pred_pct, touched, 0

    best_solution = None

    for k_all in k_all_candidates:
        obj = float(all_info.get("weight", 1.0)) * (k_all - float(all_info.get("prior", 0.0))) ** 2
        values = {"All Students": int(k_all)}

        feasible = True

        for all_s, a_s, b_s in active_identities:
            split = best_split_for_sum_42a2(
                k_sum=int(k_all),
                a_info=entries[a_s],
                b_info=entries[b_s],
            )

            if split is None:
                feasible = False
                break

            values[a_s] = split["ka"]
            values[b_s] = split["kb"]
            obj += split["obj"]

        if not feasible:
            continue

        if best_solution is None or obj < best_solution["obj"]:
            best_solution = {
                "obj": float(obj),
                "values": values,
            }

    if best_solution is None:
        return pred_pct, touched, 0

    solved_subgroups = set()
    for all_s, a_s, b_s in active_identities:
        solved_subgroups.update([all_s, a_s, b_s])

    for subgroup in solved_subgroups:
        if subgroup not in local_for_subgroup:
            continue

        j = local_for_subgroup[subgroup]
        n = entries[subgroup]["n"]
        k = best_solution["values"].get(subgroup, None)

        if k is None or n <= 0:
            continue

        pred_pct[j] = np.float32(np.clip(100.0 * float(k) / float(n), 0, 100))
        touched[j] = True

    return pred_pct, touched, len(active_identities)

def solve_many_integer_42a2(query_df, known_lookup, prior_pct):
    query_df = query_df.reset_index(drop=True).copy()
    query_df["local_pos"] = np.arange(len(query_df))

    prior_pct = np.asarray(prior_pct, dtype=np.float32).reshape(-1)

    pred = np.full(len(query_df), np.nan, dtype=np.float32)
    touched = np.zeros(len(query_df), dtype=bool)
    n_eq = np.zeros(len(query_df), dtype=np.float32)

    for group_key, g in query_df.groupby("group_key", sort=False):
        loc = g["local_pos"].to_numpy(dtype=np.int64)
        known_group = known_lookup.get(str(group_key), {})

        pred_g, touched_g, eq_count_g = solve_group_integer_42a2(
            query_group_df=g.drop(columns=["local_pos"]).reset_index(drop=True),
            known_group=known_group,
            prior_pct_query=prior_pct[loc],
        )

        pred[loc] = pred_g
        touched[loc] = touched_g
        n_eq[loc] = eq_count_g

    return pred, touched, n_eq

# ------------------------------------------------------------
# 5. OOF solving for candidate priors
# ------------------------------------------------------------

folds42a2 = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_42a2))
)

solver_oof_42a2 = {}
solver_touched_oof_42a2 = {}
fold_diag_rows_42a2 = []

for prior_name, prior_obj in prior_preds_42a2.items():
    print("\n" + "=" * 90)
    print(f"OOF integer solver with prior: {prior_name}")
    print("=" * 90)

    prior_oof = prior_obj["oof"]

    pred_all = np.full(n_train_42a2, np.nan, dtype=np.float32)
    touched_all = np.zeros(n_train_42a2, dtype=bool)

    for fold_num, (tr_idx, va_idx) in enumerate(folds42a2, start=1):
        known_lookup = build_known_lookup_42a2(train42, tr_idx)

        query_fold = train42.iloc[va_idx].reset_index(drop=True)
        prior_fold = prior_oof[va_idx]

        pred_fold, touched_fold, eq_count_fold = solve_many_integer_42a2(
            query_df=query_fold,
            known_lookup=known_lookup,
            prior_pct=prior_fold,
        )

        pred_all[va_idx] = pred_fold
        touched_all[va_idx] = touched_fold

        covered = touched_fold & np.isfinite(pred_fold)

        if int(covered.sum()) > 0:
            solver_mse_cov = float(mean_squared_error(y_42a2[va_idx][covered], pred_fold[covered]))
            base39c_mse_cov = float(mean_squared_error(y_42a2[va_idx][covered], base39c_oof_42a2[va_idx][covered]))
            prior_mse_cov = float(mean_squared_error(y_42a2[va_idx][covered], prior_fold[covered]))
        else:
            solver_mse_cov = np.nan
            base39c_mse_cov = np.nan
            prior_mse_cov = np.nan

        fold_diag_rows_42a2.append({
            "prior": prior_name,
            "fold": fold_num,
            "n_valid": len(va_idx),
            "covered_rows": int(covered.sum()),
            "coverage_rate": float(covered.mean()),
            "base39c_mse_on_covered": base39c_mse_cov,
            "prior_mse_on_covered": prior_mse_cov,
            "solver_mse_on_covered": solver_mse_cov,
        })

        print(
            f"fold {fold_num}: covered {int(covered.sum())}/{len(va_idx)} "
            f"| solver MSE covered {solver_mse_cov:.6f}"
        )

    solver_oof_42a2[prior_name] = pred_all
    solver_touched_oof_42a2[prior_name] = touched_all & np.isfinite(pred_all)

# ------------------------------------------------------------
# 6. Lambda scan against protected 39C anchor
# ------------------------------------------------------------

lambda_grid_42a2 = np.unique(
    np.concatenate([
        np.linspace(-0.50, 1.50, 501),
        np.array([0.0, 0.25, 0.50, 0.65, 0.75, 0.924, 0.992, 1.0]),
    ])
)

def scan_lambda_global_42a2(solver_pred, covered):
    best = None

    for lam in lambda_grid_42a2:
        pred = base39c_oof_42a2.copy()
        pred[covered] = base39c_oof_42a2[covered] + float(lam) * (solver_pred[covered] - base39c_oof_42a2[covered])
        pred = np.clip(pred, 0, 100)

        mse = float(mean_squared_error(y_42a2, pred))

        row = {
            "lambda_global": float(lam),
            "oof_mse": mse,
        }

        if best is None or mse < best["oof_mse"]:
            best = row

    return best

def scan_lambda_tier_42a2(solver_pred, covered):
    # Optimize tiers independently because the masks are disjoint.
    tier_lams = {}
    tier_sse = {}
    pred = base39c_oof_42a2.copy().astype(np.float64)

    for tier, tier_mask in {
        "overlap": None,
        "solver_only": None,
        "none": None,
    }.items():
        pass

    tier_masks = {}

    # If 39A/39B raw tier variables are not in memory, derive from saved raw files.
    raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
    raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
    if "row_index" in raw39a_oof.columns:
        raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
    if "row_index" in raw39b_oof.columns:
        raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

    direct_cov = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
    solver_cov = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

    tier_masks["overlap"] = covered & direct_cov & solver_cov
    tier_masks["solver_only"] = covered & solver_cov & ~direct_cov
    tier_masks["none"] = covered & ~(direct_cov | solver_cov)

    for tier, mask in tier_masks.items():
        if int(mask.sum()) == 0:
            tier_lams[tier] = 0.0
            continue

        best_t = None

        for lam in lambda_grid_42a2:
            pred_t = base39c_oof_42a2[mask] + float(lam) * (solver_pred[mask] - base39c_oof_42a2[mask])
            pred_t = np.clip(pred_t, 0, 100)
            mse_t = float(mean_squared_error(y_42a2[mask], pred_t))
            sse_t = mse_t * int(mask.sum())

            if best_t is None or sse_t < best_t["sse"]:
                best_t = {
                    "lambda": float(lam),
                    "mse": mse_t,
                    "sse": sse_t,
                }

        tier_lams[tier] = best_t["lambda"]
        pred[mask] = base39c_oof_42a2[mask] + tier_lams[tier] * (solver_pred[mask] - base39c_oof_42a2[mask])

    pred = np.clip(pred, 0, 100).astype(np.float32)
    mse = float(mean_squared_error(y_42a2, pred))

    return {
        "lambda_overlap": tier_lams["overlap"],
        "lambda_solver_only": tier_lams["solver_only"],
        "lambda_none": tier_lams["none"],
        "oof_mse": mse,
        "pred": pred,
    }

screen_rows_42a2 = []

for prior_name, solver_pred in solver_oof_42a2.items():
    covered = solver_touched_oof_42a2[prior_name]

    if int(covered.sum()) == 0:
        continue

    best_global = scan_lambda_global_42a2(solver_pred, covered)

    screen_rows_42a2.append({
        "candidate_type": "global_lambda",
        "prior": prior_name,
        "covered_rows": int(covered.sum()),
        "coverage_rate": float(covered.mean()),
        "lambda_global": best_global["lambda_global"],
        "lambda_overlap": np.nan,
        "lambda_solver_only": np.nan,
        "lambda_none": np.nan,
        "oof_mse": best_global["oof_mse"],
        "gain_vs_39c": mse39c_42a2 - best_global["oof_mse"],
    })

    best_tier = scan_lambda_tier_42a2(solver_pred, covered)

    screen_rows_42a2.append({
        "candidate_type": "tier_specific_lambda",
        "prior": prior_name,
        "covered_rows": int(covered.sum()),
        "coverage_rate": float(covered.mean()),
        "lambda_global": np.nan,
        "lambda_overlap": best_tier["lambda_overlap"],
        "lambda_solver_only": best_tier["lambda_solver_only"],
        "lambda_none": best_tier["lambda_none"],
        "oof_mse": best_tier["oof_mse"],
        "gain_vs_39c": mse39c_42a2 - best_tier["oof_mse"],
    })

screen42a2 = pd.DataFrame(screen_rows_42a2).sort_values("oof_mse").reset_index(drop=True)

if len(screen42a2) == 0:
    raise ValueError("No integer solver candidates covered any OOF rows.")

best42a2 = screen42a2.iloc[0]
best_prior42a2 = best42a2["prior"]
best_solver_oof42a2 = solver_oof_42a2[best_prior42a2]
best_covered_oof42a2 = solver_touched_oof_42a2[best_prior42a2]

# Build final OOF according to best row.
if best42a2["candidate_type"] == "global_lambda":
    lam = float(best42a2["lambda_global"])
    final_oof42a2 = base39c_oof_42a2.copy()
    final_oof42a2[best_covered_oof42a2] = (
        base39c_oof_42a2[best_covered_oof42a2]
        + lam * (best_solver_oof42a2[best_covered_oof42a2] - base39c_oof_42a2[best_covered_oof42a2])
    )
    final_oof42a2 = np.clip(final_oof42a2, 0, 100).astype(np.float32)
else:
    raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
    raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
    if "row_index" in raw39a_oof.columns:
        raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
    if "row_index" in raw39b_oof.columns:
        raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

    direct_cov = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
    solver_cov = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

    tier_masks_best = {
        "overlap": best_covered_oof42a2 & direct_cov & solver_cov,
        "solver_only": best_covered_oof42a2 & solver_cov & ~direct_cov,
        "none": best_covered_oof42a2 & ~(direct_cov | solver_cov),
    }

    tier_lams = {
        "overlap": float(best42a2["lambda_overlap"]) if np.isfinite(best42a2["lambda_overlap"]) else 0.0,
        "solver_only": float(best42a2["lambda_solver_only"]) if np.isfinite(best42a2["lambda_solver_only"]) else 0.0,
        "none": float(best42a2["lambda_none"]) if np.isfinite(best42a2["lambda_none"]) else 0.0,
    }

    final_oof42a2 = base39c_oof_42a2.copy().astype(np.float64)

    for tier, mask in tier_masks_best.items():
        final_oof42a2[mask] = (
            base39c_oof_42a2[mask]
            + tier_lams[tier] * (best_solver_oof42a2[mask] - base39c_oof_42a2[mask])
        )

    final_oof42a2 = np.clip(final_oof42a2, 0, 100).astype(np.float32)

best_mse42a2 = float(mean_squared_error(y_42a2, final_oof42a2))
best_gain42a2 = mse39c_42a2 - best_mse42a2

# ------------------------------------------------------------
# 7. Solve test with best prior
# ------------------------------------------------------------

known_lookup_full = build_known_lookup_42a2(train42, np.arange(n_train_42a2))

prior_test_best = prior_preds_42a2[best_prior42a2]["test"]

solver_test42a2, touched_test42a2, eq_count_test42a2 = solve_many_integer_42a2(
    query_df=test42.reset_index(drop=True),
    known_lookup=known_lookup_full,
    prior_pct=prior_test_best,
)

covered_test42a2 = touched_test42a2 & np.isfinite(solver_test42a2)

if best42a2["candidate_type"] == "global_lambda":
    lam = float(best42a2["lambda_global"])
    final_test42a2 = base39c_test_42a2.copy()
    final_test42a2[covered_test42a2] = (
        base39c_test_42a2[covered_test42a2]
        + lam * (solver_test42a2[covered_test42a2] - base39c_test_42a2[covered_test42a2])
    )
    final_test42a2 = np.clip(final_test42a2, 0, 100).astype(np.float32)
else:
    raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
    raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

    direct_cov_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
    solver_cov_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

    tier_masks_test_best = {
        "overlap": covered_test42a2 & direct_cov_test & solver_cov_test,
        "solver_only": covered_test42a2 & solver_cov_test & ~direct_cov_test,
        "none": covered_test42a2 & ~(direct_cov_test | solver_cov_test),
    }

    tier_lams = {
        "overlap": float(best42a2["lambda_overlap"]) if np.isfinite(best42a2["lambda_overlap"]) else 0.0,
        "solver_only": float(best42a2["lambda_solver_only"]) if np.isfinite(best42a2["lambda_solver_only"]) else 0.0,
        "none": float(best42a2["lambda_none"]) if np.isfinite(best42a2["lambda_none"]) else 0.0,
    }

    final_test42a2 = base39c_test_42a2.copy().astype(np.float64)

    for tier, mask in tier_masks_test_best.items():
        final_test42a2[mask] = (
            base39c_test_42a2[mask]
            + tier_lams[tier] * (solver_test42a2[mask] - base39c_test_42a2[mask])
        )

    final_test42a2 = np.clip(final_test42a2, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 8. Fold and tier diagnostics
# ------------------------------------------------------------

fold_gain_rows42a2 = []

for fold_num, (_, va_idx) in enumerate(folds42a2, start=1):
    row = {
        "fold": fold_num,
        "mse_39c": float(mean_squared_error(y_42a2[va_idx], base39c_oof_42a2[va_idx])),
        "mse_42a2": float(mean_squared_error(y_42a2[va_idx], final_oof42a2[va_idx])),
    }
    row["gain_42a2_vs_39c"] = row["mse_39c"] - row["mse_42a2"]
    row["covered_rows"] = int(best_covered_oof42a2[va_idx].sum())
    fold_gain_rows42a2.append(row)

fold_gains42a2 = pd.DataFrame(fold_gain_rows42a2)

# Tier diagnostics.
raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_cov = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_cov = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

tier_masks_oof = {
    "overlap": direct_cov & solver_cov,
    "solver_only": solver_cov & ~direct_cov,
    "none": ~(direct_cov | solver_cov),
}

tier_perf_rows42a2 = []

for tier, mask in tier_masks_oof.items():
    if int(mask.sum()) == 0:
        continue

    tier_perf_rows42a2.append({
        "tier": tier,
        "n_rows": int(mask.sum()),
        "mse_39c": float(mean_squared_error(y_42a2[mask], base39c_oof_42a2[mask])),
        "mse_42a2": float(mean_squared_error(y_42a2[mask], final_oof42a2[mask])),
        "gain_vs_39c": float(mean_squared_error(y_42a2[mask], base39c_oof_42a2[mask]) - mean_squared_error(y_42a2[mask], final_oof42a2[mask])),
        "covered_by_42a2": int(best_covered_oof42a2[mask].sum()),
    })

tier_perf42a2 = pd.DataFrame(tier_perf_rows42a2)

# ------------------------------------------------------------
# 9. Save artifacts
# ------------------------------------------------------------

screen_path42a2 = "model_results/int42a2_solver_screen.csv"
fold_diag_path42a2 = "model_results/int42a2_fold_diag.csv"
fold_gains_path42a2 = "model_results/int42a2_best_fold_gains.csv"
tier_perf_path42a2 = "model_results/int42a2_tier_performance.csv"

raw_oof_path42a2 = "model_results/int42a2_raw_oof_solver.csv"
raw_test_path42a2 = "model_results/int42a2_raw_test_solver.csv"
oof_path42a2 = "model_results/oof_int42a2_best.csv"
testpred_path42a2 = "model_results/testpred_int42a2_best.csv"
submission_path42a2 = "submission_int42a2_best.csv"

screen42a2.to_csv(screen_path42a2, index=False)
pd.DataFrame(fold_diag_rows_42a2).to_csv(fold_diag_path42a2, index=False)
fold_gains42a2.to_csv(fold_gains_path42a2, index=False)
tier_perf42a2.to_csv(tier_perf_path42a2, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_42a2),
    TARGET_COL: y_42a2,
    "pred_39c": base39c_oof_42a2,
    "prior": best_prior42a2,
    "solver_pred": best_solver_oof42a2,
    "solver_covered": best_covered_oof42a2.astype(int),
    "pred_clipped": final_oof42a2,
}).to_csv(oof_path42a2, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_42a2),
    TARGET_COL: y_42a2,
    "pred_39c": base39c_oof_42a2,
    "prior": best_prior42a2,
    "solver_pred": best_solver_oof42a2,
    "solver_covered": best_covered_oof42a2.astype(int),
}).to_csv(raw_oof_path42a2, index=False)

pd.DataFrame({
    ID_COL: test_ids_42a2,
    "pred_39c": base39c_test_42a2,
    "prior": best_prior42a2,
    "solver_pred": solver_test42a2,
    "solver_covered": covered_test42a2.astype(int),
    TARGET_COL: final_test42a2,
}).to_csv(testpred_path42a2, index=False)

pd.DataFrame({
    ID_COL: test_ids_42a2,
    "pred_39c": base39c_test_42a2,
    "prior": best_prior42a2,
    "solver_pred": solver_test42a2,
    "solver_covered": covered_test42a2.astype(int),
}).to_csv(raw_test_path42a2, index=False)

pd.DataFrame({
    ID_COL: test_ids_42a2,
    TARGET_COL: final_test42a2,
}).to_csv(submission_path42a2, index=False)

# Validate submission.
sub42a2 = pd.read_csv(submission_path42a2)
assert sub42a2.shape == (n_test_42a2, 2)
assert list(sub42a2.columns) == [ID_COL, TARGET_COL]
assert sub42a2[ID_COL].notna().all()
assert sub42a2[TARGET_COL].notna().all()
assert np.isfinite(sub42a2[TARGET_COL]).all()
assert sub42a2[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 10. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("42A-2 integer interval accounting solver complete")
print("=" * 90)

print("\nBaseline")
print("--------")
print(f"39C OOF MSE: {mse39c_42a2:.6f}")

print("\nTop 42A-2 candidates")
print("---------------------")
display(screen42a2.head(20))

print("\nBest 42A-2 candidate")
print("--------------------")
print(best42a2.to_string())
print(f"\nBest 42A-2 OOF MSE: {best_mse42a2:.6f}")
print(f"Gain vs 39C:         {best_gain42a2:.6f}")

print("\nFold gains")
print("----------")
print(fold_gains42a2.to_string(index=False))
print("Min fold gain vs 39C:", float(fold_gains42a2["gain_42a2_vs_39c"].min()))

print("\nTier performance")
print("----------------")
print(tier_perf42a2.to_string(index=False))

print("\nTest coverage")
print("-------------")
print("Covered test rows:", int(covered_test42a2.sum()), "of", n_test_42a2, "| rate:", float(covered_test42a2.mean()))

print("\nSaved files")
print("-----------")
print(screen_path42a2)
print(fold_diag_path42a2)
print(fold_gains_path42a2)
print(tier_perf_path42a2)
print(raw_oof_path42a2)
print(raw_test_path42a2)
print(oof_path42a2)
print(testpred_path42a2)
print(submission_path42a2)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path42a2)
print("Shape:", sub42a2.shape)
print(sub42a2[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
if best_gain42a2 >= 3.0:
    print("42A-2 clears the 3+ OOF threshold versus 39C. Consider submission only if tier gains are accounting-covered, not just none-tier.")
elif best_gain42a2 >= 1.0:
    print("42A-2 has some signal but does not clear 3+. Save artifact; do not submit without strong tier/public justification.")
elif best_gain42a2 > 0:
    print("42A-2 gives only a marginal OOF gain. Do not submit under the current threshold.")
else:
    print("42A-2 does not improve over 39C. Keep 39C as current best.")

42A-2. Integer interval accounting solver
pct_tol: 0.5
Loaded prior account39c | OOF MSE 53.226631
Loaded prior account39d | OOF MSE 52.820927
Loaded prior knn35a     | OOF MSE 76.453072

OOF integer solver with prior: account39c
fold 1: covered 16333/28985 | solver MSE covered 24.005636
fold 2: covered 16243/28984 | solver MSE covered 23.823112
fold 3: covered 16362/28984 | solver MSE covered 27.214041
fold 4: covered 16366/28984 | solver MSE covered 23.943897
fold 5: covered 16286/28984 | solver MSE covered 25.223650

OOF integer solver with prior: account39d
fold 1: covered 16333/28985 | solver MSE covered 23.044277
fold 2: covered 16243/28984 | solver MSE covered 22.311846
fold 3: covered 16362/28984 | solver MSE covered 25.334303
fold 4: covered 16366/28984 | solver MSE covered 22.622257
fold 5: covered 16286/28984 | solver MSE covered 24.740620

OOF integer solver with prior: knn35a
fold 1: covered 16333/28985 | solver MSE covered 24.022051
fold 2: covered 16243/28984 | solver MS

,candidate_type,prior,covered_rows,coverage_rate,lambda_global,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_39c
0,tier_specific_lambda,account39d,81590,0.562996,NaN,1.028,0.264,0.0,53.123951,0.102680
1,global_lambda,account39d,81590,0.562996,0.288,NaN,NaN,NaN,53.142159,0.084473
2,tier_specific_lambda,knn35a,81590,0.562996,NaN,0.640,-0.028,0.0,53.154541,0.072090
3,tier_specific_lambda,account39c,81590,0.562996,NaN,1.028,-0.024,0.0,53.191689,0.034943
4,global_lambda,knn35a,81590,0.562996,0.060,NaN,NaN,NaN,53.221767,0.004864
5,global_lambda,account39c,81590,0.562996,0.004,NaN,NaN,NaN,53.226604,0.000027



Best 42A-2 candidate
--------------------
candidate_type        tier_specific_lambda
prior                           account39d
covered_rows                         81590
coverage_rate                     0.562996
lambda_global                          NaN
lambda_overlap                       1.028
lambda_solver_only                   0.264
lambda_none                            0.0
oof_mse                          53.123951
gain_vs_39c                        0.10268

Best 42A-2 OOF MSE: 53.123951
Gain vs 39C:         0.102680

Fold gains
----------
 fold   mse_39c  mse_42a2  gain_42a2_vs_39c  covered_rows
    1 53.559204 53.503445          0.055759         16333
    2 51.060661 50.909088          0.151573         16243
    3 54.383560 54.265594          0.117966         16362
    4 53.151001 53.027317          0.123684         16366
    5 53.978729 53.914284          0.064445         16286
Min fold gain vs 39C: 0.055759429931640625

Tier performance
----------------
       tier  n_ro

### 42A. Integer Interval Accounting Solver

This section tested whether the accounting branch could be improved by treating `PERCENT_PROFICIENT` as arising from an underlying integer count of proficient students. Instead of fixing each known row at `round(PERCENT_PROFICIENT / 100 × N_STUDENTS)`, the method used a feasible integer interval:

`ceil(N × (y - 0.5) / 100) ≤ k ≤ floor(N × (y + 0.5) / 100)`

The 42A-1 audit showed that a tolerance of `0.5` percentage points was appropriate. At this tolerance, all rows had valid count intervals, and the two subgroup identities were nearly perfectly feasible:

`All Students = Female + Male`

`All Students = Economically Disadvantaged + Not Economically Disadvantaged`

The 42A-2 solver used integer candidate search within each `SCHOOL × ASSESSMENT_NAME` group. The best candidate used `account39d` as the prior with tier-specific shrinkage:

- overlap lambda: `1.028`
- solver-only lambda: `0.264`
- none lambda: `0.000`

Results:

- 39C OOF MSE: `53.226631`
- 42A-2 OOF MSE: `53.123951`
- Gain versus 39C: `0.102680`

Tier gains:

- overlap: `0.092306`
- solver-only: `0.356613`
- none: `0.000000`

Conclusion: the integer interval solver is methodologically sound and improves accounting-covered rows slightly, but the gain is far below the current submission threshold. `submission_int42a2_best.csv` should not be submitted unless submission slots are abundant and the goal is only to probe a tiny structural variant.

In [38]:
# ============================================================
# 42B-1. Compact count-scale prior feature setup
# ============================================================
#
# Goal:
#   Build a compact prior feature matrix for a better accounting-solver prior.
#
# This is NOT final prediction modeling yet.
#
# The prior will later be passed into the integer accounting solver.
# ============================================================

import os
import re
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("42B-1. Compact count-scale prior feature setup")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required checks
# ------------------------------------------------------------

required_42b1 = ["raw_train_te", "raw_test_te", "X_train_proc_model", "X_test_proc_model", "y_train"]
missing_42b1 = [x for x in required_42b1 if x not in globals()]

if missing_42b1:
    raise ValueError(f"Missing required objects: {missing_42b1}. Rerun safe recovery/setup first.")

y_42b = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_42b = len(y_42b)
n_test_42b = len(raw_test_te)

if len(raw_train_te) != n_train_42b:
    raise ValueError("raw_train_te and y_train length mismatch.")
if len(raw_test_te) != X_test_proc_model.shape[0]:
    raise ValueError("raw_test_te and X_test_proc_model length mismatch.")

# ------------------------------------------------------------
# 2. Prediction artifact helpers
# ------------------------------------------------------------

def load_oof_pred_42b(path, y_ref):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_col = None
    for c in ["pred_clipped", "prediction", "pred", "oof_pred"]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_diff > 1e-5:
            raise ValueError(f"Target mismatch in {path}: max diff {max_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col

def load_test_pred_42b(path):
    df = pd.read_csv(path)

    if len(df) != n_test_42b:
        raise ValueError(f"Test row mismatch for {path}: got {len(df)}, expected {n_test_42b}")

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} in {path} and no global test_ids.")

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col

artifact_specs_42b = [
    ("anchor33a", "model_results/oof_seg33a_arcsine_adaptive.csv", "model_results/testpred_seg33a_arcsine_adaptive.csv"),
    ("knn35a", "model_results/oof_knn35a_local_residual_best.csv", "model_results/testpred_knn35a_local_residual_best.csv"),
    ("nn37b", "model_results/oof_nn37b38b_cpu_safe_rank1.csv", "model_results/testpred_nn37b38b_cpu_safe_rank1.csv"),
    ("account39a", "model_results/oof_account39a_best.csv", "model_results/testpred_account39a_best.csv"),
    ("account39b", "model_results/oof_account39b_solver_best.csv", "model_results/testpred_account39b_solver_best.csv"),
    ("account39c", "model_results/oof_account39c_final_best.csv", "model_results/testpred_account39c_final_best.csv"),
    ("account39d", "model_results/oof_account39d_best.csv", "model_results/testpred_account39d_best.csv"),
    ("int42a2", "model_results/oof_int42a2_best.csv", "model_results/testpred_int42a2_best.csv"),
]

pred_oof_42b = {}
pred_test_42b = {}
test_ids_42b = None
artifact_rows_42b = []

for name, oof_path, test_path in artifact_specs_42b:
    if not Path(oof_path).exists() or not Path(test_path).exists():
        print(f"Skipping missing artifact: {name}")
        continue

    try:
        p_oof, oof_col = load_oof_pred_42b(oof_path, y_42b)
        p_test, ids, test_col = load_test_pred_42b(test_path)

        if test_ids_42b is None:
            test_ids_42b = ids

        pred_oof_42b[name] = p_oof
        pred_test_42b[name] = p_test

        artifact_rows_42b.append({
            "name": name,
            "oof_mse": float(mean_squared_error(y_42b, p_oof)),
            "oof_col": oof_col,
            "test_col": test_col,
        })

        print(f"Loaded {name:12s} | OOF MSE {mean_squared_error(y_42b, p_oof):.6f}")

    except Exception as e:
        print(f"Skipping {name} due to error: {repr(e)}")

artifact_summary42b = pd.DataFrame(artifact_rows_42b).sort_values("oof_mse").reset_index(drop=True)

if "account39c" not in pred_oof_42b:
    raise ValueError("account39c is required as protected anchor.")

mse39c_42b = float(mean_squared_error(y_42b, pred_oof_42b["account39c"]))

print("\nLoaded artifact summary")
print("-----------------------")
display(artifact_summary42b)

# ------------------------------------------------------------
# 3. Recover accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_cov_oof_42b = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test_42b = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)

solver_cov_oof_42b = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test_42b = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof_42b = direct_cov_oof_42b & solver_cov_oof_42b
solver_only_oof_42b = solver_cov_oof_42b & ~direct_cov_oof_42b
none_oof_42b = ~(direct_cov_oof_42b | solver_cov_oof_42b)

overlap_test_42b = direct_cov_test_42b & solver_cov_test_42b
solver_only_test_42b = solver_cov_test_42b & ~direct_cov_test_42b
none_test_42b = ~(direct_cov_test_42b | solver_cov_test_42b)

tier_masks_oof_42b = {
    "overlap": overlap_oof_42b,
    "solver_only": solver_only_oof_42b,
    "none": none_oof_42b,
}

tier_masks_test_42b = {
    "overlap": overlap_test_42b,
    "solver_only": solver_only_test_42b,
    "none": none_test_42b,
}

print("\nTier coverage")
print("-------------")
for tier in ["overlap", "solver_only", "none"]:
    print(
        f"{tier:12s} train n={int(tier_masks_oof_42b[tier].sum()):6d} "
        f"| test n={int(tier_masks_test_42b[tier].sum()):6d}"
    )

# ------------------------------------------------------------
# 4. Assessment-name parser
# ------------------------------------------------------------

def parse_assessment_features_42b(s):
    s = pd.Series(s).astype("string").fillna("<NA>").astype(str)
    upper = s.str.upper()

    out = pd.DataFrame(index=np.arange(len(s)))

    out["assess_has_math"] = upper.str.contains("MATH|MATHEMATICS|ALGEBRA|GEOMETRY|CALCULUS", regex=True).astype(np.float32)
    out["assess_has_ela"] = upper.str.contains("ELA|ENGLISH|LITERACY|READING|WRITING", regex=True).astype(np.float32)
    out["assess_has_science"] = upper.str.contains("SCIENCE|BIOLOGY|CHEMISTRY|PHYSICS|EARTH", regex=True).astype(np.float32)
    out["assess_has_social"] = upper.str.contains("HISTORY|SOCIAL|GLOBAL|US HISTORY|CIVICS", regex=True).astype(np.float32)
    out["assess_has_regents"] = upper.str.contains("REGENTS", regex=True).astype(np.float32)

    # Extract grade-like tokens. Defensive and simple.
    grade = np.full(len(s), -1, dtype=np.float32)

    patterns = [
        r"GRADE\s*([0-9]{1,2})",
        r"GR\s*([0-9]{1,2})",
        r"\b([0-9]{1,2})(?:ST|ND|RD|TH)?\s*GRADE\b",
    ]

    for pat in patterns:
        extracted = upper.str.extract(pat, expand=False)
        mask = extracted.notna() & (grade < 0)
        grade[mask.to_numpy()] = pd.to_numeric(extracted[mask], errors="coerce").fillna(-1).to_numpy(dtype=np.float32)

    out["assess_grade_num"] = grade
    out["assess_is_elementary"] = ((grade >= 3) & (grade <= 5)).astype(np.float32)
    out["assess_is_middle"] = ((grade >= 6) & (grade <= 8)).astype(np.float32)
    out["assess_is_high"] = ((grade >= 9) | (out["assess_has_regents"].to_numpy(dtype=bool))).astype(np.float32)

    return out

assess_train_42b = parse_assessment_features_42b(raw_train_te["ASSESSMENT_NAME"])
assess_test_42b = parse_assessment_features_42b(raw_test_te["ASSESSMENT_NAME"])

# ------------------------------------------------------------
# 5. Build compact feature matrix
# ------------------------------------------------------------

X_train_42b = pd.DataFrame(index=np.arange(n_train_42b))
X_test_42b = pd.DataFrame(index=np.arange(n_test_42b))

# Saved prediction artifacts and differences.
for name in pred_oof_42b:
    X_train_42b[f"pred_{name}"] = pred_oof_42b[name].astype(np.float32)
    X_test_42b[f"pred_{name}"] = pred_test_42b[name].astype(np.float32)

for name in pred_oof_42b:
    if name == "account39c":
        continue
    X_train_42b[f"diff_{name}_minus_39c"] = (pred_oof_42b[name] - pred_oof_42b["account39c"]).astype(np.float32)
    X_test_42b[f"diff_{name}_minus_39c"] = (pred_test_42b[name] - pred_test_42b["account39c"]).astype(np.float32)

# Tier flags.
X_train_42b["tier_overlap"] = overlap_oof_42b.astype(np.float32)
X_train_42b["tier_solver_only"] = solver_only_oof_42b.astype(np.float32)
X_train_42b["tier_none"] = none_oof_42b.astype(np.float32)

X_test_42b["tier_overlap"] = overlap_test_42b.astype(np.float32)
X_test_42b["tier_solver_only"] = solver_only_test_42b.astype(np.float32)
X_test_42b["tier_none"] = none_test_42b.astype(np.float32)

# Numeric covariates.
numeric_covariates_42b = [
    "N_STUDENTS",
    "PERCENT_FREE_LUNCH",
    "PERCENT_REDUCED_LUNCH",
    "PERCENT_ECONOMICALLY_DISADVANTAGED",
    "PERCENT_ENGLISH_LANGUAGE_LEARNERS",
    "PERCENT_ENGLISH_LANGUAGE_LEANERS",
    "PERCENT_WITH_DISABILITIES",
    "PERCENT_STUDENTS_WITH_DISABILITIES",
    "PERCENT_HOMELESS",
    "PERCENT_MIGRANT",
    "PERCENT_FEMALE",
    "PERCENT_MALE",
    "ATTENDANCE_RATE",
]

added_numeric_42b = []

for c in numeric_covariates_42b:
    if c in X_train_proc_model.columns and c in X_test_proc_model.columns:
        if pd.api.types.is_numeric_dtype(X_train_proc_model[c]):
            if c not in added_numeric_42b:
                X_train_42b[f"num_{c}"] = (
                    pd.to_numeric(X_train_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )
                X_test_42b[f"num_{c}"] = (
                    pd.to_numeric(X_test_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )
                added_numeric_42b.append(c)

if "num_N_STUDENTS" in X_train_42b.columns:
    X_train_42b["num_log1p_N_STUDENTS"] = np.log1p(np.maximum(X_train_42b["num_N_STUDENTS"].to_numpy(dtype=np.float32), 0))
    X_test_42b["num_log1p_N_STUDENTS"] = np.log1p(np.maximum(X_test_42b["num_N_STUDENTS"].to_numpy(dtype=np.float32), 0))

# Assessment parsed features.
X_train_42b = pd.concat([X_train_42b.reset_index(drop=True), assess_train_42b.reset_index(drop=True)], axis=1)
X_test_42b = pd.concat([X_test_42b.reset_index(drop=True), assess_test_42b.reset_index(drop=True)], axis=1)

# Low-cardinality dummies.
dummy_cols_42b = []
for c in ["SUBGROUP_NAME", "ASSESSMENT_NAME", "DISTRICT_TYPE", "COUNTY", "REGION"]:
    if c in raw_train_te.columns and c in raw_test_te.columns:
        nunique = raw_train_te[c].nunique(dropna=False)
        if nunique <= 100:
            dummy_cols_42b.append(c)

if len(dummy_cols_42b) > 0:
    cat_train = raw_train_te[dummy_cols_42b].astype("string").fillna("<NA>").astype(str)
    cat_test = raw_test_te[dummy_cols_42b].astype("string").fillna("<NA>").astype(str)
    cat_all = pd.concat([cat_train, cat_test], axis=0).reset_index(drop=True)

    dummies_all = pd.get_dummies(cat_all, columns=dummy_cols_42b, prefix=dummy_cols_42b, dtype=np.float32)
    dummies_train = dummies_all.iloc[:n_train_42b].reset_index(drop=True)
    dummies_test = dummies_all.iloc[n_train_42b:].reset_index(drop=True)

    X_train_42b = pd.concat([X_train_42b.reset_index(drop=True), dummies_train], axis=1)
    X_test_42b = pd.concat([X_test_42b.reset_index(drop=True), dummies_test], axis=1)

X_train_42b = X_train_42b.loc[:, ~X_train_42b.columns.duplicated()]
X_test_42b = X_test_42b[X_train_42b.columns]

print("\n42B prior feature matrix")
print("------------------------")
print("Train:", X_train_42b.shape)
print("Test: ", X_test_42b.shape)
print("Approx train MB:", round(X_train_42b.memory_usage(deep=True).sum() / 1024**2, 2))
print("Numeric covariates added:", added_numeric_42b)
print("Dummy cols:", dummy_cols_42b)

# ------------------------------------------------------------
# 6. Save setup arrays
# ------------------------------------------------------------

feature_npz_path_42b1 = "model_results/prior42b_features.npz"
feature_cols_path_42b1 = "model_results/prior42b_feature_columns.csv"
artifact_summary_path_42b1 = "model_results/prior42b_loaded_artifacts.csv"

np.savez_compressed(
    feature_npz_path_42b1,
    X_train=X_train_42b.to_numpy(dtype=np.float32),
    X_test=X_test_42b.to_numpy(dtype=np.float32),
    y=y_42b.astype(np.float32),
    pred39c_oof=pred_oof_42b["account39c"].astype(np.float32),
    pred39c_test=pred_test_42b["account39c"].astype(np.float32),
    overlap_oof=overlap_oof_42b.astype(np.int8),
    solver_only_oof=solver_only_oof_42b.astype(np.int8),
    none_oof=none_oof_42b.astype(np.int8),
    overlap_test=overlap_test_42b.astype(np.int8),
    solver_only_test=solver_only_test_42b.astype(np.int8),
    none_test=none_test_42b.astype(np.int8),
    test_ids=np.array(test_ids_42b, dtype=object),
    columns=np.array(X_train_42b.columns, dtype=object),
)

pd.DataFrame({"feature": X_train_42b.columns}).to_csv(feature_cols_path_42b1, index=False)
artifact_summary42b.to_csv(artifact_summary_path_42b1, index=False)

print("\nSaved files")
print("-----------")
print(feature_npz_path_42b1)
print(feature_cols_path_42b1)
print(artifact_summary_path_42b1)

print("\n42B-1 complete.")

42B-1. Compact count-scale prior feature setup
Loaded anchor33a    | OOF MSE 77.049866
Loaded knn35a       | OOF MSE 76.453072
Loaded nn37b        | OOF MSE 76.849442
Loaded account39a   | OOF MSE 55.642780
Loaded account39b   | OOF MSE 53.720547
Loaded account39c   | OOF MSE 53.226631
Loaded account39d   | OOF MSE 52.820927
Loaded int42a2      | OOF MSE 53.123951

Loaded artifact summary
-----------------------


,name,oof_mse,oof_col,test_col
0,account39d,52.820927,pred_clipped,PERCENT_PROFICIENT
1,int42a2,53.123951,pred_clipped,PERCENT_PROFICIENT
2,account39c,53.226631,pred_clipped,PERCENT_PROFICIENT
3,account39b,53.720547,pred_clipped,PERCENT_PROFICIENT
4,account39a,55.642780,pred_clipped,PERCENT_PROFICIENT
5,knn35a,76.453072,pred_clipped,PERCENT_PROFICIENT
6,nn37b,76.849442,pred_clipped,PERCENT_PROFICIENT
7,anchor33a,77.049866,pred_clipped,PERCENT_PROFICIENT



Tier coverage
-------------
overlap      train n= 53786 | test n= 27298
solver_only  train n= 27807 | test n= 17807
none         train n= 63328 | test n=  3202

42B prior feature matrix
------------------------
Train: (144921, 155)
Test:  (48307, 155)
Approx train MB: 85.69
Numeric covariates added: ['N_STUDENTS', 'PERCENT_FREE_LUNCH', 'PERCENT_REDUCED_LUNCH', 'PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_WITH_DISABILITIES', 'PERCENT_HOMELESS', 'PERCENT_MIGRANT', 'PERCENT_FEMALE', 'PERCENT_MALE', 'ATTENDANCE_RATE']
Dummy cols: ['SUBGROUP_NAME', 'ASSESSMENT_NAME', 'DISTRICT_TYPE', 'COUNTY', 'REGION']

Saved files
-----------
model_results/prior42b_features.npz
model_results/prior42b_feature_columns.csv
model_results/prior42b_loaded_artifacts.csv

42B-1 complete.


In [39]:
# ============================================================
# 42B-2. Train compact count-scale / percent prior models
# ============================================================
#
# Uses:
#   model_results/prior42b_features.npz
#
# Trains a few small OOF prior models:
#   - Ridge on percent
#   - HistGradientBoosting on percent
#   - HistGradientBoosting on logit-percent
#
# These priors are NOT final submissions.
# They will be plugged into the integer accounting solver in 42B-3.
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

try:
    from sklearn.ensemble import HistGradientBoostingRegressor
    HAVE_HGB_42B2 = True
except Exception as e:
    HAVE_HGB_42B2 = False
    print("HistGradientBoostingRegressor unavailable:", repr(e))

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("42B-2. Train compact count-scale / percent prior models")
print("=" * 90)
print("Have HGB:", HAVE_HGB_42B2)

feature_npz_path = "model_results/prior42b_features.npz"

if not Path(feature_npz_path).exists():
    raise FileNotFoundError("Run 42B-1 first.")

data42b = np.load(feature_npz_path, allow_pickle=True)

X_train = data42b["X_train"].astype(np.float32)
X_test = data42b["X_test"].astype(np.float32)
y = data42b["y"].astype(np.float32)

pred39c_oof = data42b["pred39c_oof"].astype(np.float32)
pred39c_test = data42b["pred39c_test"].astype(np.float32)

overlap_oof = data42b["overlap_oof"].astype(bool)
solver_only_oof = data42b["solver_only_oof"].astype(bool)
none_oof = data42b["none_oof"].astype(bool)

overlap_test = data42b["overlap_test"].astype(bool)
solver_only_test = data42b["solver_only_test"].astype(bool)
none_test = data42b["none_test"].astype(bool)

test_ids = data42b["test_ids"]
feature_cols = [str(x) for x in data42b["columns"]]

n_train = len(y)
n_test = X_test.shape[0]

mse39c = float(mean_squared_error(y, pred39c_oof))

print("\nLoaded feature block")
print("--------------------")
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print(f"39C OOF MSE: {mse39c:.6f}")

def pct_to_logit_42b2(p):
    p = np.asarray(p, dtype=np.float64)
    q = np.clip(p / 100.0, 1e-4, 1 - 1e-4)
    return np.log(q / (1 - q)).astype(np.float32)

def logit_to_pct_42b2(z):
    z = np.asarray(z, dtype=np.float64)
    q = 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
    return np.clip(100.0 * q, 0, 100).astype(np.float32)

def make_prior_weights_42b2(idx, scheme):
    w = np.ones(len(idx), dtype=np.float32)

    ov = overlap_oof[idx]
    so = solver_only_oof[idx]
    no = none_oof[idx]

    if scheme == "all":
        return w

    if scheme == "solver_public":
        # Public-heavy tier weighting, but keep overlap low because it is already solved.
        w[ov] = 0.15
        w[so] = 2.25
        w[no] = 0.40
        return w

    if scheme == "balanced_unsolved":
        w[ov] = 0.10
        w[so] = 1.50
        w[no] = 1.00
        return w

    raise ValueError(f"Unknown weighting scheme: {scheme}")

configs42b2 = [
    {
        "name": "prior42b_ridge_percent_solver_public",
        "model_type": "ridge_percent",
        "weight_scheme": "solver_public",
        "alpha": 5000.0,
    },
]

if HAVE_HGB_42B2:
    configs42b2.extend([
        {
            "name": "prior42b_hgb_percent_solver_public",
            "model_type": "hgb_percent",
            "weight_scheme": "solver_public",
            "params": {
                "max_iter": 180,
                "learning_rate": 0.04,
                "max_leaf_nodes": 15,
                "max_depth": 4,
                "min_samples_leaf": 120,
                "l2_regularization": 5.0,
                "random_state": RANDOM_STATE,
            },
        },
        {
            "name": "prior42b_hgb_logit_solver_public",
            "model_type": "hgb_logit",
            "weight_scheme": "solver_public",
            "params": {
                "max_iter": 180,
                "learning_rate": 0.04,
                "max_leaf_nodes": 15,
                "max_depth": 4,
                "min_samples_leaf": 120,
                "l2_regularization": 5.0,
                "random_state": RANDOM_STATE + 1,
            },
        },
        {
            "name": "prior42b_hgb_percent_balanced_unsolved",
            "model_type": "hgb_percent",
            "weight_scheme": "balanced_unsolved",
            "params": {
                "max_iter": 160,
                "learning_rate": 0.04,
                "max_leaf_nodes": 15,
                "max_depth": 4,
                "min_samples_leaf": 100,
                "l2_regularization": 4.0,
                "random_state": RANDOM_STATE + 2,
            },
        },
    ])

print("\nPrior model configs")
print("-------------------")
for cfg in configs42b2:
    print(cfg)

folds42b2 = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

prior_oof = {cfg["name"]: np.full(n_train, np.nan, dtype=np.float32) for cfg in configs42b2}
prior_test_sum = {cfg["name"]: np.zeros(n_test, dtype=np.float64) for cfg in configs42b2}
fold_rows = []

for cfg in configs42b2:
    name = cfg["name"]
    print("\n" + "=" * 90)
    print("Training prior:", name)
    print("=" * 90)

    for fold_num, (tr_idx, va_idx) in enumerate(folds42b2, start=1):
        sw = make_prior_weights_42b2(tr_idx, cfg["weight_scheme"])

        if cfg["model_type"] == "ridge_percent":
            imputer = SimpleImputer(strategy="median")
            scaler = StandardScaler()

            X_tr_imp = imputer.fit_transform(X_train[tr_idx])
            X_va_imp = imputer.transform(X_train[va_idx])
            X_te_imp = imputer.transform(X_test)

            X_tr_s = scaler.fit_transform(X_tr_imp)
            X_va_s = scaler.transform(X_va_imp)
            X_te_s = scaler.transform(X_te_imp)

            model = Ridge(alpha=float(cfg["alpha"]))
            model.fit(X_tr_s, y[tr_idx], sample_weight=sw)

            pred_va = model.predict(X_va_s).astype(np.float32)
            pred_te = model.predict(X_te_s).astype(np.float32)

            del X_tr_imp, X_va_imp, X_te_imp, X_tr_s, X_va_s, X_te_s

        elif cfg["model_type"] == "hgb_percent":
            imputer = SimpleImputer(strategy="median")

            X_tr_imp = imputer.fit_transform(X_train[tr_idx])
            X_va_imp = imputer.transform(X_train[va_idx])
            X_te_imp = imputer.transform(X_test)

            model = HistGradientBoostingRegressor(**cfg["params"])
            model.fit(X_tr_imp, y[tr_idx], sample_weight=sw)

            pred_va = model.predict(X_va_imp).astype(np.float32)
            pred_te = model.predict(X_te_imp).astype(np.float32)

            del X_tr_imp, X_va_imp, X_te_imp

        elif cfg["model_type"] == "hgb_logit":
            imputer = SimpleImputer(strategy="median")

            y_logit = pct_to_logit_42b2(y)

            X_tr_imp = imputer.fit_transform(X_train[tr_idx])
            X_va_imp = imputer.transform(X_train[va_idx])
            X_te_imp = imputer.transform(X_test)

            model = HistGradientBoostingRegressor(**cfg["params"])
            model.fit(X_tr_imp, y_logit[tr_idx], sample_weight=sw)

            pred_va = logit_to_pct_42b2(model.predict(X_va_imp))
            pred_te = logit_to_pct_42b2(model.predict(X_te_imp))

            del X_tr_imp, X_va_imp, X_te_imp

        else:
            raise ValueError(f"Unknown model_type: {cfg['model_type']}")

        pred_va = np.clip(pred_va, 0, 100).astype(np.float32)
        pred_te = np.clip(pred_te, 0, 100).astype(np.float32)

        prior_oof[name][va_idx] = pred_va
        prior_test_sum[name] += pred_te.astype(np.float64)

        row = {
            "config": name,
            "fold": fold_num,
            "mse_prior": float(mean_squared_error(y[va_idx], pred_va)),
            "mse_39c": float(mean_squared_error(y[va_idx], pred39c_oof[va_idx])),
        }
        row["gain_prior_vs_39c"] = row["mse_39c"] - row["mse_prior"]

        for tier, mask_all in {
            "overlap": overlap_oof,
            "solver_only": solver_only_oof,
            "none": none_oof,
        }.items():
            m = mask_all[va_idx]
            row[f"n_{tier}"] = int(m.sum())
            if int(m.sum()) > 0:
                row[f"mse_prior_{tier}"] = float(mean_squared_error(y[va_idx][m], pred_va[m]))
                row[f"mse_39c_{tier}"] = float(mean_squared_error(y[va_idx][m], pred39c_oof[va_idx][m]))

        fold_rows.append(row)

        print(
            f"fold {fold_num} | prior MSE {row['mse_prior']:.6f} "
            f"| 39C MSE {row['mse_39c']:.6f} | gain {row['gain_prior_vs_39c']:.6f}"
        )

        del model
        gc.collect()

# Save priors.
prior_screen_rows = []
for cfg in configs42b2:
    name = cfg["name"]
    pred_oof = prior_oof[name]
    pred_test = (prior_test_sum[name] / N_SPLITS).astype(np.float32)

    prior_screen_rows.append({
        "config": name,
        "prior_oof_mse": float(mean_squared_error(y, pred_oof)),
        "gain_vs_39c": mse39c - float(mean_squared_error(y, pred_oof)),
        "overlap_mse": float(mean_squared_error(y[overlap_oof], pred_oof[overlap_oof])),
        "solver_only_mse": float(mean_squared_error(y[solver_only_oof], pred_oof[solver_only_oof])),
        "none_mse": float(mean_squared_error(y[none_oof], pred_oof[none_oof])),
    })

    pd.DataFrame({
        "row_index": np.arange(n_train),
        TARGET_COL: y,
        "pred_clipped": pred_oof,
    }).to_csv(f"model_results/oof_{name}.csv", index=False)

    pd.DataFrame({
        ID_COL: test_ids,
        TARGET_COL: pred_test,
    }).to_csv(f"model_results/testpred_{name}.csv", index=False)

prior_screen42b2 = pd.DataFrame(prior_screen_rows).sort_values("prior_oof_mse").reset_index(drop=True)
fold_metrics42b2 = pd.DataFrame(fold_rows)

prior_screen_path = "model_results/prior42b_model_screen.csv"
fold_metrics_path = "model_results/prior42b_fold_metrics.csv"

prior_screen42b2.to_csv(prior_screen_path, index=False)
fold_metrics42b2.to_csv(fold_metrics_path, index=False)

print("\n" + "=" * 90)
print("42B-2 prior training complete")
print("=" * 90)

print("\nPrior model screen")
print("------------------")
display(prior_screen42b2)

print("\nSaved files")
print("-----------")
print(prior_screen_path)
print(fold_metrics_path)
for cfg in configs42b2:
    print(f"model_results/oof_{cfg['name']}.csv")
    print(f"model_results/testpred_{cfg['name']}.csv")

print("\n42B-2 complete.")

42B-2. Train compact count-scale / percent prior models
Have HGB: True

Loaded feature block
--------------------
X_train: (144921, 155)
X_test:  (48307, 155)
39C OOF MSE: 53.226631

Prior model configs
-------------------
{'name': 'prior42b_ridge_percent_solver_public', 'model_type': 'ridge_percent', 'weight_scheme': 'solver_public', 'alpha': 5000.0}
{'name': 'prior42b_hgb_percent_solver_public', 'model_type': 'hgb_percent', 'weight_scheme': 'solver_public', 'params': {'max_iter': 180, 'learning_rate': 0.04, 'max_leaf_nodes': 15, 'max_depth': 4, 'min_samples_leaf': 120, 'l2_regularization': 5.0, 'random_state': 9890}}
{'name': 'prior42b_hgb_logit_solver_public', 'model_type': 'hgb_logit', 'weight_scheme': 'solver_public', 'params': {'max_iter': 180, 'learning_rate': 0.04, 'max_leaf_nodes': 15, 'max_depth': 4, 'min_samples_leaf': 120, 'l2_regularization': 5.0, 'random_state': 9891}}
{'name': 'prior42b_hgb_percent_balanced_unsolved', 'model_type': 'hgb_percent', 'weight_scheme': 'balanc

,config,prior_oof_mse,gain_vs_39c,overlap_mse,solver_only_mse,none_mse
0,prior42b_hgb_percent_balanced_unsolved,52.920437,0.306194,1.065013,64.092369,92.056969
1,prior42b_ridge_percent_solver_public,52.992805,0.233826,0.812928,64.205910,92.386826
2,prior42b_hgb_percent_solver_public,53.030693,0.195938,1.079570,64.110359,92.289009
3,prior42b_hgb_logit_solver_public,59.901684,-6.675053,5.375255,72.223740,100.801727



Saved files
-----------
model_results/prior42b_model_screen.csv
model_results/prior42b_fold_metrics.csv
model_results/oof_prior42b_ridge_percent_solver_public.csv
model_results/testpred_prior42b_ridge_percent_solver_public.csv
model_results/oof_prior42b_hgb_percent_solver_public.csv
model_results/testpred_prior42b_hgb_percent_solver_public.csv
model_results/oof_prior42b_hgb_logit_solver_public.csv
model_results/testpred_prior42b_hgb_logit_solver_public.csv
model_results/oof_prior42b_hgb_percent_balanced_unsolved.csv
model_results/testpred_prior42b_hgb_percent_balanced_unsolved.csv

42B-2 complete.


In [40]:
# ============================================================
# 42B-3. Plug learned priors into integer accounting solver
# ============================================================
#
# Uses:
#   learned priors from 42B-2
#   existing priors 39C / 39D / 42A
#
# Final candidate:
#   39C anchor + tier-specific shrinkage toward integer-solver prediction.
#
# Submit only if printed decision rule says submission-worthy.
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")
PCT_TOL_42B3 = 0.5

print("=" * 90)
print("42B-3. Integer accounting solver with learned priors")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required checks / frames
# ------------------------------------------------------------

required_42b3 = ["raw_train_te", "raw_test_te", "y_train"]
missing_42b3 = [x for x in required_42b3 if x not in globals()]
if missing_42b3:
    raise ValueError(f"Missing required objects: {missing_42b3}")

y = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train = len(y)
n_test = len(raw_test_te)

def clean_str(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def safe_num(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)

train_df = pd.DataFrame({
    "row_index": np.arange(n_train),
    "SCHOOL": clean_str(raw_train_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_str(raw_train_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str(raw_train_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_num(raw_train_te["N_STUDENTS"]),
    TARGET_COL: y,
})
test_df = pd.DataFrame({
    "row_index": np.arange(n_test),
    "SCHOOL": clean_str(raw_test_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_str(raw_test_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str(raw_test_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_num(raw_test_te["N_STUDENTS"]),
})

train_df["group_key"] = train_df["SCHOOL"] + "||" + train_df["ASSESSMENT_NAME"]
test_df["group_key"] = test_df["SCHOOL"] + "||" + test_df["ASSESSMENT_NAME"]

train_df["N_STUDENTS"] = np.where(np.isfinite(train_df["N_STUDENTS"]) & (train_df["N_STUDENTS"] > 0), train_df["N_STUDENTS"], np.nan)
test_df["N_STUDENTS"] = np.where(np.isfinite(test_df["N_STUDENTS"]) & (test_df["N_STUDENTS"] > 0), test_df["N_STUDENTS"], np.nan)

# ------------------------------------------------------------
# 2. Load priors
# ------------------------------------------------------------

def load_oof(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)
    if "pred_clipped" in df.columns:
        p = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        p = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(f"No prediction column in {path}")
    return np.clip(p, 0, 100).astype(np.float32)

def load_test(path):
    df = pd.read_csv(path)
    if TARGET_COL in df.columns:
        p = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [c for c in df.columns if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])]
        if not numeric_cols:
            raise ValueError(f"No test prediction column in {path}")
        p = df[numeric_cols[0]].to_numpy(dtype=np.float32)
    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs.")
    return np.clip(p, 0, 100).astype(np.float32), ids

prior_specs = [
    ("account39c", "model_results/oof_account39c_final_best.csv", "model_results/testpred_account39c_final_best.csv"),
    ("account39d", "model_results/oof_account39d_best.csv", "model_results/testpred_account39d_best.csv"),
    ("int42a2", "model_results/oof_int42a2_best.csv", "model_results/testpred_int42a2_best.csv"),
]

# Add learned priors if present.
if Path("model_results/prior42b_model_screen.csv").exists():
    prior_screen = pd.read_csv("model_results/prior42b_model_screen.csv")
    for name in prior_screen["config"].tolist():
        oof_path = f"model_results/oof_{name}.csv"
        test_path = f"model_results/testpred_{name}.csv"
        if Path(oof_path).exists() and Path(test_path).exists():
            prior_specs.append((name, oof_path, test_path))

prior_oof = {}
prior_test = {}
test_ids = None

for name, oof_path, test_path in prior_specs:
    if not Path(oof_path).exists() or not Path(test_path).exists():
        continue
    p_oof = load_oof(oof_path)
    p_test, ids = load_test(test_path)

    if len(p_oof) != n_train or len(p_test) != n_test:
        print(f"Skipping {name}: length mismatch")
        continue

    if test_ids is None:
        test_ids = ids

    prior_oof[name] = p_oof
    prior_test[name] = p_test
    print(f"Loaded prior {name:40s} | OOF MSE {mean_squared_error(y, p_oof):.6f}")

if "account39c" not in prior_oof:
    raise ValueError("account39c prior required.")

base39c_oof = prior_oof["account39c"]
base39c_test = prior_test["account39c"]
mse39c = float(mean_squared_error(y, base39c_oof))

# ------------------------------------------------------------
# 3. Tier flags and weighted metric
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_cov_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_cov_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

tier_masks_oof = {
    "overlap": direct_cov_oof & solver_cov_oof,
    "solver_only": solver_cov_oof & ~direct_cov_oof,
    "none": ~(direct_cov_oof | solver_cov_oof),
}
tier_masks_test = {
    "overlap": direct_cov_test & solver_cov_test,
    "solver_only": solver_cov_test & ~direct_cov_test,
    "none": ~(direct_cov_test | solver_cov_test),
}
test_tier_rates = {k: float(v.mean()) for k, v in tier_masks_test.items()}

def tier_weighted_mse(pred):
    pred = np.clip(pred, 0, 100)
    total = 0.0
    for tier, mask in tier_masks_oof.items():
        total += test_tier_rates[tier] * float(mean_squared_error(y[mask], pred[mask]))
    return total

weighted39c = tier_weighted_mse(base39c_oof)

# ------------------------------------------------------------
# 4. Integer solver helpers
# ------------------------------------------------------------

IDENTITIES = [
    ("All Students", "Female", "Male"),
    ("All Students", "Economically Disadvantaged", "Not Economically Disadvantaged"),
]

def n_tol(n):
    return max(1.0, 0.02 * float(n)) if np.isfinite(n) else 1.0

def interval_bounds(percent, n_students, pct_tol=PCT_TOL_42B3):
    if not np.isfinite(percent) or not np.isfinite(n_students) or n_students <= 0:
        return None
    n_int = int(round(float(n_students)))
    lo = int(np.ceil(n_int * (float(percent) - pct_tol) / 100.0 - 1e-12))
    hi = int(np.floor(n_int * (float(percent) + pct_tol) / 100.0 + 1e-12))
    lo = max(0, lo)
    hi = min(n_int, hi)
    if lo > hi:
        return None
    return lo, hi, n_int

def round_count(percent, n_students):
    if not np.isfinite(percent) or not np.isfinite(n_students) or n_students <= 0:
        return np.nan
    n_int = int(round(float(n_students)))
    return int(np.clip(round(float(percent) / 100.0 * n_int), 0, n_int))

def build_known_lookup(df, idx):
    out = {}
    sub = df.iloc[idx]
    for r in sub.itertuples(index=False):
        bounds = interval_bounds(getattr(r, TARGET_COL), r.N_STUDENTS)
        if bounds is None:
            continue
        lo, hi, n_int = bounds
        prior = round_count(getattr(r, TARGET_COL), r.N_STUDENTS)
        g = str(r.group_key)
        s = str(r.SUBGROUP_NAME)
        out.setdefault(g, {})
        out[g][s] = {"n": n_int, "lo": lo, "hi": hi, "prior": float(prior), "weight": 1.0, "query": False}
    return out

def best_split(k_sum, a, b):
    la, ua = a["lo"], a["hi"]
    lb, ub = b["lo"], b["hi"]
    low = max(la, k_sum - ub)
    high = min(ua, k_sum - lb)
    if low > high:
        return None
    wa, wb = float(a.get("weight", 1.0)), float(b.get("weight", 1.0))
    pa, pb = float(a.get("prior", 0.0)), float(b.get("prior", 0.0))
    opt = (wa * pa + wb * (k_sum - pb)) / max(wa + wb, 1e-12)
    candidates = set()
    for v in [low, high, np.floor(opt), np.ceil(opt), round(opt)]:
        vv = int(max(low, min(high, int(v))))
        candidates.add(vv)
    best = None
    for ka in candidates:
        kb = int(k_sum - ka)
        if lb <= kb <= ub:
            obj = wa * (ka - pa) ** 2 + wb * (kb - pb) ** 2
            if best is None or obj < best["obj"]:
                best = {"ka": ka, "kb": kb, "obj": obj}
    return best

def candidate_k_all_values(all_info, active_pairs, max_enum=4000):
    lo, hi = all_info["lo"], all_info["hi"]
    for a, b in active_pairs:
        lo = max(lo, a["lo"] + b["lo"])
        hi = min(hi, a["hi"] + b["hi"])
    if hi < lo:
        return []
    width = hi - lo + 1
    if width <= max_enum:
        return range(lo, hi + 1)
    center = int(round(all_info.get("prior", 0.5 * (lo + hi))))
    vals = set([lo, hi, center])
    for d in range(-250, 251):
        v = center + d
        if lo <= v <= hi:
            vals.add(v)
    for v in np.linspace(lo, hi, 301):
        vals.add(int(round(v)))
    return sorted(vals)

def solve_group(query_group, known_group, prior_pct):
    query_group = query_group.reset_index(drop=True)
    prior_pct = np.asarray(prior_pct, dtype=np.float64)

    n = len(query_group)
    pred = np.full(n, np.nan, dtype=np.float32)
    touched = np.zeros(n, dtype=bool)

    entries = {}
    if known_group:
        for s, info in known_group.items():
            entries[str(s)] = dict(info)

    local = {}
    for j, r in query_group.iterrows():
        s = str(r["SUBGROUP_NAME"])
        n_students = float(r["N_STUDENTS"])
        if not np.isfinite(n_students) or n_students <= 0:
            continue
        n_int = int(round(n_students))
        prior_k = np.clip(float(prior_pct[j]), 0, 100) / 100.0 * n_int
        entries[s] = {"n": n_int, "lo": 0, "hi": n_int, "prior": prior_k, "weight": 1.0, "query": True, "local": j}
        local[s] = j

    active = []
    for all_s, a_s, b_s in IDENTITIES:
        if all_s in entries and a_s in entries and b_s in entries:
            if abs(entries[all_s]["n"] - (entries[a_s]["n"] + entries[b_s]["n"])) <= n_tol(entries[all_s]["n"]):
                active.append((all_s, a_s, b_s))

    if not active or "All Students" not in entries:
        return pred, touched, 0

    active_pairs = [(entries[a], entries[b]) for _, a, b in active]
    candidates = candidate_k_all_values(entries["All Students"], active_pairs)

    if not candidates:
        return pred, touched, 0

    best = None
    for k_all in candidates:
        values = {"All Students": int(k_all)}
        obj = (k_all - float(entries["All Students"].get("prior", 0.0))) ** 2
        ok = True
        for _, a_s, b_s in active:
            split = best_split(int(k_all), entries[a_s], entries[b_s])
            if split is None:
                ok = False
                break
            values[a_s] = split["ka"]
            values[b_s] = split["kb"]
            obj += split["obj"]
        if ok and (best is None or obj < best["obj"]):
            best = {"obj": obj, "values": values}

    if best is None:
        return pred, touched, 0

    solved = set()
    for all_s, a_s, b_s in active:
        solved.update([all_s, a_s, b_s])

    for s in solved:
        if s in local and s in best["values"]:
            j = local[s]
            n_int = entries[s]["n"]
            pred[j] = np.float32(np.clip(100.0 * best["values"][s] / n_int, 0, 100))
            touched[j] = True

    return pred, touched, len(active)

def solve_many(query_df, known_lookup, prior_pct):
    query_df = query_df.reset_index(drop=True).copy()
    query_df["local_pos"] = np.arange(len(query_df))
    prior_pct = np.asarray(prior_pct, dtype=np.float32)

    pred = np.full(len(query_df), np.nan, dtype=np.float32)
    touched = np.zeros(len(query_df), dtype=bool)

    for gkey, g in query_df.groupby("group_key", sort=False):
        loc = g["local_pos"].to_numpy(dtype=int)
        pred_g, touched_g, _ = solve_group(
            query_group=g.drop(columns=["local_pos"]),
            known_group=known_lookup.get(str(gkey), {}),
            prior_pct=prior_pct[loc],
        )
        pred[loc] = pred_g
        touched[loc] = touched_g

    return pred, touched

# ------------------------------------------------------------
# 5. OOF solve every prior
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))
solver_oof = {}
solver_cov = {}
fold_diag = []

for prior_name, p_oof in prior_oof.items():
    print("\n" + "=" * 90)
    print("Solving integer accounting with prior:", prior_name)
    print("=" * 90)

    out = np.full(n_train, np.nan, dtype=np.float32)
    cov = np.zeros(n_train, dtype=bool)

    for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
        known = build_known_lookup(train_df, tr_idx)
        pred_fold, touched_fold = solve_many(
            query_df=train_df.iloc[va_idx],
            known_lookup=known,
            prior_pct=p_oof[va_idx],
        )
        out[va_idx] = pred_fold
        cov[va_idx] = touched_fold & np.isfinite(pred_fold)

        mse_cov = float(mean_squared_error(y[va_idx][cov[va_idx]], pred_fold[cov[va_idx]])) if cov[va_idx].sum() else np.nan
        print(f"fold {fold_num}: covered {int(cov[va_idx].sum())}/{len(va_idx)} | solver covered MSE {mse_cov:.6f}")

        fold_diag.append({
            "prior": prior_name,
            "fold": fold_num,
            "covered_rows": int(cov[va_idx].sum()),
            "coverage_rate": float(cov[va_idx].mean()),
            "solver_mse_on_covered": mse_cov,
        })

    solver_oof[prior_name] = out
    solver_cov[prior_name] = cov

# ------------------------------------------------------------
# 6. Lambda scans
# ------------------------------------------------------------

lambda_grid = np.unique(np.concatenate([
    np.linspace(-0.50, 1.50, 501),
    np.array([0.0, 0.25, 0.50, 0.65, 0.75, 0.924, 0.992, 1.0]),
]))

def scan_tier_lambdas(solver_pred, covered):
    pred = base39c_oof.copy().astype(np.float64)
    rows = {}

    tier_masks = {
        "overlap": covered & tier_masks_oof["overlap"],
        "solver_only": covered & tier_masks_oof["solver_only"],
        "none": covered & tier_masks_oof["none"],
    }

    lams = {}
    tier_mses = {}

    for tier, mask in tier_masks.items():
        if mask.sum() == 0:
            lams[tier] = 0.0
            tier_mses[tier] = np.nan
            continue

        best = None
        for lam in lambda_grid:
            p = np.clip(base39c_oof[mask] + float(lam) * (solver_pred[mask] - base39c_oof[mask]), 0, 100)
            mse = float(mean_squared_error(y[mask], p))
            if best is None or mse < best["mse"]:
                best = {"lambda": float(lam), "mse": mse}

        lams[tier] = best["lambda"]
        tier_mses[tier] = best["mse"]
        pred[mask] = base39c_oof[mask] + lams[tier] * (solver_pred[mask] - base39c_oof[mask])

    pred = np.clip(pred, 0, 100).astype(np.float32)
    return pred, lams, tier_mses

screen_rows = []

for prior_name in solver_oof:
    pred, lams, tier_mses = scan_tier_lambdas(solver_oof[prior_name], solver_cov[prior_name])
    mse = float(mean_squared_error(y, pred))
    weighted = tier_weighted_mse(pred)

    screen_rows.append({
        "prior": prior_name,
        "covered_rows": int(solver_cov[prior_name].sum()),
        "coverage_rate": float(solver_cov[prior_name].mean()),
        "lambda_overlap": lams["overlap"],
        "lambda_solver_only": lams["solver_only"],
        "lambda_none": lams["none"],
        "oof_mse": mse,
        "gain_vs_39c": mse39c - mse,
        "test_tier_weighted_mse": weighted,
        "weighted_gain_vs_39c": weighted39c - weighted,
        "overlap_mse": tier_mses["overlap"],
        "solver_only_mse": tier_mses["solver_only"],
        "none_mse": tier_mses["none"],
    })

screen42b3 = pd.DataFrame(screen_rows).sort_values(["test_tier_weighted_mse", "oof_mse"]).reset_index(drop=True)

best = screen42b3.iloc[0]
best_prior = best["prior"]
best_solver_oof = solver_oof[best_prior]
best_cov_oof = solver_cov[best_prior]

final_oof = base39c_oof.copy().astype(np.float64)
best_lams = {
    "overlap": float(best["lambda_overlap"]),
    "solver_only": float(best["lambda_solver_only"]),
    "none": float(best["lambda_none"]),
}
for tier, mask0 in tier_masks_oof.items():
    mask = best_cov_oof & mask0
    final_oof[mask] = base39c_oof[mask] + best_lams[tier] * (best_solver_oof[mask] - base39c_oof[mask])
final_oof = np.clip(final_oof, 0, 100).astype(np.float32)

best_mse = float(mean_squared_error(y, final_oof))
best_weighted = tier_weighted_mse(final_oof)

# ------------------------------------------------------------
# 7. Solve test with best prior
# ------------------------------------------------------------

known_full = build_known_lookup(train_df, np.arange(n_train))
solver_test, cov_test = solve_many(
    query_df=test_df,
    known_lookup=known_full,
    prior_pct=prior_test[best_prior],
)

final_test = base39c_test.copy().astype(np.float64)

for tier, mask0 in tier_masks_test.items():
    mask = cov_test & mask0
    final_test[mask] = base39c_test[mask] + best_lams[tier] * (solver_test[mask] - base39c_test[mask])

final_test = np.clip(final_test, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 8. Diagnostics and save
# ------------------------------------------------------------

tier_rows = []
for tier, mask in tier_masks_oof.items():
    tier_rows.append({
        "tier": tier,
        "n_rows": int(mask.sum()),
        "mse_39c": float(mean_squared_error(y[mask], base39c_oof[mask])),
        "mse_42b": float(mean_squared_error(y[mask], final_oof[mask])),
        "gain_vs_39c": float(mean_squared_error(y[mask], base39c_oof[mask]) - mean_squared_error(y[mask], final_oof[mask])),
        "covered_by_solver": int((best_cov_oof & mask).sum()),
        "test_rows": int(tier_masks_test[tier].sum()),
    })
tier_perf42b3 = pd.DataFrame(tier_rows)

fold_rows = []
for fold_num, (_, va_idx) in enumerate(folds, start=1):
    fold_rows.append({
        "fold": fold_num,
        "mse_39c": float(mean_squared_error(y[va_idx], base39c_oof[va_idx])),
        "mse_42b": float(mean_squared_error(y[va_idx], final_oof[va_idx])),
        "gain_vs_39c": float(mean_squared_error(y[va_idx], base39c_oof[va_idx]) - mean_squared_error(y[va_idx], final_oof[va_idx])),
        "covered_rows": int(best_cov_oof[va_idx].sum()),
    })
fold_gains42b3 = pd.DataFrame(fold_rows)

screen_path = "model_results/int42b3_prior_solver_screen.csv"
fold_diag_path = "model_results/int42b3_fold_diag.csv"
tier_perf_path = "model_results/int42b3_tier_performance.csv"
fold_gains_path = "model_results/int42b3_best_fold_gains.csv"

oof_path = "model_results/oof_int42b3_best.csv"
testpred_path = "model_results/testpred_int42b3_best.csv"
submission_path = "submission_int42b3_best.csv"

screen42b3.to_csv(screen_path, index=False)
pd.DataFrame(fold_diag).to_csv(fold_diag_path, index=False)
tier_perf42b3.to_csv(tier_perf_path, index=False)
fold_gains42b3.to_csv(fold_gains_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y,
    "pred_39c": base39c_oof,
    "prior": best_prior,
    "solver_pred": best_solver_oof,
    "solver_covered": best_cov_oof.astype(int),
    "pred_clipped": final_oof,
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids,
    "pred_39c": base39c_test,
    "prior": best_prior,
    "solver_pred": solver_test,
    "solver_covered": cov_test.astype(int),
    TARGET_COL: final_test,
}).to_csv(testpred_path, index=False)

pd.DataFrame({
    ID_COL: test_ids,
    TARGET_COL: final_test,
}).to_csv(submission_path, index=False)

sub = pd.read_csv(submission_path)
assert sub.shape == (n_test, 2)
assert list(sub.columns) == [ID_COL, TARGET_COL]
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

print("\n" + "=" * 90)
print("42B-3 complete")
print("=" * 90)

print("\nReference")
print("---------")
print(f"39C OOF MSE:             {mse39c:.6f}")
print(f"39C test-tier weighted:  {weighted39c:.6f}")

print("\nTop 42B solver candidates")
print("-------------------------")
display(screen42b3)

print("\nBest 42B candidate")
print("------------------")
print(best.to_string())
print(f"\nBest 42B OOF MSE:            {best_mse:.6f}")
print(f"Best 42B gain vs 39C:        {mse39c - best_mse:.6f}")
print(f"Best 42B weighted MSE:       {best_weighted:.6f}")
print(f"Best 42B weighted gain:      {weighted39c - best_weighted:.6f}")

print("\nTier performance")
print("----------------")
print(tier_perf42b3.to_string(index=False))

print("\nFold gains")
print("----------")
print(fold_gains42b3.to_string(index=False))
print("Min fold gain:", float(fold_gains42b3["gain_vs_39c"].min()))

print("\nTest coverage")
print("-------------")
print("Covered test rows:", int(cov_test.sum()), "of", n_test, "| rate:", float(cov_test.mean()))

print("\nSaved files")
print("-----------")
print(screen_path)
print(fold_diag_path)
print(tier_perf_path)
print(fold_gains_path)
print(oof_path)
print(testpred_path)
print(submission_path)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path)
print("Shape:", sub.shape)
print(sub[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
ordinary_gain = mse39c - best_mse
weighted_gain = weighted39c - best_weighted

overlap_gain = float(tier_perf42b3.loc[tier_perf42b3["tier"] == "overlap", "gain_vs_39c"].iloc[0])
solver_gain = float(tier_perf42b3.loc[tier_perf42b3["tier"] == "solver_only", "gain_vs_39c"].iloc[0])

if ordinary_gain >= 3.0:
    print("42B clears ordinary 3+ OOF threshold. Consider submitting if fold gains are stable and overlap is not harmed.")
elif weighted_gain >= 2.0 and solver_gain >= 2.0 and overlap_gain >= -0.05:
    print("42B clears public-relevant weighted/solver thresholds. Consider submitting.")
elif weighted_gain >= 1.0 and solver_gain >= 1.0 and overlap_gain >= -0.05:
    print("42B has some public-relevant signal but does not clear your current threshold. Save artifact; do not submit unless probing.")
else:
    print("42B does not clear submission threshold. Keep 39C as protected best.")

42B-3. Integer accounting solver with learned priors
Loaded prior account39c                               | OOF MSE 53.226631
Loaded prior account39d                               | OOF MSE 52.820927
Loaded prior int42a2                                  | OOF MSE 53.123951
Loaded prior prior42b_hgb_percent_balanced_unsolved   | OOF MSE 52.920437
Loaded prior prior42b_ridge_percent_solver_public     | OOF MSE 52.992805
Loaded prior prior42b_hgb_percent_solver_public       | OOF MSE 53.030693
Loaded prior prior42b_hgb_logit_solver_public         | OOF MSE 59.901684

Solving integer accounting with prior: account39c
fold 1: covered 16333/28985 | solver covered MSE 24.005636
fold 2: covered 16243/28984 | solver covered MSE 23.823112
fold 3: covered 16362/28984 | solver covered MSE 27.214041
fold 4: covered 16366/28984 | solver covered MSE 23.943897
fold 5: covered 16286/28984 | solver covered MSE 25.223650

Solving integer accounting with prior: account39d
fold 1: covered 16333/28985 | so

,prior,covered_rows,coverage_rate,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_39c,test_tier_weighted_mse,weighted_gain_vs_39c,overlap_mse,solver_only_mse,none_mse
0,prior42b_hgb_percent_solver_public,81590,0.562996,0.952,0.336,0.0,53.073307,0.153324,30.466082,0.281525,0.507146,65.261948,NaN
1,prior42b_ridge_percent_solver_public,81590,0.562996,0.968,0.308,0.0,53.091537,0.135094,30.501400,0.246207,0.505101,65.360893,NaN
2,prior42b_hgb_percent_balanced_unsolved,81590,0.562996,0.948,0.304,0.0,53.100243,0.126389,30.517891,0.229715,0.506669,65.403229,NaN
3,account39d,81590,0.562996,1.028,0.264,0.0,53.123951,0.102680,30.563990,0.183617,0.502933,65.534012,NaN
4,prior42b_hgb_logit_solver_public,81590,0.562996,0.908,0.184,0.0,53.128616,0.098015,30.574666,0.172941,0.491322,65.580772,NaN
5,int42a2,81590,0.562996,0.984,0.072,0.0,53.186779,0.039852,30.684719,0.062888,0.502779,65.861763,NaN
6,account39c,81590,0.562996,1.028,-0.024,0.0,53.191689,0.034943,30.694124,0.053483,0.502933,65.887039,NaN



Best 42B candidate
------------------
prior                     prior42b_hgb_percent_solver_public
covered_rows                                           81590
coverage_rate                                       0.562996
lambda_overlap                                         0.952
lambda_solver_only                                     0.336
lambda_none                                              0.0
oof_mse                                            53.073307
gain_vs_39c                                         0.153324
test_tier_weighted_mse                             30.466082
weighted_gain_vs_39c                                0.281525
overlap_mse                                         0.507146
solver_only_mse                                    65.261948
none_mse                                                 NaN

Best 42B OOF MSE:            53.073307
Best 42B gain vs 39C:        0.153324
Best 42B weighted MSE:       30.466082
Best 42B weighted gain:      0.281525

Tier perform

In [41]:
# ============================================================
# 43A. 39D-protected tier/segment stacking screen
# ============================================================
#
# Current public-best anchor:
#   39D public MSE = 30.752
#
# Goal:
#   Try to improve 39D using only saved prediction artifacts.
#
# Safe:
#   - No LightGBM
#   - No PyTorch
#   - No target encoding construction
#   - No giant feature matrices
#
# Strategy:
#   OOF-safe selection of candidate predictions by tier and segment.
#
#   Tiers:
#     overlap
#     solver_only
#     none
#
#   Segment options:
#     tier-only
#     solver_only by SUBGROUP_NAME
#     solver_only by ASSESSMENT_NAME
#     solver_only by ASSESSMENT_NAME x SUBGROUP_NAME
#
# Important:
#   This does NOT mean submit automatically.
#   It tells us if any saved model source helps 39D in a public-relevant way.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("43A. 39D-protected tier/segment stacking screen")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required setup checks
# ------------------------------------------------------------

required_43a = ["raw_train_te", "raw_test_te"]
missing_43a = [x for x in required_43a if x not in globals()]

if missing_43a:
    raise ValueError(f"Missing required objects: {missing_43a}. Rerun safe recovery/setup first.")

# ------------------------------------------------------------
# 2. Load saved prediction artifacts
# ------------------------------------------------------------

artifact_specs_43a = [
    ("anchor33a", "model_results/oof_seg33a_arcsine_adaptive.csv", "model_results/testpred_seg33a_arcsine_adaptive.csv", "ml"),
    ("knn35a", "model_results/oof_knn35a_local_residual_best.csv", "model_results/testpred_knn35a_local_residual_best.csv", "ml"),
    ("nn37b", "model_results/oof_nn37b38b_cpu_safe_rank1.csv", "model_results/testpred_nn37b38b_cpu_safe_rank1.csv", "ml"),
    ("step36a", "model_results/oof_step36a_residual_bins_best.csv", "model_results/testpred_step36a_residual_bins_best.csv", "ml"),
    ("blend33c", "model_results/oof_blend33c_33a_huber33b.csv", "model_results/testpred_blend33c_33a_huber33b.csv", "ml"),

    ("account39a", "model_results/oof_account39a_best.csv", "model_results/testpred_account39a_best.csv", "accounting"),
    ("account39b", "model_results/oof_account39b_solver_best.csv", "model_results/testpred_account39b_solver_best.csv", "accounting"),
    ("account39c", "model_results/oof_account39c_final_best.csv", "model_results/testpred_account39c_final_best.csv", "accounting"),
    ("account39d", "model_results/oof_account39d_best.csv", "model_results/testpred_account39d_best.csv", "accounting"),

    ("hte40a3", "model_results/oof_hte40a3_residual_best.csv", "model_results/testpred_hte40a3_residual_best.csv", "risky"),
    ("entity41a", "model_results/oof_entity41a_best.csv", "model_results/testpred_entity41a_best.csv", "risky"),
    ("int42a2", "model_results/oof_int42a2_best.csv", "model_results/testpred_int42a2_best.csv", "accounting"),
    ("int42b3", "model_results/oof_int42b3_best.csv", "model_results/testpred_int42b3_best.csv", "accounting"),
]

def load_oof_pred_43a(path):
    df = pd.read_csv(path)

    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    pred_col = None
    for c in ["pred_clipped", "prediction", "pred", "oof_pred"]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if TARGET_COL in df.columns:
        y = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
    else:
        y = None

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}")

    return np.clip(pred, 0, 100).astype(np.float32), y, pred_col

def load_test_pred_43a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} in {path} and no global test_ids found.")

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col

oof_preds_43a = {}
test_preds_43a = {}
artifact_kind_43a = {}
artifact_rows_43a = []

y_43a = None
test_ids_43a = None

for name, oof_path, test_path, kind in artifact_specs_43a:
    if not Path(oof_path).exists() or not Path(test_path).exists():
        print(f"Skipping missing artifact: {name}")
        continue

    try:
        oof_pred, y_file, oof_col = load_oof_pred_43a(oof_path)
        test_pred, ids, test_col = load_test_pred_43a(test_path)

        if y_43a is None:
            if y_file is not None:
                y_43a = y_file.astype(np.float32)
            elif "y_train" in globals():
                y_43a = np.asarray(y_train, dtype=np.float32).reshape(-1)
            else:
                raise ValueError("Could not recover y_train.")
        else:
            if y_file is not None:
                max_diff = float(np.nanmax(np.abs(y_file - y_43a)))
                if max_diff > 1e-5:
                    raise ValueError(f"Target mismatch for {name}: max diff {max_diff}")

        if test_ids_43a is None:
            test_ids_43a = ids

        if len(oof_pred) != len(y_43a):
            raise ValueError(f"OOF length mismatch for {name}")
        if len(test_pred) != len(test_ids_43a):
            raise ValueError(f"Test length mismatch for {name}")

        oof_preds_43a[name] = oof_pred
        test_preds_43a[name] = test_pred
        artifact_kind_43a[name] = kind

        mse = float(mean_squared_error(y_43a, oof_pred))

        artifact_rows_43a.append({
            "name": name,
            "kind": kind,
            "oof_mse": mse,
            "oof_path": oof_path,
            "test_path": test_path,
            "oof_col": oof_col,
            "test_col": test_col,
        })

        print(f"Loaded {name:12s} | kind={kind:10s} | OOF MSE {mse:.6f}")

    except Exception as e:
        print(f"Skipping {name} due to error: {repr(e)}")

artifact_summary43a = (
    pd.DataFrame(artifact_rows_43a)
    .sort_values("oof_mse")
    .reset_index(drop=True)
)

if "account39d" not in oof_preds_43a:
    raise ValueError("account39d is required as protected anchor.")

n_train_43a = len(y_43a)
n_test_43a = len(test_ids_43a)

mse39d_43a = float(mean_squared_error(y_43a, oof_preds_43a["account39d"]))

print("\nArtifact summary")
print("----------------")
display(artifact_summary43a)

print("\nProtected anchor")
print("----------------")
print(f"39D OOF MSE: {mse39d_43a:.6f}")
print("39D public MSE: 30.752")

# ------------------------------------------------------------
# 3. Recover accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_cov_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_cov_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_cov_oof & solver_cov_oof
solver_only_oof = solver_cov_oof & ~direct_cov_oof
none_oof = ~(direct_cov_oof | solver_cov_oof)

overlap_test = direct_cov_test & solver_cov_test
solver_only_test = solver_cov_test & ~direct_cov_test
none_test = ~(direct_cov_test | solver_cov_test)

tier_masks_oof_43a = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test_43a = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

test_tier_rates_43a = {
    tier: float(mask.mean())
    for tier, mask in tier_masks_test_43a.items()
}

print("\nTier coverage")
print("-------------")
for tier in ["overlap", "solver_only", "none"]:
    print(
        f"{tier:12s} train n={int(tier_masks_oof_43a[tier].sum()):6d} "
        f"| test n={int(tier_masks_test_43a[tier].sum()):6d}"
    )

# ------------------------------------------------------------
# 4. Segment labels
# ------------------------------------------------------------

def clean_str_43a(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str).to_numpy()

subgroup_train_43a = clean_str_43a(raw_train_te["SUBGROUP_NAME"])
subgroup_test_43a = clean_str_43a(raw_test_te["SUBGROUP_NAME"])

assessment_train_43a = clean_str_43a(raw_train_te["ASSESSMENT_NAME"])
assessment_test_43a = clean_str_43a(raw_test_te["ASSESSMENT_NAME"])

assess_subgroup_train_43a = np.array(
    [f"{a}||{b}" for a, b in zip(assessment_train_43a, subgroup_train_43a)],
    dtype=object,
)

assess_subgroup_test_43a = np.array(
    [f"{a}||{b}" for a, b in zip(assessment_test_43a, subgroup_test_43a)],
    dtype=object,
)

segment_defs_43a = {
    "tier_only": {
        "train": np.array(["GLOBAL"] * n_train_43a, dtype=object),
        "test": np.array(["GLOBAL"] * n_test_43a, dtype=object),
        "min_rows": 1,
    },
    "subgroup": {
        "train": subgroup_train_43a.astype(object),
        "test": subgroup_test_43a.astype(object),
        "min_rows": 500,
    },
    "assessment": {
        "train": assessment_train_43a.astype(object),
        "test": assessment_test_43a.astype(object),
        "min_rows": 500,
    },
    "assessment_subgroup": {
        "train": assess_subgroup_train_43a,
        "test": assess_subgroup_test_43a,
        "min_rows": 400,
    },
}

# ------------------------------------------------------------
# 5. Metric helpers
# ------------------------------------------------------------

def ordinary_mse_43a(pred):
    return float(mean_squared_error(y_43a, np.clip(pred, 0, 100)))

def tier_weighted_mse_43a(pred):
    pred = np.clip(pred, 0, 100)
    total = 0.0

    for tier, mask in tier_masks_oof_43a.items():
        if int(mask.sum()) == 0:
            continue
        total += test_tier_rates_43a[tier] * float(mean_squared_error(y_43a[mask], pred[mask]))

    return float(total)

def tier_mse_dict_43a(pred):
    pred = np.clip(pred, 0, 100)
    out = {}

    for tier, mask in tier_masks_oof_43a.items():
        if int(mask.sum()) > 0:
            out[f"mse_{tier}"] = float(mean_squared_error(y_43a[mask], pred[mask]))
        else:
            out[f"mse_{tier}"] = np.nan

    return out

baseline_rows_43a = []

for name, pred in oof_preds_43a.items():
    row = {
        "name": name,
        "kind": artifact_kind_43a[name],
        "ordinary_oof_mse": ordinary_mse_43a(pred),
        "test_tier_weighted_mse": tier_weighted_mse_43a(pred),
        "gain_vs_39d_oof": mse39d_43a - ordinary_mse_43a(pred),
        "gain_vs_39d_weighted": tier_weighted_mse_43a(oof_preds_43a["account39d"]) - tier_weighted_mse_43a(pred),
    }
    row.update(tier_mse_dict_43a(pred))
    baseline_rows_43a.append(row)

baseline_metrics43a = (
    pd.DataFrame(baseline_rows_43a)
    .sort_values("test_tier_weighted_mse")
    .reset_index(drop=True)
)

print("\nBaseline model metrics vs 39D")
print("-----------------------------")
display(baseline_metrics43a)

# ------------------------------------------------------------
# 6. Candidate pools
# ------------------------------------------------------------

safe_pool_43a = [
    name for name in [
        "account39a", "account39b", "account39c", "account39d", "int42a2", "int42b3"
    ]
    if name in oof_preds_43a
]

with_ml_pool_43a = [
    name for name in safe_pool_43a + ["knn35a", "nn37b", "step36a", "blend33c", "anchor33a"]
    if name in oof_preds_43a
]

with_risky_pool_43a = [
    name for name in with_ml_pool_43a + ["hte40a3", "entity41a"]
    if name in oof_preds_43a
]

candidate_pools_43a = {
    "safe_accounting": safe_pool_43a,
    "safe_plus_ml": with_ml_pool_43a,
    "safe_plus_risky": with_risky_pool_43a,
}

print("\nCandidate pools")
print("---------------")
for pool_name, names in candidate_pools_43a.items():
    print(pool_name, ":", names)

# ------------------------------------------------------------
# 7. OOF-safe segment selector
# ------------------------------------------------------------

folds43a = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_43a))
)

def choose_best_model_43a(idx, names):
    best_name = None
    best_mse = np.inf

    for name in names:
        pred = oof_preds_43a[name][idx]
        mse = float(mean_squared_error(y_43a[idx], pred))

        if mse < best_mse:
            best_mse = mse
            best_name = name

    return best_name, best_mse

def build_segment_selected_candidate_43a(
    strategy_name,
    pool_names,
    segment_mode,
    protected_overlap=True,
    protected_none=True,
):
    """
    OOF-safe:
      For each fold, select model choices using train-fold rows only.
      Apply choices to validation fold and test rows.
    """
    seg_train = segment_defs_43a[segment_mode]["train"]
    seg_test = segment_defs_43a[segment_mode]["test"]
    min_rows = int(segment_defs_43a[segment_mode]["min_rows"])

    pred_oof = np.full(n_train_43a, np.nan, dtype=np.float32)
    pred_test_sum = np.zeros(n_test_43a, dtype=np.float64)
    choice_rows = []

    for fold_num, (tr_idx, va_idx) in enumerate(folds43a, start=1):
        pred_va = np.zeros(len(va_idx), dtype=np.float64)
        pred_test_fold = np.zeros(n_test_43a, dtype=np.float64)

        for tier in ["overlap", "solver_only", "none"]:
            tr_tier_mask = tier_masks_oof_43a[tier][tr_idx]
            va_tier_mask = tier_masks_oof_43a[tier][va_idx]
            test_tier_mask = tier_masks_test_43a[tier]

            if int(va_tier_mask.sum()) == 0 and int(test_tier_mask.sum()) == 0:
                continue

            if protected_overlap and tier == "overlap":
                tier_pool = ["account39d"]
            elif protected_none and tier == "none":
                tier_pool = ["account39d"]
            else:
                tier_pool = pool_names

            # Global fallback for this tier.
            tr_tier_idx = tr_idx[tr_tier_mask]

            if len(tr_tier_idx) >= 50:
                global_best, global_mse = choose_best_model_43a(tr_tier_idx, tier_pool)
            else:
                global_best = "account39d"
                global_mse = np.nan

            # Segment-specific choices only for solver_only tier unless segment_mode is tier_only.
            if segment_mode == "tier_only" or tier != "solver_only":
                # Apply global tier choice.
                if int(va_tier_mask.sum()) > 0:
                    pred_va[va_tier_mask] = oof_preds_43a[global_best][va_idx[va_tier_mask]]
                if int(test_tier_mask.sum()) > 0:
                    pred_test_fold[test_tier_mask] = test_preds_43a[global_best][test_tier_mask]

                choice_rows.append({
                    "strategy": strategy_name,
                    "fold": fold_num,
                    "tier": tier,
                    "segment_mode": segment_mode,
                    "segment": "GLOBAL",
                    "chosen_model": global_best,
                    "n_train": int(len(tr_tier_idx)),
                    "train_mse": global_mse,
                    "fallback_model": global_best,
                })

            else:
                # Solver-only segment-specific choice.
                tr_solver_segments = seg_train[tr_tier_idx]
                va_solver_idx = va_idx[va_tier_mask]
                test_solver_idx = np.where(test_tier_mask)[0]

                # Apply default global first.
                if len(va_solver_idx) > 0:
                    pred_va[va_tier_mask] = oof_preds_43a[global_best][va_solver_idx]
                if len(test_solver_idx) > 0:
                    pred_test_fold[test_tier_mask] = test_preds_43a[global_best][test_solver_idx]

                unique_segments = sorted(pd.Series(tr_solver_segments).dropna().astype(str).unique().tolist())

                for seg in unique_segments:
                    tr_seg_mask = tr_tier_mask & (seg_train[tr_idx].astype(str) == str(seg))
                    tr_seg_idx = tr_idx[tr_seg_mask]

                    if len(tr_seg_idx) < min_rows:
                        continue

                    chosen, mse_chosen = choose_best_model_43a(tr_seg_idx, tier_pool)

                    va_seg_mask_local = va_tier_mask & (seg_train[va_idx].astype(str) == str(seg))
                    test_seg_mask = test_tier_mask & (seg_test.astype(str) == str(seg))

                    if int(va_seg_mask_local.sum()) > 0:
                        pred_va[va_seg_mask_local] = oof_preds_43a[chosen][va_idx[va_seg_mask_local]]

                    if int(test_seg_mask.sum()) > 0:
                        pred_test_fold[test_seg_mask] = test_preds_43a[chosen][test_seg_mask]

                    choice_rows.append({
                        "strategy": strategy_name,
                        "fold": fold_num,
                        "tier": tier,
                        "segment_mode": segment_mode,
                        "segment": str(seg),
                        "chosen_model": chosen,
                        "n_train": int(len(tr_seg_idx)),
                        "train_mse": mse_chosen,
                        "fallback_model": global_best,
                    })

        pred_oof[va_idx] = np.clip(pred_va, 0, 100).astype(np.float32)
        pred_test_sum += np.clip(pred_test_fold, 0, 100)

    pred_test = (pred_test_sum / N_SPLITS).astype(np.float32)

    return pred_oof, pred_test, pd.DataFrame(choice_rows)

# ------------------------------------------------------------
# 8. Run strategies
# ------------------------------------------------------------

strategy_specs_43a = []

for pool_name, pool_names in candidate_pools_43a.items():
    for segment_mode in ["tier_only", "subgroup", "assessment", "assessment_subgroup"]:
        strategy_specs_43a.append({
            "strategy_name": f"{pool_name}_{segment_mode}",
            "pool_name": pool_name,
            "pool_names": pool_names,
            "segment_mode": segment_mode,
            "protected_overlap": True,
            "protected_none": True,
        })

print("\nStrategies")
print("----------")
for s in strategy_specs_43a:
    print(s["strategy_name"])

screen_rows_43a = []
choice_frames_43a = []
candidate_store_43a = {}

for spec in strategy_specs_43a:
    strategy_name = spec["strategy_name"]

    print("\nRunning strategy:", strategy_name)

    pred_oof, pred_test, choices = build_segment_selected_candidate_43a(
        strategy_name=strategy_name,
        pool_names=spec["pool_names"],
        segment_mode=spec["segment_mode"],
        protected_overlap=spec["protected_overlap"],
        protected_none=spec["protected_none"],
    )

    mse_oof = ordinary_mse_43a(pred_oof)
    mse_weighted = tier_weighted_mse_43a(pred_oof)

    row = {
        "strategy": strategy_name,
        "pool_name": spec["pool_name"],
        "segment_mode": spec["segment_mode"],
        "ordinary_oof_mse": mse_oof,
        "test_tier_weighted_mse": mse_weighted,
        "gain_vs_39d_oof": mse39d_43a - mse_oof,
        "gain_vs_39d_weighted": tier_weighted_mse_43a(oof_preds_43a["account39d"]) - mse_weighted,
    }
    row.update(tier_mse_dict_43a(pred_oof))

    screen_rows_43a.append(row)

    choices["strategy"] = strategy_name
    choice_frames_43a.append(choices)

    candidate_store_43a[strategy_name] = {
        "oof": pred_oof,
        "test": pred_test,
        "choices": choices,
    }

screen43a = pd.DataFrame(screen_rows_43a).sort_values(
    ["test_tier_weighted_mse", "ordinary_oof_mse"]
).reset_index(drop=True)

screen43a_by_oof = screen43a.sort_values(
    ["ordinary_oof_mse", "test_tier_weighted_mse"]
).reset_index(drop=True)

best_weighted_name_43a = screen43a.iloc[0]["strategy"]
best_oof_name_43a = screen43a_by_oof.iloc[0]["strategy"]

best_weighted_oof_43a = candidate_store_43a[best_weighted_name_43a]["oof"]
best_weighted_test_43a = candidate_store_43a[best_weighted_name_43a]["test"]

best_oof_oof_43a = candidate_store_43a[best_oof_name_43a]["oof"]
best_oof_test_43a = candidate_store_43a[best_oof_name_43a]["test"]

# ------------------------------------------------------------
# 9. Fold diagnostics for best weighted and best OOF
# ------------------------------------------------------------

def fold_diag_for_pred_43a(pred, label):
    rows = []

    for fold_num, (_, va_idx) in enumerate(folds43a, start=1):
        row = {
            "label": label,
            "fold": fold_num,
            "mse_39d": float(mean_squared_error(y_43a[va_idx], oof_preds_43a["account39d"][va_idx])),
            "mse_candidate": float(mean_squared_error(y_43a[va_idx], pred[va_idx])),
        }
        row["gain_vs_39d"] = row["mse_39d"] - row["mse_candidate"]

        for tier, mask_full in tier_masks_oof_43a.items():
            mask_fold = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask_fold.sum())

            if int(mask_fold.sum()) > 0:
                row[f"mse39d_{tier}"] = float(mean_squared_error(
                    y_43a[va_idx][mask_fold],
                    oof_preds_43a["account39d"][va_idx][mask_fold],
                ))
                row[f"mse_candidate_{tier}"] = float(mean_squared_error(
                    y_43a[va_idx][mask_fold],
                    pred[va_idx][mask_fold],
                ))
                row[f"gain_{tier}"] = row[f"mse39d_{tier}"] - row[f"mse_candidate_{tier}"]

        rows.append(row)

    return pd.DataFrame(rows)

fold_diag_weighted43a = fold_diag_for_pred_43a(best_weighted_oof_43a, "best_weighted")
fold_diag_oof43a = fold_diag_for_pred_43a(best_oof_oof_43a, "best_oof")
fold_diag43a = pd.concat([fold_diag_weighted43a, fold_diag_oof43a], axis=0).reset_index(drop=True)

# ------------------------------------------------------------
# 10. Save artifacts
# ------------------------------------------------------------

screen_path43a = "model_results/stack43a_39d_protected_screen.csv"
baseline_path43a = "model_results/stack43a_baseline_metrics.csv"
choices_path43a = "model_results/stack43a_segment_choices.csv"
fold_diag_path43a = "model_results/stack43a_best_fold_diag.csv"

oof_weighted_path43a = "model_results/oof_stack43a_best_weighted.csv"
test_weighted_path43a = "model_results/testpred_stack43a_best_weighted.csv"
submission_weighted_path43a = "submission_stack43a_best_weighted.csv"

oof_oof_path43a = "model_results/oof_stack43a_best_oof.csv"
test_oof_path43a = "model_results/testpred_stack43a_best_oof.csv"
submission_oof_path43a = "submission_stack43a_best_oof.csv"

screen43a.to_csv(screen_path43a, index=False)
baseline_metrics43a.to_csv(baseline_path43a, index=False)

if len(choice_frames_43a) > 0:
    pd.concat(choice_frames_43a, axis=0).to_csv(choices_path43a, index=False)
else:
    pd.DataFrame().to_csv(choices_path43a, index=False)

fold_diag43a.to_csv(fold_diag_path43a, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_43a),
    TARGET_COL: y_43a,
    "pred_39d": oof_preds_43a["account39d"],
    "pred_clipped": best_weighted_oof_43a,
}).to_csv(oof_weighted_path43a, index=False)

pd.DataFrame({
    ID_COL: test_ids_43a,
    TARGET_COL: best_weighted_test_43a,
}).to_csv(test_weighted_path43a, index=False)

pd.DataFrame({
    ID_COL: test_ids_43a,
    TARGET_COL: best_weighted_test_43a,
}).to_csv(submission_weighted_path43a, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_43a),
    TARGET_COL: y_43a,
    "pred_39d": oof_preds_43a["account39d"],
    "pred_clipped": best_oof_oof_43a,
}).to_csv(oof_oof_path43a, index=False)

pd.DataFrame({
    ID_COL: test_ids_43a,
    TARGET_COL: best_oof_test_43a,
}).to_csv(test_oof_path43a, index=False)

pd.DataFrame({
    ID_COL: test_ids_43a,
    TARGET_COL: best_oof_test_43a,
}).to_csv(submission_oof_path43a, index=False)

# Validate submissions.
for p in [submission_weighted_path43a, submission_oof_path43a]:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test_43a, 2)
    assert list(sub.columns) == [ID_COL, TARGET_COL]
    assert sub[ID_COL].notna().all()
    assert sub[TARGET_COL].notna().all()
    assert np.isfinite(sub[TARGET_COL]).all()
    assert sub[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 11. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("43A 39D-protected segment stack complete")
print("=" * 90)

print("\nReference 39D")
print("-------------")
print(f"39D ordinary OOF MSE:       {ordinary_mse_43a(oof_preds_43a['account39d']):.6f}")
print(f"39D test-tier weighted MSE: {tier_weighted_mse_43a(oof_preds_43a['account39d']):.6f}")
print("39D public MSE:             30.752")

print("\nTop strategies by test-tier weighted MSE")
print("----------------------------------------")
display(screen43a.head(20))

print("\nTop strategies by ordinary OOF MSE")
print("----------------------------------")
display(screen43a_by_oof.head(20))

print("\nBest weighted strategy")
print("----------------------")
print(screen43a.iloc[0].to_string())

print("\nBest OOF strategy")
print("-----------------")
print(screen43a_by_oof.iloc[0].to_string())

print("\nFold diagnostics")
print("----------------")
print(fold_diag43a.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path43a)
print(baseline_path43a)
print(choices_path43a)
print(fold_diag_path43a)
print(oof_weighted_path43a)
print(test_weighted_path43a)
print(submission_weighted_path43a)
print(oof_oof_path43a)
print(test_oof_path43a)
print(submission_oof_path43a)

print("\nDecision rule")
print("-------------")
best_weighted_gain = float(screen43a.iloc[0]["gain_vs_39d_weighted"])
best_oof_gain = float(screen43a_by_oof.iloc[0]["gain_vs_39d_oof"])

if best_weighted_gain >= 1.5 and best_oof_gain >= 0.0:
    print("43A has meaningful weighted improvement over 39D. Consider as a serious candidate, but inspect whether it uses risky HTE/entity artifacts.")
elif best_oof_gain >= 1.5 and best_weighted_gain >= 0.0:
    print("43A has meaningful ordinary OOF improvement, but weighted improvement is smaller. Treat cautiously.")
elif best_weighted_gain > 0.3 and best_oof_gain >= 0.0:
    print("43A has modest improvement over 39D. Save artifact; submit only if you want a speculative slot.")
else:
    print("43A does not improve enough over 39D. Keep 39D as protected public-best.")

43A. 39D-protected tier/segment stacking screen
Loaded anchor33a    | kind=ml         | OOF MSE 77.049866
Loaded knn35a       | kind=ml         | OOF MSE 76.453072
Loaded nn37b        | kind=ml         | OOF MSE 76.849442
Loaded step36a      | kind=ml         | OOF MSE 76.954796
Loaded blend33c     | kind=ml         | OOF MSE 76.859200
Loaded account39a   | kind=accounting | OOF MSE 55.642780
Loaded account39b   | kind=accounting | OOF MSE 53.720547
Loaded account39c   | kind=accounting | OOF MSE 53.226631
Loaded account39d   | kind=accounting | OOF MSE 52.820927
Loaded hte40a3      | kind=risky      | OOF MSE 49.689606
Loaded entity41a    | kind=risky      | OOF MSE 52.760006
Loaded int42a2      | kind=accounting | OOF MSE 53.123951
Loaded int42b3      | kind=accounting | OOF MSE 53.073307

Artifact summary
----------------


,name,kind,oof_mse,oof_path,test_path,oof_col,test_col
0,hte40a3,risky,49.689606,model_results/oof_hte40a3_residual_best.csv,model_results/testpred_hte40a3_residual_best.csv,pred_clipped,PERCENT_PROFICIENT
1,entity41a,risky,52.760006,model_results/oof_entity41a_best.csv,model_results/testpred_entity41a_best.csv,pred_clipped,PERCENT_PROFICIENT
2,account39d,accounting,52.820927,model_results/oof_account39d_best.csv,model_results/testpred_account39d_best.csv,pred_clipped,PERCENT_PROFICIENT
3,int42b3,accounting,53.073307,model_results/oof_int42b3_best.csv,model_results/testpred_int42b3_best.csv,pred_clipped,PERCENT_PROFICIENT
4,int42a2,accounting,53.123951,model_results/oof_int42a2_best.csv,model_results/testpred_int42a2_best.csv,pred_clipped,PERCENT_PROFICIENT
5,account39c,accounting,53.226631,model_results/oof_account39c_final_best.csv,model_results/testpred_account39c_final_best.csv,pred_clipped,PERCENT_PROFICIENT
6,account39b,accounting,53.720547,model_results/oof_account39b_solver_best.csv,model_results/testpred_account39b_solver_best.csv,pred_clipped,PERCENT_PROFICIENT
7,account39a,accounting,55.642780,model_results/oof_account39a_best.csv,model_results/testpred_account39a_best.csv,pred_clipped,PERCENT_PROFICIENT
8,knn35a,ml,76.453072,model_results/oof_knn35a_local_residual_best.csv,model_results/testpred_knn35a_local_residual_b...,pred_clipped,PERCENT_PROFICIENT
9,nn37b,ml,76.849442,model_results/oof_nn37b38b_cpu_safe_rank1.csv,model_results/testpred_nn37b38b_cpu_safe_rank1...,pred_clipped,PERCENT_PROFICIENT



Protected anchor
----------------
39D OOF MSE: 52.820927
39D public MSE: 30.752

Tier coverage
-------------
overlap      train n= 53786 | test n= 27298
solver_only  train n= 27807 | test n= 17807
none         train n= 63328 | test n=  3202

Baseline model metrics vs 39D
-----------------------------


,name,kind,ordinary_oof_mse,test_tier_weighted_mse,gain_vs_39d_oof,gain_vs_39d_weighted,mse_overlap,mse_solver_only,mse_none
0,hte40a3,risky,49.689606,29.890315,3.131321,0.257172,0.595326,64.945831,84.687637
1,account39d,accounting,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
2,int42b3,accounting,53.073307,30.466082,-0.252380,-0.318595,0.507233,65.261948,92.366982
3,int42a2,accounting,53.123951,30.563990,-0.303024,-0.416503,0.503020,65.534012,92.366982
4,entity41a,risky,52.760006,30.664958,0.060921,-0.517471,0.595309,65.855698,91.314491
5,account39c,accounting,53.226631,30.747607,-0.405704,-0.600120,0.595326,65.890625,92.366982
6,account39b,accounting,53.720547,31.651285,-0.899620,-1.503798,0.862888,67.930656,92.374245
7,account39a,accounting,55.642780,35.148235,-2.821854,-5.000748,0.595420,77.772476,92.678802
8,knn35a,ml,76.453072,66.983315,-23.632145,-36.835828,57.247768,77.342117,92.374245
9,blend33c,ml,76.859200,67.434008,-24.038273,-37.286521,57.788399,77.677948,92.696976



Candidate pools
---------------
safe_accounting : ['account39a', 'account39b', 'account39c', 'account39d', 'int42a2', 'int42b3']
safe_plus_ml : ['account39a', 'account39b', 'account39c', 'account39d', 'int42a2', 'int42b3', 'knn35a', 'nn37b', 'step36a', 'blend33c', 'anchor33a']
safe_plus_risky : ['account39a', 'account39b', 'account39c', 'account39d', 'int42a2', 'int42b3', 'knn35a', 'nn37b', 'step36a', 'blend33c', 'anchor33a', 'hte40a3', 'entity41a']

Strategies
----------
safe_accounting_tier_only
safe_accounting_subgroup
safe_accounting_assessment
safe_accounting_assessment_subgroup
safe_plus_ml_tier_only
safe_plus_ml_subgroup
safe_plus_ml_assessment
safe_plus_ml_assessment_subgroup
safe_plus_risky_tier_only
safe_plus_risky_subgroup
safe_plus_risky_assessment
safe_plus_risky_assessment_subgroup

Running strategy: safe_accounting_tier_only

Running strategy: safe_accounting_subgroup

Running strategy: safe_accounting_assessment

Running strategy: safe_accounting_assessment_subgroup

R

,strategy,pool_name,segment_mode,ordinary_oof_mse,test_tier_weighted_mse,gain_vs_39d_oof,gain_vs_39d_weighted,mse_overlap,mse_solver_only,mse_none
0,safe_plus_risky_subgroup,safe_plus_risky,subgroup,52.809963,30.126422,0.010963,0.021065,0.595325,64.247169,92.135094
1,safe_accounting_subgroup,safe_accounting,subgroup,52.811234,30.128875,0.009693,0.018612,0.595325,64.253822,92.135094
2,safe_plus_ml_subgroup,safe_plus_ml,subgroup,52.811234,30.128875,0.009693,0.018612,0.595325,64.253822,92.135094
3,safe_accounting_tier_only,safe_accounting,tier_only,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
4,safe_accounting_assessment_subgroup,safe_accounting,assessment_subgroup,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
5,safe_plus_ml_tier_only,safe_plus_ml,tier_only,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
6,safe_plus_ml_assessment_subgroup,safe_plus_ml,assessment_subgroup,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
7,safe_plus_risky_tier_only,safe_plus_risky,tier_only,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
8,safe_plus_risky_assessment_subgroup,safe_plus_risky,assessment_subgroup,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
9,safe_plus_risky_assessment,safe_plus_risky,assessment,52.867645,30.237241,-0.046719,-0.089754,0.595325,64.547798,92.135094



Top strategies by ordinary OOF MSE
----------------------------------


,strategy,pool_name,segment_mode,ordinary_oof_mse,test_tier_weighted_mse,gain_vs_39d_oof,gain_vs_39d_weighted,mse_overlap,mse_solver_only,mse_none
0,safe_plus_risky_subgroup,safe_plus_risky,subgroup,52.809963,30.126422,0.010963,0.021065,0.595325,64.247169,92.135094
1,safe_accounting_subgroup,safe_accounting,subgroup,52.811234,30.128875,0.009693,0.018612,0.595325,64.253822,92.135094
2,safe_plus_ml_subgroup,safe_plus_ml,subgroup,52.811234,30.128875,0.009693,0.018612,0.595325,64.253822,92.135094
3,safe_accounting_tier_only,safe_accounting,tier_only,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
4,safe_accounting_assessment_subgroup,safe_accounting,assessment_subgroup,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
5,safe_plus_ml_tier_only,safe_plus_ml,tier_only,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
6,safe_plus_ml_assessment_subgroup,safe_plus_ml,assessment_subgroup,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
7,safe_plus_risky_tier_only,safe_plus_risky,tier_only,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
8,safe_plus_risky_assessment_subgroup,safe_plus_risky,assessment_subgroup,52.820927,30.147487,0.000000,0.000000,0.595325,64.304314,92.135094
9,safe_plus_risky_assessment,safe_plus_risky,assessment,52.867645,30.237241,-0.046719,-0.089754,0.595325,64.547798,92.135094



Best weighted strategy
----------------------
strategy                  safe_plus_risky_subgroup
pool_name                          safe_plus_risky
segment_mode                              subgroup
ordinary_oof_mse                         52.809963
test_tier_weighted_mse                   30.126422
gain_vs_39d_oof                           0.010963
gain_vs_39d_weighted                      0.021065
mse_overlap                               0.595325
mse_solver_only                          64.247169
mse_none                                 92.135094

Best OOF strategy
-----------------
strategy                  safe_plus_risky_subgroup
pool_name                          safe_plus_risky
segment_mode                              subgroup
ordinary_oof_mse                         52.809963
test_tier_weighted_mse                   30.126422
gain_vs_39d_oof                           0.010963
gain_vs_39d_weighted                      0.021065
mse_overlap                               0.59532

In [42]:
# ============================================================
# 43B-1. Assessment / grade / enrollment structure diagnostic
# ============================================================
#
# Goal:
#   Look for new structural signal outside accounting:
#   assessment grade/subject/type + grade-level enrollment features.
#
# This is a diagnostic only:
#   - parses ASSESSMENT_NAME
#   - audits available enrollment/grade-span columns
#   - computes residual patterns after 39D
#   - checks whether grade/subject structure explains remaining error
#
# No model training yet.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("43B-1. Assessment / grade / enrollment structure diagnostic")
print("=" * 90)

required_43b = ["raw_train_te", "raw_test_te", "X_train_proc_model", "X_test_proc_model", "y_train"]
missing_43b = [x for x in required_43b if x not in globals()]
if missing_43b:
    raise ValueError(f"Missing required objects: {missing_43b}. Rerun safe recovery/setup first.")

y_43b = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_43b = len(y_43b)
n_test_43b = len(raw_test_te)

# ------------------------------------------------------------
# 1. Load protected 39D anchor
# ------------------------------------------------------------

oof39d_path = "model_results/oof_account39d_best.csv"
test39d_path = "model_results/testpred_account39d_best.csv"

if not Path(oof39d_path).exists() or not Path(test39d_path).exists():
    raise FileNotFoundError("Missing 39D artifacts.")

oof39d = pd.read_csv(oof39d_path)
test39d = pd.read_csv(test39d_path)

if "row_index" in oof39d.columns:
    oof39d = oof39d.sort_values("row_index").reset_index(drop=True)

if "pred_clipped" in oof39d.columns:
    pred39d_oof = oof39d["pred_clipped"].to_numpy(dtype=np.float32)
elif TARGET_COL in oof39d.columns:
    pred39d_oof = oof39d[TARGET_COL].to_numpy(dtype=np.float32)
else:
    raise ValueError("Could not find 39D OOF prediction column.")

if TARGET_COL in test39d.columns:
    pred39d_test = test39d[TARGET_COL].to_numpy(dtype=np.float32)
else:
    numeric_cols = [c for c in test39d.columns if c != ID_COL and pd.api.types.is_numeric_dtype(test39d[c])]
    pred39d_test = test39d[numeric_cols[0]].to_numpy(dtype=np.float32)

pred39d_oof = np.clip(pred39d_oof, 0, 100)
pred39d_test = np.clip(pred39d_test, 0, 100)

resid39d = (y_43b - pred39d_oof).astype(np.float32)
mse39d = float(mean_squared_error(y_43b, pred39d_oof))

print("\nProtected anchor")
print("----------------")
print(f"39D OOF MSE: {mse39d:.6f}")
print("39D public MSE: 30.752")
print("Residual mean:", float(np.mean(resid39d)))
print("Residual std: ", float(np.std(resid39d)))

# ------------------------------------------------------------
# 2. Recover tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_cov_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_cov_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

tier_oof = np.array(["none"] * n_train_43b, dtype=object)
tier_oof[solver_cov_oof & ~direct_cov_oof] = "solver_only"
tier_oof[solver_cov_oof & direct_cov_oof] = "overlap"

tier_test = np.array(["none"] * n_test_43b, dtype=object)
tier_test[solver_cov_test & ~direct_cov_test] = "solver_only"
tier_test[solver_cov_test & direct_cov_test] = "overlap"

# ------------------------------------------------------------
# 3. Parse assessment names
# ------------------------------------------------------------

def clean_text_43b(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_assessment_43b(x):
    s = clean_text_43b(x)
    sl = s.lower()

    # Subject.
    if "math" in sl or "algebra" in sl or "geometry" in sl:
        subject = "math"
    elif "ela" in sl or "english" in sl or "reading" in sl or "literacy" in sl:
        subject = "ela"
    elif "science" in sl or "biology" in sl or "earth" in sl or "physics" in sl or "chemistry" in sl:
        subject = "science"
    elif "history" in sl or "global" in sl or "social" in sl or "government" in sl:
        subject = "social_studies"
    else:
        subject = "other"

    # Assessment type.
    if "regents" in sl:
        assess_type = "regents"
    elif "grade" in sl or re.search(r"\bgr(?:ade)?\s*[0-9]{1,2}\b", sl):
        assess_type = "grade_exam"
    else:
        assess_type = "other"

    # Grade parsing.
    grade = np.nan

    patterns = [
        r"grade\s*([0-9]{1,2})",
        r"\bgr\s*([0-9]{1,2})\b",
        r"\bg([0-9]{1,2})\b",
    ]

    for pat in patterns:
        m = re.search(pat, sl)
        if m:
            grade = float(m.group(1))
            break

    # Regents usually high-school if grade not explicit.
    if not np.isfinite(grade) and assess_type == "regents":
        grade = 11.0

    if np.isfinite(grade):
        if grade <= 5:
            grade_band = "elementary"
        elif grade <= 8:
            grade_band = "middle"
        else:
            grade_band = "high"
    else:
        grade_band = "unknown"

    return {
        "assessment_subject": subject,
        "assessment_type": assess_type,
        "assessment_grade": grade,
        "assessment_grade_band": grade_band,
    }

train_assess_parse = pd.DataFrame([parse_assessment_43b(x) for x in raw_train_te["ASSESSMENT_NAME"]])
test_assess_parse = pd.DataFrame([parse_assessment_43b(x) for x in raw_test_te["ASSESSMENT_NAME"]])

print("\nAssessment parse summary: train")
print("-------------------------------")
print(train_assess_parse[["assessment_subject", "assessment_type", "assessment_grade_band"]].value_counts(dropna=False).head(30).to_string())

print("\nAssessment parse summary: test")
print("------------------------------")
print(test_assess_parse[["assessment_subject", "assessment_type", "assessment_grade_band"]].value_counts(dropna=False).head(30).to_string())

# ------------------------------------------------------------
# 4. Audit possible grade/enrollment columns
# ------------------------------------------------------------

all_model_cols = list(X_train_proc_model.columns)

grade_col_candidates = []
enroll_col_candidates = []
other_school_structure_cols = []

for c in all_model_cols:
    cl = c.lower()

    if any(tok in cl for tok in ["enroll", "enrollment", "students", "student_count", "grade"]):
        if pd.api.types.is_numeric_dtype(X_train_proc_model[c]):
            if re.search(r"(grade|gr|g)[_\s-]*0?[0-9]{1,2}", cl) or re.search(r"[_\s-]k", cl):
                grade_col_candidates.append(c)
            elif "enroll" in cl or "student" in cl:
                enroll_col_candidates.append(c)
            else:
                other_school_structure_cols.append(c)

grade_col_candidates = sorted(set(grade_col_candidates))
enroll_col_candidates = sorted(set(enroll_col_candidates))
other_school_structure_cols = sorted(set(other_school_structure_cols))

print("\nGrade-like numeric columns")
print("--------------------------")
for c in grade_col_candidates[:100]:
    print(c)

print("\nEnrollment/student-count-like numeric columns")
print("---------------------------------------------")
for c in enroll_col_candidates[:100]:
    print(c)

print("\nOther school-structure numeric columns")
print("--------------------------------------")
for c in other_school_structure_cols[:100]:
    print(c)

# ------------------------------------------------------------
# 5. Attempt automatic mapping from parsed grade to columns
# ------------------------------------------------------------

def infer_grade_from_col_43b(c):
    cl = c.lower()

    # Avoid target/tested grade one-hot columns if not actual enrollment.
    # We only need diagnostics, so permissive matching is okay.
    m = re.search(r"(?:grade|gr|g)[_\s-]*0?([0-9]{1,2})", cl)
    if m:
        return int(m.group(1))

    return None

grade_to_cols = {}

for c in grade_col_candidates + enroll_col_candidates + other_school_structure_cols:
    g = infer_grade_from_col_43b(c)
    if g is not None and 0 <= g <= 12:
        grade_to_cols.setdefault(g, []).append(c)

print("\nInferred grade-to-column mapping")
print("--------------------------------")
for g in sorted(grade_to_cols.keys()):
    print(f"grade {g}: {grade_to_cols[g][:5]}")

# Pick one representative numeric column per grade, preferring enrollment/student words.
grade_rep_col = {}

for g, cols in grade_to_cols.items():
    preferred = [
        c for c in cols
        if ("enroll" in c.lower() or "student" in c.lower()) and pd.api.types.is_numeric_dtype(X_train_proc_model[c])
    ]
    if len(preferred) > 0:
        grade_rep_col[g] = preferred[0]
    else:
        numeric = [c for c in cols if pd.api.types.is_numeric_dtype(X_train_proc_model[c])]
        if len(numeric) > 0:
            grade_rep_col[g] = numeric[0]

print("\nRepresentative grade columns")
print("----------------------------")
for g in sorted(grade_rep_col.keys()):
    print(f"grade {g}: {grade_rep_col[g]}")

# ------------------------------------------------------------
# 6. Build diagnostic frame
# ------------------------------------------------------------

diag43b = pd.DataFrame({
    "row_index": np.arange(n_train_43b),
    "tier": tier_oof,
    "ASSESSMENT_NAME": raw_train_te["ASSESSMENT_NAME"].astype(str).to_numpy(),
    "SUBGROUP_NAME": raw_train_te["SUBGROUP_NAME"].astype(str).to_numpy(),
    "N_STUDENTS": pd.to_numeric(raw_train_te["N_STUDENTS"], errors="coerce").to_numpy(dtype=np.float64),
    TARGET_COL: y_43b,
    "pred39d": pred39d_oof,
    "resid39d": resid39d,
})

diag43b = pd.concat([diag43b, train_assess_parse.reset_index(drop=True)], axis=1)

# Add grade-representative enrollment for tested grade when possible.
tested_grade_enroll = np.full(n_train_43b, np.nan, dtype=np.float32)

for g, col in grade_rep_col.items():
    mask = diag43b["assessment_grade"].to_numpy(dtype=np.float64) == float(g)
    if mask.any():
        vals = pd.to_numeric(X_train_proc_model[col], errors="coerce").to_numpy(dtype=np.float32)
        tested_grade_enroll[mask] = vals[mask]

diag43b["tested_grade_enroll_feature"] = tested_grade_enroll
diag43b["tested_grade_enroll_missing"] = (~np.isfinite(tested_grade_enroll)).astype(int)

# Some useful ratios.
diag43b["log1p_N_STUDENTS"] = np.log1p(np.maximum(diag43b["N_STUDENTS"].to_numpy(dtype=np.float64), 0))

diag43b["N_to_tested_grade_enroll_ratio"] = np.where(
    np.isfinite(diag43b["tested_grade_enroll_feature"])
    & (diag43b["tested_grade_enroll_feature"] > 0)
    & np.isfinite(diag43b["N_STUDENTS"]),
    diag43b["N_STUDENTS"] / diag43b["tested_grade_enroll_feature"],
    np.nan,
)

# ------------------------------------------------------------
# 7. Residual summaries by parsed fields
# ------------------------------------------------------------

def summarize_group_43b(df, group_cols, min_n=100):
    out = (
        df.groupby(group_cols, dropna=False)
        .agg(
            n=("resid39d", "size"),
            mse_39d=("resid39d", lambda x: float(np.mean(np.asarray(x) ** 2))),
            mean_resid=("resid39d", "mean"),
            std_resid=("resid39d", "std"),
            mean_abs_resid=("resid39d", lambda x: float(np.mean(np.abs(np.asarray(x))))),
            mean_N=("N_STUDENTS", "mean"),
            mean_ratio=("N_to_tested_grade_enroll_ratio", "mean"),
        )
        .reset_index()
    )

    out = out[out["n"] >= min_n].sort_values("mse_39d", ascending=False).reset_index(drop=True)
    return out

summaries = {
    "tier_subject": summarize_group_43b(diag43b, ["tier", "assessment_subject"], min_n=300),
    "tier_type_band": summarize_group_43b(diag43b, ["tier", "assessment_type", "assessment_grade_band"], min_n=300),
    "tier_assessment": summarize_group_43b(diag43b, ["tier", "ASSESSMENT_NAME"], min_n=300),
    "tier_subgroup": summarize_group_43b(diag43b, ["tier", "SUBGROUP_NAME"], min_n=300),
}

print("\nResidual summary: tier × subject")
print("--------------------------------")
print(summaries["tier_subject"].head(30).to_string(index=False))

print("\nResidual summary: tier × type × grade band")
print("------------------------------------------")
print(summaries["tier_type_band"].head(30).to_string(index=False))

print("\nResidual summary: tier × assessment")
print("-----------------------------------")
print(summaries["tier_assessment"].head(30).to_string(index=False))

print("\nResidual summary: tier × subgroup")
print("---------------------------------")
print(summaries["tier_subgroup"].head(30).to_string(index=False))

# ------------------------------------------------------------
# 8. Test composition for parsed fields
# ------------------------------------------------------------

test_diag43b = pd.DataFrame({
    "row_index": np.arange(n_test_43b),
    "tier": tier_test,
    "ASSESSMENT_NAME": raw_test_te["ASSESSMENT_NAME"].astype(str).to_numpy(),
    "SUBGROUP_NAME": raw_test_te["SUBGROUP_NAME"].astype(str).to_numpy(),
    "N_STUDENTS": pd.to_numeric(raw_test_te["N_STUDENTS"], errors="coerce").to_numpy(dtype=np.float64),
})

test_diag43b = pd.concat([test_diag43b, test_assess_parse.reset_index(drop=True)], axis=1)

test_comp_subject = (
    test_diag43b.groupby(["tier", "assessment_subject"], dropna=False)
    .size()
    .reset_index(name="test_rows")
    .sort_values("test_rows", ascending=False)
)

test_comp_assessment = (
    test_diag43b.groupby(["tier", "ASSESSMENT_NAME"], dropna=False)
    .size()
    .reset_index(name="test_rows")
    .sort_values("test_rows", ascending=False)
)

print("\nTest composition: tier × subject")
print("--------------------------------")
print(test_comp_subject.head(30).to_string(index=False))

print("\nTest composition: tier × assessment")
print("-----------------------------------")
print(test_comp_assessment.head(30).to_string(index=False))

# ------------------------------------------------------------
# 9. Save artifacts
# ------------------------------------------------------------

parse_summary_train_path = "model_results/parse43b_train_diag.csv"
parse_summary_test_path = "model_results/parse43b_test_diag.csv"
grade_columns_path = "model_results/parse43b_grade_column_audit.csv"

diag43b.to_csv(parse_summary_train_path, index=False)
test_diag43b.to_csv(parse_summary_test_path, index=False)

grade_audit_rows = []
for g, cols in grade_to_cols.items():
    for c in cols:
        grade_audit_rows.append({
            "grade": g,
            "column": c,
            "is_representative": int(grade_rep_col.get(g) == c),
        })

pd.DataFrame(grade_audit_rows).to_csv(grade_columns_path, index=False)

for name, df in summaries.items():
    p = f"model_results/parse43b_summary_{name}.csv"
    df.to_csv(p, index=False)

test_comp_subject.to_csv("model_results/parse43b_test_comp_tier_subject.csv", index=False)
test_comp_assessment.to_csv("model_results/parse43b_test_comp_tier_assessment.csv", index=False)

print("\nSaved files")
print("-----------")
print(parse_summary_train_path)
print(parse_summary_test_path)
print(grade_columns_path)
for name in summaries:
    print(f"model_results/parse43b_summary_{name}.csv")
print("model_results/parse43b_test_comp_tier_subject.csv")
print("model_results/parse43b_test_comp_tier_assessment.csv")

print("\n43B-1 complete.")
print("Next step: if diagnostics show strong structure, build 43B-2 as a small prior/correction model.")

43B-1. Assessment / grade / enrollment structure diagnostic

Protected anchor
----------------
39D OOF MSE: 52.820927
39D public MSE: 30.752
Residual mean: -0.11891642957925797
Residual std:  7.266827583312988

Assessment parse summary: train
-------------------------------
assessment_subject  assessment_type  assessment_grade_band
math                other            unknown                  42899
ela                 other            unknown                  40555
math                regents          high                     18462
science             other            unknown                  12971
                    regents          high                     10017
social_studies      regents          high                      8966
other               regents          high                      6354
ela                 regents          high                      4697

Assessment parse summary: test
------------------------------
assessment_subject  assessment_type  assessment_grade_band


In [43]:
# ============================================================
# 43B-2. 39D-protected feature-level residual stack/calibrator
# ============================================================
#
# Current public-best:
#   39D public MSE = 30.752
#
# Goal:
#   Test feature-level stacking around 39D, not just prediction averaging.
#
# Target:
#   residual_39D = y - pred_39D
#
# Features:
#   - saved model differences vs 39D
#   - tier flags
#   - subgroup / assessment / parsed subject/type/grade band
#   - N_STUDENTS and demographics
#   - grade enrollment proxy from 43B parsing
#
# Prediction:
#   final = pred_39D + lambda_tier * predicted_residual
#
# Safety:
#   - no target encoding construction
#   - no torch
#   - no LightGBM
#   - small dense matrix only
#   - protects overlap tier by default
# ============================================================

import os
import re
import gc
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("43B-2. 39D-protected feature-level residual stack/calibrator")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required setup checks
# ------------------------------------------------------------

required_43b2 = ["raw_train_te", "raw_test_te", "X_train_proc_model", "X_test_proc_model", "y_train"]
missing_43b2 = [x for x in required_43b2 if x not in globals()]

if missing_43b2:
    raise ValueError(f"Missing required objects: {missing_43b2}. Rerun safe recovery/setup first.")

y_43b2 = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_43b2 = len(y_43b2)
n_test_43b2 = len(raw_test_te)

# ------------------------------------------------------------
# 2. Prediction artifact loaders
# ------------------------------------------------------------

def load_oof_pred_43b2(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    pred_col = None
    for c in ["pred_clipped", "prediction", "pred", "oof_pred"]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No OOF prediction column found in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF prediction in {path}")

    if TARGET_COL in df.columns:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        max_diff = float(np.nanmax(np.abs(y_file - y_ref)))
        if max_diff > 1e-5:
            raise ValueError(f"Target mismatch for {path}: max diff {max_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col

def load_test_pred_43b2(path):
    df = pd.read_csv(path)

    if len(df) != n_test_43b2:
        raise ValueError(f"Test row mismatch for {path}: got {len(df)}, expected {n_test_43b2}")

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No test prediction column found in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no global test_ids found.")

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test prediction in {path}")

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col

artifact_specs_43b2 = [
    ("anchor33a", "model_results/oof_seg33a_arcsine_adaptive.csv", "model_results/testpred_seg33a_arcsine_adaptive.csv"),
    ("knn35a", "model_results/oof_knn35a_local_residual_best.csv", "model_results/testpred_knn35a_local_residual_best.csv"),
    ("nn37b", "model_results/oof_nn37b38b_cpu_safe_rank1.csv", "model_results/testpred_nn37b38b_cpu_safe_rank1.csv"),
    ("step36a", "model_results/oof_step36a_residual_bins_best.csv", "model_results/testpred_step36a_residual_bins_best.csv"),
    ("blend33c", "model_results/oof_blend33c_33a_huber33b.csv", "model_results/testpred_blend33c_33a_huber33b.csv"),

    ("account39a", "model_results/oof_account39a_best.csv", "model_results/testpred_account39a_best.csv"),
    ("account39b", "model_results/oof_account39b_solver_best.csv", "model_results/testpred_account39b_solver_best.csv"),
    ("account39c", "model_results/oof_account39c_final_best.csv", "model_results/testpred_account39c_final_best.csv"),
    ("account39d", "model_results/oof_account39d_best.csv", "model_results/testpred_account39d_best.csv"),

    ("hte40a3", "model_results/oof_hte40a3_residual_best.csv", "model_results/testpred_hte40a3_residual_best.csv"),
    ("entity41a", "model_results/oof_entity41a_best.csv", "model_results/testpred_entity41a_best.csv"),
    ("int42a2", "model_results/oof_int42a2_best.csv", "model_results/testpred_int42a2_best.csv"),
    ("int42b3", "model_results/oof_int42b3_best.csv", "model_results/testpred_int42b3_best.csv"),
]

oof_preds_43b2 = {}
test_preds_43b2 = {}
test_ids_43b2 = None
loaded_rows_43b2 = []

for name, oof_path, test_path in artifact_specs_43b2:
    if not Path(oof_path).exists() or not Path(test_path).exists():
        print(f"Skipping missing artifact: {name}")
        continue

    try:
        oof_pred, oof_col = load_oof_pred_43b2(oof_path, y_43b2)
        test_pred, ids, test_col = load_test_pred_43b2(test_path)

        if test_ids_43b2 is None:
            test_ids_43b2 = ids

        oof_preds_43b2[name] = oof_pred
        test_preds_43b2[name] = test_pred

        mse = float(mean_squared_error(y_43b2, oof_pred))

        loaded_rows_43b2.append({
            "name": name,
            "oof_mse": mse,
            "oof_col": oof_col,
            "test_col": test_col,
        })

        print(f"Loaded {name:12s} | OOF MSE {mse:.6f}")

    except Exception as e:
        print(f"Skipping {name} because of error: {repr(e)}")

if "account39d" not in oof_preds_43b2:
    raise ValueError("account39d is required as protected anchor.")

loaded43b2 = pd.DataFrame(loaded_rows_43b2).sort_values("oof_mse").reset_index(drop=True)

print("\nLoaded prediction artifacts")
print("---------------------------")
display(loaded43b2)

pred39d_oof_43b2 = oof_preds_43b2["account39d"]
pred39d_test_43b2 = test_preds_43b2["account39d"]
resid39d_43b2 = (y_43b2 - pred39d_oof_43b2).astype(np.float32)
mse39d_43b2 = float(mean_squared_error(y_43b2, pred39d_oof_43b2))

print("\nProtected anchor")
print("----------------")
print(f"39D OOF MSE: {mse39d_43b2:.6f}")
print("39D public MSE: 30.752")

# ------------------------------------------------------------
# 3. Recover accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_cov_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
direct_cov_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)

solver_cov_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
solver_cov_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_cov_oof & solver_cov_oof
solver_only_oof = solver_cov_oof & ~direct_cov_oof
none_oof = ~(direct_cov_oof | solver_cov_oof)

overlap_test = direct_cov_test & solver_cov_test
solver_only_test = solver_cov_test & ~direct_cov_test
none_test = ~(direct_cov_test | solver_cov_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

print("\nTier coverage")
print("-------------")
for tier in ["overlap", "solver_only", "none"]:
    print(
        f"{tier:12s} train n={int(tier_masks_oof[tier].sum()):6d} "
        f"| test n={int(tier_masks_test[tier].sum()):6d} "
        f"| 39D tier MSE={mean_squared_error(y_43b2[tier_masks_oof[tier]], pred39d_oof_43b2[tier_masks_oof[tier]]):.6f}"
    )

# ------------------------------------------------------------
# 4. Assessment parser, improved for compact names like MATH3 / ELA4
# ------------------------------------------------------------

def clean_text_43b2(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_assessment_43b2(x):
    s = clean_text_43b2(x)
    sl = s.lower().replace("_", " ")

    if "math" in sl or "algebra" in sl or "geometry" in sl:
        subject = "math"
    elif "ela" in sl or "english" in sl or "reading" in sl or "literacy" in sl:
        subject = "ela"
    elif "science" in sl or "biology" in sl or "earth" in sl or "physics" in sl or "chemistry" in sl:
        subject = "science"
    elif "history" in sl or "global" in sl or "social" in sl or "government" in sl:
        subject = "social_studies"
    else:
        subject = "other"

    if "regents" in sl:
        assess_type = "regents"
    elif re.search(r"(math|ela|science)\s*[0-9]{1,2}\b", sl):
        assess_type = "grade_exam"
    elif re.search(r"combined\s*[0-9]{1,2}", sl):
        assess_type = "grade_exam"
    else:
        assess_type = "other"

    grade = np.nan

    patterns = [
        r"grade\s*([0-9]{1,2})",
        r"\bgr\s*([0-9]{1,2})\b",
        r"\bg([0-9]{1,2})\b",
        r"\bmath\s*([0-9]{1,2})\b",
        r"\bela\s*([0-9]{1,2})\b",
        r"\bscience\s*([0-9]{1,2})\b",
        r"\bcombined\s*([0-9]{1,2})",
        r"\bregents\s*math\s*([0-9]{1,2})\b",
        r"\bregents\s*science\s*([0-9]{1,2})\b",
    ]

    for pat in patterns:
        m = re.search(pat, sl)
        if m:
            grade = float(m.group(1))
            break

    if not np.isfinite(grade) and assess_type == "regents":
        grade = 11.0

    if np.isfinite(grade):
        if grade <= 5:
            grade_band = "elementary"
        elif grade <= 8:
            grade_band = "middle"
        else:
            grade_band = "high"
    else:
        grade_band = "unknown"

    return {
        "assessment_subject": subject,
        "assessment_type": assess_type,
        "assessment_grade": grade,
        "assessment_grade_band": grade_band,
    }

train_parse = pd.DataFrame([parse_assessment_43b2(x) for x in raw_train_te["ASSESSMENT_NAME"]])
test_parse = pd.DataFrame([parse_assessment_43b2(x) for x in raw_test_te["ASSESSMENT_NAME"]])

# ------------------------------------------------------------
# 5. Build small feature matrix
# ------------------------------------------------------------

X_train_meta = pd.DataFrame(index=np.arange(n_train_43b2))
X_test_meta = pd.DataFrame(index=np.arange(n_test_43b2))

# Protected anchor and model-difference features.
X_train_meta["pred39d"] = pred39d_oof_43b2.astype(np.float32)
X_test_meta["pred39d"] = pred39d_test_43b2.astype(np.float32)

for name in sorted(oof_preds_43b2.keys()):
    if name == "account39d":
        continue

    X_train_meta[f"diff_{name}_minus_39d"] = (oof_preds_43b2[name] - pred39d_oof_43b2).astype(np.float32)
    X_test_meta[f"diff_{name}_minus_39d"] = (test_preds_43b2[name] - pred39d_test_43b2).astype(np.float32)

# Tier flags.
X_train_meta["tier_overlap"] = overlap_oof.astype(np.float32)
X_train_meta["tier_solver_only"] = solver_only_oof.astype(np.float32)
X_train_meta["tier_none"] = none_oof.astype(np.float32)

X_test_meta["tier_overlap"] = overlap_test.astype(np.float32)
X_test_meta["tier_solver_only"] = solver_only_test.astype(np.float32)
X_test_meta["tier_none"] = none_test.astype(np.float32)

# Numeric covariates.
numeric_covariates = [
    "N_STUDENTS",
    "PERCENT_FREE_LUNCH",
    "PERCENT_REDUCED_LUNCH",
    "PERCENT_ECONOMICALLY_DISADVANTAGED",
    "PERCENT_ENGLISH_LANGUAGE_LEARNERS",
    "PERCENT_ENGLISH_LANGUAGE_LEANERS",
    "PERCENT_WITH_DISABILITIES",
    "PERCENT_STUDENTS_WITH_DISABILITIES",
    "PERCENT_HOMELESS",
    "PERCENT_MIGRANT",
    "PERCENT_FEMALE",
    "PERCENT_MALE",
    "ATTENDANCE_RATE",
]

added_numeric = []

for c in numeric_covariates:
    if c in X_train_proc_model.columns and c in X_test_proc_model.columns:
        if pd.api.types.is_numeric_dtype(X_train_proc_model[c]):
            if c not in added_numeric:
                X_train_meta[f"num_{c}"] = (
                    pd.to_numeric(X_train_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )
                X_test_meta[f"num_{c}"] = (
                    pd.to_numeric(X_test_proc_model[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .to_numpy(dtype=np.float32)
                )
                added_numeric.append(c)

if "num_N_STUDENTS" in X_train_meta.columns:
    X_train_meta["num_log1p_N_STUDENTS"] = np.log1p(np.maximum(X_train_meta["num_N_STUDENTS"].to_numpy(dtype=np.float32), 0))
    X_test_meta["num_log1p_N_STUDENTS"] = np.log1p(np.maximum(X_test_meta["num_N_STUDENTS"].to_numpy(dtype=np.float32), 0))

# Grade enrollment proxy.
grade_cols = {}
for g in range(1, 13):
    candidates = [
        f"GRADE_{g:02d}",
        f"GRADE_{g}",
    ]
    for c in candidates:
        if c in X_train_proc_model.columns and c in X_test_proc_model.columns:
            grade_cols[g] = c
            break

tested_grade_enroll_train = np.full(n_train_43b2, np.nan, dtype=np.float32)
tested_grade_enroll_test = np.full(n_test_43b2, np.nan, dtype=np.float32)

for g, c in grade_cols.items():
    mask_train = train_parse["assessment_grade"].to_numpy(dtype=np.float64) == float(g)
    mask_test = test_parse["assessment_grade"].to_numpy(dtype=np.float64) == float(g)

    vals_train = pd.to_numeric(X_train_proc_model[c], errors="coerce").to_numpy(dtype=np.float32)
    vals_test = pd.to_numeric(X_test_proc_model[c], errors="coerce").to_numpy(dtype=np.float32)

    tested_grade_enroll_train[mask_train] = vals_train[mask_train]
    tested_grade_enroll_test[mask_test] = vals_test[mask_test]

X_train_meta["tested_grade_enroll"] = tested_grade_enroll_train
X_test_meta["tested_grade_enroll"] = tested_grade_enroll_test

X_train_meta["tested_grade_enroll_missing"] = (~np.isfinite(tested_grade_enroll_train)).astype(np.float32)
X_test_meta["tested_grade_enroll_missing"] = (~np.isfinite(tested_grade_enroll_test)).astype(np.float32)

X_train_meta["N_to_tested_grade_enroll_ratio"] = np.where(
    np.isfinite(tested_grade_enroll_train)
    & (tested_grade_enroll_train > 0)
    & ("num_N_STUDENTS" in X_train_meta.columns),
    X_train_meta["num_N_STUDENTS"].to_numpy(dtype=np.float32) / tested_grade_enroll_train,
    np.nan,
)

X_test_meta["N_to_tested_grade_enroll_ratio"] = np.where(
    np.isfinite(tested_grade_enroll_test)
    & (tested_grade_enroll_test > 0)
    & ("num_N_STUDENTS" in X_test_meta.columns),
    X_test_meta["num_N_STUDENTS"].to_numpy(dtype=np.float32) / tested_grade_enroll_test,
    np.nan,
)

# Categorical one-hot features.
cat_train = pd.DataFrame({
    "SUBGROUP_NAME": raw_train_te["SUBGROUP_NAME"].astype("string").fillna("<NA>").astype(str),
    "ASSESSMENT_NAME": raw_train_te["ASSESSMENT_NAME"].astype("string").fillna("<NA>").astype(str),
    "DISTRICT_TYPE": raw_train_te["DISTRICT_TYPE"].astype("string").fillna("<NA>").astype(str) if "DISTRICT_TYPE" in raw_train_te.columns else "<NA>",
    "assessment_subject": train_parse["assessment_subject"].astype(str),
    "assessment_type": train_parse["assessment_type"].astype(str),
    "assessment_grade_band": train_parse["assessment_grade_band"].astype(str),
})

cat_test = pd.DataFrame({
    "SUBGROUP_NAME": raw_test_te["SUBGROUP_NAME"].astype("string").fillna("<NA>").astype(str),
    "ASSESSMENT_NAME": raw_test_te["ASSESSMENT_NAME"].astype("string").fillna("<NA>").astype(str),
    "DISTRICT_TYPE": raw_test_te["DISTRICT_TYPE"].astype("string").fillna("<NA>").astype(str) if "DISTRICT_TYPE" in raw_test_te.columns else "<NA>",
    "assessment_subject": test_parse["assessment_subject"].astype(str),
    "assessment_type": test_parse["assessment_type"].astype(str),
    "assessment_grade_band": test_parse["assessment_grade_band"].astype(str),
})

cat_all = pd.concat([cat_train, cat_test], axis=0).reset_index(drop=True)
cat_dummies_all = pd.get_dummies(cat_all, dtype=np.float32)

cat_dummies_train = cat_dummies_all.iloc[:n_train_43b2].reset_index(drop=True)
cat_dummies_test = cat_dummies_all.iloc[n_train_43b2:].reset_index(drop=True)

X_train_meta = pd.concat([X_train_meta.reset_index(drop=True), cat_dummies_train], axis=1)
X_test_meta = pd.concat([X_test_meta.reset_index(drop=True), cat_dummies_test], axis=1)

X_train_meta = X_train_meta.loc[:, ~X_train_meta.columns.duplicated()]
X_test_meta = X_test_meta[X_train_meta.columns]

print("\nMeta-feature matrix")
print("-------------------")
print("Train:", X_train_meta.shape)
print("Test: ", X_test_meta.shape)
print("Approx train MB:", round(X_train_meta.memory_usage(deep=True).sum() / 1024**2, 2))

# ------------------------------------------------------------
# 6. OOF residual calibrator training
# ------------------------------------------------------------

configs_43b2 = [
    {
        "name": "meta43b2_ridge_solver_public",
        "model_type": "ridge",
        "alpha": 5000.0,
        "weight_scheme": "solver_public",
    },
    {
        "name": "meta43b2_ridge_balanced",
        "model_type": "ridge",
        "alpha": 10000.0,
        "weight_scheme": "balanced",
    },
    {
        "name": "meta43b2_hgb_solver_public",
        "model_type": "hgb",
        "weight_scheme": "solver_public",
        "params": {
            "max_iter": 160,
            "learning_rate": 0.04,
            "max_leaf_nodes": 15,
            "max_depth": 4,
            "min_samples_leaf": 120,
            "l2_regularization": 5.0,
            "random_state": RANDOM_STATE,
        },
    },
]

def make_weights_43b2(idx, scheme):
    w = np.ones(len(idx), dtype=np.float32)

    ov = overlap_oof[idx]
    so = solver_only_oof[idx]
    no = none_oof[idx]

    if scheme == "solver_public":
        w[ov] = 0.05
        w[so] = 2.00
        w[no] = 0.25
        return w

    if scheme == "balanced":
        w[ov] = 0.10
        w[so] = 1.00
        w[no] = 0.75
        return w

    raise ValueError(f"Unknown weight scheme: {scheme}")

folds43b2 = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_43b2))
)

resid_oof_pred = {cfg["name"]: np.full(n_train_43b2, np.nan, dtype=np.float32) for cfg in configs_43b2}
resid_test_sum = {cfg["name"]: np.zeros(n_test_43b2, dtype=np.float64) for cfg in configs_43b2}

fold_rows = []

for cfg in configs_43b2:
    print("\n" + "=" * 90)
    print("Training", cfg["name"])
    print("=" * 90)

    for fold_num, (tr_idx, va_idx) in enumerate(folds43b2, start=1):
        X_tr = X_train_meta.iloc[tr_idx]
        X_va = X_train_meta.iloc[va_idx]
        X_te = X_test_meta

        sw = make_weights_43b2(tr_idx, cfg["weight_scheme"])

        if cfg["model_type"] == "ridge":
            imputer = SimpleImputer(strategy="median")
            scaler = StandardScaler()

            X_tr_i = imputer.fit_transform(X_tr)
            X_va_i = imputer.transform(X_va)
            X_te_i = imputer.transform(X_te)

            X_tr_s = scaler.fit_transform(X_tr_i)
            X_va_s = scaler.transform(X_va_i)
            X_te_s = scaler.transform(X_te_i)

            model = Ridge(alpha=float(cfg["alpha"]))
            model.fit(X_tr_s, resid39d_43b2[tr_idx], sample_weight=sw)

            pred_va_resid = model.predict(X_va_s).astype(np.float32)
            pred_te_resid = model.predict(X_te_s).astype(np.float32)

            del X_tr_i, X_va_i, X_te_i, X_tr_s, X_va_s, X_te_s

        elif cfg["model_type"] == "hgb":
            imputer = SimpleImputer(strategy="median")

            X_tr_i = imputer.fit_transform(X_tr)
            X_va_i = imputer.transform(X_va)
            X_te_i = imputer.transform(X_te)

            model = HistGradientBoostingRegressor(**cfg["params"])
            model.fit(X_tr_i, resid39d_43b2[tr_idx], sample_weight=sw)

            pred_va_resid = model.predict(X_va_i).astype(np.float32)
            pred_te_resid = model.predict(X_te_i).astype(np.float32)

            del X_tr_i, X_va_i, X_te_i

        else:
            raise ValueError(cfg["model_type"])

        resid_oof_pred[cfg["name"]][va_idx] = pred_va_resid
        resid_test_sum[cfg["name"]] += pred_te_resid.astype(np.float64)

        pred_lam1 = np.clip(pred39d_oof_43b2[va_idx] + pred_va_resid, 0, 100)

        row = {
            "config": cfg["name"],
            "fold": fold_num,
            "mse_39d": float(mean_squared_error(y_43b2[va_idx], pred39d_oof_43b2[va_idx])),
            "mse_lam1": float(mean_squared_error(y_43b2[va_idx], pred_lam1)),
        }
        row["gain_lam1_vs_39d"] = row["mse_39d"] - row["mse_lam1"]

        fold_rows.append(row)

        print(
            f"fold {fold_num} | lam1 MSE {row['mse_lam1']:.6f} "
            f"| gain {row['gain_lam1_vs_39d']:.6f}"
        )

        del model
        gc.collect()

# ------------------------------------------------------------
# 7. Lambda scan, protecting overlap by default
# ------------------------------------------------------------

lambda_grid = np.unique(
    np.concatenate([
        np.linspace(-1.0, 1.5, 626),
        np.array([0.0, 0.05, 0.10, 0.25, 0.50, 0.75, 1.0]),
    ])
)

def opt_lambda_for_mask(resid_pred, mask):
    mask = np.asarray(mask, dtype=bool)
    if int(mask.sum()) == 0:
        return 0.0, np.nan

    y_m = y_43b2[mask].astype(np.float64)
    base_m = pred39d_oof_43b2[mask].astype(np.float64)
    r_m = resid_pred[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid:
        pred = np.clip(base_m + float(lam) * r_m, 0, 100)
        mse = float(mean_squared_error(y_m, pred))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam, best_mse

def make_pred_from_lams(resid_pred, lams, is_test=False):
    base = pred39d_test_43b2.copy() if is_test else pred39d_oof_43b2.copy()

    masks = {
        "overlap": overlap_test if is_test else overlap_oof,
        "solver_only": solver_only_test if is_test else solver_only_oof,
        "none": none_test if is_test else none_oof,
    }

    pred = base.astype(np.float64)

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        if int(mask.sum()) > 0:
            pred[mask] = base[mask] + lam * resid_pred[mask]

    return np.clip(pred, 0, 100).astype(np.float32)

screen_rows = []

for cfg in configs_43b2:
    name = cfg["name"]
    r_oof = resid_oof_pred[name]

    if not np.isfinite(r_oof).all():
        print("Skipping incomplete", name)
        continue

    # We force overlap lambda to 0, because overlap is already solved.
    lam_solver, mse_solver = opt_lambda_for_mask(r_oof, solver_only_oof)
    lam_none, mse_none = opt_lambda_for_mask(r_oof, none_oof)

    lams_solver_only = {
        "overlap": 0.0,
        "solver_only": lam_solver,
        "none": 0.0,
    }

    pred_solver_only = make_pred_from_lams(r_oof, lams_solver_only, is_test=False)
    mse_solver_only_total = float(mean_squared_error(y_43b2, pred_solver_only))

    screen_rows.append({
        "candidate_type": "solver_only_correction",
        "config": name,
        "lambda_overlap": 0.0,
        "lambda_solver_only": lam_solver,
        "lambda_none": 0.0,
        "oof_mse": mse_solver_only_total,
        "gain_vs_39d": mse39d_43b2 - mse_solver_only_total,
        "solver_only_mse": float(mean_squared_error(y_43b2[solver_only_oof], pred_solver_only[solver_only_oof])),
        "none_mse": float(mean_squared_error(y_43b2[none_oof], pred_solver_only[none_oof])),
    })

    lams_solver_none = {
        "overlap": 0.0,
        "solver_only": lam_solver,
        "none": lam_none,
    }

    pred_solver_none = make_pred_from_lams(r_oof, lams_solver_none, is_test=False)
    mse_solver_none_total = float(mean_squared_error(y_43b2, pred_solver_none))

    screen_rows.append({
        "candidate_type": "solver_and_none_correction",
        "config": name,
        "lambda_overlap": 0.0,
        "lambda_solver_only": lam_solver,
        "lambda_none": lam_none,
        "oof_mse": mse_solver_none_total,
        "gain_vs_39d": mse39d_43b2 - mse_solver_none_total,
        "solver_only_mse": float(mean_squared_error(y_43b2[solver_only_oof], pred_solver_none[solver_only_oof])),
        "none_mse": float(mean_squared_error(y_43b2[none_oof], pred_solver_none[none_oof])),
    })

screen43b2 = pd.DataFrame(screen_rows).sort_values("oof_mse").reset_index(drop=True)

best43b2 = screen43b2.iloc[0]
best_name = best43b2["config"]
best_resid_oof = resid_oof_pred[best_name]
best_resid_test = (resid_test_sum[best_name] / N_SPLITS).astype(np.float32)

best_lams = {
    "overlap": float(best43b2["lambda_overlap"]),
    "solver_only": float(best43b2["lambda_solver_only"]),
    "none": float(best43b2["lambda_none"]),
}

final_oof = make_pred_from_lams(best_resid_oof, best_lams, is_test=False)
final_test = make_pred_from_lams(best_resid_test, best_lams, is_test=True)

best_mse = float(mean_squared_error(y_43b2, final_oof))
best_gain = mse39d_43b2 - best_mse

# ------------------------------------------------------------
# 8. Fold / tier diagnostics and save
# ------------------------------------------------------------

fold_diag_rows = []

for fold_num, (_, va_idx) in enumerate(folds43b2, start=1):
    row = {
        "fold": fold_num,
        "mse_39d": float(mean_squared_error(y_43b2[va_idx], pred39d_oof_43b2[va_idx])),
        "mse_43b2": float(mean_squared_error(y_43b2[va_idx], final_oof[va_idx])),
    }
    row["gain_vs_39d"] = row["mse_39d"] - row["mse_43b2"]

    for tier, mask_full in {
        "overlap": overlap_oof,
        "solver_only": solver_only_oof,
        "none": none_oof,
    }.items():
        mask_fold = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask_fold.sum())

        if int(mask_fold.sum()) > 0:
            row[f"mse39d_{tier}"] = float(mean_squared_error(
                y_43b2[va_idx][mask_fold],
                pred39d_oof_43b2[va_idx][mask_fold],
            ))
            row[f"mse43b2_{tier}"] = float(mean_squared_error(
                y_43b2[va_idx][mask_fold],
                final_oof[va_idx][mask_fold],
            ))
            row[f"gain_{tier}"] = row[f"mse39d_{tier}"] - row[f"mse43b2_{tier}"]

    fold_diag_rows.append(row)

fold_diag43b2 = pd.DataFrame(fold_diag_rows)

tier_perf_rows = []

for tier, mask in {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}.items():
    tier_perf_rows.append({
        "tier": tier,
        "n_rows": int(mask.sum()),
        "mse_39d": float(mean_squared_error(y_43b2[mask], pred39d_oof_43b2[mask])),
        "mse_43b2": float(mean_squared_error(y_43b2[mask], final_oof[mask])),
        "gain_vs_39d": float(mean_squared_error(y_43b2[mask], pred39d_oof_43b2[mask]) - mean_squared_error(y_43b2[mask], final_oof[mask])),
    })

tier_perf43b2 = pd.DataFrame(tier_perf_rows)

screen_path = "model_results/meta43b2_screen.csv"
fold_metrics_path = "model_results/meta43b2_fold_metrics.csv"
fold_diag_path = "model_results/meta43b2_best_fold_diag.csv"
tier_perf_path = "model_results/meta43b2_tier_performance.csv"
oof_path = "model_results/oof_meta43b2_best.csv"
testpred_path = "model_results/testpred_meta43b2_best.csv"
submission_path = "submission_meta43b2_best.csv"

screen43b2.to_csv(screen_path, index=False)
pd.DataFrame(fold_rows).to_csv(fold_metrics_path, index=False)
fold_diag43b2.to_csv(fold_diag_path, index=False)
tier_perf43b2.to_csv(tier_perf_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_43b2),
    TARGET_COL: y_43b2,
    "pred_39d": pred39d_oof_43b2,
    "residual_39d": resid39d_43b2,
    "residual_pred": best_resid_oof,
    "pred_clipped": final_oof,
    "overlap": overlap_oof.astype(int),
    "solver_only": solver_only_oof.astype(int),
    "none": none_oof.astype(int),
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids_43b2,
    "pred_39d": pred39d_test_43b2,
    "residual_pred": best_resid_test,
    TARGET_COL: final_test,
    "overlap": overlap_test.astype(int),
    "solver_only": solver_only_test.astype(int),
    "none": none_test.astype(int),
}).to_csv(testpred_path, index=False)

pd.DataFrame({
    ID_COL: test_ids_43b2,
    TARGET_COL: final_test,
}).to_csv(submission_path, index=False)

sub = pd.read_csv(submission_path)
assert sub.shape == (n_test_43b2, 2)
assert list(sub.columns) == [ID_COL, TARGET_COL]
assert sub[ID_COL].notna().all()
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

print("\n" + "=" * 90)
print("43B-2 feature-level stack/calibrator complete")
print("=" * 90)

print("\nProtected anchor")
print("----------------")
print(f"39D OOF MSE: {mse39d_43b2:.6f}")
print("39D public MSE: 30.752")

print("\nTop candidates")
print("--------------")
display(screen43b2)

print("\nBest candidate")
print("--------------")
print(best43b2.to_string())
print(f"\nBest 43B-2 OOF MSE: {best_mse:.6f}")
print(f"Gain vs 39D:         {best_gain:.6f}")

print("\nFold diagnostics")
print("----------------")
print(fold_diag43b2.to_string(index=False))
print("Min fold gain:", float(fold_diag43b2["gain_vs_39d"].min()))

print("\nTier performance")
print("----------------")
print(tier_perf43b2.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path)
print(fold_metrics_path)
print(fold_diag_path)
print(tier_perf_path)
print(oof_path)
print(testpred_path)
print(submission_path)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path)
print("Shape:", sub.shape)
print(sub[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
if best_gain >= 1.5 and fold_diag43b2["gain_vs_39d"].min() >= 0:
    print("43B-2 has meaningful stable gain over 39D. Consider submission.")
elif best_gain >= 0.5 and fold_diag43b2["gain_vs_39d"].min() >= 0:
    print("43B-2 has modest stable gain. Save artifact; submit only if using a speculative slot.")
elif best_gain > 0:
    print("43B-2 has only marginal or unstable gain. Do not submit under current threshold.")
else:
    print("43B-2 does not improve over 39D. Keep 39D protected.")

43B-2. 39D-protected feature-level residual stack/calibrator
Loaded anchor33a    | OOF MSE 77.049866
Loaded knn35a       | OOF MSE 76.453072
Loaded nn37b        | OOF MSE 76.849442
Loaded step36a      | OOF MSE 76.954796
Loaded blend33c     | OOF MSE 76.859200
Loaded account39a   | OOF MSE 55.642780
Loaded account39b   | OOF MSE 53.720547
Loaded account39c   | OOF MSE 53.226631
Loaded account39d   | OOF MSE 52.820927
Loaded hte40a3      | OOF MSE 49.689606
Loaded entity41a    | OOF MSE 52.760006
Loaded int42a2      | OOF MSE 53.123951
Loaded int42b3      | OOF MSE 53.073307

Loaded prediction artifacts
---------------------------


,name,oof_mse,oof_col,test_col
0,hte40a3,49.689606,pred_clipped,PERCENT_PROFICIENT
1,entity41a,52.760006,pred_clipped,PERCENT_PROFICIENT
2,account39d,52.820927,pred_clipped,PERCENT_PROFICIENT
3,int42b3,53.073307,pred_clipped,PERCENT_PROFICIENT
4,int42a2,53.123951,pred_clipped,PERCENT_PROFICIENT
5,account39c,53.226631,pred_clipped,PERCENT_PROFICIENT
6,account39b,53.720547,pred_clipped,PERCENT_PROFICIENT
7,account39a,55.642780,pred_clipped,PERCENT_PROFICIENT
8,knn35a,76.453072,pred_clipped,PERCENT_PROFICIENT
9,nn37b,76.849442,pred_clipped,PERCENT_PROFICIENT



Protected anchor
----------------
39D OOF MSE: 52.820927
39D public MSE: 30.752

Tier coverage
-------------
overlap      train n= 53786 | test n= 27298 | 39D tier MSE=0.595325
solver_only  train n= 27807 | test n= 17807 | 39D tier MSE=64.304314
none         train n= 63328 | test n=  3202 | 39D tier MSE=92.135094

Meta-feature matrix
-------------------
Train: (144921, 87)
Test:  (48307, 87)
Approx train MB: 48.1

Training meta43b2_ridge_solver_public
fold 1 | lam1 MSE 49.020737 | gain 4.046127
fold 2 | lam1 MSE 51.542343 | gain -0.692780
fold 3 | lam1 MSE 50.496567 | gain 3.314045
fold 4 | lam1 MSE 48.806881 | gain 3.879032
fold 5 | lam1 MSE 50.204582 | gain 3.487083

Training meta43b2_ridge_balanced
fold 1 | lam1 MSE 48.823097 | gain 4.243767
fold 2 | lam1 MSE 51.842590 | gain -0.993027
fold 3 | lam1 MSE 50.539749 | gain 3.270863
fold 4 | lam1 MSE 48.780174 | gain 3.905739
fold 5 | lam1 MSE 49.933628 | gain 3.758038

Training meta43b2_hgb_solver_public
fold 1 | lam1 MSE 48.853138 | 

,candidate_type,config,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_39d,solver_only_mse,none_mse
0,solver_and_none_correction,meta43b2_hgb_solver_public,0.0,0.816,0.848,49.768635,3.052292,63.268414,85.605034
1,solver_and_none_correction,meta43b2_ridge_balanced,0.0,0.760,0.856,49.837719,2.983208,63.534943,85.646095
2,solver_and_none_correction,meta43b2_ridge_solver_public,0.0,0.792,0.872,49.891781,2.929146,63.394299,85.831573
3,solver_only_correction,meta43b2_hgb_solver_public,0.0,0.816,0.000,52.622158,0.198769,63.268414,92.135094
4,solver_only_correction,meta43b2_ridge_solver_public,0.0,0.792,0.000,52.646313,0.174614,63.394299,92.135094
5,solver_only_correction,meta43b2_ridge_balanced,0.0,0.760,0.000,52.673302,0.147625,63.534943,92.135094



Best candidate
--------------
candidate_type        solver_and_none_correction
config                meta43b2_hgb_solver_public
lambda_overlap                               0.0
lambda_solver_only                         0.816
lambda_none                                0.848
oof_mse                                49.768635
gain_vs_39d                             3.052292
solver_only_mse                        63.268414
none_mse                               85.605034

Best 43B-2 OOF MSE: 49.768635
Gain vs 39D:         3.052292

Fold diagnostics
----------------
 fold   mse_39d  mse_43b2  gain_vs_39d  n_overlap  mse39d_overlap  mse43b2_overlap  gain_overlap  n_solver_only  mse39d_solver_only  mse43b2_solver_only  gain_solver_only  n_none  mse39d_none  mse43b2_none  gain_none
    1 53.066864 49.107651     3.959213      10837        0.488835         0.488835           0.0           5496           62.327160            61.012974          1.314186   12652    94.079628     85.580177   8.49945

39D public:        30.752   current best


int42b3 public:    30.923   close


entity41a public:  31.452   worse, but not disastrous


meta43b2 public:   31.944   clearly worse

39D remains protected best.


int42b3 is the only “near miss” worth investigating.


entity41a is not worth chasing yet.


meta43b2 confirms residual/meta models are public-risky.

In [44]:
# ============================================================
# 44A. 39D vs int42b3 disagreement audit + conservative hybrid
# ============================================================
#
# Motivation:
#   39D public:     30.752
#   int42b3 public: 30.923
#
# int42b3 is close publicly despite worse OOF.
# It improves the overlap tier OOF but hurts solver_only.
#
# Goal:
#   Test simple protected hybrids:
#     overlap: 39D / int42b3 / blend
#     solver_only: mostly 39D
#     none: 39D
#
# This is small and safe.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("44A. 39D vs int42b3 disagreement audit + conservative hybrid")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load OOF/test predictions
# ------------------------------------------------------------

def load_oof(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(f"No prediction column in {path}")

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [c for c in df.columns if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])]
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test ids.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

oof39d_df, y44, pred39d_oof = load_oof("model_results/oof_account39d_best.csv")
test39d_df, test_ids44, pred39d_test = load_test("model_results/testpred_account39d_best.csv")

oof42b_df, y42_file, pred42b_oof = load_oof("model_results/oof_int42b3_best.csv")
test42b_df, _, pred42b_test = load_test("model_results/testpred_int42b3_best.csv")

assert len(y44) == len(pred42b_oof)
assert np.max(np.abs(y44 - y42_file)) < 1e-5

n_train = len(y44)
n_test = len(pred39d_test)

mse39d = mean_squared_error(y44, pred39d_oof)
mse42b = mean_squared_error(y44, pred42b_oof)

print("\nBase models")
print("-----------")
print(f"39D OOF MSE:    {mse39d:.6f}")
print(f"int42b3 OOF MSE:{mse42b:.6f}")
print("39D public:     30.752")
print("int42b3 public: 30.923")

# ------------------------------------------------------------
# 2. Recover tier flags
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}
tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

# ------------------------------------------------------------
# 3. Tier comparison and disagreement audit
# ------------------------------------------------------------

rows = []

for tier, mask in tier_masks_oof.items():
    diff = pred42b_oof[mask] - pred39d_oof[mask]

    rows.append({
        "tier": tier,
        "train_rows": int(mask.sum()),
        "test_rows": int(tier_masks_test[tier].sum()),
        "mse_39d": float(mean_squared_error(y44[mask], pred39d_oof[mask])),
        "mse_int42b3": float(mean_squared_error(y44[mask], pred42b_oof[mask])),
        "gain_int42b3_vs_39d": float(mean_squared_error(y44[mask], pred39d_oof[mask]) - mean_squared_error(y44[mask], pred42b_oof[mask])),
        "mean_diff_42b_minus_39d": float(np.mean(diff)),
        "std_diff": float(np.std(diff)),
        "p50_abs_diff": float(np.percentile(np.abs(diff), 50)),
        "p90_abs_diff": float(np.percentile(np.abs(diff), 90)),
        "p99_abs_diff": float(np.percentile(np.abs(diff), 99)),
    })

tier_compare44 = pd.DataFrame(rows)

print("\nTier comparison")
print("---------------")
print(tier_compare44.to_string(index=False))

# Optional subgroup-level comparison.
if "raw_train_te" in globals() and "SUBGROUP_NAME" in raw_train_te.columns:
    subgroup = pd.Series(raw_train_te["SUBGROUP_NAME"]).astype(str).to_numpy()
    sub_rows = []

    for tier, tier_mask in tier_masks_oof.items():
        for sg in sorted(pd.Series(subgroup[tier_mask]).unique()):
            mask = tier_mask & (subgroup == sg)
            if mask.sum() < 300:
                continue

            sub_rows.append({
                "tier": tier,
                "subgroup": sg,
                "n": int(mask.sum()),
                "mse_39d": float(mean_squared_error(y44[mask], pred39d_oof[mask])),
                "mse_int42b3": float(mean_squared_error(y44[mask], pred42b_oof[mask])),
                "gain_int42b3_vs_39d": float(mean_squared_error(y44[mask], pred39d_oof[mask]) - mean_squared_error(y44[mask], pred42b_oof[mask])),
            })

    subgroup_compare44 = pd.DataFrame(sub_rows).sort_values("gain_int42b3_vs_39d", ascending=False)
else:
    subgroup_compare44 = pd.DataFrame()

print("\nSubgroup comparison, positive means int42b3 beats 39D")
print("-----------------------------------------------------")
display(subgroup_compare44.head(30))

# ------------------------------------------------------------
# 4. Simple protected hybrid scan
# ------------------------------------------------------------

lambda_grid = np.unique(np.concatenate([
    np.linspace(0.0, 1.0, 501),
    np.array([0.0, 0.25, 0.5, 0.75, 1.0])
]))

def make_hybrid_oof(lam_overlap, lam_solver, lam_none):
    pred = pred39d_oof.copy().astype(np.float64)

    pred[overlap_oof] = pred39d_oof[overlap_oof] + lam_overlap * (pred42b_oof[overlap_oof] - pred39d_oof[overlap_oof])
    pred[solver_only_oof] = pred39d_oof[solver_only_oof] + lam_solver * (pred42b_oof[solver_only_oof] - pred39d_oof[solver_only_oof])
    pred[none_oof] = pred39d_oof[none_oof] + lam_none * (pred42b_oof[none_oof] - pred39d_oof[none_oof])

    return np.clip(pred, 0, 100).astype(np.float32)

def make_hybrid_test(lam_overlap, lam_solver, lam_none):
    pred = pred39d_test.copy().astype(np.float64)

    pred[overlap_test] = pred39d_test[overlap_test] + lam_overlap * (pred42b_test[overlap_test] - pred39d_test[overlap_test])
    pred[solver_only_test] = pred39d_test[solver_only_test] + lam_solver * (pred42b_test[solver_only_test] - pred39d_test[solver_only_test])
    pred[none_test] = pred39d_test[none_test] + lam_none * (pred42b_test[none_test] - pred39d_test[none_test])

    return np.clip(pred, 0, 100).astype(np.float32)

screen = []

# Intentionally conservative families.
candidate_lambda_sets = []

# Overlap-only borrowing from int42b3.
for lo in lambda_grid:
    candidate_lambda_sets.append(("overlap_only", lo, 0.0, 0.0))

# Overlap plus tiny solver adjustment.
for lo in np.linspace(0.0, 1.0, 251):
    for ls in [0.0, 0.05, 0.10, 0.15, 0.20]:
        candidate_lambda_sets.append(("overlap_plus_tiny_solver", float(lo), float(ls), 0.0))

# Full tier independent, but still no none borrowing because int42b3 leaves none as 39C-ish, not useful.
for lo in np.linspace(0.0, 1.0, 151):
    for ls in np.linspace(0.0, 0.5, 101):
        candidate_lambda_sets.append(("overlap_solver_grid", float(lo), float(ls), 0.0))

seen = set()
unique_candidate_lambda_sets = []

for item in candidate_lambda_sets:
    key = (item[0], round(item[1], 6), round(item[2], 6), round(item[3], 6))
    if key not in seen:
        seen.add(key)
        unique_candidate_lambda_sets.append(item)

for family, lo, ls, ln in unique_candidate_lambda_sets:
    pred = make_hybrid_oof(lo, ls, ln)
    mse = float(mean_squared_error(y44, pred))

    row = {
        "family": family,
        "lambda_overlap": lo,
        "lambda_solver_only": ls,
        "lambda_none": ln,
        "oof_mse": mse,
        "gain_vs_39d": mse39d - mse,
    }

    for tier, mask in tier_masks_oof.items():
        row[f"mse_{tier}"] = float(mean_squared_error(y44[mask], pred[mask]))

    screen.append(row)

screen44 = pd.DataFrame(screen).sort_values("oof_mse").reset_index(drop=True)

best44 = screen44.iloc[0]
best_oof44 = make_hybrid_oof(best44["lambda_overlap"], best44["lambda_solver_only"], best44["lambda_none"])
best_test44 = make_hybrid_test(best44["lambda_overlap"], best44["lambda_solver_only"], best44["lambda_none"])

# ------------------------------------------------------------
# 5. Fold diagnostics
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

fold_rows = []

for fold_num, (_, va_idx) in enumerate(folds, start=1):
    row = {
        "fold": fold_num,
        "mse_39d": float(mean_squared_error(y44[va_idx], pred39d_oof[va_idx])),
        "mse_hybrid": float(mean_squared_error(y44[va_idx], best_oof44[va_idx])),
    }
    row["gain_vs_39d"] = row["mse_39d"] - row["mse_hybrid"]

    for tier, mask_full in tier_masks_oof.items():
        mask = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask.sum())
        if mask.sum():
            row[f"mse39d_{tier}"] = float(mean_squared_error(y44[va_idx][mask], pred39d_oof[va_idx][mask]))
            row[f"msehybrid_{tier}"] = float(mean_squared_error(y44[va_idx][mask], best_oof44[va_idx][mask]))
            row[f"gain_{tier}"] = row[f"mse39d_{tier}"] - row[f"msehybrid_{tier}"]

    fold_rows.append(row)

fold_diag44 = pd.DataFrame(fold_rows)

# ------------------------------------------------------------
# 6. Save artifacts
# ------------------------------------------------------------

screen_path = "model_results/hybrid44a_39d_int42b3_screen.csv"
tier_compare_path = "model_results/hybrid44a_tier_compare.csv"
subgroup_compare_path = "model_results/hybrid44a_subgroup_compare.csv"
fold_diag_path = "model_results/hybrid44a_best_fold_diag.csv"
oof_path = "model_results/oof_hybrid44a_39d_int42b3_best.csv"
test_path = "model_results/testpred_hybrid44a_39d_int42b3_best.csv"
submission_path = "submission_hybrid44a_39d_int42b3_best.csv"

screen44.to_csv(screen_path, index=False)
tier_compare44.to_csv(tier_compare_path, index=False)
subgroup_compare44.to_csv(subgroup_compare_path, index=False)
fold_diag44.to_csv(fold_diag_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y44,
    "pred_39d": pred39d_oof,
    "pred_int42b3": pred42b_oof,
    "pred_clipped": best_oof44,
    "overlap": overlap_oof.astype(int),
    "solver_only": solver_only_oof.astype(int),
    "none": none_oof.astype(int),
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids44,
    "pred_39d": pred39d_test,
    "pred_int42b3": pred42b_test,
    TARGET_COL: best_test44,
    "overlap": overlap_test.astype(int),
    "solver_only": solver_only_test.astype(int),
    "none": none_test.astype(int),
}).to_csv(test_path, index=False)

pd.DataFrame({
    ID_COL: test_ids44,
    TARGET_COL: best_test44,
}).to_csv(submission_path, index=False)

sub = pd.read_csv(submission_path)
assert sub.shape == (n_test, 2)
assert list(sub.columns) == [ID_COL, TARGET_COL]
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 7. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("44A hybrid audit complete")
print("=" * 90)

print("\nTier comparison")
print("---------------")
print(tier_compare44.to_string(index=False))

print("\nTop 20 hybrid candidates")
print("------------------------")
display(screen44.head(20))

print("\nBest hybrid")
print("-----------")
print(best44.to_string())

print("\nFold diagnostics")
print("----------------")
print(fold_diag44.to_string(index=False))
print("Min fold gain:", float(fold_diag44["gain_vs_39d"].min()))

print("\nSaved files")
print("-----------")
print(screen_path)
print(tier_compare_path)
print(subgroup_compare_path)
print(fold_diag_path)
print(oof_path)
print(test_path)
print(submission_path)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path)
print("Shape:", sub.shape)
print(sub[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
if float(best44["gain_vs_39d"]) >= 0.5 and float(fold_diag44["gain_vs_39d"].min()) >= 0:
    print("Hybrid gives stable OOF gain over 39D. Consider if using speculative slot.")
elif float(best44["gain_vs_39d"]) > 0:
    print("Hybrid gives only small OOF gain. Save artifact; do not submit unless you want a low-risk probe.")
else:
    print("Hybrid does not improve OOF. Keep 39D.")

44A. 39D vs int42b3 disagreement audit + conservative hybrid

Base models
-----------
39D OOF MSE:    52.820927
int42b3 OOF MSE:53.073307
39D public:     30.752
int42b3 public: 30.923

Tier comparison
---------------
       tier  train_rows  test_rows   mse_39d  mse_int42b3  gain_int42b3_vs_39d  mean_diff_42b_minus_39d  std_diff  p50_abs_diff  p90_abs_diff  p99_abs_diff
    overlap       53786      27298  0.595325     0.507233             0.088092                -0.006248  0.296541      0.027443      0.111807      0.959676
solver_only       27807      17807 64.304314    65.261948            -0.957634                -0.005122  1.181424      0.439041      1.566211      4.700071
       none       63328       3202 92.135094    92.366982            -0.231888                 0.046192  0.627395      0.128893      1.065721      2.164416

Subgroup comparison, positive means int42b3 beats 39D
-----------------------------------------------------


,tier,subgroup,n,mse_39d,mse_int42b3,gain_int42b3_vs_39d
4,overlap,Not Economically Disadvantaged,8718,1.791970,1.394579,0.397392
6,solver_only,Economically Disadvantaged,4913,63.574211,63.221199,0.353012
1,overlap,Economically Disadvantaged,8706,1.244122,1.176057,0.068065
2,overlap,Female,10199,0.211041,0.176194,0.034848
3,overlap,Male,10189,0.195781,0.163880,0.031901
0,overlap,All Students,15974,0.088843,0.088804,0.000039
12,none,Female,13100,86.754730,86.755379,-0.000648
13,none,Male,13353,81.401337,81.409882,-0.008545
14,none,Not Economically Disadvantaged,10981,111.773598,111.865974,-0.092377
10,none,All Students,14376,101.315857,101.620293,-0.304436



44A hybrid audit complete

Tier comparison
---------------
       tier  train_rows  test_rows   mse_39d  mse_int42b3  gain_int42b3_vs_39d  mean_diff_42b_minus_39d  std_diff  p50_abs_diff  p90_abs_diff  p99_abs_diff
    overlap       53786      27298  0.595325     0.507233             0.088092                -0.006248  0.296541      0.027443      0.111807      0.959676
solver_only       27807      17807 64.304314    65.261948            -0.957634                -0.005122  1.181424      0.439041      1.566211      4.700071
       none       63328       3202 92.135094    92.366982            -0.231888                 0.046192  0.627395      0.128893      1.065721      2.164416

Top 20 hybrid candidates
------------------------


,family,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_39d,mse_overlap,mse_solver_only,mse_none
0,overlap_solver_grid,0.993333,0.155,0.0,52.781635,0.039291,0.507237,64.269936,92.135094
1,overlap_solver_grid,1.000000,0.160,0.0,52.781635,0.039291,0.507233,64.269943,92.135094
2,overlap_solver_grid,1.000000,0.155,0.0,52.781635,0.039291,0.507233,64.269936,92.135094
3,overlap_solver_grid,0.993333,0.160,0.0,52.781639,0.039288,0.507237,64.269943,92.135094
4,overlap_solver_grid,0.986667,0.155,0.0,52.781639,0.039288,0.507250,64.269936,92.135094
5,overlap_solver_grid,0.986667,0.160,0.0,52.781643,0.039284,0.507250,64.269943,92.135094
6,overlap_plus_tiny_solver,1.000000,0.150,0.0,52.781647,0.039280,0.507233,64.269997,92.135094
7,overlap_solver_grid,0.980000,0.160,0.0,52.781647,0.039280,0.507270,64.269943,92.135094
8,overlap_solver_grid,0.980000,0.155,0.0,52.781647,0.039280,0.507270,64.269936,92.135094
9,overlap_solver_grid,0.993333,0.150,0.0,52.781647,0.039280,0.507237,64.269997,92.135094



Best hybrid
-----------
family                overlap_solver_grid
lambda_overlap                   0.993333
lambda_solver_only                  0.155
lambda_none                           0.0
oof_mse                         52.781635
gain_vs_39d                      0.039291
mse_overlap                      0.507237
mse_solver_only                 64.269936
mse_none                        92.135094

Fold diagnostics
----------------
 fold   mse_39d  mse_hybrid  gain_vs_39d  n_overlap  mse39d_overlap  msehybrid_overlap  gain_overlap  n_solver_only  mse39d_solver_only  msehybrid_solver_only  gain_solver_only  n_none  mse39d_none  msehybrid_none  gain_none
    1 53.066864   53.056427     0.010437      10837        0.488835           0.418884      0.069952           5496           62.327160              62.410019         -0.082859   12652    94.079628       94.079628        0.0
    2 50.849564   50.803844     0.045719      10622        0.460458           0.394795      0.065664           5

### Public Leaderboard Check: 44A Conservative Accounting Hybrid

The 44A conservative hybrid of 39D and the integer/count solver produced a public leaderboard MSE of `30.602`, improving over the previous protected anchor 39D public MSE of `30.752`.

This model blended 39D with `int42b3` in a highly constrained way:

- nearly full `int42b3` on overlap rows,
- only a small `int42b3` contribution on solver-only rows,
- no change on none-tier rows.

The OOF gain was very small (`0.039291`), but the public improvement was meaningful because the change targeted public-heavy accounting-covered rows rather than the training-heavy none tier. This confirms that conservative accounting-structure improvements are more reliable than broad residual/meta corrections.

Current protected public-best model:

`submission_hybrid44a_39d_int42b3_best.csv`

Public MSE: `30.602`

In [45]:
# ============================================================
# 44B. Subgroup-specific conservative hybrid of 44A components
# ============================================================
#
# Protected public-best:
#   44A public MSE = 30.602
#
# Inputs:
#   39D and int42b3
#
# Idea:
#   44A used one lambda for overlap and one tiny lambda for solver_only.
#   44B tunes lambda by tier x SUBGROUP_NAME, with shrinkage and protections.
#
# This is still a conservative accounting hybrid:
#   - no learned residual model
#   - no HTE/meta/entity correction
#   - no change to none tier
#
# Submission only if fold-stable and structurally sensible.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("44B. Subgroup-specific conservative hybrid of 39D and int42b3")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load artifacts
# ------------------------------------------------------------

def load_oof(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(f"No prediction column in {path}")

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [c for c in df.columns if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])]
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

oof39d_df, y44b, pred39d_oof = load_oof("model_results/oof_account39d_best.csv")
test39d_df, test_ids44b, pred39d_test = load_test("model_results/testpred_account39d_best.csv")

oof42b_df, y42b_file, pred42b_oof = load_oof("model_results/oof_int42b3_best.csv")
test42b_df, _, pred42b_test = load_test("model_results/testpred_int42b3_best.csv")

oof44a_df, y44a_file, pred44a_oof = load_oof("model_results/oof_hybrid44a_39d_int42b3_best.csv")
test44a_df, _, pred44a_test = load_test("model_results/testpred_hybrid44a_39d_int42b3_best.csv")

assert np.max(np.abs(y44b - y42b_file)) < 1e-5
assert np.max(np.abs(y44b - y44a_file)) < 1e-5

n_train = len(y44b)
n_test = len(pred39d_test)

mse39d = float(mean_squared_error(y44b, pred39d_oof))
mse42b = float(mean_squared_error(y44b, pred42b_oof))
mse44a = float(mean_squared_error(y44b, pred44a_oof))

print("\nBase models")
print("-----------")
print(f"39D OOF MSE:    {mse39d:.6f} | public 30.752")
print(f"int42b3 OOF MSE:{mse42b:.6f} | public 30.923")
print(f"44A OOF MSE:    {mse44a:.6f} | public 30.602")

# ------------------------------------------------------------
# 2. Tiers and subgroups
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)
direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

tier_oof = np.array(["none"] * n_train, dtype=object)
tier_oof[solver_oof & ~direct_oof] = "solver_only"
tier_oof[solver_oof & direct_oof] = "overlap"

tier_test = np.array(["none"] * n_test, dtype=object)
tier_test[solver_test & ~direct_test] = "solver_only"
tier_test[solver_test & direct_test] = "overlap"

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("raw_train_te/raw_test_te needed for subgroup-specific hybrid.")

subgroup_oof = pd.Series(raw_train_te["SUBGROUP_NAME"]).astype(str).to_numpy()
subgroup_test = pd.Series(raw_test_te["SUBGROUP_NAME"]).astype(str).to_numpy()

cell_oof = np.array([f"{t}||{s}" for t, s in zip(tier_oof, subgroup_oof)], dtype=object)
cell_test = np.array([f"{t}||{s}" for t, s in zip(tier_test, subgroup_test)], dtype=object)

# ------------------------------------------------------------
# 3. Lambda tuning helpers
# ------------------------------------------------------------

lambda_grid = np.unique(np.concatenate([
    np.linspace(0.0, 1.0, 501),
    np.array([0.0, 0.155, 0.5, 0.75, 0.993333, 1.0])
]))

def best_lambda_for_mask(mask, default_lam=None):
    if int(mask.sum()) == 0:
        return 0.0, np.nan

    y = y44b[mask].astype(np.float64)
    p0 = pred39d_oof[mask].astype(np.float64)
    p1 = pred42b_oof[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid:
        pred = np.clip(p0 + float(lam) * (p1 - p0), 0, 100)
        mse = float(mean_squared_error(y, pred))

        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam, best_mse

def apply_cell_lambdas(lam_by_cell, fallback_by_tier, is_test=False):
    if is_test:
        base = pred39d_test.copy().astype(np.float64)
        alt = pred42b_test.astype(np.float64)
        cells = cell_test
        tiers = tier_test
    else:
        base = pred39d_oof.copy().astype(np.float64)
        alt = pred42b_oof.astype(np.float64)
        cells = cell_oof
        tiers = tier_oof

    pred = base.copy()

    for i in range(len(pred)):
        t = str(tiers[i])

        if t == "none":
            lam = 0.0
        else:
            lam = lam_by_cell.get(str(cells[i]), fallback_by_tier.get(t, 0.0))

        pred[i] = base[i] + lam * (alt[i] - base[i])

    return np.clip(pred, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 4. OOF-safe cell lambda fitting
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

min_rows_grid = [300, 700, 1200]
shrink_grid = [0.0, 0.25, 0.50, 0.75]

screen_rows = []
candidate_store = {}
choice_rows = []

# Base fallback lambdas from 44A.
fallback_44a = {
    "overlap": 0.993333,
    "solver_only": 0.155,
    "none": 0.0,
}

for min_rows in min_rows_grid:
    for shrink in shrink_grid:
        pred_oof = np.full(n_train, np.nan, dtype=np.float32)
        pred_test_sum = np.zeros(n_test, dtype=np.float64)
        all_choice_rows = []

        for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
            lam_by_cell = {}

            # Fit fallback by tier on train fold, but restricted:
            # overlap can vary; solver_only conservative; none fixed 0.
            fallback_by_tier = dict(fallback_44a)

            for tier in ["overlap", "solver_only"]:
                mask_t = np.isin(np.arange(n_train), tr_idx) & (tier_oof == tier)
                lam_t, mse_t = best_lambda_for_mask(mask_t)

                # Conservative shrink back to 44A fallback.
                lam_t_shrunk = (1.0 - shrink) * lam_t + shrink * fallback_44a[tier]

                if tier == "solver_only":
                    # Hard cap: don't let solver_only borrow too much from int42b3.
                    lam_t_shrunk = min(lam_t_shrunk, 0.30)

                fallback_by_tier[tier] = float(lam_t_shrunk)

            # Fit cell lambdas.
            train_cells = sorted(pd.Series(cell_oof[tr_idx]).unique().tolist())

            for cell in train_cells:
                tier = cell.split("||", 1)[0]

                if tier == "none":
                    continue

                mask = np.isin(np.arange(n_train), tr_idx) & (cell_oof == cell)

                if int(mask.sum()) < min_rows:
                    continue

                lam_cell, mse_cell = best_lambda_for_mask(mask)

                fallback = fallback_by_tier.get(tier, 0.0)

                lam_cell_shrunk = (1.0 - shrink) * lam_cell + shrink * fallback

                if tier == "solver_only":
                    # Hard cap because int42b3 hurts many solver-only cells.
                    lam_cell_shrunk = min(lam_cell_shrunk, 0.50)

                lam_by_cell[cell] = float(lam_cell_shrunk)

                all_choice_rows.append({
                    "fold": fold_num,
                    "min_rows": min_rows,
                    "shrink": shrink,
                    "cell": cell,
                    "tier": tier,
                    "lambda_cell_raw": float(lam_cell),
                    "lambda_cell_shrunk": float(lam_cell_shrunk),
                    "n_train": int(mask.sum()),
                    "mse_cell_best": float(mse_cell),
                    "fallback_lambda": float(fallback),
                })

            # Apply to val/test.
            pred_val_full = apply_cell_lambdas(lam_by_cell, fallback_by_tier, is_test=False)
            pred_test_fold = apply_cell_lambdas(lam_by_cell, fallback_by_tier, is_test=True)

            pred_oof[va_idx] = pred_val_full[va_idx]
            pred_test_sum += pred_test_fold

        pred_test = (pred_test_sum / N_SPLITS).astype(np.float32)

        mse = float(mean_squared_error(y44b, pred_oof))

        row = {
            "min_rows": min_rows,
            "shrink": shrink,
            "oof_mse": mse,
            "gain_vs_44a": mse44a - mse,
            "gain_vs_39d": mse39d - mse,
        }

        for tier in ["overlap", "solver_only", "none"]:
            mask = tier_oof == tier
            row[f"mse_{tier}"] = float(mean_squared_error(y44b[mask], pred_oof[mask]))

        screen_rows.append(row)

        key = f"min{min_rows}_shrink{str(shrink).replace('.', 'p')}"
        candidate_store[key] = {
            "oof": pred_oof,
            "test": pred_test,
            "choices": pd.DataFrame(all_choice_rows),
            "row": row,
        }

screen44b = pd.DataFrame(screen_rows).sort_values("oof_mse").reset_index(drop=True)

best_row = screen44b.iloc[0]
best_key = f"min{int(best_row['min_rows'])}_shrink{str(float(best_row['shrink'])).replace('.', 'p')}"
best_oof = candidate_store[best_key]["oof"]
best_test = candidate_store[best_key]["test"]
best_choices = candidate_store[best_key]["choices"]

# ------------------------------------------------------------
# 5. Fold diagnostics
# ------------------------------------------------------------

fold_rows = []

for fold_num, (_, va_idx) in enumerate(folds, start=1):
    row = {
        "fold": fold_num,
        "mse_44a": float(mean_squared_error(y44b[va_idx], pred44a_oof[va_idx])),
        "mse_44b": float(mean_squared_error(y44b[va_idx], best_oof[va_idx])),
    }
    row["gain_vs_44a"] = row["mse_44a"] - row["mse_44b"]

    for tier in ["overlap", "solver_only", "none"]:
        mask = (tier_oof[va_idx] == tier)
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse44a_{tier}"] = float(mean_squared_error(y44b[va_idx][mask], pred44a_oof[va_idx][mask]))
            row[f"mse44b_{tier}"] = float(mean_squared_error(y44b[va_idx][mask], best_oof[va_idx][mask]))
            row[f"gain_{tier}"] = row[f"mse44a_{tier}"] - row[f"mse44b_{tier}"]

    fold_rows.append(row)

fold_diag44b = pd.DataFrame(fold_rows)

# ------------------------------------------------------------
# 6. Save artifacts
# ------------------------------------------------------------

screen_path = "model_results/hybrid44b_cell_lambda_screen.csv"
choices_path = "model_results/hybrid44b_best_cell_choices.csv"
fold_path = "model_results/hybrid44b_best_fold_diag.csv"
oof_path = "model_results/oof_hybrid44b_best.csv"
test_path = "model_results/testpred_hybrid44b_best.csv"
submission_path = "submission_hybrid44b_best.csv"

screen44b.to_csv(screen_path, index=False)
best_choices.to_csv(choices_path, index=False)
fold_diag44b.to_csv(fold_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y44b,
    "pred_39d": pred39d_oof,
    "pred_int42b3": pred42b_oof,
    "pred_44a": pred44a_oof,
    "pred_clipped": best_oof,
    "tier": tier_oof,
    "subgroup": subgroup_oof,
    "cell": cell_oof,
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids44b,
    "pred_39d": pred39d_test,
    "pred_int42b3": pred42b_test,
    "pred_44a": pred44a_test,
    TARGET_COL: best_test,
    "tier": tier_test,
    "subgroup": subgroup_test,
    "cell": cell_test,
}).to_csv(test_path, index=False)

pd.DataFrame({
    ID_COL: test_ids44b,
    TARGET_COL: best_test,
}).to_csv(submission_path, index=False)

sub = pd.read_csv(submission_path)
assert sub.shape == (n_test, 2)
assert list(sub.columns) == [ID_COL, TARGET_COL]
assert sub[ID_COL].notna().all()
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 7. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("44B subgroup-specific accounting hybrid complete")
print("=" * 90)

print("\nReference")
print("---------")
print(f"39D OOF MSE: {mse39d:.6f} | public 30.752")
print(f"44A OOF MSE: {mse44a:.6f} | public 30.602")

print("\nTop 20 candidates")
print("-----------------")
display(screen44b.head(20))

print("\nBest 44B")
print("--------")
print(best_row.to_string())

print("\nFold diagnostics")
print("----------------")
print(fold_diag44b.to_string(index=False))
print("Min fold gain vs 44A:", float(fold_diag44b["gain_vs_44a"].min()))

print("\nBest cell choices")
print("-----------------")
if len(best_choices) > 0:
    display(best_choices.sort_values(["tier", "cell", "fold"]).head(50))
else:
    print("No segment choices beyond fallback.")

print("\nSaved files")
print("-----------")
print(screen_path)
print(choices_path)
print(fold_path)
print(oof_path)
print(test_path)
print(submission_path)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path)
print("Shape:", sub.shape)
print(sub[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
if float(best_row["gain_vs_44a"]) >= 0.05 and float(fold_diag44b["gain_vs_44a"].min()) >= 0:
    print("44B gives small stable improvement over 44A. Reasonable low-risk probe if submission slot remains.")
elif float(best_row["gain_vs_44a"]) > 0:
    print("44B gives marginal/unstable gain. Save artifact; 44A remains protected.")
else:
    print("44B does not improve over 44A. Keep 44A protected.")

44B. Subgroup-specific conservative hybrid of 39D and int42b3

Base models
-----------
39D OOF MSE:    52.820927 | public 30.752
int42b3 OOF MSE:53.073307 | public 30.923
44A OOF MSE:    52.781635 | public 30.602

44B subgroup-specific accounting hybrid complete

Reference
---------
39D OOF MSE: 52.820927 | public 30.752
44A OOF MSE: 52.781635 | public 30.602

Top 20 candidates
-----------------


,min_rows,shrink,oof_mse,gain_vs_44a,gain_vs_39d,mse_overlap,mse_solver_only,mse_none
0,300,0.25,52.780037,0.001598,0.040890,0.507021,64.262032,92.135094
1,700,0.25,52.780037,0.001598,0.040890,0.507021,64.262032,92.135094
2,1200,0.25,52.780037,0.001598,0.040890,0.507021,64.262032,92.135094
3,300,0.50,52.780579,0.001057,0.040348,0.507041,64.264786,92.135094
4,700,0.50,52.780579,0.001057,0.040348,0.507041,64.264786,92.135094
5,1200,0.50,52.780579,0.001057,0.040348,0.507041,64.264786,92.135094
6,300,0.00,52.780624,0.001011,0.040302,0.507054,64.265030,92.135094
7,700,0.00,52.780624,0.001011,0.040302,0.507054,64.265030,92.135094
8,1200,0.00,52.780624,0.001011,0.040302,0.507054,64.265030,92.135094
9,300,0.75,52.780689,0.000946,0.040237,0.507113,64.265228,92.135094



Best 44B
--------
min_rows           300.000000
shrink               0.250000
oof_mse             52.780037
gain_vs_44a          0.001598
gain_vs_39d          0.040890
mse_overlap          0.507021
mse_solver_only     64.262032
mse_none            92.135094

Fold diagnostics
----------------
 fold   mse_44a   mse_44b  gain_vs_44a  n_overlap  mse44a_overlap  mse44b_overlap  gain_overlap  n_solver_only  mse44a_solver_only  mse44b_solver_only  gain_solver_only  n_none  mse44a_none  mse44b_none  gain_none
    1 53.056427 53.054913     0.001514      10837        0.418884        0.418953     -0.000069           5496           62.410019           62.401917          0.008102   12652    94.079628    94.079628        0.0
    2 50.803844 50.813717    -0.009872      10622        0.394795        0.394440      0.000355           5623           61.309315           61.360851         -0.051537   12739    88.198669    88.198669        0.0
    3 53.775944 53.784950    -0.009007      10838        0.45424

,fold,min_rows,shrink,cell,tier,lambda_cell_raw,lambda_cell_shrunk,n_train,mse_cell_best,fallback_lambda
0,1,300,0.25,overlap||All Students,overlap,0.526,0.644083,12775,0.088187,0.998333
10,2,300,0.25,overlap||All Students,overlap,0.462,0.596083,12778,0.087606,0.998333
20,3,300,0.25,overlap||All Students,overlap,0.598,0.692833,12799,0.088242,0.977333
30,4,300,0.25,overlap||All Students,overlap,0.484,0.612583,12677,0.088390,0.998333
40,5,300,0.25,overlap||All Students,overlap,0.462,0.594958,12867,0.087175,0.993833
1,1,300,0.25,overlap||Economically Disadvantaged,overlap,0.832,0.873583,6941,1.253839,0.998333
11,2,300,0.25,overlap||Economically Disadvantaged,overlap,0.974,0.980083,7004,1.261213,0.998333
21,3,300,0.25,overlap||Economically Disadvantaged,overlap,0.880,0.904333,6992,1.277042,0.977333
31,4,300,0.25,overlap||Economically Disadvantaged,overlap,0.858,0.893083,6956,1.070761,0.998333
41,5,300,0.25,overlap||Economically Disadvantaged,overlap,0.890,0.915958,6931,1.009354,0.993833



Saved files
-----------
model_results/hybrid44b_cell_lambda_screen.csv
model_results/hybrid44b_best_cell_choices.csv
model_results/hybrid44b_best_fold_diag.csv
model_results/oof_hybrid44b_best.csv
model_results/testpred_hybrid44b_best.csv
submission_hybrid44b_best.csv

Submission validation
---------------------
File: submission_hybrid44b_best.csv
Shape: (48307, 2)
count    48307.000000
mean        54.161816
std         25.825435
min          0.000897
25%         33.347084
50%         52.813282
75%         75.268853
max        100.000000
Name: PERCENT_PROFICIENT, dtype: float64

Decision rule
-------------
44B gives marginal/unstable gain. Save artifact; 44A remains protected.


### 44B. Subgroup-Specific Conservative Accounting Hybrid

This section tested whether the successful 44A hybrid could be improved by tuning the 39D/int42b3 blend separately by `tier × SUBGROUP_NAME`.

The protected public-best model at this point was 44A:

- 39D public MSE: `30.752`
- 44A public MSE: `30.602`

The best 44B candidate had:

- OOF MSE: `52.780037`
- Gain versus 44A: `0.001598`
- Gain versus 39D: `0.040890`

The gain over 44A was essentially zero. Fold-level results were not stable: folds 2 and 3 were slightly worse than 44A, while the other folds showed only tiny gains.

Tier performance was also almost unchanged:

- overlap MSE: `0.507237 → 0.507021`
- solver-only MSE: `64.269936 → 64.262032`
- none MSE: unchanged at `92.135094`

Conclusion: subgroup-specific lambda tuning does not reveal meaningful additional signal beyond 44A. The added complexity is not justified. The current protected submission remains `submission_hybrid44a_39d_int42b3_best.csv` with public MSE `30.602`.

In [48]:
# ============================================================
# 44C. Full-train refit of 44B tier x subgroup accounting hybrid
# ============================================================
#
# Protected public-best:
#   44B public MSE = 30.556
#
# Motivation:
#   44B used OOF fold-specific tier x subgroup lambdas and averaged test predictions.
#   Since that configuration transferred publicly, refit the same lambda rule
#   on the full training set and apply to test.
#
# Inputs:
#   39D prediction
#   int42b3 prediction
#   tier labels: overlap / solver_only / none
#   subgroup labels
#
# Output:
#   submission_hybrid44c_fullfit_best.csv
#
# Important:
#   Full-train fit MSE is not honest validation. Use it only to generate
#   the final refit artifact after 44B has been publicly validated.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("44C. Full-train refit of 44B tier x subgroup accounting hybrid")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load artifacts
# ------------------------------------------------------------

def load_oof(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(f"No prediction column in {path}")

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

oof39d_df, y44c, pred39d_oof = load_oof("model_results/oof_account39d_best.csv")
test39d_df, test_ids44c, pred39d_test = load_test("model_results/testpred_account39d_best.csv")

oof42b_df, y42b_file, pred42b_oof = load_oof("model_results/oof_int42b3_best.csv")
test42b_df, _, pred42b_test = load_test("model_results/testpred_int42b3_best.csv")

oof44a_df, y44a_file, pred44a_oof = load_oof("model_results/oof_hybrid44a_39d_int42b3_best.csv")
test44a_df, _, pred44a_test = load_test("model_results/testpred_hybrid44a_39d_int42b3_best.csv")

oof44b_df, y44b_file, pred44b_oof = load_oof("model_results/oof_hybrid44b_best.csv")
test44b_df, _, pred44b_test = load_test("model_results/testpred_hybrid44b_best.csv")

assert np.max(np.abs(y44c - y42b_file)) < 1e-5
assert np.max(np.abs(y44c - y44a_file)) < 1e-5
assert np.max(np.abs(y44c - y44b_file)) < 1e-5

n_train = len(y44c)
n_test = len(pred39d_test)

mse39d = float(mean_squared_error(y44c, pred39d_oof))
mse42b = float(mean_squared_error(y44c, pred42b_oof))
mse44a = float(mean_squared_error(y44c, pred44a_oof))
mse44b = float(mean_squared_error(y44c, pred44b_oof))

print("\nReference models")
print("----------------")
print(f"39D OOF MSE:    {mse39d:.6f} | public 30.752")
print(f"int42b3 OOF MSE:{mse42b:.6f} | public 30.923")
print(f"44A OOF MSE:    {mse44a:.6f} | public 30.602")
print(f"44B OOF MSE:    {mse44b:.6f} | public 30.556")

# ------------------------------------------------------------
# 2. Recover tiers and subgroups
# ------------------------------------------------------------

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("raw_train_te and raw_test_te are required.")

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

tier_oof = np.array(["none"] * n_train, dtype=object)
tier_oof[solver_oof & ~direct_oof] = "solver_only"
tier_oof[solver_oof & direct_oof] = "overlap"

tier_test = np.array(["none"] * n_test, dtype=object)
tier_test[solver_test & ~direct_test] = "solver_only"
tier_test[solver_test & direct_test] = "overlap"

subgroup_oof = pd.Series(raw_train_te["SUBGROUP_NAME"]).astype(str).to_numpy()
subgroup_test = pd.Series(raw_test_te["SUBGROUP_NAME"]).astype(str).to_numpy()

cell_oof = np.array([f"{t}||{s}" for t, s in zip(tier_oof, subgroup_oof)], dtype=object)
cell_test = np.array([f"{t}||{s}" for t, s in zip(tier_test, subgroup_test)], dtype=object)

print("\nTier coverage")
print("-------------")
for tier in ["overlap", "solver_only", "none"]:
    print(
        f"{tier:12s} train={int((tier_oof == tier).sum()):6d} "
        f"test={int((tier_test == tier).sum()):6d}"
    )

# ------------------------------------------------------------
# 3. Lambda helpers
# ------------------------------------------------------------

lambda_grid = np.unique(
    np.concatenate([
        np.linspace(0.0, 1.0, 1001),
        np.array([0.0, 0.155, 0.25, 0.5, 0.75, 0.993333, 1.0])
    ])
)

fallback_44a = {
    "overlap": 0.993333,
    "solver_only": 0.155,
    "none": 0.0,
}

def best_lambda_for_mask(mask):
    mask = np.asarray(mask, dtype=bool)

    if int(mask.sum()) == 0:
        return 0.0, np.nan

    y = y44c[mask].astype(np.float64)
    p0 = pred39d_oof[mask].astype(np.float64)
    p1 = pred42b_oof[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid:
        pred = np.clip(p0 + float(lam) * (p1 - p0), 0, 100)
        mse = float(mean_squared_error(y, pred))

        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam, best_mse

def fit_full_lambdas(min_rows=300, shrink=0.25, solver_cell_cap=0.50, solver_fallback_cap=0.30):
    """
    Refit the 44B rule on full training data.
    """
    lambda_by_cell = {}
    rows = []

    fallback_by_tier = dict(fallback_44a)

    for tier in ["overlap", "solver_only"]:
        mask_tier = tier_oof == tier
        raw_lam, raw_mse = best_lambda_for_mask(mask_tier)

        shrunk_lam = (1.0 - shrink) * raw_lam + shrink * fallback_44a[tier]

        if tier == "solver_only":
            shrunk_lam = min(shrunk_lam, solver_fallback_cap)

        fallback_by_tier[tier] = float(shrunk_lam)

        rows.append({
            "level": "tier_fallback",
            "cell": tier,
            "tier": tier,
            "subgroup": "",
            "n_train": int(mask_tier.sum()),
            "lambda_raw": float(raw_lam),
            "lambda_shrunk": float(shrunk_lam),
            "mse_best": float(raw_mse),
            "fallback_used": fallback_44a[tier],
            "min_rows": min_rows,
            "shrink": shrink,
            "solver_cell_cap": solver_cell_cap,
            "solver_fallback_cap": solver_fallback_cap,
        })

    for cell in sorted(pd.Series(cell_oof).unique().tolist()):
        tier, subgroup = str(cell).split("||", 1)

        if tier == "none":
            continue

        mask_cell = cell_oof == cell

        if int(mask_cell.sum()) < int(min_rows):
            continue

        raw_lam, raw_mse = best_lambda_for_mask(mask_cell)

        fallback = fallback_by_tier.get(tier, 0.0)
        shrunk_lam = (1.0 - shrink) * raw_lam + shrink * fallback

        if tier == "solver_only":
            shrunk_lam = min(shrunk_lam, solver_cell_cap)

        lambda_by_cell[str(cell)] = float(shrunk_lam)

        rows.append({
            "level": "cell",
            "cell": str(cell),
            "tier": tier,
            "subgroup": subgroup,
            "n_train": int(mask_cell.sum()),
            "lambda_raw": float(raw_lam),
            "lambda_shrunk": float(shrunk_lam),
            "mse_best": float(raw_mse),
            "fallback_used": float(fallback),
            "min_rows": min_rows,
            "shrink": shrink,
            "solver_cell_cap": solver_cell_cap,
            "solver_fallback_cap": solver_fallback_cap,
        })

    return lambda_by_cell, fallback_by_tier, pd.DataFrame(rows)

def apply_lambdas(lambda_by_cell, fallback_by_tier, is_test=False):
    if is_test:
        p0 = pred39d_test.astype(np.float64)
        p1 = pred42b_test.astype(np.float64)
        cells = cell_test
        tiers = tier_test
    else:
        p0 = pred39d_oof.astype(np.float64)
        p1 = pred42b_oof.astype(np.float64)
        cells = cell_oof
        tiers = tier_oof

    pred = p0.copy()

    for i in range(len(pred)):
        tier = str(tiers[i])

        if tier == "none":
            lam = 0.0
        else:
            lam = lambda_by_cell.get(str(cells[i]), fallback_by_tier.get(tier, 0.0))

        pred[i] = p0[i] + float(lam) * (p1[i] - p0[i])

    return np.clip(pred, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 4. Fit several full-train refit variants
# ------------------------------------------------------------

variant_specs = [
    {"name": "fullfit_min300_shrink0p25_cap050", "min_rows": 300, "shrink": 0.25, "solver_cell_cap": 0.50, "solver_fallback_cap": 0.30},
    {"name": "fullfit_min300_shrink0p50_cap050", "min_rows": 300, "shrink": 0.50, "solver_cell_cap": 0.50, "solver_fallback_cap": 0.30},
    {"name": "fullfit_min300_shrink0p75_cap050", "min_rows": 300, "shrink": 0.75, "solver_cell_cap": 0.50, "solver_fallback_cap": 0.30},
    {"name": "fullfit_min300_shrink0p25_cap030", "min_rows": 300, "shrink": 0.25, "solver_cell_cap": 0.30, "solver_fallback_cap": 0.30},
]

variant_rows = []
variant_store = {}

for spec in variant_specs:
    lambda_by_cell, fallback_by_tier, lambda_table = fit_full_lambdas(
        min_rows=spec["min_rows"],
        shrink=spec["shrink"],
        solver_cell_cap=spec["solver_cell_cap"],
        solver_fallback_cap=spec["solver_fallback_cap"],
    )

    pred_oof_fullfit = apply_lambdas(lambda_by_cell, fallback_by_tier, is_test=False)
    pred_test_fullfit = apply_lambdas(lambda_by_cell, fallback_by_tier, is_test=True)

    trainfit_mse = float(mean_squared_error(y44c, pred_oof_fullfit))

    row = dict(spec)
    row["trainfit_mse_not_oof"] = trainfit_mse
    row["trainfit_gain_vs_44a_not_oof"] = mse44a - trainfit_mse
    row["trainfit_gain_vs_44b_not_oof"] = mse44b - trainfit_mse
    row["mean_abs_diff_vs_44b_test"] = float(np.mean(np.abs(pred_test_fullfit - pred44b_test)))
    row["p95_abs_diff_vs_44b_test"] = float(np.percentile(np.abs(pred_test_fullfit - pred44b_test), 95))
    row["max_abs_diff_vs_44b_test"] = float(np.max(np.abs(pred_test_fullfit - pred44b_test)))

    for tier in ["overlap", "solver_only", "none"]:
        mask_oof = tier_oof == tier
        mask_test = tier_test == tier

        row[f"trainfit_mse_{tier}"] = float(mean_squared_error(y44c[mask_oof], pred_oof_fullfit[mask_oof]))
        row[f"test_mean_abs_diff_vs_44b_{tier}"] = float(np.mean(np.abs(pred_test_fullfit[mask_test] - pred44b_test[mask_test])))

    variant_rows.append(row)

    variant_store[spec["name"]] = {
        "lambda_by_cell": lambda_by_cell,
        "fallback_by_tier": fallback_by_tier,
        "lambda_table": lambda_table,
        "pred_oof_fullfit": pred_oof_fullfit,
        "pred_test_fullfit": pred_test_fullfit,
    }

variant_screen = pd.DataFrame(variant_rows)

print("\nFull-fit variant screen")
print("-----------------------")
display(variant_screen)

# Select the direct analogue of public-best 44B configuration.
selected_name = "fullfit_min300_shrink0p25_cap050"
selected = variant_store[selected_name]
selected_test = selected["pred_test_fullfit"]
selected_oof_fullfit = selected["pred_oof_fullfit"]

# ------------------------------------------------------------
# 5. Save artifacts
# ------------------------------------------------------------

screen_path = "model_results/hybrid44c_fullfit_variant_screen.csv"
lambda_path = "model_results/hybrid44c_fullfit_selected_lambdas.csv"
oof_path = "model_results/oof_hybrid44c_fullfit_best_trainfit_not_oof.csv"
test_path = "model_results/testpred_hybrid44c_fullfit_best.csv"
submission_path = "submission_hybrid44c_fullfit_best.csv"

variant_screen.to_csv(screen_path, index=False)
selected["lambda_table"].to_csv(lambda_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y44c,
    "pred_39d": pred39d_oof,
    "pred_int42b3": pred42b_oof,
    "pred_44a": pred44a_oof,
    "pred_44b": pred44b_oof,
    "pred_fullfit_44c": selected_oof_fullfit,
    "tier": tier_oof,
    "subgroup": subgroup_oof,
    "cell": cell_oof,
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids44c,
    "pred_39d": pred39d_test,
    "pred_int42b3": pred42b_test,
    "pred_44a": pred44a_test,
    "pred_44b": pred44b_test,
    TARGET_COL: selected_test,
    "tier": tier_test,
    "subgroup": subgroup_test,
    "cell": cell_test,
}).to_csv(test_path, index=False)

pd.DataFrame({
    ID_COL: test_ids44c,
    TARGET_COL: selected_test,
}).to_csv(submission_path, index=False)

# Save additional variant submissions for inspection, not automatic submission.
for name, obj in variant_store.items():
    path = f"submission_hybrid44c_{name}.csv"
    pd.DataFrame({
        ID_COL: test_ids44c,
        TARGET_COL: obj["pred_test_fullfit"],
    }).to_csv(path, index=False)

# Validate selected submission.
sub = pd.read_csv(submission_path)
assert sub.shape == (n_test, 2)
assert list(sub.columns) == [ID_COL, TARGET_COL]
assert sub[ID_COL].notna().all()
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 6. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("44C full-train refit complete")
print("=" * 90)

print("\nReferences")
print("----------")
print(f"44A public: 30.602")
print(f"44B public: 30.556")
print(f"Selected 44C variant: {selected_name}")

print("\nSelected full-fit lambdas")
print("-------------------------")
print(selected["lambda_table"].to_string(index=False))

print("\nSelected test difference vs 44B")
print("-------------------------------")
diff = selected_test - pred44b_test
for tier in ["overlap", "solver_only", "none"]:
    mask = tier_test == tier
    print(
        f"{tier:12s} n={int(mask.sum()):6d} "
        f"mean_diff={float(np.mean(diff[mask])):.6f} "
        f"mean_abs_diff={float(np.mean(np.abs(diff[mask]))):.6f} "
        f"p95_abs_diff={float(np.percentile(np.abs(diff[mask]), 95)):.6f}"
    )

print("\nSaved files")
print("-----------")
print(screen_path)
print(lambda_path)
print(oof_path)
print(test_path)
print(submission_path)
for name in variant_store:
    print(f"submission_hybrid44c_{name}.csv")

print("\nSubmission validation")
print("---------------------")
print("Selected file:", submission_path)
print("Shape:", sub.shape)
print(sub[TARGET_COL].describe())

print("\nRecommendation")
print("--------------")
print("If you have a submission slot, the most natural next probe is:")
print(submission_path)
print("It is the full-training refit of the publicly validated 44B configuration.")

44C. Full-train refit of 44B tier x subgroup accounting hybrid

Reference models
----------------
39D OOF MSE:    52.820927 | public 30.752
int42b3 OOF MSE:53.073307 | public 30.923
44A OOF MSE:    52.781635 | public 30.602
44B OOF MSE:    52.780037 | public 30.556

Tier coverage
-------------
overlap      train= 53786 test= 27298
solver_only  train= 27807 test= 17807
none         train= 63328 test=  3202

Full-fit variant screen
-----------------------


,name,min_rows,shrink,solver_cell_cap,solver_fallback_cap,trainfit_mse_not_oof,trainfit_gain_vs_44a_not_oof,trainfit_gain_vs_44b_not_oof,mean_abs_diff_vs_44b_test,p95_abs_diff_vs_44b_test,max_abs_diff_vs_44b_test,trainfit_mse_overlap,test_mean_abs_diff_vs_44b_overlap,trainfit_mse_solver_only,test_mean_abs_diff_vs_44b_solver_only,trainfit_mse_none,test_mean_abs_diff_vs_44b_none
0,fullfit_min300_shrink0p25_cap050,300,0.25,0.5,0.3,52.770508,0.011127,0.009529,0.000607,0.003114,0.058172,0.506754,0.000106,64.212891,0.001484,92.135094,0.0
1,fullfit_min300_shrink0p50_cap050,300,0.50,0.5,0.3,52.772346,0.009289,0.007690,0.007373,0.037010,0.379890,0.506853,0.002119,64.222252,0.016753,92.135094,0.0
2,fullfit_min300_shrink0p75_cap050,300,0.75,0.5,0.3,52.776348,0.005287,0.003689,0.019105,0.094168,1.290718,0.507016,0.004160,64.242805,0.045452,92.135094,0.0
3,fullfit_min300_shrink0p25_cap030,300,0.25,0.3,0.3,52.773918,0.007717,0.006119,0.008487,0.044569,1.296646,0.506754,0.000106,64.230629,0.022859,92.135094,0.0



44C full-train refit complete

References
----------
44A public: 30.602
44B public: 30.556
Selected 44C variant: fullfit_min300_shrink0p25_cap050

Selected full-fit lambdas
-------------------------
        level                                        cell        tier                       subgroup  n_train  lambda_raw  lambda_shrunk   mse_best  fallback_used  min_rows  shrink  solver_cell_cap  solver_fallback_cap
tier_fallback                                     overlap     overlap                                   53786       1.000       0.998333   0.507233       0.993333       300    0.25              0.5                  0.3
tier_fallback                                 solver_only solver_only                                   27807       0.157       0.156500  64.269927       0.155000       300    0.25              0.5                  0.3
         cell                       overlap||All Students     overlap                   All Students    15974       0.505       0.628333   0.08

In [49]:
# ============================================================
# 44D. Liberal but controlled solver-only / none hybrid screen
# ============================================================
#
# Protected public-best:
#   44B public MSE = 30.556
#
# Goal:
#   Before using a submission slot on tiny 44C, test whether any
#   saved model artifact can improve 44B on solver_only or none rows.
#
# Strategy:
#   base = 44B
#   candidate correction = base + lambda * (candidate - base)
#
#   Try corrections only on:
#      - solver_only
#      - solver_only + none
#
#   Segment options:
#      - global within tier
#      - subgroup within tier
#      - assessment within tier
#      - assessment x subgroup within tier
#
# This is more liberal than 44B/44C, but still safer than full residual/meta models.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("44D. Liberal but controlled solver-only / none hybrid screen")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load artifacts
# ------------------------------------------------------------

def load_oof_44d(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(f"No prediction column in {path}")

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_44d(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

base_oof_df, y44d, pred44b_oof = load_oof_44d("model_results/oof_hybrid44b_best.csv")
base_test_df, test_ids44d, pred44b_test = load_test_44d("model_results/testpred_hybrid44b_best.csv")

n_train44d = len(y44d)
n_test44d = len(pred44b_test)

mse44b = float(mean_squared_error(y44d, pred44b_oof))

candidate_specs_44d = [
    ("meta43b2", "model_results/oof_meta43b2_best.csv", "model_results/testpred_meta43b2_best.csv", "risky_meta"),
    ("hte40a3", "model_results/oof_hte40a3_residual_best.csv", "model_results/testpred_hte40a3_residual_best.csv", "risky_hte"),
    ("entity41a", "model_results/oof_entity41a_best.csv", "model_results/testpred_entity41a_best.csv", "risky_entity"),
    ("int42b3", "model_results/oof_int42b3_best.csv", "model_results/testpred_int42b3_best.csv", "accounting"),
    ("int42a2", "model_results/oof_int42a2_best.csv", "model_results/testpred_int42a2_best.csv", "accounting"),
    ("hybrid44a", "model_results/oof_hybrid44a_39d_int42b3_best.csv", "model_results/testpred_hybrid44a_39d_int42b3_best.csv", "accounting"),
    ("account39d", "model_results/oof_account39d_best.csv", "model_results/testpred_account39d_best.csv", "accounting"),
    ("account39c", "model_results/oof_account39c_final_best.csv", "model_results/testpred_account39c_final_best.csv", "accounting"),
    ("stack43a", "model_results/oof_stack43a_best_weighted.csv", "model_results/testpred_stack43a_best_weighted.csv", "stack"),
]

cand_oof = {}
cand_test = {}
cand_kind = {}
cand_rows = []

for name, oof_path, test_path, kind in candidate_specs_44d:
    if not Path(oof_path).exists() or not Path(test_path).exists():
        print(f"Skipping missing candidate: {name}")
        continue

    try:
        _, y_file, p_oof = load_oof_44d(oof_path)
        _, _, p_test = load_test_44d(test_path)

        if len(p_oof) != n_train44d or len(p_test) != n_test44d:
            raise ValueError("length mismatch")

        max_y_diff = float(np.max(np.abs(y_file - y44d)))
        if max_y_diff > 1e-5:
            raise ValueError(f"target mismatch: {max_y_diff}")

        cand_oof[name] = p_oof
        cand_test[name] = p_test
        cand_kind[name] = kind

        cand_rows.append({
            "candidate": name,
            "kind": kind,
            "oof_mse": float(mean_squared_error(y44d, p_oof)),
            "mean_abs_diff_vs_44b": float(np.mean(np.abs(p_oof - pred44b_oof))),
        })

        print(f"Loaded {name:10s} | kind={kind:12s} | OOF MSE {mean_squared_error(y44d, p_oof):.6f}")

    except Exception as e:
        print(f"Skipping {name} due to error: {repr(e)}")

cand_summary44d = pd.DataFrame(cand_rows).sort_values("oof_mse").reset_index(drop=True)

print("\nCandidate summary")
print("-----------------")
display(cand_summary44d)

print("\nProtected base")
print("--------------")
print(f"44B OOF MSE: {mse44b:.6f}")
print("44B public MSE: 30.556")

# ------------------------------------------------------------
# 2. Tiers and segments
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

tier_oof = np.array(["none"] * n_train44d, dtype=object)
tier_oof[solver_oof & ~direct_oof] = "solver_only"
tier_oof[solver_oof & direct_oof] = "overlap"

tier_test = np.array(["none"] * n_test44d, dtype=object)
tier_test[solver_test & ~direct_test] = "solver_only"
tier_test[solver_test & direct_test] = "overlap"

tier_masks_oof = {
    "overlap": tier_oof == "overlap",
    "solver_only": tier_oof == "solver_only",
    "none": tier_oof == "none",
}

tier_masks_test = {
    "overlap": tier_test == "overlap",
    "solver_only": tier_test == "solver_only",
    "none": tier_test == "none",
}

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("raw_train_te/raw_test_te required for segment labels.")

subgroup_train = pd.Series(raw_train_te["SUBGROUP_NAME"]).astype(str).to_numpy()
subgroup_test = pd.Series(raw_test_te["SUBGROUP_NAME"]).astype(str).to_numpy()

assessment_train = pd.Series(raw_train_te["ASSESSMENT_NAME"]).astype(str).to_numpy()
assessment_test = pd.Series(raw_test_te["ASSESSMENT_NAME"]).astype(str).to_numpy()

assess_subgroup_train = np.array([f"{a}||{s}" for a, s in zip(assessment_train, subgroup_train)], dtype=object)
assess_subgroup_test = np.array([f"{a}||{s}" for a, s in zip(assessment_test, subgroup_test)], dtype=object)

segment_defs = {
    "global": {
        "train": np.array(["GLOBAL"] * n_train44d, dtype=object),
        "test": np.array(["GLOBAL"] * n_test44d, dtype=object),
        "min_rows": 1,
    },
    "subgroup": {
        "train": subgroup_train.astype(object),
        "test": subgroup_test.astype(object),
        "min_rows": 500,
    },
    "assessment": {
        "train": assessment_train.astype(object),
        "test": assessment_test.astype(object),
        "min_rows": 700,
    },
    "assessment_subgroup": {
        "train": assess_subgroup_train,
        "test": assess_subgroup_test,
        "min_rows": 500,
    },
}

print("\nTier coverage")
print("-------------")
for tier in ["overlap", "solver_only", "none"]:
    mask = tier_masks_oof[tier]
    print(
        f"{tier:12s} train={int(mask.sum()):6d} test={int(tier_masks_test[tier].sum()):6d} "
        f"44B MSE={mean_squared_error(y44d[mask], pred44b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 3. Metrics and lambda fit
# ------------------------------------------------------------

test_tier_rates = {
    tier: float(mask.mean())
    for tier, mask in tier_masks_test.items()
}

def tier_weighted_mse(pred):
    pred = np.clip(pred, 0, 100)
    total = 0.0
    for tier, mask in tier_masks_oof.items():
        total += test_tier_rates[tier] * float(mean_squared_error(y44d[mask], pred[mask]))
    return float(total)

def tier_mses(pred):
    pred = np.clip(pred, 0, 100)
    return {
        f"mse_{tier}": float(mean_squared_error(y44d[mask], pred[mask]))
        for tier, mask in tier_masks_oof.items()
    }

lambda_grid = np.unique(np.concatenate([
    np.linspace(-0.5, 1.5, 801),
    np.array([0.0, 0.05, 0.10, 0.155, 0.25, 0.5, 0.75, 1.0])
]))

def best_lambda_on_indices(idx, candidate_name):
    if len(idx) == 0:
        return 0.0, np.nan

    y = y44d[idx].astype(np.float64)
    p0 = pred44b_oof[idx].astype(np.float64)
    p1 = cand_oof[candidate_name][idx].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid:
        pred = np.clip(p0 + float(lam) * (p1 - p0), 0, 100)
        mse = float(mean_squared_error(y, pred))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam, best_mse

# ------------------------------------------------------------
# 4. OOF-safe strategy builder
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train44d)))

strategy_specs = []

for candidate in cand_oof.keys():
    for active_tiers_name, active_tiers in [
        ("solver_only", ["solver_only"]),
        ("solver_plus_none", ["solver_only", "none"]),
        ("none_only", ["none"]),
    ]:
        for segment_mode in ["global", "subgroup", "assessment", "assessment_subgroup"]:
            # Keep the search reasonable: none with assessment_subgroup is too overfit-prone.
            if active_tiers_name != "solver_only" and segment_mode == "assessment_subgroup":
                continue

            strategy_specs.append({
                "candidate": candidate,
                "active_tiers_name": active_tiers_name,
                "active_tiers": active_tiers,
                "segment_mode": segment_mode,
            })

print("\nNumber of strategies:", len(strategy_specs))

screen_rows = []
store = {}

for spec in strategy_specs:
    candidate = spec["candidate"]
    active_tiers = spec["active_tiers"]
    segment_mode = spec["segment_mode"]

    seg_train = segment_defs[segment_mode]["train"]
    seg_test = segment_defs[segment_mode]["test"]
    min_rows = int(segment_defs[segment_mode]["min_rows"])

    strategy_name = f"{candidate}_{spec['active_tiers_name']}_{segment_mode}"

    pred_oof = np.full(n_train44d, np.nan, dtype=np.float32)
    pred_test_sum = np.zeros(n_test44d, dtype=np.float64)
    choice_rows = []

    for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
        pred_val_full = pred44b_oof.copy().astype(np.float64)
        pred_test_fold = pred44b_test.copy().astype(np.float64)

        for tier in active_tiers:
            tr_tier_idx = tr_idx[tier_oof[tr_idx] == tier]
            va_tier_idx = va_idx[tier_oof[va_idx] == tier]
            test_tier_idx = np.where(tier_test == tier)[0]

            if len(va_tier_idx) == 0 and len(test_tier_idx) == 0:
                continue

            # Global fallback lambda for the fold/tier.
            lam_global, mse_global = best_lambda_on_indices(tr_tier_idx, candidate)

            if segment_mode == "global":
                if len(va_tier_idx) > 0:
                    pred_val_full[va_tier_idx] = (
                        pred44b_oof[va_tier_idx]
                        + lam_global * (cand_oof[candidate][va_tier_idx] - pred44b_oof[va_tier_idx])
                    )

                if len(test_tier_idx) > 0:
                    pred_test_fold[test_tier_idx] = (
                        pred44b_test[test_tier_idx]
                        + lam_global * (cand_test[candidate][test_tier_idx] - pred44b_test[test_tier_idx])
                    )

                choice_rows.append({
                    "strategy": strategy_name,
                    "fold": fold_num,
                    "tier": tier,
                    "segment": "GLOBAL",
                    "lambda": float(lam_global),
                    "n_train": int(len(tr_tier_idx)),
                    "train_mse_at_lambda": float(mse_global),
                })

            else:
                # Apply global fallback first.
                if len(va_tier_idx) > 0:
                    pred_val_full[va_tier_idx] = (
                        pred44b_oof[va_tier_idx]
                        + lam_global * (cand_oof[candidate][va_tier_idx] - pred44b_oof[va_tier_idx])
                    )

                if len(test_tier_idx) > 0:
                    pred_test_fold[test_tier_idx] = (
                        pred44b_test[test_tier_idx]
                        + lam_global * (cand_test[candidate][test_tier_idx] - pred44b_test[test_tier_idx])
                    )

                # Segment-specific lambdas.
                train_segments = sorted(pd.Series(seg_train[tr_tier_idx]).dropna().astype(str).unique().tolist())

                for seg in train_segments:
                    tr_seg_idx = tr_tier_idx[seg_train[tr_tier_idx].astype(str) == str(seg)]

                    if len(tr_seg_idx) < min_rows:
                        continue

                    lam_seg, mse_seg = best_lambda_on_indices(tr_seg_idx, candidate)

                    va_seg_idx = va_tier_idx[seg_train[va_tier_idx].astype(str) == str(seg)]
                    test_seg_idx = test_tier_idx[seg_test[test_tier_idx].astype(str) == str(seg)]

                    if len(va_seg_idx) > 0:
                        pred_val_full[va_seg_idx] = (
                            pred44b_oof[va_seg_idx]
                            + lam_seg * (cand_oof[candidate][va_seg_idx] - pred44b_oof[va_seg_idx])
                        )

                    if len(test_seg_idx) > 0:
                        pred_test_fold[test_seg_idx] = (
                            pred44b_test[test_seg_idx]
                            + lam_seg * (cand_test[candidate][test_seg_idx] - pred44b_test[test_seg_idx])
                        )

                    choice_rows.append({
                        "strategy": strategy_name,
                        "fold": fold_num,
                        "tier": tier,
                        "segment": str(seg),
                        "lambda": float(lam_seg),
                        "n_train": int(len(tr_seg_idx)),
                        "train_mse_at_lambda": float(mse_seg),
                    })

        pred_oof[va_idx] = np.clip(pred_val_full[va_idx], 0, 100).astype(np.float32)
        pred_test_sum += np.clip(pred_test_fold, 0, 100)

    pred_test = (pred_test_sum / N_SPLITS).astype(np.float32)

    mse_oof = float(mean_squared_error(y44d, pred_oof))
    mse_weighted = tier_weighted_mse(pred_oof)

    row = {
        "strategy": strategy_name,
        "candidate": candidate,
        "kind": cand_kind.get(candidate, ""),
        "active_tiers": spec["active_tiers_name"],
        "segment_mode": segment_mode,
        "ordinary_oof_mse": mse_oof,
        "test_tier_weighted_mse": mse_weighted,
        "gain_vs_44b_oof": mse44b - mse_oof,
        "gain_vs_44b_weighted": tier_weighted_mse(pred44b_oof) - mse_weighted,
    }
    row.update(tier_mses(pred_oof))

    screen_rows.append(row)

    store[strategy_name] = {
        "oof": pred_oof,
        "test": pred_test,
        "choices": pd.DataFrame(choice_rows),
    }

screen44d = pd.DataFrame(screen_rows).sort_values(
    ["ordinary_oof_mse", "test_tier_weighted_mse"]
).reset_index(drop=True)

screen44d_weighted = screen44d.sort_values(
    ["test_tier_weighted_mse", "ordinary_oof_mse"]
).reset_index(drop=True)

best_oof_name = screen44d.iloc[0]["strategy"]
best_weighted_name = screen44d_weighted.iloc[0]["strategy"]

best_oof_pred = store[best_oof_name]["oof"]
best_oof_test = store[best_oof_name]["test"]

best_weighted_pred = store[best_weighted_name]["oof"]
best_weighted_test = store[best_weighted_name]["test"]

# ------------------------------------------------------------
# 5. Fold diagnostics
# ------------------------------------------------------------

def fold_diag(pred, label):
    rows = []

    for fold_num, (_, va_idx) in enumerate(folds, start=1):
        row = {
            "label": label,
            "fold": fold_num,
            "mse_44b": float(mean_squared_error(y44d[va_idx], pred44b_oof[va_idx])),
            "mse_candidate": float(mean_squared_error(y44d[va_idx], pred[va_idx])),
        }
        row["gain_vs_44b"] = row["mse_44b"] - row["mse_candidate"]

        for tier in ["overlap", "solver_only", "none"]:
            mask = tier_oof[va_idx] == tier
            row[f"n_{tier}"] = int(mask.sum())

            if int(mask.sum()) > 0:
                row[f"mse44b_{tier}"] = float(mean_squared_error(y44d[va_idx][mask], pred44b_oof[va_idx][mask]))
                row[f"msecand_{tier}"] = float(mean_squared_error(y44d[va_idx][mask], pred[va_idx][mask]))
                row[f"gain_{tier}"] = row[f"mse44b_{tier}"] - row[f"msecand_{tier}"]

        rows.append(row)

    return pd.DataFrame(rows)

fold_diag_oof = fold_diag(best_oof_pred, "best_oof")
fold_diag_weighted = fold_diag(best_weighted_pred, "best_weighted")
fold_diag44d = pd.concat([fold_diag_oof, fold_diag_weighted], axis=0).reset_index(drop=True)

# ------------------------------------------------------------
# 6. Save artifacts
# ------------------------------------------------------------

screen_path = "model_results/liberal44d_screen.csv"
fold_path = "model_results/liberal44d_best_fold_diag.csv"
choice_oof_path = "model_results/liberal44d_best_oof_choices.csv"
choice_weighted_path = "model_results/liberal44d_best_weighted_choices.csv"

oof_oof_path = "model_results/oof_liberal44d_best_oof.csv"
test_oof_path = "model_results/testpred_liberal44d_best_oof.csv"
submission_oof_path = "submission_liberal44d_best_oof.csv"

oof_weighted_path = "model_results/oof_liberal44d_best_weighted.csv"
test_weighted_path = "model_results/testpred_liberal44d_best_weighted.csv"
submission_weighted_path = "submission_liberal44d_best_weighted.csv"

screen44d.to_csv(screen_path, index=False)
fold_diag44d.to_csv(fold_path, index=False)
store[best_oof_name]["choices"].to_csv(choice_oof_path, index=False)
store[best_weighted_name]["choices"].to_csv(choice_weighted_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train44d),
    TARGET_COL: y44d,
    "pred_44b": pred44b_oof,
    "pred_clipped": best_oof_pred,
    "tier": tier_oof,
}).to_csv(oof_oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids44d,
    "pred_44b": pred44b_test,
    TARGET_COL: best_oof_test,
    "tier": tier_test,
}).to_csv(test_oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids44d,
    TARGET_COL: best_oof_test,
}).to_csv(submission_oof_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train44d),
    TARGET_COL: y44d,
    "pred_44b": pred44b_oof,
    "pred_clipped": best_weighted_pred,
    "tier": tier_oof,
}).to_csv(oof_weighted_path, index=False)

pd.DataFrame({
    ID_COL: test_ids44d,
    "pred_44b": pred44b_test,
    TARGET_COL: best_weighted_test,
    "tier": tier_test,
}).to_csv(test_weighted_path, index=False)

pd.DataFrame({
    ID_COL: test_ids44d,
    TARGET_COL: best_weighted_test,
}).to_csv(submission_weighted_path, index=False)

for p in [submission_oof_path, submission_weighted_path]:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test44d, 2)
    assert list(sub.columns) == [ID_COL, TARGET_COL]
    assert sub[ID_COL].notna().all()
    assert sub[TARGET_COL].notna().all()
    assert np.isfinite(sub[TARGET_COL]).all()
    assert sub[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 7. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("44D liberal controlled hybrid screen complete")
print("=" * 90)

print("\nReference")
print("---------")
print(f"44B OOF MSE: {mse44b:.6f}")
print("44B public MSE: 30.556")

print("\nTop 20 by ordinary OOF")
print("----------------------")
display(screen44d.head(20))

print("\nTop 20 by test-tier weighted OOF")
print("--------------------------------")
display(screen44d_weighted.head(20))

print("\nBest ordinary OOF strategy")
print("--------------------------")
print(screen44d.iloc[0].to_string())

print("\nBest weighted strategy")
print("----------------------")
print(screen44d_weighted.iloc[0].to_string())

print("\nFold diagnostics")
print("----------------")
print(fold_diag44d.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path)
print(fold_path)
print(choice_oof_path)
print(choice_weighted_path)
print(oof_oof_path)
print(test_oof_path)
print(submission_oof_path)
print(oof_weighted_path)
print(test_weighted_path)
print(submission_weighted_path)

print("\nDecision rule")
print("-------------")
best_oof_gain = float(screen44d.iloc[0]["gain_vs_44b_oof"])
best_weighted_gain = float(screen44d_weighted.iloc[0]["gain_vs_44b_weighted"])
min_fold_oof = float(fold_diag_oof["gain_vs_44b"].min())
min_fold_weighted = float(fold_diag_weighted["gain_vs_44b"].min())

if best_oof_gain >= 0.25 and min_fold_oof >= 0:
    print("Best OOF strategy has meaningful stable gain over 44B. Consider submission_liberal44d_best_oof.csv.")
elif best_weighted_gain >= 0.25 and min_fold_weighted >= 0:
    print("Best weighted strategy has meaningful stable gain over 44B. Consider submission_liberal44d_best_weighted.csv.")
elif best_oof_gain > 0.05 or best_weighted_gain > 0.05:
    print("44D has modest gain. Inspect candidate/risk before deciding.")
else:
    print("44D does not find enough extra signal. 44B remains protected.")

44D. Liberal but controlled solver-only / none hybrid screen
Loaded meta43b2   | kind=risky_meta   | OOF MSE 49.768635
Loaded hte40a3    | kind=risky_hte    | OOF MSE 49.689606
Loaded entity41a  | kind=risky_entity | OOF MSE 52.760006
Loaded int42b3    | kind=accounting   | OOF MSE 53.073307
Loaded int42a2    | kind=accounting   | OOF MSE 53.123951
Loaded hybrid44a  | kind=accounting   | OOF MSE 52.781635
Loaded account39d | kind=accounting   | OOF MSE 52.820927
Loaded account39c | kind=accounting   | OOF MSE 53.226631
Loaded stack43a   | kind=stack        | OOF MSE 52.809963

Candidate summary
-----------------


,candidate,kind,oof_mse,mean_abs_diff_vs_44b
0,hte40a3,risky_hte,49.689606,1.086022
1,meta43b2,risky_meta,49.768635,0.856697
2,entity41a,risky_entity,52.760006,0.544102
3,hybrid44a,accounting,52.781635,0.018396
4,stack43a,stack,52.809963,0.060194
5,account39d,accounting,52.820927,0.046868
6,int42b3,accounting,53.073307,0.277995
7,int42a2,accounting,53.123951,0.279354
8,account39c,accounting,53.226631,0.312034



Protected base
--------------
44B OOF MSE: 52.780037
44B public MSE: 30.556

Tier coverage
-------------
overlap      train= 53786 test= 27298 44B MSE=0.507021
solver_only  train= 27807 test= 17807 44B MSE=64.262032
none         train= 63328 test=  3202 44B MSE=92.135094

Number of strategies: 90

44D liberal controlled hybrid screen complete

Reference
---------
44B OOF MSE: 52.780037
44B public MSE: 30.556

Top 20 by ordinary OOF
----------------------


,strategy,candidate,kind,active_tiers,segment_mode,ordinary_oof_mse,test_tier_weighted_mse,gain_vs_44b_oof,gain_vs_44b_weighted,mse_overlap,mse_solver_only,mse_none
0,hte40a3_solver_plus_none_subgroup,hte40a3,risky_hte,solver_plus_none,subgroup,49.774822,29.585762,3.005215,0.496239,0.507021,64.143074,85.310135
1,hte40a3_none_only_subgroup,hte40a3,risky_hte,none_only,subgroup,49.797646,29.629613,2.982391,0.452388,0.507021,64.262032,85.310135
2,hte40a3_solver_plus_none_global,hte40a3,risky_hte,solver_plus_none,global,49.829922,29.576105,2.950115,0.505896,0.507021,64.090012,85.459526
3,hte40a3_none_only_global,hte40a3,risky_hte,none_only,global,49.862930,29.639515,2.917107,0.442486,0.507021,64.262032,85.459526
4,hte40a3_solver_plus_none_assessment,hte40a3,risky_hte,solver_plus_none,assessment,49.894722,29.622309,2.885315,0.459692,0.507021,64.197151,85.560760
5,hte40a3_none_only_assessment,hte40a3,risky_hte,none_only,assessment,49.907169,29.646225,2.872868,0.435776,0.507021,64.262032,85.560760
6,meta43b2_solver_plus_none_subgroup,meta43b2,risky_meta,solver_plus_none,subgroup,50.412422,29.451136,2.367615,0.630864,0.507021,63.461693,87.068405
7,meta43b2_solver_plus_none_global,meta43b2,risky_meta,solver_plus_none,global,50.440876,29.449467,2.339161,0.632534,0.507021,63.444061,87.141266
8,meta43b2_solver_plus_none_assessment,meta43b2,risky_meta,solver_plus_none,assessment,50.477741,29.452770,2.302296,0.629231,0.507021,63.437321,87.228584
9,meta43b2_none_only_subgroup,meta43b2,risky_meta,none_only,subgroup,50.565983,29.746159,2.214054,0.335842,0.507021,64.262032,87.068405



Top 20 by test-tier weighted OOF
--------------------------------


,strategy,candidate,kind,active_tiers,segment_mode,ordinary_oof_mse,test_tier_weighted_mse,gain_vs_44b_oof,gain_vs_44b_weighted,mse_overlap,mse_solver_only,mse_none
0,meta43b2_solver_plus_none_global,meta43b2,risky_meta,solver_plus_none,global,50.440876,29.449467,2.339161,0.632534,0.507021,63.444061,87.141266
1,meta43b2_solver_plus_none_subgroup,meta43b2,risky_meta,solver_plus_none,subgroup,50.412422,29.451136,2.367615,0.630864,0.507021,63.461693,87.068405
2,meta43b2_solver_plus_none_assessment,meta43b2,risky_meta,solver_plus_none,assessment,50.477741,29.452770,2.302296,0.629231,0.507021,63.437321,87.228584
3,hte40a3_solver_plus_none_global,hte40a3,risky_hte,solver_plus_none,global,49.829922,29.576105,2.950115,0.505896,0.507021,64.090012,85.459526
4,hte40a3_solver_plus_none_subgroup,hte40a3,risky_hte,solver_plus_none,subgroup,49.774822,29.585762,3.005215,0.496239,0.507021,64.143074,85.310135
5,hte40a3_solver_plus_none_assessment,hte40a3,risky_hte,solver_plus_none,assessment,49.894722,29.622309,2.885315,0.459692,0.507021,64.197151,85.560760
6,hte40a3_none_only_subgroup,hte40a3,risky_hte,none_only,subgroup,49.797646,29.629613,2.982391,0.452388,0.507021,64.262032,85.310135
7,hte40a3_none_only_global,hte40a3,risky_hte,none_only,global,49.862930,29.639515,2.917107,0.442486,0.507021,64.262032,85.459526
8,hte40a3_none_only_assessment,hte40a3,risky_hte,none_only,assessment,49.907169,29.646225,2.872868,0.435776,0.507021,64.262032,85.560760
9,meta43b2_none_only_subgroup,meta43b2,risky_meta,none_only,subgroup,50.565983,29.746159,2.214054,0.335842,0.507021,64.262032,87.068405



Best ordinary OOF strategy
--------------------------
strategy                  hte40a3_solver_plus_none_subgroup
candidate                                           hte40a3
kind                                              risky_hte
active_tiers                               solver_plus_none
segment_mode                                       subgroup
ordinary_oof_mse                                  49.774822
test_tier_weighted_mse                            29.585762
gain_vs_44b_oof                                    3.005215
gain_vs_44b_weighted                               0.496239
mse_overlap                                        0.507021
mse_solver_only                                   64.143074
mse_none                                          85.310135

Best weighted strategy
----------------------
strategy                  meta43b2_solver_plus_none_global
candidate                                         meta43b2
kind                                            risky_meta
a

In [50]:
# ============================================================
# 45A. Sparse mixed-effects style direct model on arcsine scale
# ============================================================
#
# New branch beyond accounting:
#   Fit a direct statistical model over the sparse structure:
#
#       SCHOOL
#       DISTRICT
#       COUNTY
#       ASSESSMENT_NAME
#       SUBGROUP_NAME
#       ASSESSMENT_NAME × SUBGROUP_NAME
#       DISTRICT × ASSESSMENT_NAME
#       COUNTY × ASSESSMENT_NAME
#       optional SCHOOL × ASSESSMENT_NAME
#
# Target:
#   z = arcsin(sqrt(y / 100))
#
# Prediction:
#   y_hat = 100 * sin(z_hat)^2
#
# Then compare as:
#   1. standalone direct model
#   2. protected hybrid with 44B:
#        overlap unchanged
#        solver_only / none get OOF-tuned blend toward 45A
#
# Safe:
#   - sparse one-hot matrix
#   - ridge regression only
#   - no LightGBM
#   - no neural net
#   - no huge dense matrix
# ============================================================

import os
import re
import gc
import numpy as np
import pandas as pd
from pathlib import Path

from scipy import sparse
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("45A. Sparse mixed-effects style direct model on arcsine scale")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_45a = ["raw_train_te", "raw_test_te", "X_train_proc_model", "X_test_proc_model", "y_train"]

missing_45a = [x for x in required_45a if x not in globals()]
if missing_45a:
    raise ValueError(f"Missing required objects: {missing_45a}. Rerun safe recovery/setup first.")

y_45a = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train_45a = len(y_45a)
n_test_45a = len(raw_test_te)

if len(raw_train_te) != n_train_45a:
    raise ValueError("raw_train_te and y_train row count mismatch.")
if len(raw_test_te) != n_test_45a:
    raise ValueError("raw_test_te row count mismatch.")

# ------------------------------------------------------------
# 2. Load protected 44B anchor
# ------------------------------------------------------------

def load_oof_45a(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(f"No prediction column in {path}")

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column in {path}")

    y_file = df[TARGET_COL].to_numpy(dtype=np.float32)

    return df, y_file, np.clip(pred, 0, 100).astype(np.float32)

def load_test_45a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

oof44b_df, y44b_file, pred44b_oof = load_oof_45a("model_results/oof_hybrid44b_best.csv")
test44b_df, test_ids_45a, pred44b_test = load_test_45a("model_results/testpred_hybrid44b_best.csv")

if np.max(np.abs(y44b_file - y_45a)) > 1e-5:
    raise ValueError("44B OOF target mismatch.")

mse44b_45a = float(mean_squared_error(y_45a, pred44b_oof))

print("\nProtected anchor")
print("----------------")
print(f"44B OOF MSE: {mse44b_45a:.6f}")
print("44B public MSE: 30.556")

# ------------------------------------------------------------
# 3. Recover tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

tier_oof = np.array(["none"] * n_train_45a, dtype=object)
tier_oof[solver_oof & ~direct_oof] = "solver_only"
tier_oof[solver_oof & direct_oof] = "overlap"

tier_test = np.array(["none"] * n_test_45a, dtype=object)
tier_test[solver_test & ~direct_test] = "solver_only"
tier_test[solver_test & direct_test] = "overlap"

tier_masks_oof = {
    "overlap": tier_oof == "overlap",
    "solver_only": tier_oof == "solver_only",
    "none": tier_oof == "none",
}

tier_masks_test = {
    "overlap": tier_test == "overlap",
    "solver_only": tier_test == "solver_only",
    "none": tier_test == "none",
}

print("\nTier coverage")
print("-------------")
for tier in ["overlap", "solver_only", "none"]:
    m = tier_masks_oof[tier]
    print(
        f"{tier:12s} train={int(m.sum()):6d} test={int(tier_masks_test[tier].sum()):6d} "
        f"44B tier MSE={mean_squared_error(y_45a[m], pred44b_oof[m]):.6f}"
    )

# ------------------------------------------------------------
# 4. Feature construction helpers
# ------------------------------------------------------------

def clean_str_45a(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def add_interaction_col_45a(df, col_a, col_b, new_name):
    if col_a in df.columns and col_b in df.columns:
        df[new_name] = clean_str_45a(df[col_a]).to_numpy() + "||" + clean_str_45a(df[col_b]).to_numpy()
    return df

def make_raw_feature_frame_45a(raw_df):
    out = pd.DataFrame(index=np.arange(len(raw_df)))

    required_cat_cols = [
        "SCHOOL",
        "DISTRICT",
        "COUNTY",
        "REGION",
        "DISTRICT_TYPE",
        "ASSESSMENT_NAME",
        "SUBGROUP_NAME",
    ]

    for c in required_cat_cols:
        if c in raw_df.columns:
            out[c] = clean_str_45a(raw_df[c]).to_numpy()
        else:
            out[c] = "<NA>"

    out = add_interaction_col_45a(out, "ASSESSMENT_NAME", "SUBGROUP_NAME", "ASSESSMENT_NAME__X__SUBGROUP_NAME")
    out = add_interaction_col_45a(out, "DISTRICT", "ASSESSMENT_NAME", "DISTRICT__X__ASSESSMENT_NAME")
    out = add_interaction_col_45a(out, "COUNTY", "ASSESSMENT_NAME", "COUNTY__X__ASSESSMENT_NAME")
    out = add_interaction_col_45a(out, "SCHOOL", "SUBGROUP_NAME", "SCHOOL__X__SUBGROUP_NAME")
    out = add_interaction_col_45a(out, "SCHOOL", "ASSESSMENT_NAME", "SCHOOL__X__ASSESSMENT_NAME")

    return out

cat_train_all_45a = make_raw_feature_frame_45a(raw_train_te)
cat_test_all_45a = make_raw_feature_frame_45a(raw_test_te)

numeric_covariates_45a = [
    "N_STUDENTS",
    "PERCENT_FREE_LUNCH",
    "PERCENT_REDUCED_LUNCH",
    "PERCENT_ECONOMICALLY_DISADVANTAGED",
    "PERCENT_ENGLISH_LANGUAGE_LEARNERS",
    "PERCENT_ENGLISH_LANGUAGE_LEANERS",
    "PERCENT_WITH_DISABILITIES",
    "PERCENT_STUDENTS_WITH_DISABILITIES",
    "PERCENT_HOMELESS",
    "PERCENT_MIGRANT",
    "PERCENT_FEMALE",
    "PERCENT_MALE",
    "ATTENDANCE_RATE",
    "GRADE_03",
    "GRADE_04",
    "GRADE_05",
    "GRADE_06",
    "GRADE_07",
    "GRADE_08",
    "GRADE_09",
    "GRADE_10",
    "GRADE_11",
    "GRADE_12",
]

num_train_all_45a = pd.DataFrame(index=np.arange(n_train_45a))
num_test_all_45a = pd.DataFrame(index=np.arange(n_test_45a))

for c in numeric_covariates_45a:
    if c in X_train_proc_model.columns and c in X_test_proc_model.columns:
        if pd.api.types.is_numeric_dtype(X_train_proc_model[c]):
            num_train_all_45a[c] = (
                pd.to_numeric(X_train_proc_model[c], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .to_numpy(dtype=np.float32)
            )
            num_test_all_45a[c] = (
                pd.to_numeric(X_test_proc_model[c], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .to_numpy(dtype=np.float32)
            )

if "N_STUDENTS" in num_train_all_45a.columns:
    num_train_all_45a["log1p_N_STUDENTS"] = np.log1p(np.maximum(num_train_all_45a["N_STUDENTS"].to_numpy(dtype=np.float32), 0))
    num_test_all_45a["log1p_N_STUDENTS"] = np.log1p(np.maximum(num_test_all_45a["N_STUDENTS"].to_numpy(dtype=np.float32), 0))

# Add 44B as a numeric context feature but not as target.
num_train_all_45a["pred44b"] = pred44b_oof.astype(np.float32)
num_test_all_45a["pred44b"] = pred44b_test.astype(np.float32)

num_train_all_45a["tier_overlap"] = tier_masks_oof["overlap"].astype(np.float32)
num_train_all_45a["tier_solver_only"] = tier_masks_oof["solver_only"].astype(np.float32)
num_train_all_45a["tier_none"] = tier_masks_oof["none"].astype(np.float32)

num_test_all_45a["tier_overlap"] = tier_masks_test["overlap"].astype(np.float32)
num_test_all_45a["tier_solver_only"] = tier_masks_test["solver_only"].astype(np.float32)
num_test_all_45a["tier_none"] = tier_masks_test["none"].astype(np.float32)

print("\nNumeric features used")
print("---------------------")
print(list(num_train_all_45a.columns))
print("n numeric:", num_train_all_45a.shape[1])

# ------------------------------------------------------------
# 5. Target transform
# ------------------------------------------------------------

def percent_to_arcsine_45a(y):
    p = np.clip(np.asarray(y, dtype=np.float64) / 100.0, 1e-6, 1 - 1e-6)
    return np.arcsin(np.sqrt(p)).astype(np.float32)

def arcsine_to_percent_45a(z):
    p = np.sin(np.asarray(z, dtype=np.float64)) ** 2
    return np.clip(100.0 * p, 0, 100).astype(np.float32)

z_45a = percent_to_arcsine_45a(y_45a)

# ------------------------------------------------------------
# 6. Configs
# ------------------------------------------------------------

configs_45a = [
    {
        "name": "mixed45a_core_ridge_a100",
        "alpha": 100.0,
        "cat_cols": [
            "SCHOOL",
            "DISTRICT",
            "COUNTY",
            "REGION",
            "DISTRICT_TYPE",
            "ASSESSMENT_NAME",
            "SUBGROUP_NAME",
            "ASSESSMENT_NAME__X__SUBGROUP_NAME",
            "DISTRICT__X__ASSESSMENT_NAME",
            "COUNTY__X__ASSESSMENT_NAME",
        ],
        "include_school_assessment": False,
    },
    {
        "name": "mixed45a_core_ridge_a1000",
        "alpha": 1000.0,
        "cat_cols": [
            "SCHOOL",
            "DISTRICT",
            "COUNTY",
            "REGION",
            "DISTRICT_TYPE",
            "ASSESSMENT_NAME",
            "SUBGROUP_NAME",
            "ASSESSMENT_NAME__X__SUBGROUP_NAME",
            "DISTRICT__X__ASSESSMENT_NAME",
            "COUNTY__X__ASSESSMENT_NAME",
        ],
        "include_school_assessment": False,
    },
    {
        "name": "mixed45a_rich_ridge_a1000",
        "alpha": 1000.0,
        "cat_cols": [
            "SCHOOL",
            "DISTRICT",
            "COUNTY",
            "REGION",
            "DISTRICT_TYPE",
            "ASSESSMENT_NAME",
            "SUBGROUP_NAME",
            "ASSESSMENT_NAME__X__SUBGROUP_NAME",
            "DISTRICT__X__ASSESSMENT_NAME",
            "COUNTY__X__ASSESSMENT_NAME",
            "SCHOOL__X__SUBGROUP_NAME",
            "SCHOOL__X__ASSESSMENT_NAME",
        ],
        "include_school_assessment": True,
    },
]

print("\n45A configs")
print("-----------")
for cfg in configs_45a:
    print(cfg["name"], "| alpha", cfg["alpha"], "| n_cat", len(cfg["cat_cols"]))

# ------------------------------------------------------------
# 7. Build sparse matrix per fold
# ------------------------------------------------------------

def make_ohe_45a():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True, dtype=np.float32)

def build_sparse_design_45a(cat_train, cat_apply, num_train, num_apply, cat_cols):
    # Categorical sparse one-hot.
    ohe = make_ohe_45a()
    X_cat_train = ohe.fit_transform(cat_train[cat_cols])
    X_cat_apply = ohe.transform(cat_apply[cat_cols])

    # Numeric sparse block.
    imp = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_num_train = imp.fit_transform(num_train)
    X_num_apply = imp.transform(num_apply)

    X_num_train = scaler.fit_transform(X_num_train).astype(np.float32)
    X_num_apply = scaler.transform(X_num_apply).astype(np.float32)

    X_num_train_sp = sparse.csr_matrix(X_num_train)
    X_num_apply_sp = sparse.csr_matrix(X_num_apply)

    X_train_sp = sparse.hstack([X_cat_train, X_num_train_sp], format="csr", dtype=np.float32)
    X_apply_sp = sparse.hstack([X_cat_apply, X_num_apply_sp], format="csr", dtype=np.float32)

    return X_train_sp, X_apply_sp, ohe

def make_sample_weight_45a(idx):
    # Stabilize count-scale fitting.
    if "N_STUDENTS" in num_train_all_45a.columns:
        n = num_train_all_45a.iloc[idx]["N_STUDENTS"].to_numpy(dtype=np.float32)
        n = np.where(np.isfinite(n) & (n > 0), n, np.nanmedian(n[np.isfinite(n) & (n > 0)]))
        w = np.sqrt(np.clip(n, 1, 500)).astype(np.float32)
    else:
        w = np.ones(len(idx), dtype=np.float32)

    # Do not let overlap dominate; it is already solved.
    ov = tier_masks_oof["overlap"][idx]
    so = tier_masks_oof["solver_only"][idx]
    no = tier_masks_oof["none"][idx]

    w[ov] *= 0.10
    w[so] *= 1.25
    w[no] *= 0.75

    return w.astype(np.float32)

# ------------------------------------------------------------
# 8. OOF training
# ------------------------------------------------------------

folds45a = list(
    KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    .split(np.arange(n_train_45a))
)

pred_oof_by_config_45a = {
    cfg["name"]: np.full(n_train_45a, np.nan, dtype=np.float32)
    for cfg in configs_45a
}
pred_test_sum_by_config_45a = {
    cfg["name"]: np.zeros(n_test_45a, dtype=np.float64)
    for cfg in configs_45a
}

fold_rows_45a = []

for cfg in configs_45a:
    name = cfg["name"]
    cat_cols = cfg["cat_cols"]

    print("\n" + "=" * 90)
    print("Training config:", name)
    print("=" * 90)

    for fold_num, (tr_idx, va_idx) in enumerate(folds45a, start=1):
        cat_tr = cat_train_all_45a.iloc[tr_idx].reset_index(drop=True)
        cat_va = cat_train_all_45a.iloc[va_idx].reset_index(drop=True)
        cat_te = cat_test_all_45a.reset_index(drop=True)

        num_tr = num_train_all_45a.iloc[tr_idx].reset_index(drop=True)
        num_va = num_train_all_45a.iloc[va_idx].reset_index(drop=True)
        num_te = num_test_all_45a.reset_index(drop=True)

        X_tr, X_va, _ = build_sparse_design_45a(cat_tr, cat_va, num_tr, num_va, cat_cols)
        _, X_te, _ = build_sparse_design_45a(cat_tr, cat_te, num_tr, num_te, cat_cols)

        sw = make_sample_weight_45a(tr_idx)

        model = Ridge(
            alpha=float(cfg["alpha"]),
            solver="lsqr",
            fit_intercept=True,
            random_state=RANDOM_STATE,
        )

        model.fit(X_tr, z_45a[tr_idx], sample_weight=sw)

        z_va_pred = model.predict(X_va).astype(np.float32)
        z_te_pred = model.predict(X_te).astype(np.float32)

        p_va = arcsine_to_percent_45a(z_va_pred)
        p_te = arcsine_to_percent_45a(z_te_pred)

        pred_oof_by_config_45a[name][va_idx] = p_va
        pred_test_sum_by_config_45a[name] += p_te.astype(np.float64)

        row = {
            "config": name,
            "fold": fold_num,
            "mse_44b": float(mean_squared_error(y_45a[va_idx], pred44b_oof[va_idx])),
            "mse_direct_45a": float(mean_squared_error(y_45a[va_idx], p_va)),
            "n_features_sparse": int(X_tr.shape[1]),
            "n_train": int(len(tr_idx)),
            "n_valid": int(len(va_idx)),
        }
        row["gain_direct_vs_44b"] = row["mse_44b"] - row["mse_direct_45a"]

        for tier, mask_full in tier_masks_oof.items():
            mask_fold = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask_fold.sum())
            if int(mask_fold.sum()) > 0:
                row[f"mse44b_{tier}"] = float(mean_squared_error(y_45a[va_idx][mask_fold], pred44b_oof[va_idx][mask_fold]))
                row[f"mse45a_{tier}"] = float(mean_squared_error(y_45a[va_idx][mask_fold], p_va[mask_fold]))
                row[f"gain_direct_{tier}"] = row[f"mse44b_{tier}"] - row[f"mse45a_{tier}"]

        fold_rows_45a.append(row)

        print(
            f"fold {fold_num} | direct MSE {row['mse_direct_45a']:.6f} "
            f"| gain vs 44B {row['gain_direct_vs_44b']:.6f} "
            f"| sparse features {X_tr.shape[1]}"
        )

        del X_tr, X_va, X_te, model
        gc.collect()

fold_metrics45a = pd.DataFrame(fold_rows_45a)

# ------------------------------------------------------------
# 9. Protected blend with 44B
# ------------------------------------------------------------

lambda_grid_45a = np.unique(
    np.concatenate([
        np.linspace(-0.5, 1.5, 801),
        np.array([0.0, 0.05, 0.10, 0.155, 0.25, 0.50, 0.75, 1.0])
    ])
)

def best_lambda_for_mask_45a(candidate_pred, mask):
    if int(mask.sum()) == 0:
        return 0.0, np.nan

    y = y_45a[mask].astype(np.float64)
    p0 = pred44b_oof[mask].astype(np.float64)
    p1 = candidate_pred[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid_45a:
        p = np.clip(p0 + float(lam) * (p1 - p0), 0, 100)
        mse = float(mean_squared_error(y, p))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam, best_mse

def make_protected_blend_45a(candidate_oof, candidate_test, lams, is_test=False):
    if is_test:
        base = pred44b_test.copy().astype(np.float64)
        cand = candidate_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred44b_oof.copy().astype(np.float64)
        cand = candidate_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    # Always protect overlap by default.
    for tier in ["overlap", "solver_only", "none"]:
        lam = float(lams.get(tier, 0.0))
        mask = masks[tier]
        pred[mask] = base[mask] + lam * (cand[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

screen_rows_45a = []

for cfg in configs_45a:
    name = cfg["name"]

    cand_oof = pred_oof_by_config_45a[name]
    cand_test = (pred_test_sum_by_config_45a[name] / N_SPLITS).astype(np.float32)

    if not np.isfinite(cand_oof).all():
        print("Skipping incomplete:", name)
        continue

    # Candidate direct.
    direct_mse = float(mean_squared_error(y_45a, cand_oof))

    # Solver-only correction.
    lam_solver, mse_solver_tier = best_lambda_for_mask_45a(cand_oof, tier_masks_oof["solver_only"])
    lams_solver = {"overlap": 0.0, "solver_only": lam_solver, "none": 0.0}
    pred_solver = make_protected_blend_45a(cand_oof, cand_test, lams_solver, is_test=False)

    screen_rows_45a.append({
        "candidate_type": "solver_only_correction",
        "config": name,
        "direct_oof_mse": direct_mse,
        "lambda_overlap": 0.0,
        "lambda_solver_only": lam_solver,
        "lambda_none": 0.0,
        "oof_mse": float(mean_squared_error(y_45a, pred_solver)),
        "gain_vs_44b": mse44b_45a - float(mean_squared_error(y_45a, pred_solver)),
        "mse_overlap": float(mean_squared_error(y_45a[tier_masks_oof["overlap"]], pred_solver[tier_masks_oof["overlap"]])),
        "mse_solver_only": float(mean_squared_error(y_45a[tier_masks_oof["solver_only"]], pred_solver[tier_masks_oof["solver_only"]])),
        "mse_none": float(mean_squared_error(y_45a[tier_masks_oof["none"]], pred_solver[tier_masks_oof["none"]])),
    })

    # Solver + none correction.
    lam_none, mse_none_tier = best_lambda_for_mask_45a(cand_oof, tier_masks_oof["none"])
    lams_solver_none = {"overlap": 0.0, "solver_only": lam_solver, "none": lam_none}
    pred_solver_none = make_protected_blend_45a(cand_oof, cand_test, lams_solver_none, is_test=False)

    screen_rows_45a.append({
        "candidate_type": "solver_and_none_correction",
        "config": name,
        "direct_oof_mse": direct_mse,
        "lambda_overlap": 0.0,
        "lambda_solver_only": lam_solver,
        "lambda_none": lam_none,
        "oof_mse": float(mean_squared_error(y_45a, pred_solver_none)),
        "gain_vs_44b": mse44b_45a - float(mean_squared_error(y_45a, pred_solver_none)),
        "mse_overlap": float(mean_squared_error(y_45a[tier_masks_oof["overlap"]], pred_solver_none[tier_masks_oof["overlap"]])),
        "mse_solver_only": float(mean_squared_error(y_45a[tier_masks_oof["solver_only"]], pred_solver_none[tier_masks_oof["solver_only"]])),
        "mse_none": float(mean_squared_error(y_45a[tier_masks_oof["none"]], pred_solver_none[tier_masks_oof["none"]])),
    })

screen45a = pd.DataFrame(screen_rows_45a).sort_values("oof_mse").reset_index(drop=True)

best45a = screen45a.iloc[0]
best_name45a = best45a["config"]
best_cand_oof45a = pred_oof_by_config_45a[best_name45a]
best_cand_test45a = (pred_test_sum_by_config_45a[best_name45a] / N_SPLITS).astype(np.float32)

best_lams45a = {
    "overlap": float(best45a["lambda_overlap"]),
    "solver_only": float(best45a["lambda_solver_only"]),
    "none": float(best45a["lambda_none"]),
}

final_oof45a = make_protected_blend_45a(best_cand_oof45a, best_cand_test45a, best_lams45a, is_test=False)
final_test45a = make_protected_blend_45a(best_cand_oof45a, best_cand_test45a, best_lams45a, is_test=True)

# ------------------------------------------------------------
# 10. Fold diagnostics for best
# ------------------------------------------------------------

fold_diag_rows_45a = []

for fold_num, (_, va_idx) in enumerate(folds45a, start=1):
    row = {
        "fold": fold_num,
        "mse_44b": float(mean_squared_error(y_45a[va_idx], pred44b_oof[va_idx])),
        "mse_45a": float(mean_squared_error(y_45a[va_idx], final_oof45a[va_idx])),
    }
    row["gain_vs_44b"] = row["mse_44b"] - row["mse_45a"]

    for tier, mask_full in tier_masks_oof.items():
        mask_fold = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask_fold.sum())
        if int(mask_fold.sum()) > 0:
            row[f"mse44b_{tier}"] = float(mean_squared_error(y_45a[va_idx][mask_fold], pred44b_oof[va_idx][mask_fold]))
            row[f"mse45a_{tier}"] = float(mean_squared_error(y_45a[va_idx][mask_fold], final_oof45a[va_idx][mask_fold]))
            row[f"gain_{tier}"] = row[f"mse44b_{tier}"] - row[f"mse45a_{tier}"]

    fold_diag_rows_45a.append(row)

fold_diag45a = pd.DataFrame(fold_diag_rows_45a)

# ------------------------------------------------------------
# 11. Save artifacts
# ------------------------------------------------------------

screen_path45a = "model_results/mixed45a_screen.csv"
fold_metrics_path45a = "model_results/mixed45a_fold_metrics.csv"
fold_diag_path45a = "model_results/mixed45a_best_fold_diag.csv"
oof_path45a = "model_results/oof_mixed45a_best.csv"
test_path45a = "model_results/testpred_mixed45a_best.csv"
submission_path45a = "submission_mixed45a_best.csv"

screen45a.to_csv(screen_path45a, index=False)
fold_metrics45a.to_csv(fold_metrics_path45a, index=False)
fold_diag45a.to_csv(fold_diag_path45a, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_45a),
    TARGET_COL: y_45a,
    "pred_44b": pred44b_oof,
    "pred_mixed_direct": best_cand_oof45a,
    "pred_clipped": final_oof45a,
    "tier": tier_oof,
}).to_csv(oof_path45a, index=False)

pd.DataFrame({
    ID_COL: test_ids_45a,
    "pred_44b": pred44b_test,
    "pred_mixed_direct": best_cand_test45a,
    TARGET_COL: final_test45a,
    "tier": tier_test,
}).to_csv(test_path45a, index=False)

pd.DataFrame({
    ID_COL: test_ids_45a,
    TARGET_COL: final_test45a,
}).to_csv(submission_path45a, index=False)

sub45a = pd.read_csv(submission_path45a)
assert sub45a.shape == (n_test_45a, 2)
assert list(sub45a.columns) == [ID_COL, TARGET_COL]
assert sub45a[ID_COL].notna().all()
assert sub45a[TARGET_COL].notna().all()
assert np.isfinite(sub45a[TARGET_COL]).all()
assert sub45a[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 12. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("45A sparse mixed-effects direct model complete")
print("=" * 90)

print("\nProtected reference")
print("-------------------")
print(f"44B OOF MSE: {mse44b_45a:.6f}")
print("44B public MSE: 30.556")

print("\nFold metrics for direct mixed model")
print("-----------------------------------")
display(fold_metrics45a)

print("\nTop 45A protected blend candidates")
print("----------------------------------")
display(screen45a)

print("\nBest 45A")
print("--------")
print(best45a.to_string())
print(f"\nBest 45A OOF MSE: {mean_squared_error(y_45a, final_oof45a):.6f}")
print(f"Gain vs 44B:       {mse44b_45a - mean_squared_error(y_45a, final_oof45a):.6f}")

print("\nFold diagnostics for best 45A")
print("-----------------------------")
print(fold_diag45a.to_string(index=False))
print("Min fold gain:", float(fold_diag45a["gain_vs_44b"].min()))

print("\nSaved files")
print("-----------")
print(screen_path45a)
print(fold_metrics_path45a)
print(fold_diag_path45a)
print(oof_path45a)
print(test_path45a)
print(submission_path45a)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path45a)
print("Shape:", sub45a.shape)
print(sub45a[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
gain45a = mse44b_45a - mean_squared_error(y_45a, final_oof45a)
min_fold_gain45a = float(fold_diag45a["gain_vs_44b"].min())

if gain45a >= 0.25 and min_fold_gain45a >= 0:
    print("45A gives meaningful stable gain over 44B. Consider submission.")
elif gain45a >= 0.05:
    print("45A gives modest gain. Inspect fold/tier behavior before using a submission.")
elif gain45a > 0:
    print("45A gives tiny gain. Save artifact; probably do not submit.")
else:
    print("45A does not improve over 44B. Keep 44B protected.")

45A. Sparse mixed-effects style direct model on arcsine scale

Protected anchor
----------------
44B OOF MSE: 52.780037
44B public MSE: 30.556

Tier coverage
-------------
overlap      train= 53786 test= 27298 44B tier MSE=0.507021
solver_only  train= 27807 test= 17807 44B tier MSE=64.262032
none         train= 63328 test=  3202 44B tier MSE=92.135094

Numeric features used
---------------------
['N_STUDENTS', 'PERCENT_FREE_LUNCH', 'PERCENT_REDUCED_LUNCH', 'PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_WITH_DISABILITIES', 'PERCENT_HOMELESS', 'PERCENT_MIGRANT', 'PERCENT_FEMALE', 'PERCENT_MALE', 'ATTENDANCE_RATE', 'GRADE_03', 'GRADE_04', 'GRADE_05', 'GRADE_06', 'GRADE_07', 'GRADE_08', 'GRADE_09', 'GRADE_10', 'GRADE_11', 'GRADE_12', 'log1p_N_STUDENTS', 'pred44b', 'tier_overlap', 'tier_solver_only', 'tier_none']
n numeric: 26

45A configs
-----------
mixed45a_core_ridge_a100 | alpha 100.0 | n_cat 10
mixed45a_core_ridge_a1000 | alpha 1000.0 | n_cat 10
mix

,config,fold,mse_44b,mse_direct_45a,n_features_sparse,n_train,n_valid,gain_direct_vs_44b,n_overlap,mse44b_overlap,mse45a_overlap,gain_direct_overlap,n_solver_only,mse44b_solver_only,mse45a_solver_only,gain_direct_solver_only,n_none,mse44b_none,mse45a_none,gain_direct_none
0,mixed45a_core_ridge_a100,1,53.054913,61.474426,25887,115936,28985,-8.419514,10837,0.418953,7.076604,-6.657651,5496,62.401917,70.326385,-7.924469,12652,94.079628,104.223305,-10.143677
1,mixed45a_core_ridge_a100,2,50.813717,59.449856,25877,115937,28984,-8.636139,10622,0.394440,7.243136,-6.848696,5623,61.360851,68.684822,-7.323971,12739,88.198669,98.904411,-10.705742
2,mixed45a_core_ridge_a100,3,53.784950,62.109329,25850,115937,28984,-8.324379,10838,0.453856,7.244027,-6.790171,5524,69.692871,77.393394,-7.700523,12622,92.616127,102.530876,-9.914749
3,mixed45a_core_ridge_a100,4,52.625896,61.405590,25903,115937,28984,-8.779694,10796,0.571821,7.174565,-6.602743,5571,61.890247,68.641716,-6.751469,12617,93.076424,104.614410,-11.537987
4,mixed45a_core_ridge_a100,5,53.620712,61.637749,25873,115937,28984,-8.017036,10693,0.696569,7.454687,-6.758119,5593,66.005226,72.540138,-6.534912,12698,92.733269,102.463264,-9.729996
5,mixed45a_core_ridge_a1000,1,53.054913,58.926868,25887,115936,28985,-5.871956,10837,0.418953,5.950816,-5.531863,5496,62.401917,68.154610,-5.752693,12652,94.079628,100.294693,-6.215065
6,mixed45a_core_ridge_a1000,2,50.813717,56.555950,25877,115937,28984,-5.742233,10622,0.394440,6.058781,-5.664341,5623,61.360851,66.296387,-4.935535,12739,88.198669,94.361938,-6.163269
7,mixed45a_core_ridge_a1000,3,53.784950,59.589455,25850,115937,28984,-5.804504,10838,0.453856,6.147872,-5.694017,5524,69.692871,76.062523,-6.369652,12622,92.616127,98.268166,-5.652039
8,mixed45a_core_ridge_a1000,4,52.625896,58.775829,25903,115937,28984,-6.149933,10796,0.571821,6.071691,-5.499870,5571,61.890247,66.841850,-4.951603,12617,93.076424,100.311691,-7.235268
9,mixed45a_core_ridge_a1000,5,53.620712,58.994228,25873,115937,28984,-5.373516,10693,0.696569,6.294008,-5.597440,5593,66.005226,70.801018,-4.795792,12698,92.733269,98.172684,-5.439415



Top 45A protected blend candidates
----------------------------------


,candidate_type,config,direct_oof_mse,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_44b,mse_overlap,mse_solver_only,mse_none
0,solver_and_none_correction,mixed45a_core_ridge_a100,61.215393,0.0,-0.0425,-0.2475,52.592197,0.187840,0.507021,64.249306,91.710831
1,solver_and_none_correction,mixed45a_rich_ridge_a1000,58.654045,0.0,-0.0100,-0.0900,52.760887,0.019150,0.507021,64.261513,92.091492
2,solver_and_none_correction,mixed45a_core_ridge_a1000,58.568470,0.0,-0.0025,-0.0625,52.770977,0.009060,0.507021,64.262001,92.114372
3,solver_only_correction,mixed45a_core_ridge_a100,61.215393,0.0,-0.0425,0.0000,52.777599,0.002438,0.507021,64.249306,92.135094
4,solver_only_correction,mixed45a_rich_ridge_a1000,58.654045,0.0,-0.0100,0.0000,52.779938,0.000099,0.507021,64.261513,92.135094
5,solver_only_correction,mixed45a_core_ridge_a1000,58.568470,0.0,-0.0025,0.0000,52.780033,0.000004,0.507021,64.262001,92.135094



Best 45A
--------
candidate_type        solver_and_none_correction
config                  mixed45a_core_ridge_a100
direct_oof_mse                         61.215393
lambda_overlap                               0.0
lambda_solver_only                       -0.0425
lambda_none                              -0.2475
oof_mse                                52.592197
gain_vs_44b                              0.18784
mse_overlap                             0.507021
mse_solver_only                        64.249306
mse_none                               91.710831

Best 45A OOF MSE: 52.592197
Gain vs 44B:       0.187840

Fold diagnostics for best 45A
-----------------------------
 fold   mse_44b   mse_45a  gain_vs_44b  n_overlap  mse44b_overlap  mse45a_overlap  gain_overlap  n_solver_only  mse44b_solver_only  mse45a_solver_only  gain_solver_only  n_none  mse44b_none  mse45a_none  gain_none
    1 53.054913 52.868919     0.185993      10837        0.418953        0.418953           0.0           5496

### Public Leaderboard Check: 45A Sparse Mixed-Effects Model

The 45A sparse mixed-effects style model produced a public leaderboard MSE of `30.626`. This did not improve over the current protected best 44B public MSE of `30.556`, but it was close enough to be considered a credible alternate model rather than a failed branch.

Unlike the HTE/meta residual branches, 45A represented a genuinely different modeling structure. It used sparse school, district, county, assessment, subgroup, and interaction effects on an arcsine-transformed target, then blended the resulting prediction into 44B only on unsolved tiers.

The best 45A OOF model improved 44B from `52.780037` to `52.592197`, a gain of `0.187840`, with all fold gains positive. However, much of the improvement came from the `none` tier, which is underrepresented in the public test set. This likely explains why the public score did not beat 44B.

Conclusion: 45A is not the current final model, but it is a useful private-test hedge and evidence that sparse mixed-effect structure contains residual signal beyond deterministic accounting.

In [51]:
# ============================================================
# 46A. 44B / 45A private-hedge tier blend
# ============================================================
#
# Current protected public-best:
#   44B public MSE = 30.556
#
# Motivation:
#   45A did not beat 44B publicly, but it was close and had different
#   error behavior, especially on none-tier rows.
#
# Model:
#   pred_46A = pred_44B + lambda_tier * (pred_45A - pred_44B)
#
# Default protection:
#   overlap      lambda = 0
#   solver_only  small grid
#   none         larger grid
#
# Output:
#   Several candidate submissions:
#      submission_hedge46a_best_oof.csv
#      submission_hedge46a_none025.csv
#      submission_hedge46a_none050.csv
#      submission_hedge46a_none075.csv
#      submission_hedge46a_none100.csv
#      submission_hedge46a_solver005_none050.csv
#
# Do not submit automatically. Inspect the output first.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("46A. 44B / 45A private-hedge tier blend")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load OOF/test helpers
# ------------------------------------------------------------

def load_oof_46a(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(f"No prediction column found in {path}")

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column found in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_46a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 2. Load 44B and 45A
# ------------------------------------------------------------

oof44b_df, y_46a, pred44b_oof = load_oof_46a("model_results/oof_hybrid44b_best.csv")
test44b_df, test_ids_46a, pred44b_test = load_test_46a("model_results/testpred_hybrid44b_best.csv")

oof45a_df, y45a_file, pred45a_oof = load_oof_46a("model_results/oof_mixed45a_best.csv")
test45a_df, _, pred45a_test = load_test_46a("model_results/testpred_mixed45a_best.csv")

assert len(y_46a) == len(pred45a_oof)
assert np.max(np.abs(y_46a - y45a_file)) < 1e-5

n_train_46a = len(y_46a)
n_test_46a = len(pred44b_test)

mse44b = float(mean_squared_error(y_46a, pred44b_oof))
mse45a = float(mean_squared_error(y_46a, pred45a_oof))

print("\nBase models")
print("-----------")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")
print(f"45A OOF MSE: {mse45a:.6f} | public 30.626")

# ------------------------------------------------------------
# 3. Recover accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

print("\nTier coverage and tier MSE")
print("--------------------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"44B MSE={mean_squared_error(y_46a[mask], pred44b_oof[mask]):.6f} "
        f"45A MSE={mean_squared_error(y_46a[mask], pred45a_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 4. Blend helpers
# ------------------------------------------------------------

def make_blend_46a(lambda_overlap, lambda_solver_only, lambda_none, is_test=False):
    if is_test:
        base = pred44b_test.astype(np.float64)
        alt = pred45a_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred44b_oof.astype(np.float64)
        alt = pred45a_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    lams = {
        "overlap": float(lambda_overlap),
        "solver_only": float(lambda_solver_only),
        "none": float(lambda_none),
    }

    for tier, mask in masks.items():
        pred[mask] = base[mask] + lams[tier] * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

def tier_mse_dict_46a(pred):
    out = {}
    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y_46a[mask], pred[mask]))
        out[f"gain_{tier}_vs_44b"] = float(
            mean_squared_error(y_46a[mask], pred44b_oof[mask])
            - mean_squared_error(y_46a[mask], pred[mask])
        )
    return out

# ------------------------------------------------------------
# 5. Grid screen
# ------------------------------------------------------------

lambda_overlap_grid = [0.0]   # protect overlap
lambda_solver_grid = [0.0, 0.025, 0.05, 0.075, 0.10, 0.15]
lambda_none_grid = [0.0, 0.10, 0.25, 0.40, 0.50, 0.60, 0.75, 0.90, 1.00]

screen_rows = []

for lo in lambda_overlap_grid:
    for ls in lambda_solver_grid:
        for ln in lambda_none_grid:
            pred = make_blend_46a(lo, ls, ln, is_test=False)
            mse = float(mean_squared_error(y_46a, pred))

            row = {
                "lambda_overlap": lo,
                "lambda_solver_only": ls,
                "lambda_none": ln,
                "oof_mse": mse,
                "gain_vs_44b": mse44b - mse,
            }
            row.update(tier_mse_dict_46a(pred))
            screen_rows.append(row)

screen46a = pd.DataFrame(screen_rows).sort_values("oof_mse").reset_index(drop=True)

best46a = screen46a.iloc[0]
best_oof46a = make_blend_46a(
    best46a["lambda_overlap"],
    best46a["lambda_solver_only"],
    best46a["lambda_none"],
    is_test=False,
)
best_test46a = make_blend_46a(
    best46a["lambda_overlap"],
    best46a["lambda_solver_only"],
    best46a["lambda_none"],
    is_test=True,
)

# ------------------------------------------------------------
# 6. Stress-weighted metrics
# ------------------------------------------------------------
# These are not validation labels. They simulate private tier mixtures.
# If private has more none rows than public, a hedge can look more attractive.

def weighted_tier_mse_46a(pred, none_rate):
    none_rate = float(none_rate)
    public_overlap_rate = float(overlap_test.mean())
    public_solver_rate = float(solver_only_test.mean())

    remaining = 1.0 - none_rate
    total_public_accounting = public_overlap_rate + public_solver_rate

    if total_public_accounting <= 0:
        overlap_rate = remaining * 0.6
        solver_rate = remaining * 0.4
    else:
        overlap_rate = remaining * public_overlap_rate / total_public_accounting
        solver_rate = remaining * public_solver_rate / total_public_accounting

    return (
        overlap_rate * mean_squared_error(y_46a[overlap_oof], pred[overlap_oof])
        + solver_rate * mean_squared_error(y_46a[solver_only_oof], pred[solver_only_oof])
        + none_rate * mean_squared_error(y_46a[none_oof], pred[none_oof])
    )

stress_rows = []
for idx, row in screen46a.iterrows():
    pred = make_blend_46a(row["lambda_overlap"], row["lambda_solver_only"], row["lambda_none"], is_test=False)

    stress_row = {
        "lambda_overlap": row["lambda_overlap"],
        "lambda_solver_only": row["lambda_solver_only"],
        "lambda_none": row["lambda_none"],
        "oof_mse": row["oof_mse"],
        "gain_vs_44b": row["gain_vs_44b"],
    }

    for none_rate in [0.066284, 0.10, 0.15, 0.20, 0.30, 0.40]:
        stress_row[f"weighted_mse_none_rate_{none_rate:.3f}"] = weighted_tier_mse_46a(pred, none_rate)

    stress_rows.append(stress_row)

stress46a = pd.DataFrame(stress_rows)

# ------------------------------------------------------------
# 7. Fold diagnostics
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train_46a)))

def fold_diag_for_pred_46a(pred, label):
    rows = []
    for fold_num, (_, va_idx) in enumerate(folds, start=1):
        row = {
            "label": label,
            "fold": fold_num,
            "mse_44b": float(mean_squared_error(y_46a[va_idx], pred44b_oof[va_idx])),
            "mse_candidate": float(mean_squared_error(y_46a[va_idx], pred[va_idx])),
        }
        row["gain_vs_44b"] = row["mse_44b"] - row["mse_candidate"]

        for tier, mask_full in tier_masks_oof.items():
            mask = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask.sum())
            if int(mask.sum()) > 0:
                row[f"mse44b_{tier}"] = float(mean_squared_error(
                    y_46a[va_idx][mask],
                    pred44b_oof[va_idx][mask],
                ))
                row[f"msecand_{tier}"] = float(mean_squared_error(
                    y_46a[va_idx][mask],
                    pred[va_idx][mask],
                ))
                row[f"gain_{tier}"] = row[f"mse44b_{tier}"] - row[f"msecand_{tier}"]

        rows.append(row)

    return pd.DataFrame(rows)

fold_best46a = fold_diag_for_pred_46a(best_oof46a, "best_oof")

# Also evaluate named hedge candidates.
named_hedges = {
    "none025": {"lambda_overlap": 0.0, "lambda_solver_only": 0.0, "lambda_none": 0.25},
    "none050": {"lambda_overlap": 0.0, "lambda_solver_only": 0.0, "lambda_none": 0.50},
    "none075": {"lambda_overlap": 0.0, "lambda_solver_only": 0.0, "lambda_none": 0.75},
    "none100": {"lambda_overlap": 0.0, "lambda_solver_only": 0.0, "lambda_none": 1.00},
    "solver005_none050": {"lambda_overlap": 0.0, "lambda_solver_only": 0.05, "lambda_none": 0.50},
}

named_rows = []
named_fold_frames = []

for name, lams in named_hedges.items():
    pred = make_blend_46a(
        lams["lambda_overlap"],
        lams["lambda_solver_only"],
        lams["lambda_none"],
        is_test=False,
    )
    test_pred = make_blend_46a(
        lams["lambda_overlap"],
        lams["lambda_solver_only"],
        lams["lambda_none"],
        is_test=True,
    )

    row = {
        "name": name,
        **lams,
        "oof_mse": float(mean_squared_error(y_46a, pred)),
        "gain_vs_44b": mse44b - float(mean_squared_error(y_46a, pred)),
    }
    row.update(tier_mse_dict_46a(pred))
    named_rows.append(row)

    fd = fold_diag_for_pred_46a(pred, name)
    named_fold_frames.append(fd)

    # Save named submission.
    sub_path = f"submission_hedge46a_{name}.csv"
    pd.DataFrame({
        ID_COL: test_ids_46a,
        TARGET_COL: test_pred,
    }).to_csv(sub_path, index=False)

named46a = pd.DataFrame(named_rows).sort_values("oof_mse").reset_index(drop=True)
named_fold46a = pd.concat(named_fold_frames, axis=0).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Save artifacts
# ------------------------------------------------------------

screen_path = "model_results/hedge46a_grid_screen.csv"
stress_path = "model_results/hedge46a_stress_weighted_screen.csv"
fold_best_path = "model_results/hedge46a_best_oof_fold_diag.csv"
named_path = "model_results/hedge46a_named_candidates.csv"
named_fold_path = "model_results/hedge46a_named_fold_diag.csv"

oof_best_path = "model_results/oof_hedge46a_best_oof.csv"
test_best_path = "model_results/testpred_hedge46a_best_oof.csv"
submission_best_path = "submission_hedge46a_best_oof.csv"

screen46a.to_csv(screen_path, index=False)
stress46a.to_csv(stress_path, index=False)
fold_best46a.to_csv(fold_best_path, index=False)
named46a.to_csv(named_path, index=False)
named_fold46a.to_csv(named_fold_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train_46a),
    TARGET_COL: y_46a,
    "pred_44b": pred44b_oof,
    "pred_45a": pred45a_oof,
    "pred_clipped": best_oof46a,
    "tier_overlap": overlap_oof.astype(int),
    "tier_solver_only": solver_only_oof.astype(int),
    "tier_none": none_oof.astype(int),
}).to_csv(oof_best_path, index=False)

pd.DataFrame({
    ID_COL: test_ids_46a,
    "pred_44b": pred44b_test,
    "pred_45a": pred45a_test,
    TARGET_COL: best_test46a,
    "tier_overlap": overlap_test.astype(int),
    "tier_solver_only": solver_only_test.astype(int),
    "tier_none": none_test.astype(int),
}).to_csv(test_best_path, index=False)

pd.DataFrame({
    ID_COL: test_ids_46a,
    TARGET_COL: best_test46a,
}).to_csv(submission_best_path, index=False)

# Validate all submissions.
submission_paths = [submission_best_path] + [f"submission_hedge46a_{name}.csv" for name in named_hedges]

for p in submission_paths:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test_46a, 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[ID_COL].notna().all()
    assert sub[TARGET_COL].notna().all()
    assert np.isfinite(sub[TARGET_COL]).all()
    assert sub[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 9. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("46A hedge blend complete")
print("=" * 90)

print("\nReference")
print("---------")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")
print(f"45A OOF MSE: {mse45a:.6f} | public 30.626")

print("\nTop 20 grid candidates by ordinary OOF")
print("--------------------------------------")
display(screen46a.head(20))

print("\nNamed hedge candidates")
print("----------------------")
display(named46a)

print("\nStress-weighted screen, top 10 by private none-rate 0.20")
print("--------------------------------------------------------")
display(stress46a.sort_values("weighted_mse_none_rate_0.200").head(10))

print("\nBest OOF candidate")
print("------------------")
print(best46a.to_string())

print("\nBest OOF fold diagnostics")
print("-------------------------")
print(fold_best46a.to_string(index=False))
print("Min fold gain:", float(fold_best46a["gain_vs_44b"].min()))

print("\nNamed fold diagnostics")
print("----------------------")
print(named_fold46a.to_string(index=False))

print("\nSaved files")
print("-----------")
print(screen_path)
print(stress_path)
print(fold_best_path)
print(named_path)
print(named_fold_path)
print(oof_best_path)
print(test_best_path)
print(submission_best_path)
for name in named_hedges:
    print(f"submission_hedge46a_{name}.csv")

print("\nDecision guidance")
print("-----------------")
print("Public-protected model remains submission_hybrid44b_best.csv unless a hedge is intentionally chosen.")
print("If optimizing public, likely do not submit 46A.")
print("If hedging private none-tier risk, inspect named candidates none025/none050 before choosing.")

46A. 44B / 45A private-hedge tier blend

Base models
-----------
44B OOF MSE: 52.780037 | public 30.556
45A OOF MSE: 52.592197 | public 30.626

Tier coverage and tier MSE
--------------------------
overlap      train= 53786 test= 27298 44B MSE=0.507021 45A MSE=0.507021
solver_only  train= 27807 test= 17807 44B MSE=64.262032 45A MSE=64.249306
none         train= 63328 test=  3202 44B MSE=92.135094 45A MSE=91.710831

46A hedge blend complete

Reference
---------
44B OOF MSE: 52.780037 | public 30.556
45A OOF MSE: 52.592197 | public 30.626

Top 20 grid candidates by ordinary OOF
--------------------------------------


,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_44b,mse_overlap,gain_overlap_vs_44b,mse_solver_only,gain_solver_only_vs_44b,mse_none,gain_none_vs_44b
0,0.0,0.150,1.00,52.593983,0.186054,0.507021,0.0,64.258591,0.003441,91.710831,0.424263
1,0.0,0.100,1.00,52.594193,0.185844,0.507021,0.0,64.259674,0.002357,91.710831,0.424263
2,0.0,0.075,1.00,52.594296,0.185741,0.507021,0.0,64.260239,0.001793,91.710831,0.424263
3,0.0,0.050,1.00,52.594406,0.185631,0.507021,0.0,64.260826,0.001205,91.710831,0.424263
4,0.0,0.025,1.00,52.594524,0.185513,0.507021,0.0,64.261421,0.000610,91.710831,0.424263
5,0.0,0.000,1.00,52.594643,0.185394,0.507021,0.0,64.262032,0.000000,91.710831,0.424263
6,0.0,0.150,0.90,52.595753,0.184284,0.507021,0.0,64.258591,0.003441,91.714882,0.420212
7,0.0,0.100,0.90,52.595963,0.184074,0.507021,0.0,64.259674,0.002357,91.714882,0.420212
8,0.0,0.075,0.90,52.596069,0.183968,0.507021,0.0,64.260239,0.001793,91.714882,0.420212
9,0.0,0.050,0.90,52.596180,0.183857,0.507021,0.0,64.260826,0.001205,91.714882,0.420212



Named hedge candidates
----------------------


,name,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_44b,mse_overlap,gain_overlap_vs_44b,mse_solver_only,gain_solver_only_vs_44b,mse_none,gain_none_vs_44b
0,none100,0.0,0.00,1.00,52.594643,0.185394,0.507021,0.0,64.262032,0.000000,91.710831,0.424263
1,none075,0.0,0.00,0.75,52.606056,0.173981,0.507021,0.0,64.262032,0.000000,91.736954,0.398140
2,solver005_none050,0.0,0.05,0.50,52.640533,0.139503,0.507021,0.0,64.260826,0.001205,91.816368,0.318726
3,none050,0.0,0.00,0.50,52.640762,0.139275,0.507021,0.0,64.262032,0.000000,91.816368,0.318726
4,none025,0.0,0.00,0.25,52.698757,0.081280,0.507021,0.0,64.262032,0.000000,91.949089,0.186005



Stress-weighted screen, top 10 by private none-rate 0.20
--------------------------------------------------------


,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_44b,weighted_mse_none_rate_0.066,weighted_mse_none_rate_0.100,weighted_mse_none_rate_0.150,weighted_mse_none_rate_0.200,weighted_mse_none_rate_0.300,weighted_mse_none_rate_0.400
0,0.0,0.150,1.0,52.593983,0.186054,30.052585,32.279032,35.580799,38.882565,45.486098,52.089632
1,0.0,0.100,1.0,52.594193,0.185844,30.052984,32.279417,35.581162,38.882907,45.486398,52.089888
2,0.0,0.075,1.0,52.594296,0.185741,30.053192,32.279618,35.581352,38.883086,45.486554,52.090022
3,0.0,0.050,1.0,52.594406,0.185631,30.053409,32.279826,35.581549,38.883271,45.486716,52.090161
6,0.0,0.150,0.9,52.595753,0.184284,30.052853,32.279437,35.581406,38.883375,45.487314,52.091252
4,0.0,0.025,1.0,52.594524,0.185513,30.053628,32.280038,35.581748,38.883459,45.486881,52.090302
5,0.0,0.000,1.0,52.594643,0.185394,30.053853,32.280255,35.581953,38.883652,45.487049,52.090447
7,0.0,0.100,0.9,52.595963,0.184074,30.053253,32.279822,35.581770,38.883718,45.487613,52.091509
8,0.0,0.075,0.9,52.596069,0.183968,30.053461,32.280023,35.581959,38.883896,45.487769,52.091642
9,0.0,0.050,0.9,52.596180,0.183857,30.053677,32.280231,35.582156,38.884081,45.487932,52.091782



Best OOF candidate
------------------
lambda_overlap              0.000000
lambda_solver_only          0.150000
lambda_none                 1.000000
oof_mse                    52.593983
gain_vs_44b                 0.186054
mse_overlap                 0.507021
gain_overlap_vs_44b         0.000000
mse_solver_only            64.258591
gain_solver_only_vs_44b     0.003441
mse_none                   91.710831
gain_none_vs_44b            0.424263

Best OOF fold diagnostics
-------------------------
   label  fold   mse_44b  mse_candidate  gain_vs_44b  n_overlap  mse44b_overlap  msecand_overlap  gain_overlap  n_solver_only  mse44b_solver_only  msecand_solver_only  gain_solver_only  n_none  mse44b_none  msecand_none  gain_none
best_oof     1 53.054913      52.875408     0.179504      10837        0.418953         0.418953           0.0           5496           62.401917            62.394070          0.007847   12652    94.079628     93.671814   0.407814
best_oof     2 50.813717      50.574280

In [52]:
# ============================================================
# 47A. N_STUDENTS-bin accounting hybrid refinement
# ============================================================
#
# Protected public-best:
#   44B public MSE = 30.556
#
# Goal:
#   Improve the successful 39D/int42b3 accounting hybrid by allowing
#   blend weights to vary by:
#
#       tier × SUBGROUP_NAME × N_STUDENTS_BIN
#
# Rationale:
#   The integer/count solver advantage should depend on N_STUDENTS,
#   because rounding error and feasible count intervals are N-dependent.
#
# Conservative design:
#   - no broad residual/meta model
#   - no HTE/meta artifacts
#   - no change to none tier by default
#   - only blends 39D and int42b3
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("47A. N_STUDENTS-bin accounting hybrid refinement")
print("=" * 90)

def load_oof(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)
    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(f"No prediction column in {path}")
    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test(path):
    df = pd.read_csv(path)
    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [c for c in df.columns if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])]
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 1. Load core artifacts
# ------------------------------------------------------------

_, y47, pred39d_oof = load_oof("model_results/oof_account39d_best.csv")
_, test_ids47, pred39d_test = load_test("model_results/testpred_account39d_best.csv")

_, y42, pred42b_oof = load_oof("model_results/oof_int42b3_best.csv")
_, _, pred42b_test = load_test("model_results/testpred_int42b3_best.csv")

_, y44b, pred44b_oof = load_oof("model_results/oof_hybrid44b_best.csv")
_, _, pred44b_test = load_test("model_results/testpred_hybrid44b_best.csv")

assert np.max(np.abs(y47 - y42)) < 1e-5
assert np.max(np.abs(y47 - y44b)) < 1e-5

n_train = len(y47)
n_test = len(pred39d_test)

mse39d = float(mean_squared_error(y47, pred39d_oof))
mse42b = float(mean_squared_error(y47, pred42b_oof))
mse44b = float(mean_squared_error(y47, pred44b_oof))

print("\nReference")
print("---------")
print(f"39D OOF:    {mse39d:.6f} | public 30.752")
print(f"int42b3 OOF:{mse42b:.6f} | public 30.923")
print(f"44B OOF:    {mse44b:.6f} | public 30.556")

# ------------------------------------------------------------
# 2. Recover tiers
# ------------------------------------------------------------

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("raw_train_te/raw_test_te required.")

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

tier_oof = np.array(["none"] * n_train, dtype=object)
tier_oof[solver_oof & ~direct_oof] = "solver_only"
tier_oof[solver_oof & direct_oof] = "overlap"

tier_test = np.array(["none"] * n_test, dtype=object)
tier_test[solver_test & ~direct_test] = "solver_only"
tier_test[solver_test & direct_test] = "overlap"

# ------------------------------------------------------------
# 3. Build segmentation labels
# ------------------------------------------------------------

subgroup_oof = pd.Series(raw_train_te["SUBGROUP_NAME"]).astype(str).to_numpy()
subgroup_test = pd.Series(raw_test_te["SUBGROUP_NAME"]).astype(str).to_numpy()

n_students_oof = pd.to_numeric(raw_train_te["N_STUDENTS"], errors="coerce").to_numpy(dtype=np.float64)
n_students_test = pd.to_numeric(raw_test_te["N_STUDENTS"], errors="coerce").to_numpy(dtype=np.float64)

def n_bin(x):
    return pd.cut(
        pd.Series(x).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, 200, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", "101-200", ">200"],
    ).astype("string").fillna("<NA>").astype(str).to_numpy()

nbin_oof = n_bin(n_students_oof)
nbin_test = n_bin(n_students_test)

assessment_oof = pd.Series(raw_train_te["ASSESSMENT_NAME"]).astype(str).to_numpy()
assessment_test = pd.Series(raw_test_te["ASSESSMENT_NAME"]).astype(str).to_numpy()

def subject_family(arr):
    out = []
    for s in arr:
        sl = str(s).lower()
        if "math" in sl or "algebra" in sl or "geometry" in sl:
            out.append("math")
        elif "ela" in sl or "english" in sl:
            out.append("ela")
        elif "science" in sl or "chemistry" in sl or "physics" in sl or "biology" in sl or "earth" in sl:
            out.append("science")
        elif "history" in sl or "global" in sl or "social" in sl or "government" in sl:
            out.append("social")
        else:
            out.append("other")
    return np.asarray(out, dtype=object)

subject_oof = subject_family(assessment_oof)
subject_test = subject_family(assessment_test)

segment_defs = {
    "tier": {
        "train": tier_oof,
        "test": tier_test,
        "min_rows": 1,
    },
    "tier_subgroup": {
        "train": np.array([f"{t}||{s}" for t, s in zip(tier_oof, subgroup_oof)], dtype=object),
        "test": np.array([f"{t}||{s}" for t, s in zip(tier_test, subgroup_test)], dtype=object),
        "min_rows": 300,
    },
    "tier_nbin": {
        "train": np.array([f"{t}||{b}" for t, b in zip(tier_oof, nbin_oof)], dtype=object),
        "test": np.array([f"{t}||{b}" for t, b in zip(tier_test, nbin_test)], dtype=object),
        "min_rows": 300,
    },
    "tier_subgroup_nbin": {
        "train": np.array([f"{t}||{s}||{b}" for t, s, b in zip(tier_oof, subgroup_oof, nbin_oof)], dtype=object),
        "test": np.array([f"{t}||{s}||{b}" for t, s, b in zip(tier_test, subgroup_test, nbin_test)], dtype=object),
        "min_rows": 300,
    },
    "tier_subgroup_subject": {
        "train": np.array([f"{t}||{s}||{f}" for t, s, f in zip(tier_oof, subgroup_oof, subject_oof)], dtype=object),
        "test": np.array([f"{t}||{s}||{f}" for t, s, f in zip(tier_test, subgroup_test, subject_test)], dtype=object),
        "min_rows": 300,
    },
}

# ------------------------------------------------------------
# 4. Lambda helpers
# ------------------------------------------------------------

lambda_grid = np.unique(np.concatenate([
    np.linspace(0.0, 1.0, 1001),
    np.array([0.0, 0.155, 0.25, 0.5, 0.75, 0.993333, 1.0])
]))

fallback_44b_like = {
    "overlap": 0.993333,
    "solver_only": 0.155,
    "none": 0.0,
}

def best_lambda(idx):
    idx = np.asarray(idx, dtype=np.int64)
    if len(idx) == 0:
        return 0.0, np.nan

    y = y47[idx].astype(np.float64)
    p0 = pred39d_oof[idx].astype(np.float64)
    p1 = pred42b_oof[idx].astype(np.float64)

    best_l = 0.0
    best_m = np.inf

    for l in lambda_grid:
        p = np.clip(p0 + float(l) * (p1 - p0), 0, 100)
        m = float(mean_squared_error(y, p))
        if m < best_m:
            best_m = m
            best_l = float(l)

    return best_l, best_m

def apply_lambdas(lambda_by_segment, fallback_by_tier, seg_arr, is_test=False):
    if is_test:
        p0 = pred39d_test.astype(np.float64)
        p1 = pred42b_test.astype(np.float64)
        tiers = tier_test
    else:
        p0 = pred39d_oof.astype(np.float64)
        p1 = pred42b_oof.astype(np.float64)
        tiers = tier_oof

    pred = p0.copy()

    for i in range(len(pred)):
        t = str(tiers[i])
        if t == "none":
            lam = 0.0
        else:
            lam = lambda_by_segment.get(str(seg_arr[i]), fallback_by_tier.get(t, 0.0))
        pred[i] = p0[i] + lam * (p1[i] - p0[i])

    return np.clip(pred, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 5. OOF-safe screen
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

configs = []
for segment_name in segment_defs:
    for shrink in [0.0, 0.25, 0.50, 0.75]:
        for solver_cap in [0.30, 0.50]:
            configs.append({
                "segment_name": segment_name,
                "shrink": shrink,
                "solver_cap": solver_cap,
            })

screen_rows = []
store = {}

for cfg in configs:
    seg_name = cfg["segment_name"]
    shrink = float(cfg["shrink"])
    solver_cap = float(cfg["solver_cap"])

    seg_train = segment_defs[seg_name]["train"]
    seg_test = segment_defs[seg_name]["test"]
    min_rows = int(segment_defs[seg_name]["min_rows"])

    pred_oof = np.full(n_train, np.nan, dtype=np.float32)
    pred_test_sum = np.zeros(n_test, dtype=np.float64)
    choice_rows = []

    for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
        lambda_by_segment = {}
        fallback_by_tier = {}

        # Fit tier fallback lambdas on training fold.
        for tier in ["overlap", "solver_only"]:
            tier_tr_idx = tr_idx[tier_oof[tr_idx] == tier]
            raw_lam, raw_mse = best_lambda(tier_tr_idx)
            lam = (1.0 - shrink) * raw_lam + shrink * fallback_44b_like[tier]

            if tier == "solver_only":
                lam = min(lam, solver_cap)

            fallback_by_tier[tier] = float(lam)

        fallback_by_tier["none"] = 0.0

        # Fit segment lambdas.
        for seg in sorted(pd.Series(seg_train[tr_idx]).astype(str).unique().tolist()):
            tier = str(seg).split("||", 1)[0]
            if tier == "none":
                continue

            idx = tr_idx[seg_train[tr_idx].astype(str) == str(seg)]

            if len(idx) < min_rows:
                continue

            raw_lam, raw_mse = best_lambda(idx)
            fallback = fallback_by_tier.get(tier, fallback_44b_like.get(tier, 0.0))

            lam = (1.0 - shrink) * raw_lam + shrink * fallback

            if tier == "solver_only":
                lam = min(lam, solver_cap)

            lambda_by_segment[str(seg)] = float(lam)

            choice_rows.append({
                "fold": fold_num,
                "segment_name": seg_name,
                "segment": str(seg),
                "tier": tier,
                "n_train": int(len(idx)),
                "lambda_raw": float(raw_lam),
                "lambda_final": float(lam),
                "fallback": float(fallback),
                "mse_best": float(raw_mse),
            })

        pred_full = apply_lambdas(lambda_by_segment, fallback_by_tier, seg_train, is_test=False)
        pred_test_fold = apply_lambdas(lambda_by_segment, fallback_by_tier, seg_test, is_test=True)

        pred_oof[va_idx] = pred_full[va_idx]
        pred_test_sum += pred_test_fold

    pred_test = (pred_test_sum / N_SPLITS).astype(np.float32)

    mse = float(mean_squared_error(y47, pred_oof))

    row = {
        **cfg,
        "oof_mse": mse,
        "gain_vs_44b": mse44b - mse,
    }

    for tier in ["overlap", "solver_only", "none"]:
        mask = tier_oof == tier
        row[f"mse_{tier}"] = float(mean_squared_error(y47[mask], pred_oof[mask]))

    key = f"{seg_name}_shrink{str(shrink).replace('.', 'p')}_cap{str(solver_cap).replace('.', 'p')}"
    screen_rows.append(row)
    store[key] = {
        "oof": pred_oof,
        "test": pred_test,
        "choices": pd.DataFrame(choice_rows),
        "config": cfg,
    }

screen47a = pd.DataFrame(screen_rows).sort_values("oof_mse").reset_index(drop=True)

best_row = screen47a.iloc[0]
best_key = (
    f"{best_row['segment_name']}_"
    f"shrink{str(float(best_row['shrink'])).replace('.', 'p')}_"
    f"cap{str(float(best_row['solver_cap'])).replace('.', 'p')}"
)

best_oof = store[best_key]["oof"]
best_test = store[best_key]["test"]
best_choices = store[best_key]["choices"]

# ------------------------------------------------------------
# 6. Fold diagnostics
# ------------------------------------------------------------

fold_rows = []
for fold_num, (_, va_idx) in enumerate(folds, start=1):
    row = {
        "fold": fold_num,
        "mse_44b": float(mean_squared_error(y47[va_idx], pred44b_oof[va_idx])),
        "mse_47a": float(mean_squared_error(y47[va_idx], best_oof[va_idx])),
    }
    row["gain_vs_44b"] = row["mse_44b"] - row["mse_47a"]

    for tier in ["overlap", "solver_only", "none"]:
        mask = tier_oof[va_idx] == tier
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse44b_{tier}"] = float(mean_squared_error(y47[va_idx][mask], pred44b_oof[va_idx][mask]))
            row[f"mse47a_{tier}"] = float(mean_squared_error(y47[va_idx][mask], best_oof[va_idx][mask]))
            row[f"gain_{tier}"] = row[f"mse44b_{tier}"] - row[f"mse47a_{tier}"]

    fold_rows.append(row)

fold_diag47a = pd.DataFrame(fold_rows)

# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------

screen_path = "model_results/hybrid47a_nbin_segment_screen.csv"
choices_path = "model_results/hybrid47a_best_choices.csv"
fold_path = "model_results/hybrid47a_best_fold_diag.csv"
oof_path = "model_results/oof_hybrid47a_best.csv"
test_path = "model_results/testpred_hybrid47a_best.csv"
submission_path = "submission_hybrid47a_best.csv"

screen47a.to_csv(screen_path, index=False)
best_choices.to_csv(choices_path, index=False)
fold_diag47a.to_csv(fold_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y47,
    "pred_39d": pred39d_oof,
    "pred_int42b3": pred42b_oof,
    "pred_44b": pred44b_oof,
    "pred_clipped": best_oof,
    "tier": tier_oof,
    "subgroup": subgroup_oof,
    "n_bin": nbin_oof,
    "subject": subject_oof,
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids47,
    "pred_39d": pred39d_test,
    "pred_int42b3": pred42b_test,
    "pred_44b": pred44b_test,
    TARGET_COL: best_test,
    "tier": tier_test,
    "subgroup": subgroup_test,
    "n_bin": nbin_test,
    "subject": subject_test,
}).to_csv(test_path, index=False)

pd.DataFrame({
    ID_COL: test_ids47,
    TARGET_COL: best_test,
}).to_csv(submission_path, index=False)

sub = pd.read_csv(submission_path)
assert sub.shape == (n_test, 2)
assert list(sub.columns) == [ID_COL, TARGET_COL]
assert sub[ID_COL].notna().all()
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 8. Output
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("47A N_STUDENTS-bin accounting hybrid refinement complete")
print("=" * 90)

print("\nReference")
print("---------")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")

print("\nTop 20 47A configs")
print("------------------")
display(screen47a.head(20))

print("\nBest 47A")
print("--------")
print(best_row.to_string())

print("\nFold diagnostics")
print("----------------")
print(fold_diag47a.to_string(index=False))
print("Min fold gain:", float(fold_diag47a["gain_vs_44b"].min()))

print("\nBest choices head")
print("-----------------")
display(best_choices.head(50))

print("\nSaved files")
print("-----------")
print(screen_path)
print(choices_path)
print(fold_path)
print(oof_path)
print(test_path)
print(submission_path)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path)
print("Shape:", sub.shape)
print(sub[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
gain = float(best_row["gain_vs_44b"])
min_fold_gain = float(fold_diag47a["gain_vs_44b"].min())

if gain >= 0.05 and min_fold_gain >= 0:
    print("47A gives small stable gain over 44B. Worth a public probe if slots remain.")
elif gain > 0:
    print("47A gives marginal/unstable gain. Save artifact; inspect before submitting.")
else:
    print("47A does not improve over 44B. Keep 44B protected.")

47A. N_STUDENTS-bin accounting hybrid refinement

Reference
---------
39D OOF:    52.820927 | public 30.752
int42b3 OOF:53.073307 | public 30.923
44B OOF:    52.780037 | public 30.556

47A N_STUDENTS-bin accounting hybrid refinement complete

Reference
---------
44B OOF MSE: 52.780037 | public 30.556

Top 20 47A configs
------------------


,segment_name,shrink,solver_cap,oof_mse,gain_vs_44b,mse_overlap,mse_solver_only,mse_none
0,tier_subgroup_nbin,0.00,0.5,52.746933,0.033104,0.508814,64.086021,92.135094
1,tier_subgroup_nbin,0.25,0.5,52.747627,0.032410,0.507998,64.091217,92.135094
2,tier_subgroup_nbin,0.50,0.5,52.748554,0.031483,0.507466,64.097076,92.135094
3,tier_nbin,0.00,0.5,52.752487,0.027550,0.512517,64.107788,92.135094
4,tier_nbin,0.25,0.5,52.756283,0.023754,0.509678,64.133080,92.135094
5,tier_nbin,0.00,0.3,52.759766,0.020271,0.512517,64.145744,92.135094
6,tier_subgroup_nbin,0.75,0.5,52.760353,0.019684,0.507213,64.159042,92.135094
7,tier_subgroup_nbin,0.00,0.3,52.760971,0.019066,0.508814,64.159180,92.135094
8,tier_nbin,0.25,0.3,52.761734,0.018303,0.509678,64.161491,92.135094
9,tier_nbin,0.50,0.5,52.761917,0.018120,0.507853,64.165985,92.135094



Best 47A
--------
segment_name       tier_subgroup_nbin
shrink                            0.0
solver_cap                        0.5
oof_mse                     52.746933
gain_vs_44b                  0.033104
mse_overlap                  0.508814
mse_solver_only             64.086021
mse_none                    92.135094

Fold diagnostics
----------------
 fold   mse_44b   mse_47a  gain_vs_44b  n_overlap  mse44b_overlap  mse47a_overlap  gain_overlap  n_solver_only  mse44b_solver_only  mse47a_solver_only  gain_solver_only  n_none  mse44b_none  mse47a_none  gain_none
    1 53.054913 53.015087     0.039825      10837        0.418953        0.422295     -0.003343           5496           62.401917           62.185287          0.216629   12652    94.079628    94.079628        0.0
    2 50.813717 50.775639     0.038078      10622        0.394440        0.394144      0.000296           5623           61.360851           61.165169          0.195683   12739    88.198669    88.198669        0.0


,fold,segment_name,segment,tier,n_train,lambda_raw,lambda_final,fallback,mse_best
0,1,tier_subgroup_nbin,overlap||All Students||101-200,overlap,2452,0.870,0.870,1.000,0.095801
1,1,tier_subgroup_nbin,overlap||All Students||11-20,overlap,665,0.926,0.926,1.000,0.063767
2,1,tier_subgroup_nbin,overlap||All Students||21-50,overlap,3610,1.000,1.000,1.000,0.079970
3,1,tier_subgroup_nbin,overlap||All Students||51-100,overlap,4982,1.000,1.000,1.000,0.081632
4,1,tier_subgroup_nbin,overlap||All Students||>200,overlap,1044,0.240,0.240,1.000,0.140924
5,1,tier_subgroup_nbin,overlap||Economically Disadvantaged||101-200,overlap,650,0.734,0.734,1.000,0.328178
6,1,tier_subgroup_nbin,overlap||Economically Disadvantaged||11-20,overlap,1180,0.536,0.536,1.000,0.968351
7,1,tier_subgroup_nbin,overlap||Economically Disadvantaged||21-50,overlap,2476,0.862,0.862,1.000,0.515702
8,1,tier_subgroup_nbin,overlap||Economically Disadvantaged||51-100,overlap,1707,1.000,1.000,1.000,0.373500
9,1,tier_subgroup_nbin,overlap||Economically Disadvantaged||6-10,overlap,574,1.000,1.000,1.000,3.069396



Saved files
-----------
model_results/hybrid47a_nbin_segment_screen.csv
model_results/hybrid47a_best_choices.csv
model_results/hybrid47a_best_fold_diag.csv
model_results/oof_hybrid47a_best.csv
model_results/testpred_hybrid47a_best.csv
submission_hybrid47a_best.csv

Submission validation
---------------------
File: submission_hybrid47a_best.csv
Shape: (48307, 2)
count    48307.000000
mean        54.162129
std         25.825294
min          0.000421
25%         33.342754
50%         52.806038
75%         75.279837
max        100.000000
Name: PERCENT_PROFICIENT, dtype: float64

Decision rule
-------------
47A gives marginal/unstable gain. Save artifact; inspect before submitting.


In [53]:
# ============================================================
# 48A. Raw-table deterministic structure audit
# ============================================================
#
# Goal:
#   Search for new high-value structure beyond the known subgroup accounting:
#
#   Known:
#       All Students = Female + Male
#       All Students = Econ Disadvantaged + Not Econ Disadvantaged
#
#   New audits:
#       1. Raw schema / column audit
#       2. Duplicate row / repeated-key audit
#       3. Train-test exact key overlap audit
#       4. Exhaustive subgroup pair-sum identity scan
#       5. Possible district/county/region aggregate-row audit
#       6. N_STUDENTS consistency and suspicious aggregation candidates
#
# This cell does NOT build a submission.
# It tells us whether there is another deterministic path worth exploiting.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

os.makedirs("model_results", exist_ok=True)

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("48A. Raw-table deterministic structure audit")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load raw CSVs if present; otherwise use notebook objects
# ------------------------------------------------------------

csv_paths = {
    "scores_training": "/mnt/data/scores_training.csv",
    "scores_test": "/mnt/data/scores_test.csv",
    "school_covariates": "/mnt/data/school_covariates.csv",
    "district_covariates": "/mnt/data/district_covariates.csv",
}

loaded_csvs = {}

for name, path in csv_paths.items():
    if Path(path).exists():
        loaded_csvs[name] = pd.read_csv(path)
        print(f"Loaded {name:20s} {loaded_csvs[name].shape} from {path}")
    else:
        print(f"Missing {name:20s} at {path}")

if "raw_train_te" in globals() and "raw_test_te" in globals():
    train_raw_48 = raw_train_te.copy()
    test_raw_48 = raw_test_te.copy()
    source_used_48 = "notebook raw_train_te/raw_test_te"
else:
    if "scores_training" not in loaded_csvs or "scores_test" not in loaded_csvs:
        raise ValueError("Need either raw_train_te/raw_test_te in notebook or uploaded scores_training/scores_test CSVs.")
    train_raw_48 = loaded_csvs["scores_training"].copy()
    test_raw_48 = loaded_csvs["scores_test"].copy()
    source_used_48 = "uploaded scores CSVs"

print("\nSource used for score audit:", source_used_48)
print("Train raw shape:", train_raw_48.shape)
print("Test raw shape: ", test_raw_48.shape)

# ------------------------------------------------------------
# 2. Flexible column resolver
# ------------------------------------------------------------

def find_col_48(df, candidates, required=False):
    cols_lower = {c.lower(): c for c in df.columns}
    
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    
    # fuzzy fallback
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    
    if required:
        raise ValueError(f"Could not find required column among candidates: {candidates}")
    return None

colmap_48 = {
    "id": find_col_48(train_raw_48, [ID_COL, "ASSESSMENT_ID", "ID"], required=False),
    "target": find_col_48(train_raw_48, [TARGET_COL, "PERCENT_PROFICIENT"], required=False),
    "school": find_col_48(train_raw_48, ["SCHOOL", "SCHOOL_NAME", "ENTITY_NAME"], required=True),
    "district": find_col_48(train_raw_48, ["DISTRICT", "DISTRICT_NAME"], required=False),
    "county": find_col_48(train_raw_48, ["COUNTY", "COUNTY_NAME"], required=False),
    "region": find_col_48(train_raw_48, ["REGION"], required=False),
    "district_type": find_col_48(train_raw_48, ["DISTRICT_TYPE"], required=False),
    "assessment": find_col_48(train_raw_48, ["ASSESSMENT_NAME", "ASSESSMENT"], required=True),
    "subgroup": find_col_48(train_raw_48, ["SUBGROUP_NAME", "SUBGROUP"], required=True),
    "n": find_col_48(train_raw_48, ["N_STUDENTS", "NUM_STUDENTS", "STUDENTS"], required=True),
}

print("\nResolved columns")
print("----------------")
for k, v in colmap_48.items():
    print(f"{k:15s}: {v}")

if colmap_48["target"] is None:
    if "y_train" in globals():
        train_raw_48[TARGET_COL] = np.asarray(y_train, dtype=np.float32).reshape(-1)
        colmap_48["target"] = TARGET_COL
    else:
        raise ValueError("Could not find target column and no y_train in notebook.")

# ------------------------------------------------------------
# 3. Build compact train/test frames
# ------------------------------------------------------------

def clean_str_48(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def safe_num_48(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)

def make_compact_48(df, source_name):
    out = pd.DataFrame(index=np.arange(len(df)))
    out["source"] = source_name
    out["row_index"] = np.arange(len(df))
    
    for new_name, key in [
        ("ASSESSMENT_ID", "id"),
        ("SCHOOL", "school"),
        ("DISTRICT", "district"),
        ("COUNTY", "county"),
        ("REGION", "region"),
        ("DISTRICT_TYPE", "district_type"),
        ("ASSESSMENT_NAME", "assessment"),
        ("SUBGROUP_NAME", "subgroup"),
    ]:
        c = colmap_48.get(key)
        if c is not None and c in df.columns:
            out[new_name] = clean_str_48(df[c]).to_numpy()
        else:
            out[new_name] = "<NA>"
    
    n_col = colmap_48["n"]
    out["N_STUDENTS"] = safe_num_48(df[n_col]).to_numpy(dtype=np.float64)
    
    if source_name == "train":
        target_col = colmap_48["target"]
        out[TARGET_COL] = safe_num_48(df[target_col]).to_numpy(dtype=np.float64)
    else:
        out[TARGET_COL] = np.nan
    
    out["school_assessment_key"] = out["SCHOOL"] + "||" + out["ASSESSMENT_NAME"]
    out["school_assessment_subgroup_key"] = out["SCHOOL"] + "||" + out["ASSESSMENT_NAME"] + "||" + out["SUBGROUP_NAME"]
    out["district_assessment_subgroup_key"] = out["DISTRICT"] + "||" + out["ASSESSMENT_NAME"] + "||" + out["SUBGROUP_NAME"]
    out["county_assessment_subgroup_key"] = out["COUNTY"] + "||" + out["ASSESSMENT_NAME"] + "||" + out["SUBGROUP_NAME"]
    out["region_assessment_subgroup_key"] = out["REGION"] + "||" + out["ASSESSMENT_NAME"] + "||" + out["SUBGROUP_NAME"]
    
    if source_name == "train":
        y = np.clip(out[TARGET_COL].to_numpy(dtype=np.float64), 0, 100)
        n = out["N_STUDENTS"].to_numpy(dtype=np.float64)
        out["k_float"] = y / 100.0 * n
        out["k_round"] = np.rint(out["k_float"])
        out["k_round"] = np.where(np.isfinite(out["k_round"]), out["k_round"], np.nan)
        out["k_round"] = np.minimum(np.maximum(out["k_round"], 0), n)
        out["k_round_error_abs"] = np.abs(out["k_float"] - out["k_round"])
    else:
        out["k_float"] = np.nan
        out["k_round"] = np.nan
        out["k_round_error_abs"] = np.nan
    
    return out

train48 = make_compact_48(train_raw_48, "train")
test48 = make_compact_48(test_raw_48, "test")
all48 = pd.concat([train48, test48], axis=0).reset_index(drop=True)

print("\nCompact frames")
print("--------------")
print("train48:", train48.shape)
print("test48: ", test48.shape)
print("all48:  ", all48.shape)

# ------------------------------------------------------------
# 4. Schema / distribution audit
# ------------------------------------------------------------

schema_rows = []

for source_name, df in [("train", train48), ("test", test48), ("all", all48)]:
    for c in ["SCHOOL", "DISTRICT", "COUNTY", "REGION", "DISTRICT_TYPE", "ASSESSMENT_NAME", "SUBGROUP_NAME"]:
        schema_rows.append({
            "source": source_name,
            "column": c,
            "n_unique": int(df[c].nunique(dropna=False)),
            "n_missing_like": int((df[c].astype(str).isin(["", "<NA>", "nan", "None"])).sum()),
        })
    
    schema_rows.append({
        "source": source_name,
        "column": "N_STUDENTS",
        "n_unique": int(df["N_STUDENTS"].nunique(dropna=False)),
        "n_missing_like": int(~np.isfinite(df["N_STUDENTS"]).sum()) if False else int(df["N_STUDENTS"].isna().sum()),
    })

schema48 = pd.DataFrame(schema_rows)

print("\nSchema audit")
print("------------")
print(schema48.to_string(index=False))

print("\nSubgroup counts")
print("---------------")
subgroup_counts48 = (
    all48.groupby(["source", "SUBGROUP_NAME"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["source", "rows"], ascending=[True, False])
)
print(subgroup_counts48.to_string(index=False))

# ------------------------------------------------------------
# 5. Duplicate and repeated-key audit
# ------------------------------------------------------------

duplicate_key_specs = {
    "school_assessment_subgroup": ["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"],
    "school_assessment_subgroup_n": ["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"],
    "district_assessment_subgroup_n": ["DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"],
    "county_assessment_subgroup_n": ["COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"],
}

dup_summary_rows = []
dup_detail_frames = []

for spec_name, cols in duplicate_key_specs.items():
    for source_name, df in [("train", train48), ("test", test48), ("all", all48)]:
        key_counts = df.groupby(cols, dropna=False).size().reset_index(name="rows")
        dup = key_counts[key_counts["rows"] > 1].copy()
        
        row = {
            "key_spec": spec_name,
            "source": source_name,
            "n_duplicate_keys": int(len(dup)),
            "rows_in_duplicate_keys": int(dup["rows"].sum()) if len(dup) else 0,
            "max_rows_per_key": int(dup["rows"].max()) if len(dup) else 0,
        }
        dup_summary_rows.append(row)
        
        if len(dup) > 0:
            detail = dup.sort_values("rows", ascending=False).head(50).copy()
            detail.insert(0, "source", source_name)
            detail.insert(0, "key_spec", spec_name)
            dup_detail_frames.append(detail)

dup_summary48 = pd.DataFrame(dup_summary_rows)
dup_detail48 = pd.concat(dup_detail_frames, axis=0).reset_index(drop=True) if dup_detail_frames else pd.DataFrame()

print("\nDuplicate key summary")
print("---------------------")
print(dup_summary48.to_string(index=False))

if len(dup_detail48):
    print("\nTop duplicate keys")
    print("------------------")
    display(dup_detail48.head(50))

# Train duplicate target consistency.
target_dup_rows = []

for spec_name, cols in duplicate_key_specs.items():
    g = (
        train48.groupby(cols, dropna=False)
        .agg(
            rows=(TARGET_COL, "size"),
            target_mean=(TARGET_COL, "mean"),
            target_std=(TARGET_COL, "std"),
            target_min=(TARGET_COL, "min"),
            target_max=(TARGET_COL, "max"),
            n_min=("N_STUDENTS", "min"),
            n_max=("N_STUDENTS", "max"),
        )
        .reset_index()
    )
    g = g[g["rows"] > 1].copy()
    if len(g) == 0:
        continue
    g["target_range"] = g["target_max"] - g["target_min"]
    g["key_spec"] = spec_name
    target_dup_rows.append(g.sort_values("target_range", ascending=False).head(100))

target_dup_consistency48 = pd.concat(target_dup_rows, axis=0).reset_index(drop=True) if target_dup_rows else pd.DataFrame()

print("\nTrain duplicate target consistency")
print("----------------------------------")
if len(target_dup_consistency48):
    display(target_dup_consistency48.head(50))
else:
    print("No duplicate train keys found under tested key specs.")

# ------------------------------------------------------------
# 6. Train-test exact-key overlap audit
# ------------------------------------------------------------

overlap_rows = []

for spec_name, cols in duplicate_key_specs.items():
    train_keys = train48[cols].drop_duplicates()
    test_keys = test48[cols].drop_duplicates()
    
    merged = test_keys.merge(train_keys, on=cols, how="inner")
    
    # row-level coverage in test
    test_key_series = test48[cols].astype(str).agg("||".join, axis=1)
    train_key_set = set(train48[cols].astype(str).agg("||".join, axis=1))
    coverage = test_key_series.isin(train_key_set)
    
    overlap_rows.append({
        "key_spec": spec_name,
        "n_train_unique_keys": int(len(train_keys)),
        "n_test_unique_keys": int(len(test_keys)),
        "n_shared_unique_keys": int(len(merged)),
        "test_row_coverage_shared_key": float(coverage.mean()),
        "test_rows_with_shared_key": int(coverage.sum()),
    })

train_test_overlap48 = pd.DataFrame(overlap_rows)

print("\nTrain-test exact key overlap")
print("----------------------------")
print(train_test_overlap48.to_string(index=False))

# For exact train-test keys, evaluate how stable train target is.
exact_key_detail_frames = []

for spec_name, cols in duplicate_key_specs.items():
    train_stats = (
        train48.groupby(cols, dropna=False)
        .agg(
            train_rows=(TARGET_COL, "size"),
            train_target_mean=(TARGET_COL, "mean"),
            train_target_std=(TARGET_COL, "std"),
            train_target_min=(TARGET_COL, "min"),
            train_target_max=(TARGET_COL, "max"),
        )
        .reset_index()
    )
    
    test_keys = test48[["row_index"] + cols].copy()
    merged = test_keys.merge(train_stats, on=cols, how="left")
    merged["key_spec"] = spec_name
    merged["has_train_key"] = merged["train_rows"].notna()
    
    exact_key_detail_frames.append(merged)

exact_key_detail48 = pd.concat(exact_key_detail_frames, axis=0).reset_index(drop=True)

print("\nExact key detail sample")
print("-----------------------")
display(exact_key_detail48[exact_key_detail48["has_train_key"]].head(30))

# ------------------------------------------------------------
# 7. Exhaustive subgroup pair-sum identity scan
# ------------------------------------------------------------

def n_tolerance(n):
    if not np.isfinite(n):
        return 1.0
    return max(1.0, 0.02 * float(n))

def pair_sum_scan(df, source_name):
    """
    Within SCHOOL x ASSESSMENT_NAME, scan all ordered identities:
        subgroup_target = subgroup_a + subgroup_b
    using N_STUDENTS and, for train, rounded proficient counts.
    """
    subgroups = sorted(df["SUBGROUP_NAME"].dropna().astype(str).unique().tolist())
    
    wide_n = df.pivot_table(
        index="school_assessment_key",
        columns="SUBGROUP_NAME",
        values="N_STUDENTS",
        aggfunc="first",
    )
    
    if source_name == "train":
        wide_k = df.pivot_table(
            index="school_assessment_key",
            columns="SUBGROUP_NAME",
            values="k_round",
            aggfunc="first",
        )
    else:
        wide_k = None
    
    rows = []
    
    for target in subgroups:
        if target not in wide_n.columns:
            continue
        
        for i, a in enumerate(subgroups):
            for b in subgroups[i+1:]:
                if a == target or b == target:
                    continue
                if a not in wide_n.columns or b not in wide_n.columns:
                    continue
                
                complete = wide_n[target].notna() & wide_n[a].notna() & wide_n[b].notna()
                n_complete = int(complete.sum())
                
                if n_complete == 0:
                    continue
                
                n_t = wide_n.loc[complete, target].to_numpy(dtype=np.float64)
                n_a = wide_n.loc[complete, a].to_numpy(dtype=np.float64)
                n_b = wide_n.loc[complete, b].to_numpy(dtype=np.float64)
                
                tol = np.array([n_tolerance(x) for x in n_t])
                n_match = np.abs(n_t - (n_a + n_b)) <= tol
                
                row = {
                    "source": source_name,
                    "target_subgroup": target,
                    "part_a": a,
                    "part_b": b,
                    "complete_groups": n_complete,
                    "n_match_rate": float(n_match.mean()),
                    "n_match_groups": int(n_match.sum()),
                }
                
                if wide_k is not None:
                    complete_k = (
                        complete
                        & wide_k[target].notna()
                        & wide_k[a].notna()
                        & wide_k[b].notna()
                    )
                    if int(complete_k.sum()) > 0:
                        k_t = wide_k.loc[complete_k, target].to_numpy(dtype=np.float64)
                        k_a = wide_k.loc[complete_k, a].to_numpy(dtype=np.float64)
                        k_b = wide_k.loc[complete_k, b].to_numpy(dtype=np.float64)
                        k_match = k_t == (k_a + k_b)
                        row["k_complete_groups"] = int(complete_k.sum())
                        row["k_rounded_match_rate"] = float(k_match.mean())
                        row["k_rounded_match_groups"] = int(k_match.sum())
                    else:
                        row["k_complete_groups"] = 0
                        row["k_rounded_match_rate"] = np.nan
                        row["k_rounded_match_groups"] = 0
                
                rows.append(row)
    
    return pd.DataFrame(rows).sort_values(
        ["n_match_rate", "complete_groups"],
        ascending=[False, False],
    ).reset_index(drop=True)

pair_scan_train48 = pair_sum_scan(train48, "train")
pair_scan_test48 = pair_sum_scan(test48, "test")

print("\nTop train subgroup pair-sum identities by N_STUDENTS")
print("----------------------------------------------------")
display(pair_scan_train48.head(30))

print("\nTop test subgroup pair-sum identities by N_STUDENTS")
print("---------------------------------------------------")
display(pair_scan_test48.head(30))

# ------------------------------------------------------------
# 8. Possible aggregate-row audit
# ------------------------------------------------------------
# This checks whether within a geo x assessment x subgroup group, one row's N_STUDENTS
# looks like the sum of the other rows' N_STUDENTS.
#
# If such rows exist, they may represent aggregate rows mixed with component rows.

def aggregate_row_audit(df, source_name, geo_col, min_group_size=3):
    key_cols = [geo_col, "ASSESSMENT_NAME", "SUBGROUP_NAME"]
    
    work = df.copy()
    work = work[np.isfinite(work["N_STUDENTS"]) & (work["N_STUDENTS"] > 0)].copy()
    
    rows = []
    
    for key, g in work.groupby(key_cols, dropna=False, sort=False):
        if len(g) < min_group_size:
            continue
        
        n_vals = g["N_STUDENTS"].to_numpy(dtype=np.float64)
        total = np.nansum(n_vals)
        
        for idx_local, r in enumerate(g.itertuples(index=False)):
            n_i = float(r.N_STUDENTS)
            other_sum = total - n_i
            
            if other_sum <= 0:
                continue
            
            abs_diff = abs(n_i - other_sum)
            rel_diff = abs_diff / max(other_sum, 1.0)
            
            if abs_diff <= max(1.0, 0.02 * other_sum):
                rows.append({
                    "source": source_name,
                    "geo_col": geo_col,
                    "geo_value": getattr(r, geo_col) if hasattr(r, geo_col) else str(key[0]),
                    "ASSESSMENT_NAME": r.ASSESSMENT_NAME,
                    "SUBGROUP_NAME": r.SUBGROUP_NAME,
                    "candidate_row_index": int(r.row_index),
                    "candidate_school": r.SCHOOL,
                    "candidate_n": n_i,
                    "other_rows_sum_n": other_sum,
                    "abs_diff": abs_diff,
                    "rel_diff": rel_diff,
                    "group_size": int(len(g)),
                })
    
    return pd.DataFrame(rows).sort_values(["rel_diff", "abs_diff"]).reset_index(drop=True) if rows else pd.DataFrame()

aggregate_audits = []

for source_name, df in [("train", train48), ("test", test48), ("all", all48)]:
    for geo_col in ["DISTRICT", "COUNTY", "REGION"]:
        if geo_col in df.columns:
            agg = aggregate_row_audit(df, source_name, geo_col)
            if len(agg):
                aggregate_audits.append(agg)

aggregate_candidates48 = pd.concat(aggregate_audits, axis=0).reset_index(drop=True) if aggregate_audits else pd.DataFrame()

print("\nPossible aggregate-row candidates")
print("---------------------------------")
if len(aggregate_candidates48):
    display(aggregate_candidates48.head(100))
else:
    print("No candidate rows found where one row's N_STUDENTS approximately equals the sum of sibling rows.")

# ------------------------------------------------------------
# 9. Cross-source train/test possible aggregate coverage
# ------------------------------------------------------------
# For each test row, check if its N_STUDENTS approximates the sum of training rows
# in the same geo x assessment x subgroup group, and vice versa.

cross_agg_rows = []

for geo_col in ["DISTRICT", "COUNTY", "REGION"]:
    key_cols = [geo_col, "ASSESSMENT_NAME", "SUBGROUP_NAME"]
    
    train_sum = (
        train48.groupby(key_cols, dropna=False)
        .agg(
            train_sum_n=("N_STUDENTS", "sum"),
            train_rows=("N_STUDENTS", "size"),
            train_schools=("SCHOOL", "nunique"),
        )
        .reset_index()
    )
    
    test_with = test48.merge(train_sum, on=key_cols, how="left")
    valid = test_with["train_sum_n"].notna() & np.isfinite(test_with["N_STUDENTS"])
    
    if valid.any():
        diff = np.abs(test_with.loc[valid, "N_STUDENTS"] - test_with.loc[valid, "train_sum_n"])
        tol = np.maximum(1.0, 0.02 * test_with.loc[valid, "train_sum_n"])
        match = diff <= tol
        
        cross_agg_rows.append({
            "geo_col": geo_col,
            "direction": "test_row_equals_sum_of_train_rows_same_geo_assessment_subgroup",
            "candidate_rows": int(match.sum()),
            "candidate_rate_among_valid": float(match.mean()),
            "valid_test_rows_with_train_group": int(valid.sum()),
        })

cross_agg48 = pd.DataFrame(cross_agg_rows)

print("\nCross-source aggregate coverage")
print("-------------------------------")
if len(cross_agg48):
    print(cross_agg48.to_string(index=False))
else:
    print("No cross-source aggregate candidates found.")

# ------------------------------------------------------------
# 10. Save artifacts
# ------------------------------------------------------------

paths = {
    "schema": "model_results/audit48a_schema.csv",
    "subgroup_counts": "model_results/audit48a_subgroup_counts.csv",
    "duplicate_summary": "model_results/audit48a_duplicate_summary.csv",
    "duplicate_detail": "model_results/audit48a_duplicate_detail.csv",
    "duplicate_target_consistency": "model_results/audit48a_duplicate_target_consistency.csv",
    "train_test_overlap": "model_results/audit48a_train_test_exact_key_overlap.csv",
    "exact_key_detail": "model_results/audit48a_exact_key_detail.csv",
    "pair_scan_train": "model_results/audit48a_pair_sum_scan_train.csv",
    "pair_scan_test": "model_results/audit48a_pair_sum_scan_test.csv",
    "aggregate_candidates": "model_results/audit48a_aggregate_row_candidates.csv",
    "cross_agg": "model_results/audit48a_cross_source_aggregate_candidates.csv",
}

schema48.to_csv(paths["schema"], index=False)
subgroup_counts48.to_csv(paths["subgroup_counts"], index=False)
dup_summary48.to_csv(paths["duplicate_summary"], index=False)
dup_detail48.to_csv(paths["duplicate_detail"], index=False)
target_dup_consistency48.to_csv(paths["duplicate_target_consistency"], index=False)
train_test_overlap48.to_csv(paths["train_test_overlap"], index=False)
exact_key_detail48.to_csv(paths["exact_key_detail"], index=False)
pair_scan_train48.to_csv(paths["pair_scan_train"], index=False)
pair_scan_test48.to_csv(paths["pair_scan_test"], index=False)
aggregate_candidates48.to_csv(paths["aggregate_candidates"], index=False)
cross_agg48.to_csv(paths["cross_agg"], index=False)

print("\nSaved files")
print("-----------")
for p in paths.values():
    print(p)

# ------------------------------------------------------------
# 11. Interpretation hints
# ------------------------------------------------------------

print("\n48A complete.")
print("Interpretation guide:")
print("1. If pair-sum scan finds identities beyond known gender/econ splits, build a new accounting solver.")
print("2. If aggregate-row candidates exist, inspect whether rows are true district/county/region aggregates.")
print("3. If train-test exact key overlap is high and duplicate train targets are stable, build a repeat-key predictor.")
print("4. If none of these appear, another 5-10 public MSE jump probably requires external hidden structure not visible in current tables.")

48A. Raw-table deterministic structure audit
Missing scores_training      at /mnt/data/scores_training.csv
Missing scores_test          at /mnt/data/scores_test.csv
Missing school_covariates    at /mnt/data/school_covariates.csv
Missing district_covariates  at /mnt/data/district_covariates.csv

Source used for score audit: notebook raw_train_te/raw_test_te
Train raw shape: (144921, 63)
Test raw shape:  (48307, 62)

Resolved columns
----------------
id             : ASSESSMENT_ID
target         : PERCENT_PROFICIENT
school         : SCHOOL
district       : DISTRICT
county         : COUNTY
region         : REGION
district_type  : DISTRICT_TYPE
assessment     : ASSESSMENT_NAME
subgroup       : SUBGROUP_NAME
n              : N_STUDENTS

Compact frames
--------------
train48: (144921, 20)
test48:  (48307, 20)
all48:   (193228, 20)

Schema audit
------------
source          column  n_unique  n_missing_like
 train          SCHOOL      4469               0
 train        DISTRICT       710      

,key_spec,source,DISTRICT,ASSESSMENT_NAME,SUBGROUP_NAME,N_STUDENTS,rows,COUNTY
0,district_assessment_subgroup_n,train,03759bb1,MATH3,Not Economically Disadvantaged,5.0,19,NaN
1,district_assessment_subgroup_n,train,03759bb1,MATH4,Not Economically Disadvantaged,5.0,18,NaN
2,district_assessment_subgroup_n,train,03759bb1,ELA3,Not Economically Disadvantaged,5.0,18,NaN
3,district_assessment_subgroup_n,train,03759bb1,ELA4,Not Economically Disadvantaged,5.0,16,NaN
4,district_assessment_subgroup_n,train,03759bb1,MATH6,Not Economically Disadvantaged,7.0,15,NaN
5,district_assessment_subgroup_n,train,03759bb1,MATH7,Not Economically Disadvantaged,6.0,14,NaN
6,district_assessment_subgroup_n,train,03759bb1,ELA6,Not Economically Disadvantaged,7.0,14,NaN
7,district_assessment_subgroup_n,train,03759bb1,ELA3,Not Economically Disadvantaged,7.0,14,NaN
8,district_assessment_subgroup_n,train,03759bb1,ELA6,Not Economically Disadvantaged,5.0,14,NaN
9,district_assessment_subgroup_n,train,03759bb1,MATH4,Not Economically Disadvantaged,7.0,13,NaN



Train duplicate target consistency
----------------------------------


,DISTRICT,ASSESSMENT_NAME,SUBGROUP_NAME,N_STUDENTS,rows,target_mean,target_std,target_min,target_max,n_min,n_max,target_range,key_spec,COUNTY
0,03759bb1,Regents Common Core Geometry,Male,14.0,2,50.000000,70.710678,0.0,100.0,14.0,14.0,100.0,district_assessment_subgroup_n,NaN
1,0caf2225,ELA6,Female,6.0,2,50.000000,70.710678,0.0,100.0,6.0,6.0,100.0,district_assessment_subgroup_n,NaN
2,03759bb1,Regents Algebra I,Not Economically Disadvantaged,5.0,8,62.500000,39.188191,0.0,100.0,5.0,5.0,100.0,district_assessment_subgroup_n,NaN
3,03759bb1,Regents Algebra I,Female,7.0,3,38.000000,54.147945,0.0,100.0,7.0,7.0,100.0,district_assessment_subgroup_n,NaN
4,03759bb1,ELA5,Not Economically Disadvantaged,5.0,9,42.222222,32.317866,0.0,100.0,5.0,5.0,100.0,district_assessment_subgroup_n,NaN
5,03759bb1,MATH8,Female,17.0,3,35.333333,56.083271,0.0,100.0,17.0,17.0,100.0,district_assessment_subgroup_n,NaN
6,03759bb1,MATH8,Female,18.0,6,40.833333,47.050682,0.0,100.0,18.0,18.0,100.0,district_assessment_subgroup_n,NaN
7,03759bb1,MATH5,Not Economically Disadvantaged,5.0,9,55.555556,39.721251,0.0,100.0,5.0,5.0,100.0,district_assessment_subgroup_n,NaN
8,03759bb1,MATH3,Not Economically Disadvantaged,5.0,19,71.578947,26.090262,0.0,100.0,5.0,5.0,100.0,district_assessment_subgroup_n,NaN
9,03759bb1,MATH8,Economically Disadvantaged,19.0,4,33.000000,45.482597,0.0,100.0,19.0,19.0,100.0,district_assessment_subgroup_n,NaN



Train-test exact key overlap
----------------------------
                      key_spec  n_train_unique_keys  n_test_unique_keys  n_shared_unique_keys  test_row_coverage_shared_key  test_rows_with_shared_key
    school_assessment_subgroup               144921               48307                     0                      0.000000                          0
  school_assessment_subgroup_n               144921               48307                     0                      0.000000                          0
district_assessment_subgroup_n               131112               46290                  6148                      0.154677                       7472
  county_assessment_subgroup_n               105955               42203                 15897                      0.424390                      20501

Exact key detail sample
-----------------------


...


Top test subgroup pair-sum identities by N_STUDENTS
---------------------------------------------------


,source,target_subgroup,part_a,part_b,complete_groups,n_match_rate,n_match_groups
0,test,All Students,Female,Male,615,1.000000,615
1,test,All Students,Economically Disadvantaged,Not Economically Disadvantaged,516,1.000000,516
2,test,All Students,Male,Not Economically Disadvantaged,474,0.099156,47
3,test,All Students,Female,Not Economically Disadvantaged,484,0.092975,45
4,test,All Students,Economically Disadvantaged,Male,511,0.076321,39
5,test,All Students,Economically Disadvantaged,Female,477,0.073375,35
6,test,Economically Disadvantaged,Male,Not Economically Disadvantaged,460,0.039130,18
7,test,Not Economically Disadvantaged,Economically Disadvantaged,Female,466,0.036481,17
8,test,Economically Disadvantaged,Female,Male,479,0.035491,17
9,test,Economically Disadvantaged,Female,Not Economically Disadvantaged,466,0.032189,15



Possible aggregate-row candidates
---------------------------------


,source,geo_col,geo_value,ASSESSMENT_NAME,SUBGROUP_NAME,candidate_row_index,candidate_school,candidate_n,other_rows_sum_n,abs_diff,rel_diff,group_size
0,train,DISTRICT,609c25c9,ELA3,Economically Disadvantaged,15116,13f2177c,33.0,33.0,0.0,0.000000,4
1,train,DISTRICT,3c1108fe,RegentsScience8,All Students,124209,1f893100,81.0,81.0,0.0,0.000000,3
2,train,DISTRICT,3b4ff962,Regents Phy Set/Physics,Female,71962,d4644dc3,130.0,130.0,0.0,0.000000,3
3,train,DISTRICT,609c25c9,ELA5,Economically Disadvantaged,91881,13f2177c,31.0,31.0,0.0,0.000000,4
4,train,DISTRICT,4db5d948,ELA4,Not Economically Disadvantaged,8432,23e0b7e8,45.0,45.0,0.0,0.000000,3
...,...,...,...,...,...,...,...,...,...,...,...,...
95,train,DISTRICT,aff42a2a,Science5,Male,2105,48577ce9,36.0,35.0,1.0,0.028571,3
96,train,DISTRICT,13359051,MATH4,Female,18425,3b79e25d,34.0,35.0,1.0,0.028571,3
97,train,DISTRICT,2d2885c3,ELA5,Female,121756,de8ce39e,36.0,35.0,1.0,0.028571,3
98,train,DISTRICT,99ee9c55,MATH4,Male,122348,a38cc48c,36.0,35.0,1.0,0.028571,3



Cross-source aggregate coverage
-------------------------------
 geo_col                                                      direction  candidate_rows  candidate_rate_among_valid  valid_test_rows_with_train_group
DISTRICT test_row_equals_sum_of_train_rows_same_geo_assessment_subgroup             269                    0.008499                             31651
  COUNTY test_row_equals_sum_of_train_rows_same_geo_assessment_subgroup              58                    0.001204                             48192
  REGION test_row_equals_sum_of_train_rows_same_geo_assessment_subgroup               0                    0.000000                             48307

Saved files
-----------
model_results/audit48a_schema.csv
model_results/audit48a_subgroup_counts.csv
model_results/audit48a_duplicate_summary.csv
model_results/audit48a_duplicate_detail.csv
model_results/audit48a_duplicate_target_consistency.csv
model_results/audit48a_train_test_exact_key_overlap.csv
model_results/audit48a_exact_key

In [54]:
# ============================================================
# 48B. OOF validation of district/county repeat-key and aggregate predictors
# ============================================================
#
# Motivation from 48A:
#   - No new subgroup pair-sum identities were found.
#   - But broader exact-key overlap exists:
#       district × assessment × subgroup × N_STUDENTS
#       county × assessment × subgroup × N_STUDENTS
#   - Possible aggregate-row candidates also exist.
#
# Goal:
#   Validate whether these structures improve the protected 44B model.
#
# Candidate predictors:
#   1. Repeat-key mean predictors:
#        DISTRICT/COUNTY × ASSESSMENT_NAME × SUBGROUP_NAME × N_STUDENTS
#        DISTRICT/COUNTY × ASSESSMENT_NAME × SUBGROUP_NAME
#
#   2. Aggregate-count predictors:
#        test/validation row N_STUDENTS approximately equals sum of training
#        rows in same DISTRICT/COUNTY × ASSESSMENT_NAME × SUBGROUP_NAME group.
#
# Protected base:
#   44B, public MSE = 30.556
#
# This cell creates:
#   submission_repeat48b_best_oof.csv
#   submission_repeat48b_best_weighted.csv
#
# Do not submit automatically; inspect output first.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("48B. OOF validation of district/county repeat-key and aggregate predictors")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load protected 44B
# ------------------------------------------------------------

def load_oof_48b(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(f"No prediction column found in {path}")

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column found in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_48b(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

oof44b_df, y48b, pred44b_oof = load_oof_48b("model_results/oof_hybrid44b_best.csv")
test44b_df, test_ids48b, pred44b_test = load_test_48b("model_results/testpred_hybrid44b_best.csv")

n_train48b = len(y48b)
n_test48b = len(pred44b_test)

mse44b = float(mean_squared_error(y48b, pred44b_oof))

print("\nProtected reference")
print("-------------------")
print(f"44B OOF MSE: {mse44b:.6f}")
print("44B public MSE: 30.556")

# ------------------------------------------------------------
# 2. Build compact raw frames
# ------------------------------------------------------------

required = ["raw_train_te", "raw_test_te"]
missing = [x for x in required if x not in globals()]
if missing:
    raise ValueError(f"Missing {missing}. Rerun setup/recovery cells first.")

def clean_str_48b(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def safe_num_48b(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)

required_cols = ["SCHOOL", "DISTRICT", "COUNTY", "REGION", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]

for c in required_cols:
    if c not in raw_train_te.columns or c not in raw_test_te.columns:
        raise ValueError(f"Missing required raw column: {c}")

train48b = pd.DataFrame({
    "row_index": np.arange(n_train48b),
    "SCHOOL": clean_str_48b(raw_train_te["SCHOOL"]),
    "DISTRICT": clean_str_48b(raw_train_te["DISTRICT"]),
    "COUNTY": clean_str_48b(raw_train_te["COUNTY"]),
    "REGION": clean_str_48b(raw_train_te["REGION"]),
    "ASSESSMENT_NAME": clean_str_48b(raw_train_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str_48b(raw_train_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_num_48b(raw_train_te["N_STUDENTS"]).to_numpy(dtype=np.float64),
    TARGET_COL: y48b.astype(np.float64),
})

test48b = pd.DataFrame({
    "row_index": np.arange(n_test48b),
    "SCHOOL": clean_str_48b(raw_test_te["SCHOOL"]),
    "DISTRICT": clean_str_48b(raw_test_te["DISTRICT"]),
    "COUNTY": clean_str_48b(raw_test_te["COUNTY"]),
    "REGION": clean_str_48b(raw_test_te["REGION"]),
    "ASSESSMENT_NAME": clean_str_48b(raw_test_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_str_48b(raw_test_te["SUBGROUP_NAME"]),
    "N_STUDENTS": safe_num_48b(raw_test_te["N_STUDENTS"]).to_numpy(dtype=np.float64),
})

train48b["N_INT"] = np.rint(train48b["N_STUDENTS"]).astype("Int64")
test48b["N_INT"] = np.rint(test48b["N_STUDENTS"]).astype("Int64")

train48b["k_round"] = np.rint(np.clip(train48b[TARGET_COL], 0, 100) / 100.0 * train48b["N_STUDENTS"])
train48b["k_round"] = np.minimum(np.maximum(train48b["k_round"], 0), train48b["N_STUDENTS"])

# ------------------------------------------------------------
# 3. Recover accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

tier_oof = np.array(["none"] * n_train48b, dtype=object)
tier_oof[solver_oof & ~direct_oof] = "solver_only"
tier_oof[solver_oof & direct_oof] = "overlap"

tier_test = np.array(["none"] * n_test48b, dtype=object)
tier_test[solver_test & ~direct_test] = "solver_only"
tier_test[solver_test & direct_test] = "overlap"

tier_masks_oof = {
    "overlap": tier_oof == "overlap",
    "solver_only": tier_oof == "solver_only",
    "none": tier_oof == "none",
}

tier_masks_test = {
    "overlap": tier_test == "overlap",
    "solver_only": tier_test == "solver_only",
    "none": tier_test == "none",
}

test_tier_rates = {
    tier: float(mask.mean())
    for tier, mask in tier_masks_test.items()
}

print("\nTier coverage")
print("-------------")
for tier in ["overlap", "solver_only", "none"]:
    mask = tier_masks_oof[tier]
    print(
        f"{tier:12s} train={int(mask.sum()):6d} test={int(tier_masks_test[tier].sum()):6d} "
        f"44B tier MSE={mean_squared_error(y48b[mask], pred44b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 4. Repeat-key candidate builders
# ------------------------------------------------------------

def fit_repeat_stats(df_fit, key_cols):
    stats = (
        df_fit.groupby(key_cols, dropna=False)
        .agg(
            repeat_rows=(TARGET_COL, "size"),
            target_mean=(TARGET_COL, "mean"),
            target_std=(TARGET_COL, "std"),
            target_min=(TARGET_COL, "min"),
            target_max=(TARGET_COL, "max"),
            sum_k=("k_round", "sum"),
            sum_n=("N_STUDENTS", "sum"),
        )
        .reset_index()
    )

    stats["target_std"] = stats["target_std"].fillna(0.0)
    stats["count_rate_pred"] = np.where(
        stats["sum_n"] > 0,
        100.0 * stats["sum_k"] / stats["sum_n"],
        np.nan,
    )
    stats["target_range"] = stats["target_max"] - stats["target_min"]

    return stats

def apply_repeat_stats(df_query, stats, key_cols, pred_col, min_rows=1, max_std=None, max_range=None):
    merged = df_query[["row_index"] + key_cols].merge(stats, on=key_cols, how="left")

    pred = np.full(len(df_query), np.nan, dtype=np.float32)
    covered = np.zeros(len(df_query), dtype=bool)

    valid = merged["repeat_rows"].notna()
    valid &= merged["repeat_rows"].astype(float) >= float(min_rows)

    if max_std is not None:
        valid &= merged["target_std"].astype(float) <= float(max_std)

    if max_range is not None:
        valid &= merged["target_range"].astype(float) <= float(max_range)

    vals = pd.to_numeric(merged[pred_col], errors="coerce").to_numpy(dtype=np.float64)
    valid &= np.isfinite(vals)

    pred[valid.to_numpy()] = np.clip(vals[valid.to_numpy()], 0, 100).astype(np.float32)
    covered[valid.to_numpy()] = True

    return pred, covered

repeat_specs = [
    {
        "name": "repeat_district_n_mean_min1",
        "key_cols": ["DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_INT"],
        "pred_col": "target_mean",
        "min_rows": 1,
        "max_std": None,
        "max_range": None,
    },
    {
        "name": "repeat_district_n_mean_min2",
        "key_cols": ["DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_INT"],
        "pred_col": "target_mean",
        "min_rows": 2,
        "max_std": None,
        "max_range": None,
    },
    {
        "name": "repeat_district_n_count_min1",
        "key_cols": ["DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_INT"],
        "pred_col": "count_rate_pred",
        "min_rows": 1,
        "max_std": None,
        "max_range": None,
    },
    {
        "name": "repeat_district_n_lowvar_s10",
        "key_cols": ["DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_INT"],
        "pred_col": "target_mean",
        "min_rows": 2,
        "max_std": 10.0,
        "max_range": None,
    },
    {
        "name": "repeat_county_n_mean_min1",
        "key_cols": ["COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_INT"],
        "pred_col": "target_mean",
        "min_rows": 1,
        "max_std": None,
        "max_range": None,
    },
    {
        "name": "repeat_county_n_mean_min2",
        "key_cols": ["COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_INT"],
        "pred_col": "target_mean",
        "min_rows": 2,
        "max_std": None,
        "max_range": None,
    },
    {
        "name": "repeat_county_n_count_min1",
        "key_cols": ["COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_INT"],
        "pred_col": "count_rate_pred",
        "min_rows": 1,
        "max_std": None,
        "max_range": None,
    },
    {
        "name": "repeat_county_n_lowvar_s10",
        "key_cols": ["COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_INT"],
        "pred_col": "target_mean",
        "min_rows": 2,
        "max_std": 10.0,
        "max_range": None,
    },
    {
        "name": "repeat_district_noN_min3",
        "key_cols": ["DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME"],
        "pred_col": "target_mean",
        "min_rows": 3,
        "max_std": None,
        "max_range": None,
    },
    {
        "name": "repeat_county_noN_min5",
        "key_cols": ["COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"],
        "pred_col": "target_mean",
        "min_rows": 5,
        "max_std": None,
        "max_range": None,
    },
]

# ------------------------------------------------------------
# 5. Aggregate-count candidate builders
# ------------------------------------------------------------

def fit_aggregate_stats(df_fit, geo_col):
    key_cols = [geo_col, "ASSESSMENT_NAME", "SUBGROUP_NAME"]

    stats = (
        df_fit.groupby(key_cols, dropna=False)
        .agg(
            agg_rows=(TARGET_COL, "size"),
            agg_sum_n=("N_STUDENTS", "sum"),
            agg_sum_k=("k_round", "sum"),
            agg_mean_target=(TARGET_COL, "mean"),
        )
        .reset_index()
    )

    stats["agg_pred_pct"] = np.where(
        stats["agg_sum_n"] > 0,
        100.0 * stats["agg_sum_k"] / stats["agg_sum_n"],
        np.nan,
    )

    return stats

def apply_aggregate_stats(df_query, stats, geo_col, min_rows=2, tol_frac=0.02):
    key_cols = [geo_col, "ASSESSMENT_NAME", "SUBGROUP_NAME"]

    merged = df_query[["row_index", "N_STUDENTS"] + key_cols].merge(stats, on=key_cols, how="left")

    pred = np.full(len(df_query), np.nan, dtype=np.float32)
    covered = np.zeros(len(df_query), dtype=bool)

    qn = pd.to_numeric(merged["N_STUDENTS"], errors="coerce").to_numpy(dtype=np.float64)
    sn = pd.to_numeric(merged["agg_sum_n"], errors="coerce").to_numpy(dtype=np.float64)
    sk = pd.to_numeric(merged["agg_sum_k"], errors="coerce").to_numpy(dtype=np.float64)
    rows = pd.to_numeric(merged["agg_rows"], errors="coerce").to_numpy(dtype=np.float64)

    tol = np.maximum(1.0, float(tol_frac) * np.maximum(sn, 1.0))
    valid = (
        np.isfinite(qn)
        & np.isfinite(sn)
        & np.isfinite(sk)
        & (rows >= float(min_rows))
        & (sn > 0)
        & (np.abs(qn - sn) <= tol)
    )

    vals = np.where(qn > 0, 100.0 * sk / qn, np.nan)
    valid &= np.isfinite(vals)

    pred[valid] = np.clip(vals[valid], 0, 100).astype(np.float32)
    covered[valid] = True

    return pred, covered

aggregate_specs = [
    {"name": "aggregate_district_tol02_min2", "geo_col": "DISTRICT", "min_rows": 2, "tol_frac": 0.02},
    {"name": "aggregate_district_tol01_min2", "geo_col": "DISTRICT", "min_rows": 2, "tol_frac": 0.01},
    {"name": "aggregate_county_tol02_min2", "geo_col": "COUNTY", "min_rows": 2, "tol_frac": 0.02},
    {"name": "aggregate_county_tol01_min2", "geo_col": "COUNTY", "min_rows": 2, "tol_frac": 0.01},
]

# ------------------------------------------------------------
# 6. Build OOF candidates
# ------------------------------------------------------------

folds48b = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train48b)))

candidate_oof = {}
candidate_test_sum = {}
candidate_covered_oof = {}
candidate_covered_test_sum = {}
candidate_meta_rows = []

# Repeat-key candidates.
for spec in repeat_specs:
    name = spec["name"]
    print("\nBuilding repeat candidate:", name)

    pred_oof = np.full(n_train48b, np.nan, dtype=np.float32)
    covered_oof = np.zeros(n_train48b, dtype=bool)

    pred_test_sum = np.zeros(n_test48b, dtype=np.float64)
    covered_test_sum = np.zeros(n_test48b, dtype=np.float64)

    for fold_num, (tr_idx, va_idx) in enumerate(folds48b, start=1):
        stats = fit_repeat_stats(train48b.iloc[tr_idx], spec["key_cols"])

        p_va, c_va = apply_repeat_stats(
            train48b.iloc[va_idx].reset_index(drop=True),
            stats,
            spec["key_cols"],
            spec["pred_col"],
            min_rows=spec["min_rows"],
            max_std=spec["max_std"],
            max_range=spec["max_range"],
        )

        pred_oof[va_idx] = p_va
        covered_oof[va_idx] = c_va

        p_te, c_te = apply_repeat_stats(
            test48b.reset_index(drop=True),
            stats,
            spec["key_cols"],
            spec["pred_col"],
            min_rows=spec["min_rows"],
            max_std=spec["max_std"],
            max_range=spec["max_range"],
        )

        # For test, average predictions only over folds that cover the row.
        pred_test_sum[c_te] += p_te[c_te]
        covered_test_sum[c_te] += 1.0

    pred_test = np.full(n_test48b, np.nan, dtype=np.float32)
    covered_test = covered_test_sum > 0
    pred_test[covered_test] = (pred_test_sum[covered_test] / covered_test_sum[covered_test]).astype(np.float32)

    candidate_oof[name] = pred_oof
    candidate_test_sum[name] = pred_test
    candidate_covered_oof[name] = covered_oof
    candidate_covered_test_sum[name] = covered_test

# Aggregate candidates.
for spec in aggregate_specs:
    name = spec["name"]
    print("\nBuilding aggregate candidate:", name)

    pred_oof = np.full(n_train48b, np.nan, dtype=np.float32)
    covered_oof = np.zeros(n_train48b, dtype=bool)

    pred_test_sum = np.zeros(n_test48b, dtype=np.float64)
    covered_test_sum = np.zeros(n_test48b, dtype=np.float64)

    for fold_num, (tr_idx, va_idx) in enumerate(folds48b, start=1):
        stats = fit_aggregate_stats(train48b.iloc[tr_idx], spec["geo_col"])

        p_va, c_va = apply_aggregate_stats(
            train48b.iloc[va_idx].reset_index(drop=True),
            stats,
            spec["geo_col"],
            min_rows=spec["min_rows"],
            tol_frac=spec["tol_frac"],
        )

        pred_oof[va_idx] = p_va
        covered_oof[va_idx] = c_va

        p_te, c_te = apply_aggregate_stats(
            test48b.reset_index(drop=True),
            stats,
            spec["geo_col"],
            min_rows=spec["min_rows"],
            tol_frac=spec["tol_frac"],
        )

        pred_test_sum[c_te] += p_te[c_te]
        covered_test_sum[c_te] += 1.0

    pred_test = np.full(n_test48b, np.nan, dtype=np.float32)
    covered_test = covered_test_sum > 0
    pred_test[covered_test] = (pred_test_sum[covered_test] / covered_test_sum[covered_test]).astype(np.float32)

    candidate_oof[name] = pred_oof
    candidate_test_sum[name] = pred_test
    candidate_covered_oof[name] = covered_oof
    candidate_covered_test_sum[name] = covered_test

# Candidate coverage summary.
coverage_rows = []

for name in candidate_oof:
    covered = candidate_covered_oof[name]
    p = candidate_oof[name]

    row = {
        "candidate": name,
        "oof_covered_rows": int(covered.sum()),
        "oof_coverage_rate": float(covered.mean()),
        "test_covered_rows": int(candidate_covered_test_sum[name].sum()),
        "test_coverage_rate": float(candidate_covered_test_sum[name].mean()),
    }

    if int(covered.sum()) > 0:
        row["candidate_mse_on_covered"] = float(mean_squared_error(y48b[covered], p[covered]))
        row["base44b_mse_on_covered"] = float(mean_squared_error(y48b[covered], pred44b_oof[covered]))
        row["covered_gain_vs_44b"] = row["base44b_mse_on_covered"] - row["candidate_mse_on_covered"]
    else:
        row["candidate_mse_on_covered"] = np.nan
        row["base44b_mse_on_covered"] = np.nan
        row["covered_gain_vs_44b"] = np.nan

    for tier, mask in tier_masks_oof.items():
        cm = covered & mask
        row[f"covered_{tier}"] = int(cm.sum())
        if int(cm.sum()) > 0:
            row[f"candidate_mse_{tier}_covered"] = float(mean_squared_error(y48b[cm], p[cm]))
            row[f"base_mse_{tier}_covered"] = float(mean_squared_error(y48b[cm], pred44b_oof[cm]))
            row[f"gain_{tier}_covered"] = row[f"base_mse_{tier}_covered"] - row[f"candidate_mse_{tier}_covered"]
        else:
            row[f"candidate_mse_{tier}_covered"] = np.nan
            row[f"base_mse_{tier}_covered"] = np.nan
            row[f"gain_{tier}_covered"] = np.nan

    coverage_rows.append(row)

coverage48b = pd.DataFrame(coverage_rows).sort_values("covered_gain_vs_44b", ascending=False).reset_index(drop=True)

print("\nCandidate coverage and covered-row performance")
print("----------------------------------------------")
display(coverage48b)

# ------------------------------------------------------------
# 7. Blend scan
# ------------------------------------------------------------

lambda_grid = np.unique(
    np.concatenate([
        np.linspace(-0.50, 1.50, 801),
        np.array([0.0, 0.05, 0.10, 0.155, 0.25, 0.50, 0.75, 1.0])
    ])
)

def make_blend_candidate(name, lams, is_test=False):
    if is_test:
        base = pred44b_test.copy().astype(np.float64)
        cand = candidate_test_sum[name].astype(np.float64)
        covered = candidate_covered_test_sum[name]
        masks = tier_masks_test
    else:
        base = pred44b_oof.copy().astype(np.float64)
        cand = candidate_oof[name].astype(np.float64)
        covered = candidate_covered_oof[name]
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        active = covered & mask & np.isfinite(cand)
        if int(active.sum()) == 0:
            continue
        lam = float(lams.get(tier, 0.0))
        pred[active] = base[active] + lam * (cand[active] - base[active])

    return np.clip(pred, 0, 100).astype(np.float32)

def best_lambda_for_active(name, mask):
    covered = candidate_covered_oof[name]
    cand = candidate_oof[name]
    active = covered & mask & np.isfinite(cand)

    if int(active.sum()) == 0:
        return 0.0, np.nan

    y = y48b[active].astype(np.float64)
    base = pred44b_oof[active].astype(np.float64)
    alt = cand[active].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid:
        pred = np.clip(base + float(lam) * (alt - base), 0, 100)
        mse = float(mean_squared_error(y, pred))

        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam, best_mse

def tier_weighted_mse(pred):
    pred = np.clip(pred, 0, 100)
    total = 0.0
    for tier, mask in tier_masks_oof.items():
        total += test_tier_rates[tier] * float(mean_squared_error(y48b[mask], pred[mask]))
    return float(total)

def tier_mse_dict(pred):
    pred = np.clip(pred, 0, 100)
    out = {}
    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y48b[mask], pred[mask]))
        out[f"gain_{tier}_vs_44b"] = (
            float(mean_squared_error(y48b[mask], pred44b_oof[mask]))
            - out[f"mse_{tier}"]
        )
    return out

screen_rows = []

for name in candidate_oof:
    covered = candidate_covered_oof[name]

    if int(covered.sum()) == 0:
        continue

    # Global lambda on all covered rows.
    lam_global, _ = best_lambda_for_active(name, np.ones(n_train48b, dtype=bool))
    lams_global = {"overlap": lam_global, "solver_only": lam_global, "none": lam_global}
    pred_global = make_blend_candidate(name, lams_global, is_test=False)

    row = {
        "candidate": name,
        "strategy": "global_covered",
        "lambda_overlap": lam_global,
        "lambda_solver_only": lam_global,
        "lambda_none": lam_global,
        "covered_rows": int(covered.sum()),
        "oof_mse": float(mean_squared_error(y48b, pred_global)),
        "gain_vs_44b": mse44b - float(mean_squared_error(y48b, pred_global)),
        "test_tier_weighted_mse": tier_weighted_mse(pred_global),
        "weighted_gain_vs_44b": tier_weighted_mse(pred44b_oof) - tier_weighted_mse(pred_global),
    }
    row.update(tier_mse_dict(pred_global))
    screen_rows.append(row)

    # Tier-specific lambdas, allow overlap.
    tier_lams = {}
    for tier, mask in tier_masks_oof.items():
        tier_lams[tier], _ = best_lambda_for_active(name, mask)

    pred_tier = make_blend_candidate(name, tier_lams, is_test=False)

    row = {
        "candidate": name,
        "strategy": "tier_specific",
        "lambda_overlap": tier_lams["overlap"],
        "lambda_solver_only": tier_lams["solver_only"],
        "lambda_none": tier_lams["none"],
        "covered_rows": int(covered.sum()),
        "oof_mse": float(mean_squared_error(y48b, pred_tier)),
        "gain_vs_44b": mse44b - float(mean_squared_error(y48b, pred_tier)),
        "test_tier_weighted_mse": tier_weighted_mse(pred_tier),
        "weighted_gain_vs_44b": tier_weighted_mse(pred44b_oof) - tier_weighted_mse(pred_tier),
    }
    row.update(tier_mse_dict(pred_tier))
    screen_rows.append(row)

    # Protected overlap: only solver_only and none can move.
    prot_lams = {
        "overlap": 0.0,
        "solver_only": tier_lams["solver_only"],
        "none": tier_lams["none"],
    }
    pred_prot = make_blend_candidate(name, prot_lams, is_test=False)

    row = {
        "candidate": name,
        "strategy": "protected_solver_none",
        "lambda_overlap": 0.0,
        "lambda_solver_only": prot_lams["solver_only"],
        "lambda_none": prot_lams["none"],
        "covered_rows": int(covered.sum()),
        "oof_mse": float(mean_squared_error(y48b, pred_prot)),
        "gain_vs_44b": mse44b - float(mean_squared_error(y48b, pred_prot)),
        "test_tier_weighted_mse": tier_weighted_mse(pred_prot),
        "weighted_gain_vs_44b": tier_weighted_mse(pred44b_oof) - tier_weighted_mse(pred_prot),
    }
    row.update(tier_mse_dict(pred_prot))
    screen_rows.append(row)

screen48b = pd.DataFrame(screen_rows).sort_values(["oof_mse", "test_tier_weighted_mse"]).reset_index(drop=True)
screen48b_weighted = screen48b.sort_values(["test_tier_weighted_mse", "oof_mse"]).reset_index(drop=True)

best_oof = screen48b.iloc[0]
best_weighted = screen48b_weighted.iloc[0]

def row_to_lams(row):
    return {
        "overlap": float(row["lambda_overlap"]) if np.isfinite(row["lambda_overlap"]) else 0.0,
        "solver_only": float(row["lambda_solver_only"]) if np.isfinite(row["lambda_solver_only"]) else 0.0,
        "none": float(row["lambda_none"]) if np.isfinite(row["lambda_none"]) else 0.0,
    }

best_oof_pred = make_blend_candidate(best_oof["candidate"], row_to_lams(best_oof), is_test=False)
best_oof_test = make_blend_candidate(best_oof["candidate"], row_to_lams(best_oof), is_test=True)

best_weighted_pred = make_blend_candidate(best_weighted["candidate"], row_to_lams(best_weighted), is_test=False)
best_weighted_test = make_blend_candidate(best_weighted["candidate"], row_to_lams(best_weighted), is_test=True)

# ------------------------------------------------------------
# 8. Fold diagnostics
# ------------------------------------------------------------

def fold_diag(pred, label):
    rows = []

    for fold_num, (_, va_idx) in enumerate(folds48b, start=1):
        row = {
            "label": label,
            "fold": fold_num,
            "mse_44b": float(mean_squared_error(y48b[va_idx], pred44b_oof[va_idx])),
            "mse_candidate": float(mean_squared_error(y48b[va_idx], pred[va_idx])),
        }
        row["gain_vs_44b"] = row["mse_44b"] - row["mse_candidate"]

        for tier, mask_full in tier_masks_oof.items():
            mask = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask.sum())
            if int(mask.sum()) > 0:
                row[f"mse44b_{tier}"] = float(mean_squared_error(
                    y48b[va_idx][mask],
                    pred44b_oof[va_idx][mask],
                ))
                row[f"msecand_{tier}"] = float(mean_squared_error(
                    y48b[va_idx][mask],
                    pred[va_idx][mask],
                ))
                row[f"gain_{tier}"] = row[f"mse44b_{tier}"] - row[f"msecand_{tier}"]

        rows.append(row)

    return pd.DataFrame(rows)

fold_oof48b = fold_diag(best_oof_pred, "best_oof")
fold_weighted48b = fold_diag(best_weighted_pred, "best_weighted")
fold_diag48b = pd.concat([fold_oof48b, fold_weighted48b], axis=0).reset_index(drop=True)

# ------------------------------------------------------------
# 9. Save artifacts
# ------------------------------------------------------------

coverage_path = "model_results/repeat48b_candidate_coverage.csv"
screen_path = "model_results/repeat48b_screen.csv"
fold_path = "model_results/repeat48b_best_fold_diag.csv"

oof_oof_path = "model_results/oof_repeat48b_best_oof.csv"
test_oof_path = "model_results/testpred_repeat48b_best_oof.csv"
submission_oof_path = "submission_repeat48b_best_oof.csv"

oof_weighted_path = "model_results/oof_repeat48b_best_weighted.csv"
test_weighted_path = "model_results/testpred_repeat48b_best_weighted.csv"
submission_weighted_path = "submission_repeat48b_best_weighted.csv"

coverage48b.to_csv(coverage_path, index=False)
screen48b.to_csv(screen_path, index=False)
fold_diag48b.to_csv(fold_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train48b),
    TARGET_COL: y48b,
    "pred_44b": pred44b_oof,
    "pred_clipped": best_oof_pred,
    "tier": tier_oof,
}).to_csv(oof_oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids48b,
    "pred_44b": pred44b_test,
    TARGET_COL: best_oof_test,
    "tier": tier_test,
}).to_csv(test_oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids48b,
    TARGET_COL: best_oof_test,
}).to_csv(submission_oof_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train48b),
    TARGET_COL: y48b,
    "pred_44b": pred44b_oof,
    "pred_clipped": best_weighted_pred,
    "tier": tier_oof,
}).to_csv(oof_weighted_path, index=False)

pd.DataFrame({
    ID_COL: test_ids48b,
    "pred_44b": pred44b_test,
    TARGET_COL: best_weighted_test,
    "tier": tier_test,
}).to_csv(test_weighted_path, index=False)

pd.DataFrame({
    ID_COL: test_ids48b,
    TARGET_COL: best_weighted_test,
}).to_csv(submission_weighted_path, index=False)

# Validate submissions.
for p in [submission_oof_path, submission_weighted_path]:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test48b, 2)
    assert list(sub.columns) == [ID_COL, TARGET_COL]
    assert sub[ID_COL].notna().all()
    assert sub[TARGET_COL].notna().all()
    assert np.isfinite(sub[TARGET_COL]).all()
    assert sub[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 10. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("48B repeat/aggregate validation complete")
print("=" * 90)

print("\nProtected reference")
print("-------------------")
print(f"44B OOF MSE: {mse44b:.6f}")
print("44B public MSE: 30.556")

print("\nCandidate coverage and covered-row performance")
print("----------------------------------------------")
display(coverage48b)

print("\nTop 20 by ordinary OOF")
print("----------------------")
display(screen48b.head(20))

print("\nTop 20 by test-tier-weighted OOF")
print("--------------------------------")
display(screen48b_weighted.head(20))

print("\nBest ordinary OOF candidate")
print("---------------------------")
print(best_oof.to_string())

print("\nBest weighted candidate")
print("-----------------------")
print(best_weighted.to_string())

print("\nFold diagnostics")
print("----------------")
print(fold_diag48b.to_string(index=False))

print("\nSaved files")
print("-----------")
print(coverage_path)
print(screen_path)
print(fold_path)
print(oof_oof_path)
print(test_oof_path)
print(submission_oof_path)
print(oof_weighted_path)
print(test_weighted_path)
print(submission_weighted_path)

print("\nSubmission validation")
print("---------------------")
for p in [submission_oof_path, submission_weighted_path]:
    sub = pd.read_csv(p)
    print(p, sub.shape, sub[TARGET_COL].describe().to_dict())

print("\nDecision rule")
print("-------------")
best_gain = float(best_oof["gain_vs_44b"])
best_weighted_gain = float(best_weighted["weighted_gain_vs_44b"])
min_fold_oof = float(fold_oof48b["gain_vs_44b"].min())
min_fold_weighted = float(fold_weighted48b["gain_vs_44b"].min())

if best_gain >= 0.25 and min_fold_oof >= 0:
    print("Best OOF repeat/aggregate candidate has meaningful stable gain. Consider submission_repeat48b_best_oof.csv.")
elif best_weighted_gain >= 0.25 and min_fold_weighted >= 0:
    print("Best weighted repeat/aggregate candidate has meaningful stable gain. Consider submission_repeat48b_best_weighted.csv.")
elif best_gain > 0.05 or best_weighted_gain > 0.05:
    print("48B has modest gain. Inspect candidate coverage and fold stability before submitting.")
else:
    print("48B does not show enough repeat/aggregate signal. Do not submit; keep 44B protected.")

48B. OOF validation of district/county repeat-key and aggregate predictors

Protected reference
-------------------
44B OOF MSE: 52.780037
44B public MSE: 30.556

Tier coverage
-------------
overlap      train= 53786 test= 27298 44B tier MSE=0.507021
solver_only  train= 27807 test= 17807 44B tier MSE=64.262032
none         train= 63328 test=  3202 44B tier MSE=92.135094

Building repeat candidate: repeat_district_n_mean_min1

Building repeat candidate: repeat_district_n_mean_min2

Building repeat candidate: repeat_district_n_count_min1

Building repeat candidate: repeat_district_n_lowvar_s10

Building repeat candidate: repeat_county_n_mean_min1

Building repeat candidate: repeat_county_n_mean_min2

Building repeat candidate: repeat_county_n_count_min1

Building repeat candidate: repeat_county_n_lowvar_s10

Building repeat candidate: repeat_district_noN_min3

Building repeat candidate: repeat_county_noN_min5

Building aggregate candidate: aggregate_district_tol02_min2

Building aggregat

,candidate,oof_covered_rows,oof_coverage_rate,test_covered_rows,test_coverage_rate,candidate_mse_on_covered,base44b_mse_on_covered,covered_gain_vs_44b,covered_overlap,candidate_mse_overlap_covered,base_mse_overlap_covered,gain_overlap_covered,covered_solver_only,candidate_mse_solver_only_covered,base_mse_solver_only_covered,gain_solver_only_covered,covered_none,candidate_mse_none_covered,base_mse_none_covered,gain_none_covered
0,aggregate_county_tol02_min2,127,0.000876,163,0.003374,417.169891,28.428671,-388.741220,46,517.028259,0.339981,-516.688279,25,513.624390,56.113522,-457.510868,56,292.083405,39.142216,-252.941189
1,repeat_district_noN_min3,71958,0.496533,25412,0.526052,457.766113,56.451042,-401.315071,26565,445.507935,0.490852,-445.017083,13886,447.915314,69.657074,-378.258240,31507,472.443115,97.813385,-374.629730
2,repeat_county_noN_min5,130351,0.899462,45194,0.935558,459.473083,51.835274,-407.637810,48466,446.083862,0.545697,-445.538165,25077,449.336853,63.330601,-386.006252,56808,475.370667,90.518784,-384.851883
3,aggregate_county_tol01_min2,112,0.000773,149,0.003084,444.812012,31.861345,-412.950666,42,555.632141,0.333451,-555.298690,19,544.352295,72.267761,-472.084534,51,316.464752,42.772118,-273.692635
4,aggregate_district_tol02_min2,157,0.001083,193,0.003995,493.526367,42.144306,-451.382061,61,504.563812,1.507072,-503.056740,35,420.557587,65.553741,-355.003845,61,524.356201,69.349907,-455.006294
5,aggregate_district_tol01_min2,144,0.000994,172,0.003561,514.141602,42.928364,-471.213238,56,537.805359,1.570005,-536.235354,34,431.872345,67.473869,-364.398476,54,541.400513,70.363945,-471.036568
6,repeat_county_n_mean_min2,27039,0.186578,10852,0.224647,656.632690,77.764671,-578.868019,9983,647.019348,0.683881,-646.335468,5418,628.557617,103.084305,-525.473312,11638,677.949158,132.096695,-545.852463
7,repeat_county_n_lowvar_s10,7015,0.048406,4729,0.097895,652.145691,58.782280,-593.363411,2633,657.413879,0.557004,-656.856875,1383,615.991577,75.458229,-540.533348,2999,664.192993,102.211533,-561.981461
8,repeat_district_n_mean_min2,8040,0.055479,3151,0.065229,705.095703,77.653854,-627.441849,2999,677.243774,0.757792,-676.485982,1494,671.359985,109.073166,-562.286819,3547,742.854126,129.435867,-613.418259
9,repeat_district_n_lowvar_s10,1949,0.013449,1245,0.025773,685.352417,57.879513,-627.472904,748,648.457703,0.930064,-647.527638,355,719.518433,74.952187,-644.566246,846,703.636353,101.067909,-602.568443



48B repeat/aggregate validation complete

Protected reference
-------------------
44B OOF MSE: 52.780037
44B public MSE: 30.556

Candidate coverage and covered-row performance
----------------------------------------------


,candidate,oof_covered_rows,oof_coverage_rate,test_covered_rows,test_coverage_rate,candidate_mse_on_covered,base44b_mse_on_covered,covered_gain_vs_44b,covered_overlap,candidate_mse_overlap_covered,base_mse_overlap_covered,gain_overlap_covered,covered_solver_only,candidate_mse_solver_only_covered,base_mse_solver_only_covered,gain_solver_only_covered,covered_none,candidate_mse_none_covered,base_mse_none_covered,gain_none_covered
0,aggregate_county_tol02_min2,127,0.000876,163,0.003374,417.169891,28.428671,-388.741220,46,517.028259,0.339981,-516.688279,25,513.624390,56.113522,-457.510868,56,292.083405,39.142216,-252.941189
1,repeat_district_noN_min3,71958,0.496533,25412,0.526052,457.766113,56.451042,-401.315071,26565,445.507935,0.490852,-445.017083,13886,447.915314,69.657074,-378.258240,31507,472.443115,97.813385,-374.629730
2,repeat_county_noN_min5,130351,0.899462,45194,0.935558,459.473083,51.835274,-407.637810,48466,446.083862,0.545697,-445.538165,25077,449.336853,63.330601,-386.006252,56808,475.370667,90.518784,-384.851883
3,aggregate_county_tol01_min2,112,0.000773,149,0.003084,444.812012,31.861345,-412.950666,42,555.632141,0.333451,-555.298690,19,544.352295,72.267761,-472.084534,51,316.464752,42.772118,-273.692635
4,aggregate_district_tol02_min2,157,0.001083,193,0.003995,493.526367,42.144306,-451.382061,61,504.563812,1.507072,-503.056740,35,420.557587,65.553741,-355.003845,61,524.356201,69.349907,-455.006294
5,aggregate_district_tol01_min2,144,0.000994,172,0.003561,514.141602,42.928364,-471.213238,56,537.805359,1.570005,-536.235354,34,431.872345,67.473869,-364.398476,54,541.400513,70.363945,-471.036568
6,repeat_county_n_mean_min2,27039,0.186578,10852,0.224647,656.632690,77.764671,-578.868019,9983,647.019348,0.683881,-646.335468,5418,628.557617,103.084305,-525.473312,11638,677.949158,132.096695,-545.852463
7,repeat_county_n_lowvar_s10,7015,0.048406,4729,0.097895,652.145691,58.782280,-593.363411,2633,657.413879,0.557004,-656.856875,1383,615.991577,75.458229,-540.533348,2999,664.192993,102.211533,-561.981461
8,repeat_district_n_mean_min2,8040,0.055479,3151,0.065229,705.095703,77.653854,-627.441849,2999,677.243774,0.757792,-676.485982,1494,671.359985,109.073166,-562.286819,3547,742.854126,129.435867,-613.418259
9,repeat_district_n_lowvar_s10,1949,0.013449,1245,0.025773,685.352417,57.879513,-627.472904,748,648.457703,0.930064,-647.527638,355,719.518433,74.952187,-644.566246,846,703.636353,101.067909,-602.568443



Top 20 by ordinary OOF
----------------------


,candidate,strategy,lambda_overlap,lambda_solver_only,lambda_none,covered_rows,oof_mse,gain_vs_44b,test_tier_weighted_mse,weighted_gain_vs_44b,mse_overlap,gain_overlap_vs_44b,mse_solver_only,gain_solver_only_vs_44b,mse_none,gain_none_vs_44b
0,repeat_county_noN_min5,tier_specific,0.0000,-0.0025,-0.0150,130351,52.750393,0.029644,30.076063,0.005938,0.507021,0.000000,64.257790,0.004242,92.069107,0.065987
1,repeat_county_noN_min5,protected_solver_none,0.0000,-0.0025,-0.0150,130351,52.750393,0.029644,30.076063,0.005938,0.507021,0.000000,64.257790,0.004242,92.069107,0.065987
2,repeat_district_noN_min3,tier_specific,0.0000,0.0000,-0.0150,71958,52.760960,0.019077,30.079107,0.002894,0.507021,0.000000,64.262032,0.000000,92.091431,0.043663
3,repeat_district_noN_min3,protected_solver_none,0.0000,0.0000,-0.0150,71958,52.760960,0.019077,30.079107,0.002894,0.507021,0.000000,64.262032,0.000000,92.091431,0.043663
4,repeat_county_noN_min5,global_covered,-0.0075,-0.0075,-0.0075,130351,52.765820,0.014217,30.091229,-0.009229,0.529368,-0.022347,64.262077,-0.000046,92.083549,0.051544
5,repeat_county_n_count_min1,tier_specific,0.0000,0.0075,-0.0100,55284,52.768356,0.011681,30.075517,0.006484,0.507021,0.000000,64.248154,0.013878,92.114449,0.020645
6,repeat_county_n_count_min1,protected_solver_none,0.0000,0.0075,-0.0100,55284,52.768356,0.011681,30.075517,0.006484,0.507021,0.000000,64.248154,0.013878,92.114449,0.020645
7,repeat_county_n_mean_min1,tier_specific,0.0000,0.0075,-0.0100,55284,52.768360,0.011677,30.075538,0.006463,0.507021,0.000000,64.248215,0.013817,92.114433,0.020660
8,repeat_county_n_mean_min1,protected_solver_none,0.0000,0.0075,-0.0100,55284,52.768360,0.011677,30.075538,0.006463,0.507021,0.000000,64.248215,0.013817,92.114433,0.020660
9,repeat_district_n_count_min1,tier_specific,0.0000,0.0075,-0.0150,19993,52.769924,0.010113,30.078386,0.003615,0.507021,0.000000,64.255905,0.006126,92.114632,0.020462



Top 20 by test-tier-weighted OOF
--------------------------------


,candidate,strategy,lambda_overlap,lambda_solver_only,lambda_none,covered_rows,oof_mse,gain_vs_44b,test_tier_weighted_mse,weighted_gain_vs_44b,mse_overlap,gain_overlap_vs_44b,mse_solver_only,gain_solver_only_vs_44b,mse_none,gain_none_vs_44b
0,repeat_county_n_mean_min2,tier_specific,0.0000,0.0125,-0.0050,27039,52.775669,0.004368,30.075080,0.006921,0.507021,0.000000,64.243599,0.018433,92.133194,0.001900
1,repeat_county_n_mean_min2,protected_solver_none,0.0000,0.0125,-0.0050,27039,52.775669,0.004368,30.075080,0.006921,0.507021,0.000000,64.243599,0.018433,92.133194,0.001900
2,repeat_county_n_count_min1,tier_specific,0.0000,0.0075,-0.0100,55284,52.768356,0.011681,30.075517,0.006484,0.507021,0.000000,64.248154,0.013878,92.114449,0.020645
3,repeat_county_n_count_min1,protected_solver_none,0.0000,0.0075,-0.0100,55284,52.768356,0.011681,30.075517,0.006484,0.507021,0.000000,64.248154,0.013878,92.114449,0.020645
4,repeat_county_n_mean_min1,tier_specific,0.0000,0.0075,-0.0100,55284,52.768360,0.011677,30.075538,0.006463,0.507021,0.000000,64.248215,0.013817,92.114433,0.020660
5,repeat_county_n_mean_min1,protected_solver_none,0.0000,0.0075,-0.0100,55284,52.768360,0.011677,30.075538,0.006463,0.507021,0.000000,64.248215,0.013817,92.114433,0.020660
6,repeat_county_noN_min5,tier_specific,0.0000,-0.0025,-0.0150,130351,52.750393,0.029644,30.076063,0.005938,0.507021,0.000000,64.257790,0.004242,92.069107,0.065987
7,repeat_county_noN_min5,protected_solver_none,0.0000,-0.0025,-0.0150,130351,52.750393,0.029644,30.076063,0.005938,0.507021,0.000000,64.257790,0.004242,92.069107,0.065987
8,repeat_district_n_count_min1,tier_specific,0.0000,0.0075,-0.0150,19993,52.769924,0.010113,30.078386,0.003615,0.507021,0.000000,64.255905,0.006126,92.114632,0.020462
9,repeat_district_n_count_min1,protected_solver_none,0.0000,0.0075,-0.0150,19993,52.769924,0.010113,30.078386,0.003615,0.507021,0.000000,64.255905,0.006126,92.114632,0.020462



Best ordinary OOF candidate
---------------------------
candidate                  repeat_county_noN_min5
strategy                            tier_specific
lambda_overlap                                0.0
lambda_solver_only                        -0.0025
lambda_none                                -0.015
covered_rows                               130351
oof_mse                                 52.750393
gain_vs_44b                              0.029644
test_tier_weighted_mse                  30.076063
weighted_gain_vs_44b                     0.005938
mse_overlap                              0.507021
gain_overlap_vs_44b                           0.0
mse_solver_only                          64.25779
gain_solver_only_vs_44b                  0.004242
mse_none                                92.069107
gain_none_vs_44b                         0.065987

Best weighted candidate
-----------------------
candidate                  repeat_county_n_mean_min2
strategy                               ti

In [55]:
# ============================================================
# 49A. Target reporting / quantization screen
# ============================================================
#
# Goal:
#   Test whether final predictions should be snapped/rounded to the
#   reported-percentage grid implied by integer proficient counts.
#
# Motivation:
#   The target PERCENT_PROFICIENT appears count-derived:
#
#       y ≈ round(100 * k / N_STUDENTS)
#
#   If hidden test labels are reported percentages rather than continuous
#   rates, continuous submissions may be slightly misaligned.
#
# Candidates:
#   - raw 44B
#   - nearest integer percentage
#   - nearest feasible reported percentage round(100*k/N)
#   - nearest exact count percentage 100*k/N
#   - tier-specific snapping/blending
#
# Protected current best:
#   44B public MSE = 30.556
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("49A. Target reporting / quantization screen")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load protected 44B and useful alternates
# ------------------------------------------------------------

def load_oof_49a(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_49a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

artifact_specs = [
    ("hybrid44b", "model_results/oof_hybrid44b_best.csv", "model_results/testpred_hybrid44b_best.csv"),
    ("hybrid44a", "model_results/oof_hybrid44a_39d_int42b3_best.csv", "model_results/testpred_hybrid44a_39d_int42b3_best.csv"),
    ("account39d", "model_results/oof_account39d_best.csv", "model_results/testpred_account39d_best.csv"),
    ("int42b3", "model_results/oof_int42b3_best.csv", "model_results/testpred_int42b3_best.csv"),
    ("mixed45a", "model_results/oof_mixed45a_best.csv", "model_results/testpred_mixed45a_best.csv"),
]

oof_preds = {}
test_preds = {}
test_ids_49a = None
y_49a = None
loaded_rows = []

for name, oof_path, test_path in artifact_specs:
    if not Path(oof_path).exists() or not Path(test_path).exists():
        print(f"Skipping missing artifact: {name}")
        continue

    oof_df, y_file, p_oof = load_oof_49a(oof_path)
    test_df, ids, p_test = load_test_49a(test_path)

    if y_49a is None:
        y_49a = y_file
    else:
        assert np.max(np.abs(y_49a - y_file)) < 1e-5, f"target mismatch: {name}"

    if test_ids_49a is None:
        test_ids_49a = ids

    oof_preds[name] = p_oof
    test_preds[name] = p_test

    loaded_rows.append({
        "name": name,
        "oof_mse": float(mean_squared_error(y_49a, p_oof)),
    })

loaded49a = pd.DataFrame(loaded_rows).sort_values("oof_mse").reset_index(drop=True)

if "hybrid44b" not in oof_preds:
    raise ValueError("hybrid44b is required.")

n_train = len(y_49a)
n_test = len(test_ids_49a)

pred44b_oof = oof_preds["hybrid44b"]
pred44b_test = test_preds["hybrid44b"]
mse44b = float(mean_squared_error(y_49a, pred44b_oof))

print("\nLoaded artifacts")
print("----------------")
display(loaded49a)

print("\nProtected reference")
print("-------------------")
print(f"44B OOF MSE: {mse44b:.6f}")
print("44B public MSE: 30.556")

# ------------------------------------------------------------
# 2. Raw N_STUDENTS and tier masks
# ------------------------------------------------------------

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("raw_train_te/raw_test_te required.")

n_train_students = pd.to_numeric(raw_train_te["N_STUDENTS"], errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)
n_test_students = pd.to_numeric(raw_test_te["N_STUDENTS"], errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)

n_train_int = np.rint(n_train_students)
n_test_int = np.rint(n_test_students)

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

print("\nTier coverage")
print("-------------")
for tier in ["overlap", "solver_only", "none"]:
    mask = tier_masks_oof[tier]
    print(
        f"{tier:12s} train={int(mask.sum()):6d} test={int(tier_masks_test[tier].sum()):6d} "
        f"44B tier MSE={mean_squared_error(y_49a[mask], pred44b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 3. Target discreteness audit
# ------------------------------------------------------------

target_decimal = np.abs(y_49a - np.rint(y_49a))
integer_share = float((target_decimal < 1e-8).mean())

k_float = y_49a.astype(np.float64) / 100.0 * n_train_students
k_round = np.rint(k_float)
k_error = np.abs(k_float - k_round)

print("\nTarget/reporting audit")
print("----------------------")
print("Share of train targets exactly integer:", integer_share)
print("Target decimal summary:")
print(pd.Series(target_decimal).describe(percentiles=[0.5, 0.9, 0.99]).to_string())
print("\nCount reconstruction error summary | y/100*N - round |:")
print(pd.Series(k_error).describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]).to_string())

# ------------------------------------------------------------
# 4. Quantization helpers
# ------------------------------------------------------------

def round_integer(p):
    return np.clip(np.rint(p), 0, 100).astype(np.float32)

def snap_exact_count_pct(p, n):
    p = np.asarray(p, dtype=np.float64)
    n = np.asarray(n, dtype=np.float64)

    out = p.copy()
    valid = np.isfinite(n) & (n > 0) & np.isfinite(p)

    k = np.rint(np.clip(p[valid], 0, 100) / 100.0 * n[valid])
    k = np.minimum(np.maximum(k, 0), n[valid])
    out[valid] = 100.0 * k / n[valid]

    return np.clip(out, 0, 100).astype(np.float32)

_report_grid_cache = {}

def possible_report_values_for_n(n):
    n_int = int(round(float(n)))
    if n_int <= 0:
        return np.arange(0, 101, dtype=np.float32)
    if n_int in _report_grid_cache:
        return _report_grid_cache[n_int]

    k = np.arange(n_int + 1, dtype=np.float64)
    vals = np.unique(np.rint(100.0 * k / n_int))
    vals = np.clip(vals, 0, 100).astype(np.float32)

    _report_grid_cache[n_int] = vals
    return vals

def snap_reported_percent_grid(p, n):
    p = np.asarray(p, dtype=np.float64)
    n = np.asarray(n, dtype=np.float64)

    out = np.clip(p.copy(), 0, 100)

    # Process by unique N for speed.
    n_rounded = np.rint(n)
    valid = np.isfinite(n_rounded) & (n_rounded > 0) & np.isfinite(p)

    for n_val in np.unique(n_rounded[valid]):
        mask = valid & (n_rounded == n_val)
        vals = possible_report_values_for_n(n_val)

        # nearest grid value
        p_sub = out[mask]
        idx = np.abs(p_sub[:, None] - vals[None, :]).argmin(axis=1)
        out[mask] = vals[idx]

    return np.clip(out, 0, 100).astype(np.float32)

def transform_pred(pred, n, transform_name):
    if transform_name == "raw":
        return np.clip(pred, 0, 100).astype(np.float32)
    if transform_name == "round_integer":
        return round_integer(pred)
    if transform_name == "snap_exact_count_pct":
        return snap_exact_count_pct(pred, n)
    if transform_name == "snap_reported_percent_grid":
        return snap_reported_percent_grid(pred, n)
    raise ValueError(transform_name)

# ------------------------------------------------------------
# 5. Direct transform screen
# ------------------------------------------------------------

transform_names = [
    "raw",
    "round_integer",
    "snap_exact_count_pct",
    "snap_reported_percent_grid",
]

direct_rows = []

for model_name, pred_oof in oof_preds.items():
    pred_test = test_preds[model_name]

    for transform_name in transform_names:
        q_oof = transform_pred(pred_oof, n_train_int, transform_name)
        q_test = transform_pred(pred_test, n_test_int, transform_name)

        row = {
            "model": model_name,
            "transform": transform_name,
            "oof_mse": float(mean_squared_error(y_49a, q_oof)),
            "gain_vs_raw_model": float(mean_squared_error(y_49a, pred_oof) - mean_squared_error(y_49a, q_oof)),
            "gain_vs_44b": mse44b - float(mean_squared_error(y_49a, q_oof)),
        }

        for tier, mask in tier_masks_oof.items():
            row[f"mse_{tier}"] = float(mean_squared_error(y_49a[mask], q_oof[mask]))
            row[f"gain_{tier}_vs_44b"] = (
                float(mean_squared_error(y_49a[mask], pred44b_oof[mask]))
                - row[f"mse_{tier}"]
            )

        direct_rows.append(row)

direct49a = pd.DataFrame(direct_rows).sort_values("oof_mse").reset_index(drop=True)

print("\nDirect quantization screen")
print("--------------------------")
display(direct49a.head(30))

# ------------------------------------------------------------
# 6. Tier-protected quantized blending around 44B
# ------------------------------------------------------------

lambda_grid = np.unique(
    np.concatenate([
        np.linspace(-0.5, 1.5, 801),
        np.array([0.0, 0.05, 0.10, 0.155, 0.25, 0.50, 0.75, 1.0])
    ])
)

def make_tier_blend(q_oof, lams, is_test=False, q_test=None):
    if is_test:
        base = pred44b_test.copy().astype(np.float64)
        alt = q_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred44b_oof.copy().astype(np.float64)
        alt = q_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

def best_lambda_for_mask(q_oof, mask):
    if int(mask.sum()) == 0:
        return 0.0, np.nan

    y = y_49a[mask].astype(np.float64)
    base = pred44b_oof[mask].astype(np.float64)
    alt = q_oof[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid:
        pred = np.clip(base + float(lam) * (alt - base), 0, 100)
        mse = float(mean_squared_error(y, pred))

        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam, best_mse

blend_rows = []
blend_store = {}

for model_name, pred_oof in oof_preds.items():
    pred_test = test_preds[model_name]

    for transform_name in transform_names:
        q_oof = transform_pred(pred_oof, n_train_int, transform_name)
        q_test = transform_pred(pred_test, n_test_int, transform_name)

        # protect overlap by default
        lam_solver, _ = best_lambda_for_mask(q_oof, solver_only_oof)
        lam_none, _ = best_lambda_for_mask(q_oof, none_oof)

        candidate_lam_sets = {
            "solver_only": {"overlap": 0.0, "solver_only": lam_solver, "none": 0.0},
            "none_only": {"overlap": 0.0, "solver_only": 0.0, "none": lam_none},
            "solver_plus_none": {"overlap": 0.0, "solver_only": lam_solver, "none": lam_none},
        }

        # Also allow overlap for same-family quantization only.
        lam_overlap, _ = best_lambda_for_mask(q_oof, overlap_oof)
        candidate_lam_sets["all_tiers"] = {"overlap": lam_overlap, "solver_only": lam_solver, "none": lam_none}

        for strategy, lams in candidate_lam_sets.items():
            pred_blend = make_tier_blend(q_oof, lams, is_test=False)
            test_blend = make_tier_blend(q_oof, lams, is_test=True, q_test=q_test)

            mse = float(mean_squared_error(y_49a, pred_blend))

            row = {
                "model": model_name,
                "transform": transform_name,
                "strategy": strategy,
                "lambda_overlap": float(lams["overlap"]),
                "lambda_solver_only": float(lams["solver_only"]),
                "lambda_none": float(lams["none"]),
                "oof_mse": mse,
                "gain_vs_44b": mse44b - mse,
            }

            for tier, mask in tier_masks_oof.items():
                row[f"mse_{tier}"] = float(mean_squared_error(y_49a[mask], pred_blend[mask]))
                row[f"gain_{tier}_vs_44b"] = (
                    float(mean_squared_error(y_49a[mask], pred44b_oof[mask]))
                    - row[f"mse_{tier}"]
                )

            key = f"{model_name}__{transform_name}__{strategy}"
            row["key"] = key

            blend_rows.append(row)
            blend_store[key] = {
                "oof": pred_blend,
                "test": test_blend,
                "q_oof": q_oof,
                "q_test": q_test,
                "lams": lams,
            }

blend49a = pd.DataFrame(blend_rows).sort_values("oof_mse").reset_index(drop=True)

best_key = blend49a.iloc[0]["key"]
best_oof = blend_store[best_key]["oof"]
best_test = blend_store[best_key]["test"]

# ------------------------------------------------------------
# 7. Fold diagnostics
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

fold_rows = []

for fold_num, (_, va_idx) in enumerate(folds, start=1):
    row = {
        "fold": fold_num,
        "mse_44b": float(mean_squared_error(y_49a[va_idx], pred44b_oof[va_idx])),
        "mse_49a": float(mean_squared_error(y_49a[va_idx], best_oof[va_idx])),
    }
    row["gain_vs_44b"] = row["mse_44b"] - row["mse_49a"]

    for tier, mask_full in tier_masks_oof.items():
        mask = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse44b_{tier}"] = float(mean_squared_error(y_49a[va_idx][mask], pred44b_oof[va_idx][mask]))
            row[f"mse49a_{tier}"] = float(mean_squared_error(y_49a[va_idx][mask], best_oof[va_idx][mask]))
            row[f"gain_{tier}"] = row[f"mse44b_{tier}"] - row[f"mse49a_{tier}"]

    fold_rows.append(row)

fold49a = pd.DataFrame(fold_rows)

# ------------------------------------------------------------
# 8. Save artifacts
# ------------------------------------------------------------

direct_path = "model_results/quant49a_direct_screen.csv"
blend_path = "model_results/quant49a_blend_screen.csv"
fold_path = "model_results/quant49a_best_fold_diag.csv"
oof_path = "model_results/oof_quant49a_best.csv"
test_path = "model_results/testpred_quant49a_best.csv"
submission_path = "submission_quant49a_best.csv"

direct49a.to_csv(direct_path, index=False)
blend49a.to_csv(blend_path, index=False)
fold49a.to_csv(fold_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y_49a,
    "pred_44b": pred44b_oof,
    "pred_clipped": best_oof,
    "tier_overlap": overlap_oof.astype(int),
    "tier_solver_only": solver_only_oof.astype(int),
    "tier_none": none_oof.astype(int),
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids_49a,
    "pred_44b": pred44b_test,
    TARGET_COL: best_test,
    "tier_overlap": overlap_test.astype(int),
    "tier_solver_only": solver_only_test.astype(int),
    "tier_none": none_test.astype(int),
}).to_csv(test_path, index=False)

pd.DataFrame({
    ID_COL: test_ids_49a,
    TARGET_COL: best_test,
}).to_csv(submission_path, index=False)

sub = pd.read_csv(submission_path)
assert sub.shape == (n_test, 2)
assert list(sub.columns) == [ID_COL, TARGET_COL]
assert sub[ID_COL].notna().all()
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

# Save selected named variants for public probes if needed.
named_variants = [
    ("44b_round_integer_all_tiers", "hybrid44b", "round_integer", "all_tiers"),
    ("44b_report_grid_all_tiers", "hybrid44b", "snap_reported_percent_grid", "all_tiers"),
    ("44b_report_grid_solver_none", "hybrid44b", "snap_reported_percent_grid", "solver_plus_none"),
]

for out_name, model_name, transform_name, strategy in named_variants:
    key = f"{model_name}__{transform_name}__{strategy}"
    if key in blend_store:
        pd.DataFrame({
            ID_COL: test_ids_49a,
            TARGET_COL: blend_store[key]["test"],
        }).to_csv(f"submission_quant49a_{out_name}.csv", index=False)

# ------------------------------------------------------------
# 9. Output
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("49A quantization screen complete")
print("=" * 90)

print("\nProtected reference")
print("-------------------")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")

print("\nDirect quantization screen, top 25")
print("----------------------------------")
display(direct49a.head(25))

print("\nProtected/tier blend quantization screen, top 25")
print("------------------------------------------------")
display(blend49a.head(25))

print("\nBest 49A")
print("--------")
print(blend49a.iloc[0].to_string())

print("\nFold diagnostics")
print("----------------")
print(fold49a.to_string(index=False))
print("Min fold gain:", float(fold49a["gain_vs_44b"].min()))

print("\nSaved files")
print("-----------")
print(direct_path)
print(blend_path)
print(fold_path)
print(oof_path)
print(test_path)
print(submission_path)
for out_name, _, _, _ in named_variants:
    print(f"submission_quant49a_{out_name}.csv")

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path)
print("Shape:", sub.shape)
print(sub[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
gain = float(blend49a.iloc[0]["gain_vs_44b"])
min_fold_gain = float(fold49a["gain_vs_44b"].min())

if gain >= 0.05 and min_fold_gain >= 0:
    print("49A gives stable quantization gain. Worth a public probe.")
elif gain > 0:
    print("49A gives marginal quantization gain. Inspect before submitting.")
else:
    print("49A does not improve. Keep 44B protected.")

49A. Target reporting / quantization screen

Loaded artifacts
----------------


,name,oof_mse
0,mixed45a,52.592197
1,hybrid44b,52.780037
2,hybrid44a,52.781635
3,account39d,52.820927
4,int42b3,53.073307



Protected reference
-------------------
44B OOF MSE: 52.780037
44B public MSE: 30.556

Tier coverage
-------------
overlap      train= 53786 test= 27298 44B tier MSE=0.507021
solver_only  train= 27807 test= 17807 44B tier MSE=64.262032
none         train= 63328 test=  3202 44B tier MSE=92.135094

Target/reporting audit
----------------------
Share of train targets exactly integer: 1.0
Target decimal summary:
count    144921.0
mean          0.0
std           0.0
min           0.0
50%           0.0
90%           0.0
99%           0.0
max           0.0

Count reconstruction error summary | y/100*N - round |:
count    144921.000000
mean          0.111673
std           0.117793
min           0.000000
50%           0.080000
90%           0.290000
95%           0.380000
99%           0.480000
99.9%         0.500000
max           0.500000

Direct quantization screen
--------------------------


,model,transform,oof_mse,gain_vs_raw_model,gain_vs_44b,mse_overlap,gain_overlap_vs_44b,mse_solver_only,gain_solver_only_vs_44b,mse_none,gain_none_vs_44b
0,mixed45a,raw,52.592197,0.000000,0.187840,0.507021,0.000000,64.249306,0.012726,91.710831,0.424263
1,mixed45a,round_integer,52.617466,-0.025269,0.162571,0.452999,0.054022,64.288414,-0.026382,91.797356,0.337738
2,hybrid44b,raw,52.780037,0.000000,0.000000,0.507021,0.000000,64.262032,0.000000,92.135094,0.000000
3,hybrid44a,raw,52.781635,0.000000,-0.001598,0.507237,-0.000217,64.269936,-0.007904,92.135094,0.000000
4,hybrid44b,round_integer,52.815319,-0.035282,-0.035282,0.452999,0.054022,64.350739,-0.088707,92.222763,-0.087669
5,hybrid44a,round_integer,52.815796,-0.034161,-0.035759,0.451363,0.055658,64.356384,-0.094353,92.222763,-0.087669
6,account39d,raw,52.820927,0.000000,-0.040890,0.595325,-0.088304,64.304314,-0.042282,92.135094,0.000000
7,account39d,round_integer,52.848179,-0.027252,-0.068142,0.562674,-0.055654,64.309853,-0.047821,92.222763,-0.087669
8,int42b3,raw,53.073307,0.000000,-0.293270,0.507233,-0.000212,65.261948,-0.999916,92.366982,-0.231888
9,int42b3,round_integer,53.099358,-0.026051,-0.319321,0.450991,0.056030,65.354301,-1.092270,92.433807,-0.298714



49A quantization screen complete

Protected reference
-------------------
44B OOF MSE: 52.780037 | public 30.556

Direct quantization screen, top 25
----------------------------------


,model,transform,oof_mse,gain_vs_raw_model,gain_vs_44b,mse_overlap,gain_overlap_vs_44b,mse_solver_only,gain_solver_only_vs_44b,mse_none,gain_none_vs_44b
0,mixed45a,raw,52.592197,0.000000,0.187840,0.507021,0.000000,64.249306,0.012726,91.710831,0.424263
1,mixed45a,round_integer,52.617466,-0.025269,0.162571,0.452999,0.054022,64.288414,-0.026382,91.797356,0.337738
2,hybrid44b,raw,52.780037,0.000000,0.000000,0.507021,0.000000,64.262032,0.000000,92.135094,0.000000
3,hybrid44a,raw,52.781635,0.000000,-0.001598,0.507237,-0.000217,64.269936,-0.007904,92.135094,0.000000
4,hybrid44b,round_integer,52.815319,-0.035282,-0.035282,0.452999,0.054022,64.350739,-0.088707,92.222763,-0.087669
5,hybrid44a,round_integer,52.815796,-0.034161,-0.035759,0.451363,0.055658,64.356384,-0.094353,92.222763,-0.087669
6,account39d,raw,52.820927,0.000000,-0.040890,0.595325,-0.088304,64.304314,-0.042282,92.135094,0.000000
7,account39d,round_integer,52.848179,-0.027252,-0.068142,0.562674,-0.055654,64.309853,-0.047821,92.222763,-0.087669
8,int42b3,raw,53.073307,0.000000,-0.293270,0.507233,-0.000212,65.261948,-0.999916,92.366982,-0.231888
9,int42b3,round_integer,53.099358,-0.026051,-0.319321,0.450991,0.056030,65.354301,-1.092270,92.433807,-0.298714



Protected/tier blend quantization screen, top 25
------------------------------------------------


,model,transform,strategy,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_44b,mse_overlap,gain_overlap_vs_44b,mse_solver_only,gain_solver_only_vs_44b,mse_none,gain_none_vs_44b,key
0,mixed45a,raw,all_tiers,-0.5000,1.0300,0.9975,52.592197,0.187840,0.507021,0.000000,64.249290,0.012741,91.710823,0.424271,mixed45a__raw__all_tiers
1,mixed45a,raw,solver_plus_none,0.0000,1.0300,0.9975,52.592197,0.187840,0.507021,0.000000,64.249290,0.012741,91.710823,0.424271,mixed45a__raw__solver_plus_none
2,mixed45a,raw,none_only,0.0000,0.0000,0.9975,52.594643,0.185394,0.507021,0.000000,64.262032,0.000000,91.710823,0.424271,mixed45a__raw__none_only
3,mixed45a,round_integer,all_tiers,0.8525,0.3600,0.8300,52.603069,0.176968,0.451363,0.055658,64.249878,0.012154,91.782730,0.352364,mixed45a__round_integer__all_tiers
4,mixed45a,round_integer,solver_plus_none,0.0000,0.3600,0.8300,52.623730,0.156307,0.507021,0.000000,64.249878,0.012154,91.782730,0.352364,mixed45a__round_integer__solver_plus_none
5,mixed45a,round_integer,none_only,0.0000,0.0000,0.8300,52.626057,0.153980,0.507021,0.000000,64.262032,0.000000,91.782730,0.352364,mixed45a__round_integer__none_only
6,mixed45a,snap_reported_percent_grid,all_tiers,0.8600,-0.0575,0.1225,52.737095,0.042942,0.450058,0.056963,64.254158,0.007874,92.088646,0.046448,mixed45a__snap_reported_percent_grid__all_tiers
7,hybrid44a,snap_reported_percent_grid,all_tiers,0.8600,-0.1250,0.0275,52.750919,0.029118,0.449992,0.057028,64.225128,0.036903,92.133095,0.001999,hybrid44a__snap_reported_percent_grid__all_tiers
8,int42b3,round_integer,all_tiers,0.8650,-0.0200,0.1900,52.751041,0.028996,0.449598,0.057422,64.261574,0.000458,92.117706,0.017387,int42b3__round_integer__all_tiers
9,hybrid44b,snap_reported_percent_grid,all_tiers,0.8600,-0.0925,0.0275,52.754230,0.025806,0.450058,0.056963,64.242279,0.019753,92.133095,0.001999,hybrid44b__snap_reported_percent_grid__all_tiers



Best 49A
--------
model                                      mixed45a
transform                                       raw
strategy                                  all_tiers
lambda_overlap                                 -0.5
lambda_solver_only                             1.03
lambda_none                                  0.9975
oof_mse                                   52.592197
gain_vs_44b                                 0.18784
mse_overlap                                0.507021
gain_overlap_vs_44b                             0.0
mse_solver_only                            64.24929
gain_solver_only_vs_44b                    0.012741
mse_none                                  91.710823
gain_none_vs_44b                           0.424271
key                        mixed45a__raw__all_tiers

Fold diagnostics
----------------
 fold   mse_44b   mse_49a  gain_vs_44b  n_overlap  mse44b_overlap  mse49a_overlap  gain_overlap  n_solver_only  mse44b_solver_only  mse49a_solver_only  gain_solver_on

In [56]:
# ============================================================
# 50A. Low-rank matrix-factorization residual model
# ============================================================
#
# Protected public-best:
#   44B public MSE = 30.556
#
# Motivation:
#   Accounting identities are mostly exhausted.
#   Repeat-key/aggregate predictors were noisy.
#   Quantization did not help.
#
# New signal family:
#   Low-rank latent structure over:
#
#       SCHOOL × (ASSESSMENT_NAME × SUBGROUP_NAME)
#
# Model:
#   Work on arcsine scale.
#
#   z_i = arcsin(sqrt(y_i / 100))
#   b_i = arcsin(sqrt(pred44b_i / 100))
#   residual_i = z_i - b_i
#
#   residual_i ≈ global
#              + school_bias[SCHOOL_i]
#              + assessment_subgroup_bias[ASSESSMENT_NAME_i, SUBGROUP_NAME_i]
#              + school_factor[SCHOOL_i] · assessment_subgroup_factor[ASSESSMENT_NAME_i, SUBGROUP_NAME_i]
#
# Then:
#   candidate_i = 100 * sin(b_i + residual_hat_i)^2
#
# Final protected blend:
#   overlap unchanged
#   solver_only / none get OOF-tuned blend toward candidate
#
# This is a genuine matrix-factorization / recommender-style branch.
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path

from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("50A. Low-rank matrix-factorization residual model")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load protected 44B
# ------------------------------------------------------------

def load_oof_50a(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column found in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_50a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

oof44b_df, y50a, pred44b_oof = load_oof_50a("model_results/oof_hybrid44b_best.csv")
test44b_df, test_ids50a, pred44b_test = load_test_50a("model_results/testpred_hybrid44b_best.csv")

n_train50a = len(y50a)
n_test50a = len(pred44b_test)

mse44b = float(mean_squared_error(y50a, pred44b_oof))

print("\nProtected reference")
print("-------------------")
print(f"44B OOF MSE: {mse44b:.6f}")
print("44B public MSE: 30.556")

# ------------------------------------------------------------
# 2. Required raw columns
# ------------------------------------------------------------

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("raw_train_te/raw_test_te required. Rerun setup/recovery cells first.")

for c in ["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]:
    if c not in raw_train_te.columns or c not in raw_test_te.columns:
        raise ValueError(f"Missing required raw column: {c}")

def clean_str_50a(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str).to_numpy()

school_train = clean_str_50a(raw_train_te["SCHOOL"])
school_test = clean_str_50a(raw_test_te["SCHOOL"])

assessment_train = clean_str_50a(raw_train_te["ASSESSMENT_NAME"])
assessment_test = clean_str_50a(raw_test_te["ASSESSMENT_NAME"])

subgroup_train = clean_str_50a(raw_train_te["SUBGROUP_NAME"])
subgroup_test = clean_str_50a(raw_test_te["SUBGROUP_NAME"])

asg_train = np.array([f"{a}||{s}" for a, s in zip(assessment_train, subgroup_train)], dtype=object)
asg_test = np.array([f"{a}||{s}" for a, s in zip(assessment_test, subgroup_test)], dtype=object)

n_students_train = (
    pd.to_numeric(raw_train_te["N_STUDENTS"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .to_numpy(dtype=np.float64)
)

n_students_test = (
    pd.to_numeric(raw_test_te["N_STUDENTS"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .to_numpy(dtype=np.float64)
)

# Category maps built from train+test labels only. This uses no target values.
all_schools = pd.Index(pd.Series(np.concatenate([school_train, school_test])).astype(str).unique())
all_asg = pd.Index(pd.Series(np.concatenate([asg_train, asg_test])).astype(str).unique())

school_to_idx = {v: i for i, v in enumerate(all_schools)}
asg_to_idx = {v: i for i, v in enumerate(all_asg)}

school_idx_train = np.array([school_to_idx[x] for x in school_train], dtype=np.int32)
school_idx_test = np.array([school_to_idx[x] for x in school_test], dtype=np.int32)

asg_idx_train = np.array([asg_to_idx[x] for x in asg_train], dtype=np.int32)
asg_idx_test = np.array([asg_to_idx[x] for x in asg_test], dtype=np.int32)

n_schools = len(all_schools)
n_asg = len(all_asg)

print("\nMatrix dimensions")
print("-----------------")
print("n_schools:", n_schools)
print("n_assessment_subgroups:", n_asg)
print("train observed rows:", n_train50a)
print("test rows:", n_test50a)

# ------------------------------------------------------------
# 3. Recover accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

print("\nTier coverage")
print("-------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"44B MSE={mean_squared_error(y50a[mask], pred44b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 4. Transform helpers
# ------------------------------------------------------------

def pct_to_arc_50a(pct):
    p = np.clip(np.asarray(pct, dtype=np.float64) / 100.0, 1e-6, 1 - 1e-6)
    return np.arcsin(np.sqrt(p)).astype(np.float32)

def arc_to_pct_50a(z):
    p = np.sin(np.asarray(z, dtype=np.float64)) ** 2
    return np.clip(100.0 * p, 0, 100).astype(np.float32)

z_y = pct_to_arc_50a(y50a)
z_44b_oof = pct_to_arc_50a(pred44b_oof)
z_44b_test = pct_to_arc_50a(pred44b_test)

resid_z = (z_y - z_44b_oof).astype(np.float32)

# ------------------------------------------------------------
# 5. Bias and low-rank fit helpers
# ------------------------------------------------------------

def make_weights(idx, scheme):
    idx = np.asarray(idx, dtype=np.int64)

    n = n_students_train[idx].astype(np.float64)
    valid_n = np.isfinite(n) & (n > 0)

    if valid_n.any():
        med_n = np.nanmedian(n[valid_n])
    else:
        med_n = 30.0

    n = np.where(valid_n, n, med_n)

    if scheme == "uniform":
        w = np.ones(len(idx), dtype=np.float64)
    elif scheme == "sqrt_n":
        w = np.sqrt(np.clip(n, 1, 500))
    elif scheme == "solver_public":
        w = np.sqrt(np.clip(n, 1, 500))
        local_overlap = overlap_oof[idx]
        local_solver = solver_only_oof[idx]
        local_none = none_oof[idx]
        w[local_overlap] *= 0.10
        w[local_solver] *= 1.25
        w[local_none] *= 0.75
    elif scheme == "none_focus":
        w = np.sqrt(np.clip(n, 1, 500))
        local_overlap = overlap_oof[idx]
        local_solver = solver_only_oof[idx]
        local_none = none_oof[idx]
        w[local_overlap] *= 0.05
        w[local_solver] *= 0.75
        w[local_none] *= 1.50
    else:
        raise ValueError(f"Unknown weight scheme: {scheme}")

    return w.astype(np.float64)

def fit_biases(idx, reg_school=20.0, reg_asg=20.0, weight_scheme="sqrt_n", n_iter=3):
    idx = np.asarray(idx, dtype=np.int64)

    s_idx = school_idx_train[idx]
    a_idx = asg_idx_train[idx]
    r = resid_z[idx].astype(np.float64)
    w = make_weights(idx, weight_scheme)

    global_bias = float(np.sum(w * r) / max(np.sum(w), 1e-12))

    school_bias = np.zeros(n_schools, dtype=np.float64)
    asg_bias = np.zeros(n_asg, dtype=np.float64)

    for _ in range(n_iter):
        # Update school bias.
        residual_for_school = r - global_bias - asg_bias[a_idx]
        sum_w_s = np.bincount(s_idx, weights=w, minlength=n_schools)
        sum_wr_s = np.bincount(s_idx, weights=w * residual_for_school, minlength=n_schools)
        school_bias = sum_wr_s / (sum_w_s + float(reg_school))

        # Update ASG bias.
        residual_for_asg = r - global_bias - school_bias[s_idx]
        sum_w_a = np.bincount(a_idx, weights=w, minlength=n_asg)
        sum_wr_a = np.bincount(a_idx, weights=w * residual_for_asg, minlength=n_asg)
        asg_bias = sum_wr_a / (sum_w_a + float(reg_asg))

    debiased = r - global_bias - school_bias[s_idx] - asg_bias[a_idx]

    return global_bias, school_bias.astype(np.float32), asg_bias.astype(np.float32), debiased.astype(np.float32)

def fit_svd_matrix(idx, debiased_values, rank, random_state):
    idx = np.asarray(idx, dtype=np.int64)

    rows = school_idx_train[idx]
    cols = asg_idx_train[idx]

    M = sparse.csr_matrix(
        (debiased_values.astype(np.float32), (rows, cols)),
        shape=(n_schools, n_asg),
        dtype=np.float32,
    )

    rank_eff = int(min(rank, max(1, min(n_schools, n_asg) - 1)))

    if rank_eff < 1:
        return None, None

    svd = TruncatedSVD(
        n_components=rank_eff,
        n_iter=7,
        random_state=random_state,
    )

    row_factors = svd.fit_transform(M).astype(np.float32)
    col_components = svd.components_.astype(np.float32)

    return row_factors, col_components

def predict_factorized(
    base_z,
    s_idx,
    a_idx,
    global_bias,
    school_bias,
    asg_bias,
    row_factors,
    col_components,
    factor_scale=1.0,
    bias_scale=1.0,
):
    bias_part = global_bias + school_bias[s_idx].astype(np.float64) + asg_bias[a_idx].astype(np.float64)
    bias_part = float(bias_scale) * bias_part

    if row_factors is None or col_components is None:
        factor_part = np.zeros(len(s_idx), dtype=np.float64)
    else:
        # row_factors includes singular values; components are V.
        factor_part = np.sum(row_factors[s_idx] * col_components[:, a_idx].T, axis=1)
        factor_part = float(factor_scale) * factor_part.astype(np.float64)

    z_pred = base_z.astype(np.float64) + bias_part + factor_part

    return arc_to_pct_50a(z_pred)

# ------------------------------------------------------------
# 6. Configs
# ------------------------------------------------------------

configs50a = []

for rank in [2, 4, 8, 12]:
    for reg in [10.0, 30.0, 100.0]:
        configs50a.append({
            "name": f"mf50a_rank{rank}_reg{str(reg).replace('.', 'p')}_sqrt",
            "rank": rank,
            "reg_school": reg,
            "reg_asg": reg,
            "weight_scheme": "sqrt_n",
            "factor_scale": 1.0,
            "bias_scale": 1.0,
        })

# Add a couple focused configs.
configs50a += [
    {
        "name": "mf50a_rank8_reg30_solver_public",
        "rank": 8,
        "reg_school": 30.0,
        "reg_asg": 30.0,
        "weight_scheme": "solver_public",
        "factor_scale": 1.0,
        "bias_scale": 1.0,
    },
    {
        "name": "mf50a_rank8_reg30_none_focus",
        "rank": 8,
        "reg_school": 30.0,
        "reg_asg": 30.0,
        "weight_scheme": "none_focus",
        "factor_scale": 1.0,
        "bias_scale": 1.0,
    },
]

print("\n50A configs")
print("-----------")
for cfg in configs50a:
    print(cfg)

# ------------------------------------------------------------
# 7. OOF training
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train50a)))

cand_oof = {cfg["name"]: np.full(n_train50a, np.nan, dtype=np.float32) for cfg in configs50a}
cand_test_sum = {cfg["name"]: np.zeros(n_test50a, dtype=np.float64) for cfg in configs50a}
fold_rows = []

for cfg in configs50a:
    name = cfg["name"]
    print("\n" + "=" * 90)
    print("Training", name)
    print("=" * 90)

    for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
        global_bias, school_bias, asg_bias, debiased = fit_biases(
            tr_idx,
            reg_school=cfg["reg_school"],
            reg_asg=cfg["reg_asg"],
            weight_scheme=cfg["weight_scheme"],
            n_iter=3,
        )

        row_factors, col_components = fit_svd_matrix(
            tr_idx,
            debiased,
            rank=cfg["rank"],
            random_state=RANDOM_STATE + fold_num + cfg["rank"],
        )

        p_va = predict_factorized(
            base_z=z_44b_oof[va_idx],
            s_idx=school_idx_train[va_idx],
            a_idx=asg_idx_train[va_idx],
            global_bias=global_bias,
            school_bias=school_bias,
            asg_bias=asg_bias,
            row_factors=row_factors,
            col_components=col_components,
            factor_scale=cfg["factor_scale"],
            bias_scale=cfg["bias_scale"],
        )

        p_te = predict_factorized(
            base_z=z_44b_test,
            s_idx=school_idx_test,
            a_idx=asg_idx_test,
            global_bias=global_bias,
            school_bias=school_bias,
            asg_bias=asg_bias,
            row_factors=row_factors,
            col_components=col_components,
            factor_scale=cfg["factor_scale"],
            bias_scale=cfg["bias_scale"],
        )

        cand_oof[name][va_idx] = p_va
        cand_test_sum[name] += p_te.astype(np.float64)

        row = {
            "config": name,
            "fold": fold_num,
            "mse_44b": float(mean_squared_error(y50a[va_idx], pred44b_oof[va_idx])),
            "mse_direct_mf": float(mean_squared_error(y50a[va_idx], p_va)),
        }
        row["gain_direct_vs_44b"] = row["mse_44b"] - row["mse_direct_mf"]

        for tier, mask_full in tier_masks_oof.items():
            mask = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask.sum())
            if int(mask.sum()) > 0:
                row[f"mse44b_{tier}"] = float(mean_squared_error(y50a[va_idx][mask], pred44b_oof[va_idx][mask]))
                row[f"msemf_{tier}"] = float(mean_squared_error(y50a[va_idx][mask], p_va[mask]))
                row[f"gain_direct_{tier}"] = row[f"mse44b_{tier}"] - row[f"msemf_{tier}"]

        fold_rows.append(row)

        print(
            f"fold {fold_num} | direct MF MSE {row['mse_direct_mf']:.6f} "
            f"| gain {row['gain_direct_vs_44b']:.6f}"
        )

        del row_factors, col_components, school_bias, asg_bias, debiased
        gc.collect()

fold_metrics50a = pd.DataFrame(fold_rows)

# ------------------------------------------------------------
# 8. Protected blend scan
# ------------------------------------------------------------

lambda_grid = np.unique(
    np.concatenate([
        np.linspace(-1.0, 1.5, 1001),
        np.array([0.0, 0.05, 0.10, 0.155, 0.25, 0.50, 0.75, 1.0])
    ])
)

def best_lambda_for_mask(candidate_pred, mask):
    if int(mask.sum()) == 0:
        return 0.0, np.nan

    y = y50a[mask].astype(np.float64)
    base = pred44b_oof[mask].astype(np.float64)
    alt = candidate_pred[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid:
        p = np.clip(base + float(lam) * (alt - base), 0, 100)
        mse = float(mean_squared_error(y, p))

        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam, best_mse

def make_blend(candidate_oof, candidate_test, lams, is_test=False):
    if is_test:
        base = pred44b_test.astype(np.float64)
        alt = candidate_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred44b_oof.astype(np.float64)
        alt = candidate_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

def tier_metrics(pred):
    out = {}
    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y50a[mask], pred[mask]))
        out[f"gain_{tier}_vs_44b"] = (
            float(mean_squared_error(y50a[mask], pred44b_oof[mask]))
            - out[f"mse_{tier}"]
        )
    return out

screen_rows = []

for cfg in configs50a:
    name = cfg["name"]
    p_oof = cand_oof[name]
    p_test = (cand_test_sum[name] / N_SPLITS).astype(np.float32)

    if not np.isfinite(p_oof).all():
        print("Skipping incomplete:", name)
        continue

    direct_mse = float(mean_squared_error(y50a, p_oof))

    lam_overlap, _ = best_lambda_for_mask(p_oof, overlap_oof)
    lam_solver, _ = best_lambda_for_mask(p_oof, solver_only_oof)
    lam_none, _ = best_lambda_for_mask(p_oof, none_oof)

    strategies = {
        "solver_only": {
            "overlap": 0.0,
            "solver_only": lam_solver,
            "none": 0.0,
        },
        "none_only": {
            "overlap": 0.0,
            "solver_only": 0.0,
            "none": lam_none,
        },
        "solver_plus_none": {
            "overlap": 0.0,
            "solver_only": lam_solver,
            "none": lam_none,
        },
        "all_tiers": {
            "overlap": lam_overlap,
            "solver_only": lam_solver,
            "none": lam_none,
        },
    }

    for strategy_name, lams in strategies.items():
        pred = make_blend(p_oof, p_test, lams, is_test=False)
        mse = float(mean_squared_error(y50a, pred))

        row = {
            "config": name,
            "strategy": strategy_name,
            "direct_oof_mse": direct_mse,
            "lambda_overlap": lams["overlap"],
            "lambda_solver_only": lams["solver_only"],
            "lambda_none": lams["none"],
            "oof_mse": mse,
            "gain_vs_44b": mse44b - mse,
        }
        row.update(tier_metrics(pred))
        screen_rows.append(row)

screen50a = pd.DataFrame(screen_rows).sort_values("oof_mse").reset_index(drop=True)

best50a = screen50a.iloc[0]
best_config = best50a["config"]
best_test_direct = (cand_test_sum[best_config] / N_SPLITS).astype(np.float32)

best_lams = {
    "overlap": float(best50a["lambda_overlap"]),
    "solver_only": float(best50a["lambda_solver_only"]),
    "none": float(best50a["lambda_none"]),
}

final_oof50a = make_blend(cand_oof[best_config], best_test_direct, best_lams, is_test=False)
final_test50a = make_blend(cand_oof[best_config], best_test_direct, best_lams, is_test=True)

# ------------------------------------------------------------
# 9. Fold diagnostics
# ------------------------------------------------------------

fold_diag_rows = []

for fold_num, (_, va_idx) in enumerate(folds, start=1):
    row = {
        "fold": fold_num,
        "mse_44b": float(mean_squared_error(y50a[va_idx], pred44b_oof[va_idx])),
        "mse_50a": float(mean_squared_error(y50a[va_idx], final_oof50a[va_idx])),
    }
    row["gain_vs_44b"] = row["mse_44b"] - row["mse_50a"]

    for tier, mask_full in tier_masks_oof.items():
        mask = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse44b_{tier}"] = float(mean_squared_error(y50a[va_idx][mask], pred44b_oof[va_idx][mask]))
            row[f"mse50a_{tier}"] = float(mean_squared_error(y50a[va_idx][mask], final_oof50a[va_idx][mask]))
            row[f"gain_{tier}"] = row[f"mse44b_{tier}"] - row[f"mse50a_{tier}"]

    fold_diag_rows.append(row)

fold_diag50a = pd.DataFrame(fold_diag_rows)

# ------------------------------------------------------------
# 10. Save artifacts
# ------------------------------------------------------------

screen_path = "model_results/mf50a_screen.csv"
fold_metrics_path = "model_results/mf50a_fold_metrics.csv"
fold_diag_path = "model_results/mf50a_best_fold_diag.csv"
oof_path = "model_results/oof_mf50a_best.csv"
test_path = "model_results/testpred_mf50a_best.csv"
submission_path = "submission_mf50a_best.csv"

screen50a.to_csv(screen_path, index=False)
fold_metrics50a.to_csv(fold_metrics_path, index=False)
fold_diag50a.to_csv(fold_diag_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train50a),
    TARGET_COL: y50a,
    "pred_44b": pred44b_oof,
    "pred_mf_direct": cand_oof[best_config],
    "pred_clipped": final_oof50a,
    "tier_overlap": overlap_oof.astype(int),
    "tier_solver_only": solver_only_oof.astype(int),
    "tier_none": none_oof.astype(int),
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids50a,
    "pred_44b": pred44b_test,
    "pred_mf_direct": best_test_direct,
    TARGET_COL: final_test50a,
    "tier_overlap": overlap_test.astype(int),
    "tier_solver_only": solver_only_test.astype(int),
    "tier_none": none_test.astype(int),
}).to_csv(test_path, index=False)

pd.DataFrame({
    ID_COL: test_ids50a,
    TARGET_COL: final_test50a,
}).to_csv(submission_path, index=False)

sub = pd.read_csv(submission_path)
assert sub.shape == (n_test50a, 2)
assert list(sub.columns) == [ID_COL, TARGET_COL]
assert sub[ID_COL].notna().all()
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

# ------------------------------------------------------------
# 11. Output
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("50A low-rank matrix-factorization residual model complete")
print("=" * 90)

print("\nProtected reference")
print("-------------------")
print(f"44B OOF MSE: {mse44b:.6f}")
print("44B public MSE: 30.556")

print("\nDirect MF fold metrics")
print("----------------------")
display(fold_metrics50a)

print("\nTop 25 protected blend candidates")
print("---------------------------------")
display(screen50a.head(25))

print("\nBest 50A")
print("--------")
print(best50a.to_string())
print(f"\nBest 50A OOF MSE: {mean_squared_error(y50a, final_oof50a):.6f}")
print(f"Gain vs 44B:       {mse44b - mean_squared_error(y50a, final_oof50a):.6f}")

print("\nFold diagnostics")
print("----------------")
print(fold_diag50a.to_string(index=False))
print("Min fold gain:", float(fold_diag50a["gain_vs_44b"].min()))

print("\nSaved files")
print("-----------")
print(screen_path)
print(fold_metrics_path)
print(fold_diag_path)
print(oof_path)
print(test_path)
print(submission_path)

print("\nSubmission validation")
print("---------------------")
print("File:", submission_path)
print("Shape:", sub.shape)
print(sub[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
gain = float(mse44b - mean_squared_error(y50a, final_oof50a))
min_fold_gain = float(fold_diag50a["gain_vs_44b"].min())

if gain >= 0.25 and min_fold_gain >= 0:
    print("50A has meaningful stable gain. Worth a public probe.")
elif gain >= 0.05 and min_fold_gain >= 0:
    print("50A has modest stable gain. Consider if slots remain.")
elif gain > 0:
    print("50A has small or unstable gain. Save artifact; likely do not submit.")
else:
    print("50A does not improve over 44B. Keep 44B protected.")

50A. Low-rank matrix-factorization residual model

Protected reference
-------------------
44B OOF MSE: 52.780037
44B public MSE: 30.556

Matrix dimensions
-----------------
n_schools: 4469
n_assessment_subgroups: 132
train observed rows: 144921
test rows: 48307

Tier coverage
-------------
overlap      train= 53786 test= 27298 44B MSE=0.507021
solver_only  train= 27807 test= 17807 44B MSE=64.262032
none         train= 63328 test=  3202 44B MSE=92.135094

50A configs
-----------
{'name': 'mf50a_rank2_reg10p0_sqrt', 'rank': 2, 'reg_school': 10.0, 'reg_asg': 10.0, 'weight_scheme': 'sqrt_n', 'factor_scale': 1.0, 'bias_scale': 1.0}
{'name': 'mf50a_rank2_reg30p0_sqrt', 'rank': 2, 'reg_school': 30.0, 'reg_asg': 30.0, 'weight_scheme': 'sqrt_n', 'factor_scale': 1.0, 'bias_scale': 1.0}
{'name': 'mf50a_rank2_reg100p0_sqrt', 'rank': 2, 'reg_school': 100.0, 'reg_asg': 100.0, 'weight_scheme': 'sqrt_n', 'factor_scale': 1.0, 'bias_scale': 1.0}
{'name': 'mf50a_rank4_reg10p0_sqrt', 'rank': 4, 'reg_scho

,config,fold,mse_44b,mse_direct_mf,gain_direct_vs_44b,n_overlap,mse44b_overlap,msemf_overlap,gain_direct_overlap,n_solver_only,mse44b_solver_only,msemf_solver_only,gain_direct_solver_only,n_none,mse44b_none,msemf_none,gain_direct_none
0,mf50a_rank2_reg10p0_sqrt,1,53.054913,56.363224,-3.308311,10837,0.418953,2.594359,-2.175406,5496,62.401917,65.642479,-3.240562,12652,94.079628,98.387749,-4.308121
1,mf50a_rank2_reg10p0_sqrt,2,50.813717,54.447952,-3.634235,10622,0.394440,2.616906,-2.222466,5623,61.360851,64.474312,-3.113461,12739,88.198669,93.239944,-5.041275
2,mf50a_rank2_reg10p0_sqrt,3,53.784950,57.154560,-3.369610,10838,0.453856,2.664881,-2.211025,5524,69.692871,72.086891,-2.394020,12622,92.616127,97.407524,-4.791397
3,mf50a_rank2_reg10p0_sqrt,4,52.625896,56.232792,-3.606895,10796,0.571821,2.702220,-2.130399,5571,61.890247,64.148888,-2.258640,12617,93.076424,98.542030,-5.465607
4,mf50a_rank2_reg10p0_sqrt,5,53.620712,56.873981,-3.253269,10693,0.696569,2.838497,-2.141928,5593,66.005226,68.638687,-2.633461,12698,92.733269,97.195396,-4.462128
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,mf50a_rank8_reg30_none_focus,1,53.054913,58.047405,-4.992493,10837,0.418953,5.263987,-4.845034,5496,62.401917,66.934410,-4.532494,12652,94.079628,99.398254,-5.318626
66,mf50a_rank8_reg30_none_focus,2,50.813717,55.970238,-5.156521,10622,0.394440,5.417705,-5.023265,5623,61.360851,65.881836,-4.520985,12739,88.198669,93.746841,-5.548172
67,mf50a_rank8_reg30_none_focus,3,53.784950,58.888664,-5.103714,10838,0.453856,5.516395,-5.062539,5524,69.692871,73.235481,-3.542610,12622,92.616127,98.438400,-5.822273
68,mf50a_rank8_reg30_none_focus,4,52.625896,57.903076,-5.277180,10796,0.571821,5.409229,-4.837407,5571,61.890247,65.704002,-3.813755,12617,93.076424,99.376053,-6.299629



Top 25 protected blend candidates
---------------------------------


,config,strategy,direct_oof_mse,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_44b,mse_overlap,gain_overlap_vs_44b,mse_solver_only,gain_solver_only_vs_44b,mse_none,gain_none_vs_44b
0,mf50a_rank12_reg100p0_sqrt,all_tiers,53.915245,0.0050,0.3950,0.4950,52.247093,0.532944,0.506949,0.000072,63.799431,0.462601,91.118683,1.016411
1,mf50a_rank12_reg100p0_sqrt,solver_plus_none,53.915245,0.0000,0.3950,0.4950,52.247120,0.532917,0.507021,0.000000,63.799431,0.462601,91.118683,1.016411
2,mf50a_rank8_reg100p0_sqrt,all_tiers,53.849091,0.0075,0.3825,0.4775,52.335449,0.444588,0.506929,0.000091,63.886898,0.375134,91.282478,0.852615
3,mf50a_rank8_reg100p0_sqrt,solver_plus_none,53.849091,0.0000,0.3825,0.4775,52.335484,0.444553,0.507021,0.000000,63.886898,0.375134,91.282478,0.852615
4,mf50a_rank12_reg100p0_sqrt,none_only,53.915245,0.0000,0.0000,0.4950,52.335884,0.444153,0.507021,0.000000,64.262032,0.000000,91.118683,1.016411
5,mf50a_rank8_reg100p0_sqrt,none_only,53.849091,0.0000,0.0000,0.4775,52.407463,0.372574,0.507021,0.000000,64.262032,0.000000,91.282478,0.852615
6,mf50a_rank12_reg30p0_sqrt,all_tiers,55.040867,0.0050,0.2750,0.3150,52.520924,0.259113,0.506957,0.000063,63.988609,0.273422,91.662231,0.472862
7,mf50a_rank12_reg30p0_sqrt,solver_plus_none,55.040867,0.0000,0.2750,0.3150,52.520947,0.259090,0.507021,0.000000,63.988609,0.273422,91.662231,0.472862
8,mf50a_rank12_reg30p0_sqrt,none_only,55.040867,0.0000,0.0000,0.3150,52.573410,0.206627,0.507021,0.000000,64.262032,0.000000,91.662231,0.472862
9,mf50a_rank8_reg30p0_sqrt,all_tiers,54.954964,0.0050,0.2500,0.2925,52.578297,0.201740,0.506941,0.000080,64.060074,0.201958,91.762177,0.372917



Best 50A
--------
config                     mf50a_rank12_reg100p0_sqrt
strategy                                    all_tiers
direct_oof_mse                              53.915245
lambda_overlap                                  0.005
lambda_solver_only                              0.395
lambda_none                                     0.495
oof_mse                                     52.247093
gain_vs_44b                                  0.532944
mse_overlap                                  0.506949
gain_overlap_vs_44b                          0.000072
mse_solver_only                             63.799431
gain_solver_only_vs_44b                      0.462601
mse_none                                    91.118683
gain_none_vs_44b                             1.016411

Best 50A OOF MSE: 52.247093
Gain vs 44B:       0.532944

Fold diagnostics
----------------
 fold   mse_44b   mse_50a  gain_vs_44b  n_overlap  mse44b_overlap  mse50a_overlap  gain_overlap  n_solver_only  mse44b_solver_only  m

### Public Leaderboard Check: 50A Low-Rank Matrix Factorization Breakthrough

The 50A low-rank matrix-factorization residual model produced a public leaderboard MSE of `29.742`, improving substantially over the previous protected best 44B public MSE of `30.556`.

This branch was different from the previous accounting-only refinements. It modeled residual structure over the sparse matrix:

`SCHOOL × (ASSESSMENT_NAME × SUBGROUP_NAME)`

using low-rank latent factors on the arcsine-transformed residual from 44B.

The best 50A candidate was:

- config: `mf50a_rank12_reg100p0_sqrt`
- strategy: `all_tiers`
- lambda_overlap: `0.005`
- lambda_solver_only: `0.395`
- lambda_none: `0.495`

OOF results:

- 44B OOF MSE: `52.780037`
- 50A OOF MSE: `52.247093`
- OOF gain: `0.532944`

Tier results:

- overlap MSE: `0.507021 → 0.506949`
- solver-only MSE: `64.262032 → 63.799431`
- none MSE: `92.135094 → 91.118683`

All fold gains were positive, with minimum fold gain `0.382893`.

Conclusion: 50A becomes the new protected model. The low-rank matrix-factorization branch captured residual school-assessment-subgroup structure that deterministic accounting did not capture.

In [ ]:
# ============================================================
# 50B. Extensive low-rank matrix-factorization refinement + full refit
# ============================================================
#
# Current protected public-best:
#   50A public MSE = 29.742
#
# Base model:
#   44B predictions
#
# New branch:
#   Local hyperparameter refinement around 50A:
#
#       residual_z = arcsin(sqrt(y/100)) - arcsin(sqrt(pred44b/100))
#
#       residual_z ≈ global
#                  + entity_bias[entity]
#                  + item_bias[ASSESSMENT_NAME × SUBGROUP_NAME]
#                  + entity_factor[entity] · item_factor[item]
#
# Main entity:
#   SCHOOL
#
# Additional exploratory entities:
#   DISTRICT, COUNTY
#
# Output:
#   OOF-safe fold-averaged candidate submissions
#   Full-train refit candidate submissions
#   Blends of fold-averaged and full-train refit candidates
#
# Safety:
#   - sparse matrices only
#   - no GPU
#   - no torch
#   - no huge dense feature matrices
#   - checkpoints every config
# ============================================================

import os
import gc
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path

from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# ------------------------------------------------------------
# 0. Settings
# ------------------------------------------------------------

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

CHECKPOINT_DIR = Path("model_results/mf50b_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
Path("model_results").mkdir(exist_ok=True)

RESET_50B_CHECKPOINTS = False

# Set to True for the broad search. This is still kernel-safe, just longer.
RUN_EXTENSIVE_50B = True

# Number of top unique configs to refit on full training data.
N_FULLFIT_CONFIGS = 5

if RESET_50B_CHECKPOINTS:
    for p in CHECKPOINT_DIR.glob("*"):
        p.unlink()

print("=" * 90)
print("50B. Extensive low-rank matrix-factorization refinement + full refit")
print("=" * 90)
print("Checkpoint dir:", CHECKPOINT_DIR)

# ------------------------------------------------------------
# 1. Load prediction artifacts
# ------------------------------------------------------------

def load_oof_50b(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column found in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_50b(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

# Required base.
oof44b_df, y50b, pred44b_oof = load_oof_50b("model_results/oof_hybrid44b_best.csv")
test44b_df, test_ids50b, pred44b_test = load_test_50b("model_results/testpred_hybrid44b_best.csv")

n_train50b = len(y50b)
n_test50b = len(pred44b_test)
mse44b = float(mean_squared_error(y50b, pred44b_oof))

# Optional current best 50A.
has_50a = Path("model_results/oof_mf50a_best.csv").exists() and Path("model_results/testpred_mf50a_best.csv").exists()
if has_50a:
    _, y50a_file, pred50a_oof = load_oof_50b("model_results/oof_mf50a_best.csv")
    _, _, pred50a_test = load_test_50b("model_results/testpred_mf50a_best.csv")
    assert np.max(np.abs(y50a_file - y50b)) < 1e-5
    mse50a = float(mean_squared_error(y50b, pred50a_oof))
else:
    pred50a_oof = None
    pred50a_test = None
    mse50a = np.nan

print("\nProtected references")
print("--------------------")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")
if has_50a:
    print(f"50A OOF MSE: {mse50a:.6f} | public 29.742")
else:
    print("50A artifact not found; continuing without 50A comparison.")

# ------------------------------------------------------------
# 2. Raw columns and category mappings
# ------------------------------------------------------------

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("raw_train_te/raw_test_te required. Rerun setup/recovery cells first.")

for c in ["SCHOOL", "DISTRICT", "COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]:
    if c not in raw_train_te.columns or c not in raw_test_te.columns:
        raise ValueError(f"Missing required raw column: {c}")

def clean_str_50b(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str).to_numpy()

school_train = clean_str_50b(raw_train_te["SCHOOL"])
school_test = clean_str_50b(raw_test_te["SCHOOL"])

district_train = clean_str_50b(raw_train_te["DISTRICT"])
district_test = clean_str_50b(raw_test_te["DISTRICT"])

county_train = clean_str_50b(raw_train_te["COUNTY"])
county_test = clean_str_50b(raw_test_te["COUNTY"])

assessment_train = clean_str_50b(raw_train_te["ASSESSMENT_NAME"])
assessment_test = clean_str_50b(raw_test_te["ASSESSMENT_NAME"])

subgroup_train = clean_str_50b(raw_train_te["SUBGROUP_NAME"])
subgroup_test = clean_str_50b(raw_test_te["SUBGROUP_NAME"])

asg_train = np.array([f"{a}||{s}" for a, s in zip(assessment_train, subgroup_train)], dtype=object)
asg_test = np.array([f"{a}||{s}" for a, s in zip(assessment_test, subgroup_test)], dtype=object)

n_students_train = (
    pd.to_numeric(raw_train_te["N_STUDENTS"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .to_numpy(dtype=np.float64)
)

n_students_test = (
    pd.to_numeric(raw_test_te["N_STUDENTS"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .to_numpy(dtype=np.float64)
)

def make_index_pair(train_arr, test_arr):
    all_values = pd.Index(pd.Series(np.concatenate([train_arr, test_arr])).astype(str).unique())
    mapper = {v: i for i, v in enumerate(all_values)}
    train_idx = np.array([mapper[x] for x in train_arr], dtype=np.int32)
    test_idx = np.array([mapper[x] for x in test_arr], dtype=np.int32)
    return all_values, mapper, train_idx, test_idx

all_schools, _, school_idx_train, school_idx_test = make_index_pair(school_train, school_test)
all_districts, _, district_idx_train, district_idx_test = make_index_pair(district_train, district_test)
all_counties, _, county_idx_train, county_idx_test = make_index_pair(county_train, county_test)
all_asg, _, asg_idx_train, asg_idx_test = make_index_pair(asg_train, asg_test)

entity_index_data = {
    "school": {
        "train_idx": school_idx_train,
        "test_idx": school_idx_test,
        "n_entities": len(all_schools),
    },
    "district": {
        "train_idx": district_idx_train,
        "test_idx": district_idx_test,
        "n_entities": len(all_districts),
    },
    "county": {
        "train_idx": county_idx_train,
        "test_idx": county_idx_test,
        "n_entities": len(all_counties),
    },
}

n_items = len(all_asg)

print("\nMatrix dimensions")
print("-----------------")
print("n schools:", len(all_schools))
print("n districts:", len(all_districts))
print("n counties:", len(all_counties))
print("n assessment-subgroup items:", n_items)
print("train rows:", n_train50b)
print("test rows:", n_test50b)

# ------------------------------------------------------------
# 3. Accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

test_tier_rates = {
    tier: float(mask.mean())
    for tier, mask in tier_masks_test.items()
}

print("\nTier coverage")
print("-------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"44B MSE={mean_squared_error(y50b[mask], pred44b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 4. Transform and fitting helpers
# ------------------------------------------------------------

def pct_to_arc_50b(pct):
    p = np.clip(np.asarray(pct, dtype=np.float64) / 100.0, 1e-6, 1 - 1e-6)
    return np.arcsin(np.sqrt(p)).astype(np.float32)

def arc_to_pct_50b(z):
    p = np.sin(np.asarray(z, dtype=np.float64)) ** 2
    return np.clip(100.0 * p, 0, 100).astype(np.float32)

z_y = pct_to_arc_50b(y50b)
z_44b_oof = pct_to_arc_50b(pred44b_oof)
z_44b_test = pct_to_arc_50b(pred44b_test)
resid_z = (z_y - z_44b_oof).astype(np.float32)

def make_weights_50b(idx, scheme):
    idx = np.asarray(idx, dtype=np.int64)

    n = n_students_train[idx].astype(np.float64)
    valid_n = np.isfinite(n) & (n > 0)
    med_n = np.nanmedian(n[valid_n]) if valid_n.any() else 30.0
    n = np.where(valid_n, n, med_n)

    if scheme == "uniform":
        w = np.ones(len(idx), dtype=np.float64)
    elif scheme == "sqrt_n":
        w = np.sqrt(np.clip(n, 1, 500))
    elif scheme == "log_n":
        w = np.log1p(np.clip(n, 1, 500))
    elif scheme == "solver_mild":
        w = np.sqrt(np.clip(n, 1, 500))
        w[overlap_oof[idx]] *= 0.25
        w[solver_only_oof[idx]] *= 1.25
        w[none_oof[idx]] *= 0.75
    elif scheme == "solver_public":
        w = np.sqrt(np.clip(n, 1, 500))
        w[overlap_oof[idx]] *= 0.10
        w[solver_only_oof[idx]] *= 1.25
        w[none_oof[idx]] *= 0.75
    elif scheme == "none_mild":
        w = np.sqrt(np.clip(n, 1, 500))
        w[overlap_oof[idx]] *= 0.10
        w[solver_only_oof[idx]] *= 0.85
        w[none_oof[idx]] *= 1.25
    else:
        raise ValueError(f"Unknown weight scheme: {scheme}")

    return w.astype(np.float64)

def fit_biases_50b(
    idx,
    entity_idx_all,
    item_idx_all,
    n_entities,
    n_items,
    reg_entity,
    reg_item,
    weight_scheme,
    n_iter,
):
    idx = np.asarray(idx, dtype=np.int64)

    e_idx = entity_idx_all[idx]
    i_idx = item_idx_all[idx]
    r = resid_z[idx].astype(np.float64)
    w = make_weights_50b(idx, weight_scheme)

    global_bias = float(np.sum(w * r) / max(np.sum(w), 1e-12))

    entity_bias = np.zeros(n_entities, dtype=np.float64)
    item_bias = np.zeros(n_items, dtype=np.float64)

    for _ in range(int(n_iter)):
        residual_entity = r - global_bias - item_bias[i_idx]
        sum_w_e = np.bincount(e_idx, weights=w, minlength=n_entities)
        sum_wr_e = np.bincount(e_idx, weights=w * residual_entity, minlength=n_entities)
        entity_bias = sum_wr_e / (sum_w_e + float(reg_entity))

        residual_item = r - global_bias - entity_bias[e_idx]
        sum_w_i = np.bincount(i_idx, weights=w, minlength=n_items)
        sum_wr_i = np.bincount(i_idx, weights=w * residual_item, minlength=n_items)
        item_bias = sum_wr_i / (sum_w_i + float(reg_item))

    debiased = r - global_bias - entity_bias[e_idx] - item_bias[i_idx]

    return (
        global_bias,
        entity_bias.astype(np.float32),
        item_bias.astype(np.float32),
        debiased.astype(np.float32),
    )

def fit_svd_50b(
    idx,
    debiased_values,
    entity_idx_all,
    item_idx_all,
    n_entities,
    n_items,
    rank,
    random_state,
):
    idx = np.asarray(idx, dtype=np.int64)

    e_idx = entity_idx_all[idx]
    i_idx = item_idx_all[idx]

    matrix = sparse.csr_matrix(
        (debiased_values.astype(np.float32), (e_idx, i_idx)),
        shape=(n_entities, n_items),
        dtype=np.float32,
    )

    rank_eff = int(min(rank, max(1, min(n_entities, n_items) - 1)))

    svd = TruncatedSVD(
        n_components=rank_eff,
        n_iter=8,
        random_state=random_state,
    )

    entity_factors = svd.fit_transform(matrix).astype(np.float32)
    item_components = svd.components_.astype(np.float32)

    return entity_factors, item_components

def predict_mf_50b(
    base_z,
    entity_idx,
    item_idx,
    global_bias,
    entity_bias,
    item_bias,
    entity_factors,
    item_components,
    bias_scale,
    factor_scale,
):
    bias = (
        float(global_bias)
        + entity_bias[entity_idx].astype(np.float64)
        + item_bias[item_idx].astype(np.float64)
    )
    bias *= float(bias_scale)

    if entity_factors is None or item_components is None:
        factor = np.zeros(len(entity_idx), dtype=np.float64)
    else:
        factor = np.sum(
            entity_factors[entity_idx] * item_components[:, item_idx].T,
            axis=1,
        ).astype(np.float64)
        factor *= float(factor_scale)

    z_pred = base_z.astype(np.float64) + bias + factor
    return arc_to_pct_50b(z_pred)

def train_predict_config_oof_50b(cfg):
    entity_data = entity_index_data[cfg["entity_key"]]
    entity_idx_train = entity_data["train_idx"]
    entity_idx_test = entity_data["test_idx"]
    n_entities = entity_data["n_entities"]

    item_idx_train = asg_idx_train
    item_idx_test = asg_idx_test

    oof = np.full(n_train50b, np.nan, dtype=np.float32)
    test_sum = np.zeros(n_test50b, dtype=np.float64)
    fold_rows = []

    folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train50b)))

    for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
        global_bias, entity_bias, item_bias, debiased = fit_biases_50b(
            tr_idx,
            entity_idx_train,
            item_idx_train,
            n_entities,
            n_items,
            reg_entity=cfg["reg_entity"],
            reg_item=cfg["reg_item"],
            weight_scheme=cfg["weight_scheme"],
            n_iter=cfg["bias_iter"],
        )

        entity_factors, item_components = fit_svd_50b(
            tr_idx,
            debiased,
            entity_idx_train,
            item_idx_train,
            n_entities,
            n_items,
            rank=cfg["rank"],
            random_state=RANDOM_STATE + 1000 * fold_num + cfg["rank"],
        )

        pred_va = predict_mf_50b(
            base_z=z_44b_oof[va_idx],
            entity_idx=entity_idx_train[va_idx],
            item_idx=item_idx_train[va_idx],
            global_bias=global_bias,
            entity_bias=entity_bias,
            item_bias=item_bias,
            entity_factors=entity_factors,
            item_components=item_components,
            bias_scale=cfg["bias_scale"],
            factor_scale=cfg["factor_scale"],
        )

        pred_test = predict_mf_50b(
            base_z=z_44b_test,
            entity_idx=entity_idx_test,
            item_idx=item_idx_test,
            global_bias=global_bias,
            entity_bias=entity_bias,
            item_bias=item_bias,
            entity_factors=entity_factors,
            item_components=item_components,
            bias_scale=cfg["bias_scale"],
            factor_scale=cfg["factor_scale"],
        )

        oof[va_idx] = pred_va
        test_sum += pred_test.astype(np.float64)

        row = {
            "config": cfg["name"],
            "fold": fold_num,
            "mse_44b": float(mean_squared_error(y50b[va_idx], pred44b_oof[va_idx])),
            "mse_direct": float(mean_squared_error(y50b[va_idx], pred_va)),
        }
        row["gain_direct_vs_44b"] = row["mse_44b"] - row["mse_direct"]

        for tier, mask_full in tier_masks_oof.items():
            mask = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask.sum())
            if int(mask.sum()) > 0:
                row[f"mse44b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred44b_oof[va_idx][mask]))
                row[f"mse_direct_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred_va[mask]))
                row[f"gain_direct_{tier}"] = row[f"mse44b_{tier}"] - row[f"mse_direct_{tier}"]

        fold_rows.append(row)

        del entity_factors, item_components, entity_bias, item_bias, debiased
        gc.collect()

    test_avg = (test_sum / N_SPLITS).astype(np.float32)
    return oof, test_avg, pd.DataFrame(fold_rows)

def train_predict_config_fullfit_50b(cfg):
    entity_data = entity_index_data[cfg["entity_key"]]
    entity_idx_train = entity_data["train_idx"]
    entity_idx_test = entity_data["test_idx"]
    n_entities = entity_data["n_entities"]

    item_idx_train = asg_idx_train
    item_idx_test = asg_idx_test

    tr_idx = np.arange(n_train50b)

    global_bias, entity_bias, item_bias, debiased = fit_biases_50b(
        tr_idx,
        entity_idx_train,
        item_idx_train,
        n_entities,
        n_items,
        reg_entity=cfg["reg_entity"],
        reg_item=cfg["reg_item"],
        weight_scheme=cfg["weight_scheme"],
        n_iter=cfg["bias_iter"],
    )

    entity_factors, item_components = fit_svd_50b(
        tr_idx,
        debiased,
        entity_idx_train,
        item_idx_train,
        n_entities,
        n_items,
        rank=cfg["rank"],
        random_state=RANDOM_STATE + 777 + cfg["rank"],
    )

    pred_train_fullfit = predict_mf_50b(
        base_z=z_44b_oof,
        entity_idx=entity_idx_train,
        item_idx=item_idx_train,
        global_bias=global_bias,
        entity_bias=entity_bias,
        item_bias=item_bias,
        entity_factors=entity_factors,
        item_components=item_components,
        bias_scale=cfg["bias_scale"],
        factor_scale=cfg["factor_scale"],
    )

    pred_test_fullfit = predict_mf_50b(
        base_z=z_44b_test,
        entity_idx=entity_idx_test,
        item_idx=item_idx_test,
        global_bias=global_bias,
        entity_bias=entity_bias,
        item_bias=item_bias,
        entity_factors=entity_factors,
        item_components=item_components,
        bias_scale=cfg["bias_scale"],
        factor_scale=cfg["factor_scale"],
    )

    return pred_train_fullfit, pred_test_fullfit

# ------------------------------------------------------------
# 5. Config grid
# ------------------------------------------------------------

def safe_float_str(x):
    return str(x).replace(".", "p").replace("-", "m")

configs50b = []

if RUN_EXTENSIVE_50B:
    # Main local school-level search around 50A winner.
    for rank in [6, 8, 10, 12, 14, 16, 20, 24, 32]:
        for reg in [50.0, 75.0, 100.0, 125.0, 150.0, 200.0, 300.0, 500.0]:
            for weight_scheme in ["sqrt_n", "uniform"]:
                cfg = {
                    "entity_key": "school",
                    "rank": rank,
                    "reg_entity": reg,
                    "reg_item": reg,
                    "weight_scheme": weight_scheme,
                    "bias_iter": 4,
                    "bias_scale": 1.0,
                    "factor_scale": 1.0,
                }
                cfg["name"] = (
                    f"mf50b_{cfg['entity_key']}_rank{rank}"
                    f"_reg{safe_float_str(reg)}_{weight_scheme}"
                    f"_b{safe_float_str(cfg['bias_scale'])}"
                    f"_f{safe_float_str(cfg['factor_scale'])}"
                )
                configs50b.append(cfg)

    # A focused factor-scale search near the 50A winner.
    for rank in [8, 10, 12, 14, 16, 20]:
        for reg in [75.0, 100.0, 125.0, 150.0]:
            for factor_scale in [0.50, 0.75, 1.25, 1.50]:
                cfg = {
                    "entity_key": "school",
                    "rank": rank,
                    "reg_entity": reg,
                    "reg_item": reg,
                    "weight_scheme": "sqrt_n",
                    "bias_iter": 4,
                    "bias_scale": 1.0,
                    "factor_scale": factor_scale,
                }
                cfg["name"] = (
                    f"mf50b_{cfg['entity_key']}_rank{rank}"
                    f"_reg{safe_float_str(reg)}_sqrt_n"
                    f"_b{safe_float_str(cfg['bias_scale'])}"
                    f"_f{safe_float_str(cfg['factor_scale'])}"
                )
                configs50b.append(cfg)

    # Mild alternate weighting.
    for rank in [8, 12, 16, 20]:
        for reg in [75.0, 100.0, 150.0, 250.0]:
            for weight_scheme in ["log_n", "solver_mild", "none_mild"]:
                cfg = {
                    "entity_key": "school",
                    "rank": rank,
                    "reg_entity": reg,
                    "reg_item": reg,
                    "weight_scheme": weight_scheme,
                    "bias_iter": 4,
                    "bias_scale": 1.0,
                    "factor_scale": 1.0,
                }
                cfg["name"] = (
                    f"mf50b_{cfg['entity_key']}_rank{rank}"
                    f"_reg{safe_float_str(reg)}_{weight_scheme}"
                    f"_b{safe_float_str(cfg['bias_scale'])}"
                    f"_f{safe_float_str(cfg['factor_scale'])}"
                )
                configs50b.append(cfg)

    # Smaller district/county searches.
    for entity_key in ["district", "county"]:
        for rank in [2, 4, 8, 12, 16]:
            for reg in [25.0, 50.0, 100.0, 200.0]:
                cfg = {
                    "entity_key": entity_key,
                    "rank": rank,
                    "reg_entity": reg,
                    "reg_item": reg,
                    "weight_scheme": "sqrt_n",
                    "bias_iter": 4,
                    "bias_scale": 1.0,
                    "factor_scale": 1.0,
                }
                cfg["name"] = (
                    f"mf50b_{cfg['entity_key']}_rank{rank}"
                    f"_reg{safe_float_str(reg)}_sqrt_n"
                    f"_b{safe_float_str(cfg['bias_scale'])}"
                    f"_f{safe_float_str(cfg['factor_scale'])}"
                )
                configs50b.append(cfg)
else:
    # Short version if you need a quick rerun.
    for rank in [8, 10, 12, 14, 16]:
        for reg in [75.0, 100.0, 125.0, 150.0, 200.0]:
            cfg = {
                "entity_key": "school",
                "rank": rank,
                "reg_entity": reg,
                "reg_item": reg,
                "weight_scheme": "sqrt_n",
                "bias_iter": 4,
                "bias_scale": 1.0,
                "factor_scale": 1.0,
            }
            cfg["name"] = (
                f"mf50b_{cfg['entity_key']}_rank{rank}"
                f"_reg{safe_float_str(reg)}_sqrt_n"
                f"_b{safe_float_str(cfg['bias_scale'])}"
                f"_f{safe_float_str(cfg['factor_scale'])}"
            )
            configs50b.append(cfg)

# Deduplicate configs by name.
cfg_by_name = {}
for cfg in configs50b:
    cfg_by_name[cfg["name"]] = cfg
configs50b = list(cfg_by_name.values())

print("\nNumber of 50B configs:", len(configs50b))
print("First 10 configs:")
for cfg in configs50b[:10]:
    print(cfg)

# Save config manifest.
with open("model_results/mf50b_config_manifest.json", "w") as f:
    json.dump(configs50b, f, indent=2)

# ------------------------------------------------------------
# 6. Run or load checkpoints
# ------------------------------------------------------------

cand_oof = {}
cand_test = {}
fold_metric_frames = []
completed_rows = []

t0 = time.time()

for cfg_num, cfg in enumerate(configs50b, start=1):
    name = cfg["name"]

    oof_path = CHECKPOINT_DIR / f"{name}_oof.npy"
    test_path = CHECKPOINT_DIR / f"{name}_test.npy"
    folds_path = CHECKPOINT_DIR / f"{name}_folds.csv"
    meta_path = CHECKPOINT_DIR / f"{name}_meta.json"

    print("\n" + "=" * 90)
    print(f"Config {cfg_num}/{len(configs50b)}: {name}")
    print("=" * 90)

    if oof_path.exists() and test_path.exists() and folds_path.exists():
        print("Loading checkpoint.")
        oof_pred = np.load(oof_path).astype(np.float32)
        test_pred = np.load(test_path).astype(np.float32)
        fold_df = pd.read_csv(folds_path)
    else:
        start = time.time()
        oof_pred, test_pred, fold_df = train_predict_config_oof_50b(cfg)
        np.save(oof_path, oof_pred)
        np.save(test_path, test_pred)
        fold_df.to_csv(folds_path, index=False)
        with open(meta_path, "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Finished config in {time.time() - start:.1f}s")

    cand_oof[name] = oof_pred
    cand_test[name] = test_pred
    fold_metric_frames.append(fold_df)

    direct_mse = float(mean_squared_error(y50b, oof_pred))
    direct_gain_vs_44b = mse44b - direct_mse

    if has_50a:
        direct_gain_vs_50a = mse50a - direct_mse
    else:
        direct_gain_vs_50a = np.nan

    completed_rows.append({
        "config": name,
        "entity_key": cfg["entity_key"],
        "rank": cfg["rank"],
        "reg_entity": cfg["reg_entity"],
        "reg_item": cfg["reg_item"],
        "weight_scheme": cfg["weight_scheme"],
        "bias_iter": cfg["bias_iter"],
        "bias_scale": cfg["bias_scale"],
        "factor_scale": cfg["factor_scale"],
        "direct_oof_mse": direct_mse,
        "direct_gain_vs_44b": direct_gain_vs_44b,
        "direct_gain_vs_50a": direct_gain_vs_50a,
    })

    pd.DataFrame(completed_rows).to_csv("model_results/mf50b_direct_progress.csv", index=False)
    print(f"Direct OOF MSE: {direct_mse:.6f} | gain vs 44B: {direct_gain_vs_44b:.6f}")

print("\nTotal 50B config pass elapsed seconds:", round(time.time() - t0, 1))

fold_metrics50b = pd.concat(fold_metric_frames, axis=0).reset_index(drop=True)

# ------------------------------------------------------------
# 7. Blend scan
# ------------------------------------------------------------

lambda_grid = np.unique(
    np.concatenate([
        np.linspace(-0.25, 1.25, 601),
        np.array([0.0, 0.005, 0.05, 0.10, 0.155, 0.25, 0.395, 0.495, 0.50, 0.75, 1.0])
    ])
)

def best_lambda_for_mask_50b(candidate_pred, mask):
    if int(mask.sum()) == 0:
        return 0.0, np.nan

    y = y50b[mask].astype(np.float64)
    base = pred44b_oof[mask].astype(np.float64)
    alt = candidate_pred[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid:
        pred = np.clip(base + float(lam) * (alt - base), 0, 100)
        mse = float(mean_squared_error(y, pred))

        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam, best_mse

def make_blend_50b(candidate_oof, candidate_test, lams, is_test=False):
    if is_test:
        base = pred44b_test.astype(np.float64)
        alt = candidate_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred44b_oof.astype(np.float64)
        alt = candidate_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

def tier_metrics_50b(pred):
    out = {}
    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y50b[mask], pred[mask]))
        out[f"gain_{tier}_vs_44b"] = (
            float(mean_squared_error(y50b[mask], pred44b_oof[mask]))
            - out[f"mse_{tier}"]
        )
        if has_50a:
            out[f"gain_{tier}_vs_50a"] = (
                float(mean_squared_error(y50b[mask], pred50a_oof[mask]))
                - out[f"mse_{tier}"]
            )
    return out

def tier_weighted_mse_50b(pred):
    pred = np.clip(pred, 0, 100)
    total = 0.0
    for tier, mask in tier_masks_oof.items():
        total += test_tier_rates[tier] * float(mean_squared_error(y50b[mask], pred[mask]))
    return float(total)

screen_rows = []
blend_store = {}

for cfg in configs50b:
    name = cfg["name"]
    p_oof = cand_oof[name]
    p_test = cand_test[name]

    direct_mse = float(mean_squared_error(y50b, p_oof))

    lam_overlap, _ = best_lambda_for_mask_50b(p_oof, overlap_oof)
    lam_solver, _ = best_lambda_for_mask_50b(p_oof, solver_only_oof)
    lam_none, _ = best_lambda_for_mask_50b(p_oof, none_oof)

    strategy_lams = {
        "solver_only": {
            "overlap": 0.0,
            "solver_only": lam_solver,
            "none": 0.0,
        },
        "none_only": {
            "overlap": 0.0,
            "solver_only": 0.0,
            "none": lam_none,
        },
        "solver_plus_none": {
            "overlap": 0.0,
            "solver_only": lam_solver,
            "none": lam_none,
        },
        "all_tiers": {
            "overlap": lam_overlap,
            "solver_only": lam_solver,
            "none": lam_none,
        },
    }

    for strategy, lams in strategy_lams.items():
        pred = make_blend_50b(p_oof, p_test, lams, is_test=False)
        test_pred = make_blend_50b(p_oof, p_test, lams, is_test=True)

        mse = float(mean_squared_error(y50b, pred))

        row = {
            "config": name,
            "strategy": strategy,
            "entity_key": cfg["entity_key"],
            "rank": cfg["rank"],
            "reg_entity": cfg["reg_entity"],
            "reg_item": cfg["reg_item"],
            "weight_scheme": cfg["weight_scheme"],
            "bias_iter": cfg["bias_iter"],
            "bias_scale": cfg["bias_scale"],
            "factor_scale": cfg["factor_scale"],
            "direct_oof_mse": direct_mse,
            "lambda_overlap": float(lams["overlap"]),
            "lambda_solver_only": float(lams["solver_only"]),
            "lambda_none": float(lams["none"]),
            "oof_mse": mse,
            "gain_vs_44b": mse44b - mse,
            "test_tier_weighted_mse": tier_weighted_mse_50b(pred),
            "weighted_gain_vs_44b": tier_weighted_mse_50b(pred44b_oof) - tier_weighted_mse_50b(pred),
        }

        if has_50a:
            row["gain_vs_50a"] = mse50a - mse
            row["weighted_gain_vs_50a"] = tier_weighted_mse_50b(pred50a_oof) - tier_weighted_mse_50b(pred)
        else:
            row["gain_vs_50a"] = np.nan
            row["weighted_gain_vs_50a"] = np.nan

        row.update(tier_metrics_50b(pred))

        key = f"{name}__{strategy}"
        row["key"] = key

        screen_rows.append(row)
        blend_store[key] = {
            "oof": pred,
            "test": test_pred,
            "lams": lams,
            "config": cfg,
        }

screen50b = pd.DataFrame(screen_rows).sort_values(["oof_mse", "test_tier_weighted_mse"]).reset_index(drop=True)
screen50b_weighted = screen50b.sort_values(["test_tier_weighted_mse", "oof_mse"]).reset_index(drop=True)

best_key = screen50b.iloc[0]["key"]
best_row = screen50b.iloc[0]
best_oof = blend_store[best_key]["oof"]
best_test = blend_store[best_key]["test"]
best_cfg = blend_store[best_key]["config"]

# ------------------------------------------------------------
# 8. Fold diagnostics for best OOF candidate
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train50b)))

fold_diag_rows = []

for fold_num, (_, va_idx) in enumerate(folds, start=1):
    row = {
        "fold": fold_num,
        "mse_44b": float(mean_squared_error(y50b[va_idx], pred44b_oof[va_idx])),
        "mse_50b": float(mean_squared_error(y50b[va_idx], best_oof[va_idx])),
    }
    row["gain_vs_44b"] = row["mse_44b"] - row["mse_50b"]

    if has_50a:
        row["mse_50a"] = float(mean_squared_error(y50b[va_idx], pred50a_oof[va_idx]))
        row["gain_vs_50a"] = row["mse_50a"] - row["mse_50b"]

    for tier, mask_full in tier_masks_oof.items():
        mask = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse44b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred44b_oof[va_idx][mask]))
            row[f"mse50b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], best_oof[va_idx][mask]))
            row[f"gain_{tier}_vs_44b"] = row[f"mse44b_{tier}"] - row[f"mse50b_{tier}"]
            if has_50a:
                row[f"mse50a_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred50a_oof[va_idx][mask]))
                row[f"gain_{tier}_vs_50a"] = row[f"mse50a_{tier}"] - row[f"mse50b_{tier}"]

    fold_diag_rows.append(row)

fold_diag50b = pd.DataFrame(fold_diag_rows)

# ------------------------------------------------------------
# 9. Full-train refit for top configs
# ------------------------------------------------------------

# Pick top unique configs by OOF screen.
top_unique_configs = []
seen_cfg = set()

for _, row in screen50b.iterrows():
    cfg_name = row["config"]
    if cfg_name not in seen_cfg:
        seen_cfg.add(cfg_name)
        top_unique_configs.append(cfg_name)
    if len(top_unique_configs) >= N_FULLFIT_CONFIGS:
        break

cfg_lookup = {cfg["name"]: cfg for cfg in configs50b}

fullfit_rows = []
fullfit_store = {}

for cfg_name in top_unique_configs:
    cfg = cfg_lookup[cfg_name]
    print("\n" + "=" * 90)
    print("Full-train refit:", cfg_name)
    print("=" * 90)

    full_oof_direct, full_test_direct = train_predict_config_fullfit_50b(cfg)

    # Use the best OOF-selected strategy for this config.
    cfg_screen = screen50b[screen50b["config"] == cfg_name].sort_values("oof_mse").reset_index(drop=True)
    cfg_best = cfg_screen.iloc[0]
    cfg_lams = {
        "overlap": float(cfg_best["lambda_overlap"]),
        "solver_only": float(cfg_best["lambda_solver_only"]),
        "none": float(cfg_best["lambda_none"]),
    }

    full_oof_blend = make_blend_50b(full_oof_direct, full_test_direct, cfg_lams, is_test=False)
    full_test_blend = make_blend_50b(full_oof_direct, full_test_direct, cfg_lams, is_test=True)

    # In-sample metric only; not honest validation.
    trainfit_mse = float(mean_squared_error(y50b, full_oof_blend))

    foldavg_test_for_cfg = blend_store[cfg_best["key"]]["test"]

    row = {
        "config": cfg_name,
        "selected_strategy": cfg_best["strategy"],
        "lambda_overlap": cfg_lams["overlap"],
        "lambda_solver_only": cfg_lams["solver_only"],
        "lambda_none": cfg_lams["none"],
        "oof_selected_mse": float(cfg_best["oof_mse"]),
        "oof_gain_vs_44b": float(cfg_best["gain_vs_44b"]),
        "oof_gain_vs_50a": float(cfg_best["gain_vs_50a"]) if has_50a else np.nan,
        "trainfit_mse_not_oof": trainfit_mse,
        "mean_abs_diff_fullfit_vs_foldavg_test": float(np.mean(np.abs(full_test_blend - foldavg_test_for_cfg))),
        "p95_abs_diff_fullfit_vs_foldavg_test": float(np.percentile(np.abs(full_test_blend - foldavg_test_for_cfg), 95)),
        "max_abs_diff_fullfit_vs_foldavg_test": float(np.max(np.abs(full_test_blend - foldavg_test_for_cfg))),
    }

    if has_50a:
        row["mean_abs_diff_fullfit_vs_50a_test"] = float(np.mean(np.abs(full_test_blend - pred50a_test)))
        row["p95_abs_diff_fullfit_vs_50a_test"] = float(np.percentile(np.abs(full_test_blend - pred50a_test), 95))

    fullfit_rows.append(row)

    fullfit_store[cfg_name] = {
        "full_oof_direct": full_oof_direct,
        "full_test_direct": full_test_direct,
        "full_oof_blend": full_oof_blend,
        "full_test_blend": full_test_blend,
        "foldavg_test": foldavg_test_for_cfg,
        "cfg_best_row": cfg_best,
        "lams": cfg_lams,
    }

fullfit50b = pd.DataFrame(fullfit_rows).sort_values(["oof_selected_mse", "mean_abs_diff_fullfit_vs_foldavg_test"]).reset_index(drop=True)

# ------------------------------------------------------------
# 10. Save artifacts and submissions
# ------------------------------------------------------------

screen_path = "model_results/mf50b_screen.csv"
weighted_path = "model_results/mf50b_screen_weighted.csv"
direct_progress_path = "model_results/mf50b_direct_progress.csv"
fold_metrics_path = "model_results/mf50b_fold_metrics.csv"
fold_diag_path = "model_results/mf50b_best_fold_diag.csv"
fullfit_path = "model_results/mf50b_fullfit_screen.csv"

oof_best_path = "model_results/oof_mf50b_best_oof.csv"
test_best_path = "model_results/testpred_mf50b_best_oof.csv"
submission_best_path = "submission_mf50b_best_oof.csv"

screen50b.to_csv(screen_path, index=False)
screen50b_weighted.to_csv(weighted_path, index=False)
pd.DataFrame(completed_rows).to_csv(direct_progress_path, index=False)
fold_metrics50b.to_csv(fold_metrics_path, index=False)
fold_diag50b.to_csv(fold_diag_path, index=False)
fullfit50b.to_csv(fullfit_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train50b),
    TARGET_COL: y50b,
    "pred_44b": pred44b_oof,
    "pred_50a": pred50a_oof if has_50a else np.nan,
    "pred_clipped": best_oof,
    "tier_overlap": overlap_oof.astype(int),
    "tier_solver_only": solver_only_oof.astype(int),
    "tier_none": none_oof.astype(int),
}).to_csv(oof_best_path, index=False)

pd.DataFrame({
    ID_COL: test_ids50b,
    "pred_44b": pred44b_test,
    "pred_50a": pred50a_test if has_50a else np.nan,
    TARGET_COL: best_test,
    "tier_overlap": overlap_test.astype(int),
    "tier_solver_only": solver_only_test.astype(int),
    "tier_none": none_test.astype(int),
}).to_csv(test_best_path, index=False)

pd.DataFrame({
    ID_COL: test_ids50b,
    TARGET_COL: best_test,
}).to_csv(submission_best_path, index=False)

# Fullfit and fold/fullfit blend submissions.
fullfit_submission_paths = []

if len(fullfit50b) > 0:
    for row_idx, row in fullfit50b.iterrows():
        cfg_name = row["config"]
        obj = fullfit_store[cfg_name]

        tag = f"rank{int(cfg_lookup[cfg_name]['rank'])}_{cfg_lookup[cfg_name]['entity_key']}_reg{safe_float_str(cfg_lookup[cfg_name]['reg_entity'])}_{cfg_lookup[cfg_name]['weight_scheme']}"
        base_name = f"mf50b_fullfit_{row_idx+1}_{tag}"

        # Fullfit submission.
        path_full = f"submission_{base_name}.csv"
        pd.DataFrame({
            ID_COL: test_ids50b,
            TARGET_COL: obj["full_test_blend"],
        }).to_csv(path_full, index=False)
        fullfit_submission_paths.append(path_full)

        # Blends between OOF fold-averaged test prediction and fullfit test prediction.
        for w_full in [0.25, 0.50, 0.75]:
            mix = (
                float(w_full) * obj["full_test_blend"].astype(np.float64)
                + (1.0 - float(w_full)) * obj["foldavg_test"].astype(np.float64)
            )
            mix = np.clip(mix, 0, 100).astype(np.float32)

            path_mix = f"submission_{base_name}_mixfull{safe_float_str(w_full)}.csv"
            pd.DataFrame({
                ID_COL: test_ids50b,
                TARGET_COL: mix,
            }).to_csv(path_mix, index=False)
            fullfit_submission_paths.append(path_mix)

# Validate submissions.
all_submission_paths = [submission_best_path] + fullfit_submission_paths

for p in all_submission_paths:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test50b, 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[ID_COL].notna().all(), p
    assert sub[TARGET_COL].notna().all(), p
    assert np.isfinite(sub[TARGET_COL]).all(), p
    assert sub[TARGET_COL].between(0, 100).all(), p

# ------------------------------------------------------------
# 11. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("50B extensive MF refinement complete")
print("=" * 90)

print("\nReference")
print("---------")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")
if has_50a:
    print(f"50A OOF MSE: {mse50a:.6f} | public 29.742")

print("\nTop 30 OOF-safe 50B candidates")
print("------------------------------")
display(screen50b.head(30))

print("\nTop 20 weighted candidates")
print("--------------------------")
display(screen50b_weighted.head(20))

print("\nBest OOF-safe 50B")
print("-----------------")
print(best_row.to_string())

print("\nBest fold diagnostics")
print("---------------------")
print(fold_diag50b.to_string(index=False))
print("Min fold gain vs 44B:", float(fold_diag50b["gain_vs_44b"].min()))
if has_50a and "gain_vs_50a" in fold_diag50b.columns:
    print("Min fold gain vs 50A:", float(fold_diag50b["gain_vs_50a"].min()))

print("\nFull-train refit screen")
print("-----------------------")
display(fullfit50b)

print("\nSaved files")
print("-----------")
print(screen_path)
print(weighted_path)
print(direct_progress_path)
print(fold_metrics_path)
print(fold_diag_path)
print(fullfit_path)
print(oof_best_path)
print(test_best_path)
print(submission_best_path)
for p in fullfit_submission_paths:
    print(p)

print("\nSubmission validation")
print("---------------------")
for p in all_submission_paths[:10]:
    sub = pd.read_csv(p)
    print(p, sub.shape, sub[TARGET_COL].describe().to_dict())
if len(all_submission_paths) > 10:
    print(f"... {len(all_submission_paths) - 10} additional submission files validated.")

print("\nDecision guidance")
print("-----------------")
print("Primary OOF-safe candidate:")
print(submission_best_path)

if len(fullfit50b) > 0:
    print("\nMost natural full-refit probes are the first row in the fullfit table:")
    print(f"submission_mf50b_fullfit_1_*.csv")
    print("and its mixfull0p25 / mixfull0p50 variants.")
print("\nUse public probing order only after inspecting whether 50B materially improves 50A OOF and has stable folds.")

In [57]:
# ============================================================
# 50B. Extensive low-rank matrix-factorization refinement + full refit
# ============================================================
#
# Current protected public-best:
#   50A public MSE = 29.742
#
# Base model:
#   44B predictions
#
# New branch:
#   Local hyperparameter refinement around 50A:
#
#       residual_z = arcsin(sqrt(y/100)) - arcsin(sqrt(pred44b/100))
#
#       residual_z ≈ global
#                  + entity_bias[entity]
#                  + item_bias[ASSESSMENT_NAME × SUBGROUP_NAME]
#                  + entity_factor[entity] · item_factor[item]
#
# Main entity:
#   SCHOOL
#
# Additional exploratory entities:
#   DISTRICT, COUNTY
#
# Output:
#   OOF-safe fold-averaged candidate submissions
#   Full-train refit candidate submissions
#   Blends of fold-averaged and full-train refit candidates
#
# Safety:
#   - sparse matrices only
#   - no GPU
#   - no torch
#   - no huge dense feature matrices
#   - checkpoints every config
# ============================================================

import os
import gc
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path

from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# ------------------------------------------------------------
# 0. Settings
# ------------------------------------------------------------

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

CHECKPOINT_DIR = Path("model_results/mf50b_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
Path("model_results").mkdir(exist_ok=True)

RESET_50B_CHECKPOINTS = False

# Set to True for the broad search. This is still kernel-safe, just longer.
RUN_EXTENSIVE_50B = True

# Number of top unique configs to refit on full training data.
N_FULLFIT_CONFIGS = 5

if RESET_50B_CHECKPOINTS:
    for p in CHECKPOINT_DIR.glob("*"):
        p.unlink()

print("=" * 90)
print("50B. Extensive low-rank matrix-factorization refinement + full refit")
print("=" * 90)
print("Checkpoint dir:", CHECKPOINT_DIR)

# ------------------------------------------------------------
# 1. Load prediction artifacts
# ------------------------------------------------------------

def load_oof_50b(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column found in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_50b(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

# Required base.
oof44b_df, y50b, pred44b_oof = load_oof_50b("model_results/oof_hybrid44b_best.csv")
test44b_df, test_ids50b, pred44b_test = load_test_50b("model_results/testpred_hybrid44b_best.csv")

n_train50b = len(y50b)
n_test50b = len(pred44b_test)
mse44b = float(mean_squared_error(y50b, pred44b_oof))

# Optional current best 50A.
has_50a = Path("model_results/oof_mf50a_best.csv").exists() and Path("model_results/testpred_mf50a_best.csv").exists()
if has_50a:
    _, y50a_file, pred50a_oof = load_oof_50b("model_results/oof_mf50a_best.csv")
    _, _, pred50a_test = load_test_50b("model_results/testpred_mf50a_best.csv")
    assert np.max(np.abs(y50a_file - y50b)) < 1e-5
    mse50a = float(mean_squared_error(y50b, pred50a_oof))
else:
    pred50a_oof = None
    pred50a_test = None
    mse50a = np.nan

print("\nProtected references")
print("--------------------")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")
if has_50a:
    print(f"50A OOF MSE: {mse50a:.6f} | public 29.742")
else:
    print("50A artifact not found; continuing without 50A comparison.")

# ------------------------------------------------------------
# 2. Raw columns and category mappings
# ------------------------------------------------------------

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("raw_train_te/raw_test_te required. Rerun setup/recovery cells first.")

for c in ["SCHOOL", "DISTRICT", "COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]:
    if c not in raw_train_te.columns or c not in raw_test_te.columns:
        raise ValueError(f"Missing required raw column: {c}")

def clean_str_50b(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str).to_numpy()

school_train = clean_str_50b(raw_train_te["SCHOOL"])
school_test = clean_str_50b(raw_test_te["SCHOOL"])

district_train = clean_str_50b(raw_train_te["DISTRICT"])
district_test = clean_str_50b(raw_test_te["DISTRICT"])

county_train = clean_str_50b(raw_train_te["COUNTY"])
county_test = clean_str_50b(raw_test_te["COUNTY"])

assessment_train = clean_str_50b(raw_train_te["ASSESSMENT_NAME"])
assessment_test = clean_str_50b(raw_test_te["ASSESSMENT_NAME"])

subgroup_train = clean_str_50b(raw_train_te["SUBGROUP_NAME"])
subgroup_test = clean_str_50b(raw_test_te["SUBGROUP_NAME"])

asg_train = np.array([f"{a}||{s}" for a, s in zip(assessment_train, subgroup_train)], dtype=object)
asg_test = np.array([f"{a}||{s}" for a, s in zip(assessment_test, subgroup_test)], dtype=object)

n_students_train = (
    pd.to_numeric(raw_train_te["N_STUDENTS"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .to_numpy(dtype=np.float64)
)

n_students_test = (
    pd.to_numeric(raw_test_te["N_STUDENTS"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .to_numpy(dtype=np.float64)
)

def make_index_pair(train_arr, test_arr):
    all_values = pd.Index(pd.Series(np.concatenate([train_arr, test_arr])).astype(str).unique())
    mapper = {v: i for i, v in enumerate(all_values)}
    train_idx = np.array([mapper[x] for x in train_arr], dtype=np.int32)
    test_idx = np.array([mapper[x] for x in test_arr], dtype=np.int32)
    return all_values, mapper, train_idx, test_idx

all_schools, _, school_idx_train, school_idx_test = make_index_pair(school_train, school_test)
all_districts, _, district_idx_train, district_idx_test = make_index_pair(district_train, district_test)
all_counties, _, county_idx_train, county_idx_test = make_index_pair(county_train, county_test)
all_asg, _, asg_idx_train, asg_idx_test = make_index_pair(asg_train, asg_test)

entity_index_data = {
    "school": {
        "train_idx": school_idx_train,
        "test_idx": school_idx_test,
        "n_entities": len(all_schools),
    },
    "district": {
        "train_idx": district_idx_train,
        "test_idx": district_idx_test,
        "n_entities": len(all_districts),
    },
    "county": {
        "train_idx": county_idx_train,
        "test_idx": county_idx_test,
        "n_entities": len(all_counties),
    },
}

n_items = len(all_asg)

print("\nMatrix dimensions")
print("-----------------")
print("n schools:", len(all_schools))
print("n districts:", len(all_districts))
print("n counties:", len(all_counties))
print("n assessment-subgroup items:", n_items)
print("train rows:", n_train50b)
print("test rows:", n_test50b)

# ------------------------------------------------------------
# 3. Accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

test_tier_rates = {
    tier: float(mask.mean())
    for tier, mask in tier_masks_test.items()
}

print("\nTier coverage")
print("-------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"44B MSE={mean_squared_error(y50b[mask], pred44b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 4. Transform and fitting helpers
# ------------------------------------------------------------

def pct_to_arc_50b(pct):
    p = np.clip(np.asarray(pct, dtype=np.float64) / 100.0, 1e-6, 1 - 1e-6)
    return np.arcsin(np.sqrt(p)).astype(np.float32)

def arc_to_pct_50b(z):
    p = np.sin(np.asarray(z, dtype=np.float64)) ** 2
    return np.clip(100.0 * p, 0, 100).astype(np.float32)

z_y = pct_to_arc_50b(y50b)
z_44b_oof = pct_to_arc_50b(pred44b_oof)
z_44b_test = pct_to_arc_50b(pred44b_test)
resid_z = (z_y - z_44b_oof).astype(np.float32)

def make_weights_50b(idx, scheme):
    idx = np.asarray(idx, dtype=np.int64)

    n = n_students_train[idx].astype(np.float64)
    valid_n = np.isfinite(n) & (n > 0)
    med_n = np.nanmedian(n[valid_n]) if valid_n.any() else 30.0
    n = np.where(valid_n, n, med_n)

    if scheme == "uniform":
        w = np.ones(len(idx), dtype=np.float64)
    elif scheme == "sqrt_n":
        w = np.sqrt(np.clip(n, 1, 500))
    elif scheme == "log_n":
        w = np.log1p(np.clip(n, 1, 500))
    elif scheme == "solver_mild":
        w = np.sqrt(np.clip(n, 1, 500))
        w[overlap_oof[idx]] *= 0.25
        w[solver_only_oof[idx]] *= 1.25
        w[none_oof[idx]] *= 0.75
    elif scheme == "solver_public":
        w = np.sqrt(np.clip(n, 1, 500))
        w[overlap_oof[idx]] *= 0.10
        w[solver_only_oof[idx]] *= 1.25
        w[none_oof[idx]] *= 0.75
    elif scheme == "none_mild":
        w = np.sqrt(np.clip(n, 1, 500))
        w[overlap_oof[idx]] *= 0.10
        w[solver_only_oof[idx]] *= 0.85
        w[none_oof[idx]] *= 1.25
    else:
        raise ValueError(f"Unknown weight scheme: {scheme}")

    return w.astype(np.float64)

def fit_biases_50b(
    idx,
    entity_idx_all,
    item_idx_all,
    n_entities,
    n_items,
    reg_entity,
    reg_item,
    weight_scheme,
    n_iter,
):
    idx = np.asarray(idx, dtype=np.int64)

    e_idx = entity_idx_all[idx]
    i_idx = item_idx_all[idx]
    r = resid_z[idx].astype(np.float64)
    w = make_weights_50b(idx, weight_scheme)

    global_bias = float(np.sum(w * r) / max(np.sum(w), 1e-12))

    entity_bias = np.zeros(n_entities, dtype=np.float64)
    item_bias = np.zeros(n_items, dtype=np.float64)

    for _ in range(int(n_iter)):
        residual_entity = r - global_bias - item_bias[i_idx]
        sum_w_e = np.bincount(e_idx, weights=w, minlength=n_entities)
        sum_wr_e = np.bincount(e_idx, weights=w * residual_entity, minlength=n_entities)
        entity_bias = sum_wr_e / (sum_w_e + float(reg_entity))

        residual_item = r - global_bias - entity_bias[e_idx]
        sum_w_i = np.bincount(i_idx, weights=w, minlength=n_items)
        sum_wr_i = np.bincount(i_idx, weights=w * residual_item, minlength=n_items)
        item_bias = sum_wr_i / (sum_w_i + float(reg_item))

    debiased = r - global_bias - entity_bias[e_idx] - item_bias[i_idx]

    return (
        global_bias,
        entity_bias.astype(np.float32),
        item_bias.astype(np.float32),
        debiased.astype(np.float32),
    )

def fit_svd_50b(
    idx,
    debiased_values,
    entity_idx_all,
    item_idx_all,
    n_entities,
    n_items,
    rank,
    random_state,
):
    idx = np.asarray(idx, dtype=np.int64)

    e_idx = entity_idx_all[idx]
    i_idx = item_idx_all[idx]

    matrix = sparse.csr_matrix(
        (debiased_values.astype(np.float32), (e_idx, i_idx)),
        shape=(n_entities, n_items),
        dtype=np.float32,
    )

    rank_eff = int(min(rank, max(1, min(n_entities, n_items) - 1)))

    svd = TruncatedSVD(
        n_components=rank_eff,
        n_iter=8,
        random_state=random_state,
    )

    entity_factors = svd.fit_transform(matrix).astype(np.float32)
    item_components = svd.components_.astype(np.float32)

    return entity_factors, item_components

def predict_mf_50b(
    base_z,
    entity_idx,
    item_idx,
    global_bias,
    entity_bias,
    item_bias,
    entity_factors,
    item_components,
    bias_scale,
    factor_scale,
):
    bias = (
        float(global_bias)
        + entity_bias[entity_idx].astype(np.float64)
        + item_bias[item_idx].astype(np.float64)
    )
    bias *= float(bias_scale)

    if entity_factors is None or item_components is None:
        factor = np.zeros(len(entity_idx), dtype=np.float64)
    else:
        factor = np.sum(
            entity_factors[entity_idx] * item_components[:, item_idx].T,
            axis=1,
        ).astype(np.float64)
        factor *= float(factor_scale)

    z_pred = base_z.astype(np.float64) + bias + factor
    return arc_to_pct_50b(z_pred)

def train_predict_config_oof_50b(cfg):
    entity_data = entity_index_data[cfg["entity_key"]]
    entity_idx_train = entity_data["train_idx"]
    entity_idx_test = entity_data["test_idx"]
    n_entities = entity_data["n_entities"]

    item_idx_train = asg_idx_train
    item_idx_test = asg_idx_test

    oof = np.full(n_train50b, np.nan, dtype=np.float32)
    test_sum = np.zeros(n_test50b, dtype=np.float64)
    fold_rows = []

    folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train50b)))

    for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
        global_bias, entity_bias, item_bias, debiased = fit_biases_50b(
            tr_idx,
            entity_idx_train,
            item_idx_train,
            n_entities,
            n_items,
            reg_entity=cfg["reg_entity"],
            reg_item=cfg["reg_item"],
            weight_scheme=cfg["weight_scheme"],
            n_iter=cfg["bias_iter"],
        )

        entity_factors, item_components = fit_svd_50b(
            tr_idx,
            debiased,
            entity_idx_train,
            item_idx_train,
            n_entities,
            n_items,
            rank=cfg["rank"],
            random_state=RANDOM_STATE + 1000 * fold_num + cfg["rank"],
        )

        pred_va = predict_mf_50b(
            base_z=z_44b_oof[va_idx],
            entity_idx=entity_idx_train[va_idx],
            item_idx=item_idx_train[va_idx],
            global_bias=global_bias,
            entity_bias=entity_bias,
            item_bias=item_bias,
            entity_factors=entity_factors,
            item_components=item_components,
            bias_scale=cfg["bias_scale"],
            factor_scale=cfg["factor_scale"],
        )

        pred_test = predict_mf_50b(
            base_z=z_44b_test,
            entity_idx=entity_idx_test,
            item_idx=item_idx_test,
            global_bias=global_bias,
            entity_bias=entity_bias,
            item_bias=item_bias,
            entity_factors=entity_factors,
            item_components=item_components,
            bias_scale=cfg["bias_scale"],
            factor_scale=cfg["factor_scale"],
        )

        oof[va_idx] = pred_va
        test_sum += pred_test.astype(np.float64)

        row = {
            "config": cfg["name"],
            "fold": fold_num,
            "mse_44b": float(mean_squared_error(y50b[va_idx], pred44b_oof[va_idx])),
            "mse_direct": float(mean_squared_error(y50b[va_idx], pred_va)),
        }
        row["gain_direct_vs_44b"] = row["mse_44b"] - row["mse_direct"]

        for tier, mask_full in tier_masks_oof.items():
            mask = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask.sum())
            if int(mask.sum()) > 0:
                row[f"mse44b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred44b_oof[va_idx][mask]))
                row[f"mse_direct_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred_va[mask]))
                row[f"gain_direct_{tier}"] = row[f"mse44b_{tier}"] - row[f"mse_direct_{tier}"]

        fold_rows.append(row)

        del entity_factors, item_components, entity_bias, item_bias, debiased
        gc.collect()

    test_avg = (test_sum / N_SPLITS).astype(np.float32)
    return oof, test_avg, pd.DataFrame(fold_rows)

def train_predict_config_fullfit_50b(cfg):
    entity_data = entity_index_data[cfg["entity_key"]]
    entity_idx_train = entity_data["train_idx"]
    entity_idx_test = entity_data["test_idx"]
    n_entities = entity_data["n_entities"]

    item_idx_train = asg_idx_train
    item_idx_test = asg_idx_test

    tr_idx = np.arange(n_train50b)

    global_bias, entity_bias, item_bias, debiased = fit_biases_50b(
        tr_idx,
        entity_idx_train,
        item_idx_train,
        n_entities,
        n_items,
        reg_entity=cfg["reg_entity"],
        reg_item=cfg["reg_item"],
        weight_scheme=cfg["weight_scheme"],
        n_iter=cfg["bias_iter"],
    )

    entity_factors, item_components = fit_svd_50b(
        tr_idx,
        debiased,
        entity_idx_train,
        item_idx_train,
        n_entities,
        n_items,
        rank=cfg["rank"],
        random_state=RANDOM_STATE + 777 + cfg["rank"],
    )

    pred_train_fullfit = predict_mf_50b(
        base_z=z_44b_oof,
        entity_idx=entity_idx_train,
        item_idx=item_idx_train,
        global_bias=global_bias,
        entity_bias=entity_bias,
        item_bias=item_bias,
        entity_factors=entity_factors,
        item_components=item_components,
        bias_scale=cfg["bias_scale"],
        factor_scale=cfg["factor_scale"],
    )

    pred_test_fullfit = predict_mf_50b(
        base_z=z_44b_test,
        entity_idx=entity_idx_test,
        item_idx=item_idx_test,
        global_bias=global_bias,
        entity_bias=entity_bias,
        item_bias=item_bias,
        entity_factors=entity_factors,
        item_components=item_components,
        bias_scale=cfg["bias_scale"],
        factor_scale=cfg["factor_scale"],
    )

    return pred_train_fullfit, pred_test_fullfit

# ------------------------------------------------------------
# 5. Config grid
# ------------------------------------------------------------

def safe_float_str(x):
    return str(x).replace(".", "p").replace("-", "m")

configs50b = []

if RUN_EXTENSIVE_50B:
    # Main local school-level search around 50A winner.
    for rank in [6, 8, 10, 12, 14, 16, 20, 24, 32]:
        for reg in [50.0, 75.0, 100.0, 125.0, 150.0, 200.0, 300.0, 500.0]:
            for weight_scheme in ["sqrt_n", "uniform"]:
                cfg = {
                    "entity_key": "school",
                    "rank": rank,
                    "reg_entity": reg,
                    "reg_item": reg,
                    "weight_scheme": weight_scheme,
                    "bias_iter": 4,
                    "bias_scale": 1.0,
                    "factor_scale": 1.0,
                }
                cfg["name"] = (
                    f"mf50b_{cfg['entity_key']}_rank{rank}"
                    f"_reg{safe_float_str(reg)}_{weight_scheme}"
                    f"_b{safe_float_str(cfg['bias_scale'])}"
                    f"_f{safe_float_str(cfg['factor_scale'])}"
                )
                configs50b.append(cfg)

    # A focused factor-scale search near the 50A winner.
    for rank in [8, 10, 12, 14, 16, 20]:
        for reg in [75.0, 100.0, 125.0, 150.0]:
            for factor_scale in [0.50, 0.75, 1.25, 1.50]:
                cfg = {
                    "entity_key": "school",
                    "rank": rank,
                    "reg_entity": reg,
                    "reg_item": reg,
                    "weight_scheme": "sqrt_n",
                    "bias_iter": 4,
                    "bias_scale": 1.0,
                    "factor_scale": factor_scale,
                }
                cfg["name"] = (
                    f"mf50b_{cfg['entity_key']}_rank{rank}"
                    f"_reg{safe_float_str(reg)}_sqrt_n"
                    f"_b{safe_float_str(cfg['bias_scale'])}"
                    f"_f{safe_float_str(cfg['factor_scale'])}"
                )
                configs50b.append(cfg)

    # Mild alternate weighting.
    for rank in [8, 12, 16, 20]:
        for reg in [75.0, 100.0, 150.0, 250.0]:
            for weight_scheme in ["log_n", "solver_mild", "none_mild"]:
                cfg = {
                    "entity_key": "school",
                    "rank": rank,
                    "reg_entity": reg,
                    "reg_item": reg,
                    "weight_scheme": weight_scheme,
                    "bias_iter": 4,
                    "bias_scale": 1.0,
                    "factor_scale": 1.0,
                }
                cfg["name"] = (
                    f"mf50b_{cfg['entity_key']}_rank{rank}"
                    f"_reg{safe_float_str(reg)}_{weight_scheme}"
                    f"_b{safe_float_str(cfg['bias_scale'])}"
                    f"_f{safe_float_str(cfg['factor_scale'])}"
                )
                configs50b.append(cfg)

    # Smaller district/county searches.
    for entity_key in ["district", "county"]:
        for rank in [2, 4, 8, 12, 16]:
            for reg in [25.0, 50.0, 100.0, 200.0]:
                cfg = {
                    "entity_key": entity_key,
                    "rank": rank,
                    "reg_entity": reg,
                    "reg_item": reg,
                    "weight_scheme": "sqrt_n",
                    "bias_iter": 4,
                    "bias_scale": 1.0,
                    "factor_scale": 1.0,
                }
                cfg["name"] = (
                    f"mf50b_{cfg['entity_key']}_rank{rank}"
                    f"_reg{safe_float_str(reg)}_sqrt_n"
                    f"_b{safe_float_str(cfg['bias_scale'])}"
                    f"_f{safe_float_str(cfg['factor_scale'])}"
                )
                configs50b.append(cfg)
else:
    # Short version if you need a quick rerun.
    for rank in [8, 10, 12, 14, 16]:
        for reg in [75.0, 100.0, 125.0, 150.0, 200.0]:
            cfg = {
                "entity_key": "school",
                "rank": rank,
                "reg_entity": reg,
                "reg_item": reg,
                "weight_scheme": "sqrt_n",
                "bias_iter": 4,
                "bias_scale": 1.0,
                "factor_scale": 1.0,
            }
            cfg["name"] = (
                f"mf50b_{cfg['entity_key']}_rank{rank}"
                f"_reg{safe_float_str(reg)}_sqrt_n"
                f"_b{safe_float_str(cfg['bias_scale'])}"
                f"_f{safe_float_str(cfg['factor_scale'])}"
            )
            configs50b.append(cfg)

# Deduplicate configs by name.
cfg_by_name = {}
for cfg in configs50b:
    cfg_by_name[cfg["name"]] = cfg
configs50b = list(cfg_by_name.values())

print("\nNumber of 50B configs:", len(configs50b))
print("First 10 configs:")
for cfg in configs50b[:10]:
    print(cfg)

# Save config manifest.
with open("model_results/mf50b_config_manifest.json", "w") as f:
    json.dump(configs50b, f, indent=2)

# ------------------------------------------------------------
# 6. Run or load checkpoints
# ------------------------------------------------------------

cand_oof = {}
cand_test = {}
fold_metric_frames = []
completed_rows = []

t0 = time.time()

for cfg_num, cfg in enumerate(configs50b, start=1):
    name = cfg["name"]

    oof_path = CHECKPOINT_DIR / f"{name}_oof.npy"
    test_path = CHECKPOINT_DIR / f"{name}_test.npy"
    folds_path = CHECKPOINT_DIR / f"{name}_folds.csv"
    meta_path = CHECKPOINT_DIR / f"{name}_meta.json"

    print("\n" + "=" * 90)
    print(f"Config {cfg_num}/{len(configs50b)}: {name}")
    print("=" * 90)

    if oof_path.exists() and test_path.exists() and folds_path.exists():
        print("Loading checkpoint.")
        oof_pred = np.load(oof_path).astype(np.float32)
        test_pred = np.load(test_path).astype(np.float32)
        fold_df = pd.read_csv(folds_path)
    else:
        start = time.time()
        oof_pred, test_pred, fold_df = train_predict_config_oof_50b(cfg)
        np.save(oof_path, oof_pred)
        np.save(test_path, test_pred)
        fold_df.to_csv(folds_path, index=False)
        with open(meta_path, "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Finished config in {time.time() - start:.1f}s")

    cand_oof[name] = oof_pred
    cand_test[name] = test_pred
    fold_metric_frames.append(fold_df)

    direct_mse = float(mean_squared_error(y50b, oof_pred))
    direct_gain_vs_44b = mse44b - direct_mse

    if has_50a:
        direct_gain_vs_50a = mse50a - direct_mse
    else:
        direct_gain_vs_50a = np.nan

    completed_rows.append({
        "config": name,
        "entity_key": cfg["entity_key"],
        "rank": cfg["rank"],
        "reg_entity": cfg["reg_entity"],
        "reg_item": cfg["reg_item"],
        "weight_scheme": cfg["weight_scheme"],
        "bias_iter": cfg["bias_iter"],
        "bias_scale": cfg["bias_scale"],
        "factor_scale": cfg["factor_scale"],
        "direct_oof_mse": direct_mse,
        "direct_gain_vs_44b": direct_gain_vs_44b,
        "direct_gain_vs_50a": direct_gain_vs_50a,
    })

    pd.DataFrame(completed_rows).to_csv("model_results/mf50b_direct_progress.csv", index=False)
    print(f"Direct OOF MSE: {direct_mse:.6f} | gain vs 44B: {direct_gain_vs_44b:.6f}")

print("\nTotal 50B config pass elapsed seconds:", round(time.time() - t0, 1))

fold_metrics50b = pd.concat(fold_metric_frames, axis=0).reset_index(drop=True)

# ------------------------------------------------------------
# 7. Blend scan
# ------------------------------------------------------------

lambda_grid = np.unique(
    np.concatenate([
        np.linspace(-0.25, 1.25, 601),
        np.array([0.0, 0.005, 0.05, 0.10, 0.155, 0.25, 0.395, 0.495, 0.50, 0.75, 1.0])
    ])
)

def best_lambda_for_mask_50b(candidate_pred, mask):
    if int(mask.sum()) == 0:
        return 0.0, np.nan

    y = y50b[mask].astype(np.float64)
    base = pred44b_oof[mask].astype(np.float64)
    alt = candidate_pred[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid:
        pred = np.clip(base + float(lam) * (alt - base), 0, 100)
        mse = float(mean_squared_error(y, pred))

        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam, best_mse

def make_blend_50b(candidate_oof, candidate_test, lams, is_test=False):
    if is_test:
        base = pred44b_test.astype(np.float64)
        alt = candidate_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred44b_oof.astype(np.float64)
        alt = candidate_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

def tier_metrics_50b(pred):
    out = {}
    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y50b[mask], pred[mask]))
        out[f"gain_{tier}_vs_44b"] = (
            float(mean_squared_error(y50b[mask], pred44b_oof[mask]))
            - out[f"mse_{tier}"]
        )
        if has_50a:
            out[f"gain_{tier}_vs_50a"] = (
                float(mean_squared_error(y50b[mask], pred50a_oof[mask]))
                - out[f"mse_{tier}"]
            )
    return out

def tier_weighted_mse_50b(pred):
    pred = np.clip(pred, 0, 100)
    total = 0.0
    for tier, mask in tier_masks_oof.items():
        total += test_tier_rates[tier] * float(mean_squared_error(y50b[mask], pred[mask]))
    return float(total)

screen_rows = []
blend_store = {}

for cfg in configs50b:
    name = cfg["name"]
    p_oof = cand_oof[name]
    p_test = cand_test[name]

    direct_mse = float(mean_squared_error(y50b, p_oof))

    lam_overlap, _ = best_lambda_for_mask_50b(p_oof, overlap_oof)
    lam_solver, _ = best_lambda_for_mask_50b(p_oof, solver_only_oof)
    lam_none, _ = best_lambda_for_mask_50b(p_oof, none_oof)

    strategy_lams = {
        "solver_only": {
            "overlap": 0.0,
            "solver_only": lam_solver,
            "none": 0.0,
        },
        "none_only": {
            "overlap": 0.0,
            "solver_only": 0.0,
            "none": lam_none,
        },
        "solver_plus_none": {
            "overlap": 0.0,
            "solver_only": lam_solver,
            "none": lam_none,
        },
        "all_tiers": {
            "overlap": lam_overlap,
            "solver_only": lam_solver,
            "none": lam_none,
        },
    }

    for strategy, lams in strategy_lams.items():
        pred = make_blend_50b(p_oof, p_test, lams, is_test=False)
        test_pred = make_blend_50b(p_oof, p_test, lams, is_test=True)

        mse = float(mean_squared_error(y50b, pred))

        row = {
            "config": name,
            "strategy": strategy,
            "entity_key": cfg["entity_key"],
            "rank": cfg["rank"],
            "reg_entity": cfg["reg_entity"],
            "reg_item": cfg["reg_item"],
            "weight_scheme": cfg["weight_scheme"],
            "bias_iter": cfg["bias_iter"],
            "bias_scale": cfg["bias_scale"],
            "factor_scale": cfg["factor_scale"],
            "direct_oof_mse": direct_mse,
            "lambda_overlap": float(lams["overlap"]),
            "lambda_solver_only": float(lams["solver_only"]),
            "lambda_none": float(lams["none"]),
            "oof_mse": mse,
            "gain_vs_44b": mse44b - mse,
            "test_tier_weighted_mse": tier_weighted_mse_50b(pred),
            "weighted_gain_vs_44b": tier_weighted_mse_50b(pred44b_oof) - tier_weighted_mse_50b(pred),
        }

        if has_50a:
            row["gain_vs_50a"] = mse50a - mse
            row["weighted_gain_vs_50a"] = tier_weighted_mse_50b(pred50a_oof) - tier_weighted_mse_50b(pred)
        else:
            row["gain_vs_50a"] = np.nan
            row["weighted_gain_vs_50a"] = np.nan

        row.update(tier_metrics_50b(pred))

        key = f"{name}__{strategy}"
        row["key"] = key

        screen_rows.append(row)
        blend_store[key] = {
            "oof": pred,
            "test": test_pred,
            "lams": lams,
            "config": cfg,
        }

screen50b = pd.DataFrame(screen_rows).sort_values(["oof_mse", "test_tier_weighted_mse"]).reset_index(drop=True)
screen50b_weighted = screen50b.sort_values(["test_tier_weighted_mse", "oof_mse"]).reset_index(drop=True)

best_key = screen50b.iloc[0]["key"]
best_row = screen50b.iloc[0]
best_oof = blend_store[best_key]["oof"]
best_test = blend_store[best_key]["test"]
best_cfg = blend_store[best_key]["config"]

# ------------------------------------------------------------
# 8. Fold diagnostics for best OOF candidate
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train50b)))

fold_diag_rows = []

for fold_num, (_, va_idx) in enumerate(folds, start=1):
    row = {
        "fold": fold_num,
        "mse_44b": float(mean_squared_error(y50b[va_idx], pred44b_oof[va_idx])),
        "mse_50b": float(mean_squared_error(y50b[va_idx], best_oof[va_idx])),
    }
    row["gain_vs_44b"] = row["mse_44b"] - row["mse_50b"]

    if has_50a:
        row["mse_50a"] = float(mean_squared_error(y50b[va_idx], pred50a_oof[va_idx]))
        row["gain_vs_50a"] = row["mse_50a"] - row["mse_50b"]

    for tier, mask_full in tier_masks_oof.items():
        mask = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse44b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred44b_oof[va_idx][mask]))
            row[f"mse50b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], best_oof[va_idx][mask]))
            row[f"gain_{tier}_vs_44b"] = row[f"mse44b_{tier}"] - row[f"mse50b_{tier}"]
            if has_50a:
                row[f"mse50a_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred50a_oof[va_idx][mask]))
                row[f"gain_{tier}_vs_50a"] = row[f"mse50a_{tier}"] - row[f"mse50b_{tier}"]

    fold_diag_rows.append(row)

fold_diag50b = pd.DataFrame(fold_diag_rows)

# ------------------------------------------------------------
# 9. Full-train refit for top configs
# ------------------------------------------------------------

# Pick top unique configs by OOF screen.
top_unique_configs = []
seen_cfg = set()

for _, row in screen50b.iterrows():
    cfg_name = row["config"]
    if cfg_name not in seen_cfg:
        seen_cfg.add(cfg_name)
        top_unique_configs.append(cfg_name)
    if len(top_unique_configs) >= N_FULLFIT_CONFIGS:
        break

cfg_lookup = {cfg["name"]: cfg for cfg in configs50b}

fullfit_rows = []
fullfit_store = {}

for cfg_name in top_unique_configs:
    cfg = cfg_lookup[cfg_name]
    print("\n" + "=" * 90)
    print("Full-train refit:", cfg_name)
    print("=" * 90)

    full_oof_direct, full_test_direct = train_predict_config_fullfit_50b(cfg)

    # Use the best OOF-selected strategy for this config.
    cfg_screen = screen50b[screen50b["config"] == cfg_name].sort_values("oof_mse").reset_index(drop=True)
    cfg_best = cfg_screen.iloc[0]
    cfg_lams = {
        "overlap": float(cfg_best["lambda_overlap"]),
        "solver_only": float(cfg_best["lambda_solver_only"]),
        "none": float(cfg_best["lambda_none"]),
    }

    full_oof_blend = make_blend_50b(full_oof_direct, full_test_direct, cfg_lams, is_test=False)
    full_test_blend = make_blend_50b(full_oof_direct, full_test_direct, cfg_lams, is_test=True)

    # In-sample metric only; not honest validation.
    trainfit_mse = float(mean_squared_error(y50b, full_oof_blend))

    foldavg_test_for_cfg = blend_store[cfg_best["key"]]["test"]

    row = {
        "config": cfg_name,
        "selected_strategy": cfg_best["strategy"],
        "lambda_overlap": cfg_lams["overlap"],
        "lambda_solver_only": cfg_lams["solver_only"],
        "lambda_none": cfg_lams["none"],
        "oof_selected_mse": float(cfg_best["oof_mse"]),
        "oof_gain_vs_44b": float(cfg_best["gain_vs_44b"]),
        "oof_gain_vs_50a": float(cfg_best["gain_vs_50a"]) if has_50a else np.nan,
        "trainfit_mse_not_oof": trainfit_mse,
        "mean_abs_diff_fullfit_vs_foldavg_test": float(np.mean(np.abs(full_test_blend - foldavg_test_for_cfg))),
        "p95_abs_diff_fullfit_vs_foldavg_test": float(np.percentile(np.abs(full_test_blend - foldavg_test_for_cfg), 95)),
        "max_abs_diff_fullfit_vs_foldavg_test": float(np.max(np.abs(full_test_blend - foldavg_test_for_cfg))),
    }

    if has_50a:
        row["mean_abs_diff_fullfit_vs_50a_test"] = float(np.mean(np.abs(full_test_blend - pred50a_test)))
        row["p95_abs_diff_fullfit_vs_50a_test"] = float(np.percentile(np.abs(full_test_blend - pred50a_test), 95))

    fullfit_rows.append(row)

    fullfit_store[cfg_name] = {
        "full_oof_direct": full_oof_direct,
        "full_test_direct": full_test_direct,
        "full_oof_blend": full_oof_blend,
        "full_test_blend": full_test_blend,
        "foldavg_test": foldavg_test_for_cfg,
        "cfg_best_row": cfg_best,
        "lams": cfg_lams,
    }

fullfit50b = pd.DataFrame(fullfit_rows).sort_values(["oof_selected_mse", "mean_abs_diff_fullfit_vs_foldavg_test"]).reset_index(drop=True)

# ------------------------------------------------------------
# 10. Save artifacts and submissions
# ------------------------------------------------------------

screen_path = "model_results/mf50b_screen.csv"
weighted_path = "model_results/mf50b_screen_weighted.csv"
direct_progress_path = "model_results/mf50b_direct_progress.csv"
fold_metrics_path = "model_results/mf50b_fold_metrics.csv"
fold_diag_path = "model_results/mf50b_best_fold_diag.csv"
fullfit_path = "model_results/mf50b_fullfit_screen.csv"

oof_best_path = "model_results/oof_mf50b_best_oof.csv"
test_best_path = "model_results/testpred_mf50b_best_oof.csv"
submission_best_path = "submission_mf50b_best_oof.csv"

screen50b.to_csv(screen_path, index=False)
screen50b_weighted.to_csv(weighted_path, index=False)
pd.DataFrame(completed_rows).to_csv(direct_progress_path, index=False)
fold_metrics50b.to_csv(fold_metrics_path, index=False)
fold_diag50b.to_csv(fold_diag_path, index=False)
fullfit50b.to_csv(fullfit_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train50b),
    TARGET_COL: y50b,
    "pred_44b": pred44b_oof,
    "pred_50a": pred50a_oof if has_50a else np.nan,
    "pred_clipped": best_oof,
    "tier_overlap": overlap_oof.astype(int),
    "tier_solver_only": solver_only_oof.astype(int),
    "tier_none": none_oof.astype(int),
}).to_csv(oof_best_path, index=False)

pd.DataFrame({
    ID_COL: test_ids50b,
    "pred_44b": pred44b_test,
    "pred_50a": pred50a_test if has_50a else np.nan,
    TARGET_COL: best_test,
    "tier_overlap": overlap_test.astype(int),
    "tier_solver_only": solver_only_test.astype(int),
    "tier_none": none_test.astype(int),
}).to_csv(test_best_path, index=False)

pd.DataFrame({
    ID_COL: test_ids50b,
    TARGET_COL: best_test,
}).to_csv(submission_best_path, index=False)

# Fullfit and fold/fullfit blend submissions.
fullfit_submission_paths = []

if len(fullfit50b) > 0:
    for row_idx, row in fullfit50b.iterrows():
        cfg_name = row["config"]
        obj = fullfit_store[cfg_name]

        tag = f"rank{int(cfg_lookup[cfg_name]['rank'])}_{cfg_lookup[cfg_name]['entity_key']}_reg{safe_float_str(cfg_lookup[cfg_name]['reg_entity'])}_{cfg_lookup[cfg_name]['weight_scheme']}"
        base_name = f"mf50b_fullfit_{row_idx+1}_{tag}"

        # Fullfit submission.
        path_full = f"submission_{base_name}.csv"
        pd.DataFrame({
            ID_COL: test_ids50b,
            TARGET_COL: obj["full_test_blend"],
        }).to_csv(path_full, index=False)
        fullfit_submission_paths.append(path_full)

        # Blends between OOF fold-averaged test prediction and fullfit test prediction.
        for w_full in [0.25, 0.50, 0.75]:
            mix = (
                float(w_full) * obj["full_test_blend"].astype(np.float64)
                + (1.0 - float(w_full)) * obj["foldavg_test"].astype(np.float64)
            )
            mix = np.clip(mix, 0, 100).astype(np.float32)

            path_mix = f"submission_{base_name}_mixfull{safe_float_str(w_full)}.csv"
            pd.DataFrame({
                ID_COL: test_ids50b,
                TARGET_COL: mix,
            }).to_csv(path_mix, index=False)
            fullfit_submission_paths.append(path_mix)

# Validate submissions.
all_submission_paths = [submission_best_path] + fullfit_submission_paths

for p in all_submission_paths:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test50b, 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[ID_COL].notna().all(), p
    assert sub[TARGET_COL].notna().all(), p
    assert np.isfinite(sub[TARGET_COL]).all(), p
    assert sub[TARGET_COL].between(0, 100).all(), p

# ------------------------------------------------------------
# 11. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("50B extensive MF refinement complete")
print("=" * 90)

print("\nReference")
print("---------")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")
if has_50a:
    print(f"50A OOF MSE: {mse50a:.6f} | public 29.742")

print("\nTop 30 OOF-safe 50B candidates")
print("------------------------------")
display(screen50b.head(30))

print("\nTop 20 weighted candidates")
print("--------------------------")
display(screen50b_weighted.head(20))

print("\nBest OOF-safe 50B")
print("-----------------")
print(best_row.to_string())

print("\nBest fold diagnostics")
print("---------------------")
print(fold_diag50b.to_string(index=False))
print("Min fold gain vs 44B:", float(fold_diag50b["gain_vs_44b"].min()))
if has_50a and "gain_vs_50a" in fold_diag50b.columns:
    print("Min fold gain vs 50A:", float(fold_diag50b["gain_vs_50a"].min()))

print("\nFull-train refit screen")
print("-----------------------")
display(fullfit50b)

print("\nSaved files")
print("-----------")
print(screen_path)
print(weighted_path)
print(direct_progress_path)
print(fold_metrics_path)
print(fold_diag_path)
print(fullfit_path)
print(oof_best_path)
print(test_best_path)
print(submission_best_path)
for p in fullfit_submission_paths:
    print(p)

print("\nSubmission validation")
print("---------------------")
for p in all_submission_paths[:10]:
    sub = pd.read_csv(p)
    print(p, sub.shape, sub[TARGET_COL].describe().to_dict())
if len(all_submission_paths) > 10:
    print(f"... {len(all_submission_paths) - 10} additional submission files validated.")

print("\nDecision guidance")
print("-----------------")
print("Primary OOF-safe candidate:")
print(submission_best_path)

if len(fullfit50b) > 0:
    print("\nMost natural full-refit probes are the first row in the fullfit table:")
    print(f"submission_mf50b_fullfit_1_*.csv")
    print("and its mixfull0p25 / mixfull0p50 variants.")
print("\nUse public probing order only after inspecting whether 50B materially improves 50A OOF and has stable folds.")

50B. Extensive low-rank matrix-factorization refinement + full refit
Checkpoint dir: model_results/mf50b_checkpoints

Protected references
--------------------
44B OOF MSE: 52.780037 | public 30.556
50A OOF MSE: 52.247093 | public 29.742

Matrix dimensions
-----------------
n schools: 4469
n districts: 710
n counties: 62
n assessment-subgroup items: 132
train rows: 144921
test rows: 48307

Tier coverage
-------------
overlap      train= 53786 test= 27298 44B MSE=0.507021
solver_only  train= 27807 test= 17807 44B MSE=64.262032
none         train= 63328 test=  3202 44B MSE=92.135094

Number of 50B configs: 328
First 10 configs:
{'entity_key': 'school', 'rank': 6, 'reg_entity': 50.0, 'reg_item': 50.0, 'weight_scheme': 'sqrt_n', 'bias_iter': 4, 'bias_scale': 1.0, 'factor_scale': 1.0, 'name': 'mf50b_school_rank6_reg50p0_sqrt_n_b1p0_f1p0'}
{'entity_key': 'school', 'rank': 6, 'reg_entity': 50.0, 'reg_item': 50.0, 'weight_scheme': 'uniform', 'bias_iter': 4, 'bias_scale': 1.0, 'factor_scale': 1

,config,strategy,entity_key,rank,reg_entity,reg_item,weight_scheme,bias_iter,bias_scale,factor_scale,direct_oof_mse,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_44b,test_tier_weighted_mse,weighted_gain_vs_44b,gain_vs_50a,weighted_gain_vs_50a,mse_overlap,gain_overlap_vs_44b,gain_overlap_vs_50a,mse_solver_only,gain_solver_only_vs_44b,gain_solver_only_vs_50a,mse_none,gain_none_vs_44b,gain_none_vs_50a,key
0,mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0,all_tiers,school,16,500.0,500.0,uniform,4,1.0,1.0,52.522476,0.0050,0.5475,0.8250,51.463158,1.316879,29.618224,0.463776,0.783936,0.225839,0.506983,0.000038,-0.000034,63.484444,0.777588,0.314987,89.462982,2.672112,1.655701,mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0...
1,mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0,solver_plus_none,school,16,500.0,500.0,uniform,4,1.0,1.0,52.522476,0.0000,0.5475,0.8250,51.463173,1.316864,29.618246,0.463755,0.783920,0.225817,0.507021,0.000000,-0.000072,63.484444,0.777588,0.314987,89.462982,2.672112,1.655701,mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0...
2,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,all_tiers,school,20,500.0,500.0,uniform,4,1.0,1.0,52.583138,0.0025,0.4975,0.8200,51.475582,1.304455,29.668911,0.413090,0.771511,0.175152,0.507008,0.000013,-0.000059,63.628147,0.633884,0.171284,89.428291,2.706802,1.690392,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...
3,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,solver_plus_none,school,20,500.0,500.0,uniform,4,1.0,1.0,52.583138,0.0000,0.4975,0.8200,51.475586,1.304451,29.668919,0.413082,0.771507,0.175145,0.507021,0.000000,-0.000072,63.628147,0.633884,0.171284,89.428291,2.706802,1.690392,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...
4,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0,all_tiers,school,14,500.0,500.0,uniform,4,1.0,1.0,52.490196,0.0050,0.5775,0.8275,51.483036,1.297001,29.597466,0.484535,0.764057,0.246597,0.506985,0.000036,-0.000036,63.414417,0.847614,0.385014,89.539223,2.595871,1.579460,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0...
5,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0,solver_plus_none,school,14,500.0,500.0,uniform,4,1.0,1.0,52.490196,0.0000,0.5775,0.8275,51.483047,1.296989,29.597486,0.484515,0.764046,0.246577,0.507021,0.000000,-0.000072,63.414417,0.847614,0.385014,89.539223,2.595871,1.579460,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0...
6,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,all_tiers,school,12,500.0,500.0,uniform,4,1.0,1.0,52.431503,0.0050,0.6150,0.8375,51.484856,1.295181,29.577413,0.504588,0.762238,0.266651,0.506955,0.000065,-0.000007,63.354584,0.907448,0.444847,89.569687,2.565407,1.548996,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...
7,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,solver_plus_none,school,12,500.0,500.0,uniform,4,1.0,1.0,52.431503,0.0000,0.6150,0.8375,51.484879,1.295158,29.577450,0.504551,0.762215,0.266614,0.507021,0.000000,-0.000072,63.354584,0.907448,0.444847,89.569687,2.565407,1.548996,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...
8,mf50b_school_rank16_reg300p0_uniform_b1p0_f1p0,all_tiers,school,16,300.0,300.0,uniform,4,1.0,1.0,52.589603,0.0050,0.5400,0.8075,51.505215,1.274822,29.629632,0.452369,0.741879,0.214431,0.506982,0.000039,-0.000033,63.499252,0.762779,0.300179,89.552734,2.582359,1.565948,mf50b_school_rank16_reg300p0_uniform_b1p0_f1p0...
9,mf50b_school_rank16_reg300p0_uniform_b1p0_f1p0,solver_plus_none,school,16,300.0,300.0,uniform,4,1.0,1.0,52.589603,0.0000,0.5400,0.8075,51.505234,1.274803,29.629654,0.452347,0.741859,0.214410,0.507021,0.000000,-0.000072,63.499252,0.762779,0.300179,89.552734,2.582359,1.565948,mf50b_school_rank16_reg300p0_uniform_b1p0_f1p0...



Top 20 weighted candidates
--------------------------


,config,strategy,entity_key,rank,reg_entity,reg_item,weight_scheme,bias_iter,bias_scale,factor_scale,direct_oof_mse,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_44b,test_tier_weighted_mse,weighted_gain_vs_44b,gain_vs_50a,weighted_gain_vs_50a,mse_overlap,gain_overlap_vs_44b,gain_overlap_vs_50a,mse_solver_only,gain_solver_only_vs_44b,gain_solver_only_vs_50a,mse_none,gain_none_vs_44b,gain_none_vs_50a,key
0,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,all_tiers,school,12,500.0,500.0,uniform,4,1.0,1.0,52.431503,0.005,0.6150,0.8375,51.484856,1.295181,29.577413,0.504588,0.762238,0.266651,0.506955,0.000065,-6.556511e-06,63.354584,0.907448,0.444847,89.569687,2.565407,1.548996,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...
1,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,solver_plus_none,school,12,500.0,500.0,uniform,4,1.0,1.0,52.431503,0.000,0.6150,0.8375,51.484879,1.295158,29.577450,0.504551,0.762215,0.266614,0.507021,0.000000,-7.200241e-05,63.354584,0.907448,0.444847,89.569687,2.565407,1.548996,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...
2,mf50b_school_rank12_reg300p0_uniform_b1p0_f1p0,all_tiers,school,12,300.0,300.0,uniform,4,1.0,1.0,52.521049,0.005,0.6025,0.8100,51.543404,1.236633,29.593361,0.488639,0.703690,0.250702,0.506952,0.000069,-3.099442e-06,63.375408,0.886623,0.424023,89.694519,2.440575,1.424164,mf50b_school_rank12_reg300p0_uniform_b1p0_f1p0...
3,mf50b_school_rank12_reg300p0_uniform_b1p0_f1p0,solver_plus_none,school,12,300.0,300.0,uniform,4,1.0,1.0,52.521049,0.000,0.6025,0.8100,51.543427,1.236610,29.593400,0.488600,0.703667,0.250663,0.507021,0.000000,-7.200241e-05,63.375408,0.886623,0.424023,89.694519,2.440575,1.424164,mf50b_school_rank12_reg300p0_uniform_b1p0_f1p0...
4,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,all_tiers,school,10,500.0,500.0,uniform,4,1.0,1.0,52.388744,0.005,0.6275,0.8450,51.536339,1.243698,29.594585,0.487416,0.710754,0.249479,0.506958,0.000063,-9.059906e-06,63.382156,0.879875,0.417274,89.675392,2.459702,1.443291,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...
5,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,solver_plus_none,school,10,500.0,500.0,uniform,4,1.0,1.0,52.388744,0.000,0.6275,0.8450,51.536366,1.243671,29.594620,0.487381,0.710728,0.249443,0.507021,0.000000,-7.200241e-05,63.382156,0.879875,0.417274,89.675392,2.459702,1.443291,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...
6,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0,all_tiers,school,14,500.0,500.0,uniform,4,1.0,1.0,52.490196,0.005,0.5775,0.8275,51.483036,1.297001,29.597466,0.484535,0.764057,0.246597,0.506985,0.000036,-3.635883e-05,63.414417,0.847614,0.385014,89.539223,2.595871,1.579460,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0...
7,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0,solver_plus_none,school,14,500.0,500.0,uniform,4,1.0,1.0,52.490196,0.000,0.5775,0.8275,51.483047,1.296989,29.597486,0.484515,0.764046,0.246577,0.507021,0.000000,-7.200241e-05,63.414417,0.847614,0.385014,89.539223,2.595871,1.579460,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0...
8,mf50b_school_rank12_reg200p0_uniform_b1p0_f1p0,all_tiers,school,12,200.0,200.0,uniform,4,1.0,1.0,52.616024,0.005,0.5900,0.7850,51.605522,1.174515,29.611342,0.470659,0.641571,0.232721,0.506948,0.000073,5.364418e-07,63.400620,0.861412,0.398811,89.825607,2.309486,1.293076,mf50b_school_rank12_reg200p0_uniform_b1p0_f1p0...
9,mf50b_school_rank12_reg200p0_uniform_b1p0_f1p0,solver_plus_none,school,12,200.0,200.0,uniform,4,1.0,1.0,52.616024,0.000,0.5900,0.7850,51.605549,1.174488,29.611383,0.470618,0.641544,0.232680,0.507021,0.000000,-7.200241e-05,63.400620,0.861412,0.398811,89.825607,2.309486,1.293076,mf50b_school_rank12_reg200p0_uniform_b1p0_f1p0...



Best OOF-safe 50B
-----------------
config                        mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0
strategy                                                           all_tiers
entity_key                                                            school
rank                                                                      16
reg_entity                                                             500.0
reg_item                                                               500.0
weight_scheme                                                        uniform
bias_iter                                                                  4
bias_scale                                                               1.0
factor_scale                                                             1.0
direct_oof_mse                                                     52.522476
lambda_overlap                                                         0.005
lambda_solver_only                     

,config,selected_strategy,lambda_overlap,lambda_solver_only,lambda_none,oof_selected_mse,oof_gain_vs_44b,oof_gain_vs_50a,trainfit_mse_not_oof,mean_abs_diff_fullfit_vs_foldavg_test,p95_abs_diff_fullfit_vs_foldavg_test,max_abs_diff_fullfit_vs_foldavg_test,mean_abs_diff_fullfit_vs_50a_test,p95_abs_diff_fullfit_vs_50a_test
0,mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0,all_tiers,0.0050,0.5475,0.8250,51.463158,1.316879,0.783936,36.218914,0.146578,0.767232,7.185913,0.270790,1.373751
1,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,all_tiers,0.0025,0.4975,0.8200,51.475582,1.304455,0.771511,33.887459,0.149343,0.773160,6.193687,0.285557,1.439026
2,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0,all_tiers,0.0050,0.5775,0.8275,51.483036,1.297001,0.764057,37.499912,0.142140,0.758881,6.466343,0.264242,1.327232
3,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,all_tiers,0.0050,0.6150,0.8375,51.484856,1.295181,0.762238,38.809891,0.144438,0.786332,7.571281,0.265682,1.347692
4,mf50b_school_rank16_reg300p0_uniform_b1p0_f1p0,all_tiers,0.0050,0.5400,0.8075,51.505215,1.274822,0.741879,36.364399,0.144880,0.755127,7.393681,0.263126,1.338106



Saved files
-----------
model_results/mf50b_screen.csv
model_results/mf50b_screen_weighted.csv
model_results/mf50b_direct_progress.csv
model_results/mf50b_fold_metrics.csv
model_results/mf50b_best_fold_diag.csv
model_results/mf50b_fullfit_screen.csv
model_results/oof_mf50b_best_oof.csv
model_results/testpred_mf50b_best_oof.csv
submission_mf50b_best_oof.csv
submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform.csv
submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform_mixfull0p25.csv
submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform_mixfull0p5.csv
submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform_mixfull0p75.csv
submission_mf50b_fullfit_2_rank20_school_reg500p0_uniform.csv
submission_mf50b_fullfit_2_rank20_school_reg500p0_uniform_mixfull0p25.csv
submission_mf50b_fullfit_2_rank20_school_reg500p0_uniform_mixfull0p5.csv
submission_mf50b_fullfit_2_rank20_school_reg500p0_uniform_mixfull0p75.csv
submission_mf50b_fullfit_3_rank14_school_reg500p0_uniform.csv
submission_m

In [58]:
# ============================================================
# 50C. Repeated-CV bagged MF ensemble around 50B winner
# ============================================================
#
# Current known public-best submitted:
#   50A public MSE = 29.742
#
# Current best OOF-safe candidate:
#   50B best OOF:
#       config = mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0
#       OOF MSE = 51.463158
#       gain vs 50A = 0.783936
#
# Goal:
#   Bag the best MF configurations across several CV seeds, then
#   produce OOF-safe ensemble candidates.
#
# This is safe:
#   - reuses sparse MF functions from 50B
#   - no neural nets
#   - no dense giant matrices
#   - checkpointed
# ============================================================

import os
import gc
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

Path("model_results").mkdir(exist_ok=True)
CHECKPOINT_DIR_50C = Path("model_results/mf50c_bag_checkpoints")
CHECKPOINT_DIR_50C.mkdir(parents=True, exist_ok=True)

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5

print("=" * 90)
print("50C. Repeated-CV bagged MF ensemble around 50B winner")
print("=" * 90)

# ------------------------------------------------------------
# 0. Verify 50B objects/functions are available
# ------------------------------------------------------------

required_names_50c = [
    "y50b",
    "pred44b_oof",
    "pred44b_test",
    "test_ids50b",
    "z_44b_oof",
    "z_44b_test",
    "tier_masks_oof",
    "tier_masks_test",
    "overlap_oof",
    "solver_only_oof",
    "none_oof",
    "fit_biases_50b",
    "fit_svd_50b",
    "predict_mf_50b",
    "entity_index_data",
    "asg_idx_train",
    "asg_idx_test",
    "n_items",
]

missing = [x for x in required_names_50c if x not in globals()]
if missing:
    raise ValueError(
        "Missing objects from 50B run. Run the 50B cell first or keep the same kernel. "
        f"Missing: {missing}"
    )

mse44b_50c = float(mean_squared_error(y50b, pred44b_oof))

has_50a_50c = "pred50a_oof" in globals() and pred50a_oof is not None
if has_50a_50c:
    mse50a_50c = float(mean_squared_error(y50b, pred50a_oof))
else:
    mse50a_50c = np.nan

has_50b_50c = Path("model_results/oof_mf50b_best_oof.csv").exists() and Path("model_results/testpred_mf50b_best_oof.csv").exists()

def load_oof_simple_50c(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)
    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        raise ValueError(path)
    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_simple_50c(path):
    df = pd.read_csv(path)
    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)
    return np.clip(pred, 0, 100).astype(np.float32)

if has_50b_50c:
    y50b_file, pred50b_oof = load_oof_simple_50c("model_results/oof_mf50b_best_oof.csv")
    pred50b_test = load_test_simple_50c("model_results/testpred_mf50b_best_oof.csv")
    assert np.max(np.abs(y50b_file - y50b)) < 1e-5
    mse50b_50c = float(mean_squared_error(y50b, pred50b_oof))
else:
    pred50b_oof = None
    pred50b_test = None
    mse50b_50c = np.nan

print("\nReferences")
print("----------")
print(f"44B OOF MSE: {mse44b_50c:.6f} | public 30.556")
if has_50a_50c:
    print(f"50A OOF MSE: {mse50a_50c:.6f} | public 29.742")
if has_50b_50c:
    print(f"50B OOF MSE: {mse50b_50c:.6f} | pending public")

# ------------------------------------------------------------
# 1. Load top 50B configs
# ------------------------------------------------------------

screen50b_path = "model_results/mf50b_screen.csv"
if not Path(screen50b_path).exists():
    raise ValueError("model_results/mf50b_screen.csv not found.")

screen50b_loaded = pd.read_csv(screen50b_path).sort_values("oof_mse").reset_index(drop=True)

# Pick top unique configs. These are all school/uniform/local-MF variants.
top_unique_cfg_rows = []
seen_cfg = set()

for _, row in screen50b_loaded.iterrows():
    cfg_name = row["config"]
    if cfg_name not in seen_cfg:
        seen_cfg.add(cfg_name)
        top_unique_cfg_rows.append(row)
    if len(top_unique_cfg_rows) >= 8:
        break

top_cfg_df = pd.DataFrame(top_unique_cfg_rows).reset_index(drop=True)

print("\nTop configs selected for bagging")
print("--------------------------------")
display(top_cfg_df[[
    "config", "rank", "reg_entity", "weight_scheme",
    "lambda_overlap", "lambda_solver_only", "lambda_none",
    "oof_mse", "gain_vs_50a"
]])

# Build cfg dicts from the saved manifest if available.
manifest_path = Path("model_results/mf50b_config_manifest.json")
if manifest_path.exists():
    with open(manifest_path, "r") as f:
        manifest_cfgs = json.load(f)
    cfg_lookup_50c = {cfg["name"]: cfg for cfg in manifest_cfgs}
else:
    # Fallback parse from screen rows.
    cfg_lookup_50c = {}
    for _, row in top_cfg_df.iterrows():
        cfg_lookup_50c[row["config"]] = {
            "name": row["config"],
            "entity_key": row["entity_key"],
            "rank": int(row["rank"]),
            "reg_entity": float(row["reg_entity"]),
            "reg_item": float(row["reg_item"]),
            "weight_scheme": row["weight_scheme"],
            "bias_iter": int(row["bias_iter"]),
            "bias_scale": float(row["bias_scale"]),
            "factor_scale": float(row["factor_scale"]),
        }

# ------------------------------------------------------------
# 2. Bagged training helper with custom CV seed
# ------------------------------------------------------------

def train_predict_config_oof_seed_50c(cfg, cv_seed):
    entity_data = entity_index_data[cfg["entity_key"]]
    entity_idx_train = entity_data["train_idx"]
    entity_idx_test = entity_data["test_idx"]
    n_entities = entity_data["n_entities"]

    item_idx_train = asg_idx_train
    item_idx_test = asg_idx_test

    oof = np.full(len(y50b), np.nan, dtype=np.float32)
    test_sum = np.zeros(len(pred44b_test), dtype=np.float64)

    folds = list(
        KFold(n_splits=N_SPLITS, shuffle=True, random_state=int(cv_seed))
        .split(np.arange(len(y50b)))
    )

    fold_rows = []

    for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
        global_bias, entity_bias, item_bias, debiased = fit_biases_50b(
            tr_idx,
            entity_idx_train,
            item_idx_train,
            n_entities,
            n_items,
            reg_entity=cfg["reg_entity"],
            reg_item=cfg["reg_item"],
            weight_scheme=cfg["weight_scheme"],
            n_iter=cfg["bias_iter"],
        )

        entity_factors, item_components = fit_svd_50b(
            tr_idx,
            debiased,
            entity_idx_train,
            item_idx_train,
            n_entities,
            n_items,
            rank=cfg["rank"],
            random_state=int(cv_seed) + 1000 * fold_num + int(cfg["rank"]),
        )

        pred_va = predict_mf_50b(
            base_z=z_44b_oof[va_idx],
            entity_idx=entity_idx_train[va_idx],
            item_idx=item_idx_train[va_idx],
            global_bias=global_bias,
            entity_bias=entity_bias,
            item_bias=item_bias,
            entity_factors=entity_factors,
            item_components=item_components,
            bias_scale=cfg["bias_scale"],
            factor_scale=cfg["factor_scale"],
        )

        pred_te = predict_mf_50b(
            base_z=z_44b_test,
            entity_idx=entity_idx_test,
            item_idx=item_idx_test,
            global_bias=global_bias,
            entity_bias=entity_bias,
            item_bias=item_bias,
            entity_factors=entity_factors,
            item_components=item_components,
            bias_scale=cfg["bias_scale"],
            factor_scale=cfg["factor_scale"],
        )

        oof[va_idx] = pred_va
        test_sum += pred_te.astype(np.float64)

        fold_rows.append({
            "config": cfg["name"],
            "cv_seed": int(cv_seed),
            "fold": fold_num,
            "mse_direct": float(mean_squared_error(y50b[va_idx], pred_va)),
            "mse_44b": float(mean_squared_error(y50b[va_idx], pred44b_oof[va_idx])),
        })

        del entity_factors, item_components, entity_bias, item_bias, debiased
        gc.collect()

    test_avg = (test_sum / N_SPLITS).astype(np.float32)
    return oof, test_avg, pd.DataFrame(fold_rows)

def make_tier_blend_from_direct_50c(direct_oof, direct_test, lams):
    pred_oof = pred44b_oof.copy().astype(np.float64)
    pred_test = pred44b_test.copy().astype(np.float64)

    for tier, mask in tier_masks_oof.items():
        lam = float(lams.get(tier, 0.0))
        pred_oof[mask] = pred44b_oof[mask] + lam * (direct_oof[mask] - pred44b_oof[mask])

    for tier, mask in tier_masks_test.items():
        lam = float(lams.get(tier, 0.0))
        pred_test[mask] = pred44b_test[mask] + lam * (direct_test[mask] - pred44b_test[mask])

    return (
        np.clip(pred_oof, 0, 100).astype(np.float32),
        np.clip(pred_test, 0, 100).astype(np.float32),
    )

# ------------------------------------------------------------
# 3. Run bagging
# ------------------------------------------------------------

# Increase this if you want more runtime. 10 seeds x 8 configs is already useful.
CV_SEEDS_50C = [
    RANDOM_STATE,
    RANDOM_STATE + 11,
    RANDOM_STATE + 23,
    RANDOM_STATE + 37,
    RANDOM_STATE + 53,
    RANDOM_STATE + 71,
    RANDOM_STATE + 97,
    RANDOM_STATE + 131,
    RANDOM_STATE + 173,
    RANDOM_STATE + 211,
]

bag_oof = {}
bag_test = {}
bag_rows = []
bag_fold_frames = []

t0 = time.time()

for cfg_idx, row in top_cfg_df.iterrows():
    cfg_name = row["config"]
    cfg = cfg_lookup_50c[cfg_name]

    # Use the best lambdas from that config's top row.
    lams = {
        "overlap": float(row["lambda_overlap"]),
        "solver_only": float(row["lambda_solver_only"]),
        "none": float(row["lambda_none"]),
    }

    for cv_seed in CV_SEEDS_50C:
        tag = f"{cfg_name}__seed{cv_seed}"
        oof_path = CHECKPOINT_DIR_50C / f"{tag}_oof.npy"
        test_path = CHECKPOINT_DIR_50C / f"{tag}_test.npy"
        fold_path = CHECKPOINT_DIR_50C / f"{tag}_folds.csv"

        print("\n" + "-" * 90)
        print("Bag config:", tag)
        print("-" * 90)

        if oof_path.exists() and test_path.exists() and fold_path.exists():
            direct_oof = np.load(oof_path).astype(np.float32)
            direct_test = np.load(test_path).astype(np.float32)
            fold_df = pd.read_csv(fold_path)
            print("Loaded checkpoint.")
        else:
            start = time.time()
            direct_oof, direct_test, fold_df = train_predict_config_oof_seed_50c(cfg, cv_seed)
            np.save(oof_path, direct_oof)
            np.save(test_path, direct_test)
            fold_df.to_csv(fold_path, index=False)
            print(f"Finished in {time.time() - start:.1f}s")

        blended_oof, blended_test = make_tier_blend_from_direct_50c(direct_oof, direct_test, lams)

        bag_oof[tag] = blended_oof
        bag_test[tag] = blended_test
        bag_fold_frames.append(fold_df)

        mse = float(mean_squared_error(y50b, blended_oof))
        row_out = {
            "tag": tag,
            "config": cfg_name,
            "cv_seed": int(cv_seed),
            "rank": int(cfg["rank"]),
            "reg_entity": float(cfg["reg_entity"]),
            "weight_scheme": cfg["weight_scheme"],
            "lambda_overlap": lams["overlap"],
            "lambda_solver_only": lams["solver_only"],
            "lambda_none": lams["none"],
            "oof_mse": mse,
            "gain_vs_44b": mse44b_50c - mse,
            "gain_vs_50a": mse50a_50c - mse if has_50a_50c else np.nan,
            "gain_vs_50b": mse50b_50c - mse if has_50b_50c else np.nan,
        }

        for tier, mask in tier_masks_oof.items():
            row_out[f"mse_{tier}"] = float(mean_squared_error(y50b[mask], blended_oof[mask]))

        bag_rows.append(row_out)
        pd.DataFrame(bag_rows).to_csv("model_results/mf50c_bag_progress.csv", index=False)
        print(f"OOF MSE: {mse:.6f}")

print("\nBagging elapsed seconds:", round(time.time() - t0, 1))

bag50c = pd.DataFrame(bag_rows).sort_values("oof_mse").reset_index(drop=True)
bag_fold50c = pd.concat(bag_fold_frames, axis=0).reset_index(drop=True) if bag_fold_frames else pd.DataFrame()

# ------------------------------------------------------------
# 4. Build ensemble candidates
# ------------------------------------------------------------

candidate_oof = {}
candidate_test = {}

# Core candidates.
candidate_oof["44b"] = pred44b_oof
candidate_test["44b"] = pred44b_test

if has_50a_50c:
    candidate_oof["50a"] = pred50a_oof
    candidate_test["50a"] = pred50a_test

if has_50b_50c:
    candidate_oof["50b_best_oof"] = pred50b_oof
    candidate_test["50b_best_oof"] = pred50b_test

# Bag candidates.
for tag in bag_oof:
    candidate_oof[tag] = bag_oof[tag]
    candidate_test[tag] = bag_test[tag]

# Simple averages over top bag candidates.
avg_rows = []
avg_oof = {}
avg_test = {}

for k in [3, 5, 10, 20, 40]:
    tags = bag50c.head(min(k, len(bag50c)))["tag"].tolist()
    if not tags:
        continue

    po = np.mean([candidate_oof[t].astype(np.float64) for t in tags], axis=0)
    pt = np.mean([candidate_test[t].astype(np.float64) for t in tags], axis=0)

    po = np.clip(po, 0, 100).astype(np.float32)
    pt = np.clip(pt, 0, 100).astype(np.float32)

    name = f"avg_top{k}_bags"
    candidate_oof[name] = po
    candidate_test[name] = pt
    avg_oof[name] = po
    avg_test[name] = pt

    mse = float(mean_squared_error(y50b, po))
    avg_rows.append({
        "name": name,
        "n_bags": len(tags),
        "oof_mse": mse,
        "gain_vs_44b": mse44b_50c - mse,
        "gain_vs_50a": mse50a_50c - mse if has_50a_50c else np.nan,
        "gain_vs_50b": mse50b_50c - mse if has_50b_50c else np.nan,
    })

avg50c = pd.DataFrame(avg_rows).sort_values("oof_mse").reset_index(drop=True)

# ------------------------------------------------------------
# 5. Greedy ensemble on top candidates
# ------------------------------------------------------------

# Use a limited candidate pool for stability.
candidate_screen_rows = []
for name, pred in candidate_oof.items():
    candidate_screen_rows.append({
        "name": name,
        "oof_mse": float(mean_squared_error(y50b, pred)),
        "gain_vs_44b": mse44b_50c - float(mean_squared_error(y50b, pred)),
        "gain_vs_50a": mse50a_50c - float(mean_squared_error(y50b, pred)) if has_50a_50c else np.nan,
        "gain_vs_50b": mse50b_50c - float(mean_squared_error(y50b, pred)) if has_50b_50c else np.nan,
    })

candidate_screen50c = pd.DataFrame(candidate_screen_rows).sort_values("oof_mse").reset_index(drop=True)

pool_names = candidate_screen50c.head(min(60, len(candidate_screen50c)))["name"].tolist()

# Start from best single OOF candidate.
start_name = candidate_screen50c.iloc[0]["name"]
current_oof = candidate_oof[start_name].copy().astype(np.float32)
current_test = candidate_test[start_name].copy().astype(np.float32)
current_mse = float(mean_squared_error(y50b, current_oof))

greedy_rows = [{
    "iteration": 0,
    "chosen": start_name,
    "alpha": 1.0,
    "oof_mse": current_mse,
    "gain_vs_44b": mse44b_50c - current_mse,
    "gain_vs_50a": mse50a_50c - current_mse if has_50a_50c else np.nan,
    "gain_vs_50b": mse50b_50c - current_mse if has_50b_50c else np.nan,
}]

alpha_grid = np.array([0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25], dtype=np.float64)

for it in range(1, 31):
    best = None

    for name in pool_names:
        cand_o = candidate_oof[name].astype(np.float64)

        for alpha in alpha_grid:
            trial = (1.0 - alpha) * current_oof.astype(np.float64) + alpha * cand_o
            trial = np.clip(trial, 0, 100).astype(np.float32)
            mse = float(mean_squared_error(y50b, trial))

            if best is None or mse < best["mse"]:
                best = {
                    "name": name,
                    "alpha": float(alpha),
                    "mse": mse,
                    "trial": trial,
                }

    if best is None or best["mse"] >= current_mse - 1e-5:
        break

    current_oof = best["trial"]
    current_test = (
        (1.0 - best["alpha"]) * current_test.astype(np.float64)
        + best["alpha"] * candidate_test[best["name"]].astype(np.float64)
    )
    current_test = np.clip(current_test, 0, 100).astype(np.float32)
    current_mse = best["mse"]

    greedy_rows.append({
        "iteration": it,
        "chosen": best["name"],
        "alpha": best["alpha"],
        "oof_mse": current_mse,
        "gain_vs_44b": mse44b_50c - current_mse,
        "gain_vs_50a": mse50a_50c - current_mse if has_50a_50c else np.nan,
        "gain_vs_50b": mse50b_50c - current_mse if has_50b_50c else np.nan,
    })

greedy50c = pd.DataFrame(greedy_rows)

# ------------------------------------------------------------
# 6. Fold diagnostics
# ------------------------------------------------------------

folds_diag = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(len(y50b))))

fold_diag_rows = []

for fold_num, (_, va_idx) in enumerate(folds_diag, start=1):
    row = {
        "fold": fold_num,
        "mse_44b": float(mean_squared_error(y50b[va_idx], pred44b_oof[va_idx])),
        "mse_50c": float(mean_squared_error(y50b[va_idx], current_oof[va_idx])),
    }
    row["gain_vs_44b"] = row["mse_44b"] - row["mse_50c"]

    if has_50a_50c:
        row["mse_50a"] = float(mean_squared_error(y50b[va_idx], pred50a_oof[va_idx]))
        row["gain_vs_50a"] = row["mse_50a"] - row["mse_50c"]

    if has_50b_50c:
        row["mse_50b"] = float(mean_squared_error(y50b[va_idx], pred50b_oof[va_idx]))
        row["gain_vs_50b"] = row["mse_50b"] - row["mse_50c"]

    for tier, mask_full in tier_masks_oof.items():
        mask = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse50c_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], current_oof[va_idx][mask]))
            row[f"mse44b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred44b_oof[va_idx][mask]))
            row[f"gain_{tier}_vs_44b"] = row[f"mse44b_{tier}"] - row[f"mse50c_{tier}"]
            if has_50a_50c:
                row[f"mse50a_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred50a_oof[va_idx][mask]))
                row[f"gain_{tier}_vs_50a"] = row[f"mse50a_{tier}"] - row[f"mse50c_{tier}"]
            if has_50b_50c:
                row[f"mse50b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred50b_oof[va_idx][mask]))
                row[f"gain_{tier}_vs_50b"] = row[f"mse50b_{tier}"] - row[f"mse50c_{tier}"]

    fold_diag_rows.append(row)

fold_diag50c = pd.DataFrame(fold_diag_rows)

# ------------------------------------------------------------
# 7. Save artifacts
# ------------------------------------------------------------

bag_path = "model_results/mf50c_bag_screen.csv"
avg_path = "model_results/mf50c_avg_screen.csv"
candidate_screen_path = "model_results/mf50c_candidate_screen.csv"
greedy_path = "model_results/mf50c_greedy_path.csv"
fold_diag_path = "model_results/mf50c_fold_diag.csv"
oof_path = "model_results/oof_mf50c_greedy_best.csv"
test_path = "model_results/testpred_mf50c_greedy_best.csv"
submission_path = "submission_mf50c_greedy_best.csv"

bag50c.to_csv(bag_path, index=False)
avg50c.to_csv(avg_path, index=False)
candidate_screen50c.to_csv(candidate_screen_path, index=False)
greedy50c.to_csv(greedy_path, index=False)
fold_diag50c.to_csv(fold_diag_path, index=False)

pd.DataFrame({
    "row_index": np.arange(len(y50b)),
    TARGET_COL: y50b,
    "pred_44b": pred44b_oof,
    "pred_50a": pred50a_oof if has_50a_50c else np.nan,
    "pred_50b": pred50b_oof if has_50b_50c else np.nan,
    "pred_clipped": current_oof,
    "tier_overlap": overlap_oof.astype(int),
    "tier_solver_only": solver_only_oof.astype(int),
    "tier_none": none_oof.astype(int),
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids50b,
    "pred_44b": pred44b_test,
    "pred_50a": pred50a_test if has_50a_50c else np.nan,
    "pred_50b": pred50b_test if has_50b_50c else np.nan,
    TARGET_COL: current_test,
    "tier_overlap": tier_masks_test["overlap"].astype(int),
    "tier_solver_only": tier_masks_test["solver_only"].astype(int),
    "tier_none": tier_masks_test["none"].astype(int),
}).to_csv(test_path, index=False)

pd.DataFrame({
    ID_COL: test_ids50b,
    TARGET_COL: current_test,
}).to_csv(submission_path, index=False)

# Save simple average submissions too.
avg_submission_paths = []
for name, pred_test in avg_test.items():
    p = f"submission_mf50c_{name}.csv"
    pd.DataFrame({
        ID_COL: test_ids50b,
        TARGET_COL: pred_test,
    }).to_csv(p, index=False)
    avg_submission_paths.append(p)

# Validate.
for p in [submission_path] + avg_submission_paths:
    sub = pd.read_csv(p)
    assert sub.shape == (len(test_ids50b), 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[ID_COL].notna().all(), p
    assert sub[TARGET_COL].notna().all(), p
    assert np.isfinite(sub[TARGET_COL]).all(), p
    assert sub[TARGET_COL].between(0, 100).all(), p

# ------------------------------------------------------------
# 8. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("50C bagged MF ensemble complete")
print("=" * 90)

print("\nReferences")
print("----------")
print(f"44B OOF MSE: {mse44b_50c:.6f} | public 30.556")
if has_50a_50c:
    print(f"50A OOF MSE: {mse50a_50c:.6f} | public 29.742")
if has_50b_50c:
    print(f"50B OOF MSE: {mse50b_50c:.6f} | pending public")

print("\nTop 20 bagged single candidates")
print("-------------------------------")
display(bag50c.head(20))

print("\nSimple average candidates")
print("-------------------------")
display(avg50c)

print("\nTop 20 candidate pool")
print("---------------------")
display(candidate_screen50c.head(20))

print("\nGreedy path")
print("-----------")
display(greedy50c)

print("\nFold diagnostics")
print("----------------")
print(fold_diag50c.to_string(index=False))
print("Min fold gain vs 44B:", float(fold_diag50c["gain_vs_44b"].min()))
if has_50a_50c:
    print("Min fold gain vs 50A:", float(fold_diag50c["gain_vs_50a"].min()))
if has_50b_50c:
    print("Min fold gain vs 50B:", float(fold_diag50c["gain_vs_50b"].min()))

print("\nSaved files")
print("-----------")
print(bag_path)
print(avg_path)
print(candidate_screen_path)
print(greedy_path)
print(fold_diag_path)
print(oof_path)
print(test_path)
print(submission_path)
for p in avg_submission_paths:
    print(p)

print("\nSubmission validation")
print("---------------------")
sub = pd.read_csv(submission_path)
print(sub.shape)
print(sub[TARGET_COL].describe())

print("\nDecision guidance")
print("-----------------")
print("Do not submit before the 50B public result is known.")
print("If 50B improves public, compare 50C OOF/fold stability vs 50B and consider mf50c_greedy_best next.")

50C. Repeated-CV bagged MF ensemble around 50B winner

References
----------
44B OOF MSE: 52.780037 | public 30.556
50A OOF MSE: 52.247093 | public 29.742
50B OOF MSE: 51.463158 | pending public

Top configs selected for bagging
--------------------------------


,config,rank,reg_entity,weight_scheme,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50a
0,mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0,16,500.0,uniform,0.0050,0.5475,0.8250,51.463158,0.783936
1,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,20,500.0,uniform,0.0025,0.4975,0.8200,51.475582,0.771511
2,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0,14,500.0,uniform,0.0050,0.5775,0.8275,51.483036,0.764057
3,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,12,500.0,uniform,0.0050,0.6150,0.8375,51.484856,0.762238
4,mf50b_school_rank16_reg300p0_uniform_b1p0_f1p0,16,300.0,uniform,0.0050,0.5400,0.8075,51.505215,0.741879
5,mf50b_school_rank20_reg300p0_uniform_b1p0_f1p0,20,300.0,uniform,0.0025,0.4950,0.8025,51.512085,0.735008
6,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,10,500.0,uniform,0.0050,0.6275,0.8450,51.536339,0.710754
7,mf50b_school_rank14_reg300p0_uniform_b1p0_f1p0,14,300.0,uniform,0.0050,0.5650,0.8050,51.539810,0.707283



------------------------------------------------------------------------------------------
Bag config: mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0__seed9890
------------------------------------------------------------------------------------------
Finished in 5.4s
OOF MSE: 51.463158

------------------------------------------------------------------------------------------
Bag config: mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0__seed9901
------------------------------------------------------------------------------------------
Finished in 3.0s
OOF MSE: 50.650623

------------------------------------------------------------------------------------------
Bag config: mf50b_school_rank16_reg500p0_uniform_b1p0_f1p0__seed9913
------------------------------------------------------------------------------------------
Finished in 2.8s
OOF MSE: 50.670719

------------------------------------------------------------------------------------------
Bag config: mf50b_school_rank16_reg500p0_uni

,tag,config,cv_seed,rank,reg_entity,weight_scheme,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_44b,gain_vs_50a,gain_vs_50b,mse_overlap,mse_solver_only,mse_none
0,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,10101,20,500.0,uniform,0.0025,0.4975,0.8200,50.529709,2.250328,1.717384,0.933449,0.507017,61.277107,88.296059
1,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,9913,20,500.0,uniform,0.0025,0.4975,0.8200,50.540794,2.239243,1.706299,0.922363,0.506988,61.316990,88.303940
2,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,9943,20,500.0,uniform,0.0025,0.4975,0.8200,50.542572,2.237465,1.704521,0.920586,0.507031,61.124691,88.392418
3,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,9927,20,500.0,uniform,0.0025,0.4975,0.8200,50.544636,2.235401,1.702457,0.918522,0.507032,61.248470,88.342789
4,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,10021,20,500.0,uniform,0.0025,0.4975,0.8200,50.552036,2.228001,1.695057,0.911121,0.507004,61.161461,88.397942
5,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,9901,20,500.0,uniform,0.0025,0.4975,0.8200,50.561131,2.218906,1.685963,0.902027,0.506999,61.153355,88.422325
6,mf50b_school_rank20_reg300p0_uniform_b1p0_f1p0...,mf50b_school_rank20_reg300p0_uniform_b1p0_f1p0,10101,20,300.0,uniform,0.0025,0.4950,0.8025,50.565742,2.214294,1.681351,0.897415,0.507016,61.279434,88.377510
7,mf50b_school_rank20_reg300p0_uniform_b1p0_f1p0...,mf50b_school_rank20_reg300p0_uniform_b1p0_f1p0,9943,20,300.0,uniform,0.0025,0.4950,0.8025,50.577644,2.202393,1.669449,0.885513,0.507027,61.122402,88.473686
8,mf50b_school_rank20_reg300p0_uniform_b1p0_f1p0...,mf50b_school_rank20_reg300p0_uniform_b1p0_f1p0,9913,20,300.0,uniform,0.0025,0.4950,0.8025,50.579205,2.200832,1.667889,0.883953,0.506988,61.307064,88.396210
9,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0,9987,20,500.0,uniform,0.0025,0.4975,0.8200,50.581795,2.198242,1.665298,0.881363,0.506999,61.409649,88.357079



Simple average candidates
-------------------------


...


Top 20 candidate pool
---------------------


,name,oof_mse,gain_vs_44b,gain_vs_50a,gain_vs_50b
0,avg_top5_bags,50.178162,2.601875,2.068932,1.284996
1,avg_top10_bags,50.179420,2.600616,2.067673,1.283737
2,avg_top20_bags,50.184086,2.595951,2.063007,1.279072
3,avg_top40_bags,50.231853,2.548183,2.015240,1.231304
4,avg_top3_bags,50.237537,2.542500,2.009556,1.225620
5,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,50.529709,2.250328,1.717384,0.933449
6,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,50.540794,2.239243,1.706299,0.922363
7,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,50.542572,2.237465,1.704521,0.920586
8,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,50.544636,2.235401,1.702457,0.918522
9,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,50.552036,2.228001,1.695057,0.911121



Greedy path
-----------


,iteration,chosen,alpha,oof_mse,gain_vs_44b,gain_vs_50a,gain_vs_50b
0,0,avg_top5_bags,1.000,50.178162,2.601875,2.068932,1.284996
1,1,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,0.150,50.165878,2.614159,2.081215,1.297279
2,2,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,0.100,50.158710,2.621326,2.088383,1.304447
3,3,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,0.050,50.157501,2.622536,2.089592,1.305656
4,4,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...,0.030,50.156528,2.623508,2.090565,1.306629
5,5,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,0.030,50.155987,2.624050,2.091106,1.307171
6,6,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...,0.020,50.155781,2.624256,2.091312,1.307377
7,7,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,0.020,50.155628,2.624409,2.091465,1.307529
8,8,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,0.020,50.155472,2.624565,2.091621,1.307686
9,9,mf50b_school_rank20_reg500p0_uniform_b1p0_f1p0...,0.010,50.155418,2.624619,2.091675,1.307739



Fold diagnostics
----------------
 fold   mse_44b   mse_50c  gain_vs_44b   mse_50a  gain_vs_50a   mse_50b  gain_vs_50b  n_overlap  mse50c_overlap  mse44b_overlap  gain_overlap_vs_44b  mse50a_overlap  gain_overlap_vs_50a  mse50b_overlap  gain_overlap_vs_50b  n_solver_only  mse50c_solver_only  mse44b_solver_only  gain_solver_only_vs_44b  mse50a_solver_only  gain_solver_only_vs_50a  mse50b_solver_only  gain_solver_only_vs_50b  n_none  mse50c_none  mse44b_none  gain_none_vs_44b  mse50a_none  gain_none_vs_50a  mse50b_none  gain_none_vs_50b
    1 53.054913 50.722122     2.332790 52.599960     1.877838 51.927963     1.205841      10837        0.418915        0.418953             0.000038        0.418754            -0.000161        0.418636            -0.000279           5496           59.282471           62.401917                 3.119446           62.097149                 2.814678           61.760151                 2.477680   12652    90.090462    94.079628          3.989166    93.169922 

In [59]:
# ============================================================
# 51A. Second-pass MF residual on top of 50B
# ============================================================
#
# Purpose:
#   Test whether residual low-rank structure remains after 50B.
#
# Base:
#   pred50b_oof / pred50b_test from model_results/oof_mf50b_best_oof.csv
#
# Model:
#   residual_z = arcsin(sqrt(y/100)) - arcsin(sqrt(pred50b/100))
#
# Then same school × assessment-subgroup MF residual model.
#
# This is intentionally smaller than 50B because it is a second pass.
# ============================================================

import os
import gc
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

Path("model_results").mkdir(exist_ok=True)

print("=" * 90)
print("51A. Second-pass MF residual on top of 50B")
print("=" * 90)

if not Path("model_results/oof_mf50b_best_oof.csv").exists():
    raise ValueError("Need model_results/oof_mf50b_best_oof.csv from 50B.")

# Load 50B base.
y51a_file, pred50b_base_oof = load_oof_simple_50c("model_results/oof_mf50b_best_oof.csv")
pred50b_base_test = load_test_simple_50c("model_results/testpred_mf50b_best_oof.csv")
assert np.max(np.abs(y51a_file - y50b)) < 1e-5

mse50b_base = float(mean_squared_error(y50b, pred50b_base_oof))

z_base_oof_51a = pct_to_arc_50b(pred50b_base_oof)
z_base_test_51a = pct_to_arc_50b(pred50b_base_test)
resid_z_original_50b = resid_z.copy()
z_44b_oof_original_50b = z_44b_oof.copy()
z_44b_test_original_50b = z_44b_test.copy()

# Temporarily redirect the global residual/base used by 50B helper functions.
resid_z = (z_y - z_base_oof_51a).astype(np.float32)
z_44b_oof = z_base_oof_51a
z_44b_test = z_base_test_51a

configs51a = []

for rank in [2, 4, 6, 8, 10, 12]:
    for reg in [100.0, 200.0, 300.0, 500.0, 800.0, 1200.0]:
        cfg = {
            "name": f"mf51a_school_rank{rank}_reg{str(reg).replace('.', 'p')}_uniform",
            "entity_key": "school",
            "rank": rank,
            "reg_entity": reg,
            "reg_item": reg,
            "weight_scheme": "uniform",
            "bias_iter": 4,
            "bias_scale": 1.0,
            "factor_scale": 1.0,
        }
        configs51a.append(cfg)

print("\nNumber of 51A configs:", len(configs51a))

CHECKPOINT_DIR_51A = Path("model_results/mf51a_checkpoints")
CHECKPOINT_DIR_51A.mkdir(exist_ok=True)

cand_oof_51a = {}
cand_test_51a = {}
direct_rows_51a = []
fold_frames_51a = []

for cfg_num, cfg in enumerate(configs51a, start=1):
    name = cfg["name"]
    print("\n" + "-" * 90)
    print(f"51A config {cfg_num}/{len(configs51a)}: {name}")
    print("-" * 90)

    oof_path = CHECKPOINT_DIR_51A / f"{name}_oof.npy"
    test_path = CHECKPOINT_DIR_51A / f"{name}_test.npy"
    fold_path = CHECKPOINT_DIR_51A / f"{name}_folds.csv"

    if oof_path.exists() and test_path.exists() and fold_path.exists():
        p_oof = np.load(oof_path).astype(np.float32)
        p_test = np.load(test_path).astype(np.float32)
        fd = pd.read_csv(fold_path)
        print("Loaded checkpoint.")
    else:
        start = time.time()
        p_oof, p_test, fd = train_predict_config_oof_50b(cfg)
        np.save(oof_path, p_oof)
        np.save(test_path, p_test)
        fd.to_csv(fold_path, index=False)
        print(f"Finished in {time.time() - start:.1f}s")

    cand_oof_51a[name] = p_oof
    cand_test_51a[name] = p_test
    fold_frames_51a.append(fd)

    mse = float(mean_squared_error(y50b, p_oof))
    direct_rows_51a.append({
        "config": name,
        "rank": cfg["rank"],
        "reg": cfg["reg_entity"],
        "direct_oof_mse": mse,
        "gain_vs_50b_base": mse50b_base - mse,
    })
    print(f"Direct OOF MSE: {mse:.6f} | gain vs 50B base: {mse50b_base - mse:.6f}")

direct51a = pd.DataFrame(direct_rows_51a).sort_values("direct_oof_mse").reset_index(drop=True)
fold_metrics51a = pd.concat(fold_frames_51a, axis=0).reset_index(drop=True)

# Restore original globals.
resid_z = resid_z_original_50b
z_44b_oof = z_44b_oof_original_50b
z_44b_test = z_44b_test_original_50b

# Blend scan around 50B base.
lambda_grid_51a = np.unique(
    np.concatenate([
        np.linspace(-0.25, 1.25, 601),
        np.array([0.0, 0.005, 0.05, 0.10, 0.155, 0.25, 0.50, 0.75, 1.0])
    ])
)

def best_lambda_mask_51a(candidate_pred, mask):
    if int(mask.sum()) == 0:
        return 0.0
    y = y50b[mask].astype(np.float64)
    base = pred50b_base_oof[mask].astype(np.float64)
    alt = candidate_pred[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf
    for lam in lambda_grid_51a:
        p = np.clip(base + float(lam) * (alt - base), 0, 100)
        mse = float(mean_squared_error(y, p))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)
    return best_lam

def make_blend_51a(candidate_oof, candidate_test, lams, is_test=False):
    if is_test:
        base = pred50b_base_test.astype(np.float64)
        alt = candidate_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred50b_base_oof.astype(np.float64)
        alt = candidate_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()
    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

screen_rows_51a = []

for cfg in configs51a:
    name = cfg["name"]
    p_oof = cand_oof_51a[name]
    p_test = cand_test_51a[name]

    lam_overlap = best_lambda_mask_51a(p_oof, overlap_oof)
    lam_solver = best_lambda_mask_51a(p_oof, solver_only_oof)
    lam_none = best_lambda_mask_51a(p_oof, none_oof)

    strategies = {
        "solver_only": {"overlap": 0.0, "solver_only": lam_solver, "none": 0.0},
        "none_only": {"overlap": 0.0, "solver_only": 0.0, "none": lam_none},
        "solver_plus_none": {"overlap": 0.0, "solver_only": lam_solver, "none": lam_none},
        "all_tiers": {"overlap": lam_overlap, "solver_only": lam_solver, "none": lam_none},
    }

    for strategy, lams in strategies.items():
        pred = make_blend_51a(p_oof, p_test, lams, is_test=False)
        mse = float(mean_squared_error(y50b, pred))

        row = {
            "config": name,
            "strategy": strategy,
            "rank": cfg["rank"],
            "reg": cfg["reg_entity"],
            "lambda_overlap": lams["overlap"],
            "lambda_solver_only": lams["solver_only"],
            "lambda_none": lams["none"],
            "oof_mse": mse,
            "gain_vs_50b_base": mse50b_base - mse,
            "gain_vs_50a": mse50a_50c - mse if has_50a_50c else np.nan,
            "gain_vs_44b": mse44b_50c - mse,
        }

        for tier, mask in tier_masks_oof.items():
            row[f"mse_{tier}"] = float(mean_squared_error(y50b[mask], pred[mask]))

        screen_rows_51a.append(row)

screen51a = pd.DataFrame(screen_rows_51a).sort_values("oof_mse").reset_index(drop=True)

best51a = screen51a.iloc[0]
best_name = best51a["config"]
best_lams = {
    "overlap": float(best51a["lambda_overlap"]),
    "solver_only": float(best51a["lambda_solver_only"]),
    "none": float(best51a["lambda_none"]),
}

final_oof51a = make_blend_51a(cand_oof_51a[best_name], cand_test_51a[best_name], best_lams, is_test=False)
final_test51a = make_blend_51a(cand_oof_51a[best_name], cand_test_51a[best_name], best_lams, is_test=True)

# Fold diagnostics.
folds_diag = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(len(y50b))))
fold_rows_51a = []

for fold_num, (_, va_idx) in enumerate(folds_diag, start=1):
    row = {
        "fold": fold_num,
        "mse_50b_base": float(mean_squared_error(y50b[va_idx], pred50b_base_oof[va_idx])),
        "mse_51a": float(mean_squared_error(y50b[va_idx], final_oof51a[va_idx])),
    }
    row["gain_vs_50b_base"] = row["mse_50b_base"] - row["mse_51a"]

    if has_50a_50c:
        row["mse_50a"] = float(mean_squared_error(y50b[va_idx], pred50a_oof[va_idx]))
        row["gain_vs_50a"] = row["mse_50a"] - row["mse_51a"]

    for tier, mask_full in tier_masks_oof.items():
        mask = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse51a_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], final_oof51a[va_idx][mask]))
            row[f"mse50b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred50b_base_oof[va_idx][mask]))
            row[f"gain_{tier}_vs_50b"] = row[f"mse50b_{tier}"] - row[f"mse51a_{tier}"]

    fold_rows_51a.append(row)

fold51a = pd.DataFrame(fold_rows_51a)

# Save.
direct51a.to_csv("model_results/mf51a_direct_screen.csv", index=False)
screen51a.to_csv("model_results/mf51a_screen.csv", index=False)
fold_metrics51a.to_csv("model_results/mf51a_fold_metrics.csv", index=False)
fold51a.to_csv("model_results/mf51a_best_fold_diag.csv", index=False)

pd.DataFrame({
    "row_index": np.arange(len(y50b)),
    TARGET_COL: y50b,
    "pred_50b_base": pred50b_base_oof,
    "pred_clipped": final_oof51a,
    "tier_overlap": overlap_oof.astype(int),
    "tier_solver_only": solver_only_oof.astype(int),
    "tier_none": none_oof.astype(int),
}).to_csv("model_results/oof_mf51a_best.csv", index=False)

pd.DataFrame({
    ID_COL: test_ids50b,
    "pred_50b_base": pred50b_base_test,
    TARGET_COL: final_test51a,
    "tier_overlap": tier_masks_test["overlap"].astype(int),
    "tier_solver_only": tier_masks_test["solver_only"].astype(int),
    "tier_none": tier_masks_test["none"].astype(int),
}).to_csv("model_results/testpred_mf51a_best.csv", index=False)

pd.DataFrame({
    ID_COL: test_ids50b,
    TARGET_COL: final_test51a,
}).to_csv("submission_mf51a_best.csv", index=False)

sub = pd.read_csv("submission_mf51a_best.csv")
assert sub.shape == (len(test_ids50b), 2)
assert list(sub.columns) == [ID_COL, TARGET_COL]
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

print("\n" + "=" * 90)
print("51A second-pass MF complete")
print("=" * 90)

print("\n50B base OOF MSE:", mse50b_base)

print("\nTop 20 direct second-pass configs")
print("--------------------------------")
display(direct51a.head(20))

print("\nTop 20 blended second-pass candidates")
print("------------------------------------")
display(screen51a.head(20))

print("\nBest 51A")
print("--------")
print(best51a.to_string())

print("\nFold diagnostics")
print("----------------")
print(fold51a.to_string(index=False))
print("Min fold gain vs 50B base:", float(fold51a["gain_vs_50b_base"].min()))

print("\nSaved submission")
print("----------------")
print("submission_mf51a_best.csv")
print(sub[TARGET_COL].describe())

51A. Second-pass MF residual on top of 50B

Number of 51A configs: 36

------------------------------------------------------------------------------------------
51A config 1/36: mf51a_school_rank2_reg100p0_uniform
------------------------------------------------------------------------------------------
Finished in 2.6s
Direct OOF MSE: 52.778824 | gain vs 50B base: -1.315666

------------------------------------------------------------------------------------------
51A config 2/36: mf51a_school_rank2_reg200p0_uniform
------------------------------------------------------------------------------------------
Finished in 2.5s
Direct OOF MSE: 52.491909 | gain vs 50B base: -1.028751

------------------------------------------------------------------------------------------
51A config 3/36: mf51a_school_rank2_reg300p0_uniform
------------------------------------------------------------------------------------------
Finished in 2.5s
Direct OOF MSE: 52.377686 | gain vs 50B base: -0.914528

--

,config,rank,reg,direct_oof_mse,gain_vs_50b_base
0,mf51a_school_rank2_reg1200p0_uniform,2,1200.0,52.142902,-0.679745
1,mf51a_school_rank2_reg800p0_uniform,2,800.0,52.193180,-0.730022
2,mf51a_school_rank2_reg500p0_uniform,2,500.0,52.268829,-0.805672
3,mf51a_school_rank2_reg300p0_uniform,2,300.0,52.377686,-0.914528
4,mf51a_school_rank2_reg200p0_uniform,2,200.0,52.491909,-1.028751
5,mf51a_school_rank4_reg1200p0_uniform,4,1200.0,52.651501,-1.188343
6,mf51a_school_rank4_reg800p0_uniform,4,800.0,52.697227,-1.234070
7,mf51a_school_rank4_reg500p0_uniform,4,500.0,52.768257,-1.305099
8,mf51a_school_rank2_reg100p0_uniform,2,100.0,52.778824,-1.315666
9,mf51a_school_rank4_reg300p0_uniform,4,300.0,52.884033,-1.420876



Top 20 blended second-pass candidates
------------------------------------


,config,strategy,rank,reg,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b_base,gain_vs_50a,gain_vs_44b,mse_overlap,mse_solver_only,mse_none
0,mf51a_school_rank12_reg100p0_uniform,all_tiers,12,100.0,-0.005,-0.25,-0.25,51.201221,0.261936,1.045872,1.578815,0.506963,63.257629,88.963173
1,mf51a_school_rank12_reg100p0_uniform,solver_plus_none,12,100.0,0.000,-0.25,-0.25,51.201229,0.261929,1.045864,1.578808,0.506983,63.257629,88.963173
2,mf51a_school_rank10_reg100p0_uniform,all_tiers,10,100.0,-0.005,-0.25,-0.25,51.229382,0.233776,1.017712,1.550655,0.506950,63.289829,89.013489
3,mf51a_school_rank10_reg100p0_uniform,solver_plus_none,10,100.0,0.000,-0.25,-0.25,51.229393,0.233765,1.017700,1.550644,0.506983,63.289829,89.013489
4,mf51a_school_rank12_reg200p0_uniform,all_tiers,12,200.0,-0.005,-0.25,-0.25,51.239479,0.223679,1.007614,1.540558,0.506953,63.281990,89.040039
5,mf51a_school_rank12_reg200p0_uniform,solver_plus_none,12,200.0,0.000,-0.25,-0.25,51.239491,0.223667,1.007603,1.540546,0.506983,63.281990,89.040039
6,mf51a_school_rank8_reg100p0_uniform,all_tiers,8,100.0,-0.005,-0.25,-0.25,51.243511,0.219646,1.003582,1.536526,0.506965,63.292957,89.044434
7,mf51a_school_rank8_reg100p0_uniform,solver_plus_none,8,100.0,0.000,-0.25,-0.25,51.243519,0.219639,1.003574,1.536518,0.506983,63.292957,89.044434
8,mf51a_school_rank12_reg100p0_uniform,none_only,12,100.0,0.000,0.00,-0.25,51.244755,0.218403,1.002338,1.535282,0.506983,63.484444,88.963173
9,mf51a_school_rank12_reg300p0_uniform,all_tiers,12,300.0,-0.005,-0.25,-0.25,51.253311,0.209846,0.993782,1.526726,0.506949,63.292248,89.067184



Best 51A
--------
config                mf51a_school_rank12_reg100p0_uniform
strategy                                         all_tiers
rank                                                    12
reg                                                  100.0
lambda_overlap                                      -0.005
lambda_solver_only                                   -0.25
lambda_none                                          -0.25
oof_mse                                          51.201221
gain_vs_50b_base                                  0.261936
gain_vs_50a                                       1.045872
gain_vs_44b                                       1.578815
mse_overlap                                       0.506963
mse_solver_only                                  63.257629
mse_none                                         88.963173

Fold diagnostics
----------------
 fold  mse_50b_base   mse_51a  gain_vs_50b_base   mse_50a  gain_vs_50a  n_overlap  mse51a_overlap  mse50b_overlap  gain_

In [60]:
# ============================================================
# 51B. Wider anti-signal lambda search for second-pass MF
# ============================================================
#
# Motivation:
#   51A best hit lambda_solver_only = -0.25 and lambda_none = -0.25,
#   meaning the optimum may be farther negative.
#
# This cell requires the same kernel state after 51A:
#   cand_oof_51a, cand_test_51a, pred50b_base_oof, pred50b_base_test, etc.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

print("=" * 90)
print("51B. Wider anti-signal lambda search for second-pass MF")
print("=" * 90)

required_51b = [
    "cand_oof_51a", "cand_test_51a",
    "pred50b_base_oof", "pred50b_base_test",
    "y50b", "test_ids50b",
    "tier_masks_oof", "tier_masks_test",
    "overlap_oof", "solver_only_oof", "none_oof",
]

missing_51b = [x for x in required_51b if x not in globals()]
if missing_51b:
    raise ValueError(f"Missing 51A objects from memory: {missing_51b}")

mse50b_base = float(mean_squared_error(y50b, pred50b_base_oof))

lambda_grid_51b = np.unique(
    np.concatenate([
        np.linspace(-2.0, 1.0, 1201),
        np.array([-2.0, -1.5, -1.0, -0.75, -0.50, -0.25, -0.10, 0.0, 0.10, 0.25, 0.50, 1.0])
    ])
)

def best_lambda_mask_51b(candidate_pred, mask):
    if int(mask.sum()) == 0:
        return 0.0

    y = y50b[mask].astype(np.float64)
    base = pred50b_base_oof[mask].astype(np.float64)
    alt = candidate_pred[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid_51b:
        pred = np.clip(base + float(lam) * (alt - base), 0, 100)
        mse = float(mean_squared_error(y, pred))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam

def make_blend_51b(candidate_oof, candidate_test, lams, is_test=False):
    if is_test:
        base = pred50b_base_test.astype(np.float64)
        alt = candidate_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred50b_base_oof.astype(np.float64)
        alt = candidate_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

screen_rows_51b = []
store_51b = {}

for name in cand_oof_51a:
    p_oof = cand_oof_51a[name]
    p_test = cand_test_51a[name]

    lam_overlap = best_lambda_mask_51b(p_oof, overlap_oof)
    lam_solver = best_lambda_mask_51b(p_oof, solver_only_oof)
    lam_none = best_lambda_mask_51b(p_oof, none_oof)

    strategies = {
        "solver_only": {"overlap": 0.0, "solver_only": lam_solver, "none": 0.0},
        "none_only": {"overlap": 0.0, "solver_only": 0.0, "none": lam_none},
        "solver_plus_none": {"overlap": 0.0, "solver_only": lam_solver, "none": lam_none},
        "all_tiers": {"overlap": lam_overlap, "solver_only": lam_solver, "none": lam_none},
    }

    for strategy, lams in strategies.items():
        pred_oof = make_blend_51b(p_oof, p_test, lams, is_test=False)
        pred_test = make_blend_51b(p_oof, p_test, lams, is_test=True)

        mse = float(mean_squared_error(y50b, pred_oof))

        row = {
            "config": name,
            "strategy": strategy,
            "lambda_overlap": float(lams["overlap"]),
            "lambda_solver_only": float(lams["solver_only"]),
            "lambda_none": float(lams["none"]),
            "oof_mse": mse,
            "gain_vs_50b_base": mse50b_base - mse,
        }

        for tier, mask in tier_masks_oof.items():
            row[f"mse_{tier}"] = float(mean_squared_error(y50b[mask], pred_oof[mask]))
            row[f"gain_{tier}_vs_50b"] = (
                float(mean_squared_error(y50b[mask], pred50b_base_oof[mask]))
                - row[f"mse_{tier}"]
            )

        key = f"{name}__{strategy}"
        row["key"] = key
        screen_rows_51b.append(row)
        store_51b[key] = {"oof": pred_oof, "test": pred_test, "lams": lams}

screen51b = pd.DataFrame(screen_rows_51b).sort_values("oof_mse").reset_index(drop=True)

best51b = screen51b.iloc[0]
best_key51b = best51b["key"]
final_oof51b = store_51b[best_key51b]["oof"]
final_test51b = store_51b[best_key51b]["test"]

# Fold diagnostics using original fold split.
folds = list(KFold(n_splits=5, shuffle=True, random_state=9890).split(np.arange(len(y50b))))

fold_rows = []
for fold_num, (_, va_idx) in enumerate(folds, start=1):
    row = {
        "fold": fold_num,
        "mse_50b_base": float(mean_squared_error(y50b[va_idx], pred50b_base_oof[va_idx])),
        "mse_51b": float(mean_squared_error(y50b[va_idx], final_oof51b[va_idx])),
    }
    row["gain_vs_50b_base"] = row["mse_50b_base"] - row["mse_51b"]

    for tier, mask_full in tier_masks_oof.items():
        mask = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse50b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], pred50b_base_oof[va_idx][mask]))
            row[f"mse51b_{tier}"] = float(mean_squared_error(y50b[va_idx][mask], final_oof51b[va_idx][mask]))
            row[f"gain_{tier}_vs_50b"] = row[f"mse50b_{tier}"] - row[f"mse51b_{tier}"]

    fold_rows.append(row)

fold51b = pd.DataFrame(fold_rows)

# Save artifacts.
screen51b.to_csv("model_results/mf51b_wide_lambda_screen.csv", index=False)
fold51b.to_csv("model_results/mf51b_best_fold_diag.csv", index=False)

pd.DataFrame({
    "row_index": np.arange(len(y50b)),
    TARGET_COL: y50b,
    "pred_50b_base": pred50b_base_oof,
    "pred_clipped": final_oof51b,
    "tier_overlap": tier_masks_oof["overlap"].astype(int),
    "tier_solver_only": tier_masks_oof["solver_only"].astype(int),
    "tier_none": tier_masks_oof["none"].astype(int),
}).to_csv("model_results/oof_mf51b_best.csv", index=False)

pd.DataFrame({
    ID_COL: test_ids50b,
    "pred_50b_base": pred50b_base_test,
    TARGET_COL: final_test51b,
    "tier_overlap": tier_masks_test["overlap"].astype(int),
    "tier_solver_only": tier_masks_test["solver_only"].astype(int),
    "tier_none": tier_masks_test["none"].astype(int),
}).to_csv("model_results/testpred_mf51b_best.csv", index=False)

pd.DataFrame({
    ID_COL: test_ids50b,
    TARGET_COL: final_test51b,
}).to_csv("submission_mf51b_best.csv", index=False)

sub = pd.read_csv("submission_mf51b_best.csv")
assert sub.shape == (len(test_ids50b), 2)
assert list(sub.columns) == [ID_COL, TARGET_COL]
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

print("\nTop 20 51B wide-lambda candidates")
print("---------------------------------")
display(screen51b.head(20))

print("\nBest 51B")
print("--------")
print(best51b.to_string())

print("\nFold diagnostics")
print("----------------")
print(fold51b.to_string(index=False))
print("Min fold gain vs 50B:", float(fold51b["gain_vs_50b_base"].min()))

print("\nSaved")
print("-----")
print("model_results/mf51b_wide_lambda_screen.csv")
print("model_results/mf51b_best_fold_diag.csv")
print("model_results/oof_mf51b_best.csv")
print("model_results/testpred_mf51b_best.csv")
print("submission_mf51b_best.csv")
print(sub[TARGET_COL].describe())

51B. Wider anti-signal lambda search for second-pass MF

Top 20 51B wide-lambda candidates
---------------------------------


,config,strategy,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b_base,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b,key
0,mf51a_school_rank12_reg100p0_uniform,all_tiers,-0.005,-0.4350,-0.5700,51.091057,0.372101,0.506963,0.000020,63.207626,0.276817,88.733017,0.729965,mf51a_school_rank12_reg100p0_uniform__all_tiers
1,mf51a_school_rank12_reg100p0_uniform,solver_plus_none,0.000,-0.4350,-0.5700,51.091061,0.372097,0.506983,0.000000,63.207626,0.276817,88.733017,0.729965,mf51a_school_rank12_reg100p0_uniform__solver_p...
2,mf51a_school_rank10_reg100p0_uniform,all_tiers,-0.005,-0.4150,-0.5675,51.133072,0.330086,0.506950,0.000032,63.253178,0.231266,88.809181,0.653801,mf51a_school_rank10_reg100p0_uniform__all_tiers
3,mf51a_school_rank10_reg100p0_uniform,solver_plus_none,0.000,-0.4150,-0.5675,51.133087,0.330070,0.506983,0.000000,63.253178,0.231266,88.809181,0.653801,mf51a_school_rank10_reg100p0_uniform__solver_p...
4,mf51a_school_rank8_reg100p0_uniform,all_tiers,-0.005,-0.4475,-0.5975,51.140945,0.322212,0.506965,0.000017,63.247036,0.237408,88.829887,0.633095,mf51a_school_rank8_reg100p0_uniform__all_tiers
5,mf51a_school_rank8_reg100p0_uniform,solver_plus_none,0.000,-0.4475,-0.5975,51.140949,0.322208,0.506983,0.000000,63.247036,0.237408,88.829887,0.633095,mf51a_school_rank8_reg100p0_uniform__solver_pl...
6,mf51a_school_rank12_reg100p0_uniform,none_only,0.000,0.0000,-0.5700,51.144176,0.318981,0.506983,0.000000,63.484444,0.000000,88.733017,0.729965,mf51a_school_rank12_reg100p0_uniform__none_only
7,mf51a_school_rank12_reg200p0_uniform,all_tiers,-0.005,-0.4150,-0.5150,51.165401,0.297756,0.506953,0.000029,63.244114,0.240330,88.887146,0.575836,mf51a_school_rank12_reg200p0_uniform__all_tiers
8,mf51a_school_rank12_reg200p0_uniform,solver_plus_none,0.000,-0.4150,-0.5150,51.165413,0.297745,0.506983,0.000000,63.244114,0.240330,88.887146,0.575836,mf51a_school_rank12_reg200p0_uniform__solver_p...
9,mf51a_school_rank10_reg100p0_uniform,none_only,0.000,0.0000,-0.5675,51.177456,0.285702,0.506983,0.000000,63.484444,0.000000,88.809181,0.653801,mf51a_school_rank10_reg100p0_uniform__none_only



Best 51B
--------
config                                mf51a_school_rank12_reg100p0_uniform
strategy                                                         all_tiers
lambda_overlap                                                      -0.005
lambda_solver_only                                                  -0.435
lambda_none                                                          -0.57
oof_mse                                                          51.091057
gain_vs_50b_base                                                  0.372101
mse_overlap                                                       0.506963
gain_overlap_vs_50b                                                0.00002
mse_solver_only                                                  63.207626
gain_solver_only_vs_50b                                           0.276817
mse_none                                                         88.733017
gain_none_vs_50b                                                  0.729965
key   

In [61]:
# ============================================================
# 52A. Observed-entry ALS matrix factorization
# ============================================================
#
# Current public-protected model:
#   50B best OOF public MSE = 29.448
#
# Why this branch:
#   50B/50C used SVD-style factorization on a sparse residual matrix.
#   That is fast, but it implicitly treats unobserved matrix cells as zeros.
#
#   This branch fits matrix factorization only on observed rows:
#
#       residual_z ≈ μ + school_bias + item_bias + school_factor · item_factor
#
#   using alternating least squares.
#
# Candidate bases:
#   1. 44B base: learns a new first-pass MF correction
#   2. 50B base: learns a second-pass observed-entry MF correction
#
# Output:
#   submission_als52a_best_oof.csv
#   submission_als52a_top1.csv ... top5
#
# Safe:
#   - no torch
#   - no GPU
#   - no dense giant matrix
#   - checkpointed per config
# ============================================================

import os
import gc
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

Path("model_results").mkdir(exist_ok=True)
CHECKPOINT_DIR_52A = Path("model_results/als52a_checkpoints")
CHECKPOINT_DIR_52A.mkdir(exist_ok=True)

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5

print("=" * 90)
print("52A. Observed-entry ALS matrix factorization")
print("=" * 90)
print("Checkpoint dir:", CHECKPOINT_DIR_52A)

# ------------------------------------------------------------
# 1. Load artifacts
# ------------------------------------------------------------

def load_oof_52a(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column found in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_52a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

# 44B base
_, y52a, pred44b_oof = load_oof_52a("model_results/oof_hybrid44b_best.csv")
_, test_ids52a, pred44b_test = load_test_52a("model_results/testpred_hybrid44b_best.csv")

# 50B base
if not Path("model_results/oof_mf50b_best_oof.csv").exists():
    raise ValueError("Need model_results/oof_mf50b_best_oof.csv from 50B.")

_, y50b_file, pred50b_oof = load_oof_52a("model_results/oof_mf50b_best_oof.csv")
_, _, pred50b_test = load_test_52a("model_results/testpred_mf50b_best_oof.csv")

assert np.max(np.abs(y52a - y50b_file)) < 1e-5

# Optional 50A and 50C for comparison only.
has_50a = Path("model_results/oof_mf50a_best.csv").exists()
if has_50a:
    _, y50a_file, pred50a_oof = load_oof_52a("model_results/oof_mf50a_best.csv")
    _, _, pred50a_test = load_test_52a("model_results/testpred_mf50a_best.csv")
    assert np.max(np.abs(y52a - y50a_file)) < 1e-5
else:
    pred50a_oof = None
    pred50a_test = None

has_50c = Path("model_results/oof_mf50c_greedy_best.csv").exists()
if has_50c:
    _, y50c_file, pred50c_oof = load_oof_52a("model_results/oof_mf50c_greedy_best.csv")
    _, _, pred50c_test = load_test_52a("model_results/testpred_mf50c_greedy_best.csv")
    assert np.max(np.abs(y52a - y50c_file)) < 1e-5
else:
    pred50c_oof = None
    pred50c_test = None

n_train = len(y52a)
n_test = len(pred44b_test)

mse44b = float(mean_squared_error(y52a, pred44b_oof))
mse50b = float(mean_squared_error(y52a, pred50b_oof))
mse50a = float(mean_squared_error(y52a, pred50a_oof)) if has_50a else np.nan
mse50c = float(mean_squared_error(y52a, pred50c_oof)) if has_50c else np.nan

print("\nReferences")
print("----------")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")
if has_50a:
    print(f"50A OOF MSE: {mse50a:.6f} | public 29.742")
print(f"50B OOF MSE: {mse50b:.6f} | public 29.448")
if has_50c:
    print(f"50C greedy OOF MSE: {mse50c:.6f} | public transfer uncertain")

# ------------------------------------------------------------
# 2. Raw labels and index maps
# ------------------------------------------------------------

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise ValueError("raw_train_te/raw_test_te required.")

for c in ["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]:
    if c not in raw_train_te.columns or c not in raw_test_te.columns:
        raise ValueError(f"Missing required column: {c}")

def clean_str_52a(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str).to_numpy()

school_train = clean_str_52a(raw_train_te["SCHOOL"])
school_test = clean_str_52a(raw_test_te["SCHOOL"])

assessment_train = clean_str_52a(raw_train_te["ASSESSMENT_NAME"])
assessment_test = clean_str_52a(raw_test_te["ASSESSMENT_NAME"])

subgroup_train = clean_str_52a(raw_train_te["SUBGROUP_NAME"])
subgroup_test = clean_str_52a(raw_test_te["SUBGROUP_NAME"])

item_train = np.array([f"{a}||{s}" for a, s in zip(assessment_train, subgroup_train)], dtype=object)
item_test = np.array([f"{a}||{s}" for a, s in zip(assessment_test, subgroup_test)], dtype=object)

all_schools = pd.Index(pd.Series(np.concatenate([school_train, school_test])).astype(str).unique())
school_to_idx = {v: i for i, v in enumerate(all_schools)}
school_idx_train = np.array([school_to_idx[x] for x in school_train], dtype=np.int32)
school_idx_test = np.array([school_to_idx[x] for x in school_test], dtype=np.int32)

all_items = pd.Index(pd.Series(np.concatenate([item_train, item_test])).astype(str).unique())
item_to_idx = {v: i for i, v in enumerate(all_items)}
item_idx_train = np.array([item_to_idx[x] for x in item_train], dtype=np.int32)
item_idx_test = np.array([item_to_idx[x] for x in item_test], dtype=np.int32)

n_schools = len(all_schools)
n_items = len(all_items)

n_students_train = (
    pd.to_numeric(raw_train_te["N_STUDENTS"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .to_numpy(dtype=np.float64)
)

print("\nALS dimensions")
print("--------------")
print("n_schools:", n_schools)
print("n_items:", n_items)
print("train rows:", n_train)
print("test rows:", n_test)

# ------------------------------------------------------------
# 3. Tier masks
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

print("\nTier coverage")
print("-------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} test={int(tier_masks_test[tier].sum()):6d} "
        f"50B MSE={mean_squared_error(y52a[mask], pred50b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 4. Transform helpers
# ------------------------------------------------------------

def pct_to_arc_52a(pct):
    p = np.clip(np.asarray(pct, dtype=np.float64) / 100.0, 1e-6, 1 - 1e-6)
    return np.arcsin(np.sqrt(p)).astype(np.float32)

def arc_to_pct_52a(z):
    p = np.sin(np.asarray(z, dtype=np.float64)) ** 2
    return np.clip(100.0 * p, 0, 100).astype(np.float32)

z_y = pct_to_arc_52a(y52a)

base_pred_oof = {
    "44b": pred44b_oof,
    "50b": pred50b_oof,
}

base_pred_test = {
    "44b": pred44b_test,
    "50b": pred50b_test,
}

base_z_oof = {
    "44b": pct_to_arc_52a(pred44b_oof),
    "50b": pct_to_arc_52a(pred50b_oof),
}

base_z_test = {
    "44b": pct_to_arc_52a(pred44b_test),
    "50b": pct_to_arc_52a(pred50b_test),
}

# ------------------------------------------------------------
# 5. ALS helpers
# ------------------------------------------------------------

def build_groups(indices, n_groups):
    groups = [[] for _ in range(n_groups)]
    for pos, g in enumerate(indices):
        groups[int(g)].append(pos)
    return [np.asarray(x, dtype=np.int32) for x in groups]

def make_weight_52a(train_indices, scheme):
    if scheme == "uniform":
        return np.ones(len(train_indices), dtype=np.float64)

    n = n_students_train[train_indices].astype(np.float64)
    valid = np.isfinite(n) & (n > 0)
    med = np.nanmedian(n[valid]) if valid.any() else 30.0
    n = np.where(valid, n, med)

    if scheme == "sqrt_n":
        return np.sqrt(np.clip(n, 1, 500)).astype(np.float64)

    if scheme == "public_tier":
        w = np.ones(len(train_indices), dtype=np.float64)
        w[overlap_oof[train_indices]] *= 0.10
        w[solver_only_oof[train_indices]] *= 1.25
        w[none_oof[train_indices]] *= 0.75
        return w

    raise ValueError(f"Unknown weight scheme: {scheme}")

def fit_biases_52a(resid, tr_idx, reg_school, reg_item, weight_scheme, n_iter=5):
    s = school_idx_train[tr_idx]
    it = item_idx_train[tr_idx]
    r = resid[tr_idx].astype(np.float64)
    w = make_weight_52a(tr_idx, weight_scheme)

    mu = float(np.sum(w * r) / max(np.sum(w), 1e-12))

    sb = np.zeros(n_schools, dtype=np.float64)
    ib = np.zeros(n_items, dtype=np.float64)

    for _ in range(int(n_iter)):
        tmp = r - mu - ib[it]
        sw = np.bincount(s, weights=w, minlength=n_schools)
        sr = np.bincount(s, weights=w * tmp, minlength=n_schools)
        sb = sr / (sw + float(reg_school))

        tmp = r - mu - sb[s]
        iw = np.bincount(it, weights=w, minlength=n_items)
        ir = np.bincount(it, weights=w * tmp, minlength=n_items)
        ib = ir / (iw + float(reg_item))

    return mu, sb.astype(np.float32), ib.astype(np.float32)

def fit_als_observed_52a(
    resid,
    tr_idx,
    rank=16,
    reg_bias=500.0,
    reg_factor=100.0,
    weight_scheme="uniform",
    n_epochs=6,
    seed=9890,
):
    rng = np.random.default_rng(int(seed))

    tr_idx = np.asarray(tr_idx, dtype=np.int64)
    s = school_idx_train[tr_idx]
    it = item_idx_train[tr_idx]
    r = resid[tr_idx].astype(np.float64)
    w = make_weight_52a(tr_idx, weight_scheme)

    mu, sb, ib = fit_biases_52a(
        resid=resid,
        tr_idx=tr_idx,
        reg_school=reg_bias,
        reg_item=reg_bias,
        weight_scheme=weight_scheme,
        n_iter=5,
    )

    sb = sb.astype(np.float64)
    ib = ib.astype(np.float64)

    # Residual after biases.
    y = r - mu - sb[s] - ib[it]

    rank = int(rank)
    U = rng.normal(0, 0.01, size=(n_schools, rank)).astype(np.float64)
    V = rng.normal(0, 0.01, size=(n_items, rank)).astype(np.float64)

    school_groups = build_groups(s, n_schools)
    item_groups = build_groups(it, n_items)

    eye = np.eye(rank, dtype=np.float64)

    for epoch in range(int(n_epochs)):
        # Update school factors.
        for school_id, obs in enumerate(school_groups):
            if obs.size == 0:
                U[school_id, :] = 0.0
                continue

            item_obs = it[obs]
            V_obs = V[item_obs]
            w_obs = w[obs]
            y_obs = y[obs]

            A = V_obs.T @ (V_obs * w_obs[:, None]) + float(reg_factor) * eye
            b = V_obs.T @ (w_obs * y_obs)

            try:
                U[school_id, :] = np.linalg.solve(A, b)
            except np.linalg.LinAlgError:
                U[school_id, :] = np.linalg.lstsq(A, b, rcond=None)[0]

        # Update item factors.
        for item_id, obs in enumerate(item_groups):
            if obs.size == 0:
                V[item_id, :] = 0.0
                continue

            school_obs = s[obs]
            U_obs = U[school_obs]
            w_obs = w[obs]
            y_obs = y[obs]

            A = U_obs.T @ (U_obs * w_obs[:, None]) + float(reg_factor) * eye
            b = U_obs.T @ (w_obs * y_obs)

            try:
                V[item_id, :] = np.linalg.solve(A, b)
            except np.linalg.LinAlgError:
                V[item_id, :] = np.linalg.lstsq(A, b, rcond=None)[0]

    return {
        "mu": float(mu),
        "school_bias": sb.astype(np.float32),
        "item_bias": ib.astype(np.float32),
        "U": U.astype(np.float32),
        "V": V.astype(np.float32),
    }

def predict_als_52a(base_z, school_idx, item_idx, model, bias_scale=1.0, factor_scale=1.0):
    mu = model["mu"]
    sb = model["school_bias"]
    ib = model["item_bias"]
    U = model["U"]
    V = model["V"]

    bias = mu + sb[school_idx].astype(np.float64) + ib[item_idx].astype(np.float64)
    factor = np.sum(U[school_idx] * V[item_idx], axis=1).astype(np.float64)

    z = base_z.astype(np.float64) + float(bias_scale) * bias + float(factor_scale) * factor
    return arc_to_pct_52a(z)

# ------------------------------------------------------------
# 6. Config grid
# ------------------------------------------------------------

configs52a = []

# First-pass observed-entry ALS around 44B.
for rank in [8, 12, 16, 20]:
    for reg_factor in [50.0, 100.0, 200.0, 500.0]:
        configs52a.append({
            "name": f"als52a_base44b_rank{rank}_regf{str(reg_factor).replace('.', 'p')}_uniform",
            "base": "44b",
            "rank": rank,
            "reg_bias": 500.0,
            "reg_factor": reg_factor,
            "weight_scheme": "uniform",
            "n_epochs": 6,
            "bias_scale": 1.0,
            "factor_scale": 1.0,
        })

# Second-pass observed-entry ALS around public-validated 50B.
for rank in [2, 4, 6, 8, 12]:
    for reg_factor in [100.0, 200.0, 500.0, 1000.0, 2000.0]:
        configs52a.append({
            "name": f"als52a_base50b_rank{rank}_regf{str(reg_factor).replace('.', 'p')}_uniform",
            "base": "50b",
            "rank": rank,
            "reg_bias": 500.0,
            "reg_factor": reg_factor,
            "weight_scheme": "uniform",
            "n_epochs": 6,
            "bias_scale": 1.0,
            "factor_scale": 1.0,
        })

# A tiny number of sqrt_n variants for comparison.
for rank in [8, 16]:
    for reg_factor in [200.0, 500.0]:
        configs52a.append({
            "name": f"als52a_base44b_rank{rank}_regf{str(reg_factor).replace('.', 'p')}_sqrt",
            "base": "44b",
            "rank": rank,
            "reg_bias": 500.0,
            "reg_factor": reg_factor,
            "weight_scheme": "sqrt_n",
            "n_epochs": 6,
            "bias_scale": 1.0,
            "factor_scale": 1.0,
        })

print("\nNumber of ALS configs:", len(configs52a))
print("First configs:")
for c in configs52a[:8]:
    print(c)

with open("model_results/als52a_config_manifest.json", "w") as f:
    json.dump(configs52a, f, indent=2)

# ------------------------------------------------------------
# 7. Train OOF candidates
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

cand_oof = {}
cand_test = {}
fold_frames = []
direct_rows = []

t0 = time.time()

for cfg_num, cfg in enumerate(configs52a, start=1):
    name = cfg["name"]
    base = cfg["base"]

    print("\n" + "=" * 90)
    print(f"ALS config {cfg_num}/{len(configs52a)}: {name}")
    print("=" * 90)

    oof_path = CHECKPOINT_DIR_52A / f"{name}_oof.npy"
    test_path = CHECKPOINT_DIR_52A / f"{name}_test.npy"
    fold_path = CHECKPOINT_DIR_52A / f"{name}_folds.csv"

    if oof_path.exists() and test_path.exists() and fold_path.exists():
        pred_oof = np.load(oof_path).astype(np.float32)
        pred_test = np.load(test_path).astype(np.float32)
        fold_df = pd.read_csv(fold_path)
        print("Loaded checkpoint.")
    else:
        resid = (z_y - base_z_oof[base]).astype(np.float32)
        pred_oof = np.full(n_train, np.nan, dtype=np.float32)
        test_sum = np.zeros(n_test, dtype=np.float64)
        fold_rows = []

        start = time.time()

        for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
            model = fit_als_observed_52a(
                resid=resid,
                tr_idx=tr_idx,
                rank=cfg["rank"],
                reg_bias=cfg["reg_bias"],
                reg_factor=cfg["reg_factor"],
                weight_scheme=cfg["weight_scheme"],
                n_epochs=cfg["n_epochs"],
                seed=RANDOM_STATE + 1000 * fold_num + cfg["rank"],
            )

            p_va = predict_als_52a(
                base_z=base_z_oof[base][va_idx],
                school_idx=school_idx_train[va_idx],
                item_idx=item_idx_train[va_idx],
                model=model,
                bias_scale=cfg["bias_scale"],
                factor_scale=cfg["factor_scale"],
            )

            p_te = predict_als_52a(
                base_z=base_z_test[base],
                school_idx=school_idx_test,
                item_idx=item_idx_test,
                model=model,
                bias_scale=cfg["bias_scale"],
                factor_scale=cfg["factor_scale"],
            )

            pred_oof[va_idx] = p_va
            test_sum += p_te.astype(np.float64)

            row = {
                "config": name,
                "base": base,
                "fold": fold_num,
                "direct_mse": float(mean_squared_error(y52a[va_idx], p_va)),
                "base_mse": float(mean_squared_error(y52a[va_idx], base_pred_oof[base][va_idx])),
            }
            row["direct_gain_vs_base"] = row["base_mse"] - row["direct_mse"]

            for tier, mask_full in tier_masks_oof.items():
                mask = mask_full[va_idx]
                row[f"n_{tier}"] = int(mask.sum())
                if int(mask.sum()) > 0:
                    row[f"direct_mse_{tier}"] = float(mean_squared_error(y52a[va_idx][mask], p_va[mask]))
                    row[f"base_mse_{tier}"] = float(mean_squared_error(y52a[va_idx][mask], base_pred_oof[base][va_idx][mask]))
                    row[f"direct_gain_{tier}"] = row[f"base_mse_{tier}"] - row[f"direct_mse_{tier}"]

            fold_rows.append(row)

            del model
            gc.collect()

            print(
                f"fold {fold_num} | direct MSE {row['direct_mse']:.6f} "
                f"| gain vs base {row['direct_gain_vs_base']:.6f}"
            )

        pred_test = (test_sum / N_SPLITS).astype(np.float32)
        fold_df = pd.DataFrame(fold_rows)

        np.save(oof_path, pred_oof)
        np.save(test_path, pred_test)
        fold_df.to_csv(fold_path, index=False)

        print(f"Finished config in {time.time() - start:.1f}s")

    cand_oof[name] = pred_oof
    cand_test[name] = pred_test
    fold_frames.append(fold_df)

    direct_mse = float(mean_squared_error(y52a, pred_oof))
    direct_rows.append({
        **cfg,
        "direct_oof_mse": direct_mse,
        "direct_gain_vs_44b": mse44b - direct_mse,
        "direct_gain_vs_50b": mse50b - direct_mse,
        "direct_gain_vs_50a": mse50a - direct_mse if has_50a else np.nan,
    })

    pd.DataFrame(direct_rows).to_csv("model_results/als52a_direct_progress.csv", index=False)
    print(f"Direct OOF MSE: {direct_mse:.6f} | gain vs 50B: {mse50b - direct_mse:.6f}")

print("\nALS config pass elapsed seconds:", round(time.time() - t0, 1))

direct52a = pd.DataFrame(direct_rows).sort_values("direct_oof_mse").reset_index(drop=True)
fold_metrics52a = pd.concat(fold_frames, axis=0).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Tier blend scan
# ------------------------------------------------------------

lambda_grid = np.unique(
    np.concatenate([
        np.linspace(-1.0, 1.5, 1001),
        np.array([-1.0, -0.75, -0.50, -0.25, -0.10, 0.0, 0.005, 0.05, 0.10, 0.25, 0.50, 0.75, 1.0])
    ])
)

def best_lambda_mask(candidate_pred, base_pred, mask):
    if int(mask.sum()) == 0:
        return 0.0

    y = y52a[mask].astype(np.float64)
    b = base_pred[mask].astype(np.float64)
    a = candidate_pred[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = np.inf

    for lam in lambda_grid:
        p = np.clip(b + float(lam) * (a - b), 0, 100)
        mse = float(mean_squared_error(y, p))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam

def make_blend(candidate_oof, candidate_test, base_name, lams, is_test=False):
    if is_test:
        base = base_pred_test[base_name].astype(np.float64)
        alt = candidate_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = base_pred_oof[base_name].astype(np.float64)
        alt = candidate_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

screen_rows = []
store = {}

for cfg in configs52a:
    name = cfg["name"]
    base = cfg["base"]
    p_oof = cand_oof[name]
    p_test = cand_test[name]
    base_oof = base_pred_oof[base]

    lam_overlap = best_lambda_mask(p_oof, base_oof, overlap_oof)
    lam_solver = best_lambda_mask(p_oof, base_oof, solver_only_oof)
    lam_none = best_lambda_mask(p_oof, base_oof, none_oof)

    strategies = {
        "solver_only": {"overlap": 0.0, "solver_only": lam_solver, "none": 0.0},
        "none_only": {"overlap": 0.0, "solver_only": 0.0, "none": lam_none},
        "solver_plus_none": {"overlap": 0.0, "solver_only": lam_solver, "none": lam_none},
        "all_tiers": {"overlap": lam_overlap, "solver_only": lam_solver, "none": lam_none},
    }

    for strategy, lams in strategies.items():
        pred_oof = make_blend(p_oof, p_test, base, lams, is_test=False)
        pred_test = make_blend(p_oof, p_test, base, lams, is_test=True)

        mse = float(mean_squared_error(y52a, pred_oof))

        row = {
            **cfg,
            "strategy": strategy,
            "lambda_overlap": float(lams["overlap"]),
            "lambda_solver_only": float(lams["solver_only"]),
            "lambda_none": float(lams["none"]),
            "oof_mse": mse,
            "gain_vs_44b": mse44b - mse,
            "gain_vs_50a": mse50a - mse if has_50a else np.nan,
            "gain_vs_50b": mse50b - mse,
        }

        for tier, mask in tier_masks_oof.items():
            row[f"mse_{tier}"] = float(mean_squared_error(y52a[mask], pred_oof[mask]))
            row[f"gain_{tier}_vs_50b"] = (
                float(mean_squared_error(y52a[mask], pred50b_oof[mask])) - row[f"mse_{tier}"]
            )

        key = f"{name}__{strategy}"
        row["key"] = key
        screen_rows.append(row)
        store[key] = {
            "oof": pred_oof,
            "test": pred_test,
            "lams": lams,
            "cfg": cfg,
        }

screen52a = pd.DataFrame(screen_rows).sort_values("oof_mse").reset_index(drop=True)

best = screen52a.iloc[0]
best_key = best["key"]
best_oof = store[best_key]["oof"]
best_test = store[best_key]["test"]

# ------------------------------------------------------------
# 9. Fold diagnostics for best
# ------------------------------------------------------------

fold_diag_rows = []

for fold_num, (_, va_idx) in enumerate(folds, start=1):
    row = {
        "fold": fold_num,
        "mse_44b": float(mean_squared_error(y52a[va_idx], pred44b_oof[va_idx])),
        "mse_50b": float(mean_squared_error(y52a[va_idx], pred50b_oof[va_idx])),
        "mse_52a": float(mean_squared_error(y52a[va_idx], best_oof[va_idx])),
    }
    row["gain_vs_44b"] = row["mse_44b"] - row["mse_52a"]
    row["gain_vs_50b"] = row["mse_50b"] - row["mse_52a"]

    if has_50a:
        row["mse_50a"] = float(mean_squared_error(y52a[va_idx], pred50a_oof[va_idx]))
        row["gain_vs_50a"] = row["mse_50a"] - row["mse_52a"]

    for tier, mask_full in tier_masks_oof.items():
        mask = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse52a_{tier}"] = float(mean_squared_error(y52a[va_idx][mask], best_oof[va_idx][mask]))
            row[f"mse50b_{tier}"] = float(mean_squared_error(y52a[va_idx][mask], pred50b_oof[va_idx][mask]))
            row[f"gain_{tier}_vs_50b"] = row[f"mse50b_{tier}"] - row[f"mse52a_{tier}"]

    fold_diag_rows.append(row)

fold_diag52a = pd.DataFrame(fold_diag_rows)

# ------------------------------------------------------------
# 10. Save artifacts and top submissions
# ------------------------------------------------------------

direct_path = "model_results/als52a_direct_screen.csv"
screen_path = "model_results/als52a_blend_screen.csv"
fold_metrics_path = "model_results/als52a_fold_metrics.csv"
fold_diag_path = "model_results/als52a_best_fold_diag.csv"

oof_path = "model_results/oof_als52a_best.csv"
test_path = "model_results/testpred_als52a_best.csv"
submission_path = "submission_als52a_best_oof.csv"

direct52a.to_csv(direct_path, index=False)
screen52a.to_csv(screen_path, index=False)
fold_metrics52a.to_csv(fold_metrics_path, index=False)
fold_diag52a.to_csv(fold_diag_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y52a,
    "pred_44b": pred44b_oof,
    "pred_50b": pred50b_oof,
    "pred_clipped": best_oof,
    "tier_overlap": overlap_oof.astype(int),
    "tier_solver_only": solver_only_oof.astype(int),
    "tier_none": none_oof.astype(int),
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids52a,
    "pred_44b": pred44b_test,
    "pred_50b": pred50b_test,
    TARGET_COL: best_test,
    "tier_overlap": overlap_test.astype(int),
    "tier_solver_only": solver_only_test.astype(int),
    "tier_none": none_test.astype(int),
}).to_csv(test_path, index=False)

pd.DataFrame({
    ID_COL: test_ids52a,
    TARGET_COL: best_test,
}).to_csv(submission_path, index=False)

# Save top 5 submissions for later probing.
top_submission_paths = []
seen_keys = []

for i, row in screen52a.head(20).iterrows():
    key = row["key"]
    if key in seen_keys:
        continue
    seen_keys.append(key)

    pred_test = store[key]["test"]
    out_path = f"submission_als52a_top{len(seen_keys)}.csv"

    pd.DataFrame({
        ID_COL: test_ids52a,
        TARGET_COL: pred_test,
    }).to_csv(out_path, index=False)

    top_submission_paths.append(out_path)

    if len(seen_keys) >= 5:
        break

# Validate.
for p in [submission_path] + top_submission_paths:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test, 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[ID_COL].notna().all(), p
    assert sub[TARGET_COL].notna().all(), p
    assert np.isfinite(sub[TARGET_COL]).all(), p
    assert sub[TARGET_COL].between(0, 100).all(), p

# ------------------------------------------------------------
# 11. Output
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("52A observed-entry ALS complete")
print("=" * 90)

print("\nReferences")
print("----------")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")
if has_50a:
    print(f"50A OOF MSE: {mse50a:.6f} | public 29.742")
print(f"50B OOF MSE: {mse50b:.6f} | public 29.448")

print("\nTop 20 direct ALS candidates")
print("----------------------------")
display(direct52a.head(20))

print("\nTop 30 blended ALS candidates")
print("-----------------------------")
display(screen52a.head(30))

print("\nBest 52A")
print("--------")
print(best.to_string())

print("\nFold diagnostics")
print("----------------")
print(fold_diag52a.to_string(index=False))
print("Min fold gain vs 50B:", float(fold_diag52a["gain_vs_50b"].min()))

print("\nSaved files")
print("-----------")
print(direct_path)
print(screen_path)
print(fold_metrics_path)
print(fold_diag_path)
print(oof_path)
print(test_path)
print(submission_path)
for p in top_submission_paths:
    print(p)

print("\nSubmission validation")
print("---------------------")
sub = pd.read_csv(submission_path)
print(sub.shape)
print(sub[TARGET_COL].describe())

print("\nDecision rule")
print("-------------")
best_gain_vs_50b = float(best["gain_vs_50b"])
min_fold_gain_vs_50b = float(fold_diag52a["gain_vs_50b"].min())

if best_gain_vs_50b >= 0.25 and min_fold_gain_vs_50b >= 0:
    print("52A has meaningful stable gain over 50B. Worth a public probe after tomorrow's slots renew.")
elif best_gain_vs_50b > 0:
    print("52A has some OOF gain over 50B. Inspect fold/tier behavior carefully before submitting.")
else:
    print("52A does not beat public-validated 50B. Keep 50B protected.")

52A. Observed-entry ALS matrix factorization
Checkpoint dir: model_results/als52a_checkpoints

References
----------
44B OOF MSE: 52.780037 | public 30.556
50A OOF MSE: 52.247093 | public 29.742
50B OOF MSE: 51.463158 | public 29.448
50C greedy OOF MSE: 50.155319 | public transfer uncertain

ALS dimensions
--------------
n_schools: 4469
n_items: 132
train rows: 144921
test rows: 48307

Tier coverage
-------------
overlap      train= 53786 test= 27298 50B MSE=0.506983
solver_only  train= 27807 test= 17807 50B MSE=63.484444
none         train= 63328 test=  3202 50B MSE=89.462982

Number of ALS configs: 45
First configs:
{'name': 'als52a_base44b_rank8_regf50p0_uniform', 'base': '44b', 'rank': 8, 'reg_bias': 500.0, 'reg_factor': 50.0, 'weight_scheme': 'uniform', 'n_epochs': 6, 'bias_scale': 1.0, 'factor_scale': 1.0}
{'name': 'als52a_base44b_rank8_regf100p0_uniform', 'base': '44b', 'rank': 8, 'reg_bias': 500.0, 'reg_factor': 100.0, 'weight_scheme': 'uniform', 'n_epochs': 6, 'bias_scale': 1.

,name,base,rank,reg_bias,reg_factor,weight_scheme,n_epochs,bias_scale,factor_scale,direct_oof_mse,direct_gain_vs_44b,direct_gain_vs_50b,direct_gain_vs_50a
0,als52a_base50b_rank4_regf200p0_uniform,50b,4,500.0,200.0,uniform,6,1.0,1.0,51.772667,1.00737,-0.309509,0.474426
1,als52a_base50b_rank4_regf2000p0_uniform,50b,4,500.0,2000.0,uniform,6,1.0,1.0,51.772667,1.00737,-0.309509,0.474426
2,als52a_base50b_rank4_regf100p0_uniform,50b,4,500.0,100.0,uniform,6,1.0,1.0,51.772667,1.00737,-0.309509,0.474426
3,als52a_base50b_rank2_regf2000p0_uniform,50b,2,500.0,2000.0,uniform,6,1.0,1.0,51.772667,1.00737,-0.309509,0.474426
4,als52a_base50b_rank2_regf1000p0_uniform,50b,2,500.0,1000.0,uniform,6,1.0,1.0,51.772667,1.00737,-0.309509,0.474426
5,als52a_base50b_rank2_regf500p0_uniform,50b,2,500.0,500.0,uniform,6,1.0,1.0,51.772667,1.00737,-0.309509,0.474426
6,als52a_base50b_rank2_regf200p0_uniform,50b,2,500.0,200.0,uniform,6,1.0,1.0,51.772667,1.00737,-0.309509,0.474426
7,als52a_base50b_rank2_regf100p0_uniform,50b,2,500.0,100.0,uniform,6,1.0,1.0,51.772667,1.00737,-0.309509,0.474426
8,als52a_base50b_rank6_regf100p0_uniform,50b,6,500.0,100.0,uniform,6,1.0,1.0,51.772667,1.00737,-0.309509,0.474426
9,als52a_base50b_rank6_regf200p0_uniform,50b,6,500.0,200.0,uniform,6,1.0,1.0,51.772667,1.00737,-0.309509,0.474426



Top 30 blended ALS candidates
-----------------------------


,name,base,rank,reg_bias,reg_factor,weight_scheme,n_epochs,bias_scale,factor_scale,strategy,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_44b,gain_vs_50a,gain_vs_50b,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b,key
0,als52a_base50b_rank6_regf100p0_uniform,50b,6,500.0,100.0,uniform,6,1.0,1.0,all_tiers,0.0125,-1.0,-1.0,51.262241,1.517796,0.984852,0.200916,0.506974,0.000009,63.382011,0.102432,89.048195,0.414787,als52a_base50b_rank6_regf100p0_uniform__all_tiers
1,als52a_base50b_rank8_regf200p0_uniform,50b,8,500.0,200.0,uniform,6,1.0,1.0,all_tiers,0.0125,-1.0,-1.0,51.262241,1.517796,0.984852,0.200916,0.506974,0.000009,63.382011,0.102432,89.048195,0.414787,als52a_base50b_rank8_regf200p0_uniform__all_tiers
2,als52a_base50b_rank2_regf1000p0_uniform,50b,2,500.0,1000.0,uniform,6,1.0,1.0,all_tiers,0.0125,-1.0,-1.0,51.262241,1.517796,0.984852,0.200916,0.506974,0.000009,63.382011,0.102432,89.048195,0.414787,als52a_base50b_rank2_regf1000p0_uniform__all_t...
3,als52a_base50b_rank8_regf1000p0_uniform,50b,8,500.0,1000.0,uniform,6,1.0,1.0,all_tiers,0.0125,-1.0,-1.0,51.262241,1.517796,0.984852,0.200916,0.506974,0.000009,63.382011,0.102432,89.048195,0.414787,als52a_base50b_rank8_regf1000p0_uniform__all_t...
4,als52a_base50b_rank8_regf100p0_uniform,50b,8,500.0,100.0,uniform,6,1.0,1.0,all_tiers,0.0125,-1.0,-1.0,51.262241,1.517796,0.984852,0.200916,0.506974,0.000009,63.382011,0.102432,89.048195,0.414787,als52a_base50b_rank8_regf100p0_uniform__all_tiers
5,als52a_base50b_rank8_regf2000p0_uniform,50b,8,500.0,2000.0,uniform,6,1.0,1.0,all_tiers,0.0125,-1.0,-1.0,51.262241,1.517796,0.984852,0.200916,0.506974,0.000009,63.382011,0.102432,89.048195,0.414787,als52a_base50b_rank8_regf2000p0_uniform__all_t...
6,als52a_base50b_rank2_regf2000p0_uniform,50b,2,500.0,2000.0,uniform,6,1.0,1.0,all_tiers,0.0125,-1.0,-1.0,51.262241,1.517796,0.984852,0.200916,0.506974,0.000009,63.382011,0.102432,89.048195,0.414787,als52a_base50b_rank2_regf2000p0_uniform__all_t...
7,als52a_base50b_rank2_regf200p0_uniform,50b,2,500.0,200.0,uniform,6,1.0,1.0,all_tiers,0.0125,-1.0,-1.0,51.262241,1.517796,0.984852,0.200916,0.506974,0.000009,63.382011,0.102432,89.048195,0.414787,als52a_base50b_rank2_regf200p0_uniform__all_tiers
8,als52a_base50b_rank12_regf100p0_uniform,50b,12,500.0,100.0,uniform,6,1.0,1.0,all_tiers,0.0125,-1.0,-1.0,51.262241,1.517796,0.984852,0.200916,0.506974,0.000009,63.382011,0.102432,89.048195,0.414787,als52a_base50b_rank12_regf100p0_uniform__all_t...
9,als52a_base50b_rank6_regf2000p0_uniform,50b,6,500.0,2000.0,uniform,6,1.0,1.0,all_tiers,0.0125,-1.0,-1.0,51.262241,1.517796,0.984852,0.200916,0.506974,0.000009,63.382011,0.102432,89.048195,0.414787,als52a_base50b_rank6_regf2000p0_uniform__all_t...



Best 52A
--------
name                                  als52a_base50b_rank6_regf100p0_uniform
base                                                                     50b
rank                                                                       6
reg_bias                                                               500.0
reg_factor                                                             100.0
weight_scheme                                                        uniform
n_epochs                                                                   6
bias_scale                                                               1.0
factor_scale                                                             1.0
strategy                                                           all_tiers
lambda_overlap                                                        0.0125
lambda_solver_only                                                      -1.0
lambda_none                                              

this was useless 51A

In [4]:
# ============================================================
# POST-RESTART RAW ALIASES FOR RECOVERY
# ============================================================

TARGET_COL = "PERCENT_PROFICIENT"
TARGET = TARGET_COL
ID_COL = "ASSESSMENT_ID"

raw_train_te = train_full.copy()
raw_test_te = test_full.copy()

y_train = train_full[TARGET_COL].to_numpy(dtype="float32")
test_ids = test_full[ID_COL].copy()

print("Raw alias recovery OK")
print("raw_train_te:", raw_train_te.shape)
print("raw_test_te: ", raw_test_te.shape)
print("y_train:     ", y_train.shape)
print("test_ids:    ", test_ids.shape)

Raw alias recovery OK
raw_train_te: (144921, 62)
raw_test_te:  (48307, 61)
y_train:      (144921,)
test_ids:     (48307,)


In [5]:
# ============================================================
# Minimal recovery / preflight after kernel crash
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("Minimal recovery / preflight")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required in-memory raw objects
# ------------------------------------------------------------

required_objects = ["raw_train_te", "raw_test_te"]

missing_objects = [x for x in required_objects if x not in globals()]
if missing_objects:
    raise RuntimeError(
        f"Missing {missing_objects}. Run only the notebook setup/data-load cells "
        "until raw_train_te and raw_test_te exist. Do not rerun model training cells."
    )

if "y_train" not in globals():
    if TARGET_COL in raw_train_te.columns:
        y_train = raw_train_te[TARGET_COL].to_numpy(dtype=np.float32)
        print("Recovered y_train from raw_train_te target column.")
    else:
        raise RuntimeError("Missing y_train and target column not found in raw_train_te.")

y_train = np.asarray(y_train, dtype=np.float32).reshape(-1)

print("raw_train_te:", raw_train_te.shape)
print("raw_test_te: ", raw_test_te.shape)
print("y_train:     ", y_train.shape)

# ------------------------------------------------------------
# 2. Required artifact files
# ------------------------------------------------------------

required_files = [
    "model_results/oof_mf50b_best_oof.csv",
    "model_results/testpred_mf50b_best_oof.csv",
    "model_results/oof_hybrid44b_best.csv",
    "model_results/testpred_hybrid44b_best.csv",
    "model_results/account39a_raw_oof_reconstruction.csv",
    "model_results/account39a_raw_test_reconstruction.csv",
    "model_results/account39b_solver_raw_oof.csv",
    "model_results/account39b_solver_raw_test.csv",
]

print("\nArtifact checks")
print("---------------")
for p in required_files:
    print(f"{p:65s}", Path(p).exists())

missing_files = [p for p in required_files if not Path(p).exists()]
if missing_files:
    raise RuntimeError(f"Missing required artifact files: {missing_files}")

# ------------------------------------------------------------
# 3. Safe load helpers
# ------------------------------------------------------------

def load_oof_recovery(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column found in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)

def load_test_recovery(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 4. Load 50B and 44B anchors
# ------------------------------------------------------------

_, y50b_file, pred50b_oof = load_oof_recovery("model_results/oof_mf50b_best_oof.csv")
_, test_ids, pred50b_test = load_test_recovery("model_results/testpred_mf50b_best_oof.csv")

_, y44b_file, pred44b_oof = load_oof_recovery("model_results/oof_hybrid44b_best.csv")
_, _, pred44b_test = load_test_recovery("model_results/testpred_hybrid44b_best.csv")

assert len(y50b_file) == len(y_train), "50B OOF length mismatch"
assert len(pred50b_test) == len(raw_test_te), "50B test length mismatch"
assert np.max(np.abs(y50b_file - y_train)) < 1e-5, "50B target mismatch"
assert np.max(np.abs(y44b_file - y_train)) < 1e-5, "44B target mismatch"

print("\nAnchor checks")
print("-------------")
print(f"50B OOF MSE: {mean_squared_error(y_train, pred50b_oof):.6f} | public 29.448")
print(f"44B OOF MSE: {mean_squared_error(y_train, pred44b_oof):.6f} | public 30.556")

# ------------------------------------------------------------
# 5. Reconstruct accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_oof = np.array(["none"] * len(y_train), dtype=object)
tier_oof[solver_only_oof] = "solver_only"
tier_oof[overlap_oof] = "overlap"

tier_test = np.array(["none"] * len(raw_test_te), dtype=object)
tier_test[solver_only_test] = "solver_only"
tier_test[overlap_test] = "overlap"

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

print("\nTier recovery")
print("-------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"50B MSE={mean_squared_error(y_train[mask], pred50b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 6. LightGBM checkpoint cleanup
# ------------------------------------------------------------

ckpt_dir = Path("model_results/lgb53a_checkpoints")
ckpt_dir.mkdir(exist_ok=True)

print("\nExisting 53A checkpoint files")
print("-----------------------------")
for p in sorted(ckpt_dir.glob("*"))[:20]:
    print(p)

# Remove partial files only if they exist without complete matching triplet.
for stem in [
    "lgb53a_base_public_mimic_l31_leaf500",
    "lgb53a_base_solver_focus_l31_leaf700",
    "lgb53a_base_solver_none_l63_leaf1000",
    "lgb53a_rich_public_mimic_l31_leaf1000",
    "lgb53a_rich_solver_focus_l63_leaf1500",
    "lgb53a_base_uniform_l31_leaf800",
]:
    files = [
        ckpt_dir / f"{stem}_delta_oof.npy",
        ckpt_dir / f"{stem}_delta_test.npy",
        ckpt_dir / f"{stem}_folds.csv",
    ]
    exists = [p.exists() for p in files]
    if any(exists) and not all(exists):
        print("Removing partial checkpoint for", stem)
        for p in files:
            if p.exists():
                p.unlink()

gc.collect()

print("\nRecovery OK.")
print("Next: do NOT rerun full 53A yet. Run a tiny LightGBM smoke test first.")

Minimal recovery / preflight
raw_train_te: (144921, 62)
raw_test_te:  (48307, 61)
y_train:      (144921,)

Artifact checks
---------------
model_results/oof_mf50b_best_oof.csv                              True
model_results/testpred_mf50b_best_oof.csv                         True
model_results/oof_hybrid44b_best.csv                              True
model_results/testpred_hybrid44b_best.csv                         True
model_results/account39a_raw_oof_reconstruction.csv               True
model_results/account39a_raw_test_reconstruction.csv              True
model_results/account39b_solver_raw_oof.csv                       True
model_results/account39b_solver_raw_test.csv                      True

Anchor checks
-------------
50B OOF MSE: 51.463158 | public 29.448
44B OOF MSE: 52.780037 | public 30.556

Tier recovery
-------------
overlap      train= 53786 test= 27298 50B MSE=0.506983
solver_only  train= 27807 test= 17807 50B MSE=63.484444
none         train= 63328 test=  3202 50B MSE=

In [6]:
# ============================================================
# 53A smoke test: LightGBM with low-cardinality categoricals only
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except Exception as e:
    raise ImportError(f"LightGBM import failed: {repr(e)}")

print("=" * 90)
print("53A LightGBM smoke test")
print("=" * 90)

def pct_to_arc_smoke(pct):
    p = np.clip(np.asarray(pct, dtype=np.float64) / 100.0, 1e-6, 1 - 1e-6)
    return np.arcsin(np.sqrt(p)).astype(np.float32)

def arc_to_pct_smoke(z):
    p = np.sin(np.asarray(z, dtype=np.float64)) ** 2
    return np.clip(100.0 * p, 0, 100).astype(np.float32)

z_y = pct_to_arc_smoke(y_train)
z_50b_oof = pct_to_arc_smoke(pred50b_oof)
resid_z = (z_y - z_50b_oof).astype(np.float32)

# Keep only low/medium-cardinality categories for smoke test.
low_cat_cols = [
    "SUBGROUP_NAME",
    "ASSESSMENT_NAME",
    "DISTRICT_TYPE",
    "COUNTY",
    "REGION",
]

num_cols = [
    "N_STUDENTS",
    "PERCENT_FREE_LUNCH",
    "PERCENT_REDUCED_LUNCH",
    "PERCENT_ECONOMICALLY_DISADVANTAGED",
    "PERCENT_ENGLISH_LANGUAGE_LEARNERS",
    "PERCENT_WITH_DISABILITIES",
    "PERCENT_HOMELESS",
    "PERCENT_MIGRANT",
    "PERCENT_FEMALE",
    "PERCENT_MALE",
    "ATTENDANCE_RATE",
]

low_cat_cols = [c for c in low_cat_cols if c in raw_train_te.columns and c in raw_test_te.columns]
num_cols = [c for c in num_cols if c in raw_train_te.columns and c in raw_test_te.columns]

X_smoke = pd.DataFrame(index=np.arange(len(raw_train_te)))

for c in num_cols:
    X_smoke[c] = (
        pd.to_numeric(raw_train_te[c], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .astype(np.float32)
    )

X_smoke["pred50b"] = pred50b_oof.astype(np.float32)
X_smoke["tier_solver_only"] = solver_only_oof.astype(np.float32)
X_smoke["tier_none"] = none_oof.astype(np.float32)
X_smoke["tier_overlap"] = overlap_oof.astype(np.float32)

for c in low_cat_cols:
    s = pd.Series(raw_train_te[c]).astype("string").fillna("<NA>").astype(str)
    X_smoke[c] = pd.Categorical(s)

cat_cols_smoke = low_cat_cols

print("Smoke features:", X_smoke.shape)
print("Smoke categorical columns:", cat_cols_smoke)

folds = list(KFold(n_splits=5, shuffle=True, random_state=9890).split(np.arange(len(y_train))))
tr_idx, va_idx = folds[0]

# Use only fold 1 first.
X_tr = X_smoke.iloc[tr_idx]
X_va = X_smoke.iloc[va_idx]

w_tr = np.ones(len(tr_idx), dtype=np.float32)
w_tr[overlap_oof[tr_idx]] *= 0.05
w_tr[solver_only_oof[tr_idx]] *= 1.25
w_tr[none_oof[tr_idx]] *= 0.50

model = LGBMRegressor(
    objective="regression",
    n_estimators=150,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=1000,
    subsample=0.80,
    subsample_freq=1,
    colsample_bytree=0.80,
    reg_lambda=100.0,
    n_jobs=1,
    verbosity=-1,
    random_state=9890,
    max_bin=127,
)

model.fit(
    X_tr,
    resid_z[tr_idx],
    sample_weight=w_tr,
    categorical_feature=cat_cols_smoke,
)

d_va = model.predict(X_va).astype(np.float32)
pred_va = arc_to_pct_smoke(z_50b_oof[va_idx] + np.clip(d_va, -0.20, 0.20))

base_mse = mean_squared_error(y_train[va_idx], pred50b_oof[va_idx])
cand_mse = mean_squared_error(y_train[va_idx], pred_va)

print("\nSmoke result")
print("------------")
print("Fold 1 50B MSE:", float(base_mse))
print("Fold 1 LGB candidate MSE:", float(cand_mse))
print("Gain:", float(base_mse - cand_mse))

print("\nSmoke test completed without kernel crash.")

53A LightGBM smoke test
Smoke features: (144921, 19)
Smoke categorical columns: ['SUBGROUP_NAME', 'ASSESSMENT_NAME', 'DISTRICT_TYPE', 'COUNTY', 'REGION']

Smoke result
------------
Fold 1 50B MSE: 51.92796325683594
Fold 1 LGB candidate MSE: 53.04200744628906
Gain: -1.114044189453125

Smoke test completed without kernel crash.


In [7]:
# ============================================================
# PRECHECK BEFORE 53B
# ============================================================

needed = [
    "raw_train_te", "raw_test_te", "y_train",
    "pred50b_oof", "pred50b_test",
    "pred44b_oof", "pred44b_test",
    "overlap_oof", "solver_only_oof", "none_oof",
    "overlap_test", "solver_only_test", "none_test",
]

missing = [x for x in needed if x not in globals()]
print("Missing:", missing)

if "test_ids" not in globals():
    if "ASSESSMENT_ID" in raw_test_te.columns:
        test_ids = raw_test_te["ASSESSMENT_ID"].to_numpy()
        print("Recovered test_ids from raw_test_te.")
    else:
        raise RuntimeError("test_ids missing and ASSESSMENT_ID not in raw_test_te.")

print("Ready for 53B." if not missing else "Do not run 53B yet.")

Missing: []
Ready for 53B.


In [8]:
# ============================================================
# 53B. Safe numeric-only LightGBM residual model with OOF encodings
# ============================================================
#
# Purpose:
#   Test tree-based residual modeling on top of public-validated 50B
#   without using LightGBM native categorical features.
#
# Why:
#   Native categorical 53A likely caused kernel crash.
#   Smoke test survived but low-cardinality features were bad.
#   This version encodes high-cardinality categories numerically using:
#       - OOF residual target encodings
#       - count/frequency encodings
#
# Current protected model:
#   submission_mf50b_best_oof.csv
#   public MSE = 29.448
#
# Do not submit automatically.
# ============================================================

import os
import gc
import json
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except Exception as e:
    raise ImportError(f"LightGBM import failed: {repr(e)}")

Path("model_results").mkdir(exist_ok=True)
CHECKPOINT_DIR_53B = Path("model_results/lgb53b_safe_checkpoints")
CHECKPOINT_DIR_53B.mkdir(exist_ok=True)

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5

print("=" * 90)
print("53B. Safe numeric-only LightGBM residual model with OOF encodings")
print("=" * 90)

# ------------------------------------------------------------
# 1. Sanity checks from recovery
# ------------------------------------------------------------

required = [
    "raw_train_te", "raw_test_te", "y_train",
    "pred50b_oof", "pred50b_test",
    "pred44b_oof", "pred44b_test",
    "tier_masks_oof", "tier_masks_test",
    "overlap_oof", "solver_only_oof", "none_oof",
    "overlap_test", "solver_only_test", "none_test",
    "test_ids"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing recovery variables: {missing}")

y53b = np.asarray(y_train, dtype=np.float32).reshape(-1)
n_train = len(y53b)
n_test = len(pred50b_test)

mse50b = float(mean_squared_error(y53b, pred50b_oof))

print("\nReference")
print("---------")
print(f"50B OOF MSE: {mse50b:.6f} | public 29.448")

for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"50B MSE={mean_squared_error(y53b[mask], pred50b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 2. Target transform
# ------------------------------------------------------------

def pct_to_arc_53b(pct):
    p = np.clip(np.asarray(pct, dtype=np.float64) / 100.0, 1e-6, 1 - 1e-6)
    return np.arcsin(np.sqrt(p)).astype(np.float32)

def arc_to_pct_53b(z):
    p = np.sin(np.asarray(z, dtype=np.float64)) ** 2
    return np.clip(100.0 * p, 0, 100).astype(np.float32)

z_y = pct_to_arc_53b(y53b)
z_50b_oof = pct_to_arc_53b(pred50b_oof)
z_50b_test = pct_to_arc_53b(pred50b_test)
resid_z = (z_y - z_50b_oof).astype(np.float32)

# ------------------------------------------------------------
# 3. Base numeric features
# ------------------------------------------------------------

def clean_str_53b(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def subject_family_53b(x):
    s = str(x).lower()
    if "math" in s or "algebra" in s or "geometry" in s:
        return "math"
    if "ela" in s or "english" in s:
        return "ela"
    if "science" in s or "chemistry" in s or "physics" in s or "biology" in s or "earth" in s:
        return "science"
    if "history" in s or "global" in s or "social" in s or "government" in s:
        return "social"
    return "other"

def n_bin_53b(x):
    return pd.cut(
        pd.Series(x).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, 200, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", "101-200", ">200"],
    ).astype("string").fillna("<NA>").astype(str)

def build_base_numeric_53b(raw_df, pred50b, pred44b, tier_arr):
    out = pd.DataFrame(index=np.arange(len(raw_df)))

    shared_cols = [c for c in raw_train_te.columns if c in raw_test_te.columns]
    exclude = {TARGET_COL, ID_COL, "ASSESSMENT_ID", "row_index"}
    shared_cols = [c for c in shared_cols if c not in exclude]

    for c in shared_cols:
        if pd.api.types.is_numeric_dtype(raw_train_te[c]) or pd.api.types.is_numeric_dtype(raw_test_te[c]):
            out[c] = (
                pd.to_numeric(raw_df[c], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .astype(np.float32)
            )

    if "N_STUDENTS" in raw_df.columns:
        n = (
            pd.to_numeric(raw_df["N_STUDENTS"], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .to_numpy(dtype=np.float64)
        )
        out["N_STUDENTS_NUM"] = n.astype(np.float32)
        out["LOG1P_N_STUDENTS"] = np.log1p(np.maximum(n, 0)).astype(np.float32)
    else:
        out["N_STUDENTS_NUM"] = np.nan
        out["LOG1P_N_STUDENTS"] = np.nan

    out["pred50b"] = pred50b.astype(np.float32)
    out["pred44b"] = pred44b.astype(np.float32)
    out["diff50b_44b"] = (pred50b - pred44b).astype(np.float32)

    out["tier_overlap"] = (pd.Series(tier_arr).astype(str) == "overlap").astype(np.float32)
    out["tier_solver_only"] = (pd.Series(tier_arr).astype(str) == "solver_only").astype(np.float32)
    out["tier_none"] = (pd.Series(tier_arr).astype(str) == "none").astype(np.float32)

    return out

tier_oof_arr = np.array(["none"] * n_train, dtype=object)
tier_oof_arr[solver_only_oof] = "solver_only"
tier_oof_arr[overlap_oof] = "overlap"

tier_test_arr = np.array(["none"] * n_test, dtype=object)
tier_test_arr[solver_only_test] = "solver_only"
tier_test_arr[overlap_test] = "overlap"

X_base_train = build_base_numeric_53b(raw_train_te, pred50b_oof, pred44b_oof, tier_oof_arr)
X_base_test = build_base_numeric_53b(raw_test_te, pred50b_test, pred44b_test, tier_test_arr)

# ------------------------------------------------------------
# 4. Categorical keys for safe encodings
# ------------------------------------------------------------

def build_cat_frame_53b(raw_df, tier_arr):
    out = pd.DataFrame(index=np.arange(len(raw_df)))

    for c in ["SCHOOL", "DISTRICT", "COUNTY", "REGION", "DISTRICT_TYPE", "ASSESSMENT_NAME", "SUBGROUP_NAME"]:
        if c in raw_df.columns:
            out[c] = clean_str_53b(raw_df[c]).to_numpy()
        else:
            out[c] = "<NA>"

    n = pd.to_numeric(raw_df["N_STUDENTS"], errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)
    out["N_STUDENTS_BIN"] = n_bin_53b(n)

    out["ASSESSMENT_SUBGROUP"] = out["ASSESSMENT_NAME"].astype(str) + "||" + out["SUBGROUP_NAME"].astype(str)
    out["SCHOOL_ASSESSMENT"] = out["SCHOOL"].astype(str) + "||" + out["ASSESSMENT_NAME"].astype(str)
    out["SCHOOL_SUBGROUP"] = out["SCHOOL"].astype(str) + "||" + out["SUBGROUP_NAME"].astype(str)
    out["DISTRICT_ASSESSMENT"] = out["DISTRICT"].astype(str) + "||" + out["ASSESSMENT_NAME"].astype(str)
    out["COUNTY_ASSESSMENT"] = out["COUNTY"].astype(str) + "||" + out["ASSESSMENT_NAME"].astype(str)

    out["SUBJECT_FAMILY"] = pd.Series(out["ASSESSMENT_NAME"]).map(subject_family_53b).astype(str).to_numpy()
    out["SUBJECT_SUBGROUP"] = out["SUBJECT_FAMILY"].astype(str) + "||" + out["SUBGROUP_NAME"].astype(str)
    out["TIER"] = pd.Series(tier_arr).astype(str).to_numpy()

    return out

cat_train = build_cat_frame_53b(raw_train_te, tier_oof_arr)
cat_test = build_cat_frame_53b(raw_test_te, tier_test_arr)

encoding_cols = [
    "SCHOOL",
    "DISTRICT",
    "COUNTY",
    "REGION",
    "DISTRICT_TYPE",
    "ASSESSMENT_NAME",
    "SUBGROUP_NAME",
    "ASSESSMENT_SUBGROUP",
    "SCHOOL_ASSESSMENT",
    "SCHOOL_SUBGROUP",
    "DISTRICT_ASSESSMENT",
    "COUNTY_ASSESSMENT",
    "N_STUDENTS_BIN",
    "SUBJECT_FAMILY",
    "SUBJECT_SUBGROUP",
    "TIER",
]

encoding_cols = [c for c in encoding_cols if c in cat_train.columns]

print("\nBase feature shapes")
print("-------------------")
print("X_base_train:", X_base_train.shape)
print("X_base_test: ", X_base_test.shape)
print("encoding columns:", len(encoding_cols), encoding_cols)

# ------------------------------------------------------------
# 5. Fold-safe OOF target/frequency encoding
# ------------------------------------------------------------

def fit_encoding_stats(keys, target, global_mean, smoothing):
    df = pd.DataFrame({"key": keys.astype(str), "target": target.astype(np.float32)})
    stats = df.groupby("key")["target"].agg(["mean", "count"])
    stats["te"] = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
    return stats

def map_te(keys, stats, global_mean):
    return (
        pd.Series(keys.astype(str))
        .map(stats["te"])
        .fillna(global_mean)
        .to_numpy(dtype=np.float32)
    )

def map_count(keys, stats):
    return (
        pd.Series(keys.astype(str))
        .map(stats["count"])
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

def add_oof_encoded_features_for_outer_fold(
    X_tr_base,
    X_va_base,
    X_te_base,
    tr_idx,
    va_idx,
    cols,
    smoothing=200.0,
    inner_splits=4,
):
    X_tr = X_tr_base.copy().reset_index(drop=True)
    X_va = X_va_base.copy().reset_index(drop=True)
    X_te = X_te_base.copy().reset_index(drop=True)

    tr_idx = np.asarray(tr_idx, dtype=np.int64)
    va_idx = np.asarray(va_idx, dtype=np.int64)

    target_tr = resid_z[tr_idx]
    global_mean = float(np.mean(target_tr))

    inner = list(KFold(n_splits=inner_splits, shuffle=True, random_state=RANDOM_STATE + 222).split(np.arange(len(tr_idx))))

    for col in cols:
        keys_tr_all = cat_train.iloc[tr_idx][col].astype(str).reset_index(drop=True)
        keys_va = cat_train.iloc[va_idx][col].astype(str).reset_index(drop=True)
        keys_te = cat_test[col].astype(str).reset_index(drop=True)

        # OOF target encoding for training rows.
        tr_te = np.full(len(tr_idx), global_mean, dtype=np.float32)

        for inner_tr_pos, inner_va_pos in inner:
            stats_inner = fit_encoding_stats(
                keys_tr_all.iloc[inner_tr_pos].to_numpy(),
                target_tr[inner_tr_pos],
                global_mean=global_mean,
                smoothing=smoothing,
            )
            tr_te[inner_va_pos] = map_te(
                keys_tr_all.iloc[inner_va_pos].to_numpy(),
                stats_inner,
                global_mean=global_mean,
            )

        # Full fold stats for validation/test and count encodings.
        stats_full = fit_encoding_stats(
            keys_tr_all.to_numpy(),
            target_tr,
            global_mean=global_mean,
            smoothing=smoothing,
        )

        safe_col = col.lower().replace(" ", "_").replace("/", "_").replace("-", "_")

        X_tr[f"te_{safe_col}"] = tr_te
        X_va[f"te_{safe_col}"] = map_te(keys_va.to_numpy(), stats_full, global_mean=global_mean)
        X_te[f"te_{safe_col}"] = map_te(keys_te.to_numpy(), stats_full, global_mean=global_mean)

        X_tr[f"cnt_{safe_col}"] = np.log1p(map_count(keys_tr_all.to_numpy(), stats_full)).astype(np.float32)
        X_va[f"cnt_{safe_col}"] = np.log1p(map_count(keys_va.to_numpy(), stats_full)).astype(np.float32)
        X_te[f"cnt_{safe_col}"] = np.log1p(map_count(keys_te.to_numpy(), stats_full)).astype(np.float32)

    # Ensure numeric float32.
    for df in [X_tr, X_va, X_te]:
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype(np.float32)

    return X_tr, X_va, X_te

# ------------------------------------------------------------
# 6. Weights, scoring, blending
# ------------------------------------------------------------

def make_weight_53b(idx, scheme):
    idx = np.asarray(idx, dtype=np.int64)
    w = np.ones(len(idx), dtype=np.float32)

    if scheme == "uniform":
        pass
    elif scheme == "public_mimic":
        w[overlap_oof[idx]] *= 0.05
        w[solver_only_oof[idx]] *= 1.25
        w[none_oof[idx]] *= 0.35
    elif scheme == "solver_focus":
        w[overlap_oof[idx]] *= 0.03
        w[solver_only_oof[idx]] *= 1.50
        w[none_oof[idx]] *= 0.50
    elif scheme == "solver_none":
        w[overlap_oof[idx]] *= 0.03
        w[solver_only_oof[idx]] *= 1.00
        w[none_oof[idx]] *= 1.00
    else:
        raise ValueError(scheme)

    return w

test_tier_rates = {tier: float(mask.mean()) for tier, mask in tier_masks_test.items()}

def tier_weighted_mse_53b(pred):
    total = 0.0
    for tier, mask in tier_masks_oof.items():
        total += test_tier_rates[tier] * float(mean_squared_error(y53b[mask], pred[mask]))
    return total

def candidate_from_delta(delta_oof, delta_test, cap):
    delta_oof = np.clip(delta_oof.astype(np.float64), -cap, cap)
    delta_test = np.clip(delta_test.astype(np.float64), -cap, cap)

    cand_oof = arc_to_pct_53b(z_50b_oof.astype(np.float64) + delta_oof)
    cand_test = arc_to_pct_53b(z_50b_test.astype(np.float64) + delta_test)

    return cand_oof, cand_test

def best_lambda_nonnegative(cand_oof, mask):
    grid = np.unique(np.concatenate([
        np.linspace(0.0, 1.25, 501),
        np.array([0.0, 0.005, 0.05, 0.10, 0.25, 0.50, 0.75, 1.0])
    ]))

    y = y53b[mask].astype(np.float64)
    b = pred50b_oof[mask].astype(np.float64)
    a = cand_oof[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = float(mean_squared_error(y, b))

    for lam in grid:
        p = np.clip(b + float(lam) * (a - b), 0, 100)
        mse = float(mean_squared_error(y, p))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam

def apply_tier_blend(cand_oof, cand_test, lams, is_test=False):
    if is_test:
        base = pred50b_test.astype(np.float64)
        alt = cand_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred50b_oof.astype(np.float64)
        alt = cand_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

def tier_metrics(pred):
    out = {}
    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y53b[mask], pred[mask]))
        out[f"gain_{tier}_vs_50b"] = (
            float(mean_squared_error(y53b[mask], pred50b_oof[mask]))
            - out[f"mse_{tier}"]
        )
    return out

# ------------------------------------------------------------
# 7. Configs
# ------------------------------------------------------------

configs53b = [
    {
        "name": "lgb53b_te_public_mimic",
        "weight_scheme": "public_mimic",
        "encoding_smoothing": 300.0,
        "delta_cap": 0.20,
        "params": {
            "n_estimators": 350,
            "learning_rate": 0.025,
            "num_leaves": 31,
            "min_child_samples": 1200,
            "subsample": 0.80,
            "subsample_freq": 1,
            "colsample_bytree": 0.80,
            "reg_lambda": 150.0,
        },
    },
    {
        "name": "lgb53b_te_solver_focus",
        "weight_scheme": "solver_focus",
        "encoding_smoothing": 500.0,
        "delta_cap": 0.20,
        "params": {
            "n_estimators": 400,
            "learning_rate": 0.020,
            "num_leaves": 31,
            "min_child_samples": 1500,
            "subsample": 0.80,
            "subsample_freq": 1,
            "colsample_bytree": 0.80,
            "reg_lambda": 250.0,
        },
    },
    {
        "name": "lgb53b_te_solver_none",
        "weight_scheme": "solver_none",
        "encoding_smoothing": 500.0,
        "delta_cap": 0.25,
        "params": {
            "n_estimators": 450,
            "learning_rate": 0.020,
            "num_leaves": 31,
            "min_child_samples": 1800,
            "subsample": 0.80,
            "subsample_freq": 1,
            "colsample_bytree": 0.75,
            "reg_lambda": 300.0,
        },
    },
]

with open("model_results/lgb53b_config_manifest.json", "w") as f:
    json.dump(configs53b, f, indent=2)

# ------------------------------------------------------------
# 8. Train
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

delta_oof_by_cfg = {}
delta_test_by_cfg = {}
fold_rows_all = []
direct_rows = []

for cfg_idx, cfg in enumerate(configs53b, start=1):
    name = cfg["name"]

    print("\n" + "=" * 90)
    print(f"Training 53B config {cfg_idx}/{len(configs53b)}: {name}")
    print("=" * 90)

    ckpt_oof = CHECKPOINT_DIR_53B / f"{name}_delta_oof.npy"
    ckpt_test = CHECKPOINT_DIR_53B / f"{name}_delta_test.npy"
    ckpt_folds = CHECKPOINT_DIR_53B / f"{name}_folds.csv"

    if ckpt_oof.exists() and ckpt_test.exists() and ckpt_folds.exists():
        print("Loading checkpoint.")
        delta_oof = np.load(ckpt_oof).astype(np.float32)
        delta_test = np.load(ckpt_test).astype(np.float32)
        fold_df = pd.read_csv(ckpt_folds)
    else:
        delta_oof = np.full(n_train, np.nan, dtype=np.float32)
        delta_test_sum = np.zeros(n_test, dtype=np.float64)
        fold_rows = []

        for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
            X_tr, X_va, X_te = add_oof_encoded_features_for_outer_fold(
                X_base_train.iloc[tr_idx],
                X_base_train.iloc[va_idx],
                X_base_test,
                tr_idx,
                va_idx,
                encoding_cols,
                smoothing=cfg["encoding_smoothing"],
                inner_splits=4,
            )

            w_tr = make_weight_53b(tr_idx, cfg["weight_scheme"])
            w_va = make_weight_53b(va_idx, cfg["weight_scheme"])

            params = dict(cfg["params"])
            params.update({
                "objective": "regression",
                "random_state": RANDOM_STATE + 1000 * cfg_idx + fold_num,
                "n_jobs": 1,
                "verbosity": -1,
                "force_col_wise": True,
                "max_bin": 127,
            })

            model = LGBMRegressor(**params)

            model.fit(
                X_tr,
                resid_z[tr_idx],
                sample_weight=w_tr,
                eval_set=[(X_va, resid_z[va_idx])],
                eval_sample_weight=[w_va],
                eval_metric="l2",
                callbacks=[
                    lgb.early_stopping(stopping_rounds=60, verbose=False),
                    lgb.log_evaluation(period=0),
                ],
            )

            d_va = model.predict(X_va).astype(np.float32)
            d_te = model.predict(X_te).astype(np.float32)

            delta_oof[va_idx] = d_va
            delta_test_sum += d_te.astype(np.float64)

            cand_va = arc_to_pct_53b(
                z_50b_oof[va_idx].astype(np.float64)
                + np.clip(d_va.astype(np.float64), -cfg["delta_cap"], cfg["delta_cap"])
            )

            row = {
                "config": name,
                "fold": fold_num,
                "best_iteration": int(getattr(model, "best_iteration_", 0) or params["n_estimators"]),
                "mse_50b": float(mean_squared_error(y53b[va_idx], pred50b_oof[va_idx])),
                "mse_direct": float(mean_squared_error(y53b[va_idx], cand_va)),
            }
            row["gain_direct_vs_50b"] = row["mse_50b"] - row["mse_direct"]

            for tier, mask_full in tier_masks_oof.items():
                mask = mask_full[va_idx]
                row[f"n_{tier}"] = int(mask.sum())
                if int(mask.sum()) > 0:
                    row[f"mse50b_{tier}"] = float(mean_squared_error(y53b[va_idx][mask], pred50b_oof[va_idx][mask]))
                    row[f"msedir_{tier}"] = float(mean_squared_error(y53b[va_idx][mask], cand_va[mask]))
                    row[f"gain_direct_{tier}_vs_50b"] = row[f"mse50b_{tier}"] - row[f"msedir_{tier}"]

            fold_rows.append(row)

            print(
                f"fold {fold_num} | direct MSE {row['mse_direct']:.6f} "
                f"| gain vs 50B {row['gain_direct_vs_50b']:.6f} "
                f"| best_iter {row['best_iteration']}"
            )

            del model, X_tr, X_va, X_te
            gc.collect()

        delta_test = (delta_test_sum / N_SPLITS).astype(np.float32)
        fold_df = pd.DataFrame(fold_rows)

        np.save(ckpt_oof, delta_oof)
        np.save(ckpt_test, delta_test)
        fold_df.to_csv(ckpt_folds, index=False)

    delta_oof_by_cfg[name] = delta_oof
    delta_test_by_cfg[name] = delta_test
    fold_rows_all.append(fold_df)

    cand_oof, cand_test = candidate_from_delta(delta_oof, delta_test, cfg["delta_cap"])
    direct_mse = float(mean_squared_error(y53b, cand_oof))

    row = {
        "config": name,
        "direct_oof_mse": direct_mse,
        "direct_gain_vs_50b": mse50b - direct_mse,
        "direct_public_weighted_mse": tier_weighted_mse_53b(cand_oof),
        "direct_public_weighted_gain_vs_50b": tier_weighted_mse_53b(pred50b_oof) - tier_weighted_mse_53b(cand_oof),
    }
    row.update(tier_metrics(cand_oof))
    direct_rows.append(row)

    print(f"Direct OOF MSE: {direct_mse:.6f} | gain vs 50B: {mse50b - direct_mse:.6f}")

fold_metrics53b = pd.concat(fold_rows_all, axis=0).reset_index(drop=True)
direct53b = pd.DataFrame(direct_rows).sort_values("direct_oof_mse").reset_index(drop=True)

# ------------------------------------------------------------
# 9. Tier blend screen
# ------------------------------------------------------------

screen_rows = []
store = {}

for cfg in configs53b:
    name = cfg["name"]
    cand_oof, cand_test = candidate_from_delta(delta_oof_by_cfg[name], delta_test_by_cfg[name], cfg["delta_cap"])

    lam_solver = best_lambda_nonnegative(cand_oof, solver_only_oof)
    lam_none = best_lambda_nonnegative(cand_oof, none_oof)

    strategies = {
        "solver_only": {"overlap": 0.0, "solver_only": lam_solver, "none": 0.0},
        "none_only": {"overlap": 0.0, "solver_only": 0.0, "none": lam_none},
        "solver_plus_none": {"overlap": 0.0, "solver_only": lam_solver, "none": lam_none},
    }

    for strategy, lams in strategies.items():
        pred_oof = apply_tier_blend(cand_oof, cand_test, lams, is_test=False)
        pred_test = apply_tier_blend(cand_oof, cand_test, lams, is_test=True)

        mse = float(mean_squared_error(y53b, pred_oof))
        weighted = tier_weighted_mse_53b(pred_oof)

        row = {
            "config": name,
            "strategy": strategy,
            "lambda_overlap": lams["overlap"],
            "lambda_solver_only": lams["solver_only"],
            "lambda_none": lams["none"],
            "oof_mse": mse,
            "gain_vs_50b": mse50b - mse,
            "public_weighted_mse": weighted,
            "public_weighted_gain_vs_50b": tier_weighted_mse_53b(pred50b_oof) - weighted,
        }
        row.update(tier_metrics(pred_oof))

        key = f"{name}__{strategy}"
        row["key"] = key
        screen_rows.append(row)
        store[key] = {"oof": pred_oof, "test": pred_test, "lams": lams}

screen53b = pd.DataFrame(screen_rows).sort_values(["public_weighted_mse", "oof_mse"]).reset_index(drop=True)
screen53b_oof = screen53b.sort_values(["oof_mse", "public_weighted_mse"]).reset_index(drop=True)

best_weighted = screen53b.iloc[0]
best_key = best_weighted["key"]
best_oof = store[best_key]["oof"]
best_test = store[best_key]["test"]

# ------------------------------------------------------------
# 10. Fold diagnostics
# ------------------------------------------------------------

fold_diag_rows = []

for fold_num, (_, va_idx) in enumerate(folds, start=1):
    row = {
        "fold": fold_num,
        "mse_50b": float(mean_squared_error(y53b[va_idx], pred50b_oof[va_idx])),
        "mse_53b": float(mean_squared_error(y53b[va_idx], best_oof[va_idx])),
    }
    row["gain_vs_50b"] = row["mse_50b"] - row["mse_53b"]

    for tier, mask_full in tier_masks_oof.items():
        mask = mask_full[va_idx]
        row[f"n_{tier}"] = int(mask.sum())
        if int(mask.sum()) > 0:
            row[f"mse50b_{tier}"] = float(mean_squared_error(y53b[va_idx][mask], pred50b_oof[va_idx][mask]))
            row[f"mse53b_{tier}"] = float(mean_squared_error(y53b[va_idx][mask], best_oof[va_idx][mask]))
            row[f"gain_{tier}_vs_50b"] = row[f"mse50b_{tier}"] - row[f"mse53b_{tier}"]

    fold_diag_rows.append(row)

fold_diag53b = pd.DataFrame(fold_diag_rows)

# ------------------------------------------------------------
# 11. Save
# ------------------------------------------------------------

direct53b.to_csv("model_results/lgb53b_direct_screen.csv", index=False)
screen53b.to_csv("model_results/lgb53b_blend_screen.csv", index=False)
screen53b_oof.to_csv("model_results/lgb53b_blend_screen_oof_sorted.csv", index=False)
fold_metrics53b.to_csv("model_results/lgb53b_fold_metrics.csv", index=False)
fold_diag53b.to_csv("model_results/lgb53b_best_fold_diag.csv", index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y53b,
    "pred_50b": pred50b_oof,
    "pred_clipped": best_oof,
    "tier": tier_oof_arr,
}).to_csv("model_results/oof_lgb53b_best_weighted.csv", index=False)

pd.DataFrame({
    ID_COL: test_ids,
    "pred_50b": pred50b_test,
    TARGET_COL: best_test,
    "tier": tier_test_arr,
}).to_csv("model_results/testpred_lgb53b_best_weighted.csv", index=False)

pd.DataFrame({
    ID_COL: test_ids,
    TARGET_COL: best_test,
}).to_csv("submission_lgb53b_best_weighted.csv", index=False)

# Top 3 submissions for later only.
for i, row in screen53b.head(3).iterrows():
    pd.DataFrame({
        ID_COL: test_ids,
        TARGET_COL: store[row["key"]]["test"],
    }).to_csv(f"submission_lgb53b_top{i+1}.csv", index=False)

# Validate.
for p in ["submission_lgb53b_best_weighted.csv", "submission_lgb53b_top1.csv", "submission_lgb53b_top2.csv", "submission_lgb53b_top3.csv"]:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test, 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[TARGET_COL].notna().all(), p
    assert np.isfinite(sub[TARGET_COL]).all(), p
    assert sub[TARGET_COL].between(0, 100).all(), p

# ------------------------------------------------------------
# 12. Output
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("53B safe LightGBM complete")
print("=" * 90)

print("\nReference")
print("---------")
print(f"50B OOF MSE: {mse50b:.6f} | public 29.448")

print("\nDirect candidates")
print("-----------------")
display(direct53b)

print("\nTop blended candidates by public-weighted OOF")
print("---------------------------------------------")
display(screen53b.head(20))

print("\nTop blended candidates by ordinary OOF")
print("--------------------------------------")
display(screen53b_oof.head(20))

print("\nBest 53B")
print("--------")
print(best_weighted.to_string())

print("\nFold diagnostics")
print("----------------")
print(fold_diag53b.to_string(index=False))
print("Min fold gain vs 50B:", float(fold_diag53b["gain_vs_50b"].min()))

print("\nSaved")
print("-----")
print("model_results/lgb53b_direct_screen.csv")
print("model_results/lgb53b_blend_screen.csv")
print("model_results/lgb53b_best_fold_diag.csv")
print("model_results/oof_lgb53b_best_weighted.csv")
print("model_results/testpred_lgb53b_best_weighted.csv")
print("submission_lgb53b_best_weighted.csv")
print("submission_lgb53b_top1.csv")
print("submission_lgb53b_top2.csv")
print("submission_lgb53b_top3.csv")

print("\nDecision rule")
print("-------------")
gain = float(best_weighted["gain_vs_50b"])
weighted_gain = float(best_weighted["public_weighted_gain_vs_50b"])
min_fold_gain = float(fold_diag53b["gain_vs_50b"].min())
solver_gain = float(best_weighted["gain_solver_only_vs_50b"])
overlap_gain = float(best_weighted["gain_overlap_vs_50b"])

if gain >= 1.0 and weighted_gain >= 0.50 and min_fold_gain >= 0 and solver_gain > 0 and overlap_gain > -0.002:
    print("53B has strong enough tree residual signal to consider a public probe.")
elif gain >= 0.50 and weighted_gain >= 0.25 and solver_gain > 0 and overlap_gain > -0.002:
    print("53B has moderate signal. Inspect carefully; do not submit blindly.")
else:
    print("53B is not strong enough versus public-validated 50B. Keep 50B protected.")

53B. Safe numeric-only LightGBM residual model with OOF encodings

Reference
---------
50B OOF MSE: 51.463158 | public 29.448
overlap      train= 53786 test= 27298 50B MSE=0.506983
solver_only  train= 27807 test= 17807 50B MSE=63.484444
none         train= 63328 test=  3202 50B MSE=89.462982

Base feature shapes
-------------------
X_base_train: (144921, 61)
X_base_test:  (48307, 61)
encoding columns: 16 ['SCHOOL', 'DISTRICT', 'COUNTY', 'REGION', 'DISTRICT_TYPE', 'ASSESSMENT_NAME', 'SUBGROUP_NAME', 'ASSESSMENT_SUBGROUP', 'SCHOOL_ASSESSMENT', 'SCHOOL_SUBGROUP', 'DISTRICT_ASSESSMENT', 'COUNTY_ASSESSMENT', 'N_STUDENTS_BIN', 'SUBJECT_FAMILY', 'SUBJECT_SUBGROUP', 'TIER']

Training 53B config 1/3: lgb53b_te_public_mimic
fold 1 | direct MSE 53.805340 | gain vs 50B -1.877377 | best_iter 107
fold 2 | direct MSE 50.554596 | gain vs 50B -1.073002 | best_iter 29
fold 3 | direct MSE 53.933765 | gain vs 50B -1.687370 | best_iter 100
fold 4 | direct MSE 53.780788 | gain vs 50B -2.310841 | best_iter 1

,config,direct_oof_mse,direct_gain_vs_50b,direct_public_weighted_mse,direct_public_weighted_gain_vs_50b,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b
0,lgb53b_te_solver_none,53.100510,-1.637352,31.019613,-1.401389,2.012019,-1.505036,64.625084,-1.140640,91.430824,-1.967842
1,lgb53b_te_public_mimic,53.211666,-1.748508,30.807774,-1.189549,1.523118,-1.016135,64.684143,-1.199699,92.074493,-2.611511
2,lgb53b_te_solver_focus,53.337017,-1.873859,30.795879,-1.177655,1.544819,-1.037836,64.560585,-1.076141,92.397171,-2.934189



Top blended candidates by public-weighted OOF
---------------------------------------------


,config,strategy,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b,public_weighted_mse,public_weighted_gain_vs_50b,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b,key
0,lgb53b_te_solver_none,solver_only,0.0,0.1025,0.0,51.460320,0.002838,29.612777,0.005448,0.506983,0.0,63.469666,0.014778,89.462982,0.0,lgb53b_te_solver_none__solver_only
1,lgb53b_te_solver_none,solver_plus_none,0.0,0.1025,0.0,51.460320,0.002838,29.612777,0.005448,0.506983,0.0,63.469666,0.014778,89.462982,0.0,lgb53b_te_solver_none__solver_plus_none
2,lgb53b_te_public_mimic,solver_only,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_public_mimic__solver_only
3,lgb53b_te_public_mimic,none_only,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_public_mimic__none_only
4,lgb53b_te_public_mimic,solver_plus_none,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_public_mimic__solver_plus_none
5,lgb53b_te_solver_focus,solver_only,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_solver_focus__solver_only
6,lgb53b_te_solver_focus,none_only,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_solver_focus__none_only
7,lgb53b_te_solver_focus,solver_plus_none,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_solver_focus__solver_plus_none
8,lgb53b_te_solver_none,none_only,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_solver_none__none_only



Top blended candidates by ordinary OOF
--------------------------------------


,config,strategy,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b,public_weighted_mse,public_weighted_gain_vs_50b,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b,key
0,lgb53b_te_solver_none,solver_only,0.0,0.1025,0.0,51.460320,0.002838,29.612777,0.005448,0.506983,0.0,63.469666,0.014778,89.462982,0.0,lgb53b_te_solver_none__solver_only
1,lgb53b_te_solver_none,solver_plus_none,0.0,0.1025,0.0,51.460320,0.002838,29.612777,0.005448,0.506983,0.0,63.469666,0.014778,89.462982,0.0,lgb53b_te_solver_none__solver_plus_none
2,lgb53b_te_public_mimic,solver_only,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_public_mimic__solver_only
3,lgb53b_te_public_mimic,none_only,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_public_mimic__none_only
4,lgb53b_te_public_mimic,solver_plus_none,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_public_mimic__solver_plus_none
5,lgb53b_te_solver_focus,solver_only,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_solver_focus__solver_only
6,lgb53b_te_solver_focus,none_only,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_solver_focus__none_only
7,lgb53b_te_solver_focus,solver_plus_none,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_solver_focus__solver_plus_none
8,lgb53b_te_solver_none,none_only,0.0,0.0000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,lgb53b_te_solver_none__none_only



Best 53B
--------
config                                      lgb53b_te_solver_none
strategy                                              solver_only
lambda_overlap                                                0.0
lambda_solver_only                                         0.1025
lambda_none                                                   0.0
oof_mse                                                  51.46032
gain_vs_50b                                              0.002838
public_weighted_mse                                     29.612777
public_weighted_gain_vs_50b                              0.005448
mse_overlap                                              0.506983
gain_overlap_vs_50b                                           0.0
mse_solver_only                                         63.469666
gain_solver_only_vs_50b                                  0.014778
mse_none                                                89.462982
gain_none_vs_50b                                         

In [6]:
# ============================================================
# Cleanup failed 54A partial checkpoints
# ============================================================

from pathlib import Path
import os

ckpt = Path("model_results/binom54a_checkpoints")

if ckpt.exists():
    print("54A checkpoint files:")
    for p in sorted(ckpt.glob("*")):
        print(" ", p)

    # Only remove partial 54A files. This does NOT touch 50B/50C artifacts.
    for p in sorted(ckpt.glob("binom54a_*")):
        print("Removing:", p)
        p.unlink()

    print("54A checkpoint cleanup complete.")
else:
    print("No 54A checkpoint directory found.")

54A checkpoint files:
54A checkpoint cleanup complete.


In [7]:
# ============================================================
# 55A. Binned residual tables / empirical-Bayes smoothing
# ============================================================
#
# Current protected public model:
#   submission_mf50b_best_oof.csv
#   public MSE = 29.448
#
# Why this branch:
#   - 53B LightGBM residual model was effectively noise.
#   - 52A ALS gave only a small anti-signal-style OOF gain.
#   - This branch is simple, interpretable, and kernel-safe.
#
# Model:
#   Work on arcsine residual from 50B:
#
#       r_i = arcsin(sqrt(y_i / 100)) - arcsin(sqrt(pred50B_i / 100))
#
#   Estimate residual means in binned/categorical cells using hierarchical
#   empirical-Bayes smoothing:
#
#       global/tier prior
#       -> tier × assessment-subgroup
#       -> tier × assessment-subgroup × N_STUDENTS_BIN
#       -> etc.
#
#   Then:
#
#       candidate = 100 * sin(arcsin_sqrt(pred50B) + smoothed_residual)^2
#
# Final:
#   Protected blend back into 50B:
#       overlap unchanged
#       solver_only / none get nonnegative OOF-tuned lambdas
#
# No Torch. No LightGBM. No native categorical code.
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

Path("model_results").mkdir(exist_ok=True)

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5

print("=" * 90)
print("55A. Binned residual tables / empirical-Bayes smoothing")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required raw data
# ------------------------------------------------------------

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise RuntimeError(
        "raw_train_te/raw_test_te missing. Run your setup cells first:\n"
        "1. # 0. Setup and raw data loading\n"
        "2. # 1. Merge datasets\n"
        "3. # 4. Build X and y + preserve IDs\n"
        "then raw_train_te/raw_test_te aliases."
    )

print("\nRaw data")
print("--------")
print("raw_train_te:", raw_train_te.shape)
print("raw_test_te: ", raw_test_te.shape)

# ------------------------------------------------------------
# 2. Artifact helpers
# ------------------------------------------------------------

def load_oof_55a(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column found in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)


def load_test_55a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if len(numeric_cols) == 0:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column and no test_ids available.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

required_files_55a = [
    "model_results/oof_mf50b_best_oof.csv",
    "model_results/testpred_mf50b_best_oof.csv",
    "model_results/oof_hybrid44b_best.csv",
    "model_results/testpred_hybrid44b_best.csv",
    "model_results/account39a_raw_oof_reconstruction.csv",
    "model_results/account39a_raw_test_reconstruction.csv",
    "model_results/account39b_solver_raw_oof.csv",
    "model_results/account39b_solver_raw_test.csv",
]

print("\nArtifact checks")
print("---------------")
for p in required_files_55a:
    print(f"{p:65s}", Path(p).exists())

missing = [p for p in required_files_55a if not Path(p).exists()]
if missing:
    raise RuntimeError(f"Missing required artifacts: {missing}")

# ------------------------------------------------------------
# 3. Load 50B and 44B
# ------------------------------------------------------------

_, y55_file, pred50b_oof = load_oof_55a("model_results/oof_mf50b_best_oof.csv")
_, test_ids55a, pred50b_test = load_test_55a("model_results/testpred_mf50b_best_oof.csv")

_, y44_file, pred44b_oof = load_oof_55a("model_results/oof_hybrid44b_best.csv")
_, _, pred44b_test = load_test_55a("model_results/testpred_hybrid44b_best.csv")

if "y_train" in globals():
    y55a = np.asarray(y_train, dtype=np.float32).reshape(-1)
else:
    y55a = y55_file.copy()

assert len(y55a) == len(pred50b_oof)
assert len(raw_train_te) == len(pred50b_oof)
assert len(raw_test_te) == len(pred50b_test)
assert np.max(np.abs(y55a - y55_file)) < 1e-5
assert np.max(np.abs(y55a - y44_file)) < 1e-5

n_train = len(y55a)
n_test = len(pred50b_test)

mse50b = float(mean_squared_error(y55a, pred50b_oof))
mse44b = float(mean_squared_error(y55a, pred44b_oof))

print("\nReference")
print("---------")
print(f"50B OOF MSE: {mse50b:.6f} | public 29.448")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")

# ------------------------------------------------------------
# 4. Reconstruct tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_oof = np.array(["none"] * n_train, dtype=object)
tier_oof[solver_only_oof] = "solver_only"
tier_oof[overlap_oof] = "overlap"

tier_test = np.array(["none"] * n_test, dtype=object)
tier_test[solver_only_test] = "solver_only"
tier_test[overlap_test] = "overlap"

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

test_tier_rates = {tier: float(mask.mean()) for tier, mask in tier_masks_test.items()}

print("\nTier coverage")
print("-------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"50B tier MSE={mean_squared_error(y55a[mask], pred50b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 5. Transform target and residual
# ------------------------------------------------------------

def pct_to_arc_55a(pct):
    p = np.clip(np.asarray(pct, dtype=np.float64) / 100.0, 1e-6, 1 - 1e-6)
    return np.arcsin(np.sqrt(p)).astype(np.float32)

def arc_to_pct_55a(z):
    p = np.sin(np.asarray(z, dtype=np.float64)) ** 2
    return np.clip(100.0 * p, 0, 100).astype(np.float32)

z_y = pct_to_arc_55a(y55a)
z_50b_oof = pct_to_arc_55a(pred50b_oof)
z_50b_test = pct_to_arc_55a(pred50b_test)

resid_z = (z_y - z_50b_oof).astype(np.float32)

print("\n50B arcsine residual summary")
print("----------------------------")
print(pd.Series(resid_z).describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).to_string())

# ------------------------------------------------------------
# 6. Build categorical/binned key frames
# ------------------------------------------------------------

def clean_str_55a(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)

def safe_num_55a(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)

def n_bin_55a(x):
    return pd.cut(
        pd.Series(x).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, 200, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", "101-200", ">200"],
    ).astype("string").fillna("<NA>").astype(str)

def subject_family_55a(x):
    s = str(x).lower()
    if "math" in s or "algebra" in s or "geometry" in s:
        return "math"
    if "ela" in s or "english" in s:
        return "ela"
    if "science" in s or "chemistry" in s or "physics" in s or "biology" in s or "earth" in s:
        return "science"
    if "history" in s or "global" in s or "social" in s or "government" in s:
        return "social"
    return "other"

def make_key_frame_55a(raw_df, tier_arr):
    out = pd.DataFrame(index=np.arange(len(raw_df)))

    for c in ["SCHOOL", "DISTRICT", "COUNTY", "REGION", "DISTRICT_TYPE", "ASSESSMENT_NAME", "SUBGROUP_NAME"]:
        if c in raw_df.columns:
            out[c] = clean_str_55a(raw_df[c]).to_numpy()
        else:
            out[c] = "<NA>"

    if "N_STUDENTS" in raw_df.columns:
        n = safe_num_55a(raw_df["N_STUDENTS"])
    else:
        n = np.full(len(raw_df), np.nan)

    out["N_STUDENTS_BIN"] = n_bin_55a(n)
    out["TIER"] = pd.Series(tier_arr).astype(str).to_numpy()

    out["ASSESSMENT_SUBGROUP"] = out["ASSESSMENT_NAME"].astype(str) + "||" + out["SUBGROUP_NAME"].astype(str)
    out["SUBJECT_FAMILY"] = pd.Series(out["ASSESSMENT_NAME"]).map(subject_family_55a).astype(str).to_numpy()
    out["SUBJECT_SUBGROUP"] = out["SUBJECT_FAMILY"].astype(str) + "||" + out["SUBGROUP_NAME"].astype(str)

    # Tiered keys.
    out["TIER_ASG"] = out["TIER"].astype(str) + "||" + out["ASSESSMENT_SUBGROUP"].astype(str)
    out["TIER_ASG_NBIN"] = out["TIER_ASG"].astype(str) + "||" + out["N_STUDENTS_BIN"].astype(str)

    out["TIER_SUBJECT_SUBGROUP"] = out["TIER"].astype(str) + "||" + out["SUBJECT_SUBGROUP"].astype(str)
    out["TIER_SUBJECT_SUBGROUP_NBIN"] = out["TIER_SUBJECT_SUBGROUP"].astype(str) + "||" + out["N_STUDENTS_BIN"].astype(str)

    out["TIER_SUBGROUP_NBIN"] = out["TIER"].astype(str) + "||" + out["SUBGROUP_NAME"].astype(str) + "||" + out["N_STUDENTS_BIN"].astype(str)
    out["TIER_ASSESSMENT_NBIN"] = out["TIER"].astype(str) + "||" + out["ASSESSMENT_NAME"].astype(str) + "||" + out["N_STUDENTS_BIN"].astype(str)

    out["TIER_DISTRICT_TYPE_ASG"] = out["TIER"].astype(str) + "||" + out["DISTRICT_TYPE"].astype(str) + "||" + out["ASSESSMENT_SUBGROUP"].astype(str)
    out["TIER_DISTRICT_TYPE_ASG_NBIN"] = out["TIER_DISTRICT_TYPE_ASG"].astype(str) + "||" + out["N_STUDENTS_BIN"].astype(str)

    out["TIER_COUNTY_ASG"] = out["TIER"].astype(str) + "||" + out["COUNTY"].astype(str) + "||" + out["ASSESSMENT_SUBGROUP"].astype(str)
    out["TIER_DISTRICT_ASG"] = out["TIER"].astype(str) + "||" + out["DISTRICT"].astype(str) + "||" + out["ASSESSMENT_SUBGROUP"].astype(str)

    out["TIER_COUNTY_SUBJECT_SUBGROUP"] = out["TIER"].astype(str) + "||" + out["COUNTY"].astype(str) + "||" + out["SUBJECT_SUBGROUP"].astype(str)

    return out

keys_train = make_key_frame_55a(raw_train_te, tier_oof)
keys_test = make_key_frame_55a(raw_test_te, tier_test)

print("\nKey frame")
print("---------")
print("keys_train:", keys_train.shape)
print("keys_test: ", keys_test.shape)

# ------------------------------------------------------------
# 7. Empirical-Bayes hierarchy functions
# ------------------------------------------------------------

def make_sample_weight_55a(idx, scheme):
    idx = np.asarray(idx, dtype=np.int64)

    if scheme == "uniform":
        w = np.ones(len(idx), dtype=np.float64)

    elif scheme == "public_mimic":
        w = np.ones(len(idx), dtype=np.float64)
        w[overlap_oof[idx]] *= 0.05
        w[solver_only_oof[idx]] *= 1.25
        w[none_oof[idx]] *= 0.35

    elif scheme == "solver_none":
        w = np.ones(len(idx), dtype=np.float64)
        w[overlap_oof[idx]] *= 0.03
        w[solver_only_oof[idx]] *= 1.00
        w[none_oof[idx]] *= 1.00

    elif scheme == "solver_focus":
        w = np.ones(len(idx), dtype=np.float64)
        w[overlap_oof[idx]] *= 0.03
        w[solver_only_oof[idx]] *= 1.50
        w[none_oof[idx]] *= 0.30

    else:
        raise ValueError(f"Unknown weight scheme: {scheme}")

    w = w / max(np.mean(w), 1e-12)
    return w.astype(np.float64)


def map_post_55a(key_values, post_series):
    mapped = pd.Series(key_values.astype(str)).map(post_series)
    return mapped.to_numpy(dtype=np.float64)


def fit_apply_hierarchy_55a(tr_idx, va_idx, cfg):
    """
    Fit hierarchical EB residual corrections on tr_idx.
    Apply to va_idx and test.
    """
    tr_idx = np.asarray(tr_idx, dtype=np.int64)
    va_idx = np.asarray(va_idx, dtype=np.int64)

    w = make_sample_weight_55a(tr_idx, cfg["weight_scheme"])
    r = resid_z[tr_idx].astype(np.float64)

    global_prior = float(np.sum(w * r) / max(np.sum(w), 1e-12))

    prior_tr = np.full(len(tr_idx), global_prior, dtype=np.float64)
    prior_va = np.full(len(va_idx), global_prior, dtype=np.float64)
    prior_te = np.full(n_test, global_prior, dtype=np.float64)

    fit_rows = []

    for level in cfg["levels"]:
        key = level["key"]
        smooth = float(level["smooth"])
        min_rows = int(level["min_rows"])

        key_tr = keys_train.iloc[tr_idx][key].astype(str).to_numpy()
        key_va = keys_train.iloc[va_idx][key].astype(str).to_numpy()
        key_te = keys_test[key].astype(str).to_numpy()

        df = pd.DataFrame({
            "key": key_tr,
            "resid": r,
            "prior": prior_tr,
            "w": w,
        })
        df["wr"] = df["w"] * df["resid"]
        df["wp"] = df["w"] * df["prior"]

        stats = (
            df.groupby("key", sort=False)
            .agg(
                n=("resid", "size"),
                sum_w=("w", "sum"),
                sum_wr=("wr", "sum"),
                sum_wp=("wp", "sum"),
            )
        )

        stats = stats[stats["sum_w"] > 0].copy()
        stats["mean_resid"] = stats["sum_wr"] / stats["sum_w"]
        stats["mean_prior"] = stats["sum_wp"] / stats["sum_w"]
        stats["alpha"] = stats["sum_w"] / (stats["sum_w"] + smooth)
        stats["post"] = stats["mean_prior"] + stats["alpha"] * (stats["mean_resid"] - stats["mean_prior"])

        # Do not use too-small cells.
        stats_use = stats[stats["n"] >= min_rows].copy()

        fit_rows.append({
            "level": key,
            "smooth": smooth,
            "min_rows": min_rows,
            "n_groups_all": int(len(stats)),
            "n_groups_used": int(len(stats_use)),
            "median_group_n_used": float(stats_use["n"].median()) if len(stats_use) else np.nan,
            "max_group_n_used": int(stats_use["n"].max()) if len(stats_use) else 0,
        })

        if len(stats_use) == 0:
            continue

        post = stats_use["post"]

        tr_mapped = map_post_55a(key_tr, post)
        va_mapped = map_post_55a(key_va, post)
        te_mapped = map_post_55a(key_te, post)

        prior_tr = np.where(np.isfinite(tr_mapped), tr_mapped, prior_tr)
        prior_va = np.where(np.isfinite(va_mapped), va_mapped, prior_va)
        prior_te = np.where(np.isfinite(te_mapped), te_mapped, prior_te)

    return prior_va.astype(np.float32), prior_te.astype(np.float32), pd.DataFrame(fit_rows)

# ------------------------------------------------------------
# 8. Configs
# ------------------------------------------------------------

configs55a = [
    {
        "name": "eb55a_asg_nbin_public",
        "weight_scheme": "public_mimic",
        "resid_cap": 0.20,
        "levels": [
            {"key": "TIER", "smooth": 100.0, "min_rows": 1},
            {"key": "TIER_ASG", "smooth": 300.0, "min_rows": 100},
            {"key": "TIER_ASG_NBIN", "smooth": 500.0, "min_rows": 150},
        ],
    },
    {
        "name": "eb55a_subject_nbin_public",
        "weight_scheme": "public_mimic",
        "resid_cap": 0.20,
        "levels": [
            {"key": "TIER", "smooth": 100.0, "min_rows": 1},
            {"key": "TIER_SUBJECT_SUBGROUP", "smooth": 250.0, "min_rows": 100},
            {"key": "TIER_SUBJECT_SUBGROUP_NBIN", "smooth": 500.0, "min_rows": 150},
        ],
    },
    {
        "name": "eb55a_subgroup_assessment_nbin_solver",
        "weight_scheme": "solver_focus",
        "resid_cap": 0.18,
        "levels": [
            {"key": "TIER", "smooth": 150.0, "min_rows": 1},
            {"key": "TIER_SUBGROUP_NBIN", "smooth": 400.0, "min_rows": 150},
            {"key": "TIER_ASSESSMENT_NBIN", "smooth": 600.0, "min_rows": 150},
        ],
    },
    {
        "name": "eb55a_district_type_asg_public",
        "weight_scheme": "public_mimic",
        "resid_cap": 0.20,
        "levels": [
            {"key": "TIER", "smooth": 100.0, "min_rows": 1},
            {"key": "TIER_ASG", "smooth": 300.0, "min_rows": 100},
            {"key": "TIER_DISTRICT_TYPE_ASG", "smooth": 800.0, "min_rows": 250},
            {"key": "TIER_DISTRICT_TYPE_ASG_NBIN", "smooth": 1000.0, "min_rows": 300},
        ],
    },
    {
        "name": "eb55a_county_asg_conservative",
        "weight_scheme": "solver_none",
        "resid_cap": 0.15,
        "levels": [
            {"key": "TIER", "smooth": 200.0, "min_rows": 1},
            {"key": "TIER_ASG", "smooth": 500.0, "min_rows": 150},
            {"key": "TIER_COUNTY_ASG", "smooth": 1500.0, "min_rows": 400},
        ],
    },
    {
        "name": "eb55a_district_asg_very_conservative",
        "weight_scheme": "solver_none",
        "resid_cap": 0.12,
        "levels": [
            {"key": "TIER", "smooth": 300.0, "min_rows": 1},
            {"key": "TIER_ASG", "smooth": 700.0, "min_rows": 200},
            {"key": "TIER_DISTRICT_ASG", "smooth": 2500.0, "min_rows": 800},
        ],
    },
    {
        "name": "eb55a_county_subject_public",
        "weight_scheme": "public_mimic",
        "resid_cap": 0.15,
        "levels": [
            {"key": "TIER", "smooth": 150.0, "min_rows": 1},
            {"key": "TIER_SUBJECT_SUBGROUP", "smooth": 300.0, "min_rows": 100},
            {"key": "TIER_COUNTY_SUBJECT_SUBGROUP", "smooth": 1200.0, "min_rows": 300},
        ],
    },
]

with open("model_results/eb55a_config_manifest.json", "w") as f:
    json.dump(configs55a, f, indent=2)

print("\n55A configs")
print("-----------")
for cfg in configs55a:
    print(cfg["name"], "| weight:", cfg["weight_scheme"], "| levels:", [x["key"] for x in cfg["levels"]])

# ------------------------------------------------------------
# 9. OOF train/apply
# ------------------------------------------------------------

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

corr_oof_by_cfg = {}
corr_test_by_cfg = {}
fit_detail_frames = []
direct_rows = []

for cfg_idx, cfg in enumerate(configs55a, start=1):
    name = cfg["name"]

    print("\n" + "=" * 90)
    print(f"55A config {cfg_idx}/{len(configs55a)}: {name}")
    print("=" * 90)

    corr_oof = np.full(n_train, np.nan, dtype=np.float32)
    corr_test_sum = np.zeros(n_test, dtype=np.float64)

    fold_fit_details = []

    for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
        corr_va, corr_te, fit_detail = fit_apply_hierarchy_55a(tr_idx, va_idx, cfg)

        corr_oof[va_idx] = corr_va
        corr_test_sum += corr_te.astype(np.float64)

        fit_detail.insert(0, "fold", fold_num)
        fit_detail.insert(0, "config", name)
        fold_fit_details.append(fit_detail)

        cand_va = arc_to_pct_55a(
            z_50b_oof[va_idx].astype(np.float64)
            + np.clip(corr_va.astype(np.float64), -cfg["resid_cap"], cfg["resid_cap"])
        )

        fold_gain = float(mean_squared_error(y55a[va_idx], pred50b_oof[va_idx]) - mean_squared_error(y55a[va_idx], cand_va))

        print(
            f"fold {fold_num} direct candidate gain vs 50B: {fold_gain:.6f} "
            f"| corr mean={float(np.mean(corr_va)):.6f} std={float(np.std(corr_va)):.6f}"
        )

    corr_test = (corr_test_sum / N_SPLITS).astype(np.float32)

    corr_oof_by_cfg[name] = corr_oof
    corr_test_by_cfg[name] = corr_test

    fit_details_cfg = pd.concat(fold_fit_details, axis=0).reset_index(drop=True)
    fit_detail_frames.append(fit_details_cfg)

    direct_oof = arc_to_pct_55a(
        z_50b_oof.astype(np.float64)
        + np.clip(corr_oof.astype(np.float64), -cfg["resid_cap"], cfg["resid_cap"])
    )
    direct_test = arc_to_pct_55a(
        z_50b_test.astype(np.float64)
        + np.clip(corr_test.astype(np.float64), -cfg["resid_cap"], cfg["resid_cap"])
    )

    direct_mse = float(mean_squared_error(y55a, direct_oof))

    row = {
        "config": name,
        "direct_oof_mse": direct_mse,
        "direct_gain_vs_50b": mse50b - direct_mse,
        "direct_public_weighted_mse": None,
        "resid_cap": cfg["resid_cap"],
        "corr_oof_mean": float(np.mean(corr_oof)),
        "corr_oof_std": float(np.std(corr_oof)),
        "corr_test_mean": float(np.mean(corr_test)),
        "corr_test_std": float(np.std(corr_test)),
    }

    for tier, mask in tier_masks_oof.items():
        row[f"direct_mse_{tier}"] = float(mean_squared_error(y55a[mask], direct_oof[mask]))
        row[f"direct_gain_{tier}_vs_50b"] = (
            float(mean_squared_error(y55a[mask], pred50b_oof[mask]))
            - row[f"direct_mse_{tier}"]
        )

    # Fill public-weighted from tier MSE.
    row["direct_public_weighted_mse"] = sum(
        test_tier_rates[tier] * row[f"direct_mse_{tier}"]
        for tier in ["overlap", "solver_only", "none"]
    )
    row["direct_public_weighted_gain_vs_50b"] = (
        sum(test_tier_rates[tier] * float(mean_squared_error(y55a[tier_masks_oof[tier]], pred50b_oof[tier_masks_oof[tier]])) for tier in ["overlap", "solver_only", "none"])
        - row["direct_public_weighted_mse"]
    )

    direct_rows.append(row)

    print(f"Direct OOF MSE: {direct_mse:.6f} | gain vs 50B: {mse50b - direct_mse:.6f}")

fit_details55a = pd.concat(fit_detail_frames, axis=0).reset_index(drop=True)
direct55a = pd.DataFrame(direct_rows).sort_values("direct_oof_mse").reset_index(drop=True)

# ------------------------------------------------------------
# 10. Protected tier blending
# ------------------------------------------------------------

lambda_grid = np.unique(np.concatenate([
    np.linspace(0.0, 1.5, 601),
    np.array([0.0, 0.005, 0.05, 0.10, 0.25, 0.50, 0.75, 1.0])
]))

def best_lambda_nonneg_55a(candidate_oof, mask):
    if int(mask.sum()) == 0:
        return 0.0

    y = y55a[mask].astype(np.float64)
    base = pred50b_oof[mask].astype(np.float64)
    alt = candidate_oof[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = float(mean_squared_error(y, base))

    for lam in lambda_grid:
        pred = np.clip(base + float(lam) * (alt - base), 0, 100)
        mse = float(mean_squared_error(y, pred))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam

def make_candidate_55a(name):
    cfg = next(x for x in configs55a if x["name"] == name)
    cap = float(cfg["resid_cap"])

    cand_oof = arc_to_pct_55a(
        z_50b_oof.astype(np.float64)
        + np.clip(corr_oof_by_cfg[name].astype(np.float64), -cap, cap)
    )
    cand_test = arc_to_pct_55a(
        z_50b_test.astype(np.float64)
        + np.clip(corr_test_by_cfg[name].astype(np.float64), -cap, cap)
    )
    return cand_oof, cand_test

def apply_blend_55a(cand_oof, cand_test, lams, is_test=False):
    if is_test:
        base = pred50b_test.astype(np.float64)
        alt = cand_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred50b_oof.astype(np.float64)
        alt = cand_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

def tier_weighted_mse_55a(pred):
    total = 0.0
    for tier, mask in tier_masks_oof.items():
        total += test_tier_rates[tier] * float(mean_squared_error(y55a[mask], pred[mask]))
    return float(total)

def tier_metrics_55a(pred):
    out = {}
    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y55a[mask], pred[mask]))
        out[f"gain_{tier}_vs_50b"] = (
            float(mean_squared_error(y55a[mask], pred50b_oof[mask]))
            - out[f"mse_{tier}"]
        )
    return out

screen_rows = []
store = {}

for cfg in configs55a:
    name = cfg["name"]
    cand_oof, cand_test = make_candidate_55a(name)

    lam_solver = best_lambda_nonneg_55a(cand_oof, solver_only_oof)
    lam_none = best_lambda_nonneg_55a(cand_oof, none_oof)

    strategies = {
        "solver_only": {"overlap": 0.0, "solver_only": lam_solver, "none": 0.0},
        "none_only": {"overlap": 0.0, "solver_only": 0.0, "none": lam_none},
        "solver_plus_none": {"overlap": 0.0, "solver_only": lam_solver, "none": lam_none},
    }

    for strategy, lams in strategies.items():
        pred_oof = apply_blend_55a(cand_oof, cand_test, lams, is_test=False)
        pred_test = apply_blend_55a(cand_oof, cand_test, lams, is_test=True)

        mse = float(mean_squared_error(y55a, pred_oof))
        weighted = tier_weighted_mse_55a(pred_oof)

        row = {
            "config": name,
            "strategy": strategy,
            "lambda_overlap": lams["overlap"],
            "lambda_solver_only": lams["solver_only"],
            "lambda_none": lams["none"],
            "oof_mse": mse,
            "gain_vs_50b": mse50b - mse,
            "public_weighted_mse": weighted,
            "public_weighted_gain_vs_50b": tier_weighted_mse_55a(pred50b_oof) - weighted,
        }
        row.update(tier_metrics_55a(pred_oof))

        key = f"{name}__{strategy}"
        row["key"] = key
        screen_rows.append(row)
        store[key] = {"oof": pred_oof, "test": pred_test, "lams": lams}

screen55a = pd.DataFrame(screen_rows).sort_values(["public_weighted_mse", "oof_mse"]).reset_index(drop=True)
screen55a_oof = screen55a.sort_values(["oof_mse", "public_weighted_mse"]).reset_index(drop=True)

best_weighted = screen55a.iloc[0]
best_weighted_key = best_weighted["key"]
best_weighted_oof = store[best_weighted_key]["oof"]
best_weighted_test = store[best_weighted_key]["test"]

best_oof = screen55a_oof.iloc[0]
best_oof_key = best_oof["key"]
best_oof_pred = store[best_oof_key]["oof"]
best_oof_test = store[best_oof_key]["test"]

# ------------------------------------------------------------
# 11. Fold diagnostics
# ------------------------------------------------------------

def fold_diag_55a(pred, label):
    rows = []

    for fold_num, (_, va_idx) in enumerate(folds, start=1):
        row = {
            "label": label,
            "fold": fold_num,
            "mse_50b": float(mean_squared_error(y55a[va_idx], pred50b_oof[va_idx])),
            "mse_candidate": float(mean_squared_error(y55a[va_idx], pred[va_idx])),
        }
        row["gain_vs_50b"] = row["mse_50b"] - row["mse_candidate"]

        for tier, mask_full in tier_masks_oof.items():
            mask = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask.sum())
            if int(mask.sum()) > 0:
                row[f"mse50b_{tier}"] = float(mean_squared_error(y55a[va_idx][mask], pred50b_oof[va_idx][mask]))
                row[f"msecand_{tier}"] = float(mean_squared_error(y55a[va_idx][mask], pred[va_idx][mask]))
                row[f"gain_{tier}_vs_50b"] = row[f"mse50b_{tier}"] - row[f"msecand_{tier}"]

        rows.append(row)

    return pd.DataFrame(rows)

fold_weighted55a = fold_diag_55a(best_weighted_oof, "best_weighted")
fold_oof55a = fold_diag_55a(best_oof_pred, "best_oof")
fold_diag55a = pd.concat([fold_weighted55a, fold_oof55a], axis=0).reset_index(drop=True)

# ------------------------------------------------------------
# 12. Save artifacts
# ------------------------------------------------------------

direct_path = "model_results/eb55a_direct_screen.csv"
screen_path = "model_results/eb55a_blend_screen.csv"
screen_oof_path = "model_results/eb55a_blend_screen_oof_sorted.csv"
fit_details_path = "model_results/eb55a_fit_details.csv"
fold_diag_path = "model_results/eb55a_best_fold_diag.csv"

oof_weighted_path = "model_results/oof_eb55a_best_weighted.csv"
test_weighted_path = "model_results/testpred_eb55a_best_weighted.csv"
submission_weighted_path = "submission_eb55a_best_weighted.csv"

oof_oof_path = "model_results/oof_eb55a_best_oof.csv"
test_oof_path = "model_results/testpred_eb55a_best_oof.csv"
submission_oof_path = "submission_eb55a_best_oof.csv"

direct55a.to_csv(direct_path, index=False)
screen55a.to_csv(screen_path, index=False)
screen55a_oof.to_csv(screen_oof_path, index=False)
fit_details55a.to_csv(fit_details_path, index=False)
fold_diag55a.to_csv(fold_diag_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y55a,
    "pred_50b": pred50b_oof,
    "pred_clipped": best_weighted_oof,
    "tier": tier_oof,
}).to_csv(oof_weighted_path, index=False)

pd.DataFrame({
    ID_COL: test_ids55a,
    "pred_50b": pred50b_test,
    TARGET_COL: best_weighted_test,
    "tier": tier_test,
}).to_csv(test_weighted_path, index=False)

pd.DataFrame({
    ID_COL: test_ids55a,
    TARGET_COL: best_weighted_test,
}).to_csv(submission_weighted_path, index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y55a,
    "pred_50b": pred50b_oof,
    "pred_clipped": best_oof_pred,
    "tier": tier_oof,
}).to_csv(oof_oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids55a,
    "pred_50b": pred50b_test,
    TARGET_COL: best_oof_test,
    "tier": tier_test,
}).to_csv(test_oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids55a,
    TARGET_COL: best_oof_test,
}).to_csv(submission_oof_path, index=False)

top_paths = []
for i, row in screen55a.head(5).iterrows():
    key = row["key"]
    out_path = f"submission_eb55a_top{i+1}_weighted.csv"
    pd.DataFrame({
        ID_COL: test_ids55a,
        TARGET_COL: store[key]["test"],
    }).to_csv(out_path, index=False)
    top_paths.append(out_path)

# Validate.
for p in [submission_weighted_path, submission_oof_path] + top_paths:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test, 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[ID_COL].notna().all(), p
    assert sub[TARGET_COL].notna().all(), p
    assert np.isfinite(sub[TARGET_COL]).all(), p
    assert sub[TARGET_COL].between(0, 100).all(), p

# ------------------------------------------------------------
# 13. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("55A empirical-Bayes binned residual model complete")
print("=" * 90)

print("\nReference")
print("---------")
print(f"50B OOF MSE: {mse50b:.6f} | public 29.448")

print("\nDirect EB candidates")
print("--------------------")
display(direct55a)

print("\nTop blended candidates by public-weighted OOF")
print("---------------------------------------------")
display(screen55a.head(25))

print("\nTop blended candidates by ordinary OOF")
print("--------------------------------------")
display(screen55a_oof.head(25))

print("\nBest public-weighted candidate")
print("------------------------------")
print(best_weighted.to_string())

print("\nBest ordinary OOF candidate")
print("---------------------------")
print(best_oof.to_string())

print("\nFold diagnostics")
print("----------------")
print(fold_diag55a.to_string(index=False))

print("\nMin fold gains")
print("--------------")
for label, fd in [("best_weighted", fold_weighted55a), ("best_oof", fold_oof55a)]:
    print(
        f"{label:15s} min_gain={fd['gain_vs_50b'].min(): .6f} "
        f"mean_gain={fd['gain_vs_50b'].mean(): .6f}"
    )

print("\nSaved files")
print("-----------")
print(direct_path)
print(screen_path)
print(screen_oof_path)
print(fit_details_path)
print(fold_diag_path)
print(oof_weighted_path)
print(test_weighted_path)
print(submission_weighted_path)
print(oof_oof_path)
print(test_oof_path)
print(submission_oof_path)
for p in top_paths:
    print(p)

print("\nSubmission validation")
print("---------------------")
for p in [submission_weighted_path, submission_oof_path]:
    sub = pd.read_csv(p)
    print(p, sub.shape, sub[TARGET_COL].describe().to_dict())

print("\nDecision rule")
print("-------------")
gain = float(best_weighted["gain_vs_50b"])
weighted_gain = float(best_weighted["public_weighted_gain_vs_50b"])
solver_gain = float(best_weighted["gain_solver_only_vs_50b"])
none_gain = float(best_weighted["gain_none_vs_50b"])
overlap_gain = float(best_weighted["gain_overlap_vs_50b"])
min_fold_gain = float(fold_weighted55a["gain_vs_50b"].min())

if gain >= 1.0 and weighted_gain >= 0.50 and solver_gain > 0 and overlap_gain > -0.002 and min_fold_gain >= 0:
    print("55A has strong enough EB residual signal to consider a public probe.")
elif gain >= 0.50 and weighted_gain >= 0.25 and solver_gain > 0 and overlap_gain > -0.002:
    print("55A has moderate signal. Inspect carefully; do not submit blindly.")
else:
    print("55A is not strong enough versus public-validated 50B. Keep 50B protected.")

55A. Binned residual tables / empirical-Bayes smoothing

Raw data
--------
raw_train_te: (144921, 62)
raw_test_te:  (48307, 61)

Artifact checks
---------------
model_results/oof_mf50b_best_oof.csv                              True
model_results/testpred_mf50b_best_oof.csv                         True
model_results/oof_hybrid44b_best.csv                              True
model_results/testpred_hybrid44b_best.csv                         True
model_results/account39a_raw_oof_reconstruction.csv               True
model_results/account39a_raw_test_reconstruction.csv              True
model_results/account39b_solver_raw_oof.csv                       True
model_results/account39b_solver_raw_test.csv                      True

Reference
---------
50B OOF MSE: 51.463158 | public 29.448
44B OOF MSE: 52.780037 | public 30.556

Tier coverage
-------------
overlap      train= 53786 test= 27298 50B tier MSE=0.506983
solver_only  train= 27807 test= 17807 50B tier MSE=63.484444
none         train= 63

,config,direct_oof_mse,direct_gain_vs_50b,direct_public_weighted_mse,resid_cap,corr_oof_mean,corr_oof_std,corr_test_mean,corr_test_std,direct_mse_overlap,direct_gain_overlap_vs_50b,direct_mse_solver_only,direct_gain_solver_only_vs_50b,direct_mse_none,direct_gain_none_vs_50b,direct_public_weighted_gain_vs_50b
0,eb55a_county_subject_public,51.564743,-0.101585,29.642101,0.15,0.000415,0.002950,0.000477,0.002834,0.507161,-0.000179,63.509117,-0.024673,89.684464,-0.221481,-0.023877
1,eb55a_subgroup_assessment_nbin_solver,51.585693,-0.122536,29.688747,0.18,0.000413,0.002945,0.000393,0.003090,0.507000,-0.000017,63.637386,-0.152943,89.676224,-0.213242,-0.070522
2,eb55a_subject_nbin_public,51.593365,-0.130207,29.652570,0.20,0.000459,0.003637,0.000580,0.003561,0.507141,-0.000158,63.527195,-0.042751,89.742043,-0.279060,-0.034346
3,eb55a_district_asg_very_conservative,51.660427,-0.197269,29.669888,0.12,0.000549,0.002922,0.000560,0.002005,0.507199,-0.000216,63.548153,-0.063709,89.886261,-0.423279,-0.051664
4,eb55a_county_asg_conservative,51.702671,-0.239513,29.679291,0.15,0.000519,0.003501,0.000541,0.002502,0.507136,-0.000154,63.557068,-0.072624,89.979080,-0.516098,-0.061067
5,eb55a_district_type_asg_public,51.730419,-0.267262,29.746520,0.20,0.000505,0.004044,0.000541,0.003829,0.507067,-0.000085,63.742790,-0.258347,89.961075,-0.498093,-0.128296
6,eb55a_asg_nbin_public,51.740562,-0.277405,29.750431,0.20,0.000510,0.004130,0.000570,0.003894,0.507068,-0.000086,63.749775,-0.265331,89.981224,-0.518242,-0.132207



Top blended candidates by public-weighted OOF
---------------------------------------------


,config,strategy,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b,public_weighted_mse,public_weighted_gain_vs_50b,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b,key
0,eb55a_subject_nbin_public,solver_only,0.0,0.410,0.0,51.455441,0.007717,29.603405,0.014820,0.506983,0.0,63.444241,0.040203,89.462982,0.0,eb55a_subject_nbin_public__solver_only
1,eb55a_subject_nbin_public,solver_plus_none,0.0,0.410,0.0,51.455441,0.007717,29.603405,0.014820,0.506983,0.0,63.444241,0.040203,89.462982,0.0,eb55a_subject_nbin_public__solver_plus_none
2,eb55a_county_subject_public,solver_only,0.0,0.415,0.0,51.458275,0.004883,29.608847,0.009378,0.506983,0.0,63.459003,0.025440,89.462982,0.0,eb55a_county_subject_public__solver_only
3,eb55a_county_subject_public,solver_plus_none,0.0,0.415,0.0,51.458275,0.004883,29.608847,0.009378,0.506983,0.0,63.459003,0.025440,89.462982,0.0,eb55a_county_subject_public__solver_plus_none
4,eb55a_subgroup_assessment_nbin_solver,solver_only,0.0,0.065,0.0,51.463013,0.000145,29.617943,0.000281,0.506983,0.0,63.483681,0.000763,89.462982,0.0,eb55a_subgroup_assessment_nbin_solver__solver_...
5,eb55a_subgroup_assessment_nbin_solver,solver_plus_none,0.0,0.065,0.0,51.463013,0.000145,29.617943,0.000281,0.506983,0.0,63.483681,0.000763,89.462982,0.0,eb55a_subgroup_assessment_nbin_solver__solver_...
6,eb55a_asg_nbin_public,solver_only,0.0,0.000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,eb55a_asg_nbin_public__solver_only
7,eb55a_asg_nbin_public,none_only,0.0,0.000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,eb55a_asg_nbin_public__none_only
8,eb55a_asg_nbin_public,solver_plus_none,0.0,0.000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,eb55a_asg_nbin_public__solver_plus_none
9,eb55a_subject_nbin_public,none_only,0.0,0.000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,eb55a_subject_nbin_public__none_only



Top blended candidates by ordinary OOF
--------------------------------------


,config,strategy,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b,public_weighted_mse,public_weighted_gain_vs_50b,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b,key
0,eb55a_subject_nbin_public,solver_only,0.0,0.410,0.0,51.455441,0.007717,29.603405,0.014820,0.506983,0.0,63.444241,0.040203,89.462982,0.0,eb55a_subject_nbin_public__solver_only
1,eb55a_subject_nbin_public,solver_plus_none,0.0,0.410,0.0,51.455441,0.007717,29.603405,0.014820,0.506983,0.0,63.444241,0.040203,89.462982,0.0,eb55a_subject_nbin_public__solver_plus_none
2,eb55a_county_subject_public,solver_only,0.0,0.415,0.0,51.458275,0.004883,29.608847,0.009378,0.506983,0.0,63.459003,0.025440,89.462982,0.0,eb55a_county_subject_public__solver_only
3,eb55a_county_subject_public,solver_plus_none,0.0,0.415,0.0,51.458275,0.004883,29.608847,0.009378,0.506983,0.0,63.459003,0.025440,89.462982,0.0,eb55a_county_subject_public__solver_plus_none
4,eb55a_subgroup_assessment_nbin_solver,solver_only,0.0,0.065,0.0,51.463013,0.000145,29.617943,0.000281,0.506983,0.0,63.483681,0.000763,89.462982,0.0,eb55a_subgroup_assessment_nbin_solver__solver_...
5,eb55a_subgroup_assessment_nbin_solver,solver_plus_none,0.0,0.065,0.0,51.463013,0.000145,29.617943,0.000281,0.506983,0.0,63.483681,0.000763,89.462982,0.0,eb55a_subgroup_assessment_nbin_solver__solver_...
6,eb55a_asg_nbin_public,solver_only,0.0,0.000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,eb55a_asg_nbin_public__solver_only
7,eb55a_asg_nbin_public,none_only,0.0,0.000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,eb55a_asg_nbin_public__none_only
8,eb55a_asg_nbin_public,solver_plus_none,0.0,0.000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,eb55a_asg_nbin_public__solver_plus_none
9,eb55a_subject_nbin_public,none_only,0.0,0.000,0.0,51.463158,0.000000,29.618224,0.000000,0.506983,0.0,63.484444,0.000000,89.462982,0.0,eb55a_subject_nbin_public__none_only



Best public-weighted candidate
------------------------------
config                                      eb55a_subject_nbin_public
strategy                                                  solver_only
lambda_overlap                                                    0.0
lambda_solver_only                                               0.41
lambda_none                                                       0.0
oof_mse                                                     51.455441
gain_vs_50b                                                  0.007717
public_weighted_mse                                         29.603405
public_weighted_gain_vs_50b                                   0.01482
mse_overlap                                                  0.506983
gain_overlap_vs_50b                                               0.0
mse_solver_only                                             63.444241
gain_solver_only_vs_50b                                      0.040203
mse_none                   

In [8]:
# ============================================================
# 56A. Mine existing 50B checkpoints for alternative public-weighted candidates
# ============================================================
#
# Current protected public best:
#   submission_mf50b_best_oof.csv = 29.448
#
# Why:
#   50B transferred publicly.
#   Later residual families did not.
#
#   50B best was selected by ordinary OOF:
#       rank16_reg500_uniform
#
#   But several nearby 50B configs had better public-tier-weighted OOF:
#       rank12_reg500_uniform
#       rank14_reg500_uniform
#       rank10_reg500_uniform
#       rank12_reg300_uniform
#       etc.
#
# Goal:
#   Reconstruct fold-averaged OOF/test submissions for promising 50B configs
#   from existing checkpoint .npy files, then create:
#       - best public-weighted 50B variant submissions
#       - conservative blends with current 50B public anchor
#
# No retraining.
# No native libraries beyond numpy/pandas/sklearn.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

print("=" * 90)
print("56A. Mine existing 50B checkpoints for alternative public-weighted candidates")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required files
# ------------------------------------------------------------

required = [
    "model_results/mf50b_screen.csv",
    "model_results/oof_mf50b_best_oof.csv",
    "model_results/testpred_mf50b_best_oof.csv",
    "model_results/oof_hybrid44b_best.csv",
    "model_results/testpred_hybrid44b_best.csv",
]

for p in required:
    print(f"{p:60s}", Path(p).exists())

missing = [p for p in required if not Path(p).exists()]
if missing:
    raise RuntimeError(f"Missing required files: {missing}")

ckpt_dir = Path("model_results/mf50b_checkpoints")
if not ckpt_dir.exists():
    raise RuntimeError("Missing model_results/mf50b_checkpoints directory.")

# ------------------------------------------------------------
# 2. Load base artifacts
# ------------------------------------------------------------

def load_oof_56a(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)


def load_test_56a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError("No test IDs found.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)


_, y56, pred50b_oof = load_oof_56a("model_results/oof_mf50b_best_oof.csv")
_, test_ids56, pred50b_test = load_test_56a("model_results/testpred_mf50b_best_oof.csv")

_, y44, pred44b_oof = load_oof_56a("model_results/oof_hybrid44b_best.csv")
_, _, pred44b_test = load_test_56a("model_results/testpred_hybrid44b_best.csv")

assert np.max(np.abs(y56 - y44)) < 1e-5

n_train = len(y56)
n_test = len(pred50b_test)

mse50b = float(mean_squared_error(y56, pred50b_oof))

print("\nReference")
print("---------")
print(f"50B OOF MSE: {mse50b:.6f} | public 29.448")

# ------------------------------------------------------------
# 3. Reconstruct tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

test_tier_rates = {tier: float(mask.mean()) for tier, mask in tier_masks_test.items()}

def tier_weighted_mse(pred):
    out = 0.0
    for tier, mask in tier_masks_oof.items():
        out += test_tier_rates[tier] * float(mean_squared_error(y56[mask], pred[mask]))
    return float(out)

mse50b_weighted = tier_weighted_mse(pred50b_oof)

print("\nTier coverage")
print("-------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"50B MSE={mean_squared_error(y56[mask], pred50b_oof[mask]):.6f}"
    )

print("50B public-tier-weighted OOF:", mse50b_weighted)

# ------------------------------------------------------------
# 4. Load 50B screen and pick candidates
# ------------------------------------------------------------

screen = pd.read_csv("model_results/mf50b_screen.csv")

# Only use fold-averaged OOF-safe 50B configs, not fullfit.
screen = screen.copy()
screen = screen[screen["entity_key"].astype(str) == "school"].copy()
screen = screen[screen["weight_scheme"].astype(str) == "uniform"].copy()
screen = screen[screen["strategy"].isin(["all_tiers", "solver_plus_none"])].copy()

# Exclude if config checkpoint missing.
def has_ckpt(config):
    return (
        (ckpt_dir / f"{config}_oof.npy").exists()
        and (ckpt_dir / f"{config}_test.npy").exists()
    )

screen["has_checkpoint"] = screen["config"].map(has_ckpt)
screen = screen[screen["has_checkpoint"]].copy()

# Top by public-weighted and top by ordinary OOF.
top_weighted = screen.sort_values(["test_tier_weighted_mse", "oof_mse"]).head(12)
top_oof = screen.sort_values(["oof_mse", "test_tier_weighted_mse"]).head(12)

candidate_rows = pd.concat([top_weighted, top_oof], axis=0).drop_duplicates("key").reset_index(drop=True)

print("\nTop candidate rows selected")
print("---------------------------")
display(candidate_rows[[
    "config", "strategy", "rank", "reg_entity",
    "lambda_overlap", "lambda_solver_only", "lambda_none",
    "oof_mse", "gain_vs_50a", "test_tier_weighted_mse",
    "weighted_gain_vs_50a"
]].head(30))

# ------------------------------------------------------------
# 5. Reconstruct predictions for candidate rows
# ------------------------------------------------------------

def make_blend(candidate_oof_direct, candidate_test_direct, lams, is_test=False):
    if is_test:
        base = pred44b_test.astype(np.float64)
        alt = candidate_test_direct.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred44b_oof.astype(np.float64)
        alt = candidate_oof_direct.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)


store = {}
rows = []

for _, row in candidate_rows.iterrows():
    cfg = row["config"]
    strategy = row["strategy"]
    key = row["key"]

    p_oof_direct = np.load(ckpt_dir / f"{cfg}_oof.npy").astype(np.float32)
    p_test_direct = np.load(ckpt_dir / f"{cfg}_test.npy").astype(np.float32)

    lams = {
        "overlap": float(row["lambda_overlap"]),
        "solver_only": float(row["lambda_solver_only"]),
        "none": float(row["lambda_none"]),
    }

    p_oof = make_blend(p_oof_direct, p_test_direct, lams, is_test=False)
    p_test = make_blend(p_oof_direct, p_test_direct, lams, is_test=True)

    mse = float(mean_squared_error(y56, p_oof))
    weighted = tier_weighted_mse(p_oof)

    out = {
        "key": key,
        "config": cfg,
        "strategy": strategy,
        "rank": int(row["rank"]),
        "reg_entity": float(row["reg_entity"]),
        "lambda_overlap": lams["overlap"],
        "lambda_solver_only": lams["solver_only"],
        "lambda_none": lams["none"],
        "oof_mse_recomputed": mse,
        "gain_vs_50b": mse50b - mse,
        "public_weighted_mse_recomputed": weighted,
        "public_weighted_gain_vs_50b": mse50b_weighted - weighted,
        "mean_abs_diff_vs_50b_test": float(np.mean(np.abs(p_test - pred50b_test))),
        "p95_abs_diff_vs_50b_test": float(np.percentile(np.abs(p_test - pred50b_test), 95)),
        "max_abs_diff_vs_50b_test": float(np.max(np.abs(p_test - pred50b_test))),
    }

    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y56[mask], p_oof[mask]))
        out[f"gain_{tier}_vs_50b"] = (
            float(mean_squared_error(y56[mask], pred50b_oof[mask]))
            - out[f"mse_{tier}"]
        )

    rows.append(out)
    store[key] = {"oof": p_oof, "test": p_test, "row": out}

recon = pd.DataFrame(rows).sort_values(
    ["public_weighted_mse_recomputed", "oof_mse_recomputed"]
).reset_index(drop=True)

print("\nReconstructed candidate screen")
print("------------------------------")
display(recon.head(30))

# ------------------------------------------------------------
# 6. Build conservative blends with current public anchor 50B
# ------------------------------------------------------------

blend_rows = []
blend_store = {}

for _, row in recon.iterrows():
    key = row["key"]
    p_oof_alt = store[key]["oof"]
    p_test_alt = store[key]["test"]

    for w_alt in [0.25, 0.50, 0.75, 1.00]:
        p_oof_mix = np.clip(
            (1.0 - w_alt) * pred50b_oof.astype(np.float64)
            + w_alt * p_oof_alt.astype(np.float64),
            0, 100
        ).astype(np.float32)

        p_test_mix = np.clip(
            (1.0 - w_alt) * pred50b_test.astype(np.float64)
            + w_alt * p_test_alt.astype(np.float64),
            0, 100
        ).astype(np.float32)

        mse = float(mean_squared_error(y56, p_oof_mix))
        weighted = tier_weighted_mse(p_oof_mix)

        out = {
            "source_key": key,
            "config": row["config"],
            "rank": row["rank"],
            "reg_entity": row["reg_entity"],
            "w_alt": float(w_alt),
            "oof_mse": mse,
            "gain_vs_50b": mse50b - mse,
            "public_weighted_mse": weighted,
            "public_weighted_gain_vs_50b": mse50b_weighted - weighted,
            "mean_abs_diff_vs_50b_test": float(np.mean(np.abs(p_test_mix - pred50b_test))),
            "p95_abs_diff_vs_50b_test": float(np.percentile(np.abs(p_test_mix - pred50b_test), 95)),
            "max_abs_diff_vs_50b_test": float(np.max(np.abs(p_test_mix - pred50b_test))),
        }

        for tier, mask in tier_masks_oof.items():
            out[f"mse_{tier}"] = float(mean_squared_error(y56[mask], p_oof_mix[mask]))
            out[f"gain_{tier}_vs_50b"] = (
                float(mean_squared_error(y56[mask], pred50b_oof[mask]))
                - out[f"mse_{tier}"]
            )

        blend_key = f"{key}__mix{str(w_alt).replace('.', 'p')}"
        out["blend_key"] = blend_key

        blend_rows.append(out)
        blend_store[blend_key] = {"oof": p_oof_mix, "test": p_test_mix, "row": out}

blend_screen = pd.DataFrame(blend_rows).sort_values(
    ["public_weighted_mse", "oof_mse"]
).reset_index(drop=True)

print("\nConservative blend screen")
print("-------------------------")
display(blend_screen.head(30))

# ------------------------------------------------------------
# 7. Fold diagnostics for top candidates
# ------------------------------------------------------------

folds = list(KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

top_blend_keys = blend_screen.head(5)["blend_key"].tolist()
fold_rows = []

for label, obj in [("50b_anchor", {"oof": pred50b_oof})] + [(k, blend_store[k]) for k in top_blend_keys]:
    pred = obj["oof"]

    for fold_num, (_, va_idx) in enumerate(folds, start=1):
        row = {
            "label": label,
            "fold": fold_num,
            "mse_50b": float(mean_squared_error(y56[va_idx], pred50b_oof[va_idx])),
            "mse_candidate": float(mean_squared_error(y56[va_idx], pred[va_idx])),
        }
        row["gain_vs_50b"] = row["mse_50b"] - row["mse_candidate"]

        for tier, mask_full in tier_masks_oof.items():
            mask = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask.sum())
            if int(mask.sum()) > 0:
                row[f"mse50b_{tier}"] = float(mean_squared_error(y56[va_idx][mask], pred50b_oof[va_idx][mask]))
                row[f"msecand_{tier}"] = float(mean_squared_error(y56[va_idx][mask], pred[va_idx][mask]))
                row[f"gain_{tier}_vs_50b"] = row[f"mse50b_{tier}"] - row[f"msecand_{tier}"]

        fold_rows.append(row)

fold_diag = pd.DataFrame(fold_rows)

print("\nFold diagnostics for top blends")
print("-------------------------------")
print(fold_diag.to_string(index=False))

# ------------------------------------------------------------
# 8. Save submissions
# ------------------------------------------------------------

Path("model_results").mkdir(exist_ok=True)

recon.to_csv("model_results/mf56a_reconstructed_50b_variants.csv", index=False)
blend_screen.to_csv("model_results/mf56a_conservative_blend_screen.csv", index=False)
fold_diag.to_csv("model_results/mf56a_top_blend_fold_diag.csv", index=False)

submission_paths = []

# Save top 5 blend candidates.
for i, blend_key in enumerate(top_blend_keys, start=1):
    p_test = blend_store[blend_key]["test"]
    out_path = f"submission_mf56a_top{i}_weightedblend.csv"
    pd.DataFrame({
        ID_COL: test_ids56,
        TARGET_COL: p_test,
    }).to_csv(out_path, index=False)
    submission_paths.append(out_path)

# Also save top direct public-weighted alternatives.
for i, (_, row) in enumerate(recon.head(5).iterrows(), start=1):
    key = row["key"]
    p_test = store[key]["test"]
    out_path = f"submission_mf56a_direct_top{i}_weighted.csv"
    pd.DataFrame({
        ID_COL: test_ids56,
        TARGET_COL: p_test,
    }).to_csv(out_path, index=False)
    submission_paths.append(out_path)

# Validate.
for p in submission_paths:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test, 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[ID_COL].notna().all(), p
    assert sub[TARGET_COL].notna().all(), p
    assert np.isfinite(sub[TARGET_COL]).all(), p
    assert sub[TARGET_COL].between(0, 100).all(), p

print("\nSaved files")
print("-----------")
print("model_results/mf56a_reconstructed_50b_variants.csv")
print("model_results/mf56a_conservative_blend_screen.csv")
print("model_results/mf56a_top_blend_fold_diag.csv")
for p in submission_paths:
    print(p)

print("\nDecision guidance")
print("-----------------")
print("These are not new residual families. They are nearby variants within the public-validated 50B family.")
print("If using a public slot, the safest probe is submission_mf56a_top1_weightedblend.csv.")
print("If top1 improves, try top direct weighted or top2 blend next.")

56A. Mine existing 50B checkpoints for alternative public-weighted candidates
model_results/mf50b_screen.csv                               True
model_results/oof_mf50b_best_oof.csv                         True
model_results/testpred_mf50b_best_oof.csv                    True
model_results/oof_hybrid44b_best.csv                         True
model_results/testpred_hybrid44b_best.csv                    True

Reference
---------
50B OOF MSE: 51.463158 | public 29.448

Tier coverage
-------------
overlap      train= 53786 test= 27298 50B MSE=0.506983
solver_only  train= 27807 test= 17807 50B MSE=63.484444
none         train= 63328 test=  3202 50B MSE=89.462982
50B public-tier-weighted OOF: 29.618224391869244

Top candidate rows selected
---------------------------


,config,strategy,rank,reg_entity,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50a,test_tier_weighted_mse,weighted_gain_vs_50a
0,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,all_tiers,12,500.0,0.0050,0.6150,0.8375,51.484856,0.762238,29.577413,0.266651
1,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,solver_plus_none,12,500.0,0.0000,0.6150,0.8375,51.484879,0.762215,29.577450,0.266614
2,mf50b_school_rank12_reg300p0_uniform_b1p0_f1p0,all_tiers,12,300.0,0.0050,0.6025,0.8100,51.543404,0.703690,29.593361,0.250702
3,mf50b_school_rank12_reg300p0_uniform_b1p0_f1p0,solver_plus_none,12,300.0,0.0000,0.6025,0.8100,51.543427,0.703667,29.593400,0.250663
4,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,all_tiers,10,500.0,0.0050,0.6275,0.8450,51.536339,0.710754,29.594585,0.249479
5,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,solver_plus_none,10,500.0,0.0000,0.6275,0.8450,51.536366,0.710728,29.594620,0.249443
6,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0,all_tiers,14,500.0,0.0050,0.5775,0.8275,51.483036,0.764057,29.597466,0.246597
7,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0,solver_plus_none,14,500.0,0.0000,0.5775,0.8275,51.483047,0.764046,29.597486,0.246577
8,mf50b_school_rank12_reg200p0_uniform_b1p0_f1p0,all_tiers,12,200.0,0.0050,0.5900,0.7850,51.605522,0.641571,29.611342,0.232721
9,mf50b_school_rank12_reg200p0_uniform_b1p0_f1p0,solver_plus_none,12,200.0,0.0000,0.5900,0.7850,51.605549,0.641544,29.611383,0.232680



Reconstructed candidate screen
------------------------------


,key,config,strategy,rank,reg_entity,lambda_overlap,lambda_solver_only,lambda_none,oof_mse_recomputed,gain_vs_50b,public_weighted_mse_recomputed,public_weighted_gain_vs_50b,mean_abs_diff_vs_50b_test,p95_abs_diff_vs_50b_test,max_abs_diff_vs_50b_test,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b
0,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,all_tiers,12,500.0,0.0050,0.6150,0.8375,51.484856,-0.021698,29.577413,0.040812,0.098552,0.529694,5.481434,0.506955,2.729893e-05,63.354584,0.129860,89.569687,-0.106705
1,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,solver_plus_none,12,500.0,0.0000,0.6150,0.8375,51.484879,-0.021721,29.577450,0.040775,0.100235,0.529694,5.481434,0.507021,-3.814697e-05,63.354584,0.129860,89.569687,-0.106705
2,mf50b_school_rank12_reg300p0_uniform_b1p0_f1p0...,mf50b_school_rank12_reg300p0_uniform_b1p0_f1p0,all_tiers,12,300.0,0.0050,0.6025,0.8100,51.543404,-0.080246,29.593361,0.024863,0.101263,0.532943,5.092205,0.506952,3.075600e-05,63.375408,0.109035,89.694519,-0.231537
3,mf50b_school_rank12_reg300p0_uniform_b1p0_f1p0...,mf50b_school_rank12_reg300p0_uniform_b1p0_f1p0,solver_plus_none,12,300.0,0.0000,0.6025,0.8100,51.543427,-0.080269,29.593400,0.024824,0.102923,0.532943,5.092205,0.507021,-3.814697e-05,63.375408,0.109035,89.694519,-0.231537
4,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,all_tiers,10,500.0,0.0050,0.6275,0.8450,51.536339,-0.073181,29.594585,0.023640,0.142509,0.776480,6.832275,0.506958,2.479553e-05,63.382156,0.102287,89.675392,-0.212410
5,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,solver_plus_none,10,500.0,0.0000,0.6275,0.8450,51.536366,-0.073208,29.594620,0.023604,0.143804,0.776480,6.832275,0.507021,-3.814697e-05,63.382156,0.102287,89.675392,-0.212410
6,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0,all_tiers,14,500.0,0.0050,0.5775,0.8275,51.483036,-0.019878,29.597466,0.020758,0.060775,0.337305,2.809227,0.506985,-2.503395e-06,63.414417,0.070026,89.539223,-0.076241
7,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank14_reg500p0_uniform_b1p0_f1p0,solver_plus_none,14,500.0,0.0000,0.5775,0.8275,51.483047,-0.019890,29.597486,0.020738,0.062816,0.337305,2.809227,0.507021,-3.814697e-05,63.414417,0.070026,89.539223,-0.076241
8,mf50b_school_rank12_reg200p0_uniform_b1p0_f1p0...,mf50b_school_rank12_reg200p0_uniform_b1p0_f1p0,all_tiers,12,200.0,0.0050,0.5900,0.7850,51.605522,-0.142365,29.611342,0.006882,0.105476,0.540277,4.715252,0.506948,3.439188e-05,63.400620,0.083824,89.825607,-0.362625
9,mf50b_school_rank12_reg200p0_uniform_b1p0_f1p0...,mf50b_school_rank12_reg200p0_uniform_b1p0_f1p0,solver_plus_none,12,200.0,0.0000,0.5900,0.7850,51.605549,-0.142391,29.611383,0.006841,0.107093,0.540277,4.715252,0.507021,-3.814697e-05,63.400620,0.083824,89.825607,-0.362625



Conservative blend screen
-------------------------


,source_key,config,rank,reg_entity,w_alt,oof_mse,gain_vs_50b,public_weighted_mse,public_weighted_gain_vs_50b,mean_abs_diff_vs_50b_test,p95_abs_diff_vs_50b_test,max_abs_diff_vs_50b_test,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b,blend_key
0,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,10,500.0,0.50,51.344170,0.118988,29.541988,0.076236,0.071255,0.388238,3.416138,0.506962,0.000021,63.313091,0.171352,89.265945,0.197037,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...
1,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,10,500.0,0.50,51.344181,0.118977,29.542002,0.076223,0.071902,0.388238,3.416138,0.506986,-0.000004,63.313091,0.171352,89.265945,0.197037,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...
2,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,10,500.0,0.75,51.401360,0.061798,29.552183,0.066041,0.106882,0.582359,5.124207,0.506958,0.000025,63.317574,0.166870,89.394859,0.068123,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...
3,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0,10,500.0,0.75,51.401379,0.061779,29.552207,0.066018,0.107853,0.582359,5.124207,0.507000,-0.000017,63.317574,0.166870,89.394859,0.068123,mf50b_school_rank10_reg500p0_uniform_b1p0_f1p0...
4,mf50b_school_rank10_reg300p0_uniform_b1p0_f1p0...,mf50b_school_rank10_reg300p0_uniform_b1p0_f1p0,10,300.0,0.50,51.379276,0.083881,29.553454,0.064771,0.071858,0.386409,3.510311,0.506961,0.000021,63.331177,0.153267,89.338348,0.124634,mf50b_school_rank10_reg300p0_uniform_b1p0_f1p0...
5,mf50b_school_rank10_reg300p0_uniform_b1p0_f1p0...,mf50b_school_rank10_reg300p0_uniform_b1p0_f1p0,10,300.0,0.50,51.379288,0.083870,29.553468,0.064757,0.072498,0.386409,3.510311,0.506986,-0.000004,63.331177,0.153267,89.338348,0.124634,mf50b_school_rank10_reg300p0_uniform_b1p0_f1p0...
6,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,12,500.0,0.50,51.372356,0.090801,29.554827,0.063397,0.049276,0.264847,2.740715,0.506963,0.000019,63.338314,0.146130,89.319366,0.143616,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...
7,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,12,500.0,0.50,51.372360,0.090797,29.554841,0.063384,0.050118,0.264847,2.740715,0.506986,-0.000004,63.338314,0.146130,89.319366,0.143616,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...
8,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,12,500.0,0.75,51.403191,0.059967,29.555372,0.062852,0.073914,0.397271,4.111076,0.506958,0.000025,63.326149,0.158295,89.395287,0.067696,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...
9,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0,12,500.0,0.75,51.403210,0.059948,29.555396,0.062828,0.075176,0.397271,4.111076,0.507000,-0.000017,63.326149,0.158295,89.395287,0.067696,mf50b_school_rank12_reg500p0_uniform_b1p0_f1p0...



Fold diagnostics for top blends
-------------------------------
                                                                    label  fold   mse_50b  mse_candidate  gain_vs_50b  n_overlap  mse50b_overlap  msecand_overlap  gain_overlap_vs_50b  n_solver_only  mse50b_solver_only  msecand_solver_only  gain_solver_only_vs_50b  n_none  mse50b_none  msecand_none  gain_none_vs_50b
                                                               50b_anchor     1 51.927963      51.927963     0.000000      10837        0.418636         0.418636             0.000000           5496           61.760151            61.760151                 0.000000   12652    91.776917     91.776917          0.000000
                                                               50b_anchor     2 49.481594      49.481594     0.000000      10622        0.394585         0.394585             0.000000           5623           60.256847            60.256847                 0.000000   12739    85.654999     85.654999   

In [9]:
# ============================================================
# 57A. Global integer-count feasibility solver audit
# ============================================================
#
# Current protected public model:
#   submission_mf50b_best_oof.csv
#   public MSE = 29.448
#
# Why this branch:
#   External data is disallowed.
#   Generic residual modeling has failed or given tiny/noisy gains.
#
#   The only plausible remaining high-value path is another deterministic
#   or semi-deterministic integer-count structure.
#
# Goal:
#   Treat each row as generated by an integer proficient count:
#
#       k_i integer in [0, N_i]
#       percent_i ≈ reported function of 100 * k_i / N_i
#
#   For each school/district/assessment block, enforce known count equations:
#
#       All Students = Female + Male
#       All Students = Economically Disadvantaged + Not Economically Disadvantaged
#
#   and, when All Students is missing but both partitions exist:
#
#       Female + Male = Econ Disadvantaged + Not Econ Disadvantaged
#
#   Projection objective:
#       choose feasible integer counts closest to 50B prior percentages.
#
# Outputs:
#   - audit of constraint coverage
#   - OOF validation of projected candidates
#   - protected tier-blended candidates
#   - submissions only for inspection, not automatic submission
#
# Submission rule:
#   Do not submit unless gain vs 50B is large and fold-stable.
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

Path("model_results").mkdir(exist_ok=True)

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5

print("=" * 90)
print("57A. Global integer-count feasibility solver audit")
print("=" * 90)

# ------------------------------------------------------------
# 1. Required raw data and artifacts
# ------------------------------------------------------------

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise RuntimeError(
        "raw_train_te/raw_test_te missing. Run setup/merge/build cells first, then raw aliases."
    )

required_files = [
    "model_results/oof_mf50b_best_oof.csv",
    "model_results/testpred_mf50b_best_oof.csv",
    "model_results/oof_hybrid44b_best.csv",
    "model_results/testpred_hybrid44b_best.csv",
    "model_results/account39a_raw_oof_reconstruction.csv",
    "model_results/account39a_raw_test_reconstruction.csv",
    "model_results/account39b_solver_raw_oof.csv",
    "model_results/account39b_solver_raw_test.csv",
]

print("\nArtifact checks")
print("---------------")
for p in required_files:
    print(f"{p:65s}", Path(p).exists())

missing = [p for p in required_files if not Path(p).exists()]
if missing:
    raise RuntimeError(f"Missing required artifacts: {missing}")

print("\nRaw shapes")
print("----------")
print("raw_train_te:", raw_train_te.shape)
print("raw_test_te: ", raw_test_te.shape)

# ------------------------------------------------------------
# 2. Load helpers
# ------------------------------------------------------------

def load_oof_57a(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column found in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)


def load_test_57a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column and no test_ids available.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)

_, y57_file, pred50b_oof = load_oof_57a("model_results/oof_mf50b_best_oof.csv")
_, test_ids57a, pred50b_test = load_test_57a("model_results/testpred_mf50b_best_oof.csv")

_, y44_file, pred44b_oof = load_oof_57a("model_results/oof_hybrid44b_best.csv")
_, _, pred44b_test = load_test_57a("model_results/testpred_hybrid44b_best.csv")

if "y_train" in globals():
    y57 = np.asarray(y_train, dtype=np.float32).reshape(-1)
else:
    y57 = y57_file.copy()

assert len(y57) == len(pred50b_oof)
assert len(raw_train_te) == len(pred50b_oof)
assert len(raw_test_te) == len(pred50b_test)
assert np.max(np.abs(y57 - y57_file)) < 1e-5
assert np.max(np.abs(y57 - y44_file)) < 1e-5

n_train = len(y57)
n_test = len(pred50b_test)

mse50b = float(mean_squared_error(y57, pred50b_oof))
mse44b = float(mean_squared_error(y57, pred44b_oof))

print("\nReferences")
print("----------")
print(f"50B OOF MSE: {mse50b:.6f} | public 29.448")
print(f"44B OOF MSE: {mse44b:.6f} | public 30.556")

# ------------------------------------------------------------
# 3. Reconstruct accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_oof = np.array(["none"] * n_train, dtype=object)
tier_oof[solver_only_oof] = "solver_only"
tier_oof[overlap_oof] = "overlap"

tier_test = np.array(["none"] * n_test, dtype=object)
tier_test[solver_only_test] = "solver_only"
tier_test[overlap_test] = "overlap"

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

test_tier_rates = {
    tier: float(mask.mean())
    for tier, mask in tier_masks_test.items()
}

print("\nTier coverage")
print("-------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"50B MSE={mean_squared_error(y57[mask], pred50b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 4. Build compact row frames
# ------------------------------------------------------------

def clean_str_57a(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str).to_numpy()

def safe_num_57a(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)

def build_frame_57a(raw_df, pred, tier_arr, source):
    out = pd.DataFrame(index=np.arange(len(raw_df)))
    out["source"] = source
    out["row_index"] = np.arange(len(raw_df))

    for c in ["DISTRICT", "SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"]:
        if c in raw_df.columns:
            out[c] = clean_str_57a(raw_df[c])
        else:
            out[c] = "<NA>"

    if "N_STUDENTS" not in raw_df.columns:
        raise RuntimeError("N_STUDENTS is required for 57A.")

    n = safe_num_57a(raw_df["N_STUDENTS"])
    n_int = np.rint(n)
    n_int = np.where(np.isfinite(n_int) & (n_int > 0), n_int, 1)
    n_int = np.clip(n_int, 1, 1000000).astype(np.int64)

    out["N_STUDENTS"] = n
    out["N_INT"] = n_int
    out["PRED_50B"] = np.clip(pred, 0, 100).astype(np.float64)
    out["tier"] = pd.Series(tier_arr).astype(str).to_numpy()

    out["BLOCK_DISTRICT_SCHOOL_ASSESSMENT"] = (
        out["DISTRICT"].astype(str)
        + "||" + out["SCHOOL"].astype(str)
        + "||" + out["ASSESSMENT_NAME"].astype(str)
    )

    out["BLOCK_SCHOOL_ASSESSMENT"] = (
        out["SCHOOL"].astype(str)
        + "||" + out["ASSESSMENT_NAME"].astype(str)
    )

    return out

train57 = build_frame_57a(raw_train_te, pred50b_oof, tier_oof, "train")
test57 = build_frame_57a(raw_test_te, pred50b_test, tier_test, "test")

# True count reconstruction for train.
train57["Y_TRUE"] = y57.astype(np.float64)
train57["K_TRUE_ROUND"] = np.rint(train57["Y_TRUE"].to_numpy() / 100.0 * train57["N_INT"].to_numpy()).astype(np.int64)
train57["K_TRUE_ROUND"] = np.minimum(np.maximum(train57["K_TRUE_ROUND"], 0), train57["N_INT"].to_numpy())

print("\nCompact frames")
print("--------------")
print("train57:", train57.shape)
print("test57: ", test57.shape)

# ------------------------------------------------------------
# 5. Count projection helper functions
# ------------------------------------------------------------

SG_ALL = "All Students"
SG_FEMALE = "Female"
SG_MALE = "Male"
SG_ED = "Economically Disadvantaged"
SG_NOT_ED = "Not Economically Disadvantaged"

KNOWN_SUBGROUPS = [SG_ALL, SG_FEMALE, SG_MALE, SG_ED, SG_NOT_ED]

def best_split_for_total(T, n1, n2, c1, c2, w1, w2):
    """
    Minimize w1*(k1-c1)^2 + w2*(k2-c2)^2 subject to:
        k1 + k2 = T
        0 <= k1 <= n1
        0 <= k2 <= n2
    """
    T = int(T)
    lo = max(0, T - int(n2))
    hi = min(int(n1), T)

    if lo > hi:
        return None

    denom = max(float(w1 + w2), 1e-12)
    k1_cont = (float(w1) * float(c1) + float(w2) * (float(T) - float(c2))) / denom

    candidates = set()
    for d in range(-3, 4):
        candidates.add(int(np.floor(k1_cont)) + d)
        candidates.add(int(np.ceil(k1_cont)) + d)

    candidates.add(lo)
    candidates.add(hi)

    best = None
    for k1 in candidates:
        if k1 < lo or k1 > hi:
            continue
        k2 = T - k1
        obj = float(w1) * (k1 - c1) ** 2 + float(w2) * (k2 - c2) ** 2
        if best is None or obj < best[0]:
            best = (obj, int(k1), int(k2))

    return best


def discrete_convex_minimize(eval_func, lo, hi):
    """
    Ternary-ish search for discrete convex objective, with local cleanup.
    """
    lo = int(lo)
    hi = int(hi)

    if lo > hi:
        return None

    # Small interval: brute force.
    if hi - lo <= 80:
        best_t = lo
        best_v = eval_func(lo)
        for t in range(lo + 1, hi + 1):
            v = eval_func(t)
            if v < best_v:
                best_v = v
                best_t = t
        return best_t, best_v

    l, r = lo, hi
    while r - l > 40:
        m1 = l + (r - l) // 3
        m2 = r - (r - l) // 3
        v1 = eval_func(m1)
        v2 = eval_func(m2)
        if v1 <= v2:
            r = m2 - 1
        else:
            l = m1 + 1

    best_t = l
    best_v = eval_func(l)
    for t in range(l + 1, r + 1):
        v = eval_func(t)
        if v < best_v:
            best_v = v
            best_t = t

    return best_t, best_v


def project_block_counts(block_df, prior_col="PRED_50B"):
    """
    Project known subgroup rows in one block to feasible integer counts.
    Returns:
        dict row_index -> projected integer count
        metadata dict
    """
    sub_to_rows = {}
    for sg, grp in block_df.groupby("SUBGROUP_NAME", sort=False):
        if sg in KNOWN_SUBGROUPS:
            sub_to_rows[str(sg)] = grp.index.to_list()

    # Only solve blocks with unique subgroup rows. Duplicates are skipped.
    if any(len(v) != 1 for v in sub_to_rows.values()):
        return {}, {"status": "duplicate_subgroup_rows", "n_projected": 0}

    row_by_sg = {sg: rows[0] for sg, rows in sub_to_rows.items()}

    if len(row_by_sg) < 3:
        return {}, {"status": "too_few_known_rows", "n_projected": 0}

    N = {sg: int(block_df.loc[idx, "N_INT"]) for sg, idx in row_by_sg.items()}
    p = {sg: float(block_df.loc[idx, prior_col]) for sg, idx in row_by_sg.items()}

    # Prior counts and percent-MSE weights.
    c = {sg: p[sg] / 100.0 * N[sg] for sg in row_by_sg}
    w = {sg: (100.0 / max(N[sg], 1)) ** 2 for sg in row_by_sg}

    has_all = SG_ALL in row_by_sg
    has_gender = SG_FEMALE in row_by_sg and SG_MALE in row_by_sg
    has_econ = SG_ED in row_by_sg and SG_NOT_ED in row_by_sg

    gender_n_ok = has_all and has_gender and (N[SG_ALL] == N[SG_FEMALE] + N[SG_MALE])
    econ_n_ok = has_all and has_econ and (N[SG_ALL] == N[SG_ED] + N[SG_NOT_ED])

    pair_pair_ok = (
        (not has_all)
        and has_gender
        and has_econ
        and (N[SG_FEMALE] + N[SG_MALE] == N[SG_ED] + N[SG_NOT_ED])
    )

    projected = {}
    status = "no_valid_identity"

    # Case 1: All + both partitions.
    if has_all and gender_n_ok and econ_n_ok:
        lo = 0
        hi = min(
            N[SG_ALL],
            N[SG_FEMALE] + N[SG_MALE],
            N[SG_ED] + N[SG_NOT_ED],
        )

        def eval_T(T):
            gender = best_split_for_total(
                T, N[SG_FEMALE], N[SG_MALE],
                c[SG_FEMALE], c[SG_MALE],
                w[SG_FEMALE], w[SG_MALE],
            )
            econ = best_split_for_total(
                T, N[SG_ED], N[SG_NOT_ED],
                c[SG_ED], c[SG_NOT_ED],
                w[SG_ED], w[SG_NOT_ED],
            )
            if gender is None or econ is None:
                return np.inf
            return (
                w[SG_ALL] * (T - c[SG_ALL]) ** 2
                + gender[0]
                + econ[0]
            )

        result = discrete_convex_minimize(eval_T, lo, hi)
        if result is not None and np.isfinite(result[1]):
            T = int(result[0])

            gender = best_split_for_total(
                T, N[SG_FEMALE], N[SG_MALE],
                c[SG_FEMALE], c[SG_MALE],
                w[SG_FEMALE], w[SG_MALE],
            )
            econ = best_split_for_total(
                T, N[SG_ED], N[SG_NOT_ED],
                c[SG_ED], c[SG_NOT_ED],
                w[SG_ED], w[SG_NOT_ED],
            )

            projected[row_by_sg[SG_ALL]] = T
            projected[row_by_sg[SG_FEMALE]] = gender[1]
            projected[row_by_sg[SG_MALE]] = gender[2]
            projected[row_by_sg[SG_ED]] = econ[1]
            projected[row_by_sg[SG_NOT_ED]] = econ[2]

            status = "all_gender_econ"

    # Case 2: All + gender only.
    elif has_all and gender_n_ok:
        lo = 0
        hi = min(N[SG_ALL], N[SG_FEMALE] + N[SG_MALE])

        def eval_T(T):
            gender = best_split_for_total(
                T, N[SG_FEMALE], N[SG_MALE],
                c[SG_FEMALE], c[SG_MALE],
                w[SG_FEMALE], w[SG_MALE],
            )
            if gender is None:
                return np.inf
            return w[SG_ALL] * (T - c[SG_ALL]) ** 2 + gender[0]

        result = discrete_convex_minimize(eval_T, lo, hi)
        if result is not None and np.isfinite(result[1]):
            T = int(result[0])
            gender = best_split_for_total(
                T, N[SG_FEMALE], N[SG_MALE],
                c[SG_FEMALE], c[SG_MALE],
                w[SG_FEMALE], w[SG_MALE],
            )

            projected[row_by_sg[SG_ALL]] = T
            projected[row_by_sg[SG_FEMALE]] = gender[1]
            projected[row_by_sg[SG_MALE]] = gender[2]

            status = "all_gender"

    # Case 3: All + econ only.
    elif has_all and econ_n_ok:
        lo = 0
        hi = min(N[SG_ALL], N[SG_ED] + N[SG_NOT_ED])

        def eval_T(T):
            econ = best_split_for_total(
                T, N[SG_ED], N[SG_NOT_ED],
                c[SG_ED], c[SG_NOT_ED],
                w[SG_ED], w[SG_NOT_ED],
            )
            if econ is None:
                return np.inf
            return w[SG_ALL] * (T - c[SG_ALL]) ** 2 + econ[0]

        result = discrete_convex_minimize(eval_T, lo, hi)
        if result is not None and np.isfinite(result[1]):
            T = int(result[0])
            econ = best_split_for_total(
                T, N[SG_ED], N[SG_NOT_ED],
                c[SG_ED], c[SG_NOT_ED],
                w[SG_ED], w[SG_NOT_ED],
            )

            projected[row_by_sg[SG_ALL]] = T
            projected[row_by_sg[SG_ED]] = econ[1]
            projected[row_by_sg[SG_NOT_ED]] = econ[2]

            status = "all_econ"

    # Case 4: no All, but both partitions exist and total N matches.
    elif pair_pair_ok:
        lo = 0
        hi = min(
            N[SG_FEMALE] + N[SG_MALE],
            N[SG_ED] + N[SG_NOT_ED],
        )

        def eval_T(T):
            gender = best_split_for_total(
                T, N[SG_FEMALE], N[SG_MALE],
                c[SG_FEMALE], c[SG_MALE],
                w[SG_FEMALE], w[SG_MALE],
            )
            econ = best_split_for_total(
                T, N[SG_ED], N[SG_NOT_ED],
                c[SG_ED], c[SG_NOT_ED],
                w[SG_ED], w[SG_NOT_ED],
            )
            if gender is None or econ is None:
                return np.inf
            return gender[0] + econ[0]

        result = discrete_convex_minimize(eval_T, lo, hi)
        if result is not None and np.isfinite(result[1]):
            T = int(result[0])
            gender = best_split_for_total(
                T, N[SG_FEMALE], N[SG_MALE],
                c[SG_FEMALE], c[SG_MALE],
                w[SG_FEMALE], w[SG_MALE],
            )
            econ = best_split_for_total(
                T, N[SG_ED], N[SG_NOT_ED],
                c[SG_ED], c[SG_NOT_ED],
                w[SG_ED], w[SG_NOT_ED],
            )

            projected[row_by_sg[SG_FEMALE]] = gender[1]
            projected[row_by_sg[SG_MALE]] = gender[2]
            projected[row_by_sg[SG_ED]] = econ[1]
            projected[row_by_sg[SG_NOT_ED]] = econ[2]

            status = "gender_econ_no_all"

    meta = {
        "status": status,
        "n_projected": int(len(projected)),
        "has_all": bool(has_all),
        "has_gender": bool(has_gender),
        "has_econ": bool(has_econ),
        "gender_n_ok": bool(gender_n_ok),
        "econ_n_ok": bool(econ_n_ok),
        "pair_pair_ok": bool(pair_pair_ok),
    }

    return projected, meta

# ------------------------------------------------------------
# 6. Constraint audit on true train counts
# ------------------------------------------------------------

def audit_true_constraints(df, block_col):
    rows = []

    for block, g in df.groupby(block_col, sort=False):
        sub = {}
        for sg, gg in g.groupby("SUBGROUP_NAME", sort=False):
            if str(sg) in KNOWN_SUBGROUPS and len(gg) == 1:
                sub[str(sg)] = gg.index[0]

        if len(sub) < 3:
            continue

        row = {
            "block_col": block_col,
            "block": block,
            "n_known_subgroups": len(sub),
            "has_all": SG_ALL in sub,
            "has_gender": SG_FEMALE in sub and SG_MALE in sub,
            "has_econ": SG_ED in sub and SG_NOT_ED in sub,
        }

        if row["has_all"] and row["has_gender"]:
            n_ok = (
                int(df.loc[sub[SG_ALL], "N_INT"])
                == int(df.loc[sub[SG_FEMALE], "N_INT"]) + int(df.loc[sub[SG_MALE], "N_INT"])
            )
            k_ok = (
                int(df.loc[sub[SG_ALL], "K_TRUE_ROUND"])
                == int(df.loc[sub[SG_FEMALE], "K_TRUE_ROUND"]) + int(df.loc[sub[SG_MALE], "K_TRUE_ROUND"])
            )
            row["gender_N_identity"] = bool(n_ok)
            row["gender_K_identity"] = bool(k_ok)

        if row["has_all"] and row["has_econ"]:
            n_ok = (
                int(df.loc[sub[SG_ALL], "N_INT"])
                == int(df.loc[sub[SG_ED], "N_INT"]) + int(df.loc[sub[SG_NOT_ED], "N_INT"])
            )
            k_ok = (
                int(df.loc[sub[SG_ALL], "K_TRUE_ROUND"])
                == int(df.loc[sub[SG_ED], "K_TRUE_ROUND"]) + int(df.loc[sub[SG_NOT_ED], "K_TRUE_ROUND"])
            )
            row["econ_N_identity"] = bool(n_ok)
            row["econ_K_identity"] = bool(k_ok)

        rows.append(row)

    out = pd.DataFrame(rows)
    return out

audit_dsa = audit_true_constraints(train57, "BLOCK_DISTRICT_SCHOOL_ASSESSMENT")
audit_sa = audit_true_constraints(train57, "BLOCK_SCHOOL_ASSESSMENT")

def summarize_audit(audit, name):
    print(f"\nConstraint audit: {name}")
    print("-" * (18 + len(name)))
    if len(audit) == 0:
        print("No auditable blocks.")
        return

    for c in ["gender_N_identity", "gender_K_identity", "econ_N_identity", "econ_K_identity"]:
        if c in audit.columns:
            valid = audit[c].notna()
            if valid.any():
                print(
                    f"{c:20s} valid_blocks={int(valid.sum()):6d} "
                    f"match_rate={float(audit.loc[valid, c].mean()):.6f}"
                )

summarize_audit(audit_dsa, "DISTRICT × SCHOOL × ASSESSMENT")
summarize_audit(audit_sa, "SCHOOL × ASSESSMENT")

# ------------------------------------------------------------
# 7. Apply projection to train/test frames
# ------------------------------------------------------------

def apply_projection(df, block_col, prior_col="PRED_50B"):
    k_prior = np.rint(df[prior_col].to_numpy(dtype=np.float64) / 100.0 * df["N_INT"].to_numpy(dtype=np.float64))
    k_prior = np.minimum(np.maximum(k_prior, 0), df["N_INT"].to_numpy(dtype=np.int64)).astype(np.int64)

    k_proj = k_prior.copy()
    projected_mask = np.zeros(len(df), dtype=bool)

    meta_rows = []

    for block, g in df.groupby(block_col, sort=False):
        projected, meta = project_block_counts(g, prior_col=prior_col)

        for idx, k in projected.items():
            k_proj[int(idx)] = int(k)
            projected_mask[int(idx)] = True

        meta["block_col"] = block_col
        meta["block"] = block
        meta["block_rows"] = int(len(g))
        meta_rows.append(meta)

    meta_df = pd.DataFrame(meta_rows)

    n = df["N_INT"].to_numpy(dtype=np.float64)

    pct_exact = np.clip(100.0 * k_proj.astype(np.float64) / np.maximum(n, 1), 0, 100).astype(np.float32)
    pct_reported_integer = np.clip(np.rint(pct_exact), 0, 100).astype(np.float32)

    return {
        "k_prior": k_prior,
        "k_projected": k_proj,
        "projected_mask": projected_mask,
        "pct_exact": pct_exact,
        "pct_reported_integer": pct_reported_integer,
        "meta": meta_df,
    }

projection_specs = [
    {
        "name": "dsa",
        "block_col": "BLOCK_DISTRICT_SCHOOL_ASSESSMENT",
    },
    {
        "name": "sa",
        "block_col": "BLOCK_SCHOOL_ASSESSMENT",
    },
]

projection_outputs = {}

for spec in projection_specs:
    name = spec["name"]
    block_col = spec["block_col"]

    print("\n" + "=" * 90)
    print(f"Applying projection: {name} using {block_col}")
    print("=" * 90)

    train_proj = apply_projection(train57, block_col, prior_col="PRED_50B")
    test_proj = apply_projection(test57, block_col, prior_col="PRED_50B")

    projection_outputs[name] = {
        "train": train_proj,
        "test": test_proj,
        "block_col": block_col,
    }

    print("Train projected rows:", int(train_proj["projected_mask"].sum()), "/", n_train)
    print("Test projected rows: ", int(test_proj["projected_mask"].sum()), "/", n_test)

    print("\nProjection status counts train:")
    print(train_proj["meta"]["status"].value_counts(dropna=False).to_string())

    print("\nProjection status counts test:")
    print(test_proj["meta"]["status"].value_counts(dropna=False).to_string())

# ------------------------------------------------------------
# 8. Direct candidate and protected blend scoring
# ------------------------------------------------------------

def tier_weighted_mse_57a(pred):
    total = 0.0
    for tier, mask in tier_masks_oof.items():
        total += test_tier_rates[tier] * float(mean_squared_error(y57[mask], pred[mask]))
    return float(total)

def tier_metrics_57a(pred):
    out = {}
    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y57[mask], pred[mask]))
        out[f"gain_{tier}_vs_50b"] = (
            float(mean_squared_error(y57[mask], pred50b_oof[mask])) - out[f"mse_{tier}"]
        )
    return out

def protected_blend(candidate_oof, candidate_test, lams, is_test=False):
    if is_test:
        base = pred50b_test.astype(np.float64)
        alt = candidate_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred50b_oof.astype(np.float64)
        alt = candidate_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)

lambda_grid = np.unique(np.concatenate([
    np.linspace(0.0, 1.50, 601),
    np.array([0.0, 0.005, 0.05, 0.10, 0.25, 0.50, 0.75, 1.0])
]))

def best_lambda_nonnegative(candidate_oof, mask):
    if int(mask.sum()) == 0:
        return 0.0

    y = y57[mask].astype(np.float64)
    base = pred50b_oof[mask].astype(np.float64)
    alt = candidate_oof[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = float(mean_squared_error(y, base))

    for lam in lambda_grid:
        p = np.clip(base + float(lam) * (alt - base), 0, 100)
        mse = float(mean_squared_error(y, p))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam

direct_rows = []
screen_rows = []
store = {}

for proj_name, obj in projection_outputs.items():
    for pct_kind in ["pct_exact", "pct_reported_integer"]:
        cand_oof = pred50b_oof.copy().astype(np.float32)
        cand_test = pred50b_test.copy().astype(np.float32)

        # Only projected rows are changed.
        train_mask = obj["train"]["projected_mask"]
        test_mask = obj["test"]["projected_mask"]

        cand_oof[train_mask] = obj["train"][pct_kind][train_mask]
        cand_test[test_mask] = obj["test"][pct_kind][test_mask]

        direct_mse = float(mean_squared_error(y57, cand_oof))
        direct_weighted = tier_weighted_mse_57a(cand_oof)

        drow = {
            "projection": proj_name,
            "pct_kind": pct_kind,
            "direct_oof_mse": direct_mse,
            "direct_gain_vs_50b": mse50b - direct_mse,
            "direct_public_weighted_mse": direct_weighted,
            "direct_public_weighted_gain_vs_50b": tier_weighted_mse_57a(pred50b_oof) - direct_weighted,
            "train_projected_rows": int(train_mask.sum()),
            "test_projected_rows": int(test_mask.sum()),
            "mean_abs_diff_vs_50b_test": float(np.mean(np.abs(cand_test - pred50b_test))),
            "p95_abs_diff_vs_50b_test": float(np.percentile(np.abs(cand_test - pred50b_test), 95)),
            "max_abs_diff_vs_50b_test": float(np.max(np.abs(cand_test - pred50b_test))),
        }
        drow.update(tier_metrics_57a(cand_oof))
        direct_rows.append(drow)

        # Tier-protected blends.
        lam_solver = best_lambda_nonnegative(cand_oof, solver_only_oof)
        lam_none = best_lambda_nonnegative(cand_oof, none_oof)

        # Tiny overlap allowed only if it helps.
        overlap_grid = [0.0, 0.0025, 0.005, 0.01]
        best_overlap = 0.0
        base_overlap_mse = float(mean_squared_error(y57[overlap_oof], pred50b_oof[overlap_oof]))
        best_overlap_mse = base_overlap_mse

        for lo in overlap_grid:
            p_tmp = pred50b_oof[overlap_oof] + float(lo) * (cand_oof[overlap_oof] - pred50b_oof[overlap_oof])
            p_tmp = np.clip(p_tmp, 0, 100)
            m = float(mean_squared_error(y57[overlap_oof], p_tmp))
            if m < best_overlap_mse:
                best_overlap_mse = m
                best_overlap = float(lo)

        strategies = {
            "solver_only": {
                "overlap": 0.0,
                "solver_only": lam_solver,
                "none": 0.0,
            },
            "none_only": {
                "overlap": 0.0,
                "solver_only": 0.0,
                "none": lam_none,
            },
            "solver_plus_none": {
                "overlap": 0.0,
                "solver_only": lam_solver,
                "none": lam_none,
            },
            "all_tiers_tiny_overlap": {
                "overlap": best_overlap,
                "solver_only": lam_solver,
                "none": lam_none,
            },
        }

        for strategy, lams in strategies.items():
            pred_oof = protected_blend(cand_oof, cand_test, lams, is_test=False)
            pred_test = protected_blend(cand_oof, cand_test, lams, is_test=True)

            mse = float(mean_squared_error(y57, pred_oof))
            weighted = tier_weighted_mse_57a(pred_oof)

            row = {
                "projection": proj_name,
                "pct_kind": pct_kind,
                "strategy": strategy,
                "lambda_overlap": float(lams["overlap"]),
                "lambda_solver_only": float(lams["solver_only"]),
                "lambda_none": float(lams["none"]),
                "oof_mse": mse,
                "gain_vs_50b": mse50b - mse,
                "public_weighted_mse": weighted,
                "public_weighted_gain_vs_50b": tier_weighted_mse_57a(pred50b_oof) - weighted,
                "mean_abs_diff_vs_50b_test": float(np.mean(np.abs(pred_test - pred50b_test))),
                "p95_abs_diff_vs_50b_test": float(np.percentile(np.abs(pred_test - pred50b_test), 95)),
                "max_abs_diff_vs_50b_test": float(np.max(np.abs(pred_test - pred50b_test))),
            }
            row.update(tier_metrics_57a(pred_oof))

            key = f"{proj_name}__{pct_kind}__{strategy}"
            row["key"] = key

            screen_rows.append(row)
            store[key] = {
                "oof": pred_oof,
                "test": pred_test,
                "lams": lams,
                "direct_oof": cand_oof,
                "direct_test": cand_test,
            }

direct57a = pd.DataFrame(direct_rows).sort_values(["direct_public_weighted_mse", "direct_oof_mse"]).reset_index(drop=True)
screen57a = pd.DataFrame(screen_rows).sort_values(["public_weighted_mse", "oof_mse"]).reset_index(drop=True)
screen57a_oof = screen57a.sort_values(["oof_mse", "public_weighted_mse"]).reset_index(drop=True)

print("\nDirect projection candidates")
print("----------------------------")
display(direct57a)

print("\nTop protected blend candidates by public-weighted OOF")
print("-----------------------------------------------------")
display(screen57a.head(20))

print("\nTop protected blend candidates by ordinary OOF")
print("----------------------------------------------")
display(screen57a_oof.head(20))

# ------------------------------------------------------------
# 9. Fold diagnostics for best candidates
# ------------------------------------------------------------

best_weighted = screen57a.iloc[0]
best_oof = screen57a_oof.iloc[0]

best_weighted_key = best_weighted["key"]
best_oof_key = best_oof["key"]

best_weighted_oof = store[best_weighted_key]["oof"]
best_weighted_test = store[best_weighted_key]["test"]

best_oof_pred = store[best_oof_key]["oof"]
best_oof_test = store[best_oof_key]["test"]

folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

def fold_diag(pred, label):
    rows = []

    for fold_num, (_, va_idx) in enumerate(folds, start=1):
        row = {
            "label": label,
            "fold": fold_num,
            "mse_50b": float(mean_squared_error(y57[va_idx], pred50b_oof[va_idx])),
            "mse_candidate": float(mean_squared_error(y57[va_idx], pred[va_idx])),
        }
        row["gain_vs_50b"] = row["mse_50b"] - row["mse_candidate"]

        for tier, mask_full in tier_masks_oof.items():
            mask = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask.sum())
            if int(mask.sum()) > 0:
                row[f"mse50b_{tier}"] = float(mean_squared_error(y57[va_idx][mask], pred50b_oof[va_idx][mask]))
                row[f"msecand_{tier}"] = float(mean_squared_error(y57[va_idx][mask], pred[va_idx][mask]))
                row[f"gain_{tier}_vs_50b"] = row[f"mse50b_{tier}"] - row[f"msecand_{tier}"]

        rows.append(row)

    return pd.DataFrame(rows)

fold_weighted57a = fold_diag(best_weighted_oof, "best_weighted")
fold_oof57a = fold_diag(best_oof_pred, "best_oof")
fold_diag57a = pd.concat([fold_weighted57a, fold_oof57a], axis=0).reset_index(drop=True)

print("\nBest public-weighted candidate")
print("------------------------------")
print(best_weighted.to_string())

print("\nBest ordinary OOF candidate")
print("---------------------------")
print(best_oof.to_string())

print("\nFold diagnostics")
print("----------------")
print(fold_diag57a.to_string(index=False))

print("\nMin fold gains")
print("--------------")
for label, fd in [("best_weighted", fold_weighted57a), ("best_oof", fold_oof57a)]:
    print(
        f"{label:15s} min_gain={fd['gain_vs_50b'].min(): .6f} "
        f"mean_gain={fd['gain_vs_50b'].mean(): .6f}"
    )

# ------------------------------------------------------------
# 10. Save artifacts and submissions
# ------------------------------------------------------------

audit_dsa.to_csv("model_results/int57a_true_constraint_audit_district_school_assessment.csv", index=False)
audit_sa.to_csv("model_results/int57a_true_constraint_audit_school_assessment.csv", index=False)

for proj_name, obj in projection_outputs.items():
    obj["train"]["meta"].to_csv(f"model_results/int57a_projection_meta_train_{proj_name}.csv", index=False)
    obj["test"]["meta"].to_csv(f"model_results/int57a_projection_meta_test_{proj_name}.csv", index=False)

direct57a.to_csv("model_results/int57a_direct_projection_screen.csv", index=False)
screen57a.to_csv("model_results/int57a_blend_screen.csv", index=False)
screen57a_oof.to_csv("model_results/int57a_blend_screen_oof_sorted.csv", index=False)
fold_diag57a.to_csv("model_results/int57a_best_fold_diag.csv", index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y57,
    "pred_50b": pred50b_oof,
    "pred_clipped": best_weighted_oof,
    "tier": tier_oof,
}).to_csv("model_results/oof_int57a_best_weighted.csv", index=False)

pd.DataFrame({
    ID_COL: test_ids57a,
    "pred_50b": pred50b_test,
    TARGET_COL: best_weighted_test,
    "tier": tier_test,
}).to_csv("model_results/testpred_int57a_best_weighted.csv", index=False)

pd.DataFrame({
    ID_COL: test_ids57a,
    TARGET_COL: best_weighted_test,
}).to_csv("submission_int57a_best_weighted.csv", index=False)

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y57,
    "pred_50b": pred50b_oof,
    "pred_clipped": best_oof_pred,
    "tier": tier_oof,
}).to_csv("model_results/oof_int57a_best_oof.csv", index=False)

pd.DataFrame({
    ID_COL: test_ids57a,
    "pred_50b": pred50b_test,
    TARGET_COL: best_oof_test,
    "tier": tier_test,
}).to_csv("model_results/testpred_int57a_best_oof.csv", index=False)

pd.DataFrame({
    ID_COL: test_ids57a,
    TARGET_COL: best_oof_test,
}).to_csv("submission_int57a_best_oof.csv", index=False)

# Save top 5 weighted candidates.
top_paths = []
for i, row in screen57a.head(5).iterrows():
    key = row["key"]
    out_path = f"submission_int57a_top{i+1}_weighted.csv"
    pd.DataFrame({
        ID_COL: test_ids57a,
        TARGET_COL: store[key]["test"],
    }).to_csv(out_path, index=False)
    top_paths.append(out_path)

# Validate submissions.
for p in ["submission_int57a_best_weighted.csv", "submission_int57a_best_oof.csv"] + top_paths:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test, 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[ID_COL].notna().all(), p
    assert sub[TARGET_COL].notna().all(), p
    assert np.isfinite(sub[TARGET_COL]).all(), p
    assert sub[TARGET_COL].between(0, 100).all(), p

print("\nSaved files")
print("-----------")
print("model_results/int57a_true_constraint_audit_district_school_assessment.csv")
print("model_results/int57a_true_constraint_audit_school_assessment.csv")
print("model_results/int57a_direct_projection_screen.csv")
print("model_results/int57a_blend_screen.csv")
print("model_results/int57a_blend_screen_oof_sorted.csv")
print("model_results/int57a_best_fold_diag.csv")
print("model_results/oof_int57a_best_weighted.csv")
print("model_results/testpred_int57a_best_weighted.csv")
print("submission_int57a_best_weighted.csv")
print("model_results/oof_int57a_best_oof.csv")
print("model_results/testpred_int57a_best_oof.csv")
print("submission_int57a_best_oof.csv")
for p in top_paths:
    print(p)

print("\nSubmission validation")
print("---------------------")
for p in ["submission_int57a_best_weighted.csv", "submission_int57a_best_oof.csv"]:
    sub = pd.read_csv(p)
    print(p, sub.shape, sub[TARGET_COL].describe().to_dict())

# ------------------------------------------------------------
# 11. Decision rule
# ------------------------------------------------------------

print("\nDecision rule")
print("-------------")

gain = float(best_weighted["gain_vs_50b"])
weighted_gain = float(best_weighted["public_weighted_gain_vs_50b"])
solver_gain = float(best_weighted["gain_solver_only_vs_50b"])
none_gain = float(best_weighted["gain_none_vs_50b"])
overlap_gain = float(best_weighted["gain_overlap_vs_50b"])
min_fold_gain = float(fold_weighted57a["gain_vs_50b"].min())

if gain >= 1.0 and weighted_gain >= 0.50 and min_fold_gain >= 0 and solver_gain > 0 and overlap_gain > -0.002:
    print("57A found strong count-projection signal. Worth a public probe.")
elif gain >= 0.50 and weighted_gain >= 0.25 and min_fold_gain >= 0 and solver_gain > 0 and overlap_gain > -0.002:
    print("57A found moderate count-projection signal. Inspect carefully before submitting.")
elif gain > 0 and solver_gain > 0 and overlap_gain > -0.002:
    print("57A found only small count-projection signal. Probably not worth submitting unless public slots are expendable.")
else:
    print("57A does not improve enough over public-validated 50B. Keep 50B protected.")

print("\nProtected current best remains:")
print("submission_mf50b_best_oof.csv | public MSE 29.448")

57A. Global integer-count feasibility solver audit

Artifact checks
---------------
model_results/oof_mf50b_best_oof.csv                              True
model_results/testpred_mf50b_best_oof.csv                         True
model_results/oof_hybrid44b_best.csv                              True
model_results/testpred_hybrid44b_best.csv                         True
model_results/account39a_raw_oof_reconstruction.csv               True
model_results/account39a_raw_test_reconstruction.csv              True
model_results/account39b_solver_raw_oof.csv                       True
model_results/account39b_solver_raw_test.csv                      True

Raw shapes
----------
raw_train_te: (144921, 62)
raw_test_te:  (48307, 61)

References
----------
50B OOF MSE: 51.463158 | public 29.448
44B OOF MSE: 52.780037 | public 30.556

Tier coverage
-------------
overlap      train= 53786 test= 27298 50B MSE=0.506983
solver_only  train= 27807 test= 17807 50B MSE=63.484444
none         train= 63328 test=

,projection,pct_kind,direct_oof_mse,direct_gain_vs_50b,direct_public_weighted_mse,direct_public_weighted_gain_vs_50b,train_projected_rows,test_projected_rows,mean_abs_diff_vs_50b_test,p95_abs_diff_vs_50b_test,max_abs_diff_vs_50b_test,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b
0,dsa,pct_reported_integer,51.812840,-0.349682,30.292468,-0.674243,91337,3676,0.077876,0.404477,10.129333,0.567062,-0.060080,65.224075,-1.739632,89.448303,0.014679
1,sa,pct_reported_integer,51.812840,-0.349682,30.292468,-0.674243,91337,3676,0.077876,0.404477,10.129333,0.567062,-0.060080,65.224075,-1.739632,89.448303,0.014679
2,dsa,pct_exact,51.821228,-0.358070,30.312444,-0.694219,91337,3676,0.073869,0.342083,10.129333,0.617919,-0.110937,65.202950,-1.718506,89.433586,0.029396
3,sa,pct_exact,51.821228,-0.358070,30.312444,-0.694219,91337,3676,0.073869,0.342083,10.129333,0.617919,-0.110937,65.202950,-1.718506,89.433586,0.029396



Top protected blend candidates by public-weighted OOF
-----------------------------------------------------


,projection,pct_kind,strategy,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b,public_weighted_mse,public_weighted_gain_vs_50b,mean_abs_diff_vs_50b_test,p95_abs_diff_vs_50b_test,max_abs_diff_vs_50b_test,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b,key
0,dsa,pct_reported_integer,all_tiers_tiny_overlap,0.01,0.195,0.5075,51.345879,0.117279,29.562583,0.055641,0.013956,0.056841,1.975220,0.505413,0.001569,63.375309,0.109135,89.243851,0.219131,dsa__pct_reported_integer__all_tiers_tiny_overlap
1,sa,pct_reported_integer,all_tiers_tiny_overlap,0.01,0.195,0.5075,51.345879,0.117279,29.562583,0.055641,0.013956,0.056841,1.975220,0.505413,0.001569,63.375309,0.109135,89.243851,0.219131,sa__pct_reported_integer__all_tiers_tiny_overlap
2,dsa,pct_reported_integer,solver_plus_none,0.00,0.195,0.5075,51.346462,0.116695,29.563470,0.054754,0.013889,0.056841,1.975220,0.506983,0.000000,63.375309,0.109135,89.243851,0.219131,dsa__pct_reported_integer__solver_plus_none
3,sa,pct_reported_integer,solver_plus_none,0.00,0.195,0.5075,51.346462,0.116695,29.563470,0.054754,0.013889,0.056841,1.975220,0.506983,0.000000,63.375309,0.109135,89.243851,0.219131,sa__pct_reported_integer__solver_plus_none
4,dsa,pct_exact,all_tiers_tiny_overlap,0.01,0.190,0.5175,51.345642,0.117516,29.566038,0.052187,0.013156,0.050422,1.924573,0.506628,0.000355,63.383766,0.100677,89.238579,0.224403,dsa__pct_exact__all_tiers_tiny_overlap
5,sa,pct_exact,all_tiers_tiny_overlap,0.01,0.190,0.5175,51.345642,0.117516,29.566038,0.052187,0.013156,0.050422,1.924573,0.506628,0.000355,63.383766,0.100677,89.238579,0.224403,sa__pct_exact__all_tiers_tiny_overlap
6,dsa,pct_exact,solver_plus_none,0.00,0.190,0.5175,51.345776,0.117382,29.566238,0.051986,0.013107,0.050422,1.924573,0.506983,0.000000,63.383766,0.100677,89.238579,0.224403,dsa__pct_exact__solver_plus_none
7,sa,pct_exact,solver_plus_none,0.00,0.190,0.5175,51.345776,0.117382,29.566238,0.051986,0.013107,0.050422,1.924573,0.506983,0.000000,63.383766,0.100677,89.238579,0.224403,sa__pct_exact__solver_plus_none
8,dsa,pct_reported_integer,solver_only,0.00,0.195,0.0000,51.442215,0.020943,29.577995,0.040229,0.013889,0.056841,1.975220,0.506983,0.000000,63.375309,0.109135,89.462982,0.000000,dsa__pct_reported_integer__solver_only
9,sa,pct_reported_integer,solver_only,0.00,0.195,0.0000,51.442215,0.020943,29.577995,0.040229,0.013889,0.056841,1.975220,0.506983,0.000000,63.375309,0.109135,89.462982,0.000000,sa__pct_reported_integer__solver_only



Top protected blend candidates by ordinary OOF
----------------------------------------------


,projection,pct_kind,strategy,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b,public_weighted_mse,public_weighted_gain_vs_50b,mean_abs_diff_vs_50b_test,p95_abs_diff_vs_50b_test,max_abs_diff_vs_50b_test,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b,key
0,dsa,pct_exact,all_tiers_tiny_overlap,0.01,0.190,0.5175,51.345642,0.117516,29.566038,0.052187,0.013156,0.050422,1.924573,0.506628,0.000355,63.383766,0.100677,89.238579,0.224403,dsa__pct_exact__all_tiers_tiny_overlap
1,sa,pct_exact,all_tiers_tiny_overlap,0.01,0.190,0.5175,51.345642,0.117516,29.566038,0.052187,0.013156,0.050422,1.924573,0.506628,0.000355,63.383766,0.100677,89.238579,0.224403,sa__pct_exact__all_tiers_tiny_overlap
2,dsa,pct_exact,solver_plus_none,0.00,0.190,0.5175,51.345776,0.117382,29.566238,0.051986,0.013107,0.050422,1.924573,0.506983,0.000000,63.383766,0.100677,89.238579,0.224403,dsa__pct_exact__solver_plus_none
3,sa,pct_exact,solver_plus_none,0.00,0.190,0.5175,51.345776,0.117382,29.566238,0.051986,0.013107,0.050422,1.924573,0.506983,0.000000,63.383766,0.100677,89.238579,0.224403,sa__pct_exact__solver_plus_none
4,dsa,pct_reported_integer,all_tiers_tiny_overlap,0.01,0.195,0.5075,51.345879,0.117279,29.562583,0.055641,0.013956,0.056841,1.975220,0.505413,0.001569,63.375309,0.109135,89.243851,0.219131,dsa__pct_reported_integer__all_tiers_tiny_overlap
5,sa,pct_reported_integer,all_tiers_tiny_overlap,0.01,0.195,0.5075,51.345879,0.117279,29.562583,0.055641,0.013956,0.056841,1.975220,0.505413,0.001569,63.375309,0.109135,89.243851,0.219131,sa__pct_reported_integer__all_tiers_tiny_overlap
6,dsa,pct_reported_integer,solver_plus_none,0.00,0.195,0.5075,51.346462,0.116695,29.563470,0.054754,0.013889,0.056841,1.975220,0.506983,0.000000,63.375309,0.109135,89.243851,0.219131,dsa__pct_reported_integer__solver_plus_none
7,sa,pct_reported_integer,solver_plus_none,0.00,0.195,0.5075,51.346462,0.116695,29.563470,0.054754,0.013889,0.056841,1.975220,0.506983,0.000000,63.375309,0.109135,89.243851,0.219131,sa__pct_reported_integer__solver_plus_none
8,dsa,pct_exact,none_only,0.00,0.000,0.5175,51.365097,0.098061,29.603350,0.014874,0.000000,0.000000,0.000000,0.506983,0.000000,63.484444,0.000000,89.238579,0.224403,dsa__pct_exact__none_only
9,sa,pct_exact,none_only,0.00,0.000,0.5175,51.365097,0.098061,29.603350,0.014874,0.000000,0.000000,0.000000,0.506983,0.000000,63.484444,0.000000,89.238579,0.224403,sa__pct_exact__none_only



Best public-weighted candidate
------------------------------
projection                                                                   dsa
pct_kind                                                    pct_reported_integer
strategy                                                  all_tiers_tiny_overlap
lambda_overlap                                                              0.01
lambda_solver_only                                                         0.195
lambda_none                                                               0.5075
oof_mse                                                                51.345879
gain_vs_50b                                                             0.117279
public_weighted_mse                                                    29.562583
public_weighted_gain_vs_50b                                             0.055641
mean_abs_diff_vs_50b_test                                               0.013956
p95_abs_diff_vs_50b_test                      

In [11]:
# ============================================================
# 58A. Expanded fullfit MF local search around public-winning 50B fullfit
# ============================================================
#
# Current protected public best:
#   submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform.csv
#   public MSE = 29.334
#
# Motivation:
#   Pure full-train refit of 50B rank16/reg500 beat fold-averaged 50B:
#
#       50B fold-averaged public: 29.448
#       50B fullfit rank16/reg500: 29.334
#
#   Therefore, search harder inside the successful MF/fullfit family.
#
# Model family:
#   base = 44B accounting hybrid
#   residual_z = arcsin_sqrt(y) - arcsin_sqrt(base)
#   residual_z ≈ global
#              + school_bias
#              + assessment_subgroup_bias
#              + school_factor · assessment_subgroup_factor
#
# This cell:
#   1. Runs an OOF-safe local screen around rank16/reg500.
#   2. Selects top local configs by OOF / public-weighted OOF / stability.
#   3. Full-train refits selected configs.
#   4. Saves pure fullfit submissions.
#   5. Saves blends with the current public-best fullfit anchor.
#
# No Torch. No LightGBM. No external data.
# ============================================================

import os
import gc
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

Path("model_results").mkdir(exist_ok=True)
CHECKPOINT_DIR_58A = Path("model_results/mf58a_checkpoints")
CHECKPOINT_DIR_58A.mkdir(exist_ok=True)

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5

print("=" * 90)
print("58A. Expanded fullfit MF local search around public-winning 50B fullfit")
print("=" * 90)
print("Checkpoint dir:", CHECKPOINT_DIR_58A)

# ------------------------------------------------------------
# 1. Required raw data and artifacts
# ------------------------------------------------------------

if "raw_train_te" not in globals() or "raw_test_te" not in globals():
    raise RuntimeError(
        "raw_train_te/raw_test_te missing. Run setup/merge/build cells first, then raw aliases."
    )

required_files = [
    "model_results/oof_hybrid44b_best.csv",
    "model_results/testpred_hybrid44b_best.csv",
    "model_results/oof_mf50b_best_oof.csv",
    "model_results/testpred_mf50b_best_oof.csv",
    "model_results/account39a_raw_oof_reconstruction.csv",
    "model_results/account39a_raw_test_reconstruction.csv",
    "model_results/account39b_solver_raw_oof.csv",
    "model_results/account39b_solver_raw_test.csv",
    "submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform.csv",
]

print("\nArtifact checks")
print("---------------")
for p in required_files:
    print(f"{p:75s}", Path(p).exists())

missing = [p for p in required_files if not Path(p).exists()]
if missing:
    raise RuntimeError(f"Missing required files: {missing}")

print("\nRaw shapes")
print("----------")
print("raw_train_te:", raw_train_te.shape)
print("raw_test_te: ", raw_test_te.shape)

# ------------------------------------------------------------
# 2. Load helpers
# ------------------------------------------------------------

def load_oof_58a(path):
    df = pd.read_csv(path)
    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if "pred_clipped" in df.columns:
        pred = df["pred_clipped"].to_numpy(dtype=np.float32)
    elif TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column found in {path}")

    y = df[TARGET_COL].to_numpy(dtype=np.float32)
    return df, y, np.clip(pred, 0, 100).astype(np.float32)


def load_test_58a(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred = df[TARGET_COL].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"No prediction column found in {path}")
        pred = df[numeric_cols[0]].to_numpy(dtype=np.float32)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column and no test_ids available.")

    return df, ids, np.clip(pred, 0, 100).astype(np.float32)


# 44B is the MF base.
_, y58_file, pred44b_oof = load_oof_58a("model_results/oof_hybrid44b_best.csv")
_, test_ids58a, pred44b_test = load_test_58a("model_results/testpred_hybrid44b_best.csv")

# 50B fold-averaged anchor for comparison.
_, y50b_file, pred50b_oof = load_oof_58a("model_results/oof_mf50b_best_oof.csv")
_, _, pred50b_test = load_test_58a("model_results/testpred_mf50b_best_oof.csv")

if "y_train" in globals():
    y58 = np.asarray(y_train, dtype=np.float32).reshape(-1)
else:
    y58 = y58_file.copy()

assert len(y58) == len(pred44b_oof)
assert len(raw_train_te) == len(pred44b_oof)
assert len(raw_test_te) == len(pred44b_test)
assert np.max(np.abs(y58 - y58_file)) < 1e-5
assert np.max(np.abs(y58 - y50b_file)) < 1e-5

n_train = len(y58)
n_test = len(pred44b_test)

mse44b = float(mean_squared_error(y58, pred44b_oof))
mse50b = float(mean_squared_error(y58, pred50b_oof))

# Current public-best fullfit anchor.
anchor_sub = pd.read_csv("submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform.csv")
assert anchor_sub.shape == (n_test, 2)
assert list(anchor_sub.columns) == [ID_COL, TARGET_COL]
assert np.array_equal(anchor_sub[ID_COL].to_numpy(), test_ids58a)

pred_fullfit_anchor_test = anchor_sub[TARGET_COL].to_numpy(dtype=np.float32)
pred_fullfit_anchor_test = np.clip(pred_fullfit_anchor_test, 0, 100).astype(np.float32)

print("\nReferences")
print("----------")
print(f"44B OOF MSE:       {mse44b:.6f} | public 30.556")
print(f"50B fold OOF MSE:  {mse50b:.6f} | public 29.448")
print("50B fullfit anchor: submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform.csv | public 29.334")

# ------------------------------------------------------------
# 3. Reconstruct tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

tier_oof = np.array(["none"] * n_train, dtype=object)
tier_oof[solver_only_oof] = "solver_only"
tier_oof[overlap_oof] = "overlap"

tier_test = np.array(["none"] * n_test, dtype=object)
tier_test[solver_only_test] = "solver_only"
tier_test[overlap_test] = "overlap"

test_tier_rates = {
    tier: float(mask.mean())
    for tier, mask in tier_masks_test.items()
}

print("\nTier coverage")
print("-------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"50B OOF tier MSE={mean_squared_error(y58[mask], pred50b_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 4. Build matrix indices
# ------------------------------------------------------------

def clean_str_58a(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str).to_numpy()

for c in ["SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]:
    if c not in raw_train_te.columns or c not in raw_test_te.columns:
        raise RuntimeError(f"Missing required raw column: {c}")

school_train = clean_str_58a(raw_train_te["SCHOOL"])
school_test = clean_str_58a(raw_test_te["SCHOOL"])

assessment_train = clean_str_58a(raw_train_te["ASSESSMENT_NAME"])
assessment_test = clean_str_58a(raw_test_te["ASSESSMENT_NAME"])

subgroup_train = clean_str_58a(raw_train_te["SUBGROUP_NAME"])
subgroup_test = clean_str_58a(raw_test_te["SUBGROUP_NAME"])

asg_train = np.array([f"{a}||{s}" for a, s in zip(assessment_train, subgroup_train)], dtype=object)
asg_test = np.array([f"{a}||{s}" for a, s in zip(assessment_test, subgroup_test)], dtype=object)

n_students_train = (
    pd.to_numeric(raw_train_te["N_STUDENTS"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .to_numpy(dtype=np.float64)
)

def make_index_pair(train_arr, test_arr):
    all_values = pd.Index(pd.Series(np.concatenate([train_arr, test_arr])).astype(str).unique())
    mapper = {v: i for i, v in enumerate(all_values)}
    train_idx = np.array([mapper[x] for x in train_arr], dtype=np.int32)
    test_idx = np.array([mapper[x] for x in test_arr], dtype=np.int32)
    return all_values, mapper, train_idx, test_idx

all_schools, _, school_idx_train, school_idx_test = make_index_pair(school_train, school_test)
all_asg, _, asg_idx_train, asg_idx_test = make_index_pair(asg_train, asg_test)

n_schools = len(all_schools)
n_asg = len(all_asg)

print("\nMF dimensions")
print("-------------")
print("n_schools:", n_schools)
print("n_assessment_subgroups:", n_asg)
print("train rows:", n_train)
print("test rows:", n_test)

# ------------------------------------------------------------
# 5. Transform helpers and residual target
# ------------------------------------------------------------

def pct_to_arc_58a(pct):
    p = np.clip(np.asarray(pct, dtype=np.float64) / 100.0, 1e-6, 1 - 1e-6)
    return np.arcsin(np.sqrt(p)).astype(np.float32)

def arc_to_pct_58a(z):
    p = np.sin(np.asarray(z, dtype=np.float64)) ** 2
    return np.clip(100.0 * p, 0, 100).astype(np.float32)

z_y = pct_to_arc_58a(y58)
z_base_oof = pct_to_arc_58a(pred44b_oof)
z_base_test = pct_to_arc_58a(pred44b_test)

resid_z = (z_y - z_base_oof).astype(np.float32)

# ------------------------------------------------------------
# 6. MF fit/predict helpers
# ------------------------------------------------------------

def make_weights_58a(idx, scheme="uniform"):
    idx = np.asarray(idx, dtype=np.int64)

    if scheme == "uniform":
        return np.ones(len(idx), dtype=np.float64)

    if scheme == "sqrt_n":
        n = n_students_train[idx].astype(np.float64)
        valid = np.isfinite(n) & (n > 0)
        med = np.nanmedian(n[valid]) if valid.any() else 30.0
        n = np.where(valid, n, med)
        return np.sqrt(np.clip(n, 1, 500)).astype(np.float64)

    raise ValueError(f"Unknown weight scheme: {scheme}")


def fit_biases_58a(idx, reg_school, reg_asg, weight_scheme="uniform", n_iter=4):
    idx = np.asarray(idx, dtype=np.int64)

    s_idx = school_idx_train[idx]
    a_idx = asg_idx_train[idx]
    r = resid_z[idx].astype(np.float64)
    w = make_weights_58a(idx, weight_scheme)

    global_bias = float(np.sum(w * r) / max(np.sum(w), 1e-12))

    school_bias = np.zeros(n_schools, dtype=np.float64)
    asg_bias = np.zeros(n_asg, dtype=np.float64)

    for _ in range(int(n_iter)):
        residual_for_school = r - global_bias - asg_bias[a_idx]
        sum_w_s = np.bincount(s_idx, weights=w, minlength=n_schools)
        sum_wr_s = np.bincount(s_idx, weights=w * residual_for_school, minlength=n_schools)
        school_bias = sum_wr_s / (sum_w_s + float(reg_school))

        residual_for_asg = r - global_bias - school_bias[s_idx]
        sum_w_a = np.bincount(a_idx, weights=w, minlength=n_asg)
        sum_wr_a = np.bincount(a_idx, weights=w * residual_for_asg, minlength=n_asg)
        asg_bias = sum_wr_a / (sum_w_a + float(reg_asg))

    debiased = r - global_bias - school_bias[s_idx] - asg_bias[a_idx]

    return global_bias, school_bias.astype(np.float32), asg_bias.astype(np.float32), debiased.astype(np.float32)


def fit_svd_58a(idx, debiased_values, rank, random_state):
    idx = np.asarray(idx, dtype=np.int64)

    rows = school_idx_train[idx]
    cols = asg_idx_train[idx]

    mat = sparse.csr_matrix(
        (debiased_values.astype(np.float32), (rows, cols)),
        shape=(n_schools, n_asg),
        dtype=np.float32,
    )

    rank_eff = int(min(rank, max(1, min(n_schools, n_asg) - 1)))

    svd = TruncatedSVD(
        n_components=rank_eff,
        n_iter=8,
        random_state=int(random_state),
    )

    school_factors = svd.fit_transform(mat).astype(np.float32)
    asg_components = svd.components_.astype(np.float32)

    return school_factors, asg_components


def predict_mf_58a(
    base_z,
    school_idx,
    asg_idx,
    global_bias,
    school_bias,
    asg_bias,
    school_factors,
    asg_components,
    bias_scale=1.0,
    factor_scale=1.0,
):
    bias_part = (
        float(global_bias)
        + school_bias[school_idx].astype(np.float64)
        + asg_bias[asg_idx].astype(np.float64)
    )
    bias_part *= float(bias_scale)

    factor_part = np.sum(
        school_factors[school_idx] * asg_components[:, asg_idx].T,
        axis=1,
    ).astype(np.float64)
    factor_part *= float(factor_scale)

    z_pred = base_z.astype(np.float64) + bias_part + factor_part
    return arc_to_pct_58a(z_pred)


def train_predict_oof_config_58a(cfg):
    folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

    oof = np.full(n_train, np.nan, dtype=np.float32)
    test_sum = np.zeros(n_test, dtype=np.float64)
    fold_rows = []

    for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
        global_bias, school_bias, asg_bias, debiased = fit_biases_58a(
            tr_idx,
            reg_school=cfg["reg"],
            reg_asg=cfg["reg"],
            weight_scheme=cfg["weight_scheme"],
            n_iter=cfg["bias_iter"],
        )

        school_factors, asg_components = fit_svd_58a(
            tr_idx,
            debiased,
            rank=cfg["rank"],
            random_state=RANDOM_STATE + 1000 * fold_num + int(cfg["rank"]),
        )

        pred_va = predict_mf_58a(
            base_z=z_base_oof[va_idx],
            school_idx=school_idx_train[va_idx],
            asg_idx=asg_idx_train[va_idx],
            global_bias=global_bias,
            school_bias=school_bias,
            asg_bias=asg_bias,
            school_factors=school_factors,
            asg_components=asg_components,
            bias_scale=cfg["bias_scale"],
            factor_scale=cfg["factor_scale"],
        )

        pred_te = predict_mf_58a(
            base_z=z_base_test,
            school_idx=school_idx_test,
            asg_idx=asg_idx_test,
            global_bias=global_bias,
            school_bias=school_bias,
            asg_bias=asg_bias,
            school_factors=school_factors,
            asg_components=asg_components,
            bias_scale=cfg["bias_scale"],
            factor_scale=cfg["factor_scale"],
        )

        oof[va_idx] = pred_va
        test_sum += pred_te.astype(np.float64)

        row = {
            "config": cfg["name"],
            "fold": fold_num,
            "direct_mse": float(mean_squared_error(y58[va_idx], pred_va)),
            "base44b_mse": float(mean_squared_error(y58[va_idx], pred44b_oof[va_idx])),
            "foldavg50b_mse": float(mean_squared_error(y58[va_idx], pred50b_oof[va_idx])),
        }
        row["direct_gain_vs_44b"] = row["base44b_mse"] - row["direct_mse"]
        row["direct_gain_vs_50b"] = row["foldavg50b_mse"] - row["direct_mse"]

        fold_rows.append(row)

        del school_factors, asg_components, school_bias, asg_bias, debiased
        gc.collect()

    test = (test_sum / N_SPLITS).astype(np.float32)
    return oof, test, pd.DataFrame(fold_rows)


def train_predict_fullfit_config_58a(cfg):
    tr_idx = np.arange(n_train)

    global_bias, school_bias, asg_bias, debiased = fit_biases_58a(
        tr_idx,
        reg_school=cfg["reg"],
        reg_asg=cfg["reg"],
        weight_scheme=cfg["weight_scheme"],
        n_iter=cfg["bias_iter"],
    )

    school_factors, asg_components = fit_svd_58a(
        tr_idx,
        debiased,
        rank=cfg["rank"],
        random_state=RANDOM_STATE + 9999 + int(cfg["rank"]) + int(cfg["reg"]),
    )

    pred_train = predict_mf_58a(
        base_z=z_base_oof,
        school_idx=school_idx_train,
        asg_idx=asg_idx_train,
        global_bias=global_bias,
        school_bias=school_bias,
        asg_bias=asg_bias,
        school_factors=school_factors,
        asg_components=asg_components,
        bias_scale=cfg["bias_scale"],
        factor_scale=cfg["factor_scale"],
    )

    pred_test = predict_mf_58a(
        base_z=z_base_test,
        school_idx=school_idx_test,
        asg_idx=asg_idx_test,
        global_bias=global_bias,
        school_bias=school_bias,
        asg_bias=asg_bias,
        school_factors=school_factors,
        asg_components=asg_components,
        bias_scale=cfg["bias_scale"],
        factor_scale=cfg["factor_scale"],
    )

    return pred_train, pred_test

# ------------------------------------------------------------
# 7. Scoring / blend helpers
# ------------------------------------------------------------

def tier_weighted_mse_58a(pred):
    total = 0.0
    for tier, mask in tier_masks_oof.items():
        total += test_tier_rates[tier] * float(mean_squared_error(y58[mask], pred[mask]))
    return float(total)


lambda_grid = np.unique(np.concatenate([
    np.linspace(0.0, 1.25, 501),
    np.array([0.0, 0.0025, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.5475, 0.615, 0.75, 0.825, 1.0])
]))

def best_lambda_mask_58a(candidate_pred, mask, tiny_overlap=False):
    if int(mask.sum()) == 0:
        return 0.0

    if tiny_overlap:
        grid = np.array([0.0, 0.0025, 0.005, 0.01, 0.02], dtype=np.float64)
    else:
        grid = lambda_grid

    y = y58[mask].astype(np.float64)
    base = pred44b_oof[mask].astype(np.float64)
    alt = candidate_pred[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = float(mean_squared_error(y, base))

    for lam in grid:
        p = np.clip(base + float(lam) * (alt - base), 0, 100)
        mse = float(mean_squared_error(y, p))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam


def apply_tier_blend_58a(candidate_oof, candidate_test, lams, is_test=False):
    if is_test:
        base = pred44b_test.astype(np.float64)
        alt = candidate_test.astype(np.float64)
        masks = tier_masks_test
    else:
        base = pred44b_oof.astype(np.float64)
        alt = candidate_oof.astype(np.float64)
        masks = tier_masks_oof

    pred = base.copy()

    for tier, mask in masks.items():
        lam = float(lams.get(tier, 0.0))
        pred[mask] = base[mask] + lam * (alt[mask] - base[mask])

    return np.clip(pred, 0, 100).astype(np.float32)


def tier_metrics_58a(pred):
    out = {}
    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y58[mask], pred[mask]))
        out[f"gain_{tier}_vs_50b"] = (
            float(mean_squared_error(y58[mask], pred50b_oof[mask])) - out[f"mse_{tier}"]
        )
    return out

# ------------------------------------------------------------
# 8. Config grid around public-winning rank16/reg500
# ------------------------------------------------------------

def safe_float_str(x):
    return str(x).replace(".", "p").replace("-", "m")

ranks = [10, 12, 14, 15, 16, 17, 18, 20, 22]
regs = [350.0, 400.0, 450.0, 500.0, 600.0, 750.0, 1000.0]
factor_scales = [0.90, 1.00, 1.10]

configs58a = []

for rank in ranks:
    for reg in regs:
        for fscale in factor_scales:
            cfg = {
                "rank": int(rank),
                "reg": float(reg),
                "weight_scheme": "uniform",
                "bias_iter": 4,
                "bias_scale": 1.0,
                "factor_scale": float(fscale),
            }
            cfg["name"] = (
                f"mf58a_school_rank{rank}_reg{safe_float_str(reg)}"
                f"_uniform_b1p0_f{safe_float_str(fscale)}"
            )
            configs58a.append(cfg)

# Ensure exact public-winning 50B config is represented.
# rank16/reg500/f1.00 should be in the grid already.
cfg_names = {c["name"] for c in configs58a}

print("\n58A config count:", len(configs58a))
print("First 10 configs:")
for cfg in configs58a[:10]:
    print(cfg)

with open("model_results/mf58a_config_manifest.json", "w") as f:
    json.dump(configs58a, f, indent=2)

# ------------------------------------------------------------
# 9. OOF local screen
# ------------------------------------------------------------

screen_rows = []
fold_metric_frames = []

t0 = time.time()

for cfg_idx, cfg in enumerate(configs58a, start=1):
    name = cfg["name"]

    ckpt_oof = CHECKPOINT_DIR_58A / f"{name}_direct_oof.npy"
    ckpt_test = CHECKPOINT_DIR_58A / f"{name}_direct_test.npy"
    ckpt_fold = CHECKPOINT_DIR_58A / f"{name}_folds.csv"

    print("\n" + "=" * 90)
    print(f"58A OOF config {cfg_idx}/{len(configs58a)}: {name}")
    print("=" * 90)

    if ckpt_oof.exists() and ckpt_test.exists() and ckpt_fold.exists():
        print("Loading checkpoint.")
        direct_oof = np.load(ckpt_oof).astype(np.float32)
        direct_test = np.load(ckpt_test).astype(np.float32)
        fold_df = pd.read_csv(ckpt_fold)
    else:
        start = time.time()
        direct_oof, direct_test, fold_df = train_predict_oof_config_58a(cfg)
        np.save(ckpt_oof, direct_oof)
        np.save(ckpt_test, direct_test)
        fold_df.to_csv(ckpt_fold, index=False)
        print(f"Finished config in {time.time() - start:.1f}s")

    fold_metric_frames.append(fold_df)

    lam_overlap = best_lambda_mask_58a(direct_oof, overlap_oof, tiny_overlap=True)
    lam_solver = best_lambda_mask_58a(direct_oof, solver_only_oof, tiny_overlap=False)
    lam_none = best_lambda_mask_58a(direct_oof, none_oof, tiny_overlap=False)

    lams = {
        "overlap": lam_overlap,
        "solver_only": lam_solver,
        "none": lam_none,
    }

    blended_oof = apply_tier_blend_58a(direct_oof, direct_test, lams, is_test=False)

    mse = float(mean_squared_error(y58, blended_oof))
    weighted = tier_weighted_mse_58a(blended_oof)

    row = {
        "config": name,
        "rank": cfg["rank"],
        "reg": cfg["reg"],
        "factor_scale": cfg["factor_scale"],
        "direct_oof_mse": float(mean_squared_error(y58, direct_oof)),
        "lambda_overlap": lam_overlap,
        "lambda_solver_only": lam_solver,
        "lambda_none": lam_none,
        "oof_mse": mse,
        "gain_vs_50b_fold": mse50b - mse,
        "public_weighted_mse": weighted,
        "public_weighted_gain_vs_50b_fold": tier_weighted_mse_58a(pred50b_oof) - weighted,
    }
    row.update(tier_metrics_58a(blended_oof))

    screen_rows.append(row)
    pd.DataFrame(screen_rows).to_csv("model_results/mf58a_oof_screen_progress.csv", index=False)

    print(
        f"OOF blended MSE={mse:.6f} | gain vs 50B fold={mse50b - mse:.6f} "
        f"| weighted={weighted:.6f}"
    )

print("\nOOF local screen elapsed seconds:", round(time.time() - t0, 1))

screen58a = pd.DataFrame(screen_rows)
fold_metrics58a = pd.concat(fold_metric_frames, axis=0).reset_index(drop=True)

screen58a = screen58a.sort_values(["oof_mse", "public_weighted_mse"]).reset_index(drop=True)
screen58a_weighted = screen58a.sort_values(["public_weighted_mse", "oof_mse"]).reset_index(drop=True)

# ------------------------------------------------------------
# 10. Select configs for fullfit
# ------------------------------------------------------------

selected_names = []
selected_rows = []

# Always include best by OOF and weighted.
for df_sel in [screen58a, screen58a_weighted]:
    for _, row in df_sel.iterrows():
        name = row["config"]
        if name not in selected_names:
            selected_names.append(name)
            selected_rows.append(row)
        if len(selected_names) >= 16:
            break
    if len(selected_names) >= 16:
        break

# Ensure exact public-winning neighborhood candidates included.
force_patterns = [
    "rank16_reg500p0_uniform_b1p0_f1p0",
    "rank14_reg500p0_uniform_b1p0_f1p0",
    "rank12_reg500p0_uniform_b1p0_f1p0",
    "rank20_reg500p0_uniform_b1p0_f1p0",
    "rank16_reg600p0_uniform_b1p0_f1p0",
    "rank16_reg750p0_uniform_b1p0_f1p0",
]

for pat in force_patterns:
    matches = [r for _, r in screen58a.iterrows() if pat in r["config"]]
    for row in matches[:1]:
        if row["config"] not in selected_names:
            selected_names.append(row["config"])
            selected_rows.append(row)

selected_df = pd.DataFrame(selected_rows).drop_duplicates("config").reset_index(drop=True)

cfg_lookup = {cfg["name"]: cfg for cfg in configs58a}

print("\nSelected configs for fullfit")
print("----------------------------")
display(selected_df[[
    "config", "rank", "reg", "factor_scale",
    "lambda_overlap", "lambda_solver_only", "lambda_none",
    "oof_mse", "gain_vs_50b_fold", "public_weighted_mse",
    "public_weighted_gain_vs_50b_fold",
    "mse_overlap", "mse_solver_only", "mse_none"
]])

# ------------------------------------------------------------
# 11. Full-train refits and submission files
# ------------------------------------------------------------

fullfit_rows = []
submission_paths = []

def row_to_lams(row):
    return {
        "overlap": float(row["lambda_overlap"]),
        "solver_only": float(row["lambda_solver_only"]),
        "none": float(row["lambda_none"]),
    }

for i, row in selected_df.iterrows():
    cfg_name = row["config"]
    cfg = cfg_lookup[cfg_name]
    lams = row_to_lams(row)

    print("\n" + "=" * 90)
    print(f"58A fullfit {i+1}/{len(selected_df)}: {cfg_name}")
    print("=" * 90)

    ckpt_train = CHECKPOINT_DIR_58A / f"{cfg_name}_fullfit_direct_train.npy"
    ckpt_test = CHECKPOINT_DIR_58A / f"{cfg_name}_fullfit_direct_test.npy"

    if ckpt_train.exists() and ckpt_test.exists():
        print("Loading fullfit checkpoint.")
        full_direct_train = np.load(ckpt_train).astype(np.float32)
        full_direct_test = np.load(ckpt_test).astype(np.float32)
    else:
        start = time.time()
        full_direct_train, full_direct_test = train_predict_fullfit_config_58a(cfg)
        np.save(ckpt_train, full_direct_train)
        np.save(ckpt_test, full_direct_test)
        print(f"Finished fullfit in {time.time() - start:.1f}s")

    full_blend_train = apply_tier_blend_58a(full_direct_train, full_direct_test, lams, is_test=False)
    full_blend_test = apply_tier_blend_58a(full_direct_train, full_direct_test, lams, is_test=True)

    trainfit_mse_not_oof = float(mean_squared_error(y58, full_blend_train))

    # Main pure fullfit submission.
    tag = (
        f"rank{cfg['rank']}_reg{safe_float_str(cfg['reg'])}"
        f"_f{safe_float_str(cfg['factor_scale'])}"
    )
    out_pure = f"submission_mf58a_fullfit_{i+1}_{tag}.csv"

    pd.DataFrame({
        ID_COL: test_ids58a,
        TARGET_COL: full_blend_test,
    }).to_csv(out_pure, index=False)

    submission_paths.append(out_pure)

    # Blends with current public-best fullfit anchor.
    blend_weights = [0.25, 0.50, 0.75]
    for w_new in blend_weights:
        mix_test = np.clip(
            (1.0 - float(w_new)) * pred_fullfit_anchor_test.astype(np.float64)
            + float(w_new) * full_blend_test.astype(np.float64),
            0, 100
        ).astype(np.float32)

        out_mix = f"submission_mf58a_fullfit_{i+1}_{tag}_mixanchor{safe_float_str(w_new)}.csv"

        pd.DataFrame({
            ID_COL: test_ids58a,
            TARGET_COL: mix_test,
        }).to_csv(out_mix, index=False)

        submission_paths.append(out_mix)

    fullfit_rows.append({
        "rank_order": int(i + 1),
        "config": cfg_name,
        "rank": cfg["rank"],
        "reg": cfg["reg"],
        "factor_scale": cfg["factor_scale"],
        "lambda_overlap": lams["overlap"],
        "lambda_solver_only": lams["solver_only"],
        "lambda_none": lams["none"],
        "oof_screen_mse": float(row["oof_mse"]),
        "oof_gain_vs_50b_fold": float(row["gain_vs_50b_fold"]),
        "public_weighted_oof": float(row["public_weighted_mse"]),
        "public_weighted_gain_vs_50b_fold": float(row["public_weighted_gain_vs_50b_fold"]),
        "trainfit_mse_not_oof": trainfit_mse_not_oof,
        "mean_abs_diff_vs_anchor_test": float(np.mean(np.abs(full_blend_test - pred_fullfit_anchor_test))),
        "p95_abs_diff_vs_anchor_test": float(np.percentile(np.abs(full_blend_test - pred_fullfit_anchor_test), 95)),
        "max_abs_diff_vs_anchor_test": float(np.max(np.abs(full_blend_test - pred_fullfit_anchor_test))),
        "pure_submission": out_pure,
    })

fullfit58a = pd.DataFrame(fullfit_rows).sort_values(
    ["oof_screen_mse", "public_weighted_oof"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 12. Fold diagnostics for top OOF-screen candidates
# ------------------------------------------------------------

top_diag_names = selected_df.head(8)["config"].tolist()
folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(np.arange(n_train)))

fold_diag_rows = []

for cfg_name in top_diag_names:
    cfg = cfg_lookup[cfg_name]
    direct_oof = np.load(CHECKPOINT_DIR_58A / f"{cfg_name}_direct_oof.npy").astype(np.float32)

    row_screen = screen58a[screen58a["config"] == cfg_name].iloc[0]
    lams = row_to_lams(row_screen)

    dummy_test = np.zeros(n_test, dtype=np.float32)
    blended_oof = apply_tier_blend_58a(direct_oof, dummy_test, lams, is_test=False)

    for fold_num, (_, va_idx) in enumerate(folds, start=1):
        row = {
            "config": cfg_name,
            "fold": fold_num,
            "mse_50b_fold": float(mean_squared_error(y58[va_idx], pred50b_oof[va_idx])),
            "mse_58a_oof": float(mean_squared_error(y58[va_idx], blended_oof[va_idx])),
        }
        row["gain_vs_50b_fold"] = row["mse_50b_fold"] - row["mse_58a_oof"]

        for tier, mask_full in tier_masks_oof.items():
            mask = mask_full[va_idx]
            row[f"n_{tier}"] = int(mask.sum())
            if int(mask.sum()) > 0:
                row[f"mse50b_{tier}"] = float(mean_squared_error(y58[va_idx][mask], pred50b_oof[va_idx][mask]))
                row[f"mse58a_{tier}"] = float(mean_squared_error(y58[va_idx][mask], blended_oof[va_idx][mask]))
                row[f"gain_{tier}_vs_50b_fold"] = row[f"mse50b_{tier}"] - row[f"mse58a_{tier}"]

        fold_diag_rows.append(row)

fold_diag58a = pd.DataFrame(fold_diag_rows)

# ------------------------------------------------------------
# 13. Save screens and validate submissions
# ------------------------------------------------------------

screen58a.to_csv("model_results/mf58a_oof_screen.csv", index=False)
screen58a_weighted.to_csv("model_results/mf58a_oof_screen_weighted.csv", index=False)
fold_metrics58a.to_csv("model_results/mf58a_fold_metrics.csv", index=False)
selected_df.to_csv("model_results/mf58a_selected_fullfit_configs.csv", index=False)
fullfit58a.to_csv("model_results/mf58a_fullfit_screen.csv", index=False)
fold_diag58a.to_csv("model_results/mf58a_top_fold_diag.csv", index=False)

for p in submission_paths:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test, 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[ID_COL].notna().all(), p
    assert sub[TARGET_COL].notna().all(), p
    assert np.isfinite(sub[TARGET_COL]).all(), p
    assert sub[TARGET_COL].between(0, 100).all(), p

# ------------------------------------------------------------
# 14. Output summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("58A expanded fullfit MF local search complete")
print("=" * 90)

print("\nReference")
print("---------")
print("Current public best:")
print("submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform.csv | public 29.334")
print(f"50B fold OOF MSE: {mse50b:.6f}")

print("\nTop 30 OOF-screen configs")
print("-------------------------")
display(screen58a.head(30))

print("\nTop 30 public-weighted OOF-screen configs")
print("-----------------------------------------")
display(screen58a_weighted.head(30))

print("\nSelected fullfit configs")
print("------------------------")
display(selected_df)

print("\nFullfit screen")
print("--------------")
display(fullfit58a)

print("\nTop fold diagnostics")
print("--------------------")
print(fold_diag58a.to_string(index=False))

print("\nSaved files")
print("-----------")
print("model_results/mf58a_oof_screen.csv")
print("model_results/mf58a_oof_screen_weighted.csv")
print("model_results/mf58a_fold_metrics.csv")
print("model_results/mf58a_selected_fullfit_configs.csv")
print("model_results/mf58a_fullfit_screen.csv")
print("model_results/mf58a_top_fold_diag.csv")
print("\nSubmission files:")
for p in submission_paths:
    print(p)

print("\nSubmission guidance")
print("-------------------")
print("Prioritize pure fullfit submissions first, because pure rank16/reg500 fullfit already beat its mix/fold anchor publicly.")
print("Start with the top rows in model_results/mf58a_fullfit_screen.csv, but avoid submitting near-duplicates on the same day.")
print("Current protected public best remains:")
print("submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform.csv | public 29.334")

58A. Expanded fullfit MF local search around public-winning 50B fullfit
Checkpoint dir: model_results/mf58a_checkpoints

Artifact checks
---------------
model_results/oof_hybrid44b_best.csv                                        True
model_results/testpred_hybrid44b_best.csv                                   True
model_results/oof_mf50b_best_oof.csv                                        True
model_results/testpred_mf50b_best_oof.csv                                   True
model_results/account39a_raw_oof_reconstruction.csv                         True
model_results/account39a_raw_test_reconstruction.csv                        True
model_results/account39b_solver_raw_oof.csv                                 True
model_results/account39b_solver_raw_test.csv                                True
submission_mf50b_fullfit_1_rank16_school_reg500p0_uniform.csv               True

Raw shapes
----------
raw_train_te: (144921, 62)
raw_test_te:  (48307, 61)

References
----------
44B OOF MSE:       

,config,rank,reg,factor_scale,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b_fold,public_weighted_mse,public_weighted_gain_vs_50b_fold,mse_overlap,mse_solver_only,mse_none
0,mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p1,15,1000.0,1.1,0.0050,0.5275,0.7850,51.367054,0.096104,29.576583,0.041642,0.506959,63.404762,89.278076
1,mf58a_school_rank15_reg750p0_uniform_b1p0_f1p1,15,750.0,1.1,0.0050,0.5275,0.7800,51.378925,0.084232,29.578793,0.039431,0.506959,63.405972,89.304703
2,mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p0,15,1000.0,1.0,0.0050,0.5775,0.8575,51.378979,0.084179,29.580125,0.038100,0.506958,63.409870,89.303116
3,mf58a_school_rank15_reg600p0_uniform_b1p0_f1p1,15,600.0,1.1,0.0050,0.5275,0.7775,51.389980,0.073177,29.580304,0.037920,0.506958,63.405483,89.330215
4,mf58a_school_rank15_reg750p0_uniform_b1p0_f1p0,15,750.0,1.0,0.0050,0.5750,0.8500,51.393631,0.069527,29.583107,0.035118,0.506957,63.412109,89.335663
5,mf58a_school_rank15_reg1000p0_uniform_b1p0_f0p9,15,1000.0,0.9,0.0050,0.6375,0.9425,51.394222,0.068935,29.584706,0.033518,0.506958,63.416553,89.335068
6,mf58a_school_rank15_reg500p0_uniform_b1p0_f1p1,15,500.0,1.1,0.0050,0.5275,0.7725,51.401600,0.061558,29.581349,0.036876,0.506959,63.403370,89.357727
7,mf58a_school_rank15_reg600p0_uniform_b1p0_f1p0,15,600.0,1.0,0.0050,0.5750,0.8450,51.407230,0.055927,29.585342,0.032882,0.506957,63.412621,89.366547
8,mf58a_school_rank15_reg450p0_uniform_b1p0_f1p1,15,450.0,1.1,0.0050,0.5275,0.7700,51.410248,0.052910,29.582189,0.036035,0.506959,63.401981,89.378128
9,mf58a_school_rank15_reg750p0_uniform_b1p0_f0p9,15,750.0,0.9,0.0050,0.6325,0.9350,51.412411,0.050747,29.588685,0.029539,0.506957,63.420151,89.375099



58A fullfit 1/22: mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p1
Finished fullfit in 0.1s

58A fullfit 2/22: mf58a_school_rank15_reg750p0_uniform_b1p0_f1p1
Finished fullfit in 0.2s

58A fullfit 3/22: mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p0
Finished fullfit in 0.1s

58A fullfit 4/22: mf58a_school_rank15_reg600p0_uniform_b1p0_f1p1
Finished fullfit in 0.1s

58A fullfit 5/22: mf58a_school_rank15_reg750p0_uniform_b1p0_f1p0
Finished fullfit in 0.1s

58A fullfit 6/22: mf58a_school_rank15_reg1000p0_uniform_b1p0_f0p9
Finished fullfit in 0.1s

58A fullfit 7/22: mf58a_school_rank15_reg500p0_uniform_b1p0_f1p1
Finished fullfit in 0.1s

58A fullfit 8/22: mf58a_school_rank15_reg600p0_uniform_b1p0_f1p0
Finished fullfit in 0.1s

58A fullfit 9/22: mf58a_school_rank15_reg450p0_uniform_b1p0_f1p1
Finished fullfit in 0.1s

58A fullfit 10/22: mf58a_school_rank15_reg750p0_uniform_b1p0_f0p9
Finished fullfit in 0.1s

58A fullfit 11/22: mf58a_school_rank16_reg1000p0_uniform_b1p0_f1p1
Finished fullfit i

,config,rank,reg,factor_scale,direct_oof_mse,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b_fold,public_weighted_mse,public_weighted_gain_vs_50b_fold,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b
0,mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p1,15,1000.0,1.1,52.669220,0.0050,0.5275,0.7850,51.367054,0.096104,29.576583,0.041642,0.506959,2.306700e-05,63.404762,0.079681,89.278076,0.184906
1,mf58a_school_rank15_reg750p0_uniform_b1p0_f1p1,15,750.0,1.1,52.689793,0.0050,0.5275,0.7800,51.378925,0.084232,29.578793,0.039431,0.506959,2.378225e-05,63.405972,0.078472,89.304703,0.158279
2,mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p0,15,1000.0,1.0,52.394753,0.0050,0.5775,0.8575,51.378979,0.084179,29.580125,0.038100,0.506958,2.443790e-05,63.409870,0.074574,89.303116,0.159866
3,mf58a_school_rank15_reg600p0_uniform_b1p0_f1p1,15,600.0,1.1,52.707172,0.0050,0.5275,0.7775,51.389980,0.073177,29.580304,0.037920,0.506958,2.408028e-05,63.405483,0.078960,89.330215,0.132767
4,mf58a_school_rank15_reg750p0_uniform_b1p0_f1p0,15,750.0,1.0,52.417538,0.0050,0.5750,0.8500,51.393631,0.069527,29.583107,0.035118,0.506957,2.521276e-05,63.412109,0.072334,89.335663,0.127319
5,mf58a_school_rank15_reg1000p0_uniform_b1p0_f0p9,15,1000.0,0.9,52.179047,0.0050,0.6375,0.9425,51.394222,0.068935,29.584706,0.033518,0.506958,2.467632e-05,63.416553,0.067890,89.335068,0.127914
6,mf58a_school_rank15_reg500p0_uniform_b1p0_f1p1,15,500.0,1.1,52.724335,0.0050,0.5275,0.7725,51.401600,0.061558,29.581349,0.036876,0.506959,2.390146e-05,63.403370,0.081074,89.357727,0.105255
7,mf58a_school_rank15_reg600p0_uniform_b1p0_f1p0,15,600.0,1.0,52.437386,0.0050,0.5750,0.8450,51.407230,0.055927,29.585342,0.032882,0.506957,2.551079e-05,63.412621,0.071823,89.366547,0.096436
8,mf58a_school_rank15_reg450p0_uniform_b1p0_f1p1,15,450.0,1.1,52.737324,0.0050,0.5275,0.7700,51.410248,0.052910,29.582189,0.036035,0.506959,2.378225e-05,63.401981,0.082462,89.378128,0.084854
9,mf58a_school_rank15_reg750p0_uniform_b1p0_f0p9,15,750.0,0.9,52.203968,0.0050,0.6325,0.9350,51.412411,0.050747,29.588685,0.029539,0.506957,2.545118e-05,63.420151,0.064293,89.375099,0.087883



Top 30 public-weighted OOF-screen configs
-----------------------------------------


...


Selected fullfit configs
------------------------


,config,rank,reg,factor_scale,direct_oof_mse,lambda_overlap,lambda_solver_only,lambda_none,oof_mse,gain_vs_50b_fold,public_weighted_mse,public_weighted_gain_vs_50b_fold,mse_overlap,gain_overlap_vs_50b,mse_solver_only,gain_solver_only_vs_50b,mse_none,gain_none_vs_50b
0,mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p1,15,1000.0,1.1,52.669220,0.0050,0.5275,0.7850,51.367054,0.096104,29.576583,0.041642,0.506959,2.306700e-05,63.404762,0.079681,89.278076,0.184906
1,mf58a_school_rank15_reg750p0_uniform_b1p0_f1p1,15,750.0,1.1,52.689793,0.0050,0.5275,0.7800,51.378925,0.084232,29.578793,0.039431,0.506959,2.378225e-05,63.405972,0.078472,89.304703,0.158279
2,mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p0,15,1000.0,1.0,52.394753,0.0050,0.5775,0.8575,51.378979,0.084179,29.580125,0.038100,0.506958,2.443790e-05,63.409870,0.074574,89.303116,0.159866
3,mf58a_school_rank15_reg600p0_uniform_b1p0_f1p1,15,600.0,1.1,52.707172,0.0050,0.5275,0.7775,51.389980,0.073177,29.580304,0.037920,0.506958,2.408028e-05,63.405483,0.078960,89.330215,0.132767
4,mf58a_school_rank15_reg750p0_uniform_b1p0_f1p0,15,750.0,1.0,52.417538,0.0050,0.5750,0.8500,51.393631,0.069527,29.583107,0.035118,0.506957,2.521276e-05,63.412109,0.072334,89.335663,0.127319
5,mf58a_school_rank15_reg1000p0_uniform_b1p0_f0p9,15,1000.0,0.9,52.179047,0.0050,0.6375,0.9425,51.394222,0.068935,29.584706,0.033518,0.506958,2.467632e-05,63.416553,0.067890,89.335068,0.127914
6,mf58a_school_rank15_reg500p0_uniform_b1p0_f1p1,15,500.0,1.1,52.724335,0.0050,0.5275,0.7725,51.401600,0.061558,29.581349,0.036876,0.506959,2.390146e-05,63.403370,0.081074,89.357727,0.105255
7,mf58a_school_rank15_reg600p0_uniform_b1p0_f1p0,15,600.0,1.0,52.437386,0.0050,0.5750,0.8450,51.407230,0.055927,29.585342,0.032882,0.506957,2.551079e-05,63.412621,0.071823,89.366547,0.096436
8,mf58a_school_rank15_reg450p0_uniform_b1p0_f1p1,15,450.0,1.1,52.737324,0.0050,0.5275,0.7700,51.410248,0.052910,29.582189,0.036035,0.506959,2.378225e-05,63.401981,0.082462,89.378128,0.084854
9,mf58a_school_rank15_reg750p0_uniform_b1p0_f0p9,15,750.0,0.9,52.203968,0.0050,0.6325,0.9350,51.412411,0.050747,29.588685,0.029539,0.506957,2.545118e-05,63.420151,0.064293,89.375099,0.087883



Fullfit screen
--------------


,rank_order,config,rank,reg,factor_scale,lambda_overlap,lambda_solver_only,lambda_none,oof_screen_mse,oof_gain_vs_50b_fold,public_weighted_oof,public_weighted_gain_vs_50b_fold,trainfit_mse_not_oof,mean_abs_diff_vs_anchor_test,p95_abs_diff_vs_anchor_test,max_abs_diff_vs_anchor_test,pure_submission
0,1,mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p1,15,1000.0,1.1,0.0050,0.5275,0.7850,51.367054,0.096104,29.576583,0.041642,36.518612,0.116203,0.605875,9.260651,submission_mf58a_fullfit_1_rank15_reg1000p0_f1...
1,2,mf58a_school_rank15_reg750p0_uniform_b1p0_f1p1,15,750.0,1.1,0.0050,0.5275,0.7800,51.378925,0.084232,29.578793,0.039431,36.617764,0.106815,0.564495,7.708443,submission_mf58a_fullfit_2_rank15_reg750p0_f1p...
2,3,mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p0,15,1000.0,1.0,0.0050,0.5775,0.8575,51.378979,0.084179,29.580125,0.038100,36.538387,0.113664,0.597335,9.222710,submission_mf58a_fullfit_3_rank15_reg1000p0_f1...
3,4,mf58a_school_rank15_reg600p0_uniform_b1p0_f1p1,15,600.0,1.1,0.0050,0.5275,0.7775,51.389980,0.073177,29.580304,0.037920,36.634262,0.111318,0.597931,8.098549,submission_mf58a_fullfit_4_rank15_reg600p0_f1p...
4,5,mf58a_school_rank15_reg750p0_uniform_b1p0_f1p0,15,750.0,1.0,0.0050,0.5750,0.8500,51.393631,0.069527,29.583107,0.035118,36.661572,0.104290,0.558703,7.596443,submission_mf58a_fullfit_5_rank15_reg750p0_f1p...
5,6,mf58a_school_rank15_reg1000p0_uniform_b1p0_f0p9,15,1000.0,0.9,0.0050,0.6375,0.9425,51.394222,0.068935,29.584706,0.033518,36.598476,0.111475,0.587787,9.153481,submission_mf58a_fullfit_6_rank15_reg1000p0_f0...
6,7,mf58a_school_rank15_reg500p0_uniform_b1p0_f1p1,15,500.0,1.1,0.0050,0.5275,0.7725,51.401600,0.061558,29.581349,0.036876,36.608547,0.094553,0.494507,8.475410,submission_mf58a_fullfit_7_rank15_reg500p0_f1p...
7,8,mf58a_school_rank15_reg600p0_uniform_b1p0_f1p0,15,600.0,1.0,0.0050,0.5750,0.8450,51.407230,0.055927,29.585342,0.032882,36.697304,0.109239,0.588513,8.032310,submission_mf58a_fullfit_8_rank15_reg600p0_f1p...
8,9,mf58a_school_rank15_reg450p0_uniform_b1p0_f1p1,15,450.0,1.1,0.0050,0.5275,0.7700,51.410248,0.052910,29.582189,0.036035,36.661572,0.100164,0.545195,6.295620,submission_mf58a_fullfit_9_rank15_reg450p0_f1p...
9,10,mf58a_school_rank15_reg750p0_uniform_b1p0_f0p9,15,750.0,0.9,0.0050,0.6325,0.9350,51.412411,0.050747,29.588685,0.029539,36.720360,0.102912,0.551552,7.471668,submission_mf58a_fullfit_10_rank15_reg750p0_f0...



Top fold diagnostics
--------------------
                                         config  fold  mse_50b_fold  mse_58a_oof  gain_vs_50b_fold  n_overlap  mse50b_overlap  mse58a_overlap  gain_overlap_vs_50b_fold  n_solver_only  mse50b_solver_only  mse58a_solver_only  gain_solver_only_vs_50b_fold  n_none  mse50b_none  mse58a_none  gain_none_vs_50b_fold
mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p1     1     51.927963    51.911369          0.016594      10837        0.418636        0.418624              1.195073e-05           5496           61.760151           61.736187                      0.023964   12652    91.776917    91.749306               0.027611
mf58a_school_rank15_reg1000p0_uniform_b1p0_f1p1     2     49.481594    49.304161          0.177433      10622        0.394585        0.394604             -1.874566e-05           5623           60.256847           60.146107                      0.110741   12739    85.654999    85.300148               0.354851
mf58a_school_rank15_reg1000

In [12]:
# ============================================================
# 59A. Global saved OOF audit across model_results
# ============================================================
#
# Purpose:
#   Build a real leaderboard of every saved OOF prediction file.
#
# This answers:
#   "Across all saved models in the notebook, what are the OOF MSEs?"
#
# It does not decide final model alone, because public/private transfer
# can differ. But it gives us the complete evidence table.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("59A. Global saved OOF audit across model_results")
print("=" * 90)

model_dir = Path("model_results")
assert model_dir.exists(), "model_results folder not found"

# ------------------------------------------------------------
# 1. Collect OOF files
# ------------------------------------------------------------

oof_files = sorted(model_dir.glob("oof_*.csv"))

print("Number of oof_*.csv files:", len(oof_files))

rows = []

def find_pred_column(df, path):
    # Highest-confidence prediction columns first.
    preferred = [
        "pred_clipped",
        "prediction",
        "pred",
        "PRED",
        "oof_pred",
        "OOF_PRED",
    ]

    for c in preferred:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            return c

    # If TARGET_COL appears and this is not just target, be careful.
    # In many testpred/submission files TARGET_COL is prediction,
    # but in OOF files TARGET_COL is usually y_true.
    numeric_cols = [
        c for c in df.columns
        if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    # Prefer columns containing pred.
    pred_like = [c for c in numeric_cols if "pred" in c.lower() or "clip" in c.lower()]
    if pred_like:
        return pred_like[0]

    if len(numeric_cols) == 1:
        return numeric_cols[0]

    return None

for path in oof_files:
    try:
        df = pd.read_csv(path)

        if "row_index" in df.columns:
            df = df.sort_values("row_index").reset_index(drop=True)

        if TARGET_COL not in df.columns:
            rows.append({
                "file": path.name,
                "status": "skip_no_target_col",
                "n_rows": len(df),
                "oof_mse": np.nan,
                "pred_col": None,
            })
            continue

        pred_col = find_pred_column(df, path)

        if pred_col is None:
            rows.append({
                "file": path.name,
                "status": "skip_no_pred_col",
                "n_rows": len(df),
                "oof_mse": np.nan,
                "pred_col": None,
            })
            continue

        y = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        p = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

        ok = np.isfinite(y) & np.isfinite(p)

        if ok.sum() == 0:
            rows.append({
                "file": path.name,
                "status": "skip_no_finite",
                "n_rows": len(df),
                "oof_mse": np.nan,
                "pred_col": pred_col,
            })
            continue

        mse = float(mean_squared_error(y[ok], np.clip(p[ok], 0, 100)))

        rows.append({
            "file": path.name,
            "status": "ok",
            "n_rows": len(df),
            "n_finite": int(ok.sum()),
            "oof_mse": mse,
            "pred_col": pred_col,
            "mean_pred": float(np.nanmean(p)),
            "std_pred": float(np.nanstd(p)),
            "min_pred": float(np.nanmin(p)),
            "max_pred": float(np.nanmax(p)),
        })

    except Exception as e:
        rows.append({
            "file": path.name,
            "status": f"error: {type(e).__name__}: {e}",
            "n_rows": np.nan,
            "oof_mse": np.nan,
            "pred_col": None,
        })

audit = pd.DataFrame(rows)

ok_audit = audit[audit["status"] == "ok"].copy()
ok_audit = ok_audit.sort_values("oof_mse").reset_index(drop=True)

bad_audit = audit[audit["status"] != "ok"].copy().reset_index(drop=True)

# ------------------------------------------------------------
# 2. Add known public scores manually where we know them
# ------------------------------------------------------------

public_scores = {
    "oof_hybrid44b_best.csv": 30.556,
    "oof_mf50a_best.csv": 29.742,
    "oof_mf50b_best_oof.csv": 29.448,
    "oof_mf50c_greedy_best.csv": 29.489,
    "oof_mf51b_best.csv": 29.904,
    "oof_lgb53b_best_weighted.csv": None,
    "oof_eb55a_best_weighted.csv": None,
    "oof_int57a_best_weighted.csv": None,
}

ok_audit["known_public_mse"] = ok_audit["file"].map(public_scores)

# ------------------------------------------------------------
# 3. Save and display
# ------------------------------------------------------------

audit.to_csv("model_results/global_oof_audit_all_files.csv", index=False)
ok_audit.to_csv("model_results/global_oof_audit_ranked_ok.csv", index=False)
bad_audit.to_csv("model_results/global_oof_audit_skipped_or_error.csv", index=False)

print("\nTop 50 saved OOF files by computed OOF MSE")
print("------------------------------------------")
display(ok_audit.head(50))

print("\nSkipped/error files")
print("-------------------")
display(bad_audit)

print("\nSaved")
print("-----")
print("model_results/global_oof_audit_all_files.csv")
print("model_results/global_oof_audit_ranked_ok.csv")
print("model_results/global_oof_audit_skipped_or_error.csv")

59A. Global saved OOF audit across model_results
Number of oof_*.csv files: 102

Top 50 saved OOF files by computed OOF MSE
------------------------------------------


,file,status,n_rows,n_finite,oof_mse,pred_col,mean_pred,std_pred,min_pred,max_pred,known_public_mse
0,oof_stack40b_best_oof.csv,ok,144921,144921.0,49.621123,pred_clipped,54.363338,25.365803,1.962399e-17,100.00000,NaN
1,oof_stack40b_best_weighted.csv,ok,144921,144921.0,49.621123,pred_clipped,54.363338,25.365803,1.962399e-17,100.00000,NaN
2,oof_hte40a3_residual_best.csv,ok,144921,144921.0,49.689606,pred_clipped,54.357800,25.392454,0.000000e+00,100.00000,NaN
3,oof_meta43b2_best.csv,ok,144921,144921.0,49.768634,pred_clipped,54.282383,25.403962,0.000000e+00,100.00000,NaN
4,oof_liberal44d_best_oof.csv,ok,144921,144921.0,49.774822,pred_clipped,54.399572,25.389364,0.000000e+00,100.00000,NaN
5,oof_mf50c_greedy_best.csv,ok,144921,144921.0,50.155320,pred_clipped,54.326333,25.398939,1.952223e-04,100.00000,29.489
6,oof_liberal44d_best_weighted.csv,ok,144921,144921.0,50.440873,pred_clipped,54.393854,25.416742,0.000000e+00,100.00000,NaN
7,oof_mf51b_best.csv,ok,144921,144921.0,51.091054,pred_clipped,54.327912,25.365736,0.000000e+00,100.00000,29.904
8,oof_mf51a_best.csv,ok,144921,144921.0,51.201223,pred_clipped,54.328474,25.389191,0.000000e+00,100.00000,NaN
9,oof_als52a_best.csv,ok,144921,144921.0,51.262243,pred_clipped,54.313049,25.365975,1.800258e-04,100.00000,NaN



Skipped/error files
-------------------


,file,status,n_rows,n_finite,oof_mse,pred_col,mean_pred,std_pred,min_pred,max_pred
0,oof_blend_rf500_extratrees_safe.csv,skip_no_target_col,144921,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,oof_blend_rf_et_lgbm_weighted.csv,skip_no_target_col,144921,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,oof_extratrees_safe_base.csv,skip_no_target_col,144921,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,oof_hist_group_means_30A_lite.csv,skip_no_target_col,144921,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,oof_hist_group_selected_30C.csv,skip_no_target_col,144921,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,oof_lgbm_t03_base_5fold_oof.csv,skip_no_target_col,144921,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv,skip_no_target_col,144921,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv,skip_no_target_col,144921,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,oof_rf_500_base.csv,skip_no_target_col,144921,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Saved
-----
model_results/global_oof_audit_all_files.csv
model_results/global_oof_audit_ranked_ok.csv
model_results/global_oof_audit_skipped_or_error.csv


In [14]:
required_memory = ["raw_train_te", "raw_test_te"]
[x for x in required_memory if x not in globals()]

[]

In [15]:
# ============================================================
# 60A. Interpretable fallback audit for accounting solver models
# ============================================================
#
# Purpose:
#   Test whether the accounting models can use a simpler/interpretable
#   fallback baseline instead of the complicated later baseline.
#
# Why:
#   39A/39B/39C/39D depend on a baseline/prior whenever accounting
#   reconstruction is missing or ambiguous.
#
# This cell:
#   1. Scans saved OOF prediction files in model_results.
#   2. Scores candidate fallback models by tier.
#   3. Starts from the best available accounting model, preferably 39D or 39C.
#   4. Tests replacing/blending only solver_only and none tiers with simpler models.
#   5. Saves screens and possible submissions.
#
# This is for interpretability/reporting, not necessarily leaderboard best.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

Path("model_results").mkdir(exist_ok=True)

print("=" * 90)
print("60A. Interpretable fallback audit for accounting solver models")
print("=" * 90)

# ------------------------------------------------------------
# 1. Basic artifact checks
# ------------------------------------------------------------

required_tier_files = [
    "model_results/account39a_raw_oof_reconstruction.csv",
    "model_results/account39a_raw_test_reconstruction.csv",
    "model_results/account39b_solver_raw_oof.csv",
    "model_results/account39b_solver_raw_test.csv",
]

for p in required_tier_files:
    print(f"{p:65s}", Path(p).exists())

missing = [p for p in required_tier_files if not Path(p).exists()]
if missing:
    raise RuntimeError(f"Missing tier reconstruction files: {missing}")

# ------------------------------------------------------------
# 2. Load helper functions
# ------------------------------------------------------------

def find_pred_col(df, path_name=""):
    preferred = [
        "pred_clipped",
        "prediction",
        "pred",
        "oof_pred",
        "OOF_PRED",
        "PRED",
    ]

    for c in preferred:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            return c

    numeric_cols = [
        c for c in df.columns
        if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    pred_like = [c for c in numeric_cols if "pred" in c.lower() or "clip" in c.lower()]
    if pred_like:
        return pred_like[0]

    if len(numeric_cols) == 1:
        return numeric_cols[0]

    return None


def load_oof_pred(path):
    df = pd.read_csv(path)

    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if TARGET_COL not in df.columns:
        return None

    pred_col = find_pred_col(df, path.name)
    if pred_col is None:
        return None

    y = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
    p = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    return {
        "df": df,
        "y": y,
        "pred": np.clip(p, 0, 100).astype(np.float32),
        "pred_col": pred_col,
    }


def load_test_pred(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        pred_col = find_pred_col(df, path.name)

    if pred_col is None:
        return None

    p = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        return None

    return {
        "df": df,
        "ids": ids,
        "pred": np.clip(p, 0, 100).astype(np.float32),
        "pred_col": pred_col,
    }


def candidate_test_paths_for_oof(oof_path):
    name = oof_path.name
    candidates = []

    if name.startswith("oof_"):
        candidates.append(oof_path.parent / name.replace("oof_", "testpred_", 1))

    candidates.append(oof_path.parent / name.replace("oof", "testpred"))
    candidates.append(oof_path.parent / name.replace("_oof", "_test"))
    candidates.append(oof_path.parent / name.replace("oof_", "submission_"))

    # Also try project-root submissions with matching stem fragments.
    stem = name
    stem = stem.replace("oof_", "")
    stem = stem.replace(".csv", "")
    candidates.extend(sorted(Path(".").glob(f"submission*{stem}*.csv")))

    # Deduplicate.
    out = []
    seen = set()
    for p in candidates:
        p = Path(p)
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out


def find_matching_test_for_oof(oof_path, n_test_expected=None, test_ids_expected=None):
    for p in candidate_test_paths_for_oof(oof_path):
        if not p.exists():
            continue
        loaded = load_test_pred(p)
        if loaded is None:
            continue
        if n_test_expected is not None and len(loaded["pred"]) != n_test_expected:
            continue
        if test_ids_expected is not None:
            if len(loaded["ids"]) != len(test_ids_expected):
                continue
            # Usually exact order should match.
            if not np.array_equal(loaded["ids"], test_ids_expected):
                continue
        return p, loaded
    return None, None


# ------------------------------------------------------------
# 3. Detect accounting base model to start from
# ------------------------------------------------------------

accounting_base_specs = [
    ("account39d_best", "model_results/oof_account39d_best.csv", "model_results/testpred_account39d_best.csv"),
    ("account39c_final", "model_results/oof_account39c_final.csv", "model_results/testpred_account39c_final.csv"),
    ("account39c_best", "model_results/oof_account39c_best.csv", "model_results/testpred_account39c_best.csv"),
    ("account39b_best", "model_results/oof_account39b_best.csv", "model_results/testpred_account39b_best.csv"),
    ("account39a_best", "model_results/oof_account39a_best.csv", "model_results/testpred_account39a_best.csv"),
]

account_base_name = None
account_oof_loaded = None
account_test_loaded = None

for name, oof_path, test_path in accounting_base_specs:
    if Path(oof_path).exists() and Path(test_path).exists():
        temp_oof = load_oof_pred(Path(oof_path))
        temp_test = load_test_pred(Path(test_path))
        if temp_oof is not None and temp_test is not None:
            account_base_name = name
            account_oof_loaded = temp_oof
            account_test_loaded = temp_test
            break

if account_base_name is None:
    raise RuntimeError(
        "Could not find an accounting base OOF/test pair. "
        "Expected one of oof_account39d_best, oof_account39c_final, etc."
    )

y = account_oof_loaded["y"].astype(np.float32)
account_oof = account_oof_loaded["pred"].astype(np.float32)
account_test = account_test_loaded["pred"].astype(np.float32)
test_ids60 = account_test_loaded["ids"]

n_train = len(y)
n_test = len(account_test)

print("\nAccounting base selected")
print("------------------------")
print("account_base_name:", account_base_name)
print("OOF MSE:", float(mean_squared_error(y, account_oof)))
print("n_train:", n_train)
print("n_test:", n_test)

# ------------------------------------------------------------
# 4. Reconstruct tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

tier_masks_oof = {
    "overlap": overlap_oof,
    "solver_only": solver_only_oof,
    "none": none_oof,
}

tier_masks_test = {
    "overlap": overlap_test,
    "solver_only": solver_only_test,
    "none": none_test,
}

test_tier_rates = {tier: float(mask.mean()) for tier, mask in tier_masks_test.items()}

print("\nTier coverage")
print("-------------")
for tier, mask in tier_masks_oof.items():
    print(
        f"{tier:12s} train={int(mask.sum()):6d} "
        f"test={int(tier_masks_test[tier].sum()):6d} "
        f"account_base_mse={mean_squared_error(y[mask], account_oof[mask]):.6f}"
    )

# ------------------------------------------------------------
# 5. Scan all saved OOF models as possible fallback components
# ------------------------------------------------------------

exclude_for_interpretable_fallback = [
    "account39",
    "hybrid44",
    "mf50",
    "mf51",
    "mf52",
    "mf56",
    "mf58",
    "als52",
    "lgb53",
    "eb55",
    "int57",
    "entity41",
    "binom54",
]

def classify_candidate(file_name):
    low = file_name.lower()

    if any(tok in low for tok in ["mean", "ridge", "linear", "lasso", "elastic"]):
        return "simple_stat_linear"
    if any(tok in low for tok in ["knn"]):
        return "knn"
    if any(tok in low for tok in ["lgb", "lgbm", "lightgbm"]):
        return "lgbm_tree"
    if any(tok in low for tok in ["rf", "random", "et", "extra"]):
        return "tree_ensemble"
    if any(tok in low for tok in ["blend"]):
        return "linear_blend_or_meta"
    if any(tok in low for tok in ["anchor", "33a", "33b", "33c", "35a", "36a", "37b"]):
        return "earlier_pipeline"
    return "other"

candidate_rows = []
candidate_store = {}

for oof_path in sorted(Path("model_results").glob("oof_*.csv")):
    low = oof_path.name.lower()

    # Skip current accounting/MF branches unless you deliberately want them as baselines.
    if any(tok in low for tok in exclude_for_interpretable_fallback):
        continue

    loaded = load_oof_pred(oof_path)
    if loaded is None:
        continue

    if len(loaded["pred"]) != n_train:
        continue

    # Must match target.
    if np.nanmax(np.abs(loaded["y"] - y)) > 1e-5:
        continue

    test_path, test_loaded = find_matching_test_for_oof(
        oof_path,
        n_test_expected=n_test,
        test_ids_expected=test_ids60,
    )

    pred = loaded["pred"]

    row = {
        "name": oof_path.stem.replace("oof_", ""),
        "oof_file": str(oof_path),
        "test_file": str(test_path) if test_path is not None else "",
        "has_test": test_loaded is not None,
        "pred_col": loaded["pred_col"],
        "candidate_class": classify_candidate(oof_path.name),
        "oof_mse": float(mean_squared_error(y, pred)),
    }

    for tier, mask in tier_masks_oof.items():
        row[f"mse_{tier}"] = float(mean_squared_error(y[mask], pred[mask]))
        row[f"gain_{tier}_vs_account_base"] = (
            float(mean_squared_error(y[mask], account_oof[mask])) - row[f"mse_{tier}"]
        )

    row["test_tier_weighted_mse"] = sum(
        test_tier_rates[t] * row[f"mse_{t}"]
        for t in ["overlap", "solver_only", "none"]
    )

    candidate_rows.append(row)
    candidate_store[row["name"]] = {
        "oof": pred.astype(np.float32),
        "test": test_loaded["pred"].astype(np.float32) if test_loaded is not None else None,
        "oof_file": oof_path,
        "test_file": test_path,
    }

candidate_screen = (
    pd.DataFrame(candidate_rows)
    .sort_values(["mse_none", "test_tier_weighted_mse", "oof_mse"])
    .reset_index(drop=True)
)

candidate_screen.to_csv("model_results/interp60a_fallback_candidate_screen.csv", index=False)

print("\nCandidate fallback models ranked by none-tier MSE")
print("-------------------------------------------------")
display(candidate_screen.head(40))

# ------------------------------------------------------------
# 6. Build accounting + fallback blend candidates
# ------------------------------------------------------------

lambda_grid = np.unique(np.concatenate([
    np.linspace(0.0, 1.0, 401),
    np.array([0.0, 0.05, 0.10, 0.25, 0.50, 0.75, 1.0])
]))

def best_lambda_between(base_pred, alt_pred, mask):
    if int(mask.sum()) == 0:
        return 0.0

    yy = y[mask].astype(np.float64)
    b = base_pred[mask].astype(np.float64)
    a = alt_pred[mask].astype(np.float64)

    best_lam = 0.0
    best_mse = float(mean_squared_error(yy, b))

    for lam in lambda_grid:
        p = np.clip((1.0 - float(lam)) * b + float(lam) * a, 0, 100)
        mse = float(mean_squared_error(yy, p))
        if mse < best_mse:
            best_mse = mse
            best_lam = float(lam)

    return best_lam

def score_pred(pred):
    out = {
        "oof_mse": float(mean_squared_error(y, pred)),
        "gain_vs_account_base": float(mean_squared_error(y, account_oof) - mean_squared_error(y, pred)),
    }

    for tier, mask in tier_masks_oof.items():
        out[f"mse_{tier}"] = float(mean_squared_error(y[mask], pred[mask]))
        out[f"gain_{tier}_vs_account_base"] = (
            float(mean_squared_error(y[mask], account_oof[mask])) - out[f"mse_{tier}"]
        )

    out["test_tier_weighted_mse"] = sum(
        test_tier_rates[t] * out[f"mse_{t}"]
        for t in ["overlap", "solver_only", "none"]
    )

    base_weighted = sum(
        test_tier_rates[t] * float(mean_squared_error(y[tier_masks_oof[t]], account_oof[tier_masks_oof[t]]))
        for t in ["overlap", "solver_only", "none"]
    )

    out["test_tier_weighted_gain_vs_account_base"] = base_weighted - out["test_tier_weighted_mse"]

    return out

fallback_rows = []
fallback_store = {}

for _, row in candidate_screen.iterrows():
    name = row["name"]

    alt_oof = candidate_store[name]["oof"]
    alt_test = candidate_store[name]["test"]

    # Need test predictions to save a submission; still score OOF without test.
    lam_none = best_lambda_between(account_oof, alt_oof, none_oof)
    lam_solver = best_lambda_between(account_oof, alt_oof, solver_only_oof)

    strategies = {
        "none_only": {"solver_only": 0.0, "none": lam_none},
        "solver_only_proxy": {"solver_only": lam_solver, "none": 0.0},
        "solver_plus_none_proxy": {"solver_only": lam_solver, "none": lam_none},
    }

    for strategy, lams in strategies.items():
        pred_oof = account_oof.copy().astype(np.float64)

        # Keep overlap untouched for interpretability and safety.
        if lams["solver_only"] > 0:
            mask = solver_only_oof
            pred_oof[mask] = (
                (1.0 - lams["solver_only"]) * account_oof[mask]
                + lams["solver_only"] * alt_oof[mask]
            )

        if lams["none"] > 0:
            mask = none_oof
            pred_oof[mask] = (
                (1.0 - lams["none"]) * account_oof[mask]
                + lams["none"] * alt_oof[mask]
            )

        pred_oof = np.clip(pred_oof, 0, 100).astype(np.float32)

        scored = score_pred(pred_oof)

        out = {
            "fallback_name": name,
            "candidate_class": row["candidate_class"],
            "strategy": strategy,
            "lambda_solver_only_to_fallback": float(lams["solver_only"]),
            "lambda_none_to_fallback": float(lams["none"]),
            "fallback_oof_file": row["oof_file"],
            "fallback_test_file": row["test_file"],
            "has_test": bool(row["has_test"]),
        }
        out.update(scored)

        key = f"{name}__{strategy}"
        out["key"] = key

        fallback_rows.append(out)

        if alt_test is not None:
            pred_test = account_test.copy().astype(np.float64)

            if lams["solver_only"] > 0:
                mask = solver_only_test
                pred_test[mask] = (
                    (1.0 - lams["solver_only"]) * account_test[mask]
                    + lams["solver_only"] * alt_test[mask]
                )

            if lams["none"] > 0:
                mask = none_test
                pred_test[mask] = (
                    (1.0 - lams["none"]) * account_test[mask]
                    + lams["none"] * alt_test[mask]
                )

            pred_test = np.clip(pred_test, 0, 100).astype(np.float32)
        else:
            pred_test = None

        fallback_store[key] = {
            "oof": pred_oof,
            "test": pred_test,
            "meta": out,
        }

fallback_screen = (
    pd.DataFrame(fallback_rows)
    .sort_values(["oof_mse", "test_tier_weighted_mse"])
    .reset_index(drop=True)
)

fallback_screen.to_csv("model_results/interp60a_accounting_fallback_swap_screen.csv", index=False)

print("\nAccounting + interpretable fallback candidates")
print("----------------------------------------------")
display(fallback_screen.head(60))

# ------------------------------------------------------------
# 7. Save top test submissions when test predictions exist
# ------------------------------------------------------------

submission_paths = []

for i, row in fallback_screen[fallback_screen["has_test"]].head(10).iterrows():
    key = row["key"]
    pred_test = fallback_store[key]["test"]

    if pred_test is None:
        continue

    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", key)[:120]
    out_path = f"submission_interp60a_{len(submission_paths)+1}_{safe_name}.csv"

    pd.DataFrame({
        ID_COL: test_ids60,
        TARGET_COL: pred_test,
    }).to_csv(out_path, index=False)

    submission_paths.append(out_path)

# Validate.
for p in submission_paths:
    sub = pd.read_csv(p)
    assert sub.shape == (n_test, 2), (p, sub.shape)
    assert list(sub.columns) == [ID_COL, TARGET_COL], (p, sub.columns.tolist())
    assert sub[ID_COL].notna().all(), p
    assert sub[TARGET_COL].notna().all(), p
    assert np.isfinite(sub[TARGET_COL]).all(), p
    assert sub[TARGET_COL].between(0, 100).all(), p

print("\nSaved")
print("-----")
print("model_results/interp60a_fallback_candidate_screen.csv")
print("model_results/interp60a_accounting_fallback_swap_screen.csv")
for p in submission_paths:
    print(p)

print("\nInterpretation guide")
print("--------------------")
print("Use this for the report / interpretable model search.")
print("A good fallback candidate should improve none-tier MSE and test-tier-weighted OOF without touching overlap.")
print("If the best lambda_none is 0, the existing fallback is already better than that candidate.")
print("If the best strategy uses only a small lambda, use that as a simple closed-form blend.")

60A. Interpretable fallback audit for accounting solver models
model_results/account39a_raw_oof_reconstruction.csv               True
model_results/account39a_raw_test_reconstruction.csv              True
model_results/account39b_solver_raw_oof.csv                       True
model_results/account39b_solver_raw_test.csv                      True

Accounting base selected
------------------------
account_base_name: account39d_best
OOF MSE: 52.820926666259766
n_train: 144921
n_test: 48307

Tier coverage
-------------
overlap      train= 53786 test= 27298 account_base_mse=0.595325
solver_only  train= 27807 test= 17807 account_base_mse=64.304314
none         train= 63328 test=  3202 account_base_mse=92.135094

Candidate fallback models ranked by none-tier MSE
-------------------------------------------------


,name,oof_file,test_file,has_test,pred_col,candidate_class,oof_mse,mse_overlap,gain_overlap_vs_account_base,mse_solver_only,gain_solver_only_vs_account_base,mse_none,gain_none_vs_account_base,test_tier_weighted_mse
0,hte40a3_residual_best,model_results/oof_hte40a3_residual_best.csv,model_results/testpred_hte40a3_residual_best.csv,True,pred_clipped,other,49.689606,0.595326,-0.000001,64.945831,-0.641518,84.687637,7.447456,29.890315
1,stack40b_best_oof,model_results/oof_stack40b_best_oof.csv,model_results/testpred_stack40b_best_oof.csv,True,pred_clipped,other,49.621124,0.595349,-0.000024,64.070160,0.234154,84.915390,7.219704,29.582633
2,stack40b_best_weighted,model_results/oof_stack40b_best_weighted.csv,model_results/testpred_stack40b_best_weighted.csv,True,pred_clipped,other,49.621124,0.595349,-0.000024,64.070160,0.234154,84.915390,7.219704,29.582633
3,liberal44d_best_oof,model_results/oof_liberal44d_best_oof.csv,model_results/testpred_liberal44d_best_oof.csv,True,pred_clipped,other,49.774822,0.507021,0.088304,64.143074,0.161240,85.310135,6.824959,29.585762
4,meta43b2_best,model_results/oof_meta43b2_best.csv,model_results/testpred_meta43b2_best.csv,True,pred_clipped,tree_ensemble,49.768635,0.595325,0.000000,63.268414,1.035900,85.605034,6.530060,29.332791
5,group_resid_meta_on_oldnew_internal_oldw_0p025...,model_results/oof_group_resid_meta_on_oldnew_i...,model_results/testpred_group_resid_meta_on_old...,True,OOF_group_resid_meta_on_oldnew_internal_oldw_0...,tree_ensemble,70.801170,53.841152,-53.245828,69.015907,-4.711594,85.989601,6.145493,61.565834
6,liberal44d_best_weighted,model_results/oof_liberal44d_best_weighted.csv,model_results/testpred_liberal44d_best_weighte...,True,pred_clipped,other,50.440876,0.507021,0.088304,63.444061,0.860252,87.141266,4.993828,29.449467
7,quant49a_best,model_results/oof_quant49a_best.csv,model_results/testpred_quant49a_best.csv,True,pred_clipped,other,52.592197,0.507021,0.088304,64.249290,0.055023,91.710823,0.424271,30.049182
8,mixed45a_best,model_results/oof_mixed45a_best.csv,model_results/testpred_mixed45a_best.csv,True,pred_clipped,other,52.592197,0.507021,0.088304,64.249306,0.055008,91.710831,0.424263,30.049188
9,hedge46a_best_oof,model_results/oof_hedge46a_best_oof.csv,model_results/testpred_hedge46a_best_oof.csv,True,pred_clipped,other,52.593983,0.507021,0.088304,64.258591,0.045723,91.710831,0.424263,30.052610



Accounting + interpretable fallback candidates
----------------------------------------------


,fallback_name,candidate_class,strategy,lambda_solver_only_to_fallback,lambda_none_to_fallback,fallback_oof_file,fallback_test_file,has_test,oof_mse,gain_vs_account_base,mse_overlap,gain_overlap_vs_account_base,mse_solver_only,gain_solver_only_vs_account_base,mse_none,gain_none_vs_account_base,test_tier_weighted_mse,test_tier_weighted_gain_vs_account_base,key
0,group_resid_meta_on_oldnew_internal_oldw_0p025...,tree_ensemble,solver_plus_none_proxy,0.4275,0.6125,model_results/oof_group_resid_meta_on_oldnew_i...,model_results/testpred_group_resid_meta_on_old...,True,47.184929,5.635998,0.595325,0.0,58.235905,6.068409,81.902176,10.232918,27.232258,2.915229,group_resid_meta_on_oldnew_internal_oldw_0p025...
1,group_resid_meta_on_oldnew_internal_oldw_0p025...,tree_ensemble,none_only,0.0000,0.6125,model_results/oof_group_resid_meta_on_oldnew_i...,model_results/testpred_group_resid_meta_on_old...,True,48.349316,4.471611,0.595325,0.0,64.304314,0.000000,81.902176,10.232918,29.469204,0.678283,group_resid_meta_on_oldnew_internal_oldw_0p025...
2,hte40a3_residual_best,other,solver_plus_none_proxy,0.3425,0.9525,model_results/oof_hte40a3_residual_best.csv,model_results/testpred_hte40a3_residual_best.csv,True,49.512684,3.308243,0.595325,0.0,64.065193,0.239120,84.669449,7.465645,29.564486,0.583001,hte40a3_residual_best__solver_plus_none_proxy
3,hte40a3_residual_best,other,none_only,0.0000,0.9525,model_results/oof_hte40a3_residual_best.csv,model_results/testpred_hte40a3_residual_best.csv,True,49.558567,3.262360,0.595325,0.0,64.304314,0.000000,84.669449,7.465645,29.652631,0.494856,hte40a3_residual_best__none_only
4,stack40b_best_oof,other,solver_plus_none_proxy,0.9875,0.9900,model_results/oof_stack40b_best_oof.csv,model_results/testpred_stack40b_best_oof.csv,True,49.620861,3.200066,0.595325,0.0,64.070114,0.234200,84.914841,7.220253,29.582566,0.564921,stack40b_best_oof__solver_plus_none_proxy
5,stack40b_best_weighted,other,solver_plus_none_proxy,0.9875,0.9900,model_results/oof_stack40b_best_weighted.csv,model_results/testpred_stack40b_best_weighted.csv,True,49.620861,3.200066,0.595325,0.0,64.070114,0.234200,84.914841,7.220253,29.582566,0.564921,stack40b_best_weighted__solver_plus_none_proxy
6,stack40b_best_oof,other,none_only,0.0000,0.9900,model_results/oof_stack40b_best_oof.csv,model_results/testpred_stack40b_best_oof.csv,True,49.665798,3.155128,0.595325,0.0,64.304314,0.000000,84.914841,7.220253,29.668897,0.478590,stack40b_best_oof__none_only
7,stack40b_best_weighted,other,none_only,0.0000,0.9900,model_results/oof_stack40b_best_weighted.csv,model_results/testpred_stack40b_best_weighted.csv,True,49.665798,3.155128,0.595325,0.0,64.304314,0.000000,84.914841,7.220253,29.668897,0.478590,stack40b_best_weighted__none_only
8,liberal44d_best_oof,other,solver_plus_none_proxy,0.7350,0.8925,model_results/oof_liberal44d_best_oof.csv,model_results/testpred_liberal44d_best_oof.csv,True,49.758305,3.062622,0.595325,0.0,64.118851,0.185463,85.207962,6.927132,29.619961,0.527526,liberal44d_best_oof__solver_plus_none_proxy
9,meta43b2_best,tree_ensemble,solver_plus_none_proxy,0.9975,0.9975,model_results/oof_meta43b2_best.csv,model_results/testpred_meta43b2_best.csv,True,49.768604,3.052322,0.595325,0.0,63.268402,1.035912,85.604965,6.530128,29.332782,0.814705,meta43b2_best__solver_plus_none_proxy



Saved
-----
model_results/interp60a_fallback_candidate_screen.csv
model_results/interp60a_accounting_fallback_swap_screen.csv
submission_interp60a_1_group_resid_meta_on_oldnew_internal_oldw_0p025_neww_0p875_lambda_1p015__solver_plus_none_proxy.csv
submission_interp60a_2_group_resid_meta_on_oldnew_internal_oldw_0p025_neww_0p875_lambda_1p015__none_only.csv
submission_interp60a_3_hte40a3_residual_best__solver_plus_none_proxy.csv
submission_interp60a_4_hte40a3_residual_best__none_only.csv
submission_interp60a_5_stack40b_best_oof__solver_plus_none_proxy.csv
submission_interp60a_6_stack40b_best_weighted__solver_plus_none_proxy.csv
submission_interp60a_7_stack40b_best_oof__none_only.csv
submission_interp60a_8_stack40b_best_weighted__none_only.csv
submission_interp60a_9_liberal44d_best_oof__solver_plus_none_proxy.csv
submission_interp60a_10_meta43b2_best__solver_plus_none_proxy.csv

Interpretation guide
--------------------
Use this for the report / interpretable model search.
A good fallback

In [16]:
# ============================================================
# Standalone simple-fallback accounting solver helpers
# Paste this cell at the end of the notebook.
# ============================================================

import os
import glob
import gc
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

try:
    from scipy.optimize import lsq_linear
    HAVE_LSQ_LINEAR_SIMPLE_39B = True
except Exception:
    HAVE_LSQ_LINEAR_SIMPLE_39B = False

os.makedirs("model_results", exist_ok=True)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
N_SPLITS = 5
TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("Have scipy.optimize.lsq_linear:", HAVE_LSQ_LINEAR_SIMPLE_39B)


# ------------------------------------------------------------
# 1. Load raw score data from notebook globals or CSV files
# ------------------------------------------------------------

def _first_existing_path(patterns):
    hits = []
    for pat in patterns:
        hits.extend(glob.glob(pat))
    hits = sorted(set(hits))
    if not hits:
        return None
    return hits[0]


def _load_train_test_scores():
    # Prefer already-loaded notebook objects if present.
    train_candidates = [
        "raw_train_te",
        "raw_train",
        "train_raw",
        "scores_train",
        "scores_training",
        "train_scores",
        "train",
    ]
    test_candidates = [
        "raw_test_te",
        "raw_test",
        "test_raw",
        "scores_test",
        "test_scores",
        "test",
    ]

    train_df = None
    test_df = None

    for name in train_candidates:
        obj = globals().get(name, None)
        if isinstance(obj, pd.DataFrame):
            if {"SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"}.issubset(obj.columns):
                train_df = obj.copy()
                print(f"Using training rows from notebook variable: {name}")
                break

    for name in test_candidates:
        obj = globals().get(name, None)
        if isinstance(obj, pd.DataFrame):
            if {"SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"}.issubset(obj.columns):
                test_df = obj.copy()
                print(f"Using test rows from notebook variable: {name}")
                break

    # If globals were not found, read CSV files.
    if train_df is None:
        train_path = _first_existing_path([
            "scores_training.csv",
            "scores_training*.csv",
            "./scores_training.csv",
            "./scores_training*.csv",
            "/mnt/data/scores_training.csv",
            "/mnt/data/scores_training*.csv",
        ])
        if train_path is None:
            raise FileNotFoundError("Could not find scores_training CSV and no raw training DataFrame was found.")
        train_df = pd.read_csv(train_path)
        print(f"Loaded training rows from: {train_path}")

    if test_df is None:
        test_path = _first_existing_path([
            "scores_test.csv",
            "scores_test*.csv",
            "./scores_test.csv",
            "./scores_test*.csv",
            "/mnt/data/scores_test.csv",
            "/mnt/data/scores_test*.csv",
        ])
        if test_path is None:
            raise FileNotFoundError("Could not find scores_test CSV and no raw test DataFrame was found.")
        test_df = pd.read_csv(test_path)
        print(f"Loaded test rows from: {test_path}")

    # Add target if train_df came from a processed object without the target.
    if TARGET_COL not in train_df.columns:
        if "y_train" in globals():
            y = np.asarray(globals()["y_train"]).reshape(-1)
            if len(y) != len(train_df):
                raise ValueError("y_train exists but its length does not match the training DataFrame.")
            train_df[TARGET_COL] = y
            print("Added target from notebook variable: y_train")
        else:
            raise ValueError(f"Training DataFrame does not contain {TARGET_COL}, and y_train was not found.")

    required_train_cols = [ID_COL, "SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS", TARGET_COL]
    required_test_cols = [ID_COL, "SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME", "N_STUDENTS"]

    missing_train = [c for c in required_train_cols if c not in train_df.columns]
    missing_test = [c for c in required_test_cols if c not in test_df.columns]

    if missing_train:
        raise ValueError(f"Training data missing required columns: {missing_train}")
    if missing_test:
        raise ValueError(f"Test data missing required columns: {missing_test}")

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


raw_train_simple, raw_test_simple = _load_train_test_scores()

y_arr_simple = pd.to_numeric(raw_train_simple[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
if not np.isfinite(y_arr_simple).all():
    raise ValueError("Training target contains non-finite values.")

n_train_simple = len(raw_train_simple)
n_test_simple = len(raw_test_simple)

print("Training rows:", n_train_simple)
print("Test rows:", n_test_simple)


# ------------------------------------------------------------
# 2. Prediction file loaders
# ------------------------------------------------------------

def _load_oof_prediction(path, y_ref):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    df = pd.read_csv(path)

    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row mismatch for {path}: got {len(df)}, expected {len(y_ref)}")

    # Prefer explicit prediction columns.
    preferred_cols = ["pred_clipped", "prediction", "pred", "oof_pred", "pred_raw"]

    pred_col = None
    for c in preferred_cols:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            pred_col = c
            break

    # If no explicit prediction column exists, use TARGET_COL only if it is the only viable numeric column.
    if pred_col is None:
        numeric_cols = [
            c for c in df.columns
            if c not in ["row_index", ID_COL, "fold"]
            and pd.api.types.is_numeric_dtype(df[c])
        ]

        # Avoid accidentally picking the true target if other numeric columns exist.
        if TARGET_COL in numeric_cols and len(numeric_cols) > 1:
            numeric_cols = [c for c in numeric_cols if c != TARGET_COL]

        if not numeric_cols:
            raise ValueError(f"Could not find prediction column in {path}")

        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions in {path}, column {pred_col}")

    # If the OOF file includes the target, validate alignment.
    if TARGET_COL in df.columns and pred_col != TARGET_COL:
        y_file = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
        if np.isfinite(y_file).all():
            max_diff = float(np.max(np.abs(y_file - y_ref)))
            if max_diff > 1e-5:
                raise ValueError(f"OOF target mismatch for {path}; max difference = {max_diff}")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col


def _load_test_prediction(path, raw_test_df):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    df = pd.read_csv(path)

    if ID_COL in df.columns:
        if df[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in {path}")

        raw_ids = raw_test_df[ID_COL].to_numpy()

        # Reorder prediction file to raw_test order if necessary.
        if len(df) != len(raw_test_df):
            raise ValueError(f"Test row mismatch for {path}: got {len(df)}, expected {len(raw_test_df)}")

        df = df.set_index(ID_COL).loc[raw_ids].reset_index()

    else:
        if len(df) != len(raw_test_df):
            raise ValueError(f"Test row mismatch for {path}: got {len(df)}, expected {len(raw_test_df)}")
        df[ID_COL] = raw_test_df[ID_COL].to_numpy()

    if TARGET_COL in df.columns and pd.api.types.is_numeric_dtype(df[TARGET_COL]):
        pred_col = TARGET_COL
    else:
        numeric_cols = [
            c for c in df.columns
            if c != ID_COL and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not numeric_cols:
            raise ValueError(f"Could not find test prediction column in {path}")
        pred_col = numeric_cols[0]

    pred = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions in {path}, column {pred_col}")

    ids = df[ID_COL].to_numpy()

    return np.clip(pred, 0, 100).astype(np.float32), ids, pred_col


# ------------------------------------------------------------
# 3. Compact accounting frames
# ------------------------------------------------------------

def _clean_str(s):
    return pd.Series(s).astype("string").fillna("<NA>").astype(str)


def _safe_numeric(s):
    return (
        pd.to_numeric(s, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=np.float64)
    )


train_simple_39b = pd.DataFrame({
    "row_index": np.arange(n_train_simple),
    ID_COL: raw_train_simple[ID_COL].to_numpy(),
    "SCHOOL": _clean_str(raw_train_simple["SCHOOL"]),
    "ASSESSMENT_NAME": _clean_str(raw_train_simple["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": _clean_str(raw_train_simple["SUBGROUP_NAME"]),
    "N_STUDENTS": _safe_numeric(raw_train_simple["N_STUDENTS"]),
    TARGET_COL: y_arr_simple,
})

test_simple_39b = pd.DataFrame({
    "row_index": np.arange(n_test_simple),
    ID_COL: raw_test_simple[ID_COL].to_numpy(),
    "SCHOOL": _clean_str(raw_test_simple["SCHOOL"]),
    "ASSESSMENT_NAME": _clean_str(raw_test_simple["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": _clean_str(raw_test_simple["SUBGROUP_NAME"]),
    "N_STUDENTS": _safe_numeric(raw_test_simple["N_STUDENTS"]),
})

train_simple_39b["group_key"] = train_simple_39b["SCHOOL"] + "||" + train_simple_39b["ASSESSMENT_NAME"]
test_simple_39b["group_key"] = test_simple_39b["SCHOOL"] + "||" + test_simple_39b["ASSESSMENT_NAME"]

for df in [train_simple_39b, test_simple_39b]:
    df["N_STUDENTS"] = np.where(
        np.isfinite(df["N_STUDENTS"]) & (df["N_STUDENTS"] > 0),
        df["N_STUDENTS"],
        np.nan,
    )

train_simple_39b["prof_count"] = np.rint(
    np.clip(train_simple_39b[TARGET_COL].to_numpy(dtype=np.float64), 0, 100)
    / 100.0
    * train_simple_39b["N_STUDENTS"].to_numpy(dtype=np.float64)
)

train_simple_39b["prof_count"] = np.clip(
    train_simple_39b["prof_count"],
    0,
    train_simple_39b["N_STUDENTS"],
)

print("\nSubgroup counts:")
print(
    pd.concat([
        train_simple_39b["SUBGROUP_NAME"].value_counts().rename("train"),
        test_simple_39b["SUBGROUP_NAME"].value_counts().rename("test"),
    ], axis=1).fillna(0).astype(int).to_string()
)


# ------------------------------------------------------------
# 4. Accounting identities
# ------------------------------------------------------------

candidate_identities_simple_39b = [
    ("All Students", "Female", "Male"),
    ("All Students", "Economically Disadvantaged", "Not Economically Disadvantaged"),
]

subgroups_seen = set(train_simple_39b["SUBGROUP_NAME"]).union(set(test_simple_39b["SUBGROUP_NAME"]))

accounting_identities_simple_39b = [
    tup for tup in candidate_identities_simple_39b
    if all(s in subgroups_seen for s in tup)
]

if not accounting_identities_simple_39b:
    raise ValueError("No usable accounting identities found from SUBGROUP_NAME values.")

print("\nAccounting identities used:")
for all_s, a_s, b_s in accounting_identities_simple_39b:
    print(f"  {all_s} = {a_s} + {b_s}")


def _n_tolerance(n):
    if not np.isfinite(n):
        return 1.0
    return max(1.0, 0.02 * float(n))


# ------------------------------------------------------------
# 5. Known-count lookup and solver
# ------------------------------------------------------------

def build_known_lookup_simple_39b(df, row_indices):
    sub = df.iloc[row_indices]

    lookup = {}

    for group_key, g in sub.groupby("group_key", sort=False):
        group_dict = {}

        for subgroup, sg in g.groupby("SUBGROUP_NAME", sort=False):
            n_vals = sg["N_STUDENTS"].to_numpy(dtype=np.float64)
            k_vals = sg["prof_count"].to_numpy(dtype=np.float64)

            good = np.isfinite(n_vals) & (n_vals > 0) & np.isfinite(k_vals)

            if not good.any():
                continue

            n = float(np.mean(n_vals[good]))
            k = float(np.mean(k_vals[good]))

            group_dict[str(subgroup)] = {
                "n": n,
                "k": float(np.clip(k, 0, n)),
            }

        if group_dict:
            lookup[str(group_key)] = group_dict

    return lookup


def solve_one_group_simple_39b(
    query_group_df,
    known_group,
    base_prior_pct,
    identities,
    equation_weight,
    prior_weight,
):
    n_query = len(query_group_df)

    pred_pct = np.full(n_query, np.nan, dtype=np.float32)
    touched = np.zeros(n_query, dtype=bool)
    eq_count_used = 0

    subgroup_to_var = {}
    var_to_local = []
    var_n = []
    var_prior_k = []

    for j, row in query_group_df.iterrows():
        subgroup = str(row["SUBGROUP_NAME"])
        n = float(row["N_STUDENTS"])
        prior_pct = float(base_prior_pct[j])

        if not np.isfinite(n) or n <= 0 or not np.isfinite(prior_pct):
            continue

        if subgroup in subgroup_to_var:
            continue

        v = len(var_to_local)
        subgroup_to_var[subgroup] = v
        var_to_local.append(j)
        var_n.append(n)
        var_prior_k.append(float(np.clip(prior_pct, 0, 100) / 100.0 * n))

    n_vars = len(var_to_local)

    if n_vars == 0:
        return pred_pct, touched, eq_count_used

    def subgroup_present(s):
        return (s in subgroup_to_var) or (known_group is not None and s in known_group)

    def subgroup_n(s):
        if s in subgroup_to_var:
            return var_n[subgroup_to_var[s]]
        return known_group[s]["n"]

    def subgroup_known_k(s):
        return known_group[s]["k"]

    A_rows = []
    b_vals = []

    sqrt_prior = float(np.sqrt(prior_weight))
    sqrt_eq = float(np.sqrt(equation_weight))

    # Prior rows: keep unknown counts close to fallback predicted counts.
    for v in range(n_vars):
        row = np.zeros(n_vars, dtype=np.float64)
        row[v] = sqrt_prior
        A_rows.append(row)
        b_vals.append(sqrt_prior * var_prior_k[v])

    touched_vars = set()

    # Accounting equations.
    for all_s, a_s, b_s in identities:
        if not (subgroup_present(all_s) and subgroup_present(a_s) and subgroup_present(b_s)):
            continue

        n_all = subgroup_n(all_s)
        n_a = subgroup_n(a_s)
        n_b = subgroup_n(b_s)

        if not (np.isfinite(n_all) and np.isfinite(n_a) and np.isfinite(n_b)):
            continue

        # Require the subgroup sizes to be consistent enough to use the identity.
        if abs(n_all - (n_a + n_b)) > _n_tolerance(n_all):
            continue

        # x_all - x_a - x_b = 0
        signs = {
            all_s: 1.0,
            a_s: -1.0,
            b_s: -1.0,
        }

        row = np.zeros(n_vars, dtype=np.float64)
        known_sum = 0.0
        vars_in_eq = []

        for s, sign in signs.items():
            if s in subgroup_to_var:
                v = subgroup_to_var[s]
                row[v] += sign
                vars_in_eq.append(v)
            else:
                known_sum += sign * subgroup_known_k(s)

        if not vars_in_eq:
            continue

        A_rows.append(sqrt_eq * row)
        b_vals.append(sqrt_eq * (-known_sum))

        eq_count_used += 1
        touched_vars.update(vars_in_eq)

    if eq_count_used == 0 or not touched_vars:
        return pred_pct, touched, eq_count_used

    A = np.vstack(A_rows)
    b = np.asarray(b_vals, dtype=np.float64)

    lower = np.zeros(n_vars, dtype=np.float64)
    upper = np.asarray(var_n, dtype=np.float64)

    try:
        if HAVE_LSQ_LINEAR_SIMPLE_39B:
            sol = lsq_linear(
                A,
                b,
                bounds=(lower, upper),
                method="trf",
                lsmr_tol="auto",
                max_iter=100,
            )
            x = sol.x
        else:
            x, *_ = np.linalg.lstsq(A, b, rcond=None)
            x = np.clip(x, lower, upper)
    except Exception:
        x, *_ = np.linalg.lstsq(A, b, rcond=None)
        x = np.clip(x, lower, upper)

    for v in touched_vars:
        local_j = var_to_local[v]
        n = var_n[v]

        if np.isfinite(n) and n > 0:
            p = 100.0 * float(x[v]) / n
            pred_pct[local_j] = np.float32(np.clip(p, 0, 100))
            touched[local_j] = True

    return pred_pct, touched, eq_count_used


def solve_many_simple_39b(
    query_df,
    known_lookup,
    base_prior_pct,
    identities,
    equation_weight,
    prior_weight,
):
    query_df = query_df.reset_index(drop=True).copy()
    query_df["local_pos"] = np.arange(len(query_df))

    base_prior_pct = np.asarray(base_prior_pct, dtype=np.float32).reshape(-1)

    pred = np.full(len(query_df), np.nan, dtype=np.float32)
    touched = np.zeros(len(query_df), dtype=bool)
    eq_counts = np.zeros(len(query_df), dtype=np.float32)

    for group_key, g in query_df.groupby("group_key", sort=False):
        local_positions = g["local_pos"].to_numpy(dtype=np.int64)
        known_group = known_lookup.get(str(group_key), {})

        pred_g, touched_g, eq_count_g = solve_one_group_simple_39b(
            query_group_df=g.reset_index(drop=True),
            known_group=known_group,
            base_prior_pct=base_prior_pct[local_positions],
            identities=identities,
            equation_weight=equation_weight,
            prior_weight=prior_weight,
        )

        pred[local_positions] = pred_g
        touched[local_positions] = touched_g
        eq_counts[local_positions] = eq_count_g

    return pred, touched, eq_counts


# ------------------------------------------------------------
# 6. Main runner for one fallback candidate
# ------------------------------------------------------------

def run_simple_fallback_accounting_solver(
    name,
    oof_path,
    test_path,
    equation_weight_grid=(100.0, 1000.0, 10000.0, 100000.0),
    prior_weight=1.0,
    lambda_grid=None,
):
    if lambda_grid is None:
        lambda_grid = np.unique(
            np.concatenate([
                np.linspace(-0.50, 1.50, 501),
                np.array([0.0, 0.25, 0.50, 0.75, 0.924, 0.992, 1.0]),
            ])
        )

    print("\n" + "=" * 100)
    print(f"Running simple fallback accounting solver: {name}")
    print("=" * 100)

    base_oof, oof_col = _load_oof_prediction(oof_path, y_arr_simple)
    base_test, test_ids, test_col = _load_test_prediction(test_path, raw_test_simple)

    base_oof_mse = float(mean_squared_error(y_arr_simple, base_oof))

    print("OOF path:", oof_path)
    print("Test path:", test_path)
    print("OOF pred column:", oof_col)
    print("Test pred column:", test_col)
    print(f"Base-only OOF MSE: {base_oof_mse:.6f}")

    # Save base-only submission too.
    base_submission_path = f"submission_base_simple_{name}.csv"
    pd.DataFrame({
        ID_COL: test_ids,
        TARGET_COL: np.clip(base_test, 0, 100),
    }).to_csv(base_submission_path, index=False)

    folds = list(
        KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        .split(np.arange(n_train_simple))
    )

    screen_rows = []
    solver_oof_by_eq = {}
    touched_oof_by_eq = {}

    for eq_weight in equation_weight_grid:
        eq_weight = float(eq_weight)

        solver_oof = np.full(n_train_simple, np.nan, dtype=np.float32)
        touched_oof = np.zeros(n_train_simple, dtype=bool)

        print(f"\n  Equation weight = {eq_weight:g}")

        for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
            known_lookup_fold = build_known_lookup_simple_39b(train_simple_39b, tr_idx)
            query_fold = train_simple_39b.iloc[va_idx].reset_index(drop=True)
            base_prior_fold = base_oof[va_idx]

            pred_fold, touched_fold, eq_counts_fold = solve_many_simple_39b(
                query_df=query_fold,
                known_lookup=known_lookup_fold,
                base_prior_pct=base_prior_fold,
                identities=accounting_identities_simple_39b,
                equation_weight=eq_weight,
                prior_weight=prior_weight,
            )

            solver_oof[va_idx] = pred_fold
            touched_oof[va_idx] = touched_fold

            print(
                f"    fold {fold_num}: covered {int(touched_fold.sum())} / {len(va_idx)}"
            )

        solver_oof_by_eq[eq_weight] = solver_oof
        touched_oof_by_eq[eq_weight] = touched_oof

        covered = np.isfinite(solver_oof) & touched_oof
        print(f"  total OOF covered: {int(covered.sum())} / {n_train_simple}")

        if int(covered.sum()) == 0:
            continue

        for lam in lambda_grid:
            pred = base_oof.copy()
            pred[covered] = base_oof[covered] + float(lam) * (solver_oof[covered] - base_oof[covered])
            pred = np.clip(pred, 0, 100).astype(np.float32)

            mse = float(mean_squared_error(y_arr_simple, pred))

            screen_rows.append({
                "fallback_name": name,
                "equation_weight": eq_weight,
                "prior_weight": float(prior_weight),
                "lambda_solver": float(lam),
                "covered_rows": int(covered.sum()),
                "coverage_rate": float(covered.mean()),
                "base_oof_mse": base_oof_mse,
                "oof_mse": mse,
                "gain_vs_base": base_oof_mse - mse,
            })

    if not screen_rows:
        raise ValueError(f"No accounting solver rows covered for fallback {name}.")

    screen = pd.DataFrame(screen_rows).sort_values("oof_mse").reset_index(drop=True)
    best = screen.iloc[0].copy()

    best_eq = float(best["equation_weight"])
    best_lambda = float(best["lambda_solver"])

    best_solver_oof = solver_oof_by_eq[best_eq]
    best_touched_oof = touched_oof_by_eq[best_eq]
    best_covered_oof = np.isfinite(best_solver_oof) & best_touched_oof

    best_oof = base_oof.copy()
    best_oof[best_covered_oof] = (
        base_oof[best_covered_oof]
        + best_lambda * (best_solver_oof[best_covered_oof] - base_oof[best_covered_oof])
    )
    best_oof = np.clip(best_oof, 0, 100).astype(np.float32)

    best_oof_mse = float(mean_squared_error(y_arr_simple, best_oof))

    # Test solve uses all training rows as known rows.
    known_lookup_full = build_known_lookup_simple_39b(train_simple_39b, np.arange(n_train_simple))

    solver_test, touched_test, eq_counts_test = solve_many_simple_39b(
        query_df=test_simple_39b.reset_index(drop=True),
        known_lookup=known_lookup_full,
        base_prior_pct=base_test,
        identities=accounting_identities_simple_39b,
        equation_weight=best_eq,
        prior_weight=prior_weight,
    )

    covered_test = np.isfinite(solver_test) & touched_test

    best_test = base_test.copy()
    best_test[covered_test] = (
        base_test[covered_test]
        + best_lambda * (solver_test[covered_test] - base_test[covered_test])
    )
    best_test = np.clip(best_test, 0, 100).astype(np.float32)

    # Save artifacts.
    screen_path = f"model_results/account39b_simple_{name}_screen.csv"
    oof_path_out = f"model_results/oof_account39b_simple_{name}.csv"
    testpred_path_out = f"model_results/testpred_account39b_simple_{name}.csv"
    submission_path = f"submission_account39b_simple_{name}.csv"

    screen.to_csv(screen_path, index=False)

    pd.DataFrame({
        "row_index": np.arange(n_train_simple),
        TARGET_COL: y_arr_simple,
        "fallback_name": name,
        "base_pred": base_oof,
        "solver_pred": best_solver_oof,
        "solver_covered": best_covered_oof.astype(int),
        "equation_weight": best_eq,
        "prior_weight": prior_weight,
        "lambda_solver": best_lambda,
        "pred_clipped": best_oof,
    }).to_csv(oof_path_out, index=False)

    pd.DataFrame({
        ID_COL: test_ids,
        "fallback_name": name,
        "base_pred": base_test,
        "solver_pred": solver_test,
        "solver_covered": covered_test.astype(int),
        "equation_weight": best_eq,
        "prior_weight": prior_weight,
        "lambda_solver": best_lambda,
        TARGET_COL: best_test,
    }).to_csv(testpred_path_out, index=False)

    sub = pd.DataFrame({
        ID_COL: test_ids,
        TARGET_COL: best_test,
    })

    sub.to_csv(submission_path, index=False)

    # Validate submission.
    assert sub.shape == (n_test_simple, 2), sub.shape
    assert list(sub.columns) == [ID_COL, TARGET_COL], list(sub.columns)
    assert sub[ID_COL].notna().all()
    assert sub[TARGET_COL].notna().all()
    assert np.isfinite(sub[TARGET_COL]).all()
    assert sub[TARGET_COL].between(0, 100).all()

    summary = {
        "fallback_name": name,
        "base_oof_mse": base_oof_mse,
        "best_accounting_oof_mse": best_oof_mse,
        "gain_vs_base": base_oof_mse - best_oof_mse,
        "equation_weight": best_eq,
        "prior_weight": prior_weight,
        "lambda_solver": best_lambda,
        "oof_covered_rows": int(best_covered_oof.sum()),
        "oof_coverage_rate": float(best_covered_oof.mean()),
        "test_covered_rows": int(covered_test.sum()),
        "test_coverage_rate": float(covered_test.mean()),
        "base_submission_path": base_submission_path,
        "accounting_submission_path": submission_path,
        "screen_path": screen_path,
        "oof_path": oof_path_out,
        "testpred_path": testpred_path_out,
    }

    print("\nBest config for", name)
    print("----------------" + "-" * len(name))
    print(pd.Series(summary).to_string())

    print("\nTop 10 configs:")
    display(screen.head(10))

    print("\nSaved:")
    print(" ", base_submission_path)
    print(" ", submission_path)
    print(" ", screen_path)
    print(" ", oof_path_out)
    print(" ", testpred_path_out)

    gc.collect()

    return summary, screen

Have scipy.optimize.lsq_linear: True
Using training rows from notebook variable: raw_train_te
Using test rows from notebook variable: raw_test_te
Training rows: 144921
Test rows: 48307

Subgroup counts:
                                train   test
SUBGROUP_NAME                               
All Students                    36711  12428
Male                            29363   9589
Female                          29110   9777
Economically Disadvantaged      25137   8487
Not Economically Disadvantaged  24600   8026

Accounting identities used:
  All Students = Female + Male
  All Students = Economically Disadvantaged + Not Economically Disadvantaged


In [17]:
# ============================================================
# Run the three simple fallback accounting submissions
# ============================================================

simple_fallback_candidates = [
    {
        "name": "lgbm_te_28e",
        "oof_path": "model_results/oof_lgbm_te_base_5fold_oof_v1.csv",
        "test_path": "model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv",
    },
    {
        "name": "blend28g",
        "oof_path": "model_results/oof_blend_te_verified_noresid_weighted.csv",
        "test_path": "model_results/testpred_blend_te_verified_noresid_weighted.csv",
    },
    {
        "name": "blend31a",
        "oof_path": "model_results/oof_blend_overnight31_linear_bounded_te_weighted.csv",
        "test_path": "model_results/testpred_blend_overnight31_linear_bounded_te_weighted.csv",
    },
]

# Full screen. This tests the same equation-weight grid style as 39B.
EQUATION_WEIGHT_GRID = [100.0, 1000.0, 10000.0, 100000.0]

# For a faster run, replace the line above with:
# EQUATION_WEIGHT_GRID = [100000.0]

all_summaries = []
all_screens = []

for cand in simple_fallback_candidates:
    name = cand["name"]
    oof_path = cand["oof_path"]
    test_path = cand["test_path"]

    if not Path(oof_path).exists() or not Path(test_path).exists():
        print("\nSkipping missing candidate:", name)
        print("  OOF path exists:", Path(oof_path).exists(), "|", oof_path)
        print("  Test path exists:", Path(test_path).exists(), "|", test_path)
        continue

    summary, screen = run_simple_fallback_accounting_solver(
        name=name,
        oof_path=oof_path,
        test_path=test_path,
        equation_weight_grid=EQUATION_WEIGHT_GRID,
        prior_weight=1.0,
    )

    all_summaries.append(summary)
    all_screens.append(screen)

if not all_summaries:
    raise ValueError("No simple fallback candidates were run. Check that the model_results files exist.")

summary_df = pd.DataFrame(all_summaries).sort_values("best_accounting_oof_mse").reset_index(drop=True)
summary_path = "model_results/account39b_simple_fallback_comparison_summary.csv"
summary_df.to_csv(summary_path, index=False)

if all_screens:
    full_screen_df = pd.concat(all_screens, ignore_index=True)
    full_screen_df = full_screen_df.sort_values("oof_mse").reset_index(drop=True)
    full_screen_path = "model_results/account39b_simple_fallback_full_screen.csv"
    full_screen_df.to_csv(full_screen_path, index=False)
else:
    full_screen_path = None

print("\n" + "=" * 100)
print("Simple fallback accounting comparison complete")
print("=" * 100)

display(summary_df)

print("\nBest simple/component-wise-simple fallback:")
best_row = summary_df.iloc[0]
print(best_row.to_string())

print("\nAccounting-solver submissions created:")
for p in summary_df["accounting_submission_path"]:
    print(" ", p)

print("\nBase-only submissions also created:")
for p in summary_df["base_submission_path"]:
    print(" ", p)

print("\nSaved comparison files:")
print(" ", summary_path)
if full_screen_path is not None:
    print(" ", full_screen_path)


Running simple fallback accounting solver: lgbm_te_28e
OOF path: model_results/oof_lgbm_te_base_5fold_oof_v1.csv
Test path: model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv
OOF pred column: pred_clipped
Test pred column: PERCENT_PROFICIENT
Base-only OOF MSE: 82.907614

  Equation weight = 100
    fold 1: covered 16333 / 28985
    fold 2: covered 16245 / 28984
    fold 3: covered 16362 / 28984
    fold 4: covered 16367 / 28984
    fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

  Equation weight = 1000
    fold 1: covered 16333 / 28985
    fold 2: covered 16245 / 28984
    fold 3: covered 16362 / 28984
    fold 4: covered 16367 / 28984
    fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

  Equation weight = 10000
    fold 1: covered 16333 / 28985
    fold 2: covered 16245 / 28984
    fold 3: covered 16362 / 28984
    fold 4: covered 16367 / 28984
    fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

  Equation weight = 1

,fallback_name,equation_weight,prior_weight,lambda_solver,covered_rows,coverage_rate,base_oof_mse,oof_mse,gain_vs_base
0,lgbm_te_28e,100000.0,1.0,0.916,81593,0.563017,82.907614,58.908045,23.999568
1,lgbm_te_28e,100000.0,1.0,0.912,81593,0.563017,82.907614,58.908163,23.999450
2,lgbm_te_28e,10000.0,1.0,0.916,81593,0.563017,82.907614,58.908255,23.999359
3,lgbm_te_28e,10000.0,1.0,0.912,81593,0.563017,82.907614,58.908392,23.999222
4,lgbm_te_28e,100000.0,1.0,0.920,81593,0.563017,82.907614,58.908845,23.998768
5,lgbm_te_28e,10000.0,1.0,0.920,81593,0.563017,82.907614,58.909037,23.998577
6,lgbm_te_28e,100000.0,1.0,0.908,81593,0.563017,82.907614,58.909200,23.998414
7,lgbm_te_28e,10000.0,1.0,0.908,81593,0.563017,82.907614,58.909446,23.998167
8,lgbm_te_28e,1000.0,1.0,0.916,81593,0.563017,82.907614,58.910377,23.997236
9,lgbm_te_28e,100000.0,1.0,0.924,81593,0.563017,82.907614,58.910564,23.997050



Saved:
  submission_base_simple_lgbm_te_28e.csv
  submission_account39b_simple_lgbm_te_28e.csv
  model_results/account39b_simple_lgbm_te_28e_screen.csv
  model_results/oof_account39b_simple_lgbm_te_28e.csv
  model_results/testpred_account39b_simple_lgbm_te_28e.csv

Running simple fallback accounting solver: blend28g
OOF path: model_results/oof_blend_te_verified_noresid_weighted.csv
Test path: model_results/testpred_blend_te_verified_noresid_weighted.csv
OOF pred column: pred_clipped
Test pred column: PERCENT_PROFICIENT
Base-only OOF MSE: 78.071510

  Equation weight = 100
    fold 1: covered 16333 / 28985
    fold 2: covered 16245 / 28984
    fold 3: covered 16362 / 28984
    fold 4: covered 16367 / 28984
    fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

  Equation weight = 1000
    fold 1: covered 16333 / 28985
    fold 2: covered 16245 / 28984
    fold 3: covered 16362 / 28984
    fold 4: covered 16367 / 28984
    fold 5: covered 16286 / 28984
  total OOF covere

,fallback_name,equation_weight,prior_weight,lambda_solver,covered_rows,coverage_rate,base_oof_mse,oof_mse,gain_vs_base
0,blend28g,100000.0,1.0,0.932,81593,0.563017,78.07151,54.292981,23.778529
1,blend28g,10000.0,1.0,0.932,81593,0.563017,78.07151,54.293135,23.778376
2,blend28g,100000.0,1.0,0.928,81593,0.563017,78.07151,54.293156,23.778355
3,blend28g,10000.0,1.0,0.928,81593,0.563017,78.07151,54.293327,23.778183
4,blend28g,100000.0,1.0,0.936,81593,0.563017,78.07151,54.293685,23.777825
5,blend28g,10000.0,1.0,0.936,81593,0.563017,78.07151,54.293820,23.777690
6,blend28g,100000.0,1.0,0.924,81593,0.563017,78.07151,54.294208,23.777302
7,blend28g,100000.0,1.0,0.924,81593,0.563017,78.07151,54.294208,23.777302
8,blend28g,10000.0,1.0,0.924,81593,0.563017,78.07151,54.294398,23.777112
9,blend28g,10000.0,1.0,0.924,81593,0.563017,78.07151,54.294398,23.777112



Saved:
  submission_base_simple_blend28g.csv
  submission_account39b_simple_blend28g.csv
  model_results/account39b_simple_blend28g_screen.csv
  model_results/oof_account39b_simple_blend28g.csv
  model_results/testpred_account39b_simple_blend28g.csv

Running simple fallback accounting solver: blend31a
OOF path: model_results/oof_blend_overnight31_linear_bounded_te_weighted.csv
Test path: model_results/testpred_blend_overnight31_linear_bounded_te_weighted.csv
OOF pred column: pred_clipped
Test pred column: PERCENT_PROFICIENT
Base-only OOF MSE: 77.148170

  Equation weight = 100
    fold 1: covered 16333 / 28985
    fold 2: covered 16245 / 28984
    fold 3: covered 16362 / 28984
    fold 4: covered 16367 / 28984
    fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

  Equation weight = 1000
    fold 1: covered 16333 / 28985
    fold 2: covered 16245 / 28984
    fold 3: covered 16362 / 28984
    fold 4: covered 16367 / 28984
    fold 5: covered 16286 / 28984
  total OOF c

,fallback_name,equation_weight,prior_weight,lambda_solver,covered_rows,coverage_rate,base_oof_mse,oof_mse,gain_vs_base
0,blend31a,100000.0,1.0,0.932,81593,0.563017,77.14817,53.930163,23.218007
1,blend31a,10000.0,1.0,0.932,81593,0.563017,77.14817,53.930315,23.217855
2,blend31a,100000.0,1.0,0.928,81593,0.563017,77.14817,53.930418,23.217752
3,blend31a,10000.0,1.0,0.928,81593,0.563017,77.14817,53.930588,23.217582
4,blend31a,100000.0,1.0,0.936,81593,0.563017,77.14817,53.930765,23.217406
5,blend31a,10000.0,1.0,0.936,81593,0.563017,77.14817,53.930899,23.217271
6,blend31a,100000.0,1.0,0.924,81593,0.563017,77.14817,53.931530,23.216640
7,blend31a,100000.0,1.0,0.924,81593,0.563017,77.14817,53.931530,23.216640
8,blend31a,10000.0,1.0,0.924,81593,0.563017,77.14817,53.931718,23.216452
9,blend31a,10000.0,1.0,0.924,81593,0.563017,77.14817,53.931718,23.216452



Saved:
  submission_base_simple_blend31a.csv
  submission_account39b_simple_blend31a.csv
  model_results/account39b_simple_blend31a_screen.csv
  model_results/oof_account39b_simple_blend31a.csv
  model_results/testpred_account39b_simple_blend31a.csv

Simple fallback accounting comparison complete


,fallback_name,base_oof_mse,best_accounting_oof_mse,gain_vs_base,equation_weight,prior_weight,lambda_solver,oof_covered_rows,oof_coverage_rate,test_covered_rows,test_coverage_rate,base_submission_path,accounting_submission_path,screen_path,oof_path,testpred_path
0,blend31a,77.148170,53.930163,23.218007,100000.0,1.0,0.932,81593,0.563017,45105,0.933716,submission_base_simple_blend31a.csv,submission_account39b_simple_blend31a.csv,model_results/account39b_simple_blend31a_scree...,model_results/oof_account39b_simple_blend31a.csv,model_results/testpred_account39b_simple_blend...
1,blend28g,78.071510,54.292981,23.778529,100000.0,1.0,0.932,81593,0.563017,45105,0.933716,submission_base_simple_blend28g.csv,submission_account39b_simple_blend28g.csv,model_results/account39b_simple_blend28g_scree...,model_results/oof_account39b_simple_blend28g.csv,model_results/testpred_account39b_simple_blend...
2,lgbm_te_28e,82.907614,58.908045,23.999568,100000.0,1.0,0.916,81593,0.563017,45105,0.933716,submission_base_simple_lgbm_te_28e.csv,submission_account39b_simple_lgbm_te_28e.csv,model_results/account39b_simple_lgbm_te_28e_sc...,model_results/oof_account39b_simple_lgbm_te_28...,model_results/testpred_account39b_simple_lgbm_...



Best simple/component-wise-simple fallback:
fallback_name                                                          blend31a
base_oof_mse                                                           77.14817
best_accounting_oof_mse                                               53.930163
gain_vs_base                                                          23.218007
equation_weight                                                        100000.0
prior_weight                                                                1.0
lambda_solver                                                             0.932
oof_covered_rows                                                          81593
oof_coverage_rate                                                      0.563017
test_covered_rows                                                         45105
test_coverage_rate                                                     0.933716
base_submission_path                        submission_base_simple_blend31a

In [18]:
# ============================================================
# 60B. Create interpretable stack-ridge fallback submission
# ============================================================
#
# Model:
#   A_i = account39d_best accounting/solver prediction
#   R_i = stack_ridge_components prediction
#
#   if tier_i = overlap:
#       pred_i = A_i
#
#   if tier_i = solver_only:
#       pred_i = 0.9000 * A_i + 0.1000 * R_i
#
#   if tier_i = none:
#       pred_i = 0.8225 * A_i + 0.1775 * R_i
#
# Output:
#   submission_interp60b_account39d_stack_ridge_components_solver_plus_none.csv
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

print("=" * 90)
print("60B. Stack-ridge fallback to account39d accounting solver")
print("=" * 90)

# ------------------------------------------------------------
# 1. Files
# ------------------------------------------------------------

required_files = [
    "model_results/oof_account39d_best.csv",
    "model_results/testpred_account39d_best.csv",
    "model_results/oof_stack_ridge_oof_components.csv",
    "model_results/testpred_stack_ridge_oof_components.csv",
    "model_results/account39a_raw_oof_reconstruction.csv",
    "model_results/account39a_raw_test_reconstruction.csv",
    "model_results/account39b_solver_raw_oof.csv",
    "model_results/account39b_solver_raw_test.csv",
]

print("\nFile checks")
print("-----------")
for p in required_files:
    print(f"{p:65s}", Path(p).exists())

missing = [p for p in required_files if not Path(p).exists()]
if missing:
    raise RuntimeError(f"Missing required files: {missing}")

# ------------------------------------------------------------
# 2. Load helpers
# ------------------------------------------------------------

def find_pred_col(df):
    preferred = [
        "pred_clipped",
        "prediction",
        "pred",
        "oof_pred",
        "OOF_PRED",
        "PRED",
    ]

    for c in preferred:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            return c

    numeric_cols = [
        c for c in df.columns
        if c not in ["row_index", ID_COL, TARGET_COL, "fold"]
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    pred_like = [c for c in numeric_cols if "pred" in c.lower() or "clip" in c.lower()]
    if pred_like:
        return pred_like[0]

    if len(numeric_cols) == 1:
        return numeric_cols[0]

    return None


def load_oof(path):
    df = pd.read_csv(path)

    if "row_index" in df.columns:
        df = df.sort_values("row_index").reset_index(drop=True)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No target column in {path}")

    pred_col = find_pred_col(df)
    if pred_col is None:
        raise ValueError(f"No prediction column found in {path}")

    y = pd.to_numeric(df[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)
    p = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    return df, y.astype(np.float32), np.clip(p, 0, 100).astype(np.float32), pred_col


def load_test(path):
    df = pd.read_csv(path)

    if TARGET_COL in df.columns:
        pred_col = TARGET_COL
    else:
        pred_col = find_pred_col(df)

    if pred_col is None:
        raise ValueError(f"No prediction column found in {path}")

    p = pd.to_numeric(df[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if ID_COL in df.columns:
        ids = df[ID_COL].to_numpy()
    elif "test_ids" in globals():
        ids = np.asarray(test_ids)
    else:
        raise ValueError(f"No {ID_COL} column in {path} and no test_ids in memory")

    return df, ids, np.clip(p, 0, 100).astype(np.float32), pred_col

# ------------------------------------------------------------
# 3. Load account39d and stack-ridge fallback
# ------------------------------------------------------------

_, y_acc, pred_acc_oof, acc_oof_col = load_oof("model_results/oof_account39d_best.csv")
_, test_ids_60b, pred_acc_test, acc_test_col = load_test("model_results/testpred_account39d_best.csv")

_, y_ridge, pred_ridge_oof, ridge_oof_col = load_oof("model_results/oof_stack_ridge_oof_components.csv")
_, test_ids_ridge, pred_ridge_test, ridge_test_col = load_test("model_results/testpred_stack_ridge_oof_components.csv")

assert len(y_acc) == len(pred_acc_oof)
assert len(y_ridge) == len(pred_ridge_oof)
assert len(pred_acc_oof) == len(pred_ridge_oof)
assert len(pred_acc_test) == len(pred_ridge_test)
assert np.max(np.abs(y_acc - y_ridge)) < 1e-5, "OOF target mismatch"
assert np.array_equal(test_ids_60b, test_ids_ridge), "test ID/order mismatch"

y = y_acc
n_train = len(y)
n_test = len(pred_acc_test)

print("\nLoaded predictions")
print("------------------")
print("account39d OOF pred col:", acc_oof_col)
print("account39d test pred col:", acc_test_col)
print("stack-ridge OOF pred col:", ridge_oof_col)
print("stack-ridge test pred col:", ridge_test_col)
print("n_train:", n_train)
print("n_test: ", n_test)

# ------------------------------------------------------------
# 4. Reconstruct accounting tiers
# ------------------------------------------------------------

raw39a_oof = pd.read_csv("model_results/account39a_raw_oof_reconstruction.csv")
raw39a_test = pd.read_csv("model_results/account39a_raw_test_reconstruction.csv")
raw39b_oof = pd.read_csv("model_results/account39b_solver_raw_oof.csv")
raw39b_test = pd.read_csv("model_results/account39b_solver_raw_test.csv")

if "row_index" in raw39a_oof.columns:
    raw39a_oof = raw39a_oof.sort_values("row_index").reset_index(drop=True)
if "row_index" in raw39b_oof.columns:
    raw39b_oof = raw39b_oof.sort_values("row_index").reset_index(drop=True)

direct_oof = raw39a_oof["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_oof = raw39b_oof["solver_covered"].astype(int).to_numpy().astype(bool)

direct_test = raw39a_test["accounting_covered"].astype(int).to_numpy().astype(bool)
solver_test = raw39b_test["solver_covered"].astype(int).to_numpy().astype(bool)

assert len(direct_oof) == n_train
assert len(solver_oof) == n_train
assert len(direct_test) == n_test
assert len(solver_test) == n_test

overlap_oof = direct_oof & solver_oof
solver_only_oof = solver_oof & ~direct_oof
none_oof = ~(direct_oof | solver_oof)

overlap_test = direct_test & solver_test
solver_only_test = solver_test & ~direct_test
none_test = ~(direct_test | solver_test)

print("\nTier coverage")
print("-------------")
for name, tr_mask, te_mask in [
    ("overlap", overlap_oof, overlap_test),
    ("solver_only", solver_only_oof, solver_only_test),
    ("none", none_oof, none_test),
]:
    print(f"{name:12s} train={int(tr_mask.sum()):6d} test={int(te_mask.sum()):6d}")

# ------------------------------------------------------------
# 5. Apply fixed 60A stack-ridge fallback lambdas
# ------------------------------------------------------------

lambda_solver = 0.1000
lambda_none = 0.1775

pred_60b_oof = pred_acc_oof.copy().astype(np.float64)
pred_60b_test = pred_acc_test.copy().astype(np.float64)

# overlap stays exactly account39d
pred_60b_oof[solver_only_oof] = (
    (1.0 - lambda_solver) * pred_acc_oof[solver_only_oof]
    + lambda_solver * pred_ridge_oof[solver_only_oof]
)
pred_60b_test[solver_only_test] = (
    (1.0 - lambda_solver) * pred_acc_test[solver_only_test]
    + lambda_solver * pred_ridge_test[solver_only_test]
)

pred_60b_oof[none_oof] = (
    (1.0 - lambda_none) * pred_acc_oof[none_oof]
    + lambda_none * pred_ridge_oof[none_oof]
)
pred_60b_test[none_test] = (
    (1.0 - lambda_none) * pred_acc_test[none_test]
    + lambda_none * pred_ridge_test[none_test]
)

pred_60b_oof = np.clip(pred_60b_oof, 0, 100).astype(np.float32)
pred_60b_test = np.clip(pred_60b_test, 0, 100).astype(np.float32)

# ------------------------------------------------------------
# 6. Diagnostics
# ------------------------------------------------------------

def mse(mask, pred):
    return float(mean_squared_error(y[mask], pred[mask]))

base_mse = float(mean_squared_error(y, pred_acc_oof))
new_mse = float(mean_squared_error(y, pred_60b_oof))

diag_rows = []

for tier, mask in [
    ("overlap", overlap_oof),
    ("solver_only", solver_only_oof),
    ("none", none_oof),
]:
    diag_rows.append({
        "tier": tier,
        "n_train": int(mask.sum()),
        "account39d_mse": mse(mask, pred_acc_oof),
        "stack_ridge_mse": mse(mask, pred_ridge_oof),
        "new_60b_mse": mse(mask, pred_60b_oof),
        "gain_vs_account39d": mse(mask, pred_acc_oof) - mse(mask, pred_60b_oof),
    })

diag = pd.DataFrame(diag_rows)

print("\nOOF diagnostics")
print("---------------")
print(f"account39d OOF MSE: {base_mse:.6f}")
print(f"60B OOF MSE:        {new_mse:.6f}")
print(f"gain:               {base_mse - new_mse:.6f}")
display(diag)

print("\nTest prediction shift vs account39d")
print("-----------------------------------")
abs_diff = np.abs(pred_60b_test - pred_acc_test)
print("mean_abs_diff:", float(abs_diff.mean()))
print("p95_abs_diff: ", float(np.percentile(abs_diff, 95)))
print("max_abs_diff: ", float(abs_diff.max()))

# ------------------------------------------------------------
# 7. Save OOF/test/submission
# ------------------------------------------------------------

tier_oof = np.array(["none"] * n_train, dtype=object)
tier_oof[solver_only_oof] = "solver_only"
tier_oof[overlap_oof] = "overlap"

tier_test = np.array(["none"] * n_test, dtype=object)
tier_test[solver_only_test] = "solver_only"
tier_test[overlap_test] = "overlap"

oof_path = "model_results/oof_interp60b_account39d_stack_ridge_components_solver_plus_none.csv"
testpred_path = "model_results/testpred_interp60b_account39d_stack_ridge_components_solver_plus_none.csv"
submission_path = "submission_interp60b_account39d_stack_ridge_components_solver_plus_none.csv"
diag_path = "model_results/interp60b_account39d_stack_ridge_components_diag.csv"

pd.DataFrame({
    "row_index": np.arange(n_train),
    TARGET_COL: y,
    "pred_account39d": pred_acc_oof,
    "pred_stack_ridge_components": pred_ridge_oof,
    "pred_clipped": pred_60b_oof,
    "tier": tier_oof,
}).to_csv(oof_path, index=False)

pd.DataFrame({
    ID_COL: test_ids_60b,
    "pred_account39d": pred_acc_test,
    "pred_stack_ridge_components": pred_ridge_test,
    TARGET_COL: pred_60b_test,
    "tier": tier_test,
}).to_csv(testpred_path, index=False)

pd.DataFrame({
    ID_COL: test_ids_60b,
    TARGET_COL: pred_60b_test,
}).to_csv(submission_path, index=False)

diag.to_csv(diag_path, index=False)

# Validate submission.
sub = pd.read_csv(submission_path)
assert sub.shape == (n_test, 2), sub.shape
assert list(sub.columns) == [ID_COL, TARGET_COL], sub.columns.tolist()
assert sub[ID_COL].notna().all()
assert sub[TARGET_COL].notna().all()
assert np.isfinite(sub[TARGET_COL]).all()
assert sub[TARGET_COL].between(0, 100).all()

print("\nSaved")
print("-----")
print(oof_path)
print(testpred_path)
print(diag_path)
print(submission_path)

print("\nClosed-form model")
print("-----------------")
print("overlap:      account39d")
print("solver_only:  0.9000 * account39d + 0.1000 * stack_ridge_components")
print("none:         0.8225 * account39d + 0.1775 * stack_ridge_components")

60B. Stack-ridge fallback to account39d accounting solver

File checks
-----------
model_results/oof_account39d_best.csv                             True
model_results/testpred_account39d_best.csv                        True
model_results/oof_stack_ridge_oof_components.csv                  True
model_results/testpred_stack_ridge_oof_components.csv             True
model_results/account39a_raw_oof_reconstruction.csv               True
model_results/account39a_raw_test_reconstruction.csv              True
model_results/account39b_solver_raw_oof.csv                       True
model_results/account39b_solver_raw_test.csv                      True

Loaded predictions
------------------
account39d OOF pred col: pred_clipped
account39d test pred col: PERCENT_PROFICIENT
stack-ridge OOF pred col: OOF_stack_ridge_oof_components
stack-ridge test pred col: TESTPRED_stack_ridge_oof_components
n_train: 144921
n_test:  48307

Tier coverage
-------------
overlap      train= 53786 test= 27298
solver_on

,tier,n_train,account39d_mse,stack_ridge_mse,new_60b_mse,gain_vs_account39d
0,overlap,53786,0.595325,70.803017,0.595325,0.000000
1,solver_only,27807,64.304314,86.821724,64.021759,0.282555
2,none,63328,92.135094,103.069366,91.595741,0.539352



Test prediction shift vs account39d
-----------------------------------
mean_abs_diff: 0.16755472123622894
p95_abs_diff:  0.8156028985977173
max_abs_diff:  5.802070617675781

Saved
-----
model_results/oof_interp60b_account39d_stack_ridge_components_solver_plus_none.csv
model_results/testpred_interp60b_account39d_stack_ridge_components_solver_plus_none.csv
model_results/interp60b_account39d_stack_ridge_components_diag.csv
submission_interp60b_account39d_stack_ridge_components_solver_plus_none.csv

Closed-form model
-----------------
overlap:      account39d
solver_only:  0.9000 * account39d + 0.1000 * stack_ridge_components
none:         0.8225 * account39d + 0.1775 * stack_ridge_components


The original notebook cell that created stack_ridge_oof_components is: # 22A. Post-80.662: update tracker + refine/stack saved OOF artifacts